In [2]:
# ==========================================
# CELL 1: Setup and Functions
# ==========================================
nvars = 2 # number of variables
BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"] # polynomial ring in x1, ..., xnvars, z
# BR is a global variable
BR.inject_variables() # make it so you can use those variables
SymmetricFunctions(QQ).inject_shorthands(verbose=False) # define the bases of symmetric functions

def do_P_k(k, n):
    # we are computing p_k[s_n] = s_n[p_k]
    # evaluate at the generators gens = BR.gens()[:-1] = (x1, x2, x3, x4)
    global BR, nvars
    CR = s[1].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR.gens()[i] : BR('x'+str(i+1))**k for i in range(nvars)})

@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = BR(expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1))-BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars))))
    return sum(c * mul(BR('x'+str(i+1))^(v[i]-(nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))

"""
This is the denominator that you are trying to determine
"""
@cached_function
def den_guess():
    global BR
    m1 = 0#z*x1**5
    m2 = 0#z*x1**4*x2
    m3 = 0#z**2*x1**5*x2**5
    m4 = 0#z**8*x1**20*x2**20
    m5 = 0#z**4*x1**12*x2**4*x3**4
    m6 = 0#z**2*x1**4*x2**4*x3**2
    m7 = 0#z**3*x1**5*x2**5*x3**5
    m8 = 0#-z**6*x1**10*x2**10*x3**10
    
    return BR((1-m1)*(1-m2)*(1-m3)*(1-m4)*(1-m5)*(1-m6)*(1-m7)*(1-m8))

# Global cache to expand the denominator exactly once
EXPANDED_DENOMINATOR = None

def get_den_expanded():
    global EXPANDED_DENOMINATOR
    if EXPANDED_DENOMINATOR is None:
        EXPANDED_DENOMINATOR = den_guess()
    return EXPANDED_DENOMINATOR

@cached_function
def den_coeff(d):
    global BR, z
    # Extremely fast native coefficient extraction (bypassing SR completely)
    return get_den_expanded().coefficient({z: d})

def calc_num(la, d):
    return sum(den_coeff(d-r) * do_P_lambda(Partition(la), r) for r in range(d+1))

Defining x1, x2, x3, x4, x5, z


In [4]:
# ==========================================
# CELL 2: Execution Loop
# ==========================================
out = 0

print("Warming up native denominator expansion... (takes a few seconds)")
get_den_expanded()
print("Expansion complete! Starting degrees...\n")

for d in range(0, 45):
    CC = calc_num([4,1], d)
    if CC:
        CC_list = list(CC)
        if len(CC_list) > 6:
            front = CC_list[:3]
            back = CC_list[-3:]
            print(d, len(CC_list), "FRONT:", front, "BACK:", back)
        else:
            print(d, len(CC_list), CC_list)
    else:
        print(d, 0, "[]")
    
    out += z**d * CC

Warming up native denominator expansion... (takes a few seconds)
Expansion complete! Starting degrees...

0 1 [(-1, 1)]
1 5 [(1, x1^4*x2), (1, x1^3*x2^2), (-1, x1^3*x2*x3), (-1, x1^2*x2^2*x3), (1, x1^2*x2*x3*x4)]
2 14 FRONT: [(-1, x1^8*x2^2), (-1, x1^6*x2^4), (-1, x1^5*x2^5)] BACK: [(-1, x1^3*x2^3*x3^3*x4), (-1, x1^4*x2^2*x3^2*x4^2), (1, x1^3*x2^3*x3^2*x4^2)]
3 33 FRONT: [(1, x1^10*x2^5), (1, x1^9*x2^6), (1, x1^8*x2^7)] BACK: [(1, x1^6*x2^3*x3^3*x4^3), (-1, x1^5*x2^4*x3^3*x4^3), (1, x1^4*x2^4*x3^4*x4^3)]
4 62 FRONT: [(-1, x1^13*x2^7), (-1, x1^12*x2^8), (-1, x1^10*x2^10)] BACK: [(2, x1^7*x2^5*x3^4*x4^4), (-1, x1^6*x2^6*x3^4*x4^4), (-2, x1^6*x2^5*x3^5*x4^4)]
5 90 FRONT: [(1, x1^15*x2^10), (1, x1^14*x2^11), (-1, x1^16*x2^8*x3)] BACK: [(2, x1^8*x2^7*x3^5*x4^5), (2, x1^8*x2^6*x3^6*x4^5), (-2, x1^7*x2^7*x3^6*x4^5)]
6 143 FRONT: [(-1, x1^18*x2^12), (1, x1^18*x2^11*x3), (1, x1^17*x2^12*x3)] BACK: [(-2, x1^10*x2^7*x3^7*x4^6), (3, x1^9*x2^8*x3^7*x4^6), (-1, x1^8*x2^8*x3^8*x4^6)]
7 213 FRONT: [(-

KeyboardInterrupt: 

In [18]:
factor(den_guess())

(x1^2*x2*x3*x4*z - 1) * (x1^2*x2^2*x3*z - 1) * (x1^2*x2^2*x3*z + 1) * (x1^3*x2*x3*z + 1) * (x1^4*x2*z - 1) * (x1^5*z - 1) * (x1^3*x2^3*x3^2*x4^2*z^2 - 1) * (x1^4*x2^2*x3^2*x4^2*z^2 + 1) * (x1^6*x2^2*x3^2*z^2 + 1) * (x1^5*x2^5*z^2 + 1) * (x1^4*x2^4*x3^4*x4^3*z^3 - 1) * (x1^5*x2^5*x3^5*z^3 - 1) * (x1^5*x2^5*x3^5*x4^5*z^4 + 1) * (x1^6*x2^6*x3^4*x4^4*z^4 + 1) * (x1^10*x2^10*z^4 + 1) * (x1^10*x2^10*x3^10*z^6 + 1)

In [16]:
factor(out)

(-1) * (x1^68*x2^53*x3^34*x4^15*z^34 - x1^66*x2^52*x3^33*x4^14*z^33 + x1^66*x2^51*x3^33*x4^15*z^33 + x1^65*x2^52*x3^33*x4^15*z^33 - x1^65*x2^51*x3^34*x4^15*z^33 - x1^64*x2^52*x3^34*x4^15*z^33 - x1^65*x2^50*x3^32*x4^13*z^32 + x1^64*x2^51*x3^32*x4^13*z^32 + x1^65*x2^50*x3^31*x4^14*z^32 - x1^64*x2^50*x3^32*x4^14*z^32 - x1^63*x2^51*x3^32*x4^14*z^32 + 2*x1^62*x2^51*x3^33*x4^14*z^32 - x1^64*x2^49*x3^32*x4^15*z^32 + x1^63*x2^50*x3^32*x4^15*z^32 + x1^62*x2^51*x3^32*x4^15*z^32 - 2*x1^62*x2^50*x3^33*x4^15*z^32 - 2*x1^61*x2^51*x3^33*x4^15*z^32 + x1^63*x2^48*x3^34*x4^15*z^32 + x1^62*x2^49*x3^34*x4^15*z^32 + x1^60*x2^51*x3^34*x4^15*z^32 - x1^64*x2^49*x3^30*x4^12*z^31 + x1^63*x2^49*x3^31*x4^12*z^31 - x1^62*x2^50*x3^31*x4^12*z^31 - x1^63*x2^48*x3^31*x4^13*z^31 - x1^62*x2^49*x3^31*x4^13*z^31 + 2*x1^61*x2^50*x3^31*x4^13*z^31 + x1^62*x2^48*x3^32*x4^13*z^31 + x1^61*x2^49*x3^32*x4^13*z^31 - 2*x1^60*x2^50*x3^32*x4^13*z^31 + x1^62*x2^49*x3^30*x4^14*z^31 + x1^62*x2^48*x3^31*x4^14*z^31 - 2*x1^61*x2^49*x3^31*x

In [54]:
factor(out)

(-1) * (x1^82*x2^65*x3^41*x4^17*z^41 + x1^80*x2^64*x3^40*x4^16*z^40 - 2*x1^79*x2^63*x3^41*x4^17*z^40 + x1^78*x2^63*x3^39*x4^15*z^39 - 2*x1^79*x2^62*x3^38*x4^16*z^39 + x1^78*x2^62*x3^39*x4^16*z^39 - x1^77*x2^63*x3^39*x4^16*z^39 - x1^77*x2^62*x3^40*x4^16*z^39 - x1^76*x2^63*x3^40*x4^16*z^39 + x1^78*x2^61*x3^39*x4^17*z^39 - x1^76*x2^63*x3^39*x4^17*z^39 - x1^77*x2^61*x3^40*x4^17*z^39 + x1^76*x2^62*x3^40*x4^17*z^39 + x1^76*x2^61*x3^41*x4^17*z^39 + x1^75*x2^62*x3^41*x4^17*z^39 + x1^74*x2^63*x3^41*x4^17*z^39 - x1^77*x2^61*x3^38*x4^14*z^38 + x1^76*x2^62*x3^38*x4^14*z^38 - x1^77*x2^61*x3^37*x4^15*z^38 - x1^75*x2^61*x3^39*x4^15*z^38 - x1^74*x2^62*x3^39*x4^15*z^38 - x1^77*x2^60*x3^37*x4^16*z^38 + 3*x1^76*x2^60*x3^38*x4^16*z^38 + x1^75*x2^61*x3^38*x4^16*z^38 + x1^74*x2^62*x3^38*x4^16*z^38 - x1^76*x2^59*x3^39*x4^16*z^38 - x1^75*x2^60*x3^39*x4^16*z^38 + x1^74*x2^61*x3^39*x4^16*z^38 - x1^73*x2^62*x3^39*x4^16*z^38 + x1^75*x2^59*x3^40*x4^16*z^38 - x1^74*x2^60*x3^40*x4^16*z^38 + x1^73*x2^61*x3^40*x4^16*z

In [ ]:
((-1) * (x1^2*x2*x3*x4*z - 1)^2 * (x1^82*x2^65*x3^41*x4^17*z^41 + x1^80*x2^64*x3^40*x4^16*z^40 - 2*x1^79*x2^63*x3^41*x4^17*z^40 + x1^78*x2^63*x3^39*x4^15*z^39 - 2*x1^79*x2^62*x3^38*x4^16*z^39 + x1^78*x2^62*x3^39*x4^16*z^39 - x1^77*x2^63*x3^39*x4^16*z^39 - x1^77*x2^62*x3^40*x4^16*z^39 - x1^76*x2^63*x3^40*x4^16*z^39 + x1^78*x2^61*x3^39*x4^17*z^39 - x1^76*x2^63*x3^39*x4^17*z^39 - x1^77*x2^61*x3^40*x4^17*z^39 + x1^76*x2^62*x3^40*x4^17*z^39 + x1^76*x2^61*x3^41*x4^17*z^39 + x1^75*x2^62*x3^41*x4^17*z^39 + x1^74*x2^63*x3^41*x4^17*z^39 - x1^77*x2^61*x3^38*x4^14*z^38 + x1^76*x2^62*x3^38*x4^14*z^38 - x1^77*x2^61*x3^37*x4^15*z^38 - x1^75*x2^61*x3^39*x4^15*z^38 - x1^74*x2^62*x3^39*x4^15*z^38 - x1^77*x2^60*x3^37*x4^16*z^38 + 3*x1^76*x2^60*x3^38*x4^16*z^38 + x1^75*x2^61*x3^38*x4^16*z^38 + x1^74*x2^62*x3^38*x4^16*z^38 - x1^76*x2^59*x3^39*x4^16*z^38 - x1^75*x2^60*x3^39*x4^16*z^38 + x1^74*x2^61*x3^39*x4^16*z^38 - x1^73*x2^62*x3^39*x4^16*z^38 + x1^75*x2^59*x3^40*x4^16*z^38 - x1^74*x2^60*x3^40*x4^16*z^38 + x1^73*x2^61*x3^40*x4^16*z^38 - x1^76*x2^60*x3^37*x4^17*z^38 + x1^75*x2^61*x3^37*x4^17*z^38 + x1^76*x2^59*x3^38*x4^17*z^38 + x1^75*x2^60*x3^38*x4^17*z^38 - 2*x1^74*x2^61*x3^38*x4^17*z^38 - x1^75*x2^59*x3^39*x4^17*z^38 - x1^74*x2^60*x3^39*x4^17*z^38 + x1^72*x2^62*x3^39*x4^17*z^38 + x1^75*x2^58*x3^40*x4^17*z^38 + x1^74*x2^59*x3^40*x4^17*z^38 - x1^73*x2^60*x3^40*x4^17*z^38 + x1^72*x2^61*x3^40*x4^17*z^38 - x1^74*x2^58*x3^41*x4^17*z^38 + x1^73*x2^59*x3^41*x4^17*z^38 - x1^72*x2^60*x3^41*x4^17*z^38 - x1^71*x2^61*x3^41*x4^17*z^38 - x1^76*x2^60*x3^36*x4^13*z^37 - x1^76*x2^59*x3^37*x4^13*z^37 + x1^74*x2^61*x3^37*x4^13*z^37 + x1^76*x2^59*x3^36*x4^14*z^37 - x1^75*x2^60*x3^36*x4^14*z^37 + x1^75*x2^59*x3^37*x4^14*z^37 - x1^74*x2^60*x3^37*x4^14*z^37 - x1^73*x2^61*x3^37*x4^14*z^37 + x1^75*x2^58*x3^38*x4^14*z^37 + x1^74*x2^59*x3^38*x4^14*z^37 + x1^76*x2^59*x3^35*x4^15*z^37 - x1^75*x2^59*x3^36*x4^15*z^37 + x1^74*x2^60*x3^36*x4^15*z^37 + x1^75*x2^58*x3^37*x4^15*z^37 + x1^74*x2^59*x3^37*x4^15*z^37 + x1^73*x2^60*x3^37*x4^15*z^37 - x1^74*x2^58*x3^38*x4^15*z^37 + x1^73*x2^59*x3^38*x4^15*z^37 + x1^71*x2^60*x3^39*x4^15*z^37 + x1^75*x2^59*x3^35*x4^16*z^37 - x1^75*x2^58*x3^36*x4^16*z^37 - x1^74*x2^59*x3^36*x4^16*z^37 + x1^73*x2^60*x3^36*x4^16*z^37 + 2*x1^74*x2^58*x3^37*x4^16*z^37 - x1^73*x2^59*x3^37*x4^16*z^37 + x1^72*x2^60*x3^37*x4^16*z^37 - 2*x1^74*x2^57*x3^38*x4^16*z^37 - x1^73*x2^58*x3^38*x4^16*z^37 - 3*x1^71*x2^60*x3^38*x4^16*z^37 - x1^70*x2^61*x3^38*x4^16*z^37 + 2*x1^73*x2^57*x3^39*x4^16*z^37 + x1^71*x2^59*x3^39*x4^16*z^37 - x1^73*x2^56*x3^40*x4^16*z^37 - x1^72*x2^57*x3^40*x4^16*z^37 + x1^71*x2^58*x3^40*x4^16*z^37 - x1^70*x2^59*x3^40*x4^16*z^37 + x1^75*x2^58*x3^35*x4^17*z^37 - 2*x1^74*x2^58*x3^36*x4^17*z^37 + x1^73*x2^59*x3^36*x4^17*z^37 + 2*x1^73*x2^58*x3^37*x4^17*z^37 - x1^72*x2^59*x3^37*x4^17*z^37 - x1^70*x2^61*x3^37*x4^17*z^37 - 2*x1^73*x2^57*x3^38*x4^17*z^37 - x1^72*x2^58*x3^38*x4^17*z^37 + x1^69*x2^61*x3^38*x4^17*z^37 + x1^73*x2^56*x3^39*x4^17*z^37 + 2*x1^70*x2^59*x3^39*x4^17*z^37 - x1^72*x2^56*x3^40*x4^17*z^37 - x1^71*x2^57*x3^40*x4^17*z^37 - x1^70*x2^58*x3^40*x4^17*z^37 + x1^72*x2^55*x3^41*x4^17*z^37 + x1^71*x2^56*x3^41*x4^17*z^37 - x1^70*x2^57*x3^41*x4^17*z^37 + x1^69*x2^58*x3^41*x4^17*z^37 - x1^74*x2^59*x3^35*x4^12*z^36 + x1^74*x2^58*x3^36*x4^12*z^36 - x1^73*x2^59*x3^36*x4^12*z^36 + x1^75*x2^58*x3^34*x4^13*z^36 + x1^74*x2^58*x3^35*x4^13*z^36 - 2*x1^74*x2^57*x3^36*x4^13*z^36 + x1^73*x2^58*x3^36*x4^13*z^36 + x1^72*x2^59*x3^36*x4^13*z^36 + x1^73*x2^57*x3^37*x4^13*z^36 + x1^72*x2^58*x3^37*x4^13*z^36 - x1^70*x2^60*x3^37*x4^13*z^36 - x1^74*x2^58*x3^34*x4^14*z^36 - x1^74*x2^57*x3^35*x4^14*z^36 + x1^73*x2^58*x3^35*x4^14*z^36 + x1^72*x2^59*x3^35*x4^14*z^36 - x1^73*x2^57*x3^36*x4^14*z^36 - x1^72*x2^58*x3^36*x4^14*z^36 + x1^73*x2^56*x3^37*x4^14*z^36 - x1^72*x2^57*x3^37*x4^14*z^36 + x1^69*x2^60*x3^37*x4^14*z^36 - x1^72*x2^56*x3^38*x4^14*z^36 - x1^71*x2^57*x3^38*x4^14*z^36 - 2*x1^70*x2^58*x3^38*x4^14*z^36 - x1^68*x2^60*x3^38*x4^14*z^36 + x1^74*x2^57*x3^34*x4^15*z^36 - x1^73*x2^57*x3^35*x4^15*z^36 - x1^72*x2^58*x3^35*x4^15*z^36 - x1^71*x2^59*x3^35*x4^15*z^36 + 2*x1^73*x2^56*x3^36*x4^15*z^36 - x1^71*x2^58*x3^36*x4^15*z^36 + x1^70*x2^59*x3^36*x4^15*z^36 - x1^69*x2^60*x3^36*x4^15*z^36 - 2*x1^72*x2^56*x3^37*x4^15*z^36 - 2*x1^70*x2^58*x3^37*x4^15*z^36 + x1^72*x2^55*x3^38*x4^15*z^36 + x1^71*x2^56*x3^38*x4^15*z^36 - x1^70*x2^57*x3^38*x4^15*z^36 + x1^69*x2^58*x3^38*x4^15*z^36 - x1^74*x2^57*x3^33*x4^16*z^36 + 2*x1^73*x2^57*x3^34*x4^16*z^36 - x1^72*x2^58*x3^34*x4^16*z^36 - 2*x1^73*x2^56*x3^35*x4^16*z^36 - 2*x1^72*x2^57*x3^35*x4^16*z^36 + 2*x1^71*x2^58*x3^35*x4^16*z^36 - x1^70*x2^59*x3^35*x4^16*z^36 + 4*x1^72*x2^56*x3^36*x4^16*z^36 + x1^69*x2^59*x3^36*x4^16*z^36 - 2*x1^72*x2^55*x3^37*x4^16*z^36 - 2*x1^71*x2^56*x3^37*x4^16*z^36 + 3*x1^70*x2^57*x3^37*x4^16*z^36 - 3*x1^69*x2^58*x3^37*x4^16*z^36 + 4*x1^71*x2^55*x3^38*x4^16*z^36 + x1^69*x2^57*x3^38*x4^16*z^36 + x1^68*x2^58*x3^38*x4^16*z^36 - 2*x1^71*x2^54*x3^39*x4^16*z^36 - x1^70*x2^55*x3^39*x4^16*z^36 + x1^69*x2^56*x3^39*x4^16*z^36 - 3*x1^68*x2^57*x3^39*x4^16*z^36 + x1^70*x2^54*x3^40*x4^16*z^36 + x1^69*x2^55*x3^40*x4^16*z^36 + x1^68*x2^56*x3^40*x4^16*z^36 - x1^73*x2^57*x3^33*x4^17*z^36 + x1^73*x2^56*x3^34*x4^17*z^36 + x1^72*x2^57*x3^34*x4^17*z^36 - x1^71*x2^58*x3^34*x4^17*z^36 - 2*x1^72*x2^56*x3^35*x4^17*z^36 - x1^71*x2^57*x3^35*x4^17*z^36 + x1^72*x2^55*x3^36*x4^17*z^36 + 2*x1^71*x2^56*x3^36*x4^17*z^36 + x1^69*x2^58*x3^36*x4^17*z^36 - x1^68*x2^59*x3^36*x4^17*z^36 - 2*x1^71*x2^55*x3^37*x4^17*z^36 - x1^68*x2^58*x3^37*x4^17*z^36 + x1^66*x2^60*x3^37*x4^17*z^36 + 2*x1^70*x2^55*x3^38*x4^17*z^36 + 2*x1^68*x2^57*x3^38*x4^17*z^36 - x1^66*x2^59*x3^38*x4^17*z^36 - x1^65*x2^60*x3^38*x4^17*z^36 - 2*x1^70*x2^54*x3^39*x4^17*z^36 - x1^67*x2^57*x3^39*x4^17*z^36 + x1^70*x2^53*x3^40*x4^17*z^36 + 2*x1^67*x2^56*x3^40*x4^17*z^36 - x1^69*x2^53*x3^41*x4^17*z^36 - x1^68*x2^54*x3^41*x4^17*z^36 - x1^67*x2^55*x3^41*x4^17*z^36 + x1^74*x2^57*x3^33*x4^11*z^35 - x1^73*x2^57*x3^34*x4^11*z^35 + x1^73*x2^57*x3^33*x4^12*z^35 + x1^72*x2^56*x3^35*x4^12*z^35 + x1^70*x2^58*x3^35*x4^12*z^35 - x1^71*x2^56*x3^36*x4^12*z^35 - x1^69*x2^58*x3^36*x4^12*z^35 - x1^73*x2^56*x3^33*x4^13*z^35 - x1^72*x2^55*x3^35*x4^13*z^35 - x1^71*x2^56*x3^35*x4^13*z^35 - 2*x1^70*x2^57*x3^35*x4^13*z^35 + 2*x1^71*x2^55*x3^36*x4^13*z^35 + 3*x1^70*x2^56*x3^36*x4^13*z^35 + x1^69*x2^57*x3^36*x4^13*z^35 - x1^71*x2^54*x3^37*x4^13*z^35 - x1^69*x2^56*x3^37*x4^13*z^35 - x1^68*x2^57*x3^37*x4^13*z^35 + x1^73*x2^56*x3^32*x4^14*z^35 - x1^71*x2^57*x3^33*x4^14*z^35 - x1^72*x2^55*x3^34*x4^14*z^35 + 3*x1^70*x2^57*x3^34*x4^14*z^35 - 2*x1^70*x2^56*x3^35*x4^14*z^35 - x1^69*x2^57*x3^35*x4^14*z^35 - x1^68*x2^58*x3^35*x4^14*z^35 - x1^67*x2^59*x3^35*x4^14*z^35 - x1^71*x2^54*x3^36*x4^14*z^35 + 3*x1^69*x2^56*x3^36*x4^14*z^35 + x1^67*x2^58*x3^36*x4^14*z^35 - x1^70*x2^54*x3^37*x4^14*z^35 - 2*x1^69*x2^55*x3^37*x4^14*z^35 + x1^67*x2^57*x3^37*x4^14*z^35 - x1^66*x2^58*x3^37*x4^14*z^35 + x1^70*x2^53*x3^38*x4^14*z^35 + x1^68*x2^55*x3^38*x4^14*z^35 + x1^67*x2^56*x3^38*x4^14*z^35 + x1^66*x2^57*x3^38*x4^14*z^35 + x1^65*x2^58*x3^38*x4^14*z^35 - x1^72*x2^56*x3^32*x4^15*z^35 + x1^71*x2^56*x3^33*x4^15*z^35 - 2*x1^71*x2^55*x3^34*x4^15*z^35 - x1^69*x2^57*x3^34*x4^15*z^35 + 2*x1^71*x2^54*x3^35*x4^15*z^35 + x1^70*x2^55*x3^35*x4^15*z^35 - x1^69*x2^56*x3^35*x4^15*z^35 + 2*x1^68*x2^57*x3^35*x4^15*z^35 + x1^67*x2^58*x3^35*x4^15*z^35 - 4*x1^70*x2^54*x3^36*x4^15*z^35 + x1^69*x2^55*x3^36*x4^15*z^35 - x1^68*x2^56*x3^36*x4^15*z^35 - x1^67*x2^57*x3^36*x4^15*z^35 + x1^66*x2^58*x3^36*x4^15*z^35 + x1^65*x2^59*x3^36*x4^15*z^35 + 2*x1^70*x2^53*x3^37*x4^15*z^35 + x1^69*x2^54*x3^37*x4^15*z^35 - x1^68*x2^55*x3^37*x4^15*z^35 + 3*x1^67*x2^56*x3^37*x4^15*z^35 + x1^66*x2^57*x3^37*x4^15*z^35 - x1^69*x2^53*x3^38*x4^15*z^35 - x1^68*x2^54*x3^38*x4^15*z^35 - x1^67*x2^55*x3^38*x4^15*z^35 - x1^72*x2^55*x3^32*x4^16*z^35 + 4*x1^71*x2^55*x3^33*x4^16*z^35 - x1^70*x2^56*x3^33*x4^16*z^35 + x1^69*x2^57*x3^33*x4^16*z^35 - 2*x1^71*x2^54*x3^34*x4^16*z^35 - 4*x1^70*x2^55*x3^34*x4^16*z^35 + 2*x1^69*x2^56*x3^34*x4^16*z^35 - x1^68*x2^57*x3^34*x4^16*z^35 + x1^67*x2^58*x3^34*x4^16*z^35 + 6*x1^70*x2^54*x3^35*x4^16*z^35 + x1^69*x2^55*x3^35*x4^16*z^35 + x1^67*x2^57*x3^35*x4^16*z^35 - x1^66*x2^58*x3^35*x4^16*z^35 - 2*x1^70*x2^53*x3^36*x4^16*z^35 - 4*x1^69*x2^54*x3^36*x4^16*z^35 + x1^68*x2^55*x3^36*x4^16*z^35 - 3*x1^67*x2^56*x3^36*x4^16*z^35 + x1^66*x2^57*x3^36*x4^16*z^35 + 5*x1^69*x2^53*x3^37*x4^16*z^35 + x1^68*x2^54*x3^37*x4^16*z^35 + 2*x1^66*x2^56*x3^37*x4^16*z^35 - x1^64*x2^58*x3^37*x4^16*z^35 - 2*x1^69*x2^52*x3^38*x4^16*z^35 - 2*x1^68*x2^53*x3^38*x4^16*z^35 - 4*x1^66*x2^55*x3^38*x4^16*z^35 + 3*x1^68*x2^52*x3^39*x4^16*z^35 + x1^67*x2^53*x3^39*x4^16*z^35 + x1^66*x2^54*x3^39*x4^16*z^35 + x1^65*x2^55*x3^39*x4^16*z^35 - x1^68*x2^51*x3^40*x4^16*z^35 - 2*x1^65*x2^54*x3^40*x4^16*z^35 + x1^72*x2^55*x3^31*x4^17*z^35 - 2*x1^71*x2^55*x3^32*x4^17*z^35 + x1^70*x2^56*x3^32*x4^17*z^35 + 2*x1^70*x2^55*x3^33*x4^17*z^35 + x1^68*x2^57*x3^33*x4^17*z^35 - 2*x1^70*x2^54*x3^34*x4^17*z^35 - x1^69*x2^55*x3^34*x4^17*z^35 - x1^68*x2^56*x3^34*x4^17*z^35 + x1^66*x2^58*x3^34*x4^17*z^35 + x1^70*x2^53*x3^35*x4^17*z^35 + 2*x1^69*x2^54*x3^35*x4^17*z^35 + x1^67*x2^56*x3^35*x4^17*z^35 - x1^65*x2^58*x3^35*x4^17*z^35 - 2*x1^69*x2^53*x3^36*x4^17*z^35 - x1^68*x2^54*x3^36*x4^17*z^35 - x1^67*x2^55*x3^36*x4^17*z^35 - x1^66*x2^56*x3^36*x4^17*z^35 + x1^64*x2^58*x3^36*x4^17*z^35 + x1^69*x2^52*x3^37*x4^17*z^35 + 2*x1^68*x2^53*x3^37*x4^17*z^35 + x1^66*x2^55*x3^37*x4^17*z^35 - x1^65*x2^56*x3^37*x4^17*z^35 + x1^64*x2^57*x3^37*x4^17*z^35 - x1^63*x2^58*x3^37*x4^17*z^35 - 2*x1^68*x2^52*x3^38*x4^17*z^35 - x1^65*x2^55*x3^38*x4^17*z^35 - x1^64*x2^56*x3^38*x4^17*z^35 + x1^63*x2^57*x3^38*x4^17*z^35 + x1^62*x2^58*x3^38*x4^17*z^35 + x1^67*x2^52*x3^39*x4^17*z^35 + x1^66*x2^53*x3^39*x4^17*z^35 + x1^65*x2^54*x3^39*x4^17*z^35 - 2*x1^67*x2^51*x3^40*x4^17*z^35 - x1^64*x2^54*x3^40*x4^17*z^35 + 2*x1^64*x2^53*x3^41*x4^17*z^35 + x1^72*x2^55*x3^33*x4^10*z^34 - x1^70*x2^57*x3^33*x4^10*z^34 + x1^72*x2^55*x3^32*x4^11*z^34 + x1^71*x2^56*x3^32*x4^11*z^34 - 3*x1^71*x2^55*x3^33*x4^11*z^34 - x1^69*x2^57*x3^33*x4^11*z^34 + x1^70*x2^55*x3^34*x4^11*z^34 + 2*x1^68*x2^57*x3^34*x4^11*z^34 - x1^70*x2^54*x3^35*x4^11*z^34 - x1^68*x2^56*x3^35*x4^11*z^34 - x1^72*x2^55*x3^31*x4^12*z^34 + x1^71*x2^55*x3^32*x4^12*z^34 - 2*x1^70*x2^56*x3^32*x4^12*z^34 - 2*x1^70*x2^55*x3^33*x4^12*z^34 + x1^69*x2^56*x3^33*x4^12*z^34 - x1^68*x2^57*x3^33*x4^12*z^34 - x1^69*x2^55*x3^34*x4^12*z^34 - x1^69*x2^54*x3^35*x4^12*z^34 + x1^68*x2^55*x3^35*x4^12*z^34 - 2*x1^67*x2^56*x3^35*x4^12*z^34 + x1^69*x2^53*x3^36*x4^12*z^34 - x1^68*x2^54*x3^36*x4^12*z^34 + x1^67*x2^55*x3^36*x4^12*z^34 + x1^66*x2^56*x3^36*x4^12*z^34 + x1^71*x2^55*x3^31*x4^13*z^34 - x1^70*x2^55*x3^32*x4^13*z^34 - x1^69*x2^56*x3^32*x4^13*z^34 + 3*x1^70*x2^54*x3^33*x4^13*z^34 + 3*x1^69*x2^55*x3^33*x4^13*z^34 + x1^68*x2^56*x3^33*x4^13*z^34 - x1^69*x2^54*x3^34*x4^13*z^34 - 4*x1^68*x2^55*x3^34*x4^13*z^34 + x1^67*x2^56*x3^34*x4^13*z^34 - x1^65*x2^58*x3^34*x4^13*z^34 + 2*x1^69*x2^53*x3^35*x4^13*z^34 + 2*x1^68*x2^54*x3^35*x4^13*z^34 + 3*x1^67*x2^55*x3^35*x4^13*z^34 + x1^66*x2^56*x3^35*x4^13*z^34 + 2*x1^65*x2^57*x3^35*x4^13*z^34 - 4*x1^67*x2^54*x3^36*x4^13*z^34 - 2*x1^66*x2^55*x3^36*x4^13*z^34 - 2*x1^65*x2^56*x3^36*x4^13*z^34 + 2*x1^68*x2^52*x3^37*x4^13*z^34 + x1^67*x2^53*x3^37*x4^13*z^34 + x1^66*x2^54*x3^37*x4^13*z^34 + x1^64*x2^56*x3^37*x4^13*z^34 - x1^71*x2^54*x3^31*x4^14*z^34 - 2*x1^69*x2^55*x3^32*x4^14*z^34 - x1^68*x2^56*x3^32*x4^14*z^34 + x1^69*x2^54*x3^33*x4^14*z^34 + 2*x1^68*x2^55*x3^33*x4^14*z^34 + 2*x1^66*x2^57*x3^33*x4^14*z^34 + x1^69*x2^53*x3^34*x4^14*z^34 + x1^68*x2^54*x3^34*x4^14*z^34 - x1^67*x2^55*x3^34*x4^14*z^34 - x1^65*x2^57*x3^34*x4^14*z^34 + x1^64*x2^58*x3^34*x4^14*z^34 - x1^69*x2^52*x3^35*x4^14*z^34 + x1^68*x2^53*x3^35*x4^14*z^34 + 4*x1^67*x2^54*x3^35*x4^14*z^34 + x1^64*x2^57*x3^35*x4^14*z^34 + x1^67*x2^53*x3^36*x4^14*z^34 - x1^65*x2^55*x3^36*x4^14*z^34 - 2*x1^64*x2^56*x3^36*x4^14*z^34 - x1^68*x2^51*x3^37*x4^14*z^34 + 3*x1^66*x2^53*x3^37*x4^14*z^34 + x1^65*x2^54*x3^37*x4^14*z^34 + x1^64*x2^55*x3^37*x4^14*z^34 - 2*x1^67*x2^51*x3^38*x4^14*z^34 - x1^66*x2^52*x3^38*x4^14*z^34 - x1^65*x2^53*x3^38*x4^14*z^34 - x1^63*x2^55*x3^38*x4^14*z^34 + x1^71*x2^54*x3^30*x4^15*z^34 - x1^70*x2^54*x3^31*x4^15*z^34 + 2*x1^70*x2^53*x3^32*x4^15*z^34 + x1^69*x2^54*x3^32*x4^15*z^34 + x1^67*x2^56*x3^32*x4^15*z^34 - 4*x1^69*x2^53*x3^33*x4^15*z^34 + x1^68*x2^54*x3^33*x4^15*z^34 - 2*x1^66*x2^56*x3^33*x4^15*z^34 + 2*x1^69*x2^52*x3^34*x4^15*z^34 + 2*x1^68*x2^53*x3^34*x4^15*z^34 - 2*x1^67*x2^54*x3^34*x4^15*z^34 + 3*x1^66*x2^55*x3^34*x4^15*z^34 - 5*x1^68*x2^52*x3^35*x4^15*z^34 - x1^67*x2^53*x3^35*x4^15*z^34 - 2*x1^65*x2^55*x3^35*x4^15*z^34 + 2*x1^68*x2^51*x3^36*x4^15*z^34 + 2*x1^67*x2^52*x3^36*x4^15*z^34 + 3*x1^65*x2^54*x3^36*x4^15*z^34 + x1^64*x2^55*x3^36*x4^15*z^34 + x1^63*x2^56*x3^36*x4^15*z^34 - x1^62*x2^57*x3^36*x4^15*z^34 - 3*x1^67*x2^51*x3^37*x4^15*z^34 - x1^66*x2^52*x3^37*x4^15*z^34 - x1^65*x2^53*x3^37*x4^15*z^34 - x1^64*x2^54*x3^37*x4^15*z^34 + x1^67*x2^50*x3^38*x4^15*z^34 + 2*x1^64*x2^53*x3^38*x4^15*z^34 + x1^70*x2^54*x3^30*x4^16*z^34 - 2*x1^70*x2^53*x3^31*x4^16*z^34 - x1^69*x2^54*x3^31*x4^16*z^34 + 2*x1^68*x2^55*x3^31*x4^16*z^34 + 5*x1^69*x2^53*x3^32*x4^16*z^34 - x1^67*x2^55*x3^32*x4^16*z^34 - 2*x1^69*x2^52*x3^33*x4^16*z^34 - 5*x1^68*x2^53*x3^33*x4^16*z^34 + x1^67*x2^54*x3^33*x4^16*z^34 - 2*x1^66*x2^55*x3^33*x4^16*z^34 + x1^65*x2^56*x3^33*x4^16*z^34 + 6*x1^68*x2^52*x3^34*x4^16*z^34 + 2*x1^67*x2^53*x3^34*x4^16*z^34 + x1^66*x2^54*x3^34*x4^16*z^34 + 2*x1^65*x2^55*x3^34*x4^16*z^34 - x1^64*x2^56*x3^34*x4^16*z^34 - x1^63*x2^57*x3^34*x4^16*z^34 - 2*x1^68*x2^51*x3^35*x4^16*z^34 - 6*x1^67*x2^52*x3^35*x4^16*z^34 - 4*x1^65*x2^54*x3^35*x4^16*z^34 + 2*x1^63*x2^56*x3^35*x4^16*z^34 + 2*x1^62*x2^57*x3^35*x4^16*z^34 + 6*x1^67*x2^51*x3^36*x4^16*z^34 + x1^66*x2^52*x3^36*x4^16*z^34 + x1^65*x2^53*x3^36*x4^16*z^34 + 3*x1^64*x2^54*x3^36*x4^16*z^34 - x1^63*x2^55*x3^36*x4^16*z^34 - 2*x1^62*x2^56*x3^36*x4^16*z^34 - 2*x1^67*x2^50*x3^37*x4^16*z^34 - 3*x1^66*x2^51*x3^37*x4^16*z^34 - x1^65*x2^52*x3^37*x4^16*z^34 - 4*x1^64*x2^53*x3^37*x4^16*z^34 + x1^63*x2^54*x3^37*x4^16*z^34 - x1^62*x2^55*x3^37*x4^16*z^34 + x1^61*x2^56*x3^37*x4^16*z^34 + 4*x1^66*x2^50*x3^38*x4^16*z^34 + 2*x1^64*x2^52*x3^38*x4^16*z^34 + 2*x1^63*x2^53*x3^38*x4^16*z^34 - x1^66*x2^49*x3^39*x4^16*z^34 - x1^65*x2^50*x3^39*x4^16*z^34 - x1^64*x2^51*x3^39*x4^16*z^34 - 3*x1^63*x2^52*x3^39*x4^16*z^34 + x1^65*x2^49*x3^40*x4^16*z^34 + x1^62*x2^52*x3^40*x4^16*z^34 + x1^70*x2^53*x3^30*x4^17*z^34 - 2*x1^69*x2^53*x3^31*x4^17*z^34 - x1^68*x2^54*x3^31*x4^17*z^34 - x1^67*x2^55*x3^31*x4^17*z^34 + x1^69*x2^52*x3^32*x4^17*z^34 + 2*x1^68*x2^53*x3^32*x4^17*z^34 + x1^66*x2^55*x3^32*x4^17*z^34 - x1^65*x2^56*x3^32*x4^17*z^34 - 2*x1^68*x2^52*x3^33*x4^17*z^34 - x1^65*x2^55*x3^33*x4^17*z^34 + 2*x1^67*x2^52*x3^34*x4^17*z^34 + 2*x1^65*x2^54*x3^34*x4^17*z^34 - x1^62*x2^57*x3^34*x4^17*z^34 - 2*x1^67*x2^51*x3^35*x4^17*z^34 - x1^66*x2^52*x3^35*x4^17*z^34 - x1^65*x2^53*x3^35*x4^17*z^34 + x1^63*x2^55*x3^35*x4^17*z^34 + x1^62*x2^56*x3^35*x4^17*z^34 + x1^61*x2^57*x3^35*x4^17*z^34 + x1^67*x2^50*x3^36*x4^17*z^34 + 2*x1^66*x2^51*x3^36*x4^17*z^34 + x1^64*x2^53*x3^36*x4^17*z^34 + x1^63*x2^54*x3^36*x4^17*z^34 - 2*x1^61*x2^56*x3^36*x4^17*z^34 - 2*x1^66*x2^50*x3^37*x4^17*z^34 - x1^65*x2^51*x3^37*x4^17*z^34 - x1^64*x2^52*x3^37*x4^17*z^34 - x1^63*x2^53*x3^37*x4^17*z^34 + x1^62*x2^54*x3^37*x4^17*z^34 + x1^61*x2^55*x3^37*x4^17*z^34 + x1^66*x2^49*x3^38*x4^17*z^34 + 2*x1^65*x2^50*x3^38*x4^17*z^34 + x1^63*x2^52*x3^38*x4^17*z^34 - x1^62*x2^53*x3^38*x4^17*z^34 + x1^61*x2^54*x3^38*x4^17*z^34 - x1^60*x2^55*x3^38*x4^17*z^34 - x1^65*x2^49*x3^39*x4^17*z^34 + x1^64*x2^50*x3^39*x4^17*z^34 - x1^63*x2^51*x3^39*x4^17*z^34 - x1^62*x2^52*x3^39*x4^17*z^34 + x1^64*x2^49*x3^40*x4^17*z^34 + x1^63*x2^50*x3^40*x4^17*z^34 + x1^62*x2^51*x3^40*x4^17*z^34 - x1^61*x2^51*x3^41*x4^17*z^34 - x1^70*x2^54*x3^32*x4^9*z^33 + x1^69*x2^55*x3^32*x4^9*z^33 - x1^70*x2^53*x3^33*x4^9*z^33 - x1^69*x2^54*x3^33*x4^9*z^33 - x1^71*x2^54*x3^30*x4^10*z^33 + x1^69*x2^53*x3^33*x4^10*z^33 - x1^68*x2^54*x3^33*x4^10*z^33 + x1^66*x2^56*x3^33*x4^10*z^33 - x1^66*x2^55*x3^34*x4^10*z^33 - x1^70*x2^54*x3^30*x4^11*z^33 + 2*x1^70*x2^53*x3^31*x4^11*z^33 + x1^69*x2^54*x3^31*x4^11*z^33 - x1^68*x2^55*x3^31*x4^11*z^33 - 3*x1^69*x2^53*x3^32*x4^11*z^33 - 2*x1^67*x2^55*x3^32*x4^11*z^33 - x1^66*x2^56*x3^32*x4^11*z^33 + x1^69*x2^52*x3^33*x4^11*z^33 + 2*x1^68*x2^53*x3^33*x4^11*z^33 + x1^67*x2^54*x3^33*x4^11*z^33 + 4*x1^66*x2^55*x3^33*x4^11*z^33 + x1^65*x2^56*x3^33*x4^11*z^33 - x1^68*x2^52*x3^34*x4^11*z^33 - x1^65*x2^55*x3^34*x4^11*z^33 + x1^68*x2^51*x3^35*x4^11*z^33 + x1^67*x2^52*x3^35*x4^11*z^33 + 2*x1^65*x2^54*x3^35*x4^11*z^33 + x1^64*x2^55*x3^35*x4^11*z^33 - x1^70*x2^53*x3^30*x4^12*z^33 + x1^69*x2^53*x3^31*x4^12*z^33 + x1^67*x2^55*x3^31*x4^12*z^33 - x1^68*x2^53*x3^32*x4^12*z^33 + 3*x1^67*x2^54*x3^32*x4^12*z^33 + 2*x1^65*x2^56*x3^32*x4^12*z^33 - x1^67*x2^53*x3^33*x4^12*z^33 - x1^66*x2^54*x3^33*x4^12*z^33 + x1^65*x2^55*x3^33*x4^12*z^33 - x1^68*x2^51*x3^34*x4^12*z^33 + 3*x1^66*x2^53*x3^34*x4^12*z^33 - x1^65*x2^54*x3^34*x4^12*z^33 - x1^67*x2^51*x3^35*x4^12*z^33 - x1^67*x2^50*x3^36*x4^12*z^33 - x1^66*x2^51*x3^36*x4^12*z^33 + x1^65*x2^52*x3^36*x4^12*z^33 - x1^64*x2^53*x3^36*x4^12*z^33 - x1^70*x2^53*x3^29*x4^13*z^33 + x1^69*x2^53*x3^30*x4^13*z^33 + x1^68*x2^54*x3^30*x4^13*z^33 - x1^68*x2^53*x3^31*x4^13*z^33 - 4*x1^67*x2^54*x3^31*x4^13*z^33 + x1^68*x2^52*x3^32*x4^13*z^33 + 4*x1^67*x2^53*x3^32*x4^13*z^33 + 3*x1^66*x2^54*x3^32*x4^13*z^33 - x1^65*x2^55*x3^32*x4^13*z^33 + x1^64*x2^56*x3^32*x4^13*z^33 - 2*x1^67*x2^52*x3^33*x4^13*z^33 - 8*x1^66*x2^53*x3^33*x4^13*z^33 - 2*x1^64*x2^55*x3^33*x4^13*z^33 + x1^67*x2^51*x3^34*x4^13*z^33 + 2*x1^66*x2^52*x3^34*x4^13*z^33 + 4*x1^65*x2^53*x3^34*x4^13*z^33 + x1^64*x2^54*x3^34*x4^13*z^33 + 2*x1^63*x2^55*x3^34*x4^13*z^33 + x1^61*x2^57*x3^34*x4^13*z^33 - x1^66*x2^51*x3^35*x4^13*z^33 - 5*x1^65*x2^52*x3^35*x4^13*z^33 - x1^64*x2^53*x3^35*x4^13*z^33 - 2*x1^63*x2^54*x3^35*x4^13*z^33 - 2*x1^62*x2^55*x3^35*x4^13*z^33 + 2*x1^66*x2^50*x3^36*x4^13*z^33 + x1^64*x2^52*x3^36*x4^13*z^33 + x1^63*x2^53*x3^36*x4^13*z^33 + 3*x1^62*x2^54*x3^36*x4^13*z^33 - x1^65*x2^50*x3^37*x4^13*z^33 - 2*x1^64*x2^51*x3^37*x4^13*z^33 - 2*x1^63*x2^52*x3^37*x4^13*z^33 - x1^62*x2^53*x3^37*x4^13*z^33 + x1^69*x2^53*x3^29*x4^14*z^33 - x1^68*x2^53*x3^30*x4^14*z^33 + x1^67*x2^54*x3^30*x4^14*z^33 + x1^68*x2^52*x3^31*x4^14*z^33 + x1^67*x2^53*x3^31*x4^14*z^33 + x1^65*x2^55*x3^31*x4^14*z^33 - x1^68*x2^51*x3^32*x4^14*z^33 - x1^67*x2^52*x3^32*x4^14*z^33 + 2*x1^66*x2^53*x3^32*x4^14*z^33 + x1^65*x2^54*x3^32*x4^14*z^33 + x1^64*x2^55*x3^32*x4^14*z^33 + 2*x1^67*x2^51*x3^33*x4^14*z^33 - 3*x1^66*x2^52*x3^33*x4^14*z^33 - 3*x1^65*x2^53*x3^33*x4^14*z^33 + 2*x1^64*x2^54*x3^33*x4^14*z^33 - x1^63*x2^55*x3^33*x4^14*z^33 - x1^62*x2^56*x3^33*x4^14*z^33 + 2*x1^65*x2^52*x3^34*x4^14*z^33 - 3*x1^64*x2^53*x3^34*x4^14*z^33 + x1^62*x2^55*x3^34*x4^14*z^33 - x1^60*x2^57*x3^34*x4^14*z^33 + x1^66*x2^50*x3^35*x4^14*z^33 - 2*x1^64*x2^52*x3^35*x4^14*z^33 - x1^63*x2^53*x3^35*x4^14*z^33 - x1^62*x2^54*x3^35*x4^14*z^33 + x1^61*x2^55*x3^35*x4^14*z^33 + x1^59*x2^57*x3^35*x4^14*z^33 + x1^65*x2^50*x3^36*x4^14*z^33 + x1^64*x2^51*x3^36*x4^14*z^33 - x1^63*x2^52*x3^36*x4^14*z^33 + x1^61*x2^54*x3^36*x4^14*z^33 + x1^64*x2^50*x3^37*x4^14*z^33 - x1^62*x2^52*x3^37*x4^14*z^33 - 2*x1^61*x2^53*x3^37*x4^14*z^33 + x1^64*x2^49*x3^38*x4^14*z^33 + 2*x1^63*x2^50*x3^38*x4^14*z^33 + 2*x1^62*x2^51*x3^38*x4^14*z^33 + x1^61*x2^52*x3^38*x4^14*z^33 - 2*x1^68*x2^52*x3^30*x4^15*z^33 - x1^66*x2^54*x3^30*x4^15*z^33 + 2*x1^68*x2^51*x3^31*x4^15*z^33 + 2*x1^67*x2^52*x3^31*x4^15*z^33 - x1^66*x2^53*x3^31*x4^15*z^33 + x1^65*x2^54*x3^31*x4^15*z^33 - 6*x1^67*x2^51*x3^32*x4^15*z^33 - x1^65*x2^53*x3^32*x4^15*z^33 - x1^64*x2^54*x3^32*x4^15*z^33 + 2*x1^67*x2^50*x3^33*x4^15*z^33 + 6*x1^66*x2^51*x3^33*x4^15*z^33 - 2*x1^65*x2^52*x3^33*x4^15*z^33 + 2*x1^64*x2^53*x3^33*x4^15*z^33 - 6*x1^66*x2^50*x3^34*x4^15*z^33 - x1^65*x2^51*x3^34*x4^15*z^33 + x1^64*x2^52*x3^34*x4^15*z^33 - 3*x1^63*x2^53*x3^34*x4^15*z^33 + x1^61*x2^55*x3^34*x4^15*z^33 + 2*x1^66*x2^49*x3^35*x4^15*z^33 + 3*x1^65*x2^50*x3^35*x4^15*z^33 + x1^64*x2^51*x3^35*x4^15*z^33 + 4*x1^63*x2^52*x3^35*x4^15*z^33 - 4*x1^65*x2^49*x3^36*x4^15*z^33 - 2*x1^63*x2^51*x3^36*x4^15*z^33 - 2*x1^62*x2^52*x3^36*x4^15*z^33 + x1^65*x2^48*x3^37*x4^15*z^33 + x1^64*x2^49*x3^37*x4^15*z^33 + x1^63*x2^50*x3^37*x4^15*z^33 + 3*x1^62*x2^51*x3^37*x4^15*z^33 - 2*x1^64*x2^48*x3^38*x4^15*z^33 - x1^61*x2^51*x3^38*x4^15*z^33 - x1^63*x2^48*x3^39*x4^15*z^33 - x1^69*x2^52*x3^28*x4^16*z^33 + 3*x1^68*x2^52*x3^29*x4^16*z^33 - 2*x1^67*x2^53*x3^29*x4^16*z^33 - 2*x1^68*x2^51*x3^30*x4^16*z^33 - 3*x1^67*x2^52*x3^30*x4^16*z^33 + 2*x1^66*x2^53*x3^30*x4^16*z^33 - x1^65*x2^54*x3^30*x4^16*z^33 + 6*x1^67*x2^51*x3^31*x4^16*z^33 + x1^66*x2^52*x3^31*x4^16*z^33 - x1^64*x2^54*x3^31*x4^16*z^33 - 2*x1^63*x2^55*x3^31*x4^16*z^33 - 2*x1^67*x2^50*x3^32*x4^16*z^33 - 6*x1^66*x2^51*x3^32*x4^16*z^33 - 2*x1^64*x2^53*x3^32*x4^16*z^33 + 2*x1^63*x2^54*x3^32*x4^16*z^33 + 2*x1^62*x2^55*x3^32*x4^16*z^33 + 6*x1^66*x2^50*x3^33*x4^16*z^33 + 2*x1^65*x2^51*x3^33*x4^16*z^33 + 2*x1^64*x2^52*x3^33*x4^16*z^33 + x1^63*x2^53*x3^33*x4^16*z^33 - x1^62*x2^54*x3^33*x4^16*z^33 - 2*x1^61*x2^55*x3^33*x4^16*z^33 - 2*x1^66*x2^49*x3^34*x4^16*z^33 - 6*x1^65*x2^50*x3^34*x4^16*z^33 - 4*x1^63*x2^52*x3^34*x4^16*z^33 + 3*x1^60*x2^55*x3^34*x4^16*z^33 + 6*x1^65*x2^49*x3^35*x4^16*z^33 + 2*x1^64*x2^50*x3^35*x4^16*z^33 + 2*x1^63*x2^51*x3^35*x4^16*z^33 + 2*x1^62*x2^52*x3^35*x4^16*z^33 + x1^61*x2^53*x3^35*x4^16*z^33 - 2*x1^60*x2^54*x3^35*x4^16*z^33 - 2*x1^59*x2^55*x3^35*x4^16*z^33 - 2*x1^65*x2^48*x3^36*x4^16*z^33 - 5*x1^64*x2^49*x3^36*x4^16*z^33 - x1^63*x2^50*x3^36*x4^16*z^33 - 3*x1^62*x2^51*x3^36*x4^16*z^33 - x1^60*x2^53*x3^36*x4^16*z^33 + 3*x1^59*x2^54*x3^36*x4^16*z^33 + 5*x1^64*x2^48*x3^37*x4^16*z^33 + 2*x1^62*x2^50*x3^37*x4^16*z^33 + 3*x1^61*x2^51*x3^37*x4^16*z^33 - x1^60*x2^52*x3^37*x4^16*z^33 - x1^59*x2^53*x3^37*x4^16*z^33 - 2*x1^63*x2^48*x3^38*x4^16*z^33 - 4*x1^61*x2^50*x3^38*x4^16*z^33 + 2*x1^63*x2^47*x3^39*x4^16*z^33 + x1^61*x2^49*x3^39*x4^16*z^33 + 2*x1^60*x2^50*x3^39*x4^16*z^33 - x1^60*x2^49*x3^40*x4^16*z^33 - x1^68*x2^52*x3^28*x4^17*z^33 + x1^67*x2^52*x3^29*x4^17*z^33 - 2*x1^67*x2^51*x3^30*x4^17*z^33 - x1^65*x2^53*x3^30*x4^17*z^33 + x1^67*x2^50*x3^31*x4^17*z^33 + 2*x1^66*x2^51*x3^31*x4^17*z^33 + 2*x1^64*x2^53*x3^31*x4^17*z^33 + x1^63*x2^54*x3^31*x4^17*z^33 - 2*x1^66*x2^50*x3^32*x4^17*z^33 - x1^65*x2^51*x3^32*x4^17*z^33 - x1^64*x2^52*x3^32*x4^17*z^33 + x1^61*x2^55*x3^32*x4^17*z^33 + x1^66*x2^49*x3^33*x4^17*z^33 + 2*x1^65*x2^50*x3^33*x4^17*z^33 + x1^63*x2^52*x3^33*x4^17*z^33 - 2*x1^62*x2^53*x3^33*x4^17*z^33 - x1^60*x2^55*x3^33*x4^17*z^33 - 2*x1^65*x2^49*x3^34*x4^17*z^33 + x1^59*x2^55*x3^34*x4^17*z^33 + 2*x1^64*x2^49*x3^35*x4^17*z^33 + 2*x1^62*x2^51*x3^35*x4^17*z^33 - x1^61*x2^52*x3^35*x4^17*z^33 - 2*x1^59*x2^54*x3^35*x4^17*z^33 - x1^58*x2^55*x3^35*x4^17*z^33 - 2*x1^64*x2^48*x3^36*x4^17*z^33 - x1^63*x2^49*x3^36*x4^17*z^33 - x1^62*x2^50*x3^36*x4^17*z^33 + x1^58*x2^54*x3^36*x4^17*z^33 + x1^64*x2^47*x3^37*x4^17*z^33 + 2*x1^63*x2^48*x3^37*x4^17*z^33 + x1^61*x2^50*x3^37*x4^17*z^33 + x1^60*x2^51*x3^37*x4^17*z^33 - 2*x1^58*x2^53*x3^37*x4^17*z^33 - 2*x1^63*x2^47*x3^38*x4^17*z^33 - x1^62*x2^48*x3^38*x4^17*z^33 - x1^61*x2^49*x3^38*x4^17*z^33 - x1^60*x2^50*x3^38*x4^17*z^33 + x1^59*x2^51*x3^38*x4^17*z^33 + x1^58*x2^52*x3^38*x4^17*z^33 + x1^62*x2^47*x3^39*x4^17*z^33 + x1^60*x2^49*x3^39*x4^17*z^33 - x1^60*x2^48*x3^40*x4^17*z^33 - x1^59*x2^49*x3^40*x4^17*z^33 - x1^68*x2^54*x3^30*x4^8*z^32 + x1^69*x2^52*x3^31*x4^8*z^32 - x1^67*x2^53*x3^32*x4^8*z^32 + x1^67*x2^53*x3^31*x4^9*z^32 - x1^68*x2^51*x3^32*x4^9*z^32 + x1^67*x2^52*x3^32*x4^9*z^32 + x1^65*x2^54*x3^32*x4^9*z^32 + x1^67*x2^51*x3^33*x4^9*z^32 + 2*x1^66*x2^52*x3^33*x4^9*z^32 + 2*x1^65*x2^53*x3^33*x4^9*z^32 + x1^68*x2^52*x3^30*x4^10*z^32 + x1^67*x2^53*x3^30*x4^10*z^32 + x1^66*x2^54*x3^30*x4^10*z^32 - x1^68*x2^51*x3^31*x4^10*z^32 - x1^67*x2^52*x3^31*x4^10*z^32 + x1^66*x2^53*x3^31*x4^10*z^32 - x1^65*x2^54*x3^31*x4^10*z^32 + x1^67*x2^51*x3^32*x4^10*z^32 + x1^65*x2^53*x3^32*x4^10*z^32 - 2*x1^66*x2^51*x3^33*x4^10*z^32 - x1^65*x2^52*x3^33*x4^10*z^32 - x1^64*x2^53*x3^33*x4^10*z^32 + x1^65*x2^51*x3^34*x4^10*z^32 + x1^63*x2^53*x3^34*x4^10*z^32 + x1^62*x2^54*x3^34*x4^10*z^32 + x1^69*x2^52*x3^28*x4^11*z^32 - 3*x1^68*x2^52*x3^29*x4^11*z^32 + x1^67*x2^53*x3^29*x4^11*z^32 + 2*x1^68*x2^51*x3^30*x4^11*z^32 + 3*x1^67*x2^52*x3^30*x4^11*z^32 - 2*x1^66*x2^53*x3^30*x4^11*z^32 + 2*x1^65*x2^54*x3^30*x4^11*z^32 - 3*x1^67*x2^51*x3^31*x4^11*z^32 - x1^65*x2^53*x3^31*x4^11*z^32 - x1^64*x2^54*x3^31*x4^11*z^32 + x1^63*x2^55*x3^31*x4^11*z^32 + x1^67*x2^50*x3^32*x4^11*z^32 + 3*x1^66*x2^51*x3^32*x4^11*z^32 + 3*x1^64*x2^53*x3^32*x4^11*z^32 + x1^63*x2^54*x3^32*x4^11*z^32 + x1^62*x2^55*x3^32*x4^11*z^32 - 2*x1^66*x2^50*x3^33*x4^11*z^32 + x1^65*x2^51*x3^33*x4^11*z^32 - 2*x1^64*x2^52*x3^33*x4^11*z^32 - 3*x1^63*x2^53*x3^33*x4^11*z^32 - x1^62*x2^54*x3^33*x4^11*z^32 + x1^66*x2^49*x3^34*x4^11*z^32 + x1^65*x2^50*x3^34*x4^11*z^32 + 2*x1^63*x2^52*x3^34*x4^11*z^32 - x1^62*x2^53*x3^34*x4^11*z^32 - x1^65*x2^49*x3^35*x4^11*z^32 - 2*x1^64*x2^50*x3^35*x4^11*z^32 - x1^63*x2^51*x3^35*x4^11*z^32 - 2*x1^61*x2^53*x3^35*x4^11*z^32 + x1^68*x2^52*x3^28*x4^12*z^32 - x1^67*x2^52*x3^29*x4^12*z^32 + x1^66*x2^53*x3^29*x4^12*z^32 - x1^67*x2^50*x3^31*x4^12*z^32 + 4*x1^65*x2^52*x3^31*x4^12*z^32 - 2*x1^64*x2^53*x3^31*x4^12*z^32 + x1^62*x2^55*x3^31*x4^12*z^32 + x1^66*x2^50*x3^32*x4^12*z^32 - 3*x1^65*x2^51*x3^32*x4^12*z^32 - 2*x1^64*x2^52*x3^32*x4^12*z^32 + x1^63*x2^53*x3^32*x4^12*z^32 - 2*x1^62*x2^54*x3^32*x4^12*z^32 - 2*x1^61*x2^55*x3^32*x4^12*z^32 - x1^65*x2^50*x3^33*x4^12*z^32 + 5*x1^64*x2^51*x3^33*x4^12*z^32 + 2*x1^62*x2^53*x3^33*x4^12*z^32 + x1^60*x2^55*x3^33*x4^12*z^32 - x1^63*x2^51*x3^34*x4^12*z^32 - 2*x1^61*x2^53*x3^34*x4^12*z^32 - x1^65*x2^48*x3^35*x4^12*z^32 + x1^64*x2^49*x3^35*x4^12*z^32 + 2*x1^63*x2^50*x3^35*x4^12*z^32 + x1^61*x2^52*x3^35*x4^12*z^32 + x1^64*x2^48*x3^36*x4^12*z^32 + x1^63*x2^49*x3^36*x4^12*z^32 + x1^62*x2^50*x3^36*x4^12*z^32 + 3*x1^66*x2^52*x3^29*x4^13*z^32 + x1^65*x2^53*x3^29*x4^13*z^32 - 5*x1^65*x2^52*x3^30*x4^13*z^32 - x1^63*x2^54*x3^30*x4^13*z^32 + 2*x1^65*x2^51*x3^31*x4^13*z^32 + 5*x1^64*x2^52*x3^31*x4^13*z^32 - x1^63*x2^53*x3^31*x4^13*z^32 + 2*x1^62*x2^54*x3^31*x4^13*z^32 - x1^61*x2^55*x3^31*x4^13*z^32 - 2*x1^65*x2^50*x3^32*x4^13*z^32 - 7*x1^64*x2^51*x3^32*x4^13*z^32 - 3*x1^63*x2^52*x3^32*x4^13*z^32 - 2*x1^62*x2^53*x3^32*x4^13*z^32 - x1^61*x2^54*x3^32*x4^13*z^32 + x1^60*x2^55*x3^32*x4^13*z^32 + 2*x1^64*x2^50*x3^33*x4^13*z^32 + 6*x1^63*x2^51*x3^33*x4^13*z^32 - x1^62*x2^52*x3^33*x4^13*z^32 + 3*x1^61*x2^53*x3^33*x4^13*z^32 - x1^60*x2^54*x3^33*x4^13*z^32 - 2*x1^64*x2^49*x3^34*x4^13*z^32 - 5*x1^63*x2^50*x3^34*x4^13*z^32 - 3*x1^61*x2^52*x3^34*x4^13*z^32 - x1^60*x2^53*x3^34*x4^13*z^32 + x1^59*x2^54*x3^34*x4^13*z^32 + x1^64*x2^48*x3^35*x4^13*z^32 + x1^63*x2^49*x3^35*x4^13*z^32 + 3*x1^62*x2^50*x3^35*x4^13*z^32 + x1^61*x2^51*x3^35*x4^13*z^32 + 4*x1^60*x2^52*x3^35*x4^13*z^32 - 2*x1^63*x2^48*x3^36*x4^13*z^32 - 3*x1^62*x2^49*x3^36*x4^13*z^32 - x1^61*x2^50*x3^36*x4^13*z^32 - x1^60*x2^51*x3^36*x4^13*z^32 - x1^59*x2^52*x3^36*x4^13*z^32 + x1^63*x2^47*x3^37*x4^13*z^32 + x1^61*x2^49*x3^37*x4^13*z^32 + x1^60*x2^50*x3^37*x4^13*z^32 + 2*x1^59*x2^51*x3^37*x4^13*z^32 - x1^68*x2^51*x3^27*x4^14*z^32 + x1^67*x2^51*x3^28*x4^14*z^32 - x1^66*x2^52*x3^28*x4^14*z^32 - x1^66*x2^51*x3^29*x4^14*z^32 - 2*x1^64*x2^53*x3^29*x4^14*z^32 + x1^66*x2^50*x3^30*x4^14*z^32 + x1^63*x2^53*x3^30*x4^14*z^32 - x1^62*x2^54*x3^30*x4^14*z^32 - x1^66*x2^49*x3^31*x4^14*z^32 - x1^65*x2^50*x3^31*x4^14*z^32 + 2*x1^64*x2^51*x3^31*x4^14*z^32 - x1^63*x2^52*x3^31*x4^14*z^32 - x1^62*x2^53*x3^31*x4^14*z^32 - x1^60*x2^55*x3^31*x4^14*z^32 + 2*x1^65*x2^49*x3^32*x4^14*z^32 - x1^63*x2^51*x3^32*x4^14*z^32 + 3*x1^62*x2^52*x3^32*x4^14*z^32 + x1^59*x2^55*x3^32*x4^14*z^32 - x1^65*x2^48*x3^33*x4^14*z^32 - x1^64*x2^49*x3^33*x4^14*z^32 + x1^63*x2^50*x3^33*x4^14*z^32 + 2*x1^62*x2^51*x3^33*x4^14*z^32 + x1^61*x2^52*x3^33*x4^14*z^32 - 2*x1^60*x2^53*x3^33*x4^14*z^32 - x1^58*x2^55*x3^33*x4^14*z^32 + 2*x1^64*x2^48*x3^34*x4^14*z^32 - 2*x1^63*x2^49*x3^34*x4^14*z^32 - 3*x1^62*x2^50*x3^34*x4^14*z^32 + 2*x1^61*x2^51*x3^34*x4^14*z^32 - x1^58*x2^54*x3^34*x4^14*z^32 + x1^57*x2^55*x3^34*x4^14*z^32 + x1^63*x2^48*x3^35*x4^14*z^32 + x1^62*x2^49*x3^35*x4^14*z^32 - 2*x1^61*x2^50*x3^35*x4^14*z^32 + x1^60*x2^51*x3^35*x4^14*z^32 + x1^59*x2^52*x3^35*x4^14*z^32 - x1^58*x2^53*x3^35*x4^14*z^32 - x1^57*x2^54*x3^35*x4^14*z^32 - x1^56*x2^55*x3^35*x4^14*z^32 + x1^63*x2^47*x3^36*x4^14*z^32 - x1^62*x2^48*x3^36*x4^14*z^32 - 2*x1^61*x2^49*x3^36*x4^14*z^32 - x1^59*x2^51*x3^36*x4^14*z^32 + x1^62*x2^47*x3^37*x4^14*z^32 + x1^61*x2^48*x3^37*x4^14*z^32 - x1^60*x2^49*x3^37*x4^14*z^32 + x1^58*x2^51*x3^37*x4^14*z^32 - x1^61*x2^47*x3^38*x4^14*z^32 - x1^60*x2^48*x3^38*x4^14*z^32 - x1^59*x2^49*x3^38*x4^14*z^32 - 2*x1^58*x2^50*x3^38*x4^14*z^32 + x1^67*x2^50*x3^28*x4^15*z^32 - x1^65*x2^52*x3^28*x4^15*z^32 - 3*x1^66*x2^50*x3^29*x4^15*z^32 + x1^65*x2^51*x3^29*x4^15*z^32 + x1^64*x2^52*x3^29*x4^15*z^32 + 2*x1^66*x2^49*x3^30*x4^15*z^32 + 3*x1^65*x2^50*x3^30*x4^15*z^32 - 2*x1^64*x2^51*x3^30*x4^15*z^32 + x1^63*x2^52*x3^30*x4^15*z^32 - 6*x1^65*x2^49*x3^31*x4^15*z^32 - x1^64*x2^50*x3^31*x4^15*z^32 - x1^62*x2^52*x3^31*x4^15*z^32 + x1^61*x2^53*x3^31*x4^15*z^32 + 2*x1^65*x2^48*x3^32*x4^15*z^32 + 6*x1^64*x2^49*x3^32*x4^15*z^32 + 3*x1^62*x2^51*x3^32*x4^15*z^32 - x1^60*x2^53*x3^32*x4^15*z^32 - x1^59*x2^54*x3^32*x4^15*z^32 - 6*x1^64*x2^48*x3^33*x4^15*z^32 - 2*x1^63*x2^49*x3^33*x4^15*z^32 - 2*x1^62*x2^50*x3^33*x4^15*z^32 - 2*x1^61*x2^51*x3^33*x4^15*z^32 + x1^60*x2^52*x3^33*x4^15*z^32 + x1^59*x2^53*x3^33*x4^15*z^32 + 2*x1^64*x2^47*x3^34*x4^15*z^32 + 5*x1^63*x2^48*x3^34*x4^15*z^32 + x1^62*x2^49*x3^34*x4^15*z^32 + 3*x1^61*x2^50*x3^34*x4^15*z^32 - 2*x1^58*x2^53*x3^34*x4^15*z^32 - 5*x1^63*x2^47*x3^35*x4^15*z^32 - 2*x1^61*x2^49*x3^35*x4^15*z^32 - 3*x1^60*x2^50*x3^35*x4^15*z^32 + x1^59*x2^51*x3^35*x4^15*z^32 + x1^58*x2^52*x3^35*x4^15*z^32 + 2*x1^63*x2^46*x3^36*x4^15*z^32 + 2*x1^62*x2^47*x3^36*x4^15*z^32 + 4*x1^60*x2^49*x3^36*x4^15*z^32 - 2*x1^62*x2^46*x3^37*x4^15*z^32 + x1^61*x2^47*x3^37*x4^15*z^32 - x1^60*x2^48*x3^37*x4^15*z^32 - 2*x1^59*x2^49*x3^37*x4^15*z^32 + x1^61*x2^46*x3^38*x4^15*z^32 + x1^60*x2^47*x3^38*x4^15*z^32 + x1^59*x2^48*x3^38*x4^15*z^32 + x1^60*x2^46*x3^39*x4^15*z^32 + x1^59*x2^47*x3^39*x4^15*z^32 - 2*x1^67*x2^50*x3^27*x4^16*z^32 + 4*x1^66*x2^50*x3^28*x4^16*z^32 + x1^64*x2^52*x3^28*x4^16*z^32 - 2*x1^66*x2^49*x3^29*x4^16*z^32 - 4*x1^65*x2^50*x3^29*x4^16*z^32 + x1^64*x2^51*x3^29*x4^16*z^32 - x1^63*x2^52*x3^29*x4^16*z^32 + 2*x1^62*x2^53*x3^29*x4^16*z^32 + 6*x1^65*x2^49*x3^30*x4^16*z^32 + 2*x1^64*x2^50*x3^30*x4^16*z^32 + x1^63*x2^51*x3^30*x4^16*z^32 - 2*x1^61*x2^53*x3^30*x4^16*z^32 - 2*x1^65*x2^48*x3^31*x4^16*z^32 - 6*x1^64*x2^49*x3^31*x4^16*z^32 - 4*x1^62*x2^51*x3^31*x4^16*z^32 + 2*x1^61*x2^52*x3^31*x4^16*z^32 + 2*x1^60*x2^53*x3^31*x4^16*z^32 + 2*x1^59*x2^54*x3^31*x4^16*z^32 + 6*x1^64*x2^48*x3^32*x4^16*z^32 + 2*x1^63*x2^49*x3^32*x4^16*z^32 + 2*x1^62*x2^50*x3^32*x4^16*z^32 + x1^61*x2^51*x3^32*x4^16*z^32 - 2*x1^60*x2^52*x3^32*x4^16*z^32 - 3*x1^59*x2^53*x3^32*x4^16*z^32 - 2*x1^58*x2^54*x3^32*x4^16*z^32 - 2*x1^64*x2^47*x3^33*x4^16*z^32 - 6*x1^63*x2^48*x3^33*x4^16*z^32 - 4*x1^61*x2^50*x3^33*x4^16*z^32 + x1^60*x2^51*x3^33*x4^16*z^32 + 5*x1^58*x2^53*x3^33*x4^16*z^32 + 6*x1^63*x2^47*x3^34*x4^16*z^32 + 2*x1^62*x2^48*x3^34*x4^16*z^32 + 2*x1^61*x2^49*x3^34*x4^16*z^32 + x1^60*x2^50*x3^34*x4^16*z^32 - x1^59*x2^51*x3^34*x4^16*z^32 - x1^58*x2^52*x3^34*x4^16*z^32 - 2*x1^57*x2^53*x3^34*x4^16*z^32 - 2*x1^63*x2^46*x3^35*x4^16*z^32 - 6*x1^62*x2^47*x3^35*x4^16*z^32 - 4*x1^60*x2^49*x3^35*x4^16*z^32 + 4*x1^57*x2^52*x3^35*x4^16*z^32 + 5*x1^62*x2^46*x3^36*x4^16*z^32 + x1^61*x2^47*x3^36*x4^16*z^32 + 3*x1^60*x2^48*x3^36*x4^16*z^32 + 2*x1^59*x2^49*x3^36*x4^16*z^32 - x1^58*x2^50*x3^36*x4^16*z^32 - x1^57*x2^51*x3^36*x4^16*z^32 - x1^56*x2^52*x3^36*x4^16*z^32 - x1^62*x2^45*x3^37*x4^16*z^32 - 4*x1^61*x2^46*x3^37*x4^16*z^32 - 3*x1^59*x2^48*x3^37*x4^16*z^32 - x1^58*x2^49*x3^37*x4^16*z^32 + 2*x1^56*x2^51*x3^37*x4^16*z^32 + x1^61*x2^45*x3^38*x4^16*z^32 - x1^60*x2^46*x3^38*x4^16*z^32 + x1^59*x2^47*x3^38*x4^16*z^32 + 2*x1^58*x2^48*x3^38*x4^16*z^32 - x1^60*x2^45*x3^39*x4^16*z^32 - x1^59*x2^46*x3^39*x4^16*z^32 - x1^58*x2^47*x3^39*x4^16*z^32 + x1^57*x2^47*x3^40*x4^16*z^32 - x1^66*x2^50*x3^27*x4^17*z^32 + x1^66*x2^49*x3^28*x4^17*z^32 + x1^65*x2^50*x3^28*x4^17*z^32 + x1^63*x2^52*x3^28*x4^17*z^32 - 2*x1^65*x2^49*x3^29*x4^17*z^32 - x1^62*x2^52*x3^29*x4^17*z^32 + 2*x1^64*x2^49*x3^30*x4^17*z^32 + x1^62*x2^51*x3^30*x4^17*z^32 - 2*x1^64*x2^48*x3^31*x4^17*z^32 - x1^63*x2^49*x3^31*x4^17*z^32 - x1^62*x2^50*x3^31*x4^17*z^32 + x1^64*x2^47*x3^32*x4^17*z^32 + 2*x1^63*x2^48*x3^32*x4^17*z^32 + x1^61*x2^50*x3^32*x4^17*z^32 - x1^60*x2^51*x3^32*x4^17*z^32 - 2*x1^58*x2^53*x3^32*x4^17*z^32 - 2*x1^63*x2^47*x3^33*x4^17*z^32 - x1^62*x2^48*x3^33*x4^17*z^32 - x1^61*x2^49*x3^33*x4^17*z^32 + x1^59*x2^51*x3^33*x4^17*z^32 + x1^58*x2^52*x3^33*x4^17*z^32 + 2*x1^57*x2^53*x3^33*x4^17*z^32 + x1^63*x2^46*x3^34*x4^17*z^32 + 2*x1^62*x2^47*x3^34*x4^17*z^32 + x1^60*x2^49*x3^34*x4^17*z^32 - 2*x1^59*x2^50*x3^34*x4^17*z^32 - 2*x1^57*x2^52*x3^34*x4^17*z^32 - 2*x1^62*x2^46*x3^35*x4^17*z^32 + 2*x1^56*x2^52*x3^35*x4^17*z^32 + 2*x1^61*x2^46*x3^36*x4^17*z^32 + 2*x1^59*x2^48*x3^36*x4^17*z^32 - x1^57*x2^50*x3^36*x4^17*z^32 - x1^56*x2^51*x3^36*x4^17*z^32 - x1^61*x2^45*x3^37*x4^17*z^32 - 2*x1^60*x2^46*x3^37*x4^17*z^32 - x1^59*x2^47*x3^37*x4^17*z^32 + x1^55*x2^51*x3^37*x4^17*z^32 + x1^60*x2^45*x3^38*x4^17*z^32 + 2*x1^59*x2^46*x3^38*x4^17*z^32 + x1^58*x2^47*x3^38*x4^17*z^32 + x1^57*x2^48*x3^38*x4^17*z^32 - 2*x1^55*x2^50*x3^38*x4^17*z^32 - x1^58*x2^46*x3^39*x4^17*z^32 - x1^57*x2^47*x3^39*x4^17*z^32 - x1^65*x2^52*x3^31*x4^7*z^31 - x1^66*x2^52*x3^29*x4^8*z^31 + x1^66*x2^51*x3^30*x4^8*z^31 + 2*x1^65*x2^52*x3^30*x4^8*z^31 - x1^66*x2^50*x3^31*x4^8*z^31 - x1^65*x2^51*x3^31*x4^8*z^31 + x1^66*x2^49*x3^32*x4^8*z^31 + x1^64*x2^51*x3^32*x4^8*z^31 + x1^63*x2^52*x3^32*x4^8*z^31 + x1^66*x2^52*x3^28*x4^9*z^31 + x1^66*x2^51*x3^29*x4^9*z^31 - 2*x1^65*x2^52*x3^29*x4^9*z^31 + x1^64*x2^53*x3^29*x4^9*z^31 + x1^66*x2^50*x3^30*x4^9*z^31 - 2*x1^64*x2^51*x3^31*x4^9*z^31 + x1^63*x2^52*x3^31*x4^9*z^31 + x1^65*x2^49*x3^32*x4^9*z^31 + x1^64*x2^50*x3^32*x4^9*z^31 - x1^62*x2^52*x3^32*x4^9*z^31 - x1^65*x2^48*x3^33*x4^9*z^31 - x1^63*x2^50*x3^33*x4^9*z^31 - 2*x1^62*x2^51*x3^33*x4^9*z^31 - x1^61*x2^52*x3^33*x4^9*z^31 - x1^67*x2^50*x3^28*x4^10*z^31 + 2*x1^66*x2^50*x3^29*x4^10*z^31 - x1^65*x2^51*x3^29*x4^10*z^31 + x1^63*x2^53*x3^29*x4^10*z^31 - x1^66*x2^49*x3^30*x4^10*z^31 - 2*x1^65*x2^50*x3^30*x4^10*z^31 + x1^64*x2^51*x3^30*x4^10*z^31 - 3*x1^63*x2^52*x3^30*x4^10*z^31 - x1^62*x2^53*x3^30*x4^10*z^31 + x1^61*x2^54*x3^30*x4^10*z^31 + 2*x1^65*x2^49*x3^31*x4^10*z^31 + 2*x1^64*x2^50*x3^31*x4^10*z^31 + x1^63*x2^51*x3^31*x4^10*z^31 + x1^62*x2^52*x3^31*x4^10*z^31 - x1^61*x2^53*x3^31*x4^10*z^31 - x1^65*x2^48*x3^32*x4^10*z^31 - x1^64*x2^49*x3^32*x4^10*z^31 - x1^63*x2^50*x3^32*x4^10*z^31 - 2*x1^62*x2^51*x3^32*x4^10*z^31 - x1^61*x2^52*x3^32*x4^10*z^31 - x1^64*x2^48*x3^33*x4^10*z^31 + x1^63*x2^49*x3^33*x4^10*z^31 + 2*x1^62*x2^50*x3^33*x4^10*z^31 + 2*x1^61*x2^51*x3^33*x4^10*z^31 + x1^60*x2^52*x3^33*x4^10*z^31 - 2*x1^62*x2^49*x3^34*x4^10*z^31 - x1^59*x2^52*x3^34*x4^10*z^31 + 2*x1^67*x2^50*x3^27*x4^11*z^31 - 4*x1^66*x2^50*x3^28*x4^11*z^31 + x1^65*x2^51*x3^28*x4^11*z^31 - x1^64*x2^52*x3^28*x4^11*z^31 + x1^66*x2^49*x3^29*x4^11*z^31 + 3*x1^65*x2^50*x3^29*x4^11*z^31 - x1^64*x2^51*x3^29*x4^11*z^31 + 2*x1^63*x2^52*x3^29*x4^11*z^31 - 2*x1^62*x2^53*x3^29*x4^11*z^31 - 4*x1^65*x2^49*x3^30*x4^11*z^31 - 3*x1^64*x2^50*x3^30*x4^11*z^31 - 2*x1^62*x2^52*x3^30*x4^11*z^31 + 2*x1^65*x2^48*x3^31*x4^11*z^31 + 2*x1^64*x2^49*x3^31*x4^11*z^31 - 3*x1^63*x2^50*x3^31*x4^11*z^31 + 3*x1^62*x2^51*x3^31*x4^11*z^31 + x1^61*x2^52*x3^31*x4^11*z^31 - x1^60*x2^53*x3^31*x4^11*z^31 - x1^59*x2^54*x3^31*x4^11*z^31 - 2*x1^64*x2^48*x3^32*x4^11*z^31 - x1^63*x2^49*x3^32*x4^11*z^31 - 2*x1^62*x2^50*x3^32*x4^11*z^31 - x1^61*x2^51*x3^32*x4^11*z^31 + x1^59*x2^53*x3^32*x4^11*z^31 + 2*x1^64*x2^47*x3^33*x4^11*z^31 + x1^63*x2^48*x3^33*x4^11*z^31 - x1^62*x2^49*x3^33*x4^11*z^31 + x1^61*x2^50*x3^33*x4^11*z^31 + x1^59*x2^52*x3^33*x4^11*z^31 - x1^63*x2^47*x3^34*x4^11*z^31 - x1^61*x2^49*x3^34*x4^11*z^31 - 2*x1^60*x2^50*x3^34*x4^11*z^31 + x1^63*x2^46*x3^35*x4^11*z^31 + 2*x1^61*x2^48*x3^35*x4^11*z^31 + 2*x1^60*x2^49*x3^35*x4^11*z^31 + x1^58*x2^51*x3^35*x4^11*z^31 - x1^67*x2^50*x3^26*x4^12*z^31 + x1^66*x2^50*x3^27*x4^12*z^31 - x1^65*x2^51*x3^27*x4^12*z^31 - x1^65*x2^50*x3^28*x4^12*z^31 + x1^64*x2^51*x3^28*x4^12*z^31 - x1^63*x2^52*x3^28*x4^12*z^31 + x1^65*x2^49*x3^29*x4^12*z^31 - 2*x1^64*x2^50*x3^29*x4^12*z^31 - x1^63*x2^51*x3^29*x4^12*z^31 + 2*x1^62*x2^52*x3^29*x4^12*z^31 - x1^61*x2^53*x3^29*x4^12*z^31 - x1^65*x2^48*x3^30*x4^12*z^31 + x1^64*x2^49*x3^30*x4^12*z^31 + 6*x1^63*x2^50*x3^30*x4^12*z^31 - 2*x1^62*x2^51*x3^30*x4^12*z^31 + x1^61*x2^52*x3^30*x4^12*z^31 + x1^60*x2^53*x3^30*x4^12*z^31 + 2*x1^64*x2^48*x3^31*x4^12*z^31 - x1^63*x2^49*x3^31*x4^12*z^31 - 5*x1^62*x2^50*x3^31*x4^12*z^31 + x1^61*x2^51*x3^31*x4^12*z^31 - x1^60*x2^52*x3^31*x4^12*z^31 - x1^59*x2^53*x3^31*x4^12*z^31 - x1^58*x2^54*x3^31*x4^12*z^31 - x1^64*x2^47*x3^32*x4^12*z^31 + 5*x1^62*x2^49*x3^32*x4^12*z^31 + 2*x1^60*x2^51*x3^32*x4^12*z^31 - x1^59*x2^52*x3^32*x4^12*z^31 + 2*x1^58*x2^53*x3^32*x4^12*z^31 + x1^63*x2^47*x3^33*x4^12*z^31 - x1^62*x2^48*x3^33*x4^12*z^31 - 3*x1^61*x2^49*x3^33*x4^12*z^31 - 4*x1^59*x2^51*x3^33*x4^12*z^31 - x1^58*x2^52*x3^33*x4^12*z^31 - x1^57*x2^53*x3^33*x4^12*z^31 + x1^62*x2^47*x3^34*x4^12*z^31 + 2*x1^61*x2^48*x3^34*x4^12*z^31 + x1^59*x2^50*x3^34*x4^12*z^31 + x1^58*x2^51*x3^34*x4^12*z^31 + x1^62*x2^46*x3^35*x4^12*z^31 - x1^60*x2^48*x3^35*x4^12*z^31 - 2*x1^58*x2^50*x3^35*x4^12*z^31 - x1^62*x2^45*x3^36*x4^12*z^31 - 2*x1^59*x2^48*x3^36*x4^12*z^31 + x1^66*x2^49*x3^27*x4^13*z^31 - 2*x1^64*x2^51*x3^27*x4^13*z^31 - x1^65*x2^49*x3^28*x4^13*z^31 + x1^64*x2^50*x3^28*x4^13*z^31 + 2*x1^63*x2^51*x3^28*x4^13*z^31 - 2*x1^62*x2^52*x3^28*x4^13*z^31 + x1^64*x2^49*x3^29*x4^13*z^31 - 5*x1^63*x2^50*x3^29*x4^13*z^31 - x1^62*x2^51*x3^29*x4^13*z^31 - x1^61*x2^52*x3^29*x4^13*z^31 + x1^63*x2^49*x3^30*x4^13*z^31 + 5*x1^62*x2^50*x3^30*x4^13*z^31 - 3*x1^61*x2^51*x3^30*x4^13*z^31 + 2*x1^60*x2^52*x3^30*x4^13*z^31 - x1^59*x2^53*x3^30*x4^13*z^31 - 6*x1^62*x2^49*x3^31*x4^13*z^31 - x1^61*x2^50*x3^31*x4^13*z^31 - x1^60*x2^51*x3^31*x4^13*z^31 - x1^59*x2^52*x3^31*x4^13*z^31 + x1^58*x2^53*x3^31*x4^13*z^31 + x1^57*x2^54*x3^31*x4^13*z^31 + 3*x1^62*x2^48*x3^32*x4^13*z^31 + 7*x1^61*x2^49*x3^32*x4^13*z^31 + 4*x1^59*x2^51*x3^32*x4^13*z^31 - x1^57*x2^53*x3^32*x4^13*z^31 - 2*x1^56*x2^54*x3^32*x4^13*z^31 - 2*x1^62*x2^47*x3^33*x4^13*z^31 - 4*x1^61*x2^48*x3^33*x4^13*z^31 - x1^60*x2^49*x3^33*x4^13*z^31 - 3*x1^59*x2^50*x3^33*x4^13*z^31 + 2*x1^57*x2^52*x3^33*x4^13*z^31 + 2*x1^56*x2^53*x3^33*x4^13*z^31 + 3*x1^61*x2^47*x3^34*x4^13*z^31 + 4*x1^60*x2^48*x3^34*x4^13*z^31 + 4*x1^58*x2^50*x3^34*x4^13*z^31 - x1^57*x2^51*x3^34*x4^13*z^31 - x1^55*x2^53*x3^34*x4^13*z^31 - x1^62*x2^45*x3^35*x4^13*z^31 - x1^61*x2^46*x3^35*x4^13*z^31 - 3*x1^60*x2^47*x3^35*x4^13*z^31 - x1^59*x2^48*x3^35*x4^13*z^31 - 2*x1^58*x2^49*x3^35*x4^13*z^31 - 2*x1^57*x2^50*x3^35*x4^13*z^31 + x1^61*x2^45*x3^36*x4^13*z^31 + x1^60*x2^46*x3^36*x4^13*z^31 + 3*x1^59*x2^47*x3^36*x4^13*z^31 + x1^58*x2^48*x3^36*x4^13*z^31 + 3*x1^57*x2^49*x3^36*x4^13*z^31 - x1^60*x2^45*x3^37*x4^13*z^31 - 2*x1^59*x2^46*x3^37*x4^13*z^31 - x1^58*x2^47*x3^37*x4^13*z^31 - x1^56*x2^49*x3^37*x4^13*z^31 + x1^64*x2^50*x3^27*x4^14*z^31 + x1^63*x2^51*x3^27*x4^14*z^31 - x1^65*x2^48*x3^28*x4^14*z^31 + x1^63*x2^50*x3^28*x4^14*z^31 + x1^61*x2^52*x3^28*x4^14*z^31 + 2*x1^64*x2^48*x3^29*x4^14*z^31 - 2*x1^63*x2^49*x3^29*x4^14*z^31 - x1^62*x2^50*x3^29*x4^14*z^31 + 2*x1^61*x2^51*x3^29*x4^14*z^31 + x1^60*x2^52*x3^29*x4^14*z^31 + x1^59*x2^53*x3^29*x4^14*z^31 - 2*x1^63*x2^48*x3^30*x4^14*z^31 + 3*x1^62*x2^49*x3^30*x4^14*z^31 - 2*x1^61*x2^50*x3^30*x4^14*z^31 - 2*x1^60*x2^51*x3^30*x4^14*z^31 - x1^59*x2^52*x3^30*x4^14*z^31 + 2*x1^63*x2^47*x3^31*x4^14*z^31 + x1^62*x2^48*x3^31*x4^14*z^31 - 2*x1^61*x2^49*x3^31*x4^14*z^31 + x1^60*x2^50*x3^31*x4^14*z^31 - 2*x1^59*x2^51*x3^31*x4^14*z^31 + x1^58*x2^52*x3^31*x4^14*z^31 + x1^56*x2^54*x3^31*x4^14*z^31 - x1^63*x2^46*x3^32*x4^14*z^31 - 2*x1^62*x2^47*x3^32*x4^14*z^31 + 2*x1^61*x2^48*x3^32*x4^14*z^31 - 2*x1^59*x2^50*x3^32*x4^14*z^31 - x1^58*x2^51*x3^32*x4^14*z^31 - 2*x1^57*x2^52*x3^32*x4^14*z^31 - x1^56*x2^53*x3^32*x4^14*z^31 - x1^55*x2^54*x3^32*x4^14*z^31 + x1^62*x2^46*x3^33*x4^14*z^31 - x1^61*x2^47*x3^33*x4^14*z^31 + x1^59*x2^49*x3^33*x4^14*z^31 - 3*x1^58*x2^50*x3^33*x4^14*z^31 - x1^57*x2^51*x3^33*x4^14*z^31 + x1^56*x2^52*x3^33*x4^14*z^31 + 2*x1^55*x2^53*x3^33*x4^14*z^31 - x1^62*x2^45*x3^34*x4^14*z^31 - x1^61*x2^46*x3^34*x4^14*z^31 + x1^60*x2^47*x3^34*x4^14*z^31 + x1^59*x2^48*x3^34*x4^14*z^31 + 2*x1^58*x2^49*x3^34*x4^14*z^31 - x1^57*x2^50*x3^34*x4^14*z^31 - x1^56*x2^51*x3^34*x4^14*z^31 - x1^55*x2^52*x3^34*x4^14*z^31 + x1^61*x2^45*x3^35*x4^14*z^31 - 3*x1^60*x2^46*x3^35*x4^14*z^31 - 2*x1^59*x2^47*x3^35*x4^14*z^31 + 2*x1^58*x2^48*x3^35*x4^14*z^31 - x1^57*x2^49*x3^35*x4^14*z^31 + x1^56*x2^50*x3^35*x4^14*z^31 + x1^54*x2^52*x3^35*x4^14*z^31 + x1^57*x2^48*x3^36*x4^14*z^31 + x1^56*x2^49*x3^36*x4^14*z^31 - x1^58*x2^46*x3^37*x4^14*z^31 - x1^56*x2^48*x3^37*x4^14*z^31 + x1^58*x2^45*x3^38*x4^14*z^31 + x1^57*x2^46*x3^38*x4^14*z^31 + x1^55*x2^48*x3^38*x4^14*z^31 - x1^65*x2^49*x3^26*x4^15*z^31 + x1^64*x2^50*x3^26*x4^15*z^31 + 2*x1^65*x2^48*x3^27*x4^15*z^31 + x1^64*x2^49*x3^27*x4^15*z^31 - 2*x1^63*x2^50*x3^27*x4^15*z^31 - 5*x1^64*x2^48*x3^28*x4^15*z^31 + x1^63*x2^49*x3^28*x4^15*z^31 + x1^62*x2^50*x3^28*x4^15*z^31 + x1^61*x2^51*x3^28*x4^15*z^31 + x1^60*x2^52*x3^28*x4^15*z^31 + 2*x1^64*x2^47*x3^29*x4^15*z^31 + 5*x1^63*x2^48*x3^29*x4^15*z^31 - 2*x1^62*x2^49*x3^29*x4^15*z^31 + x1^61*x2^50*x3^29*x4^15*z^31 - 2*x1^60*x2^51*x3^29*x4^15*z^31 - x1^59*x2^52*x3^29*x4^15*z^31 - 6*x1^63*x2^47*x3^30*x4^15*z^31 - 2*x1^62*x2^48*x3^30*x4^15*z^31 + x1^60*x2^50*x3^30*x4^15*z^31 + 2*x1^59*x2^51*x3^30*x4^15*z^31 + x1^58*x2^52*x3^30*x4^15*z^31 + 2*x1^63*x2^46*x3^31*x4^15*z^31 + 6*x1^62*x2^47*x3^31*x4^15*z^31 + 3*x1^60*x2^49*x3^31*x4^15*z^31 - x1^59*x2^50*x3^31*x4^15*z^31 - 2*x1^58*x2^51*x3^31*x4^15*z^31 - 2*x1^57*x2^52*x3^31*x4^15*z^31 - 6*x1^62*x2^46*x3^32*x4^15*z^31 - 2*x1^61*x2^47*x3^32*x4^15*z^31 - 2*x1^60*x2^48*x3^32*x4^15*z^31 - x1^59*x2^49*x3^32*x4^15*z^31 + 2*x1^57*x2^51*x3^32*x4^15*z^31 + x1^56*x2^52*x3^32*x4^15*z^31 + 2*x1^62*x2^45*x3^33*x4^15*z^31 + 6*x1^61*x2^46*x3^33*x4^15*z^31 + 4*x1^59*x2^48*x3^33*x4^15*z^31 - 4*x1^56*x2^51*x3^33*x4^15*z^31 - 5*x1^61*x2^45*x3^34*x4^15*z^31 - x1^60*x2^46*x3^34*x4^15*z^31 - 3*x1^59*x2^47*x3^34*x4^15*z^31 - 2*x1^58*x2^48*x3^34*x4^15*z^31 + x1^57*x2^49*x3^34*x4^15*z^31 + x1^56*x2^50*x3^34*x4^15*z^31 + x1^55*x2^51*x3^34*x4^15*z^31 + x1^61*x2^44*x3^35*x4^15*z^31 + 4*x1^60*x2^45*x3^35*x4^15*z^31 + 3*x1^58*x2^47*x3^35*x4^15*z^31 + x1^57*x2^48*x3^35*x4^15*z^31 - 2*x1^55*x2^50*x3^35*x4^15*z^31 - 3*x1^60*x2^44*x3^36*x4^15*z^31 - x1^59*x2^45*x3^36*x4^15*z^31 - 2*x1^58*x2^46*x3^36*x4^15*z^31 - 2*x1^57*x2^47*x3^36*x4^15*z^31 + x1^59*x2^44*x3^37*x4^15*z^31 - x1^58*x2^45*x3^37*x4^15*z^31 + 2*x1^57*x2^46*x3^37*x4^15*z^31 - x1^57*x2^45*x3^38*x4^15*z^31 - x1^56*x2^46*x3^38*x4^15*z^31 - x1^56*x2^45*x3^39*x4^15*z^31 + x1^65*x2^49*x3^25*x4^16*z^31 - x1^65*x2^48*x3^26*x4^16*z^31 - 2*x1^64*x2^49*x3^26*x4^16*z^31 + x1^63*x2^50*x3^26*x4^16*z^31 + 5*x1^64*x2^48*x3^27*x4^16*z^31 + x1^63*x2^49*x3^27*x4^16*z^31 + x1^62*x2^50*x3^27*x4^16*z^31 - 2*x1^64*x2^47*x3^28*x4^16*z^31 - 5*x1^63*x2^48*x3^28*x4^16*z^31 - 3*x1^61*x2^50*x3^28*x4^16*z^31 + 6*x1^63*x2^47*x3^29*x4^16*z^31 + x1^62*x2^48*x3^29*x4^16*z^31 + 2*x1^61*x2^49*x3^29*x4^16*z^31 - x1^59*x2^51*x3^29*x4^16*z^31 - 2*x1^58*x2^52*x3^29*x4^16*z^31 - 2*x1^63*x2^46*x3^30*x4^16*z^31 - 6*x1^62*x2^47*x3^30*x4^16*z^31 - 5*x1^60*x2^49*x3^30*x4^16*z^31 + 2*x1^59*x2^50*x3^30*x4^16*z^31 + x1^58*x2^51*x3^30*x4^16*z^31 + 3*x1^57*x2^52*x3^30*x4^16*z^31 + 6*x1^62*x2^46*x3^31*x4^16*z^31 + 2*x1^61*x2^47*x3^31*x4^16*z^31 + 2*x1^60*x2^48*x3^31*x4^16*z^31 - x1^58*x2^50*x3^31*x4^16*z^31 - 2*x1^57*x2^51*x3^31*x4^16*z^31 - 3*x1^56*x2^52*x3^31*x4^16*z^31 - 2*x1^62*x2^45*x3^32*x4^16*z^31 - 6*x1^61*x2^46*x3^32*x4^16*z^31 - 4*x1^59*x2^48*x3^32*x4^16*z^31 + 4*x1^58*x2^49*x3^32*x4^16*z^31 + 5*x1^56*x2^51*x3^32*x4^16*z^31 + 2*x1^55*x2^52*x3^32*x4^16*z^31 + 6*x1^61*x2^45*x3^33*x4^16*z^31 + 2*x1^60*x2^46*x3^33*x4^16*z^31 + 2*x1^59*x2^47*x3^33*x4^16*z^31 - x1^57*x2^49*x3^33*x4^16*z^31 - x1^56*x2^50*x3^33*x4^16*z^31 - 4*x1^55*x2^51*x3^33*x4^16*z^31 - 2*x1^61*x2^44*x3^34*x4^16*z^31 - 6*x1^60*x2^45*x3^34*x4^16*z^31 - 4*x1^58*x2^47*x3^34*x4^16*z^31 + x1^57*x2^48*x3^34*x4^16*z^31 + x1^56*x2^49*x3^34*x4^16*z^31 + 5*x1^55*x2^50*x3^34*x4^16*z^31 + 5*x1^60*x2^44*x3^35*x4^16*z^31 + 2*x1^59*x2^45*x3^35*x4^16*z^31 + 2*x1^58*x2^46*x3^35*x4^16*z^31 + 2*x1^57*x2^47*x3^35*x4^16*z^31 - 2*x1^55*x2^49*x3^35*x4^16*z^31 - 2*x1^54*x2^50*x3^35*x4^16*z^31 - x1^60*x2^43*x3^36*x4^16*z^31 - 4*x1^59*x2^44*x3^36*x4^16*z^31 - 4*x1^57*x2^46*x3^36*x4^16*z^31 - x1^56*x2^47*x3^36*x4^16*z^31 + x1^55*x2^48*x3^36*x4^16*z^31 + 3*x1^54*x2^49*x3^36*x4^16*z^31 + 2*x1^59*x2^43*x3^37*x4^16*z^31 + 2*x1^58*x2^44*x3^37*x4^16*z^31 + 2*x1^57*x2^45*x3^37*x4^16*z^31 + x1^56*x2^46*x3^37*x4^16*z^31 - x1^53*x2^49*x3^37*x4^16*z^31 - x1^58*x2^43*x3^38*x4^16*z^31 - x1^56*x2^45*x3^38*x4^16*z^31 + x1^55*x2^46*x3^38*x4^16*z^31 + x1^56*x2^44*x3^39*x4^16*z^31 + x1^55*x2^45*x3^39*x4^16*z^31 - x1^62*x2^50*x3^26*x4^17*z^31 + x1^64*x2^47*x3^27*x4^17*z^31 + 2*x1^63*x2^48*x3^27*x4^17*z^31 - x1^62*x2^49*x3^27*x4^17*z^31 + x1^61*x2^50*x3^27*x4^17*z^31 - 2*x1^63*x2^47*x3^28*x4^17*z^31 - x1^62*x2^48*x3^28*x4^17*z^31 + x1^63*x2^46*x3^29*x4^17*z^31 + 2*x1^62*x2^47*x3^29*x4^17*z^31 + x1^60*x2^49*x3^29*x4^17*z^31 - x1^59*x2^50*x3^29*x4^17*z^31 - 2*x1^62*x2^46*x3^30*x4^17*z^31 + x1^57*x2^51*x3^30*x4^17*z^31 + 2*x1^61*x2^46*x3^31*x4^17*z^31 + 2*x1^59*x2^48*x3^31*x4^17*z^31 - x1^58*x2^49*x3^31*x4^17*z^31 - 2*x1^56*x2^51*x3^31*x4^17*z^31 - 2*x1^61*x2^45*x3^32*x4^17*z^31 - x1^60*x2^46*x3^32*x4^17*z^31 - x1^59*x2^47*x3^32*x4^17*z^31 + x1^57*x2^49*x3^32*x4^17*z^31 + x1^56*x2^50*x3^32*x4^17*z^31 + 2*x1^55*x2^51*x3^32*x4^17*z^31 + x1^61*x2^44*x3^33*x4^17*z^31 + 2*x1^60*x2^45*x3^33*x4^17*z^31 + x1^58*x2^47*x3^33*x4^17*z^31 - x1^57*x2^48*x3^33*x4^17*z^31 - 2*x1^55*x2^50*x3^33*x4^17*z^31 - x1^54*x2^51*x3^33*x4^17*z^31 - 2*x1^60*x2^44*x3^34*x4^17*z^31 - x1^59*x2^45*x3^34*x4^17*z^31 - x1^58*x2^46*x3^34*x4^17*z^31 + x1^56*x2^48*x3^34*x4^17*z^31 + x1^55*x2^49*x3^34*x4^17*z^31 + 2*x1^54*x2^50*x3^34*x4^17*z^31 + 2*x1^59*x2^44*x3^35*x4^17*z^31 + x1^57*x2^46*x3^35*x4^17*z^31 - 2*x1^56*x2^47*x3^35*x4^17*z^31 - 2*x1^54*x2^49*x3^35*x4^17*z^31 - x1^58*x2^44*x3^36*x4^17*z^31 - x1^56*x2^46*x3^36*x4^17*z^31 - x1^55*x2^47*x3^36*x4^17*z^31 + x1^54*x2^48*x3^36*x4^17*z^31 + x1^53*x2^49*x3^36*x4^17*z^31 + x1^57*x2^44*x3^37*x4^17*z^31 + 2*x1^56*x2^45*x3^37*x4^17*z^31 + x1^55*x2^46*x3^37*x4^17*z^31 - x1^54*x2^47*x3^37*x4^17*z^31 - x1^53*x2^48*x3^37*x4^17*z^31 - x1^56*x2^44*x3^38*x4^17*z^31 - x1^54*x2^46*x3^38*x4^17*z^31 + x1^52*x2^48*x3^38*x4^17*z^31 + x1^66*x2^49*x3^29*x4^6*z^30 + x1^65*x2^50*x3^29*x4^6*z^30 + x1^64*x2^50*x3^30*x4^6*z^30 + x1^65*x2^51*x3^27*x4^7*z^30 - x1^66*x2^49*x3^28*x4^7*z^30 + x1^64*x2^50*x3^29*x4^7*z^30 - x1^65*x2^48*x3^30*x4^7*z^30 + x1^62*x2^51*x3^30*x4^7*z^30 + x1^62*x2^50*x3^31*x4^7*z^30 + x1^61*x2^51*x3^31*x4^7*z^30 + x1^64*x2^51*x3^27*x4^8*z^30 - 2*x1^64*x2^50*x3^28*x4^8*z^30 + x1^62*x2^52*x3^28*x4^8*z^30 + 2*x1^63*x2^50*x3^29*x4^8*z^30 - x1^62*x2^51*x3^29*x4^8*z^30 - x1^64*x2^48*x3^30*x4^8*z^30 - x1^63*x2^49*x3^30*x4^8*z^30 - 2*x1^62*x2^50*x3^30*x4^8*z^30 - x1^61*x2^51*x3^30*x4^8*z^30 - x1^60*x2^52*x3^30*x4^8*z^30 + 2*x1^62*x2^49*x3^31*x4^8*z^30 - 2*x1^63*x2^47*x3^32*x4^8*z^30 - x1^62*x2^48*x3^32*x4^8*z^30 - x1^61*x2^49*x3^32*x4^8*z^30 - x1^60*x2^50*x3^32*x4^8*z^30 + x1^64*x2^50*x3^27*x4^9*z^30 - x1^65*x2^48*x3^28*x4^9*z^30 + x1^64*x2^49*x3^28*x4^9*z^30 - 2*x1^63*x2^50*x3^28*x4^9*z^30 - x1^62*x2^51*x3^28*x4^9*z^30 - x1^61*x2^52*x3^28*x4^9*z^30 + 2*x1^62*x2^50*x3^29*x4^9*z^30 - x1^61*x2^51*x3^29*x4^9*z^30 - 2*x1^62*x2^49*x3^30*x4^9*z^30 + x1^61*x2^50*x3^30*x4^9*z^30 + x1^60*x2^51*x3^30*x4^9*z^30 + 2*x1^63*x2^47*x3^31*x4^9*z^30 - x1^62*x2^48*x3^31*x4^9*z^30 + x1^61*x2^49*x3^31*x4^9*z^30 + x1^63*x2^46*x3^32*x4^9*z^30 - x1^61*x2^48*x3^32*x4^9*z^30 - x1^60*x2^49*x3^32*x4^9*z^30 - x1^59*x2^50*x3^32*x4^9*z^30 + 2*x1^62*x2^46*x3^33*x4^9*z^30 + x1^61*x2^47*x3^33*x4^9*z^30 + x1^60*x2^48*x3^33*x4^9*z^30 + x1^58*x2^50*x3^33*x4^9*z^30 + x1^65*x2^49*x3^26*x4^10*z^30 - x1^65*x2^48*x3^27*x4^10*z^30 - x1^64*x2^49*x3^27*x4^10*z^30 + x1^63*x2^50*x3^27*x4^10*z^30 + 3*x1^64*x2^48*x3^28*x4^10*z^30 - x1^63*x2^49*x3^28*x4^10*z^30 + 2*x1^61*x2^51*x3^28*x4^10*z^30 - x1^60*x2^52*x3^28*x4^10*z^30 - 3*x1^63*x2^48*x3^29*x4^10*z^30 - x1^62*x2^49*x3^29*x4^10*z^30 - x1^61*x2^50*x3^29*x4^10*z^30 - x1^59*x2^52*x3^29*x4^10*z^30 + x1^63*x2^47*x3^30*x4^10*z^30 + 3*x1^62*x2^48*x3^30*x4^10*z^30 + x1^61*x2^49*x3^30*x4^10*z^30 + x1^60*x2^50*x3^30*x4^10*z^30 + x1^59*x2^51*x3^30*x4^10*z^30 - x1^57*x2^53*x3^30*x4^10*z^30 - x1^62*x2^47*x3^31*x4^10*z^30 - 3*x1^61*x2^48*x3^31*x4^10*z^30 - 2*x1^60*x2^49*x3^31*x4^10*z^30 - x1^59*x2^50*x3^31*x4^10*z^30 + x1^57*x2^52*x3^31*x4^10*z^30 + 2*x1^61*x2^47*x3^32*x4^10*z^30 + 2*x1^60*x2^48*x3^32*x4^10*z^30 + x1^59*x2^49*x3^32*x4^10*z^30 + 2*x1^58*x2^50*x3^32*x4^10*z^30 - x1^62*x2^45*x3^33*x4^10*z^30 + x1^61*x2^46*x3^33*x4^10*z^30 - x1^60*x2^47*x3^33*x4^10*z^30 + x1^59*x2^48*x3^33*x4^10*z^30 - 2*x1^57*x2^50*x3^33*x4^10*z^30 + x1^59*x2^47*x3^34*x4^10*z^30 + x1^58*x2^48*x3^34*x4^10*z^30 + x1^57*x2^49*x3^34*x4^10*z^30 - 2*x1^65*x2^49*x3^25*x4^11*z^30 + x1^65*x2^48*x3^26*x4^11*z^30 + 2*x1^64*x2^49*x3^26*x4^11*z^30 - 2*x1^63*x2^50*x3^26*x4^11*z^30 - 4*x1^64*x2^48*x3^27*x4^11*z^30 - x1^63*x2^49*x3^27*x4^11*z^30 + x1^64*x2^47*x3^28*x4^11*z^30 + 4*x1^63*x2^48*x3^28*x4^11*z^30 + x1^61*x2^50*x3^28*x4^11*z^30 - x1^60*x2^51*x3^28*x4^11*z^30 - 4*x1^63*x2^47*x3^29*x4^11*z^30 + 2*x1^62*x2^48*x3^29*x4^11*z^30 - 2*x1^60*x2^50*x3^29*x4^11*z^30 + x1^59*x2^51*x3^29*x4^11*z^30 + 2*x1^58*x2^52*x3^29*x4^11*z^30 + x1^63*x2^46*x3^30*x4^11*z^30 + 3*x1^62*x2^47*x3^30*x4^11*z^30 - x1^61*x2^48*x3^30*x4^11*z^30 + 4*x1^60*x2^49*x3^30*x4^11*z^30 - x1^58*x2^51*x3^30*x4^11*z^30 - 3*x1^57*x2^52*x3^30*x4^11*z^30 - 4*x1^62*x2^46*x3^31*x4^11*z^30 - 2*x1^61*x2^47*x3^31*x4^11*z^30 + x1^58*x2^50*x3^31*x4^11*z^30 + x1^56*x2^52*x3^31*x4^11*z^30 + x1^62*x2^45*x3^32*x4^11*z^30 + x1^61*x2^46*x3^32*x4^11*z^30 + 2*x1^59*x2^48*x3^32*x4^11*z^30 - x1^58*x2^49*x3^32*x4^11*z^30 + x1^57*x2^50*x3^32*x4^11*z^30 - 2*x1^56*x2^51*x3^32*x4^11*z^30 - x1^55*x2^52*x3^32*x4^11*z^30 - 3*x1^61*x2^45*x3^33*x4^11*z^30 - x1^60*x2^46*x3^33*x4^11*z^30 - 2*x1^59*x2^47*x3^33*x4^11*z^30 + x1^57*x2^49*x3^33*x4^11*z^30 + x1^61*x2^44*x3^34*x4^11*z^30 + x1^56*x2^49*x3^34*x4^11*z^30 - 2*x1^60*x2^44*x3^35*x4^11*z^30 - x1^58*x2^46*x3^35*x4^11*z^30 - 2*x1^57*x2^47*x3^35*x4^11*z^30 - x1^56*x2^48*x3^35*x4^11*z^30 + 2*x1^64*x2^48*x3^26*x4^12*z^30 + x1^62*x2^50*x3^26*x4^12*z^30 - x1^64*x2^47*x3^27*x4^12*z^30 - 2*x1^63*x2^48*x3^27*x4^12*z^30 + 3*x1^62*x2^49*x3^27*x4^12*z^30 - x1^61*x2^50*x3^27*x4^12*z^30 + x1^60*x2^51*x3^27*x4^12*z^30 + 2*x1^63*x2^47*x3^28*x4^12*z^30 - 3*x1^62*x2^48*x3^28*x4^12*z^30 - 3*x1^61*x2^49*x3^28*x4^12*z^30 + 2*x1^60*x2^50*x3^28*x4^12*z^30 - x1^59*x2^51*x3^28*x4^12*z^30 - 2*x1^62*x2^47*x3^29*x4^12*z^30 + 7*x1^61*x2^48*x3^29*x4^12*z^30 + x1^59*x2^50*x3^29*x4^12*z^30 + 2*x1^62*x2^46*x3^30*x4^12*z^30 - 2*x1^61*x2^47*x3^30*x4^12*z^30 - 7*x1^60*x2^48*x3^30*x4^12*z^30 + 2*x1^59*x2^49*x3^30*x4^12*z^30 - 3*x1^58*x2^50*x3^30*x4^12*z^30 + x1^57*x2^51*x3^30*x4^12*z^30 - x1^56*x2^52*x3^30*x4^12*z^30 - x1^62*x2^45*x3^31*x4^12*z^30 + 4*x1^60*x2^47*x3^31*x4^12*z^30 + 2*x1^58*x2^49*x3^31*x4^12*z^30 + x1^57*x2^50*x3^31*x4^12*z^30 + 2*x1^61*x2^45*x3^32*x4^12*z^30 - 2*x1^60*x2^46*x3^32*x4^12*z^30 - 3*x1^59*x2^47*x3^32*x4^12*z^30 + x1^58*x2^48*x3^32*x4^12*z^30 - 5*x1^57*x2^49*x3^32*x4^12*z^30 - x1^60*x2^45*x3^33*x4^12*z^30 + 3*x1^59*x2^46*x3^33*x4^12*z^30 + 3*x1^57*x2^48*x3^33*x4^12*z^30 + x1^56*x2^49*x3^33*x4^12*z^30 + x1^55*x2^50*x3^33*x4^12*z^30 + x1^60*x2^44*x3^34*x4^12*z^30 - 2*x1^59*x2^45*x3^34*x4^12*z^30 - 2*x1^58*x2^46*x3^34*x4^12*z^30 - 3*x1^56*x2^48*x3^34*x4^12*z^30 + x1^60*x2^43*x3^35*x4^12*z^30 + x1^59*x2^44*x3^35*x4^12*z^30 + x1^55*x2^48*x3^35*x4^12*z^30 + x1^59*x2^43*x3^36*x4^12*z^30 + x1^58*x2^44*x3^36*x4^12*z^30 + x1^56*x2^46*x3^36*x4^12*z^30 - x1^64*x2^48*x3^25*x4^13*z^30 + 2*x1^63*x2^49*x3^25*x4^13*z^30 + x1^63*x2^48*x3^26*x4^13*z^30 - 3*x1^62*x2^49*x3^26*x4^13*z^30 + 2*x1^61*x2^50*x3^26*x4^13*z^30 - x1^63*x2^47*x3^27*x4^13*z^30 + x1^62*x2^48*x3^27*x4^13*z^30 + 2*x1^61*x2^49*x3^27*x4^13*z^30 - x1^60*x2^50*x3^27*x4^13*z^30 + 2*x1^59*x2^51*x3^27*x4^13*z^30 + x1^62*x2^47*x3^28*x4^13*z^30 - 4*x1^61*x2^48*x3^28*x4^13*z^30 + 2*x1^57*x2^52*x3^28*x4^13*z^30 + 2*x1^61*x2^47*x3^29*x4^13*z^30 + 4*x1^60*x2^48*x3^29*x4^13*z^30 - 2*x1^59*x2^49*x3^29*x4^13*z^30 + x1^58*x2^50*x3^29*x4^13*z^30 - x1^57*x2^51*x3^29*x4^13*z^30 - 2*x1^56*x2^52*x3^29*x4^13*z^30 - 6*x1^60*x2^47*x3^30*x4^13*z^30 - x1^59*x2^48*x3^30*x4^13*z^30 + x1^58*x2^49*x3^30*x4^13*z^30 + x1^57*x2^50*x3^30*x4^13*z^30 + 2*x1^56*x2^51*x3^30*x4^13*z^30 + 2*x1^55*x2^52*x3^30*x4^13*z^30 + 2*x1^60*x2^46*x3^31*x4^13*z^30 + 6*x1^59*x2^47*x3^31*x4^13*z^30 - x1^58*x2^48*x3^31*x4^13*z^30 + 3*x1^57*x2^49*x3^31*x4^13*z^30 - 2*x1^56*x2^50*x3^31*x4^13*z^30 - x1^55*x2^51*x3^31*x4^13*z^30 - 3*x1^54*x2^52*x3^31*x4^13*z^30 - x1^60*x2^45*x3^32*x4^13*z^30 - 5*x1^59*x2^46*x3^32*x4^13*z^30 - 3*x1^58*x2^47*x3^32*x4^13*z^30 - 2*x1^57*x2^48*x3^32*x4^13*z^30 - x1^56*x2^49*x3^32*x4^13*z^30 + x1^55*x2^50*x3^32*x4^13*z^30 + 2*x1^54*x2^51*x3^32*x4^13*z^30 + 2*x1^53*x2^52*x3^32*x4^13*z^30 + 4*x1^59*x2^45*x3^33*x4^13*z^30 + 5*x1^58*x2^46*x3^33*x4^13*z^30 - x1^57*x2^47*x3^33*x4^13*z^30 + 5*x1^56*x2^48*x3^33*x4^13*z^30 - x1^54*x2^50*x3^33*x4^13*z^30 - 3*x1^53*x2^51*x3^33*x4^13*z^30 - x1^60*x2^43*x3^34*x4^13*z^30 - x1^59*x2^44*x3^34*x4^13*z^30 - 4*x1^58*x2^45*x3^34*x4^13*z^30 - x1^57*x2^46*x3^34*x4^13*z^30 - 2*x1^56*x2^47*x3^34*x4^13*z^30 - 2*x1^55*x2^48*x3^34*x4^13*z^30 + 2*x1^54*x2^49*x3^34*x4^13*z^30 + x1^53*x2^50*x3^34*x4^13*z^30 + 3*x1^58*x2^44*x3^35*x4^13*z^30 + 3*x1^57*x2^45*x3^35*x4^13*z^30 + 4*x1^55*x2^47*x3^35*x4^13*z^30 - x1^58*x2^43*x3^36*x4^13*z^30 - 2*x1^57*x2^44*x3^36*x4^13*z^30 - x1^56*x2^45*x3^36*x4^13*z^30 - x1^55*x2^46*x3^36*x4^13*z^30 - 2*x1^54*x2^47*x3^36*x4^13*z^30 + 2*x1^56*x2^44*x3^37*x4^13*z^30 + 2*x1^55*x2^45*x3^37*x4^13*z^30 + x1^54*x2^46*x3^37*x4^13*z^30 + x1^64*x2^47*x3^25*x4^14*z^30 - x1^62*x2^48*x3^26*x4^14*z^30 - x1^63*x2^46*x3^27*x4^14*z^30 + 3*x1^61*x2^48*x3^27*x4^14*z^30 - x1^60*x2^49*x3^27*x4^14*z^30 - x1^59*x2^50*x3^27*x4^14*z^30 + 2*x1^62*x2^46*x3^28*x4^14*z^30 - x1^61*x2^47*x3^28*x4^14*z^30 - 2*x1^60*x2^48*x3^28*x4^14*z^30 - x1^58*x2^50*x3^28*x4^14*z^30 - x1^57*x2^51*x3^28*x4^14*z^30 - x1^62*x2^45*x3^29*x4^14*z^30 - 2*x1^61*x2^46*x3^29*x4^14*z^30 + 2*x1^60*x2^47*x3^29*x4^14*z^30 + x1^59*x2^48*x3^29*x4^14*z^30 + 2*x1^58*x2^49*x3^29*x4^14*z^30 - x1^57*x2^50*x3^29*x4^14*z^30 - x1^56*x2^51*x3^29*x4^14*z^30 - x1^55*x2^52*x3^29*x4^14*z^30 + 2*x1^61*x2^45*x3^30*x4^14*z^30 - x1^60*x2^46*x3^30*x4^14*z^30 - 2*x1^59*x2^47*x3^30*x4^14*z^30 + 4*x1^56*x2^50*x3^30*x4^14*z^30 + x1^55*x2^51*x3^30*x4^14*z^30 + x1^54*x2^52*x3^30*x4^14*z^30 - 2*x1^60*x2^45*x3^31*x4^14*z^30 + 2*x1^59*x2^46*x3^31*x4^14*z^30 - 2*x1^58*x2^47*x3^31*x4^14*z^30 - x1^57*x2^48*x3^31*x4^14*z^30 + x1^56*x2^49*x3^31*x4^14*z^30 + x1^55*x2^50*x3^31*x4^14*z^30 - x1^53*x2^52*x3^31*x4^14*z^30 + 2*x1^60*x2^44*x3^32*x4^14*z^30 + x1^59*x2^45*x3^32*x4^14*z^30 - x1^58*x2^46*x3^32*x4^14*z^30 + x1^57*x2^47*x3^32*x4^14*z^30 - 2*x1^56*x2^48*x3^32*x4^14*z^30 + x1^55*x2^49*x3^32*x4^14*z^30 + 2*x1^54*x2^50*x3^32*x4^14*z^30 + 2*x1^53*x2^51*x3^32*x4^14*z^30 + x1^52*x2^52*x3^32*x4^14*z^30 - x1^60*x2^43*x3^33*x4^14*z^30 - x1^59*x2^44*x3^33*x4^14*z^30 + 3*x1^58*x2^45*x3^33*x4^14*z^30 - x1^56*x2^47*x3^33*x4^14*z^30 + x1^55*x2^48*x3^33*x4^14*z^30 + x1^54*x2^49*x3^33*x4^14*z^30 - x1^52*x2^51*x3^33*x4^14*z^30 + 2*x1^59*x2^43*x3^34*x4^14*z^30 - x1^58*x2^44*x3^34*x4^14*z^30 + x1^56*x2^46*x3^34*x4^14*z^30 - 3*x1^55*x2^47*x3^34*x4^14*z^30 - x1^54*x2^48*x3^34*x4^14*z^30 + x1^53*x2^49*x3^34*x4^14*z^30 + 2*x1^52*x2^50*x3^34*x4^14*z^30 - x1^58*x2^43*x3^35*x4^14*z^30 + x1^57*x2^44*x3^35*x4^14*z^30 + x1^56*x2^45*x3^35*x4^14*z^30 + x1^55*x2^46*x3^35*x4^14*z^30 - 2*x1^53*x2^48*x3^35*x4^14*z^30 - x1^52*x2^49*x3^35*x4^14*z^30 - x1^57*x2^43*x3^36*x4^14*z^30 + x1^55*x2^45*x3^36*x4^14*z^30 - x1^54*x2^46*x3^36*x4^14*z^30 + x1^53*x2^46*x3^37*x4^14*z^30 - x1^54*x2^44*x3^38*x4^14*z^30 + x1^64*x2^47*x3^24*x4^15*z^30 - x1^63*x2^48*x3^24*x4^15*z^30 - 2*x1^63*x2^47*x3^25*x4^15*z^30 + x1^62*x2^48*x3^25*x4^15*z^30 + 2*x1^63*x2^46*x3^26*x4^15*z^30 + 2*x1^62*x2^47*x3^26*x4^15*z^30 - x1^61*x2^48*x3^26*x4^15*z^30 - x1^59*x2^50*x3^26*x4^15*z^30 - 6*x1^62*x2^46*x3^27*x4^15*z^30 - x1^61*x2^47*x3^27*x4^15*z^30 - x1^60*x2^48*x3^27*x4^15*z^30 + x1^59*x2^49*x3^27*x4^15*z^30 + 2*x1^58*x2^50*x3^27*x4^15*z^30 + 2*x1^62*x2^45*x3^28*x4^15*z^30 + 6*x1^61*x2^46*x3^28*x4^15*z^30 - x1^60*x2^47*x3^28*x4^15*z^30 + 2*x1^59*x2^48*x3^28*x4^15*z^30 - 3*x1^58*x2^49*x3^28*x4^15*z^30 - 2*x1^57*x2^50*x3^28*x4^15*z^30 - x1^56*x2^51*x3^28*x4^15*z^30 - 6*x1^61*x2^45*x3^29*x4^15*z^30 - 2*x1^60*x2^46*x3^29*x4^15*z^30 - x1^59*x2^47*x3^29*x4^15*z^30 + 3*x1^57*x2^49*x3^29*x4^15*z^30 + 2*x1^56*x2^50*x3^29*x4^15*z^30 + x1^55*x2^51*x3^29*x4^15*z^30 + 2*x1^61*x2^44*x3^30*x4^15*z^30 + 6*x1^60*x2^45*x3^30*x4^15*z^30 + 4*x1^58*x2^47*x3^30*x4^15*z^30 - 3*x1^57*x2^48*x3^30*x4^15*z^30 - 2*x1^56*x2^49*x3^30*x4^15*z^30 - 4*x1^55*x2^50*x3^30*x4^15*z^30 - 6*x1^60*x2^44*x3^31*x4^15*z^30 - 2*x1^59*x2^45*x3^31*x4^15*z^30 - 2*x1^58*x2^46*x3^31*x4^15*z^30 + x1^56*x2^48*x3^31*x4^15*z^30 + 2*x1^55*x2^49*x3^31*x4^15*z^30 + 3*x1^54*x2^50*x3^31*x4^15*z^30 + 2*x1^60*x2^43*x3^32*x4^15*z^30 + 6*x1^59*x2^44*x3^32*x4^15*z^30 + 4*x1^57*x2^46*x3^32*x4^15*z^30 - x1^56*x2^47*x3^32*x4^15*z^30 - x1^55*x2^48*x3^32*x4^15*z^30 - 5*x1^54*x2^49*x3^32*x4^15*z^30 - 6*x1^59*x2^43*x3^33*x4^15*z^30 - 2*x1^58*x2^44*x3^33*x4^15*z^30 - 2*x1^57*x2^45*x3^33*x4^15*z^30 - 2*x1^56*x2^46*x3^33*x4^15*z^30 + 2*x1^54*x2^48*x3^33*x4^15*z^30 + 2*x1^53*x2^49*x3^33*x4^15*z^30 + x1^59*x2^42*x3^34*x4^15*z^30 + 5*x1^58*x2^43*x3^34*x4^15*z^30 - x1^57*x2^44*x3^34*x4^15*z^30 + 4*x1^56*x2^45*x3^34*x4^15*z^30 + x1^55*x2^46*x3^34*x4^15*z^30 - x1^54*x2^47*x3^34*x4^15*z^30 - 3*x1^53*x2^48*x3^34*x4^15*z^30 - 3*x1^58*x2^42*x3^35*x4^15*z^30 - 2*x1^56*x2^44*x3^35*x4^15*z^30 - x1^55*x2^45*x3^35*x4^15*z^30 + x1^52*x2^48*x3^35*x4^15*z^30 + 2*x1^57*x2^42*x3^36*x4^15*z^30 + x1^56*x2^43*x3^36*x4^15*z^30 + 3*x1^55*x2^44*x3^36*x4^15*z^30 + x1^54*x2^45*x3^36*x4^15*z^30 - x1^54*x2^44*x3^37*x4^15*z^30 + 2*x1^63*x2^47*x3^24*x4^16*z^30 - 2*x1^63*x2^46*x3^25*x4^16*z^30 - 2*x1^62*x2^47*x3^25*x4^16*z^30 + x1^61*x2^48*x3^25*x4^16*z^30 - 2*x1^60*x2^49*x3^25*x4^16*z^30 + 5*x1^62*x2^46*x3^26*x4^16*z^30 + x1^61*x2^47*x3^26*x4^16*z^30 + x1^60*x2^48*x3^26*x4^16*z^30 + x1^59*x2^49*x3^26*x4^16*z^30 - x1^58*x2^50*x3^26*x4^16*z^30 - 2*x1^62*x2^45*x3^27*x4^16*z^30 - 5*x1^61*x2^46*x3^27*x4^16*z^30 - 3*x1^59*x2^48*x3^27*x4^16*z^30 + x1^57*x2^50*x3^27*x4^16*z^30 + 6*x1^61*x2^45*x3^28*x4^16*z^30 + 2*x1^60*x2^46*x3^28*x4^16*z^30 + 2*x1^59*x2^47*x3^28*x4^16*z^30 + x1^58*x2^48*x3^28*x4^16*z^30 - x1^56*x2^50*x3^28*x4^16*z^30 - 2*x1^61*x2^44*x3^29*x4^16*z^30 - 6*x1^60*x2^45*x3^29*x4^16*z^30 - 3*x1^58*x2^47*x3^29*x4^16*z^30 + 3*x1^57*x2^48*x3^29*x4^16*z^30 + 4*x1^55*x2^50*x3^29*x4^16*z^30 + 6*x1^60*x2^44*x3^30*x4^16*z^30 + 2*x1^59*x2^45*x3^30*x4^16*z^30 + 2*x1^58*x2^46*x3^30*x4^16*z^30 - x1^56*x2^48*x3^30*x4^16*z^30 - x1^55*x2^49*x3^30*x4^16*z^30 - 4*x1^54*x2^50*x3^30*x4^16*z^30 - 2*x1^60*x2^43*x3^31*x4^16*z^30 - 6*x1^59*x2^44*x3^31*x4^16*z^30 - 4*x1^57*x2^46*x3^31*x4^16*z^30 + 4*x1^56*x2^47*x3^31*x4^16*z^30 + 6*x1^54*x2^49*x3^31*x4^16*z^30 + x1^53*x2^50*x3^31*x4^16*z^30 + 6*x1^59*x2^43*x3^32*x4^16*z^30 + 2*x1^58*x2^44*x3^32*x4^16*z^30 + 2*x1^57*x2^45*x3^32*x4^16*z^30 - 2*x1^55*x2^47*x3^32*x4^16*z^30 - 2*x1^54*x2^48*x3^32*x4^16*z^30 - 6*x1^53*x2^49*x3^32*x4^16*z^30 - x1^59*x2^42*x3^33*x4^16*z^30 - 6*x1^58*x2^43*x3^33*x4^16*z^30 - 4*x1^56*x2^45*x3^33*x4^16*z^30 + 3*x1^55*x2^46*x3^33*x4^16*z^30 + x1^54*x2^47*x3^33*x4^16*z^30 + 5*x1^53*x2^48*x3^33*x4^16*z^30 + x1^52*x2^49*x3^33*x4^16*z^30 + 4*x1^58*x2^42*x3^34*x4^16*z^30 + 3*x1^57*x2^43*x3^34*x4^16*z^30 + 2*x1^56*x2^44*x3^34*x4^16*z^30 + x1^55*x2^45*x3^34*x4^16*z^30 - 2*x1^53*x2^47*x3^34*x4^16*z^30 - 4*x1^52*x2^48*x3^34*x4^16*z^30 - 4*x1^57*x2^42*x3^35*x4^16*z^30 - 2*x1^56*x2^43*x3^35*x4^16*z^30 - 3*x1^55*x2^44*x3^35*x4^16*z^30 + 4*x1^52*x2^47*x3^35*x4^16*z^30 + x1^57*x2^41*x3^36*x4^16*z^30 + 2*x1^56*x2^42*x3^36*x4^16*z^30 + 2*x1^55*x2^43*x3^36*x4^16*z^30 + x1^53*x2^45*x3^36*x4^16*z^30 - x1^52*x2^46*x3^36*x4^16*z^30 - 2*x1^51*x2^47*x3^36*x4^16*z^30 - x1^56*x2^41*x3^37*x4^16*z^30 - 2*x1^55*x2^42*x3^37*x4^16*z^30 - 2*x1^54*x2^43*x3^37*x4^16*z^30 + x1^52*x2^45*x3^37*x4^16*z^30 + x1^51*x2^46*x3^37*x4^16*z^30 + x1^54*x2^42*x3^38*x4^16*z^30 + x1^53*x2^43*x3^38*x4^16*z^30 - x1^62*x2^46*x3^25*x4^17*z^30 + x1^60*x2^48*x3^25*x4^17*z^30 + x1^61*x2^46*x3^26*x4^17*z^30 - x1^60*x2^47*x3^26*x4^17*z^30 - 2*x1^61*x2^45*x3^27*x4^17*z^30 - x1^60*x2^46*x3^27*x4^17*z^30 - x1^59*x2^47*x3^27*x4^17*z^30 + x1^57*x2^49*x3^27*x4^17*z^30 + x1^61*x2^44*x3^28*x4^17*z^30 + 2*x1^60*x2^45*x3^28*x4^17*z^30 + x1^58*x2^47*x3^28*x4^17*z^30 - x1^57*x2^48*x3^28*x4^17*z^30 - x1^56*x2^49*x3^28*x4^17*z^30 - x1^55*x2^50*x3^28*x4^17*z^30 - 2*x1^60*x2^44*x3^29*x4^17*z^30 - x1^59*x2^45*x3^29*x4^17*z^30 - x1^58*x2^46*x3^29*x4^17*z^30 + x1^56*x2^48*x3^29*x4^17*z^30 + x1^55*x2^49*x3^29*x4^17*z^30 + x1^54*x2^50*x3^29*x4^17*z^30 + x1^60*x2^43*x3^30*x4^17*z^30 + 2*x1^59*x2^44*x3^30*x4^17*z^30 + x1^57*x2^46*x3^30*x4^17*z^30 - 2*x1^56*x2^47*x3^30*x4^17*z^30 - 2*x1^54*x2^49*x3^30*x4^17*z^30 - 2*x1^59*x2^43*x3^31*x4^17*z^30 + 2*x1^53*x2^49*x3^31*x4^17*z^30 + 2*x1^58*x2^43*x3^32*x4^17*z^30 + 2*x1^56*x2^45*x3^32*x4^17*z^30 - x1^55*x2^46*x3^32*x4^17*z^30 - 2*x1^53*x2^48*x3^32*x4^17*z^30 - x1^52*x2^49*x3^32*x4^17*z^30 - x1^58*x2^42*x3^33*x4^17*z^30 - x1^57*x2^43*x3^33*x4^17*z^30 - x1^56*x2^44*x3^33*x4^17*z^30 + x1^54*x2^46*x3^33*x4^17*z^30 + x1^53*x2^47*x3^33*x4^17*z^30 + 2*x1^52*x2^48*x3^33*x4^17*z^30 + x1^57*x2^42*x3^34*x4^17*z^30 + x1^56*x2^43*x3^34*x4^17*z^30 + x1^55*x2^44*x3^34*x4^17*z^30 - x1^54*x2^45*x3^34*x4^17*z^30 - 2*x1^52*x2^47*x3^34*x4^17*z^30 - x1^51*x2^48*x3^34*x4^17*z^30 - x1^55*x2^43*x3^35*x4^17*z^30 + x1^53*x2^45*x3^35*x4^17*z^30 + x1^52*x2^46*x3^35*x4^17*z^30 + 2*x1^51*x2^47*x3^35*x4^17*z^30 + x1^52*x2^45*x3^36*x4^17*z^30 - x1^51*x2^46*x3^36*x4^17*z^30 - x1^52*x2^44*x3^37*x4^17*z^30 + x1^50*x2^46*x3^37*x4^17*z^30 - x1^65*x2^48*x3^27*x4^5*z^29 - x1^64*x2^48*x3^27*x4^6*z^29 + x1^64*x2^47*x3^28*x4^6*z^29 - x1^63*x2^47*x3^29*x4^6*z^29 - x1^62*x2^48*x3^29*x4^6*z^29 - 2*x1^61*x2^49*x3^29*x4^6*z^29 - x1^61*x2^48*x3^30*x4^6*z^29 + x1^64*x2^47*x3^27*x4^7*z^29 + x1^63*x2^48*x3^27*x4^7*z^29 - 2*x1^62*x2^49*x3^27*x4^7*z^29 + 2*x1^63*x2^47*x3^28*x4^7*z^29 + x1^62*x2^48*x3^28*x4^7*z^29 - x1^61*x2^48*x3^29*x4^7*z^29 - x1^60*x2^49*x3^29*x4^7*z^29 + 2*x1^62*x2^46*x3^30*x4^7*z^29 + x1^60*x2^48*x3^30*x4^7*z^29 - x1^59*x2^49*x3^30*x4^7*z^29 - x1^58*x2^50*x3^30*x4^7*z^29 - x1^58*x2^49*x3^31*x4^7*z^29 - x1^63*x2^49*x3^25*x4^8*z^29 - x1^64*x2^47*x3^26*x4^8*z^29 + 3*x1^62*x2^49*x3^26*x4^8*z^29 - 2*x1^61*x2^50*x3^26*x4^8*z^29 - x1^63*x2^47*x3^27*x4^8*z^29 - x1^62*x2^48*x3^27*x4^8*z^29 - 2*x1^61*x2^49*x3^27*x4^8*z^29 + x1^60*x2^50*x3^27*x4^8*z^29 + x1^63*x2^46*x3^28*x4^8*z^29 + 3*x1^61*x2^48*x3^28*x4^8*z^29 - x1^58*x2^51*x3^28*x4^8*z^29 - x1^62*x2^46*x3^29*x4^8*z^29 - x1^60*x2^48*x3^29*x4^8*z^29 + x1^59*x2^49*x3^29*x4^8*z^29 - x1^58*x2^50*x3^29*x4^8*z^29 + x1^61*x2^46*x3^30*x4^8*z^29 + 2*x1^60*x2^47*x3^30*x4^8*z^29 + x1^59*x2^48*x3^30*x4^8*z^29 + x1^58*x2^49*x3^30*x4^8*z^29 + x1^57*x2^50*x3^30*x4^8*z^29 - 2*x1^61*x2^45*x3^31*x4^8*z^29 + x1^60*x2^46*x3^31*x4^8*z^29 - x1^59*x2^47*x3^31*x4^8*z^29 - x1^58*x2^48*x3^31*x4^8*z^29 + x1^60*x2^45*x3^32*x4^8*z^29 + 2*x1^59*x2^46*x3^32*x4^8*z^29 + 2*x1^58*x2^47*x3^32*x4^8*z^29 + x1^57*x2^48*x3^32*x4^8*z^29 - x1^62*x2^49*x3^25*x4^9*z^29 + x1^63*x2^47*x3^26*x4^9*z^29 + x1^61*x2^49*x3^26*x4^9*z^29 + x1^63*x2^46*x3^27*x4^9*z^29 - 3*x1^61*x2^48*x3^27*x4^9*z^29 - x1^59*x2^50*x3^27*x4^9*z^29 - x1^58*x2^51*x3^27*x4^9*z^29 + x1^61*x2^47*x3^28*x4^9*z^29 + 2*x1^60*x2^48*x3^28*x4^9*z^29 - x1^59*x2^49*x3^28*x4^9*z^29 + x1^58*x2^50*x3^28*x4^9*z^29 + x1^57*x2^51*x3^28*x4^9*z^29 - 2*x1^62*x2^45*x3^29*x4^9*z^29 - x1^60*x2^47*x3^29*x4^9*z^29 - 2*x1^58*x2^49*x3^29*x4^9*z^29 - x1^56*x2^51*x3^29*x4^9*z^29 + 2*x1^61*x2^45*x3^30*x4^9*z^29 - 2*x1^60*x2^46*x3^30*x4^9*z^29 + x1^59*x2^47*x3^30*x4^9*z^29 - 2*x1^56*x2^50*x3^30*x4^9*z^29 - 2*x1^60*x2^45*x3^31*x4^9*z^29 - 2*x1^58*x2^47*x3^31*x4^9*z^29 - x1^57*x2^48*x3^31*x4^9*z^29 - x1^59*x2^45*x3^32*x4^9*z^29 - x1^58*x2^46*x3^32*x4^9*z^29 + x1^56*x2^48*x3^32*x4^9*z^29 - x1^59*x2^44*x3^33*x4^9*z^29 - 2*x1^58*x2^45*x3^33*x4^9*z^29 - 2*x1^57*x2^46*x3^33*x4^9*z^29 - x1^56*x2^47*x3^33*x4^9*z^29 - x1^64*x2^47*x3^24*x4^10*z^29 + 2*x1^63*x2^47*x3^25*x4^10*z^29 - x1^62*x2^48*x3^25*x4^10*z^29 - x1^63*x2^46*x3^26*x4^10*z^29 - 2*x1^62*x2^47*x3^26*x4^10*z^29 + x1^61*x2^48*x3^26*x4^10*z^29 - 2*x1^60*x2^49*x3^26*x4^10*z^29 + 2*x1^62*x2^46*x3^27*x4^10*z^29 + 2*x1^61*x2^47*x3^27*x4^10*z^29 + x1^59*x2^49*x3^27*x4^10*z^29 - x1^58*x2^50*x3^27*x4^10*z^29 - 2*x1^61*x2^46*x3^28*x4^10*z^29 + x1^57*x2^50*x3^28*x4^10*z^29 + x1^61*x2^45*x3^29*x4^10*z^29 + x1^60*x2^46*x3^29*x4^10*z^29 + 2*x1^59*x2^47*x3^29*x4^10*z^29 + 2*x1^58*x2^48*x3^29*x4^10*z^29 - x1^56*x2^50*x3^29*x4^10*z^29 + x1^60*x2^45*x3^30*x4^10*z^29 - 2*x1^59*x2^46*x3^30*x4^10*z^29 - 3*x1^58*x2^47*x3^30*x4^10*z^29 - x1^56*x2^49*x3^30*x4^10*z^29 + x1^55*x2^50*x3^30*x4^10*z^29 + 3*x1^58*x2^46*x3^31*x4^10*z^29 + x1^55*x2^49*x3^31*x4^10*z^29 - x1^54*x2^50*x3^31*x4^10*z^29 - x1^53*x2^51*x3^31*x4^10*z^29 + x1^59*x2^44*x3^32*x4^10*z^29 - 2*x1^58*x2^45*x3^32*x4^10*z^29 - x1^56*x2^47*x3^32*x4^10*z^29 - x1^55*x2^48*x3^32*x4^10*z^29 + x1^59*x2^43*x3^33*x4^10*z^29 + x1^58*x2^44*x3^33*x4^10*z^29 + x1^57*x2^45*x3^33*x4^10*z^29 - x1^57*x2^44*x3^34*x4^10*z^29 + x1^56*x2^45*x3^34*x4^10*z^29 - x1^55*x2^46*x3^34*x4^10*z^29 - x1^54*x2^47*x3^34*x4^10*z^29 + 2*x1^64*x2^47*x3^23*x4^11*z^29 - 3*x1^63*x2^47*x3^24*x4^11*z^29 + 2*x1^62*x2^48*x3^24*x4^11*z^29 + x1^63*x2^46*x3^25*x4^11*z^29 + 3*x1^62*x2^47*x3^25*x4^11*z^29 - x1^61*x2^48*x3^25*x4^11*z^29 + 2*x1^60*x2^49*x3^25*x4^11*z^29 - 3*x1^62*x2^46*x3^26*x4^11*z^29 - 2*x1^61*x2^47*x3^26*x4^11*z^29 + x1^59*x2^49*x3^26*x4^11*z^29 + 2*x1^58*x2^50*x3^26*x4^11*z^29 + 2*x1^62*x2^45*x3^27*x4^11*z^29 + 3*x1^61*x2^46*x3^27*x4^11*z^29 - x1^60*x2^47*x3^27*x4^11*z^29 + 2*x1^59*x2^48*x3^27*x4^11*z^29 - 2*x1^58*x2^49*x3^27*x4^11*z^29 - 2*x1^57*x2^50*x3^27*x4^11*z^29 - 4*x1^61*x2^45*x3^28*x4^11*z^29 - 3*x1^58*x2^48*x3^28*x4^11*z^29 + 3*x1^56*x2^50*x3^28*x4^11*z^29 + x1^61*x2^44*x3^29*x4^11*z^29 + 3*x1^60*x2^45*x3^29*x4^11*z^29 - x1^59*x2^46*x3^29*x4^11*z^29 - 3*x1^57*x2^48*x3^29*x4^11*z^29 + x1^56*x2^49*x3^29*x4^11*z^29 - 4*x1^55*x2^50*x3^29*x4^11*z^29 - 5*x1^60*x2^44*x3^30*x4^11*z^29 + x1^59*x2^45*x3^30*x4^11*z^29 + x1^58*x2^46*x3^30*x4^11*z^29 - 2*x1^57*x2^47*x3^30*x4^11*z^29 + x1^56*x2^48*x3^30*x4^11*z^29 + x1^55*x2^49*x3^30*x4^11*z^29 + 3*x1^54*x2^50*x3^30*x4^11*z^29 + x1^53*x2^51*x3^30*x4^11*z^29 + x1^60*x2^43*x3^31*x4^11*z^29 + 2*x1^59*x2^44*x3^31*x4^11*z^29 + 4*x1^57*x2^46*x3^31*x4^11*z^29 - x1^55*x2^48*x3^31*x4^11*z^29 - 2*x1^54*x2^49*x3^31*x4^11*z^29 + x1^53*x2^50*x3^31*x4^11*z^29 - 3*x1^59*x2^43*x3^32*x4^11*z^29 - x1^57*x2^45*x3^32*x4^11*z^29 - x1^56*x2^46*x3^32*x4^11*z^29 + 3*x1^55*x2^47*x3^32*x4^11*z^29 + 2*x1^52*x2^50*x3^32*x4^11*z^29 + x1^58*x2^43*x3^33*x4^11*z^29 + 4*x1^56*x2^45*x3^33*x4^11*z^29 + x1^55*x2^46*x3^33*x4^11*z^29 - x1^54*x2^47*x3^33*x4^11*z^29 - x1^58*x2^42*x3^34*x4^11*z^29 - x1^56*x2^44*x3^34*x4^11*z^29 + x1^54*x2^46*x3^34*x4^11*z^29 + x1^57*x2^42*x3^35*x4^11*z^29 + 2*x1^56*x2^43*x3^35*x4^11*z^29 + x1^54*x2^45*x3^35*x4^11*z^29 + x1^53*x2^46*x3^35*x4^11*z^29 - x1^63*x2^46*x3^24*x4^12*z^29 + x1^61*x2^48*x3^24*x4^12*z^29 + 2*x1^62*x2^46*x3^25*x4^12*z^29 - 2*x1^61*x2^47*x3^25*x4^12*z^29 - x1^60*x2^48*x3^25*x4^12*z^29 + x1^59*x2^49*x3^25*x4^12*z^29 - x1^62*x2^45*x3^26*x4^12*z^29 - 2*x1^61*x2^46*x3^26*x4^12*z^29 + 4*x1^60*x2^47*x3^26*x4^12*z^29 - 2*x1^59*x2^48*x3^26*x4^12*z^29 - x1^58*x2^49*x3^26*x4^12*z^29 + 2*x1^61*x2^45*x3^27*x4^12*z^29 - 3*x1^59*x2^47*x3^27*x4^12*z^29 + 3*x1^58*x2^48*x3^27*x4^12*z^29 - 2*x1^57*x2^49*x3^27*x4^12*z^29 - x1^61*x2^44*x3^28*x4^12*z^29 - 2*x1^60*x2^45*x3^28*x4^12*z^29 + 6*x1^59*x2^46*x3^28*x4^12*z^29 + x1^58*x2^47*x3^28*x4^12*z^29 + 2*x1^57*x2^48*x3^28*x4^12*z^29 + 2*x1^60*x2^44*x3^29*x4^12*z^29 - 2*x1^59*x2^45*x3^29*x4^12*z^29 - 6*x1^58*x2^46*x3^29*x4^12*z^29 + x1^57*x2^47*x3^29*x4^12*z^29 - 4*x1^56*x2^48*x3^29*x4^12*z^29 + x1^55*x2^49*x3^29*x4^12*z^29 + x1^53*x2^51*x3^29*x4^12*z^29 - x1^59*x2^44*x3^30*x4^12*z^29 + 5*x1^58*x2^45*x3^30*x4^12*z^29 + x1^57*x2^46*x3^30*x4^12*z^29 + 3*x1^56*x2^47*x3^30*x4^12*z^29 + x1^55*x2^48*x3^30*x4^12*z^29 - x1^54*x2^49*x3^30*x4^12*z^29 - 2*x1^53*x2^50*x3^30*x4^12*z^29 + 2*x1^59*x2^43*x3^31*x4^12*z^29 - 3*x1^58*x2^44*x3^31*x4^12*z^29 - 4*x1^57*x2^45*x3^31*x4^12*z^29 + x1^56*x2^46*x3^31*x4^12*z^29 - 5*x1^55*x2^47*x3^31*x4^12*z^29 + 2*x1^52*x2^50*x3^31*x4^12*z^29 - x1^59*x2^42*x3^32*x4^12*z^29 - x1^58*x2^43*x3^32*x4^12*z^29 + 4*x1^57*x2^44*x3^32*x4^12*z^29 + x1^55*x2^46*x3^32*x4^12*z^29 + 2*x1^54*x2^47*x3^32*x4^12*z^29 - x1^52*x2^49*x3^32*x4^12*z^29 - 2*x1^57*x2^43*x3^33*x4^12*z^29 - 2*x1^56*x2^44*x3^33*x4^12*z^29 + x1^55*x2^45*x3^33*x4^12*z^29 - 5*x1^54*x2^46*x3^33*x4^12*z^29 - x1^53*x2^47*x3^33*x4^12*z^29 + x1^58*x2^41*x3^34*x4^12*z^29 - x1^57*x2^42*x3^34*x4^12*z^29 + 2*x1^56*x2^43*x3^34*x4^12*z^29 + x1^54*x2^45*x3^34*x4^12*z^29 + 2*x1^53*x2^46*x3^34*x4^12*z^29 - x1^57*x2^41*x3^35*x4^12*z^29 - 2*x1^56*x2^42*x3^35*x4^12*z^29 - 2*x1^55*x2^43*x3^35*x4^12*z^29 - x1^53*x2^45*x3^35*x4^12*z^29 - x1^55*x2^42*x3^36*x4^12*z^29 + x1^63*x2^46*x3^23*x4^13*z^29 + x1^61*x2^47*x3^24*x4^13*z^29 - 3*x1^60*x2^47*x3^25*x4^13*z^29 - 2*x1^58*x2^49*x3^25*x4^13*z^29 + x1^60*x2^46*x3^26*x4^13*z^29 + 3*x1^59*x2^47*x3^26*x4^13*z^29 - 2*x1^58*x2^48*x3^26*x4^13*z^29 + x1^57*x2^49*x3^26*x4^13*z^29 - 2*x1^56*x2^50*x3^26*x4^13*z^29 - 5*x1^59*x2^46*x3^27*x4^13*z^29 + x1^55*x2^50*x3^27*x4^13*z^29 + 2*x1^59*x2^45*x3^28*x4^13*z^29 + 5*x1^58*x2^46*x3^28*x4^13*z^29 - 2*x1^57*x2^47*x3^28*x4^13*z^29 + 3*x1^56*x2^48*x3^28*x4^13*z^29 - 2*x1^55*x2^49*x3^28*x4^13*z^29 - x1^54*x2^50*x3^28*x4^13*z^29 - 2*x1^53*x2^51*x3^28*x4^13*z^29 - 6*x1^58*x2^45*x3^29*x4^13*z^29 - 2*x1^57*x2^46*x3^29*x4^13*z^29 + x1^56*x2^47*x3^29*x4^13*z^29 + 2*x1^55*x2^48*x3^29*x4^13*z^29 + 3*x1^54*x2^49*x3^29*x4^13*z^29 + 3*x1^53*x2^50*x3^29*x4^13*z^29 + 2*x1^52*x2^51*x3^29*x4^13*z^29 + 2*x1^58*x2^44*x3^30*x4^13*z^29 + 6*x1^57*x2^45*x3^30*x4^13*z^29 + 4*x1^55*x2^47*x3^30*x4^13*z^29 - 4*x1^54*x2^48*x3^30*x4^13*z^29 - 2*x1^53*x2^49*x3^30*x4^13*z^29 - 5*x1^52*x2^50*x3^30*x4^13*z^29 - 6*x1^57*x2^44*x3^31*x4^13*z^29 - 2*x1^56*x2^45*x3^31*x4^13*z^29 - x1^54*x2^47*x3^31*x4^13*z^29 + x1^53*x2^48*x3^31*x4^13*z^29 + 3*x1^52*x2^49*x3^31*x4^13*z^29 + 2*x1^51*x2^50*x3^31*x4^13*z^29 + x1^58*x2^42*x3^32*x4^13*z^29 + 3*x1^57*x2^43*x3^32*x4^13*z^29 + 5*x1^56*x2^44*x3^32*x4^13*z^29 + 5*x1^54*x2^46*x3^32*x4^13*z^29 - x1^53*x2^47*x3^32*x4^13*z^29 - x1^52*x2^48*x3^32*x4^13*z^29 - 4*x1^51*x2^49*x3^32*x4^13*z^29 + x1^58*x2^41*x3^33*x4^13*z^29 - 5*x1^56*x2^43*x3^33*x4^13*z^29 - 3*x1^55*x2^44*x3^33*x4^13*z^29 - x1^54*x2^45*x3^33*x4^13*z^29 - 2*x1^53*x2^46*x3^33*x4^13*z^29 + x1^52*x2^47*x3^33*x4^13*z^29 + x1^51*x2^48*x3^33*x4^13*z^29 + x1^50*x2^49*x3^33*x4^13*z^29 + x1^57*x2^41*x3^34*x4^13*z^29 + 3*x1^56*x2^42*x3^34*x4^13*z^29 + 3*x1^55*x2^43*x3^34*x4^13*z^29 + 4*x1^53*x2^45*x3^34*x4^13*z^29 - x1^51*x2^47*x3^34*x4^13*z^29 - 2*x1^50*x2^48*x3^34*x4^13*z^29 + x1^56*x2^41*x3^35*x4^13*z^29 - 2*x1^55*x2^42*x3^35*x4^13*z^29 - 2*x1^54*x2^43*x3^35*x4^13*z^29 - 2*x1^53*x2^44*x3^35*x4^13*z^29 - 2*x1^52*x2^45*x3^35*x4^13*z^29 + x1^54*x2^42*x3^36*x4^13*z^29 + 2*x1^52*x2^44*x3^36*x4^13*z^29 - x1^52*x2^43*x3^37*x4^13*z^29 - x1^51*x2^44*x3^37*x4^13*z^29 - x1^61*x2^47*x3^23*x4^14*z^29 - x1^62*x2^45*x3^24*x4^14*z^29 + x1^61*x2^46*x3^24*x4^14*z^29 + x1^60*x2^47*x3^24*x4^14*z^29 + x1^61*x2^45*x3^25*x4^14*z^29 - 3*x1^60*x2^46*x3^25*x4^14*z^29 - 2*x1^59*x2^47*x3^25*x4^14*z^29 + x1^58*x2^48*x3^25*x4^14*z^29 - x1^60*x2^45*x3^26*x4^14*z^29 + 4*x1^59*x2^46*x3^26*x4^14*z^29 + 2*x1^60*x2^44*x3^27*x4^14*z^29 + x1^59*x2^45*x3^27*x4^14*z^29 - 3*x1^58*x2^46*x3^27*x4^14*z^29 - x1^57*x2^47*x3^27*x4^14*z^29 - 2*x1^56*x2^48*x3^27*x4^14*z^29 + x1^55*x2^49*x3^27*x4^14*z^29 - x1^60*x2^43*x3^28*x4^14*z^29 - 2*x1^59*x2^44*x3^28*x4^14*z^29 + 2*x1^58*x2^45*x3^28*x4^14*z^29 + x1^57*x2^46*x3^28*x4^14*z^29 + 2*x1^56*x2^47*x3^28*x4^14*z^29 + x1^55*x2^48*x3^28*x4^14*z^29 + 2*x1^59*x2^43*x3^29*x4^14*z^29 - x1^57*x2^45*x3^29*x4^14*z^29 - 2*x1^55*x2^47*x3^29*x4^14*z^29 - x1^54*x2^48*x3^29*x4^14*z^29 + 2*x1^52*x2^50*x3^29*x4^14*z^29 - x1^59*x2^42*x3^30*x4^14*z^29 - 2*x1^58*x2^43*x3^30*x4^14*z^29 + 2*x1^57*x2^44*x3^30*x4^14*z^29 + 2*x1^55*x2^46*x3^30*x4^14*z^29 + x1^54*x2^47*x3^30*x4^14*z^29 - 2*x1^52*x2^49*x3^30*x4^14*z^29 - 2*x1^51*x2^50*x3^30*x4^14*z^29 + 2*x1^58*x2^42*x3^31*x4^14*z^29 - x1^57*x2^43*x3^31*x4^14*z^29 - 2*x1^56*x2^44*x3^31*x4^14*z^29 + 3*x1^53*x2^47*x3^31*x4^14*z^29 - x1^52*x2^48*x3^31*x4^14*z^29 + 2*x1^51*x2^49*x3^31*x4^14*z^29 - x1^58*x2^41*x3^32*x4^14*z^29 - 2*x1^57*x2^42*x3^32*x4^14*z^29 + 2*x1^56*x2^43*x3^32*x4^14*z^29 - 2*x1^55*x2^44*x3^32*x4^14*z^29 - 2*x1^54*x2^45*x3^32*x4^14*z^29 + x1^53*x2^46*x3^32*x4^14*z^29 + x1^52*x2^47*x3^32*x4^14*z^29 - x1^51*x2^48*x3^32*x4^14*z^29 - 2*x1^50*x2^49*x3^32*x4^14*z^29 + x1^57*x2^41*x3^33*x4^14*z^29 + 2*x1^56*x2^42*x3^33*x4^14*z^29 - x1^55*x2^43*x3^33*x4^14*z^29 - x1^54*x2^44*x3^33*x4^14*z^29 - x1^53*x2^45*x3^33*x4^14*z^29 + x1^52*x2^46*x3^33*x4^14*z^29 + x1^50*x2^48*x3^33*x4^14*z^29 - x1^56*x2^41*x3^34*x4^14*z^29 - x1^53*x2^44*x3^34*x4^14*z^29 + x1^52*x2^45*x3^34*x4^14*z^29 + x1^51*x2^46*x3^34*x4^14*z^29 - x1^49*x2^48*x3^34*x4^14*z^29 + x1^55*x2^41*x3^35*x4^14*z^29 + x1^54*x2^42*x3^35*x4^14*z^29 + x1^53*x2^43*x3^35*x4^14*z^29 - x1^52*x2^44*x3^35*x4^14*z^29 + x1^50*x2^46*x3^35*x4^14*z^29 + 2*x1^49*x2^47*x3^35*x4^14*z^29 + x1^53*x2^42*x3^36*x4^14*z^29 + x1^62*x2^45*x3^23*x4^15*z^29 + x1^61*x2^46*x3^23*x4^15*z^29 - x1^60*x2^47*x3^23*x4^15*z^29 - 4*x1^61*x2^45*x3^24*x4^15*z^29 + x1^59*x2^47*x3^24*x4^15*z^29 + 2*x1^61*x2^44*x3^25*x4^15*z^29 + 4*x1^60*x2^45*x3^25*x4^15*z^29 - x1^59*x2^46*x3^25*x4^15*z^29 + x1^58*x2^47*x3^25*x4^15*z^29 - x1^57*x2^48*x3^25*x4^15*z^29 - 6*x1^60*x2^44*x3^26*x4^15*z^29 - x1^59*x2^45*x3^26*x4^15*z^29 - x1^58*x2^46*x3^26*x4^15*z^29 + x1^56*x2^48*x3^26*x4^15*z^29 + x1^55*x2^49*x3^26*x4^15*z^29 + 2*x1^60*x2^43*x3^27*x4^15*z^29 + 6*x1^59*x2^44*x3^27*x4^15*z^29 + 5*x1^57*x2^46*x3^27*x4^15*z^29 - x1^56*x2^47*x3^27*x4^15*z^29 - x1^55*x2^48*x3^27*x4^15*z^29 - 2*x1^54*x2^49*x3^27*x4^15*z^29 - 6*x1^59*x2^43*x3^28*x4^15*z^29 - 2*x1^58*x2^44*x3^28*x4^15*z^29 - 2*x1^57*x2^45*x3^28*x4^15*z^29 + 2*x1^55*x2^47*x3^28*x4^15*z^29 + 3*x1^54*x2^48*x3^28*x4^15*z^29 + 2*x1^53*x2^49*x3^28*x4^15*z^29 + 2*x1^59*x2^42*x3^29*x4^15*z^29 + 6*x1^58*x2^43*x3^29*x4^15*z^29 + 4*x1^56*x2^45*x3^29*x4^15*z^29 - 4*x1^55*x2^46*x3^29*x4^15*z^29 - x1^54*x2^47*x3^29*x4^15*z^29 - 5*x1^53*x2^48*x3^29*x4^15*z^29 - x1^52*x2^49*x3^29*x4^15*z^29 - 6*x1^58*x2^42*x3^30*x4^15*z^29 - 2*x1^57*x2^43*x3^30*x4^15*z^29 - 2*x1^56*x2^44*x3^30*x4^15*z^29 + 2*x1^54*x2^46*x3^30*x4^15*z^29 + 2*x1^53*x2^47*x3^30*x4^15*z^29 + 5*x1^52*x2^48*x3^30*x4^15*z^29 + 2*x1^58*x2^41*x3^31*x4^15*z^29 + 6*x1^57*x2^42*x3^31*x4^15*z^29 + 4*x1^55*x2^44*x3^31*x4^15*z^29 - 3*x1^54*x2^45*x3^31*x4^15*z^29 - x1^53*x2^46*x3^31*x4^15*z^29 - 5*x1^52*x2^47*x3^31*x4^15*z^29 - x1^51*x2^48*x3^31*x4^15*z^29 - 5*x1^57*x2^41*x3^32*x4^15*z^29 - 2*x1^56*x2^42*x3^32*x4^15*z^29 - 2*x1^55*x2^43*x3^32*x4^15*z^29 - x1^54*x2^44*x3^32*x4^15*z^29 + 2*x1^52*x2^46*x3^32*x4^15*z^29 + 4*x1^51*x2^47*x3^32*x4^15*z^29 + 2*x1^57*x2^40*x3^33*x4^15*z^29 + 5*x1^56*x2^41*x3^33*x4^15*z^29 + 4*x1^54*x2^43*x3^33*x4^15*z^29 - 4*x1^51*x2^46*x3^33*x4^15*z^29 - 3*x1^56*x2^40*x3^34*x4^15*z^29 - 2*x1^55*x2^41*x3^34*x4^15*z^29 - x1^54*x2^42*x3^34*x4^15*z^29 - x1^53*x2^43*x3^34*x4^15*z^29 - x1^52*x2^44*x3^34*x4^15*z^29 + x1^51*x2^45*x3^34*x4^15*z^29 + 2*x1^50*x2^46*x3^34*x4^15*z^29 + 2*x1^55*x2^40*x3^35*x4^15*z^29 + 3*x1^53*x2^42*x3^35*x4^15*z^29 - x1^52*x2^43*x3^35*x4^15*z^29 - x1^51*x2^44*x3^35*x4^15*z^29 - x1^50*x2^45*x3^35*x4^15*z^29 - x1^54*x2^40*x3^36*x4^15*z^29 - x1^53*x2^41*x3^36*x4^15*z^29 - x1^52*x2^42*x3^36*x4^15*z^29 - x1^51*x2^43*x3^36*x4^15*z^29 - x1^50*x2^44*x3^36*x4^15*z^29 - x1^51*x2^42*x3^37*x4^15*z^29 + x1^61*x2^45*x3^23*x4^16*z^29 + x1^59*x2^47*x3^23*x4^16*z^29 - x1^61*x2^44*x3^24*x4^16*z^29 - 3*x1^60*x2^45*x3^24*x4^16*z^29 - x1^58*x2^47*x3^24*x4^16*z^29 + x1^57*x2^48*x3^24*x4^16*z^29 + 5*x1^60*x2^44*x3^25*x4^16*z^29 + 2*x1^59*x2^45*x3^25*x4^16*z^29 + x1^57*x2^47*x3^25*x4^16*z^29 - 2*x1^60*x2^43*x3^26*x4^16*z^29 - 6*x1^59*x2^44*x3^26*x4^16*z^29 + x1^58*x2^45*x3^26*x4^16*z^29 - 4*x1^57*x2^46*x3^26*x4^16*z^29 + x1^56*x2^47*x3^26*x4^16*z^29 + x1^54*x2^49*x3^26*x4^16*z^29 + 6*x1^59*x2^43*x3^27*x4^16*z^29 + 2*x1^58*x2^44*x3^27*x4^16*z^29 + x1^57*x2^45*x3^27*x4^16*z^29 - x1^55*x2^47*x3^27*x4^16*z^29 - 2*x1^54*x2^48*x3^27*x4^16*z^29 - x1^53*x2^49*x3^27*x4^16*z^29 - 2*x1^59*x2^42*x3^28*x4^16*z^29 - 6*x1^58*x2^43*x3^28*x4^16*z^29 - 4*x1^56*x2^45*x3^28*x4^16*z^29 + 3*x1^55*x2^46*x3^28*x4^16*z^29 + 4*x1^53*x2^48*x3^28*x4^16*z^29 + 6*x1^58*x2^42*x3^29*x4^16*z^29 + 2*x1^57*x2^43*x3^29*x4^16*z^29 + 2*x1^56*x2^44*x3^29*x4^16*z^29 - 2*x1^54*x2^46*x3^29*x4^16*z^29 - 3*x1^53*x2^47*x3^29*x4^16*z^29 - 4*x1^52*x2^48*x3^29*x4^16*z^29 - 2*x1^58*x2^41*x3^30*x4^16*z^29 - 6*x1^57*x2^42*x3^30*x4^16*z^29 - 4*x1^55*x2^44*x3^30*x4^16*z^29 + 4*x1^54*x2^45*x3^30*x4^16*z^29 + 6*x1^52*x2^47*x3^30*x4^16*z^29 + x1^51*x2^48*x3^30*x4^16*z^29 + 6*x1^57*x2^41*x3^31*x4^16*z^29 + 2*x1^56*x2^42*x3^31*x4^16*z^29 + 2*x1^55*x2^43*x3^31*x4^16*z^29 - 2*x1^53*x2^45*x3^31*x4^16*z^29 - 2*x1^52*x2^46*x3^31*x4^16*z^29 - 6*x1^51*x2^47*x3^31*x4^16*z^29 - x1^57*x2^40*x3^32*x4^16*z^29 - 6*x1^56*x2^41*x3^32*x4^16*z^29 - 4*x1^54*x2^43*x3^32*x4^16*z^29 + 4*x1^53*x2^44*x3^32*x4^16*z^29 + 6*x1^51*x2^46*x3^32*x4^16*z^29 + 2*x1^50*x2^47*x3^32*x4^16*z^29 + 2*x1^56*x2^40*x3^33*x4^16*z^29 + 3*x1^55*x2^41*x3^33*x4^16*z^29 + x1^54*x2^42*x3^33*x4^16*z^29 + x1^53*x2^43*x3^33*x4^16*z^29 - x1^52*x2^44*x3^33*x4^16*z^29 - 3*x1^51*x2^45*x3^33*x4^16*z^29 - 5*x1^50*x2^46*x3^33*x4^16*z^29 - 2*x1^55*x2^40*x3^34*x4^16*z^29 - 2*x1^54*x2^41*x3^34*x4^16*z^29 - 3*x1^53*x2^42*x3^34*x4^16*z^29 + x1^52*x2^43*x3^34*x4^16*z^29 + 4*x1^50*x2^45*x3^34*x4^16*z^29 + x1^49*x2^46*x3^34*x4^16*z^29 + x1^54*x2^40*x3^35*x4^16*z^29 + 2*x1^53*x2^41*x3^35*x4^16*z^29 + x1^52*x2^42*x3^35*x4^16*z^29 + x1^51*x2^43*x3^35*x4^16*z^29 - 2*x1^50*x2^44*x3^35*x4^16*z^29 - 2*x1^49*x2^45*x3^35*x4^16*z^29 - x1^53*x2^40*x3^36*x4^16*z^29 - 2*x1^52*x2^41*x3^36*x4^16*z^29 + 2*x1^49*x2^44*x3^36*x4^16*z^29 + x1^52*x2^40*x3^37*x4^16*z^29 + x1^50*x2^42*x3^37*x4^16*z^29 - x1^48*x2^44*x3^37*x4^16*z^29 - x1^60*x2^44*x3^24*x4^17*z^29 + x1^58*x2^46*x3^24*x4^17*z^29 + x1^56*x2^48*x3^24*x4^17*z^29 + x1^59*x2^44*x3^25*x4^17*z^29 + x1^58*x2^45*x3^25*x4^17*z^29 - x1^57*x2^46*x3^25*x4^17*z^29 - 2*x1^56*x2^47*x3^25*x4^17*z^29 - x1^55*x2^48*x3^25*x4^17*z^29 - x1^59*x2^43*x3^26*x4^17*z^29 - x1^58*x2^44*x3^26*x4^17*z^29 + x1^57*x2^45*x3^26*x4^17*z^29 + x1^56*x2^46*x3^26*x4^17*z^29 + x1^54*x2^48*x3^26*x4^17*z^29 + 2*x1^58*x2^43*x3^27*x4^17*z^29 + 2*x1^56*x2^45*x3^27*x4^17*z^29 - x1^55*x2^46*x3^27*x4^17*z^29 - 2*x1^53*x2^48*x3^27*x4^17*z^29 - 2*x1^58*x2^42*x3^28*x4^17*z^29 - x1^57*x2^43*x3^28*x4^17*z^29 - x1^56*x2^44*x3^28*x4^17*z^29 + x1^54*x2^46*x3^28*x4^17*z^29 + x1^53*x2^47*x3^28*x4^17*z^29 + 2*x1^52*x2^48*x3^28*x4^17*z^29 + x1^58*x2^41*x3^29*x4^17*z^29 + 2*x1^57*x2^42*x3^29*x4^17*z^29 + x1^55*x2^44*x3^29*x4^17*z^29 - x1^54*x2^45*x3^29*x4^17*z^29 - 2*x1^52*x2^47*x3^29*x4^17*z^29 - x1^51*x2^48*x3^29*x4^17*z^29 - 2*x1^57*x2^41*x3^30*x4^17*z^29 - x1^56*x2^42*x3^30*x4^17*z^29 - x1^55*x2^43*x3^30*x4^17*z^29 + x1^53*x2^45*x3^30*x4^17*z^29 + x1^52*x2^46*x3^30*x4^17*z^29 + 2*x1^51*x2^47*x3^30*x4^17*z^29 + 2*x1^56*x2^41*x3^31*x4^17*z^29 + x1^54*x2^43*x3^31*x4^17*z^29 - 2*x1^53*x2^44*x3^31*x4^17*z^29 - 2*x1^51*x2^46*x3^31*x4^17*z^29 - x1^55*x2^41*x3^32*x4^17*z^29 + 2*x1^50*x2^46*x3^32*x4^17*z^29 + x1^53*x2^42*x3^33*x4^17*z^29 - x1^52*x2^43*x3^33*x4^17*z^29 - 2*x1^50*x2^45*x3^33*x4^17*z^29 - x1^49*x2^46*x3^33*x4^17*z^29 + x1^50*x2^44*x3^34*x4^17*z^29 + 2*x1^49*x2^45*x3^34*x4^17*z^29 - x1^51*x2^42*x3^35*x4^17*z^29 + x1^50*x2^43*x3^35*x4^17*z^29 - 2*x1^49*x2^44*x3^35*x4^17*z^29 - x1^48*x2^45*x3^35*x4^17*z^29 + x1^50*x2^42*x3^36*x4^17*z^29 + x1^61*x2^48*x3^27*x4^4*z^28 - x1^63*x2^46*x3^26*x4^5*z^28 - x1^62*x2^47*x3^26*x4^5*z^28 + 2*x1^62*x2^46*x3^27*x4^5*z^28 + x1^60*x2^48*x3^27*x4^5*z^28 - x1^62*x2^45*x3^28*x4^5*z^28 + x1^60*x2^47*x3^28*x4^5*z^28 - x1^59*x2^48*x3^28*x4^5*z^28 - x1^60*x2^46*x3^29*x4^5*z^28 + x1^63*x2^46*x3^25*x4^6*z^28 - 2*x1^62*x2^46*x3^26*x4^6*z^28 + x1^61*x2^47*x3^26*x4^6*z^28 + x1^62*x2^45*x3^27*x4^6*z^28 + x1^61*x2^46*x3^27*x4^6*z^28 - x1^60*x2^47*x3^27*x4^6*z^28 + x1^59*x2^48*x3^27*x4^6*z^28 - 2*x1^61*x2^45*x3^28*x4^6*z^28 - x1^59*x2^47*x3^28*x4^6*z^28 + x1^61*x2^44*x3^29*x4^6*z^28 - x1^59*x2^46*x3^29*x4^6*z^28 + 2*x1^58*x2^47*x3^29*x4^6*z^28 + x1^57*x2^48*x3^29*x4^6*z^28 + x1^59*x2^45*x3^30*x4^6*z^28 + x1^56*x2^48*x3^30*x4^6*z^28 - x1^62*x2^46*x3^25*x4^7*z^28 + x1^61*x2^47*x3^25*x4^7*z^28 - x1^59*x2^49*x3^25*x4^7*z^28 - x1^60*x2^47*x3^26*x4^7*z^28 + x1^59*x2^48*x3^26*x4^7*z^28 - 2*x1^60*x2^46*x3^27*x4^7*z^28 - x1^59*x2^47*x3^27*x4^7*z^28 + x1^57*x2^49*x3^27*x4^7*z^28 - x1^60*x2^45*x3^28*x4^7*z^28 - x1^59*x2^46*x3^28*x4^7*z^28 - x1^58*x2^47*x3^28*x4^7*z^28 + x1^56*x2^49*x3^28*x4^7*z^28 + 2*x1^60*x2^44*x3^29*x4^7*z^28 - x1^59*x2^45*x3^29*x4^7*z^28 + x1^57*x2^47*x3^29*x4^7*z^28 - x1^59*x2^44*x3^30*x4^7*z^28 - 2*x1^57*x2^46*x3^30*x4^7*z^28 - x1^56*x2^47*x3^30*x4^7*z^28 + x1^55*x2^48*x3^30*x4^7*z^28 - 2*x1^61*x2^47*x3^24*x4^8*z^28 + x1^62*x2^45*x3^25*x4^8*z^28 + 4*x1^60*x2^47*x3^25*x4^8*z^28 + x1^58*x2^49*x3^25*x4^8*z^28 + x1^61*x2^45*x3^26*x4^8*z^28 - 4*x1^59*x2^47*x3^26*x4^8*z^28 + 2*x1^58*x2^48*x3^26*x4^8*z^28 + x1^56*x2^50*x3^26*x4^8*z^28 + x1^60*x2^45*x3^27*x4^8*z^28 + 4*x1^59*x2^46*x3^27*x4^8*z^28 + x1^58*x2^47*x3^27*x4^8*z^28 - x1^55*x2^50*x3^27*x4^8*z^28 - 2*x1^60*x2^44*x3^28*x4^8*z^28 - 2*x1^59*x2^45*x3^28*x4^8*z^28 - 3*x1^58*x2^46*x3^28*x4^8*z^28 - 2*x1^56*x2^48*x3^28*x4^8*z^28 + 2*x1^59*x2^44*x3^29*x4^8*z^28 + x1^58*x2^45*x3^29*x4^8*z^28 + x1^56*x2^47*x3^29*x4^8*z^28 - x1^54*x2^49*x3^29*x4^8*z^28 - x1^59*x2^43*x3^30*x4^8*z^28 - x1^58*x2^44*x3^30*x4^8*z^28 - 2*x1^57*x2^45*x3^30*x4^8*z^28 - 2*x1^55*x2^47*x3^30*x4^8*z^28 + 2*x1^58*x2^43*x3^31*x4^8*z^28 + x1^57*x2^44*x3^31*x4^8*z^28 + x1^56*x2^45*x3^31*x4^8*z^28 + x1^55*x2^46*x3^31*x4^8*z^28 - x1^58*x2^42*x3^32*x4^8*z^28 - x1^56*x2^44*x3^32*x4^8*z^28 - x1^55*x2^45*x3^32*x4^8*z^28 - 2*x1^54*x2^46*x3^32*x4^8*z^28 + x1^61*x2^47*x3^23*x4^9*z^28 - x1^62*x2^45*x3^24*x4^9*z^28 - x1^60*x2^47*x3^24*x4^9*z^28 + x1^60*x2^46*x3^25*x4^9*z^28 + x1^59*x2^47*x3^25*x4^9*z^28 + x1^57*x2^49*x3^25*x4^9*z^28 - x1^61*x2^44*x3^26*x4^9*z^28 - 3*x1^59*x2^46*x3^26*x4^9*z^28 - x1^58*x2^47*x3^26*x4^9*z^28 - x1^57*x2^48*x3^26*x4^9*z^28 - x1^56*x2^49*x3^26*x4^9*z^28 + x1^60*x2^44*x3^27*x4^9*z^28 - 2*x1^59*x2^45*x3^27*x4^9*z^28 + 2*x1^58*x2^46*x3^27*x4^9*z^28 + x1^56*x2^48*x3^27*x4^9*z^28 + x1^55*x2^49*x3^27*x4^9*z^28 + x1^54*x2^50*x3^27*x4^9*z^28 - x1^60*x2^43*x3^28*x4^9*z^28 - x1^59*x2^44*x3^28*x4^9*z^28 + x1^58*x2^45*x3^28*x4^9*z^28 - 3*x1^57*x2^46*x3^28*x4^9*z^28 - x1^56*x2^47*x3^28*x4^9*z^28 + x1^55*x2^48*x3^28*x4^9*z^28 - x1^54*x2^49*x3^28*x4^9*z^28 + 2*x1^59*x2^43*x3^29*x4^9*z^28 + 2*x1^58*x2^44*x3^29*x4^9*z^28 + 2*x1^57*x2^45*x3^29*x4^9*z^28 + x1^55*x2^47*x3^29*x4^9*z^28 + x1^53*x2^49*x3^29*x4^9*z^28 - x1^59*x2^42*x3^30*x4^9*z^28 - 3*x1^58*x2^43*x3^30*x4^9*z^28 - x1^55*x2^46*x3^30*x4^9*z^28 - x1^54*x2^47*x3^30*x4^9*z^28 + 2*x1^53*x2^48*x3^30*x4^9*z^28 + x1^52*x2^49*x3^30*x4^9*z^28 + x1^56*x2^44*x3^31*x4^9*z^28 + x1^55*x2^45*x3^31*x4^9*z^28 + x1^54*x2^46*x3^31*x4^9*z^28 - x1^57*x2^42*x3^32*x4^9*z^28 + x1^54*x2^45*x3^32*x4^9*z^28 + x1^57*x2^41*x3^33*x4^9*z^28 + x1^55*x2^43*x3^33*x4^9*z^28 + x1^54*x2^44*x3^33*x4^9*z^28 + 2*x1^53*x2^45*x3^33*x4^9*z^28 + x1^62*x2^46*x3^22*x4^10*z^28 - x1^62*x2^45*x3^23*x4^10*z^28 - x1^61*x2^46*x3^23*x4^10*z^28 + x1^60*x2^47*x3^23*x4^10*z^28 + 3*x1^61*x2^45*x3^24*x4^10*z^28 + x1^58*x2^48*x3^24*x4^10*z^28 - 3*x1^60*x2^45*x3^25*x4^10*z^28 + x1^60*x2^44*x3^26*x4^10*z^28 + x1^59*x2^45*x3^26*x4^10*z^28 + x1^58*x2^46*x3^26*x4^10*z^28 + x1^55*x2^49*x3^26*x4^10*z^28 - x1^59*x2^44*x3^27*x4^10*z^28 - x1^58*x2^45*x3^27*x4^10*z^28 - 2*x1^57*x2^46*x3^27*x4^10*z^28 + x1^54*x2^49*x3^27*x4^10*z^28 + x1^57*x2^45*x3^28*x4^10*z^28 - x1^56*x2^46*x3^28*x4^10*z^28 - x1^55*x2^47*x3^28*x4^10*z^28 - 2*x1^54*x2^48*x3^28*x4^10*z^28 - x1^53*x2^49*x3^28*x4^10*z^28 + x1^52*x2^50*x3^28*x4^10*z^28 - 2*x1^57*x2^44*x3^29*x4^10*z^28 - 2*x1^54*x2^47*x3^29*x4^10*z^28 + 2*x1^53*x2^48*x3^29*x4^10*z^28 + x1^52*x2^49*x3^29*x4^10*z^28 + x1^58*x2^42*x3^30*x4^10*z^28 - x1^57*x2^43*x3^30*x4^10*z^28 + x1^56*x2^44*x3^30*x4^10*z^28 + x1^53*x2^47*x3^30*x4^10*z^28 - 2*x1^52*x2^48*x3^30*x4^10*z^28 - x1^51*x2^49*x3^30*x4^10*z^28 - x1^55*x2^44*x3^31*x4^10*z^28 - x1^54*x2^45*x3^31*x4^10*z^28 + x1^53*x2^46*x3^31*x4^10*z^28 + x1^50*x2^49*x3^31*x4^10*z^28 + x1^57*x2^41*x3^32*x4^10*z^28 + x1^53*x2^45*x3^32*x4^10*z^28 - x1^55*x2^42*x3^33*x4^10*z^28 - 2*x1^53*x2^44*x3^33*x4^10*z^28 + x1^55*x2^41*x3^34*x4^10*z^28 + x1^54*x2^42*x3^34*x4^10*z^28 - x1^53*x2^43*x3^34*x4^10*z^28 + x1^52*x2^44*x3^34*x4^10*z^28 + x1^62*x2^45*x3^22*x4^11*z^28 - 4*x1^61*x2^45*x3^23*x4^11*z^28 - x1^60*x2^46*x3^23*x4^11*z^28 - 2*x1^59*x2^47*x3^23*x4^11*z^28 + x1^61*x2^44*x3^24*x4^11*z^28 + 4*x1^60*x2^45*x3^24*x4^11*z^28 - x1^59*x2^46*x3^24*x4^11*z^28 - 2*x1^57*x2^48*x3^24*x4^11*z^28 - 4*x1^60*x2^44*x3^25*x4^11*z^28 + x1^59*x2^45*x3^25*x4^11*z^28 + x1^56*x2^48*x3^25*x4^11*z^28 + x1^60*x2^43*x3^26*x4^11*z^28 + 4*x1^59*x2^44*x3^26*x4^11*z^28 - 3*x1^58*x2^45*x3^26*x4^11*z^28 + 4*x1^57*x2^46*x3^26*x4^11*z^28 - 2*x1^55*x2^48*x3^26*x4^11*z^28 - 3*x1^54*x2^49*x3^26*x4^11*z^28 - 4*x1^59*x2^43*x3^27*x4^11*z^28 - 2*x1^58*x2^44*x3^27*x4^11*z^28 + x1^57*x2^45*x3^27*x4^11*z^28 + 3*x1^55*x2^47*x3^27*x4^11*z^28 + 2*x1^54*x2^48*x3^27*x4^11*z^28 + 3*x1^53*x2^49*x3^27*x4^11*z^28 + 2*x1^59*x2^42*x3^28*x4^11*z^28 + 4*x1^58*x2^43*x3^28*x4^11*z^28 - 2*x1^57*x2^44*x3^28*x4^11*z^28 + x1^56*x2^45*x3^28*x4^11*z^28 - x1^55*x2^46*x3^28*x4^11*z^28 + x1^54*x2^47*x3^28*x4^11*z^28 - 2*x1^53*x2^48*x3^28*x4^11*z^28 - x1^52*x2^49*x3^28*x4^11*z^28 - 3*x1^58*x2^42*x3^29*x4^11*z^28 + x1^57*x2^43*x3^29*x4^11*z^28 - x1^55*x2^45*x3^29*x4^11*z^28 + 3*x1^54*x2^46*x3^29*x4^11*z^28 + 3*x1^53*x2^47*x3^29*x4^11*z^28 + 2*x1^52*x2^48*x3^29*x4^11*z^28 + x1^58*x2^41*x3^30*x4^11*z^28 + 5*x1^57*x2^42*x3^30*x4^11*z^28 + x1^55*x2^44*x3^30*x4^11*z^28 - 2*x1^54*x2^45*x3^30*x4^11*z^28 - x1^52*x2^47*x3^30*x4^11*z^28 - x1^50*x2^49*x3^30*x4^11*z^28 - 4*x1^57*x2^41*x3^31*x4^11*z^28 + 2*x1^56*x2^42*x3^31*x4^11*z^28 - 2*x1^54*x2^44*x3^31*x4^11*z^28 + x1^53*x2^45*x3^31*x4^11*z^28 + 2*x1^51*x2^47*x3^31*x4^11*z^28 - x1^57*x2^40*x3^32*x4^11*z^28 + 2*x1^56*x2^41*x3^32*x4^11*z^28 + 2*x1^54*x2^43*x3^32*x4^11*z^28 + x1^53*x2^44*x3^32*x4^11*z^28 - 2*x1^52*x2^45*x3^32*x4^11*z^28 - 2*x1^51*x2^46*x3^32*x4^11*z^28 - x1^49*x2^48*x3^32*x4^11*z^28 + x1^54*x2^42*x3^33*x4^11*z^28 - x1^53*x2^43*x3^33*x4^11*z^28 - x1^52*x2^44*x3^33*x4^11*z^28 + x1^52*x2^43*x3^34*x4^11*z^28 - x1^51*x2^44*x3^34*x4^11*z^28 - x1^53*x2^41*x3^35*x4^11*z^28 - x1^51*x2^43*x3^35*x4^11*z^28 + x1^61*x2^45*x3^22*x4^12*z^28 - x1^60*x2^46*x3^22*x4^12*z^28 - x1^61*x2^44*x3^23*x4^12*z^28 - x1^60*x2^45*x3^23*x4^12*z^28 + 3*x1^59*x2^46*x3^23*x4^12*z^28 - x1^58*x2^47*x3^23*x4^12*z^28 + 3*x1^60*x2^44*x3^24*x4^12*z^28 - 2*x1^59*x2^45*x3^24*x4^12*z^28 - 2*x1^58*x2^46*x3^24*x4^12*z^28 + x1^57*x2^47*x3^24*x4^12*z^28 - x1^56*x2^48*x3^24*x4^12*z^28 - 3*x1^59*x2^44*x3^25*x4^12*z^28 + 5*x1^58*x2^45*x3^25*x4^12*z^28 - x1^57*x2^46*x3^25*x4^12*z^28 + x1^56*x2^47*x3^25*x4^12*z^28 - x1^54*x2^49*x3^25*x4^12*z^28 + 2*x1^59*x2^43*x3^26*x4^12*z^28 - x1^58*x2^44*x3^26*x4^12*z^28 - 4*x1^57*x2^45*x3^26*x4^12*z^28 + x1^56*x2^46*x3^26*x4^12*z^28 - 2*x1^55*x2^47*x3^26*x4^12*z^28 + x1^54*x2^48*x3^26*x4^12*z^28 + x1^53*x2^49*x3^26*x4^12*z^28 - x1^59*x2^42*x3^27*x4^12*z^28 - 2*x1^58*x2^43*x3^27*x4^12*z^28 + 6*x1^57*x2^44*x3^27*x4^12*z^28 - x1^55*x2^46*x3^27*x4^12*z^28 - x1^52*x2^49*x3^27*x4^12*z^28 + 2*x1^58*x2^42*x3^28*x4^12*z^28 - x1^57*x2^43*x3^28*x4^12*z^28 - 5*x1^56*x2^44*x3^28*x4^12*z^28 - 4*x1^54*x2^46*x3^28*x4^12*z^28 + 2*x1^53*x2^47*x3^28*x4^12*z^28 + x1^52*x2^48*x3^28*x4^12*z^28 + 2*x1^51*x2^49*x3^28*x4^12*z^28 - x1^58*x2^41*x3^29*x4^12*z^28 - 2*x1^57*x2^42*x3^29*x4^12*z^28 + 6*x1^56*x2^43*x3^29*x4^12*z^28 + x1^55*x2^44*x3^29*x4^12*z^28 + 2*x1^54*x2^45*x3^29*x4^12*z^28 + x1^53*x2^46*x3^29*x4^12*z^28 + x1^52*x2^47*x3^29*x4^12*z^28 - 2*x1^51*x2^48*x3^29*x4^12*z^28 - x1^50*x2^49*x3^29*x4^12*z^28 + x1^57*x2^41*x3^30*x4^12*z^28 - 3*x1^56*x2^42*x3^30*x4^12*z^28 - 5*x1^55*x2^43*x3^30*x4^12*z^28 - 5*x1^53*x2^45*x3^30*x4^12*z^28 + x1^52*x2^46*x3^30*x4^12*z^28 - x1^51*x2^47*x3^30*x4^12*z^28 + 5*x1^50*x2^48*x3^30*x4^12*z^28 + x1^57*x2^40*x3^31*x4^12*z^28 - 2*x1^56*x2^41*x3^31*x4^12*z^28 + 5*x1^55*x2^42*x3^31*x4^12*z^28 + x1^54*x2^43*x3^31*x4^12*z^28 + x1^53*x2^44*x3^31*x4^12*z^28 + 3*x1^52*x2^45*x3^31*x4^12*z^28 - x1^50*x2^47*x3^31*x4^12*z^28 - x1^49*x2^48*x3^31*x4^12*z^28 - 2*x1^54*x2^42*x3^32*x4^12*z^28 - 4*x1^52*x2^44*x3^32*x4^12*z^28 + 2*x1^49*x2^47*x3^32*x4^12*z^28 + 3*x1^54*x2^41*x3^33*x4^12*z^28 + 2*x1^53*x2^42*x3^33*x4^12*z^28 + x1^52*x2^43*x3^33*x4^12*z^28 + 2*x1^51*x2^44*x3^33*x4^12*z^28 + 2*x1^50*x2^45*x3^33*x4^12*z^28 - x1^55*x2^39*x3^34*x4^12*z^28 - x1^54*x2^40*x3^34*x4^12*z^28 - x1^53*x2^41*x3^34*x4^12*z^28 - 2*x1^51*x2^43*x3^34*x4^12*z^28 + x1^53*x2^40*x3^35*x4^12*z^28 + 2*x1^52*x2^41*x3^35*x4^12*z^28 + x1^51*x2^42*x3^35*x4^12*z^28 + x1^50*x2^43*x3^35*x4^12*z^28 + x1^60*x2^45*x3^22*x4^13*z^28 - 2*x1^59*x2^46*x3^22*x4^13*z^28 - x1^57*x2^47*x3^23*x4^13*z^28 - 4*x1^58*x2^45*x3^24*x4^13*z^28 + 3*x1^58*x2^44*x3^25*x4^13*z^28 + 4*x1^57*x2^45*x3^25*x4^13*z^28 + 2*x1^55*x2^47*x3^25*x4^13*z^28 + x1^54*x2^48*x3^25*x4^13*z^28 - 6*x1^57*x2^44*x3^26*x4^13*z^28 - x1^55*x2^46*x3^26*x4^13*z^28 + x1^54*x2^47*x3^26*x4^13*z^28 + x1^53*x2^48*x3^26*x4^13*z^28 + 2*x1^52*x2^49*x3^26*x4^13*z^28 + 2*x1^57*x2^43*x3^27*x4^13*z^28 + 6*x1^56*x2^44*x3^27*x4^13*z^28 + 3*x1^54*x2^46*x3^27*x4^13*z^28 - 3*x1^53*x2^47*x3^27*x4^13*z^28 - x1^52*x2^48*x3^27*x4^13*z^28 - 3*x1^51*x2^49*x3^27*x4^13*z^28 - 6*x1^56*x2^43*x3^28*x4^13*z^28 - 2*x1^55*x2^44*x3^28*x4^13*z^28 - 2*x1^54*x2^45*x3^28*x4^13*z^28 + x1^53*x2^46*x3^28*x4^13*z^28 + 2*x1^52*x2^47*x3^28*x4^13*z^28 + x1^51*x2^48*x3^28*x4^13*z^28 + 3*x1^50*x2^49*x3^28*x4^13*z^28 + 2*x1^56*x2^42*x3^29*x4^13*z^28 + 6*x1^55*x2^43*x3^29*x4^13*z^28 + 3*x1^53*x2^45*x3^29*x4^13*z^28 - 5*x1^52*x2^46*x3^29*x4^13*z^28 - 3*x1^51*x2^47*x3^29*x4^13*z^28 - 5*x1^50*x2^48*x3^29*x4^13*z^28 - 2*x1^49*x2^49*x3^29*x4^13*z^28 - 6*x1^55*x2^42*x3^30*x4^13*z^28 - 2*x1^54*x2^43*x3^30*x4^13*z^28 - 2*x1^52*x2^45*x3^30*x4^13*z^28 + x1^51*x2^46*x3^30*x4^13*z^28 + 3*x1^50*x2^47*x3^30*x4^13*z^28 + 4*x1^49*x2^48*x3^30*x4^13*z^28 - x1^56*x2^40*x3^31*x4^13*z^28 + 2*x1^55*x2^41*x3^31*x4^13*z^28 + 6*x1^54*x2^42*x3^31*x4^13*z^28 + 3*x1^52*x2^44*x3^31*x4^13*z^28 - 2*x1^51*x2^45*x3^31*x4^13*z^28 - 2*x1^50*x2^46*x3^31*x4^13*z^28 - 5*x1^49*x2^47*x3^31*x4^13*z^28 - 6*x1^54*x2^41*x3^32*x4^13*z^28 - 2*x1^53*x2^42*x3^32*x4^13*z^28 - x1^52*x2^43*x3^32*x4^13*z^28 - 3*x1^51*x2^44*x3^32*x4^13*z^28 + x1^50*x2^45*x3^32*x4^13*z^28 + 2*x1^49*x2^46*x3^32*x4^13*z^28 + 2*x1^48*x2^47*x3^32*x4^13*z^28 - x1^55*x2^39*x3^33*x4^13*z^28 - x1^54*x2^40*x3^33*x4^13*z^28 + 2*x1^53*x2^41*x3^33*x4^13*z^28 + x1^52*x2^42*x3^33*x4^13*z^28 + 4*x1^51*x2^43*x3^33*x4^13*z^28 - x1^50*x2^44*x3^33*x4^13*z^28 - x1^49*x2^45*x3^33*x4^13*z^28 - 3*x1^48*x2^46*x3^33*x4^13*z^28 - x1^53*x2^40*x3^34*x4^13*z^28 - 2*x1^52*x2^41*x3^34*x4^13*z^28 - 2*x1^51*x2^42*x3^34*x4^13*z^28 - x1^50*x2^43*x3^34*x4^13*z^28 + x1^49*x2^44*x3^34*x4^13*z^28 + x1^47*x2^46*x3^34*x4^13*z^28 - x1^52*x2^40*x3^35*x4^13*z^28 + 2*x1^50*x2^42*x3^35*x4^13*z^28 - x1^49*x2^42*x3^36*x4^13*z^28 + x1^60*x2^45*x3^21*x4^14*z^28 - x1^60*x2^44*x3^22*x4^14*z^28 + x1^58*x2^46*x3^22*x4^14*z^28 - 2*x1^60*x2^43*x3^23*x4^14*z^28 - x1^59*x2^44*x3^23*x4^14*z^28 + 3*x1^58*x2^45*x3^23*x4^14*z^28 + 2*x1^59*x2^43*x3^24*x4^14*z^28 - x1^58*x2^44*x3^24*x4^14*z^28 - 2*x1^57*x2^45*x3^24*x4^14*z^28 - x1^55*x2^47*x3^24*x4^14*z^28 - x1^59*x2^42*x3^25*x4^14*z^28 - 2*x1^58*x2^43*x3^25*x4^14*z^28 + 3*x1^57*x2^44*x3^25*x4^14*z^28 + 2*x1^56*x2^45*x3^25*x4^14*z^28 + 2*x1^55*x2^46*x3^25*x4^14*z^28 - x1^53*x2^48*x3^25*x4^14*z^28 + 2*x1^58*x2^42*x3^26*x4^14*z^28 - x1^57*x2^43*x3^26*x4^14*z^28 - 3*x1^56*x2^44*x3^26*x4^14*z^28 - x1^55*x2^45*x3^26*x4^14*z^28 - 2*x1^54*x2^46*x3^26*x4^14*z^28 + x1^53*x2^47*x3^26*x4^14*z^28 + x1^52*x2^48*x3^26*x4^14*z^28 - 2*x1^57*x2^42*x3^27*x4^14*z^28 + 2*x1^56*x2^43*x3^27*x4^14*z^28 - 2*x1^55*x2^44*x3^27*x4^14*z^28 + x1^54*x2^45*x3^27*x4^14*z^28 + 2*x1^53*x2^46*x3^27*x4^14*z^28 + x1^52*x2^47*x3^27*x4^14*z^28 - x1^51*x2^48*x3^27*x4^14*z^28 + 2*x1^57*x2^41*x3^28*x4^14*z^28 + x1^56*x2^42*x3^28*x4^14*z^28 - x1^55*x2^43*x3^28*x4^14*z^28 - 3*x1^53*x2^45*x3^28*x4^14*z^28 - x1^52*x2^46*x3^28*x4^14*z^28 - x1^51*x2^47*x3^28*x4^14*z^28 + 2*x1^50*x2^48*x3^28*x4^14*z^28 - x1^57*x2^40*x3^29*x4^14*z^28 - 2*x1^56*x2^41*x3^29*x4^14*z^28 + 2*x1^55*x2^42*x3^29*x4^14*z^28 + 2*x1^53*x2^44*x3^29*x4^14*z^28 + x1^51*x2^46*x3^29*x4^14*z^28 - 2*x1^49*x2^48*x3^29*x4^14*z^28 + 2*x1^56*x2^40*x3^30*x4^14*z^28 - x1^54*x2^42*x3^30*x4^14*z^28 + x1^53*x2^43*x3^30*x4^14*z^28 - x1^52*x2^44*x3^30*x4^14*z^28 - x1^51*x2^45*x3^30*x4^14*z^28 - x1^50*x2^46*x3^30*x4^14*z^28 + 2*x1^49*x2^47*x3^30*x4^14*z^28 + x1^48*x2^48*x3^30*x4^14*z^28 - 2*x1^55*x2^40*x3^31*x4^14*z^28 + 2*x1^54*x2^41*x3^31*x4^14*z^28 + 2*x1^52*x2^43*x3^31*x4^14*z^28 + x1^51*x2^44*x3^31*x4^14*z^28 - 2*x1^49*x2^46*x3^31*x4^14*z^28 - 2*x1^48*x2^47*x3^31*x4^14*z^28 + 2*x1^55*x2^39*x3^32*x4^14*z^28 + x1^54*x2^40*x3^32*x4^14*z^28 - x1^53*x2^41*x3^32*x4^14*z^28 + x1^52*x2^42*x3^32*x4^14*z^28 + x1^51*x2^43*x3^32*x4^14*z^28 + 2*x1^50*x2^44*x3^32*x4^14*z^28 + 2*x1^48*x2^46*x3^32*x4^14*z^28 - x1^54*x2^39*x3^33*x4^14*z^28 - 3*x1^52*x2^41*x3^33*x4^14*z^28 - x1^51*x2^42*x3^33*x4^14*z^28 + 2*x1^50*x2^43*x3^33*x4^14*z^28 - x1^48*x2^45*x3^33*x4^14*z^28 - x1^47*x2^46*x3^33*x4^14*z^28 - x1^51*x2^41*x3^34*x4^14*z^28 + x1^47*x2^45*x3^34*x4^14*z^28 - x1^52*x2^39*x3^35*x4^14*z^28 - x1^51*x2^40*x3^35*x4^14*z^28 - x1^48*x2^43*x3^35*x4^14*z^28 - x1^46*x2^45*x3^35*x4^14*z^28 - x1^60*x2^44*x3^21*x4^15*z^28 + x1^60*x2^43*x3^22*x4^15*z^28 + x1^59*x2^44*x3^22*x4^15*z^28 - x1^58*x2^45*x3^22*x4^15*z^28 + x1^57*x2^46*x3^22*x4^15*z^28 - 4*x1^59*x2^43*x3^23*x4^15*z^28 - x1^58*x2^44*x3^23*x4^15*z^28 - x1^57*x2^45*x3^23*x4^15*z^28 + x1^55*x2^47*x3^23*x4^15*z^28 + 2*x1^59*x2^42*x3^24*x4^15*z^28 + 5*x1^58*x2^43*x3^24*x4^15*z^28 + 2*x1^56*x2^45*x3^24*x4^15*z^28 - x1^54*x2^47*x3^24*x4^15*z^28 - 6*x1^58*x2^42*x3^25*x4^15*z^28 - 2*x1^57*x2^43*x3^25*x4^15*z^28 - 2*x1^56*x2^44*x3^25*x4^15*z^28 - x1^55*x2^45*x3^25*x4^15*z^28 + x1^54*x2^46*x3^25*x4^15*z^28 + x1^53*x2^47*x3^25*x4^15*z^28 + 2*x1^58*x2^41*x3^26*x4^15*z^28 + 6*x1^57*x2^42*x3^26*x4^15*z^28 + 3*x1^55*x2^44*x3^26*x4^15*z^28 - 2*x1^54*x2^45*x3^26*x4^15*z^28 - x1^53*x2^46*x3^26*x4^15*z^28 - 2*x1^52*x2^47*x3^26*x4^15*z^28 - 6*x1^57*x2^41*x3^27*x4^15*z^28 - 2*x1^56*x2^42*x3^27*x4^15*z^28 - 2*x1^55*x2^43*x3^27*x4^15*z^28 + x1^52*x2^46*x3^27*x4^15*z^28 + 2*x1^51*x2^47*x3^27*x4^15*z^28 + 2*x1^57*x2^40*x3^28*x4^15*z^28 + 6*x1^56*x2^41*x3^28*x4^15*z^28 + 4*x1^54*x2^43*x3^28*x4^15*z^28 - 4*x1^53*x2^44*x3^28*x4^15*z^28 - 6*x1^51*x2^46*x3^28*x4^15*z^28 - x1^50*x2^47*x3^28*x4^15*z^28 - 6*x1^56*x2^40*x3^29*x4^15*z^28 - 2*x1^55*x2^41*x3^29*x4^15*z^28 - 2*x1^54*x2^42*x3^29*x4^15*z^28 + 2*x1^52*x2^44*x3^29*x4^15*z^28 + 2*x1^51*x2^45*x3^29*x4^15*z^28 + 6*x1^50*x2^46*x3^29*x4^15*z^28 + x1^56*x2^39*x3^30*x4^15*z^28 + 6*x1^55*x2^40*x3^30*x4^15*z^28 + 4*x1^53*x2^42*x3^30*x4^15*z^28 - 4*x1^52*x2^43*x3^30*x4^15*z^28 - 6*x1^50*x2^45*x3^30*x4^15*z^28 - 2*x1^49*x2^46*x3^30*x4^15*z^28 - 5*x1^55*x2^39*x3^31*x4^15*z^28 - 2*x1^54*x2^40*x3^31*x4^15*z^28 - 2*x1^53*x2^41*x3^31*x4^15*z^28 - x1^52*x2^42*x3^31*x4^15*z^28 + x1^51*x2^43*x3^31*x4^15*z^28 + 3*x1^50*x2^44*x3^31*x4^15*z^28 + 5*x1^49*x2^45*x3^31*x4^15*z^28 + 5*x1^54*x2^39*x3^32*x4^15*z^28 + 3*x1^52*x2^41*x3^32*x4^15*z^28 - 2*x1^51*x2^42*x3^32*x4^15*z^28 - 4*x1^49*x2^44*x3^32*x4^15*z^28 - x1^48*x2^45*x3^32*x4^15*z^28 - 2*x1^54*x2^38*x3^33*x4^15*z^28 - 3*x1^53*x2^39*x3^33*x4^15*z^28 - 2*x1^52*x2^40*x3^33*x4^15*z^28 + 2*x1^49*x2^43*x3^33*x4^15*z^28 + 2*x1^48*x2^44*x3^33*x4^15*z^28 + 2*x1^53*x2^38*x3^34*x4^15*z^28 + 2*x1^52*x2^39*x3^34*x4^15*z^28 + 2*x1^51*x2^40*x3^34*x4^15*z^28 - x1^50*x2^41*x3^34*x4^15*z^28 - 2*x1^48*x2^43*x3^34*x4^15*z^28 - x1^51*x2^39*x3^35*x4^15*z^28 - x1^49*x2^41*x3^35*x4^15*z^28 + x1^48*x2^42*x3^35*x4^15*z^28 + x1^47*x2^43*x3^35*x4^15*z^28 + x1^50*x2^39*x3^36*x4^15*z^28 - x1^49*x2^40*x3^36*x4^15*z^28 - x1^48*x2^41*x3^36*x4^15*z^28 + x1^47*x2^42*x3^36*x4^15*z^28 + 2*x1^59*x2^43*x3^22*x4^16*z^28 - x1^57*x2^45*x3^22*x4^16*z^28 - 3*x1^58*x2^43*x3^23*x4^16*z^28 + x1^57*x2^44*x3^23*x4^16*z^28 + 4*x1^58*x2^42*x3^24*x4^16*z^28 + x1^57*x2^43*x3^24*x4^16*z^28 - x1^54*x2^46*x3^24*x4^16*z^28 - x1^53*x2^47*x3^24*x4^16*z^28 - x1^58*x2^41*x3^25*x4^16*z^28 - 5*x1^57*x2^42*x3^25*x4^16*z^28 - x1^56*x2^43*x3^25*x4^16*z^28 - 2*x1^55*x2^44*x3^25*x4^16*z^28 + 3*x1^54*x2^45*x3^25*x4^16*z^28 + x1^53*x2^46*x3^25*x4^16*z^28 + 2*x1^52*x2^47*x3^25*x4^16*z^28 + 6*x1^57*x2^41*x3^26*x4^16*z^28 + 2*x1^56*x2^42*x3^26*x4^16*z^28 + 2*x1^55*x2^43*x3^26*x4^16*z^28 - 2*x1^53*x2^45*x3^26*x4^16*z^28 - x1^52*x2^46*x3^26*x4^16*z^28 - 2*x1^51*x2^47*x3^26*x4^16*z^28 - 2*x1^57*x2^40*x3^27*x4^16*z^28 - 6*x1^56*x2^41*x3^27*x4^16*z^28 - 4*x1^54*x2^43*x3^27*x4^16*z^28 + 4*x1^53*x2^44*x3^27*x4^16*z^28 + x1^52*x2^45*x3^27*x4^16*z^28 + 5*x1^51*x2^46*x3^27*x4^16*z^28 + x1^50*x2^47*x3^27*x4^16*z^28 + 6*x1^56*x2^40*x3^28*x4^16*z^28 + 2*x1^55*x2^41*x3^28*x4^16*z^28 + 2*x1^54*x2^42*x3^28*x4^16*z^28 - 2*x1^52*x2^44*x3^28*x4^16*z^28 - 2*x1^51*x2^45*x3^28*x4^16*z^28 - 5*x1^50*x2^46*x3^28*x4^16*z^28 - 2*x1^56*x2^39*x3^29*x4^16*z^28 - 6*x1^55*x2^40*x3^29*x4^16*z^28 - 4*x1^53*x2^42*x3^29*x4^16*z^28 + 4*x1^52*x2^43*x3^29*x4^16*z^28 + 6*x1^50*x2^45*x3^29*x4^16*z^28 + 2*x1^49*x2^46*x3^29*x4^16*z^28 + 5*x1^55*x2^39*x3^30*x4^16*z^28 + 2*x1^54*x2^40*x3^30*x4^16*z^28 + 2*x1^53*x2^41*x3^30*x4^16*z^28 - 2*x1^51*x2^43*x3^30*x4^16*z^28 - 2*x1^50*x2^44*x3^30*x4^16*z^28 - 6*x1^49*x2^45*x3^30*x4^16*z^28 - 5*x1^54*x2^39*x3^31*x4^16*z^28 - 2*x1^53*x2^40*x3^31*x4^16*z^28 - 4*x1^52*x2^41*x3^31*x4^16*z^28 + 4*x1^51*x2^42*x3^31*x4^16*z^28 + 6*x1^49*x2^44*x3^31*x4^16*z^28 + 2*x1^48*x2^45*x3^31*x4^16*z^28 + x1^54*x2^38*x3^32*x4^16*z^28 + 2*x1^53*x2^39*x3^32*x4^16*z^28 + 3*x1^52*x2^40*x3^32*x4^16*z^28 - 2*x1^50*x2^42*x3^32*x4^16*z^28 - 2*x1^49*x2^43*x3^32*x4^16*z^28 - 6*x1^48*x2^44*x3^32*x4^16*z^28 - x1^53*x2^38*x3^33*x4^16*z^28 - x1^52*x2^39*x3^33*x4^16*z^28 - 2*x1^51*x2^40*x3^33*x4^16*z^28 + 2*x1^50*x2^41*x3^33*x4^16*z^28 - x1^49*x2^42*x3^33*x4^16*z^28 + 5*x1^48*x2^43*x3^33*x4^16*z^28 + 2*x1^47*x2^44*x3^33*x4^16*z^28 + x1^51*x2^39*x3^34*x4^16*z^28 - 2*x1^48*x2^42*x3^34*x4^16*z^28 - 3*x1^47*x2^43*x3^34*x4^16*z^28 - 2*x1^48*x2^41*x3^35*x4^16*z^28 + 2*x1^47*x2^42*x3^35*x4^16*z^28 + x1^48*x2^40*x3^36*x4^16*z^28 - x1^46*x2^42*x3^36*x4^16*z^28 + x1^55*x2^45*x3^23*x4^17*z^28 + x1^54*x2^46*x3^23*x4^17*z^28 + 2*x1^57*x2^42*x3^24*x4^17*z^28 - x1^55*x2^44*x3^24*x4^17*z^28 - x1^54*x2^45*x3^24*x4^17*z^28 - 2*x1^53*x2^46*x3^24*x4^17*z^28 - x1^52*x2^47*x3^24*x4^17*z^28 - x1^57*x2^41*x3^25*x4^17*z^28 + x1^53*x2^45*x3^25*x4^17*z^28 + 2*x1^52*x2^46*x3^25*x4^17*z^28 + x1^51*x2^47*x3^25*x4^17*z^28 + x1^56*x2^41*x3^26*x4^17*z^28 + x1^55*x2^42*x3^26*x4^17*z^28 - 2*x1^53*x2^44*x3^26*x4^17*z^28 - 2*x1^51*x2^46*x3^26*x4^17*z^28 - 2*x1^56*x2^40*x3^27*x4^17*z^28 + 2*x1^50*x2^46*x3^27*x4^17*z^28 + 2*x1^55*x2^40*x3^28*x4^17*z^28 + 2*x1^53*x2^42*x3^28*x4^17*z^28 - x1^52*x2^43*x3^28*x4^17*z^28 - 2*x1^50*x2^45*x3^28*x4^17*z^28 - x1^49*x2^46*x3^28*x4^17*z^28 - 2*x1^55*x2^39*x3^29*x4^17*z^28 - x1^54*x2^40*x3^29*x4^17*z^28 - x1^53*x2^41*x3^29*x4^17*z^28 + x1^51*x2^43*x3^29*x4^17*z^28 + x1^50*x2^44*x3^29*x4^17*z^28 + 2*x1^49*x2^45*x3^29*x4^17*z^28 + 2*x1^54*x2^39*x3^30*x4^17*z^28 + x1^52*x2^41*x3^30*x4^17*z^28 - x1^51*x2^42*x3^30*x4^17*z^28 - 2*x1^49*x2^44*x3^30*x4^17*z^28 - x1^48*x2^45*x3^30*x4^17*z^28 + x1^50*x2^42*x3^31*x4^17*z^28 + x1^49*x2^43*x3^31*x4^17*z^28 + 2*x1^48*x2^44*x3^31*x4^17*z^28 - x1^50*x2^41*x3^32*x4^17*z^28 - 2*x1^48*x2^43*x3^32*x4^17*z^28 + x1^50*x2^40*x3^33*x4^17*z^28 + 2*x1^47*x2^43*x3^33*x4^17*z^28 - x1^49*x2^40*x3^34*x4^17*z^28 - x1^47*x2^42*x3^34*x4^17*z^28 - x1^46*x2^43*x3^34*x4^17*z^28 + x1^46*x2^42*x3^35*x4^17*z^28 - x1^60*x2^46*x3^26*x4^3*z^27 + x1^62*x2^45*x3^24*x4^4*z^27 + x1^61*x2^44*x3^26*x4^4*z^27 + x1^59*x2^45*x3^27*x4^4*z^27 - x1^58*x2^46*x3^27*x4^4*z^27 - x1^57*x2^47*x3^27*x4^4*z^27 + x1^61*x2^45*x3^24*x4^5*z^27 - 2*x1^61*x2^44*x3^25*x4^5*z^27 + x1^59*x2^46*x3^25*x4^5*z^27 + 3*x1^60*x2^44*x3^26*x4^5*z^27 + 2*x1^58*x2^46*x3^26*x4^5*z^27 + x1^57*x2^47*x3^26*x4^5*z^27 - x1^60*x2^43*x3^27*x4^5*z^27 - x1^59*x2^44*x3^27*x4^5*z^27 + x1^58*x2^45*x3^27*x4^5*z^27 - 3*x1^57*x2^46*x3^27*x4^5*z^27 - x1^56*x2^47*x3^27*x4^5*z^27 + 2*x1^59*x2^43*x3^28*x4^5*z^27 - x1^55*x2^47*x3^28*x4^5*z^27 + 2*x1^57*x2^44*x3^29*x4^5*z^27 + x1^61*x2^44*x3^24*x4^6*z^27 - 2*x1^60*x2^44*x3^25*x4^6*z^27 - x1^58*x2^46*x3^25*x4^6*z^27 + x1^59*x2^44*x3^26*x4^6*z^27 - 2*x1^56*x2^47*x3^26*x4^6*z^27 - 3*x1^59*x2^43*x3^27*x4^6*z^27 + x1^58*x2^44*x3^27*x4^6*z^27 + x1^57*x2^45*x3^27*x4^6*z^27 - x1^56*x2^46*x3^27*x4^6*z^27 + x1^58*x2^43*x3^28*x4^6*z^27 - x1^57*x2^44*x3^28*x4^6*z^27 + 3*x1^56*x2^45*x3^28*x4^6*z^27 + x1^55*x2^46*x3^28*x4^6*z^27 - 2*x1^58*x2^42*x3^29*x4^6*z^27 - x1^57*x2^43*x3^29*x4^6*z^27 + x1^55*x2^45*x3^29*x4^6*z^27 - 2*x1^56*x2^43*x3^30*x4^6*z^27 - x1^53*x2^46*x3^30*x4^6*z^27 + x1^61*x2^44*x3^23*x4^7*z^27 - x1^59*x2^46*x3^23*x4^7*z^27 + x1^58*x2^47*x3^23*x4^7*z^27 - x1^60*x2^44*x3^24*x4^7*z^27 + x1^59*x2^45*x3^24*x4^7*z^27 + x1^58*x2^46*x3^24*x4^7*z^27 - 2*x1^57*x2^47*x3^24*x4^7*z^27 + x1^57*x2^46*x3^25*x4^7*z^27 + x1^55*x2^48*x3^25*x4^7*z^27 - x1^58*x2^44*x3^26*x4^7*z^27 + x1^57*x2^45*x3^26*x4^7*z^27 - x1^56*x2^46*x3^26*x4^7*z^27 - 2*x1^58*x2^43*x3^27*x4^7*z^27 + x1^57*x2^44*x3^27*x4^7*z^27 + x1^56*x2^45*x3^27*x4^7*z^27 + x1^55*x2^46*x3^27*x4^7*z^27 - x1^53*x2^48*x3^27*x4^7*z^27 + x1^58*x2^42*x3^28*x4^7*z^27 - x1^57*x2^43*x3^28*x4^7*z^27 + x1^54*x2^46*x3^28*x4^7*z^27 - x1^53*x2^47*x3^28*x4^7*z^27 - x1^52*x2^48*x3^28*x4^7*z^27 - 2*x1^57*x2^42*x3^29*x4^7*z^27 - 2*x1^55*x2^44*x3^29*x4^7*z^27 + x1^57*x2^41*x3^30*x4^7*z^27 + x1^53*x2^45*x3^30*x4^7*z^27 + 2*x1^59*x2^46*x3^22*x4^8*z^27 - x1^60*x2^44*x3^23*x4^8*z^27 - x1^59*x2^45*x3^23*x4^8*z^27 - 2*x1^58*x2^46*x3^23*x4^8*z^27 + x1^57*x2^47*x3^23*x4^8*z^27 + x1^59*x2^44*x3^24*x4^8*z^27 + 4*x1^58*x2^45*x3^24*x4^8*z^27 + x1^57*x2^46*x3^24*x4^8*z^27 + x1^56*x2^47*x3^24*x4^8*z^27 - x1^59*x2^43*x3^25*x4^8*z^27 - 3*x1^58*x2^44*x3^25*x4^8*z^27 - 5*x1^57*x2^45*x3^25*x4^8*z^27 - 2*x1^55*x2^47*x3^25*x4^8*z^27 + x1^59*x2^42*x3^26*x4^8*z^27 + 3*x1^57*x2^44*x3^26*x4^8*z^27 + x1^56*x2^45*x3^26*x4^8*z^27 + x1^55*x2^46*x3^26*x4^8*z^27 + x1^54*x2^47*x3^26*x4^8*z^27 - x1^52*x2^49*x3^26*x4^8*z^27 - x1^58*x2^42*x3^27*x4^8*z^27 - 4*x1^56*x2^44*x3^27*x4^8*z^27 - x1^55*x2^45*x3^27*x4^8*z^27 - 3*x1^54*x2^46*x3^27*x4^8*z^27 + x1^53*x2^47*x3^27*x4^8*z^27 + x1^52*x2^48*x3^27*x4^8*z^27 + x1^51*x2^49*x3^27*x4^8*z^27 + 3*x1^57*x2^42*x3^28*x4^8*z^27 + 3*x1^56*x2^43*x3^28*x4^8*z^27 + 3*x1^55*x2^44*x3^28*x4^8*z^27 + 3*x1^54*x2^45*x3^28*x4^8*z^27 - 2*x1^56*x2^42*x3^29*x4^8*z^27 - 2*x1^55*x2^43*x3^29*x4^8*z^27 + x1^54*x2^44*x3^29*x4^8*z^27 - 2*x1^53*x2^45*x3^29*x4^8*z^27 + x1^52*x2^46*x3^29*x4^8*z^27 + x1^51*x2^47*x3^29*x4^8*z^27 + x1^57*x2^40*x3^30*x4^8*z^27 + x1^56*x2^41*x3^30*x4^8*z^27 + 2*x1^55*x2^42*x3^30*x4^8*z^27 + 2*x1^54*x2^43*x3^30*x4^8*z^27 + x1^53*x2^44*x3^30*x4^8*z^27 + x1^52*x2^45*x3^30*x4^8*z^27 - x1^55*x2^41*x3^31*x4^8*z^27 - 2*x1^54*x2^42*x3^31*x4^8*z^27 - 2*x1^52*x2^44*x3^31*x4^8*z^27 + x1^56*x2^39*x3^32*x4^8*z^27 + x1^55*x2^40*x3^32*x4^8*z^27 + x1^54*x2^41*x3^32*x4^8*z^27 + x1^53*x2^42*x3^32*x4^8*z^27 + x1^51*x2^44*x3^32*x4^8*z^27 + x1^60*x2^43*x3^23*x4^9*z^27 - 2*x1^58*x2^45*x3^23*x4^9*z^27 - x1^56*x2^47*x3^23*x4^9*z^27 + 2*x1^58*x2^44*x3^24*x4^9*z^27 + 3*x1^57*x2^45*x3^24*x4^9*z^27 - x1^56*x2^46*x3^24*x4^9*z^27 + x1^55*x2^47*x3^24*x4^9*z^27 - 2*x1^59*x2^42*x3^25*x4^9*z^27 - 2*x1^57*x2^44*x3^25*x4^9*z^27 - x1^55*x2^46*x3^25*x4^9*z^27 + 4*x1^58*x2^42*x3^26*x4^9*z^27 - x1^57*x2^43*x3^26*x4^9*z^27 + 3*x1^56*x2^44*x3^26*x4^9*z^27 + 3*x1^54*x2^46*x3^26*x4^9*z^27 - x1^53*x2^47*x3^26*x4^9*z^27 + x1^52*x2^48*x3^26*x4^9*z^27 - x1^58*x2^41*x3^27*x4^9*z^27 - 3*x1^57*x2^42*x3^27*x4^9*z^27 + x1^56*x2^43*x3^27*x4^9*z^27 - x1^55*x2^44*x3^27*x4^9*z^27 + x1^54*x2^45*x3^27*x4^9*z^27 + 3*x1^57*x2^41*x3^28*x4^9*z^27 + 2*x1^56*x2^42*x3^28*x4^9*z^27 - x1^55*x2^43*x3^28*x4^9*z^27 + 2*x1^54*x2^44*x3^28*x4^9*z^27 + 3*x1^53*x2^45*x3^28*x4^9*z^27 - x1^57*x2^40*x3^29*x4^9*z^27 - x1^56*x2^41*x3^29*x4^9*z^27 - x1^55*x2^42*x3^29*x4^9*z^27 - 4*x1^54*x2^43*x3^29*x4^9*z^27 - 2*x1^53*x2^44*x3^29*x4^9*z^27 + x1^52*x2^45*x3^29*x4^9*z^27 + x1^50*x2^47*x3^29*x4^9*z^27 + x1^56*x2^40*x3^30*x4^9*z^27 + 2*x1^55*x2^41*x3^30*x4^9*z^27 + 3*x1^54*x2^42*x3^30*x4^9*z^27 - x1^51*x2^45*x3^30*x4^9*z^27 - x1^49*x2^47*x3^30*x4^9*z^27 - 2*x1^56*x2^39*x3^31*x4^9*z^27 + x1^54*x2^41*x3^31*x4^9*z^27 - x1^53*x2^42*x3^31*x4^9*z^27 + x1^52*x2^43*x3^31*x4^9*z^27 - x1^51*x2^44*x3^31*x4^9*z^27 - x1^54*x2^40*x3^32*x4^9*z^27 + x1^53*x2^41*x3^32*x4^9*z^27 + x1^52*x2^42*x3^32*x4^9*z^27 - x1^53*x2^40*x3^33*x4^9*z^27 - x1^52*x2^41*x3^33*x4^9*z^27 - x1^50*x2^43*x3^33*x4^9*z^27 - x1^61*x2^44*x3^20*x4^10*z^27 + 2*x1^60*x2^44*x3^21*x4^10*z^27 - x1^59*x2^45*x3^21*x4^10*z^27 - x1^60*x2^43*x3^22*x4^10*z^27 - 2*x1^59*x2^44*x3^22*x4^10*z^27 + x1^58*x2^45*x3^22*x4^10*z^27 - 2*x1^57*x2^46*x3^22*x4^10*z^27 + 2*x1^59*x2^43*x3^23*x4^10*z^27 + 2*x1^58*x2^44*x3^23*x4^10*z^27 - x1^55*x2^47*x3^23*x4^10*z^27 - 2*x1^58*x2^43*x3^24*x4^10*z^27 - x1^57*x2^44*x3^24*x4^10*z^27 - x1^56*x2^45*x3^24*x4^10*z^27 + x1^55*x2^46*x3^24*x4^10*z^27 - x1^53*x2^48*x3^24*x4^10*z^27 + x1^57*x2^43*x3^25*x4^10*z^27 + x1^56*x2^44*x3^25*x4^10*z^27 + x1^54*x2^46*x3^25*x4^10*z^27 - x1^53*x2^47*x3^25*x4^10*z^27 + x1^52*x2^48*x3^25*x4^10*z^27 - x1^55*x2^44*x3^26*x4^10*z^27 + 2*x1^54*x2^45*x3^26*x4^10*z^27 + x1^53*x2^46*x3^26*x4^10*z^27 + 2*x1^52*x2^47*x3^26*x4^10*z^27 - x1^51*x2^48*x3^26*x4^10*z^27 - x1^53*x2^45*x3^27*x4^10*z^27 - 2*x1^51*x2^47*x3^27*x4^10*z^27 + x1^53*x2^44*x3^28*x4^10*z^27 + 3*x1^51*x2^46*x3^28*x4^10*z^27 + x1^50*x2^47*x3^28*x4^10*z^27 - x1^55*x2^41*x3^29*x4^10*z^27 + 2*x1^54*x2^42*x3^29*x4^10*z^27 - 2*x1^52*x2^44*x3^29*x4^10*z^27 - 2*x1^49*x2^47*x3^29*x4^10*z^27 + x1^56*x2^39*x3^30*x4^10*z^27 - x1^55*x2^40*x3^30*x4^10*z^27 - x1^54*x2^41*x3^30*x4^10*z^27 - x1^52*x2^43*x3^30*x4^10*z^27 + x1^51*x2^44*x3^30*x4^10*z^27 - x1^50*x2^45*x3^30*x4^10*z^27 + 2*x1^48*x2^47*x3^30*x4^10*z^27 + x1^53*x2^41*x3^31*x4^10*z^27 - x1^52*x2^42*x3^31*x4^10*z^27 + x1^51*x2^43*x3^31*x4^10*z^27 - x1^49*x2^45*x3^31*x4^10*z^27 - x1^48*x2^46*x3^31*x4^10*z^27 + x1^55*x2^38*x3^32*x4^10*z^27 - x1^54*x2^39*x3^32*x4^10*z^27 - 2*x1^53*x2^40*x3^32*x4^10*z^27 - x1^51*x2^42*x3^32*x4^10*z^27 - x1^54*x2^38*x3^33*x4^10*z^27 + x1^53*x2^39*x3^33*x4^10*z^27 - 2*x1^51*x2^41*x3^33*x4^10*z^27 + 2*x1^50*x2^42*x3^33*x4^10*z^27 - x1^52*x2^39*x3^34*x4^10*z^27 - x1^50*x2^41*x3^34*x4^10*z^27 - x1^60*x2^44*x3^20*x4^11*z^27 + x1^60*x2^43*x3^21*x4^11*z^27 + x1^59*x2^44*x3^21*x4^11*z^27 - x1^58*x2^45*x3^21*x4^11*z^27 - 3*x1^59*x2^43*x3^22*x4^11*z^27 + x1^56*x2^46*x3^22*x4^11*z^27 + 2*x1^59*x2^42*x3^23*x4^11*z^27 + 3*x1^58*x2^43*x3^23*x4^11*z^27 - x1^57*x2^44*x3^23*x4^11*z^27 + 4*x1^56*x2^45*x3^23*x4^11*z^27 - 4*x1^58*x2^42*x3^24*x4^11*z^27 - x1^55*x2^45*x3^24*x4^11*z^27 + 2*x1^54*x2^46*x3^24*x4^11*z^27 + 3*x1^53*x2^47*x3^24*x4^11*z^27 + x1^58*x2^41*x3^25*x4^11*z^27 + 4*x1^57*x2^42*x3^25*x4^11*z^27 - 2*x1^56*x2^43*x3^25*x4^11*z^27 + x1^55*x2^44*x3^25*x4^11*z^27 - 4*x1^54*x2^45*x3^25*x4^11*z^27 - x1^53*x2^46*x3^25*x4^11*z^27 - 3*x1^52*x2^47*x3^25*x4^11*z^27 - 4*x1^57*x2^41*x3^26*x4^11*z^27 + x1^55*x2^43*x3^26*x4^11*z^27 + x1^53*x2^45*x3^26*x4^11*z^27 - 2*x1^52*x2^46*x3^26*x4^11*z^27 + 3*x1^51*x2^47*x3^26*x4^11*z^27 + x1^50*x2^48*x3^26*x4^11*z^27 + x1^57*x2^40*x3^27*x4^11*z^27 + 4*x1^56*x2^41*x3^27*x4^11*z^27 - 2*x1^55*x2^42*x3^27*x4^11*z^27 + 3*x1^54*x2^43*x3^27*x4^11*z^27 - x1^53*x2^44*x3^27*x4^11*z^27 - 2*x1^52*x2^45*x3^27*x4^11*z^27 - 4*x1^51*x2^46*x3^27*x4^11*z^27 - x1^50*x2^47*x3^27*x4^11*z^27 - x1^49*x2^48*x3^27*x4^11*z^27 - 4*x1^56*x2^40*x3^28*x4^11*z^27 - 2*x1^55*x2^41*x3^28*x4^11*z^27 + 4*x1^52*x2^44*x3^28*x4^11*z^27 + x1^51*x2^45*x3^28*x4^11*z^27 + x1^50*x2^46*x3^28*x4^11*z^27 - x1^49*x2^47*x3^28*x4^11*z^27 + 2*x1^56*x2^39*x3^29*x4^11*z^27 + 3*x1^55*x2^40*x3^29*x4^11*z^27 - 3*x1^54*x2^41*x3^29*x4^11*z^27 + x1^53*x2^42*x3^29*x4^11*z^27 - x1^52*x2^43*x3^29*x4^11*z^27 - 2*x1^51*x2^44*x3^29*x4^11*z^27 - 3*x1^50*x2^45*x3^29*x4^11*z^27 - x1^48*x2^47*x3^29*x4^11*z^27 - 2*x1^55*x2^39*x3^30*x4^11*z^27 - 2*x1^53*x2^41*x3^30*x4^11*z^27 - 2*x1^52*x2^42*x3^30*x4^11*z^27 + 2*x1^51*x2^43*x3^30*x4^11*z^27 + 3*x1^50*x2^44*x3^30*x4^11*z^27 - x1^48*x2^46*x3^30*x4^11*z^27 - x1^55*x2^38*x3^31*x4^11*z^27 + 3*x1^54*x2^39*x3^31*x4^11*z^27 + x1^52*x2^41*x3^31*x4^11*z^27 - x1^51*x2^42*x3^31*x4^11*z^27 - x1^50*x2^43*x3^31*x4^11*z^27 - x1^47*x2^46*x3^31*x4^11*z^27 + 2*x1^54*x2^38*x3^32*x4^11*z^27 + x1^53*x2^39*x3^32*x4^11*z^27 + x1^51*x2^41*x3^32*x4^11*z^27 + 2*x1^48*x2^44*x3^32*x4^11*z^27 + x1^47*x2^45*x3^32*x4^11*z^27 - x1^51*x2^40*x3^33*x4^11*z^27 - 2*x1^50*x2^41*x3^33*x4^11*z^27 + x1^50*x2^40*x3^34*x4^11*z^27 - x1^49*x2^41*x3^34*x4^11*z^27 - x1^60*x2^43*x3^20*x4^12*z^27 + x1^59*x2^44*x3^20*x4^12*z^27 + x1^59*x2^43*x3^21*x4^12*z^27 - x1^58*x2^44*x3^21*x4^12*z^27 - x1^59*x2^42*x3^22*x4^12*z^27 - 2*x1^58*x2^43*x3^22*x4^12*z^27 + 3*x1^57*x2^44*x3^22*x4^12*z^27 - x1^56*x2^45*x3^22*x4^12*z^27 + x1^55*x2^46*x3^22*x4^12*z^27 + 2*x1^58*x2^42*x3^23*x4^12*z^27 - 2*x1^56*x2^44*x3^23*x4^12*z^27 - 2*x1^54*x2^46*x3^23*x4^12*z^27 + x1^53*x2^47*x3^23*x4^12*z^27 - x1^58*x2^41*x3^24*x4^12*z^27 - 2*x1^57*x2^42*x3^24*x4^12*z^27 + 5*x1^56*x2^43*x3^24*x4^12*z^27 + 3*x1^54*x2^45*x3^24*x4^12*z^27 + x1^53*x2^46*x3^24*x4^12*z^27 - x1^52*x2^47*x3^24*x4^12*z^27 + 2*x1^57*x2^41*x3^25*x4^12*z^27 - 2*x1^56*x2^42*x3^25*x4^12*z^27 - 5*x1^55*x2^43*x3^25*x4^12*z^27 + 2*x1^54*x2^44*x3^25*x4^12*z^27 - 3*x1^53*x2^45*x3^25*x4^12*z^27 + x1^52*x2^46*x3^25*x4^12*z^27 + x1^51*x2^47*x3^25*x4^12*z^27 + x1^50*x2^48*x3^25*x4^12*z^27 - 2*x1^56*x2^41*x3^26*x4^12*z^27 + 6*x1^55*x2^42*x3^26*x4^12*z^27 + 2*x1^53*x2^44*x3^26*x4^12*z^27 - 2*x1^50*x2^47*x3^26*x4^12*z^27 - x1^49*x2^48*x3^26*x4^12*z^27 + 2*x1^56*x2^40*x3^27*x4^12*z^27 - x1^55*x2^41*x3^27*x4^12*z^27 - 5*x1^54*x2^42*x3^27*x4^12*z^27 - 4*x1^52*x2^44*x3^27*x4^12*z^27 + 4*x1^51*x2^45*x3^27*x4^12*z^27 + 4*x1^49*x2^47*x3^27*x4^12*z^27 - x1^56*x2^39*x3^28*x4^12*z^27 - 2*x1^55*x2^40*x3^28*x4^12*z^27 + 6*x1^54*x2^41*x3^28*x4^12*z^27 + x1^53*x2^42*x3^28*x4^12*z^27 + x1^52*x2^43*x3^28*x4^12*z^27 + 2*x1^51*x2^44*x3^28*x4^12*z^27 + x1^50*x2^45*x3^28*x4^12*z^27 - 3*x1^49*x2^46*x3^28*x4^12*z^27 - 3*x1^48*x2^47*x3^28*x4^12*z^27 + 2*x1^55*x2^39*x3^29*x4^12*z^27 - x1^54*x2^40*x3^29*x4^12*z^27 - 5*x1^53*x2^41*x3^29*x4^12*z^27 - 4*x1^51*x2^43*x3^29*x4^12*z^27 + x1^50*x2^44*x3^29*x4^12*z^27 + 5*x1^48*x2^46*x3^29*x4^12*z^27 - x1^54*x2^39*x3^30*x4^12*z^27 + 7*x1^53*x2^40*x3^30*x4^12*z^27 + x1^52*x2^41*x3^30*x4^12*z^27 + 3*x1^51*x2^42*x3^30*x4^12*z^27 + 3*x1^50*x2^43*x3^30*x4^12*z^27 + x1^49*x2^44*x3^30*x4^12*z^27 - 2*x1^48*x2^45*x3^30*x4^12*z^27 - 2*x1^47*x2^46*x3^30*x4^12*z^27 - x1^54*x2^38*x3^31*x4^12*z^27 - 2*x1^53*x2^39*x3^31*x4^12*z^27 - 4*x1^52*x2^40*x3^31*x4^12*z^27 - 3*x1^50*x2^42*x3^31*x4^12*z^27 + 3*x1^47*x2^45*x3^31*x4^12*z^27 + 2*x1^52*x2^39*x3^32*x4^12*z^27 - x1^51*x2^40*x3^32*x4^12*z^27 + 3*x1^49*x2^42*x3^32*x4^12*z^27 - x1^46*x2^45*x3^32*x4^12*z^27 - 2*x1^50*x2^40*x3^33*x4^12*z^27 - 4*x1^49*x2^41*x3^33*x4^12*z^27 - x1^47*x2^43*x3^33*x4^12*z^27 + x1^51*x2^38*x3^34*x4^12*z^27 + x1^50*x2^39*x3^34*x4^12*z^27 + 2*x1^49*x2^40*x3^34*x4^12*z^27 + x1^48*x2^41*x3^34*x4^12*z^27 - x1^48*x2^40*x3^35*x4^12*z^27 - x1^59*x2^43*x3^20*x4^13*z^27 + x1^58*x2^44*x3^20*x4^13*z^27 + 2*x1^59*x2^42*x3^21*x4^13*z^27 - x1^58*x2^43*x3^21*x4^13*z^27 - 2*x1^57*x2^44*x3^21*x4^13*z^27 + 2*x1^56*x2^45*x3^21*x4^13*z^27 + x1^57*x2^43*x3^22*x4^13*z^27 + 2*x1^56*x2^44*x3^22*x4^13*z^27 - x1^55*x2^45*x3^22*x4^13*z^27 + x1^54*x2^46*x3^22*x4^13*z^27 - x1^57*x2^42*x3^23*x4^13*z^27 - 5*x1^56*x2^43*x3^23*x4^13*z^27 + x1^54*x2^45*x3^23*x4^13*z^27 + x1^52*x2^47*x3^23*x4^13*z^27 + 2*x1^56*x2^42*x3^24*x4^13*z^27 + 5*x1^55*x2^43*x3^24*x4^13*z^27 - x1^54*x2^44*x3^24*x4^13*z^27 + 2*x1^53*x2^45*x3^24*x4^13*z^27 - x1^52*x2^46*x3^24*x4^13*z^27 - x1^51*x2^47*x3^24*x4^13*z^27 - 6*x1^55*x2^42*x3^25*x4^13*z^27 - 3*x1^54*x2^43*x3^25*x4^13*z^27 - 2*x1^53*x2^44*x3^25*x4^13*z^27 + x1^50*x2^47*x3^25*x4^13*z^27 + 2*x1^55*x2^41*x3^26*x4^13*z^27 + 6*x1^54*x2^42*x3^26*x4^13*z^27 + 3*x1^52*x2^44*x3^26*x4^13*z^27 - 4*x1^51*x2^45*x3^26*x4^13*z^27 - 4*x1^49*x2^47*x3^26*x4^13*z^27 - 6*x1^54*x2^41*x3^27*x4^13*z^27 - 2*x1^53*x2^42*x3^27*x4^13*z^27 - 2*x1^52*x2^43*x3^27*x4^13*z^27 + x1^50*x2^45*x3^27*x4^13*z^27 + 2*x1^49*x2^46*x3^27*x4^13*z^27 + 4*x1^48*x2^47*x3^27*x4^13*z^27 + 2*x1^54*x2^40*x3^28*x4^13*z^27 + 6*x1^53*x2^41*x3^28*x4^13*z^27 + 4*x1^51*x2^43*x3^28*x4^13*z^27 - 4*x1^50*x2^44*x3^28*x4^13*z^27 - 6*x1^48*x2^46*x3^28*x4^13*z^27 - x1^47*x2^47*x3^28*x4^13*z^27 + x1^55*x2^38*x3^29*x4^13*z^27 - 6*x1^53*x2^40*x3^29*x4^13*z^27 - 2*x1^52*x2^41*x3^29*x4^13*z^27 - x1^51*x2^42*x3^29*x4^13*z^27 - x1^50*x2^43*x3^29*x4^13*z^27 + 3*x1^49*x2^44*x3^29*x4^13*z^27 + 3*x1^48*x2^45*x3^29*x4^13*z^27 + 6*x1^47*x2^46*x3^29*x4^13*z^27 - x1^54*x2^38*x3^30*x4^13*z^27 + x1^53*x2^39*x3^30*x4^13*z^27 + 6*x1^52*x2^40*x3^30*x4^13*z^27 + 2*x1^50*x2^42*x3^30*x4^13*z^27 - 3*x1^49*x2^43*x3^30*x4^13*z^27 - x1^48*x2^44*x3^30*x4^13*z^27 - 5*x1^47*x2^45*x3^30*x4^13*z^27 - x1^46*x2^46*x3^30*x4^13*z^27 + x1^53*x2^38*x3^31*x4^13*z^27 - 2*x1^52*x2^39*x3^31*x4^13*z^27 - 2*x1^51*x2^40*x3^31*x4^13*z^27 - x1^50*x2^41*x3^31*x4^13*z^27 - 2*x1^49*x2^42*x3^31*x4^13*z^27 + x1^48*x2^43*x3^31*x4^13*z^27 + 3*x1^47*x2^44*x3^31*x4^13*z^27 + 4*x1^46*x2^45*x3^31*x4^13*z^27 - 2*x1^52*x2^38*x3^32*x4^13*z^27 + 3*x1^51*x2^39*x3^32*x4^13*z^27 + 2*x1^50*x2^40*x3^32*x4^13*z^27 + 2*x1^49*x2^41*x3^32*x4^13*z^27 - x1^48*x2^42*x3^32*x4^13*z^27 - 4*x1^46*x2^44*x3^32*x4^13*z^27 + x1^51*x2^38*x3^33*x4^13*z^27 - 3*x1^48*x2^41*x3^33*x4^13*z^27 + x1^47*x2^42*x3^33*x4^13*z^27 + x1^46*x2^43*x3^33*x4^13*z^27 + 2*x1^45*x2^44*x3^33*x4^13*z^27 - x1^50*x2^38*x3^34*x4^13*z^27 - x1^49*x2^39*x3^34*x4^13*z^27 + 2*x1^48*x2^40*x3^34*x4^13*z^27 - x1^47*x2^41*x3^34*x4^13*z^27 - 2*x1^46*x2^42*x3^34*x4^13*z^27 - x1^45*x2^43*x3^34*x4^13*z^27 + 2*x1^58*x2^42*x3^21*x4^14*z^27 - x1^57*x2^43*x3^21*x4^14*z^27 - 2*x1^56*x2^44*x3^21*x4^14*z^27 + x1^55*x2^45*x3^21*x4^14*z^27 - x1^58*x2^41*x3^22*x4^14*z^27 + 3*x1^56*x2^43*x3^22*x4^14*z^27 - x1^54*x2^45*x3^22*x4^14*z^27 + 3*x1^57*x2^41*x3^23*x4^14*z^27 + 2*x1^56*x2^42*x3^23*x4^14*z^27 - x1^53*x2^45*x3^23*x4^14*z^27 + x1^52*x2^46*x3^23*x4^14*z^27 - x1^57*x2^40*x3^24*x4^14*z^27 - 2*x1^56*x2^41*x3^24*x4^14*z^27 + 2*x1^55*x2^42*x3^24*x4^14*z^27 + x1^53*x2^44*x3^24*x4^14*z^27 + x1^52*x2^45*x3^24*x4^14*z^27 - x1^51*x2^46*x3^24*x4^14*z^27 + 2*x1^56*x2^40*x3^25*x4^14*z^27 - x1^54*x2^42*x3^25*x4^14*z^27 - 3*x1^52*x2^44*x3^25*x4^14*z^27 - x1^51*x2^45*x3^25*x4^14*z^27 + x1^50*x2^46*x3^25*x4^14*z^27 + x1^49*x2^47*x3^25*x4^14*z^27 - x1^56*x2^39*x3^26*x4^14*z^27 - 2*x1^55*x2^40*x3^26*x4^14*z^27 + 2*x1^54*x2^41*x3^26*x4^14*z^27 + 3*x1^52*x2^43*x3^26*x4^14*z^27 + x1^51*x2^44*x3^26*x4^14*z^27 + x1^50*x2^45*x3^26*x4^14*z^27 - 2*x1^49*x2^46*x3^26*x4^14*z^27 - x1^48*x2^47*x3^26*x4^14*z^27 + 2*x1^55*x2^39*x3^27*x4^14*z^27 - x1^54*x2^40*x3^27*x4^14*z^27 - 2*x1^53*x2^41*x3^27*x4^14*z^27 - x1^51*x2^43*x3^27*x4^14*z^27 + 2*x1^50*x2^44*x3^27*x4^14*z^27 - 2*x1^49*x2^45*x3^27*x4^14*z^27 + 2*x1^48*x2^46*x3^27*x4^14*z^27 - 2*x1^54*x2^39*x3^28*x4^14*z^27 + 2*x1^53*x2^40*x3^28*x4^14*z^27 - 2*x1^52*x2^41*x3^28*x4^14*z^27 + x1^51*x2^42*x3^28*x4^14*z^27 + 2*x1^49*x2^44*x3^28*x4^14*z^27 + x1^48*x2^45*x3^28*x4^14*z^27 - 2*x1^47*x2^46*x3^28*x4^14*z^27 + x1^54*x2^38*x3^29*x4^14*z^27 + x1^53*x2^39*x3^29*x4^14*z^27 - x1^52*x2^40*x3^29*x4^14*z^27 - 3*x1^50*x2^42*x3^29*x4^14*z^27 - 2*x1^48*x2^44*x3^29*x4^14*z^27 + 2*x1^47*x2^45*x3^29*x4^14*z^27 + x1^46*x2^46*x3^29*x4^14*z^27 - x1^54*x2^37*x3^30*x4^14*z^27 - x1^53*x2^38*x3^30*x4^14*z^27 + x1^52*x2^39*x3^30*x4^14*z^27 + x1^50*x2^41*x3^30*x4^14*z^27 - x1^49*x2^42*x3^30*x4^14*z^27 - 2*x1^46*x2^45*x3^30*x4^14*z^27 + x1^53*x2^37*x3^31*x4^14*z^27 - x1^52*x2^38*x3^31*x4^14*z^27 - x1^51*x2^39*x3^31*x4^14*z^27 - x1^49*x2^41*x3^31*x4^14*z^27 - x1^48*x2^42*x3^31*x4^14*z^27 - x1^47*x2^43*x3^31*x4^14*z^27 + 2*x1^46*x2^44*x3^31*x4^14*z^27 + x1^45*x2^45*x3^31*x4^14*z^27 - x1^52*x2^37*x3^32*x4^14*z^27 - x1^50*x2^39*x3^32*x4^14*z^27 - x1^47*x2^42*x3^32*x4^14*z^27 - 2*x1^46*x2^43*x3^32*x4^14*z^27 - 2*x1^45*x2^44*x3^32*x4^14*z^27 + x1^51*x2^37*x3^33*x4^14*z^27 + x1^48*x2^40*x3^33*x4^14*z^27 + x1^47*x2^41*x3^33*x4^14*z^27 - x1^46*x2^42*x3^33*x4^14*z^27 + x1^45*x2^43*x3^33*x4^14*z^27 + x1^48*x2^39*x3^34*x4^14*z^27 + x1^47*x2^40*x3^34*x4^14*z^27 - x1^44*x2^43*x3^34*x4^14*z^27 + x1^48*x2^38*x3^35*x4^14*z^27 + x1^45*x2^41*x3^35*x4^14*z^27 - x1^58*x2^42*x3^20*x4^15*z^27 + 2*x1^57*x2^42*x3^21*x4^15*z^27 + x1^56*x2^43*x3^21*x4^15*z^27 - 4*x1^57*x2^41*x3^22*x4^15*z^27 - x1^56*x2^42*x3^22*x4^15*z^27 - x1^54*x2^44*x3^22*x4^15*z^27 + x1^57*x2^40*x3^23*x4^15*z^27 + 5*x1^56*x2^41*x3^23*x4^15*z^27 + 3*x1^54*x2^43*x3^23*x4^15*z^27 - x1^53*x2^44*x3^23*x4^15*z^27 - x1^51*x2^46*x3^23*x4^15*z^27 - 6*x1^56*x2^40*x3^24*x4^15*z^27 - 2*x1^55*x2^41*x3^24*x4^15*z^27 - x1^54*x2^42*x3^24*x4^15*z^27 - x1^53*x2^43*x3^24*x4^15*z^27 + x1^52*x2^44*x3^24*x4^15*z^27 + x1^51*x2^45*x3^24*x4^15*z^27 + x1^50*x2^46*x3^24*x4^15*z^27 + 2*x1^56*x2^39*x3^25*x4^15*z^27 + 6*x1^55*x2^40*x3^25*x4^15*z^27 + 4*x1^53*x2^42*x3^25*x4^15*z^27 - 2*x1^52*x2^43*x3^25*x4^15*z^27 - 3*x1^50*x2^45*x3^25*x4^15*z^27 - 6*x1^55*x2^39*x3^26*x4^15*z^27 - 2*x1^54*x2^40*x3^26*x4^15*z^27 - 2*x1^53*x2^41*x3^26*x4^15*z^27 + 2*x1^51*x2^43*x3^26*x4^15*z^27 + 3*x1^50*x2^44*x3^26*x4^15*z^27 + 3*x1^49*x2^45*x3^26*x4^15*z^27 + 2*x1^55*x2^38*x3^27*x4^15*z^27 + 6*x1^54*x2^39*x3^27*x4^15*z^27 + 4*x1^52*x2^41*x3^27*x4^15*z^27 - 4*x1^51*x2^42*x3^27*x4^15*z^27 - 6*x1^49*x2^44*x3^27*x4^15*z^27 - 6*x1^54*x2^38*x3^28*x4^15*z^27 - 2*x1^53*x2^39*x3^28*x4^15*z^27 - 2*x1^52*x2^40*x3^28*x4^15*z^27 + 2*x1^50*x2^42*x3^28*x4^15*z^27 + 2*x1^49*x2^43*x3^28*x4^15*z^27 + 6*x1^48*x2^44*x3^28*x4^15*z^27 + 2*x1^54*x2^37*x3^29*x4^15*z^27 + 6*x1^53*x2^38*x3^29*x4^15*z^27 + 4*x1^51*x2^40*x3^29*x4^15*z^27 - 4*x1^50*x2^41*x3^29*x4^15*z^27 - 6*x1^48*x2^43*x3^29*x4^15*z^27 - 2*x1^47*x2^44*x3^29*x4^15*z^27 - 4*x1^53*x2^37*x3^30*x4^15*z^27 - 2*x1^52*x2^38*x3^30*x4^15*z^27 - x1^51*x2^39*x3^30*x4^15*z^27 + 2*x1^49*x2^41*x3^30*x4^15*z^27 + 2*x1^48*x2^42*x3^30*x4^15*z^27 + 6*x1^47*x2^43*x3^30*x4^15*z^27 + 4*x1^52*x2^37*x3^31*x4^15*z^27 + x1^51*x2^38*x3^31*x4^15*z^27 + 3*x1^50*x2^39*x3^31*x4^15*z^27 - 3*x1^49*x2^40*x3^31*x4^15*z^27 + x1^48*x2^41*x3^31*x4^15*z^27 - 5*x1^47*x2^42*x3^31*x4^15*z^27 - 2*x1^46*x2^43*x3^31*x4^15*z^27 - 2*x1^51*x2^37*x3^32*x4^15*z^27 - x1^50*x2^38*x3^32*x4^15*z^27 - x1^49*x2^39*x3^32*x4^15*z^27 + x1^48*x2^40*x3^32*x4^15*z^27 + 3*x1^47*x2^41*x3^32*x4^15*z^27 + 3*x1^46*x2^42*x3^32*x4^15*z^27 + 2*x1^50*x2^37*x3^33*x4^15*z^27 + 2*x1^49*x2^38*x3^33*x4^15*z^27 - 4*x1^46*x2^41*x3^33*x4^15*z^27 - 2*x1^49*x2^37*x3^34*x4^15*z^27 + x1^46*x2^40*x3^34*x4^15*z^27 + x1^45*x2^41*x3^34*x4^15*z^27 - x1^47*x2^38*x3^35*x4^15*z^27 + x1^46*x2^39*x3^35*x4^15*z^27 - x1^45*x2^40*x3^35*x4^15*z^27 + x1^57*x2^41*x3^21*x4^16*z^27 - x1^55*x2^43*x3^21*x4^16*z^27 - x1^54*x2^44*x3^21*x4^16*z^27 - x1^53*x2^45*x3^21*x4^16*z^27 - 3*x1^56*x2^41*x3^22*x4^16*z^27 - x1^55*x2^42*x3^22*x4^16*z^27 + 2*x1^54*x2^43*x3^22*x4^16*z^27 + 2*x1^53*x2^44*x3^22*x4^16*z^27 + x1^52*x2^45*x3^22*x4^16*z^27 + 2*x1^56*x2^40*x3^23*x4^16*z^27 + 2*x1^55*x2^41*x3^23*x4^16*z^27 - x1^54*x2^42*x3^23*x4^16*z^27 - 2*x1^53*x2^43*x3^23*x4^16*z^27 - x1^52*x2^44*x3^23*x4^16*z^27 - x1^51*x2^45*x3^23*x4^16*z^27 - 5*x1^55*x2^40*x3^24*x4^16*z^27 - x1^54*x2^41*x3^24*x4^16*z^27 + 4*x1^52*x2^43*x3^24*x4^16*z^27 + x1^51*x2^44*x3^24*x4^16*z^27 + 4*x1^50*x2^45*x3^24*x4^16*z^27 + 5*x1^55*x2^39*x3^25*x4^16*z^27 + x1^54*x2^40*x3^25*x4^16*z^27 + x1^53*x2^41*x3^25*x4^16*z^27 - 2*x1^51*x2^43*x3^25*x4^16*z^27 - 3*x1^50*x2^44*x3^25*x4^16*z^27 - 4*x1^49*x2^45*x3^25*x4^16*z^27 - 2*x1^55*x2^38*x3^26*x4^16*z^27 - 6*x1^54*x2^39*x3^26*x4^16*z^27 - 4*x1^52*x2^41*x3^26*x4^16*z^27 + 4*x1^51*x2^42*x3^26*x4^16*z^27 + 6*x1^49*x2^44*x3^26*x4^16*z^27 + x1^48*x2^45*x3^26*x4^16*z^27 + 6*x1^54*x2^38*x3^27*x4^16*z^27 + 2*x1^53*x2^39*x3^27*x4^16*z^27 + 2*x1^52*x2^40*x3^27*x4^16*z^27 - 2*x1^50*x2^42*x3^27*x4^16*z^27 - 2*x1^49*x2^43*x3^27*x4^16*z^27 - 6*x1^48*x2^44*x3^27*x4^16*z^27 - x1^54*x2^37*x3^28*x4^16*z^27 - 6*x1^53*x2^38*x3^28*x4^16*z^27 - 4*x1^51*x2^40*x3^28*x4^16*z^27 + 4*x1^50*x2^41*x3^28*x4^16*z^27 + 6*x1^48*x2^43*x3^28*x4^16*z^27 + 2*x1^47*x2^44*x3^28*x4^16*z^27 + 3*x1^53*x2^37*x3^29*x4^16*z^27 + 4*x1^52*x2^38*x3^29*x4^16*z^27 + 2*x1^51*x2^39*x3^29*x4^16*z^27 - 2*x1^49*x2^41*x3^29*x4^16*z^27 - 2*x1^48*x2^42*x3^29*x4^16*z^27 - 6*x1^47*x2^43*x3^29*x4^16*z^27 - 3*x1^52*x2^37*x3^30*x4^16*z^27 - 2*x1^51*x2^38*x3^30*x4^16*z^27 - 3*x1^50*x2^39*x3^30*x4^16*z^27 + 4*x1^49*x2^40*x3^30*x4^16*z^27 + 6*x1^47*x2^42*x3^30*x4^16*z^27 + 2*x1^46*x2^43*x3^30*x4^16*z^27 + x1^51*x2^37*x3^31*x4^16*z^27 + 2*x1^50*x2^38*x3^31*x4^16*z^27 + x1^49*x2^39*x3^31*x4^16*z^27 - 2*x1^47*x2^41*x3^31*x4^16*z^27 - 6*x1^46*x2^42*x3^31*x4^16*z^27 - 2*x1^49*x2^38*x3^32*x4^16*z^27 + 2*x1^48*x2^39*x3^32*x4^16*z^27 - 2*x1^47*x2^40*x3^32*x4^16*z^27 + 6*x1^46*x2^41*x3^32*x4^16*z^27 + 2*x1^45*x2^42*x3^32*x4^16*z^27 - x1^47*x2^39*x3^33*x4^16*z^27 - 4*x1^45*x2^41*x3^33*x4^16*z^27 + x1^47*x2^38*x3^34*x4^16*z^27 - x1^46*x2^39*x3^34*x4^16*z^27 + 2*x1^45*x2^40*x3^34*x4^16*z^27 + x1^44*x2^41*x3^34*x4^16*z^27 - x1^46*x2^38*x3^35*x4^16*z^27 + x1^53*x2^43*x3^22*x4^17*z^27 + x1^52*x2^44*x3^22*x4^17*z^27 - x1^52*x2^43*x3^23*x4^17*z^27 - x1^50*x2^45*x3^23*x4^17*z^27 - x1^54*x2^40*x3^24*x4^17*z^27 - x1^53*x2^41*x3^24*x4^17*z^27 + x1^52*x2^42*x3^24*x4^17*z^27 + x1^51*x2^43*x3^24*x4^17*z^27 + x1^50*x2^44*x3^24*x4^17*z^27 + 2*x1^49*x2^45*x3^24*x4^17*z^27 + 2*x1^54*x2^39*x3^25*x4^17*z^27 - x1^52*x2^41*x3^25*x4^17*z^27 - x1^51*x2^42*x3^25*x4^17*z^27 - 2*x1^49*x2^44*x3^25*x4^17*z^27 - x1^48*x2^45*x3^25*x4^17*z^27 - x1^54*x2^38*x3^26*x4^17*z^27 + x1^50*x2^42*x3^26*x4^17*z^27 + x1^49*x2^43*x3^26*x4^17*z^27 + 2*x1^48*x2^44*x3^26*x4^17*z^27 + x1^54*x2^37*x3^27*x4^17*z^27 + 2*x1^53*x2^38*x3^27*x4^17*z^27 + x1^51*x2^40*x3^27*x4^17*z^27 - 2*x1^50*x2^41*x3^27*x4^17*z^27 - 2*x1^48*x2^43*x3^27*x4^17*z^27 - x1^53*x2^37*x3^28*x4^17*z^27 + 2*x1^47*x2^43*x3^28*x4^17*z^27 + x1^52*x2^37*x3^29*x4^17*z^27 + 2*x1^50*x2^39*x3^29*x4^17*z^27 - x1^49*x2^40*x3^29*x4^17*z^27 - 2*x1^47*x2^42*x3^29*x4^17*z^27 - x1^46*x2^43*x3^29*x4^17*z^27 - x1^51*x2^37*x3^30*x4^17*z^27 + x1^48*x2^40*x3^30*x4^17*z^27 + x1^47*x2^41*x3^30*x4^17*z^27 + 2*x1^46*x2^42*x3^30*x4^17*z^27 - x1^49*x2^38*x3^31*x4^17*z^27 - 2*x1^48*x2^39*x3^31*x4^17*z^27 - 2*x1^46*x2^41*x3^31*x4^17*z^27 - x1^45*x2^42*x3^31*x4^17*z^27 + x1^47*x2^39*x3^32*x4^17*z^27 + 2*x1^45*x2^41*x3^32*x4^17*z^27 - x1^45*x2^40*x3^33*x4^17*z^27 + x1^44*x2^40*x3^34*x4^17*z^27 + x1^59*x2^45*x3^24*x4^2*z^26 - x1^58*x2^44*x3^25*x4^3*z^26 + x1^57*x2^44*x3^26*x4^3*z^26 - 2*x1^59*x2^43*x3^24*x4^4*z^26 - x1^58*x2^44*x3^24*x4^4*z^26 - x1^57*x2^45*x3^24*x4^4*z^26 + x1^59*x2^42*x3^25*x4^4*z^26 - x1^57*x2^44*x3^25*x4^4*z^26 + x1^56*x2^45*x3^25*x4^4*z^26 - 2*x1^58*x2^42*x3^26*x4^4*z^26 - x1^56*x2^44*x3^26*x4^4*z^26 - 2*x1^56*x2^43*x3^27*x4^4*z^26 + x1^54*x2^45*x3^27*x4^4*z^26 - x1^60*x2^43*x3^22*x4^5*z^26 + 3*x1^59*x2^43*x3^23*x4^5*z^26 - x1^58*x2^44*x3^23*x4^5*z^26 - 2*x1^59*x2^42*x3^24*x4^5*z^26 - 2*x1^58*x2^43*x3^24*x4^5*z^26 + 2*x1^57*x2^44*x3^24*x4^5*z^26 - 2*x1^56*x2^45*x3^24*x4^5*z^26 + 5*x1^58*x2^42*x3^25*x4^5*z^26 - 2*x1^57*x2^43*x3^25*x4^5*z^26 + x1^55*x2^45*x3^25*x4^5*z^26 - x1^54*x2^46*x3^25*x4^5*z^26 - 2*x1^58*x2^41*x3^26*x4^5*z^26 - 2*x1^57*x2^42*x3^26*x4^5*z^26 + 2*x1^56*x2^43*x3^26*x4^5*z^26 - 4*x1^55*x2^44*x3^26*x4^5*z^26 - x1^54*x2^45*x3^26*x4^5*z^26 - x1^53*x2^46*x3^26*x4^5*z^26 + 3*x1^57*x2^41*x3^27*x4^5*z^26 - x1^56*x2^42*x3^27*x4^5*z^26 + x1^54*x2^44*x3^27*x4^5*z^26 - x1^56*x2^41*x3^28*x4^5*z^26 + x1^55*x2^42*x3^28*x4^5*z^26 - 3*x1^54*x2^43*x3^28*x4^5*z^26 + x1^52*x2^45*x3^28*x4^5*z^26 - x1^54*x2^42*x3^29*x4^5*z^26 - x1^53*x2^43*x3^29*x4^5*z^26 - x1^52*x2^44*x3^29*x4^5*z^26 - x1^59*x2^43*x3^22*x4^6*z^26 + x1^58*x2^43*x3^23*x4^6*z^26 - x1^57*x2^44*x3^23*x4^6*z^26 - x1^58*x2^42*x3^24*x4^6*z^26 - x1^57*x2^43*x3^24*x4^6*z^26 + x1^55*x2^45*x3^24*x4^6*z^26 + x1^58*x2^41*x3^25*x4^6*z^26 + 2*x1^57*x2^42*x3^25*x4^6*z^26 - 2*x1^56*x2^43*x3^25*x4^6*z^26 + x1^55*x2^44*x3^25*x4^6*z^26 - x1^53*x2^46*x3^25*x4^6*z^26 - 2*x1^57*x2^41*x3^26*x4^6*z^26 + 4*x1^56*x2^42*x3^26*x4^6*z^26 + x1^55*x2^43*x3^26*x4^6*z^26 - 2*x1^54*x2^44*x3^26*x4^6*z^26 + x1^53*x2^45*x3^26*x4^6*z^26 + 3*x1^52*x2^46*x3^26*x4^6*z^26 + x1^57*x2^40*x3^27*x4^6*z^26 + 2*x1^56*x2^41*x3^27*x4^6*z^26 - 2*x1^55*x2^42*x3^27*x4^6*z^26 + 2*x1^54*x2^43*x3^27*x4^6*z^26 - x1^53*x2^44*x3^27*x4^6*z^26 - x1^56*x2^40*x3^28*x4^6*z^26 + x1^55*x2^41*x3^28*x4^6*z^26 + x1^54*x2^42*x3^28*x4^6*z^26 - x1^53*x2^43*x3^28*x4^6*z^26 + x1^55*x2^40*x3^29*x4^6*z^26 + x1^54*x2^41*x3^29*x4^6*z^26 + 3*x1^53*x2^42*x3^29*x4^6*z^26 - x1^51*x2^44*x3^29*x4^6*z^26 + x1^53*x2^41*x3^30*x4^6*z^26 + x1^52*x2^42*x3^30*x4^6*z^26 + x1^51*x2^43*x3^30*x4^6*z^26 + x1^58*x2^44*x3^21*x4^7*z^26 - 2*x1^57*x2^44*x3^22*x4^7*z^26 + x1^56*x2^45*x3^22*x4^7*z^26 - x1^57*x2^43*x3^23*x4^7*z^26 + x1^56*x2^44*x3^23*x4^7*z^26 - x1^55*x2^45*x3^23*x4^7*z^26 - x1^53*x2^47*x3^23*x4^7*z^26 - x1^56*x2^43*x3^24*x4^7*z^26 - x1^55*x2^44*x3^24*x4^7*z^26 - x1^54*x2^45*x3^24*x4^7*z^26 + x1^52*x2^47*x3^24*x4^7*z^26 + x1^56*x2^42*x3^25*x4^7*z^26 + x1^55*x2^43*x3^25*x4^7*z^26 - 2*x1^54*x2^44*x3^25*x4^7*z^26 - 2*x1^56*x2^41*x3^26*x4^7*z^26 + 2*x1^55*x2^42*x3^26*x4^7*z^26 + x1^54*x2^43*x3^26*x4^7*z^26 + x1^52*x2^45*x3^26*x4^7*z^26 + x1^51*x2^46*x3^26*x4^7*z^26 + x1^55*x2^41*x3^27*x4^7*z^26 + x1^53*x2^43*x3^27*x4^7*z^26 + x1^52*x2^44*x3^27*x4^7*z^26 - 2*x1^51*x2^45*x3^27*x4^7*z^26 + x1^50*x2^46*x3^27*x4^7*z^26 + x1^49*x2^47*x3^27*x4^7*z^26 - x1^56*x2^39*x3^28*x4^7*z^26 - x1^55*x2^40*x3^28*x4^7*z^26 + x1^54*x2^41*x3^28*x4^7*z^26 - x1^53*x2^42*x3^28*x4^7*z^26 + x1^52*x2^43*x3^28*x4^7*z^26 - x1^51*x2^44*x3^28*x4^7*z^26 + x1^49*x2^46*x3^28*x4^7*z^26 + x1^53*x2^41*x3^29*x4^7*z^26 + x1^52*x2^42*x3^29*x4^7*z^26 + x1^51*x2^43*x3^29*x4^7*z^26 - x1^55*x2^38*x3^30*x4^7*z^26 - x1^54*x2^39*x3^30*x4^7*z^26 - 2*x1^52*x2^41*x3^30*x4^7*z^26 + x1^51*x2^42*x3^30*x4^7*z^26 - 2*x1^58*x2^44*x3^20*x4^8*z^26 + x1^59*x2^42*x3^21*x4^8*z^26 + 3*x1^57*x2^44*x3^21*x4^8*z^26 - x1^56*x2^45*x3^21*x4^8*z^26 - x1^58*x2^42*x3^22*x4^8*z^26 - x1^57*x2^43*x3^22*x4^8*z^26 - 3*x1^56*x2^44*x3^22*x4^8*z^26 - 2*x1^54*x2^46*x3^22*x4^8*z^26 + x1^57*x2^42*x3^23*x4^8*z^26 + 5*x1^56*x2^43*x3^23*x4^8*z^26 + 3*x1^55*x2^44*x3^23*x4^8*z^26 + x1^54*x2^45*x3^23*x4^8*z^26 + x1^53*x2^46*x3^23*x4^8*z^26 - x1^52*x2^47*x3^23*x4^8*z^26 - x1^57*x2^41*x3^24*x4^8*z^26 - 2*x1^56*x2^42*x3^24*x4^8*z^26 - 5*x1^55*x2^43*x3^24*x4^8*z^26 - 2*x1^53*x2^45*x3^24*x4^8*z^26 + x1^51*x2^47*x3^24*x4^8*z^26 + x1^57*x2^40*x3^25*x4^8*z^26 + x1^56*x2^41*x3^25*x4^8*z^26 + 4*x1^55*x2^42*x3^25*x4^8*z^26 + 3*x1^54*x2^43*x3^25*x4^8*z^26 + 3*x1^53*x2^44*x3^25*x4^8*z^26 + x1^52*x2^45*x3^25*x4^8*z^26 - x1^50*x2^47*x3^25*x4^8*z^26 - x1^56*x2^40*x3^26*x4^8*z^26 - 2*x1^55*x2^41*x3^26*x4^8*z^26 - 4*x1^54*x2^42*x3^26*x4^8*z^26 - x1^53*x2^43*x3^26*x4^8*z^26 - 3*x1^52*x2^44*x3^26*x4^8*z^26 - x1^50*x2^46*x3^26*x4^8*z^26 + 2*x1^49*x2^47*x3^26*x4^8*z^26 + x1^56*x2^39*x3^27*x4^8*z^26 + 2*x1^55*x2^40*x3^27*x4^8*z^26 + 3*x1^54*x2^41*x3^27*x4^8*z^26 + x1^53*x2^42*x3^27*x4^8*z^26 + x1^52*x2^43*x3^27*x4^8*z^26 + 3*x1^51*x2^44*x3^27*x4^8*z^26 - x1^50*x2^45*x3^27*x4^8*z^26 - x1^49*x2^46*x3^27*x4^8*z^26 - x1^48*x2^47*x3^27*x4^8*z^26 - x1^55*x2^39*x3^28*x4^8*z^26 - 2*x1^54*x2^40*x3^28*x4^8*z^26 - 3*x1^53*x2^41*x3^28*x4^8*z^26 - x1^52*x2^42*x3^28*x4^8*z^26 - 5*x1^51*x2^43*x3^28*x4^8*z^26 + x1^50*x2^44*x3^28*x4^8*z^26 + x1^49*x2^45*x3^28*x4^8*z^26 + x1^54*x2^39*x3^29*x4^8*z^26 + 2*x1^53*x2^40*x3^29*x4^8*z^26 + x1^52*x2^41*x3^29*x4^8*z^26 + x1^50*x2^43*x3^29*x4^8*z^26 - 2*x1^49*x2^44*x3^29*x4^8*z^26 - x1^48*x2^45*x3^29*x4^8*z^26 - x1^54*x2^38*x3^30*x4^8*z^26 - x1^53*x2^39*x3^30*x4^8*z^26 - 2*x1^52*x2^40*x3^30*x4^8*z^26 - x1^51*x2^41*x3^30*x4^8*z^26 - 3*x1^50*x2^42*x3^30*x4^8*z^26 + x1^54*x2^37*x3^31*x4^8*z^26 + 2*x1^51*x2^40*x3^31*x4^8*z^26 + x1^49*x2^42*x3^31*x4^8*z^26 - x1^53*x2^37*x3^32*x4^8*z^26 - 2*x1^51*x2^39*x3^32*x4^8*z^26 - x1^50*x2^40*x3^32*x4^8*z^26 - x1^49*x2^41*x3^32*x4^8*z^26 - x1^58*x2^42*x3^21*x4^9*z^26 + x1^57*x2^43*x3^21*x4^9*z^26 - x1^55*x2^45*x3^21*x4^9*z^26 + x1^57*x2^42*x3^22*x4^9*z^26 - 2*x1^56*x2^43*x3^22*x4^9*z^26 + x1^55*x2^44*x3^22*x4^9*z^26 + x1^54*x2^45*x3^22*x4^9*z^26 + x1^57*x2^41*x3^23*x4^9*z^26 - x1^56*x2^42*x3^23*x4^9*z^26 + x1^55*x2^43*x3^23*x4^9*z^26 - x1^54*x2^44*x3^23*x4^9*z^26 + x1^53*x2^45*x3^23*x4^9*z^26 - x1^57*x2^40*x3^24*x4^9*z^26 - x1^56*x2^41*x3^24*x4^9*z^26 - 2*x1^54*x2^43*x3^24*x4^9*z^26 - 2*x1^53*x2^44*x3^24*x4^9*z^26 + x1^52*x2^45*x3^24*x4^9*z^26 + x1^51*x2^46*x3^24*x4^9*z^26 + 4*x1^56*x2^40*x3^25*x4^9*z^26 + 2*x1^55*x2^41*x3^25*x4^9*z^26 + 2*x1^54*x2^42*x3^25*x4^9*z^26 + x1^53*x2^43*x3^25*x4^9*z^26 + x1^52*x2^44*x3^25*x4^9*z^26 - x1^51*x2^45*x3^25*x4^9*z^26 - x1^49*x2^47*x3^25*x4^9*z^26 - 2*x1^56*x2^39*x3^26*x4^9*z^26 - 4*x1^55*x2^40*x3^26*x4^9*z^26 - 2*x1^54*x2^41*x3^26*x4^9*z^26 - x1^53*x2^42*x3^26*x4^9*z^26 + x1^50*x2^45*x3^26*x4^9*z^26 + x1^49*x2^46*x3^26*x4^9*z^26 + 3*x1^55*x2^39*x3^27*x4^9*z^26 + x1^54*x2^40*x3^27*x4^9*z^26 + 2*x1^53*x2^41*x3^27*x4^9*z^26 + x1^52*x2^42*x3^27*x4^9*z^26 + x1^51*x2^43*x3^27*x4^9*z^26 - x1^50*x2^44*x3^27*x4^9*z^26 - 2*x1^48*x2^46*x3^27*x4^9*z^26 - x1^55*x2^38*x3^28*x4^9*z^26 - 2*x1^54*x2^39*x3^28*x4^9*z^26 - 2*x1^53*x2^40*x3^28*x4^9*z^26 - 3*x1^52*x2^41*x3^28*x4^9*z^26 + x1^51*x2^42*x3^28*x4^9*z^26 - x1^50*x2^43*x3^28*x4^9*z^26 + x1^49*x2^44*x3^28*x4^9*z^26 + 3*x1^54*x2^38*x3^29*x4^9*z^26 + x1^53*x2^39*x3^29*x4^9*z^26 + x1^52*x2^40*x3^29*x4^9*z^26 + 2*x1^51*x2^41*x3^29*x4^9*z^26 + 2*x1^50*x2^42*x3^29*x4^9*z^26 + x1^49*x2^43*x3^29*x4^9*z^26 - x1^47*x2^45*x3^29*x4^9*z^26 - x1^54*x2^37*x3^30*x4^9*z^26 - x1^53*x2^38*x3^30*x4^9*z^26 - 3*x1^51*x2^40*x3^30*x4^9*z^26 + 2*x1^48*x2^43*x3^30*x4^9*z^26 + x1^47*x2^44*x3^30*x4^9*z^26 + 2*x1^53*x2^37*x3^31*x4^9*z^26 + x1^51*x2^39*x3^31*x4^9*z^26 + x1^50*x2^40*x3^31*x4^9*z^26 - x1^49*x2^41*x3^31*x4^9*z^26 + x1^51*x2^38*x3^32*x4^9*z^26 - x1^48*x2^41*x3^32*x4^9*z^26 - x1^51*x2^37*x3^33*x4^9*z^26 + x1^49*x2^39*x3^33*x4^9*z^26 + x1^48*x2^40*x3^33*x4^9*z^26 - x1^59*x2^42*x3^19*x4^10*z^26 + 2*x1^58*x2^42*x3^20*x4^10*z^26 + x1^56*x2^44*x3^20*x4^10*z^26 - 2*x1^57*x2^42*x3^21*x4^10*z^26 + x1^54*x2^45*x3^21*x4^10*z^26 + 2*x1^57*x2^41*x3^22*x4^10*z^26 + x1^56*x2^42*x3^22*x4^10*z^26 + x1^55*x2^43*x3^22*x4^10*z^26 + x1^52*x2^46*x3^22*x4^10*z^26 - 2*x1^56*x2^41*x3^23*x4^10*z^26 - 2*x1^54*x2^43*x3^23*x4^10*z^26 + x1^52*x2^45*x3^23*x4^10*z^26 + x1^51*x2^46*x3^23*x4^10*z^26 - x1^52*x2^44*x3^24*x4^10*z^26 - 2*x1^51*x2^45*x3^24*x4^10*z^26 - x1^50*x2^46*x3^24*x4^10*z^26 + x1^49*x2^47*x3^24*x4^10*z^26 + x1^52*x2^43*x3^25*x4^10*z^26 - x1^51*x2^44*x3^25*x4^10*z^26 + x1^50*x2^45*x3^25*x4^10*z^26 - x1^48*x2^47*x3^25*x4^10*z^26 - 2*x1^52*x2^42*x3^26*x4^10*z^26 - 2*x1^50*x2^44*x3^26*x4^10*z^26 - 3*x1^49*x2^45*x3^26*x4^10*z^26 + x1^50*x2^43*x3^27*x4^10*z^26 + 2*x1^49*x2^44*x3^27*x4^10*z^26 + x1^48*x2^45*x3^27*x4^10*z^26 + 2*x1^47*x2^46*x3^27*x4^10*z^26 - 2*x1^49*x2^43*x3^28*x4^10*z^26 - x1^48*x2^44*x3^28*x4^10*z^26 - x1^46*x2^46*x3^28*x4^10*z^26 - x1^53*x2^38*x3^29*x4^10*z^26 - x1^50*x2^41*x3^29*x4^10*z^26 + x1^49*x2^42*x3^29*x4^10*z^26 + x1^47*x2^44*x3^29*x4^10*z^26 + x1^46*x2^45*x3^29*x4^10*z^26 - 2*x1^53*x2^37*x3^30*x4^10*z^26 - x1^52*x2^38*x3^30*x4^10*z^26 + x1^51*x2^39*x3^30*x4^10*z^26 - x1^50*x2^40*x3^30*x4^10*z^26 + x1^49*x2^41*x3^30*x4^10*z^26 - x1^48*x2^42*x3^30*x4^10*z^26 + x1^53*x2^36*x3^31*x4^10*z^26 + x1^52*x2^37*x3^31*x4^10*z^26 - 2*x1^51*x2^38*x3^31*x4^10*z^26 + x1^49*x2^40*x3^31*x4^10*z^26 - x1^47*x2^42*x3^31*x4^10*z^26 + x1^46*x2^43*x3^31*x4^10*z^26 + x1^45*x2^44*x3^31*x4^10*z^26 - 2*x1^52*x2^36*x3^32*x4^10*z^26 + 2*x1^48*x2^40*x3^32*x4^10*z^26 + x1^51*x2^36*x3^33*x4^10*z^26 - x1^48*x2^38*x3^34*x4^10*z^26 + x1^47*x2^39*x3^34*x4^10*z^26 - x1^58*x2^42*x3^19*x4^11*z^26 + x1^57*x2^43*x3^19*x4^11*z^26 + x1^58*x2^41*x3^20*x4^11*z^26 + 2*x1^57*x2^42*x3^20*x4^11*z^26 - x1^56*x2^43*x3^20*x4^11*z^26 - 4*x1^57*x2^41*x3^21*x4^11*z^26 + x1^56*x2^42*x3^21*x4^11*z^26 + x1^53*x2^45*x3^21*x4^11*z^26 + x1^57*x2^40*x3^22*x4^11*z^26 + 4*x1^56*x2^41*x3^22*x4^11*z^26 - 3*x1^55*x2^42*x3^22*x4^11*z^26 + 2*x1^54*x2^43*x3^22*x4^11*z^26 - x1^53*x2^44*x3^22*x4^11*z^26 - 2*x1^52*x2^45*x3^22*x4^11*z^26 - x1^51*x2^46*x3^22*x4^11*z^26 - 4*x1^56*x2^40*x3^23*x4^11*z^26 - 2*x1^55*x2^41*x3^23*x4^11*z^26 + x1^54*x2^42*x3^23*x4^11*z^26 + x1^52*x2^44*x3^23*x4^11*z^26 + x1^51*x2^45*x3^23*x4^11*z^26 + x1^50*x2^46*x3^23*x4^11*z^26 + 2*x1^56*x2^39*x3^24*x4^11*z^26 + 4*x1^55*x2^40*x3^24*x4^11*z^26 - 2*x1^54*x2^41*x3^24*x4^11*z^26 + x1^53*x2^42*x3^24*x4^11*z^26 - 3*x1^52*x2^43*x3^24*x4^11*z^26 - 3*x1^50*x2^45*x3^24*x4^11*z^26 - x1^49*x2^46*x3^24*x4^11*z^26 - 4*x1^55*x2^39*x3^25*x4^11*z^26 + x1^53*x2^41*x3^25*x4^11*z^26 + 2*x1^51*x2^43*x3^25*x4^11*z^26 + 2*x1^50*x2^44*x3^25*x4^11*z^26 + 3*x1^49*x2^45*x3^25*x4^11*z^26 + x1^55*x2^38*x3^26*x4^11*z^26 + 4*x1^54*x2^39*x3^26*x4^11*z^26 - 2*x1^53*x2^40*x3^26*x4^11*z^26 + 2*x1^52*x2^41*x3^26*x4^11*z^26 - 3*x1^51*x2^42*x3^26*x4^11*z^26 - x1^50*x2^43*x3^26*x4^11*z^26 - 2*x1^49*x2^44*x3^26*x4^11*z^26 + x1^48*x2^45*x3^26*x4^11*z^26 - 4*x1^54*x2^38*x3^27*x4^11*z^26 + x1^52*x2^40*x3^27*x4^11*z^26 + x1^51*x2^41*x3^27*x4^11*z^26 + x1^50*x2^42*x3^27*x4^11*z^26 - x1^49*x2^43*x3^27*x4^11*z^26 + 4*x1^48*x2^44*x3^27*x4^11*z^26 - 2*x1^47*x2^45*x3^27*x4^11*z^26 + x1^46*x2^46*x3^27*x4^11*z^26 + 4*x1^53*x2^38*x3^28*x4^11*z^26 - 2*x1^52*x2^39*x3^28*x4^11*z^26 + 3*x1^51*x2^40*x3^28*x4^11*z^26 - 2*x1^49*x2^42*x3^28*x4^11*z^26 - 5*x1^48*x2^43*x3^28*x4^11*z^26 - x1^47*x2^44*x3^28*x4^11*z^26 + x1^46*x2^45*x3^28*x4^11*z^26 - 3*x1^52*x2^38*x3^29*x4^11*z^26 + x1^50*x2^40*x3^29*x4^11*z^26 + 2*x1^49*x2^41*x3^29*x4^11*z^26 + 2*x1^48*x2^42*x3^29*x4^11*z^26 + 2*x1^47*x2^43*x3^29*x4^11*z^26 - x1^46*x2^44*x3^29*x4^11*z^26 + x1^45*x2^45*x3^29*x4^11*z^26 + x1^52*x2^37*x3^30*x4^11*z^26 - x1^51*x2^38*x3^30*x4^11*z^26 - 4*x1^47*x2^42*x3^30*x4^11*z^26 - x1^46*x2^43*x3^30*x4^11*z^26 + x1^45*x2^44*x3^30*x4^11*z^26 + x1^52*x2^36*x3^31*x4^11*z^26 - x1^50*x2^38*x3^31*x4^11*z^26 + 2*x1^47*x2^41*x3^31*x4^11*z^26 - x1^45*x2^43*x3^31*x4^11*z^26 - x1^51*x2^36*x3^32*x4^11*z^26 - 2*x1^50*x2^37*x3^32*x4^11*z^26 - x1^49*x2^38*x3^32*x4^11*z^26 - 3*x1^48*x2^39*x3^32*x4^11*z^26 - 2*x1^47*x2^40*x3^32*x4^11*z^26 - x1^45*x2^42*x3^32*x4^11*z^26 - x1^44*x2^43*x3^32*x4^11*z^26 + x1^47*x2^39*x3^33*x4^11*z^26 - x1^58*x2^41*x3^19*x4^12*z^26 - x1^57*x2^42*x3^19*x4^12*z^26 + 2*x1^56*x2^43*x3^19*x4^12*z^26 + 2*x1^57*x2^41*x3^20*x4^12*z^26 - 2*x1^55*x2^43*x3^20*x4^12*z^26 + x1^54*x2^44*x3^20*x4^12*z^26 - 2*x1^56*x2^41*x3^21*x4^12*z^26 + 3*x1^55*x2^42*x3^21*x4^12*z^26 + x1^54*x2^43*x3^21*x4^12*z^26 + x1^53*x2^44*x3^21*x4^12*z^26 + 2*x1^56*x2^40*x3^22*x4^12*z^26 - x1^55*x2^41*x3^22*x4^12*z^26 - 2*x1^54*x2^42*x3^22*x4^12*z^26 + x1^53*x2^43*x3^22*x4^12*z^26 - 2*x1^52*x2^44*x3^22*x4^12*z^26 - x1^56*x2^39*x3^23*x4^12*z^26 - 2*x1^55*x2^40*x3^23*x4^12*z^26 + 6*x1^54*x2^41*x3^23*x4^12*z^26 - 2*x1^53*x2^42*x3^23*x4^12*z^26 + x1^52*x2^43*x3^23*x4^12*z^26 + x1^51*x2^44*x3^23*x4^12*z^26 + x1^50*x2^45*x3^23*x4^12*z^26 - x1^49*x2^46*x3^23*x4^12*z^26 + 2*x1^55*x2^39*x3^24*x4^12*z^26 - x1^54*x2^40*x3^24*x4^12*z^26 - 5*x1^53*x2^41*x3^24*x4^12*z^26 - 5*x1^51*x2^43*x3^24*x4^12*z^26 - x1^50*x2^44*x3^24*x4^12*z^26 - x1^49*x2^45*x3^24*x4^12*z^26 + 2*x1^48*x2^46*x3^24*x4^12*z^26 - x1^55*x2^38*x3^25*x4^12*z^26 - 2*x1^54*x2^39*x3^25*x4^12*z^26 + 6*x1^53*x2^40*x3^25*x4^12*z^26 + x1^52*x2^41*x3^25*x4^12*z^26 + 4*x1^51*x2^42*x3^25*x4^12*z^26 - x1^50*x2^43*x3^25*x4^12*z^26 - 2*x1^48*x2^45*x3^25*x4^12*z^26 - 2*x1^47*x2^46*x3^25*x4^12*z^26 + 2*x1^54*x2^38*x3^26*x4^12*z^26 - 2*x1^53*x2^39*x3^26*x4^12*z^26 - 6*x1^52*x2^40*x3^26*x4^12*z^26 - 4*x1^50*x2^42*x3^26*x4^12*z^26 + 4*x1^49*x2^43*x3^26*x4^12*z^26 - x1^48*x2^44*x3^26*x4^12*z^26 + 5*x1^47*x2^45*x3^26*x4^12*z^26 + x1^46*x2^46*x3^26*x4^12*z^26 - 2*x1^53*x2^38*x3^27*x4^12*z^26 + 6*x1^52*x2^39*x3^27*x4^12*z^26 + 2*x1^50*x2^41*x3^27*x4^12*z^26 + x1^49*x2^42*x3^27*x4^12*z^26 - x1^48*x2^43*x3^27*x4^12*z^26 - 2*x1^47*x2^44*x3^27*x4^12*z^26 - 5*x1^46*x2^45*x3^27*x4^12*z^26 + x1^53*x2^37*x3^28*x4^12*z^26 - x1^52*x2^38*x3^28*x4^12*z^26 - 5*x1^51*x2^39*x3^28*x4^12*z^26 - 3*x1^49*x2^41*x3^28*x4^12*z^26 + 2*x1^48*x2^42*x3^28*x4^12*z^26 - x1^47*x2^43*x3^28*x4^12*z^26 + 5*x1^46*x2^44*x3^28*x4^12*z^26 + x1^45*x2^45*x3^28*x4^12*z^26 - x1^53*x2^36*x3^29*x4^12*z^26 - x1^52*x2^37*x3^29*x4^12*z^26 + 5*x1^51*x2^38*x3^29*x4^12*z^26 + x1^50*x2^39*x3^29*x4^12*z^26 + 2*x1^49*x2^40*x3^29*x4^12*z^26 + 2*x1^48*x2^41*x3^29*x4^12*z^26 + x1^47*x2^42*x3^29*x4^12*z^26 - 2*x1^46*x2^43*x3^29*x4^12*z^26 - 4*x1^45*x2^44*x3^29*x4^12*z^26 + x1^52*x2^36*x3^30*x4^12*z^26 - x1^51*x2^37*x3^30*x4^12*z^26 - 5*x1^50*x2^38*x3^30*x4^12*z^26 - x1^49*x2^39*x3^30*x4^12*z^26 - 4*x1^48*x2^40*x3^30*x4^12*z^26 - 2*x1^46*x2^42*x3^30*x4^12*z^26 + 4*x1^45*x2^43*x3^30*x4^12*z^26 + 3*x1^50*x2^37*x3^31*x4^12*z^26 + x1^49*x2^38*x3^31*x4^12*z^26 + 3*x1^48*x2^39*x3^31*x4^12*z^26 + x1^47*x2^40*x3^31*x4^12*z^26 - x1^45*x2^42*x3^31*x4^12*z^26 - 2*x1^44*x2^43*x3^31*x4^12*z^26 + x1^50*x2^36*x3^32*x4^12*z^26 - x1^49*x2^37*x3^32*x4^12*z^26 - x1^47*x2^39*x3^32*x4^12*z^26 + 2*x1^46*x2^40*x3^32*x4^12*z^26 + x1^44*x2^42*x3^32*x4^12*z^26 - x1^48*x2^37*x3^33*x4^12*z^26 + 3*x1^46*x2^39*x3^33*x4^12*z^26 - x1^46*x2^38*x3^34*x4^12*z^26 - x1^55*x2^43*x3^19*x4^13*z^26 + x1^57*x2^40*x3^20*x4^13*z^26 + x1^56*x2^41*x3^20*x4^13*z^26 - 2*x1^55*x2^42*x3^20*x4^13*z^26 - x1^53*x2^44*x3^20*x4^13*z^26 - 2*x1^56*x2^40*x3^21*x4^13*z^26 + 3*x1^54*x2^42*x3^21*x4^13*z^26 - 2*x1^53*x2^43*x3^21*x4^13*z^26 - x1^51*x2^45*x3^21*x4^13*z^26 + x1^56*x2^39*x3^22*x4^13*z^26 - 6*x1^54*x2^41*x3^22*x4^13*z^26 + x1^51*x2^44*x3^22*x4^13*z^26 + x1^50*x2^45*x3^22*x4^13*z^26 + 2*x1^54*x2^40*x3^23*x4^13*z^26 + 7*x1^53*x2^41*x3^23*x4^13*z^26 + 3*x1^51*x2^43*x3^23*x4^13*z^26 - 2*x1^50*x2^44*x3^23*x4^13*z^26 - x1^49*x2^45*x3^23*x4^13*z^26 - x1^48*x2^46*x3^23*x4^13*z^26 - 6*x1^53*x2^40*x3^24*x4^13*z^26 - 2*x1^52*x2^41*x3^24*x4^13*z^26 - x1^51*x2^42*x3^24*x4^13*z^26 + 2*x1^49*x2^44*x3^24*x4^13*z^26 + 2*x1^48*x2^45*x3^24*x4^13*z^26 + x1^47*x2^46*x3^24*x4^13*z^26 + 2*x1^53*x2^39*x3^25*x4^13*z^26 + 6*x1^52*x2^40*x3^25*x4^13*z^26 + 4*x1^50*x2^42*x3^25*x4^13*z^26 - 2*x1^49*x2^43*x3^25*x4^13*z^26 - x1^48*x2^44*x3^25*x4^13*z^26 - 4*x1^47*x2^45*x3^25*x4^13*z^26 - 6*x1^52*x2^39*x3^26*x4^13*z^26 - 2*x1^51*x2^40*x3^26*x4^13*z^26 - 2*x1^50*x2^41*x3^26*x4^13*z^26 + 2*x1^48*x2^43*x3^26*x4^13*z^26 + 3*x1^47*x2^44*x3^26*x4^13*z^26 + 4*x1^46*x2^45*x3^26*x4^13*z^26 + 2*x1^52*x2^38*x3^27*x4^13*z^26 + 6*x1^51*x2^39*x3^27*x4^13*z^26 + 4*x1^49*x2^41*x3^27*x4^13*z^26 - 4*x1^48*x2^42*x3^27*x4^13*z^26 - 6*x1^46*x2^44*x3^27*x4^13*z^26 - x1^45*x2^45*x3^27*x4^13*z^26 - 6*x1^51*x2^38*x3^28*x4^13*z^26 - 2*x1^50*x2^39*x3^28*x4^13*z^26 - 2*x1^49*x2^40*x3^28*x4^13*z^26 + 2*x1^47*x2^42*x3^28*x4^13*z^26 + 2*x1^46*x2^43*x3^28*x4^13*z^26 + 6*x1^45*x2^44*x3^28*x4^13*z^26 - x1^51*x2^37*x3^29*x4^13*z^26 + 5*x1^50*x2^38*x3^29*x4^13*z^26 - x1^49*x2^39*x3^29*x4^13*z^26 + 3*x1^48*x2^40*x3^29*x4^13*z^26 - 3*x1^47*x2^41*x3^29*x4^13*z^26 - x1^46*x2^42*x3^29*x4^13*z^26 - 6*x1^45*x2^43*x3^29*x4^13*z^26 - 2*x1^44*x2^44*x3^29*x4^13*z^26 - x1^50*x2^37*x3^30*x4^13*z^26 - 2*x1^49*x2^38*x3^30*x4^13*z^26 - x1^48*x2^39*x3^30*x4^13*z^26 - x1^47*x2^40*x3^30*x4^13*z^26 + 3*x1^46*x2^41*x3^30*x4^13*z^26 + 3*x1^45*x2^42*x3^30*x4^13*z^26 + 5*x1^44*x2^43*x3^30*x4^13*z^26 + x1^49*x2^37*x3^31*x4^13*z^26 + x1^47*x2^39*x3^31*x4^13*z^26 - x1^45*x2^41*x3^31*x4^13*z^26 - 4*x1^44*x2^42*x3^31*x4^13*z^26 - x1^43*x2^43*x3^31*x4^13*z^26 + x1^49*x2^36*x3^32*x4^13*z^26 + 2*x1^48*x2^37*x3^32*x4^13*z^26 - x1^47*x2^38*x3^32*x4^13*z^26 - x1^46*x2^39*x3^32*x4^13*z^26 + x1^45*x2^40*x3^32*x4^13*z^26 + 2*x1^44*x2^41*x3^32*x4^13*z^26 + 2*x1^43*x2^42*x3^32*x4^13*z^26 + x1^47*x2^37*x3^33*x4^13*z^26 - 2*x1^43*x2^41*x3^33*x4^13*z^26 + x1^46*x2^37*x3^34*x4^13*z^26 - x1^45*x2^38*x3^34*x4^13*z^26 + x1^43*x2^40*x3^34*x4^13*z^26 + x1^42*x2^41*x3^34*x4^13*z^26 + 2*x1^56*x2^40*x3^20*x4^14*z^26 + x1^55*x2^41*x3^20*x4^14*z^26 - x1^54*x2^42*x3^20*x4^14*z^26 + x1^53*x2^43*x3^20*x4^14*z^26 - 2*x1^55*x2^40*x3^21*x4^14*z^26 - x1^54*x2^41*x3^21*x4^14*z^26 + x1^53*x2^42*x3^21*x4^14*z^26 - x1^50*x2^45*x3^21*x4^14*z^26 + 3*x1^55*x2^39*x3^22*x4^14*z^26 + x1^54*x2^40*x3^22*x4^14*z^26 - 2*x1^53*x2^41*x3^22*x4^14*z^26 - x1^52*x2^42*x3^22*x4^14*z^26 + x1^50*x2^44*x3^22*x4^14*z^26 + x1^49*x2^45*x3^22*x4^14*z^26 - x1^55*x2^38*x3^23*x4^14*z^26 - 2*x1^54*x2^39*x3^23*x4^14*z^26 + x1^53*x2^40*x3^23*x4^14*z^26 - 3*x1^52*x2^41*x3^23*x4^14*z^26 - x1^51*x2^42*x3^23*x4^14*z^26 - x1^48*x2^45*x3^23*x4^14*z^26 + 2*x1^54*x2^38*x3^24*x4^14*z^26 + x1^53*x2^39*x3^24*x4^14*z^26 - x1^52*x2^40*x3^24*x4^14*z^26 - 2*x1^50*x2^42*x3^24*x4^14*z^26 + 2*x1^47*x2^45*x3^24*x4^14*z^26 - x1^54*x2^37*x3^25*x4^14*z^26 - 2*x1^53*x2^38*x3^25*x4^14*z^26 + 2*x1^52*x2^39*x3^25*x4^14*z^26 + 2*x1^50*x2^41*x3^25*x4^14*z^26 + x1^48*x2^43*x3^25*x4^14*z^26 - 2*x1^46*x2^45*x3^25*x4^14*z^26 + 2*x1^53*x2^37*x3^26*x4^14*z^26 - x1^51*x2^39*x3^26*x4^14*z^26 - 2*x1^49*x2^41*x3^26*x4^14*z^26 - 2*x1^47*x2^43*x3^26*x4^14*z^26 + 2*x1^46*x2^44*x3^26*x4^14*z^26 + x1^45*x2^45*x3^26*x4^14*z^26 - 2*x1^52*x2^37*x3^27*x4^14*z^26 + 2*x1^51*x2^38*x3^27*x4^14*z^26 + 3*x1^49*x2^40*x3^27*x4^14*z^26 + x1^47*x2^42*x3^27*x4^14*z^26 - x1^46*x2^43*x3^27*x4^14*z^26 - 2*x1^45*x2^44*x3^27*x4^14*z^26 + x1^52*x2^36*x3^28*x4^14*z^26 - 2*x1^50*x2^38*x3^28*x4^14*z^26 - x1^48*x2^40*x3^28*x4^14*z^26 + 2*x1^47*x2^41*x3^28*x4^14*z^26 - 2*x1^46*x2^42*x3^28*x4^14*z^26 + 2*x1^45*x2^43*x3^28*x4^14*z^26 - x1^51*x2^36*x3^29*x4^14*z^26 + 2*x1^50*x2^37*x3^29*x4^14*z^26 + x1^48*x2^39*x3^29*x4^14*z^26 + 2*x1^46*x2^41*x3^29*x4^14*z^26 + x1^45*x2^42*x3^29*x4^14*z^26 - 2*x1^44*x2^43*x3^29*x4^14*z^26 + x1^51*x2^35*x3^30*x4^14*z^26 + x1^50*x2^36*x3^30*x4^14*z^26 - x1^49*x2^37*x3^30*x4^14*z^26 - 2*x1^47*x2^39*x3^30*x4^14*z^26 + 2*x1^44*x2^42*x3^30*x4^14*z^26 + x1^43*x2^43*x3^30*x4^14*z^26 - x1^50*x2^35*x3^31*x4^14*z^26 + x1^48*x2^37*x3^31*x4^14*z^26 + 2*x1^47*x2^38*x3^31*x4^14*z^26 - x1^46*x2^39*x3^31*x4^14*z^26 + x1^45*x2^40*x3^31*x4^14*z^26 - 2*x1^43*x2^42*x3^31*x4^14*z^26 - x1^47*x2^37*x3^32*x4^14*z^26 - x1^46*x2^38*x3^32*x4^14*z^26 - x1^45*x2^39*x3^32*x4^14*z^26 + 2*x1^43*x2^41*x3^32*x4^14*z^26 + x1^42*x2^42*x3^32*x4^14*z^26 - x1^47*x2^36*x3^33*x4^14*z^26 + x1^46*x2^37*x3^33*x4^14*z^26 + x1^45*x2^38*x3^33*x4^14*z^26 - x1^44*x2^39*x3^33*x4^14*z^26 - x1^56*x2^40*x3^19*x4^15*z^26 + 2*x1^55*x2^40*x3^20*x4^15*z^26 - 2*x1^55*x2^39*x3^21*x4^15*z^26 - x1^54*x2^40*x3^21*x4^15*z^26 + x1^52*x2^42*x3^21*x4^15*z^26 - x1^51*x2^43*x3^21*x4^15*z^26 + 5*x1^54*x2^39*x3^22*x4^15*z^26 + x1^53*x2^40*x3^22*x4^15*z^26 + x1^52*x2^41*x3^22*x4^15*z^26 - 2*x1^51*x2^42*x3^22*x4^15*z^26 - x1^49*x2^44*x3^22*x4^15*z^26 - 5*x1^54*x2^38*x3^23*x4^15*z^26 - x1^53*x2^39*x3^23*x4^15*z^26 - x1^52*x2^40*x3^23*x4^15*z^26 + x1^50*x2^42*x3^23*x4^15*z^26 + x1^49*x2^43*x3^23*x4^15*z^26 + x1^48*x2^44*x3^23*x4^15*z^26 + 2*x1^54*x2^37*x3^24*x4^15*z^26 + 6*x1^53*x2^38*x3^24*x4^15*z^26 + 4*x1^51*x2^40*x3^24*x4^15*z^26 - 4*x1^50*x2^41*x3^24*x4^15*z^26 - x1^49*x2^42*x3^24*x4^15*z^26 - 4*x1^48*x2^43*x3^24*x4^15*z^26 - x1^47*x2^44*x3^24*x4^15*z^26 - 6*x1^53*x2^37*x3^25*x4^15*z^26 - 2*x1^52*x2^38*x3^25*x4^15*z^26 - 2*x1^51*x2^39*x3^25*x4^15*z^26 + 2*x1^49*x2^41*x3^25*x4^15*z^26 + 2*x1^48*x2^42*x3^25*x4^15*z^26 + 4*x1^47*x2^43*x3^25*x4^15*z^26 + 2*x1^53*x2^36*x3^26*x4^15*z^26 + 6*x1^52*x2^37*x3^26*x4^15*z^26 + 4*x1^50*x2^39*x3^26*x4^15*z^26 - 4*x1^49*x2^40*x3^26*x4^15*z^26 - 6*x1^47*x2^42*x3^26*x4^15*z^26 - 2*x1^46*x2^43*x3^26*x4^15*z^26 - 6*x1^52*x2^36*x3^27*x4^15*z^26 - 2*x1^51*x2^37*x3^27*x4^15*z^26 - 2*x1^50*x2^38*x3^27*x4^15*z^26 + 2*x1^48*x2^40*x3^27*x4^15*z^26 + 2*x1^47*x2^41*x3^27*x4^15*z^26 + 6*x1^46*x2^42*x3^27*x4^15*z^26 + x1^52*x2^35*x3^28*x4^15*z^26 + 6*x1^51*x2^36*x3^28*x4^15*z^26 + x1^50*x2^37*x3^28*x4^15*z^26 + 4*x1^49*x2^38*x3^28*x4^15*z^26 - 4*x1^48*x2^39*x3^28*x4^15*z^26 - 6*x1^46*x2^41*x3^28*x4^15*z^26 - 2*x1^45*x2^42*x3^28*x4^15*z^26 - 3*x1^51*x2^35*x3^29*x4^15*z^26 - 3*x1^50*x2^36*x3^29*x4^15*z^26 - 3*x1^49*x2^37*x3^29*x4^15*z^26 + 2*x1^47*x2^39*x3^29*x4^15*z^26 + 2*x1^46*x2^40*x3^29*x4^15*z^26 + 6*x1^45*x2^41*x3^29*x4^15*z^26 + 3*x1^50*x2^35*x3^30*x4^15*z^26 + 2*x1^49*x2^36*x3^30*x4^15*z^26 + 3*x1^48*x2^37*x3^30*x4^15*z^26 - 4*x1^47*x2^38*x3^30*x4^15*z^26 - 6*x1^45*x2^40*x3^30*x4^15*z^26 - 2*x1^44*x2^41*x3^30*x4^15*z^26 - x1^49*x2^35*x3^31*x4^15*z^26 - 2*x1^48*x2^36*x3^31*x4^15*z^26 + x1^46*x2^38*x3^31*x4^15*z^26 + 2*x1^45*x2^39*x3^31*x4^15*z^26 + 4*x1^44*x2^40*x3^31*x4^15*z^26 + x1^47*x2^36*x3^32*x4^15*z^26 - x1^46*x2^37*x3^32*x4^15*z^26 + 2*x1^45*x2^38*x3^32*x4^15*z^26 - 4*x1^44*x2^39*x3^32*x4^15*z^26 - x1^43*x2^40*x3^32*x4^15*z^26 - x1^45*x2^37*x3^33*x4^15*z^26 + x1^44*x2^38*x3^33*x4^15*z^26 + 2*x1^43*x2^39*x3^33*x4^15*z^26 + x1^44*x2^37*x3^34*x4^15*z^26 - x1^43*x2^38*x3^34*x4^15*z^26 - 2*x1^52*x2^42*x3^20*x4^16*z^26 - x1^51*x2^43*x3^20*x4^16*z^26 - 2*x1^54*x2^39*x3^21*x4^16*z^26 + x1^52*x2^41*x3^21*x4^16*z^26 + 2*x1^51*x2^42*x3^21*x4^16*z^26 + 2*x1^50*x2^43*x3^21*x4^16*z^26 + x1^49*x2^44*x3^21*x4^16*z^26 + x1^54*x2^38*x3^22*x4^16*z^26 + x1^53*x2^39*x3^22*x4^16*z^26 + x1^52*x2^40*x3^22*x4^16*z^26 - 3*x1^51*x2^41*x3^22*x4^16*z^26 - 3*x1^50*x2^42*x3^22*x4^16*z^26 - 2*x1^49*x2^43*x3^22*x4^16*z^26 - x1^48*x2^44*x3^22*x4^16*z^26 - 4*x1^53*x2^38*x3^23*x4^16*z^26 + 4*x1^50*x2^41*x3^23*x4^16*z^26 + 2*x1^49*x2^42*x3^23*x4^16*z^26 + 5*x1^48*x2^43*x3^23*x4^16*z^26 + 3*x1^53*x2^37*x3^24*x4^16*z^26 + x1^52*x2^38*x3^24*x4^16*z^26 + x1^51*x2^39*x3^24*x4^16*z^26 - x1^50*x2^40*x3^24*x4^16*z^26 - 2*x1^49*x2^41*x3^24*x4^16*z^26 - 3*x1^48*x2^42*x3^24*x4^16*z^26 - 5*x1^47*x2^43*x3^24*x4^16*z^26 - x1^53*x2^36*x3^25*x4^16*z^26 - 6*x1^52*x2^37*x3^25*x4^16*z^26 - 2*x1^50*x2^39*x3^25*x4^16*z^26 + 4*x1^49*x2^40*x3^25*x4^16*z^26 + 6*x1^47*x2^42*x3^25*x4^16*z^26 + 2*x1^46*x2^43*x3^25*x4^16*z^26 + 6*x1^52*x2^36*x3^26*x4^16*z^26 + 2*x1^51*x2^37*x3^26*x4^16*z^26 + 2*x1^50*x2^38*x3^26*x4^16*z^26 - 2*x1^48*x2^40*x3^26*x4^16*z^26 - 2*x1^47*x2^41*x3^26*x4^16*z^26 - 6*x1^46*x2^42*x3^26*x4^16*z^26 - 6*x1^51*x2^36*x3^27*x4^16*z^26 - 4*x1^49*x2^38*x3^27*x4^16*z^26 + 4*x1^48*x2^39*x3^27*x4^16*z^26 + 6*x1^46*x2^41*x3^27*x4^16*z^26 + 2*x1^45*x2^42*x3^27*x4^16*z^26 + 2*x1^51*x2^35*x3^28*x4^16*z^26 + 2*x1^50*x2^36*x3^28*x4^16*z^26 + x1^49*x2^37*x3^28*x4^16*z^26 - 2*x1^47*x2^39*x3^28*x4^16*z^26 - 2*x1^46*x2^40*x3^28*x4^16*z^26 - 6*x1^45*x2^41*x3^28*x4^16*z^26 - 2*x1^50*x2^35*x3^29*x4^16*z^26 - x1^49*x2^36*x3^29*x4^16*z^26 - 3*x1^48*x2^37*x3^29*x4^16*z^26 + 2*x1^47*x2^38*x3^29*x4^16*z^26 + 6*x1^45*x2^40*x3^29*x4^16*z^26 + 2*x1^44*x2^41*x3^29*x4^16*z^26 + x1^48*x2^36*x3^30*x4^16*z^26 - 2*x1^45*x2^39*x3^30*x4^16*z^26 - 6*x1^44*x2^40*x3^30*x4^16*z^26 + 2*x1^46*x2^37*x3^31*x4^16*z^26 - 2*x1^45*x2^38*x3^31*x4^16*z^26 + 4*x1^44*x2^39*x3^31*x4^16*z^26 + 2*x1^43*x2^40*x3^31*x4^16*z^26 - x1^46*x2^36*x3^32*x4^16*z^26 + x1^44*x2^38*x3^32*x4^16*z^26 - 4*x1^43*x2^39*x3^32*x4^16*z^26 + x1^45*x2^36*x3^33*x4^16*z^26 + x1^43*x2^38*x3^33*x4^16*z^26 + x1^42*x2^39*x3^33*x4^16*z^26 - x1^42*x2^38*x3^34*x4^16*z^26 + x1^51*x2^41*x3^21*x4^17*z^26 - x1^50*x2^41*x3^22*x4^17*z^26 - x1^49*x2^42*x3^22*x4^17*z^26 - x1^48*x2^43*x3^22*x4^17*z^26 + x1^50*x2^40*x3^23*x4^17*z^26 + x1^49*x2^41*x3^23*x4^17*z^26 - x1^48*x2^42*x3^23*x4^17*z^26 + x1^47*x2^43*x3^23*x4^17*z^26 + x1^52*x2^37*x3^24*x4^17*z^26 - x1^51*x2^38*x3^24*x4^17*z^26 + x1^50*x2^39*x3^24*x4^17*z^26 - x1^49*x2^40*x3^24*x4^17*z^26 - 2*x1^47*x2^42*x3^24*x4^17*z^26 - x1^46*x2^43*x3^24*x4^17*z^26 - x1^51*x2^37*x3^25*x4^17*z^26 - x1^50*x2^38*x3^25*x4^17*z^26 + x1^49*x2^39*x3^25*x4^17*z^26 + x1^48*x2^40*x3^25*x4^17*z^26 + x1^47*x2^41*x3^25*x4^17*z^26 + 2*x1^46*x2^42*x3^25*x4^17*z^26 + 2*x1^51*x2^36*x3^26*x4^17*z^26 - x1^49*x2^38*x3^26*x4^17*z^26 - x1^48*x2^39*x3^26*x4^17*z^26 - 2*x1^46*x2^41*x3^26*x4^17*z^26 - x1^45*x2^42*x3^26*x4^17*z^26 - x1^51*x2^35*x3^27*x4^17*z^26 - x1^50*x2^36*x3^27*x4^17*z^26 - x1^49*x2^37*x3^27*x4^17*z^26 + x1^47*x2^39*x3^27*x4^17*z^26 + x1^46*x2^40*x3^27*x4^17*z^26 + 2*x1^45*x2^41*x3^27*x4^17*z^26 + x1^50*x2^35*x3^28*x4^17*z^26 - 2*x1^47*x2^38*x3^28*x4^17*z^26 - 2*x1^45*x2^40*x3^28*x4^17*z^26 + x1^47*x2^37*x3^29*x4^17*z^26 + 2*x1^44*x2^40*x3^29*x4^17*z^26 + x1^47*x2^36*x3^30*x4^17*z^26 - x1^46*x2^37*x3^30*x4^17*z^26 - 2*x1^44*x2^39*x3^30*x4^17*z^26 - x1^43*x2^40*x3^30*x4^17*z^26 + x1^45*x2^37*x3^31*x4^17*z^26 + x1^44*x2^38*x3^31*x4^17*z^26 + 2*x1^43*x2^39*x3^31*x4^17*z^26 - x1^42*x2^39*x3^32*x4^17*z^26 + x1^57*x2^43*x3^23*x4^2*z^25 - 2*x1^56*x2^43*x3^24*x4^2*z^25 + x1^56*x2^42*x3^25*x4^2*z^25 - x1^57*x2^43*x3^22*x4^3*z^25 - x1^58*x2^41*x3^23*x4^3*z^25 + 2*x1^56*x2^43*x3^23*x4^3*z^25 - x1^55*x2^44*x3^23*x4^3*z^25 - x1^56*x2^42*x3^24*x4^3*z^25 + 2*x1^55*x2^42*x3^25*x4^3*z^25 - x1^55*x2^41*x3^26*x4^3*z^25 - x1^52*x2^44*x3^26*x4^3*z^25 + x1^58*x2^41*x3^22*x4^4*z^25 - 2*x1^57*x2^41*x3^23*x4^4*z^25 + x1^56*x2^42*x3^23*x4^4*z^25 - x1^54*x2^44*x3^23*x4^4*z^25 + 2*x1^57*x2^40*x3^24*x4^4*z^25 + x1^56*x2^41*x3^24*x4^4*z^25 - x1^55*x2^42*x3^24*x4^4*z^25 + 3*x1^54*x2^43*x3^24*x4^4*z^25 + x1^53*x2^44*x3^24*x4^4*z^25 - x1^52*x2^45*x3^24*x4^4*z^25 - 3*x1^56*x2^40*x3^25*x4^4*z^25 + x1^55*x2^41*x3^25*x4^4*z^25 - x1^53*x2^43*x3^25*x4^4*z^25 + x1^52*x2^44*x3^25*x4^4*z^25 + x1^55*x2^40*x3^26*x4^4*z^25 - x1^54*x2^41*x3^26*x4^4*z^25 + 3*x1^53*x2^42*x3^26*x4^4*z^25 + x1^52*x2^43*x3^26*x4^4*z^25 + x1^53*x2^41*x3^27*x4^4*z^25 + x1^52*x2^42*x3^27*x4^4*z^25 + x1^51*x2^43*x3^27*x4^4*z^25 - 2*x1^58*x2^41*x3^21*x4^5*z^25 + 4*x1^57*x2^41*x3^22*x4^5*z^25 - x1^56*x2^42*x3^22*x4^5*z^25 + x1^55*x2^43*x3^22*x4^5*z^25 - 2*x1^57*x2^40*x3^23*x4^5*z^25 - 4*x1^56*x2^41*x3^23*x4^5*z^25 + x1^55*x2^42*x3^23*x4^5*z^25 - 2*x1^54*x2^43*x3^23*x4^5*z^25 + 2*x1^53*x2^44*x3^23*x4^5*z^25 + 5*x1^56*x2^40*x3^24*x4^5*z^25 + x1^55*x2^41*x3^24*x4^5*z^25 - x1^54*x2^42*x3^24*x4^5*z^25 + 2*x1^53*x2^43*x3^24*x4^5*z^25 - 2*x1^56*x2^39*x3^25*x4^5*z^25 - 4*x1^55*x2^40*x3^25*x4^5*z^25 + 3*x1^54*x2^41*x3^25*x4^5*z^25 - 2*x1^53*x2^42*x3^25*x4^5*z^25 + x1^51*x2^44*x3^25*x4^5*z^25 + 4*x1^55*x2^39*x3^26*x4^5*z^25 + 2*x1^52*x2^42*x3^26*x4^5*z^25 - x1^55*x2^38*x3^27*x4^5*z^25 - 2*x1^54*x2^39*x3^27*x4^5*z^25 + x1^53*x2^40*x3^27*x4^5*z^25 - 3*x1^52*x2^41*x3^27*x4^5*z^25 + x1^51*x2^42*x3^27*x4^5*z^25 + x1^54*x2^38*x3^28*x4^5*z^25 - x1^53*x2^39*x3^28*x4^5*z^25 - x1^52*x2^40*x3^28*x4^5*z^25 + x1^51*x2^41*x3^28*x4^5*z^25 + x1^52*x2^39*x3^29*x4^5*z^25 - x1^51*x2^40*x3^29*x4^5*z^25 + x1^50*x2^41*x3^29*x4^5*z^25 + x1^49*x2^42*x3^29*x4^5*z^25 + x1^58*x2^41*x3^20*x4^6*z^25 - x1^57*x2^41*x3^21*x4^6*z^25 + x1^56*x2^42*x3^21*x4^6*z^25 + x1^56*x2^41*x3^22*x4^6*z^25 - x1^56*x2^40*x3^23*x4^6*z^25 + x1^55*x2^41*x3^23*x4^6*z^25 + x1^53*x2^43*x3^23*x4^6*z^25 + x1^52*x2^44*x3^23*x4^6*z^25 + x1^56*x2^39*x3^24*x4^6*z^25 + 2*x1^55*x2^40*x3^24*x4^6*z^25 - 4*x1^54*x2^41*x3^24*x4^6*z^25 + 3*x1^53*x2^42*x3^24*x4^6*z^25 - 2*x1^51*x2^44*x3^24*x4^6*z^25 - x1^55*x2^39*x3^25*x4^6*z^25 - x1^54*x2^40*x3^25*x4^6*z^25 + 2*x1^53*x2^41*x3^25*x4^6*z^25 + 2*x1^50*x2^44*x3^25*x4^6*z^25 + x1^49*x2^45*x3^25*x4^6*z^25 + x1^55*x2^38*x3^26*x4^6*z^25 + 3*x1^54*x2^39*x3^26*x4^6*z^25 - 3*x1^53*x2^40*x3^26*x4^6*z^25 - 2*x1^52*x2^41*x3^26*x4^6*z^25 - x1^51*x2^42*x3^26*x4^6*z^25 + 2*x1^50*x2^43*x3^26*x4^6*z^25 - 2*x1^49*x2^44*x3^26*x4^6*z^25 - x1^48*x2^45*x3^26*x4^6*z^25 - 2*x1^54*x2^38*x3^27*x4^6*z^25 + x1^53*x2^39*x3^27*x4^6*z^25 + x1^52*x2^40*x3^27*x4^6*z^25 - 2*x1^51*x2^41*x3^27*x4^6*z^25 + x1^50*x2^42*x3^27*x4^6*z^25 + x1^49*x2^43*x3^27*x4^6*z^25 - x1^47*x2^45*x3^27*x4^6*z^25 + x1^54*x2^37*x3^28*x4^6*z^25 + x1^53*x2^38*x3^28*x4^6*z^25 - x1^52*x2^39*x3^28*x4^6*z^25 + x1^51*x2^40*x3^28*x4^6*z^25 - 2*x1^50*x2^41*x3^28*x4^6*z^25 - x1^53*x2^37*x3^29*x4^6*z^25 + x1^52*x2^38*x3^29*x4^6*z^25 - 2*x1^50*x2^40*x3^29*x4^6*z^25 - x1^49*x2^41*x3^29*x4^6*z^25 - x1^51*x2^38*x3^30*x4^6*z^25 + x1^50*x2^39*x3^30*x4^6*z^25 - x1^49*x2^40*x3^30*x4^6*z^25 - x1^48*x2^41*x3^30*x4^6*z^25 - x1^56*x2^43*x3^19*x4^7*z^25 + x1^56*x2^42*x3^20*x4^7*z^25 + x1^55*x2^43*x3^20*x4^7*z^25 - x1^54*x2^44*x3^20*x4^7*z^25 - x1^57*x2^40*x3^21*x4^7*z^25 - 2*x1^55*x2^42*x3^21*x4^7*z^25 - x1^54*x2^43*x3^21*x4^7*z^25 + x1^56*x2^40*x3^22*x4^7*z^25 + x1^55*x2^41*x3^22*x4^7*z^25 + 2*x1^54*x2^42*x3^22*x4^7*z^25 + x1^53*x2^43*x3^22*x4^7*z^25 + x1^52*x2^44*x3^22*x4^7*z^25 - x1^51*x2^45*x3^22*x4^7*z^25 - x1^55*x2^40*x3^23*x4^7*z^25 - x1^54*x2^41*x3^23*x4^7*z^25 + x1^52*x2^43*x3^23*x4^7*z^25 - x1^51*x2^44*x3^23*x4^7*z^25 + x1^49*x2^46*x3^23*x4^7*z^25 + x1^54*x2^40*x3^24*x4^7*z^25 + x1^53*x2^41*x3^24*x4^7*z^25 + 3*x1^51*x2^43*x3^24*x4^7*z^25 + 2*x1^50*x2^44*x3^24*x4^7*z^25 - x1^48*x2^46*x3^24*x4^7*z^25 - x1^54*x2^39*x3^25*x4^7*z^25 + x1^53*x2^40*x3^25*x4^7*z^25 - 2*x1^52*x2^41*x3^25*x4^7*z^25 + x1^50*x2^43*x3^25*x4^7*z^25 + x1^49*x2^44*x3^25*x4^7*z^25 + 2*x1^53*x2^39*x3^26*x4^7*z^25 - x1^51*x2^41*x3^26*x4^7*z^25 + x1^50*x2^42*x3^26*x4^7*z^25 - x1^49*x2^43*x3^26*x4^7*z^25 - x1^48*x2^44*x3^26*x4^7*z^25 - x1^53*x2^38*x3^27*x4^7*z^25 + x1^50*x2^41*x3^27*x4^7*z^25 - 2*x1^49*x2^42*x3^27*x4^7*z^25 + 2*x1^48*x2^43*x3^27*x4^7*z^25 + x1^47*x2^44*x3^27*x4^7*z^25 - x1^46*x2^45*x3^27*x4^7*z^25 + x1^53*x2^37*x3^28*x4^7*z^25 + x1^52*x2^38*x3^28*x4^7*z^25 + x1^51*x2^39*x3^28*x4^7*z^25 - 2*x1^53*x2^36*x3^29*x4^7*z^25 + x1^51*x2^38*x3^29*x4^7*z^25 - x1^50*x2^39*x3^29*x4^7*z^25 + x1^49*x2^40*x3^29*x4^7*z^25 - x1^48*x2^41*x3^29*x4^7*z^25 + x1^52*x2^36*x3^30*x4^7*z^25 + x1^50*x2^38*x3^30*x4^7*z^25 + 2*x1^49*x2^39*x3^30*x4^7*z^25 + x1^50*x2^37*x3^31*x4^7*z^25 - x1^56*x2^42*x3^19*x4^8*z^25 + 4*x1^55*x2^42*x3^20*x4^8*z^25 + x1^54*x2^43*x3^20*x4^8*z^25 + 2*x1^53*x2^44*x3^20*x4^8*z^25 - 3*x1^55*x2^41*x3^21*x4^8*z^25 - 5*x1^54*x2^42*x3^21*x4^8*z^25 + x1^53*x2^43*x3^21*x4^8*z^25 - 2*x1^52*x2^44*x3^21*x4^8*z^25 + x1^51*x2^45*x3^21*x4^8*z^25 + x1^56*x2^39*x3^22*x4^8*z^25 + 5*x1^54*x2^41*x3^22*x4^8*z^25 + x1^53*x2^42*x3^22*x4^8*z^25 + x1^51*x2^44*x3^22*x4^8*z^25 - 2*x1^55*x2^39*x3^23*x4^8*z^25 - 5*x1^53*x2^41*x3^23*x4^8*z^25 - 2*x1^52*x2^42*x3^23*x4^8*z^25 - 5*x1^51*x2^43*x3^23*x4^8*z^25 + x1^50*x2^44*x3^23*x4^8*z^25 + x1^48*x2^46*x3^23*x4^8*z^25 + 2*x1^54*x2^39*x3^24*x4^8*z^25 + 3*x1^53*x2^40*x3^24*x4^8*z^25 + 4*x1^52*x2^41*x3^24*x4^8*z^25 + x1^51*x2^42*x3^24*x4^8*z^25 + x1^50*x2^43*x3^24*x4^8*z^25 - x1^49*x2^44*x3^24*x4^8*z^25 - 2*x1^48*x2^45*x3^24*x4^8*z^25 - x1^47*x2^46*x3^24*x4^8*z^25 - 2*x1^54*x2^38*x3^25*x4^8*z^25 - 3*x1^53*x2^39*x3^25*x4^8*z^25 - 4*x1^52*x2^40*x3^25*x4^8*z^25 - 3*x1^50*x2^42*x3^25*x4^8*z^25 + x1^48*x2^44*x3^25*x4^8*z^25 + 3*x1^47*x2^45*x3^25*x4^8*z^25 + x1^54*x2^37*x3^26*x4^8*z^25 + 3*x1^52*x2^39*x3^26*x4^8*z^25 + 2*x1^51*x2^40*x3^26*x4^8*z^25 + x1^50*x2^41*x3^26*x4^8*z^25 + x1^49*x2^42*x3^26*x4^8*z^25 - x1^47*x2^44*x3^26*x4^8*z^25 - x1^46*x2^45*x3^26*x4^8*z^25 - 2*x1^53*x2^37*x3^27*x4^8*z^25 - 3*x1^52*x2^38*x3^27*x4^8*z^25 - 5*x1^51*x2^39*x3^27*x4^8*z^25 - x1^50*x2^40*x3^27*x4^8*z^25 - 2*x1^49*x2^41*x3^27*x4^8*z^25 + x1^48*x2^42*x3^27*x4^8*z^25 + 2*x1^46*x2^44*x3^27*x4^8*z^25 + x1^53*x2^36*x3^28*x4^8*z^25 + x1^52*x2^37*x3^28*x4^8*z^25 + 3*x1^51*x2^38*x3^28*x4^8*z^25 + 2*x1^50*x2^39*x3^28*x4^8*z^25 - x1^49*x2^40*x3^28*x4^8*z^25 + 3*x1^48*x2^41*x3^28*x4^8*z^25 - x1^47*x2^42*x3^28*x4^8*z^25 - x1^46*x2^43*x3^28*x4^8*z^25 - x1^52*x2^36*x3^29*x4^8*z^25 - 2*x1^50*x2^38*x3^29*x4^8*z^25 - x1^49*x2^39*x3^29*x4^8*z^25 - x1^48*x2^40*x3^29*x4^8*z^25 + x1^46*x2^42*x3^29*x4^8*z^25 + 2*x1^45*x2^43*x3^29*x4^8*z^25 + x1^52*x2^35*x3^30*x4^8*z^25 - x1^51*x2^36*x3^30*x4^8*z^25 + x1^50*x2^37*x3^30*x4^8*z^25 + x1^49*x2^38*x3^30*x4^8*z^25 + x1^48*x2^39*x3^30*x4^8*z^25 + 2*x1^47*x2^40*x3^30*x4^8*z^25 - x1^51*x2^35*x3^31*x4^8*z^25 - x1^50*x2^36*x3^31*x4^8*z^25 - x1^49*x2^37*x3^31*x4^8*z^25 - x1^48*x2^38*x3^31*x4^8*z^25 - x1^47*x2^39*x3^31*x4^8*z^25 + x1^47*x2^38*x3^32*x4^8*z^25 + x1^46*x2^39*x3^32*x4^8*z^25 + x1^57*x2^40*x3^19*x4^9*z^25 - x1^55*x2^42*x3^19*x4^9*z^25 + x1^54*x2^43*x3^19*x4^9*z^25 - x1^56*x2^40*x3^20*x4^9*z^25 + x1^55*x2^41*x3^20*x4^9*z^25 + x1^54*x2^42*x3^20*x4^9*z^25 - 2*x1^53*x2^43*x3^20*x4^9*z^25 - 2*x1^56*x2^39*x3^21*x4^9*z^25 + x1^55*x2^40*x3^21*x4^9*z^25 - x1^54*x2^41*x3^21*x4^9*z^25 + x1^53*x2^42*x3^21*x4^9*z^25 + x1^52*x2^43*x3^21*x4^9*z^25 + x1^51*x2^44*x3^21*x4^9*z^25 + x1^50*x2^45*x3^21*x4^9*z^25 + 3*x1^55*x2^39*x3^22*x4^9*z^25 - 2*x1^54*x2^40*x3^22*x4^9*z^25 + x1^53*x2^41*x3^22*x4^9*z^25 - x1^52*x2^42*x3^22*x4^9*z^25 - 2*x1^50*x2^44*x3^22*x4^9*z^25 - x1^49*x2^45*x3^22*x4^9*z^25 - x1^55*x2^38*x3^23*x4^9*z^25 - 3*x1^54*x2^39*x3^23*x4^9*z^25 - x1^52*x2^41*x3^23*x4^9*z^25 + x1^51*x2^42*x3^23*x4^9*z^25 + x1^50*x2^43*x3^23*x4^9*z^25 + x1^49*x2^44*x3^23*x4^9*z^25 + x1^48*x2^45*x3^23*x4^9*z^25 + 4*x1^54*x2^38*x3^24*x4^9*z^25 + 2*x1^53*x2^39*x3^24*x4^9*z^25 + x1^52*x2^40*x3^24*x4^9*z^25 + x1^50*x2^42*x3^24*x4^9*z^25 - x1^49*x2^43*x3^24*x4^9*z^25 - x1^48*x2^44*x3^24*x4^9*z^25 - 2*x1^47*x2^45*x3^24*x4^9*z^25 - x1^54*x2^37*x3^25*x4^9*z^25 - 4*x1^53*x2^38*x3^25*x4^9*z^25 - 2*x1^52*x2^39*x3^25*x4^9*z^25 - 3*x1^51*x2^40*x3^25*x4^9*z^25 - x1^50*x2^41*x3^25*x4^9*z^25 + x1^49*x2^42*x3^25*x4^9*z^25 + x1^47*x2^44*x3^25*x4^9*z^25 + x1^46*x2^45*x3^25*x4^9*z^25 + 4*x1^53*x2^37*x3^26*x4^9*z^25 + 3*x1^52*x2^38*x3^26*x4^9*z^25 + 4*x1^51*x2^39*x3^26*x4^9*z^25 + 2*x1^50*x2^40*x3^26*x4^9*z^25 + x1^49*x2^41*x3^26*x4^9*z^25 - 4*x1^48*x2^42*x3^26*x4^9*z^25 + x1^47*x2^43*x3^26*x4^9*z^25 - 3*x1^46*x2^44*x3^26*x4^9*z^25 - 2*x1^53*x2^36*x3^27*x4^9*z^25 - 2*x1^52*x2^37*x3^27*x4^9*z^25 - x1^51*x2^38*x3^27*x4^9*z^25 - 2*x1^50*x2^39*x3^27*x4^9*z^25 + 2*x1^49*x2^40*x3^27*x4^9*z^25 - x1^48*x2^41*x3^27*x4^9*z^25 + 2*x1^47*x2^42*x3^27*x4^9*z^25 + x1^45*x2^44*x3^27*x4^9*z^25 + 4*x1^52*x2^36*x3^28*x4^9*z^25 - x1^51*x2^37*x3^28*x4^9*z^25 + 2*x1^50*x2^38*x3^28*x4^9*z^25 + 3*x1^49*x2^39*x3^28*x4^9*z^25 - x1^46*x2^42*x3^28*x4^9*z^25 - x1^45*x2^43*x3^28*x4^9*z^25 - x1^52*x2^35*x3^29*x4^9*z^25 - 2*x1^51*x2^36*x3^29*x4^9*z^25 - 4*x1^49*x2^38*x3^29*x4^9*z^25 - x1^47*x2^40*x3^29*x4^9*z^25 - x1^45*x2^42*x3^29*x4^9*z^25 + 2*x1^51*x2^35*x3^30*x4^9*z^25 + x1^50*x2^36*x3^30*x4^9*z^25 + x1^49*x2^37*x3^30*x4^9*z^25 + x1^48*x2^38*x3^30*x4^9*z^25 - x1^45*x2^41*x3^30*x4^9*z^25 - 2*x1^44*x2^42*x3^30*x4^9*z^25 + x1^49*x2^36*x3^31*x4^9*z^25 - 2*x1^48*x2^37*x3^31*x4^9*z^25 - x1^47*x2^38*x3^31*x4^9*z^25 - x1^46*x2^38*x3^32*x4^9*z^25 + x1^47*x2^36*x3^33*x4^9*z^25 - x1^45*x2^38*x3^33*x4^9*z^25 + x1^57*x2^41*x3^17*x4^10*z^25 - x1^57*x2^40*x3^18*x4^10*z^25 - x1^56*x2^41*x3^18*x4^10*z^25 + 2*x1^55*x2^42*x3^18*x4^10*z^25 + x1^56*x2^40*x3^19*x4^10*z^25 + x1^55*x2^41*x3^19*x4^10*z^25 - x1^53*x2^43*x3^19*x4^10*z^25 - x1^56*x2^39*x3^20*x4^10*z^25 - x1^55*x2^40*x3^20*x4^10*z^25 - x1^53*x2^42*x3^20*x4^10*z^25 + x1^52*x2^43*x3^20*x4^10*z^25 + x1^55*x2^39*x3^21*x4^10*z^25 - x1^54*x2^40*x3^21*x4^10*z^25 - x1^52*x2^42*x3^21*x4^10*z^25 - x1^51*x2^43*x3^21*x4^10*z^25 - 2*x1^50*x2^44*x3^21*x4^10*z^25 - x1^54*x2^39*x3^22*x4^10*z^25 - 2*x1^52*x2^41*x3^22*x4^10*z^25 + 2*x1^51*x2^42*x3^22*x4^10*z^25 + 2*x1^49*x2^44*x3^22*x4^10*z^25 - x1^48*x2^45*x3^22*x4^10*z^25 + x1^53*x2^39*x3^23*x4^10*z^25 - x1^50*x2^42*x3^23*x4^10*z^25 - 2*x1^48*x2^44*x3^23*x4^10*z^25 + x1^51*x2^40*x3^24*x4^10*z^25 + 2*x1^50*x2^41*x3^24*x4^10*z^25 + 3*x1^48*x2^43*x3^24*x4^10*z^25 + 2*x1^47*x2^44*x3^24*x4^10*z^25 - x1^49*x2^41*x3^25*x4^10*z^25 - x1^47*x2^43*x3^25*x4^10*z^25 - x1^46*x2^44*x3^25*x4^10*z^25 + x1^45*x2^45*x3^25*x4^10*z^25 + 2*x1^48*x2^41*x3^26*x4^10*z^25 + 2*x1^47*x2^42*x3^26*x4^10*z^25 + 2*x1^45*x2^44*x3^26*x4^10*z^25 - x1^49*x2^39*x3^27*x4^10*z^25 + x1^48*x2^40*x3^27*x4^10*z^25 - x1^47*x2^41*x3^27*x4^10*z^25 - 2*x1^44*x2^44*x3^27*x4^10*z^25 + x1^52*x2^35*x3^28*x4^10*z^25 + x1^46*x2^41*x3^28*x4^10*z^25 + x1^45*x2^42*x3^28*x4^10*z^25 + x1^44*x2^43*x3^28*x4^10*z^25 - 3*x1^51*x2^35*x3^29*x4^10*z^25 + 2*x1^50*x2^36*x3^29*x4^10*z^25 + 2*x1^49*x2^37*x3^29*x4^10*z^25 - 2*x1^48*x2^38*x3^29*x4^10*z^25 + x1^47*x2^39*x3^29*x4^10*z^25 - x1^44*x2^42*x3^29*x4^10*z^25 + x1^51*x2^34*x3^30*x4^10*z^25 + 2*x1^50*x2^35*x3^30*x4^10*z^25 - x1^49*x2^36*x3^30*x4^10*z^25 + 2*x1^48*x2^37*x3^30*x4^10*z^25 + 2*x1^47*x2^38*x3^30*x4^10*z^25 - x1^46*x2^39*x3^30*x4^10*z^25 + 2*x1^44*x2^41*x3^30*x4^10*z^25 - 2*x1^50*x2^34*x3^31*x4^10*z^25 - 2*x1^49*x2^35*x3^31*x4^10*z^25 + x1^46*x2^38*x3^31*x4^10*z^25 - x1^45*x2^39*x3^31*x4^10*z^25 + x1^44*x2^40*x3^31*x4^10*z^25 - x1^43*x2^41*x3^31*x4^10*z^25 + x1^49*x2^34*x3^32*x4^10*z^25 + x1^48*x2^35*x3^32*x4^10*z^25 + 2*x1^47*x2^36*x3^32*x4^10*z^25 + x1^46*x2^37*x3^32*x4^10*z^25 - x1^45*x2^38*x3^32*x4^10*z^25 - x1^47*x2^35*x3^33*x4^10*z^25 - x1^46*x2^36*x3^33*x4^10*z^25 + x1^45*x2^37*x3^33*x4^10*z^25 - x1^56*x2^41*x3^17*x4^11*z^25 - x1^56*x2^40*x3^18*x4^11*z^25 + 2*x1^56*x2^39*x3^19*x4^11*z^25 + 3*x1^55*x2^40*x3^19*x4^11*z^25 - 2*x1^54*x2^41*x3^19*x4^11*z^25 - x1^52*x2^43*x3^19*x4^11*z^25 - 3*x1^55*x2^39*x3^20*x4^11*z^25 + 2*x1^53*x2^41*x3^20*x4^11*z^25 + 2*x1^51*x2^43*x3^20*x4^11*z^25 + x1^50*x2^44*x3^20*x4^11*z^25 + x1^55*x2^38*x3^21*x4^11*z^25 + 4*x1^54*x2^39*x3^21*x4^11*z^25 - 2*x1^53*x2^40*x3^21*x4^11*z^25 + x1^52*x2^41*x3^21*x4^11*z^25 - 3*x1^51*x2^42*x3^21*x4^11*z^25 - x1^50*x2^43*x3^21*x4^11*z^25 - x1^49*x2^44*x3^21*x4^11*z^25 - 4*x1^54*x2^38*x3^22*x4^11*z^25 + x1^52*x2^40*x3^22*x4^11*z^25 + 2*x1^50*x2^42*x3^22*x4^11*z^25 + x1^49*x2^43*x3^22*x4^11*z^25 + x1^48*x2^44*x3^22*x4^11*z^25 + x1^47*x2^45*x3^22*x4^11*z^25 + x1^54*x2^37*x3^23*x4^11*z^25 + 4*x1^53*x2^38*x3^23*x4^11*z^25 - 2*x1^52*x2^39*x3^23*x4^11*z^25 + 3*x1^51*x2^40*x3^23*x4^11*z^25 - 2*x1^50*x2^41*x3^23*x4^11*z^25 - x1^49*x2^42*x3^23*x4^11*z^25 - 4*x1^48*x2^43*x3^23*x4^11*z^25 - x1^46*x2^45*x3^23*x4^11*z^25 - 4*x1^53*x2^37*x3^24*x4^11*z^25 - 2*x1^52*x2^38*x3^24*x4^11*z^25 + 4*x1^49*x2^41*x3^24*x4^11*z^25 + x1^48*x2^42*x3^24*x4^11*z^25 + 3*x1^47*x2^43*x3^24*x4^11*z^25 - x1^46*x2^44*x3^24*x4^11*z^25 + 2*x1^53*x2^36*x3^25*x4^11*z^25 + 4*x1^52*x2^37*x3^25*x4^11*z^25 - 2*x1^51*x2^38*x3^25*x4^11*z^25 + x1^50*x2^39*x3^25*x4^11*z^25 - 4*x1^49*x2^40*x3^25*x4^11*z^25 - 3*x1^47*x2^42*x3^25*x4^11*z^25 + x1^45*x2^44*x3^25*x4^11*z^25 - 4*x1^52*x2^36*x3^26*x4^11*z^25 + x1^50*x2^38*x3^26*x4^11*z^25 - x1^49*x2^39*x3^26*x4^11*z^25 + x1^48*x2^40*x3^26*x4^11*z^25 + x1^47*x2^41*x3^26*x4^11*z^25 + 3*x1^46*x2^42*x3^26*x4^11*z^25 - 2*x1^45*x2^43*x3^26*x4^11*z^25 - x1^44*x2^44*x3^26*x4^11*z^25 - x1^52*x2^35*x3^27*x4^11*z^25 + 4*x1^51*x2^36*x3^27*x4^11*z^25 - 2*x1^50*x2^37*x3^27*x4^11*z^25 + 2*x1^49*x2^38*x3^27*x4^11*z^25 - 5*x1^48*x2^39*x3^27*x4^11*z^25 - 2*x1^47*x2^40*x3^27*x4^11*z^25 - 3*x1^46*x2^41*x3^27*x4^11*z^25 + x1^45*x2^42*x3^27*x4^11*z^25 + x1^44*x2^43*x3^27*x4^11*z^25 - x1^50*x2^36*x3^28*x4^11*z^25 + 2*x1^49*x2^37*x3^28*x4^11*z^25 + 3*x1^45*x2^41*x3^28*x4^11*z^25 - x1^44*x2^42*x3^28*x4^11*z^25 + x1^43*x2^43*x3^28*x4^11*z^25 - x1^50*x2^35*x3^29*x4^11*z^25 - x1^49*x2^36*x3^29*x4^11*z^25 + 2*x1^48*x2^37*x3^29*x4^11*z^25 - 2*x1^46*x2^39*x3^29*x4^11*z^25 - 3*x1^45*x2^40*x3^29*x4^11*z^25 - x1^44*x2^41*x3^29*x4^11*z^25 - x1^48*x2^36*x3^30*x4^11*z^25 + 2*x1^47*x2^37*x3^30*x4^11*z^25 + 2*x1^46*x2^38*x3^30*x4^11*z^25 + x1^44*x2^40*x3^30*x4^11*z^25 + x1^43*x2^41*x3^30*x4^11*z^25 - 2*x1^46*x2^37*x3^31*x4^11*z^25 + x1^45*x2^38*x3^31*x4^11*z^25 - x1^44*x2^39*x3^31*x4^11*z^25 - x1^43*x2^40*x3^31*x4^11*z^25 + x1^42*x2^41*x3^31*x4^11*z^25 + x1^47*x2^35*x3^32*x4^11*z^25 + x1^45*x2^37*x3^32*x4^11*z^25 + x1^44*x2^38*x3^32*x4^11*z^25 + x1^42*x2^40*x3^32*x4^11*z^25 + x1^56*x2^40*x3^17*x4^12*z^25 - x1^56*x2^39*x3^18*x4^12*z^25 + x1^54*x2^41*x3^18*x4^12*z^25 - x1^53*x2^42*x3^18*x4^12*z^25 + 2*x1^55*x2^39*x3^19*x4^12*z^25 + x1^54*x2^40*x3^19*x4^12*z^25 - x1^53*x2^41*x3^19*x4^12*z^25 - 2*x1^51*x2^43*x3^19*x4^12*z^25 - x1^55*x2^38*x3^20*x4^12*z^25 - x1^54*x2^39*x3^20*x4^12*z^25 + 2*x1^53*x2^40*x3^20*x4^12*z^25 + x1^51*x2^42*x3^20*x4^12*z^25 + x1^50*x2^43*x3^20*x4^12*z^25 - x1^49*x2^44*x3^20*x4^12*z^25 + x1^54*x2^38*x3^21*x4^12*z^25 - x1^53*x2^39*x3^21*x4^12*z^25 - 4*x1^52*x2^40*x3^21*x4^12*z^25 + x1^51*x2^41*x3^21*x4^12*z^25 - 2*x1^50*x2^42*x3^21*x4^12*z^25 - x1^49*x2^43*x3^21*x4^12*z^25 + x1^48*x2^44*x3^21*x4^12*z^25 - 2*x1^53*x2^38*x3^22*x4^12*z^25 + 6*x1^52*x2^39*x3^22*x4^12*z^25 + 2*x1^50*x2^41*x3^22*x4^12*z^25 + x1^48*x2^43*x3^22*x4^12*z^25 - x1^47*x2^44*x3^22*x4^12*z^25 + 2*x1^53*x2^37*x3^23*x4^12*z^25 - x1^52*x2^38*x3^23*x4^12*z^25 - 5*x1^51*x2^39*x3^23*x4^12*z^25 - 3*x1^49*x2^41*x3^23*x4^12*z^25 + 3*x1^48*x2^42*x3^23*x4^12*z^25 - x1^47*x2^43*x3^23*x4^12*z^25 + 2*x1^46*x2^44*x3^23*x4^12*z^25 - x1^53*x2^36*x3^24*x4^12*z^25 - 2*x1^52*x2^37*x3^24*x4^12*z^25 + 6*x1^51*x2^38*x3^24*x4^12*z^25 + x1^50*x2^39*x3^24*x4^12*z^25 + 3*x1^49*x2^40*x3^24*x4^12*z^25 + 2*x1^47*x2^42*x3^24*x4^12*z^25 - 2*x1^45*x2^44*x3^24*x4^12*z^25 + 2*x1^52*x2^36*x3^25*x4^12*z^25 - x1^51*x2^37*x3^25*x4^12*z^25 - 5*x1^50*x2^38*x3^25*x4^12*z^25 - 5*x1^48*x2^40*x3^25*x4^12*z^25 + 3*x1^47*x2^41*x3^25*x4^12*z^25 - 2*x1^46*x2^42*x3^25*x4^12*z^25 + 6*x1^45*x2^43*x3^25*x4^12*z^25 + x1^44*x2^44*x3^25*x4^12*z^25 - 2*x1^51*x2^36*x3^26*x4^12*z^25 + 6*x1^50*x2^37*x3^26*x4^12*z^25 + x1^49*x2^38*x3^26*x4^12*z^25 + 4*x1^48*x2^39*x3^26*x4^12*z^25 - 2*x1^45*x2^42*x3^26*x4^12*z^25 - 6*x1^44*x2^43*x3^26*x4^12*z^25 + x1^51*x2^35*x3^27*x4^12*z^25 - x1^50*x2^36*x3^27*x4^12*z^25 - 6*x1^49*x2^37*x3^27*x4^12*z^25 + x1^48*x2^38*x3^27*x4^12*z^25 - 3*x1^47*x2^39*x3^27*x4^12*z^25 + 3*x1^46*x2^40*x3^27*x4^12*z^25 - x1^45*x2^41*x3^27*x4^12*z^25 + 6*x1^44*x2^42*x3^27*x4^12*z^25 + 2*x1^43*x2^43*x3^27*x4^12*z^25 - x1^50*x2^35*x3^28*x4^12*z^25 + 5*x1^49*x2^36*x3^28*x4^12*z^25 + x1^48*x2^37*x3^28*x4^12*z^25 + 3*x1^47*x2^38*x3^28*x4^12*z^25 + x1^46*x2^39*x3^28*x4^12*z^25 - x1^45*x2^40*x3^28*x4^12*z^25 - 2*x1^44*x2^41*x3^28*x4^12*z^25 - 5*x1^43*x2^42*x3^28*x4^12*z^25 + x1^50*x2^34*x3^29*x4^12*z^25 + x1^49*x2^35*x3^29*x4^12*z^25 - 4*x1^48*x2^36*x3^29*x4^12*z^25 - x1^47*x2^37*x3^29*x4^12*z^25 - 3*x1^46*x2^38*x3^29*x4^12*z^25 - x1^44*x2^40*x3^29*x4^12*z^25 + 4*x1^43*x2^41*x3^29*x4^12*z^25 + x1^42*x2^42*x3^29*x4^12*z^25 - x1^49*x2^34*x3^30*x4^12*z^25 + 2*x1^47*x2^36*x3^30*x4^12*z^25 + 3*x1^46*x2^37*x3^30*x4^12*z^25 + x1^45*x2^38*x3^30*x4^12*z^25 - x1^43*x2^40*x3^30*x4^12*z^25 - 2*x1^42*x2^41*x3^30*x4^12*z^25 - 3*x1^45*x2^37*x3^31*x4^12*z^25 + x1^44*x2^38*x3^31*x4^12*z^25 + x1^43*x2^39*x3^31*x4^12*z^25 + 2*x1^42*x2^40*x3^31*x4^12*z^25 - x1^46*x2^35*x3^32*x4^12*z^25 + x1^45*x2^36*x3^32*x4^12*z^25 + 2*x1^44*x2^37*x3^32*x4^12*z^25 - 2*x1^43*x2^38*x3^32*x4^12*z^25 - x1^42*x2^39*x3^32*x4^12*z^25 - x1^41*x2^40*x3^32*x4^12*z^25 + x1^44*x2^36*x3^33*x4^12*z^25 - 2*x1^55*x2^39*x3^18*x4^13*z^25 + x1^53*x2^41*x3^18*x4^13*z^25 - x1^52*x2^42*x3^18*x4^13*z^25 - x1^53*x2^40*x3^19*x4^13*z^25 + x1^51*x2^42*x3^19*x4^13*z^25 + x1^50*x2^43*x3^19*x4^13*z^25 - 2*x1^54*x2^38*x3^20*x4^13*z^25 + 2*x1^52*x2^40*x3^20*x4^13*z^25 - 2*x1^51*x2^41*x3^20*x4^13*z^25 + x1^50*x2^42*x3^20*x4^13*z^25 - 2*x1^52*x2^39*x3^21*x4^13*z^25 - x1^51*x2^40*x3^21*x4^13*z^25 + 2*x1^48*x2^43*x3^21*x4^13*z^25 + x1^47*x2^44*x3^21*x4^13*z^25 - 2*x1^53*x2^37*x3^22*x4^13*z^25 + x1^52*x2^38*x3^22*x4^13*z^25 + 5*x1^51*x2^39*x3^22*x4^13*z^25 + 3*x1^49*x2^41*x3^22*x4^13*z^25 - 4*x1^48*x2^42*x3^22*x4^13*z^25 - 2*x1^47*x2^43*x3^22*x4^13*z^25 - 2*x1^46*x2^44*x3^22*x4^13*z^25 - 6*x1^51*x2^38*x3^23*x4^13*z^25 - 2*x1^50*x2^39*x3^23*x4^13*z^25 - 2*x1^49*x2^40*x3^23*x4^13*z^25 - x1^48*x2^41*x3^23*x4^13*z^25 + 2*x1^47*x2^42*x3^23*x4^13*z^25 + 2*x1^46*x2^43*x3^23*x4^13*z^25 + 2*x1^45*x2^44*x3^23*x4^13*z^25 + 2*x1^51*x2^37*x3^24*x4^13*z^25 + 6*x1^50*x2^38*x3^24*x4^13*z^25 + 4*x1^48*x2^40*x3^24*x4^13*z^25 - 4*x1^47*x2^41*x3^24*x4^13*z^25 - x1^46*x2^42*x3^24*x4^13*z^25 - 5*x1^45*x2^43*x3^24*x4^13*z^25 - x1^44*x2^44*x3^24*x4^13*z^25 - 6*x1^50*x2^37*x3^25*x4^13*z^25 - 2*x1^49*x2^38*x3^25*x4^13*z^25 - 2*x1^48*x2^39*x3^25*x4^13*z^25 + 2*x1^46*x2^41*x3^25*x4^13*z^25 + 2*x1^45*x2^42*x3^25*x4^13*z^25 + 5*x1^44*x2^43*x3^25*x4^13*z^25 + 2*x1^50*x2^36*x3^26*x4^13*z^25 + 6*x1^49*x2^37*x3^26*x4^13*z^25 + 4*x1^47*x2^39*x3^26*x4^13*z^25 - 4*x1^46*x2^40*x3^26*x4^13*z^25 - 6*x1^44*x2^42*x3^26*x4^13*z^25 - 2*x1^43*x2^43*x3^26*x4^13*z^25 - x1^51*x2^34*x3^27*x4^13*z^25 - 4*x1^49*x2^36*x3^27*x4^13*z^25 - 2*x1^48*x2^37*x3^27*x4^13*z^25 - 2*x1^47*x2^38*x3^27*x4^13*z^25 + 2*x1^45*x2^40*x3^27*x4^13*z^25 + 2*x1^44*x2^41*x3^27*x4^13*z^25 + 6*x1^43*x2^42*x3^27*x4^13*z^25 + x1^50*x2^34*x3^28*x4^13*z^25 + x1^49*x2^35*x3^28*x4^13*z^25 + 4*x1^48*x2^36*x3^28*x4^13*z^25 + 2*x1^47*x2^37*x3^28*x4^13*z^25 + 4*x1^46*x2^38*x3^28*x4^13*z^25 - 4*x1^45*x2^39*x3^28*x4^13*z^25 - 6*x1^43*x2^41*x3^28*x4^13*z^25 - 2*x1^42*x2^42*x3^28*x4^13*z^25 - x1^49*x2^34*x3^29*x4^13*z^25 - x1^48*x2^35*x3^29*x4^13*z^25 - x1^47*x2^36*x3^29*x4^13*z^25 + x1^45*x2^38*x3^29*x4^13*z^25 + 3*x1^44*x2^39*x3^29*x4^13*z^25 + 2*x1^43*x2^40*x3^29*x4^13*z^25 + 6*x1^42*x2^41*x3^29*x4^13*z^25 + x1^48*x2^34*x3^30*x4^13*z^25 + x1^47*x2^35*x3^30*x4^13*z^25 + x1^46*x2^36*x3^30*x4^13*z^25 + 2*x1^45*x2^37*x3^30*x4^13*z^25 - x1^43*x2^39*x3^30*x4^13*z^25 - 5*x1^42*x2^40*x3^30*x4^13*z^25 - 2*x1^41*x2^41*x3^30*x4^13*z^25 - x1^46*x2^35*x3^31*x4^13*z^25 + x1^43*x2^38*x3^31*x4^13*z^25 + 2*x1^42*x2^39*x3^31*x4^13*z^25 + 3*x1^41*x2^40*x3^31*x4^13*z^25 - x1^45*x2^35*x3^32*x4^13*z^25 - 2*x1^41*x2^39*x3^32*x4^13*z^25 - x1^43*x2^36*x3^33*x4^13*z^25 + x1^40*x2^39*x3^33*x4^13*z^25 - x1^54*x2^39*x3^18*x4^14*z^25 - x1^51*x2^42*x3^18*x4^14*z^25 + x1^54*x2^38*x3^19*x4^14*z^25 - x1^50*x2^42*x3^19*x4^14*z^25 - 3*x1^53*x2^38*x3^20*x4^14*z^25 - 2*x1^52*x2^39*x3^20*x4^14*z^25 + x1^50*x2^41*x3^20*x4^14*z^25 - x1^48*x2^43*x3^20*x4^14*z^25 + 2*x1^53*x2^37*x3^21*x4^14*z^25 - x1^52*x2^38*x3^21*x4^14*z^25 - x1^51*x2^39*x3^21*x4^14*z^25 + 2*x1^50*x2^40*x3^21*x4^14*z^25 + x1^47*x2^43*x3^21*x4^14*z^25 + x1^46*x2^44*x3^21*x4^14*z^25 - 2*x1^52*x2^37*x3^22*x4^14*z^25 - x1^51*x2^38*x3^22*x4^14*z^25 - x1^50*x2^39*x3^22*x4^14*z^25 + x1^49*x2^40*x3^22*x4^14*z^25 - 2*x1^46*x2^43*x3^22*x4^14*z^25 - x1^45*x2^44*x3^22*x4^14*z^25 + 4*x1^52*x2^36*x3^23*x4^14*z^25 - x1^50*x2^38*x3^23*x4^14*z^25 + 2*x1^47*x2^41*x3^23*x4^14*z^25 - x1^46*x2^42*x3^23*x4^14*z^25 + 2*x1^45*x2^43*x3^23*x4^14*z^25 - 2*x1^51*x2^36*x3^24*x4^14*z^25 + 2*x1^50*x2^37*x3^24*x4^14*z^25 - 2*x1^49*x2^38*x3^24*x4^14*z^25 + x1^48*x2^39*x3^24*x4^14*z^25 + 2*x1^46*x2^41*x3^24*x4^14*z^25 - 2*x1^44*x2^43*x3^24*x4^14*z^25 + 2*x1^51*x2^35*x3^25*x4^14*z^25 + x1^50*x2^36*x3^25*x4^14*z^25 - x1^49*x2^37*x3^25*x4^14*z^25 - 3*x1^47*x2^39*x3^25*x4^14*z^25 - 2*x1^45*x2^41*x3^25*x4^14*z^25 + 2*x1^44*x2^42*x3^25*x4^14*z^25 + x1^43*x2^43*x3^25*x4^14*z^25 - x1^51*x2^34*x3^26*x4^14*z^25 - 2*x1^50*x2^35*x3^26*x4^14*z^25 + 2*x1^49*x2^36*x3^26*x4^14*z^25 + 2*x1^47*x2^38*x3^26*x4^14*z^25 + x1^45*x2^40*x3^26*x4^14*z^25 - 2*x1^43*x2^42*x3^26*x4^14*z^25 + 2*x1^50*x2^34*x3^27*x4^14*z^25 - x1^49*x2^35*x3^27*x4^14*z^25 - 2*x1^48*x2^36*x3^27*x4^14*z^25 - 2*x1^46*x2^38*x3^27*x4^14*z^25 - 2*x1^44*x2^40*x3^27*x4^14*z^25 + 2*x1^43*x2^41*x3^27*x4^14*z^25 + x1^42*x2^42*x3^27*x4^14*z^25 - 2*x1^49*x2^34*x3^28*x4^14*z^25 + x1^48*x2^35*x3^28*x4^14*z^25 + 2*x1^46*x2^37*x3^28*x4^14*z^25 + x1^44*x2^39*x3^28*x4^14*z^25 - x1^43*x2^40*x3^28*x4^14*z^25 - 2*x1^42*x2^41*x3^28*x4^14*z^25 + x1^48*x2^34*x3^29*x4^14*z^25 - x1^47*x2^35*x3^29*x4^14*z^25 - 2*x1^46*x2^36*x3^29*x4^14*z^25 - 2*x1^45*x2^37*x3^29*x4^14*z^25 + x1^44*x2^38*x3^29*x4^14*z^25 - 2*x1^43*x2^39*x3^29*x4^14*z^25 + 2*x1^42*x2^40*x3^29*x4^14*z^25 - x1^47*x2^34*x3^30*x4^14*z^25 + x1^45*x2^36*x3^30*x4^14*z^25 + x1^44*x2^37*x3^30*x4^14*z^25 + 2*x1^43*x2^38*x3^30*x4^14*z^25 - 2*x1^41*x2^40*x3^30*x4^14*z^25 + x1^46*x2^34*x3^31*x4^14*z^25 - x1^44*x2^36*x3^31*x4^14*z^25 - x1^43*x2^37*x3^31*x4^14*z^25 + x1^41*x2^39*x3^31*x4^14*z^25 + x1^40*x2^40*x3^31*x4^14*z^25 + x1^44*x2^35*x3^32*x4^14*z^25 + 2*x1^42*x2^37*x3^32*x4^14*z^25 - x1^40*x2^39*x3^32*x4^14*z^25 + x1^51*x2^41*x3^18*x4^15*z^25 + 2*x1^53*x2^38*x3^19*x4^15*z^25 - x1^51*x2^40*x3^19*x4^15*z^25 - x1^53*x2^37*x3^20*x4^15*z^25 - x1^52*x2^38*x3^20*x4^15*z^25 + 2*x1^50*x2^40*x3^20*x4^15*z^25 + x1^49*x2^41*x3^20*x4^15*z^25 + 4*x1^52*x2^37*x3^21*x4^15*z^25 - x1^50*x2^39*x3^21*x4^15*z^25 - 4*x1^49*x2^40*x3^21*x4^15*z^25 - 2*x1^48*x2^41*x3^21*x4^15*z^25 - x1^47*x2^42*x3^21*x4^15*z^25 - 3*x1^52*x2^36*x3^22*x4^15*z^25 - x1^51*x2^37*x3^22*x4^15*z^25 - x1^50*x2^38*x3^22*x4^15*z^25 + x1^49*x2^39*x3^22*x4^15*z^25 + x1^48*x2^40*x3^22*x4^15*z^25 + 2*x1^47*x2^41*x3^22*x4^15*z^25 + 2*x1^46*x2^42*x3^22*x4^15*z^25 + x1^52*x2^35*x3^23*x4^15*z^25 + 6*x1^51*x2^36*x3^23*x4^15*z^25 + 2*x1^49*x2^38*x3^23*x4^15*z^25 - 4*x1^48*x2^39*x3^23*x4^15*z^25 - 6*x1^46*x2^41*x3^23*x4^15*z^25 - 6*x1^51*x2^35*x3^24*x4^15*z^25 - 2*x1^50*x2^36*x3^24*x4^15*z^25 - 2*x1^49*x2^37*x3^24*x4^15*z^25 + 2*x1^47*x2^39*x3^24*x4^15*z^25 + 2*x1^46*x2^40*x3^24*x4^15*z^25 + 6*x1^45*x2^41*x3^24*x4^15*z^25 + 2*x1^51*x2^34*x3^25*x4^15*z^25 + 6*x1^50*x2^35*x3^25*x4^15*z^25 + 4*x1^48*x2^37*x3^25*x4^15*z^25 - 4*x1^47*x2^38*x3^25*x4^15*z^25 - 6*x1^45*x2^40*x3^25*x4^15*z^25 - 2*x1^44*x2^41*x3^25*x4^15*z^25 - 5*x1^50*x2^34*x3^26*x4^15*z^25 - 3*x1^49*x2^35*x3^26*x4^15*z^25 - 2*x1^48*x2^36*x3^26*x4^15*z^25 + 2*x1^46*x2^38*x3^26*x4^15*z^25 + 2*x1^45*x2^39*x3^26*x4^15*z^25 + 6*x1^44*x2^40*x3^26*x4^15*z^25 + 5*x1^49*x2^34*x3^27*x4^15*z^25 + 2*x1^48*x2^35*x3^27*x4^15*z^25 + 4*x1^47*x2^36*x3^27*x4^15*z^25 - 4*x1^46*x2^37*x3^27*x4^15*z^25 - 6*x1^44*x2^39*x3^27*x4^15*z^25 - 2*x1^43*x2^40*x3^27*x4^15*z^25 - x1^49*x2^33*x3^28*x4^15*z^25 - 3*x1^48*x2^34*x3^28*x4^15*z^25 - 3*x1^47*x2^35*x3^28*x4^15*z^25 - x1^46*x2^36*x3^28*x4^15*z^25 + x1^45*x2^37*x3^28*x4^15*z^25 + 2*x1^44*x2^38*x3^28*x4^15*z^25 + 6*x1^43*x2^39*x3^28*x4^15*z^25 + x1^48*x2^33*x3^29*x4^15*z^25 + 2*x1^47*x2^34*x3^29*x4^15*z^25 + 3*x1^46*x2^35*x3^29*x4^15*z^25 - 2*x1^45*x2^36*x3^29*x4^15*z^25 + x1^44*x2^37*x3^29*x4^15*z^25 - 6*x1^43*x2^38*x3^29*x4^15*z^25 - 2*x1^42*x2^39*x3^29*x4^15*z^25 - 2*x1^46*x2^34*x3^30*x4^15*z^25 - x1^45*x2^35*x3^30*x4^15*z^25 + x1^43*x2^37*x3^30*x4^15*z^25 + 6*x1^42*x2^38*x3^30*x4^15*z^25 + x1^45*x2^34*x3^31*x4^15*z^25 - x1^44*x2^35*x3^31*x4^15*z^25 + 2*x1^43*x2^36*x3^31*x4^15*z^25 - 4*x1^42*x2^37*x3^31*x4^15*z^25 - x1^41*x2^38*x3^31*x4^15*z^25 + x1^43*x2^35*x3^32*x4^15*z^25 - x1^42*x2^36*x3^32*x4^15*z^25 + 2*x1^41*x2^37*x3^32*x4^15*z^25 - 2*x1^50*x2^40*x3^19*x4^16*z^25 - x1^49*x2^41*x3^19*x4^16*z^25 + 3*x1^49*x2^40*x3^20*x4^16*z^25 + x1^47*x2^42*x3^20*x4^16*z^25 + x1^51*x2^37*x3^21*x4^16*z^25 + x1^50*x2^38*x3^21*x4^16*z^25 - 3*x1^49*x2^39*x3^21*x4^16*z^25 - 2*x1^48*x2^40*x3^21*x4^16*z^25 - x1^47*x2^41*x3^21*x4^16*z^25 - 3*x1^46*x2^42*x3^21*x4^16*z^25 - 3*x1^51*x2^36*x3^22*x4^16*z^25 + x1^50*x2^37*x3^22*x4^16*z^25 + 3*x1^48*x2^39*x3^22*x4^16*z^25 + 2*x1^47*x2^40*x3^22*x4^16*z^25 + 5*x1^46*x2^41*x3^22*x4^16*z^25 + x1^45*x2^42*x3^22*x4^16*z^25 + 2*x1^51*x2^35*x3^23*x4^16*z^25 + 2*x1^50*x2^36*x3^23*x4^16*z^25 - 2*x1^48*x2^38*x3^23*x4^16*z^25 - 2*x1^47*x2^39*x3^23*x4^16*z^25 - 2*x1^46*x2^40*x3^23*x4^16*z^25 - 6*x1^45*x2^41*x3^23*x4^16*z^25 - x1^51*x2^34*x3^24*x4^16*z^25 - 5*x1^50*x2^35*x3^24*x4^16*z^25 + x1^49*x2^36*x3^24*x4^16*z^25 - x1^48*x2^37*x3^24*x4^16*z^25 + 4*x1^47*x2^38*x3^24*x4^16*z^25 + 6*x1^45*x2^40*x3^24*x4^16*z^25 + 2*x1^44*x2^41*x3^24*x4^16*z^25 + 3*x1^50*x2^34*x3^25*x4^16*z^25 + 2*x1^49*x2^35*x3^25*x4^16*z^25 + 2*x1^48*x2^36*x3^25*x4^16*z^25 - x1^47*x2^37*x3^25*x4^16*z^25 - 2*x1^46*x2^38*x3^25*x4^16*z^25 - 2*x1^45*x2^39*x3^25*x4^16*z^25 - 6*x1^44*x2^40*x3^25*x4^16*z^25 - x1^50*x2^33*x3^26*x4^16*z^25 - 4*x1^49*x2^34*x3^26*x4^16*z^25 - x1^48*x2^35*x3^26*x4^16*z^25 - 4*x1^47*x2^36*x3^26*x4^16*z^25 + 4*x1^46*x2^37*x3^26*x4^16*z^25 + 6*x1^44*x2^39*x3^26*x4^16*z^25 + 2*x1^43*x2^40*x3^26*x4^16*z^25 + x1^49*x2^33*x3^27*x4^16*z^25 + x1^48*x2^34*x3^27*x4^16*z^25 + x1^47*x2^35*x3^27*x4^16*z^25 - 2*x1^45*x2^37*x3^27*x4^16*z^25 - 2*x1^44*x2^38*x3^27*x4^16*z^25 - 6*x1^43*x2^39*x3^27*x4^16*z^25 - x1^48*x2^33*x3^28*x4^16*z^25 - x1^46*x2^35*x3^28*x4^16*z^25 + 4*x1^45*x2^36*x3^28*x4^16*z^25 + 6*x1^43*x2^38*x3^28*x4^16*z^25 + 2*x1^42*x2^39*x3^28*x4^16*z^25 + x1^47*x2^33*x3^29*x4^16*z^25 - x1^44*x2^36*x3^29*x4^16*z^25 - 6*x1^42*x2^38*x3^29*x4^16*z^25 + x1^45*x2^34*x3^30*x4^16*z^25 + 2*x1^44*x2^35*x3^30*x4^16*z^25 - x1^43*x2^36*x3^30*x4^16*z^25 + 3*x1^42*x2^37*x3^30*x4^16*z^25 + 2*x1^41*x2^38*x3^30*x4^16*z^25 - x1^43*x2^35*x3^31*x4^16*z^25 - 3*x1^41*x2^37*x3^31*x4^16*z^25 + x1^41*x2^36*x3^32*x4^16*z^25 - x1^40*x2^36*x3^33*x4^16*z^25 - 2*x1^48*x2^39*x3^21*x4^17*z^25 + x1^48*x2^38*x3^22*x4^17*z^25 + 2*x1^45*x2^41*x3^22*x4^17*z^25 - x1^47*x2^38*x3^23*x4^17*z^25 - x1^46*x2^39*x3^23*x4^17*z^25 - x1^45*x2^40*x3^23*x4^17*z^25 - x1^50*x2^34*x3^24*x4^17*z^25 - x1^49*x2^35*x3^24*x4^17*z^25 + x1^48*x2^36*x3^24*x4^17*z^25 + x1^47*x2^37*x3^24*x4^17*z^25 + 2*x1^44*x2^40*x3^24*x4^17*z^25 + x1^49*x2^34*x3^25*x4^17*z^25 - x1^48*x2^35*x3^25*x4^17*z^25 + x1^47*x2^36*x3^25*x4^17*z^25 - x1^46*x2^37*x3^25*x4^17*z^25 - 2*x1^44*x2^39*x3^25*x4^17*z^25 - x1^43*x2^40*x3^25*x4^17*z^25 - x1^48*x2^34*x3^26*x4^17*z^25 + x1^46*x2^36*x3^26*x4^17*z^25 + x1^45*x2^37*x3^26*x4^17*z^25 + x1^44*x2^38*x3^26*x4^17*z^25 + 2*x1^43*x2^39*x3^26*x4^17*z^25 + x1^47*x2^34*x3^27*x4^17*z^25 - x1^45*x2^36*x3^27*x4^17*z^25 - 2*x1^43*x2^38*x3^27*x4^17*z^25 - x1^42*x2^39*x3^27*x4^17*z^25 - x1^46*x2^34*x3^28*x4^17*z^25 + x1^44*x2^36*x3^28*x4^17*z^25 + x1^43*x2^37*x3^28*x4^17*z^25 + 2*x1^42*x2^38*x3^28*x4^17*z^25 - x1^44*x2^35*x3^29*x4^17*z^25 - 2*x1^42*x2^37*x3^29*x4^17*z^25 - x1^42*x2^36*x3^30*x4^17*z^25 + 2*x1^41*x2^37*x3^30*x4^17*z^25 - x1^40*x2^37*x3^31*x4^17*z^25 - x1^56*x2^42*x3^21*x4*z^24 - x1^55*x2^41*x3^23*x4*z^24 - x1^55*x2^42*x3^21*x4^2*z^24 + 2*x1^55*x2^41*x3^22*x4^2*z^24 - x1^53*x2^43*x3^22*x4^2*z^24 - 3*x1^54*x2^41*x3^23*x4^2*z^24 + x1^53*x2^42*x3^23*x4^2*z^24 + x1^54*x2^40*x3^24*x4^2*z^24 + x1^53*x2^41*x3^24*x4^2*z^24 + x1^52*x2^42*x3^24*x4^2*z^24 + x1^51*x2^43*x3^24*x4^2*z^24 - 2*x1^53*x2^40*x3^25*x4^2*z^24 - x1^55*x2^41*x3^21*x4^3*z^24 + 2*x1^54*x2^41*x3^22*x4^3*z^24 + x1^53*x2^42*x3^22*x4^3*z^24 + x1^52*x2^43*x3^22*x4^3*z^24 + 2*x1^55*x2^39*x3^23*x4^3*z^24 - 2*x1^53*x2^41*x3^23*x4^3*z^24 + x1^52*x2^42*x3^23*x4^3*z^24 + 3*x1^53*x2^40*x3^24*x4^3*z^24 - x1^52*x2^41*x3^24*x4^3*z^24 - x1^52*x2^40*x3^25*x4^3*z^24 - x1^51*x2^41*x3^25*x4^3*z^24 - x1^50*x2^42*x3^25*x4^3*z^24 + 2*x1^52*x2^39*x3^26*x4^3*z^24 + x1^49*x2^42*x3^26*x4^3*z^24 - x1^56*x2^40*x3^20*x4^4*z^24 + 2*x1^56*x2^39*x3^21*x4^4*z^24 + x1^55*x2^40*x3^21*x4^4*z^24 - x1^54*x2^41*x3^21*x4^4*z^24 - 4*x1^55*x2^39*x3^22*x4^4*z^24 + 2*x1^54*x2^40*x3^22*x4^4*z^24 - 2*x1^52*x2^42*x3^22*x4^4*z^24 + x1^51*x2^43*x3^22*x4^4*z^24 + 2*x1^55*x2^38*x3^23*x4^4*z^24 + 3*x1^54*x2^39*x3^23*x4^4*z^24 - 2*x1^53*x2^40*x3^23*x4^4*z^24 + 2*x1^52*x2^41*x3^23*x4^4*z^24 + x1^50*x2^43*x3^23*x4^4*z^24 - 4*x1^54*x2^38*x3^24*x4^4*z^24 + x1^52*x2^40*x3^24*x4^4*z^24 - 2*x1^51*x2^41*x3^24*x4^4*z^24 - x1^50*x2^42*x3^24*x4^4*z^24 + x1^49*x2^43*x3^24*x4^4*z^24 + x1^48*x2^44*x3^24*x4^4*z^24 + x1^54*x2^37*x3^25*x4^4*z^24 + 2*x1^53*x2^38*x3^25*x4^4*z^24 - x1^52*x2^39*x3^25*x4^4*z^24 + 3*x1^51*x2^40*x3^25*x4^4*z^24 - x1^53*x2^37*x3^26*x4^4*z^24 + x1^52*x2^38*x3^26*x4^4*z^24 + x1^51*x2^39*x3^26*x4^4*z^24 - x1^50*x2^40*x3^26*x4^4*z^24 - x1^51*x2^38*x3^27*x4^4*z^24 + x1^50*x2^39*x3^27*x4^4*z^24 - x1^49*x2^40*x3^27*x4^4*z^24 - x1^48*x2^41*x3^27*x4^4*z^24 + 2*x1^56*x2^40*x3^19*x4^5*z^24 - x1^56*x2^39*x3^20*x4^5*z^24 - 2*x1^55*x2^40*x3^20*x4^5*z^24 + 2*x1^54*x2^41*x3^20*x4^5*z^24 + 5*x1^55*x2^39*x3^21*x4^5*z^24 + x1^54*x2^40*x3^21*x4^5*z^24 - 2*x1^55*x2^38*x3^22*x4^5*z^24 - 5*x1^54*x2^39*x3^22*x4^5*z^24 + x1^53*x2^40*x3^22*x4^5*z^24 - x1^52*x2^41*x3^22*x4^5*z^24 + 6*x1^54*x2^38*x3^23*x4^5*z^24 + x1^52*x2^40*x3^23*x4^5*z^24 + 3*x1^51*x2^41*x3^23*x4^5*z^24 - x1^50*x2^42*x3^23*x4^5*z^24 - 2*x1^49*x2^43*x3^23*x4^5*z^24 - 2*x1^54*x2^37*x3^24*x4^5*z^24 - 5*x1^53*x2^38*x3^24*x4^5*z^24 + 2*x1^52*x2^39*x3^24*x4^5*z^24 - 4*x1^51*x2^40*x3^24*x4^5*z^24 + x1^50*x2^41*x3^24*x4^5*z^24 + x1^49*x2^42*x3^24*x4^5*z^24 + 2*x1^48*x2^43*x3^24*x4^5*z^24 + 4*x1^53*x2^37*x3^25*x4^5*z^24 + x1^52*x2^38*x3^25*x4^5*z^24 + x1^50*x2^40*x3^25*x4^5*z^24 - x1^49*x2^41*x3^25*x4^5*z^24 - 2*x1^48*x2^42*x3^25*x4^5*z^24 + x1^46*x2^44*x3^25*x4^5*z^24 - 2*x1^53*x2^36*x3^26*x4^5*z^24 - 2*x1^52*x2^37*x3^26*x4^5*z^24 - 3*x1^50*x2^39*x3^26*x4^5*z^24 + x1^49*x2^40*x3^26*x4^5*z^24 - x1^48*x2^41*x3^26*x4^5*z^24 + 2*x1^52*x2^36*x3^27*x4^5*z^24 - x1^51*x2^37*x3^27*x4^5*z^24 + x1^50*x2^38*x3^27*x4^5*z^24 + 2*x1^49*x2^39*x3^27*x4^5*z^24 - x1^48*x2^40*x3^27*x4^5*z^24 - x1^52*x2^35*x3^28*x4^5*z^24 - x1^51*x2^36*x3^28*x4^5*z^24 + x1^50*x2^37*x3^28*x4^5*z^24 - x1^49*x2^38*x3^28*x4^5*z^24 + 2*x1^48*x2^39*x3^28*x4^5*z^24 - x1^50*x2^36*x3^29*x4^5*z^24 - x1^49*x2^37*x3^29*x4^5*z^24 + x1^48*x2^38*x3^29*x4^5*z^24 - x1^47*x2^39*x3^29*x4^5*z^24 - 2*x1^55*x2^39*x3^20*x4^6*z^24 - x1^54*x2^40*x3^20*x4^6*z^24 - x1^53*x2^41*x3^20*x4^6*z^24 + x1^55*x2^38*x3^21*x4^6*z^24 + 2*x1^54*x2^39*x3^21*x4^6*z^24 - x1^53*x2^40*x3^21*x4^6*z^24 - x1^51*x2^42*x3^21*x4^6*z^24 - 2*x1^54*x2^38*x3^22*x4^6*z^24 + 3*x1^53*x2^39*x3^22*x4^6*z^24 + x1^52*x2^40*x3^22*x4^6*z^24 + x1^50*x2^42*x3^22*x4^6*z^24 + x1^49*x2^43*x3^22*x4^6*z^24 + 2*x1^53*x2^38*x3^23*x4^6*z^24 - 5*x1^52*x2^39*x3^23*x4^6*z^24 + x1^51*x2^40*x3^23*x4^6*z^24 - x1^50*x2^41*x3^23*x4^6*z^24 - x1^49*x2^42*x3^23*x4^6*z^24 - 2*x1^48*x2^43*x3^23*x4^6*z^24 - 3*x1^53*x2^37*x3^24*x4^6*z^24 - x1^52*x2^38*x3^24*x4^6*z^24 + 4*x1^51*x2^39*x3^24*x4^6*z^24 - 2*x1^50*x2^40*x3^24*x4^6*z^24 - x1^48*x2^42*x3^24*x4^6*z^24 + x1^53*x2^36*x3^25*x4^6*z^24 + x1^52*x2^37*x3^25*x4^6*z^24 - 4*x1^51*x2^38*x3^25*x4^6*z^24 + 2*x1^50*x2^39*x3^25*x4^6*z^24 - x1^49*x2^40*x3^25*x4^6*z^24 - x1^48*x2^41*x3^25*x4^6*z^24 - 2*x1^47*x2^42*x3^25*x4^6*z^24 - x1^46*x2^43*x3^25*x4^6*z^24 - x1^52*x2^36*x3^26*x4^6*z^24 - x1^51*x2^37*x3^26*x4^6*z^24 - x1^50*x2^38*x3^26*x4^6*z^24 + 4*x1^48*x2^40*x3^26*x4^6*z^24 + x1^47*x2^41*x3^26*x4^6*z^24 - x1^46*x2^42*x3^26*x4^6*z^24 + 2*x1^52*x2^35*x3^27*x4^6*z^24 + x1^51*x2^36*x3^27*x4^6*z^24 - 2*x1^50*x2^37*x3^27*x4^6*z^24 + x1^49*x2^38*x3^27*x4^6*z^24 - 2*x1^48*x2^39*x3^27*x4^6*z^24 + x1^47*x2^40*x3^27*x4^6*z^24 + x1^44*x2^43*x3^27*x4^6*z^24 - x1^51*x2^35*x3^28*x4^6*z^24 + x1^50*x2^36*x3^28*x4^6*z^24 - x1^49*x2^37*x3^28*x4^6*z^24 - 2*x1^48*x2^38*x3^28*x4^6*z^24 + 2*x1^47*x2^39*x3^28*x4^6*z^24 - x1^47*x2^38*x3^29*x4^6*z^24 + x1^46*x2^39*x3^29*x4^6*z^24 + x1^48*x2^36*x3^30*x4^6*z^24 - x1^47*x2^37*x3^30*x4^6*z^24 + x1^46*x2^38*x3^30*x4^6*z^24 + x1^55*x2^41*x3^17*x4^7*z^24 - 2*x1^54*x2^41*x3^18*x4^7*z^24 + x1^53*x2^42*x3^18*x4^7*z^24 + x1^55*x2^39*x3^19*x4^7*z^24 + 2*x1^53*x2^41*x3^19*x4^7*z^24 + x1^51*x2^43*x3^19*x4^7*z^24 - x1^54*x2^39*x3^20*x4^7*z^24 - 2*x1^53*x2^40*x3^20*x4^7*z^24 - 2*x1^52*x2^41*x3^20*x4^7*z^24 - x1^51*x2^42*x3^20*x4^7*z^24 + x1^49*x2^44*x3^20*x4^7*z^24 + x1^54*x2^38*x3^21*x4^7*z^24 + x1^53*x2^39*x3^21*x4^7*z^24 + 3*x1^52*x2^40*x3^21*x4^7*z^24 + x1^51*x2^41*x3^21*x4^7*z^24 + x1^50*x2^42*x3^21*x4^7*z^24 - x1^48*x2^44*x3^21*x4^7*z^24 - x1^53*x2^38*x3^22*x4^7*z^24 - 2*x1^52*x2^39*x3^22*x4^7*z^24 - 2*x1^51*x2^40*x3^22*x4^7*z^24 - x1^50*x2^41*x3^22*x4^7*z^24 - 2*x1^49*x2^42*x3^22*x4^7*z^24 - x1^48*x2^43*x3^22*x4^7*z^24 + x1^47*x2^44*x3^22*x4^7*z^24 + 2*x1^51*x2^39*x3^23*x4^7*z^24 + x1^50*x2^40*x3^23*x4^7*z^24 + x1^49*x2^41*x3^23*x4^7*z^24 - x1^48*x2^42*x3^23*x4^7*z^24 + x1^47*x2^43*x3^23*x4^7*z^24 - x1^46*x2^44*x3^23*x4^7*z^24 - x1^50*x2^39*x3^24*x4^7*z^24 + x1^49*x2^40*x3^24*x4^7*z^24 - x1^47*x2^42*x3^24*x4^7*z^24 - x1^46*x2^43*x3^24*x4^7*z^24 + x1^52*x2^36*x3^25*x4^7*z^24 + x1^51*x2^37*x3^25*x4^7*z^24 - x1^50*x2^38*x3^25*x4^7*z^24 + 2*x1^48*x2^40*x3^25*x4^7*z^24 - x1^46*x2^42*x3^25*x4^7*z^24 - x1^45*x2^43*x3^25*x4^7*z^24 - x1^49*x2^38*x3^26*x4^7*z^24 + x1^48*x2^39*x3^26*x4^7*z^24 - x1^47*x2^40*x3^26*x4^7*z^24 + 2*x1^46*x2^41*x3^26*x4^7*z^24 + x1^51*x2^35*x3^27*x4^7*z^24 + x1^48*x2^38*x3^27*x4^7*z^24 - x1^47*x2^39*x3^27*x4^7*z^24 - x1^44*x2^42*x3^27*x4^7*z^24 - 2*x1^48*x2^37*x3^28*x4^7*z^24 + 2*x1^50*x2^34*x3^29*x4^7*z^24 + x1^48*x2^36*x3^29*x4^7*z^24 + x1^47*x2^37*x3^29*x4^7*z^24 - x1^46*x2^38*x3^29*x4^7*z^24 - x1^47*x2^36*x3^30*x4^7*z^24 - x1^45*x2^38*x3^30*x4^7*z^24 - x1^47*x2^35*x3^31*x4^7*z^24 - x1^46*x2^36*x3^31*x4^7*z^24 + x1^54*x2^41*x3^17*x4^8*z^24 - x1^54*x2^40*x3^18*x4^8*z^24 - x1^53*x2^41*x3^18*x4^8*z^24 + x1^52*x2^42*x3^18*x4^8*z^24 - x1^55*x2^38*x3^19*x4^8*z^24 + 4*x1^53*x2^40*x3^19*x4^8*z^24 - x1^52*x2^41*x3^19*x4^8*z^24 - x1^53*x2^39*x3^20*x4^8*z^24 - 4*x1^52*x2^40*x3^20*x4^8*z^24 + x1^51*x2^41*x3^20*x4^8*z^24 - 3*x1^50*x2^42*x3^20*x4^8*z^24 - x1^49*x2^43*x3^20*x4^8*z^24 + x1^54*x2^37*x3^21*x4^8*z^24 + 3*x1^52*x2^39*x3^21*x4^8*z^24 + 2*x1^51*x2^40*x3^21*x4^8*z^24 + 2*x1^50*x2^41*x3^21*x4^8*z^24 - x1^48*x2^43*x3^21*x4^8*z^24 - x1^47*x2^44*x3^21*x4^8*z^24 - 2*x1^53*x2^37*x3^22*x4^8*z^24 - x1^52*x2^38*x3^22*x4^8*z^24 - 4*x1^51*x2^39*x3^22*x4^8*z^24 - 2*x1^49*x2^41*x3^22*x4^8*z^24 + 4*x1^48*x2^42*x3^22*x4^8*z^24 + x1^47*x2^43*x3^22*x4^8*z^24 + 2*x1^46*x2^44*x3^22*x4^8*z^24 + x1^53*x2^36*x3^23*x4^8*z^24 + 2*x1^52*x2^37*x3^23*x4^8*z^24 + 4*x1^51*x2^38*x3^23*x4^8*z^24 + x1^50*x2^39*x3^23*x4^8*z^24 + x1^49*x2^40*x3^23*x4^8*z^24 + 2*x1^48*x2^41*x3^23*x4^8*z^24 - x1^47*x2^42*x3^23*x4^8*z^24 - x1^46*x2^43*x3^23*x4^8*z^24 - 2*x1^45*x2^44*x3^23*x4^8*z^24 - 2*x1^52*x2^36*x3^24*x4^8*z^24 - x1^51*x2^37*x3^24*x4^8*z^24 - 4*x1^50*x2^38*x3^24*x4^8*z^24 - x1^49*x2^39*x3^24*x4^8*z^24 - 3*x1^48*x2^40*x3^24*x4^8*z^24 + x1^47*x2^41*x3^24*x4^8*z^24 + x1^46*x2^42*x3^24*x4^8*z^24 + 4*x1^45*x2^43*x3^24*x4^8*z^24 + x1^44*x2^44*x3^24*x4^8*z^24 + 2*x1^51*x2^36*x3^25*x4^8*z^24 + 4*x1^50*x2^37*x3^25*x4^8*z^24 + 4*x1^49*x2^38*x3^25*x4^8*z^24 - 2*x1^46*x2^41*x3^25*x4^8*z^24 - 2*x1^45*x2^42*x3^25*x4^8*z^24 - 2*x1^44*x2^43*x3^25*x4^8*z^24 - 2*x1^51*x2^35*x3^26*x4^8*z^24 - 2*x1^50*x2^36*x3^26*x4^8*z^24 - 3*x1^49*x2^37*x3^26*x4^8*z^24 - x1^47*x2^39*x3^26*x4^8*z^24 + x1^46*x2^40*x3^26*x4^8*z^24 + x1^45*x2^41*x3^26*x4^8*z^24 + 3*x1^44*x2^42*x3^26*x4^8*z^24 + x1^51*x2^34*x3^27*x4^8*z^24 + x1^50*x2^35*x3^27*x4^8*z^24 + 3*x1^49*x2^36*x3^27*x4^8*z^24 + 3*x1^48*x2^37*x3^27*x4^8*z^24 + 2*x1^47*x2^38*x3^27*x4^8*z^24 + 2*x1^46*x2^39*x3^27*x4^8*z^24 - 2*x1^45*x2^40*x3^27*x4^8*z^24 - x1^44*x2^41*x3^27*x4^8*z^24 - x1^43*x2^42*x3^27*x4^8*z^24 - 3*x1^50*x2^34*x3^28*x4^8*z^24 - 3*x1^48*x2^36*x3^28*x4^8*z^24 - 3*x1^47*x2^37*x3^28*x4^8*z^24 - x1^46*x2^38*x3^28*x4^8*z^24 + x1^45*x2^39*x3^28*x4^8*z^24 + 2*x1^43*x2^41*x3^28*x4^8*z^24 + x1^49*x2^34*x3^29*x4^8*z^24 + x1^47*x2^36*x3^29*x4^8*z^24 + x1^46*x2^37*x3^29*x4^8*z^24 + x1^45*x2^38*x3^29*x4^8*z^24 - x1^44*x2^39*x3^29*x4^8*z^24 - x1^42*x2^41*x3^29*x4^8*z^24 - x1^49*x2^33*x3^30*x4^8*z^24 - x1^45*x2^37*x3^30*x4^8*z^24 + x1^47*x2^34*x3^31*x4^8*z^24 + x1^46*x2^35*x3^31*x4^8*z^24 + x1^45*x2^36*x3^31*x4^8*z^24 + x1^44*x2^37*x3^31*x4^8*z^24 + x1^45*x2^35*x3^32*x4^8*z^24 - x1^44*x2^36*x3^32*x4^8*z^24 + x1^55*x2^39*x3^17*x4^9*z^24 + x1^55*x2^38*x3^18*x4^9*z^24 + x1^54*x2^39*x3^18*x4^9*z^24 - 2*x1^53*x2^40*x3^18*x4^9*z^24 + x1^52*x2^41*x3^18*x4^9*z^24 - x1^53*x2^39*x3^19*x4^9*z^24 + x1^52*x2^40*x3^19*x4^9*z^24 - x1^51*x2^41*x3^19*x4^9*z^24 - x1^49*x2^43*x3^19*x4^9*z^24 - x1^54*x2^37*x3^20*x4^9*z^24 - x1^52*x2^39*x3^20*x4^9*z^24 - x1^51*x2^40*x3^20*x4^9*z^24 + x1^49*x2^42*x3^20*x4^9*z^24 + 2*x1^48*x2^43*x3^20*x4^9*z^24 + 4*x1^53*x2^37*x3^21*x4^9*z^24 + 2*x1^52*x2^38*x3^21*x4^9*z^24 + 3*x1^51*x2^39*x3^21*x4^9*z^24 - x1^50*x2^40*x3^21*x4^9*z^24 - x1^49*x2^41*x3^21*x4^9*z^24 - 2*x1^48*x2^42*x3^21*x4^9*z^24 - 2*x1^47*x2^43*x3^21*x4^9*z^24 - x1^46*x2^44*x3^21*x4^9*z^24 - 2*x1^53*x2^36*x3^22*x4^9*z^24 - 4*x1^52*x2^37*x3^22*x4^9*z^24 - x1^51*x2^38*x3^22*x4^9*z^24 + 2*x1^49*x2^40*x3^22*x4^9*z^24 + x1^48*x2^41*x3^22*x4^9*z^24 + x1^47*x2^42*x3^22*x4^9*z^24 + 2*x1^46*x2^43*x3^22*x4^9*z^24 + x1^45*x2^44*x3^22*x4^9*z^24 + 4*x1^52*x2^36*x3^23*x4^9*z^24 + x1^51*x2^37*x3^23*x4^9*z^24 + 2*x1^50*x2^38*x3^23*x4^9*z^24 - 2*x1^47*x2^41*x3^23*x4^9*z^24 - 3*x1^45*x2^43*x3^23*x4^9*z^24 - x1^52*x2^35*x3^24*x4^9*z^24 - 4*x1^51*x2^36*x3^24*x4^9*z^24 - 2*x1^50*x2^37*x3^24*x4^9*z^24 - 4*x1^49*x2^38*x3^24*x4^9*z^24 + x1^47*x2^40*x3^24*x4^9*z^24 + x1^46*x2^41*x3^24*x4^9*z^24 + 2*x1^44*x2^43*x3^24*x4^9*z^24 + 4*x1^51*x2^35*x3^25*x4^9*z^24 + 2*x1^50*x2^36*x3^25*x4^9*z^24 + 3*x1^49*x2^37*x3^25*x4^9*z^24 + x1^48*x2^38*x3^25*x4^9*z^24 - x1^46*x2^40*x3^25*x4^9*z^24 - 2*x1^45*x2^41*x3^25*x4^9*z^24 - 2*x1^44*x2^42*x3^25*x4^9*z^24 - x1^51*x2^34*x3^26*x4^9*z^24 - 4*x1^50*x2^35*x3^26*x4^9*z^24 - 2*x1^49*x2^36*x3^26*x4^9*z^24 - 4*x1^48*x2^37*x3^26*x4^9*z^24 - 2*x1^47*x2^38*x3^26*x4^9*z^24 - x1^46*x2^39*x3^26*x4^9*z^24 + 3*x1^45*x2^40*x3^26*x4^9*z^24 + x1^44*x2^41*x3^26*x4^9*z^24 + x1^43*x2^42*x3^26*x4^9*z^24 + 4*x1^50*x2^34*x3^27*x4^9*z^24 + x1^49*x2^35*x3^27*x4^9*z^24 + 2*x1^48*x2^36*x3^27*x4^9*z^24 + x1^47*x2^37*x3^27*x4^9*z^24 - x1^46*x2^38*x3^27*x4^9*z^24 - 3*x1^45*x2^39*x3^27*x4^9*z^24 - 2*x1^44*x2^40*x3^27*x4^9*z^24 - x1^43*x2^41*x3^27*x4^9*z^24 - x1^50*x2^33*x3^28*x4^9*z^24 - 4*x1^49*x2^34*x3^28*x4^9*z^24 + x1^48*x2^35*x3^28*x4^9*z^24 - x1^46*x2^37*x3^28*x4^9*z^24 - x1^45*x2^38*x3^28*x4^9*z^24 + x1^44*x2^39*x3^28*x4^9*z^24 - x1^43*x2^40*x3^28*x4^9*z^24 + x1^42*x2^41*x3^28*x4^9*z^24 + 3*x1^49*x2^33*x3^29*x4^9*z^24 - x1^48*x2^34*x3^29*x4^9*z^24 + 3*x1^46*x2^36*x3^29*x4^9*z^24 + x1^45*x2^37*x3^29*x4^9*z^24 - x1^43*x2^39*x3^29*x4^9*z^24 - 2*x1^48*x2^33*x3^30*x4^9*z^24 - 3*x1^46*x2^35*x3^30*x4^9*z^24 - x1^45*x2^36*x3^30*x4^9*z^24 + x1^43*x2^38*x3^30*x4^9*z^24 + x1^41*x2^40*x3^30*x4^9*z^24 - x1^46*x2^34*x3^31*x4^9*z^24 + x1^43*x2^36*x3^32*x4^9*z^24 + x1^55*x2^39*x3^16*x4^10*z^24 - x1^54*x2^40*x3^16*x4^10*z^24 - x1^54*x2^39*x3^17*x4^10*z^24 + x1^53*x2^39*x3^18*x4^10*z^24 - 2*x1^51*x2^41*x3^18*x4^10*z^24 - x1^50*x2^42*x3^18*x4^10*z^24 - x1^53*x2^38*x3^19*x4^10*z^24 + x1^52*x2^39*x3^19*x4^10*z^24 + x1^51*x2^40*x3^19*x4^10*z^24 + x1^50*x2^41*x3^19*x4^10*z^24 + 2*x1^49*x2^42*x3^19*x4^10*z^24 + x1^48*x2^43*x3^19*x4^10*z^24 + x1^53*x2^37*x3^20*x4^10*z^24 + x1^52*x2^38*x3^20*x4^10*z^24 - x1^50*x2^40*x3^20*x4^10*z^24 - x1^49*x2^41*x3^20*x4^10*z^24 - 2*x1^48*x2^42*x3^20*x4^10*z^24 - x1^47*x2^43*x3^20*x4^10*z^24 - x1^52*x2^37*x3^21*x4^10*z^24 + x1^50*x2^39*x3^21*x4^10*z^24 + 3*x1^49*x2^40*x3^21*x4^10*z^24 + x1^48*x2^41*x3^21*x4^10*z^24 + 3*x1^47*x2^42*x3^21*x4^10*z^24 + x1^46*x2^43*x3^21*x4^10*z^24 - x1^49*x2^39*x3^22*x4^10*z^24 - x1^47*x2^41*x3^22*x4^10*z^24 - 3*x1^46*x2^42*x3^22*x4^10*z^24 - x1^49*x2^38*x3^23*x4^10*z^24 + x1^48*x2^39*x3^23*x4^10*z^24 + 2*x1^46*x2^41*x3^23*x4^10*z^24 + x1^45*x2^42*x3^23*x4^10*z^24 - x1^47*x2^39*x3^24*x4^10*z^24 - x1^46*x2^40*x3^24*x4^10*z^24 - 2*x1^45*x2^41*x3^24*x4^10*z^24 - x1^43*x2^43*x3^24*x4^10*z^24 + x1^44*x2^41*x3^25*x4^10*z^24 - x1^50*x2^34*x3^26*x4^10*z^24 + x1^46*x2^38*x3^26*x4^10*z^24 - 2*x1^45*x2^39*x3^26*x4^10*z^24 - x1^43*x2^41*x3^26*x4^10*z^24 - x1^42*x2^42*x3^26*x4^10*z^24 + x1^50*x2^33*x3^27*x4^10*z^24 + x1^49*x2^34*x3^27*x4^10*z^24 - x1^48*x2^35*x3^27*x4^10*z^24 + x1^46*x2^37*x3^27*x4^10*z^24 + x1^45*x2^38*x3^27*x4^10*z^24 + x1^44*x2^39*x3^27*x4^10*z^24 - 3*x1^49*x2^33*x3^28*x4^10*z^24 + x1^48*x2^34*x3^28*x4^10*z^24 - 2*x1^46*x2^36*x3^28*x4^10*z^24 + x1^45*x2^37*x3^28*x4^10*z^24 - x1^44*x2^38*x3^28*x4^10*z^24 + x1^43*x2^39*x3^28*x4^10*z^24 - x1^42*x2^40*x3^28*x4^10*z^24 - x1^41*x2^41*x3^28*x4^10*z^24 + 4*x1^48*x2^33*x3^29*x4^10*z^24 + x1^47*x2^34*x3^29*x4^10*z^24 - x1^46*x2^35*x3^29*x4^10*z^24 + x1^45*x2^36*x3^29*x4^10*z^24 + x1^44*x2^37*x3^29*x4^10*z^24 + x1^42*x2^39*x3^29*x4^10*z^24 - x1^48*x2^32*x3^30*x4^10*z^24 - 2*x1^47*x2^33*x3^30*x4^10*z^24 - x1^44*x2^36*x3^30*x4^10*z^24 + 2*x1^42*x2^38*x3^30*x4^10*z^24 - 2*x1^41*x2^39*x3^30*x4^10*z^24 + x1^47*x2^32*x3^31*x4^10*z^24 + 2*x1^46*x2^33*x3^31*x4^10*z^24 + 2*x1^45*x2^34*x3^31*x4^10*z^24 + x1^44*x2^35*x3^31*x4^10*z^24 - x1^43*x2^36*x3^31*x4^10*z^24 + x1^41*x2^38*x3^31*x4^10*z^24 - x1^45*x2^33*x3^32*x4^10*z^24 - x1^44*x2^34*x3^32*x4^10*z^24 - x1^43*x2^35*x3^32*x4^10*z^24 - 2*x1^54*x2^38*x3^17*x4^11*z^24 + x1^53*x2^39*x3^17*x4^11*z^24 + 2*x1^52*x2^40*x3^17*x4^11*z^24 + 3*x1^53*x2^38*x3^18*x4^11*z^24 - 2*x1^52*x2^39*x3^18*x4^11*z^24 - x1^51*x2^40*x3^18*x4^11*z^24 - 3*x1^53*x2^37*x3^19*x4^11*z^24 - 3*x1^52*x2^38*x3^19*x4^11*z^24 - x1^51*x2^39*x3^19*x4^11*z^24 + x1^49*x2^41*x3^19*x4^11*z^24 + x1^53*x2^36*x3^20*x4^11*z^24 + 3*x1^52*x2^37*x3^20*x4^11*z^24 - 2*x1^51*x2^38*x3^20*x4^11*z^24 - 4*x1^49*x2^40*x3^20*x4^11*z^24 - 2*x1^48*x2^41*x3^20*x4^11*z^24 - 2*x1^47*x2^42*x3^20*x4^11*z^24 - x1^46*x2^43*x3^20*x4^11*z^24 - 4*x1^52*x2^36*x3^21*x4^11*z^24 + x1^50*x2^38*x3^21*x4^11*z^24 + 3*x1^48*x2^40*x3^21*x4^11*z^24 + 3*x1^47*x2^41*x3^21*x4^11*z^24 + 2*x1^46*x2^42*x3^21*x4^11*z^24 + x1^52*x2^35*x3^22*x4^11*z^24 + 4*x1^51*x2^36*x3^22*x4^11*z^24 - 2*x1^50*x2^37*x3^22*x4^11*z^24 + 2*x1^49*x2^38*x3^22*x4^11*z^24 - 4*x1^48*x2^39*x3^22*x4^11*z^24 - 3*x1^46*x2^41*x3^22*x4^11*z^24 + x1^45*x2^42*x3^22*x4^11*z^24 - 4*x1^51*x2^35*x3^23*x4^11*z^24 + x1^49*x2^37*x3^23*x4^11*z^24 + 2*x1^47*x2^39*x3^23*x4^11*z^24 - x1^46*x2^40*x3^23*x4^11*z^24 + 4*x1^45*x2^41*x3^23*x4^11*z^24 - 2*x1^44*x2^42*x3^23*x4^11*z^24 + x1^43*x2^43*x3^23*x4^11*z^24 + x1^51*x2^34*x3^24*x4^11*z^24 + 4*x1^50*x2^35*x3^24*x4^11*z^24 - 2*x1^49*x2^36*x3^24*x4^11*z^24 + 3*x1^48*x2^37*x3^24*x4^11*z^24 - 2*x1^47*x2^38*x3^24*x4^11*z^24 - 4*x1^45*x2^40*x3^24*x4^11*z^24 - 2*x1^44*x2^41*x3^24*x4^11*z^24 + 2*x1^43*x2^42*x3^24*x4^11*z^24 - 2*x1^50*x2^34*x3^25*x4^11*z^24 - 2*x1^49*x2^35*x3^25*x4^11*z^24 + 4*x1^46*x2^38*x3^25*x4^11*z^24 + x1^45*x2^39*x3^25*x4^11*z^24 + 4*x1^44*x2^40*x3^25*x4^11*z^24 - 2*x1^43*x2^41*x3^25*x4^11*z^24 - x1^42*x2^42*x3^25*x4^11*z^24 + x1^50*x2^33*x3^26*x4^11*z^24 + 2*x1^49*x2^34*x3^26*x4^11*z^24 + x1^47*x2^36*x3^26*x4^11*z^24 - 3*x1^46*x2^37*x3^26*x4^11*z^24 + x1^45*x2^38*x3^26*x4^11*z^24 - 2*x1^44*x2^39*x3^26*x4^11*z^24 + 2*x1^42*x2^41*x3^26*x4^11*z^24 + x1^48*x2^34*x3^27*x4^11*z^24 + x1^47*x2^35*x3^27*x4^11*z^24 - x1^46*x2^36*x3^27*x4^11*z^24 + 2*x1^45*x2^37*x3^27*x4^11*z^24 + 3*x1^44*x2^38*x3^27*x4^11*z^24 + 5*x1^43*x2^39*x3^27*x4^11*z^24 - x1^42*x2^40*x3^27*x4^11*z^24 - x1^41*x2^41*x3^27*x4^11*z^24 - 2*x1^47*x2^34*x3^28*x4^11*z^24 + x1^46*x2^35*x3^28*x4^11*z^24 - 3*x1^45*x2^36*x3^28*x4^11*z^24 - x1^44*x2^37*x3^28*x4^11*z^24 - 2*x1^43*x2^38*x3^28*x4^11*z^24 + x1^42*x2^39*x3^28*x4^11*z^24 + x1^41*x2^40*x3^28*x4^11*z^24 - x1^47*x2^33*x3^29*x4^11*z^24 + 2*x1^46*x2^34*x3^29*x4^11*z^24 + 2*x1^42*x2^38*x3^29*x4^11*z^24 + x1^40*x2^40*x3^29*x4^11*z^24 + x1^45*x2^34*x3^30*x4^11*z^24 - x1^43*x2^36*x3^30*x4^11*z^24 + x1^41*x2^38*x3^30*x4^11*z^24 - x1^44*x2^34*x3^31*x4^11*z^24 + x1^43*x2^35*x3^31*x4^11*z^24 - x1^41*x2^37*x3^31*x4^11*z^24 + x1^40*x2^38*x3^31*x4^11*z^24 + x1^54*x2^38*x3^16*x4^12*z^24 - x1^53*x2^38*x3^17*x4^12*z^24 - x1^52*x2^39*x3^17*x4^12*z^24 + 2*x1^53*x2^37*x3^18*x4^12*z^24 - x1^49*x2^41*x3^18*x4^12*z^24 + x1^48*x2^42*x3^18*x4^12*z^24 - 2*x1^52*x2^37*x3^19*x4^12*z^24 + 3*x1^51*x2^38*x3^19*x4^12*z^24 - x1^50*x2^39*x3^19*x4^12*z^24 + x1^49*x2^40*x3^19*x4^12*z^24 + x1^48*x2^41*x3^19*x4^12*z^24 + x1^47*x2^42*x3^19*x4^12*z^24 + 3*x1^52*x2^36*x3^20*x4^12*z^24 - x1^51*x2^37*x3^20*x4^12*z^24 - 5*x1^50*x2^38*x3^20*x4^12*z^24 + 2*x1^49*x2^39*x3^20*x4^12*z^24 - 3*x1^48*x2^40*x3^20*x4^12*z^24 - x1^46*x2^42*x3^20*x4^12*z^24 + x1^45*x2^43*x3^20*x4^12*z^24 - x1^51*x2^36*x3^21*x4^12*z^24 + 5*x1^50*x2^37*x3^21*x4^12*z^24 + 2*x1^49*x2^38*x3^21*x4^12*z^24 + 2*x1^48*x2^39*x3^21*x4^12*z^24 - x1^47*x2^40*x3^21*x4^12*z^24 - x1^45*x2^42*x3^21*x4^12*z^24 - x1^44*x2^43*x3^21*x4^12*z^24 + 2*x1^51*x2^35*x3^22*x4^12*z^24 - 2*x1^50*x2^36*x3^22*x4^12*z^24 - 6*x1^49*x2^37*x3^22*x4^12*z^24 - 4*x1^47*x2^39*x3^22*x4^12*z^24 + 2*x1^46*x2^40*x3^22*x4^12*z^24 - x1^45*x2^41*x3^22*x4^12*z^24 + 3*x1^44*x2^42*x3^22*x4^12*z^24 - 2*x1^50*x2^35*x3^23*x4^12*z^24 + 6*x1^49*x2^36*x3^23*x4^12*z^24 + 3*x1^47*x2^38*x3^23*x4^12*z^24 - 3*x1^44*x2^41*x3^23*x4^12*z^24 - 3*x1^43*x2^42*x3^23*x4^12*z^24 + 2*x1^50*x2^34*x3^24*x4^12*z^24 - x1^49*x2^35*x3^24*x4^12*z^24 - 5*x1^48*x2^36*x3^24*x4^12*z^24 - 5*x1^46*x2^38*x3^24*x4^12*z^24 + 3*x1^45*x2^39*x3^24*x4^12*z^24 - 2*x1^44*x2^40*x3^24*x4^12*z^24 + 6*x1^43*x2^41*x3^24*x4^12*z^24 - x1^50*x2^33*x3^25*x4^12*z^24 - 2*x1^49*x2^34*x3^25*x4^12*z^24 + 6*x1^48*x2^35*x3^25*x4^12*z^24 + x1^47*x2^36*x3^25*x4^12*z^24 + 3*x1^46*x2^37*x3^25*x4^12*z^24 - x1^43*x2^40*x3^25*x4^12*z^24 - 6*x1^42*x2^41*x3^25*x4^12*z^24 - x1^48*x2^34*x3^26*x4^12*z^24 - 6*x1^47*x2^35*x3^26*x4^12*z^24 - 5*x1^45*x2^37*x3^26*x4^12*z^24 + 3*x1^44*x2^38*x3^26*x4^12*z^24 - 2*x1^43*x2^39*x3^26*x4^12*z^24 + 6*x1^42*x2^40*x3^26*x4^12*z^24 + 2*x1^41*x2^41*x3^26*x4^12*z^24 + 3*x1^47*x2^34*x3^27*x4^12*z^24 + 2*x1^46*x2^35*x3^27*x4^12*z^24 + 2*x1^45*x2^36*x3^27*x4^12*z^24 - x1^44*x2^37*x3^27*x4^12*z^24 - x1^43*x2^38*x3^27*x4^12*z^24 - 2*x1^42*x2^39*x3^27*x4^12*z^24 - 6*x1^41*x2^40*x3^27*x4^12*z^24 + x1^47*x2^33*x3^28*x4^12*z^24 - 3*x1^46*x2^34*x3^28*x4^12*z^24 - 3*x1^44*x2^36*x3^28*x4^12*z^24 + x1^43*x2^37*x3^28*x4^12*z^24 - x1^42*x2^38*x3^28*x4^12*z^24 + 5*x1^41*x2^39*x3^28*x4^12*z^24 + 2*x1^40*x2^40*x3^28*x4^12*z^24 - x1^46*x2^33*x3^29*x4^12*z^24 + x1^44*x2^35*x3^29*x4^12*z^24 + x1^43*x2^36*x3^29*x4^12*z^24 - 2*x1^41*x2^38*x3^29*x4^12*z^24 - 3*x1^40*x2^39*x3^29*x4^12*z^24 + x1^45*x2^33*x3^30*x4^12*z^24 - x1^44*x2^34*x3^30*x4^12*z^24 - 2*x1^43*x2^35*x3^30*x4^12*z^24 - 2*x1^42*x2^36*x3^30*x4^12*z^24 + 4*x1^40*x2^38*x3^30*x4^12*z^24 - 2*x1^40*x2^37*x3^31*x4^12*z^24 - x1^39*x2^38*x3^31*x4^12*z^24 + x1^39*x2^37*x3^32*x4^12*z^24 - x1^53*x2^37*x3^17*x4^13*z^24 - x1^52*x2^38*x3^17*x4^13*z^24 + x1^51*x2^39*x3^17*x4^13*z^24 + 2*x1^52*x2^37*x3^18*x4^13*z^24 + x1^51*x2^38*x3^18*x4^13*z^24 - 2*x1^50*x2^39*x3^18*x4^13*z^24 + x1^49*x2^40*x3^18*x4^13*z^24 + x1^47*x2^42*x3^18*x4^13*z^24 - x1^52*x2^36*x3^19*x4^13*z^24 + 3*x1^50*x2^38*x3^19*x4^13*z^24 - x1^49*x2^39*x3^19*x4^13*z^24 - 2*x1^47*x2^41*x3^19*x4^13*z^24 - 2*x1^46*x2^42*x3^19*x4^13*z^24 + x1^51*x2^36*x3^20*x4^13*z^24 - x1^50*x2^37*x3^20*x4^13*z^24 - x1^49*x2^38*x3^20*x4^13*z^24 + x1^48*x2^39*x3^20*x4^13*z^24 + 2*x1^47*x2^40*x3^20*x4^13*z^24 + x1^46*x2^41*x3^20*x4^13*z^24 + x1^45*x2^42*x3^20*x4^13*z^24 - 2*x1^51*x2^35*x3^21*x4^13*z^24 + 2*x1^50*x2^36*x3^21*x4^13*z^24 + 5*x1^49*x2^37*x3^21*x4^13*z^24 - x1^48*x2^38*x3^21*x4^13*z^24 + x1^47*x2^39*x3^21*x4^13*z^24 - 2*x1^46*x2^40*x3^21*x4^13*z^24 - x1^45*x2^41*x3^21*x4^13*z^24 - 4*x1^44*x2^42*x3^21*x4^13*z^24 + x1^50*x2^35*x3^22*x4^13*z^24 - 4*x1^49*x2^36*x3^22*x4^13*z^24 - x1^47*x2^38*x3^22*x4^13*z^24 + 2*x1^45*x2^40*x3^22*x4^13*z^24 + 3*x1^44*x2^41*x3^22*x4^13*z^24 + 4*x1^43*x2^42*x3^22*x4^13*z^24 + 2*x1^49*x2^35*x3^23*x4^13*z^24 + 6*x1^48*x2^36*x3^23*x4^13*z^24 + 4*x1^46*x2^38*x3^23*x4^13*z^24 - 4*x1^45*x2^39*x3^23*x4^13*z^24 - 6*x1^43*x2^41*x3^23*x4^13*z^24 - x1^42*x2^42*x3^23*x4^13*z^24 - 6*x1^48*x2^35*x3^24*x4^13*z^24 - 2*x1^47*x2^36*x3^24*x4^13*z^24 - 2*x1^46*x2^37*x3^24*x4^13*z^24 + 2*x1^44*x2^39*x3^24*x4^13*z^24 + 2*x1^43*x2^40*x3^24*x4^13*z^24 + 6*x1^42*x2^41*x3^24*x4^13*z^24 + x1^49*x2^33*x3^25*x4^13*z^24 + 6*x1^47*x2^35*x3^25*x4^13*z^24 + 4*x1^45*x2^37*x3^25*x4^13*z^24 - 4*x1^44*x2^38*x3^25*x4^13*z^24 - 6*x1^42*x2^40*x3^25*x4^13*z^24 - 2*x1^41*x2^41*x3^25*x4^13*z^24 - x1^48*x2^33*x3^26*x4^13*z^24 - 3*x1^47*x2^34*x3^26*x4^13*z^24 - 4*x1^46*x2^35*x3^26*x4^13*z^24 - 2*x1^45*x2^36*x3^26*x4^13*z^24 + 2*x1^43*x2^38*x3^26*x4^13*z^24 + 2*x1^42*x2^39*x3^26*x4^13*z^24 + 6*x1^41*x2^40*x3^26*x4^13*z^24 + x1^48*x2^32*x3^27*x4^13*z^24 + x1^47*x2^33*x3^27*x4^13*z^24 + 4*x1^46*x2^34*x3^27*x4^13*z^24 + x1^45*x2^35*x3^27*x4^13*z^24 + 2*x1^44*x2^36*x3^27*x4^13*z^24 - 4*x1^43*x2^37*x3^27*x4^13*z^24 - 6*x1^41*x2^39*x3^27*x4^13*z^24 - 2*x1^40*x2^40*x3^27*x4^13*z^24 - x1^47*x2^32*x3^28*x4^13*z^24 - 2*x1^46*x2^33*x3^28*x4^13*z^24 - 2*x1^45*x2^34*x3^28*x4^13*z^24 - 2*x1^44*x2^35*x3^28*x4^13*z^24 + 2*x1^41*x2^38*x3^28*x4^13*z^24 + 6*x1^40*x2^39*x3^28*x4^13*z^24 + 2*x1^45*x2^33*x3^29*x4^13*z^24 + x1^44*x2^34*x3^29*x4^13*z^24 + 3*x1^43*x2^35*x3^29*x4^13*z^24 - 3*x1^42*x2^36*x3^29*x4^13*z^24 - 6*x1^40*x2^38*x3^29*x4^13*z^24 - 2*x1^39*x2^39*x3^29*x4^13*z^24 - x1^44*x2^33*x3^30*x4^13*z^24 - x1^43*x2^34*x3^30*x4^13*z^24 + 4*x1^39*x2^38*x3^30*x4^13*z^24 + x1^42*x2^34*x3^31*x4^13*z^24 + x1^41*x2^35*x3^31*x4^13*z^24 - 2*x1^39*x2^37*x3^31*x4^13*z^24 - x1^38*x2^38*x3^31*x4^13*z^24 + x1^51*x2^37*x3^18*x4^14*z^24 + x1^50*x2^38*x3^18*x4^14*z^24 - 2*x1^49*x2^39*x3^18*x4^14*z^24 + x1^47*x2^41*x3^18*x4^14*z^24 - 2*x1^51*x2^36*x3^19*x4^14*z^24 - x1^48*x2^39*x3^19*x4^14*z^24 - x1^47*x2^40*x3^19*x4^14*z^24 + x1^51*x2^35*x3^20*x4^14*z^24 + x1^50*x2^36*x3^20*x4^14*z^24 + x1^49*x2^37*x3^20*x4^14*z^24 + x1^48*x2^38*x3^20*x4^14*z^24 - x1^47*x2^39*x3^20*x4^14*z^24 - x1^46*x2^40*x3^20*x4^14*z^24 + x1^44*x2^42*x3^20*x4^14*z^24 - x1^51*x2^34*x3^21*x4^14*z^24 - 3*x1^50*x2^35*x3^21*x4^14*z^24 + x1^49*x2^36*x3^21*x4^14*z^24 + x1^48*x2^37*x3^21*x4^14*z^24 + 2*x1^47*x2^38*x3^21*x4^14*z^24 - x1^46*x2^39*x3^21*x4^14*z^24 - 2*x1^43*x2^42*x3^21*x4^14*z^24 + 2*x1^50*x2^34*x3^22*x4^14*z^24 - x1^49*x2^35*x3^22*x4^14*z^24 - x1^48*x2^36*x3^22*x4^14*z^24 + x1^47*x2^37*x3^22*x4^14*z^24 - x1^44*x2^40*x3^22*x4^14*z^24 + 2*x1^43*x2^41*x3^22*x4^14*z^24 + x1^42*x2^42*x3^22*x4^14*z^24 - x1^50*x2^33*x3^23*x4^14*z^24 - 3*x1^49*x2^34*x3^23*x4^14*z^24 - 2*x1^47*x2^36*x3^23*x4^14*z^24 + 2*x1^46*x2^37*x3^23*x4^14*z^24 + x1^44*x2^39*x3^23*x4^14*z^24 - x1^43*x2^40*x3^23*x4^14*z^24 - 2*x1^42*x2^41*x3^23*x4^14*z^24 + 2*x1^49*x2^33*x3^24*x4^14*z^24 - x1^48*x2^34*x3^24*x4^14*z^24 - 2*x1^47*x2^35*x3^24*x4^14*z^24 - x1^45*x2^37*x3^24*x4^14*z^24 + 2*x1^44*x2^38*x3^24*x4^14*z^24 - 2*x1^43*x2^39*x3^24*x4^14*z^24 + 2*x1^42*x2^40*x3^24*x4^14*z^24 - x1^49*x2^32*x3^25*x4^14*z^24 - 2*x1^48*x2^33*x3^25*x4^14*z^24 + 2*x1^47*x2^34*x3^25*x4^14*z^24 - 2*x1^46*x2^35*x3^25*x4^14*z^24 + x1^45*x2^36*x3^25*x4^14*z^24 + 2*x1^43*x2^38*x3^25*x4^14*z^24 + x1^42*x2^39*x3^25*x4^14*z^24 - 2*x1^41*x2^40*x3^25*x4^14*z^24 + 2*x1^48*x2^32*x3^26*x4^14*z^24 + 2*x1^47*x2^33*x3^26*x4^14*z^24 - x1^46*x2^34*x3^26*x4^14*z^24 - 3*x1^44*x2^36*x3^26*x4^14*z^24 - 2*x1^42*x2^38*x3^26*x4^14*z^24 + 2*x1^41*x2^39*x3^26*x4^14*z^24 + x1^40*x2^40*x3^26*x4^14*z^24 - 2*x1^47*x2^32*x3^27*x4^14*z^24 - x1^46*x2^33*x3^27*x4^14*z^24 + x1^45*x2^34*x3^27*x4^14*z^24 + 3*x1^44*x2^35*x3^27*x4^14*z^24 + x1^42*x2^37*x3^27*x4^14*z^24 - 2*x1^40*x2^39*x3^27*x4^14*z^24 + x1^46*x2^32*x3^28*x4^14*z^24 + x1^45*x2^33*x3^28*x4^14*z^24 - x1^43*x2^35*x3^28*x4^14*z^24 + x1^42*x2^36*x3^28*x4^14*z^24 - 2*x1^41*x2^37*x3^28*x4^14*z^24 + 2*x1^40*x2^38*x3^28*x4^14*z^24 + x1^39*x2^39*x3^28*x4^14*z^24 - x1^44*x2^33*x3^29*x4^14*z^24 + x1^43*x2^34*x3^29*x4^14*z^24 + x1^42*x2^35*x3^29*x4^14*z^24 + 2*x1^41*x2^36*x3^29*x4^14*z^24 - 2*x1^39*x2^38*x3^29*x4^14*z^24 - x1^41*x2^35*x3^30*x4^14*z^24 - 2*x1^40*x2^36*x3^30*x4^14*z^24 + x1^39*x2^37*x3^30*x4^14*z^24 - x1^41*x2^34*x3^31*x4^14*z^24 + x1^40*x2^35*x3^31*x4^14*z^24 - x1^38*x2^37*x3^31*x4^14*z^24 - x1^39*x2^35*x3^32*x4^14*z^24 + x1^49*x2^39*x3^17*x4^15*z^24 - 2*x1^48*x2^39*x3^18*x4^15*z^24 - x1^50*x2^36*x3^19*x4^15*z^24 - x1^49*x2^37*x3^19*x4^15*z^24 + 4*x1^48*x2^38*x3^19*x4^15*z^24 + 2*x1^47*x2^39*x3^19*x4^15*z^24 + 3*x1^50*x2^35*x3^20*x4^15*z^24 - x1^49*x2^36*x3^20*x4^15*z^24 - 3*x1^47*x2^38*x3^20*x4^15*z^24 - 2*x1^46*x2^39*x3^20*x4^15*z^24 - 3*x1^45*x2^40*x3^20*x4^15*z^24 - 2*x1^50*x2^34*x3^21*x4^15*z^24 - 2*x1^49*x2^35*x3^21*x4^15*z^24 + 2*x1^47*x2^37*x3^21*x4^15*z^24 + 2*x1^46*x2^38*x3^21*x4^15*z^24 + 4*x1^45*x2^39*x3^21*x4^15*z^24 + 3*x1^44*x2^40*x3^21*x4^15*z^24 + x1^50*x2^33*x3^22*x4^15*z^24 + 5*x1^49*x2^34*x3^22*x4^15*z^24 - x1^48*x2^35*x3^22*x4^15*z^24 + x1^47*x2^36*x3^22*x4^15*z^24 - 4*x1^46*x2^37*x3^22*x4^15*z^24 - 6*x1^44*x2^39*x3^22*x4^15*z^24 - x1^43*x2^40*x3^22*x4^15*z^24 - 4*x1^49*x2^33*x3^23*x4^15*z^24 - 2*x1^48*x2^34*x3^23*x4^15*z^24 - 2*x1^47*x2^35*x3^23*x4^15*z^24 + x1^46*x2^36*x3^23*x4^15*z^24 + 2*x1^45*x2^37*x3^23*x4^15*z^24 + 2*x1^44*x2^38*x3^23*x4^15*z^24 + 6*x1^43*x2^39*x3^23*x4^15*z^24 + x1^49*x2^32*x3^24*x4^15*z^24 + 7*x1^48*x2^33*x3^24*x4^15*z^24 + 4*x1^46*x2^35*x3^24*x4^15*z^24 - 4*x1^45*x2^36*x3^24*x4^15*z^24 - 6*x1^43*x2^38*x3^24*x4^15*z^24 - 2*x1^42*x2^39*x3^24*x4^15*z^24 - 4*x1^48*x2^32*x3^25*x4^15*z^24 - 3*x1^47*x2^33*x3^25*x4^15*z^24 - 2*x1^46*x2^34*x3^25*x4^15*z^24 + 2*x1^44*x2^36*x3^25*x4^15*z^24 + 2*x1^43*x2^37*x3^25*x4^15*z^24 + 6*x1^42*x2^38*x3^25*x4^15*z^24 + 4*x1^47*x2^32*x3^26*x4^15*z^24 + x1^46*x2^33*x3^26*x4^15*z^24 + 4*x1^45*x2^34*x3^26*x4^15*z^24 - 3*x1^44*x2^35*x3^26*x4^15*z^24 - 6*x1^42*x2^37*x3^26*x4^15*z^24 - 2*x1^41*x2^38*x3^26*x4^15*z^24 - x1^46*x2^32*x3^27*x4^15*z^24 - x1^45*x2^33*x3^27*x4^15*z^24 - x1^44*x2^34*x3^27*x4^15*z^24 + 2*x1^42*x2^36*x3^27*x4^15*z^24 + 6*x1^41*x2^37*x3^27*x4^15*z^24 + x1^45*x2^32*x3^28*x4^15*z^24 + 2*x1^44*x2^33*x3^28*x4^15*z^24 - x1^43*x2^34*x3^28*x4^15*z^24 + 2*x1^42*x2^35*x3^28*x4^15*z^24 - 5*x1^41*x2^36*x3^28*x4^15*z^24 - 2*x1^40*x2^37*x3^28*x4^15*z^24 - x1^44*x2^32*x3^29*x4^15*z^24 - x1^42*x2^34*x3^29*x4^15*z^24 + 5*x1^40*x2^36*x3^29*x4^15*z^24 - x1^42*x2^33*x3^30*x4^15*z^24 + 2*x1^41*x2^34*x3^30*x4^15*z^24 - 2*x1^40*x2^35*x3^30*x4^15*z^24 - 2*x1^39*x2^36*x3^30*x4^15*z^24 - x1^40*x2^34*x3^31*x4^15*z^24 + 2*x1^39*x2^35*x3^31*x4^15*z^24 - x1^48*x2^38*x3^18*x4^16*z^24 + 3*x1^47*x2^38*x3^19*x4^16*z^24 + x1^46*x2^39*x3^19*x4^16*z^24 + x1^45*x2^40*x3^19*x4^16*z^24 - 2*x1^47*x2^37*x3^20*x4^16*z^24 - 2*x1^46*x2^38*x3^20*x4^16*z^24 - 2*x1^44*x2^40*x3^20*x4^16*z^24 - x1^49*x2^34*x3^21*x4^16*z^24 + x1^48*x2^35*x3^21*x4^16*z^24 - x1^47*x2^36*x3^21*x4^16*z^24 + 4*x1^46*x2^37*x3^21*x4^16*z^24 + x1^45*x2^38*x3^21*x4^16*z^24 + 3*x1^44*x2^39*x3^21*x4^16*z^24 + 2*x1^43*x2^40*x3^21*x4^16*z^24 + x1^49*x2^33*x3^22*x4^16*z^24 + 2*x1^48*x2^34*x3^22*x4^16*z^24 - 3*x1^46*x2^36*x3^22*x4^16*z^24 - x1^45*x2^37*x3^22*x4^16*z^24 - x1^44*x2^38*x3^22*x4^16*z^24 - 6*x1^43*x2^39*x3^22*x4^16*z^24 - 4*x1^48*x2^33*x3^23*x4^16*z^24 + 4*x1^45*x2^36*x3^23*x4^16*z^24 + 6*x1^43*x2^38*x3^23*x4^16*z^24 + 2*x1^42*x2^39*x3^23*x4^16*z^24 + x1^48*x2^32*x3^24*x4^16*z^24 + 3*x1^47*x2^33*x3^24*x4^16*z^24 + x1^46*x2^34*x3^24*x4^16*z^24 - 2*x1^45*x2^35*x3^24*x4^16*z^24 - 2*x1^44*x2^36*x3^24*x4^16*z^24 - 2*x1^43*x2^37*x3^24*x4^16*z^24 - 6*x1^42*x2^38*x3^24*x4^16*z^24 - 3*x1^47*x2^32*x3^25*x4^16*z^24 - x1^45*x2^34*x3^25*x4^16*z^24 + 4*x1^44*x2^35*x3^25*x4^16*z^24 + 6*x1^42*x2^37*x3^25*x4^16*z^24 + 2*x1^41*x2^38*x3^25*x4^16*z^24 + x1^47*x2^31*x3^26*x4^16*z^24 + x1^46*x2^32*x3^26*x4^16*z^24 + x1^45*x2^33*x3^26*x4^16*z^24 - x1^44*x2^34*x3^26*x4^16*z^24 - x1^43*x2^35*x3^26*x4^16*z^24 - 2*x1^42*x2^36*x3^26*x4^16*z^24 - 6*x1^41*x2^37*x3^26*x4^16*z^24 - x1^46*x2^31*x3^27*x4^16*z^24 - x1^44*x2^33*x3^27*x4^16*z^24 + 4*x1^43*x2^34*x3^27*x4^16*z^24 - x1^42*x2^35*x3^27*x4^16*z^24 + 6*x1^41*x2^36*x3^27*x4^16*z^24 + 2*x1^40*x2^37*x3^27*x4^16*z^24 - x1^43*x2^33*x3^28*x4^16*z^24 - 2*x1^42*x2^34*x3^28*x4^16*z^24 - x1^41*x2^35*x3^28*x4^16*z^24 - 6*x1^40*x2^36*x3^28*x4^16*z^24 - x1^43*x2^32*x3^29*x4^16*z^24 + x1^42*x2^33*x3^29*x4^16*z^24 + 2*x1^40*x2^35*x3^29*x4^16*z^24 + 2*x1^39*x2^36*x3^29*x4^16*z^24 - x1^41*x2^33*x3^30*x4^16*z^24 - x1^40*x2^34*x3^30*x4^16*z^24 - 2*x1^39*x2^35*x3^30*x4^16*z^24 + x1^38*x2^35*x3^31*x4^16*z^24 + x1^45*x2^37*x3^21*x4^17*z^24 + x1^44*x2^38*x3^21*x4^17*z^24 + x1^43*x2^39*x3^21*x4^17*z^24 - 2*x1^45*x2^36*x3^22*x4^17*z^24 - x1^42*x2^39*x3^22*x4^17*z^24 + x1^45*x2^35*x3^23*x4^17*z^24 + 2*x1^42*x2^38*x3^23*x4^17*z^24 + x1^47*x2^32*x3^24*x4^17*z^24 + x1^46*x2^33*x3^24*x4^17*z^24 - 2*x1^44*x2^35*x3^24*x4^17*z^24 - 2*x1^42*x2^37*x3^24*x4^17*z^24 - x1^46*x2^32*x3^25*x4^17*z^24 + x1^44*x2^34*x3^25*x4^17*z^24 + 2*x1^41*x2^37*x3^25*x4^17*z^24 + x1^44*x2^33*x3^26*x4^17*z^24 - x1^43*x2^34*x3^26*x4^17*z^24 - 2*x1^41*x2^36*x3^26*x4^17*z^24 - x1^40*x2^37*x3^26*x4^17*z^24 + x1^41*x2^35*x3^27*x4^17*z^24 + 2*x1^40*x2^36*x3^27*x4^17*z^24 + x1^41*x2^34*x3^28*x4^17*z^24 - x1^40*x2^35*x3^28*x4^17*z^24 - x1^39*x2^36*x3^28*x4^17*z^24 + x1^39*x2^35*x3^29*x4^17*z^24 + 2*x1^53*x2^40*x3^21*x4*z^23 - x1^53*x2^39*x3^22*x4*z^23 + 2*x1^52*x2^39*x3^23*x4*z^23 + x1^54*x2^40*x3^19*x4^2*z^23 - 3*x1^53*x2^40*x3^20*x4^2*z^23 + 2*x1^52*x2^41*x3^20*x4^2*z^23 + 2*x1^53*x2^39*x3^21*x4^2*z^23 + 2*x1^52*x2^40*x3^21*x4^2*z^23 - x1^51*x2^41*x3^21*x4^2*z^23 - 5*x1^52*x2^39*x3^22*x4^2*z^23 + x1^49*x2^42*x3^22*x4^2*z^23 + 2*x1^52*x2^38*x3^23*x4^2*z^23 + 2*x1^51*x2^39*x3^23*x4^2*z^23 + 2*x1^49*x2^41*x3^23*x4^2*z^23 - 3*x1^51*x2^38*x3^24*x4^2*z^23 + x1^50*x2^39*x3^24*x4^2*z^23 - x1^49*x2^40*x3^24*x4^2*z^23 - x1^48*x2^41*x3^24*x4^2*z^23 + x1^50*x2^38*x3^25*x4^2*z^23 + x1^49*x2^39*x3^25*x4^2*z^23 + x1^48*x2^40*x3^25*x4^2*z^23 + x1^53*x2^40*x3^19*x4^3*z^23 - x1^52*x2^40*x3^20*x4^3*z^23 - x1^54*x2^37*x3^21*x4^3*z^23 + 3*x1^52*x2^39*x3^21*x4^3*z^23 + x1^50*x2^41*x3^21*x4^3*z^23 + x1^49*x2^42*x3^21*x4^3*z^23 + x1^53*x2^37*x3^22*x4^3*z^23 - 2*x1^52*x2^38*x3^22*x4^3*z^23 - 2*x1^51*x2^39*x3^22*x4^3*z^23 + x1^50*x2^40*x3^22*x4^3*z^23 - x1^49*x2^41*x3^22*x4^3*z^23 - x1^48*x2^42*x3^22*x4^3*z^23 - x1^52*x2^37*x3^23*x4^3*z^23 + x1^51*x2^38*x3^23*x4^3*z^23 - x1^50*x2^39*x3^23*x4^3*z^23 + x1^49*x2^40*x3^23*x4^3*z^23 - x1^51*x2^37*x3^24*x4^3*z^23 - 2*x1^50*x2^38*x3^24*x4^3*z^23 - 2*x1^48*x2^40*x3^24*x4^3*z^23 + x1^50*x2^37*x3^25*x4^3*z^23 - x1^49*x2^38*x3^25*x4^3*z^23 + x1^48*x2^39*x3^25*x4^3*z^23 + x1^47*x2^40*x3^25*x4^3*z^23 - x1^49*x2^37*x3^26*x4^3*z^23 - x1^48*x2^38*x3^26*x4^3*z^23 - x1^47*x2^39*x3^26*x4^3*z^23 + x1^55*x2^38*x3^18*x4^4*z^23 - 2*x1^54*x2^38*x3^19*x4^4*z^23 + x1^53*x2^39*x3^19*x4^4*z^23 + 2*x1^54*x2^37*x3^20*x4^4*z^23 + 2*x1^53*x2^38*x3^20*x4^4*z^23 - x1^52*x2^39*x3^20*x4^4*z^23 + 2*x1^51*x2^40*x3^20*x4^4*z^23 - 6*x1^53*x2^37*x3^21*x4^4*z^23 - x1^52*x2^38*x3^21*x4^4*z^23 - x1^51*x2^39*x3^21*x4^4*z^23 + x1^49*x2^41*x3^21*x4^4*z^23 + 2*x1^53*x2^36*x3^22*x4^4*z^23 + 5*x1^52*x2^37*x3^22*x4^4*z^23 - 3*x1^51*x2^38*x3^22*x4^4*z^23 + x1^50*x2^39*x3^22*x4^4*z^23 - x1^48*x2^41*x3^22*x4^4*z^23 - 4*x1^52*x2^36*x3^23*x4^4*z^23 - x1^51*x2^37*x3^23*x4^4*z^23 - 3*x1^49*x2^39*x3^23*x4^4*z^23 + x1^48*x2^40*x3^23*x4^4*z^23 + 2*x1^52*x2^35*x3^24*x4^4*z^23 + 2*x1^51*x2^36*x3^24*x4^4*z^23 + 3*x1^49*x2^38*x3^24*x4^4*z^23 - x1^48*x2^39*x3^24*x4^4*z^23 + x1^47*x2^40*x3^24*x4^4*z^23 + x1^46*x2^41*x3^24*x4^4*z^23 - x1^45*x2^42*x3^24*x4^4*z^23 - 2*x1^51*x2^35*x3^25*x4^4*z^23 + x1^50*x2^36*x3^25*x4^4*z^23 - x1^49*x2^37*x3^25*x4^4*z^23 - 2*x1^48*x2^38*x3^25*x4^4*z^23 + x1^47*x2^39*x3^25*x4^4*z^23 + x1^51*x2^34*x3^26*x4^4*z^23 + x1^50*x2^35*x3^26*x4^4*z^23 - x1^49*x2^36*x3^26*x4^4*z^23 + x1^48*x2^37*x3^26*x4^4*z^23 - 2*x1^47*x2^38*x3^26*x4^4*z^23 + x1^49*x2^35*x3^27*x4^4*z^23 + x1^48*x2^36*x3^27*x4^4*z^23 - x1^47*x2^37*x3^27*x4^4*z^23 + x1^46*x2^38*x3^27*x4^4*z^23 - 2*x1^55*x2^38*x3^17*x4^5*z^23 + 3*x1^54*x2^38*x3^18*x4^5*z^23 - 2*x1^53*x2^39*x3^18*x4^5*z^23 - 2*x1^54*x2^37*x3^19*x4^5*z^23 - 3*x1^53*x2^38*x3^19*x4^5*z^23 + x1^52*x2^39*x3^19*x4^5*z^23 - 2*x1^51*x2^40*x3^19*x4^5*z^23 + 5*x1^53*x2^37*x3^20*x4^5*z^23 + x1^52*x2^38*x3^20*x4^5*z^23 - x1^50*x2^40*x3^20*x4^5*z^23 - 2*x1^49*x2^41*x3^20*x4^5*z^23 - 2*x1^53*x2^36*x3^21*x4^5*z^23 - 5*x1^52*x2^37*x3^21*x4^5*z^23 + x1^51*x2^38*x3^21*x4^5*z^23 - 3*x1^50*x2^39*x3^21*x4^5*z^23 + x1^49*x2^40*x3^21*x4^5*z^23 + 2*x1^48*x2^41*x3^21*x4^5*z^23 + 6*x1^52*x2^36*x3^22*x4^5*z^23 + x1^51*x2^37*x3^22*x4^5*z^23 + x1^50*x2^38*x3^22*x4^5*z^23 + 2*x1^49*x2^39*x3^22*x4^5*z^23 - 2*x1^47*x2^41*x3^22*x4^5*z^23 + x1^46*x2^42*x3^22*x4^5*z^23 - 2*x1^52*x2^35*x3^23*x4^5*z^23 - 6*x1^51*x2^36*x3^23*x4^5*z^23 + x1^50*x2^37*x3^23*x4^5*z^23 - 2*x1^49*x2^38*x3^23*x4^5*z^23 - x1^47*x2^40*x3^23*x4^5*z^23 + 4*x1^46*x2^41*x3^23*x4^5*z^23 + 5*x1^51*x2^35*x3^24*x4^5*z^23 + x1^50*x2^36*x3^24*x4^5*z^23 + 3*x1^48*x2^38*x3^24*x4^5*z^23 - x1^47*x2^39*x3^24*x4^5*z^23 - x1^46*x2^40*x3^24*x4^5*z^23 - 2*x1^45*x2^41*x3^24*x4^5*z^23 - 2*x1^51*x2^34*x3^25*x4^5*z^23 - 3*x1^50*x2^35*x3^25*x4^5*z^23 + 2*x1^49*x2^36*x3^25*x4^5*z^23 - 4*x1^48*x2^37*x3^25*x4^5*z^23 + x1^47*x2^38*x3^25*x4^5*z^23 + 3*x1^45*x2^40*x3^25*x4^5*z^23 - x1^43*x2^42*x3^25*x4^5*z^23 + 4*x1^50*x2^34*x3^26*x4^5*z^23 + x1^48*x2^36*x3^26*x4^5*z^23 + 3*x1^47*x2^37*x3^26*x4^5*z^23 - x1^46*x2^38*x3^26*x4^5*z^23 + x1^44*x2^40*x3^26*x4^5*z^23 + x1^43*x2^41*x3^26*x4^5*z^23 - x1^50*x2^33*x3^27*x4^5*z^23 - x1^49*x2^34*x3^27*x4^5*z^23 + 2*x1^48*x2^35*x3^27*x4^5*z^23 - 2*x1^47*x2^36*x3^27*x4^5*z^23 + x1^46*x2^37*x3^27*x4^5*z^23 + x1^49*x2^33*x3^28*x4^5*z^23 - x1^48*x2^34*x3^28*x4^5*z^23 + x1^47*x2^35*x3^28*x4^5*z^23 + 2*x1^46*x2^36*x3^28*x4^5*z^23 - 2*x1^45*x2^37*x3^28*x4^5*z^23 + x1^47*x2^34*x3^29*x4^5*z^23 + x1^46*x2^35*x3^29*x4^5*z^23 + x1^45*x2^36*x3^29*x4^5*z^23 + x1^54*x2^37*x3^18*x4^6*z^23 - 2*x1^53*x2^37*x3^19*x4^6*z^23 + x1^52*x2^38*x3^19*x4^6*z^23 + x1^53*x2^36*x3^20*x4^6*z^23 + 2*x1^52*x2^37*x3^20*x4^6*z^23 - 2*x1^51*x2^38*x3^20*x4^6*z^23 + 3*x1^50*x2^39*x3^20*x4^6*z^23 + x1^49*x2^40*x3^20*x4^6*z^23 - 2*x1^52*x2^36*x3^21*x4^6*z^23 - x1^51*x2^37*x3^21*x4^6*z^23 + x1^50*x2^38*x3^21*x4^6*z^23 - 2*x1^49*x2^39*x3^21*x4^6*z^23 + x1^48*x2^40*x3^21*x4^6*z^23 + x1^47*x2^41*x3^21*x4^6*z^23 + x1^52*x2^35*x3^22*x4^6*z^23 + 2*x1^51*x2^36*x3^22*x4^6*z^23 - 4*x1^50*x2^37*x3^22*x4^6*z^23 - x1^49*x2^38*x3^22*x4^6*z^23 - 4*x1^48*x2^39*x3^22*x4^6*z^23 - x1^46*x2^41*x3^22*x4^6*z^23 - x1^45*x2^42*x3^22*x4^6*z^23 - 2*x1^51*x2^35*x3^23*x4^6*z^23 + 2*x1^50*x2^36*x3^23*x4^6*z^23 + 4*x1^49*x2^37*x3^23*x4^6*z^23 + x1^47*x2^39*x3^23*x4^6*z^23 - x1^46*x2^40*x3^23*x4^6*z^23 + x1^45*x2^41*x3^23*x4^6*z^23 + x1^44*x2^42*x3^23*x4^6*z^23 + 3*x1^50*x2^35*x3^24*x4^6*z^23 - 2*x1^49*x2^36*x3^24*x4^6*z^23 + x1^48*x2^37*x3^24*x4^6*z^23 - 2*x1^47*x2^38*x3^24*x4^6*z^23 - x1^45*x2^40*x3^24*x4^6*z^23 + 2*x1^44*x2^41*x3^24*x4^6*z^23 + x1^43*x2^42*x3^24*x4^6*z^23 - 3*x1^50*x2^34*x3^25*x4^6*z^23 + 3*x1^48*x2^36*x3^25*x4^6*z^23 - x1^47*x2^37*x3^25*x4^6*z^23 + 3*x1^46*x2^38*x3^25*x4^6*z^23 - x1^45*x2^39*x3^25*x4^6*z^23 + x1^44*x2^40*x3^25*x4^6*z^23 + x1^50*x2^33*x3^26*x4^6*z^23 + x1^49*x2^34*x3^26*x4^6*z^23 - 4*x1^48*x2^35*x3^26*x4^6*z^23 + 3*x1^47*x2^36*x3^26*x4^6*z^23 - 2*x1^45*x2^38*x3^26*x4^6*z^23 - 3*x1^44*x2^39*x3^26*x4^6*z^23 + x1^42*x2^41*x3^26*x4^6*z^23 - 2*x1^49*x2^33*x3^27*x4^6*z^23 - 2*x1^46*x2^36*x3^27*x4^6*z^23 + 3*x1^45*x2^37*x3^27*x4^6*z^23 - x1^43*x2^39*x3^27*x4^6*z^23 - x1^42*x2^40*x3^27*x4^6*z^23 - 2*x1^47*x2^34*x3^28*x4^6*z^23 + x1^47*x2^33*x3^29*x4^6*z^23 - x1^45*x2^35*x3^29*x4^6*z^23 + x1^44*x2^36*x3^29*x4^6*z^23 - x1^45*x2^34*x3^30*x4^6*z^23 - x1^44*x2^35*x3^30*x4^6*z^23 + x1^53*x2^39*x3^16*x4^7*z^23 - x1^54*x2^37*x3^17*x4^7*z^23 - 2*x1^52*x2^39*x3^17*x4^7*z^23 - x1^51*x2^40*x3^17*x4^7*z^23 - x1^50*x2^41*x3^17*x4^7*z^23 + x1^52*x2^38*x3^18*x4^7*z^23 + 2*x1^51*x2^39*x3^18*x4^7*z^23 + x1^49*x2^41*x3^18*x4^7*z^23 - x1^48*x2^42*x3^18*x4^7*z^23 - 3*x1^51*x2^38*x3^19*x4^7*z^23 - x1^50*x2^39*x3^19*x4^7*z^23 - x1^48*x2^41*x3^19*x4^7*z^23 + x1^51*x2^37*x3^20*x4^7*z^23 + 3*x1^50*x2^38*x3^20*x4^7*z^23 + x1^49*x2^39*x3^20*x4^7*z^23 + 3*x1^48*x2^40*x3^20*x4^7*z^23 + x1^47*x2^41*x3^20*x4^7*z^23 - x1^45*x2^43*x3^20*x4^7*z^23 - x1^50*x2^37*x3^21*x4^7*z^23 - 2*x1^49*x2^38*x3^21*x4^7*z^23 - x1^48*x2^39*x3^21*x4^7*z^23 - x1^47*x2^40*x3^21*x4^7*z^23 + x1^45*x2^42*x3^21*x4^7*z^23 + x1^44*x2^43*x3^21*x4^7*z^23 + x1^49*x2^37*x3^22*x4^7*z^23 + 2*x1^48*x2^38*x3^22*x4^7*z^23 - x1^44*x2^42*x3^22*x4^7*z^23 + x1^47*x2^38*x3^23*x4^7*z^23 - 3*x1^46*x2^39*x3^23*x4^7*z^23 + x1^44*x2^41*x3^23*x4^7*z^23 - x1^46*x2^38*x3^24*x4^7*z^23 - 2*x1^44*x2^40*x3^24*x4^7*z^23 - 2*x1^43*x2^41*x3^24*x4^7*z^23 + x1^42*x2^42*x3^24*x4^7*z^23 - x1^49*x2^34*x3^25*x4^7*z^23 - x1^48*x2^35*x3^25*x4^7*z^23 + x1^46*x2^37*x3^25*x4^7*z^23 - x1^45*x2^38*x3^25*x4^7*z^23 + x1^44*x2^39*x3^25*x4^7*z^23 - x1^43*x2^40*x3^25*x4^7*z^23 + x1^42*x2^41*x3^25*x4^7*z^23 + 2*x1^49*x2^33*x3^26*x4^7*z^23 - x1^47*x2^35*x3^26*x4^7*z^23 + x1^46*x2^36*x3^26*x4^7*z^23 - x1^45*x2^37*x3^26*x4^7*z^23 - x1^43*x2^39*x3^26*x4^7*z^23 - x1^42*x2^40*x3^26*x4^7*z^23 - x1^49*x2^32*x3^27*x4^7*z^23 - 2*x1^48*x2^33*x3^27*x4^7*z^23 + 2*x1^47*x2^34*x3^27*x4^7*z^23 - x1^45*x2^36*x3^27*x4^7*z^23 + 2*x1^43*x2^38*x3^27*x4^7*z^23 - x1^42*x2^39*x3^27*x4^7*z^23 - x1^47*x2^33*x3^28*x4^7*z^23 + x1^45*x2^35*x3^28*x4^7*z^23 - x1^44*x2^35*x3^29*x4^7*z^23 + x1^44*x2^34*x3^30*x4^7*z^23 + x1^43*x2^34*x3^31*x4^7*z^23 - x1^54*x2^37*x3^16*x4^8*z^23 + 2*x1^52*x2^39*x3^16*x4^8*z^23 - x1^51*x2^40*x3^16*x4^8*z^23 + x1^53*x2^37*x3^17*x4^8*z^23 - x1^52*x2^38*x3^17*x4^8*z^23 - 2*x1^51*x2^39*x3^17*x4^8*z^23 + 2*x1^50*x2^40*x3^17*x4^8*z^23 - x1^49*x2^41*x3^17*x4^8*z^23 + x1^53*x2^36*x3^18*x4^8*z^23 - x1^52*x2^37*x3^18*x4^8*z^23 + 4*x1^51*x2^38*x3^18*x4^8*z^23 - x1^49*x2^40*x3^18*x4^8*z^23 - x1^47*x2^42*x3^18*x4^8*z^23 - x1^52*x2^36*x3^19*x4^8*z^23 + x1^51*x2^37*x3^19*x4^8*z^23 - 3*x1^50*x2^38*x3^19*x4^8*z^23 + x1^49*x2^39*x3^19*x4^8*z^23 - x1^48*x2^40*x3^19*x4^8*z^23 + 2*x1^47*x2^41*x3^19*x4^8*z^23 + x1^46*x2^42*x3^19*x4^8*z^23 + x1^51*x2^36*x3^20*x4^8*z^23 + 2*x1^50*x2^37*x3^20*x4^8*z^23 + 2*x1^49*x2^38*x3^20*x4^8*z^23 - x1^47*x2^40*x3^20*x4^8*z^23 - x1^46*x2^41*x3^20*x4^8*z^23 - x1^45*x2^42*x3^20*x4^8*z^23 - 2*x1^51*x2^35*x3^21*x4^8*z^23 - 3*x1^50*x2^36*x3^21*x4^8*z^23 - 3*x1^49*x2^37*x3^21*x4^8*z^23 + x1^48*x2^38*x3^21*x4^8*z^23 - x1^47*x2^39*x3^21*x4^8*z^23 + 3*x1^46*x2^40*x3^21*x4^8*z^23 + x1^45*x2^41*x3^21*x4^8*z^23 + 4*x1^44*x2^42*x3^21*x4^8*z^23 + x1^51*x2^34*x3^22*x4^8*z^23 + 2*x1^50*x2^35*x3^22*x4^8*z^23 + 4*x1^49*x2^36*x3^22*x4^8*z^23 + x1^48*x2^37*x3^22*x4^8*z^23 - x1^46*x2^39*x3^22*x4^8*z^23 - 2*x1^45*x2^40*x3^22*x4^8*z^23 - 3*x1^44*x2^41*x3^22*x4^8*z^23 - 4*x1^43*x2^42*x3^22*x4^8*z^23 - 2*x1^50*x2^34*x3^23*x4^8*z^23 - 2*x1^49*x2^35*x3^23*x4^8*z^23 - 5*x1^48*x2^36*x3^23*x4^8*z^23 - x1^47*x2^37*x3^23*x4^8*z^23 - 2*x1^46*x2^38*x3^23*x4^8*z^23 + 4*x1^45*x2^39*x3^23*x4^8*z^23 - x1^44*x2^40*x3^23*x4^8*z^23 + 4*x1^43*x2^41*x3^23*x4^8*z^23 + x1^42*x2^42*x3^23*x4^8*z^23 + x1^50*x2^33*x3^24*x4^8*z^23 + 2*x1^49*x2^34*x3^24*x4^8*z^23 + 4*x1^48*x2^35*x3^24*x4^8*z^23 + 2*x1^47*x2^36*x3^24*x4^8*z^23 - x1^46*x2^37*x3^24*x4^8*z^23 + x1^45*x2^38*x3^24*x4^8*z^23 - 2*x1^44*x2^39*x3^24*x4^8*z^23 - x1^43*x2^40*x3^24*x4^8*z^23 - 4*x1^42*x2^41*x3^24*x4^8*z^23 - 2*x1^49*x2^33*x3^25*x4^8*z^23 - 4*x1^47*x2^35*x3^25*x4^8*z^23 - x1^46*x2^36*x3^25*x4^8*z^23 - 2*x1^45*x2^37*x3^25*x4^8*z^23 + x1^44*x2^38*x3^25*x4^8*z^23 + 3*x1^43*x2^39*x3^25*x4^8*z^23 + 4*x1^42*x2^40*x3^25*x4^8*z^23 + x1^49*x2^32*x3^26*x4^8*z^23 + x1^48*x2^33*x3^26*x4^8*z^23 + x1^47*x2^34*x3^26*x4^8*z^23 + 5*x1^46*x2^35*x3^26*x4^8*z^23 + x1^45*x2^36*x3^26*x4^8*z^23 - x1^43*x2^38*x3^26*x4^8*z^23 - x1^42*x2^39*x3^26*x4^8*z^23 - 3*x1^41*x2^40*x3^26*x4^8*z^23 - 2*x1^48*x2^32*x3^27*x4^8*z^23 - x1^47*x2^33*x3^27*x4^8*z^23 - 2*x1^46*x2^34*x3^27*x4^8*z^23 - x1^45*x2^35*x3^27*x4^8*z^23 - 2*x1^44*x2^36*x3^27*x4^8*z^23 + x1^42*x2^38*x3^27*x4^8*z^23 + 3*x1^41*x2^39*x3^27*x4^8*z^23 + 2*x1^47*x2^32*x3^28*x4^8*z^23 + x1^46*x2^33*x3^28*x4^8*z^23 + x1^45*x2^34*x3^28*x4^8*z^23 + 3*x1^44*x2^35*x3^28*x4^8*z^23 + 3*x1^43*x2^36*x3^28*x4^8*z^23 - 2*x1^42*x2^37*x3^28*x4^8*z^23 - x1^40*x2^39*x3^28*x4^8*z^23 - x1^46*x2^32*x3^29*x4^8*z^23 - x1^45*x2^33*x3^29*x4^8*z^23 + x1^42*x2^36*x3^29*x4^8*z^23 + x1^41*x2^37*x3^29*x4^8*z^23 + x1^40*x2^38*x3^29*x4^8*z^23 + x1^45*x2^32*x3^30*x4^8*z^23 + x1^43*x2^34*x3^30*x4^8*z^23 - x1^42*x2^34*x3^31*x4^8*z^23 - x1^52*x2^39*x3^15*x4^9*z^23 + x1^51*x2^39*x3^16*x4^9*z^23 - x1^50*x2^40*x3^16*x4^9*z^23 - x1^52*x2^37*x3^17*x4^9*z^23 - 2*x1^51*x2^38*x3^17*x4^9*z^23 - 2*x1^50*x2^39*x3^17*x4^9*z^23 + x1^52*x2^36*x3^18*x4^9*z^23 - x1^51*x2^37*x3^18*x4^9*z^23 + x1^49*x2^39*x3^18*x4^9*z^23 + x1^48*x2^40*x3^18*x4^9*z^23 - x1^47*x2^41*x3^18*x4^9*z^23 - x1^52*x2^35*x3^19*x4^9*z^23 - 2*x1^51*x2^36*x3^19*x4^9*z^23 - x1^50*x2^37*x3^19*x4^9*z^23 + x1^46*x2^41*x3^19*x4^9*z^23 + x1^45*x2^42*x3^19*x4^9*z^23 + 4*x1^51*x2^35*x3^20*x4^9*z^23 + x1^50*x2^36*x3^20*x4^9*z^23 + 2*x1^49*x2^37*x3^20*x4^9*z^23 + 2*x1^47*x2^39*x3^20*x4^9*z^23 - x1^45*x2^41*x3^20*x4^9*z^23 - 2*x1^44*x2^42*x3^20*x4^9*z^23 - x1^51*x2^34*x3^21*x4^9*z^23 - 4*x1^50*x2^35*x3^21*x4^9*z^23 - 2*x1^49*x2^36*x3^21*x4^9*z^23 - 5*x1^48*x2^37*x3^21*x4^9*z^23 - 2*x1^47*x2^38*x3^21*x4^9*z^23 + x1^46*x2^39*x3^21*x4^9*z^23 + x1^45*x2^40*x3^21*x4^9*z^23 + 2*x1^44*x2^41*x3^21*x4^9*z^23 + 2*x1^43*x2^42*x3^21*x4^9*z^23 + 4*x1^50*x2^34*x3^22*x4^9*z^23 + 3*x1^49*x2^35*x3^22*x4^9*z^23 + 4*x1^48*x2^36*x3^22*x4^9*z^23 - x1^46*x2^38*x3^22*x4^9*z^23 - 5*x1^45*x2^39*x3^22*x4^9*z^23 - 2*x1^43*x2^41*x3^22*x4^9*z^23 - x1^42*x2^42*x3^22*x4^9*z^23 - 2*x1^50*x2^33*x3^23*x4^9*z^23 - 4*x1^49*x2^34*x3^23*x4^9*z^23 - 2*x1^48*x2^35*x3^23*x4^9*z^23 - 2*x1^47*x2^36*x3^23*x4^9*z^23 + 3*x1^46*x2^37*x3^23*x4^9*z^23 + x1^45*x2^38*x3^23*x4^9*z^23 + 3*x1^44*x2^39*x3^23*x4^9*z^23 + 2*x1^42*x2^41*x3^23*x4^9*z^23 + 4*x1^49*x2^33*x3^24*x4^9*z^23 + x1^48*x2^34*x3^24*x4^9*z^23 + 3*x1^47*x2^35*x3^24*x4^9*z^23 + x1^46*x2^36*x3^24*x4^9*z^23 + x1^45*x2^37*x3^24*x4^9*z^23 - x1^44*x2^38*x3^24*x4^9*z^23 - 2*x1^43*x2^39*x3^24*x4^9*z^23 - x1^42*x2^40*x3^24*x4^9*z^23 - x1^49*x2^32*x3^25*x4^9*z^23 - 4*x1^48*x2^33*x3^25*x4^9*z^23 - x1^47*x2^34*x3^25*x4^9*z^23 - 4*x1^46*x2^35*x3^25*x4^9*z^23 + 3*x1^43*x2^38*x3^25*x4^9*z^23 + x1^42*x2^39*x3^25*x4^9*z^23 + 2*x1^41*x2^40*x3^25*x4^9*z^23 + 3*x1^48*x2^32*x3^26*x4^9*z^23 + 2*x1^47*x2^33*x3^26*x4^9*z^23 + 2*x1^46*x2^34*x3^26*x4^9*z^23 + x1^45*x2^35*x3^26*x4^9*z^23 + x1^44*x2^36*x3^26*x4^9*z^23 - 2*x1^42*x2^38*x3^26*x4^9*z^23 - x1^41*x2^39*x3^26*x4^9*z^23 - 2*x1^48*x2^31*x3^27*x4^9*z^23 - 3*x1^47*x2^32*x3^27*x4^9*z^23 + x1^46*x2^33*x3^27*x4^9*z^23 - 3*x1^45*x2^34*x3^27*x4^9*z^23 + x1^43*x2^36*x3^27*x4^9*z^23 + 3*x1^42*x2^37*x3^27*x4^9*z^23 + x1^41*x2^38*x3^27*x4^9*z^23 + 2*x1^40*x2^39*x3^27*x4^9*z^23 + 3*x1^47*x2^31*x3^28*x4^9*z^23 + x1^46*x2^32*x3^28*x4^9*z^23 + x1^45*x2^33*x3^28*x4^9*z^23 - 2*x1^42*x2^36*x3^28*x4^9*z^23 - x1^41*x2^37*x3^28*x4^9*z^23 + x1^40*x2^38*x3^28*x4^9*z^23 - 2*x1^46*x2^31*x3^29*x4^9*z^23 - x1^44*x2^33*x3^29*x4^9*z^23 + x1^43*x2^34*x3^29*x4^9*z^23 - x1^42*x2^35*x3^29*x4^9*z^23 + x1^39*x2^38*x3^29*x4^9*z^23 + x1^45*x2^31*x3^30*x4^9*z^23 + x1^44*x2^32*x3^30*x4^9*z^23 + x1^43*x2^33*x3^30*x4^9*z^23 + x1^41*x2^35*x3^30*x4^9*z^23 - x1^40*x2^36*x3^30*x4^9*z^23 - x1^39*x2^37*x3^30*x4^9*z^23 + x1^42*x2^33*x3^31*x4^9*z^23 + x1^41*x2^34*x3^31*x4^9*z^23 + x1^53*x2^37*x3^15*x4^10*z^23 - x1^52*x2^37*x3^16*x4^10*z^23 + x1^50*x2^39*x3^16*x4^10*z^23 + x1^49*x2^40*x3^16*x4^10*z^23 + x1^52*x2^36*x3^17*x4^10*z^23 - x1^51*x2^37*x3^17*x4^10*z^23 - 2*x1^50*x2^38*x3^17*x4^10*z^23 - x1^48*x2^40*x3^17*x4^10*z^23 - x1^47*x2^41*x3^17*x4^10*z^23 + 2*x1^50*x2^37*x3^18*x4^10*z^23 + x1^48*x2^39*x3^18*x4^10*z^23 + x1^47*x2^40*x3^18*x4^10*z^23 + 2*x1^46*x2^41*x3^18*x4^10*z^23 - x1^49*x2^37*x3^19*x4^10*z^23 - 2*x1^48*x2^38*x3^19*x4^10*z^23 - 3*x1^47*x2^39*x3^19*x4^10*z^23 - 2*x1^46*x2^40*x3^19*x4^10*z^23 - 2*x1^45*x2^41*x3^19*x4^10*z^23 - x1^44*x2^42*x3^19*x4^10*z^23 - x1^49*x2^36*x3^20*x4^10*z^23 + x1^47*x2^38*x3^20*x4^10*z^23 + x1^46*x2^39*x3^20*x4^10*z^23 + 3*x1^45*x2^40*x3^20*x4^10*z^23 + x1^44*x2^41*x3^20*x4^10*z^23 + x1^43*x2^42*x3^20*x4^10*z^23 + x1^48*x2^36*x3^21*x4^10*z^23 - x1^46*x2^38*x3^21*x4^10*z^23 - 2*x1^45*x2^39*x3^21*x4^10*z^23 - 2*x1^44*x2^40*x3^21*x4^10*z^23 - x1^43*x2^41*x3^21*x4^10*z^23 + x1^46*x2^37*x3^22*x4^10*z^23 + 2*x1^44*x2^39*x3^22*x4^10*z^23 + x1^42*x2^41*x3^22*x4^10*z^23 + x1^44*x2^38*x3^23*x4^10*z^23 - 2*x1^43*x2^39*x3^23*x4^10*z^23 + x1^49*x2^32*x3^24*x4^10*z^23 + x1^42*x2^39*x3^24*x4^10*z^23 - 2*x1^48*x2^32*x3^25*x4^10*z^23 + x1^47*x2^33*x3^25*x4^10*z^23 + x1^48*x2^31*x3^26*x4^10*z^23 + 2*x1^47*x2^32*x3^26*x4^10*z^23 - x1^46*x2^33*x3^26*x4^10*z^23 + 2*x1^45*x2^34*x3^26*x4^10*z^23 + x1^44*x2^35*x3^26*x4^10*z^23 + x1^40*x2^39*x3^26*x4^10*z^23 - 2*x1^47*x2^31*x3^27*x4^10*z^23 - 2*x1^46*x2^32*x3^27*x4^10*z^23 + x1^43*x2^35*x3^27*x4^10*z^23 - 2*x1^42*x2^36*x3^27*x4^10*z^23 - 2*x1^40*x2^38*x3^27*x4^10*z^23 + 2*x1^46*x2^31*x3^28*x4^10*z^23 + x1^41*x2^36*x3^28*x4^10*z^23 - x1^40*x2^37*x3^28*x4^10*z^23 + x1^39*x2^38*x3^28*x4^10*z^23 - x1^45*x2^31*x3^29*x4^10*z^23 - 2*x1^44*x2^32*x3^29*x4^10*z^23 - x1^43*x2^33*x3^29*x4^10*z^23 + x1^41*x2^35*x3^29*x4^10*z^23 - 2*x1^39*x2^37*x3^29*x4^10*z^23 + x1^44*x2^31*x3^30*x4^10*z^23 + 2*x1^43*x2^32*x3^30*x4^10*z^23 - x1^43*x2^31*x3^31*x4^10*z^23 - x1^41*x2^33*x3^31*x4^10*z^23 + x1^39*x2^35*x3^31*x4^10*z^23 - x1^38*x2^36*x3^31*x4^10*z^23 - 2*x1^52*x2^36*x3^16*x4^11*z^23 - x1^51*x2^37*x3^16*x4^11*z^23 + x1^50*x2^38*x3^16*x4^11*z^23 + 2*x1^51*x2^36*x3^17*x4^11*z^23 - x1^49*x2^38*x3^17*x4^11*z^23 - 2*x1^48*x2^39*x3^17*x4^11*z^23 - x1^47*x2^40*x3^17*x4^11*z^23 + x1^46*x2^41*x3^17*x4^11*z^23 - 2*x1^51*x2^35*x3^18*x4^11*z^23 - x1^50*x2^36*x3^18*x4^11*z^23 + x1^49*x2^37*x3^18*x4^11*z^23 + 2*x1^48*x2^38*x3^18*x4^11*z^23 + x1^47*x2^39*x3^18*x4^11*z^23 - x1^45*x2^41*x3^18*x4^11*z^23 + 3*x1^50*x2^35*x3^19*x4^11*z^23 - 2*x1^49*x2^36*x3^19*x4^11*z^23 + 2*x1^48*x2^37*x3^19*x4^11*z^23 - 3*x1^45*x2^40*x3^19*x4^11*z^23 + x1^44*x2^41*x3^19*x4^11*z^23 - 3*x1^50*x2^34*x3^20*x4^11*z^23 + x1^48*x2^36*x3^20*x4^11*z^23 + 5*x1^46*x2^38*x3^20*x4^11*z^23 + x1^45*x2^39*x3^20*x4^11*z^23 + 3*x1^44*x2^40*x3^20*x4^11*z^23 + 2*x1^50*x2^33*x3^21*x4^11*z^23 + 4*x1^49*x2^34*x3^21*x4^11*z^23 - 2*x1^48*x2^35*x3^21*x4^11*z^23 + x1^47*x2^36*x3^21*x4^11*z^23 - 4*x1^46*x2^37*x3^21*x4^11*z^23 - 4*x1^44*x2^39*x3^21*x4^11*z^23 - x1^43*x2^40*x3^21*x4^11*z^23 - 4*x1^49*x2^33*x3^22*x4^11*z^23 + x1^47*x2^35*x3^22*x4^11*z^23 + 2*x1^45*x2^37*x3^22*x4^11*z^23 + 4*x1^43*x2^39*x3^22*x4^11*z^23 - 2*x1^42*x2^40*x3^22*x4^11*z^23 - x1^41*x2^41*x3^22*x4^11*z^23 - x1^49*x2^32*x3^23*x4^11*z^23 + 4*x1^48*x2^33*x3^23*x4^11*z^23 - 2*x1^47*x2^34*x3^23*x4^11*z^23 + 2*x1^46*x2^35*x3^23*x4^11*z^23 - 4*x1^45*x2^36*x3^23*x4^11*z^23 - 3*x1^43*x2^38*x3^23*x4^11*z^23 + 2*x1^41*x2^40*x3^23*x4^11*z^23 - x1^48*x2^32*x3^24*x4^11*z^23 - 2*x1^47*x2^33*x3^24*x4^11*z^23 + x1^46*x2^34*x3^24*x4^11*z^23 + 2*x1^44*x2^36*x3^24*x4^11*z^23 - x1^43*x2^37*x3^24*x4^11*z^23 + 4*x1^42*x2^38*x3^24*x4^11*z^23 - 2*x1^41*x2^39*x3^24*x4^11*z^23 + x1^47*x2^32*x3^25*x4^11*z^23 - x1^46*x2^33*x3^25*x4^11*z^23 + x1^45*x2^34*x3^25*x4^11*z^23 - 2*x1^44*x2^35*x3^25*x4^11*z^23 - 4*x1^42*x2^37*x3^25*x4^11*z^23 - 2*x1^41*x2^38*x3^25*x4^11*z^23 + 2*x1^40*x2^39*x3^25*x4^11*z^23 - x1^47*x2^31*x3^26*x4^11*z^23 - 2*x1^44*x2^34*x3^26*x4^11*z^23 + 2*x1^43*x2^35*x3^26*x4^11*z^23 + x1^42*x2^36*x3^26*x4^11*z^23 + 2*x1^41*x2^37*x3^26*x4^11*z^23 - 2*x1^40*x2^38*x3^26*x4^11*z^23 - x1^39*x2^39*x3^26*x4^11*z^23 + x1^46*x2^31*x3^27*x4^11*z^23 - x1^45*x2^32*x3^27*x4^11*z^23 - x1^44*x2^33*x3^27*x4^11*z^23 - 2*x1^43*x2^34*x3^27*x4^11*z^23 + 3*x1^42*x2^35*x3^27*x4^11*z^23 - 3*x1^41*x2^36*x3^27*x4^11*z^23 - x1^40*x2^37*x3^27*x4^11*z^23 + x1^44*x2^32*x3^28*x4^11*z^23 + x1^43*x2^33*x3^28*x4^11*z^23 + 2*x1^42*x2^34*x3^28*x4^11*z^23 + 3*x1^40*x2^36*x3^28*x4^11*z^23 - x1^39*x2^37*x3^28*x4^11*z^23 - x1^38*x2^38*x3^28*x4^11*z^23 + x1^43*x2^32*x3^29*x4^11*z^23 - x1^41*x2^34*x3^29*x4^11*z^23 + 2*x1^40*x2^35*x3^29*x4^11*z^23 + x1^39*x2^36*x3^29*x4^11*z^23 - x1^38*x2^37*x3^29*x4^11*z^23 - x1^41*x2^33*x3^30*x4^11*z^23 - x1^38*x2^36*x3^30*x4^11*z^23 + x1^52*x2^36*x3^15*x4^12*z^23 - x1^51*x2^36*x3^16*x4^12*z^23 - x1^50*x2^37*x3^16*x4^12*z^23 - x1^47*x2^40*x3^16*x4^12*z^23 + x1^51*x2^35*x3^17*x4^12*z^23 - x1^49*x2^37*x3^17*x4^12*z^23 - x1^50*x2^35*x3^18*x4^12*z^23 + x1^49*x2^36*x3^18*x4^12*z^23 - x1^48*x2^37*x3^18*x4^12*z^23 - x1^44*x2^41*x3^18*x4^12*z^23 + 2*x1^50*x2^34*x3^19*x4^12*z^23 - x1^49*x2^35*x3^19*x4^12*z^23 - 4*x1^48*x2^36*x3^19*x4^12*z^23 - 2*x1^46*x2^38*x3^19*x4^12*z^23 + 2*x1^45*x2^39*x3^19*x4^12*z^23 - x1^44*x2^40*x3^19*x4^12*z^23 + x1^43*x2^41*x3^19*x4^12*z^23 - 3*x1^49*x2^34*x3^20*x4^12*z^23 + 4*x1^48*x2^35*x3^20*x4^12*z^23 + x1^47*x2^36*x3^20*x4^12*z^23 + 2*x1^46*x2^37*x3^20*x4^12*z^23 - x1^42*x2^41*x3^20*x4^12*z^23 + x1^49*x2^33*x3^21*x4^12*z^23 - 2*x1^48*x2^34*x3^21*x4^12*z^23 - 6*x1^47*x2^35*x3^21*x4^12*z^23 - 5*x1^45*x2^37*x3^21*x4^12*z^23 + 3*x1^44*x2^38*x3^21*x4^12*z^23 + 4*x1^42*x2^40*x3^21*x4^12*z^23 + x1^41*x2^41*x3^21*x4^12*z^23 - x1^49*x2^32*x3^22*x4^12*z^23 - 2*x1^48*x2^33*x3^22*x4^12*z^23 + 6*x1^47*x2^34*x3^22*x4^12*z^23 + x1^46*x2^35*x3^22*x4^12*z^23 + 4*x1^45*x2^36*x3^22*x4^12*z^23 - 2*x1^42*x2^39*x3^22*x4^12*z^23 - 4*x1^41*x2^40*x3^22*x4^12*z^23 + 2*x1^48*x2^32*x3^23*x4^12*z^23 - 2*x1^47*x2^33*x3^23*x4^12*z^23 - 6*x1^46*x2^34*x3^23*x4^12*z^23 - 4*x1^44*x2^36*x3^23*x4^12*z^23 + 4*x1^43*x2^37*x3^23*x4^12*z^23 - 2*x1^42*x2^38*x3^23*x4^12*z^23 + 6*x1^41*x2^39*x3^23*x4^12*z^23 + 2*x1^40*x2^40*x3^23*x4^12*z^23 + x1^48*x2^31*x3^24*x4^12*z^23 - 2*x1^47*x2^32*x3^24*x4^12*z^23 + 5*x1^46*x2^33*x3^24*x4^12*z^23 + 3*x1^44*x2^35*x3^24*x4^12*z^23 - x1^41*x2^38*x3^24*x4^12*z^23 - 6*x1^40*x2^39*x3^24*x4^12*z^23 + x1^46*x2^32*x3^25*x4^12*z^23 - 4*x1^45*x2^33*x3^25*x4^12*z^23 - x1^44*x2^34*x3^25*x4^12*z^23 - 5*x1^43*x2^35*x3^25*x4^12*z^23 + 3*x1^42*x2^36*x3^25*x4^12*z^23 - 2*x1^41*x2^37*x3^25*x4^12*z^23 + 6*x1^40*x2^38*x3^25*x4^12*z^23 + 2*x1^39*x2^39*x3^25*x4^12*z^23 + 2*x1^45*x2^32*x3^26*x4^12*z^23 + 3*x1^44*x2^33*x3^26*x4^12*z^23 + 4*x1^43*x2^34*x3^26*x4^12*z^23 - x1^40*x2^37*x3^26*x4^12*z^23 - 6*x1^39*x2^38*x3^26*x4^12*z^23 - x1^45*x2^31*x3^27*x4^12*z^23 - 2*x1^44*x2^32*x3^27*x4^12*z^23 - 2*x1^43*x2^33*x3^27*x4^12*z^23 - 3*x1^42*x2^34*x3^27*x4^12*z^23 + 3*x1^41*x2^35*x3^27*x4^12*z^23 + 6*x1^39*x2^37*x3^27*x4^12*z^23 + 2*x1^38*x2^38*x3^27*x4^12*z^23 - x1^41*x2^34*x3^28*x4^12*z^23 - 2*x1^39*x2^36*x3^28*x4^12*z^23 - 4*x1^38*x2^37*x3^28*x4^12*z^23 - x1^39*x2^35*x3^29*x4^12*z^23 + 4*x1^38*x2^36*x3^29*x4^12*z^23 + x1^37*x2^37*x3^29*x4^12*z^23 + 2*x1^39*x2^34*x3^30*x4^12*z^23 + x1^38*x2^35*x3^30*x4^12*z^23 - 2*x1^37*x2^36*x3^30*x4^12*z^23 + x1^37*x2^35*x3^31*x4^12*z^23 + x1^48*x2^38*x3^16*x4^13*z^23 + x1^47*x2^39*x3^16*x4^13*z^23 + 2*x1^50*x2^35*x3^17*x4^13*z^23 + x1^49*x2^36*x3^17*x4^13*z^23 + x1^45*x2^40*x3^17*x4^13*z^23 + x1^47*x2^37*x3^18*x4^13*z^23 + x1^46*x2^38*x3^18*x4^13*z^23 - x1^45*x2^39*x3^18*x4^13*z^23 - 2*x1^44*x2^40*x3^18*x4^13*z^23 - x1^43*x2^41*x3^18*x4^13*z^23 + 2*x1^49*x2^34*x3^19*x4^13*z^23 - x1^48*x2^35*x3^19*x4^13*z^23 - 2*x1^47*x2^36*x3^19*x4^13*z^23 + x1^46*x2^37*x3^19*x4^13*z^23 + x1^44*x2^39*x3^19*x4^13*z^23 + 3*x1^43*x2^40*x3^19*x4^13*z^23 + x1^42*x2^41*x3^19*x4^13*z^23 - x1^49*x2^33*x3^20*x4^13*z^23 + x1^48*x2^34*x3^20*x4^13*z^23 + 3*x1^47*x2^35*x3^20*x4^13*z^23 - x1^46*x2^36*x3^20*x4^13*z^23 - 4*x1^44*x2^38*x3^20*x4^13*z^23 - x1^43*x2^39*x3^20*x4^13*z^23 - 5*x1^42*x2^40*x3^20*x4^13*z^23 + 2*x1^48*x2^33*x3^21*x4^13*z^23 - 3*x1^47*x2^34*x3^21*x4^13*z^23 - x1^46*x2^35*x3^21*x4^13*z^23 - x1^45*x2^36*x3^21*x4^13*z^23 + x1^44*x2^37*x3^21*x4^13*z^23 + 2*x1^43*x2^38*x3^21*x4^13*z^23 + 2*x1^42*x2^39*x3^21*x4^13*z^23 + 5*x1^41*x2^40*x3^21*x4^13*z^23 - x1^48*x2^32*x3^22*x4^13*z^23 + 2*x1^47*x2^33*x3^22*x4^13*z^23 + 5*x1^46*x2^34*x3^22*x4^13*z^23 - x1^45*x2^35*x3^22*x4^13*z^23 + 2*x1^44*x2^36*x3^22*x4^13*z^23 - 4*x1^43*x2^37*x3^22*x4^13*z^23 - 6*x1^41*x2^39*x3^22*x4^13*z^23 - 2*x1^40*x2^40*x3^22*x4^13*z^23 - x1^48*x2^31*x3^23*x4^13*z^23 - 6*x1^46*x2^33*x3^23*x4^13*z^23 - 2*x1^45*x2^34*x3^23*x4^13*z^23 - 2*x1^44*x2^35*x3^23*x4^13*z^23 + 2*x1^42*x2^37*x3^23*x4^13*z^23 + 2*x1^41*x2^38*x3^23*x4^13*z^23 + 6*x1^40*x2^39*x3^23*x4^13*z^23 + x1^46*x2^32*x3^24*x4^13*z^23 + 6*x1^45*x2^33*x3^24*x4^13*z^23 + 4*x1^43*x2^35*x3^24*x4^13*z^23 - 4*x1^42*x2^36*x3^24*x4^13*z^23 - 6*x1^40*x2^38*x3^24*x4^13*z^23 - 2*x1^39*x2^39*x3^24*x4^13*z^23 - 3*x1^45*x2^32*x3^25*x4^13*z^23 - 2*x1^44*x2^33*x3^25*x4^13*z^23 + 2*x1^41*x2^36*x3^25*x4^13*z^23 + 2*x1^40*x2^37*x3^25*x4^13*z^23 + 6*x1^39*x2^38*x3^25*x4^13*z^23 + x1^45*x2^31*x3^26*x4^13*z^23 + 3*x1^44*x2^32*x3^26*x4^13*z^23 + 2*x1^43*x2^33*x3^26*x4^13*z^23 + 3*x1^42*x2^34*x3^26*x4^13*z^23 - 2*x1^41*x2^35*x3^26*x4^13*z^23 - 6*x1^39*x2^37*x3^26*x4^13*z^23 - 2*x1^38*x2^38*x3^26*x4^13*z^23 - x1^44*x2^31*x3^27*x4^13*z^23 - 2*x1^43*x2^32*x3^27*x4^13*z^23 - 2*x1^42*x2^33*x3^27*x4^13*z^23 + x1^40*x2^35*x3^27*x4^13*z^23 + 2*x1^39*x2^36*x3^27*x4^13*z^23 + 6*x1^38*x2^37*x3^27*x4^13*z^23 + x1^43*x2^31*x3^28*x4^13*z^23 + 2*x1^42*x2^32*x3^28*x4^13*z^23 + x1^41*x2^33*x3^28*x4^13*z^23 - 2*x1^40*x2^34*x3^28*x4^13*z^23 + x1^39*x2^35*x3^28*x4^13*z^23 - 4*x1^38*x2^36*x3^28*x4^13*z^23 - 2*x1^37*x2^37*x3^28*x4^13*z^23 - x1^41*x2^32*x3^29*x4^13*z^23 - 2*x1^40*x2^33*x3^29*x4^13*z^23 - x1^39*x2^34*x3^29*x4^13*z^23 + 4*x1^37*x2^36*x3^29*x4^13*z^23 - x1^37*x2^35*x3^30*x4^13*z^23 - x1^36*x2^36*x3^30*x4^13*z^23 - x1^37*x2^34*x3^31*x4^13*z^23 + x1^36*x2^35*x3^31*x4^13*z^23 - 2*x1^47*x2^37*x3^17*x4^14*z^23 - x1^46*x2^38*x3^17*x4^14*z^23 - x1^49*x2^34*x3^18*x4^14*z^23 + x1^48*x2^35*x3^18*x4^14*z^23 - x1^47*x2^36*x3^18*x4^14*z^23 + x1^46*x2^37*x3^18*x4^14*z^23 + 2*x1^45*x2^38*x3^18*x4^14*z^23 + x1^48*x2^34*x3^19*x4^14*z^23 + x1^47*x2^35*x3^19*x4^14*z^23 - 2*x1^46*x2^36*x3^19*x4^14*z^23 + 3*x1^44*x2^38*x3^19*x4^14*z^23 + x1^43*x2^39*x3^19*x4^14*z^23 + x1^42*x2^40*x3^19*x4^14*z^23 - 3*x1^48*x2^33*x3^20*x4^14*z^23 + x1^47*x2^34*x3^20*x4^14*z^23 - x1^44*x2^37*x3^20*x4^14*z^23 + 3*x1^43*x2^38*x3^20*x4^14*z^23 + x1^42*x2^39*x3^20*x4^14*z^23 - x1^41*x2^40*x3^20*x4^14*z^23 + x1^48*x2^32*x3^21*x4^14*z^23 + 2*x1^47*x2^33*x3^21*x4^14*z^23 + x1^46*x2^34*x3^21*x4^14*z^23 - 2*x1^44*x2^36*x3^21*x4^14*z^23 - 2*x1^42*x2^38*x3^21*x4^14*z^23 + 2*x1^41*x2^39*x3^21*x4^14*z^23 + x1^40*x2^40*x3^21*x4^14*z^23 - x1^48*x2^31*x3^22*x4^14*z^23 - 3*x1^47*x2^32*x3^22*x4^14*z^23 + x1^46*x2^33*x3^22*x4^14*z^23 + x1^45*x2^34*x3^22*x4^14*z^23 + 2*x1^44*x2^35*x3^22*x4^14*z^23 - x1^43*x2^36*x3^22*x4^14*z^23 + x1^42*x2^37*x3^22*x4^14*z^23 - 2*x1^40*x2^39*x3^22*x4^14*z^23 + 3*x1^47*x2^31*x3^23*x4^14*z^23 + x1^46*x2^32*x3^23*x4^14*z^23 + x1^44*x2^34*x3^23*x4^14*z^23 - 2*x1^41*x2^37*x3^23*x4^14*z^23 + 2*x1^40*x2^38*x3^23*x4^14*z^23 + x1^39*x2^39*x3^23*x4^14*z^23 - 3*x1^46*x2^31*x3^24*x4^14*z^23 + x1^45*x2^32*x3^24*x4^14*z^23 + 3*x1^43*x2^34*x3^24*x4^14*z^23 + x1^41*x2^36*x3^24*x4^14*z^23 - x1^40*x2^37*x3^24*x4^14*z^23 - 2*x1^39*x2^38*x3^24*x4^14*z^23 + x1^46*x2^30*x3^25*x4^14*z^23 + 2*x1^45*x2^31*x3^25*x4^14*z^23 - x1^43*x2^33*x3^25*x4^14*z^23 - x1^42*x2^34*x3^25*x4^14*z^23 + 2*x1^41*x2^35*x3^25*x4^14*z^23 - 2*x1^40*x2^36*x3^25*x4^14*z^23 + 2*x1^39*x2^37*x3^25*x4^14*z^23 - x1^45*x2^30*x3^26*x4^14*z^23 - 2*x1^44*x2^31*x3^26*x4^14*z^23 - 2*x1^43*x2^32*x3^26*x4^14*z^23 + x1^42*x2^33*x3^26*x4^14*z^23 + 2*x1^40*x2^35*x3^26*x4^14*z^23 + x1^39*x2^36*x3^26*x4^14*z^23 - 2*x1^38*x2^37*x3^26*x4^14*z^23 + 2*x1^43*x2^31*x3^27*x4^14*z^23 + x1^42*x2^32*x3^27*x4^14*z^23 - x1^41*x2^33*x3^27*x4^14*z^23 - x1^40*x2^34*x3^27*x4^14*z^23 - 2*x1^39*x2^35*x3^27*x4^14*z^23 + 2*x1^38*x2^36*x3^27*x4^14*z^23 + x1^37*x2^37*x3^27*x4^14*z^23 - x1^42*x2^31*x3^28*x4^14*z^23 - x1^40*x2^33*x3^28*x4^14*z^23 + x1^39*x2^34*x3^28*x4^14*z^23 - 2*x1^37*x2^36*x3^28*x4^14*z^23 + x1^39*x2^33*x3^29*x4^14*z^23 - 2*x1^38*x2^34*x3^29*x4^14*z^23 + x1^36*x2^36*x3^29*x4^14*z^23 + x1^47*x2^37*x3^16*x4^15*z^23 - 2*x1^46*x2^37*x3^17*x4^15*z^23 + 2*x1^46*x2^36*x3^18*x4^15*z^23 + 2*x1^45*x2^37*x3^18*x4^15*z^23 + x1^44*x2^38*x3^18*x4^15*z^23 + x1^43*x2^39*x3^18*x4^15*z^23 + x1^48*x2^33*x3^19*x4^15*z^23 - x1^47*x2^34*x3^19*x4^15*z^23 + x1^46*x2^35*x3^19*x4^15*z^23 - 4*x1^45*x2^36*x3^19*x4^15*z^23 - 2*x1^44*x2^37*x3^19*x4^15*z^23 - 3*x1^43*x2^38*x3^19*x4^15*z^23 - x1^48*x2^32*x3^20*x4^15*z^23 - 2*x1^47*x2^33*x3^20*x4^15*z^23 + 3*x1^45*x2^35*x3^20*x4^15*z^23 + x1^44*x2^36*x3^20*x4^15*z^23 + x1^43*x2^37*x3^20*x4^15*z^23 + 5*x1^42*x2^38*x3^20*x4^15*z^23 + 4*x1^47*x2^32*x3^21*x4^15*z^23 - 4*x1^44*x2^35*x3^21*x4^15*z^23 - 6*x1^42*x2^37*x3^21*x4^15*z^23 - 2*x1^41*x2^38*x3^21*x4^15*z^23 - 3*x1^47*x2^31*x3^22*x4^15*z^23 - 3*x1^46*x2^32*x3^22*x4^15*z^23 - x1^45*x2^33*x3^22*x4^15*z^23 + 2*x1^44*x2^34*x3^22*x4^15*z^23 + 2*x1^43*x2^35*x3^22*x4^15*z^23 + 2*x1^42*x2^36*x3^22*x4^15*z^23 + 6*x1^41*x2^37*x3^22*x4^15*z^23 + x1^47*x2^30*x3^23*x4^15*z^23 + 4*x1^46*x2^31*x3^23*x4^15*z^23 + 3*x1^44*x2^33*x3^23*x4^15*z^23 - 4*x1^43*x2^34*x3^23*x4^15*z^23 - 6*x1^41*x2^36*x3^23*x4^15*z^23 - 2*x1^40*x2^37*x3^23*x4^15*z^23 - 2*x1^46*x2^30*x3^24*x4^15*z^23 - 2*x1^45*x2^31*x3^24*x4^15*z^23 - 3*x1^44*x2^32*x3^24*x4^15*z^23 + 2*x1^42*x2^34*x3^24*x4^15*z^23 + 2*x1^41*x2^35*x3^24*x4^15*z^23 + 6*x1^40*x2^36*x3^24*x4^15*z^23 + 2*x1^45*x2^30*x3^25*x4^15*z^23 + x1^44*x2^31*x3^25*x4^15*z^23 + 3*x1^43*x2^32*x3^25*x4^15*z^23 - 3*x1^42*x2^33*x3^25*x4^15*z^23 - 6*x1^40*x2^35*x3^25*x4^15*z^23 - 2*x1^39*x2^36*x3^25*x4^15*z^23 - x1^44*x2^30*x3^26*x4^15*z^23 - x1^43*x2^31*x3^26*x4^15*z^23 + x1^41*x2^33*x3^26*x4^15*z^23 + x1^40*x2^34*x3^26*x4^15*z^23 + 6*x1^39*x2^35*x3^26*x4^15*z^23 - x1^42*x2^31*x3^27*x4^15*z^23 - 3*x1^41*x2^32*x3^27*x4^15*z^23 + x1^40*x2^33*x3^27*x4^15*z^23 - 4*x1^39*x2^34*x3^27*x4^15*z^23 - 2*x1^38*x2^35*x3^27*x4^15*z^23 - x1^39*x2^33*x3^28*x4^15*z^23 + 4*x1^38*x2^34*x3^28*x4^15*z^23 + x1^39*x2^32*x3^29*x4^15*z^23 - x1^38*x2^33*x3^29*x4^15*z^23 - x1^37*x2^34*x3^29*x4^15*z^23 + x1^37*x2^33*x3^30*x4^15*z^23 + 2*x1^45*x2^36*x3^18*x4^16*z^23 - x1^45*x2^35*x3^19*x4^16*z^23 - x1^44*x2^36*x3^19*x4^16*z^23 - x1^43*x2^37*x3^19*x4^16*z^23 - 3*x1^42*x2^38*x3^19*x4^16*z^23 + 4*x1^44*x2^35*x3^20*x4^16*z^23 + 2*x1^42*x2^37*x3^20*x4^16*z^23 + x1^41*x2^38*x3^20*x4^16*z^23 + x1^47*x2^31*x3^21*x4^16*z^23 + x1^46*x2^32*x3^21*x4^16*z^23 - x1^45*x2^33*x3^21*x4^16*z^23 - 2*x1^44*x2^34*x3^21*x4^16*z^23 - x1^43*x2^35*x3^21*x4^16*z^23 - x1^42*x2^36*x3^21*x4^16*z^23 - 5*x1^41*x2^37*x3^21*x4^16*z^23 - 2*x1^46*x2^31*x3^22*x4^16*z^23 - x1^44*x2^33*x3^22*x4^16*z^23 + 5*x1^43*x2^34*x3^22*x4^16*z^23 + 4*x1^41*x2^36*x3^22*x4^16*z^23 + 2*x1^40*x2^37*x3^22*x4^16*z^23 + x1^46*x2^30*x3^23*x4^16*z^23 + 2*x1^45*x2^31*x3^23*x4^16*z^23 - x1^44*x2^32*x3^23*x4^16*z^23 - 2*x1^43*x2^33*x3^23*x4^16*z^23 - 2*x1^42*x2^34*x3^23*x4^16*z^23 - 2*x1^41*x2^35*x3^23*x4^16*z^23 - 6*x1^40*x2^36*x3^23*x4^16*z^23 - x1^45*x2^30*x3^24*x4^16*z^23 - x1^43*x2^32*x3^24*x4^16*z^23 + 3*x1^42*x2^33*x3^24*x4^16*z^23 + 6*x1^40*x2^35*x3^24*x4^16*z^23 + 2*x1^39*x2^36*x3^24*x4^16*z^23 + x1^44*x2^30*x3^25*x4^16*z^23 + x1^43*x2^31*x3^25*x4^16*z^23 - 2*x1^42*x2^32*x3^25*x4^16*z^23 - 2*x1^41*x2^33*x3^25*x4^16*z^23 - 2*x1^40*x2^34*x3^25*x4^16*z^23 - 6*x1^39*x2^35*x3^25*x4^16*z^23 - x1^43*x2^30*x3^26*x4^16*z^23 + 3*x1^41*x2^32*x3^26*x4^16*z^23 + 5*x1^39*x2^34*x3^26*x4^16*z^23 + 2*x1^38*x2^35*x3^26*x4^16*z^23 + x1^42*x2^30*x3^27*x4^16*z^23 - x1^40*x2^32*x3^27*x4^16*z^23 - 5*x1^38*x2^34*x3^27*x4^16*z^23 + x1^40*x2^31*x3^28*x4^16*z^23 + 2*x1^38*x2^33*x3^28*x4^16*z^23 + 2*x1^37*x2^34*x3^28*x4^16*z^23 + x1^38*x2^32*x3^29*x4^16*z^23 - 2*x1^37*x2^33*x3^29*x4^16*z^23 + x1^36*x2^33*x3^30*x4^16*z^23 - x1^43*x2^34*x3^21*x4^17*z^23 + x1^42*x2^35*x3^21*x4^17*z^23 - x1^41*x2^36*x3^21*x4^17*z^23 - x1^40*x2^37*x3^21*x4^17*z^23 + x1^42*x2^34*x3^22*x4^17*z^23 + x1^41*x2^35*x3^22*x4^17*z^23 + x1^40*x2^36*x3^22*x4^17*z^23 - 2*x1^42*x2^33*x3^23*x4^17*z^23 - x1^39*x2^36*x3^23*x4^17*z^23 - x1^43*x2^31*x3^24*x4^17*z^23 + x1^40*x2^34*x3^24*x4^17*z^23 + 2*x1^39*x2^35*x3^24*x4^17*z^23 + x1^42*x2^31*x3^25*x4^17*z^23 + x1^40*x2^33*x3^25*x4^17*z^23 - 2*x1^39*x2^34*x3^25*x4^17*z^23 - x1^39*x2^33*x3^26*x4^17*z^23 + 2*x1^38*x2^34*x3^26*x4^17*z^23 - x1^37*x2^34*x3^27*x4^17*z^23 + x1^52*x2^38*x3^20*z^22 - x1^52*x2^38*x3^19*x4*z^22 + x1^50*x2^40*x3^19*x4*z^22 + 2*x1^51*x2^38*x3^20*x4*z^22 - x1^50*x2^39*x3^20*x4*z^22 - 2*x1^51*x2^37*x3^21*x4*z^22 - x1^50*x2^38*x3^21*x4*z^22 - x1^48*x2^40*x3^21*x4*z^22 + 3*x1^50*x2^37*x3^22*x4*z^22 - x1^49*x2^38*x3^22*x4*z^22 - x1^49*x2^37*x3^23*x4*z^22 - x1^48*x2^38*x3^23*x4*z^22 - x1^47*x2^39*x3^23*x4*z^22 + 2*x1^52*x2^38*x3^18*x4^2*z^22 - 4*x1^51*x2^38*x3^19*x4^2*z^22 - x1^49*x2^40*x3^19*x4^2*z^22 + 2*x1^51*x2^37*x3^20*x4^2*z^22 + 5*x1^50*x2^38*x3^20*x4^2*z^22 - 2*x1^49*x2^39*x3^20*x4^2*z^22 - x1^47*x2^41*x3^20*x4^2*z^22 - 5*x1^50*x2^37*x3^21*x4^2*z^22 - x1^49*x2^38*x3^21*x4^2*z^22 - 2*x1^48*x2^39*x3^21*x4^2*z^22 + x1^46*x2^41*x3^21*x4^2*z^22 + 2*x1^50*x2^36*x3^22*x4^2*z^22 + 4*x1^49*x2^37*x3^22*x4^2*z^22 + 3*x1^47*x2^39*x3^22*x4^2*z^22 - 4*x1^49*x2^36*x3^23*x4^2*z^22 - 2*x1^47*x2^38*x3^23*x4^2*z^22 - x1^46*x2^39*x3^23*x4^2*z^22 + x1^49*x2^35*x3^24*x4^2*z^22 + 2*x1^48*x2^36*x3^24*x4^2*z^22 + 2*x1^46*x2^38*x3^24*x4^2*z^22 - x1^48*x2^35*x3^25*x4^2*z^22 + x1^47*x2^36*x3^25*x4^2*z^22 - x1^46*x2^37*x3^25*x4^2*z^22 - x1^45*x2^38*x3^25*x4^2*z^22 - x1^52*x2^38*x3^17*x4^3*z^22 + x1^51*x2^38*x3^18*x4^3*z^22 + x1^52*x2^36*x3^19*x4^3*z^22 - x1^51*x2^37*x3^19*x4^3*z^22 - x1^50*x2^38*x3^19*x4^3*z^22 - x1^48*x2^40*x3^19*x4^3*z^22 - x1^52*x2^35*x3^20*x4^3*z^22 - x1^51*x2^36*x3^20*x4^3*z^22 + 3*x1^50*x2^37*x3^20*x4^3*z^22 + x1^47*x2^40*x3^20*x4^3*z^22 + x1^51*x2^35*x3^21*x4^3*z^22 - 3*x1^49*x2^37*x3^21*x4^3*z^22 + x1^48*x2^38*x3^21*x4^3*z^22 - x1^47*x2^39*x3^21*x4^3*z^22 - x1^46*x2^40*x3^21*x4^3*z^22 - x1^45*x2^41*x3^21*x4^3*z^22 - x1^51*x2^34*x3^22*x4^3*z^22 - x1^50*x2^35*x3^22*x4^3*z^22 + 2*x1^49*x2^36*x3^22*x4^3*z^22 + x1^48*x2^37*x3^22*x4^3*z^22 + x1^47*x2^38*x3^22*x4^3*z^22 - x1^46*x2^39*x3^22*x4^3*z^22 + x1^50*x2^34*x3^23*x4^3*z^22 - 2*x1^49*x2^35*x3^23*x4^3*z^22 - 2*x1^48*x2^36*x3^23*x4^3*z^22 + 2*x1^47*x2^37*x3^23*x4^3*z^22 - x1^46*x2^38*x3^23*x4^3*z^22 - x1^45*x2^39*x3^23*x4^3*z^22 + x1^43*x2^41*x3^23*x4^3*z^22 + 2*x1^48*x2^35*x3^24*x4^3*z^22 + 2*x1^46*x2^37*x3^24*x4^3*z^22 + x1^45*x2^38*x3^24*x4^3*z^22 - x1^48*x2^34*x3^25*x4^3*z^22 - x1^47*x2^35*x3^25*x4^3*z^22 + x1^46*x2^36*x3^25*x4^3*z^22 - x1^45*x2^37*x3^25*x4^3*z^22 + x1^47*x2^34*x3^26*x4^3*z^22 - x1^46*x2^35*x3^26*x4^3*z^22 + x1^45*x2^36*x3^26*x4^3*z^22 + x1^44*x2^37*x3^26*x4^3*z^22 - x1^53*x2^37*x3^16*x4^4*z^22 + x1^53*x2^36*x3^17*x4^4*z^22 + x1^52*x2^37*x3^17*x4^4*z^22 - x1^51*x2^38*x3^17*x4^4*z^22 - 4*x1^52*x2^36*x3^18*x4^4*z^22 - x1^49*x2^39*x3^18*x4^4*z^22 + 2*x1^52*x2^35*x3^19*x4^4*z^22 + 4*x1^51*x2^36*x3^19*x4^4*z^22 - 2*x1^50*x2^37*x3^19*x4^4*z^22 - 6*x1^51*x2^35*x3^20*x4^4*z^22 - x1^50*x2^36*x3^20*x4^4*z^22 - x1^48*x2^38*x3^20*x4^4*z^22 - x1^46*x2^40*x3^20*x4^4*z^22 + 2*x1^51*x2^34*x3^21*x4^4*z^22 + 6*x1^50*x2^35*x3^21*x4^4*z^22 + 4*x1^48*x2^37*x3^21*x4^4*z^22 + x1^47*x2^38*x3^21*x4^4*z^22 - x1^46*x2^39*x3^21*x4^4*z^22 - 2*x1^45*x2^40*x3^21*x4^4*z^22 - 5*x1^50*x2^34*x3^22*x4^4*z^22 - 2*x1^49*x2^35*x3^22*x4^4*z^22 - x1^48*x2^36*x3^22*x4^4*z^22 - x1^47*x2^37*x3^22*x4^4*z^22 + x1^46*x2^38*x3^22*x4^4*z^22 + x1^45*x2^39*x3^22*x4^4*z^22 - x1^43*x2^41*x3^22*x4^4*z^22 + 2*x1^50*x2^33*x3^23*x4^4*z^22 + 3*x1^49*x2^34*x3^23*x4^4*z^22 - x1^48*x2^35*x3^23*x4^4*z^22 + 3*x1^47*x2^36*x3^23*x4^4*z^22 - 2*x1^44*x2^39*x3^23*x4^4*z^22 - 4*x1^49*x2^33*x3^24*x4^4*z^22 - x1^47*x2^35*x3^24*x4^4*z^22 - 3*x1^46*x2^36*x3^24*x4^4*z^22 + x1^45*x2^37*x3^24*x4^4*z^22 - x1^43*x2^39*x3^24*x4^4*z^22 - x1^42*x2^40*x3^24*x4^4*z^22 + 2*x1^49*x2^32*x3^25*x4^4*z^22 + x1^48*x2^33*x3^25*x4^4*z^22 - 2*x1^47*x2^34*x3^25*x4^4*z^22 + 2*x1^46*x2^35*x3^25*x4^4*z^22 - x1^45*x2^36*x3^25*x4^4*z^22 - x1^48*x2^32*x3^26*x4^4*z^22 + x1^47*x2^33*x3^26*x4^4*z^22 - x1^46*x2^34*x3^26*x4^4*z^22 - 2*x1^45*x2^35*x3^26*x4^4*z^22 + 2*x1^44*x2^36*x3^26*x4^4*z^22 - 2*x1^46*x2^33*x3^27*x4^4*z^22 - x1^45*x2^34*x3^27*x4^4*z^22 - x1^44*x2^35*x3^27*x4^4*z^22 - x1^53*x2^36*x3^16*x4^5*z^22 + 4*x1^52*x2^36*x3^17*x4^5*z^22 + x1^51*x2^37*x3^17*x4^5*z^22 + 2*x1^50*x2^38*x3^17*x4^5*z^22 - 2*x1^52*x2^35*x3^18*x4^5*z^22 - 4*x1^51*x2^36*x3^18*x4^5*z^22 + x1^50*x2^37*x3^18*x4^5*z^22 + 2*x1^48*x2^39*x3^18*x4^5*z^22 + 6*x1^51*x2^35*x3^19*x4^5*z^22 + x1^49*x2^37*x3^19*x4^5*z^22 - x1^47*x2^39*x3^19*x4^5*z^22 - 2*x1^51*x2^34*x3^20*x4^5*z^22 - 6*x1^50*x2^35*x3^20*x4^5*z^22 + 3*x1^49*x2^36*x3^20*x4^5*z^22 - 5*x1^48*x2^37*x3^20*x4^5*z^22 + 2*x1^47*x2^38*x3^20*x4^5*z^22 + 2*x1^46*x2^39*x3^20*x4^5*z^22 + 3*x1^45*x2^40*x3^20*x4^5*z^22 + 6*x1^50*x2^34*x3^21*x4^5*z^22 + 2*x1^49*x2^35*x3^21*x4^5*z^22 - x1^48*x2^36*x3^21*x4^5*z^22 + 2*x1^47*x2^37*x3^21*x4^5*z^22 - 2*x1^46*x2^38*x3^21*x4^5*z^22 - x1^45*x2^39*x3^21*x4^5*z^22 - 2*x1^44*x2^40*x3^21*x4^5*z^22 - 2*x1^50*x2^33*x3^22*x4^5*z^22 - 6*x1^49*x2^34*x3^22*x4^5*z^22 + 2*x1^48*x2^35*x3^22*x4^5*z^22 - 3*x1^47*x2^36*x3^22*x4^5*z^22 + x1^46*x2^37*x3^22*x4^5*z^22 + 3*x1^44*x2^39*x3^22*x4^5*z^22 - x1^42*x2^41*x3^22*x4^5*z^22 + 6*x1^49*x2^33*x3^23*x4^5*z^22 + x1^48*x2^34*x3^23*x4^5*z^22 + 2*x1^47*x2^35*x3^23*x4^5*z^22 + 2*x1^46*x2^36*x3^23*x4^5*z^22 - 2*x1^45*x2^37*x3^23*x4^5*z^22 - x1^44*x2^38*x3^23*x4^5*z^22 - x1^43*x2^39*x3^23*x4^5*z^22 - 2*x1^49*x2^32*x3^24*x4^5*z^22 - 4*x1^48*x2^33*x3^24*x4^5*z^22 + 2*x1^47*x2^34*x3^24*x4^5*z^22 - 3*x1^46*x2^35*x3^24*x4^5*z^22 + x1^45*x2^36*x3^24*x4^5*z^22 + 3*x1^43*x2^38*x3^24*x4^5*z^22 - x1^42*x2^39*x3^24*x4^5*z^22 + 5*x1^48*x2^32*x3^25*x4^5*z^22 + 3*x1^45*x2^35*x3^25*x4^5*z^22 - 2*x1^44*x2^36*x3^25*x4^5*z^22 + x1^43*x2^37*x3^25*x4^5*z^22 - x1^42*x2^38*x3^25*x4^5*z^22 - x1^48*x2^31*x3^26*x4^5*z^22 - x1^47*x2^32*x3^26*x4^5*z^22 + x1^46*x2^33*x3^26*x4^5*z^22 - 3*x1^45*x2^34*x3^26*x4^5*z^22 - x1^44*x2^35*x3^26*x4^5*z^22 + x1^42*x2^37*x3^26*x4^5*z^22 - x1^41*x2^38*x3^26*x4^5*z^22 - x1^40*x2^39*x3^26*x4^5*z^22 + x1^47*x2^31*x3^27*x4^5*z^22 + x1^44*x2^34*x3^27*x4^5*z^22 - 2*x1^43*x2^35*x3^27*x4^5*z^22 + x1^45*x2^32*x3^28*x4^5*z^22 + x1^44*x2^33*x3^28*x4^5*z^22 - 2*x1^42*x2^34*x3^29*x4^5*z^22 - x1^52*x2^36*x3^16*x4^6*z^22 + x1^52*x2^35*x3^17*x4^6*z^22 + x1^51*x2^36*x3^17*x4^6*z^22 - x1^50*x2^37*x3^17*x4^6*z^22 - 3*x1^51*x2^35*x3^18*x4^6*z^22 + 2*x1^50*x2^36*x3^18*x4^6*z^22 - x1^48*x2^38*x3^18*x4^6*z^22 + 3*x1^50*x2^35*x3^19*x4^6*z^22 - 3*x1^49*x2^36*x3^19*x4^6*z^22 + 2*x1^48*x2^37*x3^19*x4^6*z^22 - 2*x1^50*x2^34*x3^20*x4^6*z^22 + 2*x1^48*x2^36*x3^20*x4^6*z^22 - x1^47*x2^37*x3^20*x4^6*z^22 + x1^46*x2^38*x3^20*x4^6*z^22 - x1^45*x2^39*x3^20*x4^6*z^22 + x1^50*x2^33*x3^21*x4^6*z^22 + 2*x1^49*x2^34*x3^21*x4^6*z^22 - 4*x1^48*x2^35*x3^21*x4^6*z^22 + x1^47*x2^36*x3^21*x4^6*z^22 - x1^46*x2^37*x3^21*x4^6*z^22 - x1^44*x2^39*x3^21*x4^6*z^22 - 2*x1^49*x2^33*x3^22*x4^6*z^22 + 3*x1^47*x2^35*x3^22*x4^6*z^22 - x1^46*x2^36*x3^22*x4^6*z^22 + 4*x1^45*x2^37*x3^22*x4^6*z^22 + x1^44*x2^38*x3^22*x4^6*z^22 + x1^49*x2^32*x3^23*x4^6*z^22 + 2*x1^48*x2^33*x3^23*x4^6*z^22 - 4*x1^47*x2^34*x3^23*x4^6*z^22 - x1^46*x2^35*x3^23*x4^6*z^22 - 5*x1^45*x2^36*x3^23*x4^6*z^22 - x1^44*x2^37*x3^23*x4^6*z^22 + 2*x1^42*x2^39*x3^23*x4^6*z^22 - 3*x1^48*x2^32*x3^24*x4^6*z^22 + 2*x1^47*x2^33*x3^24*x4^6*z^22 + 2*x1^46*x2^34*x3^24*x4^6*z^22 - 2*x1^45*x2^35*x3^24*x4^6*z^22 + 3*x1^44*x2^36*x3^24*x4^6*z^22 - x1^43*x2^37*x3^24*x4^6*z^22 + 2*x1^42*x2^38*x3^24*x4^6*z^22 - 2*x1^41*x2^39*x3^24*x4^6*z^22 - x1^40*x2^40*x3^24*x4^6*z^22 - x1^48*x2^31*x3^25*x4^6*z^22 + 2*x1^47*x2^32*x3^25*x4^6*z^22 - 2*x1^46*x2^33*x3^25*x4^6*z^22 + x1^45*x2^34*x3^25*x4^6*z^22 - x1^44*x2^35*x3^25*x4^6*z^22 - 2*x1^43*x2^36*x3^25*x4^6*z^22 - x1^42*x2^37*x3^25*x4^6*z^22 + 2*x1^41*x2^38*x3^25*x4^6*z^22 - x1^47*x2^31*x3^26*x4^6*z^22 - x1^46*x2^32*x3^26*x4^6*z^22 + 2*x1^45*x2^33*x3^26*x4^6*z^22 + 2*x1^43*x2^35*x3^26*x4^6*z^22 - x1^42*x2^36*x3^26*x4^6*z^22 + 2*x1^41*x2^37*x3^26*x4^6*z^22 + x1^40*x2^38*x3^26*x4^6*z^22 - x1^45*x2^32*x3^27*x4^6*z^22 + x1^44*x2^33*x3^27*x4^6*z^22 + x1^43*x2^34*x3^27*x4^6*z^22 - x1^42*x2^35*x3^27*x4^6*z^22 - x1^41*x2^36*x3^27*x4^6*z^22 + x1^40*x2^37*x3^27*x4^6*z^22 + x1^39*x2^38*x3^27*x4^6*z^22 + x1^44*x2^32*x3^28*x4^6*z^22 + x1^43*x2^33*x3^28*x4^6*z^22 + 2*x1^42*x2^34*x3^28*x4^6*z^22 - x1^43*x2^32*x3^29*x4^6*z^22 + x1^41*x2^33*x3^30*x4^6*z^22 - x1^51*x2^38*x3^14*x4^7*z^22 + x1^52*x2^36*x3^15*x4^7*z^22 + x1^50*x2^38*x3^15*x4^7*z^22 + x1^52*x2^35*x3^16*x4^7*z^22 - x1^51*x2^36*x3^16*x4^7*z^22 - x1^50*x2^37*x3^16*x4^7*z^22 - x1^48*x2^39*x3^16*x4^7*z^22 + 2*x1^50*x2^36*x3^17*x4^7*z^22 + 3*x1^49*x2^37*x3^17*x4^7*z^22 + 2*x1^47*x2^39*x3^17*x4^7*z^22 + x1^46*x2^40*x3^17*x4^7*z^22 - 2*x1^49*x2^36*x3^18*x4^7*z^22 - x1^47*x2^38*x3^18*x4^7*z^22 + x1^44*x2^41*x3^18*x4^7*z^22 - x1^49*x2^35*x3^19*x4^7*z^22 + 2*x1^48*x2^36*x3^19*x4^7*z^22 + 2*x1^46*x2^38*x3^19*x4^7*z^22 - 2*x1^45*x2^39*x3^19*x4^7*z^22 - x1^43*x2^41*x3^19*x4^7*z^22 - 2*x1^47*x2^36*x3^20*x4^7*z^22 - x1^46*x2^37*x3^20*x4^7*z^22 - x1^45*x2^38*x3^20*x4^7*z^22 - x1^43*x2^40*x3^20*x4^7*z^22 + x1^42*x2^41*x3^20*x4^7*z^22 + x1^45*x2^37*x3^21*x4^7*z^22 - x1^44*x2^38*x3^21*x4^7*z^22 - x1^42*x2^40*x3^21*x4^7*z^22 - x1^41*x2^41*x3^21*x4^7*z^22 + x1^45*x2^36*x3^22*x4^7*z^22 - 2*x1^44*x2^37*x3^22*x4^7*z^22 + x1^43*x2^38*x3^22*x4^7*z^22 + 2*x1^42*x2^39*x3^22*x4^7*z^22 + x1^41*x2^40*x3^22*x4^7*z^22 - 2*x1^44*x2^36*x3^23*x4^7*z^22 - x1^41*x2^39*x3^23*x4^7*z^22 + x1^44*x2^35*x3^24*x4^7*z^22 + 2*x1^40*x2^39*x3^24*x4^7*z^22 + 2*x1^47*x2^31*x3^25*x4^7*z^22 - x1^46*x2^32*x3^25*x4^7*z^22 + 2*x1^44*x2^34*x3^25*x4^7*z^22 - x1^43*x2^35*x3^25*x4^7*z^22 - x1^42*x2^36*x3^25*x4^7*z^22 - 2*x1^46*x2^31*x3^26*x4^7*z^22 - x1^43*x2^34*x3^26*x4^7*z^22 + x1^41*x2^36*x3^26*x4^7*z^22 - x1^40*x2^37*x3^26*x4^7*z^22 + x1^39*x2^38*x3^26*x4^7*z^22 + x1^46*x2^30*x3^27*x4^7*z^22 + 2*x1^45*x2^31*x3^27*x4^7*z^22 + x1^44*x2^32*x3^27*x4^7*z^22 - x1^43*x2^33*x3^27*x4^7*z^22 - x1^42*x2^34*x3^27*x4^7*z^22 - 2*x1^40*x2^36*x3^27*x4^7*z^22 - x1^41*x2^34*x3^28*x4^7*z^22 - x1^42*x2^32*x3^29*x4^7*z^22 - x1^52*x2^35*x3^15*x4^8*z^22 - x1^51*x2^36*x3^15*x4^8*z^22 + 2*x1^50*x2^37*x3^15*x4^8*z^22 - x1^49*x2^38*x3^15*x4^8*z^22 + x1^48*x2^39*x3^15*x4^8*z^22 + x1^51*x2^35*x3^16*x4^8*z^22 - 4*x1^49*x2^37*x3^16*x4^8*z^22 + 2*x1^48*x2^38*x3^16*x4^8*z^22 - x1^47*x2^39*x3^16*x4^8*z^22 + x1^46*x2^40*x3^16*x4^8*z^22 + 2*x1^49*x2^36*x3^17*x4^8*z^22 - x1^46*x2^39*x3^17*x4^8*z^22 - 2*x1^45*x2^40*x3^17*x4^8*z^22 - 2*x1^50*x2^34*x3^18*x4^8*z^22 - x1^49*x2^35*x3^18*x4^8*z^22 - 4*x1^48*x2^36*x3^18*x4^8*z^22 + x1^47*x2^37*x3^18*x4^8*z^22 - x1^46*x2^38*x3^18*x4^8*z^22 + 3*x1^45*x2^39*x3^18*x4^8*z^22 + 2*x1^44*x2^40*x3^18*x4^8*z^22 + x1^43*x2^41*x3^18*x4^8*z^22 + x1^50*x2^33*x3^19*x4^8*z^22 + 2*x1^49*x2^34*x3^19*x4^8*z^22 + 3*x1^48*x2^35*x3^19*x4^8*z^22 - x1^46*x2^37*x3^19*x4^8*z^22 - x1^45*x2^38*x3^19*x4^8*z^22 - 2*x1^44*x2^39*x3^19*x4^8*z^22 - 3*x1^43*x2^40*x3^19*x4^8*z^22 - x1^42*x2^41*x3^19*x4^8*z^22 - 2*x1^49*x2^33*x3^20*x4^8*z^22 - x1^48*x2^34*x3^20*x4^8*z^22 - 3*x1^47*x2^35*x3^20*x4^8*z^22 + x1^46*x2^36*x3^20*x4^8*z^22 - 2*x1^45*x2^37*x3^20*x4^8*z^22 + 3*x1^44*x2^38*x3^20*x4^8*z^22 + x1^43*x2^39*x3^20*x4^8*z^22 + 5*x1^42*x2^40*x3^20*x4^8*z^22 + 2*x1^48*x2^33*x3^21*x4^8*z^22 + 4*x1^47*x2^34*x3^21*x4^8*z^22 + 4*x1^46*x2^35*x3^21*x4^8*z^22 + x1^45*x2^36*x3^21*x4^8*z^22 - 2*x1^44*x2^37*x3^21*x4^8*z^22 - 3*x1^43*x2^38*x3^21*x4^8*z^22 - 2*x1^42*x2^39*x3^21*x4^8*z^22 - 5*x1^41*x2^40*x3^21*x4^8*z^22 - 2*x1^48*x2^32*x3^22*x4^8*z^22 - 3*x1^47*x2^33*x3^22*x4^8*z^22 - 5*x1^46*x2^34*x3^22*x4^8*z^22 - x1^44*x2^36*x3^22*x4^8*z^22 + 5*x1^43*x2^37*x3^22*x4^8*z^22 + x1^42*x2^38*x3^22*x4^8*z^22 + 4*x1^41*x2^39*x3^22*x4^8*z^22 + 2*x1^40*x2^40*x3^22*x4^8*z^22 + x1^48*x2^31*x3^23*x4^8*z^22 + 2*x1^47*x2^32*x3^23*x4^8*z^22 + 4*x1^46*x2^33*x3^23*x4^8*z^22 + 2*x1^45*x2^34*x3^23*x4^8*z^22 + 2*x1^44*x2^35*x3^23*x4^8*z^22 + x1^43*x2^36*x3^23*x4^8*z^22 - 2*x1^42*x2^37*x3^23*x4^8*z^22 - 2*x1^41*x2^38*x3^23*x4^8*z^22 - 3*x1^40*x2^39*x3^23*x4^8*z^22 - 2*x1^47*x2^31*x3^24*x4^8*z^22 - 5*x1^45*x2^33*x3^24*x4^8*z^22 - x1^43*x2^35*x3^24*x4^8*z^22 + 4*x1^42*x2^36*x3^24*x4^8*z^22 + 2*x1^41*x2^37*x3^24*x4^8*z^22 + 3*x1^40*x2^38*x3^24*x4^8*z^22 + x1^39*x2^39*x3^24*x4^8*z^22 + 2*x1^46*x2^31*x3^25*x4^8*z^22 + 2*x1^44*x2^33*x3^25*x4^8*z^22 - x1^43*x2^34*x3^25*x4^8*z^22 + x1^42*x2^35*x3^25*x4^8*z^22 - 2*x1^41*x2^36*x3^25*x4^8*z^22 - x1^40*x2^37*x3^25*x4^8*z^22 - 4*x1^39*x2^38*x3^25*x4^8*z^22 - 2*x1^46*x2^30*x3^26*x4^8*z^22 + x1^44*x2^32*x3^26*x4^8*z^22 - 2*x1^43*x2^33*x3^26*x4^8*z^22 - 3*x1^42*x2^34*x3^26*x4^8*z^22 + 2*x1^40*x2^36*x3^26*x4^8*z^22 + x1^39*x2^37*x3^26*x4^8*z^22 + x1^38*x2^38*x3^26*x4^8*z^22 + 2*x1^45*x2^30*x3^27*x4^8*z^22 + x1^44*x2^31*x3^27*x4^8*z^22 + 2*x1^43*x2^32*x3^27*x4^8*z^22 + 2*x1^42*x2^33*x3^27*x4^8*z^22 - x1^39*x2^36*x3^27*x4^8*z^22 - 2*x1^38*x2^37*x3^27*x4^8*z^22 - x1^43*x2^31*x3^28*x4^8*z^22 + x1^42*x2^32*x3^28*x4^8*z^22 + x1^39*x2^35*x3^28*x4^8*z^22 + x1^38*x2^36*x3^28*x4^8*z^22 + x1^43*x2^30*x3^29*x4^8*z^22 + x1^42*x2^31*x3^29*x4^8*z^22 - x1^38*x2^35*x3^29*x4^8*z^22 - x1^37*x2^36*x3^29*x4^8*z^22 - x1^40*x2^32*x3^30*x4^8*z^22 - x1^51*x2^36*x3^14*x4^9*z^22 + 2*x1^49*x2^37*x3^15*x4^9*z^22 + x1^47*x2^39*x3^15*x4^9*z^22 - x1^51*x2^34*x3^16*x4^9*z^22 - x1^50*x2^35*x3^16*x4^9*z^22 - 3*x1^48*x2^37*x3^16*x4^9*z^22 - x1^47*x2^38*x3^16*x4^9*z^22 + x1^45*x2^40*x3^16*x4^9*z^22 + 2*x1^50*x2^34*x3^17*x4^9*z^22 + 3*x1^48*x2^36*x3^17*x4^9*z^22 + 2*x1^47*x2^37*x3^17*x4^9*z^22 + x1^46*x2^38*x3^17*x4^9*z^22 - x1^44*x2^40*x3^17*x4^9*z^22 - x1^50*x2^33*x3^18*x4^9*z^22 - 3*x1^49*x2^34*x3^18*x4^9*z^22 - x1^48*x2^35*x3^18*x4^9*z^22 - x1^46*x2^37*x3^18*x4^9*z^22 - x1^45*x2^38*x3^18*x4^9*z^22 - x1^44*x2^39*x3^18*x4^9*z^22 + x1^43*x2^40*x3^18*x4^9*z^22 + 4*x1^49*x2^33*x3^19*x4^9*z^22 + x1^48*x2^34*x3^19*x4^9*z^22 + 3*x1^47*x2^35*x3^19*x4^9*z^22 + x1^46*x2^36*x3^19*x4^9*z^22 + x1^45*x2^37*x3^19*x4^9*z^22 - x1^44*x2^38*x3^19*x4^9*z^22 + x1^43*x2^39*x3^19*x4^9*z^22 - 2*x1^42*x2^40*x3^19*x4^9*z^22 - x1^49*x2^32*x3^20*x4^9*z^22 - 4*x1^48*x2^33*x3^20*x4^9*z^22 - 2*x1^47*x2^34*x3^20*x4^9*z^22 - 3*x1^46*x2^35*x3^20*x4^9*z^22 + x1^44*x2^37*x3^20*x4^9*z^22 - x1^42*x2^39*x3^20*x4^9*z^22 + 2*x1^41*x2^40*x3^20*x4^9*z^22 + 4*x1^48*x2^32*x3^21*x4^9*z^22 + 2*x1^47*x2^33*x3^21*x4^9*z^22 + 3*x1^46*x2^34*x3^21*x4^9*z^22 + 2*x1^44*x2^36*x3^21*x4^9*z^22 - x1^43*x2^37*x3^21*x4^9*z^22 - 2*x1^41*x2^39*x3^21*x4^9*z^22 - x1^40*x2^40*x3^21*x4^9*z^22 - x1^48*x2^31*x3^22*x4^9*z^22 - 4*x1^47*x2^32*x3^22*x4^9*z^22 - 2*x1^46*x2^33*x3^22*x4^9*z^22 - 4*x1^45*x2^34*x3^22*x4^9*z^22 + x1^44*x2^35*x3^22*x4^9*z^22 + 5*x1^42*x2^37*x3^22*x4^9*z^22 + 2*x1^41*x2^38*x3^22*x4^9*z^22 + 2*x1^40*x2^39*x3^22*x4^9*z^22 + 4*x1^47*x2^31*x3^23*x4^9*z^22 + 2*x1^46*x2^32*x3^23*x4^9*z^22 + 4*x1^45*x2^33*x3^23*x4^9*z^22 - x1^43*x2^35*x3^23*x4^9*z^22 - 4*x1^42*x2^36*x3^23*x4^9*z^22 - 4*x1^41*x2^37*x3^23*x4^9*z^22 - 2*x1^40*x2^38*x3^23*x4^9*z^22 - x1^47*x2^30*x3^24*x4^9*z^22 - 4*x1^46*x2^31*x3^24*x4^9*z^22 - x1^45*x2^32*x3^24*x4^9*z^22 - 2*x1^44*x2^33*x3^24*x4^9*z^22 + x1^43*x2^34*x3^24*x4^9*z^22 - x1^42*x2^35*x3^24*x4^9*z^22 + 3*x1^41*x2^36*x3^24*x4^9*z^22 + x1^40*x2^37*x3^24*x4^9*z^22 + x1^39*x2^38*x3^24*x4^9*z^22 + 4*x1^46*x2^30*x3^25*x4^9*z^22 + 2*x1^44*x2^32*x3^25*x4^9*z^22 + x1^42*x2^34*x3^25*x4^9*z^22 - x1^41*x2^35*x3^25*x4^9*z^22 - 3*x1^40*x2^36*x3^25*x4^9*z^22 - x1^38*x2^38*x3^25*x4^9*z^22 - 4*x1^45*x2^30*x3^26*x4^9*z^22 + x1^44*x2^31*x3^26*x4^9*z^22 - 3*x1^43*x2^32*x3^26*x4^9*z^22 + x1^41*x2^34*x3^26*x4^9*z^22 + 3*x1^40*x2^35*x3^26*x4^9*z^22 + x1^39*x2^36*x3^26*x4^9*z^22 + x1^38*x2^37*x3^26*x4^9*z^22 + 2*x1^45*x2^29*x3^27*x4^9*z^22 + 3*x1^44*x2^30*x3^27*x4^9*z^22 - 2*x1^40*x2^34*x3^27*x4^9*z^22 - 3*x1^39*x2^35*x3^27*x4^9*z^22 - x1^37*x2^37*x3^27*x4^9*z^22 - 2*x1^44*x2^29*x3^28*x4^9*z^22 - 2*x1^43*x2^30*x3^28*x4^9*z^22 - x1^42*x2^31*x3^28*x4^9*z^22 + 2*x1^39*x2^34*x3^28*x4^9*z^22 + x1^38*x2^35*x3^28*x4^9*z^22 + x1^42*x2^30*x3^29*x4^9*z^22 - x1^39*x2^33*x3^29*x4^9*z^22 + x1^37*x2^35*x3^29*x4^9*z^22 - x1^41*x2^30*x3^30*x4^9*z^22 + x1^40*x2^31*x3^30*x4^9*z^22 + x1^39*x2^32*x3^30*x4^9*z^22 - x1^38*x2^33*x3^30*x4^9*z^22 + x1^36*x2^35*x3^30*x4^9*z^22 - x1^48*x2^38*x3^14*x4^10*z^22 - x1^49*x2^36*x3^15*x4^10*z^22 - x1^46*x2^39*x3^15*x4^10*z^22 - x1^45*x2^39*x3^16*x4^10*z^22 + x1^50*x2^33*x3^17*x4^10*z^22 - x1^49*x2^34*x3^17*x4^10*z^22 + 2*x1^47*x2^36*x3^17*x4^10*z^22 + 3*x1^46*x2^37*x3^17*x4^10*z^22 + x1^45*x2^38*x3^17*x4^10*z^22 + x1^44*x2^39*x3^17*x4^10*z^22 + x1^43*x2^40*x3^17*x4^10*z^22 + x1^49*x2^33*x3^18*x4^10*z^22 - x1^47*x2^35*x3^18*x4^10*z^22 - 2*x1^46*x2^36*x3^18*x4^10*z^22 - x1^45*x2^37*x3^18*x4^10*z^22 - x1^43*x2^39*x3^18*x4^10*z^22 - x1^42*x2^40*x3^18*x4^10*z^22 + 2*x1^47*x2^34*x3^19*x4^10*z^22 - x1^46*x2^35*x3^19*x4^10*z^22 + x1^45*x2^36*x3^19*x4^10*z^22 + 2*x1^44*x2^37*x3^19*x4^10*z^22 + 2*x1^43*x2^38*x3^19*x4^10*z^22 + x1^42*x2^39*x3^19*x4^10*z^22 + x1^41*x2^40*x3^19*x4^10*z^22 - x1^43*x2^37*x3^20*x4^10*z^22 - 2*x1^42*x2^38*x3^20*x4^10*z^22 - x1^40*x2^40*x3^20*x4^10*z^22 - x1^43*x2^36*x3^21*x4^10*z^22 + x1^42*x2^37*x3^21*x4^10*z^22 + x1^41*x2^38*x3^21*x4^10*z^22 - x1^47*x2^31*x3^22*x4^10*z^22 - x1^41*x2^37*x3^22*x4^10*z^22 + x1^47*x2^30*x3^23*x4^10*z^22 + x1^46*x2^31*x3^23*x4^10*z^22 - x1^45*x2^32*x3^23*x4^10*z^22 - 3*x1^46*x2^30*x3^24*x4^10*z^22 - x1^43*x2^33*x3^24*x4^10*z^22 + 3*x1^45*x2^30*x3^25*x4^10*z^22 - x1^45*x2^29*x3^26*x4^10*z^22 - x1^44*x2^30*x3^26*x4^10*z^22 - x1^43*x2^31*x3^26*x4^10*z^22 + x1^42*x2^32*x3^26*x4^10*z^22 - x1^41*x2^33*x3^26*x4^10*z^22 - 3*x1^40*x2^34*x3^26*x4^10*z^22 - x1^38*x2^36*x3^26*x4^10*z^22 + x1^44*x2^29*x3^27*x4^10*z^22 + x1^43*x2^30*x3^27*x4^10*z^22 + 2*x1^42*x2^31*x3^27*x4^10*z^22 - x1^39*x2^34*x3^27*x4^10*z^22 - x1^38*x2^35*x3^27*x4^10*z^22 + 2*x1^37*x2^36*x3^27*x4^10*z^22 - x1^42*x2^30*x3^28*x4^10*z^22 + x1^41*x2^31*x3^28*x4^10*z^22 + x1^40*x2^32*x3^28*x4^10*z^22 + x1^39*x2^33*x3^28*x4^10*z^22 - 2*x1^37*x2^35*x3^28*x4^10*z^22 + x1^39*x2^32*x3^29*x4^10*z^22 - 2*x1^38*x2^33*x3^29*x4^10*z^22 - x1^37*x2^34*x3^29*x4^10*z^22 + x1^36*x2^35*x3^29*x4^10*z^22 - x1^39*x2^31*x3^30*x4^10*z^22 + x1^37*x2^33*x3^30*x4^10*z^22 - x1^36*x2^34*x3^30*x4^10*z^22 + x1^50*x2^35*x3^14*x4^11*z^22 + x1^47*x2^38*x3^14*x4^11*z^22 - x1^48*x2^36*x3^15*x4^11*z^22 + 2*x1^47*x2^37*x3^15*x4^11*z^22 + x1^46*x2^38*x3^15*x4^11*z^22 - x1^45*x2^39*x3^15*x4^11*z^22 + 2*x1^49*x2^34*x3^16*x4^11*z^22 + 2*x1^48*x2^35*x3^16*x4^11*z^22 - 3*x1^46*x2^37*x3^16*x4^11*z^22 + x1^44*x2^39*x3^16*x4^11*z^22 - 2*x1^49*x2^33*x3^17*x4^11*z^22 + x1^48*x2^34*x3^17*x4^11*z^22 + 2*x1^47*x2^35*x3^17*x4^11*z^22 - x1^46*x2^36*x3^17*x4^11*z^22 + x1^45*x2^37*x3^17*x4^11*z^22 + x1^44*x2^38*x3^17*x4^11*z^22 - x1^42*x2^40*x3^17*x4^11*z^22 - x1^49*x2^32*x3^18*x4^11*z^22 + 3*x1^48*x2^33*x3^18*x4^11*z^22 - x1^47*x2^34*x3^18*x4^11*z^22 + x1^46*x2^35*x3^18*x4^11*z^22 - 3*x1^45*x2^36*x3^18*x4^11*z^22 - 3*x1^43*x2^38*x3^18*x4^11*z^22 + x1^42*x2^39*x3^18*x4^11*z^22 + x1^41*x2^40*x3^18*x4^11*z^22 - 3*x1^48*x2^32*x3^19*x4^11*z^22 + 2*x1^46*x2^34*x3^19*x4^11*z^22 + 2*x1^45*x2^35*x3^19*x4^11*z^22 + 2*x1^44*x2^36*x3^19*x4^11*z^22 - x1^43*x2^37*x3^19*x4^11*z^22 + 4*x1^42*x2^38*x3^19*x4^11*z^22 - x1^41*x2^39*x3^19*x4^11*z^22 + 4*x1^47*x2^32*x3^20*x4^11*z^22 - 4*x1^46*x2^33*x3^20*x4^11*z^22 + x1^45*x2^34*x3^20*x4^11*z^22 - 2*x1^44*x2^35*x3^20*x4^11*z^22 - x1^43*x2^36*x3^20*x4^11*z^22 - 4*x1^42*x2^37*x3^20*x4^11*z^22 - x1^41*x2^38*x3^20*x4^11*z^22 + x1^40*x2^39*x3^20*x4^11*z^22 - 4*x1^47*x2^31*x3^21*x4^11*z^22 - 2*x1^46*x2^32*x3^21*x4^11*z^22 + 4*x1^43*x2^35*x3^21*x4^11*z^22 + x1^42*x2^36*x3^21*x4^11*z^22 + 4*x1^41*x2^37*x3^21*x4^11*z^22 - 2*x1^40*x2^38*x3^21*x4^11*z^22 + x1^47*x2^30*x3^22*x4^11*z^22 + 4*x1^46*x2^31*x3^22*x4^11*z^22 - 2*x1^45*x2^32*x3^22*x4^11*z^22 + x1^44*x2^33*x3^22*x4^11*z^22 - 4*x1^43*x2^34*x3^22*x4^11*z^22 - 3*x1^41*x2^36*x3^22*x4^11*z^22 + 2*x1^39*x2^38*x3^22*x4^11*z^22 + x1^45*x2^31*x3^23*x4^11*z^22 + 3*x1^44*x2^32*x3^23*x4^11*z^22 + 2*x1^42*x2^34*x3^23*x4^11*z^22 + 4*x1^40*x2^36*x3^23*x4^11*z^22 - 2*x1^39*x2^37*x3^23*x4^11*z^22 - x1^38*x2^38*x3^23*x4^11*z^22 - x1^44*x2^31*x3^24*x4^11*z^22 + 2*x1^43*x2^32*x3^24*x4^11*z^22 - 2*x1^42*x2^33*x3^24*x4^11*z^22 - 3*x1^40*x2^35*x3^24*x4^11*z^22 + 2*x1^38*x2^37*x3^24*x4^11*z^22 - x1^44*x2^30*x3^25*x4^11*z^22 + x1^43*x2^31*x3^25*x4^11*z^22 + x1^41*x2^33*x3^25*x4^11*z^22 - x1^40*x2^34*x3^25*x4^11*z^22 + 4*x1^39*x2^35*x3^25*x4^11*z^22 - 2*x1^38*x2^36*x3^25*x4^11*z^22 + x1^43*x2^30*x3^26*x4^11*z^22 - x1^42*x2^31*x3^26*x4^11*z^22 + 2*x1^40*x2^33*x3^26*x4^11*z^22 - x1^39*x2^34*x3^26*x4^11*z^22 - x1^38*x2^35*x3^26*x4^11*z^22 + 2*x1^37*x2^36*x3^26*x4^11*z^22 - x1^42*x2^30*x3^27*x4^11*z^22 + 2*x1^40*x2^32*x3^27*x4^11*z^22 - 2*x1^37*x2^35*x3^27*x4^11*z^22 - x1^40*x2^31*x3^28*x4^11*z^22 - x1^38*x2^33*x3^28*x4^11*z^22 + x1^37*x2^34*x3^28*x4^11*z^22 - x1^38*x2^32*x3^29*x4^11*z^22 - x1^36*x2^34*x3^29*x4^11*z^22 - x1^47*x2^37*x3^14*x4^12*z^22 - 2*x1^49*x2^34*x3^15*x4^12*z^22 + x1^47*x2^36*x3^15*x4^12*z^22 - x1^46*x2^37*x3^15*x4^12*z^22 - x1^45*x2^37*x3^16*x4^12*z^22 + x1^44*x2^38*x3^16*x4^12*z^22 + x1^43*x2^39*x3^16*x4^12*z^22 - 2*x1^48*x2^33*x3^17*x4^12*z^22 + x1^47*x2^34*x3^17*x4^12*z^22 + 2*x1^46*x2^35*x3^17*x4^12*z^22 + x1^45*x2^36*x3^17*x4^12*z^22 + x1^44*x2^37*x3^17*x4^12*z^22 - x1^43*x2^38*x3^17*x4^12*z^22 + x1^48*x2^32*x3^18*x4^12*z^22 - x1^47*x2^33*x3^18*x4^12*z^22 - 3*x1^46*x2^34*x3^18*x4^12*z^22 + x1^44*x2^36*x3^18*x4^12*z^22 + 3*x1^43*x2^37*x3^18*x4^12*z^22 + 2*x1^41*x2^39*x3^18*x4^12*z^22 - 3*x1^47*x2^32*x3^19*x4^12*z^22 + 4*x1^46*x2^33*x3^19*x4^12*z^22 + 2*x1^44*x2^35*x3^19*x4^12*z^22 - x1^43*x2^36*x3^19*x4^12*z^22 + x1^42*x2^37*x3^19*x4^12*z^22 - 2*x1^41*x2^38*x3^19*x4^12*z^22 - 2*x1^40*x2^39*x3^19*x4^12*z^22 + x1^47*x2^31*x3^20*x4^12*z^22 - x1^46*x2^32*x3^20*x4^12*z^22 - 4*x1^45*x2^33*x3^20*x4^12*z^22 - 3*x1^43*x2^35*x3^20*x4^12*z^22 + 3*x1^42*x2^36*x3^20*x4^12*z^22 - 2*x1^41*x2^37*x3^20*x4^12*z^22 + 6*x1^40*x2^38*x3^20*x4^12*z^22 - 2*x1^46*x2^31*x3^21*x4^12*z^22 + 6*x1^45*x2^32*x3^21*x4^12*z^22 + 3*x1^44*x2^33*x3^21*x4^12*z^22 + 3*x1^43*x2^34*x3^21*x4^12*z^22 - x1^40*x2^37*x3^21*x4^12*z^22 - 6*x1^39*x2^38*x3^21*x4^12*z^22 + x1^46*x2^30*x3^22*x4^12*z^22 - 5*x1^44*x2^32*x3^22*x4^12*z^22 - 5*x1^42*x2^34*x3^22*x4^12*z^22 + 3*x1^41*x2^35*x3^22*x4^12*z^22 - 2*x1^40*x2^36*x3^22*x4^12*z^22 + 6*x1^39*x2^37*x3^22*x4^12*z^22 + 2*x1^38*x2^38*x3^22*x4^12*z^22 - x1^45*x2^30*x3^23*x4^12*z^22 + 3*x1^44*x2^31*x3^23*x4^12*z^22 + 2*x1^43*x2^32*x3^23*x4^12*z^22 + 4*x1^42*x2^33*x3^23*x4^12*z^22 - 2*x1^39*x2^36*x3^23*x4^12*z^22 - 6*x1^38*x2^37*x3^23*x4^12*z^22 - x1^45*x2^29*x3^24*x4^12*z^22 - 4*x1^43*x2^31*x3^24*x4^12*z^22 - x1^42*x2^32*x3^24*x4^12*z^22 - 3*x1^41*x2^33*x3^24*x4^12*z^22 + 4*x1^40*x2^34*x3^24*x4^12*z^22 - 2*x1^39*x2^35*x3^24*x4^12*z^22 + 6*x1^38*x2^36*x3^24*x4^12*z^22 + 2*x1^37*x2^37*x3^24*x4^12*z^22 + x1^44*x2^29*x3^25*x4^12*z^22 + x1^43*x2^30*x3^25*x4^12*z^22 + x1^42*x2^31*x3^25*x4^12*z^22 + 2*x1^41*x2^32*x3^25*x4^12*z^22 + x1^39*x2^34*x3^25*x4^12*z^22 - x1^38*x2^35*x3^25*x4^12*z^22 - 6*x1^37*x2^36*x3^25*x4^12*z^22 - x1^42*x2^30*x3^26*x4^12*z^22 - x1^41*x2^31*x3^26*x4^12*z^22 - 3*x1^40*x2^32*x3^26*x4^12*z^22 + 2*x1^39*x2^33*x3^26*x4^12*z^22 - 3*x1^38*x2^34*x3^26*x4^12*z^22 + 6*x1^37*x2^35*x3^26*x4^12*z^22 + 2*x1^36*x2^36*x3^26*x4^12*z^22 + x1^41*x2^30*x3^27*x4^12*z^22 + 2*x1^40*x2^31*x3^27*x4^12*z^22 - x1^37*x2^34*x3^27*x4^12*z^22 - 6*x1^36*x2^35*x3^27*x4^12*z^22 - x1^39*x2^31*x3^28*x4^12*z^22 + x1^38*x2^32*x3^28*x4^12*z^22 - x1^37*x2^33*x3^28*x4^12*z^22 + 4*x1^36*x2^34*x3^28*x4^12*z^22 + x1^35*x2^35*x3^28*x4^12*z^22 - 2*x1^35*x2^34*x3^29*x4^12*z^22 - x1^35*x2^33*x3^30*x4^12*z^22 + 2*x1^46*x2^36*x3^15*x4^13*z^22 + x1^45*x2^37*x3^15*x4^13*z^22 - x1^47*x2^33*x3^17*x4^13*z^22 - x1^46*x2^34*x3^17*x4^13*z^22 + x1^44*x2^36*x3^17*x4^13*z^22 - x1^43*x2^37*x3^17*x4^13*z^22 - x1^41*x2^39*x3^17*x4^13*z^22 + 2*x1^47*x2^32*x3^18*x4^13*z^22 - 2*x1^46*x2^33*x3^18*x4^13*z^22 - x1^45*x2^34*x3^18*x4^13*z^22 + x1^44*x2^35*x3^18*x4^13*z^22 + x1^41*x2^38*x3^18*x4^13*z^22 + 3*x1^40*x2^39*x3^18*x4^13*z^22 - x1^46*x2^32*x3^19*x4^13*z^22 + 2*x1^45*x2^33*x3^19*x4^13*z^22 - 3*x1^42*x2^36*x3^19*x4^13*z^22 - 5*x1^40*x2^38*x3^19*x4^13*z^22 - x1^39*x2^39*x3^19*x4^13*z^22 + x1^47*x2^30*x3^20*x4^13*z^22 + x1^46*x2^31*x3^20*x4^13*z^22 - 3*x1^45*x2^32*x3^20*x4^13*z^22 - x1^44*x2^33*x3^20*x4^13*z^22 + 2*x1^42*x2^35*x3^20*x4^13*z^22 + 2*x1^41*x2^36*x3^20*x4^13*z^22 + 2*x1^40*x2^37*x3^20*x4^13*z^22 + 6*x1^39*x2^38*x3^20*x4^13*z^22 + x1^45*x2^31*x3^21*x4^13*z^22 + 3*x1^44*x2^32*x3^21*x4^13*z^22 - x1^43*x2^33*x3^21*x4^13*z^22 + x1^42*x2^34*x3^21*x4^13*z^22 - 4*x1^41*x2^35*x3^21*x4^13*z^22 - 6*x1^39*x2^37*x3^21*x4^13*z^22 - 2*x1^38*x2^38*x3^21*x4^13*z^22 + x1^46*x2^29*x3^22*x4^13*z^22 - 3*x1^44*x2^31*x3^22*x4^13*z^22 - x1^43*x2^32*x3^22*x4^13*z^22 - 2*x1^42*x2^33*x3^22*x4^13*z^22 + x1^41*x2^34*x3^22*x4^13*z^22 + 2*x1^40*x2^35*x3^22*x4^13*z^22 + 2*x1^39*x2^36*x3^22*x4^13*z^22 + 6*x1^38*x2^37*x3^22*x4^13*z^22 + 2*x1^44*x2^30*x3^23*x4^13*z^22 + 6*x1^43*x2^31*x3^23*x4^13*z^22 + x1^42*x2^32*x3^23*x4^13*z^22 + 4*x1^41*x2^33*x3^23*x4^13*z^22 - 4*x1^40*x2^34*x3^23*x4^13*z^22 - 6*x1^38*x2^36*x3^23*x4^13*z^22 - 2*x1^37*x2^37*x3^23*x4^13*z^22 - 2*x1^43*x2^30*x3^24*x4^13*z^22 - 2*x1^42*x2^31*x3^24*x4^13*z^22 - 2*x1^41*x2^32*x3^24*x4^13*z^22 + 2*x1^39*x2^34*x3^24*x4^13*z^22 + 2*x1^38*x2^35*x3^24*x4^13*z^22 + 6*x1^37*x2^36*x3^24*x4^13*z^22 - x1^43*x2^29*x3^25*x4^13*z^22 + 2*x1^42*x2^30*x3^25*x4^13*z^22 + 2*x1^40*x2^32*x3^25*x4^13*z^22 - 5*x1^39*x2^33*x3^25*x4^13*z^22 - 6*x1^37*x2^35*x3^25*x4^13*z^22 - 2*x1^36*x2^36*x3^25*x4^13*z^22 - 2*x1^41*x2^30*x3^26*x4^13*z^22 - x1^40*x2^31*x3^26*x4^13*z^22 - x1^39*x2^32*x3^26*x4^13*z^22 + x1^38*x2^33*x3^26*x4^13*z^22 + 6*x1^36*x2^35*x3^26*x4^13*z^22 + x1^39*x2^31*x3^27*x4^13*z^22 - x1^38*x2^32*x3^27*x4^13*z^22 + x1^37*x2^33*x3^27*x4^13*z^22 - 3*x1^36*x2^34*x3^27*x4^13*z^22 - 2*x1^35*x2^35*x3^27*x4^13*z^22 - x1^38*x2^31*x3^28*x4^13*z^22 + x1^36*x2^33*x3^28*x4^13*z^22 + 3*x1^35*x2^34*x3^28*x4^13*z^22 + 2*x1^36*x2^32*x3^29*x4^13*z^22 - x1^35*x2^33*x3^29*x4^13*z^22 + x1^34*x2^33*x3^30*x4^13*z^22 + x1^45*x2^36*x3^15*x4^14*z^22 - x1^45*x2^35*x3^16*x4^14*z^22 + 3*x1^44*x2^35*x3^17*x4^14*z^22 + x1^43*x2^36*x3^17*x4^14*z^22 + x1^42*x2^37*x3^17*x4^14*z^22 + x1^47*x2^31*x3^18*x4^14*z^22 + x1^46*x2^32*x3^18*x4^14*z^22 - x1^45*x2^33*x3^18*x4^14*z^22 - x1^44*x2^34*x3^18*x4^14*z^22 + x1^43*x2^35*x3^18*x4^14*z^22 - x1^41*x2^37*x3^18*x4^14*z^22 - x1^46*x2^31*x3^19*x4^14*z^22 + x1^45*x2^32*x3^19*x4^14*z^22 - x1^44*x2^33*x3^19*x4^14*z^22 + x1^43*x2^34*x3^19*x4^14*z^22 + x1^42*x2^35*x3^19*x4^14*z^22 + 2*x1^41*x2^36*x3^19*x4^14*z^22 - x1^40*x2^37*x3^19*x4^14*z^22 - 2*x1^39*x2^38*x3^19*x4^14*z^22 + x1^46*x2^30*x3^20*x4^14*z^22 + 2*x1^45*x2^31*x3^20*x4^14*z^22 - 2*x1^43*x2^33*x3^20*x4^14*z^22 + x1^41*x2^35*x3^20*x4^14*z^22 - 2*x1^40*x2^36*x3^20*x4^14*z^22 + x1^39*x2^37*x3^20*x4^14*z^22 - x1^46*x2^29*x3^21*x4^14*z^22 - 2*x1^45*x2^30*x3^21*x4^14*z^22 + 2*x1^44*x2^31*x3^21*x4^14*z^22 - 2*x1^43*x2^32*x3^21*x4^14*z^22 - x1^41*x2^34*x3^21*x4^14*z^22 + 2*x1^40*x2^35*x3^21*x4^14*z^22 + x1^39*x2^36*x3^21*x4^14*z^22 - 2*x1^38*x2^37*x3^21*x4^14*z^22 + 2*x1^45*x2^29*x3^22*x4^14*z^22 + 2*x1^44*x2^30*x3^22*x4^14*z^22 - 2*x1^41*x2^33*x3^22*x4^14*z^22 - 2*x1^39*x2^35*x3^22*x4^14*z^22 + 2*x1^38*x2^36*x3^22*x4^14*z^22 + x1^37*x2^37*x3^22*x4^14*z^22 - 2*x1^44*x2^29*x3^23*x4^14*z^22 - 2*x1^43*x2^30*x3^23*x4^14*z^22 - x1^42*x2^31*x3^23*x4^14*z^22 + 2*x1^41*x2^32*x3^23*x4^14*z^22 - x1^40*x2^33*x3^23*x4^14*z^22 + x1^39*x2^34*x3^23*x4^14*z^22 - 2*x1^37*x2^36*x3^23*x4^14*z^22 + x1^43*x2^29*x3^24*x4^14*z^22 + x1^42*x2^30*x3^24*x4^14*z^22 - x1^40*x2^32*x3^24*x4^14*z^22 - 2*x1^38*x2^34*x3^24*x4^14*z^22 + 2*x1^37*x2^35*x3^24*x4^14*z^22 + x1^36*x2^36*x3^24*x4^14*z^22 - x1^42*x2^29*x3^25*x4^14*z^22 - 2*x1^41*x2^30*x3^25*x4^14*z^22 + x1^40*x2^31*x3^25*x4^14*z^22 + 2*x1^38*x2^33*x3^25*x4^14*z^22 - x1^37*x2^34*x3^25*x4^14*z^22 - 2*x1^36*x2^35*x3^25*x4^14*z^22 + x1^41*x2^29*x3^26*x4^14*z^22 + x1^40*x2^30*x3^26*x4^14*z^22 + x1^39*x2^31*x3^26*x4^14*z^22 + x1^38*x2^32*x3^26*x4^14*z^22 - 3*x1^37*x2^33*x3^26*x4^14*z^22 + 2*x1^36*x2^34*x3^26*x4^14*z^22 - 2*x1^38*x2^31*x3^27*x4^14*z^22 + x1^37*x2^32*x3^27*x4^14*z^22 + 2*x1^36*x2^33*x3^27*x4^14*z^22 - 2*x1^35*x2^34*x3^27*x4^14*z^22 + x1^37*x2^31*x3^28*x4^14*z^22 - x1^36*x2^32*x3^28*x4^14*z^22 + x1^34*x2^34*x3^28*x4^14*z^22 - 2*x1^44*x2^35*x3^16*x4^15*z^22 + x1^44*x2^34*x3^17*x4^15*z^22 + x1^43*x2^35*x3^17*x4^15*z^22 + x1^42*x2^36*x3^17*x4^15*z^22 + x1^41*x2^37*x3^17*x4^15*z^22 - 4*x1^43*x2^34*x3^18*x4^15*z^22 - 2*x1^41*x2^36*x3^18*x4^15*z^22 - x1^40*x2^37*x3^18*x4^15*z^22 - x1^46*x2^30*x3^19*x4^15*z^22 - x1^45*x2^31*x3^19*x4^15*z^22 + x1^44*x2^32*x3^19*x4^15*z^22 + 2*x1^43*x2^33*x3^19*x4^15*z^22 + x1^42*x2^34*x3^19*x4^15*z^22 + x1^41*x2^35*x3^19*x4^15*z^22 + 5*x1^40*x2^36*x3^19*x4^15*z^22 + 2*x1^45*x2^30*x3^20*x4^15*z^22 + x1^43*x2^32*x3^20*x4^15*z^22 - 5*x1^42*x2^33*x3^20*x4^15*z^22 - 4*x1^40*x2^35*x3^20*x4^15*z^22 - 2*x1^39*x2^36*x3^20*x4^15*z^22 - x1^45*x2^29*x3^21*x4^15*z^22 - 2*x1^44*x2^30*x3^21*x4^15*z^22 + 2*x1^42*x2^32*x3^21*x4^15*z^22 + 2*x1^41*x2^33*x3^21*x4^15*z^22 + 2*x1^40*x2^34*x3^21*x4^15*z^22 + 6*x1^39*x2^35*x3^21*x4^15*z^22 + 3*x1^44*x2^29*x3^22*x4^15*z^22 + x1^43*x2^30*x3^22*x4^15*z^22 + x1^42*x2^31*x3^22*x4^15*z^22 - 4*x1^41*x2^32*x3^22*x4^15*z^22 - 6*x1^39*x2^34*x3^22*x4^15*z^22 - 2*x1^38*x2^35*x3^22*x4^15*z^22 - x1^44*x2^28*x3^23*x4^15*z^22 - 2*x1^43*x2^29*x3^23*x4^15*z^22 + x1^41*x2^31*x3^23*x4^15*z^22 + x1^40*x2^32*x3^23*x4^15*z^22 + 2*x1^39*x2^33*x3^23*x4^15*z^22 + 6*x1^38*x2^34*x3^23*x4^15*z^22 + x1^43*x2^28*x3^24*x4^15*z^22 + 2*x1^41*x2^30*x3^24*x4^15*z^22 - 4*x1^40*x2^31*x3^24*x4^15*z^22 + x1^39*x2^32*x3^24*x4^15*z^22 - 6*x1^38*x2^33*x3^24*x4^15*z^22 - 2*x1^37*x2^34*x3^24*x4^15*z^22 + x1^40*x2^30*x3^25*x4^15*z^22 + x1^39*x2^31*x3^25*x4^15*z^22 + x1^38*x2^32*x3^25*x4^15*z^22 + 6*x1^37*x2^33*x3^25*x4^15*z^22 + x1^40*x2^29*x3^26*x4^15*z^22 - 2*x1^39*x2^30*x3^26*x4^15*z^22 + x1^38*x2^31*x3^26*x4^15*z^22 - 4*x1^37*x2^32*x3^26*x4^15*z^22 - 2*x1^36*x2^33*x3^26*x4^15*z^22 + 2*x1^38*x2^30*x3^27*x4^15*z^22 + x1^37*x2^31*x3^27*x4^15*z^22 + 4*x1^36*x2^32*x3^27*x4^15*z^22 - x1^35*x2^32*x3^28*x4^15*z^22 - x1^42*x2^34*x3^18*x4^16*z^22 - x1^41*x2^35*x3^18*x4^16*z^22 - x1^40*x2^36*x3^18*x4^16*z^22 + 3*x1^42*x2^33*x3^19*x4^16*z^22 - x1^41*x2^34*x3^19*x4^16*z^22 + x1^40*x2^35*x3^19*x4^16*z^22 + 2*x1^39*x2^36*x3^19*x4^16*z^22 - 2*x1^42*x2^32*x3^20*x4^16*z^22 - 2*x1^41*x2^33*x3^20*x4^16*z^22 - 4*x1^39*x2^35*x3^20*x4^16*z^22 - x1^44*x2^29*x3^21*x4^16*z^22 - x1^43*x2^30*x3^21*x4^16*z^22 + 5*x1^41*x2^32*x3^21*x4^16*z^22 - x1^40*x2^33*x3^21*x4^16*z^22 + 3*x1^39*x2^34*x3^21*x4^16*z^22 + 2*x1^38*x2^35*x3^21*x4^16*z^22 + x1^43*x2^29*x3^22*x4^16*z^22 - x1^41*x2^31*x3^22*x4^16*z^22 - 2*x1^40*x2^32*x3^22*x4^16*z^22 - 2*x1^39*x2^33*x3^22*x4^16*z^22 - 5*x1^38*x2^34*x3^22*x4^16*z^22 - x1^43*x2^28*x3^23*x4^16*z^22 - x1^42*x2^29*x3^23*x4^16*z^22 + 4*x1^40*x2^31*x3^23*x4^16*z^22 + 6*x1^38*x2^33*x3^23*x4^16*z^22 + 2*x1^37*x2^34*x3^23*x4^16*z^22 + x1^42*x2^28*x3^24*x4^16*z^22 - x1^40*x2^30*x3^24*x4^16*z^22 - x1^39*x2^31*x3^24*x4^16*z^22 - x1^38*x2^32*x3^24*x4^16*z^22 - 6*x1^37*x2^33*x3^24*x4^16*z^22 - x1^40*x2^29*x3^25*x4^16*z^22 + x1^39*x2^30*x3^25*x4^16*z^22 - x1^38*x2^31*x3^25*x4^16*z^22 + 4*x1^37*x2^32*x3^25*x4^16*z^22 + 2*x1^36*x2^33*x3^25*x4^16*z^22 - x1^37*x2^31*x3^26*x4^16*z^22 - 4*x1^36*x2^32*x3^26*x4^16*z^22 - x1^37*x2^30*x3^27*x4^16*z^22 + x1^36*x2^31*x3^27*x4^16*z^22 + x1^35*x2^32*x3^27*x4^16*z^22 - x1^35*x2^31*x3^28*x4^16*z^22 + x1^41*x2^31*x3^21*x4^17*z^22 + x1^40*x2^32*x3^21*x4^17*z^22 - x1^39*x2^33*x3^21*x4^17*z^22 + x1^38*x2^34*x3^21*x4^17*z^22 - x1^40*x2^31*x3^22*x4^17*z^22 + x1^39*x2^32*x3^22*x4^17*z^22 - x1^38*x2^33*x3^22*x4^17*z^22 - x1^37*x2^34*x3^22*x4^17*z^22 - x1^40*x2^30*x3^23*x4^17*z^22 + x1^38*x2^32*x3^23*x4^17*z^22 + x1^37*x2^33*x3^23*x4^17*z^22 + 2*x1^38*x2^31*x3^24*x4^17*z^22 - x1^37*x2^32*x3^24*x4^17*z^22 - x1^36*x2^33*x3^24*x4^17*z^22 - x1^37*x2^31*x3^25*x4^17*z^22 + x1^36*x2^32*x3^25*x4^17*z^22 - 2*x1^49*x2^36*x3^20*z^21 + x1^50*x2^37*x3^17*x4*z^21 - x1^49*x2^38*x3^17*x4*z^21 - 2*x1^50*x2^36*x3^18*x4*z^21 - x1^49*x2^37*x3^18*x4*z^21 + 2*x1^48*x2^38*x3^18*x4*z^21 + 4*x1^49*x2^36*x3^19*x4*z^21 - x1^48*x2^37*x3^19*x4*z^21 - x1^46*x2^39*x3^19*x4*z^21 - 2*x1^49*x2^35*x3^20*x4*z^21 - 3*x1^48*x2^36*x3^20*x4*z^21 + 2*x1^47*x2^37*x3^20*x4*z^21 - x1^46*x2^38*x3^20*x4*z^21 + 4*x1^48*x2^35*x3^21*x4*z^21 + x1^46*x2^37*x3^21*x4*z^21 - x1^48*x2^34*x3^22*x4*z^21 - 2*x1^47*x2^35*x3^22*x4*z^21 - 2*x1^45*x2^37*x3^22*x4*z^21 + x1^47*x2^34*x3^23*x4*z^21 - x1^46*x2^35*x3^23*x4*z^21 + x1^45*x2^36*x3^23*x4*z^21 + x1^44*x2^37*x3^23*x4*z^21 - 2*x1^50*x2^37*x3^16*x4^2*z^21 + x1^50*x2^36*x3^17*x4^2*z^21 + 2*x1^49*x2^37*x3^17*x4^2*z^21 - x1^48*x2^38*x3^17*x4^2*z^21 - 5*x1^49*x2^36*x3^18*x4^2*z^21 - x1^48*x2^37*x3^18*x4^2*z^21 - x1^47*x2^38*x3^18*x4^2*z^21 + 2*x1^49*x2^35*x3^19*x4^2*z^21 + 5*x1^48*x2^36*x3^19*x4^2*z^21 + 2*x1^46*x2^38*x3^19*x4^2*z^21 - 6*x1^48*x2^35*x3^20*x4^2*z^21 - 2*x1^47*x2^36*x3^20*x4^2*z^21 - 3*x1^46*x2^37*x3^20*x4^2*z^21 - x1^45*x2^38*x3^20*x4^2*z^21 + x1^43*x2^40*x3^20*x4^2*z^21 + 2*x1^48*x2^34*x3^21*x4^2*z^21 + 5*x1^47*x2^35*x3^21*x4^2*z^21 - x1^46*x2^36*x3^21*x4^2*z^21 + 4*x1^45*x2^37*x3^21*x4^2*z^21 + x1^44*x2^38*x3^21*x4^2*z^21 - x1^43*x2^39*x3^21*x4^2*z^21 - x1^42*x2^40*x3^21*x4^2*z^21 - 4*x1^47*x2^34*x3^22*x4^2*z^21 - x1^46*x2^35*x3^22*x4^2*z^21 - 3*x1^45*x2^36*x3^22*x4^2*z^21 - x1^44*x2^37*x3^22*x4^2*z^21 + 2*x1^47*x2^33*x3^23*x4^2*z^21 + 2*x1^46*x2^34*x3^23*x4^2*z^21 + 4*x1^44*x2^36*x3^23*x4^2*z^21 - 2*x1^46*x2^33*x3^24*x4^2*z^21 - 2*x1^44*x2^35*x3^24*x4^2*z^21 - x1^43*x2^36*x3^24*x4^2*z^21 + x1^46*x2^32*x3^25*x4^2*z^21 + x1^45*x2^33*x3^25*x4^2*z^21 - x1^44*x2^34*x3^25*x4^2*z^21 + x1^43*x2^35*x3^25*x4^2*z^21 - x1^51*x2^34*x3^17*x4^3*z^21 + 2*x1^49*x2^36*x3^17*x4^3*z^21 + x1^47*x2^38*x3^17*x4^3*z^21 + 2*x1^50*x2^34*x3^18*x4^3*z^21 - 2*x1^49*x2^35*x3^18*x4^3*z^21 - 2*x1^48*x2^36*x3^18*x4^3*z^21 + x1^47*x2^37*x3^18*x4^3*z^21 - x1^46*x2^38*x3^18*x4^3*z^21 - 2*x1^49*x2^34*x3^19*x4^3*z^21 + 3*x1^48*x2^35*x3^19*x4^3*z^21 - x1^47*x2^36*x3^19*x4^3*z^21 + 2*x1^49*x2^33*x3^20*x4^3*z^21 + x1^48*x2^34*x3^20*x4^3*z^21 - 2*x1^47*x2^35*x3^20*x4^3*z^21 - 2*x1^45*x2^37*x3^20*x4^3*z^21 + 2*x1^44*x2^38*x3^20*x4^3*z^21 - x1^49*x2^32*x3^21*x4^3*z^21 + 3*x1^47*x2^34*x3^21*x4^3*z^21 + x1^44*x2^37*x3^21*x4^3*z^21 + x1^48*x2^32*x3^22*x4^3*z^21 + x1^45*x2^35*x3^22*x4^3*z^21 - 3*x1^44*x2^36*x3^22*x4^3*z^21 + x1^42*x2^38*x3^22*x4^3*z^21 + x1^41*x2^39*x3^22*x4^3*z^21 - x1^48*x2^31*x3^23*x4^3*z^21 - x1^47*x2^32*x3^23*x4^3*z^21 + 2*x1^46*x2^33*x3^23*x4^3*z^21 + x1^45*x2^34*x3^23*x4^3*z^21 + 2*x1^44*x2^35*x3^23*x4^3*z^21 - x1^43*x2^36*x3^23*x4^3*z^21 - x1^40*x2^39*x3^23*x4^3*z^21 - 2*x1^46*x2^32*x3^24*x4^3*z^21 - x1^45*x2^33*x3^24*x4^3*z^21 + x1^44*x2^34*x3^24*x4^3*z^21 - 3*x1^43*x2^35*x3^24*x4^3*z^21 + x1^45*x2^32*x3^25*x4^3*z^21 + x1^44*x2^33*x3^25*x4^3*z^21 + x1^43*x2^34*x3^25*x4^3*z^21 - x1^44*x2^32*x3^26*x4^3*z^21 + x1^43*x2^33*x3^26*x4^3*z^21 - x1^42*x2^34*x3^26*x4^3*z^21 + x1^52*x2^35*x3^14*x4^4*z^21 - 2*x1^51*x2^35*x3^15*x4^4*z^21 + x1^50*x2^36*x3^15*x4^4*z^21 + 2*x1^51*x2^34*x3^16*x4^4*z^21 + 2*x1^50*x2^35*x3^16*x4^4*z^21 - x1^49*x2^36*x3^16*x4^4*z^21 + 2*x1^48*x2^37*x3^16*x4^4*z^21 - 5*x1^50*x2^34*x3^17*x4^4*z^21 + x1^46*x2^38*x3^17*x4^4*z^21 + 2*x1^50*x2^33*x3^18*x4^4*z^21 + 5*x1^49*x2^34*x3^18*x4^4*z^21 - x1^48*x2^35*x3^18*x4^4*z^21 + 2*x1^47*x2^36*x3^18*x4^4*z^21 - x1^46*x2^37*x3^18*x4^4*z^21 + x1^44*x2^39*x3^18*x4^4*z^21 - 6*x1^49*x2^33*x3^19*x4^4*z^21 - 2*x1^48*x2^34*x3^19*x4^4*z^21 - x1^47*x2^35*x3^19*x4^4*z^21 + x1^45*x2^37*x3^19*x4^4*z^21 + x1^44*x2^38*x3^19*x4^4*z^21 - x1^43*x2^39*x3^19*x4^4*z^21 + 2*x1^49*x2^32*x3^20*x4^4*z^21 + 6*x1^48*x2^33*x3^20*x4^4*z^21 + 3*x1^46*x2^35*x3^20*x4^4*z^21 - 2*x1^45*x2^36*x3^20*x4^4*z^21 - 2*x1^44*x2^37*x3^20*x4^4*z^21 - x1^43*x2^38*x3^20*x4^4*z^21 + x1^42*x2^39*x3^20*x4^4*z^21 - 6*x1^48*x2^32*x3^21*x4^4*z^21 - 2*x1^47*x2^33*x3^21*x4^4*z^21 - 2*x1^46*x2^34*x3^21*x4^4*z^21 - 2*x1^45*x2^35*x3^21*x4^4*z^21 - x1^44*x2^36*x3^21*x4^4*z^21 + x1^41*x2^39*x3^21*x4^4*z^21 + 2*x1^48*x2^31*x3^22*x4^4*z^21 + 4*x1^47*x2^32*x3^22*x4^4*z^21 - x1^46*x2^33*x3^22*x4^4*z^21 + 4*x1^45*x2^34*x3^22*x4^4*z^21 - 4*x1^42*x2^37*x3^22*x4^4*z^21 + x1^41*x2^38*x3^22*x4^4*z^21 + x1^40*x2^39*x3^22*x4^4*z^21 - 5*x1^47*x2^31*x3^23*x4^4*z^21 - x1^46*x2^32*x3^23*x4^4*z^21 - x1^45*x2^33*x3^23*x4^4*z^21 - 2*x1^44*x2^34*x3^23*x4^4*z^21 + x1^43*x2^35*x3^23*x4^4*z^21 - x1^42*x2^36*x3^23*x4^4*z^21 + x1^41*x2^37*x3^23*x4^4*z^21 + x1^47*x2^30*x3^24*x4^4*z^21 + 2*x1^46*x2^31*x3^24*x4^4*z^21 - x1^45*x2^32*x3^24*x4^4*z^21 + 3*x1^44*x2^33*x3^24*x4^4*z^21 + x1^43*x2^34*x3^24*x4^4*z^21 - x1^41*x2^36*x3^24*x4^4*z^21 + x1^40*x2^37*x3^24*x4^4*z^21 + x1^39*x2^38*x3^24*x4^4*z^21 - 3*x1^46*x2^30*x3^25*x4^4*z^21 - x1^44*x2^32*x3^25*x4^4*z^21 - x1^43*x2^33*x3^25*x4^4*z^21 + 2*x1^42*x2^34*x3^25*x4^4*z^21 - 2*x1^44*x2^31*x3^26*x4^4*z^21 + x1^43*x2^31*x3^27*x4^4*z^21 + x1^42*x2^32*x3^27*x4^4*z^21 + 2*x1^41*x2^33*x3^27*x4^4*z^21 + x1^51*x2^35*x3^14*x4^5*z^21 - x1^51*x2^34*x3^15*x4^5*z^21 - x1^50*x2^35*x3^15*x4^5*z^21 + x1^49*x2^36*x3^15*x4^5*z^21 + 4*x1^50*x2^34*x3^16*x4^5*z^21 - x1^47*x2^37*x3^16*x4^5*z^21 - 2*x1^50*x2^33*x3^17*x4^5*z^21 - 4*x1^49*x2^34*x3^17*x4^5*z^21 + x1^48*x2^35*x3^17*x4^5*z^21 - 4*x1^47*x2^36*x3^17*x4^5*z^21 + 6*x1^49*x2^33*x3^18*x4^5*z^21 + x1^47*x2^35*x3^18*x4^5*z^21 + x1^46*x2^36*x3^18*x4^5*z^21 - 2*x1^45*x2^37*x3^18*x4^5*z^21 - 3*x1^44*x2^38*x3^18*x4^5*z^21 - 2*x1^49*x2^32*x3^19*x4^5*z^21 - 6*x1^48*x2^33*x3^19*x4^5*z^21 + 2*x1^47*x2^34*x3^19*x4^5*z^21 - 3*x1^46*x2^35*x3^19*x4^5*z^21 + 3*x1^45*x2^36*x3^19*x4^5*z^21 + x1^44*x2^37*x3^19*x4^5*z^21 + 3*x1^43*x2^38*x3^19*x4^5*z^21 + 6*x1^48*x2^32*x3^20*x4^5*z^21 + x1^47*x2^33*x3^20*x4^5*z^21 - 2*x1^44*x2^36*x3^20*x4^5*z^21 - 4*x1^42*x2^38*x3^20*x4^5*z^21 - x1^41*x2^39*x3^20*x4^5*z^21 - 2*x1^48*x2^31*x3^21*x4^5*z^21 - 6*x1^47*x2^32*x3^21*x4^5*z^21 + 2*x1^46*x2^33*x3^21*x4^5*z^21 - 4*x1^45*x2^34*x3^21*x4^5*z^21 + 3*x1^44*x2^35*x3^21*x4^5*z^21 + x1^43*x2^36*x3^21*x4^5*z^21 + 3*x1^42*x2^37*x3^21*x4^5*z^21 + 6*x1^47*x2^31*x3^22*x4^5*z^21 + 2*x1^46*x2^32*x3^22*x4^5*z^21 + 2*x1^44*x2^34*x3^22*x4^5*z^21 - 3*x1^43*x2^35*x3^22*x4^5*z^21 - x1^42*x2^36*x3^22*x4^5*z^21 - 2*x1^41*x2^37*x3^22*x4^5*z^21 + x1^39*x2^39*x3^22*x4^5*z^21 - 2*x1^47*x2^30*x3^23*x4^5*z^21 - 6*x1^46*x2^31*x3^23*x4^5*z^21 + 2*x1^45*x2^32*x3^23*x4^5*z^21 - 4*x1^44*x2^33*x3^23*x4^5*z^21 + x1^42*x2^35*x3^23*x4^5*z^21 + 3*x1^41*x2^36*x3^23*x4^5*z^21 - x1^40*x2^37*x3^23*x4^5*z^21 - x1^39*x2^38*x3^23*x4^5*z^21 + 5*x1^46*x2^30*x3^24*x4^5*z^21 + 2*x1^43*x2^33*x3^24*x4^5*z^21 - 2*x1^42*x2^34*x3^24*x4^5*z^21 - x1^41*x2^35*x3^24*x4^5*z^21 - 2*x1^40*x2^36*x3^24*x4^5*z^21 + x1^39*x2^37*x3^24*x4^5*z^21 - 3*x1^45*x2^30*x3^25*x4^5*z^21 + x1^44*x2^31*x3^25*x4^5*z^21 - 3*x1^43*x2^32*x3^25*x4^5*z^21 + x1^41*x2^34*x3^25*x4^5*z^21 + x1^40*x2^35*x3^25*x4^5*z^21 - 2*x1^39*x2^36*x3^25*x4^5*z^21 + x1^45*x2^29*x3^26*x4^5*z^21 - x1^43*x2^31*x3^26*x4^5*z^21 + x1^40*x2^34*x3^26*x4^5*z^21 - x1^39*x2^35*x3^26*x4^5*z^21 + x1^38*x2^36*x3^26*x4^5*z^21 + x1^41*x2^32*x3^27*x4^5*z^21 + x1^40*x2^33*x3^27*x4^5*z^21 - x1^41*x2^31*x3^28*x4^5*z^21 - x1^40*x2^32*x3^28*x4^5*z^21 + x1^39*x2^32*x3^29*x4^5*z^21 - x1^50*x2^35*x3^14*x4^6*z^21 - 2*x1^50*x2^34*x3^15*x4^6*z^21 + x1^50*x2^33*x3^16*x4^6*z^21 + 2*x1^49*x2^34*x3^16*x4^6*z^21 - x1^48*x2^35*x3^16*x4^6*z^21 + 2*x1^47*x2^36*x3^16*x4^6*z^21 - 2*x1^49*x2^33*x3^17*x4^6*z^21 - x1^48*x2^34*x3^17*x4^6*z^21 + x1^45*x2^37*x3^17*x4^6*z^21 + x1^49*x2^32*x3^18*x4^6*z^21 + 2*x1^48*x2^33*x3^18*x4^6*z^21 - 3*x1^47*x2^34*x3^18*x4^6*z^21 - 3*x1^45*x2^36*x3^18*x4^6*z^21 + x1^43*x2^38*x3^18*x4^6*z^21 - 2*x1^48*x2^32*x3^19*x4^6*z^21 + 2*x1^47*x2^33*x3^19*x4^6*z^21 + 3*x1^46*x2^34*x3^19*x4^6*z^21 - 2*x1^45*x2^35*x3^19*x4^6*z^21 + x1^44*x2^36*x3^19*x4^6*z^21 - x1^43*x2^37*x3^19*x4^6*z^21 - x1^42*x2^38*x3^19*x4^6*z^21 + 2*x1^47*x2^32*x3^20*x4^6*z^21 - 4*x1^46*x2^33*x3^20*x4^6*z^21 + x1^45*x2^34*x3^20*x4^6*z^21 - x1^44*x2^35*x3^20*x4^6*z^21 - x1^42*x2^37*x3^20*x4^6*z^21 + x1^41*x2^38*x3^20*x4^6*z^21 - 2*x1^47*x2^31*x3^21*x4^6*z^21 + 3*x1^45*x2^33*x3^21*x4^6*z^21 + x1^44*x2^34*x3^21*x4^6*z^21 + 5*x1^43*x2^35*x3^21*x4^6*z^21 - 2*x1^42*x2^36*x3^21*x4^6*z^21 + 3*x1^41*x2^37*x3^21*x4^6*z^21 - x1^40*x2^38*x3^21*x4^6*z^21 + x1^47*x2^30*x3^22*x4^6*z^21 + 2*x1^46*x2^31*x3^22*x4^6*z^21 - 4*x1^45*x2^32*x3^22*x4^6*z^21 - x1^43*x2^34*x3^22*x4^6*z^21 - 2*x1^41*x2^36*x3^22*x4^6*z^21 + x1^40*x2^37*x3^22*x4^6*z^21 + x1^39*x2^38*x3^22*x4^6*z^21 - 2*x1^46*x2^30*x3^23*x4^6*z^21 + 3*x1^44*x2^32*x3^23*x4^6*z^21 - x1^43*x2^33*x3^23*x4^6*z^21 + 5*x1^42*x2^34*x3^23*x4^6*z^21 + 3*x1^41*x2^35*x3^23*x4^6*z^21 + 2*x1^40*x2^36*x3^23*x4^6*z^21 - 3*x1^39*x2^37*x3^23*x4^6*z^21 - x1^38*x2^38*x3^23*x4^6*z^21 + 4*x1^45*x2^30*x3^24*x4^6*z^21 - 4*x1^44*x2^31*x3^24*x4^6*z^21 - x1^43*x2^32*x3^24*x4^6*z^21 - x1^42*x2^33*x3^24*x4^6*z^21 - x1^41*x2^34*x3^24*x4^6*z^21 - x1^40*x2^35*x3^24*x4^6*z^21 + 2*x1^39*x2^36*x3^24*x4^6*z^21 + x1^38*x2^37*x3^24*x4^6*z^21 + 3*x1^44*x2^30*x3^25*x4^6*z^21 + 3*x1^43*x2^31*x3^25*x4^6*z^21 - x1^42*x2^32*x3^25*x4^6*z^21 + 2*x1^41*x2^33*x3^25*x4^6*z^21 + x1^40*x2^34*x3^25*x4^6*z^21 + 2*x1^39*x2^35*x3^25*x4^6*z^21 - 2*x1^38*x2^36*x3^25*x4^6*z^21 + x1^44*x2^29*x3^26*x4^6*z^21 - x1^43*x2^30*x3^26*x4^6*z^21 + 2*x1^42*x2^31*x3^26*x4^6*z^21 - 2*x1^40*x2^33*x3^26*x4^6*z^21 + x1^38*x2^35*x3^26*x4^6*z^21 - x1^37*x2^36*x3^26*x4^6*z^21 + x1^41*x2^31*x3^27*x4^6*z^21 + x1^40*x2^32*x3^27*x4^6*z^21 - x1^39*x2^33*x3^27*x4^6*z^21 + x1^38*x2^34*x3^27*x4^6*z^21 - x1^37*x2^35*x3^27*x4^6*z^21 - x1^40*x2^31*x3^28*x4^6*z^21 - 2*x1^39*x2^32*x3^28*x4^6*z^21 - x1^49*x2^36*x3^13*x4^7*z^21 + x1^48*x2^36*x3^14*x4^7*z^21 + x1^46*x2^38*x3^14*x4^7*z^21 + x1^50*x2^33*x3^15*x4^7*z^21 - 3*x1^48*x2^35*x3^15*x4^7*z^21 - 2*x1^47*x2^36*x3^15*x4^7*z^21 - x1^45*x2^38*x3^15*x4^7*z^21 - x1^49*x2^33*x3^16*x4^7*z^21 + x1^47*x2^35*x3^16*x4^7*z^21 - x1^46*x2^36*x3^16*x4^7*z^21 + x1^45*x2^37*x3^16*x4^7*z^21 + x1^48*x2^33*x3^17*x4^7*z^21 - x1^47*x2^34*x3^17*x4^7*z^21 - 2*x1^46*x2^35*x3^17*x4^7*z^21 - x1^45*x2^36*x3^17*x4^7*z^21 + x1^46*x2^34*x3^18*x4^7*z^21 + x1^45*x2^35*x3^18*x4^7*z^21 - 2*x1^43*x2^37*x3^18*x4^7*z^21 - 2*x1^41*x2^39*x3^18*x4^7*z^21 + x1^45*x2^34*x3^19*x4^7*z^21 + x1^42*x2^37*x3^19*x4^7*z^21 + x1^41*x2^38*x3^19*x4^7*z^21 + 2*x1^40*x2^39*x3^19*x4^7*z^21 + x1^43*x2^35*x3^20*x4^7*z^21 - 2*x1^40*x2^38*x3^20*x4^7*z^21 + x1^41*x2^36*x3^21*x4^7*z^21 + 2*x1^39*x2^38*x3^21*x4^7*z^21 - x1^43*x2^33*x3^22*x4^7*z^21 - x1^42*x2^34*x3^22*x4^7*z^21 + x1^41*x2^35*x3^22*x4^7*z^21 - x1^40*x2^36*x3^22*x4^7*z^21 - x1^38*x2^38*x3^22*x4^7*z^21 - x1^46*x2^29*x3^23*x4^7*z^21 + x1^44*x2^31*x3^23*x4^7*z^21 - x1^43*x2^32*x3^23*x4^7*z^21 + x1^40*x2^35*x3^23*x4^7*z^21 + x1^39*x2^36*x3^23*x4^7*z^21 + x1^38*x2^37*x3^23*x4^7*z^21 + x1^45*x2^29*x3^24*x4^7*z^21 - x1^44*x2^30*x3^24*x4^7*z^21 - x1^43*x2^31*x3^24*x4^7*z^21 + x1^42*x2^32*x3^24*x4^7*z^21 - x1^41*x2^33*x3^24*x4^7*z^21 - 2*x1^39*x2^35*x3^24*x4^7*z^21 + x1^38*x2^36*x3^24*x4^7*z^21 - 2*x1^44*x2^29*x3^25*x4^7*z^21 - x1^41*x2^32*x3^25*x4^7*z^21 - x1^40*x2^33*x3^25*x4^7*z^21 + 2*x1^39*x2^34*x3^25*x4^7*z^21 + x1^43*x2^29*x3^26*x4^7*z^21 - x1^39*x2^33*x3^26*x4^7*z^21 - x1^38*x2^34*x3^26*x4^7*z^21 + x1^37*x2^35*x3^26*x4^7*z^21 - x1^42*x2^29*x3^27*x4^7*z^21 - x1^41*x2^30*x3^27*x4^7*z^21 - x1^40*x2^31*x3^27*x4^7*z^21 + x1^39*x2^32*x3^27*x4^7*z^21 + x1^38*x2^33*x3^27*x4^7*z^21 + x1^36*x2^35*x3^27*x4^7*z^21 + x1^41*x2^29*x3^28*x4^7*z^21 + x1^40*x2^30*x3^28*x4^7*z^21 - x1^39*x2^31*x3^28*x4^7*z^21 + x1^38*x2^32*x3^28*x4^7*z^21 + x1^37*x2^33*x3^28*x4^7*z^21 + x1^38*x2^31*x3^29*x4^7*z^21 + x1^50*x2^34*x3^13*x4^8*z^21 - x1^48*x2^36*x3^13*x4^8*z^21 + x1^47*x2^37*x3^13*x4^8*z^21 + x1^50*x2^33*x3^14*x4^8*z^21 - x1^49*x2^34*x3^14*x4^8*z^21 + 2*x1^48*x2^35*x3^14*x4^8*z^21 + x1^47*x2^36*x3^14*x4^8*z^21 - x1^46*x2^37*x3^14*x4^8*z^21 + x1^48*x2^34*x3^15*x4^8*z^21 - x1^47*x2^35*x3^15*x4^8*z^21 + x1^46*x2^36*x3^15*x4^8*z^21 - x1^45*x2^37*x3^15*x4^8*z^21 + x1^44*x2^38*x3^15*x4^8*z^21 + x1^48*x2^33*x3^16*x4^8*z^21 + x1^47*x2^34*x3^16*x4^8*z^21 + 2*x1^46*x2^35*x3^16*x4^8*z^21 + x1^45*x2^36*x3^16*x4^8*z^21 - 2*x1^43*x2^38*x3^16*x4^8*z^21 - x1^42*x2^39*x3^16*x4^8*z^21 - 2*x1^47*x2^33*x3^17*x4^8*z^21 - 3*x1^46*x2^34*x3^17*x4^8*z^21 + x1^45*x2^35*x3^17*x4^8*z^21 - 2*x1^44*x2^36*x3^17*x4^8*z^21 + 2*x1^43*x2^37*x3^17*x4^8*z^21 + 2*x1^42*x2^38*x3^17*x4^8*z^21 + 3*x1^41*x2^39*x3^17*x4^8*z^21 + x1^48*x2^31*x3^18*x4^8*z^21 + 2*x1^47*x2^32*x3^18*x4^8*z^21 + 4*x1^46*x2^33*x3^18*x4^8*z^21 + 2*x1^45*x2^34*x3^18*x4^8*z^21 + x1^44*x2^35*x3^18*x4^8*z^21 - x1^43*x2^36*x3^18*x4^8*z^21 - 2*x1^42*x2^37*x3^18*x4^8*z^21 - 3*x1^41*x2^38*x3^18*x4^8*z^21 - 3*x1^40*x2^39*x3^18*x4^8*z^21 - 2*x1^47*x2^31*x3^19*x4^8*z^21 - 2*x1^46*x2^32*x3^19*x4^8*z^21 - 5*x1^45*x2^33*x3^19*x4^8*z^21 - x1^43*x2^35*x3^19*x4^8*z^21 + 5*x1^42*x2^36*x3^19*x4^8*z^21 + 5*x1^40*x2^38*x3^19*x4^8*z^21 + x1^39*x2^39*x3^19*x4^8*z^21 + x1^47*x2^30*x3^20*x4^8*z^21 + 2*x1^46*x2^31*x3^20*x4^8*z^21 + 4*x1^45*x2^32*x3^20*x4^8*z^21 + 2*x1^44*x2^33*x3^20*x4^8*z^21 - x1^43*x2^34*x3^20*x4^8*z^21 - x1^42*x2^35*x3^20*x4^8*z^21 - 3*x1^41*x2^36*x3^20*x4^8*z^21 - 5*x1^39*x2^38*x3^20*x4^8*z^21 - 2*x1^46*x2^30*x3^21*x4^8*z^21 - x1^45*x2^31*x3^21*x4^8*z^21 - 4*x1^44*x2^32*x3^21*x4^8*z^21 - 3*x1^42*x2^34*x3^21*x4^8*z^21 + 2*x1^41*x2^35*x3^21*x4^8*z^21 + 2*x1^40*x2^36*x3^21*x4^8*z^21 + 4*x1^39*x2^37*x3^21*x4^8*z^21 + 2*x1^38*x2^38*x3^21*x4^8*z^21 + 2*x1^45*x2^30*x3^22*x4^8*z^21 + 2*x1^44*x2^31*x3^22*x4^8*z^21 + 4*x1^43*x2^32*x3^22*x4^8*z^21 + x1^42*x2^33*x3^22*x4^8*z^21 - 4*x1^40*x2^35*x3^22*x4^8*z^21 - 3*x1^39*x2^36*x3^22*x4^8*z^21 - 4*x1^38*x2^37*x3^22*x4^8*z^21 - x1^45*x2^29*x3^23*x4^8*z^21 - 2*x1^44*x2^30*x3^23*x4^8*z^21 - 3*x1^43*x2^31*x3^23*x4^8*z^21 - x1^42*x2^32*x3^23*x4^8*z^21 - 2*x1^41*x2^33*x3^23*x4^8*z^21 + 2*x1^40*x2^34*x3^23*x4^8*z^21 + 3*x1^38*x2^36*x3^23*x4^8*z^21 + x1^37*x2^37*x3^23*x4^8*z^21 + x1^45*x2^28*x3^24*x4^8*z^21 + x1^44*x2^29*x3^24*x4^8*z^21 + x1^42*x2^31*x3^24*x4^8*z^21 - 4*x1^39*x2^34*x3^24*x4^8*z^21 - 3*x1^38*x2^35*x3^24*x4^8*z^21 - 4*x1^37*x2^36*x3^24*x4^8*z^21 - x1^44*x2^28*x3^25*x4^8*z^21 + x1^43*x2^29*x3^25*x4^8*z^21 + x1^41*x2^31*x3^25*x4^8*z^21 + 2*x1^39*x2^33*x3^25*x4^8*z^21 + 2*x1^38*x2^34*x3^25*x4^8*z^21 + 2*x1^37*x2^35*x3^25*x4^8*z^21 + x1^36*x2^36*x3^25*x4^8*z^21 + x1^43*x2^28*x3^26*x4^8*z^21 - 2*x1^40*x2^31*x3^26*x4^8*z^21 - x1^38*x2^33*x3^26*x4^8*z^21 - x1^37*x2^34*x3^26*x4^8*z^21 - 2*x1^36*x2^35*x3^26*x4^8*z^21 - x1^42*x2^28*x3^27*x4^8*z^21 - x1^41*x2^29*x3^27*x4^8*z^21 - x1^39*x2^31*x3^27*x4^8*z^21 + x1^36*x2^34*x3^27*x4^8*z^21 - x1^39*x2^30*x3^28*x4^8*z^21 - x1^38*x2^31*x3^28*x4^8*z^21 - x1^37*x2^32*x3^28*x4^8*z^21 - x1^36*x2^33*x3^28*x4^8*z^21 - x1^35*x2^34*x3^28*x4^8*z^21 - x1^39*x2^29*x3^29*x4^8*z^21 - x1^36*x2^32*x3^29*x4^8*z^21 + x1^35*x2^33*x3^29*x4^8*z^21 - x1^49*x2^34*x3^13*x4^9*z^21 + x1^48*x2^34*x3^14*x4^9*z^21 + x1^46*x2^36*x3^14*x4^9*z^21 + x1^45*x2^37*x3^14*x4^9*z^21 - x1^44*x2^38*x3^14*x4^9*z^21 - x1^49*x2^32*x3^15*x4^9*z^21 - x1^47*x2^34*x3^15*x4^9*z^21 - 2*x1^46*x2^35*x3^15*x4^9*z^21 - x1^45*x2^36*x3^15*x4^9*z^21 - x1^44*x2^37*x3^15*x4^9*z^21 + 2*x1^48*x2^32*x3^16*x4^9*z^21 + 2*x1^47*x2^33*x3^16*x4^9*z^21 + 3*x1^46*x2^34*x3^16*x4^9*z^21 + x1^45*x2^35*x3^16*x4^9*z^21 + 3*x1^44*x2^36*x3^16*x4^9*z^21 + x1^43*x2^37*x3^16*x4^9*z^21 - x1^41*x2^39*x3^16*x4^9*z^21 - 2*x1^48*x2^31*x3^17*x4^9*z^21 - 4*x1^47*x2^32*x3^17*x4^9*z^21 - 3*x1^45*x2^34*x3^17*x4^9*z^21 - x1^43*x2^36*x3^17*x4^9*z^21 + x1^41*x2^38*x3^17*x4^9*z^21 + x1^40*x2^39*x3^17*x4^9*z^21 + 2*x1^47*x2^31*x3^18*x4^9*z^21 + 2*x1^46*x2^32*x3^18*x4^9*z^21 + 2*x1^45*x2^33*x3^18*x4^9*z^21 + x1^44*x2^34*x3^18*x4^9*z^21 - x1^43*x2^35*x3^18*x4^9*z^21 - x1^42*x2^36*x3^18*x4^9*z^21 + x1^41*x2^37*x3^18*x4^9*z^21 - x1^40*x2^38*x3^18*x4^9*z^21 - 2*x1^47*x2^30*x3^19*x4^9*z^21 - 4*x1^46*x2^31*x3^19*x4^9*z^21 - 2*x1^45*x2^32*x3^19*x4^9*z^21 - 2*x1^44*x2^33*x3^19*x4^9*z^21 + x1^43*x2^34*x3^19*x4^9*z^21 + x1^41*x2^36*x3^19*x4^9*z^21 + x1^39*x2^38*x3^19*x4^9*z^21 + 4*x1^46*x2^30*x3^20*x4^9*z^21 + x1^45*x2^31*x3^20*x4^9*z^21 + 3*x1^44*x2^32*x3^20*x4^9*z^21 + x1^42*x2^34*x3^20*x4^9*z^21 - 3*x1^41*x2^35*x3^20*x4^9*z^21 - x1^40*x2^36*x3^20*x4^9*z^21 - 2*x1^39*x2^37*x3^20*x4^9*z^21 - x1^46*x2^29*x3^21*x4^9*z^21 - 4*x1^45*x2^30*x3^21*x4^9*z^21 - 2*x1^44*x2^31*x3^21*x4^9*z^21 - 4*x1^43*x2^32*x3^21*x4^9*z^21 + 2*x1^42*x2^33*x3^21*x4^9*z^21 + 5*x1^40*x2^35*x3^21*x4^9*z^21 + 2*x1^38*x2^37*x3^21*x4^9*z^21 + 4*x1^45*x2^29*x3^22*x4^9*z^21 + 2*x1^44*x2^30*x3^22*x4^9*z^21 + 3*x1^43*x2^31*x3^22*x4^9*z^21 - 2*x1^40*x2^34*x3^22*x4^9*z^21 - 4*x1^39*x2^35*x3^22*x4^9*z^21 - 2*x1^38*x2^36*x3^22*x4^9*z^21 - x1^37*x2^37*x3^22*x4^9*z^21 - 2*x1^45*x2^28*x3^23*x4^9*z^21 - 4*x1^44*x2^29*x3^23*x4^9*z^21 - 4*x1^42*x2^31*x3^23*x4^9*z^21 + 2*x1^41*x2^32*x3^23*x4^9*z^21 + 5*x1^39*x2^34*x3^23*x4^9*z^21 + 3*x1^38*x2^35*x3^23*x4^9*z^21 + 2*x1^37*x2^36*x3^23*x4^9*z^21 + 4*x1^44*x2^28*x3^24*x4^9*z^21 + x1^43*x2^29*x3^24*x4^9*z^21 + x1^42*x2^30*x3^24*x4^9*z^21 + x1^41*x2^31*x3^24*x4^9*z^21 - x1^40*x2^32*x3^24*x4^9*z^21 - 2*x1^39*x2^33*x3^24*x4^9*z^21 - 2*x1^38*x2^34*x3^24*x4^9*z^21 - x1^37*x2^35*x3^24*x4^9*z^21 - 4*x1^43*x2^28*x3^25*x4^9*z^21 - 2*x1^41*x2^30*x3^25*x4^9*z^21 + 3*x1^40*x2^31*x3^25*x4^9*z^21 - x1^39*x2^32*x3^25*x4^9*z^21 + 3*x1^38*x2^33*x3^25*x4^9*z^21 + 2*x1^37*x2^34*x3^25*x4^9*z^21 + 2*x1^42*x2^28*x3^26*x4^9*z^21 + x1^40*x2^30*x3^26*x4^9*z^21 - 2*x1^37*x2^33*x3^26*x4^9*z^21 - x1^36*x2^34*x3^26*x4^9*z^21 - x1^35*x2^35*x3^26*x4^9*z^21 - 2*x1^41*x2^28*x3^27*x4^9*z^21 - 2*x1^40*x2^29*x3^27*x4^9*z^21 + x1^38*x2^31*x3^27*x4^9*z^21 + 3*x1^37*x2^32*x3^27*x4^9*z^21 + x1^36*x2^33*x3^27*x4^9*z^21 + x1^35*x2^34*x3^27*x4^9*z^21 + 2*x1^40*x2^28*x3^28*x4^9*z^21 - x1^37*x2^31*x3^28*x4^9*z^21 + x1^38*x2^29*x3^29*x4^9*z^21 - x1^37*x2^30*x3^29*x4^9*z^21 + x1^36*x2^31*x3^29*x4^9*z^21 - x1^34*x2^33*x3^29*x4^9*z^21 + x1^48*x2^34*x3^13*x4^10*z^21 - x1^46*x2^36*x3^13*x4^10*z^21 + x1^45*x2^37*x3^13*x4^10*z^21 + x1^47*x2^34*x3^14*x4^10*z^21 + x1^44*x2^37*x3^14*x4^10*z^21 - x1^47*x2^33*x3^15*x4^10*z^21 + x1^43*x2^37*x3^15*x4^10*z^21 + x1^42*x2^38*x3^15*x4^10*z^21 + x1^46*x2^33*x3^16*x4^10*z^21 - x1^43*x2^36*x3^16*x4^10*z^21 - 2*x1^46*x2^32*x3^17*x4^10*z^21 - 2*x1^43*x2^35*x3^17*x4^10*z^21 - x1^42*x2^36*x3^17*x4^10*z^21 - x1^41*x2^37*x3^17*x4^10*z^21 + x1^47*x2^30*x3^18*x4^10*z^21 - x1^46*x2^31*x3^18*x4^10*z^21 + x1^45*x2^32*x3^18*x4^10*z^21 - 2*x1^44*x2^33*x3^18*x4^10*z^21 + x1^42*x2^35*x3^18*x4^10*z^21 + x1^41*x2^36*x3^18*x4^10*z^21 - x1^40*x2^37*x3^18*x4^10*z^21 - x1^44*x2^32*x3^19*x4^10*z^21 - x1^43*x2^33*x3^19*x4^10*z^21 - x1^42*x2^34*x3^19*x4^10*z^21 + x1^41*x2^35*x3^19*x4^10*z^21 - 2*x1^40*x2^36*x3^19*x4^10*z^21 + x1^46*x2^29*x3^20*x4^10*z^21 + x1^39*x2^36*x3^20*x4^10*z^21 - 2*x1^45*x2^29*x3^21*x4^10*z^21 + x1^44*x2^30*x3^21*x4^10*z^21 + x1^45*x2^28*x3^22*x4^10*z^21 + 2*x1^44*x2^29*x3^22*x4^10*z^21 - x1^43*x2^30*x3^22*x4^10*z^21 + 2*x1^42*x2^31*x3^22*x4^10*z^21 - 2*x1^44*x2^28*x3^23*x4^10*z^21 - 2*x1^43*x2^29*x3^23*x4^10*z^21 + x1^40*x2^32*x3^23*x4^10*z^21 + 2*x1^43*x2^28*x3^24*x4^10*z^21 + x1^42*x2^29*x3^24*x4^10*z^21 + x1^41*x2^30*x3^24*x4^10*z^21 - x1^40*x2^31*x3^24*x4^10*z^21 + x1^38*x2^33*x3^24*x4^10*z^21 - x1^42*x2^28*x3^25*x4^10*z^21 - x1^41*x2^29*x3^25*x4^10*z^21 - x1^39*x2^31*x3^25*x4^10*z^21 + x1^38*x2^32*x3^25*x4^10*z^21 - x1^37*x2^33*x3^25*x4^10*z^21 + x1^40*x2^29*x3^26*x4^10*z^21 - 3*x1^39*x2^30*x3^26*x4^10*z^21 - x1^38*x2^31*x3^26*x4^10*z^21 - x1^37*x2^32*x3^26*x4^10*z^21 + x1^36*x2^33*x3^26*x4^10*z^21 + 2*x1^35*x2^34*x3^26*x4^10*z^21 - x1^37*x2^31*x3^27*x4^10*z^21 + x1^36*x2^32*x3^27*x4^10*z^21 - x1^38*x2^29*x3^28*x4^10*z^21 + x1^37*x2^30*x3^28*x4^10*z^21 - 3*x1^36*x2^31*x3^28*x4^10*z^21 - x1^35*x2^32*x3^28*x4^10*z^21 + 2*x1^34*x2^33*x3^28*x4^10*z^21 + x1^37*x2^29*x3^29*x4^10*z^21 + x1^34*x2^32*x3^29*x4^10*z^21 - x1^47*x2^33*x3^14*x4^11*z^21 - x1^46*x2^34*x3^14*x4^11*z^21 + 2*x1^45*x2^35*x3^14*x4^11*z^21 - x1^43*x2^37*x3^14*x4^11*z^21 + x1^47*x2^32*x3^15*x4^11*z^21 - x1^46*x2^33*x3^15*x4^11*z^21 + x1^45*x2^34*x3^15*x4^11*z^21 - x1^44*x2^35*x3^15*x4^11*z^21 + x1^41*x2^38*x3^15*x4^11*z^21 + 2*x1^43*x2^35*x3^16*x4^11*z^21 + 2*x1^42*x2^36*x3^16*x4^11*z^21 + 2*x1^41*x2^37*x3^16*x4^11*z^21 - x1^40*x2^38*x3^16*x4^11*z^21 + x1^47*x2^30*x3^17*x4^11*z^21 + 3*x1^46*x2^31*x3^17*x4^11*z^21 - 2*x1^45*x2^32*x3^17*x4^11*z^21 - x1^44*x2^33*x3^17*x4^11*z^21 - 3*x1^43*x2^34*x3^17*x4^11*z^21 - x1^41*x2^36*x3^17*x4^11*z^21 + x1^39*x2^38*x3^17*x4^11*z^21 - x1^46*x2^30*x3^18*x4^11*z^21 + x1^45*x2^31*x3^18*x4^11*z^21 + 3*x1^44*x2^32*x3^18*x4^11*z^21 + x1^42*x2^34*x3^18*x4^11*z^21 + 3*x1^40*x2^36*x3^18*x4^11*z^21 - x1^39*x2^37*x3^18*x4^11*z^21 - x1^38*x2^38*x3^18*x4^11*z^21 + 4*x1^45*x2^30*x3^19*x4^11*z^21 - 2*x1^44*x2^31*x3^19*x4^11*z^21 + 2*x1^43*x2^32*x3^19*x4^11*z^21 - 4*x1^42*x2^33*x3^19*x4^11*z^21 - x1^41*x2^34*x3^19*x4^11*z^21 - 3*x1^40*x2^35*x3^19*x4^11*z^21 + x1^38*x2^37*x3^19*x4^11*z^21 - x1^45*x2^29*x3^20*x4^11*z^21 + 2*x1^43*x2^31*x3^20*x4^11*z^21 + 2*x1^42*x2^32*x3^20*x4^11*z^21 + 3*x1^41*x2^33*x3^20*x4^11*z^21 - x1^40*x2^34*x3^20*x4^11*z^21 + 4*x1^39*x2^35*x3^20*x4^11*z^21 - 2*x1^38*x2^36*x3^20*x4^11*z^21 + 3*x1^44*x2^29*x3^21*x4^11*z^21 - x1^43*x2^30*x3^21*x4^11*z^21 + 3*x1^42*x2^31*x3^21*x4^11*z^21 - 2*x1^41*x2^32*x3^21*x4^11*z^21 - 4*x1^39*x2^34*x3^21*x4^11*z^21 - 2*x1^38*x2^35*x3^21*x4^11*z^21 + 2*x1^37*x2^36*x3^21*x4^11*z^21 - x1^44*x2^28*x3^22*x4^11*z^21 - 2*x1^43*x2^29*x3^22*x4^11*z^21 - x1^41*x2^31*x3^22*x4^11*z^21 + 4*x1^40*x2^32*x3^22*x4^11*z^21 + x1^39*x2^33*x3^22*x4^11*z^21 + 4*x1^38*x2^34*x3^22*x4^11*z^21 - 2*x1^37*x2^35*x3^22*x4^11*z^21 - x1^36*x2^36*x3^22*x4^11*z^21 + x1^43*x2^28*x3^23*x4^11*z^21 - x1^42*x2^29*x3^23*x4^11*z^21 - 3*x1^41*x2^30*x3^23*x4^11*z^21 - 4*x1^40*x2^31*x3^23*x4^11*z^21 - 3*x1^38*x2^33*x3^23*x4^11*z^21 + 2*x1^36*x2^35*x3^23*x4^11*z^21 + x1^41*x2^29*x3^24*x4^11*z^21 + x1^40*x2^30*x3^24*x4^11*z^21 - 3*x1^38*x2^32*x3^24*x4^11*z^21 + 4*x1^37*x2^33*x3^24*x4^11*z^21 - 2*x1^36*x2^34*x3^24*x4^11*z^21 - x1^35*x2^35*x3^24*x4^11*z^21 + x1^40*x2^29*x3^25*x4^11*z^21 + x1^38*x2^31*x3^25*x4^11*z^21 + 2*x1^35*x2^34*x3^25*x4^11*z^21 - 3*x1^35*x2^33*x3^26*x4^11*z^21 + x1^37*x2^30*x3^27*x4^11*z^21 - 2*x1^36*x2^31*x3^27*x4^11*z^21 - x1^35*x2^32*x3^27*x4^11*z^21 + 2*x1^34*x2^33*x3^27*x4^11*z^21 + x1^35*x2^31*x3^28*x4^11*z^21 - x1^34*x2^32*x3^28*x4^11*z^21 - x1^45*x2^35*x3^13*x4^12*z^21 + x1^44*x2^35*x3^14*x4^12*z^21 + x1^46*x2^32*x3^15*x4^12*z^21 + x1^45*x2^33*x3^15*x4^12*z^21 - 2*x1^44*x2^34*x3^15*x4^12*z^21 - x1^43*x2^35*x3^15*x4^12*z^21 + x1^41*x2^37*x3^15*x4^12*z^21 - 2*x1^46*x2^31*x3^16*x4^12*z^21 + 2*x1^45*x2^32*x3^16*x4^12*z^21 + x1^44*x2^33*x3^16*x4^12*z^21 - x1^40*x2^37*x3^16*x4^12*z^21 + x1^45*x2^31*x3^17*x4^12*z^21 - 2*x1^44*x2^32*x3^17*x4^12*z^21 - x1^43*x2^33*x3^17*x4^12*z^21 + x1^41*x2^35*x3^17*x4^12*z^21 + 3*x1^39*x2^37*x3^17*x4^12*z^21 - x1^46*x2^29*x3^18*x4^12*z^21 - x1^45*x2^30*x3^18*x4^12*z^21 + 3*x1^44*x2^31*x3^18*x4^12*z^21 + x1^43*x2^32*x3^18*x4^12*z^21 + x1^42*x2^33*x3^18*x4^12*z^21 - x1^41*x2^34*x3^18*x4^12*z^21 - x1^40*x2^35*x3^18*x4^12*z^21 - 3*x1^39*x2^36*x3^18*x4^12*z^21 - 3*x1^38*x2^37*x3^18*x4^12*z^21 + x1^45*x2^29*x3^19*x4^12*z^21 - 4*x1^43*x2^31*x3^19*x4^12*z^21 - x1^41*x2^33*x3^19*x4^12*z^21 + 4*x1^40*x2^34*x3^19*x4^12*z^21 - 2*x1^39*x2^35*x3^19*x4^12*z^21 + 6*x1^38*x2^36*x3^19*x4^12*z^21 + x1^37*x2^37*x3^19*x4^12*z^21 - 3*x1^44*x2^29*x3^20*x4^12*z^21 + 6*x1^43*x2^30*x3^20*x4^12*z^21 + 3*x1^41*x2^32*x3^20*x4^12*z^21 - x1^40*x2^33*x3^20*x4^12*z^21 - x1^38*x2^35*x3^20*x4^12*z^21 - 6*x1^37*x2^36*x3^20*x4^12*z^21 - x1^44*x2^28*x3^21*x4^12*z^21 - 5*x1^42*x2^30*x3^21*x4^12*z^21 - x1^41*x2^31*x3^21*x4^12*z^21 - 5*x1^40*x2^32*x3^21*x4^12*z^21 + 3*x1^39*x2^33*x3^21*x4^12*z^21 - 2*x1^38*x2^34*x3^21*x4^12*z^21 + 6*x1^37*x2^35*x3^21*x4^12*z^21 + 2*x1^36*x2^36*x3^21*x4^12*z^21 + 3*x1^42*x2^29*x3^22*x4^12*z^21 + 2*x1^41*x2^30*x3^22*x4^12*z^21 + 2*x1^40*x2^31*x3^22*x4^12*z^21 - x1^37*x2^34*x3^22*x4^12*z^21 - 6*x1^36*x2^35*x3^22*x4^12*z^21 - x1^42*x2^28*x3^23*x4^12*z^21 - 3*x1^41*x2^29*x3^23*x4^12*z^21 - 3*x1^39*x2^31*x3^23*x4^12*z^21 + 2*x1^38*x2^32*x3^23*x4^12*z^21 - 2*x1^37*x2^33*x3^23*x4^12*z^21 + 6*x1^36*x2^34*x3^23*x4^12*z^21 + 2*x1^35*x2^35*x3^23*x4^12*z^21 + x1^41*x2^28*x3^24*x4^12*z^21 + x1^40*x2^29*x3^24*x4^12*z^21 + x1^39*x2^30*x3^24*x4^12*z^21 - x1^38*x2^31*x3^24*x4^12*z^21 + x1^37*x2^32*x3^24*x4^12*z^21 - 2*x1^36*x2^33*x3^24*x4^12*z^21 - 6*x1^35*x2^34*x3^24*x4^12*z^21 - x1^40*x2^28*x3^25*x4^12*z^21 - 2*x1^39*x2^29*x3^25*x4^12*z^21 - x1^38*x2^30*x3^25*x4^12*z^21 + 3*x1^37*x2^31*x3^25*x4^12*z^21 - 3*x1^36*x2^32*x3^25*x4^12*z^21 + 5*x1^35*x2^33*x3^25*x4^12*z^21 + 2*x1^34*x2^34*x3^25*x4^12*z^21 + x1^38*x2^29*x3^26*x4^12*z^21 + x1^35*x2^32*x3^26*x4^12*z^21 - 5*x1^34*x2^33*x3^26*x4^12*z^21 - x1^35*x2^31*x3^27*x4^12*z^21 + 2*x1^34*x2^32*x3^27*x4^12*z^21 + 2*x1^33*x2^33*x3^27*x4^12*z^21 + x1^34*x2^31*x3^28*x4^12*z^21 - 2*x1^33*x2^32*x3^28*x4^12*z^21 + x1^44*x2^34*x3^14*x4^13*z^21 + x1^43*x2^35*x3^14*x4^13*z^21 - 2*x1^43*x2^34*x3^15*x4^13*z^21 - x1^42*x2^35*x3^15*x4^13*z^21 - x1^41*x2^36*x3^15*x4^13*z^21 + x1^43*x2^33*x3^16*x4^13*z^21 - 2*x1^41*x2^35*x3^16*x4^13*z^21 - x1^39*x2^37*x3^16*x4^13*z^21 + x1^45*x2^30*x3^17*x4^13*z^21 - x1^44*x2^31*x3^17*x4^13*z^21 + x1^43*x2^32*x3^17*x4^13*z^21 + x1^41*x2^34*x3^17*x4^13*z^21 - x1^39*x2^36*x3^17*x4^13*z^21 + 2*x1^38*x2^37*x3^17*x4^13*z^21 - 2*x1^44*x2^30*x3^18*x4^13*z^21 + x1^43*x2^31*x3^18*x4^13*z^21 + x1^42*x2^32*x3^18*x4^13*z^21 - x1^41*x2^33*x3^18*x4^13*z^21 - 4*x1^40*x2^34*x3^18*x4^13*z^21 + x1^39*x2^35*x3^18*x4^13*z^21 - 3*x1^38*x2^36*x3^18*x4^13*z^21 - 2*x1^37*x2^37*x3^18*x4^13*z^21 + x1^44*x2^29*x3^19*x4^13*z^21 - 2*x1^43*x2^30*x3^19*x4^13*z^21 - x1^42*x2^31*x3^19*x4^13*z^21 + 2*x1^40*x2^33*x3^19*x4^13*z^21 + x1^38*x2^35*x3^19*x4^13*z^21 + 6*x1^37*x2^36*x3^19*x4^13*z^21 - 2*x1^43*x2^29*x3^20*x4^13*z^21 + 3*x1^42*x2^30*x3^20*x4^13*z^21 - 4*x1^39*x2^33*x3^20*x4^13*z^21 - 6*x1^37*x2^35*x3^20*x4^13*z^21 - 2*x1^36*x2^36*x3^20*x4^13*z^21 + x1^43*x2^28*x3^21*x4^13*z^21 - 3*x1^42*x2^29*x3^21*x4^13*z^21 - 3*x1^41*x2^30*x3^21*x4^13*z^21 - x1^40*x2^31*x3^21*x4^13*z^21 + 2*x1^39*x2^32*x3^21*x4^13*z^21 + 2*x1^38*x2^33*x3^21*x4^13*z^21 + 2*x1^37*x2^34*x3^21*x4^13*z^21 + 6*x1^36*x2^35*x3^21*x4^13*z^21 - x1^43*x2^27*x3^22*x4^13*z^21 + 2*x1^41*x2^29*x3^22*x4^13*z^21 + 2*x1^39*x2^31*x3^22*x4^13*z^21 - 4*x1^38*x2^32*x3^22*x4^13*z^21 - 6*x1^36*x2^34*x3^22*x4^13*z^21 - 2*x1^35*x2^35*x3^22*x4^13*z^21 + x1^42*x2^27*x3^23*x4^13*z^21 - x1^41*x2^28*x3^23*x4^13*z^21 - 2*x1^40*x2^29*x3^23*x4^13*z^21 - 3*x1^39*x2^30*x3^23*x4^13*z^21 + x1^37*x2^32*x3^23*x4^13*z^21 + 2*x1^36*x2^33*x3^23*x4^13*z^21 + 6*x1^35*x2^34*x3^23*x4^13*z^21 + x1^40*x2^28*x3^24*x4^13*z^21 + x1^39*x2^29*x3^24*x4^13*z^21 + 2*x1^38*x2^30*x3^24*x4^13*z^21 - 3*x1^37*x2^31*x3^24*x4^13*z^21 + x1^36*x2^32*x3^24*x4^13*z^21 - 6*x1^35*x2^33*x3^24*x4^13*z^21 - 2*x1^34*x2^34*x3^24*x4^13*z^21 + x1^39*x2^28*x3^25*x4^13*z^21 + 2*x1^36*x2^31*x3^25*x4^13*z^21 + x1^35*x2^32*x3^25*x4^13*z^21 + 6*x1^34*x2^33*x3^25*x4^13*z^21 + x1^37*x2^29*x3^26*x4^13*z^21 - 2*x1^34*x2^32*x3^26*x4^13*z^21 - 2*x1^33*x2^33*x3^26*x4^13*z^21 + x1^35*x2^30*x3^27*x4^13*z^21 + 2*x1^33*x2^32*x3^27*x4^13*z^21 - x1^32*x2^32*x3^28*x4^13*z^21 - x1^42*x2^34*x3^15*x4^14*z^21 - x1^41*x2^35*x3^15*x4^14*z^21 + 2*x1^42*x2^33*x3^16*x4^14*z^21 - x1^42*x2^32*x3^17*x4^14*z^21 - x1^41*x2^33*x3^17*x4^14*z^21 - x1^40*x2^34*x3^17*x4^14*z^21 - 3*x1^39*x2^35*x3^17*x4^14*z^21 - x1^44*x2^29*x3^18*x4^14*z^21 - x1^43*x2^30*x3^18*x4^14*z^21 + 3*x1^41*x2^32*x3^18*x4^14*z^21 - x1^40*x2^33*x3^18*x4^14*z^21 - x1^37*x2^36*x3^18*x4^14*z^21 + x1^44*x2^28*x3^19*x4^14*z^21 + x1^43*x2^29*x3^19*x4^14*z^21 - x1^42*x2^30*x3^19*x4^14*z^21 - x1^41*x2^31*x3^19*x4^14*z^21 + x1^40*x2^32*x3^19*x4^14*z^21 + x1^39*x2^33*x3^19*x4^14*z^21 - 3*x1^38*x2^34*x3^19*x4^14*z^21 + x1^36*x2^36*x3^19*x4^14*z^21 - 2*x1^43*x2^28*x3^20*x4^14*z^21 - x1^41*x2^30*x3^20*x4^14*z^21 + 2*x1^40*x2^31*x3^20*x4^14*z^21 + 3*x1^38*x2^33*x3^20*x4^14*z^21 - 2*x1^36*x2^35*x3^20*x4^14*z^21 + x1^43*x2^27*x3^21*x4^14*z^21 + 2*x1^42*x2^28*x3^21*x4^14*z^21 - 2*x1^40*x2^30*x3^21*x4^14*z^21 + 2*x1^38*x2^32*x3^21*x4^14*z^21 - 2*x1^37*x2^33*x3^21*x4^14*z^21 + 2*x1^36*x2^34*x3^21*x4^14*z^21 - x1^42*x2^27*x3^22*x4^14*z^21 - x1^41*x2^28*x3^22*x4^14*z^21 - 2*x1^40*x2^29*x3^22*x4^14*z^21 + x1^39*x2^30*x3^22*x4^14*z^21 - x1^38*x2^31*x3^22*x4^14*z^21 + 2*x1^37*x2^32*x3^22*x4^14*z^21 + x1^36*x2^33*x3^22*x4^14*z^21 - 2*x1^35*x2^34*x3^22*x4^14*z^21 + x1^40*x2^28*x3^23*x4^14*z^21 + x1^39*x2^29*x3^23*x4^14*z^21 - x1^38*x2^30*x3^23*x4^14*z^21 - x1^37*x2^31*x3^23*x4^14*z^21 - 2*x1^36*x2^32*x3^23*x4^14*z^21 + 2*x1^35*x2^33*x3^23*x4^14*z^21 + x1^34*x2^34*x3^23*x4^14*z^21 + x1^38*x2^29*x3^24*x4^14*z^21 - x1^37*x2^30*x3^24*x4^14*z^21 + 2*x1^36*x2^31*x3^24*x4^14*z^21 - 2*x1^34*x2^33*x3^24*x4^14*z^21 + x1^37*x2^29*x3^25*x4^14*z^21 + x1^36*x2^30*x3^25*x4^14*z^21 - 3*x1^35*x2^31*x3^25*x4^14*z^21 + x1^34*x2^32*x3^25*x4^14*z^21 + x1^33*x2^33*x3^25*x4^14*z^21 - x1^36*x2^29*x3^26*x4^14*z^21 + x1^34*x2^31*x3^26*x4^14*z^21 - x1^33*x2^32*x3^26*x4^14*z^21 + x1^41*x2^33*x3^16*x4^15*z^21 + x1^40*x2^34*x3^16*x4^15*z^21 + x1^39*x2^35*x3^16*x4^15*z^21 - 3*x1^41*x2^32*x3^17*x4^15*z^21 + x1^40*x2^33*x3^17*x4^15*z^21 - x1^39*x2^34*x3^17*x4^15*z^21 - x1^38*x2^35*x3^17*x4^15*z^21 + 2*x1^41*x2^31*x3^18*x4^15*z^21 + 2*x1^40*x2^32*x3^18*x4^15*z^21 + 4*x1^38*x2^34*x3^18*x4^15*z^21 + x1^43*x2^28*x3^19*x4^15*z^21 + x1^42*x2^29*x3^19*x4^15*z^21 - 5*x1^40*x2^31*x3^19*x4^15*z^21 + x1^39*x2^32*x3^19*x4^15*z^21 - 3*x1^38*x2^33*x3^19*x4^15*z^21 - 2*x1^37*x2^34*x3^19*x4^15*z^21 - x1^43*x2^27*x3^20*x4^15*z^21 - x1^42*x2^28*x3^20*x4^15*z^21 + x1^41*x2^29*x3^20*x4^15*z^21 + x1^40*x2^30*x3^20*x4^15*z^21 + 2*x1^39*x2^31*x3^20*x4^15*z^21 + 2*x1^38*x2^32*x3^20*x4^15*z^21 + 5*x1^37*x2^33*x3^20*x4^15*z^21 + 2*x1^42*x2^27*x3^21*x4^15*z^21 - x1^41*x2^28*x3^21*x4^15*z^21 - 4*x1^39*x2^30*x3^21*x4^15*z^21 - 6*x1^37*x2^32*x3^21*x4^15*z^21 - 2*x1^36*x2^33*x3^21*x4^15*z^21 - x1^41*x2^27*x3^22*x4^15*z^21 + 2*x1^39*x2^29*x3^22*x4^15*z^21 + 2*x1^38*x2^30*x3^22*x4^15*z^21 + 2*x1^37*x2^31*x3^22*x4^15*z^21 + 6*x1^36*x2^32*x3^22*x4^15*z^21 + x1^40*x2^27*x3^23*x4^15*z^21 - 3*x1^38*x2^29*x3^23*x4^15*z^21 - 5*x1^36*x2^31*x3^23*x4^15*z^21 - 2*x1^35*x2^32*x3^23*x4^15*z^21 - x1^39*x2^27*x3^24*x4^15*z^21 + x1^38*x2^28*x3^24*x4^15*z^21 + x1^37*x2^29*x3^24*x4^15*z^21 + x1^36*x2^30*x3^24*x4^15*z^21 + 5*x1^35*x2^31*x3^24*x4^15*z^21 - 2*x1^37*x2^28*x3^25*x4^15*z^21 - 3*x1^35*x2^30*x3^25*x4^15*z^21 - 2*x1^34*x2^31*x3^25*x4^15*z^21 - x1^35*x2^29*x3^26*x4^15*z^21 + 3*x1^34*x2^30*x3^26*x4^15*z^21 - 2*x1^33*x2^30*x3^27*x4^15*z^21 + x1^40*x2^31*x3^18*x4^16*z^21 - x1^39*x2^32*x3^18*x4^16*z^21 + x1^38*x2^33*x3^18*x4^16*z^21 + x1^37*x2^34*x3^18*x4^16*z^21 - x1^40*x2^30*x3^19*x4^16*z^21 - 2*x1^39*x2^31*x3^19*x4^16*z^21 - 2*x1^37*x2^33*x3^19*x4^16*z^21 + 4*x1^39*x2^30*x3^20*x4^16*z^21 + 2*x1^37*x2^32*x3^20*x4^16*z^21 + 2*x1^36*x2^33*x3^20*x4^16*z^21 + x1^40*x2^28*x3^21*x4^16*z^21 - 2*x1^38*x2^30*x3^21*x4^16*z^21 - x1^37*x2^31*x3^21*x4^16*z^21 - 4*x1^36*x2^32*x3^21*x4^16*z^21 - x1^39*x2^28*x3^22*x4^16*z^21 + 2*x1^38*x2^29*x3^22*x4^16*z^21 - 2*x1^37*x2^30*x3^22*x4^16*z^21 + 5*x1^36*x2^31*x3^22*x4^16*z^21 + 2*x1^35*x2^32*x3^22*x4^16*z^21 + x1^39*x2^27*x3^23*x4^16*z^21 - x1^37*x2^29*x3^23*x4^16*z^21 - x1^36*x2^30*x3^23*x4^16*z^21 - 6*x1^35*x2^31*x3^23*x4^16*z^21 - x1^38*x2^27*x3^24*x4^16*z^21 - x1^36*x2^29*x3^24*x4^16*z^21 + 2*x1^35*x2^30*x3^24*x4^16*z^21 + 2*x1^34*x2^31*x3^24*x4^16*z^21 + x1^35*x2^29*x3^25*x4^16*z^21 - 2*x1^34*x2^30*x3^25*x4^16*z^21 + x1^33*x2^30*x3^26*x4^16*z^21 - x1^38*x2^29*x3^21*x4^17*z^21 - x1^37*x2^30*x3^21*x4^17*z^21 - x1^36*x2^31*x3^21*x4^17*z^21 - x1^36*x2^30*x3^22*x4^17*z^21 + x1^35*x2^31*x3^22*x4^17*z^21 + x1^36*x2^29*x3^23*x4^17*z^21 - x1^34*x2^31*x3^23*x4^17*z^21 + x1^48*x2^34*x3^18*z^20 - x1^46*x2^36*x3^18*z^20 - x1^47*x2^34*x3^19*z^20 + x1^46*x2^35*x3^19*z^20 + x1^46*x2^34*x3^20*z^20 + x1^45*x2^35*x3^20*z^20 + x1^44*x2^36*x3^20*z^20 - x1^49*x2^35*x3^15*x4*z^20 + 2*x1^48*x2^35*x3^16*x4*z^20 - x1^47*x2^36*x3^16*x4*z^20 - 2*x1^48*x2^34*x3^17*x4*z^20 - 2*x1^47*x2^35*x3^17*x4*z^20 + x1^46*x2^36*x3^17*x4*z^20 + x1^44*x2^38*x3^17*x4*z^20 + 6*x1^47*x2^34*x3^18*x4*z^20 + x1^46*x2^35*x3^18*x4*z^20 + x1^45*x2^36*x3^18*x4*z^20 - x1^43*x2^38*x3^18*x4*z^20 - 2*x1^47*x2^33*x3^19*x4*z^20 - 5*x1^46*x2^34*x3^19*x4*z^20 + 2*x1^45*x2^35*x3^19*x4*z^20 - 2*x1^44*x2^36*x3^19*x4*z^20 + x1^43*x2^37*x3^19*x4*z^20 + 4*x1^46*x2^33*x3^20*x4*z^20 + x1^45*x2^34*x3^20*x4*z^20 + 2*x1^44*x2^35*x3^20*x4*z^20 - x1^42*x2^37*x3^20*x4*z^20 - 2*x1^46*x2^32*x3^21*x4*z^20 - 2*x1^45*x2^33*x3^21*x4*z^20 - 4*x1^43*x2^35*x3^21*x4*z^20 + 2*x1^45*x2^32*x3^22*x4*z^20 + 2*x1^43*x2^34*x3^22*x4*z^20 + x1^42*x2^35*x3^22*x4*z^20 - x1^45*x2^31*x3^23*x4*z^20 - x1^44*x2^32*x3^23*x4*z^20 + x1^43*x2^33*x3^23*x4*z^20 - x1^42*x2^34*x3^23*x4*z^20 + 2*x1^49*x2^35*x3^14*x4^2*z^20 - 3*x1^48*x2^35*x3^15*x4^2*z^20 + x1^47*x2^36*x3^15*x4^2*z^20 + 2*x1^48*x2^34*x3^16*x4^2*z^20 + 3*x1^47*x2^35*x3^16*x4^2*z^20 + 2*x1^45*x2^37*x3^16*x4^2*z^20 - 5*x1^47*x2^34*x3^17*x4^2*z^20 - x1^46*x2^35*x3^17*x4^2*z^20 - x1^45*x2^36*x3^17*x4^2*z^20 - x1^44*x2^37*x3^17*x4^2*z^20 + x1^43*x2^38*x3^17*x4^2*z^20 + 2*x1^47*x2^33*x3^18*x4^2*z^20 + 5*x1^46*x2^34*x3^18*x4^2*z^20 + 3*x1^44*x2^36*x3^18*x4^2*z^20 - x1^42*x2^38*x3^18*x4^2*z^20 - 6*x1^46*x2^33*x3^19*x4^2*z^20 - 2*x1^45*x2^34*x3^19*x4^2*z^20 - 2*x1^44*x2^35*x3^19*x4^2*z^20 - 2*x1^43*x2^36*x3^19*x4^2*z^20 + x1^41*x2^38*x3^19*x4^2*z^20 + 2*x1^46*x2^32*x3^20*x4^2*z^20 + 6*x1^45*x2^33*x3^20*x4^2*z^20 + 4*x1^43*x2^35*x3^20*x4^2*z^20 + x1^41*x2^37*x3^20*x4^2*z^20 - 2*x1^40*x2^38*x3^20*x4^2*z^20 - 5*x1^45*x2^32*x3^21*x4^2*z^20 - 2*x1^44*x2^33*x3^21*x4^2*z^20 - 2*x1^43*x2^34*x3^21*x4^2*z^20 - x1^42*x2^35*x3^21*x4^2*z^20 - x1^41*x2^36*x3^21*x4^2*z^20 + x1^40*x2^37*x3^21*x4^2*z^20 + x1^39*x2^38*x3^21*x4^2*z^20 + 2*x1^45*x2^31*x3^22*x4^2*z^20 + 3*x1^44*x2^32*x3^22*x4^2*z^20 - x1^43*x2^33*x3^22*x4^2*z^20 + 5*x1^42*x2^34*x3^22*x4^2*z^20 - x1^40*x2^36*x3^22*x4^2*z^20 - x1^39*x2^37*x3^22*x4^2*z^20 - 4*x1^44*x2^31*x3^23*x4^2*z^20 - 2*x1^42*x2^33*x3^23*x4^2*z^20 - 2*x1^41*x2^34*x3^23*x4^2*z^20 + x1^44*x2^30*x3^24*x4^2*z^20 + x1^43*x2^31*x3^24*x4^2*z^20 - x1^42*x2^32*x3^24*x4^2*z^20 + 3*x1^41*x2^33*x3^24*x4^2*z^20 - x1^43*x2^30*x3^25*x4^2*z^20 - x1^42*x2^31*x3^25*x4^2*z^20 - x1^41*x2^32*x3^25*x4^2*z^20 + x1^49*x2^33*x3^15*x4^3*z^20 - x1^48*x2^34*x3^15*x4^3*z^20 + x1^46*x2^36*x3^15*x4^3*z^20 - x1^49*x2^32*x3^16*x4^3*z^20 - x1^48*x2^33*x3^16*x4^3*z^20 + 3*x1^47*x2^34*x3^16*x4^3*z^20 - x1^46*x2^35*x3^16*x4^3*z^20 - x1^45*x2^36*x3^16*x4^3*z^20 + 2*x1^48*x2^32*x3^17*x4^3*z^20 - 2*x1^46*x2^34*x3^17*x4^3*z^20 + x1^45*x2^35*x3^17*x4^3*z^20 - x1^44*x2^36*x3^17*x4^3*z^20 - x1^48*x2^31*x3^18*x4^3*z^20 - 2*x1^47*x2^32*x3^18*x4^3*z^20 + 2*x1^46*x2^33*x3^18*x4^3*z^20 + x1^45*x2^34*x3^18*x4^3*z^20 + x1^44*x2^35*x3^18*x4^3*z^20 - x1^43*x2^36*x3^18*x4^3*z^20 - x1^42*x2^37*x3^18*x4^3*z^20 + 2*x1^47*x2^31*x3^19*x4^3*z^20 - x1^46*x2^32*x3^19*x4^3*z^20 - 2*x1^45*x2^33*x3^19*x4^3*z^20 + x1^44*x2^34*x3^19*x4^3*z^20 - x1^43*x2^35*x3^19*x4^3*z^20 + 2*x1^42*x2^36*x3^19*x4^3*z^20 + x1^41*x2^37*x3^19*x4^3*z^20 + x1^40*x2^38*x3^19*x4^3*z^20 - 2*x1^46*x2^31*x3^20*x4^3*z^20 + 2*x1^45*x2^32*x3^20*x4^3*z^20 - 2*x1^44*x2^33*x3^20*x4^3*z^20 + 2*x1^42*x2^35*x3^20*x4^3*z^20 - 2*x1^40*x2^37*x3^20*x4^3*z^20 - x1^39*x2^38*x3^20*x4^3*z^20 + 2*x1^46*x2^30*x3^21*x4^3*z^20 - 3*x1^44*x2^32*x3^21*x4^3*z^20 - 2*x1^42*x2^34*x3^21*x4^3*z^20 + x1^41*x2^35*x3^21*x4^3*z^20 + 2*x1^39*x2^37*x3^21*x4^3*z^20 - x1^46*x2^29*x3^22*x4^3*z^20 + 3*x1^44*x2^31*x3^22*x4^3*z^20 - x1^43*x2^32*x3^22*x4^3*z^20 - x1^42*x2^33*x3^22*x4^3*z^20 + x1^41*x2^34*x3^22*x4^3*z^20 + x1^40*x2^35*x3^22*x4^3*z^20 - x1^39*x2^36*x3^22*x4^3*z^20 - x1^38*x2^37*x3^22*x4^3*z^20 + x1^45*x2^29*x3^23*x4^3*z^20 + x1^42*x2^32*x3^23*x4^3*z^20 - 3*x1^41*x2^33*x3^23*x4^3*z^20 + x1^39*x2^35*x3^23*x4^3*z^20 + x1^38*x2^36*x3^23*x4^3*z^20 + 3*x1^43*x2^30*x3^24*x4^3*z^20 + x1^42*x2^31*x3^24*x4^3*z^20 + x1^41*x2^32*x3^24*x4^3*z^20 + x1^40*x2^33*x3^24*x4^3*z^20 - 2*x1^40*x2^32*x3^25*x4^3*z^20 + x1^41*x2^30*x3^26*x4^3*z^20 + x1^40*x2^31*x3^26*x4^3*z^20 + x1^50*x2^33*x3^13*x4^4*z^20 - 2*x1^49*x2^33*x3^14*x4^4*z^20 - x1^47*x2^35*x3^14*x4^4*z^20 + 2*x1^49*x2^32*x3^15*x4^4*z^20 + 2*x1^48*x2^33*x3^15*x4^4*z^20 - x1^45*x2^36*x3^15*x4^4*z^20 - 6*x1^48*x2^32*x3^16*x4^4*z^20 - x1^47*x2^33*x3^16*x4^4*z^20 - 2*x1^46*x2^34*x3^16*x4^4*z^20 - x1^43*x2^37*x3^16*x4^4*z^20 + 2*x1^48*x2^31*x3^17*x4^4*z^20 + 6*x1^47*x2^32*x3^17*x4^4*z^20 - x1^46*x2^33*x3^17*x4^4*z^20 + 3*x1^45*x2^34*x3^17*x4^4*z^20 - 2*x1^44*x2^35*x3^17*x4^4*z^20 - x1^43*x2^36*x3^17*x4^4*z^20 - x1^42*x2^37*x3^17*x4^4*z^20 - 6*x1^47*x2^31*x3^18*x4^4*z^20 - 2*x1^46*x2^32*x3^18*x4^4*z^20 - x1^45*x2^33*x3^18*x4^4*z^20 + 3*x1^43*x2^35*x3^18*x4^4*z^20 + 2*x1^42*x2^36*x3^18*x4^4*z^20 + x1^41*x2^37*x3^18*x4^4*z^20 - x1^40*x2^38*x3^18*x4^4*z^20 + 2*x1^47*x2^30*x3^19*x4^4*z^20 + 6*x1^46*x2^31*x3^19*x4^4*z^20 + 4*x1^44*x2^33*x3^19*x4^4*z^20 - 2*x1^43*x2^34*x3^19*x4^4*z^20 - 2*x1^41*x2^36*x3^19*x4^4*z^20 + x1^39*x2^38*x3^19*x4^4*z^20 - 6*x1^46*x2^30*x3^20*x4^4*z^20 - 2*x1^45*x2^31*x3^20*x4^4*z^20 - 2*x1^44*x2^32*x3^20*x4^4*z^20 - 2*x1^43*x2^33*x3^20*x4^4*z^20 + x1^42*x2^34*x3^20*x4^4*z^20 + 3*x1^41*x2^35*x3^20*x4^4*z^20 + 3*x1^40*x2^36*x3^20*x4^4*z^20 - x1^39*x2^37*x3^20*x4^4*z^20 + 2*x1^46*x2^29*x3^21*x4^4*z^20 + 6*x1^45*x2^30*x3^21*x4^4*z^20 + 4*x1^43*x2^32*x3^21*x4^4*z^20 - 3*x1^40*x2^35*x3^21*x4^4*z^20 + 2*x1^39*x2^36*x3^21*x4^4*z^20 - 6*x1^45*x2^29*x3^22*x4^4*z^20 - x1^44*x2^30*x3^22*x4^4*z^20 - 2*x1^42*x2^32*x3^22*x4^4*z^20 + x1^40*x2^34*x3^22*x4^4*z^20 + 2*x1^39*x2^35*x3^22*x4^4*z^20 - x1^38*x2^36*x3^22*x4^4*z^20 + 2*x1^45*x2^28*x3^23*x4^4*z^20 + 3*x1^44*x2^29*x3^23*x4^4*z^20 - x1^43*x2^30*x3^23*x4^4*z^20 + 4*x1^42*x2^31*x3^23*x4^4*z^20 + x1^41*x2^32*x3^23*x4^4*z^20 - x1^40*x2^33*x3^23*x4^4*z^20 - x1^39*x2^34*x3^23*x4^4*z^20 + 2*x1^38*x2^35*x3^23*x4^4*z^20 - 2*x1^44*x2^28*x3^24*x4^4*z^20 + x1^43*x2^29*x3^24*x4^4*z^20 - x1^41*x2^31*x3^24*x4^4*z^20 - x1^39*x2^33*x3^24*x4^4*z^20 + x1^38*x2^34*x3^24*x4^4*z^20 - x1^37*x2^35*x3^24*x4^4*z^20 + x1^43*x2^28*x3^25*x4^4*z^20 + 2*x1^41*x2^30*x3^25*x4^4*z^20 - x1^39*x2^32*x3^25*x4^4*z^20 + x1^41*x2^29*x3^26*x4^4*z^20 + x1^40*x2^30*x3^26*x4^4*z^20 + 2*x1^39*x2^31*x3^26*x4^4*z^20 - x1^39*x2^30*x3^27*x4^4*z^20 - x1^38*x2^31*x3^27*x4^4*z^20 + 2*x1^49*x2^33*x3^13*x4^5*z^20 - x1^48*x2^34*x3^13*x4^5*z^20 - 2*x1^49*x2^32*x3^14*x4^5*z^20 - 2*x1^48*x2^33*x3^14*x4^5*z^20 + x1^47*x2^34*x3^14*x4^5*z^20 + 5*x1^48*x2^32*x3^15*x4^5*z^20 - x1^47*x2^33*x3^15*x4^5*z^20 - x1^44*x2^36*x3^15*x4^5*z^20 - 2*x1^48*x2^31*x3^16*x4^5*z^20 - 5*x1^47*x2^32*x3^16*x4^5*z^20 + 3*x1^46*x2^33*x3^16*x4^5*z^20 - 3*x1^45*x2^34*x3^16*x4^5*z^20 + x1^44*x2^35*x3^16*x4^5*z^20 + 2*x1^43*x2^36*x3^16*x4^5*z^20 + x1^42*x2^37*x3^16*x4^5*z^20 + 6*x1^47*x2^31*x3^17*x4^5*z^20 + 2*x1^46*x2^32*x3^17*x4^5*z^20 - x1^45*x2^33*x3^17*x4^5*z^20 + x1^44*x2^34*x3^17*x4^5*z^20 - x1^43*x2^35*x3^17*x4^5*z^20 - x1^42*x2^36*x3^17*x4^5*z^20 - x1^41*x2^37*x3^17*x4^5*z^20 - 2*x1^47*x2^30*x3^18*x4^5*z^20 - 6*x1^46*x2^31*x3^18*x4^5*z^20 + 2*x1^45*x2^32*x3^18*x4^5*z^20 - 2*x1^44*x2^33*x3^18*x4^5*z^20 + 3*x1^43*x2^34*x3^18*x4^5*z^20 + 3*x1^41*x2^36*x3^18*x4^5*z^20 + x1^40*x2^37*x3^18*x4^5*z^20 + 6*x1^46*x2^30*x3^19*x4^5*z^20 + x1^45*x2^31*x3^19*x4^5*z^20 - 2*x1^42*x2^34*x3^19*x4^5*z^20 - 2*x1^41*x2^35*x3^19*x4^5*z^20 - 3*x1^40*x2^36*x3^19*x4^5*z^20 - 2*x1^46*x2^29*x3^20*x4^5*z^20 - 6*x1^45*x2^30*x3^20*x4^5*z^20 + 2*x1^44*x2^31*x3^20*x4^5*z^20 - 3*x1^43*x2^32*x3^20*x4^5*z^20 + 5*x1^42*x2^33*x3^20*x4^5*z^20 + x1^41*x2^34*x3^20*x4^5*z^20 + 5*x1^40*x2^35*x3^20*x4^5*z^20 + x1^38*x2^37*x3^20*x4^5*z^20 + 6*x1^45*x2^29*x3^21*x4^5*z^20 + x1^44*x2^30*x3^21*x4^5*z^20 + x1^42*x2^32*x3^21*x4^5*z^20 - 3*x1^41*x2^33*x3^21*x4^5*z^20 - 6*x1^39*x2^35*x3^21*x4^5*z^20 + 3*x1^38*x2^36*x3^21*x4^5*z^20 - x1^45*x2^28*x3^22*x4^5*z^20 - 6*x1^44*x2^29*x3^22*x4^5*z^20 + 2*x1^43*x2^30*x3^22*x4^5*z^20 - 4*x1^42*x2^31*x3^22*x4^5*z^20 + x1^41*x2^32*x3^22*x4^5*z^20 + 4*x1^39*x2^34*x3^22*x4^5*z^20 - x1^37*x2^36*x3^22*x4^5*z^20 + 3*x1^44*x2^28*x3^23*x4^5*z^20 + 3*x1^43*x2^29*x3^23*x4^5*z^20 + x1^42*x2^30*x3^23*x4^5*z^20 + x1^41*x2^31*x3^23*x4^5*z^20 - x1^40*x2^32*x3^23*x4^5*z^20 - x1^39*x2^33*x3^23*x4^5*z^20 - 3*x1^38*x2^34*x3^23*x4^5*z^20 + x1^37*x2^35*x3^23*x4^5*z^20 + x1^36*x2^36*x3^23*x4^5*z^20 - 3*x1^43*x2^28*x3^24*x4^5*z^20 - x1^41*x2^30*x3^24*x4^5*z^20 + x1^39*x2^32*x3^24*x4^5*z^20 + 2*x1^38*x2^33*x3^24*x4^5*z^20 - x1^37*x2^34*x3^24*x4^5*z^20 + x1^42*x2^28*x3^25*x4^5*z^20 - x1^38*x2^32*x3^25*x4^5*z^20 - 2*x1^37*x2^33*x3^25*x4^5*z^20 + 2*x1^36*x2^34*x3^25*x4^5*z^20 - x1^41*x2^28*x3^26*x4^5*z^20 + x1^39*x2^30*x3^26*x4^5*z^20 + 2*x1^38*x2^31*x3^26*x4^5*z^20 - x1^37*x2^32*x3^26*x4^5*z^20 - x1^36*x2^33*x3^26*x4^5*z^20 - x1^38*x2^30*x3^27*x4^5*z^20 + x1^37*x2^30*x3^28*x4^5*z^20 + x1^48*x2^33*x3^13*x4^6*z^20 - x1^47*x2^34*x3^13*x4^6*z^20 - 2*x1^48*x2^32*x3^14*x4^6*z^20 + x1^47*x2^33*x3^14*x4^6*z^20 + 2*x1^46*x2^34*x3^14*x4^6*z^20 - x1^45*x2^35*x3^14*x4^6*z^20 + 3*x1^47*x2^32*x3^15*x4^6*z^20 - 2*x1^47*x2^31*x3^16*x4^6*z^20 - x1^44*x2^34*x3^16*x4^6*z^20 - x1^42*x2^36*x3^16*x4^6*z^20 + x1^47*x2^30*x3^17*x4^6*z^20 + 2*x1^46*x2^31*x3^17*x4^6*z^20 - 4*x1^45*x2^32*x3^17*x4^6*z^20 + 3*x1^44*x2^33*x3^17*x4^6*z^20 - x1^42*x2^35*x3^17*x4^6*z^20 - x1^41*x2^36*x3^17*x4^6*z^20 - 2*x1^46*x2^30*x3^18*x4^6*z^20 + 3*x1^44*x2^32*x3^18*x4^6*z^20 + 4*x1^42*x2^34*x3^18*x4^6*z^20 + 3*x1^41*x2^35*x3^18*x4^6*z^20 + x1^40*x2^36*x3^18*x4^6*z^20 - x1^39*x2^37*x3^18*x4^6*z^20 + x1^46*x2^29*x3^19*x4^6*z^20 + 2*x1^45*x2^30*x3^19*x4^6*z^20 - 4*x1^44*x2^31*x3^19*x4^6*z^20 - x1^43*x2^32*x3^19*x4^6*z^20 - 4*x1^42*x2^33*x3^19*x4^6*z^20 + x1^41*x2^34*x3^19*x4^6*z^20 + 2*x1^39*x2^36*x3^19*x4^6*z^20 + x1^38*x2^37*x3^19*x4^6*z^20 - 2*x1^45*x2^29*x3^20*x4^6*z^20 + 2*x1^44*x2^30*x3^20*x4^6*z^20 + 4*x1^43*x2^31*x3^20*x4^6*z^20 + 2*x1^41*x2^33*x3^20*x4^6*z^20 - 3*x1^40*x2^34*x3^20*x4^6*z^20 + x1^39*x2^35*x3^20*x4^6*z^20 - 3*x1^38*x2^36*x3^20*x4^6*z^20 + 2*x1^44*x2^29*x3^21*x4^6*z^20 - 4*x1^43*x2^30*x3^21*x4^6*z^20 + x1^42*x2^31*x3^21*x4^6*z^20 - 3*x1^41*x2^32*x3^21*x4^6*z^20 - 2*x1^40*x2^33*x3^21*x4^6*z^20 - 2*x1^39*x2^34*x3^21*x4^6*z^20 + x1^37*x2^36*x3^21*x4^6*z^20 - x1^44*x2^28*x3^22*x4^6*z^20 + 3*x1^42*x2^30*x3^22*x4^6*z^20 + x1^41*x2^31*x3^22*x4^6*z^20 + 4*x1^40*x2^32*x3^22*x4^6*z^20 - 2*x1^39*x2^33*x3^22*x4^6*z^20 + 2*x1^38*x2^34*x3^22*x4^6*z^20 - 4*x1^37*x2^35*x3^22*x4^6*z^20 + x1^44*x2^27*x3^23*x4^6*z^20 + x1^43*x2^28*x3^23*x4^6*z^20 - 3*x1^42*x2^29*x3^23*x4^6*z^20 - x1^40*x2^31*x3^23*x4^6*z^20 - 3*x1^38*x2^33*x3^23*x4^6*z^20 - x1^37*x2^34*x3^23*x4^6*z^20 + x1^36*x2^35*x3^23*x4^6*z^20 - x1^43*x2^27*x3^24*x4^6*z^20 + x1^41*x2^29*x3^24*x4^6*z^20 + 3*x1^39*x2^31*x3^24*x4^6*z^20 + x1^38*x2^32*x3^24*x4^6*z^20 + 2*x1^37*x2^33*x3^24*x4^6*z^20 - 3*x1^36*x2^34*x3^24*x4^6*z^20 - x1^35*x2^35*x3^24*x4^6*z^20 + x1^42*x2^27*x3^25*x4^6*z^20 - 2*x1^41*x2^28*x3^25*x4^6*z^20 - 2*x1^40*x2^29*x3^25*x4^6*z^20 - 2*x1^39*x2^30*x3^25*x4^6*z^20 - x1^41*x2^27*x3^26*x4^6*z^20 + x1^38*x2^30*x3^26*x4^6*z^20 - x1^37*x2^31*x3^26*x4^6*z^20 + x1^36*x2^32*x3^26*x4^6*z^20 - x1^35*x2^33*x3^26*x4^6*z^20 + x1^39*x2^28*x3^27*x4^6*z^20 - 2*x1^37*x2^30*x3^27*x4^6*z^20 + x1^36*x2^31*x3^27*x4^6*z^20 + x1^35*x2^32*x3^27*x4^6*z^20 - x1^48*x2^33*x3^12*x4^7*z^20 - x1^45*x2^36*x3^12*x4^7*z^20 - 2*x1^48*x2^32*x3^13*x4^7*z^20 + x1^47*x2^33*x3^13*x4^7*z^20 + 3*x1^46*x2^34*x3^13*x4^7*z^20 - x1^45*x2^35*x3^13*x4^7*z^20 + x1^44*x2^36*x3^13*x4^7*z^20 - x1^46*x2^33*x3^14*x4^7*z^20 + x1^45*x2^34*x3^14*x4^7*z^20 - 2*x1^47*x2^31*x3^15*x4^7*z^20 - x1^46*x2^32*x3^15*x4^7*z^20 + x1^45*x2^33*x3^15*x4^7*z^20 + x1^44*x2^34*x3^15*x4^7*z^20 + 3*x1^43*x2^35*x3^15*x4^7*z^20 - x1^42*x2^36*x3^15*x4^7*z^20 + x1^45*x2^32*x3^16*x4^7*z^20 - x1^44*x2^33*x3^16*x4^7*z^20 + x1^43*x2^34*x3^16*x4^7*z^20 + x1^41*x2^36*x3^16*x4^7*z^20 + x1^40*x2^37*x3^16*x4^7*z^20 - x1^44*x2^32*x3^17*x4^7*z^20 + x1^42*x2^34*x3^17*x4^7*z^20 - x1^41*x2^35*x3^17*x4^7*z^20 - x1^40*x2^36*x3^17*x4^7*z^20 - 2*x1^39*x2^37*x3^17*x4^7*z^20 - x1^42*x2^33*x3^18*x4^7*z^20 + 2*x1^39*x2^36*x3^18*x4^7*z^20 + 2*x1^38*x2^37*x3^18*x4^7*z^20 - 2*x1^40*x2^34*x3^19*x4^7*z^20 + x1^39*x2^35*x3^19*x4^7*z^20 - 2*x1^38*x2^36*x3^19*x4^7*z^20 - x1^37*x2^37*x3^19*x4^7*z^20 - x1^38*x2^35*x3^20*x4^7*z^20 + 2*x1^37*x2^36*x3^20*x4^7*z^20 - x1^43*x2^29*x3^21*x4^7*z^20 - x1^36*x2^36*x3^21*x4^7*z^20 + 2*x1^42*x2^29*x3^22*x4^7*z^20 - x1^41*x2^30*x3^22*x4^7*z^20 + x1^40*x2^31*x3^22*x4^7*z^20 + x1^39*x2^32*x3^22*x4^7*z^20 + x1^38*x2^33*x3^22*x4^7*z^20 + x1^42*x2^28*x3^23*x4^7*z^20 - x1^41*x2^29*x3^23*x4^7*z^20 - x1^40*x2^30*x3^23*x4^7*z^20 + 2*x1^38*x2^32*x3^23*x4^7*z^20 - x1^37*x2^33*x3^23*x4^7*z^20 + x1^36*x2^34*x3^23*x4^7*z^20 - x1^35*x2^35*x3^23*x4^7*z^20 + x1^41*x2^28*x3^24*x4^7*z^20 + x1^40*x2^29*x3^24*x4^7*z^20 + 2*x1^39*x2^30*x3^24*x4^7*z^20 + 2*x1^36*x2^33*x3^24*x4^7*z^20 + x1^39*x2^29*x3^25*x4^7*z^20 - x1^36*x2^32*x3^25*x4^7*z^20 - x1^40*x2^27*x3^26*x4^7*z^20 - x1^39*x2^28*x3^26*x4^7*z^20 + x1^35*x2^32*x3^26*x4^7*z^20 - x1^38*x2^28*x3^27*x4^7*z^20 - x1^35*x2^31*x3^27*x4^7*z^20 - x1^37*x2^28*x3^28*x4^7*z^20 + x1^36*x2^29*x3^28*x4^7*z^20 - x1^34*x2^31*x3^28*x4^7*z^20 + x1^48*x2^32*x3^12*x4^8*z^20 - x1^46*x2^34*x3^12*x4^8*z^20 + x1^45*x2^35*x3^12*x4^8*z^20 - x1^44*x2^36*x3^12*x4^8*z^20 + x1^45*x2^34*x3^13*x4^8*z^20 - 2*x1^42*x2^37*x3^13*x4^8*z^20 - x1^47*x2^31*x3^14*x4^8*z^20 - x1^46*x2^32*x3^14*x4^8*z^20 - 3*x1^45*x2^33*x3^14*x4^8*z^20 - x1^44*x2^34*x3^14*x4^8*z^20 + 2*x1^42*x2^36*x3^14*x4^8*z^20 + 2*x1^41*x2^37*x3^14*x4^8*z^20 + x1^47*x2^30*x3^15*x4^8*z^20 + x1^46*x2^31*x3^15*x4^8*z^20 + 2*x1^45*x2^32*x3^15*x4^8*z^20 - x1^42*x2^35*x3^15*x4^8*z^20 - 2*x1^40*x2^37*x3^15*x4^8*z^20 - 2*x1^45*x2^31*x3^16*x4^8*z^20 - 3*x1^44*x2^32*x3^16*x4^8*z^20 + x1^43*x2^33*x3^16*x4^8*z^20 - 3*x1^42*x2^34*x3^16*x4^8*z^20 + 2*x1^41*x2^35*x3^16*x4^8*z^20 + 4*x1^39*x2^37*x3^16*x4^8*z^20 + x1^45*x2^30*x3^17*x4^8*z^20 + 2*x1^44*x2^31*x3^17*x4^8*z^20 + 2*x1^43*x2^32*x3^17*x4^8*z^20 + x1^42*x2^33*x3^17*x4^8*z^20 - x1^41*x2^34*x3^17*x4^8*z^20 - x1^40*x2^35*x3^17*x4^8*z^20 - x1^39*x2^36*x3^17*x4^8*z^20 - 4*x1^38*x2^37*x3^17*x4^8*z^20 - 2*x1^45*x2^29*x3^18*x4^8*z^20 - 3*x1^44*x2^30*x3^18*x4^8*z^20 - 5*x1^43*x2^31*x3^18*x4^8*z^20 - 2*x1^41*x2^33*x3^18*x4^8*z^20 + 4*x1^40*x2^34*x3^18*x4^8*z^20 + 4*x1^38*x2^36*x3^18*x4^8*z^20 + 2*x1^37*x2^37*x3^18*x4^8*z^20 + x1^45*x2^28*x3^19*x4^8*z^20 + 2*x1^44*x2^29*x3^19*x4^8*z^20 + 4*x1^43*x2^30*x3^19*x4^8*z^20 + 2*x1^42*x2^31*x3^19*x4^8*z^20 - 3*x1^39*x2^34*x3^19*x4^8*z^20 - 2*x1^38*x2^35*x3^19*x4^8*z^20 - 4*x1^37*x2^36*x3^19*x4^8*z^20 - 2*x1^44*x2^28*x3^20*x4^8*z^20 - 5*x1^42*x2^30*x3^20*x4^8*z^20 - 2*x1^40*x2^32*x3^20*x4^8*z^20 + 4*x1^39*x2^33*x3^20*x4^8*z^20 + 2*x1^38*x2^34*x3^20*x4^8*z^20 + 4*x1^37*x2^35*x3^20*x4^8*z^20 + x1^36*x2^36*x3^20*x4^8*z^20 + 2*x1^43*x2^28*x3^21*x4^8*z^20 + x1^42*x2^29*x3^21*x4^8*z^20 + 3*x1^41*x2^30*x3^21*x4^8*z^20 - x1^40*x2^31*x3^21*x4^8*z^20 - 3*x1^38*x2^33*x3^21*x4^8*z^20 - x1^37*x2^34*x3^21*x4^8*z^20 - 4*x1^36*x2^35*x3^21*x4^8*z^20 - x1^43*x2^27*x3^22*x4^8*z^20 - x1^41*x2^29*x3^22*x4^8*z^20 - x1^39*x2^31*x3^22*x4^8*z^20 + 2*x1^38*x2^32*x3^22*x4^8*z^20 + 2*x1^37*x2^33*x3^22*x4^8*z^20 + 4*x1^36*x2^34*x3^22*x4^8*z^20 + 2*x1^35*x2^35*x3^22*x4^8*z^20 + x1^42*x2^27*x3^23*x4^8*z^20 - x1^41*x2^28*x3^23*x4^8*z^20 + x1^40*x2^29*x3^23*x4^8*z^20 + x1^39*x2^30*x3^23*x4^8*z^20 - x1^38*x2^31*x3^23*x4^8*z^20 - 2*x1^37*x2^32*x3^23*x4^8*z^20 - 2*x1^36*x2^33*x3^23*x4^8*z^20 - 2*x1^35*x2^34*x3^23*x4^8*z^20 - x1^42*x2^26*x3^24*x4^8*z^20 - x1^41*x2^27*x3^24*x4^8*z^20 + x1^39*x2^29*x3^24*x4^8*z^20 + 3*x1^37*x2^31*x3^24*x4^8*z^20 + 4*x1^35*x2^33*x3^24*x4^8*z^20 + 2*x1^34*x2^34*x3^24*x4^8*z^20 + x1^41*x2^26*x3^25*x4^8*z^20 - x1^39*x2^28*x3^25*x4^8*z^20 - 3*x1^38*x2^29*x3^25*x4^8*z^20 - x1^37*x2^30*x3^25*x4^8*z^20 - 3*x1^36*x2^31*x3^25*x4^8*z^20 - 2*x1^35*x2^32*x3^25*x4^8*z^20 - 2*x1^34*x2^33*x3^25*x4^8*z^20 + x1^38*x2^28*x3^26*x4^8*z^20 + x1^37*x2^29*x3^26*x4^8*z^20 + 2*x1^36*x2^30*x3^26*x4^8*z^20 + x1^35*x2^31*x3^26*x4^8*z^20 + x1^33*x2^33*x3^26*x4^8*z^20 + x1^38*x2^27*x3^27*x4^8*z^20 - x1^37*x2^28*x3^27*x4^8*z^20 - x1^36*x2^29*x3^27*x4^8*z^20 - x1^34*x2^31*x3^27*x4^8*z^20 + x1^33*x2^31*x3^28*x4^8*z^20 - x1^47*x2^32*x3^12*x4^9*z^20 + x1^46*x2^33*x3^12*x4^9*z^20 + x1^43*x2^36*x3^12*x4^9*z^20 + x1^47*x2^31*x3^13*x4^9*z^20 + x1^46*x2^32*x3^13*x4^9*z^20 + x1^45*x2^33*x3^13*x4^9*z^20 + x1^44*x2^34*x3^13*x4^9*z^20 + x1^43*x2^35*x3^13*x4^9*z^20 - x1^46*x2^31*x3^14*x4^9*z^20 + x1^43*x2^34*x3^14*x4^9*z^20 + x1^41*x2^36*x3^14*x4^9*z^20 + x1^40*x2^37*x3^14*x4^9*z^20 + 2*x1^46*x2^30*x3^15*x4^9*z^20 + x1^45*x2^31*x3^15*x4^9*z^20 + x1^44*x2^32*x3^15*x4^9*z^20 + 2*x1^42*x2^34*x3^15*x4^9*z^20 - x1^39*x2^37*x3^15*x4^9*z^20 - x1^46*x2^29*x3^16*x4^9*z^20 - 2*x1^45*x2^30*x3^16*x4^9*z^20 - 2*x1^44*x2^31*x3^16*x4^9*z^20 - 2*x1^43*x2^32*x3^16*x4^9*z^20 - x1^41*x2^34*x3^16*x4^9*z^20 - x1^39*x2^36*x3^16*x4^9*z^20 + x1^38*x2^37*x3^16*x4^9*z^20 + 4*x1^45*x2^29*x3^17*x4^9*z^20 + 3*x1^44*x2^30*x3^17*x4^9*z^20 + 4*x1^43*x2^31*x3^17*x4^9*z^20 - x1^40*x2^34*x3^17*x4^9*z^20 - x1^38*x2^36*x3^17*x4^9*z^20 - x1^37*x2^37*x3^17*x4^9*z^20 - x1^45*x2^28*x3^18*x4^9*z^20 - 3*x1^44*x2^29*x3^18*x4^9*z^20 - 2*x1^42*x2^31*x3^18*x4^9*z^20 + 2*x1^41*x2^32*x3^18*x4^9*z^20 + x1^40*x2^33*x3^18*x4^9*z^20 + 3*x1^39*x2^34*x3^18*x4^9*z^20 + 2*x1^38*x2^35*x3^18*x4^9*z^20 + x1^37*x2^36*x3^18*x4^9*z^20 + 4*x1^44*x2^28*x3^19*x4^9*z^20 + 3*x1^43*x2^29*x3^19*x4^9*z^20 + 4*x1^42*x2^30*x3^19*x4^9*z^20 - x1^40*x2^32*x3^19*x4^9*z^20 - 4*x1^39*x2^33*x3^19*x4^9*z^20 - 2*x1^38*x2^34*x3^19*x4^9*z^20 - 2*x1^37*x2^35*x3^19*x4^9*z^20 - 2*x1^44*x2^27*x3^20*x4^9*z^20 - 4*x1^43*x2^28*x3^20*x4^9*z^20 - 2*x1^42*x2^29*x3^20*x4^9*z^20 - 2*x1^41*x2^30*x3^20*x4^9*z^20 + 3*x1^40*x2^31*x3^20*x4^9*z^20 + 4*x1^38*x2^33*x3^20*x4^9*z^20 + x1^37*x2^34*x3^20*x4^9*z^20 + 2*x1^36*x2^35*x3^20*x4^9*z^20 + 5*x1^43*x2^27*x3^21*x4^9*z^20 + 3*x1^41*x2^29*x3^21*x4^9*z^20 + x1^40*x2^30*x3^21*x4^9*z^20 + x1^39*x2^31*x3^21*x4^9*z^20 - 2*x1^38*x2^32*x3^21*x4^9*z^20 - 4*x1^37*x2^33*x3^21*x4^9*z^20 - 2*x1^36*x2^34*x3^21*x4^9*z^20 - x1^35*x2^35*x3^21*x4^9*z^20 - x1^43*x2^26*x3^22*x4^9*z^20 - 5*x1^42*x2^27*x3^22*x4^9*z^20 - 5*x1^40*x2^29*x3^22*x4^9*z^20 + x1^39*x2^30*x3^22*x4^9*z^20 + 5*x1^37*x2^32*x3^22*x4^9*z^20 + 2*x1^36*x2^33*x3^22*x4^9*z^20 + 2*x1^35*x2^34*x3^22*x4^9*z^20 + 3*x1^42*x2^26*x3^23*x4^9*z^20 + 3*x1^41*x2^27*x3^23*x4^9*z^20 + 2*x1^40*x2^28*x3^23*x4^9*z^20 + x1^39*x2^29*x3^23*x4^9*z^20 - x1^38*x2^30*x3^23*x4^9*z^20 - 2*x1^37*x2^31*x3^23*x4^9*z^20 - 4*x1^36*x2^32*x3^23*x4^9*z^20 - 2*x1^35*x2^33*x3^23*x4^9*z^20 - x1^34*x2^34*x3^23*x4^9*z^20 - 3*x1^41*x2^26*x3^24*x4^9*z^20 - 2*x1^40*x2^27*x3^24*x4^9*z^20 - 2*x1^39*x2^28*x3^24*x4^9*z^20 + 2*x1^38*x2^29*x3^24*x4^9*z^20 - x1^37*x2^30*x3^24*x4^9*z^20 + 3*x1^36*x2^31*x3^24*x4^9*z^20 + 2*x1^35*x2^32*x3^24*x4^9*z^20 + x1^40*x2^26*x3^25*x4^9*z^20 + 2*x1^39*x2^27*x3^25*x4^9*z^20 - x1^37*x2^29*x3^25*x4^9*z^20 - 2*x1^36*x2^30*x3^25*x4^9*z^20 - x1^35*x2^31*x3^25*x4^9*z^20 - x1^34*x2^32*x3^25*x4^9*z^20 - x1^38*x2^27*x3^26*x4^9*z^20 + x1^37*x2^28*x3^26*x4^9*z^20 - x1^36*x2^29*x3^26*x4^9*z^20 + x1^35*x2^30*x3^26*x4^9*z^20 - x1^34*x2^31*x3^26*x4^9*z^20 + x1^36*x2^28*x3^27*x4^9*z^20 - x1^35*x2^29*x3^27*x4^9*z^20 - 2*x1^34*x2^30*x3^27*x4^9*z^20 + x1^33*x2^31*x3^27*x4^9*z^20 - x1^32*x2^32*x3^27*x4^9*z^20 - x1^35*x2^28*x3^28*x4^9*z^20 + x1^34*x2^29*x3^28*x4^9*z^20 - x1^32*x2^31*x3^28*x4^9*z^20 - x1^44*x2^34*x3^12*x4^10*z^20 - x1^45*x2^32*x3^13*x4^10*z^20 - x1^44*x2^33*x3^13*x4^10*z^20 + x1^43*x2^34*x3^13*x4^10*z^20 - x1^42*x2^35*x3^13*x4^10*z^20 - x1^41*x2^36*x3^13*x4^10*z^20 - x1^46*x2^30*x3^14*x4^10*z^20 - x1^44*x2^32*x3^14*x4^10*z^20 - x1^43*x2^33*x3^14*x4^10*z^20 + x1^42*x2^34*x3^14*x4^10*z^20 + x1^41*x2^35*x3^14*x4^10*z^20 + x1^40*x2^36*x3^14*x4^10*z^20 - x1^45*x2^30*x3^15*x4^10*z^20 + x1^44*x2^31*x3^15*x4^10*z^20 + x1^43*x2^32*x3^15*x4^10*z^20 - x1^39*x2^36*x3^15*x4^10*z^20 - 2*x1^43*x2^31*x3^16*x4^10*z^20 + x1^42*x2^32*x3^16*x4^10*z^20 + x1^41*x2^33*x3^16*x4^10*z^20 + 2*x1^38*x2^36*x3^16*x4^10*z^20 - x1^44*x2^29*x3^17*x4^10*z^20 + 2*x1^43*x2^30*x3^17*x4^10*z^20 + x1^41*x2^32*x3^17*x4^10*z^20 - x1^40*x2^33*x3^17*x4^10*z^20 + x1^39*x2^34*x3^17*x4^10*z^20 + x1^38*x2^35*x3^17*x4^10*z^20 - x1^37*x2^36*x3^17*x4^10*z^20 - x1^44*x2^28*x3^18*x4^10*z^20 - x1^43*x2^29*x3^18*x4^10*z^20 - x1^42*x2^30*x3^18*x4^10*z^20 + x1^39*x2^33*x3^18*x4^10*z^20 - x1^38*x2^34*x3^18*x4^10*z^20 + x1^36*x2^36*x3^18*x4^10*z^20 + x1^44*x2^27*x3^19*x4^10*z^20 + x1^42*x2^29*x3^19*x4^10*z^20 - x1^41*x2^30*x3^19*x4^10*z^20 + x1^40*x2^31*x3^19*x4^10*z^20 + x1^39*x2^32*x3^19*x4^10*z^20 - 2*x1^43*x2^27*x3^20*x4^10*z^20 - x1^41*x2^29*x3^20*x4^10*z^20 + 2*x1^42*x2^27*x3^21*x4^10*z^20 - x1^39*x2^30*x3^21*x4^10*z^20 - 2*x1^42*x2^26*x3^22*x4^10*z^20 - x1^41*x2^27*x3^22*x4^10*z^20 - x1^40*x2^28*x3^22*x4^10*z^20 - x1^37*x2^31*x3^22*x4^10*z^20 + 2*x1^41*x2^26*x3^23*x4^10*z^20 + 2*x1^39*x2^28*x3^23*x4^10*z^20 - x1^37*x2^30*x3^23*x4^10*z^20 - x1^36*x2^31*x3^23*x4^10*z^20 + x1^37*x2^29*x3^24*x4^10*z^20 + 2*x1^36*x2^30*x3^24*x4^10*z^20 + x1^35*x2^31*x3^24*x4^10*z^20 - x1^34*x2^32*x3^24*x4^10*z^20 - x1^37*x2^28*x3^25*x4^10*z^20 + x1^36*x2^29*x3^25*x4^10*z^20 - x1^35*x2^30*x3^25*x4^10*z^20 + x1^33*x2^32*x3^25*x4^10*z^20 + x1^37*x2^27*x3^26*x4^10*z^20 + 2*x1^35*x2^29*x3^26*x4^10*z^20 + 3*x1^34*x2^30*x3^26*x4^10*z^20 - x1^32*x2^32*x3^26*x4^10*z^20 - x1^36*x2^27*x3^27*x4^10*z^20 - x1^34*x2^29*x3^27*x4^10*z^20 + x1^33*x2^29*x3^28*x4^10*z^20 + 2*x1^43*x2^33*x3^13*x4^11*z^20 + x1^42*x2^34*x3^13*x4^11*z^20 + x1^45*x2^30*x3^14*x4^11*z^20 - x1^44*x2^31*x3^14*x4^11*z^20 + x1^43*x2^32*x3^14*x4^11*z^20 - x1^42*x2^33*x3^14*x4^11*z^20 - x1^41*x2^34*x3^14*x4^11*z^20 + x1^45*x2^29*x3^15*x4^11*z^20 - x1^44*x2^30*x3^15*x4^11*z^20 + 2*x1^42*x2^32*x3^15*x4^11*z^20 - 2*x1^40*x2^34*x3^15*x4^11*z^20 + x1^39*x2^35*x3^15*x4^11*z^20 - x1^38*x2^36*x3^15*x4^11*z^20 + 2*x1^44*x2^29*x3^16*x4^11*z^20 - 2*x1^43*x2^30*x3^16*x4^11*z^20 - x1^42*x2^31*x3^16*x4^11*z^20 - 2*x1^41*x2^32*x3^16*x4^11*z^20 + x1^40*x2^33*x3^16*x4^11*z^20 - 3*x1^39*x2^34*x3^16*x4^11*z^20 - 3*x1^38*x2^35*x3^16*x4^11*z^20 - x1^44*x2^28*x3^17*x4^11*z^20 - 2*x1^43*x2^29*x3^17*x4^11*z^20 + x1^42*x2^30*x3^17*x4^11*z^20 + x1^41*x2^31*x3^17*x4^11*z^20 + x1^40*x2^32*x3^17*x4^11*z^20 + x1^39*x2^33*x3^17*x4^11*z^20 + 4*x1^38*x2^34*x3^17*x4^11*z^20 - 4*x1^37*x2^35*x3^17*x4^11*z^20 + x1^44*x2^27*x3^18*x4^11*z^20 + 3*x1^43*x2^28*x3^18*x4^11*z^20 - 2*x1^42*x2^29*x3^18*x4^11*z^20 - 3*x1^41*x2^30*x3^18*x4^11*z^20 - 5*x1^40*x2^31*x3^18*x4^11*z^20 + x1^39*x2^32*x3^18*x4^11*z^20 - 3*x1^38*x2^33*x3^18*x4^11*z^20 + 2*x1^36*x2^35*x3^18*x4^11*z^20 - x1^43*x2^27*x3^19*x4^11*z^20 - x1^42*x2^28*x3^19*x4^11*z^20 + 2*x1^41*x2^29*x3^19*x4^11*z^20 + x1^39*x2^31*x3^19*x4^11*z^20 + 4*x1^37*x2^33*x3^19*x4^11*z^20 - 2*x1^36*x2^34*x3^19*x4^11*z^20 - x1^35*x2^35*x3^19*x4^11*z^20 + x1^42*x2^27*x3^20*x4^11*z^20 - 3*x1^41*x2^28*x3^20*x4^11*z^20 + 2*x1^40*x2^29*x3^20*x4^11*z^20 - 5*x1^39*x2^30*x3^20*x4^11*z^20 - x1^38*x2^31*x3^20*x4^11*z^20 - 3*x1^37*x2^32*x3^20*x4^11*z^20 + 2*x1^35*x2^34*x3^20*x4^11*z^20 - x1^41*x2^27*x3^21*x4^11*z^20 + x1^40*x2^28*x3^21*x4^11*z^20 + x1^38*x2^30*x3^21*x4^11*z^20 - x1^37*x2^31*x3^21*x4^11*z^20 + 4*x1^36*x2^32*x3^21*x4^11*z^20 - 2*x1^35*x2^33*x3^21*x4^11*z^20 + x1^40*x2^27*x3^22*x4^11*z^20 + x1^39*x2^28*x3^22*x4^11*z^20 - x1^38*x2^29*x3^22*x4^11*z^20 + 2*x1^37*x2^30*x3^22*x4^11*z^20 - 3*x1^36*x2^31*x3^22*x4^11*z^20 - 2*x1^35*x2^32*x3^22*x4^11*z^20 + 2*x1^34*x2^33*x3^22*x4^11*z^20 - x1^39*x2^27*x3^23*x4^11*z^20 + 3*x1^37*x2^29*x3^23*x4^11*z^20 + 3*x1^35*x2^31*x3^23*x4^11*z^20 - 2*x1^34*x2^32*x3^23*x4^11*z^20 - x1^33*x2^33*x3^23*x4^11*z^20 - x1^37*x2^28*x3^24*x4^11*z^20 + x1^34*x2^31*x3^24*x4^11*z^20 + 2*x1^33*x2^32*x3^24*x4^11*z^20 - 2*x1^35*x2^29*x3^25*x4^11*z^20 + x1^34*x2^30*x3^25*x4^11*z^20 - 2*x1^33*x2^31*x3^25*x4^11*z^20 - x1^32*x2^32*x3^25*x4^11*z^20 + 2*x1^32*x2^31*x3^26*x4^11*z^20 + x1^32*x2^30*x3^27*x4^11*z^20 - x1^43*x2^33*x3^12*x4^12*z^20 + x1^42*x2^33*x3^13*x4^12*z^20 + x1^41*x2^34*x3^13*x4^12*z^20 - x1^42*x2^32*x3^14*x4^12*z^20 - x1^41*x2^33*x3^14*x4^12*z^20 - x1^44*x2^29*x3^15*x4^12*z^20 + x1^43*x2^30*x3^15*x4^12*z^20 - x1^42*x2^31*x3^15*x4^12*z^20 + x1^39*x2^34*x3^15*x4^12*z^20 - x1^37*x2^36*x3^15*x4^12*z^20 + 2*x1^43*x2^29*x3^16*x4^12*z^20 - x1^42*x2^30*x3^16*x4^12*z^20 - x1^41*x2^31*x3^16*x4^12*z^20 + 3*x1^39*x2^33*x3^16*x4^12*z^20 - x1^38*x2^34*x3^16*x4^12*z^20 + 3*x1^37*x2^35*x3^16*x4^12*z^20 - x1^43*x2^28*x3^17*x4^12*z^20 + 2*x1^42*x2^29*x3^17*x4^12*z^20 + x1^41*x2^30*x3^17*x4^12*z^20 + 2*x1^40*x2^31*x3^17*x4^12*z^20 - 2*x1^39*x2^32*x3^17*x4^12*z^20 - 5*x1^36*x2^35*x3^17*x4^12*z^20 + x1^43*x2^27*x3^18*x4^12*z^20 + x1^42*x2^28*x3^18*x4^12*z^20 - 3*x1^41*x2^29*x3^18*x4^12*z^20 - x1^40*x2^30*x3^18*x4^12*z^20 + 4*x1^38*x2^32*x3^18*x4^12*z^20 - 2*x1^37*x2^33*x3^18*x4^12*z^20 + 6*x1^36*x2^34*x3^18*x4^12*z^20 + 2*x1^35*x2^35*x3^18*x4^12*z^20 - x1^43*x2^26*x3^19*x4^12*z^20 + 2*x1^41*x2^28*x3^19*x4^12*z^20 + x1^40*x2^29*x3^19*x4^12*z^20 + 3*x1^39*x2^30*x3^19*x4^12*z^20 - 2*x1^38*x2^31*x3^19*x4^12*z^20 - 2*x1^36*x2^33*x3^19*x4^12*z^20 - 6*x1^35*x2^34*x3^19*x4^12*z^20 - 3*x1^40*x2^28*x3^20*x4^12*z^20 - x1^39*x2^29*x3^20*x4^12*z^20 - 3*x1^38*x2^30*x3^20*x4^12*z^20 + 4*x1^37*x2^31*x3^20*x4^12*z^20 - 2*x1^36*x2^32*x3^20*x4^12*z^20 + 6*x1^35*x2^33*x3^20*x4^12*z^20 + 2*x1^34*x2^34*x3^20*x4^12*z^20 + x1^41*x2^26*x3^21*x4^12*z^20 + 4*x1^40*x2^27*x3^21*x4^12*z^20 + 2*x1^38*x2^29*x3^21*x4^12*z^20 - x1^35*x2^32*x3^21*x4^12*z^20 - 6*x1^34*x2^33*x3^21*x4^12*z^20 - 3*x1^39*x2^27*x3^22*x4^12*z^20 - x1^38*x2^28*x3^22*x4^12*z^20 - 3*x1^37*x2^29*x3^22*x4^12*z^20 + 3*x1^36*x2^30*x3^22*x4^12*z^20 - 2*x1^35*x2^31*x3^22*x4^12*z^20 + 6*x1^34*x2^32*x3^22*x4^12*z^20 + 2*x1^33*x2^33*x3^22*x4^12*z^20 + 3*x1^38*x2^27*x3^23*x4^12*z^20 + 2*x1^37*x2^28*x3^23*x4^12*z^20 - x1^36*x2^29*x3^23*x4^12*z^20 - x1^35*x2^30*x3^23*x4^12*z^20 - 6*x1^33*x2^32*x3^23*x4^12*z^20 + 4*x1^35*x2^29*x3^24*x4^12*z^20 - x1^34*x2^30*x3^24*x4^12*z^20 + 4*x1^33*x2^31*x3^24*x4^12*z^20 + 2*x1^32*x2^32*x3^24*x4^12*z^20 + x1^35*x2^28*x3^25*x4^12*z^20 - 4*x1^32*x2^31*x3^25*x4^12*z^20 - x1^33*x2^29*x3^26*x4^12*z^20 + x1^32*x2^30*x3^26*x4^12*z^20 + x1^31*x2^31*x3^26*x4^12*z^20 - x1^31*x2^30*x3^27*x4^12*z^20 - 2*x1^41*x2^32*x3^14*x4^13*z^20 - x1^40*x2^33*x3^14*x4^13*z^20 - x1^39*x2^34*x3^14*x4^13*z^20 + 2*x1^38*x2^34*x3^15*x4^13*z^20 - 2*x1^40*x2^31*x3^16*x4^13*z^20 + x1^39*x2^32*x3^16*x4^13*z^20 + x1^38*x2^33*x3^16*x4^13*z^20 + 3*x1^36*x2^35*x3^16*x4^13*z^20 - x1^43*x2^27*x3^17*x4^13*z^20 - x1^42*x2^28*x3^17*x4^13*z^20 + x1^41*x2^29*x3^17*x4^13*z^20 - x1^39*x2^31*x3^17*x4^13*z^20 - 3*x1^38*x2^32*x3^17*x4^13*z^20 + x1^37*x2^33*x3^17*x4^13*z^20 - 2*x1^36*x2^34*x3^17*x4^13*z^20 - x1^35*x2^35*x3^17*x4^13*z^20 - x1^41*x2^28*x3^18*x4^13*z^20 + x1^40*x2^29*x3^18*x4^13*z^20 - x1^39*x2^30*x3^18*x4^13*z^20 + 2*x1^38*x2^31*x3^18*x4^13*z^20 + x1^37*x2^32*x3^18*x4^13*z^20 + x1^36*x2^33*x3^18*x4^13*z^20 + 5*x1^35*x2^34*x3^18*x4^13*z^20 - x1^42*x2^26*x3^19*x4^13*z^20 - x1^41*x2^27*x3^19*x4^13*z^20 + 4*x1^40*x2^28*x3^19*x4^13*z^20 - 4*x1^37*x2^31*x3^19*x4^13*z^20 + x1^36*x2^32*x3^19*x4^13*z^20 - 4*x1^35*x2^33*x3^19*x4^13*z^20 - 2*x1^34*x2^34*x3^19*x4^13*z^20 - x1^41*x2^26*x3^20*x4^13*z^20 + x1^38*x2^29*x3^20*x4^13*z^20 + 2*x1^37*x2^30*x3^20*x4^13*z^20 + 2*x1^36*x2^31*x3^20*x4^13*z^20 + 2*x1^35*x2^32*x3^20*x4^13*z^20 + 6*x1^34*x2^33*x3^20*x4^13*z^20 - x1^40*x2^26*x3^21*x4^13*z^20 + x1^39*x2^27*x3^21*x4^13*z^20 + 2*x1^38*x2^28*x3^21*x4^13*z^20 + 2*x1^37*x2^29*x3^21*x4^13*z^20 - 3*x1^36*x2^30*x3^21*x4^13*z^20 - 6*x1^34*x2^32*x3^21*x4^13*z^20 - 2*x1^33*x2^33*x3^21*x4^13*z^20 + x1^39*x2^26*x3^22*x4^13*z^20 - x1^38*x2^27*x3^22*x4^13*z^20 - x1^37*x2^28*x3^22*x4^13*z^20 + x1^35*x2^30*x3^22*x4^13*z^20 + 2*x1^34*x2^31*x3^22*x4^13*z^20 + 6*x1^33*x2^32*x3^22*x4^13*z^20 - x1^38*x2^26*x3^23*x4^13*z^20 + x1^36*x2^28*x3^23*x4^13*z^20 - 2*x1^35*x2^29*x3^23*x4^13*z^20 + x1^34*x2^30*x3^23*x4^13*z^20 - 5*x1^33*x2^31*x3^23*x4^13*z^20 - 2*x1^32*x2^32*x3^23*x4^13*z^20 - x1^36*x2^27*x3^24*x4^13*z^20 + 5*x1^32*x2^31*x3^24*x4^13*z^20 - 2*x1^34*x2^28*x3^25*x4^13*z^20 + x1^33*x2^29*x3^25*x4^13*z^20 - 2*x1^32*x2^30*x3^25*x4^13*z^20 - 2*x1^31*x2^31*x3^25*x4^13*z^20 - x1^32*x2^29*x3^26*x4^13*z^20 + 2*x1^31*x2^30*x3^26*x4^13*z^20 - x1^30*x2^30*x3^27*x4^13*z^20 + x1^40*x2^31*x3^15*x4^14*z^20 - x1^39*x2^32*x3^15*x4^14*z^20 + x1^38*x2^33*x3^15*x4^14*z^20 - x1^39*x2^31*x3^16*x4^14*z^20 - x1^38*x2^32*x3^16*x4^14*z^20 - x1^37*x2^33*x3^16*x4^14*z^20 + 3*x1^39*x2^30*x3^17*x4^14*z^20 - x1^38*x2^31*x3^17*x4^14*z^20 + x1^37*x2^32*x3^17*x4^14*z^20 + 2*x1^36*x2^33*x3^17*x4^14*z^20 + x1^42*x2^26*x3^18*x4^14*z^20 + x1^39*x2^29*x3^18*x4^14*z^20 - 2*x1^38*x2^30*x3^18*x4^14*z^20 - x1^37*x2^31*x3^18*x4^14*z^20 - 2*x1^36*x2^32*x3^18*x4^14*z^20 + x1^35*x2^33*x3^18*x4^14*z^20 + x1^34*x2^34*x3^18*x4^14*z^20 - x1^41*x2^26*x3^19*x4^14*z^20 - x1^40*x2^27*x3^19*x4^14*z^20 + 3*x1^38*x2^29*x3^19*x4^14*z^20 - x1^37*x2^30*x3^19*x4^14*z^20 - x1^34*x2^33*x3^19*x4^14*z^20 - x1^38*x2^28*x3^20*x4^14*z^20 - 3*x1^35*x2^31*x3^20*x4^14*z^20 + x1^33*x2^33*x3^20*x4^14*z^20 - 2*x1^38*x2^27*x3^21*x4^14*z^20 + 2*x1^37*x2^28*x3^21*x4^14*z^20 + 2*x1^35*x2^30*x3^21*x4^14*z^20 - x1^34*x2^31*x3^21*x4^14*z^20 - 2*x1^33*x2^32*x3^21*x4^14*z^20 + x1^35*x2^29*x3^22*x4^14*z^20 - 3*x1^34*x2^30*x3^22*x4^14*z^20 + 2*x1^33*x2^31*x3^22*x4^14*z^20 + x1^36*x2^27*x3^23*x4^14*z^20 - x1^35*x2^28*x3^23*x4^14*z^20 + 2*x1^34*x2^29*x3^23*x4^14*z^20 + 2*x1^33*x2^30*x3^23*x4^14*z^20 - 2*x1^32*x2^31*x3^23*x4^14*z^20 - x1^35*x2^27*x3^24*x4^14*z^20 - 2*x1^33*x2^29*x3^24*x4^14*z^20 + x1^31*x2^31*x3^24*x4^14*z^20 - x1^39*x2^30*x3^16*x4^15*z^20 + x1^38*x2^31*x3^16*x4^15*z^20 - x1^37*x2^32*x3^16*x4^15*z^20 - x1^36*x2^33*x3^16*x4^15*z^20 + x1^39*x2^29*x3^17*x4^15*z^20 + 2*x1^38*x2^30*x3^17*x4^15*z^20 + 2*x1^36*x2^32*x3^17*x4^15*z^20 - 4*x1^38*x2^29*x3^18*x4^15*z^20 - 2*x1^36*x2^31*x3^18*x4^15*z^20 - 2*x1^35*x2^32*x3^18*x4^15*z^20 + x1^38*x2^28*x3^19*x4^15*z^20 + 3*x1^37*x2^29*x3^19*x4^15*z^20 + x1^36*x2^30*x3^19*x4^15*z^20 + 4*x1^35*x2^31*x3^19*x4^15*z^20 + x1^40*x2^25*x3^20*x4^15*z^20 + x1^39*x2^26*x3^20*x4^15*z^20 - x1^38*x2^27*x3^20*x4^15*z^20 - 4*x1^37*x2^28*x3^20*x4^15*z^20 + x1^36*x2^29*x3^20*x4^15*z^20 - 5*x1^35*x2^30*x3^20*x4^15*z^20 - 2*x1^34*x2^31*x3^20*x4^15*z^20 - x1^39*x2^25*x3^21*x4^15*z^20 + x1^37*x2^27*x3^21*x4^15*z^20 + 3*x1^36*x2^28*x3^21*x4^15*z^20 + 2*x1^35*x2^29*x3^21*x4^15*z^20 + 6*x1^34*x2^30*x3^21*x4^15*z^20 - 2*x1^36*x2^27*x3^22*x4^15*z^20 - 5*x1^34*x2^29*x3^22*x4^15*z^20 - 2*x1^33*x2^30*x3^22*x4^15*z^20 + x1^35*x2^27*x3^23*x4^15*z^20 + x1^34*x2^28*x3^23*x4^15*z^20 + 5*x1^33*x2^29*x3^23*x4^15*z^20 + x1^34*x2^27*x3^24*x4^15*z^20 - 2*x1^33*x2^28*x3^24*x4^15*z^20 - x1^32*x2^29*x3^24*x4^15*z^20 + 2*x1^32*x2^28*x3^25*x4^15*z^20 - x1^38*x2^28*x3^18*x4^16*z^20 - x1^37*x2^29*x3^18*x4^16*z^20 + x1^36*x2^30*x3^18*x4^16*z^20 - x1^35*x2^31*x3^18*x4^16*z^20 + 2*x1^37*x2^28*x3^19*x4^16*z^20 + 2*x1^35*x2^30*x3^19*x4^16*z^20 + x1^34*x2^31*x3^19*x4^16*z^20 - x1^36*x2^28*x3^20*x4^16*z^20 - 4*x1^34*x2^30*x3^20*x4^16*z^20 + x1^36*x2^27*x3^21*x4^16*z^20 - 2*x1^35*x2^28*x3^21*x4^16*z^20 + 3*x1^34*x2^29*x3^21*x4^16*z^20 + 2*x1^33*x2^30*x3^21*x4^16*z^20 + x1^36*x2^26*x3^22*x4^16*z^20 + x1^34*x2^28*x3^22*x4^16*z^20 - 4*x1^33*x2^29*x3^22*x4^16*z^20 - 2*x1^34*x2^27*x3^23*x4^16*z^20 + x1^33*x2^28*x3^23*x4^16*z^20 + 2*x1^32*x2^29*x3^23*x4^16*z^20 + x1^33*x2^27*x3^24*x4^16*z^20 - x1^32*x2^28*x3^24*x4^16*z^20 + 2*x1^33*x2^29*x3^21*x4^17*z^20 - x1^46*x2^33*x3^16*z^19 + x1^45*x2^34*x3^16*z^19 + x1^46*x2^32*x3^17*z^19 + x1^45*x2^33*x3^17*z^19 - 2*x1^44*x2^34*x3^17*z^19 - x1^45*x2^32*x3^18*z^19 - x1^44*x2^33*x3^18*z^19 + x1^42*x2^35*x3^18*z^19 + x1^45*x2^31*x3^19*z^19 + x1^44*x2^32*x3^19*z^19 - x1^43*x2^33*x3^19*z^19 + x1^42*x2^34*x3^19*z^19 - x1^44*x2^31*x3^20*z^19 + x1^43*x2^32*x3^20*z^19 - x1^42*x2^33*x3^20*z^19 - x1^41*x2^34*x3^20*z^19 + x1^47*x2^34*x3^13*x4*z^19 - x1^47*x2^33*x3^14*x4*z^19 - x1^46*x2^34*x3^14*x4*z^19 + x1^45*x2^35*x3^14*x4*z^19 + 4*x1^46*x2^33*x3^15*x4*z^19 + x1^45*x2^34*x3^15*x4*z^19 - 2*x1^46*x2^32*x3^16*x4*z^19 - 4*x1^45*x2^33*x3^16*x4*z^19 + x1^44*x2^34*x3^16*x4*z^19 - x1^43*x2^35*x3^16*x4*z^19 + x1^42*x2^36*x3^16*x4*z^19 + 6*x1^45*x2^32*x3^17*x4*z^19 + x1^44*x2^33*x3^17*x4*z^19 + x1^43*x2^34*x3^17*x4*z^19 + x1^42*x2^35*x3^17*x4*z^19 - x1^40*x2^37*x3^17*x4*z^19 - 2*x1^45*x2^31*x3^18*x4*z^19 - 6*x1^44*x2^32*x3^18*x4*z^19 - 4*x1^42*x2^34*x3^18*x4*z^19 + x1^40*x2^36*x3^18*x4*z^19 + x1^39*x2^37*x3^18*x4*z^19 + 5*x1^44*x2^31*x3^19*x4*z^19 + 2*x1^43*x2^32*x3^19*x4*z^19 + 2*x1^42*x2^33*x3^19*x4*z^19 + x1^41*x2^34*x3^19*x4*z^19 - x1^39*x2^36*x3^19*x4*z^19 - 2*x1^44*x2^30*x3^20*x4*z^19 - 3*x1^43*x2^31*x3^20*x4*z^19 + x1^42*x2^32*x3^20*x4*z^19 - 5*x1^41*x2^33*x3^20*x4*z^19 + x1^39*x2^35*x3^20*x4*z^19 + x1^38*x2^36*x3^20*x4*z^19 + 4*x1^43*x2^30*x3^21*x4*z^19 + 2*x1^41*x2^32*x3^21*x4*z^19 + 2*x1^40*x2^33*x3^21*x4*z^19 - 2*x1^43*x2^29*x3^22*x4*z^19 - x1^42*x2^30*x3^22*x4*z^19 + x1^41*x2^31*x3^22*x4*z^19 - 3*x1^40*x2^32*x3^22*x4*z^19 + x1^42*x2^29*x3^23*x4*z^19 + x1^41*x2^30*x3^23*x4*z^19 + x1^40*x2^31*x3^23*x4*z^19 + x1^47*x2^33*x3^13*x4^2*z^19 - 4*x1^46*x2^33*x3^14*x4^2*z^19 - x1^45*x2^34*x3^14*x4^2*z^19 - 2*x1^44*x2^35*x3^14*x4^2*z^19 + 2*x1^46*x2^32*x3^15*x4^2*z^19 + 4*x1^45*x2^33*x3^15*x4^2*z^19 - x1^44*x2^34*x3^15*x4^2*z^19 + 2*x1^43*x2^35*x3^15*x4^2*z^19 - x1^42*x2^36*x3^15*x4^2*z^19 - 6*x1^45*x2^32*x3^16*x4^2*z^19 - x1^44*x2^33*x3^16*x4^2*z^19 - x1^43*x2^34*x3^16*x4^2*z^19 - x1^42*x2^35*x3^16*x4^2*z^19 + 2*x1^45*x2^31*x3^17*x4^2*z^19 + 6*x1^44*x2^32*x3^17*x4^2*z^19 - x1^43*x2^33*x3^17*x4^2*z^19 + 4*x1^42*x2^34*x3^17*x4^2*z^19 - 2*x1^41*x2^35*x3^17*x4^2*z^19 - x1^39*x2^37*x3^17*x4^2*z^19 - 6*x1^44*x2^31*x3^18*x4^2*z^19 - 2*x1^43*x2^32*x3^18*x4^2*z^19 - x1^42*x2^33*x3^18*x4^2*z^19 - x1^41*x2^34*x3^18*x4^2*z^19 + 2*x1^39*x2^36*x3^18*x4^2*z^19 + x1^38*x2^37*x3^18*x4^2*z^19 + 2*x1^44*x2^30*x3^19*x4^2*z^19 + 6*x1^43*x2^31*x3^19*x4^2*z^19 + 4*x1^41*x2^33*x3^19*x4^2*z^19 - x1^40*x2^34*x3^19*x4^2*z^19 - x1^39*x2^35*x3^19*x4^2*z^19 - 3*x1^38*x2^36*x3^19*x4^2*z^19 - 6*x1^43*x2^30*x3^20*x4^2*z^19 - 2*x1^42*x2^31*x3^20*x4^2*z^19 - 2*x1^41*x2^32*x3^20*x4^2*z^19 - 2*x1^40*x2^33*x3^20*x4^2*z^19 + 2*x1^38*x2^35*x3^20*x4^2*z^19 + x1^37*x2^36*x3^20*x4^2*z^19 + 2*x1^43*x2^29*x3^21*x4^2*z^19 + 4*x1^42*x2^30*x3^21*x4^2*z^19 + 5*x1^40*x2^32*x3^21*x4^2*z^19 - x1^39*x2^33*x3^21*x4^2*z^19 - 2*x1^37*x2^35*x3^21*x4^2*z^19 - 5*x1^42*x2^29*x3^22*x4^2*z^19 - x1^41*x2^30*x3^22*x4^2*z^19 - x1^40*x2^31*x3^22*x4^2*z^19 - 2*x1^39*x2^32*x3^22*x4^2*z^19 - x1^38*x2^33*x3^22*x4^2*z^19 + x1^37*x2^34*x3^22*x4^2*z^19 + x1^36*x2^35*x3^22*x4^2*z^19 + x1^42*x2^28*x3^23*x4^2*z^19 + 2*x1^41*x2^29*x3^23*x4^2*z^19 + 4*x1^39*x2^31*x3^23*x4^2*z^19 - x1^41*x2^28*x3^24*x4^2*z^19 - x1^40*x2^29*x3^24*x4^2*z^19 - x1^39*x2^30*x3^24*x4^2*z^19 - x1^38*x2^31*x3^24*x4^2*z^19 + 2*x1^38*x2^30*x3^25*x4^2*z^19 - x1^48*x2^31*x3^13*x4^3*z^19 + x1^46*x2^33*x3^13*x4^3*z^19 - x1^45*x2^34*x3^13*x4^3*z^19 + 2*x1^47*x2^31*x3^14*x4^3*z^19 - 2*x1^46*x2^32*x3^14*x4^3*z^19 - x1^45*x2^33*x3^14*x4^3*z^19 + 2*x1^44*x2^34*x3^14*x4^3*z^19 - 2*x1^46*x2^31*x3^15*x4^3*z^19 + 3*x1^45*x2^32*x3^15*x4^3*z^19 - x1^44*x2^33*x3^15*x4^3*z^19 - x1^43*x2^34*x3^15*x4^3*z^19 - x1^42*x2^35*x3^15*x4^3*z^19 - x1^41*x2^36*x3^15*x4^3*z^19 + 2*x1^46*x2^30*x3^16*x4^3*z^19 + x1^45*x2^31*x3^16*x4^3*z^19 - 2*x1^44*x2^32*x3^16*x4^3*z^19 - x1^42*x2^34*x3^16*x4^3*z^19 + 2*x1^41*x2^35*x3^16*x4^3*z^19 + x1^40*x2^36*x3^16*x4^3*z^19 - x1^46*x2^29*x3^17*x4^3*z^19 - 2*x1^45*x2^30*x3^17*x4^3*z^19 + 2*x1^44*x2^31*x3^17*x4^3*z^19 + x1^42*x2^33*x3^17*x4^3*z^19 - x1^40*x2^35*x3^17*x4^3*z^19 - x1^39*x2^36*x3^17*x4^3*z^19 + 2*x1^45*x2^29*x3^18*x4^3*z^19 - x1^43*x2^31*x3^18*x4^3*z^19 + x1^42*x2^32*x3^18*x4^3*z^19 - 2*x1^41*x2^33*x3^18*x4^3*z^19 + x1^40*x2^34*x3^18*x4^3*z^19 + 2*x1^39*x2^35*x3^18*x4^3*z^19 + 2*x1^38*x2^36*x3^18*x4^3*z^19 - x1^45*x2^28*x3^19*x4^3*z^19 - 2*x1^44*x2^29*x3^19*x4^3*z^19 + 2*x1^43*x2^30*x3^19*x4^3*z^19 + 2*x1^41*x2^32*x3^19*x4^3*z^19 - 2*x1^38*x2^35*x3^19*x4^3*z^19 - 2*x1^37*x2^36*x3^19*x4^3*z^19 + 2*x1^44*x2^28*x3^20*x4^3*z^19 - x1^43*x2^29*x3^20*x4^3*z^19 - 2*x1^42*x2^30*x3^20*x4^3*z^19 + x1^41*x2^31*x3^20*x4^3*z^19 + 2*x1^39*x2^33*x3^20*x4^3*z^19 - 2*x1^38*x2^34*x3^20*x4^3*z^19 + 2*x1^37*x2^35*x3^20*x4^3*z^19 + x1^36*x2^36*x3^20*x4^3*z^19 - x1^43*x2^28*x3^21*x4^3*z^19 + 2*x1^42*x2^29*x3^21*x4^3*z^19 - x1^41*x2^30*x3^21*x4^3*z^19 + 2*x1^39*x2^32*x3^21*x4^3*z^19 - 2*x1^37*x2^34*x3^21*x4^3*z^19 - x1^36*x2^35*x3^21*x4^3*z^19 + 2*x1^43*x2^27*x3^22*x4^3*z^19 + x1^42*x2^28*x3^22*x4^3*z^19 - 2*x1^41*x2^29*x3^22*x4^3*z^19 - x1^40*x2^30*x3^22*x4^3*z^19 - x1^39*x2^31*x3^22*x4^3*z^19 + x1^38*x2^32*x3^22*x4^3*z^19 - x1^37*x2^33*x3^22*x4^3*z^19 + x1^36*x2^34*x3^22*x4^3*z^19 + x1^41*x2^28*x3^23*x4^3*z^19 - x1^39*x2^30*x3^23*x4^3*z^19 + x1^38*x2^31*x3^23*x4^3*z^19 + x1^37*x2^32*x3^23*x4^3*z^19 - x1^36*x2^33*x3^23*x4^3*z^19 - x1^35*x2^34*x3^23*x4^3*z^19 - x1^40*x2^28*x3^24*x4^3*z^19 - x1^39*x2^29*x3^24*x4^3*z^19 - 3*x1^38*x2^30*x3^24*x4^3*z^19 + x1^37*x2^30*x3^25*x4^3*z^19 - x1^37*x2^29*x3^26*x4^3*z^19 - x1^48*x2^32*x3^11*x4^4*z^19 + x1^47*x2^32*x3^12*x4^4*z^19 - 2*x1^46*x2^33*x3^12*x4^4*z^19 - 3*x1^47*x2^31*x3^13*x4^4*z^19 - x1^46*x2^32*x3^13*x4^4*z^19 + x1^44*x2^34*x3^13*x4^4*z^19 + 2*x1^47*x2^30*x3^14*x4^4*z^19 + 3*x1^46*x2^31*x3^14*x4^4*z^19 - x1^45*x2^32*x3^14*x4^4*z^19 + x1^44*x2^33*x3^14*x4^4*z^19 - x1^43*x2^34*x3^14*x4^4*z^19 - 6*x1^46*x2^30*x3^15*x4^4*z^19 - x1^44*x2^32*x3^15*x4^4*z^19 + x1^43*x2^33*x3^15*x4^4*z^19 + x1^42*x2^34*x3^15*x4^4*z^19 + 2*x1^41*x2^35*x3^15*x4^4*z^19 + 2*x1^46*x2^29*x3^16*x4^4*z^19 + 6*x1^45*x2^30*x3^16*x4^4*z^19 + 5*x1^43*x2^32*x3^16*x4^4*z^19 - 2*x1^42*x2^33*x3^16*x4^4*z^19 - 2*x1^40*x2^35*x3^16*x4^4*z^19 + x1^39*x2^36*x3^16*x4^4*z^19 - 6*x1^45*x2^29*x3^17*x4^4*z^19 - 2*x1^44*x2^30*x3^17*x4^4*z^19 - 2*x1^43*x2^31*x3^17*x4^4*z^19 + 2*x1^41*x2^33*x3^17*x4^4*z^19 + 2*x1^40*x2^34*x3^17*x4^4*z^19 + 2*x1^39*x2^35*x3^17*x4^4*z^19 + 2*x1^45*x2^28*x3^18*x4^4*z^19 + 6*x1^44*x2^29*x3^18*x4^4*z^19 + 4*x1^42*x2^31*x3^18*x4^4*z^19 - 4*x1^41*x2^32*x3^18*x4^4*z^19 - x1^40*x2^33*x3^18*x4^4*z^19 - 6*x1^39*x2^34*x3^18*x4^4*z^19 - 2*x1^38*x2^35*x3^18*x4^4*z^19 - 6*x1^44*x2^28*x3^19*x4^4*z^19 - 2*x1^43*x2^29*x3^19*x4^4*z^19 - 2*x1^42*x2^30*x3^19*x4^4*z^19 - x1^41*x2^31*x3^19*x4^4*z^19 + 2*x1^40*x2^32*x3^19*x4^4*z^19 + x1^39*x2^33*x3^19*x4^4*z^19 + 3*x1^38*x2^34*x3^19*x4^4*z^19 - x1^36*x2^36*x3^19*x4^4*z^19 + 2*x1^44*x2^27*x3^20*x4^4*z^19 + 6*x1^43*x2^28*x3^20*x4^4*z^19 + 4*x1^41*x2^30*x3^20*x4^4*z^19 - x1^40*x2^31*x3^20*x4^4*z^19 + x1^39*x2^32*x3^20*x4^4*z^19 - 3*x1^38*x2^33*x3^20*x4^4*z^19 - x1^37*x2^34*x3^20*x4^4*z^19 - 6*x1^43*x2^27*x3^21*x4^4*z^19 - 2*x1^42*x2^28*x3^21*x4^4*z^19 - 2*x1^41*x2^29*x3^21*x4^4*z^19 - 2*x1^40*x2^30*x3^21*x4^4*z^19 + x1^38*x2^32*x3^21*x4^4*z^19 + 3*x1^37*x2^33*x3^21*x4^4*z^19 - x1^36*x2^34*x3^21*x4^4*z^19 - x1^35*x2^35*x3^21*x4^4*z^19 + x1^43*x2^26*x3^22*x4^4*z^19 + 5*x1^42*x2^27*x3^22*x4^4*z^19 + 3*x1^40*x2^29*x3^22*x4^4*z^19 - 2*x1^37*x2^32*x3^22*x4^4*z^19 + x1^36*x2^33*x3^22*x4^4*z^19 - 3*x1^42*x2^26*x3^23*x4^4*z^19 - x1^41*x2^27*x3^23*x4^4*z^19 - x1^38*x2^30*x3^23*x4^4*z^19 + x1^37*x2^31*x3^23*x4^4*z^19 + 2*x1^36*x2^32*x3^23*x4^4*z^19 - 2*x1^35*x2^33*x3^23*x4^4*z^19 + x1^41*x2^26*x3^24*x4^4*z^19 + x1^39*x2^28*x3^24*x4^4*z^19 - x1^38*x2^29*x3^24*x4^4*z^19 + x1^36*x2^31*x3^24*x4^4*z^19 + x1^35*x2^32*x3^24*x4^4*z^19 - x1^37*x2^28*x3^26*x4^4*z^19 - 2*x1^36*x2^29*x3^26*x4^4*z^19 + x1^47*x2^32*x3^11*x4^5*z^19 + 2*x1^47*x2^31*x3^12*x4^5*z^19 - x1^47*x2^30*x3^13*x4^5*z^19 - 4*x1^46*x2^31*x3^13*x4^5*z^19 + x1^45*x2^32*x3^13*x4^5*z^19 - x1^44*x2^33*x3^13*x4^5*z^19 + x1^43*x2^34*x3^13*x4^5*z^19 + 6*x1^46*x2^30*x3^14*x4^5*z^19 + x1^45*x2^31*x3^14*x4^5*z^19 - 2*x1^42*x2^34*x3^14*x4^5*z^19 - x1^41*x2^35*x3^14*x4^5*z^19 - 2*x1^46*x2^29*x3^15*x4^5*z^19 - 6*x1^45*x2^30*x3^15*x4^5*z^19 + 3*x1^44*x2^31*x3^15*x4^5*z^19 - 2*x1^43*x2^32*x3^15*x4^5*z^19 + 3*x1^42*x2^33*x3^15*x4^5*z^19 + x1^41*x2^34*x3^15*x4^5*z^19 + x1^40*x2^35*x3^15*x4^5*z^19 + 6*x1^45*x2^29*x3^16*x4^5*z^19 + x1^44*x2^30*x3^16*x4^5*z^19 - x1^43*x2^31*x3^16*x4^5*z^19 - 2*x1^41*x2^33*x3^16*x4^5*z^19 - x1^40*x2^34*x3^16*x4^5*z^19 - x1^39*x2^35*x3^16*x4^5*z^19 - x1^38*x2^36*x3^16*x4^5*z^19 - 2*x1^45*x2^28*x3^17*x4^5*z^19 - 6*x1^44*x2^29*x3^17*x4^5*z^19 + 2*x1^43*x2^30*x3^17*x4^5*z^19 - 4*x1^42*x2^31*x3^17*x4^5*z^19 + 3*x1^41*x2^32*x3^17*x4^5*z^19 + x1^40*x2^33*x3^17*x4^5*z^19 + 4*x1^39*x2^34*x3^17*x4^5*z^19 + x1^37*x2^36*x3^17*x4^5*z^19 + 6*x1^44*x2^28*x3^18*x4^5*z^19 + 2*x1^43*x2^29*x3^18*x4^5*z^19 - 4*x1^40*x2^32*x3^18*x4^5*z^19 - 2*x1^39*x2^33*x3^18*x4^5*z^19 - 3*x1^38*x2^34*x3^18*x4^5*z^19 + x1^37*x2^35*x3^18*x4^5*z^19 - 2*x1^44*x2^27*x3^19*x4^5*z^19 - 6*x1^43*x2^28*x3^19*x4^5*z^19 + 2*x1^42*x2^29*x3^19*x4^5*z^19 - 3*x1^41*x2^30*x3^19*x4^5*z^19 + 5*x1^40*x2^31*x3^19*x4^5*z^19 + 5*x1^38*x2^33*x3^19*x4^5*z^19 - x1^36*x2^35*x3^19*x4^5*z^19 + 6*x1^43*x2^27*x3^20*x4^5*z^19 + x1^42*x2^28*x3^20*x4^5*z^19 - 3*x1^39*x2^31*x3^20*x4^5*z^19 - 3*x1^38*x2^32*x3^20*x4^5*z^19 - 6*x1^37*x2^33*x3^20*x4^5*z^19 + 2*x1^36*x2^34*x3^20*x4^5*z^19 - 6*x1^42*x2^27*x3^21*x4^5*z^19 + 2*x1^41*x2^28*x3^21*x4^5*z^19 - 3*x1^40*x2^29*x3^21*x4^5*z^19 + 3*x1^39*x2^30*x3^21*x4^5*z^19 + 4*x1^37*x2^32*x3^21*x4^5*z^19 + x1^36*x2^33*x3^21*x4^5*z^19 - x1^35*x2^34*x3^21*x4^5*z^19 + 2*x1^42*x2^26*x3^22*x4^5*z^19 + 2*x1^41*x2^27*x3^22*x4^5*z^19 - x1^40*x2^28*x3^22*x4^5*z^19 + x1^39*x2^29*x3^22*x4^5*z^19 - 2*x1^38*x2^30*x3^22*x4^5*z^19 - 4*x1^36*x2^32*x3^22*x4^5*z^19 + 3*x1^35*x2^33*x3^22*x4^5*z^19 - 2*x1^41*x2^26*x3^23*x4^5*z^19 - 3*x1^39*x2^28*x3^23*x4^5*z^19 - x1^38*x2^29*x3^23*x4^5*z^19 + 3*x1^36*x2^31*x3^23*x4^5*z^19 + x1^35*x2^32*x3^23*x4^5*z^19 - x1^34*x2^33*x3^23*x4^5*z^19 + x1^39*x2^27*x3^24*x4^5*z^19 - x1^38*x2^28*x3^24*x4^5*z^19 - x1^37*x2^29*x3^24*x4^5*z^19 - 2*x1^36*x2^30*x3^24*x4^5*z^19 - x1^35*x2^31*x3^24*x4^5*z^19 + 2*x1^34*x2^32*x3^24*x4^5*z^19 - x1^39*x2^26*x3^25*x4^5*z^19 - x1^38*x2^27*x3^25*x4^5*z^19 + x1^37*x2^28*x3^25*x4^5*z^19 + x1^36*x2^28*x3^26*x4^5*z^19 - x1^35*x2^29*x3^26*x4^5*z^19 + 2*x1^33*x2^31*x3^26*x4^5*z^19 + x1^35*x2^28*x3^27*x4^5*z^19 + x1^46*x2^31*x3^12*x4^6*z^19 + x1^44*x2^33*x3^12*x4^6*z^19 - x1^45*x2^31*x3^13*x4^6*z^19 + x1^44*x2^32*x3^13*x4^6*z^19 + x1^42*x2^34*x3^13*x4^6*z^19 + 2*x1^45*x2^30*x3^14*x4^6*z^19 - 2*x1^42*x2^33*x3^14*x4^6*z^19 + x1^40*x2^35*x3^14*x4^6*z^19 - 2*x1^45*x2^29*x3^15*x4^6*z^19 + x1^44*x2^30*x3^15*x4^6*z^19 + x1^43*x2^31*x3^15*x4^6*z^19 - 2*x1^42*x2^32*x3^15*x4^6*z^19 + x1^40*x2^34*x3^15*x4^6*z^19 - x1^39*x2^35*x3^15*x4^6*z^19 + 2*x1^44*x2^29*x3^16*x4^6*z^19 - 4*x1^43*x2^30*x3^16*x4^6*z^19 + x1^42*x2^31*x3^16*x4^6*z^19 - x1^41*x2^32*x3^16*x4^6*z^19 - x1^39*x2^34*x3^16*x4^6*z^19 + x1^38*x2^35*x3^16*x4^6*z^19 - 2*x1^44*x2^28*x3^17*x4^6*z^19 + 3*x1^42*x2^30*x3^17*x4^6*z^19 + 2*x1^40*x2^32*x3^17*x4^6*z^19 - 2*x1^39*x2^33*x3^17*x4^6*z^19 + x1^38*x2^34*x3^17*x4^6*z^19 + x1^44*x2^27*x3^18*x4^6*z^19 + 2*x1^43*x2^28*x3^18*x4^6*z^19 - 4*x1^42*x2^29*x3^18*x4^6*z^19 - 2*x1^40*x2^31*x3^18*x4^6*z^19 - 3*x1^38*x2^33*x3^18*x4^6*z^19 - x1^37*x2^34*x3^18*x4^6*z^19 - 2*x1^43*x2^27*x3^19*x4^6*z^19 + 3*x1^41*x2^29*x3^19*x4^6*z^19 + 4*x1^39*x2^31*x3^19*x4^6*z^19 - x1^38*x2^32*x3^19*x4^6*z^19 + 2*x1^37*x2^33*x3^19*x4^6*z^19 - 4*x1^36*x2^34*x3^19*x4^6*z^19 - x1^35*x2^35*x3^19*x4^6*z^19 + 2*x1^42*x2^27*x3^20*x4^6*z^19 - 4*x1^41*x2^28*x3^20*x4^6*z^19 - x1^40*x2^29*x3^20*x4^6*z^19 - 4*x1^39*x2^30*x3^20*x4^6*z^19 + 2*x1^36*x2^33*x3^20*x4^6*z^19 + 4*x1^35*x2^34*x3^20*x4^6*z^19 - x1^42*x2^26*x3^21*x4^6*z^19 + x1^41*x2^27*x3^21*x4^6*z^19 + 4*x1^40*x2^28*x3^21*x4^6*z^19 + x1^39*x2^29*x3^21*x4^6*z^19 + 2*x1^38*x2^30*x3^21*x4^6*z^19 - x1^37*x2^31*x3^21*x4^6*z^19 + 4*x1^36*x2^32*x3^21*x4^6*z^19 - 4*x1^35*x2^33*x3^21*x4^6*z^19 + x1^41*x2^26*x3^22*x4^6*z^19 - 4*x1^40*x2^27*x3^22*x4^6*z^19 + x1^39*x2^28*x3^22*x4^6*z^19 - 2*x1^38*x2^29*x3^22*x4^6*z^19 - 2*x1^37*x2^30*x3^22*x4^6*z^19 - 2*x1^36*x2^31*x3^22*x4^6*z^19 + 3*x1^34*x2^33*x3^22*x4^6*z^19 - x1^41*x2^25*x3^23*x4^6*z^19 - x1^40*x2^26*x3^23*x4^6*z^19 + 3*x1^39*x2^27*x3^23*x4^6*z^19 + 3*x1^37*x2^29*x3^23*x4^6*z^19 - x1^36*x2^30*x3^23*x4^6*z^19 + x1^35*x2^31*x3^23*x4^6*z^19 - 3*x1^34*x2^32*x3^23*x4^6*z^19 + x1^33*x2^33*x3^23*x4^6*z^19 + x1^40*x2^25*x3^24*x4^6*z^19 - x1^38*x2^27*x3^24*x4^6*z^19 - x1^37*x2^28*x3^24*x4^6*z^19 + x1^36*x2^29*x3^24*x4^6*z^19 - 2*x1^35*x2^30*x3^24*x4^6*z^19 - x1^34*x2^31*x3^24*x4^6*z^19 + 2*x1^33*x2^32*x3^24*x4^6*z^19 - x1^38*x2^26*x3^25*x4^6*z^19 + 3*x1^36*x2^28*x3^25*x4^6*z^19 - 2*x1^34*x2^30*x3^25*x4^6*z^19 - 2*x1^33*x2^31*x3^25*x4^6*z^19 + x1^37*x2^26*x3^26*x4^6*z^19 - x1^36*x2^27*x3^26*x4^6*z^19 - 2*x1^35*x2^28*x3^26*x4^6*z^19 + x1^34*x2^29*x3^26*x4^6*z^19 - x1^35*x2^27*x3^27*x4^6*z^19 - x1^32*x2^30*x3^27*x4^6*z^19 + x1^46*x2^31*x3^11*x4^7*z^19 - x1^45*x2^32*x3^11*x4^7*z^19 + x1^43*x2^34*x3^11*x4^7*z^19 - x1^46*x2^30*x3^12*x4^7*z^19 + x1^45*x2^31*x3^12*x4^7*z^19 + 2*x1^44*x2^32*x3^12*x4^7*z^19 - x1^43*x2^33*x3^12*x4^7*z^19 + 2*x1^45*x2^30*x3^13*x4^7*z^19 - x1^43*x2^32*x3^13*x4^7*z^19 - x1^42*x2^33*x3^13*x4^7*z^19 - x1^41*x2^34*x3^13*x4^7*z^19 + x1^40*x2^35*x3^13*x4^7*z^19 - 2*x1^45*x2^29*x3^14*x4^7*z^19 + x1^44*x2^30*x3^14*x4^7*z^19 + x1^43*x2^31*x3^14*x4^7*z^19 - 2*x1^40*x2^34*x3^14*x4^7*z^19 - x1^39*x2^35*x3^14*x4^7*z^19 - x1^38*x2^36*x3^14*x4^7*z^19 + x1^44*x2^29*x3^15*x4^7*z^19 + 3*x1^42*x2^31*x3^15*x4^7*z^19 + x1^41*x2^32*x3^15*x4^7*z^19 - x1^40*x2^33*x3^15*x4^7*z^19 + x1^39*x2^34*x3^15*x4^7*z^19 + x1^38*x2^35*x3^15*x4^7*z^19 + x1^37*x2^36*x3^15*x4^7*z^19 - 3*x1^37*x2^35*x3^16*x4^7*z^19 + x1^39*x2^32*x3^17*x4^7*z^19 + 3*x1^36*x2^35*x3^17*x4^7*z^19 + x1^37*x2^33*x3^18*x4^7*z^19 - x1^36*x2^34*x3^18*x4^7*z^19 - x1^35*x2^35*x3^18*x4^7*z^19 + x1^41*x2^28*x3^19*x4^7*z^19 + x1^35*x2^34*x3^19*x4^7*z^19 - x1^41*x2^27*x3^20*x4^7*z^19 - x1^40*x2^28*x3^20*x4^7*z^19 + x1^39*x2^29*x3^20*x4^7*z^19 + x1^42*x2^25*x3^21*x4^7*z^19 + 2*x1^40*x2^27*x3^21*x4^7*z^19 + x1^39*x2^28*x3^21*x4^7*z^19 - x1^41*x2^25*x3^22*x4^7*z^19 - x1^40*x2^26*x3^22*x4^7*z^19 - 2*x1^39*x2^27*x3^22*x4^7*z^19 - 2*x1^38*x2^28*x3^22*x4^7*z^19 - x1^37*x2^29*x3^22*x4^7*z^19 + x1^36*x2^30*x3^22*x4^7*z^19 - 2*x1^35*x2^31*x3^22*x4^7*z^19 + x1^40*x2^25*x3^23*x4^7*z^19 + x1^39*x2^26*x3^23*x4^7*z^19 + x1^37*x2^28*x3^23*x4^7*z^19 + 2*x1^36*x2^29*x3^23*x4^7*z^19 + x1^35*x2^30*x3^23*x4^7*z^19 - x1^33*x2^32*x3^23*x4^7*z^19 - x1^39*x2^25*x3^24*x4^7*z^19 - x1^38*x2^26*x3^24*x4^7*z^19 - x1^37*x2^27*x3^24*x4^7*z^19 - 3*x1^36*x2^28*x3^24*x4^7*z^19 - 2*x1^35*x2^29*x3^24*x4^7*z^19 + x1^33*x2^31*x3^24*x4^7*z^19 - x1^32*x2^32*x3^24*x4^7*z^19 + x1^37*x2^26*x3^25*x4^7*z^19 + x1^36*x2^26*x3^26*x4^7*z^19 + x1^33*x2^29*x3^26*x4^7*z^19 + x1^34*x2^27*x3^27*x4^7*z^19 - x1^46*x2^30*x3^11*x4^8*z^19 + x1^44*x2^32*x3^11*x4^8*z^19 - x1^43*x2^33*x3^11*x4^8*z^19 - x1^42*x2^34*x3^11*x4^8*z^19 + x1^41*x2^35*x3^11*x4^8*z^19 - x1^45*x2^30*x3^12*x4^8*z^19 + x1^43*x2^32*x3^12*x4^8*z^19 - x1^40*x2^35*x3^12*x4^8*z^19 - x1^44*x2^30*x3^13*x4^8*z^19 - 2*x1^43*x2^31*x3^13*x4^8*z^19 + x1^40*x2^34*x3^13*x4^8*z^19 + 2*x1^39*x2^35*x3^13*x4^8*z^19 + 2*x1^38*x2^36*x3^13*x4^8*z^19 + x1^45*x2^28*x3^14*x4^8*z^19 + 3*x1^43*x2^30*x3^14*x4^8*z^19 + 2*x1^42*x2^31*x3^14*x4^8*z^19 - 3*x1^38*x2^35*x3^14*x4^8*z^19 - 2*x1^37*x2^36*x3^14*x4^8*z^19 - x1^44*x2^28*x3^15*x4^8*z^19 - x1^43*x2^29*x3^15*x4^8*z^19 - 3*x1^42*x2^30*x3^15*x4^8*z^19 - x1^41*x2^31*x3^15*x4^8*z^19 + 3*x1^39*x2^33*x3^15*x4^8*z^19 - x1^38*x2^34*x3^15*x4^8*z^19 + 4*x1^37*x2^35*x3^15*x4^8*z^19 + x1^44*x2^27*x3^16*x4^8*z^19 + 3*x1^42*x2^29*x3^16*x4^8*z^19 + x1^41*x2^30*x3^16*x4^8*z^19 - x1^40*x2^31*x3^16*x4^8*z^19 - 2*x1^38*x2^33*x3^16*x4^8*z^19 - 4*x1^36*x2^35*x3^16*x4^8*z^19 - x1^43*x2^27*x3^17*x4^8*z^19 - x1^42*x2^28*x3^17*x4^8*z^19 - 3*x1^41*x2^29*x3^17*x4^8*z^19 + x1^40*x2^30*x3^17*x4^8*z^19 - x1^39*x2^31*x3^17*x4^8*z^19 + 2*x1^38*x2^32*x3^17*x4^8*z^19 + x1^37*x2^33*x3^17*x4^8*z^19 + 4*x1^36*x2^34*x3^17*x4^8*z^19 + x1^35*x2^35*x3^17*x4^8*z^19 + 2*x1^42*x2^27*x3^18*x4^8*z^19 + 4*x1^41*x2^28*x3^18*x4^8*z^19 + 4*x1^40*x2^29*x3^18*x4^8*z^19 + x1^39*x2^30*x3^18*x4^8*z^19 - 4*x1^37*x2^32*x3^18*x4^8*z^19 - 2*x1^36*x2^33*x3^18*x4^8*z^19 - 4*x1^35*x2^34*x3^18*x4^8*z^19 - 2*x1^42*x2^26*x3^19*x4^8*z^19 - 2*x1^41*x2^27*x3^19*x4^8*z^19 - 5*x1^40*x2^28*x3^19*x4^8*z^19 - x1^38*x2^30*x3^19*x4^8*z^19 + 4*x1^37*x2^31*x3^19*x4^8*z^19 + 2*x1^36*x2^32*x3^19*x4^8*z^19 + 4*x1^35*x2^33*x3^19*x4^8*z^19 + x1^34*x2^34*x3^19*x4^8*z^19 + x1^42*x2^25*x3^20*x4^8*z^19 + 2*x1^41*x2^26*x3^20*x4^8*z^19 + x1^39*x2^28*x3^20*x4^8*z^19 - 2*x1^38*x2^29*x3^20*x4^8*z^19 - 3*x1^36*x2^31*x3^20*x4^8*z^19 - 2*x1^35*x2^32*x3^20*x4^8*z^19 - 4*x1^34*x2^33*x3^20*x4^8*z^19 - 2*x1^41*x2^25*x3^21*x4^8*z^19 + x1^40*x2^26*x3^21*x4^8*z^19 - x1^38*x2^28*x3^21*x4^8*z^19 + 3*x1^36*x2^30*x3^21*x4^8*z^19 + 2*x1^35*x2^31*x3^21*x4^8*z^19 + 4*x1^34*x2^32*x3^21*x4^8*z^19 + x1^33*x2^33*x3^21*x4^8*z^19 + 2*x1^40*x2^25*x3^22*x4^8*z^19 - x1^39*x2^26*x3^22*x4^8*z^19 + x1^38*x2^27*x3^22*x4^8*z^19 - x1^37*x2^28*x3^22*x4^8*z^19 - x1^36*x2^29*x3^22*x4^8*z^19 - 3*x1^35*x2^30*x3^22*x4^8*z^19 - x1^34*x2^31*x3^22*x4^8*z^19 - 4*x1^33*x2^32*x3^22*x4^8*z^19 - x1^39*x2^25*x3^23*x4^8*z^19 + x1^38*x2^26*x3^23*x4^8*z^19 + x1^37*x2^27*x3^23*x4^8*z^19 + x1^36*x2^28*x3^23*x4^8*z^19 + x1^34*x2^30*x3^23*x4^8*z^19 + 3*x1^33*x2^31*x3^23*x4^8*z^19 + x1^32*x2^32*x3^23*x4^8*z^19 + x1^38*x2^25*x3^24*x4^8*z^19 - x1^36*x2^27*x3^24*x4^8*z^19 - 2*x1^35*x2^28*x3^24*x4^8*z^19 - 3*x1^34*x2^29*x3^24*x4^8*z^19 - 2*x1^32*x2^31*x3^24*x4^8*z^19 - x1^37*x2^25*x3^25*x4^8*z^19 + x1^35*x2^27*x3^25*x4^8*z^19 + 3*x1^34*x2^28*x3^25*x4^8*z^19 + x1^32*x2^30*x3^25*x4^8*z^19 + x1^31*x2^31*x3^25*x4^8*z^19 - x1^35*x2^26*x3^26*x4^8*z^19 - 2*x1^33*x2^28*x3^26*x4^8*z^19 + x1^31*x2^29*x3^27*x4^8*z^19 + x1^42*x2^33*x3^11*x4^9*z^19 + x1^45*x2^29*x3^12*x4^9*z^19 + x1^44*x2^30*x3^12*x4^9*z^19 - x1^40*x2^34*x3^12*x4^9*z^19 - x1^39*x2^35*x3^12*x4^9*z^19 - x1^44*x2^29*x3^13*x4^9*z^19 - x1^43*x2^30*x3^13*x4^9*z^19 - x1^40*x2^33*x3^13*x4^9*z^19 + x1^39*x2^34*x3^13*x4^9*z^19 + 2*x1^44*x2^28*x3^14*x4^9*z^19 + x1^43*x2^29*x3^14*x4^9*z^19 - 2*x1^40*x2^32*x3^14*x4^9*z^19 - 2*x1^39*x2^33*x3^14*x4^9*z^19 - x1^38*x2^34*x3^14*x4^9*z^19 - 2*x1^37*x2^35*x3^14*x4^9*z^19 - x1^44*x2^27*x3^15*x4^9*z^19 - x1^43*x2^28*x3^15*x4^9*z^19 - 2*x1^42*x2^29*x3^15*x4^9*z^19 + 3*x1^40*x2^31*x3^15*x4^9*z^19 + 2*x1^39*x2^32*x3^15*x4^9*z^19 - x1^38*x2^33*x3^15*x4^9*z^19 + 2*x1^36*x2^35*x3^15*x4^9*z^19 + 4*x1^43*x2^27*x3^16*x4^9*z^19 + x1^42*x2^28*x3^16*x4^9*z^19 + 2*x1^41*x2^29*x3^16*x4^9*z^19 - x1^40*x2^30*x3^16*x4^9*z^19 + x1^39*x2^31*x3^16*x4^9*z^19 - 2*x1^38*x2^32*x3^16*x4^9*z^19 - 2*x1^36*x2^34*x3^16*x4^9*z^19 - x1^43*x2^26*x3^17*x4^9*z^19 - 3*x1^42*x2^27*x3^17*x4^9*z^19 - 2*x1^41*x2^28*x3^17*x4^9*z^19 - 4*x1^40*x2^29*x3^17*x4^9*z^19 + x1^39*x2^30*x3^17*x4^9*z^19 + 5*x1^37*x2^32*x3^17*x4^9*z^19 + 2*x1^35*x2^34*x3^17*x4^9*z^19 + 3*x1^42*x2^26*x3^18*x4^9*z^19 + 2*x1^41*x2^27*x3^18*x4^9*z^19 + 2*x1^40*x2^28*x3^18*x4^9*z^19 - x1^39*x2^29*x3^18*x4^9*z^19 - 2*x1^38*x2^30*x3^18*x4^9*z^19 - 2*x1^37*x2^31*x3^18*x4^9*z^19 - 4*x1^36*x2^32*x3^18*x4^9*z^19 - 2*x1^35*x2^33*x3^18*x4^9*z^19 - x1^34*x2^34*x3^18*x4^9*z^19 - 2*x1^42*x2^25*x3^19*x4^9*z^19 - 4*x1^41*x2^26*x3^19*x4^9*z^19 - x1^40*x2^27*x3^19*x4^9*z^19 - 5*x1^39*x2^28*x3^19*x4^9*z^19 + x1^38*x2^29*x3^19*x4^9*z^19 + 5*x1^36*x2^31*x3^19*x4^9*z^19 + 3*x1^35*x2^32*x3^19*x4^9*z^19 + 2*x1^34*x2^33*x3^19*x4^9*z^19 + 5*x1^41*x2^25*x3^20*x4^9*z^19 + 2*x1^40*x2^26*x3^20*x4^9*z^19 + 3*x1^39*x2^27*x3^20*x4^9*z^19 + 2*x1^38*x2^28*x3^20*x4^9*z^19 - x1^37*x2^29*x3^20*x4^9*z^19 - 4*x1^36*x2^30*x3^20*x4^9*z^19 - 4*x1^35*x2^31*x3^20*x4^9*z^19 - 2*x1^34*x2^32*x3^20*x4^9*z^19 - 5*x1^40*x2^25*x3^21*x4^9*z^19 - x1^39*x2^26*x3^21*x4^9*z^19 - 3*x1^38*x2^27*x3^21*x4^9*z^19 + 2*x1^37*x2^28*x3^21*x4^9*z^19 - x1^36*x2^29*x3^21*x4^9*z^19 + 3*x1^35*x2^30*x3^21*x4^9*z^19 + x1^34*x2^31*x3^21*x4^9*z^19 + 2*x1^33*x2^32*x3^21*x4^9*z^19 + x1^40*x2^24*x3^22*x4^9*z^19 + 3*x1^39*x2^25*x3^22*x4^9*z^19 + 2*x1^38*x2^26*x3^22*x4^9*z^19 + x1^37*x2^27*x3^22*x4^9*z^19 + x1^36*x2^28*x3^22*x4^9*z^19 - 3*x1^34*x2^30*x3^22*x4^9*z^19 - 2*x1^33*x2^31*x3^22*x4^9*z^19 - x1^32*x2^32*x3^22*x4^9*z^19 - x1^39*x2^24*x3^23*x4^9*z^19 - 2*x1^38*x2^25*x3^23*x4^9*z^19 - 3*x1^37*x2^26*x3^23*x4^9*z^19 + x1^36*x2^27*x3^23*x4^9*z^19 - x1^35*x2^28*x3^23*x4^9*z^19 + 4*x1^34*x2^29*x3^23*x4^9*z^19 + x1^33*x2^30*x3^23*x4^9*z^19 + 2*x1^32*x2^31*x3^23*x4^9*z^19 + 2*x1^37*x2^25*x3^24*x4^9*z^19 + x1^36*x2^26*x3^24*x4^9*z^19 - 2*x1^33*x2^29*x3^24*x4^9*z^19 - x1^36*x2^25*x3^25*x4^9*z^19 + x1^35*x2^26*x3^25*x4^9*z^19 - 2*x1^34*x2^27*x3^25*x4^9*z^19 + 4*x1^33*x2^28*x3^25*x4^9*z^19 - x1^31*x2^30*x3^25*x4^9*z^19 - x1^34*x2^26*x3^26*x4^9*z^19 + x1^33*x2^27*x3^26*x4^9*z^19 - 2*x1^32*x2^28*x3^26*x4^9*z^19 + x1^31*x2^29*x3^26*x4^9*z^19 + x1^30*x2^30*x3^26*x4^9*z^19 + x1^40*x2^33*x3^12*x4^10*z^19 + x1^41*x2^31*x3^13*x4^10*z^19 + 2*x1^42*x2^29*x3^14*x4^10*z^19 + x1^40*x2^31*x3^14*x4^10*z^19 + x1^39*x2^32*x3^14*x4^10*z^19 - x1^38*x2^33*x3^14*x4^10*z^19 - 2*x1^37*x2^34*x3^14*x4^10*z^19 - x1^36*x2^35*x3^14*x4^10*z^19 - x1^43*x2^27*x3^15*x4^10*z^19 + x1^42*x2^28*x3^15*x4^10*z^19 - x1^41*x2^29*x3^15*x4^10*z^19 + x1^39*x2^31*x3^15*x4^10*z^19 + 2*x1^36*x2^34*x3^15*x4^10*z^19 + x1^40*x2^29*x3^16*x4^10*z^19 + x1^39*x2^30*x3^16*x4^10*z^19 - 2*x1^38*x2^31*x3^16*x4^10*z^19 + x1^36*x2^33*x3^16*x4^10*z^19 - 2*x1^35*x2^34*x3^16*x4^10*z^19 - 2*x1^42*x2^26*x3^17*x4^10*z^19 - x1^38*x2^30*x3^17*x4^10*z^19 + x1^41*x2^26*x3^18*x4^10*z^19 + 2*x1^38*x2^29*x3^18*x4^10*z^19 - x1^41*x2^25*x3^19*x4^10*z^19 - 2*x1^40*x2^26*x3^19*x4^10*z^19 - x1^39*x2^27*x3^19*x4^10*z^19 + 2*x1^38*x2^28*x3^19*x4^10*z^19 - x1^37*x2^29*x3^19*x4^10*z^19 + x1^41*x2^24*x3^20*x4^10*z^19 + x1^40*x2^25*x3^20*x4^10*z^19 + x1^38*x2^27*x3^20*x4^10*z^19 - x1^37*x2^28*x3^20*x4^10*z^19 - x1^40*x2^24*x3^21*x4^10*z^19 + x1^39*x2^25*x3^21*x4^10*z^19 + x1^37*x2^27*x3^21*x4^10*z^19 + x1^36*x2^28*x3^21*x4^10*z^19 + 2*x1^35*x2^29*x3^21*x4^10*z^19 + x1^39*x2^24*x3^22*x4^10*z^19 + 2*x1^37*x2^26*x3^22*x4^10*z^19 - 2*x1^36*x2^27*x3^22*x4^10*z^19 - 2*x1^34*x2^29*x3^22*x4^10*z^19 + x1^33*x2^30*x3^22*x4^10*z^19 - x1^38*x2^24*x3^23*x4^10*z^19 + x1^35*x2^27*x3^23*x4^10*z^19 + 2*x1^33*x2^29*x3^23*x4^10*z^19 - x1^36*x2^25*x3^24*x4^10*z^19 - 2*x1^35*x2^26*x3^24*x4^10*z^19 - 3*x1^33*x2^28*x3^24*x4^10*z^19 - 2*x1^32*x2^29*x3^24*x4^10*z^19 + x1^34*x2^26*x3^25*x4^10*z^19 + x1^32*x2^28*x3^25*x4^10*z^19 + x1^31*x2^29*x3^25*x4^10*z^19 - x1^30*x2^30*x3^25*x4^10*z^19 - x1^32*x2^27*x3^26*x4^10*z^19 - x1^30*x2^29*x3^26*x4^10*z^19 + x1^31*x2^27*x3^27*x4^10*z^19 - x1^41*x2^32*x3^11*x4^11*z^19 + x1^39*x2^33*x3^12*x4^11*z^19 - 2*x1^40*x2^31*x3^13*x4^11*z^19 - x1^39*x2^32*x3^13*x4^11*z^19 - x1^38*x2^33*x3^13*x4^11*z^19 - x1^43*x2^27*x3^14*x4^11*z^19 - x1^42*x2^28*x3^14*x4^11*z^19 + x1^41*x2^29*x3^14*x4^11*z^19 + x1^40*x2^30*x3^14*x4^11*z^19 - x1^39*x2^31*x3^14*x4^11*z^19 - x1^38*x2^32*x3^14*x4^11*z^19 + x1^37*x2^33*x3^14*x4^11*z^19 - x1^42*x2^27*x3^15*x4^11*z^19 - 2*x1^41*x2^28*x3^15*x4^11*z^19 + x1^40*x2^29*x3^15*x4^11*z^19 - 2*x1^39*x2^30*x3^15*x4^11*z^19 - 2*x1^37*x2^32*x3^15*x4^11*z^19 + x1^36*x2^33*x3^15*x4^11*z^19 + x1^35*x2^34*x3^15*x4^11*z^19 - 2*x1^41*x2^27*x3^16*x4^11*z^19 + x1^40*x2^28*x3^16*x4^11*z^19 + 2*x1^39*x2^29*x3^16*x4^11*z^19 + x1^38*x2^30*x3^16*x4^11*z^19 - x1^37*x2^31*x3^16*x4^11*z^19 + 2*x1^36*x2^32*x3^16*x4^11*z^19 - 2*x1^35*x2^33*x3^16*x4^11*z^19 + x1^34*x2^34*x3^16*x4^11*z^19 + x1^42*x2^25*x3^17*x4^11*z^19 + 2*x1^41*x2^26*x3^17*x4^11*z^19 - 2*x1^40*x2^27*x3^17*x4^11*z^19 + x1^39*x2^28*x3^17*x4^11*z^19 - 3*x1^38*x2^29*x3^17*x4^11*z^19 + 2*x1^37*x2^30*x3^17*x4^11*z^19 - 2*x1^36*x2^31*x3^17*x4^11*z^19 - 2*x1^35*x2^32*x3^17*x4^11*z^19 + 3*x1^34*x2^33*x3^17*x4^11*z^19 - 2*x1^40*x2^26*x3^18*x4^11*z^19 + x1^38*x2^28*x3^18*x4^11*z^19 + 5*x1^37*x2^29*x3^18*x4^11*z^19 + x1^36*x2^30*x3^18*x4^11*z^19 + 4*x1^35*x2^31*x3^18*x4^11*z^19 - 2*x1^34*x2^32*x3^18*x4^11*z^19 - x1^33*x2^33*x3^18*x4^11*z^19 - x1^38*x2^27*x3^19*x4^11*z^19 - 4*x1^37*x2^28*x3^19*x4^11*z^19 + x1^36*x2^29*x3^19*x4^11*z^19 - 3*x1^35*x2^30*x3^19*x4^11*z^19 + 2*x1^33*x2^32*x3^19*x4^11*z^19 + x1^38*x2^26*x3^20*x4^11*z^19 + x1^36*x2^28*x3^20*x4^11*z^19 - x1^35*x2^29*x3^20*x4^11*z^19 + 4*x1^34*x2^30*x3^20*x4^11*z^19 - 2*x1^33*x2^31*x3^20*x4^11*z^19 - x1^32*x2^32*x3^20*x4^11*z^19 + x1^37*x2^26*x3^21*x4^11*z^19 - x1^36*x2^27*x3^21*x4^11*z^19 + x1^35*x2^28*x3^21*x4^11*z^19 - 2*x1^34*x2^29*x3^21*x4^11*z^19 + 2*x1^32*x2^31*x3^21*x4^11*z^19 - 2*x1^34*x2^28*x3^22*x4^11*z^19 + 3*x1^33*x2^29*x3^22*x4^11*z^19 - 3*x1^32*x2^30*x3^22*x4^11*z^19 + x1^34*x2^27*x3^23*x4^11*z^19 - 2*x1^32*x2^29*x3^23*x4^11*z^19 + 3*x1^31*x2^30*x3^23*x4^11*z^19 + x1^32*x2^28*x3^24*x4^11*z^19 - x1^31*x2^29*x3^24*x4^11*z^19 - x1^30*x2^30*x3^24*x4^11*z^19 + x1^30*x2^29*x3^25*x4^11*z^19 + 2*x1^40*x2^31*x3^12*x4^12*z^19 - x1^38*x2^32*x3^13*x4^12*z^19 - x1^37*x2^33*x3^13*x4^12*z^19 + 2*x1^39*x2^30*x3^14*x4^12*z^19 - x1^38*x2^31*x3^14*x4^12*z^19 - x1^35*x2^34*x3^14*x4^12*z^19 + x1^42*x2^26*x3^15*x4^12*z^19 + x1^41*x2^27*x3^15*x4^12*z^19 - x1^40*x2^28*x3^15*x4^12*z^19 + x1^38*x2^30*x3^15*x4^12*z^19 + 3*x1^37*x2^31*x3^15*x4^12*z^19 - x1^36*x2^32*x3^15*x4^12*z^19 + x1^35*x2^33*x3^15*x4^12*z^19 + x1^34*x2^34*x3^15*x4^12*z^19 + x1^40*x2^27*x3^16*x4^12*z^19 - x1^39*x2^28*x3^16*x4^12*z^19 + 2*x1^38*x2^29*x3^16*x4^12*z^19 - 3*x1^37*x2^30*x3^16*x4^12*z^19 - 5*x1^34*x2^33*x3^16*x4^12*z^19 + x1^40*x2^26*x3^17*x4^12*z^19 - 3*x1^39*x2^27*x3^17*x4^12*z^19 - x1^37*x2^29*x3^17*x4^12*z^19 + 3*x1^36*x2^30*x3^17*x4^12*z^19 - 2*x1^35*x2^31*x3^17*x4^12*z^19 + 4*x1^34*x2^32*x3^17*x4^12*z^19 + 2*x1^33*x2^33*x3^17*x4^12*z^19 + x1^39*x2^26*x3^18*x4^12*z^19 + x1^38*x2^27*x3^18*x4^12*z^19 + 2*x1^37*x2^28*x3^18*x4^12*z^19 - 2*x1^36*x2^29*x3^18*x4^12*z^19 - 2*x1^35*x2^30*x3^18*x4^12*z^19 - x1^34*x2^31*x3^18*x4^12*z^19 - 6*x1^33*x2^32*x3^18*x4^12*z^19 + x1^40*x2^24*x3^19*x4^12*z^19 - 3*x1^38*x2^26*x3^19*x4^12*z^19 - x1^36*x2^28*x3^19*x4^12*z^19 + 3*x1^35*x2^29*x3^19*x4^12*z^19 - 2*x1^34*x2^30*x3^19*x4^12*z^19 + 6*x1^33*x2^31*x3^19*x4^12*z^19 + 2*x1^32*x2^32*x3^19*x4^12*z^19 + 2*x1^38*x2^25*x3^20*x4^12*z^19 + x1^37*x2^26*x3^20*x4^12*z^19 + 2*x1^36*x2^27*x3^20*x4^12*z^19 - 2*x1^35*x2^28*x3^20*x4^12*z^19 + x1^34*x2^29*x3^20*x4^12*z^19 - 2*x1^33*x2^30*x3^20*x4^12*z^19 - 6*x1^32*x2^31*x3^20*x4^12*z^19 - 3*x1^37*x2^25*x3^21*x4^12*z^19 - 2*x1^36*x2^26*x3^21*x4^12*z^19 - 2*x1^35*x2^27*x3^21*x4^12*z^19 + 5*x1^34*x2^28*x3^21*x4^12*z^19 - 3*x1^33*x2^29*x3^21*x4^12*z^19 + 6*x1^32*x2^30*x3^21*x4^12*z^19 + 2*x1^31*x2^31*x3^21*x4^12*z^19 + x1^35*x2^26*x3^22*x4^12*z^19 - x1^33*x2^28*x3^22*x4^12*z^19 - 6*x1^31*x2^30*x3^22*x4^12*z^19 - 2*x1^34*x2^26*x3^23*x4^12*z^19 - x1^32*x2^28*x3^23*x4^12*z^19 + 4*x1^31*x2^29*x3^23*x4^12*z^19 + 2*x1^30*x2^30*x3^23*x4^12*z^19 - 2*x1^32*x2^27*x3^24*x4^12*z^19 - x1^31*x2^28*x3^24*x4^12*z^19 - 4*x1^30*x2^29*x3^24*x4^12*z^19 + x1^29*x2^29*x3^25*x4^12*z^19 + x1^38*x2^30*x3^14*x4^13*z^19 + x1^37*x2^31*x3^14*x4^13*z^19 + 2*x1^36*x2^32*x3^14*x4^13*z^19 - 2*x1^38*x2^29*x3^15*x4^13*z^19 + 2*x1^37*x2^30*x3^15*x4^13*z^19 + x1^36*x2^31*x3^15*x4^13*z^19 - x1^35*x2^32*x3^15*x4^13*z^19 + x1^34*x2^33*x3^15*x4^13*z^19 + x1^37*x2^29*x3^16*x4^13*z^19 - 2*x1^36*x2^30*x3^16*x4^13*z^19 + 2*x1^35*x2^31*x3^16*x4^13*z^19 - x1^34*x2^32*x3^16*x4^13*z^19 - 2*x1^33*x2^33*x3^16*x4^13*z^19 + x1^40*x2^25*x3^17*x4^13*z^19 + x1^39*x2^26*x3^17*x4^13*z^19 - x1^37*x2^28*x3^17*x4^13*z^19 + 3*x1^36*x2^29*x3^17*x4^13*z^19 + x1^35*x2^30*x3^17*x4^13*z^19 + 4*x1^33*x2^32*x3^17*x4^13*z^19 + 2*x1^38*x2^26*x3^18*x4^13*z^19 - 3*x1^35*x2^29*x3^18*x4^13*z^19 + x1^34*x2^30*x3^18*x4^13*z^19 - 3*x1^33*x2^31*x3^18*x4^13*z^19 - 2*x1^32*x2^32*x3^18*x4^13*z^19 + x1^39*x2^24*x3^19*x4^13*z^19 - x1^37*x2^26*x3^19*x4^13*z^19 - x1^36*x2^27*x3^19*x4^13*z^19 + x1^35*x2^28*x3^19*x4^13*z^19 + x1^34*x2^29*x3^19*x4^13*z^19 + 2*x1^33*x2^30*x3^19*x4^13*z^19 + 5*x1^32*x2^31*x3^19*x4^13*z^19 + 2*x1^37*x2^25*x3^20*x4^13*z^19 + x1^36*x2^26*x3^20*x4^13*z^19 - x1^35*x2^27*x3^20*x4^13*z^19 - 4*x1^34*x2^28*x3^20*x4^13*z^19 - 6*x1^32*x2^30*x3^20*x4^13*z^19 - 2*x1^31*x2^31*x3^20*x4^13*z^19 + x1^36*x2^25*x3^21*x4^13*z^19 + x1^34*x2^27*x3^21*x4^13*z^19 + x1^32*x2^29*x3^21*x4^13*z^19 + 6*x1^31*x2^30*x3^21*x4^13*z^19 + 2*x1^32*x2^28*x3^22*x4^13*z^19 - 4*x1^31*x2^29*x3^22*x4^13*z^19 - 2*x1^30*x2^30*x3^22*x4^13*z^19 + x1^33*x2^26*x3^23*x4^13*z^19 + 4*x1^30*x2^29*x3^23*x4^13*z^19 + x1^31*x2^27*x3^24*x4^13*z^19 - x1^30*x2^28*x3^24*x4^13*z^19 - x1^29*x2^29*x3^24*x4^13*z^19 + x1^29*x2^28*x3^25*x4^13*z^19 - x1^38*x2^28*x3^15*x4^14*z^19 - x1^37*x2^29*x3^15*x4^14*z^19 + x1^36*x2^30*x3^15*x4^14*z^19 - x1^35*x2^31*x3^15*x4^14*z^19 + x1^37*x2^28*x3^16*x4^14*z^19 - x1^36*x2^29*x3^16*x4^14*z^19 + x1^35*x2^30*x3^16*x4^14*z^19 + x1^34*x2^31*x3^16*x4^14*z^19 - x1^37*x2^27*x3^17*x4^14*z^19 - 2*x1^36*x2^28*x3^17*x4^14*z^19 - 2*x1^34*x2^30*x3^17*x4^14*z^19 - x1^39*x2^24*x3^18*x4^14*z^19 + x1^37*x2^26*x3^18*x4^14*z^19 + x1^36*x2^27*x3^18*x4^14*z^19 - 2*x1^35*x2^28*x3^18*x4^14*z^19 + 2*x1^34*x2^29*x3^18*x4^14*z^19 + 2*x1^33*x2^30*x3^18*x4^14*z^19 - x1^32*x2^31*x3^18*x4^14*z^19 + x1^36*x2^26*x3^19*x4^14*z^19 - x1^35*x2^27*x3^19*x4^14*z^19 - x1^34*x2^28*x3^19*x4^14*z^19 - 2*x1^33*x2^29*x3^19*x4^14*z^19 + x1^32*x2^30*x3^19*x4^14*z^19 + x1^31*x2^31*x3^19*x4^14*z^19 + x1^37*x2^24*x3^20*x4^14*z^19 + x1^36*x2^25*x3^20*x4^14*z^19 + x1^35*x2^26*x3^20*x4^14*z^19 - x1^34*x2^27*x3^20*x4^14*z^19 + 3*x1^33*x2^28*x3^20*x4^14*z^19 - x1^31*x2^30*x3^20*x4^14*z^19 - x1^35*x2^25*x3^21*x4^14*z^19 - 3*x1^32*x2^28*x3^21*x4^14*z^19 + x1^31*x2^29*x3^21*x4^14*z^19 + x1^30*x2^30*x3^21*x4^14*z^19 + x1^34*x2^25*x3^22*x4^14*z^19 + x1^32*x2^27*x3^22*x4^14*z^19 + x1^31*x2^28*x3^22*x4^14*z^19 - x1^30*x2^29*x3^22*x4^14*z^19 - x1^31*x2^27*x3^23*x4^14*z^19 + x1^30*x2^27*x3^24*x4^14*z^19 + x1^37*x2^27*x3^16*x4^15*z^19 + x1^36*x2^28*x3^16*x4^15*z^19 - x1^35*x2^29*x3^16*x4^15*z^19 + x1^34*x2^30*x3^16*x4^15*z^19 - 2*x1^36*x2^27*x3^17*x4^15*z^19 - 2*x1^34*x2^29*x3^17*x4^15*z^19 - x1^33*x2^30*x3^17*x4^15*z^19 + x1^36*x2^26*x3^18*x4^15*z^19 + 2*x1^35*x2^27*x3^18*x4^15*z^19 + 4*x1^33*x2^29*x3^18*x4^15*z^19 - 3*x1^35*x2^26*x3^19*x4^15*z^19 - 4*x1^33*x2^28*x3^19*x4^15*z^19 - 2*x1^32*x2^29*x3^19*x4^15*z^19 - x1^36*x2^24*x3^20*x4^15*z^19 + 2*x1^34*x2^26*x3^20*x4^15*z^19 + x1^33*x2^27*x3^20*x4^15*z^19 + 5*x1^32*x2^28*x3^20*x4^15*z^19 + x1^35*x2^24*x3^21*x4^15*z^19 + 2*x1^33*x2^26*x3^21*x4^15*z^19 - 5*x1^32*x2^27*x3^21*x4^15*z^19 - 2*x1^31*x2^28*x3^21*x4^15*z^19 + x1^33*x2^25*x3^22*x4^15*z^19 + 4*x1^31*x2^27*x3^22*x4^15*z^19 - 2*x1^30*x2^27*x3^23*x4^15*z^19 + x1^35*x2^26*x3^18*x4^16*z^19 + x1^34*x2^27*x3^18*x4^16*z^19 + x1^33*x2^28*x3^18*x4^16*z^19 + x1^33*x2^27*x3^19*x4^16*z^19 - 3*x1^32*x2^28*x3^19*x4^16*z^19 + x1^34*x2^25*x3^20*x4^16*z^19 + x1^32*x2^27*x3^20*x4^16*z^19 + 2*x1^31*x2^28*x3^20*x4^16*z^19 + x1^32*x2^26*x3^21*x4^16*z^19 - 2*x1^31*x2^27*x3^21*x4^16*z^19 - x1^32*x2^25*x3^22*x4^16*z^19 + x1^30*x2^27*x3^22*x4^16*z^19 - x1^30*x2^27*x3^21*x4^17*z^19 + x1^45*x2^31*x3^14*z^18 - 2*x1^44*x2^31*x3^15*z^18 + x1^43*x2^32*x3^15*z^18 + 2*x1^43*x2^31*x3^16*z^18 - x1^42*x2^32*x3^16*z^18 - x1^40*x2^34*x3^16*z^18 - 2*x1^43*x2^30*x3^17*z^18 - x1^42*x2^31*x3^17*z^18 + x1^39*x2^34*x3^17*z^18 + x1^43*x2^29*x3^18*z^18 + 2*x1^40*x2^32*x3^18*z^18 - x1^42*x2^29*x3^19*z^18 - x1^41*x2^30*x3^19*z^18 - x1^40*x2^31*x3^19*z^18 + x1^42*x2^28*x3^20*z^18 + x1^41*x2^29*x3^20*z^18 - x1^40*x2^30*x3^20*z^18 + x1^39*x2^31*x3^20*z^18 - x1^46*x2^32*x3^11*x4*z^18 + 2*x1^45*x2^32*x3^12*x4*z^18 - x1^44*x2^33*x3^12*x4*z^18 - 2*x1^45*x2^31*x3^13*x4*z^18 - 2*x1^44*x2^32*x3^13*x4*z^18 - x1^42*x2^34*x3^13*x4*z^18 + 5*x1^44*x2^31*x3^14*x4*z^18 + x1^42*x2^33*x3^14*x4*z^18 - x1^40*x2^35*x3^14*x4*z^18 - 2*x1^44*x2^30*x3^15*x4*z^18 - 5*x1^43*x2^31*x3^15*x4*z^18 - 3*x1^41*x2^33*x3^15*x4*z^18 + x1^39*x2^35*x3^15*x4*z^18 + 6*x1^43*x2^30*x3^16*x4*z^18 + 2*x1^42*x2^31*x3^16*x4*z^18 + 2*x1^41*x2^32*x3^16*x4*z^18 + 2*x1^40*x2^33*x3^16*x4*z^18 - x1^39*x2^34*x3^16*x4*z^18 - x1^38*x2^35*x3^16*x4*z^18 - 2*x1^43*x2^29*x3^17*x4*z^18 - 6*x1^42*x2^30*x3^17*x4*z^18 - 3*x1^40*x2^32*x3^17*x4*z^18 + x1^39*x2^33*x3^17*x4*z^18 + x1^37*x2^35*x3^17*x4*z^18 + 6*x1^42*x2^29*x3^18*x4*z^18 + 2*x1^41*x2^30*x3^18*x4*z^18 + 2*x1^40*x2^31*x3^18*x4*z^18 + 2*x1^39*x2^32*x3^18*x4*z^18 + x1^38*x2^33*x3^18*x4*z^18 - 2*x1^37*x2^34*x3^18*x4*z^18 - x1^36*x2^35*x3^18*x4*z^18 - 2*x1^42*x2^28*x3^19*x4*z^18 - 4*x1^41*x2^29*x3^19*x4*z^18 - 5*x1^39*x2^31*x3^19*x4*z^18 + x1^38*x2^32*x3^19*x4*z^18 + 2*x1^36*x2^34*x3^19*x4*z^18 + 5*x1^41*x2^28*x3^20*x4*z^18 + x1^40*x2^29*x3^20*x4*z^18 + x1^39*x2^30*x3^20*x4*z^18 + 2*x1^38*x2^31*x3^20*x4*z^18 + x1^37*x2^32*x3^20*x4*z^18 - x1^36*x2^33*x3^20*x4*z^18 - x1^35*x2^34*x3^20*x4*z^18 - x1^41*x2^27*x3^21*x4*z^18 - 2*x1^40*x2^28*x3^21*x4*z^18 - 4*x1^38*x2^30*x3^21*x4*z^18 + 3*x1^40*x2^27*x3^22*x4*z^18 + x1^39*x2^28*x3^22*x4*z^18 + x1^38*x2^29*x3^22*x4*z^18 + x1^37*x2^30*x3^22*x4*z^18 - 2*x1^37*x2^29*x3^23*x4*z^18 - x1^45*x2^32*x3^11*x4^2*z^18 + x1^45*x2^31*x3^12*x4^2*z^18 + x1^44*x2^32*x3^12*x4^2*z^18 - x1^43*x2^33*x3^12*x4^2*z^18 - 4*x1^44*x2^31*x3^13*x4^2*z^18 + x1^43*x2^32*x3^13*x4^2*z^18 + 2*x1^44*x2^30*x3^14*x4^2*z^18 + 4*x1^43*x2^31*x3^14*x4^2*z^18 - x1^42*x2^32*x3^14*x4^2*z^18 + 3*x1^41*x2^33*x3^14*x4^2*z^18 + x1^40*x2^34*x3^14*x4^2*z^18 - 6*x1^43*x2^30*x3^15*x4^2*z^18 - x1^42*x2^31*x3^15*x4^2*z^18 - x1^41*x2^32*x3^15*x4^2*z^18 + x1^39*x2^34*x3^15*x4^2*z^18 + x1^38*x2^35*x3^15*x4^2*z^18 + 2*x1^43*x2^29*x3^16*x4^2*z^18 + 6*x1^42*x2^30*x3^16*x4^2*z^18 + 3*x1^40*x2^32*x3^16*x4^2*z^18 - 3*x1^39*x2^33*x3^16*x4^2*z^18 - x1^38*x2^34*x3^16*x4^2*z^18 - 2*x1^37*x2^35*x3^16*x4^2*z^18 - 6*x1^42*x2^29*x3^17*x4^2*z^18 - 2*x1^41*x2^30*x3^17*x4^2*z^18 - 2*x1^40*x2^31*x3^17*x4^2*z^18 + 3*x1^38*x2^33*x3^17*x4^2*z^18 + 2*x1^37*x2^34*x3^17*x4^2*z^18 + 2*x1^36*x2^35*x3^17*x4^2*z^18 + 2*x1^42*x2^28*x3^18*x4^2*z^18 + 6*x1^41*x2^29*x3^18*x4^2*z^18 + 4*x1^39*x2^31*x3^18*x4^2*z^18 - 3*x1^38*x2^32*x3^18*x4^2*z^18 - 4*x1^36*x2^34*x3^18*x4^2*z^18 - x1^35*x2^35*x3^18*x4^2*z^18 - 6*x1^41*x2^28*x3^19*x4^2*z^18 - 2*x1^40*x2^29*x3^19*x4^2*z^18 - 2*x1^39*x2^30*x3^19*x4^2*z^18 - 2*x1^38*x2^31*x3^19*x4^2*z^18 + x1^37*x2^32*x3^19*x4^2*z^18 + 3*x1^36*x2^33*x3^19*x4^2*z^18 + 3*x1^35*x2^34*x3^19*x4^2*z^18 + 2*x1^41*x2^27*x3^20*x4^2*z^18 + 6*x1^40*x2^28*x3^20*x4^2*z^18 + 4*x1^38*x2^30*x3^20*x4^2*z^18 - 4*x1^35*x2^33*x3^20*x4^2*z^18 - 5*x1^40*x2^27*x3^21*x4^2*z^18 - x1^39*x2^28*x3^21*x4^2*z^18 - x1^38*x2^29*x3^21*x4^2*z^18 - 3*x1^37*x2^30*x3^21*x4^2*z^18 + 2*x1^35*x2^32*x3^21*x4^2*z^18 + x1^34*x2^33*x3^21*x4^2*z^18 + 3*x1^39*x2^27*x3^22*x4^2*z^18 + 2*x1^38*x2^28*x3^22*x4^2*z^18 + 4*x1^37*x2^29*x3^22*x4^2*z^18 - x1^36*x2^30*x3^22*x4^2*z^18 + x1^35*x2^31*x3^22*x4^2*z^18 - x1^34*x2^32*x3^22*x4^2*z^18 - x1^39*x2^26*x3^23*x4^2*z^18 - x1^38*x2^27*x3^23*x4^2*z^18 - 2*x1^37*x2^28*x3^23*x4^2*z^18 - 2*x1^36*x2^29*x3^23*x4^2*z^18 + 2*x1^36*x2^28*x3^24*x4^2*z^18 - x1^35*x2^28*x3^25*x4^2*z^18 - x1^46*x2^29*x3^12*x4^3*z^18 + 2*x1^44*x2^31*x3^12*x4^3*z^18 - x1^43*x2^32*x3^12*x4^3*z^18 + 2*x1^45*x2^29*x3^13*x4^3*z^18 - x1^43*x2^31*x3^13*x4^3*z^18 + x1^42*x2^32*x3^13*x4^3*z^18 + x1^40*x2^34*x3^13*x4^3*z^18 - x1^45*x2^28*x3^14*x4^3*z^18 - 2*x1^44*x2^29*x3^14*x4^3*z^18 + 2*x1^43*x2^30*x3^14*x4^3*z^18 + x1^42*x2^31*x3^14*x4^3*z^18 + x1^41*x2^32*x3^14*x4^3*z^18 - x1^40*x2^33*x3^14*x4^3*z^18 - 2*x1^39*x2^34*x3^14*x4^3*z^18 + 2*x1^44*x2^28*x3^15*x4^3*z^18 - x1^43*x2^29*x3^15*x4^3*z^18 - 2*x1^42*x2^30*x3^15*x4^3*z^18 - x1^40*x2^32*x3^15*x4^3*z^18 + 2*x1^39*x2^33*x3^15*x4^3*z^18 + 2*x1^38*x2^34*x3^15*x4^3*z^18 + x1^37*x2^35*x3^15*x4^3*z^18 - 2*x1^43*x2^28*x3^16*x4^3*z^18 + 2*x1^42*x2^29*x3^16*x4^3*z^18 - 2*x1^41*x2^30*x3^16*x4^3*z^18 + x1^40*x2^31*x3^16*x4^3*z^18 + x1^39*x2^32*x3^16*x4^3*z^18 - 2*x1^37*x2^34*x3^16*x4^3*z^18 - x1^36*x2^35*x3^16*x4^3*z^18 + 2*x1^43*x2^27*x3^17*x4^3*z^18 + x1^42*x2^28*x3^17*x4^3*z^18 - x1^41*x2^29*x3^17*x4^3*z^18 - 3*x1^39*x2^31*x3^17*x4^3*z^18 - x1^37*x2^33*x3^17*x4^3*z^18 + 3*x1^36*x2^34*x3^17*x4^3*z^18 - x1^43*x2^26*x3^18*x4^3*z^18 - 2*x1^42*x2^27*x3^18*x4^3*z^18 + 2*x1^41*x2^28*x3^18*x4^3*z^18 - x1^38*x2^31*x3^18*x4^3*z^18 - x1^36*x2^33*x3^18*x4^3*z^18 - 3*x1^35*x2^34*x3^18*x4^3*z^18 + 2*x1^42*x2^26*x3^19*x4^3*z^18 - x1^40*x2^28*x3^19*x4^3*z^18 + x1^39*x2^29*x3^19*x4^3*z^18 - 2*x1^38*x2^30*x3^19*x4^3*z^18 - x1^37*x2^31*x3^19*x4^3*z^18 - x1^36*x2^32*x3^19*x4^3*z^18 + 3*x1^35*x2^33*x3^19*x4^3*z^18 + x1^34*x2^34*x3^19*x4^3*z^18 - x1^42*x2^25*x3^20*x4^3*z^18 - 2*x1^41*x2^26*x3^20*x4^3*z^18 + 2*x1^40*x2^27*x3^20*x4^3*z^18 + 2*x1^38*x2^29*x3^20*x4^3*z^18 - x1^36*x2^31*x3^20*x4^3*z^18 - 2*x1^35*x2^32*x3^20*x4^3*z^18 - x1^34*x2^33*x3^20*x4^3*z^18 + x1^41*x2^25*x3^21*x4^3*z^18 - x1^40*x2^26*x3^21*x4^3*z^18 - 3*x1^39*x2^27*x3^21*x4^3*z^18 + x1^38*x2^28*x3^21*x4^3*z^18 - x1^35*x2^31*x3^21*x4^3*z^18 + 3*x1^34*x2^32*x3^21*x4^3*z^18 - x1^40*x2^25*x3^22*x4^3*z^18 - x1^39*x2^26*x3^22*x4^3*z^18 - 3*x1^38*x2^27*x3^22*x4^3*z^18 + 2*x1^36*x2^29*x3^22*x4^3*z^18 - x1^35*x2^30*x3^22*x4^3*z^18 - x1^34*x2^31*x3^22*x4^3*z^18 - 2*x1^37*x2^27*x3^23*x4^3*z^18 - x1^36*x2^28*x3^23*x4^3*z^18 + x1^35*x2^29*x3^23*x4^3*z^18 - x1^34*x2^30*x3^23*x4^3*z^18 + x1^33*x2^31*x3^23*x4^3*z^18 + x1^36*x2^27*x3^24*x4^3*z^18 + 2*x1^35*x2^28*x3^24*x4^3*z^18 - x1^46*x2^30*x3^10*x4^4*z^18 + x1^45*x2^31*x3^10*x4^4*z^18 + x1^46*x2^29*x3^11*x4^4*z^18 + x1^45*x2^30*x3^11*x4^4*z^18 - 4*x1^45*x2^29*x3^12*x4^4*z^18 + x1^43*x2^31*x3^12*x4^4*z^18 + 2*x1^42*x2^32*x3^12*x4^4*z^18 + x1^41*x2^33*x3^12*x4^4*z^18 + 2*x1^45*x2^28*x3^13*x4^4*z^18 + 4*x1^44*x2^29*x3^13*x4^4*z^18 - x1^43*x2^30*x3^13*x4^4*z^18 + 2*x1^42*x2^31*x3^13*x4^4*z^18 - x1^41*x2^32*x3^13*x4^4*z^18 - 2*x1^40*x2^33*x3^13*x4^4*z^18 - x1^39*x2^34*x3^13*x4^4*z^18 - 6*x1^44*x2^28*x3^14*x4^4*z^18 - 2*x1^43*x2^29*x3^14*x4^4*z^18 - x1^42*x2^30*x3^14*x4^4*z^18 + 2*x1^40*x2^32*x3^14*x4^4*z^18 + 2*x1^39*x2^33*x3^14*x4^4*z^18 + x1^38*x2^34*x3^14*x4^4*z^18 + 2*x1^44*x2^27*x3^15*x4^4*z^18 + 6*x1^43*x2^28*x3^15*x4^4*z^18 + 2*x1^41*x2^30*x3^15*x4^4*z^18 - 3*x1^40*x2^31*x3^15*x4^4*z^18 - 2*x1^39*x2^32*x3^15*x4^4*z^18 - 3*x1^38*x2^33*x3^15*x4^4*z^18 - x1^37*x2^34*x3^15*x4^4*z^18 - 6*x1^43*x2^27*x3^16*x4^4*z^18 - 2*x1^42*x2^28*x3^16*x4^4*z^18 - 2*x1^41*x2^29*x3^16*x4^4*z^18 + 2*x1^38*x2^32*x3^16*x4^4*z^18 + 3*x1^37*x2^33*x3^16*x4^4*z^18 + 2*x1^43*x2^26*x3^17*x4^4*z^18 + 6*x1^42*x2^27*x3^17*x4^4*z^18 + 4*x1^40*x2^29*x3^17*x4^4*z^18 - 4*x1^39*x2^30*x3^17*x4^4*z^18 - 6*x1^37*x2^32*x3^17*x4^4*z^18 - x1^36*x2^33*x3^17*x4^4*z^18 - 6*x1^42*x2^26*x3^18*x4^4*z^18 - 2*x1^41*x2^27*x3^18*x4^4*z^18 - 2*x1^40*x2^28*x3^18*x4^4*z^18 + 2*x1^38*x2^30*x3^18*x4^4*z^18 + 2*x1^37*x2^31*x3^18*x4^4*z^18 + 6*x1^36*x2^32*x3^18*x4^4*z^18 + x1^34*x2^34*x3^18*x4^4*z^18 + 2*x1^42*x2^25*x3^19*x4^4*z^18 + 6*x1^41*x2^26*x3^19*x4^4*z^18 + 4*x1^39*x2^28*x3^19*x4^4*z^18 - 2*x1^38*x2^29*x3^19*x4^4*z^18 + x1^37*x2^30*x3^19*x4^4*z^18 - 5*x1^36*x2^31*x3^19*x4^4*z^18 - 2*x1^35*x2^32*x3^19*x4^4*z^18 + x1^34*x2^33*x3^19*x4^4*z^18 - 5*x1^41*x2^25*x3^20*x4^4*z^18 - 2*x1^40*x2^26*x3^20*x4^4*z^18 - 2*x1^39*x2^27*x3^20*x4^4*z^18 - x1^38*x2^28*x3^20*x4^4*z^18 + x1^37*x2^29*x3^20*x4^4*z^18 + x1^36*x2^30*x3^20*x4^4*z^18 + 3*x1^35*x2^31*x3^20*x4^4*z^18 - 2*x1^34*x2^32*x3^20*x4^4*z^18 + 5*x1^40*x2^25*x3^21*x4^4*z^18 + x1^39*x2^26*x3^21*x4^4*z^18 + 4*x1^38*x2^27*x3^21*x4^4*z^18 + x1^36*x2^29*x3^21*x4^4*z^18 - 3*x1^35*x2^30*x3^21*x4^4*z^18 - x1^34*x2^31*x3^21*x4^4*z^18 + x1^33*x2^32*x3^21*x4^4*z^18 - x1^40*x2^24*x3^22*x4^4*z^18 - 3*x1^39*x2^25*x3^22*x4^4*z^18 - x1^38*x2^26*x3^22*x4^4*z^18 - x1^36*x2^28*x3^22*x4^4*z^18 + 2*x1^35*x2^29*x3^22*x4^4*z^18 + x1^34*x2^30*x3^22*x4^4*z^18 - 2*x1^33*x2^31*x3^22*x4^4*z^18 + x1^39*x2^24*x3^23*x4^4*z^18 + 2*x1^38*x2^25*x3^23*x4^4*z^18 + x1^37*x2^26*x3^23*x4^4*z^18 - x1^36*x2^27*x3^23*x4^4*z^18 - x1^35*x2^28*x3^23*x4^4*z^18 - x1^37*x2^25*x3^24*x4^4*z^18 - x1^33*x2^29*x3^24*x4^4*z^18 - 2*x1^32*x2^30*x3^24*x4^4*z^18 - x1^35*x2^26*x3^25*x4^4*z^18 - x1^34*x2^27*x3^25*x4^4*z^18 - x1^45*x2^30*x3^10*x4^5*z^18 + 2*x1^45*x2^29*x3^11*x4^5*z^18 - x1^44*x2^30*x3^11*x4^5*z^18 - 2*x1^43*x2^31*x3^11*x4^5*z^18 - x1^45*x2^28*x3^12*x4^5*z^18 - 4*x1^44*x2^29*x3^12*x4^5*z^18 + x1^43*x2^30*x3^12*x4^5*z^18 + 4*x1^44*x2^28*x3^13*x4^5*z^18 + 2*x1^43*x2^29*x3^13*x4^5*z^18 + x1^41*x2^31*x3^13*x4^5*z^18 - 2*x1^44*x2^27*x3^14*x4^5*z^18 - 6*x1^43*x2^28*x3^14*x4^5*z^18 - 3*x1^41*x2^30*x3^14*x4^5*z^18 + 3*x1^40*x2^31*x3^14*x4^5*z^18 + 2*x1^39*x2^32*x3^14*x4^5*z^18 + 2*x1^38*x2^33*x3^14*x4^5*z^18 + x1^37*x2^34*x3^14*x4^5*z^18 + 6*x1^43*x2^27*x3^15*x4^5*z^18 + x1^42*x2^28*x3^15*x4^5*z^18 - 4*x1^39*x2^31*x3^15*x4^5*z^18 - 3*x1^38*x2^32*x3^15*x4^5*z^18 - 2*x1^37*x2^33*x3^15*x4^5*z^18 - 2*x1^43*x2^26*x3^16*x4^5*z^18 - 6*x1^42*x2^27*x3^16*x4^5*z^18 + 2*x1^41*x2^28*x3^16*x4^5*z^18 - 3*x1^40*x2^29*x3^16*x4^5*z^18 + 5*x1^39*x2^30*x3^16*x4^5*z^18 + x1^38*x2^31*x3^16*x4^5*z^18 + 4*x1^37*x2^32*x3^16*x4^5*z^18 - x1^36*x2^33*x3^16*x4^5*z^18 + 6*x1^42*x2^26*x3^17*x4^5*z^18 + x1^41*x2^27*x3^17*x4^5*z^18 - 3*x1^38*x2^30*x3^17*x4^5*z^18 - 5*x1^36*x2^32*x3^17*x4^5*z^18 + 2*x1^35*x2^33*x3^17*x4^5*z^18 - x1^34*x2^34*x3^17*x4^5*z^18 - 2*x1^42*x2^25*x3^18*x4^5*z^18 - 6*x1^41*x2^26*x3^18*x4^5*z^18 + 2*x1^40*x2^27*x3^18*x4^5*z^18 - 4*x1^39*x2^28*x3^18*x4^5*z^18 + 4*x1^38*x2^29*x3^18*x4^5*z^18 + 6*x1^36*x2^31*x3^18*x4^5*z^18 + 2*x1^35*x2^32*x3^18*x4^5*z^18 - 2*x1^34*x2^33*x3^18*x4^5*z^18 + 4*x1^41*x2^25*x3^19*x4^5*z^18 + 2*x1^40*x2^26*x3^19*x4^5*z^18 - 4*x1^37*x2^29*x3^19*x4^5*z^18 - x1^36*x2^30*x3^19*x4^5*z^18 - 6*x1^35*x2^31*x3^19*x4^5*z^18 + 2*x1^34*x2^32*x3^19*x4^5*z^18 + x1^33*x2^33*x3^19*x4^5*z^18 - x1^41*x2^24*x3^20*x4^5*z^18 - 4*x1^40*x2^25*x3^20*x4^5*z^18 - 3*x1^38*x2^27*x3^20*x4^5*z^18 + 5*x1^37*x2^28*x3^20*x4^5*z^18 + 6*x1^35*x2^30*x3^20*x4^5*z^18 + 2*x1^34*x2^31*x3^20*x4^5*z^18 - x1^33*x2^32*x3^20*x4^5*z^18 + x1^40*x2^24*x3^21*x4^5*z^18 - 2*x1^36*x2^28*x3^21*x4^5*z^18 - x1^35*x2^29*x3^21*x4^5*z^18 - 4*x1^34*x2^30*x3^21*x4^5*z^18 + 3*x1^33*x2^31*x3^21*x4^5*z^18 - x1^39*x2^24*x3^22*x4^5*z^18 + x1^38*x2^25*x3^22*x4^5*z^18 - 2*x1^37*x2^26*x3^22*x4^5*z^18 + 2*x1^36*x2^27*x3^22*x4^5*z^18 + 4*x1^34*x2^29*x3^22*x4^5*z^18 + x1^33*x2^30*x3^22*x4^5*z^18 - 2*x1^32*x2^31*x3^22*x4^5*z^18 + x1^38*x2^24*x3^23*x4^5*z^18 + x1^34*x2^28*x3^23*x4^5*z^18 - x1^33*x2^29*x3^23*x4^5*z^18 + x1^32*x2^30*x3^23*x4^5*z^18 + x1^33*x2^28*x3^24*x4^5*z^18 - x1^31*x2^30*x3^24*x4^5*z^18 + x1^35*x2^25*x3^25*x4^5*z^18 + x1^32*x2^28*x3^25*x4^5*z^18 + x1^31*x2^29*x3^25*x4^5*z^18 - x1^30*x2^29*x3^26*x4^5*z^18 + x1^44*x2^29*x3^11*x4^6*z^18 + x1^41*x2^32*x3^11*x4^6*z^18 + x1^44*x2^28*x3^12*x4^6*z^18 - x1^43*x2^29*x3^12*x4^6*z^18 - 2*x1^42*x2^30*x3^12*x4^6*z^18 + x1^41*x2^31*x3^12*x4^6*z^18 - x1^39*x2^33*x3^12*x4^6*z^18 + x1^44*x2^27*x3^13*x4^6*z^18 + x1^43*x2^28*x3^13*x4^6*z^18 - 3*x1^42*x2^29*x3^13*x4^6*z^18 - 2*x1^40*x2^31*x3^13*x4^6*z^18 - x1^39*x2^32*x3^13*x4^6*z^18 - x1^38*x2^33*x3^13*x4^6*z^18 + x1^42*x2^28*x3^14*x4^6*z^18 + 3*x1^41*x2^29*x3^14*x4^6*z^18 - 3*x1^40*x2^30*x3^14*x4^6*z^18 + 2*x1^39*x2^31*x3^14*x4^6*z^18 + x1^38*x2^32*x3^14*x4^6*z^18 + x1^37*x2^33*x3^14*x4^6*z^18 - x1^36*x2^34*x3^14*x4^6*z^18 + x1^43*x2^26*x3^15*x4^6*z^18 + 2*x1^42*x2^27*x3^15*x4^6*z^18 - 2*x1^41*x2^28*x3^15*x4^6*z^18 - x1^40*x2^29*x3^15*x4^6*z^18 - 2*x1^39*x2^30*x3^15*x4^6*z^18 + 2*x1^38*x2^31*x3^15*x4^6*z^18 + x1^35*x2^34*x3^15*x4^6*z^18 - 2*x1^42*x2^26*x3^16*x4^6*z^18 + 2*x1^41*x2^27*x3^16*x4^6*z^18 + 4*x1^40*x2^28*x3^16*x4^6*z^18 + 2*x1^38*x2^30*x3^16*x4^6*z^18 - x1^37*x2^31*x3^16*x4^6*z^18 + x1^36*x2^32*x3^16*x4^6*z^18 - x1^35*x2^33*x3^16*x4^6*z^18 + 2*x1^41*x2^26*x3^17*x4^6*z^18 - 4*x1^40*x2^27*x3^17*x4^6*z^18 + x1^39*x2^28*x3^17*x4^6*z^18 - 2*x1^38*x2^29*x3^17*x4^6*z^18 - x1^36*x2^31*x3^17*x4^6*z^18 + 2*x1^35*x2^32*x3^17*x4^6*z^18 + x1^34*x2^33*x3^17*x4^6*z^18 - 2*x1^41*x2^25*x3^18*x4^6*z^18 + 3*x1^39*x2^27*x3^18*x4^6*z^18 + 4*x1^37*x2^29*x3^18*x4^6*z^18 - 2*x1^36*x2^30*x3^18*x4^6*z^18 + 2*x1^35*x2^31*x3^18*x4^6*z^18 - 4*x1^34*x2^32*x3^18*x4^6*z^18 + x1^33*x2^33*x3^18*x4^6*z^18 + x1^41*x2^24*x3^19*x4^6*z^18 + 2*x1^40*x2^25*x3^19*x4^6*z^18 - 4*x1^39*x2^26*x3^19*x4^6*z^18 - 2*x1^37*x2^28*x3^19*x4^6*z^18 - x1^35*x2^30*x3^19*x4^6*z^18 + 4*x1^33*x2^32*x3^19*x4^6*z^18 + x1^39*x2^25*x3^20*x4^6*z^18 + 4*x1^38*x2^26*x3^20*x4^6*z^18 + 4*x1^36*x2^28*x3^20*x4^6*z^18 - x1^35*x2^29*x3^20*x4^6*z^18 + 2*x1^34*x2^30*x3^20*x4^6*z^18 - 4*x1^33*x2^31*x3^20*x4^6*z^18 - 2*x1^32*x2^32*x3^20*x4^6*z^18 - 3*x1^38*x2^25*x3^21*x4^6*z^18 - x1^37*x2^26*x3^21*x4^6*z^18 - 5*x1^36*x2^27*x3^21*x4^6*z^18 - x1^34*x2^29*x3^21*x4^6*z^18 + 3*x1^32*x2^31*x3^21*x4^6*z^18 - x1^38*x2^24*x3^22*x4^6*z^18 + 3*x1^37*x2^25*x3^22*x4^6*z^18 - 2*x1^34*x2^28*x3^22*x4^6*z^18 + 2*x1^33*x2^29*x3^22*x4^6*z^18 - 3*x1^32*x2^30*x3^22*x4^6*z^18 + x1^37*x2^24*x3^23*x4^6*z^18 - x1^35*x2^26*x3^23*x4^6*z^18 - 2*x1^34*x2^27*x3^23*x4^6*z^18 - x1^33*x2^28*x3^23*x4^6*z^18 + x1^32*x2^29*x3^23*x4^6*z^18 + 2*x1^31*x2^30*x3^23*x4^6*z^18 - x1^36*x2^24*x3^24*x4^6*z^18 + x1^35*x2^25*x3^24*x4^6*z^18 + 2*x1^34*x2^26*x3^24*x4^6*z^18 + x1^33*x2^27*x3^24*x4^6*z^18 - 2*x1^31*x2^29*x3^24*x4^6*z^18 - x1^32*x2^27*x3^25*x4^6*z^18 + x1^31*x2^28*x3^25*x4^6*z^18 + 2*x1^30*x2^29*x3^25*x4^6*z^18 + x1^44*x2^29*x3^10*x4^7*z^18 - x1^43*x2^30*x3^10*x4^7*z^18 + x1^41*x2^32*x3^10*x4^7*z^18 + x1^39*x2^34*x3^10*x4^7*z^18 - 2*x1^39*x2^33*x3^11*x4^7*z^18 - x1^38*x2^34*x3^11*x4^7*z^18 + 2*x1^43*x2^28*x3^12*x4^7*z^18 - x1^41*x2^30*x3^12*x4^7*z^18 + x1^39*x2^32*x3^12*x4^7*z^18 + 2*x1^38*x2^33*x3^12*x4^7*z^18 + x1^37*x2^34*x3^12*x4^7*z^18 - x1^43*x2^27*x3^13*x4^7*z^18 + x1^42*x2^28*x3^13*x4^7*z^18 - x1^41*x2^29*x3^13*x4^7*z^18 - 2*x1^36*x2^34*x3^13*x4^7*z^18 + 2*x1^42*x2^27*x3^14*x4^7*z^18 + 2*x1^40*x2^29*x3^14*x4^7*z^18 - x1^39*x2^30*x3^14*x4^7*z^18 + 2*x1^36*x2^33*x3^14*x4^7*z^18 + 2*x1^35*x2^34*x3^14*x4^7*z^18 - x1^42*x2^26*x3^15*x4^7*z^18 - x1^38*x2^30*x3^15*x4^7*z^18 - 2*x1^37*x2^31*x3^15*x4^7*z^18 + x1^36*x2^32*x3^15*x4^7*z^18 - 2*x1^35*x2^33*x3^15*x4^7*z^18 - x1^34*x2^34*x3^15*x4^7*z^18 - x1^35*x2^32*x3^16*x4^7*z^18 + 2*x1^34*x2^33*x3^16*x4^7*z^18 - x1^40*x2^26*x3^17*x4^7*z^18 - x1^33*x2^33*x3^17*x4^7*z^18 + 2*x1^39*x2^26*x3^18*x4^7*z^18 - x1^38*x2^27*x3^18*x4^7*z^18 - x1^40*x2^24*x3^19*x4^7*z^18 - 2*x1^38*x2^26*x3^19*x4^7*z^18 - x1^36*x2^28*x3^19*x4^7*z^18 + x1^39*x2^24*x3^20*x4^7*z^18 + 2*x1^38*x2^25*x3^20*x4^7*z^18 + 2*x1^37*x2^26*x3^20*x4^7*z^18 + x1^36*x2^27*x3^20*x4^7*z^18 - x1^34*x2^29*x3^20*x4^7*z^18 - x1^39*x2^23*x3^21*x4^7*z^18 - x1^38*x2^24*x3^21*x4^7*z^18 - 3*x1^37*x2^25*x3^21*x4^7*z^18 - x1^36*x2^26*x3^21*x4^7*z^18 - x1^35*x2^27*x3^21*x4^7*z^18 + x1^33*x2^29*x3^21*x4^7*z^18 + x1^38*x2^23*x3^22*x4^7*z^18 + 2*x1^37*x2^24*x3^22*x4^7*z^18 + 2*x1^36*x2^25*x3^22*x4^7*z^18 + 3*x1^35*x2^26*x3^22*x4^7*z^18 + 2*x1^34*x2^27*x3^22*x4^7*z^18 + x1^33*x2^28*x3^22*x4^7*z^18 - 2*x1^36*x2^24*x3^23*x4^7*z^18 - x1^35*x2^25*x3^23*x4^7*z^18 - x1^34*x2^26*x3^23*x4^7*z^18 - 2*x1^32*x2^28*x3^23*x4^7*z^18 + x1^35*x2^24*x3^24*x4^7*z^18 + x1^34*x2^25*x3^24*x4^7*z^18 + 2*x1^32*x2^27*x3^24*x4^7*z^18 + x1^31*x2^28*x3^24*x4^7*z^18 - x1^30*x2^29*x3^24*x4^7*z^18 - x1^33*x2^25*x3^25*x4^7*z^18 - x1^32*x2^26*x3^25*x4^7*z^18 + x1^30*x2^28*x3^25*x4^7*z^18 - x1^29*x2^28*x3^26*x4^7*z^18 - x1^41*x2^31*x3^10*x4^8*z^18 - x1^40*x2^32*x3^10*x4^8*z^18 + x1^43*x2^28*x3^11*x4^8*z^18 + x1^42*x2^29*x3^11*x4^8*z^18 - x1^41*x2^30*x3^11*x4^8*z^18 - x1^39*x2^32*x3^11*x4^8*z^18 - x1^38*x2^33*x3^11*x4^8*z^18 - x1^37*x2^34*x3^11*x4^8*z^18 - x1^43*x2^27*x3^12*x4^8*z^18 + x1^40*x2^30*x3^12*x4^8*z^18 - x1^39*x2^31*x3^12*x4^8*z^18 + x1^38*x2^32*x3^12*x4^8*z^18 + 2*x1^36*x2^34*x3^12*x4^8*z^18 - x1^42*x2^27*x3^13*x4^8*z^18 + x1^41*x2^28*x3^13*x4^8*z^18 + 2*x1^40*x2^29*x3^13*x4^8*z^18 - x1^37*x2^32*x3^13*x4^8*z^18 - x1^36*x2^33*x3^13*x4^8*z^18 - 3*x1^35*x2^34*x3^13*x4^8*z^18 - 2*x1^42*x2^26*x3^14*x4^8*z^18 - x1^41*x2^27*x3^14*x4^8*z^18 - 3*x1^40*x2^28*x3^14*x4^8*z^18 - x1^39*x2^29*x3^14*x4^8*z^18 + 3*x1^37*x2^31*x3^14*x4^8*z^18 - x1^36*x2^32*x3^14*x4^8*z^18 + 3*x1^35*x2^33*x3^14*x4^8*z^18 + 2*x1^34*x2^34*x3^14*x4^8*z^18 + x1^41*x2^26*x3^15*x4^8*z^18 + 2*x1^40*x2^27*x3^15*x4^8*z^18 - x1^38*x2^29*x3^15*x4^8*z^18 - x1^37*x2^30*x3^15*x4^8*z^18 - 2*x1^36*x2^31*x3^15*x4^8*z^18 - 2*x1^35*x2^32*x3^15*x4^8*z^18 - 3*x1^34*x2^33*x3^15*x4^8*z^18 - 2*x1^41*x2^25*x3^16*x4^8*z^18 - x1^40*x2^26*x3^16*x4^8*z^18 - 3*x1^39*x2^27*x3^16*x4^8*z^18 + 4*x1^36*x2^30*x3^16*x4^8*z^18 + x1^35*x2^31*x3^16*x4^8*z^18 + 4*x1^34*x2^32*x3^16*x4^8*z^18 + x1^33*x2^33*x3^16*x4^8*z^18 + x1^40*x2^25*x3^17*x4^8*z^18 + 2*x1^39*x2^26*x3^17*x4^8*z^18 + x1^38*x2^27*x3^17*x4^8*z^18 - x1^37*x2^28*x3^17*x4^8*z^18 - x1^36*x2^29*x3^17*x4^8*z^18 - 3*x1^35*x2^30*x3^17*x4^8*z^18 - x1^34*x2^31*x3^17*x4^8*z^18 - 4*x1^33*x2^32*x3^17*x4^8*z^18 - 2*x1^40*x2^24*x3^18*x4^8*z^18 - 3*x1^38*x2^26*x3^18*x4^8*z^18 - x1^37*x2^27*x3^18*x4^8*z^18 - 3*x1^36*x2^28*x3^18*x4^8*z^18 + 2*x1^35*x2^29*x3^18*x4^8*z^18 + 2*x1^34*x2^30*x3^18*x4^8*z^18 + 4*x1^33*x2^31*x3^18*x4^8*z^18 + 2*x1^32*x2^32*x3^18*x4^8*z^18 + x1^40*x2^23*x3^19*x4^8*z^18 + 2*x1^39*x2^24*x3^19*x4^8*z^18 + 5*x1^37*x2^26*x3^19*x4^8*z^18 + x1^36*x2^27*x3^19*x4^8*z^18 - 4*x1^34*x2^29*x3^19*x4^8*z^18 - 3*x1^33*x2^30*x3^19*x4^8*z^18 - 4*x1^32*x2^31*x3^19*x4^8*z^18 - 2*x1^39*x2^23*x3^20*x4^8*z^18 - 2*x1^38*x2^24*x3^20*x4^8*z^18 - x1^37*x2^25*x3^20*x4^8*z^18 - x1^36*x2^26*x3^20*x4^8*z^18 + 2*x1^35*x2^27*x3^20*x4^8*z^18 + 5*x1^34*x2^28*x3^20*x4^8*z^18 + 2*x1^33*x2^29*x3^20*x4^8*z^18 + 4*x1^32*x2^30*x3^20*x4^8*z^18 + x1^31*x2^31*x3^20*x4^8*z^18 + 2*x1^38*x2^23*x3^21*x4^8*z^18 + x1^37*x2^24*x3^21*x4^8*z^18 - 2*x1^35*x2^26*x3^21*x4^8*z^18 - 2*x1^33*x2^28*x3^21*x4^8*z^18 - x1^32*x2^29*x3^21*x4^8*z^18 - 4*x1^31*x2^30*x3^21*x4^8*z^18 - x1^37*x2^23*x3^22*x4^8*z^18 - x1^36*x2^24*x3^22*x4^8*z^18 + x1^32*x2^28*x3^22*x4^8*z^18 + 2*x1^31*x2^29*x3^22*x4^8*z^18 + x1^30*x2^30*x3^22*x4^8*z^18 + x1^35*x2^24*x3^23*x4^8*z^18 - x1^34*x2^25*x3^23*x4^8*z^18 - x1^33*x2^26*x3^23*x4^8*z^18 + x1^31*x2^28*x3^23*x4^8*z^18 - x1^30*x2^29*x3^23*x4^8*z^18 + 2*x1^32*x2^26*x3^24*x4^8*z^18 + 2*x1^31*x2^27*x3^24*x4^8*z^18 + x1^30*x2^28*x3^24*x4^8*z^18 + x1^32*x2^25*x3^25*x4^8*z^18 - x1^31*x2^26*x3^25*x4^8*z^18 + x1^30*x2^26*x3^26*x4^8*z^18 + x1^40*x2^31*x3^10*x4^9*z^18 - x1^39*x2^31*x3^11*x4^9*z^18 - x1^37*x2^33*x3^11*x4^9*z^18 - x1^42*x2^27*x3^12*x4^9*z^18 - x1^41*x2^28*x3^12*x4^9*z^18 - x1^40*x2^29*x3^12*x4^9*z^18 + x1^37*x2^32*x3^12*x4^9*z^18 + x1^36*x2^33*x3^12*x4^9*z^18 + x1^42*x2^26*x3^13*x4^9*z^18 + 2*x1^40*x2^28*x3^13*x4^9*z^18 - 3*x1^39*x2^29*x3^13*x4^9*z^18 - 3*x1^38*x2^30*x3^13*x4^9*z^18 - x1^37*x2^31*x3^13*x4^9*z^18 - x1^36*x2^32*x3^13*x4^9*z^18 - 2*x1^35*x2^33*x3^13*x4^9*z^18 - 3*x1^41*x2^26*x3^14*x4^9*z^18 - x1^40*x2^27*x3^14*x4^9*z^18 + 2*x1^38*x2^29*x3^14*x4^9*z^18 + x1^37*x2^30*x3^14*x4^9*z^18 + 2*x1^36*x2^31*x3^14*x4^9*z^18 + x1^35*x2^32*x3^14*x4^9*z^18 + 2*x1^34*x2^33*x3^14*x4^9*z^18 + 3*x1^41*x2^25*x3^15*x4^9*z^18 + x1^40*x2^26*x3^15*x4^9*z^18 + x1^39*x2^27*x3^15*x4^9*z^18 - x1^37*x2^29*x3^15*x4^9*z^18 - 5*x1^36*x2^30*x3^15*x4^9*z^18 - x1^35*x2^31*x3^15*x4^9*z^18 - x1^34*x2^32*x3^15*x4^9*z^18 - x1^33*x2^33*x3^15*x4^9*z^18 - 4*x1^40*x2^25*x3^16*x4^9*z^18 - 3*x1^39*x2^26*x3^16*x4^9*z^18 - x1^38*x2^27*x3^16*x4^9*z^18 + 2*x1^37*x2^28*x3^16*x4^9*z^18 + x1^36*x2^29*x3^16*x4^9*z^18 + 4*x1^35*x2^30*x3^16*x4^9*z^18 + 2*x1^33*x2^32*x3^16*x4^9*z^18 + 4*x1^40*x2^24*x3^17*x4^9*z^18 + x1^39*x2^25*x3^17*x4^9*z^18 + 2*x1^38*x2^26*x3^17*x4^9*z^18 - x1^37*x2^27*x3^17*x4^9*z^18 + x1^36*x2^28*x3^17*x4^9*z^18 - 2*x1^35*x2^29*x3^17*x4^9*z^18 - 4*x1^34*x2^30*x3^17*x4^9*z^18 - 2*x1^33*x2^31*x3^17*x4^9*z^18 - x1^32*x2^32*x3^17*x4^9*z^18 - x1^40*x2^23*x3^18*x4^9*z^18 - 4*x1^39*x2^24*x3^18*x4^9*z^18 + x1^38*x2^25*x3^18*x4^9*z^18 - 4*x1^37*x2^26*x3^18*x4^9*z^18 + 2*x1^36*x2^27*x3^18*x4^9*z^18 + x1^35*x2^28*x3^18*x4^9*z^18 + 5*x1^34*x2^29*x3^18*x4^9*z^18 + 2*x1^33*x2^30*x3^18*x4^9*z^18 + 2*x1^32*x2^31*x3^18*x4^9*z^18 + 4*x1^39*x2^23*x3^19*x4^9*z^18 + 3*x1^38*x2^24*x3^19*x4^9*z^18 + 2*x1^37*x2^25*x3^19*x4^9*z^18 + x1^36*x2^26*x3^19*x4^9*z^18 - x1^34*x2^28*x3^19*x4^9*z^18 - 4*x1^33*x2^29*x3^19*x4^9*z^18 - 2*x1^32*x2^30*x3^19*x4^9*z^18 - x1^31*x2^31*x3^19*x4^9*z^18 - 4*x1^38*x2^23*x3^20*x4^9*z^18 - x1^37*x2^24*x3^20*x4^9*z^18 - 3*x1^36*x2^25*x3^20*x4^9*z^18 + x1^35*x2^26*x3^20*x4^9*z^18 - x1^34*x2^27*x3^20*x4^9*z^18 + 3*x1^33*x2^28*x3^20*x4^9*z^18 + 3*x1^32*x2^29*x3^20*x4^9*z^18 + 2*x1^31*x2^30*x3^20*x4^9*z^18 + x1^37*x2^23*x3^21*x4^9*z^18 + x1^36*x2^24*x3^21*x4^9*z^18 + x1^35*x2^25*x3^21*x4^9*z^18 - 2*x1^33*x2^27*x3^21*x4^9*z^18 - 2*x1^32*x2^28*x3^21*x4^9*z^18 - x1^31*x2^29*x3^21*x4^9*z^18 - x1^36*x2^23*x3^22*x4^9*z^18 - 2*x1^35*x2^24*x3^22*x4^9*z^18 + x1^34*x2^25*x3^22*x4^9*z^18 - x1^33*x2^26*x3^22*x4^9*z^18 + 3*x1^32*x2^27*x3^22*x4^9*z^18 - x1^31*x2^28*x3^22*x4^9*z^18 + x1^30*x2^29*x3^22*x4^9*z^18 + x1^35*x2^23*x3^23*x4^9*z^18 + x1^33*x2^25*x3^23*x4^9*z^18 - 4*x1^31*x2^27*x3^23*x4^9*z^18 + x1^30*x2^28*x3^23*x4^9*z^18 - x1^29*x2^29*x3^23*x4^9*z^18 + x1^33*x2^24*x3^24*x4^9*z^18 - 2*x1^32*x2^25*x3^24*x4^9*z^18 + 2*x1^31*x2^26*x3^24*x4^9*z^18 + x1^30*x2^27*x3^24*x4^9*z^18 - x1^29*x2^28*x3^24*x4^9*z^18 + x1^31*x2^25*x3^25*x4^9*z^18 - 2*x1^30*x2^26*x3^25*x4^9*z^18 + x1^28*x2^28*x3^25*x4^9*z^18 - x1^39*x2^31*x3^10*x4^10*z^18 - x1^38*x2^31*x3^11*x4^10*z^18 + x1^38*x2^30*x3^12*x4^10*z^18 - x1^37*x2^30*x3^13*x4^10*z^18 + x1^35*x2^32*x3^13*x4^10*z^18 + x1^40*x2^26*x3^14*x4^10*z^18 - 2*x1^39*x2^27*x3^14*x4^10*z^18 + x1^37*x2^29*x3^14*x4^10*z^18 - 2*x1^35*x2^31*x3^14*x4^10*z^18 + 2*x1^34*x2^32*x3^14*x4^10*z^18 + x1^33*x2^33*x3^14*x4^10*z^18 + x1^40*x2^25*x3^15*x4^10*z^18 + x1^39*x2^26*x3^15*x4^10*z^18 + x1^37*x2^28*x3^15*x4^10*z^18 - x1^36*x2^29*x3^15*x4^10*z^18 + x1^35*x2^30*x3^15*x4^10*z^18 - 2*x1^33*x2^32*x3^15*x4^10*z^18 - x1^40*x2^24*x3^16*x4^10*z^18 + x1^39*x2^25*x3^16*x4^10*z^18 - x1^38*x2^26*x3^16*x4^10*z^18 + x1^37*x2^27*x3^16*x4^10*z^18 - x1^36*x2^28*x3^16*x4^10*z^18 + x1^34*x2^30*x3^16*x4^10*z^18 + x1^33*x2^31*x3^16*x4^10*z^18 - x1^40*x2^23*x3^17*x4^10*z^18 + 2*x1^39*x2^24*x3^17*x4^10*z^18 + 2*x1^38*x2^25*x3^17*x4^10*z^18 + x1^36*x2^27*x3^17*x4^10*z^18 - x1^38*x2^24*x3^18*x4^10*z^18 + 3*x1^36*x2^26*x3^18*x4^10*z^18 - x1^35*x2^27*x3^18*x4^10*z^18 + x1^38*x2^23*x3^19*x4^10*z^18 - 2*x1^34*x2^27*x3^19*x4^10*z^18 - x1^33*x2^28*x3^19*x4^10*z^18 - x1^38*x2^22*x3^20*x4^10*z^18 - x1^37*x2^23*x3^20*x4^10*z^18 + x1^35*x2^25*x3^20*x4^10*z^18 + x1^34*x2^26*x3^20*x4^10*z^18 + 2*x1^33*x2^27*x3^20*x4^10*z^18 + x1^32*x2^28*x3^20*x4^10*z^18 + x1^37*x2^22*x3^21*x4^10*z^18 - x1^35*x2^24*x3^21*x4^10*z^18 - 3*x1^34*x2^25*x3^21*x4^10*z^18 - x1^33*x2^26*x3^21*x4^10*z^18 - 3*x1^32*x2^27*x3^21*x4^10*z^18 - x1^31*x2^28*x3^21*x4^10*z^18 + x1^34*x2^24*x3^22*x4^10*z^18 + x1^32*x2^26*x3^22*x4^10*z^18 + 3*x1^31*x2^27*x3^22*x4^10*z^18 + x1^34*x2^23*x3^23*x4^10*z^18 - x1^33*x2^24*x3^23*x4^10*z^18 - 2*x1^31*x2^26*x3^23*x4^10*z^18 - x1^30*x2^27*x3^23*x4^10*z^18 + x1^32*x2^24*x3^24*x4^10*z^18 + x1^31*x2^25*x3^24*x4^10*z^18 + 2*x1^30*x2^26*x3^24*x4^10*z^18 + x1^28*x2^28*x3^24*x4^10*z^18 - x1^29*x2^26*x3^25*x4^10*z^18 + x1^38*x2^30*x3^11*x4^11*z^18 + x1^37*x2^31*x3^11*x4^11*z^18 - x1^38*x2^29*x3^12*x4^11*z^18 + x1^37*x2^30*x3^12*x4^11*z^18 - x1^35*x2^32*x3^12*x4^11*z^18 + 2*x1^35*x2^31*x3^13*x4^11*z^18 + x1^40*x2^25*x3^14*x4^11*z^18 + x1^39*x2^26*x3^14*x4^11*z^18 - 3*x1^37*x2^28*x3^14*x4^11*z^18 + 2*x1^36*x2^29*x3^14*x4^11*z^18 + 2*x1^33*x2^32*x3^14*x4^11*z^18 - x1^40*x2^24*x3^15*x4^11*z^18 + 3*x1^38*x2^26*x3^15*x4^11*z^18 + 2*x1^37*x2^27*x3^15*x4^11*z^18 - 3*x1^35*x2^29*x3^15*x4^11*z^18 + 4*x1^34*x2^30*x3^15*x4^11*z^18 - x1^33*x2^31*x3^15*x4^11*z^18 - x1^32*x2^32*x3^15*x4^11*z^18 - x1^38*x2^25*x3^16*x4^11*z^18 + x1^37*x2^26*x3^16*x4^11*z^18 - 3*x1^36*x2^27*x3^16*x4^11*z^18 + x1^35*x2^28*x3^16*x4^11*z^18 - 3*x1^34*x2^29*x3^16*x4^11*z^18 + 3*x1^32*x2^31*x3^16*x4^11*z^18 - x1^39*x2^23*x3^17*x4^11*z^18 - 2*x1^38*x2^24*x3^17*x4^11*z^18 + x1^37*x2^25*x3^17*x4^11*z^18 + 2*x1^35*x2^27*x3^17*x4^11*z^18 - x1^34*x2^28*x3^17*x4^11*z^18 + 2*x1^33*x2^29*x3^17*x4^11*z^18 - 3*x1^32*x2^30*x3^17*x4^11*z^18 - x1^38*x2^23*x3^18*x4^11*z^18 - x1^37*x2^24*x3^18*x4^11*z^18 + x1^36*x2^25*x3^18*x4^11*z^18 - x1^35*x2^26*x3^18*x4^11*z^18 - 4*x1^33*x2^28*x3^18*x4^11*z^18 - 2*x1^32*x2^29*x3^18*x4^11*z^18 + 2*x1^31*x2^30*x3^18*x4^11*z^18 + x1^37*x2^23*x3^19*x4^11*z^18 + x1^36*x2^24*x3^19*x4^11*z^18 + 4*x1^34*x2^26*x3^19*x4^11*z^18 + x1^33*x2^27*x3^19*x4^11*z^18 + 4*x1^32*x2^28*x3^19*x4^11*z^18 - 2*x1^31*x2^29*x3^19*x4^11*z^18 - x1^30*x2^30*x3^19*x4^11*z^18 - x1^35*x2^24*x3^20*x4^11*z^18 - x1^34*x2^25*x3^20*x4^11*z^18 + 2*x1^33*x2^26*x3^20*x4^11*z^18 - x1^32*x2^27*x3^20*x4^11*z^18 + x1^31*x2^28*x3^20*x4^11*z^18 + 2*x1^30*x2^29*x3^20*x4^11*z^18 - x1^33*x2^25*x3^21*x4^11*z^18 - 3*x1^32*x2^26*x3^21*x4^11*z^18 + 2*x1^31*x2^27*x3^21*x4^11*z^18 - 2*x1^30*x2^28*x3^21*x4^11*z^18 - x1^29*x2^29*x3^21*x4^11*z^18 - x1^30*x2^27*x3^22*x4^11*z^18 + 2*x1^29*x2^28*x3^22*x4^11*z^18 - x1^28*x2^28*x3^23*x4^11*z^18 - x1^37*x2^29*x3^12*x4^12*z^18 - x1^36*x2^30*x3^12*x4^12*z^18 - x1^35*x2^31*x3^12*x4^12*z^18 + 2*x1^37*x2^28*x3^13*x4^12*z^18 - 2*x1^36*x2^29*x3^13*x4^12*z^18 - x1^35*x2^30*x3^13*x4^12*z^18 + x1^34*x2^31*x3^13*x4^12*z^18 - x1^33*x2^32*x3^13*x4^12*z^18 - x1^36*x2^28*x3^14*x4^12*z^18 + 2*x1^35*x2^29*x3^14*x4^12*z^18 - 2*x1^34*x2^30*x3^14*x4^12*z^18 + x1^33*x2^31*x3^14*x4^12*z^18 + x1^32*x2^32*x3^14*x4^12*z^18 - x1^39*x2^24*x3^15*x4^12*z^18 - x1^38*x2^25*x3^15*x4^12*z^18 + x1^36*x2^27*x3^15*x4^12*z^18 - 3*x1^35*x2^28*x3^15*x4^12*z^18 - x1^34*x2^29*x3^15*x4^12*z^18 - 4*x1^32*x2^31*x3^15*x4^12*z^18 + x1^39*x2^23*x3^16*x4^12*z^18 - 3*x1^37*x2^25*x3^16*x4^12*z^18 - x1^35*x2^27*x3^16*x4^12*z^18 + 4*x1^34*x2^28*x3^16*x4^12*z^18 - 2*x1^33*x2^29*x3^16*x4^12*z^18 + 3*x1^32*x2^30*x3^16*x4^12*z^18 + 2*x1^31*x2^31*x3^16*x4^12*z^18 + 2*x1^37*x2^24*x3^17*x4^12*z^18 + x1^36*x2^25*x3^17*x4^12*z^18 + x1^35*x2^26*x3^17*x4^12*z^18 - 3*x1^34*x2^27*x3^17*x4^12*z^18 - x1^32*x2^29*x3^17*x4^12*z^18 - 5*x1^31*x2^30*x3^17*x4^12*z^18 - 4*x1^36*x2^24*x3^18*x4^12*z^18 + x1^35*x2^25*x3^18*x4^12*z^18 + 2*x1^33*x2^27*x3^18*x4^12*z^18 - x1^32*x2^28*x3^18*x4^12*z^18 + 6*x1^31*x2^29*x3^18*x4^12*z^18 + 2*x1^30*x2^30*x3^18*x4^12*z^18 - x1^36*x2^23*x3^19*x4^12*z^18 + 2*x1^35*x2^24*x3^19*x4^12*z^18 - 2*x1^33*x2^26*x3^19*x4^12*z^18 - x1^32*x2^27*x3^19*x4^12*z^18 - x1^31*x2^28*x3^19*x4^12*z^18 - 6*x1^30*x2^29*x3^19*x4^12*z^18 - 2*x1^34*x2^24*x3^20*x4^12*z^18 - x1^33*x2^25*x3^20*x4^12*z^18 + 3*x1^32*x2^26*x3^20*x4^12*z^18 - x1^31*x2^27*x3^20*x4^12*z^18 + 5*x1^30*x2^28*x3^20*x4^12*z^18 + 2*x1^29*x2^29*x3^20*x4^12*z^18 + 2*x1^33*x2^24*x3^21*x4^12*z^18 + x1^32*x2^25*x3^21*x4^12*z^18 - x1^30*x2^27*x3^21*x4^12*z^18 - 5*x1^29*x2^28*x3^21*x4^12*z^18 + 2*x1^31*x2^25*x3^22*x4^12*z^18 - x1^30*x2^26*x3^22*x4^12*z^18 + 3*x1^29*x2^27*x3^22*x4^12*z^18 + 2*x1^28*x2^28*x3^22*x4^12*z^18 + 2*x1^29*x2^26*x3^23*x4^12*z^18 - 3*x1^28*x2^27*x3^23*x4^12*z^18 + 2*x1^27*x2^27*x3^24*x4^12*z^18 - x1^36*x2^27*x3^14*x4^13*z^18 + x1^35*x2^28*x3^14*x4^13*z^18 - x1^34*x2^29*x3^14*x4^13*z^18 - x1^33*x2^30*x3^14*x4^13*z^18 + 2*x1^35*x2^27*x3^15*x4^13*z^18 - x1^34*x2^28*x3^15*x4^13*z^18 + x1^33*x2^29*x3^15*x4^13*z^18 - x1^32*x2^30*x3^15*x4^13*z^18 - x1^31*x2^31*x3^15*x4^13*z^18 - x1^35*x2^26*x3^16*x4^13*z^18 + 2*x1^34*x2^27*x3^16*x4^13*z^18 + x1^33*x2^28*x3^16*x4^13*z^18 - x1^32*x2^29*x3^16*x4^13*z^18 + 2*x1^31*x2^30*x3^16*x4^13*z^18 + x1^37*x2^23*x3^17*x4^13*z^18 - x1^36*x2^24*x3^17*x4^13*z^18 - x1^35*x2^25*x3^17*x4^13*z^18 + x1^34*x2^26*x3^17*x4^13*z^18 - 3*x1^33*x2^27*x3^17*x4^13*z^18 - 2*x1^31*x2^29*x3^17*x4^13*z^18 - 2*x1^30*x2^30*x3^17*x4^13*z^18 - x1^34*x2^25*x3^18*x4^13*z^18 + 3*x1^33*x2^26*x3^18*x4^13*z^18 + x1^31*x2^28*x3^18*x4^13*z^18 + 4*x1^30*x2^29*x3^18*x4^13*z^18 + x1^34*x2^24*x3^19*x4^13*z^18 - x1^32*x2^26*x3^19*x4^13*z^18 + 3*x1^31*x2^27*x3^19*x4^13*z^18 - 5*x1^30*x2^28*x3^19*x4^13*z^18 - 2*x1^29*x2^29*x3^19*x4^13*z^18 - x1^33*x2^24*x3^20*x4^13*z^18 + x1^31*x2^26*x3^20*x4^13*z^18 + x1^30*x2^27*x3^20*x4^13*z^18 + 6*x1^29*x2^28*x3^20*x4^13*z^18 - 2*x1^31*x2^25*x3^21*x4^13*z^18 + x1^30*x2^26*x3^21*x4^13*z^18 - 2*x1^29*x2^27*x3^21*x4^13*z^18 - 2*x1^28*x2^28*x3^21*x4^13*z^18 - x1^29*x2^26*x3^22*x4^13*z^18 + 2*x1^28*x2^27*x3^22*x4^13*z^18 - x1^27*x2^27*x3^23*x4^13*z^18 + x1^35*x2^26*x3^15*x4^14*z^18 + x1^34*x2^27*x3^15*x4^14*z^18 + x1^33*x2^28*x3^15*x4^14*z^18 - x1^35*x2^25*x3^16*x4^14*z^18 - x1^34*x2^26*x3^16*x4^14*z^18 + x1^33*x2^27*x3^16*x4^14*z^18 - x1^32*x2^28*x3^16*x4^14*z^18 + 2*x1^34*x2^25*x3^17*x4^14*z^18 + 2*x1^32*x2^27*x3^17*x4^14*z^18 + x1^31*x2^28*x3^17*x4^14*z^18 - x1^33*x2^25*x3^18*x4^14*z^18 + x1^32*x2^26*x3^18*x4^14*z^18 - x1^31*x2^27*x3^18*x4^14*z^18 + x1^30*x2^28*x3^18*x4^14*z^18 + x1^34*x2^23*x3^19*x4^14*z^18 + x1^33*x2^24*x3^19*x4^14*z^18 - x1^32*x2^25*x3^19*x4^14*z^18 + 2*x1^31*x2^26*x3^19*x4^14*z^18 + 2*x1^30*x2^27*x3^19*x4^14*z^18 - x1^29*x2^28*x3^19*x4^14*z^18 - x1^33*x2^23*x3^20*x4^14*z^18 - x1^32*x2^24*x3^20*x4^14*z^18 - 3*x1^30*x2^26*x3^20*x4^14*z^18 + x1^28*x2^28*x3^20*x4^14*z^18 + x1^30*x2^25*x3^21*x4^14*z^18 - x1^29*x2^25*x3^22*x4^14*z^18 - x1^34*x2^25*x3^16*x4^15*z^18 - x1^33*x2^26*x3^16*x4^15*z^18 - x1^32*x2^27*x3^16*x4^15*z^18 + x1^34*x2^24*x3^17*x4^15*z^18 + x1^33*x2^25*x3^17*x4^15*z^18 - x1^32*x2^26*x3^17*x4^15*z^18 + 3*x1^31*x2^27*x3^17*x4^15*z^18 - 2*x1^33*x2^24*x3^18*x4^15*z^18 - 2*x1^31*x2^26*x3^18*x4^15*z^18 - 2*x1^30*x2^27*x3^18*x4^15*z^18 - x1^33*x2^23*x3^19*x4^15*z^18 + 5*x1^30*x2^26*x3^19*x4^15*z^18 + x1^31*x2^24*x3^20*x4^15*z^18 - 2*x1^30*x2^25*x3^20*x4^15*z^18 - 2*x1^29*x2^26*x3^20*x4^15*z^18 - 2*x1^30*x2^24*x3^21*x4^15*z^18 + 3*x1^29*x2^25*x3^21*x4^15*z^18 - x1^28*x2^25*x3^22*x4^15*z^18 - 2*x1^30*x2^26*x3^18*x4^16*z^18 + x1^29*x2^26*x3^19*x4^16*z^18 - 2*x1^29*x2^25*x3^20*x4^16*z^18 - x1^43*x2^30*x3^12*z^17 + x1^43*x2^29*x3^13*z^17 + x1^42*x2^30*x3^13*z^17 - x1^41*x2^31*x3^13*z^17 - 2*x1^42*x2^29*x3^14*z^17 - x1^41*x2^30*x3^14*z^17 + x1^42*x2^28*x3^15*z^17 + 2*x1^41*x2^29*x3^15*z^17 + x1^39*x2^31*x3^15*z^17 - x1^38*x2^32*x3^15*z^17 - 2*x1^41*x2^28*x3^16*z^17 - x1^38*x2^31*x3^16*z^17 + x1^36*x2^33*x3^16*z^17 + 2*x1^40*x2^28*x3^17*z^17 + 2*x1^38*x2^30*x3^17*z^17 - x1^36*x2^32*x3^17*z^17 - x1^35*x2^33*x3^17*z^17 - 2*x1^40*x2^27*x3^18*z^17 - x1^37*x2^30*x3^18*z^17 + x1^40*x2^26*x3^19*z^17 + 2*x1^37*x2^29*x3^19*z^17 - x1^39*x2^26*x3^20*z^17 - x1^38*x2^27*x3^20*z^17 - x1^37*x2^28*x3^20*z^17 - x1^44*x2^30*x3^10*x4*z^17 + 2*x1^43*x2^30*x3^11*x4*z^17 + x1^42*x2^31*x3^11*x4*z^17 + x1^41*x2^32*x3^11*x4*z^17 - 2*x1^43*x2^29*x3^12*x4*z^17 - 2*x1^42*x2^30*x3^12*x4*z^17 - x1^40*x2^32*x3^12*x4*z^17 + x1^39*x2^33*x3^12*x4*z^17 + 6*x1^42*x2^29*x3^13*x4*z^17 + x1^41*x2^30*x3^13*x4*z^17 + 2*x1^40*x2^31*x3^13*x4*z^17 + x1^39*x2^32*x3^13*x4*z^17 - 2*x1^42*x2^28*x3^14*x4*z^17 - 6*x1^41*x2^29*x3^14*x4*z^17 + x1^40*x2^30*x3^14*x4*z^17 - 4*x1^39*x2^31*x3^14*x4*z^17 + x1^38*x2^32*x3^14*x4*z^17 + x1^36*x2^34*x3^14*x4*z^17 + 6*x1^41*x2^28*x3^15*x4*z^17 + 2*x1^40*x2^29*x3^15*x4*z^17 + x1^39*x2^30*x3^15*x4*z^17 + x1^38*x2^31*x3^15*x4*z^17 - x1^37*x2^32*x3^15*x4*z^17 - x1^36*x2^33*x3^15*x4*z^17 - x1^35*x2^34*x3^15*x4*z^17 - 2*x1^41*x2^27*x3^16*x4*z^17 - 6*x1^40*x2^28*x3^16*x4*z^17 - 4*x1^38*x2^30*x3^16*x4*z^17 + x1^37*x2^31*x3^16*x4*z^17 + 3*x1^35*x2^33*x3^16*x4*z^17 + 6*x1^40*x2^27*x3^17*x4*z^17 + 2*x1^39*x2^28*x3^17*x4*z^17 + 2*x1^38*x2^29*x3^17*x4*z^17 + 2*x1^37*x2^30*x3^17*x4*z^17 - x1^36*x2^31*x3^17*x4*z^17 - 3*x1^35*x2^32*x3^17*x4*z^17 - x1^34*x2^33*x3^17*x4*z^17 - 2*x1^40*x2^26*x3^18*x4*z^17 - 6*x1^39*x2^27*x3^18*x4*z^17 - 4*x1^37*x2^29*x3^18*x4*z^17 + 4*x1^34*x2^32*x3^18*x4*z^17 + 6*x1^39*x2^26*x3^19*x4*z^17 + x1^38*x2^27*x3^19*x4*z^17 + x1^37*x2^28*x3^19*x4*z^17 + 3*x1^36*x2^29*x3^19*x4*z^17 - 2*x1^34*x2^31*x3^19*x4*z^17 - x1^33*x2^32*x3^19*x4*z^17 - 2*x1^39*x2^25*x3^20*x4*z^17 - 3*x1^38*x2^26*x3^20*x4*z^17 - x1^37*x2^27*x3^20*x4*z^17 - 4*x1^36*x2^28*x3^20*x4*z^17 + x1^35*x2^29*x3^20*x4*z^17 - x1^34*x2^30*x3^20*x4*z^17 + x1^33*x2^31*x3^20*x4*z^17 + 2*x1^38*x2^25*x3^21*x4*z^17 + 2*x1^36*x2^27*x3^21*x4*z^17 + 2*x1^35*x2^28*x3^21*x4*z^17 - x1^37*x2^25*x3^22*x4*z^17 - x1^36*x2^26*x3^22*x4*z^17 - 3*x1^35*x2^27*x3^22*x4*z^17 + x1^34*x2^27*x3^23*x4*z^17 - 2*x1^43*x2^30*x3^10*x4^2*z^17 + x1^42*x2^31*x3^10*x4^2*z^17 + 2*x1^43*x2^29*x3^11*x4^2*z^17 + 2*x1^42*x2^30*x3^11*x4^2*z^17 - 2*x1^41*x2^31*x3^11*x4^2*z^17 + x1^40*x2^32*x3^11*x4^2*z^17 - 5*x1^42*x2^29*x3^12*x4^2*z^17 + x1^40*x2^31*x3^12*x4^2*z^17 + x1^38*x2^33*x3^12*x4^2*z^17 + 2*x1^42*x2^28*x3^13*x4^2*z^17 + 5*x1^41*x2^29*x3^13*x4^2*z^17 - 2*x1^40*x2^30*x3^13*x4^2*z^17 + x1^39*x2^31*x3^13*x4^2*z^17 - 2*x1^38*x2^32*x3^13*x4^2*z^17 - x1^37*x2^33*x3^13*x4^2*z^17 - 6*x1^41*x2^28*x3^14*x4^2*z^17 - 2*x1^40*x2^29*x3^14*x4^2*z^17 + x1^38*x2^31*x3^14*x4^2*z^17 + x1^37*x2^32*x3^14*x4^2*z^17 + x1^36*x2^33*x3^14*x4^2*z^17 + 2*x1^41*x2^27*x3^15*x4^2*z^17 + 6*x1^40*x2^28*x3^15*x4^2*z^17 + 3*x1^38*x2^30*x3^15*x4^2*z^17 - 4*x1^37*x2^31*x3^15*x4^2*z^17 - x1^36*x2^32*x3^15*x4^2*z^17 - 4*x1^35*x2^33*x3^15*x4^2*z^17 - 6*x1^40*x2^27*x3^16*x4^2*z^17 - 2*x1^39*x2^28*x3^16*x4^2*z^17 - 2*x1^38*x2^29*x3^16*x4^2*z^17 + 2*x1^36*x2^31*x3^16*x4^2*z^17 + 3*x1^35*x2^32*x3^16*x4^2*z^17 + 4*x1^34*x2^33*x3^16*x4^2*z^17 + 2*x1^40*x2^26*x3^17*x4^2*z^17 + 6*x1^39*x2^27*x3^17*x4^2*z^17 + 4*x1^37*x2^29*x3^17*x4^2*z^17 - 4*x1^36*x2^30*x3^17*x4^2*z^17 - 7*x1^34*x2^32*x3^17*x4^2*z^17 - x1^33*x2^33*x3^17*x4^2*z^17 - 6*x1^39*x2^26*x3^18*x4^2*z^17 - 2*x1^38*x2^27*x3^18*x4^2*z^17 - 2*x1^37*x2^28*x3^18*x4^2*z^17 - x1^36*x2^29*x3^18*x4^2*z^17 + 2*x1^35*x2^30*x3^18*x4^2*z^17 + 2*x1^34*x2^31*x3^18*x4^2*z^17 + 4*x1^33*x2^32*x3^18*x4^2*z^17 + x1^39*x2^25*x3^19*x4^2*z^17 + 6*x1^38*x2^26*x3^19*x4^2*z^17 + 4*x1^36*x2^28*x3^19*x4^2*z^17 - x1^35*x2^29*x3^19*x4^2*z^17 + x1^34*x2^30*x3^19*x4^2*z^17 - 5*x1^33*x2^31*x3^19*x4^2*z^17 - x1^32*x2^32*x3^19*x4^2*z^17 - 3*x1^38*x2^25*x3^20*x4^2*z^17 - 4*x1^37*x2^26*x3^20*x4^2*z^17 - 2*x1^36*x2^27*x3^20*x4^2*z^17 - 2*x1^35*x2^28*x3^20*x4^2*z^17 + 2*x1^33*x2^30*x3^20*x4^2*z^17 + 2*x1^32*x2^31*x3^20*x4^2*z^17 + 3*x1^37*x2^25*x3^21*x4^2*z^17 + 2*x1^36*x2^26*x3^21*x4^2*z^17 + 3*x1^35*x2^27*x3^21*x4^2*z^17 + x1^33*x2^29*x3^21*x4^2*z^17 - 3*x1^32*x2^30*x3^21*x4^2*z^17 - 2*x1^35*x2^26*x3^22*x4^2*z^17 - 4*x1^34*x2^27*x3^22*x4^2*z^17 + x1^33*x2^28*x3^22*x4^2*z^17 + x1^32*x2^29*x3^22*x4^2*z^17 + 2*x1^34*x2^26*x3^23*x4^2*z^17 - x1^33*x2^26*x3^24*x4^2*z^17 + x1^43*x2^30*x3^9*x4^3*z^17 + x1^44*x2^28*x3^10*x4^3*z^17 - x1^42*x2^30*x3^10*x4^3*z^17 + x1^41*x2^31*x3^10*x4^3*z^17 - x1^43*x2^28*x3^11*x4^3*z^17 + 2*x1^42*x2^29*x3^11*x4^3*z^17 + x1^41*x2^30*x3^11*x4^3*z^17 + 2*x1^43*x2^27*x3^12*x4^3*z^17 - 2*x1^41*x2^29*x3^12*x4^3*z^17 - x1^40*x2^30*x3^12*x4^3*z^17 - x1^39*x2^31*x3^12*x4^3*z^17 + x1^38*x2^32*x3^12*x4^3*z^17 - x1^43*x2^26*x3^13*x4^3*z^17 - 2*x1^42*x2^27*x3^13*x4^3*z^17 + 2*x1^41*x2^28*x3^13*x4^3*z^17 - x1^40*x2^29*x3^13*x4^3*z^17 + x1^39*x2^30*x3^13*x4^3*z^17 - x1^37*x2^32*x3^13*x4^3*z^17 - x1^36*x2^33*x3^13*x4^3*z^17 + 2*x1^42*x2^26*x3^14*x4^3*z^17 - x1^40*x2^28*x3^14*x4^3*z^17 - 3*x1^38*x2^30*x3^14*x4^3*z^17 - x1^37*x2^31*x3^14*x4^3*z^17 + x1^36*x2^32*x3^14*x4^3*z^17 + 2*x1^35*x2^33*x3^14*x4^3*z^17 - x1^42*x2^25*x3^15*x4^3*z^17 - 2*x1^41*x2^26*x3^15*x4^3*z^17 + 2*x1^40*x2^27*x3^15*x4^3*z^17 + 3*x1^38*x2^29*x3^15*x4^3*z^17 + x1^36*x2^31*x3^15*x4^3*z^17 - 2*x1^35*x2^32*x3^15*x4^3*z^17 - 2*x1^34*x2^33*x3^15*x4^3*z^17 + 2*x1^41*x2^25*x3^16*x4^3*z^17 - x1^40*x2^26*x3^16*x4^3*z^17 - 2*x1^39*x2^27*x3^16*x4^3*z^17 - x1^37*x2^29*x3^16*x4^3*z^17 + 2*x1^36*x2^30*x3^16*x4^3*z^17 - 2*x1^35*x2^31*x3^16*x4^3*z^17 + 2*x1^34*x2^32*x3^16*x4^3*z^17 + x1^33*x2^33*x3^16*x4^3*z^17 - 2*x1^40*x2^25*x3^17*x4^3*z^17 + 2*x1^39*x2^26*x3^17*x4^3*z^17 - 2*x1^38*x2^27*x3^17*x4^3*z^17 + x1^37*x2^28*x3^17*x4^3*z^17 + 2*x1^35*x2^30*x3^17*x4^3*z^17 + x1^34*x2^31*x3^17*x4^3*z^17 - 2*x1^33*x2^32*x3^17*x4^3*z^17 + 2*x1^40*x2^24*x3^18*x4^3*z^17 + x1^39*x2^25*x3^18*x4^3*z^17 - x1^38*x2^26*x3^18*x4^3*z^17 - 2*x1^36*x2^28*x3^18*x4^3*z^17 + 2*x1^35*x2^29*x3^18*x4^3*z^17 + 3*x1^33*x2^31*x3^18*x4^3*z^17 + x1^32*x2^32*x3^18*x4^3*z^17 - x1^40*x2^23*x3^19*x4^3*z^17 - 2*x1^39*x2^24*x3^19*x4^3*z^17 + x1^38*x2^25*x3^19*x4^3*z^17 - x1^35*x2^28*x3^19*x4^3*z^17 + x1^34*x2^29*x3^19*x4^3*z^17 + x1^33*x2^30*x3^19*x4^3*z^17 - 2*x1^32*x2^31*x3^19*x4^3*z^17 + 2*x1^39*x2^23*x3^20*x4^3*z^17 + x1^36*x2^26*x3^20*x4^3*z^17 - 2*x1^35*x2^27*x3^20*x4^3*z^17 - x1^34*x2^28*x3^20*x4^3*z^17 - x1^33*x2^29*x3^20*x4^3*z^17 + 3*x1^32*x2^30*x3^20*x4^3*z^17 + x1^31*x2^31*x3^20*x4^3*z^17 - x1^38*x2^23*x3^21*x4^3*z^17 + x1^36*x2^25*x3^21*x4^3*z^17 + x1^35*x2^26*x3^21*x4^3*z^17 - x1^34*x2^27*x3^21*x4^3*z^17 - x1^33*x2^28*x3^21*x4^3*z^17 - x1^32*x2^29*x3^21*x4^3*z^17 - x1^31*x2^30*x3^21*x4^3*z^17 + x1^35*x2^25*x3^22*x4^3*z^17 + x1^34*x2^26*x3^22*x4^3*z^17 + 2*x1^31*x2^29*x3^22*x4^3*z^17 - x1^35*x2^24*x3^23*x4^3*z^17 + 2*x1^33*x2^26*x3^23*x4^3*z^17 - x1^32*x2^27*x3^23*x4^3*z^17 - x1^31*x2^28*x3^23*x4^3*z^17 + x1^44*x2^27*x3^10*x4^4*z^17 + 2*x1^43*x2^28*x3^10*x4^4*z^17 - x1^41*x2^30*x3^10*x4^4*z^17 - x1^40*x2^31*x3^10*x4^4*z^17 - 4*x1^43*x2^27*x3^11*x4^4*z^17 + x1^39*x2^31*x3^11*x4^4*z^17 + x1^38*x2^32*x3^11*x4^4*z^17 + 2*x1^43*x2^26*x3^12*x4^4*z^17 + 6*x1^42*x2^27*x3^12*x4^4*z^17 + 3*x1^40*x2^29*x3^12*x4^4*z^17 - 3*x1^39*x2^30*x3^12*x4^4*z^17 - x1^38*x2^31*x3^12*x4^4*z^17 - 2*x1^37*x2^32*x3^12*x4^4*z^17 - 6*x1^42*x2^26*x3^13*x4^4*z^17 - 2*x1^41*x2^27*x3^13*x4^4*z^17 + 2*x1^38*x2^30*x3^13*x4^4*z^17 + 2*x1^37*x2^31*x3^13*x4^4*z^17 + 2*x1^36*x2^32*x3^13*x4^4*z^17 + x1^35*x2^33*x3^13*x4^4*z^17 + 2*x1^42*x2^25*x3^14*x4^4*z^17 + 6*x1^41*x2^26*x3^14*x4^4*z^17 + 4*x1^39*x2^28*x3^14*x4^4*z^17 - 2*x1^38*x2^29*x3^14*x4^4*z^17 - x1^37*x2^30*x3^14*x4^4*z^17 - 4*x1^36*x2^31*x3^14*x4^4*z^17 - x1^35*x2^32*x3^14*x4^4*z^17 - x1^34*x2^33*x3^14*x4^4*z^17 - 6*x1^41*x2^25*x3^15*x4^4*z^17 - 2*x1^40*x2^26*x3^15*x4^4*z^17 - 2*x1^39*x2^27*x3^15*x4^4*z^17 + 2*x1^37*x2^29*x3^15*x4^4*z^17 + 4*x1^36*x2^30*x3^15*x4^4*z^17 + 3*x1^35*x2^31*x3^15*x4^4*z^17 + x1^34*x2^32*x3^15*x4^4*z^17 + 2*x1^41*x2^24*x3^16*x4^4*z^17 + 6*x1^40*x2^25*x3^16*x4^4*z^17 + 4*x1^38*x2^27*x3^16*x4^4*z^17 - 4*x1^37*x2^28*x3^16*x4^4*z^17 - 6*x1^35*x2^30*x3^16*x4^4*z^17 - x1^33*x2^32*x3^16*x4^4*z^17 - 6*x1^40*x2^24*x3^17*x4^4*z^17 - 2*x1^39*x2^25*x3^17*x4^4*z^17 - 2*x1^38*x2^26*x3^17*x4^4*z^17 + 2*x1^36*x2^28*x3^17*x4^4*z^17 + 2*x1^35*x2^29*x3^17*x4^4*z^17 + 6*x1^34*x2^30*x3^17*x4^4*z^17 + x1^40*x2^23*x3^18*x4^4*z^17 + 6*x1^39*x2^24*x3^18*x4^4*z^17 + 4*x1^37*x2^26*x3^18*x4^4*z^17 - 4*x1^36*x2^27*x3^18*x4^4*z^17 - 6*x1^34*x2^29*x3^18*x4^4*z^17 - 2*x1^33*x2^30*x3^18*x4^4*z^17 - 4*x1^39*x2^23*x3^19*x4^4*z^17 - 3*x1^38*x2^24*x3^19*x4^4*z^17 - 2*x1^37*x2^25*x3^19*x4^4*z^17 + x1^35*x2^27*x3^19*x4^4*z^17 + 4*x1^33*x2^29*x3^19*x4^4*z^17 - x1^32*x2^30*x3^19*x4^4*z^17 + 4*x1^38*x2^23*x3^20*x4^4*z^17 + x1^37*x2^24*x3^20*x4^4*z^17 + 2*x1^36*x2^25*x3^20*x4^4*z^17 - x1^35*x2^26*x3^20*x4^4*z^17 + x1^34*x2^27*x3^20*x4^4*z^17 - 5*x1^33*x2^28*x3^20*x4^4*z^17 - 2*x1^32*x2^29*x3^20*x4^4*z^17 + 2*x1^31*x2^30*x3^20*x4^4*z^17 - x1^37*x2^23*x3^21*x4^4*z^17 - x1^36*x2^24*x3^21*x4^4*z^17 - 2*x1^35*x2^25*x3^21*x4^4*z^17 - x1^34*x2^26*x3^21*x4^4*z^17 + x1^33*x2^27*x3^21*x4^4*z^17 + x1^32*x2^28*x3^21*x4^4*z^17 - x1^31*x2^29*x3^21*x4^4*z^17 + 2*x1^36*x2^23*x3^22*x4^4*z^17 + 2*x1^35*x2^24*x3^22*x4^4*z^17 + x1^33*x2^26*x3^22*x4^4*z^17 - 3*x1^32*x2^27*x3^22*x4^4*z^17 + x1^30*x2^29*x3^22*x4^4*z^17 - x1^35*x2^23*x3^23*x4^4*z^17 - x1^33*x2^25*x3^23*x4^4*z^17 + 2*x1^32*x2^26*x3^23*x4^4*z^17 - x1^31*x2^27*x3^23*x4^4*z^17 - 2*x1^30*x2^28*x3^23*x4^4*z^17 - x1^31*x2^26*x3^24*x4^4*z^17 + x1^30*x2^27*x3^24*x4^4*z^17 + x1^29*x2^28*x3^24*x4^4*z^17 - x1^43*x2^28*x3^9*x4^5*z^17 + x1^43*x2^27*x3^10*x4^5*z^17 + 2*x1^42*x2^28*x3^10*x4^5*z^17 - 4*x1^42*x2^27*x3^11*x4^5*z^17 + 2*x1^40*x2^29*x3^11*x4^5*z^17 + 2*x1^39*x2^30*x3^11*x4^5*z^17 + x1^38*x2^31*x3^11*x4^5*z^17 - x1^37*x2^32*x3^11*x4^5*z^17 + 3*x1^42*x2^26*x3^12*x4^5*z^17 + 2*x1^41*x2^27*x3^12*x4^5*z^17 - x1^39*x2^29*x3^12*x4^5*z^17 - x1^38*x2^30*x3^12*x4^5*z^17 + x1^36*x2^32*x3^12*x4^5*z^17 - 2*x1^42*x2^25*x3^13*x4^5*z^17 - 5*x1^41*x2^26*x3^13*x4^5*z^17 + x1^40*x2^27*x3^13*x4^5*z^17 - x1^39*x2^28*x3^13*x4^5*z^17 + 3*x1^38*x2^29*x3^13*x4^5*z^17 + 3*x1^36*x2^31*x3^13*x4^5*z^17 - x1^35*x2^32*x3^13*x4^5*z^17 + 6*x1^41*x2^25*x3^14*x4^5*z^17 + 2*x1^40*x2^26*x3^14*x4^5*z^17 + x1^39*x2^27*x3^14*x4^5*z^17 + x1^38*x2^28*x3^14*x4^5*z^17 - 2*x1^37*x2^29*x3^14*x4^5*z^17 - 2*x1^36*x2^30*x3^14*x4^5*z^17 - 3*x1^35*x2^31*x3^14*x4^5*z^17 - 2*x1^41*x2^24*x3^15*x4^5*z^17 - 6*x1^40*x2^25*x3^15*x4^5*z^17 + 2*x1^39*x2^26*x3^15*x4^5*z^17 - 3*x1^38*x2^27*x3^15*x4^5*z^17 + 5*x1^37*x2^28*x3^15*x4^5*z^17 + 6*x1^35*x2^30*x3^15*x4^5*z^17 + x1^34*x2^31*x3^15*x4^5*z^17 + 6*x1^40*x2^24*x3^16*x4^5*z^17 + x1^39*x2^25*x3^16*x4^5*z^17 - 3*x1^36*x2^28*x3^16*x4^5*z^17 - x1^35*x2^29*x3^16*x4^5*z^17 - 6*x1^34*x2^30*x3^16*x4^5*z^17 + 2*x1^33*x2^31*x3^16*x4^5*z^17 + x1^32*x2^32*x3^16*x4^5*z^17 - 6*x1^39*x2^24*x3^17*x4^5*z^17 + 2*x1^38*x2^25*x3^17*x4^5*z^17 - 3*x1^37*x2^26*x3^17*x4^5*z^17 + 5*x1^36*x2^27*x3^17*x4^5*z^17 + 5*x1^34*x2^29*x3^17*x4^5*z^17 + x1^33*x2^30*x3^17*x4^5*z^17 - 2*x1^32*x2^31*x3^17*x4^5*z^17 + 3*x1^39*x2^23*x3^18*x4^5*z^17 + 3*x1^38*x2^24*x3^18*x4^5*z^17 - 3*x1^35*x2^27*x3^18*x4^5*z^17 - 6*x1^33*x2^29*x3^18*x4^5*z^17 + 2*x1^32*x2^30*x3^18*x4^5*z^17 - 3*x1^38*x2^23*x3^19*x4^5*z^17 + x1^37*x2^24*x3^19*x4^5*z^17 - 2*x1^36*x2^25*x3^19*x4^5*z^17 + 4*x1^35*x2^26*x3^19*x4^5*z^17 + 6*x1^33*x2^28*x3^19*x4^5*z^17 + 2*x1^32*x2^29*x3^19*x4^5*z^17 - 2*x1^31*x2^30*x3^19*x4^5*z^17 + x1^38*x2^22*x3^20*x4^5*z^17 + x1^37*x2^23*x3^20*x4^5*z^17 + x1^35*x2^25*x3^20*x4^5*z^17 - 2*x1^34*x2^26*x3^20*x4^5*z^17 - 2*x1^33*x2^27*x3^20*x4^5*z^17 - 5*x1^32*x2^28*x3^20*x4^5*z^17 + x1^31*x2^29*x3^20*x4^5*z^17 - x1^37*x2^22*x3^21*x4^5*z^17 + x1^36*x2^23*x3^21*x4^5*z^17 + 3*x1^34*x2^25*x3^21*x4^5*z^17 - 2*x1^33*x2^26*x3^21*x4^5*z^17 + 5*x1^32*x2^27*x3^21*x4^5*z^17 + x1^31*x2^28*x3^21*x4^5*z^17 - 3*x1^30*x2^29*x3^21*x4^5*z^17 - x1^35*x2^23*x3^22*x4^5*z^17 - x1^34*x2^24*x3^22*x4^5*z^17 - x1^33*x2^25*x3^22*x4^5*z^17 + x1^32*x2^26*x3^22*x4^5*z^17 - 3*x1^31*x2^27*x3^22*x4^5*z^17 + 2*x1^30*x2^28*x3^22*x4^5*z^17 - x1^34*x2^23*x3^23*x4^5*z^17 + x1^33*x2^24*x3^23*x4^5*z^17 - 2*x1^29*x2^28*x3^23*x4^5*z^17 + x1^30*x2^26*x3^24*x4^5*z^17 + x1^29*x2^27*x3^24*x4^5*z^17 - x1^28*x2^27*x3^25*x4^5*z^17 - x1^42*x2^27*x3^10*x4^6*z^17 + x1^41*x2^28*x3^10*x4^6*z^17 - x1^39*x2^30*x3^10*x4^6*z^17 + x1^38*x2^31*x3^10*x4^6*z^17 - x1^41*x2^27*x3^11*x4^6*z^17 + x1^39*x2^29*x3^11*x4^6*z^17 - x1^37*x2^31*x3^11*x4^6*z^17 - x1^42*x2^25*x3^12*x4^6*z^17 - 2*x1^40*x2^27*x3^12*x4^6*z^17 - 2*x1^36*x2^31*x3^12*x4^6*z^17 + x1^35*x2^32*x3^12*x4^6*z^17 - x1^41*x2^25*x3^13*x4^6*z^17 - x1^40*x2^26*x3^13*x4^6*z^17 + 2*x1^39*x2^27*x3^13*x4^6*z^17 + x1^38*x2^28*x3^13*x4^6*z^17 + 3*x1^37*x2^29*x3^13*x4^6*z^17 - x1^36*x2^30*x3^13*x4^6*z^17 + 2*x1^35*x2^31*x3^13*x4^6*z^17 + x1^41*x2^24*x3^14*x4^6*z^17 + x1^40*x2^25*x3^14*x4^6*z^17 - 5*x1^39*x2^26*x3^14*x4^6*z^17 - 3*x1^38*x2^27*x3^14*x4^6*z^17 - 2*x1^37*x2^28*x3^14*x4^6*z^17 + x1^36*x2^29*x3^14*x4^6*z^17 - x1^35*x2^30*x3^14*x4^6*z^17 - x1^34*x2^31*x3^14*x4^6*z^17 - 2*x1^40*x2^24*x3^15*x4^6*z^17 + 2*x1^38*x2^26*x3^15*x4^6*z^17 - x1^37*x2^27*x3^15*x4^6*z^17 + 3*x1^36*x2^28*x3^15*x4^6*z^17 - x1^35*x2^29*x3^15*x4^6*z^17 - 2*x1^33*x2^31*x3^15*x4^6*z^17 - x1^32*x2^32*x3^15*x4^6*z^17 + x1^40*x2^23*x3^16*x4^6*z^17 + 2*x1^39*x2^24*x3^16*x4^6*z^17 - 4*x1^38*x2^25*x3^16*x4^6*z^17 - x1^37*x2^26*x3^16*x4^6*z^17 - 4*x1^36*x2^27*x3^16*x4^6*z^17 + 2*x1^33*x2^30*x3^16*x4^6*z^17 + 2*x1^32*x2^31*x3^16*x4^6*z^17 - 2*x1^39*x2^23*x3^17*x4^6*z^17 + 2*x1^38*x2^24*x3^17*x4^6*z^17 + 4*x1^37*x2^25*x3^17*x4^6*z^17 + 2*x1^35*x2^27*x3^17*x4^6*z^17 - 3*x1^34*x2^28*x3^17*x4^6*z^17 + 2*x1^33*x2^29*x3^17*x4^6*z^17 - 4*x1^32*x2^30*x3^17*x4^6*z^17 - x1^31*x2^31*x3^17*x4^6*z^17 - x1^39*x2^22*x3^18*x4^6*z^17 + 2*x1^38*x2^23*x3^18*x4^6*z^17 - 4*x1^37*x2^24*x3^18*x4^6*z^17 + x1^36*x2^25*x3^18*x4^6*z^17 - 2*x1^35*x2^26*x3^18*x4^6*z^17 - x1^33*x2^28*x3^18*x4^6*z^17 + 4*x1^31*x2^30*x3^18*x4^6*z^17 - x1^37*x2^23*x3^19*x4^6*z^17 + 3*x1^36*x2^24*x3^19*x4^6*z^17 + 4*x1^34*x2^26*x3^19*x4^6*z^17 - 2*x1^33*x2^27*x3^19*x4^6*z^17 + 2*x1^32*x2^28*x3^19*x4^6*z^17 - 4*x1^31*x2^29*x3^19*x4^6*z^17 - x1^30*x2^30*x3^19*x4^6*z^17 - 2*x1^36*x2^23*x3^20*x4^6*z^17 - 3*x1^35*x2^24*x3^20*x4^6*z^17 - 3*x1^34*x2^25*x3^20*x4^6*z^17 - x1^32*x2^27*x3^20*x4^6*z^17 + 4*x1^30*x2^29*x3^20*x4^6*z^17 + x1^36*x2^22*x3^21*x4^6*z^17 + 2*x1^35*x2^23*x3^21*x4^6*z^17 + 2*x1^34*x2^24*x3^21*x4^6*z^17 + 4*x1^33*x2^25*x3^21*x4^6*z^17 + 2*x1^31*x2^27*x3^21*x4^6*z^17 - 3*x1^30*x2^28*x3^21*x4^6*z^17 - x1^29*x2^29*x3^21*x4^6*z^17 - x1^33*x2^24*x3^22*x4^6*z^17 + x1^31*x2^26*x3^22*x4^6*z^17 + 3*x1^30*x2^27*x3^22*x4^6*z^17 + 3*x1^29*x2^28*x3^22*x4^6*z^17 + x1^31*x2^25*x3^23*x4^6*z^17 + 2*x1^30*x2^26*x3^23*x4^6*z^17 - 3*x1^29*x2^27*x3^23*x4^6*z^17 - 2*x1^30*x2^25*x3^24*x4^6*z^17 - x1^29*x2^26*x3^24*x4^6*z^17 + 2*x1^28*x2^27*x3^24*x4^6*z^17 + x1^39*x2^30*x3^9*x4^7*z^17 + x1^38*x2^31*x3^9*x4^7*z^17 + x1^37*x2^32*x3^9*x4^7*z^17 - x1^41*x2^27*x3^10*x4^7*z^17 + x1^39*x2^29*x3^10*x4^7*z^17 - x1^38*x2^30*x3^10*x4^7*z^17 - 2*x1^37*x2^31*x3^10*x4^7*z^17 - 2*x1^36*x2^32*x3^10*x4^7*z^17 - x1^35*x2^33*x3^10*x4^7*z^17 + 2*x1^41*x2^26*x3^11*x4^7*z^17 - 2*x1^40*x2^27*x3^11*x4^7*z^17 + x1^38*x2^29*x3^11*x4^7*z^17 + 2*x1^35*x2^32*x3^11*x4^7*z^17 + x1^34*x2^33*x3^11*x4^7*z^17 - x1^40*x2^26*x3^12*x4^7*z^17 - x1^38*x2^28*x3^12*x4^7*z^17 - x1^37*x2^29*x3^12*x4^7*z^17 + x1^36*x2^30*x3^12*x4^7*z^17 - x1^35*x2^31*x3^12*x4^7*z^17 - 4*x1^34*x2^32*x3^12*x4^7*z^17 + x1^41*x2^24*x3^13*x4^7*z^17 + x1^40*x2^25*x3^13*x4^7*z^17 - x1^39*x2^26*x3^13*x4^7*z^17 + x1^38*x2^27*x3^13*x4^7*z^17 - x1^37*x2^28*x3^13*x4^7*z^17 + 2*x1^36*x2^29*x3^13*x4^7*z^17 - x1^34*x2^31*x3^13*x4^7*z^17 + 3*x1^33*x2^32*x3^13*x4^7*z^17 - x1^38*x2^26*x3^14*x4^7*z^17 - x1^37*x2^27*x3^14*x4^7*z^17 - x1^36*x2^28*x3^14*x4^7*z^17 + x1^34*x2^30*x3^14*x4^7*z^17 - x1^33*x2^31*x3^14*x4^7*z^17 - x1^32*x2^32*x3^14*x4^7*z^17 + x1^40*x2^23*x3^15*x4^7*z^17 + x1^39*x2^24*x3^15*x4^7*z^17 + 2*x1^37*x2^26*x3^15*x4^7*z^17 - x1^36*x2^27*x3^15*x4^7*z^17 + x1^32*x2^31*x3^15*x4^7*z^17 - x1^38*x2^24*x3^16*x4^7*z^17 + x1^39*x2^22*x3^17*x4^7*z^17 + 2*x1^37*x2^24*x3^17*x4^7*z^17 + x1^36*x2^25*x3^17*x4^7*z^17 + x1^35*x2^26*x3^17*x4^7*z^17 - x1^37*x2^23*x3^18*x4^7*z^17 - 2*x1^36*x2^24*x3^18*x4^7*z^17 - x1^34*x2^26*x3^18*x4^7*z^17 + x1^33*x2^27*x3^18*x4^7*z^17 + 3*x1^36*x2^23*x3^19*x4^7*z^17 + x1^35*x2^24*x3^19*x4^7*z^17 + x1^33*x2^26*x3^19*x4^7*z^17 - x1^36*x2^22*x3^20*x4^7*z^17 - 3*x1^35*x2^23*x3^20*x4^7*z^17 - x1^34*x2^24*x3^20*x4^7*z^17 - 3*x1^33*x2^25*x3^20*x4^7*z^17 - x1^32*x2^26*x3^20*x4^7*z^17 + x1^30*x2^28*x3^20*x4^7*z^17 + x1^35*x2^22*x3^21*x4^7*z^17 + 2*x1^34*x2^23*x3^21*x4^7*z^17 + x1^33*x2^24*x3^21*x4^7*z^17 + x1^32*x2^25*x3^21*x4^7*z^17 - x1^30*x2^27*x3^21*x4^7*z^17 - x1^29*x2^28*x3^21*x4^7*z^17 - x1^34*x2^22*x3^22*x4^7*z^17 - 2*x1^33*x2^23*x3^22*x4^7*z^17 - x1^32*x2^24*x3^22*x4^7*z^17 - x1^31*x2^25*x3^22*x4^7*z^17 - x1^30*x2^26*x3^22*x4^7*z^17 + x1^29*x2^27*x3^22*x4^7*z^17 + x1^32*x2^23*x3^23*x4^7*z^17 + 2*x1^31*x2^24*x3^23*x4^7*z^17 - x1^29*x2^26*x3^23*x4^7*z^17 + x1^28*x2^26*x3^24*x4^7*z^17 + x1^28*x2^25*x3^25*x4^7*z^17 - x1^27*x2^26*x3^25*x4^7*z^17 - x1^39*x2^29*x3^9*x4^8*z^17 + x1^36*x2^31*x3^10*x4^8*z^17 - x1^41*x2^25*x3^11*x4^8*z^17 - x1^39*x2^27*x3^11*x4^8*z^17 + x1^37*x2^29*x3^11*x4^8*z^17 + 3*x1^36*x2^30*x3^11*x4^8*z^17 + 2*x1^34*x2^32*x3^11*x4^8*z^17 + x1^39*x2^26*x3^12*x4^8*z^17 - x1^37*x2^28*x3^12*x4^8*z^17 - 3*x1^36*x2^29*x3^12*x4^8*z^17 + x1^34*x2^31*x3^12*x4^8*z^17 - 3*x1^33*x2^32*x3^12*x4^8*z^17 - x1^40*x2^24*x3^13*x4^8*z^17 + x1^39*x2^25*x3^13*x4^8*z^17 - x1^38*x2^26*x3^13*x4^8*z^17 + x1^37*x2^27*x3^13*x4^8*z^17 + x1^36*x2^28*x3^13*x4^8*z^17 - x1^34*x2^30*x3^13*x4^8*z^17 + 4*x1^33*x2^31*x3^13*x4^8*z^17 + x1^32*x2^32*x3^13*x4^8*z^17 + x1^39*x2^24*x3^14*x4^8*z^17 + 2*x1^38*x2^25*x3^14*x4^8*z^17 + 3*x1^37*x2^26*x3^14*x4^8*z^17 + x1^36*x2^27*x3^14*x4^8*z^17 - x1^35*x2^28*x3^14*x4^8*z^17 - 2*x1^34*x2^29*x3^14*x4^8*z^17 - x1^33*x2^30*x3^14*x4^8*z^17 - 4*x1^32*x2^31*x3^14*x4^8*z^17 - x1^39*x2^23*x3^15*x4^8*z^17 - x1^38*x2^24*x3^15*x4^8*z^17 - 3*x1^37*x2^25*x3^15*x4^8*z^17 + x1^36*x2^26*x3^15*x4^8*z^17 + 2*x1^35*x2^27*x3^15*x4^8*z^17 + 4*x1^34*x2^28*x3^15*x4^8*z^17 + 2*x1^33*x2^29*x3^15*x4^8*z^17 + 4*x1^32*x2^30*x3^15*x4^8*z^17 + x1^31*x2^31*x3^15*x4^8*z^17 + 2*x1^38*x2^23*x3^16*x4^8*z^17 + 2*x1^37*x2^24*x3^16*x4^8*z^17 + x1^36*x2^25*x3^16*x4^8*z^17 - x1^34*x2^27*x3^16*x4^8*z^17 - 3*x1^33*x2^28*x3^16*x4^8*z^17 - 2*x1^32*x2^29*x3^16*x4^8*z^17 - 4*x1^31*x2^30*x3^16*x4^8*z^17 - 2*x1^38*x2^22*x3^17*x4^8*z^17 - x1^36*x2^24*x3^17*x4^8*z^17 - x1^35*x2^25*x3^17*x4^8*z^17 + 4*x1^33*x2^27*x3^17*x4^8*z^17 + 2*x1^32*x2^28*x3^17*x4^8*z^17 + 4*x1^31*x2^29*x3^17*x4^8*z^17 + x1^30*x2^30*x3^17*x4^8*z^17 + 3*x1^37*x2^22*x3^18*x4^8*z^17 + 2*x1^35*x2^24*x3^18*x4^8*z^17 - 2*x1^32*x2^27*x3^18*x4^8*z^17 - x1^31*x2^28*x3^18*x4^8*z^17 - 4*x1^30*x2^29*x3^18*x4^8*z^17 - x1^37*x2^21*x3^19*x4^8*z^17 - 2*x1^36*x2^22*x3^19*x4^8*z^17 - x1^35*x2^23*x3^19*x4^8*z^17 - x1^34*x2^24*x3^19*x4^8*z^17 - 2*x1^33*x2^25*x3^19*x4^8*z^17 + x1^31*x2^27*x3^19*x4^8*z^17 + 4*x1^30*x2^28*x3^19*x4^8*z^17 + 2*x1^29*x2^29*x3^19*x4^8*z^17 + x1^36*x2^21*x3^20*x4^8*z^17 + 2*x1^35*x2^22*x3^20*x4^8*z^17 + 2*x1^34*x2^23*x3^20*x4^8*z^17 + x1^33*x2^24*x3^20*x4^8*z^17 + x1^32*x2^25*x3^20*x4^8*z^17 - 3*x1^31*x2^26*x3^20*x4^8*z^17 - 2*x1^30*x2^27*x3^20*x4^8*z^17 - 4*x1^29*x2^28*x3^20*x4^8*z^17 - 2*x1^34*x2^22*x3^21*x4^8*z^17 - x1^33*x2^23*x3^21*x4^8*z^17 + x1^31*x2^25*x3^21*x4^8*z^17 + x1^30*x2^26*x3^21*x4^8*z^17 + x1^28*x2^28*x3^21*x4^8*z^17 + x1^33*x2^22*x3^22*x4^8*z^17 + x1^31*x2^24*x3^22*x4^8*z^17 - x1^30*x2^25*x3^22*x4^8*z^17 + x1^29*x2^26*x3^22*x4^8*z^17 - x1^30*x2^24*x3^23*x4^8*z^17 + 2*x1^29*x2^25*x3^23*x4^8*z^17 - x1^28*x2^26*x3^23*x4^8*z^17 - x1^27*x2^27*x3^23*x4^8*z^17 - x1^27*x2^26*x3^24*x4^8*z^17 + x1^38*x2^29*x3^9*x4^9*z^17 - x1^37*x2^30*x3^9*x4^9*z^17 - x1^38*x2^28*x3^10*x4^9*z^17 - x1^37*x2^29*x3^10*x4^9*z^17 - x1^36*x2^30*x3^10*x4^9*z^17 - x1^35*x2^31*x3^10*x4^9*z^17 + x1^37*x2^28*x3^11*x4^9*z^17 + x1^33*x2^32*x3^11*x4^9*z^17 + x1^40*x2^24*x3^12*x4^9*z^17 + x1^38*x2^26*x3^12*x4^9*z^17 - x1^37*x2^27*x3^12*x4^9*z^17 - x1^36*x2^28*x3^12*x4^9*z^17 - x1^35*x2^29*x3^12*x4^9*z^17 - x1^33*x2^31*x3^12*x4^9*z^17 - 2*x1^39*x2^24*x3^13*x4^9*z^17 - x1^37*x2^26*x3^13*x4^9*z^17 + x1^36*x2^27*x3^13*x4^9*z^17 + 3*x1^35*x2^28*x3^13*x4^9*z^17 + 3*x1^34*x2^29*x3^13*x4^9*z^17 + 3*x1^32*x2^31*x3^13*x4^9*z^17 + x1^39*x2^23*x3^14*x4^9*z^17 + x1^38*x2^24*x3^14*x4^9*z^17 + 2*x1^37*x2^25*x3^14*x4^9*z^17 - 2*x1^36*x2^26*x3^14*x4^9*z^17 - 2*x1^35*x2^27*x3^14*x4^9*z^17 - 3*x1^34*x2^28*x3^14*x4^9*z^17 - 3*x1^33*x2^29*x3^14*x4^9*z^17 - x1^32*x2^30*x3^14*x4^9*z^17 - x1^31*x2^31*x3^14*x4^9*z^17 - 3*x1^38*x2^23*x3^15*x4^9*z^17 - x1^37*x2^24*x3^15*x4^9*z^17 - x1^36*x2^25*x3^15*x4^9*z^17 + x1^35*x2^26*x3^15*x4^9*z^17 + 3*x1^33*x2^28*x3^15*x4^9*z^17 + 2*x1^32*x2^29*x3^15*x4^9*z^17 + 2*x1^31*x2^30*x3^15*x4^9*z^17 + 2*x1^38*x2^22*x3^16*x4^9*z^17 + 2*x1^37*x2^23*x3^16*x4^9*z^17 + 2*x1^36*x2^24*x3^16*x4^9*z^17 - 4*x1^33*x2^27*x3^16*x4^9*z^17 - 4*x1^32*x2^28*x3^16*x4^9*z^17 - 2*x1^31*x2^29*x3^16*x4^9*z^17 - x1^38*x2^21*x3^17*x4^9*z^17 - 4*x1^37*x2^22*x3^17*x4^9*z^17 - x1^36*x2^23*x3^17*x4^9*z^17 - x1^35*x2^24*x3^17*x4^9*z^17 + 3*x1^34*x2^25*x3^17*x4^9*z^17 + x1^33*x2^26*x3^17*x4^9*z^17 + 4*x1^32*x2^27*x3^17*x4^9*z^17 + x1^31*x2^28*x3^17*x4^9*z^17 + 2*x1^30*x2^29*x3^17*x4^9*z^17 + 2*x1^37*x2^21*x3^18*x4^9*z^17 + x1^36*x2^22*x3^18*x4^9*z^17 + x1^35*x2^23*x3^18*x4^9*z^17 - 2*x1^34*x2^24*x3^18*x4^9*z^17 - x1^33*x2^25*x3^18*x4^9*z^17 - x1^32*x2^26*x3^18*x4^9*z^17 - 4*x1^31*x2^27*x3^18*x4^9*z^17 - 2*x1^30*x2^28*x3^18*x4^9*z^17 - x1^29*x2^29*x3^18*x4^9*z^17 - 2*x1^36*x2^21*x3^19*x4^9*z^17 - x1^35*x2^22*x3^19*x4^9*z^17 - 4*x1^34*x2^23*x3^19*x4^9*z^17 + 2*x1^33*x2^24*x3^19*x4^9*z^17 + 4*x1^31*x2^26*x3^19*x4^9*z^17 + x1^30*x2^27*x3^19*x4^9*z^17 + 2*x1^29*x2^28*x3^19*x4^9*z^17 + x1^35*x2^21*x3^20*x4^9*z^17 + x1^34*x2^22*x3^20*x4^9*z^17 - 2*x1^32*x2^24*x3^20*x4^9*z^17 - 2*x1^31*x2^25*x3^20*x4^9*z^17 - 3*x1^30*x2^26*x3^20*x4^9*z^17 - x1^28*x2^28*x3^20*x4^9*z^17 + x1^33*x2^22*x3^21*x4^9*z^17 + 3*x1^32*x2^23*x3^21*x4^9*z^17 - x1^31*x2^24*x3^21*x4^9*z^17 + 4*x1^30*x2^25*x3^21*x4^9*z^17 + x1^29*x2^26*x3^21*x4^9*z^17 + x1^30*x2^24*x3^22*x4^9*z^17 - 4*x1^29*x2^25*x3^22*x4^9*z^17 + x1^27*x2^27*x3^22*x4^9*z^17 - x1^30*x2^23*x3^23*x4^9*z^17 + x1^29*x2^24*x3^23*x4^9*z^17 + x1^28*x2^25*x3^23*x4^9*z^17 - x1^28*x2^24*x3^24*x4^9*z^17 + x1^36*x2^29*x3^10*x4^10*z^17 + x1^35*x2^30*x3^10*x4^10*z^17 + x1^37*x2^27*x3^11*x4^10*z^17 + x1^35*x2^29*x3^11*x4^10*z^17 + x1^36*x2^27*x3^12*x4^10*z^17 - x1^35*x2^28*x3^12*x4^10*z^17 - x1^34*x2^29*x3^12*x4^10*z^17 - 2*x1^32*x2^31*x3^12*x4^10*z^17 + 2*x1^34*x2^28*x3^13*x4^10*z^17 + x1^38*x2^23*x3^14*x4^10*z^17 + x1^35*x2^26*x3^14*x4^10*z^17 - x1^34*x2^27*x3^14*x4^10*z^17 - x1^31*x2^30*x3^14*x4^10*z^17 - 2*x1^36*x2^24*x3^15*x4^10*z^17 + x1^35*x2^25*x3^15*x4^10*z^17 - x1^34*x2^26*x3^15*x4^10*z^17 + x1^33*x2^27*x3^15*x4^10*z^17 + x1^37*x2^22*x3^16*x4^10*z^17 + x1^36*x2^23*x3^16*x4^10*z^17 - 2*x1^34*x2^25*x3^16*x4^10*z^17 + x1^32*x2^27*x3^16*x4^10*z^17 - x1^31*x2^28*x3^16*x4^10*z^17 - x1^30*x2^29*x3^16*x4^10*z^17 + x1^36*x2^22*x3^17*x4^10*z^17 + x1^35*x2^23*x3^17*x4^10*z^17 - x1^33*x2^25*x3^17*x4^10*z^17 + x1^32*x2^26*x3^17*x4^10*z^17 + x1^36*x2^21*x3^18*x4^10*z^17 - x1^35*x2^22*x3^18*x4^10*z^17 + x1^34*x2^23*x3^18*x4^10*z^17 - x1^33*x2^24*x3^18*x4^10*z^17 - x1^32*x2^25*x3^18*x4^10*z^17 - 2*x1^31*x2^26*x3^18*x4^10*z^17 - x1^35*x2^21*x3^19*x4^10*z^17 + x1^34*x2^22*x3^19*x4^10*z^17 + 2*x1^33*x2^23*x3^19*x4^10*z^17 + x1^32*x2^24*x3^19*x4^10*z^17 + 2*x1^31*x2^25*x3^19*x4^10*z^17 + 2*x1^30*x2^26*x3^19*x4^10*z^17 + x1^29*x2^27*x3^19*x4^10*z^17 + x1^34*x2^21*x3^20*x4^10*z^17 - x1^32*x2^23*x3^20*x4^10*z^17 - x1^31*x2^24*x3^20*x4^10*z^17 - 3*x1^30*x2^25*x3^20*x4^10*z^17 - x1^29*x2^26*x3^20*x4^10*z^17 - x1^28*x2^27*x3^20*x4^10*z^17 - x1^33*x2^21*x3^21*x4^10*z^17 + x1^31*x2^23*x3^21*x4^10*z^17 + 2*x1^30*x2^24*x3^21*x4^10*z^17 + 2*x1^29*x2^25*x3^21*x4^10*z^17 + x1^28*x2^26*x3^21*x4^10*z^17 - x1^31*x2^22*x3^22*x4^10*z^17 - 2*x1^29*x2^24*x3^22*x4^10*z^17 - x1^27*x2^26*x3^22*x4^10*z^17 - x1^29*x2^23*x3^23*x4^10*z^17 + 2*x1^28*x2^24*x3^23*x4^10*z^17 - x1^27*x2^24*x3^24*x4^10*z^17 - x1^36*x2^27*x3^11*x4^11*z^17 + x1^35*x2^28*x3^11*x4^11*z^17 - x1^34*x2^29*x3^11*x4^11*z^17 - x1^36*x2^26*x3^12*x4^11*z^17 + x1^35*x2^27*x3^12*x4^11*z^17 - 2*x1^35*x2^26*x3^13*x4^11*z^17 + 2*x1^34*x2^27*x3^13*x4^11*z^17 + x1^33*x2^28*x3^13*x4^11*z^17 - x1^32*x2^29*x3^13*x4^11*z^17 + x1^31*x2^30*x3^13*x4^11*z^17 - x1^38*x2^22*x3^14*x4^11*z^17 - x1^35*x2^25*x3^14*x4^11*z^17 + 2*x1^34*x2^26*x3^14*x4^11*z^17 - x1^33*x2^27*x3^14*x4^11*z^17 + 2*x1^32*x2^28*x3^14*x4^11*z^17 - x1^31*x2^29*x3^14*x4^11*z^17 - 2*x1^30*x2^30*x3^14*x4^11*z^17 + x1^36*x2^23*x3^15*x4^11*z^17 - x1^35*x2^24*x3^15*x4^11*z^17 - 4*x1^34*x2^25*x3^15*x4^11*z^17 + x1^32*x2^27*x3^15*x4^11*z^17 + x1^31*x2^28*x3^15*x4^11*z^17 + x1^30*x2^29*x3^15*x4^11*z^17 + x1^36*x2^22*x3^16*x4^11*z^17 + 2*x1^35*x2^23*x3^16*x4^11*z^17 + x1^34*x2^24*x3^16*x4^11*z^17 + x1^33*x2^25*x3^16*x4^11*z^17 - 2*x1^32*x2^26*x3^16*x4^11*z^17 + 4*x1^31*x2^27*x3^16*x4^11*z^17 - x1^30*x2^28*x3^16*x4^11*z^17 - x1^29*x2^29*x3^16*x4^11*z^17 + x1^34*x2^23*x3^17*x4^11*z^17 + 2*x1^32*x2^25*x3^17*x4^11*z^17 - 4*x1^31*x2^26*x3^17*x4^11*z^17 + x1^30*x2^27*x3^17*x4^11*z^17 + 3*x1^29*x2^28*x3^17*x4^11*z^17 + 2*x1^34*x2^22*x3^18*x4^11*z^17 + x1^33*x2^23*x3^18*x4^11*z^17 + x1^32*x2^24*x3^18*x4^11*z^17 - x1^31*x2^25*x3^18*x4^11*z^17 + 5*x1^30*x2^26*x3^18*x4^11*z^17 - 2*x1^29*x2^27*x3^18*x4^11*z^17 - x1^33*x2^22*x3^19*x4^11*z^17 - x1^32*x2^23*x3^19*x4^11*z^17 - x1^30*x2^25*x3^19*x4^11*z^17 - 3*x1^29*x2^26*x3^19*x4^11*z^17 + 2*x1^28*x2^27*x3^19*x4^11*z^17 + x1^31*x2^23*x3^20*x4^11*z^17 + x1^29*x2^25*x3^20*x4^11*z^17 - 2*x1^28*x2^26*x3^20*x4^11*z^17 - x1^27*x2^27*x3^20*x4^11*z^17 + x1^29*x2^24*x3^21*x4^11*z^17 + x1^28*x2^25*x3^21*x4^11*z^17 + 2*x1^27*x2^26*x3^21*x4^11*z^17 + x1^35*x2^26*x3^12*x4^12*z^17 - x1^34*x2^27*x3^12*x4^12*z^17 + x1^33*x2^28*x3^12*x4^12*z^17 + x1^32*x2^29*x3^12*x4^12*z^17 - 2*x1^34*x2^26*x3^13*x4^12*z^17 + x1^33*x2^27*x3^13*x4^12*z^17 - x1^32*x2^28*x3^13*x4^12*z^17 + x1^31*x2^29*x3^13*x4^12*z^17 + x1^30*x2^30*x3^13*x4^12*z^17 + x1^34*x2^25*x3^14*x4^12*z^17 - 2*x1^33*x2^26*x3^14*x4^12*z^17 - x1^32*x2^27*x3^14*x4^12*z^17 + x1^31*x2^28*x3^14*x4^12*z^17 - 2*x1^30*x2^29*x3^14*x4^12*z^17 + x1^34*x2^24*x3^15*x4^12*z^17 - x1^33*x2^25*x3^15*x4^12*z^17 + 3*x1^32*x2^26*x3^15*x4^12*z^17 + 2*x1^30*x2^28*x3^15*x4^12*z^17 + 2*x1^29*x2^29*x3^15*x4^12*z^17 - x1^36*x2^21*x3^16*x4^12*z^17 + 2*x1^34*x2^23*x3^16*x4^12*z^17 + x1^33*x2^24*x3^16*x4^12*z^17 - x1^32*x2^25*x3^16*x4^12*z^17 - x1^30*x2^27*x3^16*x4^12*z^17 - 4*x1^29*x2^28*x3^16*x4^12*z^17 - 2*x1^34*x2^22*x3^17*x4^12*z^17 - x1^33*x2^23*x3^17*x4^12*z^17 - x1^32*x2^24*x3^17*x4^12*z^17 + 4*x1^31*x2^25*x3^17*x4^12*z^17 - 2*x1^30*x2^26*x3^17*x4^12*z^17 + 5*x1^29*x2^27*x3^17*x4^12*z^17 + 2*x1^28*x2^28*x3^17*x4^12*z^17 + 2*x1^33*x2^22*x3^18*x4^12*z^17 + x1^32*x2^23*x3^18*x4^12*z^17 - 3*x1^31*x2^24*x3^18*x4^12*z^17 - x1^30*x2^25*x3^18*x4^12*z^17 - 6*x1^28*x2^27*x3^18*x4^12*z^17 + x1^30*x2^24*x3^19*x4^12*z^17 - x1^29*x2^25*x3^19*x4^12*z^17 + 5*x1^28*x2^26*x3^19*x4^12*z^17 + 2*x1^27*x2^27*x3^19*x4^12*z^17 - x1^28*x2^25*x3^20*x4^12*z^17 - 5*x1^27*x2^26*x3^20*x4^12*z^17 - 2*x1^28*x2^24*x3^21*x4^12*z^17 + 2*x1^27*x2^25*x3^21*x4^12*z^17 + x1^26*x2^26*x3^21*x4^12*z^17 - 2*x1^26*x2^25*x3^22*x4^12*z^17 + x1^34*x2^24*x3^14*x4^13*z^17 + x1^33*x2^25*x3^14*x4^13*z^17 - x1^32*x2^26*x3^14*x4^13*z^17 + x1^31*x2^27*x3^14*x4^13*z^17 + x1^32*x2^25*x3^15*x4^13*z^17 - x1^31*x2^26*x3^15*x4^13*z^17 - x1^30*x2^27*x3^15*x4^13*z^17 + x1^29*x2^28*x3^15*x4^13*z^17 - 3*x1^31*x2^25*x3^16*x4^13*z^17 + x1^30*x2^26*x3^16*x4^13*z^17 - 2*x1^29*x2^27*x3^16*x4^13*z^17 - x1^28*x2^28*x3^16*x4^13*z^17 - x1^34*x2^21*x3^17*x4^13*z^17 - x1^33*x2^22*x3^17*x4^13*z^17 + x1^32*x2^23*x3^17*x4^13*z^17 + 2*x1^31*x2^24*x3^17*x4^13*z^17 - x1^30*x2^25*x3^17*x4^13*z^17 + 4*x1^28*x2^27*x3^17*x4^13*z^17 - x1^32*x2^22*x3^18*x4^13*z^17 - 2*x1^30*x2^24*x3^18*x4^13*z^17 + 2*x1^29*x2^25*x3^18*x4^13*z^17 - 3*x1^28*x2^26*x3^18*x4^13*z^17 - 2*x1^27*x2^27*x3^18*x4^13*z^17 - x1^31*x2^22*x3^19*x4^13*z^17 + 2*x1^30*x2^23*x3^19*x4^13*z^17 - 2*x1^28*x2^25*x3^19*x4^13*z^17 + 4*x1^27*x2^26*x3^19*x4^13*z^17 + x1^28*x2^24*x3^20*x4^13*z^17 - x1^27*x2^25*x3^20*x4^13*z^17 - 2*x1^26*x2^26*x3^20*x4^13*z^17 + x1^26*x2^25*x3^21*x4^13*z^17 - x1^33*x2^23*x3^15*x4^14*z^17 - 2*x1^30*x2^26*x3^15*x4^14*z^17 + x1^32*x2^23*x3^16*x4^14*z^17 + x1^31*x2^24*x3^16*x4^14*z^17 + x1^30*x2^25*x3^16*x4^14*z^17 + x1^30*x2^24*x3^17*x4^14*z^17 - 3*x1^29*x2^25*x3^17*x4^14*z^17 - x1^30*x2^23*x3^18*x4^14*z^17 + 2*x1^29*x2^24*x3^18*x4^14*z^17 - 2*x1^27*x2^26*x3^18*x4^14*z^17 - x1^30*x2^22*x3^19*x4^14*z^17 - x1^29*x2^23*x3^19*x4^14*z^17 - 2*x1^28*x2^24*x3^19*x4^14*z^17 + x1^27*x2^24*x3^20*x4^14*z^17 + 2*x1^29*x2^25*x3^16*x4^15*z^17 - x1^31*x2^22*x3^17*x4^15*z^17 - x1^30*x2^23*x3^17*x4^15*z^17 - x1^29*x2^24*x3^17*x4^15*z^17 - x1^28*x2^25*x3^17*x4^15*z^17 - x1^29*x2^23*x3^18*x4^15*z^17 + 3*x1^28*x2^24*x3^18*x4^15*z^17 + x1^29*x2^22*x3^19*x4^15*z^17 - 2*x1^27*x2^24*x3^19*x4^15*z^17 + x1^27*x2^23*x3^20*x4^15*z^17 + x1^27*x2^24*x3^18*x4^16*z^17 + x1^26*x2^23*x3^20*x4^16*z^17 + x1^42*x2^28*x3^10*z^16 - 2*x1^41*x2^28*x3^11*z^16 + x1^40*x2^29*x3^11*z^16 + 2*x1^40*x2^28*x3^12*z^16 + x1^38*x2^30*x3^12*z^16 - 2*x1^40*x2^27*x3^13*z^16 - x1^39*x2^28*x3^13*z^16 - x1^38*x2^29*x3^13*z^16 + x1^36*x2^31*x3^13*z^16 + x1^40*x2^26*x3^14*z^16 + 2*x1^39*x2^27*x3^14*z^16 + x1^37*x2^29*x3^14*z^16 - x1^35*x2^31*x3^14*z^16 - 2*x1^39*x2^26*x3^15*z^16 - x1^38*x2^27*x3^15*z^16 - x1^37*x2^28*x3^15*z^16 - x1^36*x2^29*x3^15*z^16 + x1^34*x2^31*x3^15*z^16 + x1^39*x2^25*x3^16*z^16 + 2*x1^38*x2^26*x3^16*z^16 + x1^36*x2^28*x3^16*z^16 - x1^35*x2^29*x3^16*z^16 + x1^34*x2^30*x3^16*z^16 - x1^33*x2^31*x3^16*z^16 - 2*x1^38*x2^25*x3^17*z^16 - x1^35*x2^28*x3^17*z^16 - x1^34*x2^29*x3^17*z^16 + x1^33*x2^30*x3^17*z^16 + x1^32*x2^31*x3^17*z^16 + x1^37*x2^25*x3^18*z^16 + x1^36*x2^26*x3^18*z^16 + x1^35*x2^27*x3^18*z^16 - 2*x1^37*x2^24*x3^19*z^16 - x1^34*x2^27*x3^19*z^16 + 2*x1^34*x2^26*x3^20*z^16 + x1^42*x2^29*x3^8*x4*z^16 - x1^41*x2^29*x3^9*x4*z^16 + 3*x1^41*x2^28*x3^10*x4*z^16 + x1^39*x2^30*x3^10*x4*z^16 - 2*x1^41*x2^27*x3^11*x4*z^16 - 3*x1^40*x2^28*x3^11*x4*z^16 + x1^39*x2^29*x3^11*x4*z^16 - 2*x1^38*x2^30*x3^11*x4*z^16 - x1^37*x2^31*x3^11*x4*z^16 + 6*x1^40*x2^27*x3^12*x4*z^16 + x1^38*x2^29*x3^12*x4*z^16 - x1^35*x2^32*x3^12*x4*z^16 - 2*x1^40*x2^26*x3^13*x4*z^16 - 6*x1^39*x2^27*x3^13*x4*z^16 - 4*x1^37*x2^29*x3^13*x4*z^16 + x1^36*x2^30*x3^13*x4*z^16 + x1^34*x2^32*x3^13*x4*z^16 + 6*x1^39*x2^26*x3^14*x4*z^16 + 2*x1^38*x2^27*x3^14*x4*z^16 + 2*x1^37*x2^28*x3^14*x4*z^16 - x1^35*x2^30*x3^14*x4*z^16 - x1^34*x2^31*x3^14*x4*z^16 - x1^33*x2^32*x3^14*x4*z^16 - 2*x1^39*x2^25*x3^15*x4*z^16 - 6*x1^38*x2^26*x3^15*x4*z^16 - 4*x1^36*x2^28*x3^15*x4*z^16 + 4*x1^35*x2^29*x3^15*x4*z^16 + x1^34*x2^30*x3^15*x4*z^16 + 4*x1^33*x2^31*x3^15*x4*z^16 + x1^32*x2^32*x3^15*x4*z^16 + 6*x1^38*x2^25*x3^16*x4*z^16 + 2*x1^37*x2^26*x3^16*x4*z^16 + 2*x1^36*x2^27*x3^16*x4*z^16 + x1^35*x2^28*x3^16*x4*z^16 - 2*x1^34*x2^29*x3^16*x4*z^16 - 2*x1^33*x2^30*x3^16*x4*z^16 - 3*x1^32*x2^31*x3^16*x4*z^16 - 2*x1^38*x2^24*x3^17*x4*z^16 - 6*x1^37*x2^25*x3^17*x4*z^16 - 4*x1^35*x2^27*x3^17*x4*z^16 + x1^34*x2^28*x3^17*x4*z^16 - x1^33*x2^29*x3^17*x4*z^16 + 5*x1^32*x2^30*x3^17*x4*z^16 + x1^31*x2^31*x3^17*x4*z^16 + 6*x1^37*x2^24*x3^18*x4*z^16 + 2*x1^36*x2^25*x3^18*x4*z^16 + 2*x1^35*x2^26*x3^18*x4*z^16 + 2*x1^34*x2^27*x3^18*x4*z^16 - 2*x1^32*x2^29*x3^18*x4*z^16 - 2*x1^31*x2^30*x3^18*x4*z^16 - x1^37*x2^23*x3^19*x4*z^16 - 5*x1^36*x2^24*x3^19*x4*z^16 - 2*x1^35*x2^25*x3^19*x4*z^16 - 3*x1^34*x2^26*x3^19*x4*z^16 - x1^32*x2^28*x3^19*x4*z^16 + 3*x1^31*x2^29*x3^19*x4*z^16 + 3*x1^36*x2^23*x3^20*x4*z^16 + x1^35*x2^24*x3^20*x4*z^16 + 2*x1^34*x2^25*x3^20*x4*z^16 + 3*x1^33*x2^26*x3^20*x4*z^16 - x1^32*x2^27*x3^20*x4*z^16 - x1^31*x2^28*x3^20*x4*z^16 - x1^35*x2^23*x3^21*x4*z^16 - 3*x1^33*x2^25*x3^21*x4*z^16 + x1^33*x2^24*x3^22*x4*z^16 + 2*x1^32*x2^25*x3^22*x4*z^16 - 2*x1^41*x2^28*x3^9*x4^2*z^16 + x1^40*x2^29*x3^9*x4^2*z^16 - x1^39*x2^30*x3^9*x4^2*z^16 + x1^41*x2^27*x3^10*x4^2*z^16 + 4*x1^40*x2^28*x3^10*x4^2*z^16 - 2*x1^39*x2^29*x3^10*x4^2*z^16 + x1^38*x2^30*x3^10*x4^2*z^16 - x1^37*x2^31*x3^10*x4^2*z^16 - 6*x1^40*x2^27*x3^11*x4^2*z^16 - x1^39*x2^28*x3^11*x4^2*z^16 + x1^37*x2^30*x3^11*x4^2*z^16 + 2*x1^36*x2^31*x3^11*x4^2*z^16 + 2*x1^40*x2^26*x3^12*x4^2*z^16 + 6*x1^39*x2^27*x3^12*x4^2*z^16 - x1^38*x2^28*x3^12*x4^2*z^16 + 2*x1^37*x2^29*x3^12*x4^2*z^16 - 3*x1^36*x2^30*x3^12*x4^2*z^16 - 2*x1^35*x2^31*x3^12*x4^2*z^16 - x1^34*x2^32*x3^12*x4^2*z^16 - 6*x1^39*x2^26*x3^13*x4^2*z^16 - 2*x1^38*x2^27*x3^13*x4^2*z^16 - x1^37*x2^28*x3^13*x4^2*z^16 + x1^36*x2^29*x3^13*x4^2*z^16 + 3*x1^35*x2^30*x3^13*x4^2*z^16 + 3*x1^34*x2^31*x3^13*x4^2*z^16 + x1^33*x2^32*x3^13*x4^2*z^16 + 2*x1^39*x2^25*x3^14*x4^2*z^16 + 6*x1^38*x2^26*x3^14*x4^2*z^16 + 4*x1^36*x2^28*x3^14*x4^2*z^16 - 4*x1^35*x2^29*x3^14*x4^2*z^16 - 2*x1^34*x2^30*x3^14*x4^2*z^16 - 5*x1^33*x2^31*x3^14*x4^2*z^16 - 6*x1^38*x2^25*x3^15*x4^2*z^16 - 2*x1^37*x2^26*x3^15*x4^2*z^16 - 2*x1^36*x2^27*x3^15*x4^2*z^16 + 2*x1^34*x2^29*x3^15*x4^2*z^16 + 3*x1^33*x2^30*x3^15*x4^2*z^16 + 5*x1^32*x2^31*x3^15*x4^2*z^16 + 2*x1^38*x2^24*x3^16*x4^2*z^16 + 6*x1^37*x2^25*x3^16*x4^2*z^16 + 4*x1^35*x2^27*x3^16*x4^2*z^16 - 4*x1^34*x2^28*x3^16*x4^2*z^16 - 6*x1^32*x2^30*x3^16*x4^2*z^16 - 2*x1^31*x2^31*x3^16*x4^2*z^16 - 6*x1^37*x2^24*x3^17*x4^2*z^16 - 2*x1^36*x2^25*x3^17*x4^2*z^16 - 2*x1^35*x2^26*x3^17*x4^2*z^16 + 2*x1^33*x2^28*x3^17*x4^2*z^16 + 2*x1^32*x2^29*x3^17*x4^2*z^16 + 6*x1^31*x2^30*x3^17*x4^2*z^16 + 6*x1^36*x2^24*x3^18*x4^2*z^16 + 4*x1^34*x2^26*x3^18*x4^2*z^16 - 2*x1^33*x2^27*x3^18*x4^2*z^16 - 6*x1^31*x2^29*x3^18*x4^2*z^16 - x1^30*x2^30*x3^18*x4^2*z^16 - 2*x1^36*x2^23*x3^19*x4^2*z^16 - 2*x1^35*x2^24*x3^19*x4^2*z^16 - x1^34*x2^25*x3^19*x4^2*z^16 - x1^33*x2^26*x3^19*x4^2*z^16 + x1^32*x2^27*x3^19*x4^2*z^16 + x1^31*x2^28*x3^19*x4^2*z^16 + 3*x1^30*x2^29*x3^19*x4^2*z^16 + x1^35*x2^23*x3^20*x4^2*z^16 + 2*x1^34*x2^24*x3^20*x4^2*z^16 + 4*x1^33*x2^25*x3^20*x4^2*z^16 + x1^32*x2^26*x3^20*x4^2*z^16 - 4*x1^30*x2^28*x3^20*x4^2*z^16 - x1^33*x2^24*x3^21*x4^2*z^16 - 2*x1^32*x2^25*x3^21*x4^2*z^16 + x1^30*x2^27*x3^21*x4^2*z^16 + x1^29*x2^28*x3^21*x4^2*z^16 + x1^31*x2^25*x3^22*x4^2*z^16 - 2*x1^29*x2^27*x3^22*x4^2*z^16 - x1^31*x2^24*x3^23*x4^2*z^16 + x1^42*x2^26*x3^9*x4^3*z^16 - 2*x1^40*x2^28*x3^9*x4^3*z^16 - x1^38*x2^30*x3^9*x4^3*z^16 - x1^42*x2^25*x3^10*x4^3*z^16 - x1^41*x2^26*x3^10*x4^3*z^16 + x1^39*x2^28*x3^10*x4^3*z^16 + x1^38*x2^29*x3^10*x4^3*z^16 - x1^36*x2^31*x3^10*x4^3*z^16 + 2*x1^41*x2^25*x3^11*x4^3*z^16 - 2*x1^39*x2^27*x3^11*x4^3*z^16 - x1^38*x2^28*x3^11*x4^3*z^16 - x1^37*x2^29*x3^11*x4^3*z^16 + x1^35*x2^31*x3^11*x4^3*z^16 - 2*x1^40*x2^25*x3^12*x4^3*z^16 + 2*x1^39*x2^26*x3^12*x4^3*z^16 - x1^38*x2^27*x3^12*x4^3*z^16 + 2*x1^37*x2^28*x3^12*x4^3*z^16 + 2*x1^36*x2^29*x3^12*x4^3*z^16 + x1^35*x2^30*x3^12*x4^3*z^16 - x1^34*x2^31*x3^12*x4^3*z^16 + 2*x1^40*x2^24*x3^13*x4^3*z^16 + x1^39*x2^25*x3^13*x4^3*z^16 - x1^38*x2^26*x3^13*x4^3*z^16 - 2*x1^36*x2^28*x3^13*x4^3*z^16 - x1^34*x2^30*x3^13*x4^3*z^16 + 2*x1^33*x2^31*x3^13*x4^3*z^16 - x1^40*x2^23*x3^14*x4^3*z^16 - 2*x1^39*x2^24*x3^14*x4^3*z^16 + 2*x1^38*x2^25*x3^14*x4^3*z^16 + 2*x1^36*x2^27*x3^14*x4^3*z^16 + 2*x1^34*x2^29*x3^14*x4^3*z^16 + x1^33*x2^30*x3^14*x4^3*z^16 - 2*x1^32*x2^31*x3^14*x4^3*z^16 + 2*x1^39*x2^23*x3^15*x4^3*z^16 - x1^37*x2^25*x3^15*x4^3*z^16 - 2*x1^35*x2^27*x3^15*x4^3*z^16 - 2*x1^33*x2^29*x3^15*x4^3*z^16 + 2*x1^32*x2^30*x3^15*x4^3*z^16 + x1^31*x2^31*x3^15*x4^3*z^16 - x1^39*x2^22*x3^16*x4^3*z^16 - 2*x1^38*x2^23*x3^16*x4^3*z^16 + 2*x1^37*x2^24*x3^16*x4^3*z^16 + 3*x1^35*x2^26*x3^16*x4^3*z^16 + x1^33*x2^28*x3^16*x4^3*z^16 - x1^32*x2^29*x3^16*x4^3*z^16 - 2*x1^31*x2^30*x3^16*x4^3*z^16 + 2*x1^38*x2^22*x3^17*x4^3*z^16 - 2*x1^36*x2^24*x3^17*x4^3*z^16 - x1^34*x2^26*x3^17*x4^3*z^16 + 2*x1^33*x2^27*x3^17*x4^3*z^16 - 2*x1^32*x2^28*x3^17*x4^3*z^16 + 2*x1^31*x2^29*x3^17*x4^3*z^16 - 2*x1^37*x2^22*x3^18*x4^3*z^16 + x1^36*x2^23*x3^18*x4^3*z^16 - 2*x1^35*x2^24*x3^18*x4^3*z^16 + x1^32*x2^27*x3^18*x4^3*z^16 - 4*x1^30*x2^29*x3^18*x4^3*z^16 + x1^37*x2^21*x3^19*x4^3*z^16 + 2*x1^36*x2^22*x3^19*x4^3*z^16 - x1^33*x2^25*x3^19*x4^3*z^16 + x1^32*x2^26*x3^19*x4^3*z^16 + x1^31*x2^27*x3^19*x4^3*z^16 + 2*x1^30*x2^28*x3^19*x4^3*z^16 - x1^36*x2^21*x3^20*x4^3*z^16 - x1^35*x2^22*x3^20*x4^3*z^16 - 2*x1^32*x2^25*x3^20*x4^3*z^16 + x1^31*x2^26*x3^20*x4^3*z^16 + x1^30*x2^27*x3^20*x4^3*z^16 - 2*x1^29*x2^28*x3^20*x4^3*z^16 + x1^34*x2^22*x3^21*x4^3*z^16 - x1^32*x2^24*x3^21*x4^3*z^16 + 2*x1^30*x2^26*x3^21*x4^3*z^16 + 3*x1^29*x2^27*x3^21*x4^3*z^16 + x1^32*x2^23*x3^22*x4^3*z^16 - x1^28*x2^27*x3^22*x4^3*z^16 + x1^31*x2^23*x3^23*x4^3*z^16 + x1^28*x2^26*x3^23*x4^3*z^16 - x1^42*x2^26*x3^8*x4^4*z^16 + x1^39*x2^29*x3^8*x4^4*z^16 + 2*x1^41*x2^26*x3^9*x4^4*z^16 + x1^37*x2^30*x3^9*x4^4*z^16 - 3*x1^41*x2^25*x3^10*x4^4*z^16 - 2*x1^40*x2^26*x3^10*x4^4*z^16 - x1^39*x2^27*x3^10*x4^4*z^16 + x1^36*x2^30*x3^10*x4^4*z^16 + 2*x1^41*x2^24*x3^11*x4^4*z^16 + 5*x1^40*x2^25*x3^11*x4^4*z^16 + x1^39*x2^26*x3^11*x4^4*z^16 - 2*x1^37*x2^28*x3^11*x4^4*z^16 - x1^36*x2^29*x3^11*x4^4*z^16 - x1^35*x2^30*x3^11*x4^4*z^16 - x1^34*x2^31*x3^11*x4^4*z^16 - 6*x1^40*x2^24*x3^12*x4^4*z^16 - 2*x1^39*x2^25*x3^12*x4^4*z^16 - 3*x1^38*x2^26*x3^12*x4^4*z^16 - x1^37*x2^27*x3^12*x4^4*z^16 + x1^35*x2^29*x3^12*x4^4*z^16 + x1^34*x2^30*x3^12*x4^4*z^16 + x1^33*x2^31*x3^12*x4^4*z^16 + 2*x1^40*x2^23*x3^13*x4^4*z^16 + 6*x1^39*x2^24*x3^13*x4^4*z^16 + 4*x1^37*x2^26*x3^13*x4^4*z^16 - 4*x1^36*x2^27*x3^13*x4^4*z^16 - 2*x1^35*x2^28*x3^13*x4^4*z^16 - 4*x1^34*x2^29*x3^13*x4^4*z^16 - x1^33*x2^30*x3^13*x4^4*z^16 - x1^32*x2^31*x3^13*x4^4*z^16 - 6*x1^39*x2^23*x3^14*x4^4*z^16 - 2*x1^38*x2^24*x3^14*x4^4*z^16 - 2*x1^37*x2^25*x3^14*x4^4*z^16 + 2*x1^35*x2^27*x3^14*x4^4*z^16 + 2*x1^34*x2^28*x3^14*x4^4*z^16 + 4*x1^33*x2^29*x3^14*x4^4*z^16 + x1^31*x2^31*x3^14*x4^4*z^16 + 2*x1^39*x2^22*x3^15*x4^4*z^16 + 6*x1^38*x2^23*x3^15*x4^4*z^16 + 4*x1^36*x2^25*x3^15*x4^4*z^16 - 4*x1^35*x2^26*x3^15*x4^4*z^16 - 6*x1^33*x2^28*x3^15*x4^4*z^16 - 2*x1^32*x2^29*x3^15*x4^4*z^16 - 5*x1^38*x2^22*x3^16*x4^4*z^16 - 2*x1^37*x2^23*x3^16*x4^4*z^16 - 2*x1^36*x2^24*x3^16*x4^4*z^16 + 2*x1^34*x2^26*x3^16*x4^4*z^16 + 2*x1^33*x2^27*x3^16*x4^4*z^16 + 6*x1^32*x2^28*x3^16*x4^4*z^16 + x1^38*x2^21*x3^17*x4^4*z^16 + 5*x1^37*x2^22*x3^17*x4^4*z^16 + x1^36*x2^23*x3^17*x4^4*z^16 + 4*x1^35*x2^24*x3^17*x4^4*z^16 - 4*x1^34*x2^25*x3^17*x4^4*z^16 - 6*x1^32*x2^27*x3^17*x4^4*z^16 - 2*x1^31*x2^28*x3^17*x4^4*z^16 - 2*x1^37*x2^21*x3^18*x4^4*z^16 - 2*x1^36*x2^22*x3^18*x4^4*z^16 - 2*x1^35*x2^23*x3^18*x4^4*z^16 + x1^34*x2^24*x3^18*x4^4*z^16 + 2*x1^33*x2^25*x3^18*x4^4*z^16 + 2*x1^32*x2^26*x3^18*x4^4*z^16 + 6*x1^31*x2^27*x3^18*x4^4*z^16 + 2*x1^36*x2^21*x3^19*x4^4*z^16 + 2*x1^35*x2^22*x3^19*x4^4*z^16 + 4*x1^34*x2^23*x3^19*x4^4*z^16 - 3*x1^33*x2^24*x3^19*x4^4*z^16 - 5*x1^31*x2^26*x3^19*x4^4*z^16 - x1^30*x2^27*x3^19*x4^4*z^16 + 2*x1^29*x2^28*x3^19*x4^4*z^16 - x1^35*x2^21*x3^20*x4^4*z^16 - 2*x1^34*x2^22*x3^20*x4^4*z^16 + x1^31*x2^25*x3^20*x4^4*z^16 + 2*x1^30*x2^26*x3^20*x4^4*z^16 - x1^32*x2^23*x3^21*x4^4*z^16 + 2*x1^31*x2^24*x3^21*x4^4*z^16 - 2*x1^30*x2^25*x3^21*x4^4*z^16 + 2*x1^28*x2^27*x3^21*x4^4*z^16 - x1^32*x2^22*x3^22*x4^4*z^16 - x1^31*x2^23*x3^22*x4^4*z^16 + x1^29*x2^25*x3^22*x4^4*z^16 + x1^30*x2^23*x3^23*x4^4*z^16 - x1^29*x2^24*x3^23*x4^4*z^16 + 2*x1^27*x2^26*x3^23*x4^4*z^16 - x1^38*x2^29*x3^8*x4^5*z^16 + x1^41*x2^25*x3^9*x4^5*z^16 + x1^40*x2^26*x3^9*x4^5*z^16 + 2*x1^39*x2^27*x3^9*x4^5*z^16 - 2*x1^38*x2^28*x3^9*x4^5*z^16 - x1^37*x2^29*x3^9*x4^5*z^16 + x1^36*x2^30*x3^9*x4^5*z^16 - 2*x1^40*x2^25*x3^10*x4^5*z^16 - x1^39*x2^26*x3^10*x4^5*z^16 - x1^38*x2^27*x3^10*x4^5*z^16 + 3*x1^37*x2^28*x3^10*x4^5*z^16 + 2*x1^40*x2^24*x3^11*x4^5*z^16 + x1^39*x2^25*x3^11*x4^5*z^16 - x1^37*x2^27*x3^11*x4^5*z^16 - 3*x1^36*x2^28*x3^11*x4^5*z^16 - 2*x1^35*x2^29*x3^11*x4^5*z^16 + x1^33*x2^31*x3^11*x4^5*z^16 - x1^40*x2^23*x3^12*x4^5*z^16 - 4*x1^39*x2^24*x3^12*x4^5*z^16 + x1^38*x2^25*x3^12*x4^5*z^16 + 3*x1^36*x2^27*x3^12*x4^5*z^16 + x1^35*x2^28*x3^12*x4^5*z^16 + 4*x1^34*x2^29*x3^12*x4^5*z^16 - x1^33*x2^30*x3^12*x4^5*z^16 - x1^32*x2^31*x3^12*x4^5*z^16 + 5*x1^39*x2^23*x3^13*x4^5*z^16 + 2*x1^38*x2^24*x3^13*x4^5*z^16 + x1^37*x2^25*x3^13*x4^5*z^16 - x1^36*x2^26*x3^13*x4^5*z^16 - 3*x1^35*x2^27*x3^13*x4^5*z^16 - x1^34*x2^28*x3^13*x4^5*z^16 - 5*x1^33*x2^29*x3^13*x4^5*z^16 + x1^32*x2^30*x3^13*x4^5*z^16 - 2*x1^39*x2^22*x3^14*x4^5*z^16 - 6*x1^38*x2^23*x3^14*x4^5*z^16 + x1^37*x2^24*x3^14*x4^5*z^16 - 3*x1^36*x2^25*x3^14*x4^5*z^16 + 3*x1^35*x2^26*x3^14*x4^5*z^16 - x1^34*x2^27*x3^14*x4^5*z^16 + 6*x1^33*x2^28*x3^14*x4^5*z^16 + x1^32*x2^29*x3^14*x4^5*z^16 - x1^31*x2^30*x3^14*x4^5*z^16 + 6*x1^38*x2^22*x3^15*x4^5*z^16 + 2*x1^37*x2^23*x3^15*x4^5*z^16 - 4*x1^34*x2^26*x3^15*x4^5*z^16 - x1^33*x2^27*x3^15*x4^5*z^16 - 6*x1^32*x2^28*x3^15*x4^5*z^16 + 2*x1^31*x2^29*x3^15*x4^5*z^16 - x1^38*x2^21*x3^16*x4^5*z^16 - 6*x1^37*x2^22*x3^16*x4^5*z^16 + 2*x1^36*x2^23*x3^16*x4^5*z^16 - 3*x1^35*x2^24*x3^16*x4^5*z^16 + 5*x1^34*x2^25*x3^16*x4^5*z^16 + 5*x1^32*x2^27*x3^16*x4^5*z^16 + x1^31*x2^28*x3^16*x4^5*z^16 - 2*x1^30*x2^29*x3^16*x4^5*z^16 + 2*x1^37*x2^21*x3^17*x4^5*z^16 - 2*x1^35*x2^23*x3^17*x4^5*z^16 - 3*x1^33*x2^25*x3^17*x4^5*z^16 - x1^32*x2^26*x3^17*x4^5*z^16 - 6*x1^31*x2^27*x3^17*x4^5*z^16 + 2*x1^30*x2^28*x3^17*x4^5*z^16 + x1^29*x2^29*x3^17*x4^5*z^16 - 2*x1^36*x2^21*x3^18*x4^5*z^16 + x1^35*x2^22*x3^18*x4^5*z^16 - 3*x1^34*x2^23*x3^18*x4^5*z^16 + 3*x1^33*x2^24*x3^18*x4^5*z^16 + 5*x1^31*x2^26*x3^18*x4^5*z^16 + x1^30*x2^27*x3^18*x4^5*z^16 - 2*x1^29*x2^28*x3^18*x4^5*z^16 + x1^35*x2^21*x3^19*x4^5*z^16 - x1^34*x2^22*x3^19*x4^5*z^16 - 2*x1^32*x2^24*x3^19*x4^5*z^16 - 6*x1^30*x2^26*x3^19*x4^5*z^16 + 2*x1^29*x2^27*x3^19*x4^5*z^16 - x1^34*x2^21*x3^20*x4^5*z^16 + x1^33*x2^22*x3^20*x4^5*z^16 + 2*x1^32*x2^23*x3^20*x4^5*z^16 - x1^31*x2^24*x3^20*x4^5*z^16 + 4*x1^30*x2^25*x3^20*x4^5*z^16 + x1^29*x2^26*x3^20*x4^5*z^16 - x1^28*x2^27*x3^20*x4^5*z^16 + x1^33*x2^21*x3^21*x4^5*z^16 - x1^32*x2^22*x3^21*x4^5*z^16 - x1^31*x2^23*x3^21*x4^5*z^16 - 2*x1^29*x2^25*x3^21*x4^5*z^16 + x1^28*x2^26*x3^21*x4^5*z^16 + x1^27*x2^27*x3^21*x4^5*z^16 + 2*x1^31*x2^22*x3^22*x4^5*z^16 + x1^29*x2^24*x3^22*x4^5*z^16 - x1^28*x2^25*x3^22*x4^5*z^16 - 2*x1^27*x2^26*x3^22*x4^5*z^16 + x1^29*x2^23*x3^23*x4^5*z^16 - x1^28*x2^24*x3^23*x4^5*z^16 + x1^26*x2^26*x3^23*x4^5*z^16 - x1^26*x2^25*x3^24*x4^5*z^16 - x1^40*x2^25*x3^9*x4^6*z^16 - x1^38*x2^27*x3^9*x4^6*z^16 - x1^37*x2^28*x3^9*x4^6*z^16 - x1^35*x2^30*x3^9*x4^6*z^16 - x1^40*x2^24*x3^10*x4^6*z^16 + x1^39*x2^25*x3^10*x4^6*z^16 + x1^38*x2^26*x3^10*x4^6*z^16 - x1^37*x2^27*x3^10*x4^6*z^16 + 2*x1^36*x2^28*x3^10*x4^6*z^16 - x1^39*x2^24*x3^11*x4^6*z^16 - x1^38*x2^25*x3^11*x4^6*z^16 - 2*x1^36*x2^27*x3^11*x4^6*z^16 - 2*x1^35*x2^28*x3^11*x4^6*z^16 + x1^34*x2^29*x3^11*x4^6*z^16 + x1^38*x2^24*x3^12*x4^6*z^16 + 3*x1^37*x2^25*x3^12*x4^6*z^16 + 2*x1^36*x2^26*x3^12*x4^6*z^16 - 2*x1^34*x2^28*x3^12*x4^6*z^16 + x1^33*x2^29*x3^12*x4^6*z^16 + x1^32*x2^30*x3^12*x4^6*z^16 - x1^39*x2^22*x3^13*x4^6*z^16 + x1^38*x2^23*x3^13*x4^6*z^16 - 3*x1^37*x2^24*x3^13*x4^6*z^16 - 2*x1^33*x2^28*x3^13*x4^6*z^16 + x1^32*x2^29*x3^13*x4^6*z^16 - x1^38*x2^22*x3^14*x4^6*z^16 - x1^37*x2^23*x3^14*x4^6*z^16 + 3*x1^36*x2^24*x3^14*x4^6*z^16 + 2*x1^35*x2^25*x3^14*x4^6*z^16 + 5*x1^34*x2^26*x3^14*x4^6*z^16 - 2*x1^33*x2^27*x3^14*x4^6*z^16 + 2*x1^32*x2^28*x3^14*x4^6*z^16 - 4*x1^31*x2^29*x3^14*x4^6*z^16 + x1^30*x2^30*x3^14*x4^6*z^16 + x1^38*x2^21*x3^15*x4^6*z^16 + 2*x1^37*x2^22*x3^15*x4^6*z^16 - 3*x1^36*x2^23*x3^15*x4^6*z^16 - x1^35*x2^24*x3^15*x4^6*z^16 - x1^34*x2^25*x3^15*x4^6*z^16 + x1^33*x2^26*x3^15*x4^6*z^16 - x1^32*x2^27*x3^15*x4^6*z^16 + 4*x1^30*x2^29*x3^15*x4^6*z^16 - x1^37*x2^21*x3^16*x4^6*z^16 + 3*x1^35*x2^23*x3^16*x4^6*z^16 + 4*x1^33*x2^25*x3^16*x4^6*z^16 - x1^32*x2^26*x3^16*x4^6*z^16 + 2*x1^31*x2^27*x3^16*x4^6*z^16 - 4*x1^30*x2^28*x3^16*x4^6*z^16 - 2*x1^29*x2^29*x3^16*x4^6*z^16 + x1^36*x2^21*x3^17*x4^6*z^16 - 3*x1^35*x2^22*x3^17*x4^6*z^16 - x1^34*x2^23*x3^17*x4^6*z^16 - 4*x1^33*x2^24*x3^17*x4^6*z^16 + 2*x1^30*x2^27*x3^17*x4^6*z^16 + 4*x1^29*x2^28*x3^17*x4^6*z^16 + x1^36*x2^20*x3^18*x4^6*z^16 + 4*x1^34*x2^22*x3^18*x4^6*z^16 + x1^33*x2^23*x3^18*x4^6*z^16 + 2*x1^32*x2^24*x3^18*x4^6*z^16 - 3*x1^31*x2^25*x3^18*x4^6*z^16 + 2*x1^30*x2^26*x3^18*x4^6*z^16 - 4*x1^29*x2^27*x3^18*x4^6*z^16 - x1^28*x2^28*x3^18*x4^6*z^16 - x1^35*x2^20*x3^19*x4^6*z^16 - x1^34*x2^21*x3^19*x4^6*z^16 - x1^33*x2^22*x3^19*x4^6*z^16 - 2*x1^32*x2^23*x3^19*x4^6*z^16 - x1^30*x2^25*x3^19*x4^6*z^16 + 4*x1^28*x2^27*x3^19*x4^6*z^16 + x1^33*x2^21*x3^20*x4^6*z^16 + x1^32*x2^22*x3^20*x4^6*z^16 + 3*x1^31*x2^23*x3^20*x4^6*z^16 - x1^30*x2^24*x3^20*x4^6*z^16 + 2*x1^29*x2^25*x3^20*x4^6*z^16 - 4*x1^28*x2^26*x3^20*x4^6*z^16 - x1^27*x2^27*x3^20*x4^6*z^16 - x1^32*x2^21*x3^21*x4^6*z^16 - 2*x1^31*x2^22*x3^21*x4^6*z^16 - 2*x1^29*x2^24*x3^21*x4^6*z^16 + 3*x1^27*x2^26*x3^21*x4^6*z^16 + x1^30*x2^22*x3^22*x4^6*z^16 + 2*x1^28*x2^24*x3^22*x4^6*z^16 - 3*x1^27*x2^25*x3^22*x4^6*z^16 - 2*x1^26*x2^26*x3^22*x4^6*z^16 + x1^26*x2^25*x3^23*x4^6*z^16 + x1^26*x2^24*x3^24*x4^6*z^16 - x1^37*x2^28*x3^8*x4^7*z^16 + x1^36*x2^29*x3^8*x4^7*z^16 + x1^35*x2^30*x3^8*x4^7*z^16 + x1^37*x2^27*x3^9*x4^7*z^16 - x1^36*x2^28*x3^9*x4^7*z^16 - 2*x1^35*x2^29*x3^9*x4^7*z^16 - x1^34*x2^30*x3^9*x4^7*z^16 - x1^33*x2^31*x3^9*x4^7*z^16 + x1^39*x2^24*x3^10*x4^7*z^16 - x1^38*x2^25*x3^10*x4^7*z^16 + x1^37*x2^26*x3^10*x4^7*z^16 - x1^36*x2^27*x3^10*x4^7*z^16 + 2*x1^33*x2^30*x3^10*x4^7*z^16 + 2*x1^32*x2^31*x3^10*x4^7*z^16 - 2*x1^38*x2^24*x3^11*x4^7*z^16 + x1^36*x2^26*x3^11*x4^7*z^16 - 2*x1^35*x2^27*x3^11*x4^7*z^16 - 2*x1^34*x2^28*x3^11*x4^7*z^16 + x1^33*x2^29*x3^11*x4^7*z^16 - 2*x1^32*x2^30*x3^11*x4^7*z^16 - x1^31*x2^31*x3^11*x4^7*z^16 + x1^38*x2^23*x3^12*x4^7*z^16 - x1^35*x2^26*x3^12*x4^7*z^16 + 2*x1^34*x2^27*x3^12*x4^7*z^16 - 2*x1^33*x2^28*x3^12*x4^7*z^16 - 2*x1^32*x2^29*x3^12*x4^7*z^16 + 3*x1^31*x2^30*x3^12*x4^7*z^16 - x1^38*x2^22*x3^13*x4^7*z^16 - x1^37*x2^23*x3^13*x4^7*z^16 - x1^36*x2^24*x3^13*x4^7*z^16 - x1^30*x2^30*x3^13*x4^7*z^16 + 2*x1^38*x2^21*x3^14*x4^7*z^16 + x1^35*x2^24*x3^14*x4^7*z^16 - x1^34*x2^25*x3^14*x4^7*z^16 + x1^33*x2^26*x3^14*x4^7*z^16 - 2*x1^37*x2^21*x3^15*x4^7*z^16 - 2*x1^35*x2^23*x3^15*x4^7*z^16 - 2*x1^34*x2^24*x3^15*x4^7*z^16 - x1^37*x2^20*x3^16*x4^7*z^16 + x1^36*x2^21*x3^16*x4^7*z^16 + x1^35*x2^22*x3^16*x4^7*z^16 + x1^33*x2^24*x3^16*x4^7*z^16 - 2*x1^35*x2^21*x3^17*x4^7*z^16 - 3*x1^34*x2^22*x3^17*x4^7*z^16 - 2*x1^32*x2^24*x3^17*x4^7*z^16 - x1^31*x2^25*x3^17*x4^7*z^16 + 2*x1^34*x2^21*x3^18*x4^7*z^16 + x1^32*x2^23*x3^18*x4^7*z^16 - x1^29*x2^26*x3^18*x4^7*z^16 + x1^34*x2^20*x3^19*x4^7*z^16 - 2*x1^33*x2^21*x3^19*x4^7*z^16 - 2*x1^31*x2^23*x3^19*x4^7*z^16 + 2*x1^30*x2^24*x3^19*x4^7*z^16 + x1^28*x2^26*x3^19*x4^7*z^16 + 2*x1^32*x2^21*x3^20*x4^7*z^16 + x1^31*x2^22*x3^20*x4^7*z^16 + x1^30*x2^23*x3^20*x4^7*z^16 + x1^28*x2^25*x3^20*x4^7*z^16 - x1^27*x2^26*x3^20*x4^7*z^16 - x1^30*x2^22*x3^21*x4^7*z^16 + x1^29*x2^23*x3^21*x4^7*z^16 + x1^27*x2^25*x3^21*x4^7*z^16 + x1^26*x2^26*x3^21*x4^7*z^16 + x1^29*x2^22*x3^22*x4^7*z^16 - x1^27*x2^24*x3^22*x4^7*z^16 - x1^26*x2^25*x3^22*x4^7*z^16 - 2*x1^27*x2^23*x3^23*x4^7*z^16 + x1^26*x2^24*x3^23*x4^7*z^16 + x1^25*x2^25*x3^23*x4^7*z^16 - x1^25*x2^24*x3^24*x4^7*z^16 + x1^37*x2^27*x3^8*x4^8*z^16 - x1^35*x2^29*x3^8*x4^8*z^16 + x1^36*x2^27*x3^9*x4^8*z^16 + x1^35*x2^27*x3^10*x4^8*z^16 + 2*x1^34*x2^28*x3^10*x4^8*z^16 - x1^33*x2^29*x3^10*x4^8*z^16 + 2*x1^38*x2^23*x3^11*x4^8*z^16 + x1^37*x2^24*x3^11*x4^8*z^16 - x1^34*x2^27*x3^11*x4^8*z^16 - x1^33*x2^28*x3^11*x4^8*z^16 - x1^32*x2^29*x3^11*x4^8*z^16 - 2*x1^31*x2^30*x3^11*x4^8*z^16 + x1^37*x2^23*x3^12*x4^8*z^16 + x1^35*x2^25*x3^12*x4^8*z^16 + 4*x1^33*x2^27*x3^12*x4^8*z^16 + 2*x1^31*x2^29*x3^12*x4^8*z^16 + x1^30*x2^30*x3^12*x4^8*z^16 + x1^37*x2^22*x3^13*x4^8*z^16 + x1^36*x2^23*x3^13*x4^8*z^16 - 3*x1^33*x2^26*x3^13*x4^8*z^16 - 2*x1^32*x2^27*x3^13*x4^8*z^16 + x1^31*x2^28*x3^13*x4^8*z^16 - 4*x1^30*x2^29*x3^13*x4^8*z^16 - x1^37*x2^21*x3^14*x4^8*z^16 - 2*x1^35*x2^23*x3^14*x4^8*z^16 + x1^34*x2^24*x3^14*x4^8*z^16 - 2*x1^33*x2^25*x3^14*x4^8*z^16 + 2*x1^32*x2^26*x3^14*x4^8*z^16 + x1^31*x2^27*x3^14*x4^8*z^16 + 2*x1^30*x2^28*x3^14*x4^8*z^16 + 2*x1^29*x2^29*x3^14*x4^8*z^16 + 3*x1^36*x2^21*x3^15*x4^8*z^16 - x1^35*x2^22*x3^15*x4^8*z^16 + 4*x1^34*x2^23*x3^15*x4^8*z^16 - x1^33*x2^24*x3^15*x4^8*z^16 - 2*x1^32*x2^25*x3^15*x4^8*z^16 - 4*x1^31*x2^26*x3^15*x4^8*z^16 - 3*x1^30*x2^27*x3^15*x4^8*z^16 - 4*x1^29*x2^28*x3^15*x4^8*z^16 - x1^36*x2^20*x3^16*x4^8*z^16 - x1^35*x2^21*x3^16*x4^8*z^16 - x1^33*x2^23*x3^16*x4^8*z^16 + x1^32*x2^24*x3^16*x4^8*z^16 + 3*x1^31*x2^25*x3^16*x4^8*z^16 + 2*x1^30*x2^26*x3^16*x4^8*z^16 + 4*x1^29*x2^27*x3^16*x4^8*z^16 + x1^28*x2^28*x3^16*x4^8*z^16 + 2*x1^35*x2^20*x3^17*x4^8*z^16 + x1^34*x2^21*x3^17*x4^8*z^16 + x1^33*x2^22*x3^17*x4^8*z^16 - x1^32*x2^23*x3^17*x4^8*z^16 - x1^30*x2^25*x3^17*x4^8*z^16 - 2*x1^29*x2^26*x3^17*x4^8*z^16 - 4*x1^28*x2^27*x3^17*x4^8*z^16 - x1^34*x2^20*x3^18*x4^8*z^16 - x1^33*x2^21*x3^18*x4^8*z^16 - x1^32*x2^22*x3^18*x4^8*z^16 - x1^31*x2^23*x3^18*x4^8*z^16 + x1^30*x2^24*x3^18*x4^8*z^16 + 3*x1^28*x2^26*x3^18*x4^8*z^16 + x1^27*x2^27*x3^18*x4^8*z^16 + x1^33*x2^20*x3^19*x4^8*z^16 + 2*x1^32*x2^21*x3^19*x4^8*z^16 + x1^30*x2^23*x3^19*x4^8*z^16 - x1^29*x2^24*x3^19*x4^8*z^16 + 2*x1^28*x2^25*x3^19*x4^8*z^16 - 3*x1^27*x2^26*x3^19*x4^8*z^16 - x1^32*x2^20*x3^20*x4^8*z^16 - x1^31*x2^21*x3^20*x4^8*z^16 - x1^30*x2^22*x3^20*x4^8*z^16 - x1^29*x2^23*x3^20*x4^8*z^16 + x1^28*x2^24*x3^20*x4^8*z^16 - x1^27*x2^25*x3^20*x4^8*z^16 + 2*x1^26*x2^26*x3^20*x4^8*z^16 + 2*x1^29*x2^22*x3^21*x4^8*z^16 - x1^28*x2^23*x3^21*x4^8*z^16 - x1^27*x2^24*x3^21*x4^8*z^16 + x1^26*x2^25*x3^21*x4^8*z^16 - x1^28*x2^22*x3^22*x4^8*z^16 + x1^27*x2^23*x3^22*x4^8*z^16 - x1^25*x2^25*x3^22*x4^8*z^16 - x1^36*x2^26*x3^9*x4^9*z^16 - x1^35*x2^27*x3^9*x4^9*z^16 + x1^35*x2^26*x3^10*x4^9*z^16 + x1^34*x2^27*x3^10*x4^9*z^16 + x1^33*x2^28*x3^10*x4^9*z^16 + x1^32*x2^29*x3^10*x4^9*z^16 + x1^31*x2^30*x3^10*x4^9*z^16 - 2*x1^35*x2^25*x3^11*x4^9*z^16 - x1^34*x2^26*x3^11*x4^9*z^16 - x1^33*x2^27*x3^11*x4^9*z^16 - x1^32*x2^28*x3^11*x4^9*z^16 + x1^31*x2^29*x3^11*x4^9*z^16 - x1^30*x2^30*x3^11*x4^9*z^16 - 2*x1^37*x2^22*x3^12*x4^9*z^16 - x1^36*x2^23*x3^12*x4^9*z^16 + x1^34*x2^25*x3^12*x4^9*z^16 + x1^33*x2^26*x3^12*x4^9*z^16 + 2*x1^32*x2^27*x3^12*x4^9*z^16 + x1^30*x2^29*x3^12*x4^9*z^16 + x1^36*x2^22*x3^13*x4^9*z^16 + x1^35*x2^23*x3^13*x4^9*z^16 - 3*x1^34*x2^24*x3^13*x4^9*z^16 + x1^33*x2^25*x3^13*x4^9*z^16 - 2*x1^32*x2^26*x3^13*x4^9*z^16 - 3*x1^31*x2^27*x3^13*x4^9*z^16 - x1^30*x2^28*x3^13*x4^9*z^16 - x1^29*x2^29*x3^13*x4^9*z^16 - 2*x1^36*x2^21*x3^14*x4^9*z^16 - x1^34*x2^23*x3^14*x4^9*z^16 + 2*x1^33*x2^24*x3^14*x4^9*z^16 + x1^32*x2^25*x3^14*x4^9*z^16 + 5*x1^31*x2^26*x3^14*x4^9*z^16 + 3*x1^30*x2^27*x3^14*x4^9*z^16 + 2*x1^29*x2^28*x3^14*x4^9*z^16 + x1^36*x2^20*x3^15*x4^9*z^16 + x1^35*x2^21*x3^15*x4^9*z^16 - x1^33*x2^23*x3^15*x4^9*z^16 - x1^32*x2^24*x3^15*x4^9*z^16 - 2*x1^31*x2^25*x3^15*x4^9*z^16 - 3*x1^30*x2^26*x3^15*x4^9*z^16 - x1^28*x2^28*x3^15*x4^9*z^16 - 3*x1^35*x2^20*x3^16*x4^9*z^16 - x1^34*x2^21*x3^16*x4^9*z^16 + 3*x1^32*x2^23*x3^16*x4^9*z^16 + 4*x1^30*x2^25*x3^16*x4^9*z^16 + 3*x1^29*x2^26*x3^16*x4^9*z^16 + 2*x1^28*x2^27*x3^16*x4^9*z^16 + x1^35*x2^19*x3^17*x4^9*z^16 + 2*x1^34*x2^20*x3^17*x4^9*z^16 + x1^33*x2^21*x3^17*x4^9*z^16 - x1^32*x2^22*x3^17*x4^9*z^16 - x1^31*x2^23*x3^17*x4^9*z^16 - 4*x1^30*x2^24*x3^17*x4^9*z^16 - 3*x1^29*x2^25*x3^17*x4^9*z^16 - 2*x1^28*x2^26*x3^17*x4^9*z^16 - x1^34*x2^19*x3^18*x4^9*z^16 - x1^32*x2^21*x3^18*x4^9*z^16 + 5*x1^31*x2^22*x3^18*x4^9*z^16 + 2*x1^30*x2^23*x3^18*x4^9*z^16 + 5*x1^29*x2^24*x3^18*x4^9*z^16 + 2*x1^27*x2^26*x3^18*x4^9*z^16 - x1^31*x2^21*x3^19*x4^9*z^16 - x1^29*x2^23*x3^19*x4^9*z^16 - 5*x1^28*x2^24*x3^19*x4^9*z^16 - x1^26*x2^26*x3^19*x4^9*z^16 - x1^31*x2^20*x3^20*x4^9*z^16 + 2*x1^30*x2^21*x3^20*x4^9*z^16 - x1^29*x2^22*x3^20*x4^9*z^16 + 5*x1^28*x2^23*x3^20*x4^9*z^16 + 3*x1^27*x2^24*x3^20*x4^9*z^16 - 2*x1^29*x2^21*x3^21*x4^9*z^16 - x1^28*x2^22*x3^21*x4^9*z^16 - 4*x1^27*x2^23*x3^21*x4^9*z^16 + x1^26*x2^23*x3^22*x4^9*z^16 - x1^32*x2^28*x3^10*x4^10*z^16 - 2*x1^33*x2^26*x3^11*x4^10*z^16 - x1^32*x2^27*x3^11*x4^10*z^16 - x1^30*x2^29*x3^11*x4^10*z^16 + x1^34*x2^24*x3^12*x4^10*z^16 - x1^33*x2^25*x3^12*x4^10*z^16 + x1^32*x2^26*x3^12*x4^10*z^16 - x1^31*x2^27*x3^12*x4^10*z^16 + 2*x1^29*x2^29*x3^12*x4^10*z^16 - x1^31*x2^26*x3^13*x4^10*z^16 - x1^30*x2^27*x3^13*x4^10*z^16 - x1^29*x2^28*x3^13*x4^10*z^16 + x1^36*x2^20*x3^14*x4^10*z^16 - x1^35*x2^21*x3^14*x4^10*z^16 - 2*x1^34*x2^22*x3^14*x4^10*z^16 + 2*x1^33*x2^23*x3^14*x4^10*z^16 - x1^32*x2^24*x3^14*x4^10*z^16 + x1^29*x2^27*x3^14*x4^10*z^16 + x1^34*x2^21*x3^15*x4^10*z^16 + x1^33*x2^22*x3^15*x4^10*z^16 - x1^32*x2^23*x3^15*x4^10*z^16 + x1^31*x2^24*x3^15*x4^10*z^16 - 2*x1^29*x2^26*x3^15*x4^10*z^16 - x1^33*x2^21*x3^16*x4^10*z^16 - x1^32*x2^22*x3^16*x4^10*z^16 + 2*x1^30*x2^24*x3^16*x4^10*z^16 - x1^29*x2^25*x3^16*x4^10*z^16 + x1^28*x2^26*x3^16*x4^10*z^16 + x1^34*x2^19*x3^17*x4^10*z^16 - 2*x1^32*x2^21*x3^17*x4^10*z^16 - 3*x1^31*x2^22*x3^17*x4^10*z^16 - x1^29*x2^24*x3^17*x4^10*z^16 - x1^28*x2^25*x3^17*x4^10*z^16 - x1^33*x2^19*x3^18*x4^10*z^16 + x1^31*x2^21*x3^18*x4^10*z^16 - x1^30*x2^22*x3^18*x4^10*z^16 + x1^28*x2^24*x3^18*x4^10*z^16 + x1^27*x2^25*x3^18*x4^10*z^16 + x1^31*x2^20*x3^19*x4^10*z^16 - x1^30*x2^21*x3^19*x4^10*z^16 - x1^29*x2^22*x3^19*x4^10*z^16 - 2*x1^28*x2^23*x3^19*x4^10*z^16 - x1^27*x2^24*x3^19*x4^10*z^16 - x1^26*x2^25*x3^19*x4^10*z^16 + x1^28*x2^22*x3^20*x4^10*z^16 + 2*x1^27*x2^23*x3^20*x4^10*z^16 + x1^25*x2^25*x3^20*x4^10*z^16 + x1^28*x2^21*x3^21*x4^10*z^16 - x1^27*x2^22*x3^21*x4^10*z^16 - x1^26*x2^23*x3^21*x4^10*z^16 + x1^26*x2^22*x3^22*x4^10*z^16 + x1^34*x2^24*x3^11*x4^11*z^16 + x1^33*x2^25*x3^11*x4^11*z^16 - x1^32*x2^26*x3^11*x4^11*z^16 + x1^31*x2^27*x3^11*x4^11*z^16 + x1^33*x2^24*x3^12*x4^11*z^16 + 2*x1^32*x2^25*x3^12*x4^11*z^16 - x1^30*x2^27*x3^12*x4^11*z^16 + x1^29*x2^28*x3^12*x4^11*z^16 + 2*x1^32*x2^24*x3^13*x4^11*z^16 - x1^31*x2^25*x3^13*x4^11*z^16 + x1^30*x2^26*x3^13*x4^11*z^16 - x1^29*x2^27*x3^13*x4^11*z^16 - x1^28*x2^28*x3^13*x4^11*z^16 + x1^35*x2^20*x3^14*x4^11*z^16 - x1^33*x2^22*x3^14*x4^11*z^16 - x1^32*x2^23*x3^14*x4^11*z^16 + 2*x1^31*x2^24*x3^14*x4^11*z^16 - x1^30*x2^25*x3^14*x4^11*z^16 - x1^29*x2^26*x3^14*x4^11*z^16 + 2*x1^28*x2^27*x3^14*x4^11*z^16 + x1^34*x2^20*x3^15*x4^11*z^16 + 2*x1^33*x2^21*x3^15*x4^11*z^16 - 2*x1^32*x2^22*x3^15*x4^11*z^16 + x1^31*x2^23*x3^15*x4^11*z^16 + 2*x1^30*x2^24*x3^15*x4^11*z^16 + 3*x1^29*x2^25*x3^15*x4^11*z^16 - 3*x1^28*x2^26*x3^15*x4^11*z^16 - x1^27*x2^27*x3^15*x4^11*z^16 - x1^33*x2^20*x3^16*x4^11*z^16 - 2*x1^32*x2^21*x3^16*x4^11*z^16 - 2*x1^29*x2^24*x3^16*x4^11*z^16 + x1^28*x2^25*x3^16*x4^11*z^16 + x1^27*x2^26*x3^16*x4^11*z^16 + 2*x1^31*x2^21*x3^17*x4^11*z^16 - 3*x1^29*x2^23*x3^17*x4^11*z^16 + 4*x1^28*x2^24*x3^17*x4^11*z^16 - 2*x1^27*x2^25*x3^17*x4^11*z^16 - x1^26*x2^26*x3^17*x4^11*z^16 - x1^30*x2^21*x3^18*x4^11*z^16 - x1^29*x2^22*x3^18*x4^11*z^16 - x1^27*x2^24*x3^18*x4^11*z^16 + x1^26*x2^25*x3^18*x4^11*z^16 - x1^26*x2^24*x3^19*x4^11*z^16 - x1^26*x2^23*x3^20*x4^11*z^16 + x1^25*x2^24*x3^20*x4^11*z^16 - x1^24*x2^24*x3^21*x4^11*z^16 - x1^33*x2^23*x3^12*x4^12*z^16 - x1^32*x2^24*x3^12*x4^12*z^16 + x1^31*x2^25*x3^12*x4^12*z^16 - x1^30*x2^26*x3^12*x4^12*z^16 - x1^31*x2^24*x3^13*x4^12*z^16 + x1^30*x2^25*x3^13*x4^12*z^16 + x1^29*x2^26*x3^13*x4^12*z^16 - x1^28*x2^27*x3^13*x4^12*z^16 - x1^31*x2^23*x3^14*x4^12*z^16 + 3*x1^30*x2^24*x3^14*x4^12*z^16 - x1^29*x2^25*x3^14*x4^12*z^16 + 2*x1^28*x2^26*x3^14*x4^12*z^16 + x1^27*x2^27*x3^14*x4^12*z^16 - 2*x1^30*x2^23*x3^15*x4^12*z^16 - 4*x1^27*x2^26*x3^15*x4^12*z^16 - x1^31*x2^21*x3^16*x4^12*z^16 + 2*x1^29*x2^23*x3^16*x4^12*z^16 - 3*x1^28*x2^24*x3^16*x4^12*z^16 + 4*x1^27*x2^25*x3^16*x4^12*z^16 + 2*x1^26*x2^26*x3^16*x4^12*z^16 - 2*x1^29*x2^22*x3^17*x4^12*z^16 + x1^28*x2^23*x3^17*x4^12*z^16 - x1^27*x2^24*x3^17*x4^12*z^16 - 5*x1^26*x2^25*x3^17*x4^12*z^16 - 2*x1^29*x2^21*x3^18*x4^12*z^16 + 2*x1^28*x2^22*x3^18*x4^12*z^16 - x1^27*x2^23*x3^18*x4^12*z^16 + 4*x1^26*x2^24*x3^18*x4^12*z^16 + 2*x1^25*x2^25*x3^18*x4^12*z^16 - x1^27*x2^22*x3^19*x4^12*z^16 + x1^26*x2^23*x3^19*x4^12*z^16 - 4*x1^25*x2^24*x3^19*x4^12*z^16 + 2*x1^24*x2^24*x3^20*x4^12*z^16 - x1^31*x2^22*x3^14*x4^13*z^16 - x1^30*x2^23*x3^14*x4^13*z^16 - x1^29*x2^24*x3^14*x4^13*z^16 - x1^30*x2^22*x3^15*x4^13*z^16 - 3*x1^29*x2^23*x3^15*x4^13*z^16 + x1^28*x2^24*x3^15*x4^13*z^16 - x1^27*x2^25*x3^15*x4^13*z^16 - x1^30*x2^21*x3^16*x4^13*z^16 + x1^29*x2^22*x3^16*x4^13*z^16 - x1^27*x2^24*x3^16*x4^13*z^16 + 3*x1^26*x2^25*x3^16*x4^13*z^16 + x1^30*x2^20*x3^17*x4^13*z^16 - x1^29*x2^21*x3^17*x4^13*z^16 - 3*x1^28*x2^22*x3^17*x4^13*z^16 + x1^27*x2^23*x3^17*x4^13*z^16 - x1^26*x2^24*x3^17*x4^13*z^16 - 2*x1^25*x2^25*x3^17*x4^13*z^16 + x1^28*x2^21*x3^18*x4^13*z^16 - x1^26*x2^23*x3^18*x4^13*z^16 + 2*x1^25*x2^24*x3^18*x4^13*z^16 - x1^24*x2^24*x3^19*x4^13*z^16 + x1^30*x2^21*x3^15*x4^14*z^16 + x1^27*x2^24*x3^15*x4^14*z^16 - 2*x1^27*x2^23*x3^16*x4^14*z^16 + x1^26*x2^23*x3^17*x4^14*z^16 + x1^27*x2^21*x3^18*x4^14*z^16 - 2*x1^26*x2^22*x3^18*x4^14*z^16 + x1^24*x2^24*x3^18*x4^14*z^16 + x1^25*x2^22*x3^19*x4^14*z^16 - x1^26*x2^23*x3^16*x4^15*z^16 + 2*x1^26*x2^22*x3^17*x4^15*z^16 - x1^25*x2^22*x3^18*x4^15*z^16 + x1^40*x2^26*x3^9*z^15 - 2*x1^39*x2^26*x3^10*z^15 - x1^38*x2^27*x3^10*z^15 - x1^37*x2^28*x3^10*z^15 + x1^39*x2^25*x3^11*z^15 + 2*x1^38*x2^26*x3^11*z^15 + x1^36*x2^28*x3^11*z^15 - x1^35*x2^29*x3^11*z^15 - 2*x1^38*x2^25*x3^12*z^15 - x1^35*x2^28*x3^12*z^15 + 2*x1^37*x2^25*x3^13*z^15 + 2*x1^35*x2^27*x3^13*z^15 - x1^32*x2^30*x3^13*z^15 - 2*x1^37*x2^24*x3^14*z^15 - x1^36*x2^25*x3^14*z^15 - x1^35*x2^26*x3^14*z^15 + x1^33*x2^28*x3^14*z^15 + x1^32*x2^29*x3^14*z^15 + x1^31*x2^30*x3^14*z^15 + x1^37*x2^23*x3^15*z^15 + 2*x1^36*x2^24*x3^15*z^15 + x1^34*x2^26*x3^15*z^15 + x1^33*x2^27*x3^15*z^15 - 2*x1^31*x2^29*x3^15*z^15 - 2*x1^36*x2^23*x3^16*z^15 - x1^35*x2^24*x3^16*z^15 - x1^34*x2^25*x3^16*z^15 - x1^33*x2^26*x3^16*z^15 + x1^32*x2^27*x3^16*z^15 + x1^31*x2^28*x3^16*z^15 + x1^36*x2^22*x3^17*z^15 + 2*x1^35*x2^23*x3^17*z^15 + x1^33*x2^25*x3^17*z^15 - x1^32*x2^26*x3^17*z^15 + x1^31*x2^27*x3^17*z^15 - x1^30*x2^28*x3^17*z^15 - x1^35*x2^22*x3^18*z^15 + x1^34*x2^23*x3^18*z^15 - x1^33*x2^24*x3^18*z^15 - x1^32*x2^25*x3^18*z^15 + x1^34*x2^22*x3^19*z^15 + x1^33*x2^23*x3^19*z^15 + x1^32*x2^24*x3^19*z^15 - x1^31*x2^24*x3^20*z^15 + x1^40*x2^27*x3^7*x4*z^15 - x1^40*x2^26*x3^8*x4*z^15 - x1^39*x2^27*x3^8*x4*z^15 - x1^37*x2^29*x3^8*x4*z^15 + 4*x1^39*x2^26*x3^9*x4*z^15 - x1^38*x2^27*x3^9*x4*z^15 + x1^36*x2^29*x3^9*x4*z^15 - 2*x1^39*x2^25*x3^10*x4*z^15 - 4*x1^38*x2^26*x3^10*x4*z^15 + 2*x1^37*x2^27*x3^10*x4*z^15 - 2*x1^36*x2^28*x3^10*x4*z^15 + 6*x1^38*x2^25*x3^11*x4*z^15 + 2*x1^37*x2^26*x3^11*x4*z^15 - x1^34*x2^29*x3^11*x4*z^15 - 2*x1^38*x2^24*x3^12*x4*z^15 - 6*x1^37*x2^25*x3^12*x4*z^15 - 2*x1^35*x2^27*x3^12*x4*z^15 + 3*x1^34*x2^28*x3^12*x4*z^15 + x1^33*x2^29*x3^12*x4*z^15 + 2*x1^32*x2^30*x3^12*x4*z^15 + 6*x1^37*x2^24*x3^13*x4*z^15 + 2*x1^36*x2^25*x3^13*x4*z^15 + 2*x1^35*x2^26*x3^13*x4*z^15 - x1^33*x2^28*x3^13*x4*z^15 - 2*x1^32*x2^29*x3^13*x4*z^15 - 2*x1^31*x2^30*x3^13*x4*z^15 - 2*x1^37*x2^23*x3^14*x4*z^15 - 6*x1^36*x2^24*x3^14*x4*z^15 - 4*x1^34*x2^26*x3^14*x4*z^15 + 4*x1^33*x2^27*x3^14*x4*z^15 + 6*x1^31*x2^29*x3^14*x4*z^15 + 6*x1^36*x2^23*x3^15*x4*z^15 + 2*x1^35*x2^24*x3^15*x4*z^15 + 2*x1^34*x2^25*x3^15*x4*z^15 - 2*x1^32*x2^27*x3^15*x4*z^15 - 2*x1^31*x2^28*x3^15*x4*z^15 - 6*x1^30*x2^29*x3^15*x4*z^15 - 2*x1^36*x2^22*x3^16*x4*z^15 - 6*x1^35*x2^23*x3^16*x4*z^15 - 4*x1^33*x2^25*x3^16*x4*z^15 + 2*x1^32*x2^26*x3^16*x4*z^15 + 6*x1^30*x2^28*x3^16*x4*z^15 + x1^29*x2^29*x3^16*x4*z^15 + 5*x1^35*x2^22*x3^17*x4*z^15 + 3*x1^34*x2^23*x3^17*x4*z^15 + 2*x1^33*x2^24*x3^17*x4*z^15 + x1^32*x2^25*x3^17*x4*z^15 - x1^31*x2^26*x3^17*x4*z^15 - x1^30*x2^27*x3^17*x4*z^15 - 3*x1^29*x2^28*x3^17*x4*z^15 - 5*x1^34*x2^22*x3^18*x4*z^15 - 2*x1^33*x2^23*x3^18*x4*z^15 - 4*x1^32*x2^24*x3^18*x4*z^15 + 4*x1^29*x2^27*x3^18*x4*z^15 + x1^34*x2^21*x3^19*x4*z^15 + 2*x1^33*x2^22*x3^19*x4*z^15 + 3*x1^32*x2^23*x3^19*x4*z^15 + 3*x1^31*x2^24*x3^19*x4*z^15 - x1^30*x2^25*x3^19*x4*z^15 - x1^29*x2^26*x3^19*x4*z^15 - x1^28*x2^27*x3^19*x4*z^15 - x1^33*x2^21*x3^20*x4*z^15 - 2*x1^32*x2^22*x3^20*x4*z^15 - 2*x1^31*x2^23*x3^20*x4*z^15 - x1^30*x2^24*x3^20*x4*z^15 + 2*x1^28*x2^26*x3^20*x4*z^15 + x1^31*x2^22*x3^21*x4*z^15 + 2*x1^30*x2^23*x3^21*x4*z^15 + x1^39*x2^27*x3^7*x4^2*z^15 - x1^38*x2^28*x3^7*x4^2*z^15 - 2*x1^39*x2^26*x3^8*x4^2*z^15 - x1^38*x2^27*x3^8*x4^2*z^15 + x1^37*x2^28*x3^8*x4^2*z^15 + x1^39*x2^25*x3^9*x4^2*z^15 + 4*x1^38*x2^26*x3^9*x4^2*z^15 - 2*x1^37*x2^27*x3^9*x4^2*z^15 + x1^36*x2^28*x3^9*x4^2*z^15 - x1^35*x2^29*x3^9*x4^2*z^15 - 4*x1^38*x2^25*x3^10*x4^2*z^15 - 2*x1^37*x2^26*x3^10*x4^2*z^15 - x1^36*x2^27*x3^10*x4^2*z^15 + 2*x1^34*x2^29*x3^10*x4^2*z^15 + x1^33*x2^30*x3^10*x4^2*z^15 + 2*x1^38*x2^24*x3^11*x4^2*z^15 + 6*x1^37*x2^25*x3^11*x4^2*z^15 + 4*x1^35*x2^27*x3^11*x4^2*z^15 - 3*x1^34*x2^28*x3^11*x4^2*z^15 - 2*x1^33*x2^29*x3^11*x4^2*z^15 - 3*x1^32*x2^30*x3^11*x4^2*z^15 - 6*x1^37*x2^24*x3^12*x4^2*z^15 - 2*x1^36*x2^25*x3^12*x4^2*z^15 - 2*x1^35*x2^26*x3^12*x4^2*z^15 + 3*x1^33*x2^28*x3^12*x4^2*z^15 + 3*x1^32*x2^29*x3^12*x4^2*z^15 + 3*x1^31*x2^30*x3^12*x4^2*z^15 + 2*x1^37*x2^23*x3^13*x4^2*z^15 + 6*x1^36*x2^24*x3^13*x4^2*z^15 + 4*x1^34*x2^26*x3^13*x4^2*z^15 - 4*x1^33*x2^27*x3^13*x4^2*z^15 - x1^32*x2^28*x3^13*x4^2*z^15 - 6*x1^31*x2^29*x3^13*x4^2*z^15 - x1^30*x2^30*x3^13*x4^2*z^15 - 6*x1^36*x2^23*x3^14*x4^2*z^15 - 2*x1^35*x2^24*x3^14*x4^2*z^15 - 2*x1^34*x2^25*x3^14*x4^2*z^15 + 2*x1^32*x2^27*x3^14*x4^2*z^15 + 2*x1^31*x2^28*x3^14*x4^2*z^15 + 6*x1^30*x2^29*x3^14*x4^2*z^15 + 2*x1^36*x2^22*x3^15*x4^2*z^15 + 6*x1^35*x2^23*x3^15*x4^2*z^15 + 4*x1^33*x2^25*x3^15*x4^2*z^15 - 4*x1^32*x2^26*x3^15*x4^2*z^15 - 6*x1^30*x2^28*x3^15*x4^2*z^15 - 2*x1^29*x2^29*x3^15*x4^2*z^15 - 4*x1^35*x2^22*x3^16*x4^2*z^15 - 2*x1^34*x2^23*x3^16*x4^2*z^15 - 2*x1^33*x2^24*x3^16*x4^2*z^15 + 2*x1^31*x2^26*x3^16*x4^2*z^15 + 2*x1^30*x2^27*x3^16*x4^2*z^15 + 6*x1^29*x2^28*x3^16*x4^2*z^15 + x1^35*x2^21*x3^17*x4^2*z^15 + 4*x1^34*x2^22*x3^17*x4^2*z^15 + x1^33*x2^23*x3^17*x4^2*z^15 + 4*x1^32*x2^24*x3^17*x4^2*z^15 - 4*x1^31*x2^25*x3^17*x4^2*z^15 - 6*x1^29*x2^27*x3^17*x4^2*z^15 - 2*x1^28*x2^28*x3^17*x4^2*z^15 - x1^34*x2^21*x3^18*x4^2*z^15 - x1^33*x2^22*x3^18*x4^2*z^15 - x1^32*x2^23*x3^18*x4^2*z^15 + x1^30*x2^25*x3^18*x4^2*z^15 + x1^29*x2^26*x3^18*x4^2*z^15 + 5*x1^28*x2^27*x3^18*x4^2*z^15 + x1^33*x2^21*x3^19*x4^2*z^15 + 2*x1^31*x2^23*x3^19*x4^2*z^15 - x1^30*x2^24*x3^19*x4^2*z^15 - x1^29*x2^25*x3^19*x4^2*z^15 - 5*x1^28*x2^26*x3^19*x4^2*z^15 + x1^31*x2^22*x3^20*x4^2*z^15 - x1^30*x2^23*x3^20*x4^2*z^15 + x1^28*x2^25*x3^20*x4^2*z^15 + 2*x1^27*x2^26*x3^20*x4^2*z^15 - 2*x1^27*x2^25*x3^21*x4^2*z^15 + x1^26*x2^25*x3^22*x4^2*z^15 + x1^38*x2^26*x3^8*x4^3*z^15 - x1^37*x2^27*x3^8*x4^3*z^15 - x1^36*x2^28*x3^8*x4^3*z^15 + x1^35*x2^29*x3^8*x4^3*z^15 - x1^40*x2^23*x3^9*x4^3*z^15 - 2*x1^39*x2^24*x3^9*x4^3*z^15 + x1^37*x2^26*x3^9*x4^3*z^15 + x1^36*x2^27*x3^9*x4^3*z^15 + x1^35*x2^28*x3^9*x4^3*z^15 + 2*x1^39*x2^23*x3^10*x4^3*z^15 - x1^37*x2^25*x3^10*x4^3*z^15 + x1^36*x2^26*x3^10*x4^3*z^15 - 2*x1^35*x2^27*x3^10*x4^3*z^15 - x1^34*x2^28*x3^10*x4^3*z^15 + x1^32*x2^30*x3^10*x4^3*z^15 - x1^39*x2^22*x3^11*x4^3*z^15 - 2*x1^38*x2^23*x3^11*x4^3*z^15 + 2*x1^35*x2^26*x3^11*x4^3*z^15 + x1^33*x2^28*x3^11*x4^3*z^15 - x1^32*x2^29*x3^11*x4^3*z^15 - x1^31*x2^30*x3^11*x4^3*z^15 + 2*x1^38*x2^22*x3^12*x4^3*z^15 - x1^37*x2^23*x3^12*x4^3*z^15 - 2*x1^36*x2^24*x3^12*x4^3*z^15 - x1^34*x2^26*x3^12*x4^3*z^15 - 2*x1^32*x2^28*x3^12*x4^3*z^15 + x1^31*x2^29*x3^12*x4^3*z^15 - 2*x1^37*x2^22*x3^13*x4^3*z^15 + 2*x1^36*x2^23*x3^13*x4^3*z^15 - 2*x1^35*x2^24*x3^13*x4^3*z^15 + x1^34*x2^25*x3^13*x4^3*z^15 + 2*x1^32*x2^27*x3^13*x4^3*z^15 - x1^30*x2^29*x3^13*x4^3*z^15 + 2*x1^37*x2^21*x3^14*x4^3*z^15 + x1^36*x2^22*x3^14*x4^3*z^15 - x1^35*x2^23*x3^14*x4^3*z^15 - 3*x1^33*x2^25*x3^14*x4^3*z^15 - 2*x1^31*x2^27*x3^14*x4^3*z^15 + 2*x1^30*x2^28*x3^14*x4^3*z^15 - x1^37*x2^20*x3^15*x4^3*z^15 - 2*x1^36*x2^21*x3^15*x4^3*z^15 + 2*x1^35*x2^22*x3^15*x4^3*z^15 + 2*x1^33*x2^24*x3^15*x4^3*z^15 + x1^31*x2^26*x3^15*x4^3*z^15 - 2*x1^29*x2^28*x3^15*x4^3*z^15 + 2*x1^36*x2^20*x3^16*x4^3*z^15 - x1^34*x2^22*x3^16*x4^3*z^15 - 2*x1^32*x2^24*x3^16*x4^3*z^15 - 2*x1^30*x2^26*x3^16*x4^3*z^15 + 2*x1^29*x2^27*x3^16*x4^3*z^15 + x1^28*x2^28*x3^16*x4^3*z^15 - 2*x1^35*x2^20*x3^17*x4^3*z^15 + 2*x1^32*x2^23*x3^17*x4^3*z^15 + x1^30*x2^25*x3^17*x4^3*z^15 - x1^29*x2^26*x3^17*x4^3*z^15 - 2*x1^28*x2^27*x3^17*x4^3*z^15 + x1^34*x2^20*x3^18*x4^3*z^15 + x1^31*x2^23*x3^18*x4^3*z^15 + 3*x1^30*x2^24*x3^18*x4^3*z^15 - x1^29*x2^25*x3^18*x4^3*z^15 + 2*x1^28*x2^26*x3^18*x4^3*z^15 + x1^27*x2^27*x3^18*x4^3*z^15 - x1^33*x2^20*x3^19*x4^3*z^15 - x1^32*x2^21*x3^19*x4^3*z^15 + x1^30*x2^23*x3^19*x4^3*z^15 + 2*x1^29*x2^24*x3^19*x4^3*z^15 - x1^28*x2^25*x3^19*x4^3*z^15 - 3*x1^27*x2^26*x3^19*x4^3*z^15 + x1^32*x2^20*x3^20*x4^3*z^15 - x1^29*x2^23*x3^20*x4^3*z^15 + x1^28*x2^24*x3^20*x4^3*z^15 + 2*x1^27*x2^25*x3^20*x4^3*z^15 - x1^29*x2^22*x3^21*x4^3*z^15 + x1^28*x2^23*x3^21*x4^3*z^15 - x1^27*x2^24*x3^21*x4^3*z^15 - 2*x1^26*x2^25*x3^21*x4^3*z^15 - x1^40*x2^24*x3^7*x4^4*z^15 - x1^39*x2^25*x3^7*x4^4*z^15 + x1^37*x2^27*x3^7*x4^4*z^15 - x1^36*x2^28*x3^7*x4^4*z^15 + 2*x1^39*x2^24*x3^8*x4^4*z^15 - x1^35*x2^28*x3^8*x4^4*z^15 - 2*x1^39*x2^23*x3^9*x4^4*z^15 - 2*x1^38*x2^24*x3^9*x4^4*z^15 - x1^37*x2^25*x3^9*x4^4*z^15 + x1^36*x2^26*x3^9*x4^4*z^15 + x1^35*x2^27*x3^9*x4^4*z^15 - 2*x1^34*x2^28*x3^9*x4^4*z^15 - x1^33*x2^29*x3^9*x4^4*z^15 + x1^39*x2^22*x3^10*x4^4*z^15 + 4*x1^38*x2^23*x3^10*x4^4*z^15 + x1^37*x2^24*x3^10*x4^4*z^15 - x1^35*x2^26*x3^10*x4^4*z^15 - x1^33*x2^28*x3^10*x4^4*z^15 - 5*x1^38*x2^22*x3^11*x4^4*z^15 - 3*x1^37*x2^23*x3^11*x4^4*z^15 - 3*x1^36*x2^24*x3^11*x4^4*z^15 + x1^35*x2^25*x3^11*x4^4*z^15 + x1^34*x2^26*x3^11*x4^4*z^15 + 2*x1^33*x2^27*x3^11*x4^4*z^15 + x1^32*x2^28*x3^11*x4^4*z^15 + 2*x1^38*x2^21*x3^12*x4^4*z^15 + 6*x1^37*x2^22*x3^12*x4^4*z^15 + x1^36*x2^23*x3^12*x4^4*z^15 + 3*x1^35*x2^24*x3^12*x4^4*z^15 - 3*x1^34*x2^25*x3^12*x4^4*z^15 + x1^33*x2^26*x3^12*x4^4*z^15 - 5*x1^32*x2^27*x3^12*x4^4*z^15 + x1^31*x2^28*x3^12*x4^4*z^15 - 6*x1^37*x2^21*x3^13*x4^4*z^15 - 2*x1^36*x2^22*x3^13*x4^4*z^15 - 2*x1^35*x2^23*x3^13*x4^4*z^15 + 2*x1^33*x2^25*x3^13*x4^4*z^15 + 2*x1^32*x2^26*x3^13*x4^4*z^15 + 6*x1^31*x2^27*x3^13*x4^4*z^15 + x1^37*x2^20*x3^14*x4^4*z^15 + 6*x1^36*x2^21*x3^14*x4^4*z^15 + 4*x1^34*x2^23*x3^14*x4^4*z^15 - 4*x1^33*x2^24*x3^14*x4^4*z^15 - 6*x1^31*x2^26*x3^14*x4^4*z^15 - 2*x1^30*x2^27*x3^14*x4^4*z^15 - 4*x1^36*x2^20*x3^15*x4^4*z^15 - 3*x1^35*x2^21*x3^15*x4^4*z^15 - 2*x1^34*x2^22*x3^15*x4^4*z^15 + 2*x1^32*x2^24*x3^15*x4^4*z^15 + 2*x1^31*x2^25*x3^15*x4^4*z^15 + 6*x1^30*x2^26*x3^15*x4^4*z^15 + 4*x1^35*x2^20*x3^16*x4^4*z^15 + x1^34*x2^21*x3^16*x4^4*z^15 + 2*x1^33*x2^22*x3^16*x4^4*z^15 - 4*x1^32*x2^23*x3^16*x4^4*z^15 - 6*x1^30*x2^25*x3^16*x4^4*z^15 - 2*x1^29*x2^26*x3^16*x4^4*z^15 - x1^35*x2^19*x3^17*x4^4*z^15 - 2*x1^34*x2^20*x3^17*x4^4*z^15 - 2*x1^33*x2^21*x3^17*x4^4*z^15 + x1^31*x2^23*x3^17*x4^4*z^15 + 2*x1^30*x2^24*x3^17*x4^4*z^15 + 6*x1^29*x2^25*x3^17*x4^4*z^15 + x1^34*x2^19*x3^18*x4^4*z^15 + x1^33*x2^20*x3^18*x4^4*z^15 + 2*x1^32*x2^21*x3^18*x4^4*z^15 - 3*x1^31*x2^22*x3^18*x4^4*z^15 - 7*x1^29*x2^24*x3^18*x4^4*z^15 - 2*x1^28*x2^25*x3^18*x4^4*z^15 - x1^32*x2^20*x3^19*x4^4*z^15 - x1^31*x2^21*x3^19*x4^4*z^15 + 6*x1^28*x2^24*x3^19*x4^4*z^15 - x1^26*x2^26*x3^19*x4^4*z^15 + x1^31*x2^20*x3^20*x4^4*z^15 + 2*x1^29*x2^22*x3^20*x4^4*z^15 - 3*x1^28*x2^23*x3^20*x4^4*z^15 + 2*x1^26*x2^25*x3^20*x4^4*z^15 + x1^29*x2^21*x3^21*x4^4*z^15 + 2*x1^27*x2^23*x3^21*x4^4*z^15 - x1^26*x2^24*x3^21*x4^4*z^15 - x1^25*x2^25*x3^21*x4^4*z^15 + x1^27*x2^22*x3^22*x4^4*z^15 + x1^36*x2^27*x3^7*x4^5*z^15 - 3*x1^36*x2^26*x3^8*x4^5*z^15 + x1^34*x2^28*x3^8*x4^5*z^15 - x1^38*x2^23*x3^9*x4^5*z^15 - 2*x1^36*x2^25*x3^9*x4^5*z^15 + x1^35*x2^26*x3^9*x4^5*z^15 + x1^33*x2^28*x3^9*x4^5*z^15 - x1^32*x2^29*x3^9*x4^5*z^15 + 2*x1^38*x2^22*x3^10*x4^5*z^15 + x1^37*x2^23*x3^10*x4^5*z^15 - x1^35*x2^25*x3^10*x4^5*z^15 - 3*x1^34*x2^26*x3^10*x4^5*z^15 - x1^33*x2^27*x3^10*x4^5*z^15 - 3*x1^32*x2^28*x3^10*x4^5*z^15 - 4*x1^37*x2^22*x3^11*x4^5*z^15 + 2*x1^36*x2^23*x3^11*x4^5*z^15 + 4*x1^34*x2^25*x3^11*x4^5*z^15 + x1^33*x2^26*x3^11*x4^5*z^15 + 5*x1^32*x2^27*x3^11*x4^5*z^15 + x1^31*x2^28*x3^11*x4^5*z^15 - x1^30*x2^29*x3^11*x4^5*z^15 + 4*x1^37*x2^21*x3^12*x4^5*z^15 + 2*x1^36*x2^22*x3^12*x4^5*z^15 - x1^35*x2^23*x3^12*x4^5*z^15 - 2*x1^34*x2^24*x3^12*x4^5*z^15 - 2*x1^33*x2^25*x3^12*x4^5*z^15 - x1^32*x2^26*x3^12*x4^5*z^15 - 5*x1^31*x2^27*x3^12*x4^5*z^15 + x1^30*x2^28*x3^12*x4^5*z^15 + x1^29*x2^29*x3^12*x4^5*z^15 - x1^37*x2^20*x3^13*x4^5*z^15 - 5*x1^36*x2^21*x3^13*x4^5*z^15 + x1^35*x2^22*x3^13*x4^5*z^15 - 2*x1^34*x2^23*x3^13*x4^5*z^15 + 3*x1^33*x2^24*x3^13*x4^5*z^15 + 5*x1^31*x2^26*x3^13*x4^5*z^15 + x1^30*x2^27*x3^13*x4^5*z^15 - x1^29*x2^28*x3^13*x4^5*z^15 + 5*x1^36*x2^20*x3^14*x4^5*z^15 + 2*x1^35*x2^21*x3^14*x4^5*z^15 + x1^34*x2^22*x3^14*x4^5*z^15 - x1^33*x2^23*x3^14*x4^5*z^15 - 2*x1^32*x2^24*x3^14*x4^5*z^15 - 6*x1^30*x2^26*x3^14*x4^5*z^15 + 2*x1^29*x2^27*x3^14*x4^5*z^15 - x1^36*x2^19*x3^15*x4^5*z^15 - 5*x1^35*x2^20*x3^15*x4^5*z^15 + x1^34*x2^21*x3^15*x4^5*z^15 - 4*x1^33*x2^22*x3^15*x4^5*z^15 + 4*x1^32*x2^23*x3^15*x4^5*z^15 + 6*x1^30*x2^25*x3^15*x4^5*z^15 + 2*x1^29*x2^26*x3^15*x4^5*z^15 - 2*x1^28*x2^27*x3^15*x4^5*z^15 + 2*x1^35*x2^19*x3^16*x4^5*z^15 + 2*x1^34*x2^20*x3^16*x4^5*z^15 + x1^32*x2^22*x3^16*x4^5*z^15 - 4*x1^31*x2^23*x3^16*x4^5*z^15 - x1^30*x2^24*x3^16*x4^5*z^15 - 6*x1^29*x2^25*x3^16*x4^5*z^15 + 2*x1^28*x2^26*x3^16*x4^5*z^15 + x1^27*x2^27*x3^16*x4^5*z^15 - 2*x1^34*x2^19*x3^17*x4^5*z^15 + x1^33*x2^20*x3^17*x4^5*z^15 + x1^32*x2^21*x3^17*x4^5*z^15 + 5*x1^31*x2^22*x3^17*x4^5*z^15 + 5*x1^29*x2^24*x3^17*x4^5*z^15 + x1^28*x2^25*x3^17*x4^5*z^15 - 2*x1^27*x2^26*x3^17*x4^5*z^15 + x1^33*x2^19*x3^18*x4^5*z^15 - x1^32*x2^20*x3^18*x4^5*z^15 - x1^31*x2^21*x3^18*x4^5*z^15 - x1^30*x2^22*x3^18*x4^5*z^15 + 2*x1^29*x2^23*x3^18*x4^5*z^15 - 6*x1^28*x2^24*x3^18*x4^5*z^15 + 2*x1^27*x2^25*x3^18*x4^5*z^15 + x1^26*x2^26*x3^18*x4^5*z^15 + 2*x1^30*x2^21*x3^19*x4^5*z^15 - x1^29*x2^22*x3^19*x4^5*z^15 + 2*x1^28*x2^23*x3^19*x4^5*z^15 + x1^27*x2^24*x3^19*x4^5*z^15 - 2*x1^26*x2^25*x3^19*x4^5*z^15 - x1^29*x2^21*x3^20*x4^5*z^15 - x1^28*x2^22*x3^20*x4^5*z^15 - 3*x1^27*x2^23*x3^20*x4^5*z^15 + 2*x1^26*x2^24*x3^20*x4^5*z^15 - x1^28*x2^21*x3^21*x4^5*z^15 + 2*x1^27*x2^22*x3^21*x4^5*z^15 - 2*x1^25*x2^24*x3^21*x4^5*z^15 - 2*x1^26*x2^22*x3^22*x4^5*z^15 + x1^25*x2^23*x3^22*x4^5*z^15 + x1^24*x2^24*x3^22*x4^5*z^15 - x1^35*x2^26*x3^8*x4^6*z^15 + x1^38*x2^22*x3^9*x4^6*z^15 + x1^37*x2^23*x3^9*x4^6*z^15 + 2*x1^35*x2^25*x3^9*x4^6*z^15 + 3*x1^34*x2^26*x3^9*x4^6*z^15 + x1^33*x2^27*x3^9*x4^6*z^15 + 2*x1^32*x2^28*x3^9*x4^6*z^15 + x1^31*x2^29*x3^9*x4^6*z^15 + x1^37*x2^22*x3^10*x4^6*z^15 - 2*x1^35*x2^24*x3^10*x4^6*z^15 + x1^32*x2^27*x3^10*x4^6*z^15 - x1^30*x2^29*x3^10*x4^6*z^15 - x1^37*x2^21*x3^11*x4^6*z^15 + x1^36*x2^22*x3^11*x4^6*z^15 + 4*x1^35*x2^23*x3^11*x4^6*z^15 + x1^31*x2^27*x3^11*x4^6*z^15 - x1^30*x2^28*x3^11*x4^6*z^15 - x1^37*x2^20*x3^12*x4^6*z^15 + x1^36*x2^21*x3^12*x4^6*z^15 - 2*x1^35*x2^22*x3^12*x4^6*z^15 - 2*x1^34*x2^23*x3^12*x4^6*z^15 - 2*x1^33*x2^24*x3^12*x4^6*z^15 - x1^32*x2^25*x3^12*x4^6*z^15 + 3*x1^30*x2^27*x3^12*x4^6*z^15 - x1^36*x2^20*x3^13*x4^6*z^15 + x1^35*x2^21*x3^13*x4^6*z^15 + 5*x1^34*x2^22*x3^13*x4^6*z^15 + 2*x1^33*x2^23*x3^13*x4^6*z^15 - 3*x1^31*x2^25*x3^13*x4^6*z^15 + 2*x1^30*x2^26*x3^13*x4^6*z^15 - 4*x1^29*x2^27*x3^13*x4^6*z^15 - x1^36*x2^19*x3^14*x4^6*z^15 + 2*x1^35*x2^20*x3^14*x4^6*z^15 - 4*x1^34*x2^21*x3^14*x4^6*z^15 + x1^33*x2^22*x3^14*x4^6*z^15 - x1^32*x2^23*x3^14*x4^6*z^15 - x1^31*x2^24*x3^14*x4^6*z^15 - x1^30*x2^25*x3^14*x4^6*z^15 + 4*x1^28*x2^27*x3^14*x4^6*z^15 - x1^34*x2^20*x3^15*x4^6*z^15 + 2*x1^33*x2^21*x3^15*x4^6*z^15 + x1^32*x2^22*x3^15*x4^6*z^15 + 3*x1^31*x2^23*x3^15*x4^6*z^15 - 2*x1^30*x2^24*x3^15*x4^6*z^15 + 2*x1^29*x2^25*x3^15*x4^6*z^15 - 4*x1^28*x2^26*x3^15*x4^6*z^15 - x1^27*x2^27*x3^15*x4^6*z^15 - 3*x1^33*x2^20*x3^16*x4^6*z^15 - 2*x1^32*x2^21*x3^16*x4^6*z^15 - 2*x1^31*x2^22*x3^16*x4^6*z^15 - x1^29*x2^24*x3^16*x4^6*z^15 + 4*x1^27*x2^26*x3^16*x4^6*z^15 + x1^33*x2^19*x3^17*x4^6*z^15 + 3*x1^32*x2^20*x3^17*x4^6*z^15 + 3*x1^30*x2^22*x3^17*x4^6*z^15 - x1^29*x2^23*x3^17*x4^6*z^15 + 2*x1^28*x2^24*x3^17*x4^6*z^15 - 4*x1^27*x2^25*x3^17*x4^6*z^15 - 2*x1^26*x2^26*x3^17*x4^6*z^15 - x1^32*x2^19*x3^18*x4^6*z^15 - x1^31*x2^20*x3^18*x4^6*z^15 - x1^30*x2^21*x3^18*x4^6*z^15 - x1^28*x2^23*x3^18*x4^6*z^15 + 2*x1^27*x2^24*x3^18*x4^6*z^15 + 4*x1^26*x2^25*x3^18*x4^6*z^15 + x1^31*x2^19*x3^19*x4^6*z^15 + 2*x1^30*x2^20*x3^19*x4^6*z^15 + x1^29*x2^21*x3^19*x4^6*z^15 - 2*x1^28*x2^22*x3^19*x4^6*z^15 + 3*x1^27*x2^23*x3^19*x4^6*z^15 - 4*x1^26*x2^24*x3^19*x4^6*z^15 - x1^25*x2^25*x3^19*x4^6*z^15 - x1^29*x2^20*x3^20*x4^6*z^15 - x1^26*x2^23*x3^20*x4^6*z^15 + 4*x1^25*x2^24*x3^20*x4^6*z^15 + x1^26*x2^22*x3^21*x4^6*z^15 - 2*x1^25*x2^23*x3^21*x4^6*z^15 - x1^24*x2^24*x3^21*x4^6*z^15 - x1^25*x2^22*x3^22*x4^6*z^15 + x1^24*x2^23*x3^22*x4^6*z^15 - x1^35*x2^26*x3^7*x4^7*z^15 + x1^34*x2^27*x3^7*x4^7*z^15 - x1^31*x2^29*x3^8*x4^7*z^15 - 2*x1^34*x2^25*x3^9*x4^7*z^15 + 2*x1^30*x2^29*x3^9*x4^7*z^15 - x1^37*x2^21*x3^10*x4^7*z^15 - x1^36*x2^22*x3^10*x4^7*z^15 + x1^35*x2^23*x3^10*x4^7*z^15 - x1^33*x2^25*x3^10*x4^7*z^15 + 2*x1^31*x2^27*x3^10*x4^7*z^15 - x1^30*x2^28*x3^10*x4^7*z^15 - x1^29*x2^29*x3^10*x4^7*z^15 + x1^34*x2^23*x3^11*x4^7*z^15 - x1^33*x2^24*x3^11*x4^7*z^15 + x1^32*x2^25*x3^11*x4^7*z^15 - x1^31*x2^26*x3^11*x4^7*z^15 + x1^30*x2^27*x3^11*x4^7*z^15 + 2*x1^29*x2^28*x3^11*x4^7*z^15 - x1^36*x2^20*x3^12*x4^7*z^15 - x1^35*x2^21*x3^12*x4^7*z^15 - x1^33*x2^23*x3^12*x4^7*z^15 + x1^32*x2^24*x3^12*x4^7*z^15 + x1^29*x2^27*x3^12*x4^7*z^15 + x1^36*x2^19*x3^13*x4^7*z^15 + x1^34*x2^21*x3^13*x4^7*z^15 + 2*x1^33*x2^22*x3^13*x4^7*z^15 - 2*x1^35*x2^19*x3^14*x4^7*z^15 - x1^34*x2^20*x3^14*x4^7*z^15 - 2*x1^33*x2^21*x3^14*x4^7*z^15 - x1^32*x2^22*x3^14*x4^7*z^15 + 3*x1^33*x2^20*x3^15*x4^7*z^15 + 2*x1^32*x2^21*x3^15*x4^7*z^15 + 2*x1^30*x2^23*x3^15*x4^7*z^15 + x1^34*x2^18*x3^16*x4^7*z^15 - x1^32*x2^20*x3^16*x4^7*z^15 + x1^31*x2^21*x3^16*x4^7*z^15 - x1^30*x2^22*x3^16*x4^7*z^15 - x1^33*x2^18*x3^17*x4^7*z^15 + x1^32*x2^19*x3^17*x4^7*z^15 + 2*x1^31*x2^20*x3^17*x4^7*z^15 + x1^30*x2^21*x3^17*x4^7*z^15 - x1^31*x2^19*x3^18*x4^7*z^15 - x1^30*x2^20*x3^18*x4^7*z^15 + 2*x1^28*x2^22*x3^18*x4^7*z^15 + 2*x1^26*x2^24*x3^18*x4^7*z^15 - x1^30*x2^19*x3^19*x4^7*z^15 - x1^27*x2^22*x3^19*x4^7*z^15 - x1^26*x2^23*x3^19*x4^7*z^15 - 2*x1^25*x2^24*x3^19*x4^7*z^15 - x1^28*x2^20*x3^20*x4^7*z^15 + 2*x1^25*x2^23*x3^20*x4^7*z^15 - x1^26*x2^21*x3^21*x4^7*z^15 - 2*x1^24*x2^23*x3^21*x4^7*z^15 + x1^23*x2^23*x3^22*x4^7*z^15 - x1^34*x2^25*x3^8*x4^8*z^15 - x1^33*x2^26*x3^8*x4^8*z^15 + x1^31*x2^28*x3^8*x4^8*z^15 + x1^34*x2^24*x3^9*x4^8*z^15 - x1^31*x2^27*x3^9*x4^8*z^15 + x1^33*x2^24*x3^10*x4^8*z^15 - x1^32*x2^25*x3^10*x4^8*z^15 - x1^31*x2^26*x3^10*x4^8*z^15 - 2*x1^29*x2^28*x3^10*x4^8*z^15 - x1^35*x2^21*x3^11*x4^8*z^15 - 2*x1^34*x2^22*x3^11*x4^8*z^15 + 3*x1^31*x2^25*x3^11*x4^8*z^15 + x1^29*x2^27*x3^11*x4^8*z^15 + x1^28*x2^28*x3^11*x4^8*z^15 + x1^35*x2^20*x3^12*x4^8*z^15 - x1^33*x2^22*x3^12*x4^8*z^15 - 2*x1^32*x2^23*x3^12*x4^8*z^15 - 2*x1^31*x2^24*x3^12*x4^8*z^15 - x1^30*x2^25*x3^12*x4^8*z^15 - x1^29*x2^26*x3^12*x4^8*z^15 - 3*x1^28*x2^27*x3^12*x4^8*z^15 - x1^33*x2^21*x3^13*x4^8*z^15 + x1^32*x2^22*x3^13*x4^8*z^15 - x1^31*x2^23*x3^13*x4^8*z^15 + 3*x1^30*x2^24*x3^13*x4^8*z^15 + 2*x1^29*x2^25*x3^13*x4^8*z^15 + 2*x1^28*x2^26*x3^13*x4^8*z^15 + x1^27*x2^27*x3^13*x4^8*z^15 + 2*x1^34*x2^19*x3^14*x4^8*z^15 + x1^32*x2^21*x3^14*x4^8*z^15 - x1^31*x2^22*x3^14*x4^8*z^15 - x1^30*x2^23*x3^14*x4^8*z^15 - 2*x1^29*x2^24*x3^14*x4^8*z^15 - x1^28*x2^25*x3^14*x4^8*z^15 - 3*x1^27*x2^26*x3^14*x4^8*z^15 - x1^33*x2^19*x3^15*x4^8*z^15 - x1^32*x2^20*x3^15*x4^8*z^15 + x1^29*x2^23*x3^15*x4^8*z^15 + 2*x1^28*x2^24*x3^15*x4^8*z^15 + 4*x1^27*x2^25*x3^15*x4^8*z^15 + 2*x1^26*x2^26*x3^15*x4^8*z^15 + x1^33*x2^18*x3^16*x4^8*z^15 + x1^31*x2^20*x3^16*x4^8*z^15 - x1^30*x2^21*x3^16*x4^8*z^15 - x1^29*x2^22*x3^16*x4^8*z^15 - 2*x1^28*x2^23*x3^16*x4^8*z^15 - 2*x1^27*x2^24*x3^16*x4^8*z^15 - 4*x1^26*x2^25*x3^16*x4^8*z^15 - x1^31*x2^19*x3^17*x4^8*z^15 - x1^30*x2^20*x3^17*x4^8*z^15 + 2*x1^29*x2^21*x3^17*x4^8*z^15 + 2*x1^28*x2^22*x3^17*x4^8*z^15 + x1^26*x2^24*x3^17*x4^8*z^15 + x1^25*x2^25*x3^17*x4^8*z^15 - x1^29*x2^20*x3^18*x4^8*z^15 + x1^28*x2^21*x3^18*x4^8*z^15 - x1^27*x2^22*x3^18*x4^8*z^15 + x1^26*x2^23*x3^18*x4^8*z^15 - x1^25*x2^24*x3^18*x4^8*z^15 - x1^28*x2^20*x3^19*x4^8*z^15 - x1^27*x2^21*x3^19*x4^8*z^15 + 2*x1^26*x2^22*x3^19*x4^8*z^15 - x1^25*x2^23*x3^19*x4^8*z^15 + x1^27*x2^20*x3^20*x4^8*z^15 - x1^25*x2^22*x3^20*x4^8*z^15 + x1^24*x2^23*x3^20*x4^8*z^15 + x1^33*x2^24*x3^9*x4^9*z^15 + x1^32*x2^25*x3^9*x4^9*z^15 + 2*x1^31*x2^26*x3^9*x4^9*z^15 + x1^29*x2^28*x3^9*x4^9*z^15 - x1^33*x2^23*x3^10*x4^9*z^15 - 2*x1^31*x2^25*x3^10*x4^9*z^15 - x1^28*x2^28*x3^10*x4^9*z^15 + 3*x1^32*x2^23*x3^11*x4^9*z^15 + x1^31*x2^24*x3^11*x4^9*z^15 + 2*x1^30*x2^25*x3^11*x4^9*z^15 + x1^29*x2^26*x3^11*x4^9*z^15 + x1^28*x2^27*x3^11*x4^9*z^15 + x1^34*x2^20*x3^12*x4^9*z^15 + 2*x1^33*x2^21*x3^12*x4^9*z^15 - x1^32*x2^22*x3^12*x4^9*z^15 - x1^30*x2^24*x3^12*x4^9*z^15 - 2*x1^29*x2^25*x3^12*x4^9*z^15 - x1^28*x2^26*x3^12*x4^9*z^15 - x1^33*x2^20*x3^13*x4^9*z^15 - x1^32*x2^21*x3^13*x4^9*z^15 + 5*x1^31*x2^22*x3^13*x4^9*z^15 + x1^30*x2^23*x3^13*x4^9*z^15 + 3*x1^29*x2^24*x3^13*x4^9*z^15 + 2*x1^28*x2^25*x3^13*x4^9*z^15 + x1^27*x2^26*x3^13*x4^9*z^15 + x1^34*x2^18*x3^14*x4^9*z^15 + x1^33*x2^19*x3^14*x4^9*z^15 + x1^32*x2^20*x3^14*x4^9*z^15 - 3*x1^31*x2^21*x3^14*x4^9*z^15 - x1^30*x2^22*x3^14*x4^9*z^15 - x1^29*x2^23*x3^14*x4^9*z^15 - 3*x1^28*x2^24*x3^14*x4^9*z^15 - 2*x1^27*x2^25*x3^14*x4^9*z^15 - x1^26*x2^26*x3^14*x4^9*z^15 - 2*x1^33*x2^18*x3^15*x4^9*z^15 + x1^32*x2^19*x3^15*x4^9*z^15 + 3*x1^30*x2^21*x3^15*x4^9*z^15 + x1^29*x2^22*x3^15*x4^9*z^15 + 4*x1^28*x2^23*x3^15*x4^9*z^15 + 2*x1^27*x2^24*x3^15*x4^9*z^15 + x1^26*x2^25*x3^15*x4^9*z^15 + x1^32*x2^18*x3^16*x4^9*z^15 - x1^30*x2^20*x3^16*x4^9*z^15 - 3*x1^29*x2^21*x3^16*x4^9*z^15 - 3*x1^28*x2^22*x3^16*x4^9*z^15 - 4*x1^27*x2^23*x3^16*x4^9*z^15 - x1^26*x2^24*x3^16*x4^9*z^15 - x1^25*x2^25*x3^16*x4^9*z^15 - x1^31*x2^18*x3^17*x4^9*z^15 + 2*x1^29*x2^20*x3^17*x4^9*z^15 + 5*x1^27*x2^22*x3^17*x4^9*z^15 + 2*x1^26*x2^23*x3^17*x4^9*z^15 + x1^25*x2^24*x3^17*x4^9*z^15 + x1^30*x2^18*x3^18*x4^9*z^15 - x1^29*x2^19*x3^18*x4^9*z^15 - x1^28*x2^20*x3^18*x4^9*z^15 - 3*x1^27*x2^21*x3^18*x4^9*z^15 - 5*x1^26*x2^22*x3^18*x4^9*z^15 - x1^25*x2^23*x3^18*x4^9*z^15 + 2*x1^28*x2^19*x3^19*x4^9*z^15 + 3*x1^26*x2^21*x3^19*x4^9*z^15 + x1^25*x2^22*x3^19*x4^9*z^15 + x1^24*x2^23*x3^19*x4^9*z^15 + x1^26*x2^20*x3^20*x4^9*z^15 - 3*x1^25*x2^21*x3^20*x4^9*z^15 - x1^23*x2^23*x3^20*x4^9*z^15 + 2*x1^24*x2^21*x3^21*x4^9*z^15 - x1^31*x2^23*x3^11*x4^10*z^15 + 2*x1^30*x2^24*x3^11*x4^10*z^15 + x1^28*x2^26*x3^11*x4^10*z^15 + x1^27*x2^27*x3^11*x4^10*z^15 - x1^31*x2^22*x3^12*x4^10*z^15 - x1^30*x2^23*x3^12*x4^10*z^15 - x1^29*x2^24*x3^12*x4^10*z^15 - x1^33*x2^19*x3^13*x4^10*z^15 + x1^31*x2^21*x3^13*x4^10*z^15 - x1^30*x2^22*x3^13*x4^10*z^15 + x1^29*x2^23*x3^13*x4^10*z^15 - x1^28*x2^24*x3^13*x4^10*z^15 + x1^27*x2^25*x3^13*x4^10*z^15 + x1^26*x2^26*x3^13*x4^10*z^15 - x1^33*x2^18*x3^14*x4^10*z^15 - x1^32*x2^19*x3^14*x4^10*z^15 + 2*x1^31*x2^20*x3^14*x4^10*z^15 - x1^30*x2^21*x3^14*x4^10*z^15 - x1^29*x2^22*x3^14*x4^10*z^15 - x1^27*x2^24*x3^14*x4^10*z^15 - x1^31*x2^19*x3^15*x4^10*z^15 - x1^30*x2^20*x3^15*x4^10*z^15 - x1^28*x2^22*x3^15*x4^10*z^15 - 2*x1^27*x2^23*x3^15*x4^10*z^15 + 2*x1^26*x2^24*x3^15*x4^10*z^15 + 2*x1^28*x2^21*x3^16*x4^10*z^15 - x1^27*x2^22*x3^16*x4^10*z^15 - x1^26*x2^23*x3^16*x4^10*z^15 - x1^30*x2^18*x3^17*x4^10*z^15 + x1^28*x2^20*x3^17*x4^10*z^15 + x1^27*x2^21*x3^17*x4^10*z^15 + x1^26*x2^22*x3^17*x4^10*z^15 + x1^29*x2^18*x3^18*x4^10*z^15 + x1^27*x2^20*x3^18*x4^10*z^15 - x1^26*x2^21*x3^18*x4^10*z^15 + x1^25*x2^22*x3^18*x4^10*z^15 - x1^26*x2^20*x3^19*x4^10*z^15 + 2*x1^25*x2^21*x3^19*x4^10*z^15 - x1^24*x2^21*x3^20*x4^10*z^15 - x1^31*x2^22*x3^11*x4^11*z^15 - x1^30*x2^23*x3^11*x4^11*z^15 - x1^29*x2^24*x3^11*x4^11*z^15 + x1^31*x2^21*x3^12*x4^11*z^15 - 3*x1^29*x2^23*x3^12*x4^11*z^15 - x1^28*x2^24*x3^12*x4^11*z^15 - x1^27*x2^25*x3^12*x4^11*z^15 + x1^29*x2^22*x3^13*x4^11*z^15 - x1^28*x2^23*x3^13*x4^11*z^15 - x1^27*x2^24*x3^13*x4^11*z^15 + x1^26*x2^25*x3^13*x4^11*z^15 + x1^29*x2^21*x3^14*x4^11*z^15 - 2*x1^28*x2^22*x3^14*x4^11*z^15 + 2*x1^27*x2^23*x3^14*x4^11*z^15 - 2*x1^26*x2^24*x3^14*x4^11*z^15 - x1^25*x2^25*x3^14*x4^11*z^15 - 3*x1^30*x2^19*x3^15*x4^11*z^15 - x1^29*x2^20*x3^15*x4^11*z^15 + 2*x1^28*x2^21*x3^15*x4^11*z^15 - x1^27*x2^22*x3^15*x4^11*z^15 - 4*x1^26*x2^23*x3^15*x4^11*z^15 + 2*x1^25*x2^24*x3^15*x4^11*z^15 + x1^29*x2^19*x3^16*x4^11*z^15 - x1^27*x2^21*x3^16*x4^11*z^15 + 2*x1^26*x2^22*x3^16*x4^11*z^15 - 2*x1^25*x2^23*x3^16*x4^11*z^15 - x1^24*x2^24*x3^16*x4^11*z^15 - x1^27*x2^20*x3^17*x4^11*z^15 + x1^25*x2^22*x3^17*x4^11*z^15 + x1^24*x2^23*x3^17*x4^11*z^15 + x1^25*x2^21*x3^18*x4^11*z^15 - x1^24*x2^22*x3^18*x4^11*z^15 + x1^23*x2^22*x3^19*x4^11*z^15 + x1^30*x2^21*x3^12*x4^12*z^15 + x1^29*x2^22*x3^12*x4^12*z^15 + x1^28*x2^23*x3^12*x4^12*z^15 - x1^30*x2^20*x3^13*x4^12*z^15 + 3*x1^28*x2^22*x3^13*x4^12*z^15 - x1^27*x2^23*x3^13*x4^12*z^15 + x1^26*x2^24*x3^13*x4^12*z^15 - x1^28*x2^21*x3^14*x4^12*z^15 + x1^26*x2^23*x3^14*x4^12*z^15 - 3*x1^25*x2^24*x3^14*x4^12*z^15 + x1^29*x2^19*x3^15*x4^12*z^15 + x1^28*x2^20*x3^15*x4^12*z^15 + 4*x1^27*x2^21*x3^15*x4^12*z^15 - 2*x1^26*x2^22*x3^15*x4^12*z^15 + 2*x1^25*x2^23*x3^15*x4^12*z^15 + 2*x1^24*x2^24*x3^15*x4^12*z^15 + x1^28*x2^19*x3^16*x4^12*z^15 - x1^27*x2^20*x3^16*x4^12*z^15 + 2*x1^25*x2^22*x3^16*x4^12*z^15 - 5*x1^24*x2^23*x3^16*x4^12*z^15 + 2*x1^26*x2^20*x3^17*x4^12*z^15 - 2*x1^25*x2^21*x3^17*x4^12*z^15 + 2*x1^24*x2^22*x3^17*x4^12*z^15 + 2*x1^23*x2^23*x3^17*x4^12*z^15 + x1^24*x2^21*x3^18*x4^12*z^15 - 3*x1^23*x2^22*x3^18*x4^12*z^15 + x1^22*x2^22*x3^19*x4^12*z^15 - x1^28*x2^20*x3^14*x4^13*z^15 + 2*x1^26*x2^22*x3^14*x4^13*z^15 + x1^26*x2^21*x3^15*x4^13*z^15 + 2*x1^24*x2^23*x3^15*x4^13*z^15 - x1^26*x2^20*x3^16*x4^13*z^15 + x1^25*x2^21*x3^16*x4^13*z^15 - x1^23*x2^23*x3^16*x4^13*z^15 + x1^25*x2^20*x3^17*x4^13*z^15 + x1^24*x2^21*x3^17*x4^13*z^15 + 2*x1^23*x2^22*x3^17*x4^13*z^15 - x1^25*x2^21*x3^15*x4^14*z^15 + x1^24*x2^21*x3^16*x4^14*z^15 - x1^23*x2^20*x3^17*x4^15*z^15 - x1^38*x2^25*x3^7*z^14 + x1^37*x2^25*x3^8*z^14 - 2*x1^37*x2^24*x3^9*z^14 - x1^35*x2^26*x3^9*z^14 + x1^37*x2^23*x3^10*z^14 + 2*x1^36*x2^24*x3^10*z^14 + 2*x1^34*x2^26*x3^10*z^14 + x1^33*x2^27*x3^10*z^14 - 2*x1^36*x2^23*x3^11*z^14 - x1^35*x2^24*x3^11*z^14 - x1^34*x2^25*x3^11*z^14 + x1^31*x2^28*x3^11*z^14 + x1^36*x2^22*x3^12*z^14 + 2*x1^35*x2^23*x3^12*z^14 + x1^33*x2^25*x3^12*z^14 - 2*x1^32*x2^26*x3^12*z^14 - x1^30*x2^28*x3^12*z^14 - 2*x1^35*x2^22*x3^13*z^14 + x1^29*x2^28*x3^13*z^14 + 2*x1^34*x2^22*x3^14*z^14 + 2*x1^32*x2^24*x3^14*z^14 - x1^31*x2^25*x3^14*z^14 - 2*x1^29*x2^27*x3^14*z^14 - x1^28*x2^28*x3^14*z^14 - 2*x1^34*x2^21*x3^15*z^14 - x1^33*x2^22*x3^15*z^14 - x1^32*x2^23*x3^15*z^14 + x1^28*x2^27*x3^15*z^14 + x1^34*x2^20*x3^16*z^14 + 2*x1^33*x2^21*x3^16*z^14 + x1^31*x2^23*x3^16*z^14 + x1^30*x2^24*x3^16*z^14 - 2*x1^28*x2^26*x3^16*z^14 - 2*x1^33*x2^20*x3^17*z^14 - x1^32*x2^21*x3^17*z^14 - x1^31*x2^22*x3^17*z^14 - x1^30*x2^23*x3^17*z^14 + x1^29*x2^24*x3^17*z^14 + x1^28*x2^25*x3^17*z^14 + x1^32*x2^20*x3^18*z^14 + x1^30*x2^22*x3^18*z^14 - x1^30*x2^21*x3^19*z^14 - x1^29*x2^22*x3^19*z^14 + x1^36*x2^27*x3^6*x4*z^14 - x1^38*x2^24*x3^7*x4*z^14 - 2*x1^37*x2^25*x3^7*x4*z^14 + x1^36*x2^26*x3^7*x4*z^14 - x1^35*x2^27*x3^7*x4*z^14 + 4*x1^37*x2^24*x3^8*x4*z^14 + x1^35*x2^26*x3^8*x4*z^14 - 2*x1^37*x2^23*x3^9*x4*z^14 - 6*x1^36*x2^24*x3^9*x4*z^14 + 2*x1^35*x2^25*x3^9*x4*z^14 - 2*x1^34*x2^26*x3^9*x4*z^14 + 2*x1^33*x2^27*x3^9*x4*z^14 + 6*x1^36*x2^23*x3^10*x4*z^14 + 2*x1^35*x2^24*x3^10*x4*z^14 - x1^33*x2^26*x3^10*x4*z^14 - 2*x1^32*x2^27*x3^10*x4*z^14 - x1^31*x2^28*x3^10*x4*z^14 - 2*x1^36*x2^22*x3^11*x4*z^14 - 6*x1^35*x2^23*x3^11*x4*z^14 - 4*x1^33*x2^25*x3^11*x4*z^14 + 3*x1^32*x2^26*x3^11*x4*z^14 + 2*x1^31*x2^27*x3^11*x4*z^14 + 3*x1^30*x2^28*x3^11*x4*z^14 + 6*x1^35*x2^22*x3^12*x4*z^14 + 2*x1^34*x2^23*x3^12*x4*z^14 + 2*x1^33*x2^24*x3^12*x4*z^14 - 2*x1^31*x2^26*x3^12*x4*z^14 - 4*x1^30*x2^27*x3^12*x4*z^14 - 3*x1^29*x2^28*x3^12*x4*z^14 - 2*x1^35*x2^21*x3^13*x4*z^14 - 6*x1^34*x2^22*x3^13*x4*z^14 - 4*x1^32*x2^24*x3^13*x4*z^14 + 4*x1^31*x2^25*x3^13*x4*z^14 + 6*x1^29*x2^27*x3^13*x4*z^14 + x1^28*x2^28*x3^13*x4*z^14 + 6*x1^34*x2^21*x3^14*x4*z^14 + 2*x1^33*x2^22*x3^14*x4*z^14 + 2*x1^32*x2^23*x3^14*x4*z^14 - 2*x1^30*x2^25*x3^14*x4*z^14 - 2*x1^29*x2^26*x3^14*x4*z^14 - 6*x1^28*x2^27*x3^14*x4*z^14 - x1^34*x2^20*x3^15*x4*z^14 - 6*x1^33*x2^21*x3^15*x4*z^14 - 4*x1^31*x2^23*x3^15*x4*z^14 + 4*x1^30*x2^24*x3^15*x4*z^14 + 6*x1^28*x2^26*x3^15*x4*z^14 + 2*x1^27*x2^27*x3^15*x4*z^14 + 4*x1^33*x2^20*x3^16*x4*z^14 + 3*x1^32*x2^21*x3^16*x4*z^14 + 2*x1^31*x2^22*x3^16*x4*z^14 - x1^29*x2^24*x3^16*x4*z^14 - x1^28*x2^25*x3^16*x4*z^14 - 5*x1^27*x2^26*x3^16*x4*z^14 - 4*x1^32*x2^20*x3^17*x4*z^14 - x1^31*x2^21*x3^17*x4*z^14 - 4*x1^30*x2^22*x3^17*x4*z^14 + x1^28*x2^24*x3^17*x4*z^14 + 5*x1^27*x2^25*x3^17*x4*z^14 + x1^31*x2^20*x3^18*x4*z^14 + x1^30*x2^21*x3^18*x4*z^14 + 2*x1^29*x2^22*x3^18*x4*z^14 + x1^28*x2^23*x3^18*x4*z^14 - 2*x1^27*x2^24*x3^18*x4*z^14 - 2*x1^26*x2^25*x3^18*x4*z^14 - x1^30*x2^20*x3^19*x4*z^14 - 2*x1^29*x2^21*x3^19*x4*z^14 - 2*x1^28*x2^22*x3^19*x4*z^14 + x1^27*x2^23*x3^19*x4*z^14 + 3*x1^26*x2^24*x3^19*x4*z^14 + x1^29*x2^20*x3^20*x4*z^14 + x1^28*x2^21*x3^20*x4*z^14 + x1^27*x2^22*x3^20*x4*z^14 - x1^25*x2^24*x3^20*x4*z^14 + x1^37*x2^25*x3^6*x4^2*z^14 - x1^36*x2^26*x3^6*x4^2*z^14 + x1^35*x2^27*x3^6*x4^2*z^14 - x1^37*x2^24*x3^7*x4^2*z^14 - x1^36*x2^25*x3^7*x4^2*z^14 + 2*x1^33*x2^28*x3^7*x4^2*z^14 + 4*x1^36*x2^24*x3^8*x4^2*z^14 - 2*x1^33*x2^27*x3^8*x4^2*z^14 - 2*x1^32*x2^28*x3^8*x4^2*z^14 - 3*x1^36*x2^23*x3^9*x4^2*z^14 - 3*x1^35*x2^24*x3^9*x4^2*z^14 - x1^34*x2^25*x3^9*x4^2*z^14 + x1^33*x2^26*x3^9*x4^2*z^14 + x1^32*x2^27*x3^9*x4^2*z^14 + 2*x1^31*x2^28*x3^9*x4^2*z^14 + 2*x1^36*x2^22*x3^10*x4^2*z^14 + 5*x1^35*x2^23*x3^10*x4^2*z^14 - x1^34*x2^24*x3^10*x4^2*z^14 + 3*x1^33*x2^25*x3^10*x4^2*z^14 - 3*x1^32*x2^26*x3^10*x4^2*z^14 - x1^31*x2^27*x3^10*x4^2*z^14 - 4*x1^30*x2^28*x3^10*x4^2*z^14 - 6*x1^35*x2^22*x3^11*x4^2*z^14 - 2*x1^34*x2^23*x3^11*x4^2*z^14 - 2*x1^33*x2^24*x3^11*x4^2*z^14 + x1^31*x2^26*x3^11*x4^2*z^14 + 2*x1^30*x2^27*x3^11*x4^2*z^14 + 4*x1^29*x2^28*x3^11*x4^2*z^14 + 2*x1^35*x2^21*x3^12*x4^2*z^14 + 6*x1^34*x2^22*x3^12*x4^2*z^14 + 4*x1^32*x2^24*x3^12*x4^2*z^14 - 4*x1^31*x2^25*x3^12*x4^2*z^14 - 6*x1^29*x2^27*x3^12*x4^2*z^14 - 2*x1^28*x2^28*x3^12*x4^2*z^14 - 6*x1^34*x2^21*x3^13*x4^2*z^14 - 2*x1^33*x2^22*x3^13*x4^2*z^14 - 2*x1^32*x2^23*x3^13*x4^2*z^14 + 2*x1^30*x2^25*x3^13*x4^2*z^14 + 2*x1^29*x2^26*x3^13*x4^2*z^14 + 6*x1^28*x2^27*x3^13*x4^2*z^14 + 6*x1^33*x2^21*x3^14*x4^2*z^14 + 4*x1^31*x2^23*x3^14*x4^2*z^14 - 4*x1^30*x2^24*x3^14*x4^2*z^14 - 6*x1^28*x2^26*x3^14*x4^2*z^14 - 2*x1^27*x2^27*x3^14*x4^2*z^14 - 3*x1^33*x2^20*x3^15*x4^2*z^14 - 3*x1^32*x2^21*x3^15*x4^2*z^14 - 2*x1^31*x2^22*x3^15*x4^2*z^14 + 2*x1^29*x2^24*x3^15*x4^2*z^14 + 2*x1^28*x2^25*x3^15*x4^2*z^14 + 6*x1^27*x2^26*x3^15*x4^2*z^14 + 3*x1^32*x2^20*x3^16*x4^2*z^14 + 2*x1^30*x2^22*x3^16*x4^2*z^14 - 4*x1^29*x2^23*x3^16*x4^2*z^14 - 6*x1^27*x2^25*x3^16*x4^2*z^14 - 2*x1^26*x2^26*x3^16*x4^2*z^14 - x1^32*x2^19*x3^17*x4^2*z^14 - x1^31*x2^20*x3^17*x4^2*z^14 - x1^30*x2^21*x3^17*x4^2*z^14 + x1^29*x2^22*x3^17*x4^2*z^14 + x1^28*x2^23*x3^17*x4^2*z^14 + 2*x1^27*x2^24*x3^17*x4^2*z^14 + 6*x1^26*x2^25*x3^17*x4^2*z^14 + x1^31*x2^19*x3^18*x4^2*z^14 + x1^29*x2^21*x3^18*x4^2*z^14 - 3*x1^28*x2^22*x3^18*x4^2*z^14 - 5*x1^26*x2^24*x3^18*x4^2*z^14 - x1^25*x2^25*x3^18*x4^2*z^14 + x1^28*x2^21*x3^19*x4^2*z^14 + x1^26*x2^23*x3^19*x4^2*z^14 + 4*x1^25*x2^24*x3^19*x4^2*z^14 - x1^26*x2^22*x3^20*x4^2*z^14 - 2*x1^25*x2^23*x3^20*x4^2*z^14 + x1^24*x2^23*x3^21*x4^2*z^14 - x1^37*x2^24*x3^6*x4^3*z^14 - x1^34*x2^27*x3^6*x4^3*z^14 + x1^38*x2^22*x3^7*x4^3*z^14 - x1^35*x2^25*x3^7*x4^3*z^14 - x1^34*x2^26*x3^7*x4^3*z^14 - x1^37*x2^22*x3^8*x4^3*z^14 + x1^36*x2^23*x3^8*x4^3*z^14 - x1^35*x2^24*x3^8*x4^3*z^14 - x1^34*x2^25*x3^8*x4^3*z^14 - x1^31*x2^28*x3^8*x4^3*z^14 + 2*x1^37*x2^21*x3^9*x4^3*z^14 + 2*x1^36*x2^22*x3^9*x4^3*z^14 + x1^35*x2^23*x3^9*x4^3*z^14 + x1^32*x2^26*x3^9*x4^3*z^14 + x1^30*x2^28*x3^9*x4^3*z^14 - x1^37*x2^20*x3^10*x4^3*z^14 - 2*x1^36*x2^21*x3^10*x4^3*z^14 + x1^35*x2^22*x3^10*x4^3*z^14 + x1^34*x2^23*x3^10*x4^3*z^14 + x1^33*x2^24*x3^10*x4^3*z^14 + x1^31*x2^26*x3^10*x4^3*z^14 + x1^30*x2^27*x3^10*x4^3*z^14 - x1^29*x2^28*x3^10*x4^3*z^14 + 2*x1^36*x2^20*x3^11*x4^3*z^14 + x1^33*x2^23*x3^11*x4^3*z^14 - x1^32*x2^24*x3^11*x4^3*z^14 - x1^30*x2^26*x3^11*x4^3*z^14 + x1^29*x2^27*x3^11*x4^3*z^14 + x1^28*x2^28*x3^11*x4^3*z^14 - x1^36*x2^19*x3^12*x4^3*z^14 - 2*x1^35*x2^20*x3^12*x4^3*z^14 + 2*x1^34*x2^21*x3^12*x4^3*z^14 + 3*x1^32*x2^23*x3^12*x4^3*z^14 + x1^30*x2^25*x3^12*x4^3*z^14 - x1^29*x2^26*x3^12*x4^3*z^14 - x1^28*x2^27*x3^12*x4^3*z^14 + 2*x1^35*x2^19*x3^13*x4^3*z^14 - x1^34*x2^20*x3^13*x4^3*z^14 - 2*x1^33*x2^21*x3^13*x4^3*z^14 - x1^31*x2^23*x3^13*x4^3*z^14 + 2*x1^30*x2^24*x3^13*x4^3*z^14 - 2*x1^29*x2^25*x3^13*x4^3*z^14 + 2*x1^28*x2^26*x3^13*x4^3*z^14 - 2*x1^34*x2^19*x3^14*x4^3*z^14 + 2*x1^33*x2^20*x3^14*x4^3*z^14 - 2*x1^32*x2^21*x3^14*x4^3*z^14 + x1^31*x2^22*x3^14*x4^3*z^14 + 2*x1^29*x2^24*x3^14*x4^3*z^14 + x1^28*x2^25*x3^14*x4^3*z^14 - 2*x1^27*x2^26*x3^14*x4^3*z^14 + x1^34*x2^18*x3^15*x4^3*z^14 + 2*x1^33*x2^19*x3^15*x4^3*z^14 - x1^32*x2^20*x3^15*x4^3*z^14 - x1^31*x2^21*x3^15*x4^3*z^14 - 3*x1^30*x2^22*x3^15*x4^3*z^14 - 2*x1^28*x2^24*x3^15*x4^3*z^14 + 2*x1^27*x2^25*x3^15*x4^3*z^14 + x1^26*x2^26*x3^15*x4^3*z^14 - x1^33*x2^18*x3^16*x4^3*z^14 - x1^32*x2^19*x3^16*x4^3*z^14 + x1^31*x2^20*x3^16*x4^3*z^14 + 3*x1^30*x2^21*x3^16*x4^3*z^14 + x1^28*x2^23*x3^16*x4^3*z^14 - 2*x1^26*x2^25*x3^16*x4^3*z^14 + x1^31*x2^19*x3^17*x4^3*z^14 - x1^30*x2^20*x3^17*x4^3*z^14 - x1^29*x2^21*x3^17*x4^3*z^14 - 2*x1^27*x2^23*x3^17*x4^3*z^14 + 2*x1^26*x2^24*x3^17*x4^3*z^14 + x1^25*x2^25*x3^17*x4^3*z^14 - x1^30*x2^19*x3^18*x4^3*z^14 + x1^29*x2^20*x3^18*x4^3*z^14 - 2*x1^26*x2^23*x3^18*x4^3*z^14 - 3*x1^25*x2^24*x3^18*x4^3*z^14 + x1^28*x2^20*x3^19*x4^3*z^14 - 3*x1^26*x2^22*x3^19*x4^3*z^14 + x1^24*x2^24*x3^19*x4^3*z^14 - x1^27*x2^20*x3^20*x4^3*z^14 + 2*x1^26*x2^21*x3^20*x4^3*z^14 + x1^25*x2^22*x3^20*x4^3*z^14 - 2*x1^24*x2^23*x3^20*x4^3*z^14 + x1^37*x2^22*x3^7*x4^4*z^14 + 2*x1^36*x2^23*x3^7*x4^4*z^14 + x1^35*x2^24*x3^7*x4^4*z^14 - 2*x1^34*x2^25*x3^7*x4^4*z^14 + x1^33*x2^26*x3^7*x4^4*z^14 + x1^32*x2^27*x3^7*x4^4*z^14 - 2*x1^37*x2^21*x3^8*x4^4*z^14 - x1^36*x2^22*x3^8*x4^4*z^14 - x1^35*x2^23*x3^8*x4^4*z^14 + 3*x1^34*x2^24*x3^8*x4^4*z^14 - x1^31*x2^27*x3^8*x4^4*z^14 + 4*x1^36*x2^21*x3^9*x4^4*z^14 + x1^34*x2^23*x3^9*x4^4*z^14 - 2*x1^33*x2^24*x3^9*x4^4*z^14 - 2*x1^32*x2^25*x3^9*x4^4*z^14 - 3*x1^31*x2^26*x3^9*x4^4*z^14 + 2*x1^30*x2^27*x3^9*x4^4*z^14 - 4*x1^36*x2^20*x3^10*x4^4*z^14 - 3*x1^35*x2^21*x3^10*x4^4*z^14 - x1^34*x2^22*x3^10*x4^4*z^14 + 2*x1^33*x2^23*x3^10*x4^4*z^14 + x1^32*x2^24*x3^10*x4^4*z^14 + 2*x1^31*x2^25*x3^10*x4^4*z^14 + 2*x1^30*x2^26*x3^10*x4^4*z^14 - x1^29*x2^27*x3^10*x4^4*z^14 + x1^36*x2^19*x3^11*x4^4*z^14 + 5*x1^35*x2^20*x3^11*x4^4*z^14 + x1^34*x2^21*x3^11*x4^4*z^14 + 3*x1^33*x2^22*x3^11*x4^4*z^14 - 2*x1^32*x2^23*x3^11*x4^4*z^14 - 6*x1^30*x2^25*x3^11*x4^4*z^14 - x1^29*x2^26*x3^11*x4^4*z^14 + x1^28*x2^27*x3^11*x4^4*z^14 - 6*x1^35*x2^19*x3^12*x4^4*z^14 - 3*x1^34*x2^20*x3^12*x4^4*z^14 - 3*x1^33*x2^21*x3^12*x4^4*z^14 + x1^32*x2^22*x3^12*x4^4*z^14 + x1^31*x2^23*x3^12*x4^4*z^14 + 2*x1^30*x2^24*x3^12*x4^4*z^14 + 6*x1^29*x2^25*x3^12*x4^4*z^14 - x1^27*x2^27*x3^12*x4^4*z^14 + x1^35*x2^18*x3^13*x4^4*z^14 + 6*x1^34*x2^19*x3^13*x4^4*z^14 + 4*x1^32*x2^21*x3^13*x4^4*z^14 - 4*x1^31*x2^22*x3^13*x4^4*z^14 - 6*x1^29*x2^24*x3^13*x4^4*z^14 - 2*x1^28*x2^25*x3^13*x4^4*z^14 - 4*x1^34*x2^18*x3^14*x4^4*z^14 - 2*x1^33*x2^19*x3^14*x4^4*z^14 - x1^32*x2^20*x3^14*x4^4*z^14 + 2*x1^30*x2^22*x3^14*x4^4*z^14 + 2*x1^29*x2^23*x3^14*x4^4*z^14 + 6*x1^28*x2^24*x3^14*x4^4*z^14 + 4*x1^33*x2^18*x3^15*x4^4*z^14 + 4*x1^31*x2^20*x3^15*x4^4*z^14 - 3*x1^30*x2^21*x3^15*x4^4*z^14 - 6*x1^28*x2^23*x3^15*x4^4*z^14 - 2*x1^27*x2^24*x3^15*x4^4*z^14 - x1^32*x2^18*x3^16*x4^4*z^14 + 2*x1^29*x2^21*x3^16*x4^4*z^14 + 3*x1^28*x2^22*x3^16*x4^4*z^14 + 6*x1^27*x2^23*x3^16*x4^4*z^14 + x1^31*x2^18*x3^17*x4^4*z^14 + x1^30*x2^19*x3^17*x4^4*z^14 - 2*x1^29*x2^20*x3^17*x4^4*z^14 + x1^28*x2^21*x3^17*x4^4*z^14 - 5*x1^27*x2^22*x3^17*x4^4*z^14 - 2*x1^26*x2^23*x3^17*x4^4*z^14 - x1^30*x2^18*x3^18*x4^4*z^14 - x1^28*x2^20*x3^18*x4^4*z^14 + 5*x1^26*x2^22*x3^18*x4^4*z^14 + x1^25*x2^23*x3^18*x4^4*z^14 - x1^28*x2^19*x3^19*x4^4*z^14 + x1^27*x2^20*x3^19*x4^4*z^14 - 2*x1^26*x2^21*x3^19*x4^4*z^14 - x1^25*x2^22*x3^19*x4^4*z^14 - 2*x1^26*x2^20*x3^20*x4^4*z^14 + 2*x1^25*x2^21*x3^20*x4^4*z^14 + x1^24*x2^22*x3^20*x4^4*z^14 - 2*x1^23*x2^23*x3^20*x4^4*z^14 - x1^24*x2^21*x3^21*x4^4*z^14 + x1^23*x2^22*x3^21*x4^4*z^14 + x1^34*x2^25*x3^6*x4^5*z^14 - x1^34*x2^24*x3^7*x4^5*z^14 - 2*x1^33*x2^25*x3^7*x4^5*z^14 - x1^32*x2^26*x3^7*x4^5*z^14 - x1^31*x2^27*x3^7*x4^5*z^14 + x1^35*x2^22*x3^8*x4^5*z^14 + 4*x1^33*x2^24*x3^8*x4^5*z^14 + 2*x1^32*x2^25*x3^8*x4^5*z^14 + x1^36*x2^20*x3^9*x4^5*z^14 - 3*x1^33*x2^23*x3^9*x4^5*z^14 + x1^31*x2^25*x3^9*x4^5*z^14 - 2*x1^30*x2^26*x3^9*x4^5*z^14 - 3*x1^35*x2^20*x3^10*x4^5*z^14 + 3*x1^32*x2^23*x3^10*x4^5*z^14 + 4*x1^30*x2^25*x3^10*x4^5*z^14 + 2*x1^29*x2^26*x3^10*x4^5*z^14 + x1^28*x2^27*x3^10*x4^5*z^14 + 2*x1^35*x2^19*x3^11*x4^5*z^14 + 2*x1^34*x2^20*x3^11*x4^5*z^14 - x1^33*x2^21*x3^11*x4^5*z^14 - 3*x1^32*x2^22*x3^11*x4^5*z^14 - 3*x1^31*x2^23*x3^11*x4^5*z^14 - x1^30*x2^24*x3^11*x4^5*z^14 - 7*x1^29*x2^25*x3^11*x4^5*z^14 + x1^28*x2^26*x3^11*x4^5*z^14 - 5*x1^34*x2^19*x3^12*x4^5*z^14 - x1^32*x2^21*x3^12*x4^5*z^14 + 4*x1^31*x2^22*x3^12*x4^5*z^14 + 5*x1^29*x2^24*x3^12*x4^5*z^14 + x1^28*x2^25*x3^12*x4^5*z^14 - 2*x1^27*x2^26*x3^12*x4^5*z^14 + 3*x1^34*x2^18*x3^13*x4^5*z^14 + 3*x1^33*x2^19*x3^13*x4^5*z^14 - x1^32*x2^20*x3^13*x4^5*z^14 - 2*x1^31*x2^21*x3^13*x4^5*z^14 - x1^30*x2^22*x3^13*x4^5*z^14 - x1^29*x2^23*x3^13*x4^5*z^14 - 6*x1^28*x2^24*x3^13*x4^5*z^14 + 2*x1^27*x2^25*x3^13*x4^5*z^14 + x1^26*x2^26*x3^13*x4^5*z^14 - 4*x1^33*x2^18*x3^14*x4^5*z^14 - 4*x1^31*x2^20*x3^14*x4^5*z^14 + 4*x1^30*x2^21*x3^14*x4^5*z^14 + 5*x1^28*x2^23*x3^14*x4^5*z^14 + x1^27*x2^24*x3^14*x4^5*z^14 - 2*x1^26*x2^25*x3^14*x4^5*z^14 + x1^33*x2^17*x3^15*x4^5*z^14 + 2*x1^32*x2^18*x3^15*x4^5*z^14 - 2*x1^29*x2^21*x3^15*x4^5*z^14 - 6*x1^27*x2^23*x3^15*x4^5*z^14 + 2*x1^26*x2^24*x3^15*x4^5*z^14 - x1^32*x2^17*x3^16*x4^5*z^14 - x1^31*x2^18*x3^16*x4^5*z^14 - x1^30*x2^19*x3^16*x4^5*z^14 + 3*x1^29*x2^20*x3^16*x4^5*z^14 - 2*x1^28*x2^21*x3^16*x4^5*z^14 + 5*x1^27*x2^22*x3^16*x4^5*z^14 + 2*x1^26*x2^23*x3^16*x4^5*z^14 - 2*x1^25*x2^24*x3^16*x4^5*z^14 + x1^30*x2^18*x3^17*x4^5*z^14 - x1^29*x2^19*x3^17*x4^5*z^14 - 3*x1^28*x2^20*x3^17*x4^5*z^14 - 5*x1^26*x2^22*x3^17*x4^5*z^14 + 2*x1^25*x2^23*x3^17*x4^5*z^14 + x1^24*x2^24*x3^17*x4^5*z^14 - x1^29*x2^18*x3^18*x4^5*z^14 + 2*x1^28*x2^19*x3^18*x4^5*z^14 + 2*x1^26*x2^21*x3^18*x4^5*z^14 - 2*x1^24*x2^23*x3^18*x4^5*z^14 - x1^27*x2^19*x3^19*x4^5*z^14 + x1^26*x2^20*x3^19*x4^5*z^14 - 3*x1^25*x2^21*x3^19*x4^5*z^14 + 2*x1^24*x2^22*x3^19*x4^5*z^14 + x1^23*x2^23*x3^19*x4^5*z^14 + x1^24*x2^21*x3^20*x4^5*z^14 - x1^23*x2^22*x3^20*x4^5*z^14 - x1^23*x2^21*x3^21*x4^5*z^14 + x1^22*x2^22*x3^21*x4^5*z^14 + x1^33*x2^24*x3^7*x4^6*z^14 - x1^32*x2^25*x3^7*x4^6*z^14 + 2*x1^32*x2^24*x3^8*x4^6*z^14 + x1^31*x2^25*x3^8*x4^6*z^14 - x1^35*x2^20*x3^9*x4^6*z^14 - 2*x1^34*x2^21*x3^9*x4^6*z^14 - x1^31*x2^24*x3^9*x4^6*z^14 - x1^29*x2^26*x3^9*x4^6*z^14 - 2*x1^28*x2^27*x3^9*x4^6*z^14 + x1^35*x2^19*x3^10*x4^6*z^14 + x1^32*x2^22*x3^10*x4^6*z^14 + x1^31*x2^23*x3^10*x4^6*z^14 - x1^30*x2^24*x3^10*x4^6*z^14 - 3*x1^28*x2^26*x3^10*x4^6*z^14 + x1^27*x2^27*x3^10*x4^6*z^14 + x1^34*x2^19*x3^11*x4^6*z^14 - 3*x1^32*x2^21*x3^11*x4^6*z^14 - 2*x1^31*x2^22*x3^11*x4^6*z^14 + 2*x1^30*x2^23*x3^11*x4^6*z^14 + 2*x1^29*x2^24*x3^11*x4^6*z^14 + 2*x1^27*x2^26*x3^11*x4^6*z^14 + x1^34*x2^18*x3^12*x4^6*z^14 + 3*x1^32*x2^20*x3^12*x4^6*z^14 + 2*x1^31*x2^21*x3^12*x4^6*z^14 + x1^30*x2^22*x3^12*x4^6*z^14 - x1^29*x2^23*x3^12*x4^6*z^14 + 3*x1^28*x2^24*x3^12*x4^6*z^14 - 3*x1^27*x2^25*x3^12*x4^6*z^14 - 2*x1^26*x2^26*x3^12*x4^6*z^14 + x1^33*x2^18*x3^13*x4^6*z^14 - x1^32*x2^19*x3^13*x4^6*z^14 - x1^31*x2^20*x3^13*x4^6*z^14 - 4*x1^30*x2^21*x3^13*x4^6*z^14 + 2*x1^27*x2^24*x3^13*x4^6*z^14 + 4*x1^26*x2^25*x3^13*x4^6*z^14 + x1^33*x2^17*x3^14*x4^6*z^14 + x1^32*x2^18*x3^14*x4^6*z^14 + 4*x1^31*x2^19*x3^14*x4^6*z^14 + 2*x1^30*x2^20*x3^14*x4^6*z^14 + x1^29*x2^21*x3^14*x4^6*z^14 - 3*x1^28*x2^22*x3^14*x4^6*z^14 + 2*x1^27*x2^23*x3^14*x4^6*z^14 - 4*x1^26*x2^24*x3^14*x4^6*z^14 - x1^25*x2^25*x3^14*x4^6*z^14 - x1^32*x2^17*x3^15*x4^6*z^14 - 3*x1^31*x2^18*x3^15*x4^6*z^14 + 2*x1^30*x2^19*x3^15*x4^6*z^14 - x1^29*x2^20*x3^15*x4^6*z^14 - x1^27*x2^22*x3^15*x4^6*z^14 + 4*x1^25*x2^24*x3^15*x4^6*z^14 + 3*x1^30*x2^18*x3^16*x4^6*z^14 + x1^29*x2^19*x3^16*x4^6*z^14 + 4*x1^28*x2^20*x3^16*x4^6*z^14 - x1^27*x2^21*x3^16*x4^6*z^14 + 2*x1^26*x2^22*x3^16*x4^6*z^14 - 4*x1^25*x2^23*x3^16*x4^6*z^14 - x1^24*x2^24*x3^16*x4^6*z^14 - 3*x1^29*x2^18*x3^17*x4^6*z^14 - 2*x1^28*x2^19*x3^17*x4^6*z^14 + x1^27*x2^20*x3^17*x4^6*z^14 + 4*x1^24*x2^23*x3^17*x4^6*z^14 - 4*x1^26*x2^20*x3^18*x4^6*z^14 + x1^25*x2^21*x3^18*x4^6*z^14 - 3*x1^24*x2^22*x3^18*x4^6*z^14 - 2*x1^23*x2^23*x3^18*x4^6*z^14 - x1^26*x2^19*x3^19*x4^6*z^14 + 3*x1^23*x2^22*x3^19*x4^6*z^14 + x1^24*x2^20*x3^20*x4^6*z^14 - x1^23*x2^21*x3^20*x4^6*z^14 - x1^22*x2^22*x3^20*x4^6*z^14 + x1^22*x2^21*x3^21*x4^6*z^14 + x1^32*x2^24*x3^7*x4^7*z^14 + x1^30*x2^26*x3^7*x4^7*z^14 - 2*x1^32*x2^23*x3^8*x4^7*z^14 + 2*x1^31*x2^24*x3^8*x4^7*z^14 - x1^29*x2^26*x3^8*x4^7*z^14 + x1^28*x2^27*x3^8*x4^7*z^14 + x1^31*x2^23*x3^9*x4^7*z^14 + 2*x1^29*x2^25*x3^9*x4^7*z^14 + x1^28*x2^26*x3^9*x4^7*z^14 - x1^27*x2^27*x3^9*x4^7*z^14 + x1^34*x2^19*x3^10*x4^7*z^14 + x1^33*x2^20*x3^10*x4^7*z^14 - x1^31*x2^22*x3^10*x4^7*z^14 + x1^30*x2^23*x3^10*x4^7*z^14 - x1^29*x2^24*x3^10*x4^7*z^14 - 2*x1^34*x2^18*x3^11*x4^7*z^14 + x1^32*x2^20*x3^11*x4^7*z^14 - x1^31*x2^21*x3^11*x4^7*z^14 + x1^30*x2^22*x3^11*x4^7*z^14 + x1^28*x2^24*x3^11*x4^7*z^14 + x1^27*x2^25*x3^11*x4^7*z^14 - x1^26*x2^26*x3^11*x4^7*z^14 + 2*x1^33*x2^18*x3^12*x4^7*z^14 + 2*x1^30*x2^21*x3^12*x4^7*z^14 - 2*x1^28*x2^23*x3^12*x4^7*z^14 + x1^27*x2^24*x3^12*x4^7*z^14 - x1^32*x2^18*x3^13*x4^7*z^14 - 3*x1^31*x2^19*x3^13*x4^7*z^14 - x1^29*x2^21*x3^13*x4^7*z^14 + 2*x1^31*x2^18*x3^14*x4^7*z^14 + x1^29*x2^20*x3^14*x4^7*z^14 + x1^31*x2^17*x3^15*x4^7*z^14 - 2*x1^30*x2^18*x3^15*x4^7*z^14 - x1^29*x2^19*x3^15*x4^7*z^14 - 2*x1^28*x2^20*x3^15*x4^7*z^14 + x1^27*x2^21*x3^15*x4^7*z^14 - x1^30*x2^17*x3^16*x4^7*z^14 + x1^29*x2^18*x3^16*x4^7*z^14 - x1^28*x2^19*x3^16*x4^7*z^14 - x1^26*x2^21*x3^16*x4^7*z^14 - x1^25*x2^22*x3^16*x4^7*z^14 + x1^29*x2^17*x3^17*x4^7*z^14 - x1^27*x2^19*x3^17*x4^7*z^14 + x1^26*x2^20*x3^17*x4^7*z^14 + x1^25*x2^21*x3^17*x4^7*z^14 + 2*x1^24*x2^22*x3^17*x4^7*z^14 + x1^27*x2^18*x3^18*x4^7*z^14 - 2*x1^24*x2^21*x3^18*x4^7*z^14 - 2*x1^23*x2^22*x3^18*x4^7*z^14 + 2*x1^25*x2^19*x3^19*x4^7*z^14 - x1^24*x2^20*x3^19*x4^7*z^14 + 2*x1^23*x2^21*x3^19*x4^7*z^14 + x1^22*x2^22*x3^19*x4^7*z^14 + x1^23*x2^20*x3^20*x4^7*z^14 - 2*x1^22*x2^21*x3^20*x4^7*z^14 + x1^21*x2^21*x3^21*x4^7*z^14 + x1^32*x2^22*x3^8*x4^8*z^14 + x1^30*x2^24*x3^8*x4^8*z^14 + x1^29*x2^25*x3^8*x4^8*z^14 - x1^30*x2^23*x3^9*x4^8*z^14 - x1^29*x2^24*x3^9*x4^8*z^14 + x1^28*x2^25*x3^9*x4^8*z^14 + x1^31*x2^21*x3^10*x4^8*z^14 - x1^30*x2^22*x3^10*x4^8*z^14 + x1^29*x2^23*x3^10*x4^8*z^14 - x1^28*x2^24*x3^10*x4^8*z^14 + 2*x1^26*x2^26*x3^10*x4^8*z^14 + x1^33*x2^18*x3^11*x4^8*z^14 + x1^31*x2^20*x3^11*x4^8*z^14 - 3*x1^28*x2^23*x3^11*x4^8*z^14 - 2*x1^27*x2^24*x3^11*x4^8*z^14 - x1^26*x2^25*x3^11*x4^8*z^14 - x1^32*x2^18*x3^12*x4^8*z^14 - x1^30*x2^20*x3^12*x4^8*z^14 + 2*x1^29*x2^21*x3^12*x4^8*z^14 + 4*x1^28*x2^22*x3^12*x4^8*z^14 + x1^27*x2^23*x3^12*x4^8*z^14 + x1^26*x2^24*x3^12*x4^8*z^14 + x1^25*x2^25*x3^12*x4^8*z^14 - 3*x1^29*x2^20*x3^13*x4^8*z^14 - 2*x1^28*x2^21*x3^13*x4^8*z^14 + x1^27*x2^22*x3^13*x4^8*z^14 - 2*x1^26*x2^23*x3^13*x4^8*z^14 - 3*x1^25*x2^24*x3^13*x4^8*z^14 - x1^28*x2^20*x3^14*x4^8*z^14 + x1^27*x2^21*x3^14*x4^8*z^14 - x1^26*x2^22*x3^14*x4^8*z^14 + 3*x1^25*x2^23*x3^14*x4^8*z^14 + x1^24*x2^24*x3^14*x4^8*z^14 - x1^30*x2^17*x3^15*x4^8*z^14 + x1^29*x2^18*x3^15*x4^8*z^14 - 3*x1^28*x2^19*x3^15*x4^8*z^14 - 3*x1^26*x2^21*x3^15*x4^8*z^14 + x1^25*x2^22*x3^15*x4^8*z^14 - 4*x1^24*x2^23*x3^15*x4^8*z^14 + x1^27*x2^19*x3^16*x4^8*z^14 + 2*x1^25*x2^21*x3^16*x4^8*z^14 + 2*x1^23*x2^23*x3^16*x4^8*z^14 - x1^27*x2^18*x3^17*x4^8*z^14 + x1^26*x2^19*x3^17*x4^8*z^14 - 3*x1^25*x2^20*x3^17*x4^8*z^14 - 2*x1^24*x2^21*x3^17*x4^8*z^14 + x1^26*x2^18*x3^18*x4^8*z^14 + 2*x1^24*x2^20*x3^18*x4^8*z^14 - x1^22*x2^22*x3^18*x4^8*z^14 - x1^31*x2^21*x3^9*x4^9*z^14 - x1^29*x2^23*x3^9*x4^9*z^14 - x1^28*x2^24*x3^9*x4^9*z^14 - x1^27*x2^25*x3^9*x4^9*z^14 - x1^26*x2^26*x3^9*x4^9*z^14 + 2*x1^30*x2^21*x3^10*x4^9*z^14 + 2*x1^28*x2^23*x3^10*x4^9*z^14 + x1^27*x2^24*x3^10*x4^9*z^14 - x1^30*x2^20*x3^11*x4^9*z^14 - x1^29*x2^21*x3^11*x4^9*z^14 - 2*x1^28*x2^22*x3^11*x4^9*z^14 - 2*x1^27*x2^23*x3^11*x4^9*z^14 - x1^26*x2^24*x3^11*x4^9*z^14 - x1^25*x2^25*x3^11*x4^9*z^14 - x1^31*x2^18*x3^12*x4^9*z^14 - x1^30*x2^19*x3^12*x4^9*z^14 + 2*x1^29*x2^20*x3^12*x4^9*z^14 - x1^28*x2^21*x3^12*x4^9*z^14 + 2*x1^27*x2^22*x3^12*x4^9*z^14 + 2*x1^26*x2^23*x3^12*x4^9*z^14 - 3*x1^28*x2^20*x3^13*x4^9*z^14 - 3*x1^27*x2^21*x3^13*x4^9*z^14 - 3*x1^26*x2^22*x3^13*x4^9*z^14 - 3*x1^25*x2^23*x3^13*x4^9*z^14 - x1^31*x2^16*x3^14*x4^9*z^14 - x1^30*x2^17*x3^14*x4^9*z^14 - x1^29*x2^18*x3^14*x4^9*z^14 + 3*x1^28*x2^19*x3^14*x4^9*z^14 + x1^27*x2^20*x3^14*x4^9*z^14 + 4*x1^26*x2^21*x3^14*x4^9*z^14 + x1^24*x2^23*x3^14*x4^9*z^14 + x1^30*x2^16*x3^15*x4^9*z^14 - x1^28*x2^18*x3^15*x4^9*z^14 - x1^27*x2^19*x3^15*x4^9*z^14 - x1^26*x2^20*x3^15*x4^9*z^14 - 3*x1^25*x2^21*x3^15*x4^9*z^14 - x1^23*x2^23*x3^15*x4^9*z^14 + 2*x1^27*x2^18*x3^16*x4^9*z^14 + 5*x1^25*x2^20*x3^16*x4^9*z^14 + 3*x1^24*x2^21*x3^16*x4^9*z^14 + x1^23*x2^22*x3^16*x4^9*z^14 - x1^26*x2^18*x3^17*x4^9*z^14 - x1^25*x2^19*x3^17*x4^9*z^14 - 4*x1^24*x2^20*x3^17*x4^9*z^14 - x1^23*x2^21*x3^17*x4^9*z^14 - x1^25*x2^18*x3^18*x4^9*z^14 + 2*x1^24*x2^19*x3^18*x4^9*z^14 + x1^23*x2^20*x3^18*x4^9*z^14 + x1^22*x2^21*x3^18*x4^9*z^14 - 2*x1^23*x2^19*x3^19*x4^9*z^14 - x1^29*x2^20*x3^11*x4^10*z^14 - x1^25*x2^24*x3^11*x4^10*z^14 + 2*x1^27*x2^21*x3^12*x4^10*z^14 + 2*x1^25*x2^23*x3^12*x4^10*z^14 + x1^30*x2^17*x3^13*x4^10*z^14 + x1^29*x2^18*x3^13*x4^10*z^14 - x1^28*x2^19*x3^13*x4^10*z^14 + x1^25*x2^22*x3^13*x4^10*z^14 - x1^24*x2^23*x3^13*x4^10*z^14 + x1^29*x2^17*x3^14*x4^10*z^14 - x1^27*x2^19*x3^14*x4^10*z^14 - x1^26*x2^20*x3^14*x4^10*z^14 - x1^25*x2^21*x3^14*x4^10*z^14 + 2*x1^24*x2^22*x3^14*x4^10*z^14 + x1^26*x2^19*x3^15*x4^10*z^14 - x1^25*x2^20*x3^15*x4^10*z^14 + x1^24*x2^21*x3^15*x4^10*z^14 - x1^27*x2^17*x3^16*x4^10*z^14 - x1^25*x2^19*x3^16*x4^10*z^14 + 2*x1^25*x2^18*x3^17*x4^10*z^14 - x1^24*x2^19*x3^17*x4^10*z^14 - x1^23*x2^20*x3^17*x4^10*z^14 + x1^22*x2^21*x3^17*x4^10*z^14 - x1^24*x2^18*x3^18*x4^10*z^14 + x1^23*x2^19*x3^18*x4^10*z^14 - x1^21*x2^21*x3^18*x4^10*z^14 + x1^29*x2^19*x3^11*x4^11*z^14 + 2*x1^26*x2^22*x3^11*x4^11*z^14 - x1^27*x2^20*x3^12*x4^11*z^14 + x1^25*x2^22*x3^12*x4^11*z^14 + 2*x1^24*x2^23*x3^12*x4^11*z^14 - x1^27*x2^19*x3^13*x4^11*z^14 - 3*x1^26*x2^20*x3^13*x4^11*z^14 + x1^25*x2^21*x3^13*x4^11*z^14 - x1^24*x2^22*x3^13*x4^11*z^14 + x1^26*x2^19*x3^14*x4^11*z^14 - x1^25*x2^20*x3^14*x4^11*z^14 - x1^24*x2^21*x3^14*x4^11*z^14 + 3*x1^23*x2^22*x3^14*x4^11*z^14 + 2*x1^26*x2^18*x3^15*x4^11*z^14 - x1^23*x2^21*x3^15*x4^11*z^14 + x1^24*x2^19*x3^16*x4^11*z^14 + 2*x1^22*x2^21*x3^16*x4^11*z^14 - x1^21*x2^21*x3^17*x4^11*z^14 - 2*x1^25*x2^21*x3^12*x4^12*z^14 + x1^27*x2^18*x3^13*x4^12*z^14 - 2*x1^23*x2^22*x3^13*x4^12*z^14 + x1^26*x2^18*x3^14*x4^12*z^14 + 3*x1^25*x2^19*x3^14*x4^12*z^14 - x1^24*x2^20*x3^14*x4^12*z^14 + x1^23*x2^21*x3^14*x4^12*z^14 + x1^22*x2^22*x3^14*x4^12*z^14 - x1^25*x2^18*x3^15*x4^12*z^14 - 2*x1^24*x2^19*x3^15*x4^12*z^14 - 3*x1^22*x2^21*x3^15*x4^12*z^14 - x1^23*x2^19*x3^16*x4^12*z^14 + 2*x1^21*x2^21*x3^16*x4^12*z^14 - x1^21*x2^20*x3^17*x4^12*z^14 + x1^25*x2^18*x3^14*x4^13*z^14 + x1^24*x2^19*x3^14*x4^13*z^14 - x1^23*x2^20*x3^14*x4^13*z^14 - x1^21*x2^21*x3^15*x4^13*z^14 - x1^20*x2^20*x3^17*x4^13*z^14 + x1^22*x2^19*x3^15*x4^14*z^14 - x1^36*x2^23*x3^6*z^13 + x1^36*x2^22*x3^7*z^13 + x1^35*x2^23*x3^7*z^13 + x1^33*x2^25*x3^7*z^13 - 2*x1^35*x2^22*x3^8*z^13 - x1^32*x2^25*x3^8*z^13 + 2*x1^34*x2^22*x3^9*z^13 + x1^32*x2^24*x3^9*z^13 - 2*x1^34*x2^21*x3^10*z^13 - x1^33*x2^22*x3^10*z^13 - x1^32*x2^23*x3^10*z^13 + x1^34*x2^20*x3^11*z^13 + 2*x1^33*x2^21*x3^11*z^13 + x1^31*x2^23*x3^11*z^13 - x1^30*x2^24*x3^11*z^13 - 2*x1^28*x2^26*x3^11*z^13 - 2*x1^33*x2^20*x3^12*z^13 - x1^32*x2^21*x3^12*z^13 - x1^31*x2^22*x3^12*z^13 + x1^29*x2^24*x3^12*z^13 + x1^28*x2^25*x3^12*z^13 + 2*x1^27*x2^26*x3^12*z^13 + x1^33*x2^19*x3^13*z^13 + 2*x1^32*x2^20*x3^13*z^13 + x1^30*x2^22*x3^13*z^13 - 2*x1^29*x2^23*x3^13*z^13 - 2*x1^27*x2^25*x3^13*z^13 - 2*x1^32*x2^19*x3^14*z^13 + 2*x1^26*x2^25*x3^14*z^13 + 2*x1^31*x2^19*x3^15*z^13 + 2*x1^29*x2^21*x3^15*z^13 - x1^27*x2^23*x3^15*z^13 - x1^26*x2^24*x3^15*z^13 - x1^31*x2^18*x3^16*z^13 - 2*x1^30*x2^19*x3^16*z^13 - x1^29*x2^20*x3^16*z^13 + x1^25*x2^24*x3^16*z^13 + x1^30*x2^18*x3^17*z^13 + 2*x1^29*x2^19*x3^17*z^13 + x1^28*x2^20*x3^17*z^13 + x1^27*x2^21*x3^17*z^13 - 2*x1^25*x2^23*x3^17*z^13 - x1^28*x2^19*x3^18*z^13 - x1^27*x2^20*x3^18*z^13 + x1^36*x2^23*x3^5*x4*z^13 - x1^34*x2^25*x3^5*x4*z^13 - 2*x1^35*x2^23*x3^6*x4*z^13 + 2*x1^34*x2^24*x3^6*x4*z^13 + 3*x1^35*x2^22*x3^7*x4*z^13 + 2*x1^34*x2^23*x3^7*x4*z^13 - x1^31*x2^26*x3^7*x4*z^13 - 2*x1^35*x2^21*x3^8*x4*z^13 - 5*x1^34*x2^22*x3^8*x4*z^13 + x1^33*x2^23*x3^8*x4*z^13 - 2*x1^32*x2^24*x3^8*x4*z^13 + 2*x1^31*x2^25*x3^8*x4*z^13 + x1^30*x2^26*x3^8*x4*z^13 + x1^29*x2^27*x3^8*x4*z^13 + 6*x1^34*x2^21*x3^9*x4*z^13 + 2*x1^33*x2^22*x3^9*x4*z^13 + 2*x1^32*x2^23*x3^9*x4*z^13 - 3*x1^30*x2^25*x3^9*x4*z^13 - 2*x1^29*x2^26*x3^9*x4*z^13 - x1^28*x2^27*x3^9*x4*z^13 - 2*x1^34*x2^20*x3^10*x4*z^13 - 6*x1^33*x2^21*x3^10*x4*z^13 - 4*x1^31*x2^23*x3^10*x4*z^13 + 4*x1^30*x2^24*x3^10*x4*z^13 + 2*x1^29*x2^25*x3^10*x4*z^13 + 5*x1^28*x2^26*x3^10*x4*z^13 + 6*x1^33*x2^20*x3^11*x4*z^13 + 2*x1^32*x2^21*x3^11*x4*z^13 + 2*x1^31*x2^22*x3^11*x4*z^13 - 2*x1^29*x2^24*x3^11*x4*z^13 - 2*x1^28*x2^25*x3^11*x4*z^13 - 5*x1^27*x2^26*x3^11*x4*z^13 - 2*x1^33*x2^19*x3^12*x4*z^13 - 6*x1^32*x2^20*x3^12*x4*z^13 - 4*x1^30*x2^22*x3^12*x4*z^13 + 4*x1^29*x2^23*x3^12*x4*z^13 + 6*x1^27*x2^25*x3^12*x4*z^13 + 2*x1^26*x2^26*x3^12*x4*z^13 + 5*x1^32*x2^19*x3^13*x4*z^13 + 2*x1^31*x2^20*x3^13*x4*z^13 + 2*x1^30*x2^21*x3^13*x4*z^13 - 2*x1^28*x2^23*x3^13*x4*z^13 - 2*x1^27*x2^24*x3^13*x4*z^13 - 6*x1^26*x2^25*x3^13*x4*z^13 - x1^32*x2^18*x3^14*x4*z^13 - 5*x1^31*x2^19*x3^14*x4*z^13 - x1^30*x2^20*x3^14*x4*z^13 - 4*x1^29*x2^21*x3^14*x4*z^13 + 4*x1^28*x2^22*x3^14*x4*z^13 + 6*x1^26*x2^24*x3^14*x4*z^13 + 2*x1^25*x2^25*x3^14*x4*z^13 + 2*x1^31*x2^18*x3^15*x4*z^13 + x1^30*x2^19*x3^15*x4*z^13 + 2*x1^29*x2^20*x3^15*x4*z^13 - 2*x1^27*x2^22*x3^15*x4*z^13 - 2*x1^26*x2^23*x3^15*x4*z^13 - 6*x1^25*x2^24*x3^15*x4*z^13 - 2*x1^30*x2^18*x3^16*x4*z^13 - x1^29*x2^19*x3^16*x4*z^13 - 3*x1^28*x2^20*x3^16*x4*z^13 + 2*x1^27*x2^21*x3^16*x4*z^13 + x1^26*x2^22*x3^16*x4*z^13 + 5*x1^25*x2^23*x3^16*x4*z^13 + x1^24*x2^24*x3^16*x4*z^13 + x1^29*x2^18*x3^17*x4*z^13 + x1^28*x2^19*x3^17*x4*z^13 - x1^25*x2^22*x3^17*x4*z^13 - 4*x1^24*x2^23*x3^17*x4*z^13 - x1^25*x2^21*x3^18*x4*z^13 + 3*x1^24*x2^22*x3^18*x4*z^13 + x1^25*x2^20*x3^19*x4*z^13 - 2*x1^23*x2^22*x3^19*x4*z^13 - x1^35*x2^23*x3^5*x4^2*z^13 + x1^34*x2^24*x3^5*x4^2*z^13 + x1^33*x2^25*x3^5*x4^2*z^13 - x1^32*x2^26*x3^5*x4^2*z^13 - x1^35*x2^22*x3^6*x4^2*z^13 - x1^34*x2^23*x3^6*x4^2*z^13 + x1^33*x2^24*x3^6*x4^2*z^13 + x1^31*x2^26*x3^6*x4^2*z^13 + 2*x1^34*x2^22*x3^7*x4^2*z^13 + x1^32*x2^24*x3^7*x4^2*z^13 - 2*x1^31*x2^25*x3^7*x4^2*z^13 - 2*x1^30*x2^26*x3^7*x4^2*z^13 - 2*x1^29*x2^27*x3^7*x4^2*z^13 - 2*x1^34*x2^21*x3^8*x4^2*z^13 - 2*x1^33*x2^22*x3^8*x4^2*z^13 + 2*x1^30*x2^25*x3^8*x4^2*z^13 + 3*x1^29*x2^26*x3^8*x4^2*z^13 + 2*x1^28*x2^27*x3^8*x4^2*z^13 + x1^34*x2^20*x3^9*x4^2*z^13 + 4*x1^33*x2^21*x3^9*x4^2*z^13 + 2*x1^31*x2^23*x3^9*x4^2*z^13 - 3*x1^30*x2^24*x3^9*x4^2*z^13 - 5*x1^28*x2^26*x3^9*x4^2*z^13 - 5*x1^33*x2^20*x3^10*x4^2*z^13 - 3*x1^32*x2^21*x3^10*x4^2*z^13 - x1^31*x2^22*x3^10*x4^2*z^13 + x1^30*x2^23*x3^10*x4^2*z^13 + 2*x1^29*x2^24*x3^10*x4^2*z^13 + 2*x1^28*x2^25*x3^10*x4^2*z^13 + 5*x1^27*x2^26*x3^10*x4^2*z^13 + 2*x1^33*x2^19*x3^11*x4^2*z^13 + 6*x1^32*x2^20*x3^11*x4^2*z^13 + 4*x1^30*x2^22*x3^11*x4^2*z^13 - 4*x1^29*x2^23*x3^11*x4^2*z^13 - 6*x1^27*x2^25*x3^11*x4^2*z^13 - x1^26*x2^26*x3^11*x4^2*z^13 - 6*x1^32*x2^19*x3^12*x4^2*z^13 - 2*x1^31*x2^20*x3^12*x4^2*z^13 - 2*x1^30*x2^21*x3^12*x4^2*z^13 + 2*x1^28*x2^23*x3^12*x4^2*z^13 + 2*x1^27*x2^24*x3^12*x4^2*z^13 + 6*x1^26*x2^25*x3^12*x4^2*z^13 + x1^32*x2^18*x3^13*x4^2*z^13 + 6*x1^31*x2^19*x3^13*x4^2*z^13 + 4*x1^29*x2^21*x3^13*x4^2*z^13 - 4*x1^28*x2^22*x3^13*x4^2*z^13 - 6*x1^26*x2^24*x3^13*x4^2*z^13 - 2*x1^25*x2^25*x3^13*x4^2*z^13 - 2*x1^31*x2^18*x3^14*x4^2*z^13 - x1^30*x2^19*x3^14*x4^2*z^13 + 2*x1^27*x2^22*x3^14*x4^2*z^13 + 2*x1^26*x2^23*x3^14*x4^2*z^13 + 6*x1^25*x2^24*x3^14*x4^2*z^13 + 2*x1^30*x2^18*x3^15*x4^2*z^13 + x1^29*x2^19*x3^15*x4^2*z^13 + 2*x1^28*x2^20*x3^15*x4^2*z^13 - 3*x1^27*x2^21*x3^15*x4^2*z^13 - 6*x1^25*x2^23*x3^15*x4^2*z^13 - 2*x1^24*x2^24*x3^15*x4^2*z^13 - x1^29*x2^18*x3^16*x4^2*z^13 - x1^28*x2^19*x3^16*x4^2*z^13 + x1^27*x2^20*x3^16*x4^2*z^13 + 2*x1^26*x2^21*x3^16*x4^2*z^13 + 2*x1^25*x2^22*x3^16*x4^2*z^13 + 6*x1^24*x2^23*x3^16*x4^2*z^13 + x1^28*x2^18*x3^17*x4^2*z^13 - 2*x1^26*x2^20*x3^17*x4^2*z^13 - 5*x1^24*x2^22*x3^17*x4^2*z^13 - 2*x1^23*x2^23*x3^17*x4^2*z^13 - x1^27*x2^18*x3^18*x4^2*z^13 + x1^25*x2^20*x3^18*x4^2*z^13 + x1^24*x2^21*x3^18*x4^2*z^13 + 4*x1^23*x2^22*x3^18*x4^2*z^13 - x1^25*x2^19*x3^19*x4^2*z^13 + x1^24*x2^20*x3^19*x4^2*z^13 - x1^23*x2^21*x3^19*x4^2*z^13 - x1^22*x2^22*x3^19*x4^2*z^13 + x1^22*x2^21*x3^20*x4^2*z^13 + x1^36*x2^20*x3^6*x4^3*z^13 + x1^34*x2^22*x3^6*x4^3*z^13 + x1^31*x2^25*x3^6*x4^3*z^13 + x1^30*x2^26*x3^6*x4^3*z^13 - x1^35*x2^20*x3^7*x4^3*z^13 + x1^31*x2^24*x3^7*x4^3*z^13 + x1^35*x2^19*x3^8*x4^3*z^13 + x1^34*x2^20*x3^8*x4^3*z^13 - 2*x1^32*x2^22*x3^8*x4^3*z^13 + x1^31*x2^23*x3^8*x4^3*z^13 + 3*x1^30*x2^24*x3^8*x4^3*z^13 + x1^28*x2^26*x3^8*x4^3*z^13 - 2*x1^34*x2^19*x3^9*x4^3*z^13 - 2*x1^32*x2^21*x3^9*x4^3*z^13 - x1^31*x2^22*x3^9*x4^3*z^13 - x1^30*x2^23*x3^9*x4^3*z^13 + x1^29*x2^24*x3^9*x4^3*z^13 - x1^28*x2^25*x3^9*x4^3*z^13 - 2*x1^27*x2^26*x3^9*x4^3*z^13 + 2*x1^34*x2^18*x3^10*x4^3*z^13 + 2*x1^33*x2^19*x3^10*x4^3*z^13 - x1^31*x2^21*x3^10*x4^3*z^13 - 2*x1^30*x2^22*x3^10*x4^3*z^13 - 2*x1^28*x2^24*x3^10*x4^3*z^13 + 2*x1^27*x2^25*x3^10*x4^3*z^13 - x1^34*x2^17*x3^11*x4^3*z^13 - 2*x1^33*x2^18*x3^11*x4^3*z^13 + x1^32*x2^19*x3^11*x4^3*z^13 + x1^31*x2^20*x3^11*x4^3*z^13 + x1^30*x2^21*x3^11*x4^3*z^13 - x1^29*x2^22*x3^11*x4^3*z^13 + x1^28*x2^23*x3^11*x4^3*z^13 - 2*x1^26*x2^25*x3^11*x4^3*z^13 + 2*x1^33*x2^17*x3^12*x4^3*z^13 - x1^31*x2^19*x3^12*x4^3*z^13 - 2*x1^29*x2^21*x3^12*x4^3*z^13 - 2*x1^27*x2^23*x3^12*x4^3*z^13 + 2*x1^26*x2^24*x3^12*x4^3*z^13 + x1^25*x2^25*x3^12*x4^3*z^13 - 2*x1^32*x2^17*x3^13*x4^3*z^13 + x1^31*x2^18*x3^13*x4^3*z^13 + x1^30*x2^19*x3^13*x4^3*z^13 + 3*x1^29*x2^20*x3^13*x4^3*z^13 + x1^27*x2^22*x3^13*x4^3*z^13 - x1^26*x2^23*x3^13*x4^3*z^13 - 2*x1^25*x2^24*x3^13*x4^3*z^13 + x1^31*x2^17*x3^14*x4^3*z^13 - x1^30*x2^18*x3^14*x4^3*z^13 - 2*x1^29*x2^19*x3^14*x4^3*z^13 - x1^28*x2^20*x3^14*x4^3*z^13 + 2*x1^27*x2^21*x3^14*x4^3*z^13 - 2*x1^26*x2^22*x3^14*x4^3*z^13 + 2*x1^25*x2^23*x3^14*x4^3*z^13 - x1^30*x2^17*x3^15*x4^3*z^13 - x1^29*x2^18*x3^15*x4^3*z^13 + 2*x1^28*x2^19*x3^15*x4^3*z^13 + x1^27*x2^20*x3^15*x4^3*z^13 + 3*x1^26*x2^21*x3^15*x4^3*z^13 + x1^25*x2^22*x3^15*x4^3*z^13 - 2*x1^24*x2^23*x3^15*x4^3*z^13 + x1^29*x2^17*x3^16*x4^3*z^13 - 2*x1^27*x2^19*x3^16*x4^3*z^13 - 2*x1^26*x2^20*x3^16*x4^3*z^13 - 3*x1^25*x2^21*x3^16*x4^3*z^13 + 2*x1^24*x2^22*x3^16*x4^3*z^13 + x1^23*x2^23*x3^16*x4^3*z^13 + x1^27*x2^18*x3^17*x4^3*z^13 + 2*x1^25*x2^20*x3^17*x4^3*z^13 + x1^24*x2^21*x3^17*x4^3*z^13 - 2*x1^23*x2^22*x3^17*x4^3*z^13 - 3*x1^24*x2^20*x3^18*x4^3*z^13 + x1^23*x2^21*x3^18*x4^3*z^13 + 2*x1^22*x2^22*x3^18*x4^3*z^13 - x1^24*x2^19*x3^19*x4^3*z^13 + x1^22*x2^21*x3^19*x4^3*z^13 - x1^22*x2^20*x3^20*x4^3*z^13 + x1^33*x2^23*x3^5*x4^4*z^13 - 2*x1^32*x2^23*x3^6*x4^4*z^13 + x1^30*x2^25*x3^6*x4^4*z^13 - x1^35*x2^19*x3^7*x4^4*z^13 - x1^33*x2^21*x3^7*x4^4*z^13 + x1^32*x2^22*x3^7*x4^4*z^13 + 2*x1^31*x2^23*x3^7*x4^4*z^13 + 2*x1^30*x2^24*x3^7*x4^4*z^13 + x1^29*x2^25*x3^7*x4^4*z^13 + 3*x1^34*x2^19*x3^8*x4^4*z^13 + x1^33*x2^20*x3^8*x4^4*z^13 + x1^32*x2^21*x3^8*x4^4*z^13 - 4*x1^31*x2^22*x3^8*x4^4*z^13 - x1^30*x2^23*x3^8*x4^4*z^13 - 2*x1^29*x2^24*x3^8*x4^4*z^13 + x1^28*x2^25*x3^8*x4^4*z^13 + x1^27*x2^26*x3^8*x4^4*z^13 - 2*x1^34*x2^18*x3^9*x4^4*z^13 - 2*x1^33*x2^19*x3^9*x4^4*z^13 - x1^32*x2^20*x3^9*x4^4*z^13 + 3*x1^31*x2^21*x3^9*x4^4*z^13 + x1^30*x2^22*x3^9*x4^4*z^13 + 2*x1^29*x2^23*x3^9*x4^4*z^13 + 6*x1^28*x2^24*x3^9*x4^4*z^13 + 5*x1^33*x2^18*x3^10*x4^4*z^13 + 2*x1^32*x2^19*x3^10*x4^4*z^13 + 2*x1^31*x2^20*x3^10*x4^4*z^13 - 3*x1^30*x2^21*x3^10*x4^4*z^13 - 6*x1^28*x2^23*x3^10*x4^4*z^13 - 2*x1^27*x2^24*x3^10*x4^4*z^13 + x1^26*x2^25*x3^10*x4^4*z^13 - 4*x1^33*x2^17*x3^11*x4^4*z^13 - 3*x1^32*x2^18*x3^11*x4^4*z^13 - x1^31*x2^19*x3^11*x4^4*z^13 + 2*x1^30*x2^20*x3^11*x4^4*z^13 + 2*x1^28*x2^22*x3^11*x4^4*z^13 + 6*x1^27*x2^23*x3^11*x4^4*z^13 + 2*x1^33*x2^16*x3^12*x4^4*z^13 + 5*x1^32*x2^17*x3^12*x4^4*z^13 + 3*x1^31*x2^18*x3^12*x4^4*z^13 + 5*x1^30*x2^19*x3^12*x4^4*z^13 - 3*x1^29*x2^20*x3^12*x4^4*z^13 - 6*x1^27*x2^22*x3^12*x4^4*z^13 - 2*x1^26*x2^23*x3^12*x4^4*z^13 - 3*x1^32*x2^16*x3^13*x4^4*z^13 - x1^31*x2^17*x3^13*x4^4*z^13 - 2*x1^30*x2^18*x3^13*x4^4*z^13 - x1^29*x2^19*x3^13*x4^4*z^13 + 2*x1^28*x2^20*x3^13*x4^4*z^13 + 2*x1^27*x2^21*x3^13*x4^4*z^13 + 6*x1^26*x2^22*x3^13*x4^4*z^13 + 3*x1^31*x2^16*x3^14*x4^4*z^13 + x1^30*x2^17*x3^14*x4^4*z^13 + 3*x1^29*x2^18*x3^14*x4^4*z^13 - 3*x1^28*x2^19*x3^14*x4^4*z^13 - 6*x1^26*x2^21*x3^14*x4^4*z^13 - 2*x1^25*x2^22*x3^14*x4^4*z^13 - 2*x1^30*x2^16*x3^15*x4^4*z^13 - x1^29*x2^17*x3^15*x4^4*z^13 - x1^28*x2^18*x3^15*x4^4*z^13 + x1^27*x2^19*x3^15*x4^4*z^13 + 6*x1^25*x2^21*x3^15*x4^4*z^13 - x1^28*x2^17*x3^16*x4^4*z^13 - 2*x1^27*x2^18*x3^16*x4^4*z^13 - 4*x1^25*x2^20*x3^16*x4^4*z^13 - 3*x1^24*x2^21*x3^16*x4^4*z^13 + 4*x1^24*x2^20*x3^17*x4^4*z^13 + x1^25*x2^18*x3^18*x4^4*z^13 + 2*x1^23*x2^19*x3^19*x4^4*z^13 - x1^22*x2^20*x3^19*x4^4*z^13 - x1^32*x2^22*x3^6*x4^5*z^13 - x1^31*x2^23*x3^6*x4^5*z^13 - 2*x1^30*x2^24*x3^6*x4^5*z^13 - x1^29*x2^25*x3^6*x4^5*z^13 + 2*x1^31*x2^22*x3^7*x4^5*z^13 + x1^29*x2^24*x3^7*x4^5*z^13 + x1^28*x2^25*x3^7*x4^5*z^13 + x1^27*x2^26*x3^7*x4^5*z^13 - 2*x1^32*x2^20*x3^8*x4^5*z^13 - 2*x1^31*x2^21*x3^8*x4^5*z^13 - x1^30*x2^22*x3^8*x4^5*z^13 - 2*x1^29*x2^23*x3^8*x4^5*z^13 - 3*x1^28*x2^24*x3^8*x4^5*z^13 - 2*x1^33*x2^18*x3^9*x4^5*z^13 + 4*x1^30*x2^21*x3^9*x4^5*z^13 + 2*x1^28*x2^23*x3^9*x4^5*z^13 + x1^33*x2^17*x3^10*x4^5*z^13 + x1^32*x2^18*x3^10*x4^5*z^13 - 3*x1^30*x2^20*x3^10*x4^5*z^13 - x1^29*x2^21*x3^10*x4^5*z^13 - x1^28*x2^22*x3^10*x4^5*z^13 - 5*x1^27*x2^23*x3^10*x4^5*z^13 + 2*x1^26*x2^24*x3^10*x4^5*z^13 - x1^25*x2^25*x3^10*x4^5*z^13 - 5*x1^32*x2^17*x3^11*x4^5*z^13 + x1^31*x2^18*x3^11*x4^5*z^13 - x1^30*x2^19*x3^11*x4^5*z^13 + 5*x1^29*x2^20*x3^11*x4^5*z^13 + 5*x1^27*x2^22*x3^11*x4^5*z^13 + 3*x1^26*x2^23*x3^11*x4^5*z^13 - x1^25*x2^24*x3^11*x4^5*z^13 + x1^32*x2^16*x3^12*x4^5*z^13 + 2*x1^31*x2^17*x3^12*x4^5*z^13 - x1^30*x2^18*x3^12*x4^5*z^13 - x1^29*x2^19*x3^12*x4^5*z^13 - 2*x1^28*x2^20*x3^12*x4^5*z^13 - x1^27*x2^21*x3^12*x4^5*z^13 - 6*x1^26*x2^22*x3^12*x4^5*z^13 + 2*x1^25*x2^23*x3^12*x4^5*z^13 + x1^24*x2^24*x3^12*x4^5*z^13 - 2*x1^31*x2^16*x3^13*x4^5*z^13 - x1^30*x2^17*x3^13*x4^5*z^13 - 2*x1^29*x2^18*x3^13*x4^5*z^13 + 4*x1^28*x2^19*x3^13*x4^5*z^13 + 5*x1^26*x2^21*x3^13*x4^5*z^13 + x1^25*x2^22*x3^13*x4^5*z^13 - 2*x1^24*x2^23*x3^13*x4^5*z^13 + x1^30*x2^16*x3^14*x4^5*z^13 + x1^27*x2^19*x3^14*x4^5*z^13 - 6*x1^25*x2^21*x3^14*x4^5*z^13 + 2*x1^24*x2^22*x3^14*x4^5*z^13 + x1^23*x2^23*x3^14*x4^5*z^13 - x1^29*x2^16*x3^15*x4^5*z^13 - x1^28*x2^17*x3^15*x4^5*z^13 + 2*x1^27*x2^18*x3^15*x4^5*z^13 - x1^26*x2^19*x3^15*x4^5*z^13 + 4*x1^25*x2^20*x3^15*x4^5*z^13 + x1^24*x2^21*x3^15*x4^5*z^13 - 2*x1^23*x2^22*x3^15*x4^5*z^13 + x1^28*x2^16*x3^16*x4^5*z^13 - x1^26*x2^18*x3^16*x4^5*z^13 + x1^25*x2^19*x3^16*x4^5*z^13 - 5*x1^24*x2^20*x3^16*x4^5*z^13 + 3*x1^23*x2^21*x3^16*x4^5*z^13 + x1^26*x2^17*x3^17*x4^5*z^13 - x1^25*x2^18*x3^17*x4^5*z^13 + 2*x1^24*x2^19*x3^17*x4^5*z^13 + 2*x1^23*x2^20*x3^17*x4^5*z^13 - 3*x1^22*x2^21*x3^17*x4^5*z^13 + x1^24*x2^18*x3^18*x4^5*z^13 - 3*x1^23*x2^19*x3^18*x4^5*z^13 + x1^22*x2^20*x3^18*x4^5*z^13 + x1^21*x2^21*x3^18*x4^5*z^13 + x1^22*x2^19*x3^19*x4^5*z^13 - x1^21*x2^20*x3^19*x4^5*z^13 + x1^31*x2^22*x3^6*x4^6*z^13 + x1^29*x2^24*x3^6*x4^6*z^13 + x1^31*x2^21*x3^7*x4^6*z^13 - x1^30*x2^22*x3^7*x4^6*z^13 + x1^30*x2^21*x3^8*x4^6*z^13 + x1^29*x2^22*x3^8*x4^6*z^13 - x1^28*x2^23*x3^8*x4^6*z^13 + x1^33*x2^17*x3^9*x4^6*z^13 + 2*x1^31*x2^19*x3^9*x4^6*z^13 + 2*x1^30*x2^20*x3^9*x4^6*z^13 - x1^29*x2^21*x3^9*x4^6*z^13 - 2*x1^28*x2^22*x3^9*x4^6*z^13 - 2*x1^26*x2^24*x3^9*x4^6*z^13 + x1^25*x2^25*x3^9*x4^6*z^13 - 2*x1^31*x2^18*x3^10*x4^6*z^13 - x1^29*x2^20*x3^10*x4^6*z^13 + 2*x1^28*x2^21*x3^10*x4^6*z^13 - 2*x1^26*x2^23*x3^10*x4^6*z^13 + 4*x1^25*x2^24*x3^10*x4^6*z^13 + x1^32*x2^16*x3^11*x4^6*z^13 + x1^30*x2^18*x3^11*x4^6*z^13 + 2*x1^28*x2^20*x3^11*x4^6*z^13 - x1^27*x2^21*x3^11*x4^6*z^13 - 5*x1^25*x2^23*x3^11*x4^6*z^13 - x1^24*x2^24*x3^11*x4^6*z^13 - 2*x1^30*x2^17*x3^12*x4^6*z^13 - 3*x1^29*x2^18*x3^12*x4^6*z^13 - 3*x1^28*x2^19*x3^12*x4^6*z^13 + x1^27*x2^20*x3^12*x4^6*z^13 - x1^25*x2^22*x3^12*x4^6*z^13 + 3*x1^24*x2^23*x3^12*x4^6*z^13 + x1^30*x2^16*x3^13*x4^6*z^13 + 2*x1^29*x2^17*x3^13*x4^6*z^13 - x1^28*x2^18*x3^13*x4^6*z^13 + x1^27*x2^19*x3^13*x4^6*z^13 - x1^26*x2^20*x3^13*x4^6*z^13 + 2*x1^25*x2^21*x3^13*x4^6*z^13 - 4*x1^24*x2^22*x3^13*x4^6*z^13 - 2*x1^23*x2^23*x3^13*x4^6*z^13 - 3*x1^29*x2^16*x3^14*x4^6*z^13 - 2*x1^28*x2^17*x3^14*x4^6*z^13 - 3*x1^27*x2^18*x3^14*x4^6*z^13 - x1^25*x2^20*x3^14*x4^6*z^13 + 2*x1^24*x2^21*x3^14*x4^6*z^13 + 4*x1^23*x2^22*x3^14*x4^6*z^13 + 3*x1^28*x2^16*x3^15*x4^6*z^13 + 2*x1^27*x2^17*x3^15*x4^6*z^13 - 4*x1^25*x2^19*x3^15*x4^6*z^13 + 3*x1^24*x2^20*x3^15*x4^6*z^13 - 4*x1^23*x2^21*x3^15*x4^6*z^13 - x1^22*x2^22*x3^15*x4^6*z^13 - x1^26*x2^17*x3^16*x4^6*z^13 - x1^23*x2^20*x3^16*x4^6*z^13 + 4*x1^22*x2^21*x3^16*x4^6*z^13 + 2*x1^25*x2^17*x3^17*x4^6*z^13 + x1^23*x2^19*x3^17*x4^6*z^13 - 4*x1^22*x2^20*x3^17*x4^6*z^13 - x1^21*x2^21*x3^17*x4^6*z^13 + 2*x1^23*x2^18*x3^18*x4^6*z^13 + x1^22*x2^19*x3^18*x4^6*z^13 + 4*x1^21*x2^20*x3^18*x4^6*z^13 - x1^20*x2^20*x3^19*x4^6*z^13 - x1^30*x2^21*x3^7*x4^7*z^13 + x1^29*x2^22*x3^7*x4^7*z^13 - x1^28*x2^23*x3^7*x4^7*z^13 - x1^27*x2^24*x3^7*x4^7*z^13 + 2*x1^29*x2^21*x3^8*x4^7*z^13 + x1^27*x2^23*x3^8*x4^7*z^13 - x1^29*x2^20*x3^9*x4^7*z^13 - x1^25*x2^24*x3^9*x4^7*z^13 - x1^32*x2^16*x3^10*x4^7*z^13 - x1^29*x2^19*x3^10*x4^7*z^13 + x1^28*x2^20*x3^10*x4^7*z^13 + x1^27*x2^21*x3^10*x4^7*z^13 + x1^31*x2^16*x3^11*x4^7*z^13 + 2*x1^30*x2^17*x3^11*x4^7*z^13 - x1^29*x2^18*x3^11*x4^7*z^13 - x1^26*x2^21*x3^11*x4^7*z^13 + x1^25*x2^22*x3^11*x4^7*z^13 - x1^24*x2^23*x3^11*x4^7*z^13 - x1^30*x2^16*x3^12*x4^7*z^13 - 2*x1^29*x2^17*x3^12*x4^7*z^13 + 2*x1^28*x2^18*x3^12*x4^7*z^13 + 2*x1^25*x2^21*x3^12*x4^7*z^13 - x1^30*x2^15*x3^13*x4^7*z^13 + x1^29*x2^16*x3^13*x4^7*z^13 + 2*x1^28*x2^17*x3^13*x4^7*z^13 + x1^27*x2^18*x3^13*x4^7*z^13 + x1^26*x2^19*x3^13*x4^7*z^13 - x1^25*x2^20*x3^13*x4^7*z^13 - x1^28*x2^16*x3^14*x4^7*z^13 + 2*x1^25*x2^19*x3^14*x4^7*z^13 + x1^24*x2^20*x3^14*x4^7*z^13 + x1^23*x2^21*x3^14*x4^7*z^13 - x1^27*x2^16*x3^15*x4^7*z^13 - x1^24*x2^19*x3^15*x4^7*z^13 - x1^23*x2^20*x3^15*x4^7*z^13 - x1^22*x2^21*x3^15*x4^7*z^13 + 3*x1^22*x2^20*x3^16*x4^7*z^13 - x1^24*x2^17*x3^17*x4^7*z^13 - 3*x1^21*x2^20*x3^17*x4^7*z^13 - x1^22*x2^18*x3^18*x4^7*z^13 + x1^21*x2^19*x3^18*x4^7*z^13 + x1^20*x2^20*x3^18*x4^7*z^13 - x1^20*x2^19*x3^19*x4^7*z^13 - 2*x1^29*x2^20*x3^8*x4^8*z^13 - x1^28*x2^21*x3^8*x4^8*z^13 - x1^27*x2^22*x3^8*x4^8*z^13 - x1^25*x2^24*x3^8*x4^8*z^13 - x1^28*x2^20*x3^9*x4^8*z^13 + x1^25*x2^23*x3^9*x4^8*z^13 - x1^28*x2^19*x3^10*x4^8*z^13 - x1^27*x2^20*x3^10*x4^8*z^13 - x1^26*x2^21*x3^10*x4^8*z^13 - x1^30*x2^16*x3^11*x4^8*z^13 - 2*x1^29*x2^17*x3^11*x4^8*z^13 + x1^28*x2^18*x3^11*x4^8*z^13 + x1^27*x2^19*x3^11*x4^8*z^13 + 3*x1^24*x2^22*x3^11*x4^8*z^13 + x1^23*x2^23*x3^11*x4^8*z^13 - x1^29*x2^16*x3^12*x4^8*z^13 - x1^27*x2^18*x3^12*x4^8*z^13 - 2*x1^25*x2^20*x3^12*x4^8*z^13 - 2*x1^24*x2^21*x3^12*x4^8*z^13 - 2*x1^23*x2^22*x3^12*x4^8*z^13 + x1^28*x2^16*x3^13*x4^8*z^13 - x1^27*x2^17*x3^13*x4^8*z^13 + x1^26*x2^18*x3^13*x4^8*z^13 + 3*x1^25*x2^19*x3^13*x4^8*z^13 - x1^24*x2^20*x3^13*x4^8*z^13 + x1^23*x2^21*x3^13*x4^8*z^13 + x1^22*x2^22*x3^13*x4^8*z^13 - x1^28*x2^15*x3^14*x4^8*z^13 - x1^27*x2^16*x3^14*x4^8*z^13 - x1^26*x2^17*x3^14*x4^8*z^13 - 2*x1^24*x2^19*x3^14*x4^8*z^13 + 2*x1^23*x2^20*x3^14*x4^8*z^13 - x1^22*x2^21*x3^14*x4^8*z^13 + x1^26*x2^16*x3^15*x4^8*z^13 + x1^25*x2^17*x3^15*x4^8*z^13 + x1^24*x2^18*x3^15*x4^8*z^13 + 3*x1^23*x2^19*x3^15*x4^8*z^13 + x1^21*x2^21*x3^15*x4^8*z^13 - x1^25*x2^16*x3^16*x4^8*z^13 - x1^23*x2^18*x3^16*x4^8*z^13 - x1^22*x2^19*x3^16*x4^8*z^13 + x1^22*x2^18*x3^17*x4^8*z^13 + x1^20*x2^20*x3^17*x4^8*z^13 - x1^21*x2^18*x3^18*x4^8*z^13 + 2*x1^28*x2^19*x3^9*x4^9*z^13 + x1^27*x2^20*x3^9*x4^9*z^13 + x1^26*x2^21*x3^9*x4^9*z^13 + x1^24*x2^23*x3^9*x4^9*z^13 - x1^27*x2^19*x3^10*x4^9*z^13 - x1^26*x2^20*x3^10*x4^9*z^13 - x1^25*x2^21*x3^10*x4^9*z^13 - 2*x1^24*x2^22*x3^10*x4^9*z^13 + 2*x1^27*x2^18*x3^11*x4^9*z^13 + 2*x1^25*x2^20*x3^11*x4^9*z^13 + x1^24*x2^21*x3^11*x4^9*z^13 + x1^23*x2^22*x3^11*x4^9*z^13 + x1^28*x2^16*x3^12*x4^9*z^13 - x1^26*x2^18*x3^12*x4^9*z^13 - x1^24*x2^20*x3^12*x4^9*z^13 - 2*x1^23*x2^21*x3^12*x4^9*z^13 + 2*x1^26*x2^17*x3^13*x4^9*z^13 + 3*x1^24*x2^19*x3^13*x4^9*z^13 + 2*x1^23*x2^20*x3^13*x4^9*z^13 + 2*x1^22*x2^21*x3^13*x4^9*z^13 + x1^27*x2^15*x3^14*x4^9*z^13 - x1^24*x2^18*x3^14*x4^9*z^13 - 4*x1^23*x2^19*x3^14*x4^9*z^13 - x1^22*x2^20*x3^14*x4^9*z^13 - x1^26*x2^15*x3^15*x4^9*z^13 - 2*x1^24*x2^17*x3^15*x4^9*z^13 + 4*x1^23*x2^18*x3^15*x4^9*z^13 - x1^21*x2^20*x3^15*x4^9*z^13 - x1^24*x2^16*x3^16*x4^9*z^13 - 4*x1^22*x2^18*x3^16*x4^9*z^13 - x1^20*x2^20*x3^16*x4^9*z^13 + 2*x1^21*x2^18*x3^17*x4^9*z^13 - x1^27*x2^17*x3^11*x4^10*z^13 + x1^26*x2^18*x3^11*x4^10*z^13 + 2*x1^25*x2^19*x3^11*x4^10*z^13 + x1^23*x2^21*x3^11*x4^10*z^13 - x1^25*x2^18*x3^12*x4^10*z^13 + x1^23*x2^20*x3^12*x4^10*z^13 - 2*x1^22*x2^21*x3^12*x4^10*z^13 - x1^26*x2^16*x3^13*x4^10*z^13 + x1^24*x2^18*x3^13*x4^10*z^13 + x1^23*x2^19*x3^13*x4^10*z^13 + x1^22*x2^20*x3^13*x4^10*z^13 - x1^25*x2^16*x3^14*x4^10*z^13 + x1^23*x2^18*x3^14*x4^10*z^13 + 2*x1^22*x2^19*x3^14*x4^10*z^13 - x1^23*x2^17*x3^15*x4^10*z^13 + x1^22*x2^18*x3^15*x4^10*z^13 + x1^23*x2^16*x3^16*x4^10*z^13 - x1^21*x2^18*x3^16*x4^10*z^13 + x1^20*x2^19*x3^16*x4^10*z^13 - x1^26*x2^17*x3^11*x4^11*z^13 - x1^23*x2^20*x3^11*x4^11*z^13 - x1^25*x2^17*x3^12*x4^11*z^13 - 2*x1^24*x2^18*x3^12*x4^11*z^13 + x1^23*x2^19*x3^12*x4^11*z^13 - x1^21*x2^21*x3^12*x4^11*z^13 + x1^23*x2^18*x3^13*x4^11*z^13 + 2*x1^21*x2^20*x3^13*x4^11*z^13 - x1^23*x2^17*x3^14*x4^11*z^13 + x1^22*x2^18*x3^14*x4^11*z^13 - x1^21*x2^19*x3^14*x4^11*z^13 - x1^20*x2^20*x3^14*x4^11*z^13 - x1^21*x2^18*x3^15*x4^11*z^13 + 2*x1^20*x2^19*x3^15*x4^11*z^13 - x1^19*x2^19*x3^16*x4^11*z^13 + x1^22*x2^19*x3^12*x4^12*z^13 + x1^23*x2^17*x3^13*x4^12*z^13 - x1^22*x2^18*x3^13*x4^12*z^13 + x1^20*x2^20*x3^13*x4^12*z^13 - x1^22*x2^17*x3^14*x4^12*z^13 - 2*x1^20*x2^19*x3^14*x4^12*z^13 + x1^20*x2^18*x3^15*x4^12*z^13 + x1^19*x2^19*x3^15*x4^12*z^13 - x1^21*x2^17*x3^14*x4^13*z^13 - x1^32*x2^23*x3^5*z^12 + x1^34*x2^20*x3^6*z^12 + 2*x1^33*x2^21*x3^6*z^12 - x1^32*x2^22*x3^6*z^12 + x1^31*x2^23*x3^6*z^12 - 2*x1^33*x2^20*x3^7*z^12 - x1^32*x2^21*x3^7*z^12 + x1^33*x2^19*x3^8*z^12 + 2*x1^32*x2^20*x3^8*z^12 + x1^30*x2^22*x3^8*z^12 - x1^29*x2^23*x3^8*z^12 - 2*x1^32*x2^19*x3^9*z^12 + x1^27*x2^24*x3^9*z^12 + 2*x1^31*x2^19*x3^10*z^12 + 2*x1^29*x2^21*x3^10*z^12 - x1^28*x2^22*x3^10*z^12 - 2*x1^26*x2^24*x3^10*z^12 - 2*x1^31*x2^18*x3^11*z^12 - x1^30*x2^19*x3^11*z^12 - x1^29*x2^20*x3^11*z^12 + x1^27*x2^22*x3^11*z^12 + x1^26*x2^23*x3^11*z^12 + 2*x1^25*x2^24*x3^11*z^12 + x1^31*x2^17*x3^12*z^12 + 2*x1^30*x2^18*x3^12*z^12 + x1^28*x2^20*x3^12*z^12 - x1^27*x2^21*x3^12*z^12 - 2*x1^25*x2^23*x3^12*z^12 - x1^24*x2^24*x3^12*z^12 - 2*x1^30*x2^17*x3^13*z^12 - x1^29*x2^18*x3^13*z^12 - x1^28*x2^19*x3^13*z^12 + x1^26*x2^21*x3^13*z^12 + x1^25*x2^22*x3^13*z^12 + 2*x1^24*x2^23*x3^13*z^12 + 2*x1^29*x2^17*x3^14*z^12 + x1^27*x2^19*x3^14*z^12 - 2*x1^26*x2^20*x3^14*z^12 - 2*x1^24*x2^22*x3^14*z^12 - x1^28*x2^17*x3^15*z^12 - x1^26*x2^19*x3^15*z^12 - x1^25*x2^20*x3^15*z^12 + x1^24*x2^21*x3^15*z^12 + x1^23*x2^22*x3^15*z^12 + x1^27*x2^17*x3^16*z^12 + 2*x1^26*x2^18*x3^16*z^12 + x1^25*x2^19*x3^16*z^12 - x1^24*x2^20*x3^16*z^12 - x1^23*x2^21*x3^16*z^12 - x1^26*x2^17*x3^17*z^12 - x1^24*x2^19*x3^17*z^12 + x1^22*x2^21*x3^17*z^12 + x1^34*x2^21*x3^4*x4*z^12 - x1^32*x2^23*x3^4*x4*z^12 - x1^30*x2^25*x3^4*x4*z^12 - 2*x1^33*x2^21*x3^5*x4*z^12 + 2*x1^30*x2^24*x3^5*x4*z^12 + x1^29*x2^25*x3^5*x4*z^12 + 2*x1^33*x2^20*x3^6*x4*z^12 + 2*x1^32*x2^21*x3^6*x4*z^12 - x1^31*x2^22*x3^6*x4*z^12 - x1^30*x2^23*x3^6*x4*z^12 - 2*x1^29*x2^24*x3^6*x4*z^12 - x1^28*x2^25*x3^6*x4*z^12 - x1^33*x2^19*x3^7*x4*z^12 - 4*x1^32*x2^20*x3^7*x4*z^12 - x1^30*x2^22*x3^7*x4*z^12 + 3*x1^29*x2^23*x3^7*x4*z^12 + 2*x1^28*x2^24*x3^7*x4*z^12 + 2*x1^27*x2^25*x3^7*x4*z^12 + 5*x1^32*x2^19*x3^8*x4*z^12 + 3*x1^31*x2^20*x3^8*x4*z^12 + x1^30*x2^21*x3^8*x4*z^12 - x1^29*x2^22*x3^8*x4*z^12 - x1^28*x2^23*x3^8*x4*z^12 - 3*x1^27*x2^24*x3^8*x4*z^12 - 2*x1^26*x2^25*x3^8*x4*z^12 - 2*x1^32*x2^18*x3^9*x4*z^12 - 6*x1^31*x2^19*x3^9*x4*z^12 - 4*x1^29*x2^21*x3^9*x4*z^12 + 4*x1^28*x2^22*x3^9*x4*z^12 + 6*x1^26*x2^24*x3^9*x4*z^12 + x1^25*x2^25*x3^9*x4*z^12 + 6*x1^31*x2^18*x3^10*x4*z^12 + 2*x1^30*x2^19*x3^10*x4*z^12 + 2*x1^29*x2^20*x3^10*x4*z^12 - 2*x1^27*x2^22*x3^10*x4*z^12 - 2*x1^26*x2^23*x3^10*x4*z^12 - 6*x1^25*x2^24*x3^10*x4*z^12 - x1^31*x2^17*x3^11*x4*z^12 - 6*x1^30*x2^18*x3^11*x4*z^12 - 4*x1^28*x2^20*x3^11*x4*z^12 + 4*x1^27*x2^21*x3^11*x4*z^12 + 6*x1^25*x2^23*x3^11*x4*z^12 + 2*x1^24*x2^24*x3^11*x4*z^12 + 4*x1^30*x2^17*x3^12*x4*z^12 + 3*x1^29*x2^18*x3^12*x4*z^12 + 2*x1^28*x2^19*x3^12*x4*z^12 - 2*x1^26*x2^21*x3^12*x4*z^12 - 2*x1^25*x2^22*x3^12*x4*z^12 - 6*x1^24*x2^23*x3^12*x4*z^12 - 4*x1^29*x2^17*x3^13*x4*z^12 - 3*x1^27*x2^19*x3^13*x4*z^12 + 4*x1^26*x2^20*x3^13*x4*z^12 + 6*x1^24*x2^22*x3^13*x4*z^12 + 2*x1^23*x2^23*x3^13*x4*z^12 + x1^29*x2^16*x3^14*x4*z^12 + 2*x1^28*x2^17*x3^14*x4*z^12 + x1^27*x2^18*x3^14*x4*z^12 - x1^25*x2^20*x3^14*x4*z^12 - 2*x1^24*x2^21*x3^14*x4*z^12 - 6*x1^23*x2^22*x3^14*x4*z^12 - x1^28*x2^16*x3^15*x4*z^12 - x1^26*x2^18*x3^15*x4*z^12 + 4*x1^25*x2^19*x3^15*x4*z^12 - x1^24*x2^20*x3^15*x4*z^12 + 6*x1^23*x2^21*x3^15*x4*z^12 + 2*x1^22*x2^22*x3^15*x4*z^12 - x1^25*x2^18*x3^16*x4*z^12 - 2*x1^23*x2^20*x3^16*x4*z^12 - 5*x1^22*x2^21*x3^16*x4*z^12 - x1^25*x2^17*x3^17*x4*z^12 + x1^24*x2^18*x3^17*x4*z^12 + 3*x1^22*x2^20*x3^17*x4*z^12 + x1^21*x2^21*x3^17*x4*z^12 - x1^23*x2^18*x3^18*x4*z^12 - x1^21*x2^20*x3^18*x4*z^12 + x1^31*x2^23*x3^4*x4^2*z^12 + x1^32*x2^21*x3^5*x4^2*z^12 + x1^31*x2^22*x3^5*x4^2*z^12 + x1^30*x2^23*x3^5*x4^2*z^12 + x1^29*x2^24*x3^5*x4^2*z^12 + x1^28*x2^25*x3^5*x4^2*z^12 + x1^32*x2^20*x3^6*x4^2*z^12 + x1^31*x2^21*x3^6*x4^2*z^12 + x1^30*x2^22*x3^6*x4^2*z^12 - 3*x1^29*x2^23*x3^6*x4^2*z^12 - 2*x1^27*x2^25*x3^6*x4^2*z^12 - 2*x1^32*x2^19*x3^7*x4^2*z^12 - x1^31*x2^20*x3^7*x4^2*z^12 + x1^30*x2^21*x3^7*x4^2*z^12 + x1^29*x2^22*x3^7*x4^2*z^12 + x1^28*x2^23*x3^7*x4^2*z^12 + 2*x1^27*x2^24*x3^7*x4^2*z^12 + 3*x1^26*x2^25*x3^7*x4^2*z^12 + 4*x1^31*x2^19*x3^8*x4^2*z^12 - 4*x1^28*x2^22*x3^8*x4^2*z^12 - 5*x1^26*x2^24*x3^8*x4^2*z^12 - 2*x1^25*x2^25*x3^8*x4^2*z^12 - 4*x1^31*x2^18*x3^9*x4^2*z^12 - 2*x1^30*x2^19*x3^9*x4^2*z^12 + x1^28*x2^21*x3^9*x4^2*z^12 + 2*x1^27*x2^22*x3^9*x4^2*z^12 + 2*x1^26*x2^23*x3^9*x4^2*z^12 + 5*x1^25*x2^24*x3^9*x4^2*z^12 + x1^31*x2^17*x3^10*x4^2*z^12 + 5*x1^30*x2^18*x3^10*x4^2*z^12 + x1^29*x2^19*x3^10*x4^2*z^12 + 3*x1^28*x2^20*x3^10*x4^2*z^12 - 4*x1^27*x2^21*x3^10*x4^2*z^12 - 6*x1^25*x2^23*x3^10*x4^2*z^12 - 2*x1^24*x2^24*x3^10*x4^2*z^12 - 5*x1^30*x2^17*x3^11*x4^2*z^12 - 2*x1^29*x2^18*x3^11*x4^2*z^12 - 2*x1^28*x2^19*x3^11*x4^2*z^12 + 2*x1^26*x2^21*x3^11*x4^2*z^12 + 2*x1^25*x2^22*x3^11*x4^2*z^12 + 6*x1^24*x2^23*x3^11*x4^2*z^12 + x1^30*x2^16*x3^12*x4^2*z^12 + 5*x1^29*x2^17*x3^12*x4^2*z^12 + x1^28*x2^18*x3^12*x4^2*z^12 + 4*x1^27*x2^19*x3^12*x4^2*z^12 - 4*x1^26*x2^20*x3^12*x4^2*z^12 - 6*x1^24*x2^22*x3^12*x4^2*z^12 - 2*x1^23*x2^23*x3^12*x4^2*z^12 - 2*x1^29*x2^16*x3^13*x4^2*z^12 - 3*x1^28*x2^17*x3^13*x4^2*z^12 - 2*x1^27*x2^18*x3^13*x4^2*z^12 + 2*x1^25*x2^20*x3^13*x4^2*z^12 + 2*x1^24*x2^21*x3^13*x4^2*z^12 + 6*x1^23*x2^22*x3^13*x4^2*z^12 + 2*x1^28*x2^16*x3^14*x4^2*z^12 + x1^27*x2^17*x3^14*x4^2*z^12 + x1^26*x2^18*x3^14*x4^2*z^12 - 5*x1^25*x2^19*x3^14*x4^2*z^12 - 6*x1^23*x2^21*x3^14*x4^2*z^12 - 2*x1^22*x2^22*x3^14*x4^2*z^12 - x1^27*x2^16*x3^15*x4^2*z^12 - x1^26*x2^17*x3^15*x4^2*z^12 + x1^24*x2^19*x3^15*x4^2*z^12 + x1^23*x2^20*x3^15*x4^2*z^12 + 6*x1^22*x2^21*x3^15*x4^2*z^12 + x1^25*x2^17*x3^16*x4^2*z^12 - x1^24*x2^18*x3^16*x4^2*z^12 + x1^23*x2^19*x3^16*x4^2*z^12 - 4*x1^22*x2^20*x3^16*x4^2*z^12 - 2*x1^21*x2^21*x3^16*x4^2*z^12 - x1^23*x2^18*x3^17*x4^2*z^12 + 4*x1^21*x2^20*x3^17*x4^2*z^12 + x1^22*x2^18*x3^18*x4^2*z^12 - x1^21*x2^19*x3^18*x4^2*z^12 - x1^20*x2^20*x3^18*x4^2*z^12 - x1^29*x2^23*x3^5*x4^3*z^12 - 2*x1^33*x2^18*x3^6*x4^3*z^12 - x1^32*x2^19*x3^6*x4^3*z^12 + x1^30*x2^21*x3^6*x4^3*z^12 - x1^29*x2^22*x3^6*x4^3*z^12 - x1^28*x2^23*x3^6*x4^3*z^12 - x1^27*x2^24*x3^6*x4^3*z^12 + x1^33*x2^17*x3^7*x4^3*z^12 - x1^31*x2^19*x3^7*x4^3*z^12 - x1^30*x2^20*x3^7*x4^3*z^12 + x1^29*x2^21*x3^7*x4^3*z^12 + x1^26*x2^24*x3^7*x4^3*z^12 - x1^32*x2^17*x3^8*x4^3*z^12 - x1^30*x2^19*x3^8*x4^3*z^12 + x1^29*x2^20*x3^8*x4^3*z^12 + x1^28*x2^21*x3^8*x4^3*z^12 + x1^27*x2^22*x3^8*x4^3*z^12 - 2*x1^26*x2^23*x3^8*x4^3*z^12 - x1^25*x2^24*x3^8*x4^3*z^12 + 2*x1^32*x2^16*x3^9*x4^3*z^12 + x1^31*x2^17*x3^9*x4^3*z^12 - x1^30*x2^18*x3^9*x4^3*z^12 - x1^29*x2^19*x3^9*x4^3*z^12 + 2*x1^28*x2^20*x3^9*x4^3*z^12 + 2*x1^27*x2^21*x3^9*x4^3*z^12 - 2*x1^26*x2^22*x3^9*x4^3*z^12 + 2*x1^25*x2^23*x3^9*x4^3*z^12 + x1^24*x2^24*x3^9*x4^3*z^12 - 2*x1^31*x2^16*x3^10*x4^3*z^12 + x1^30*x2^17*x3^10*x4^3*z^12 - 3*x1^29*x2^18*x3^10*x4^3*z^12 + 2*x1^26*x2^21*x3^10*x4^3*z^12 + x1^25*x2^22*x3^10*x4^3*z^12 - 2*x1^24*x2^23*x3^10*x4^3*z^12 + 2*x1^31*x2^15*x3^11*x4^3*z^12 + 2*x1^30*x2^16*x3^11*x4^3*z^12 - x1^28*x2^18*x3^11*x4^3*z^12 - 2*x1^27*x2^19*x3^11*x4^3*z^12 - 2*x1^25*x2^21*x3^11*x4^3*z^12 + 2*x1^24*x2^22*x3^11*x4^3*z^12 + x1^23*x2^23*x3^11*x4^3*z^12 - 2*x1^30*x2^15*x3^12*x4^3*z^12 + x1^28*x2^17*x3^12*x4^3*z^12 + 2*x1^27*x2^18*x3^12*x4^3*z^12 + x1^25*x2^20*x3^12*x4^3*z^12 - 2*x1^23*x2^22*x3^12*x4^3*z^12 - x1^27*x2^17*x3^13*x4^3*z^12 - 2*x1^26*x2^18*x3^13*x4^3*z^12 - x1^25*x2^19*x3^13*x4^3*z^12 - 2*x1^24*x2^20*x3^13*x4^3*z^12 + 2*x1^23*x2^21*x3^13*x4^3*z^12 + x1^22*x2^22*x3^13*x4^3*z^12 - x1^27*x2^16*x3^14*x4^3*z^12 + 2*x1^26*x2^17*x3^14*x4^3*z^12 + x1^25*x2^18*x3^14*x4^3*z^12 + 3*x1^24*x2^19*x3^14*x4^3*z^12 - x1^23*x2^20*x3^14*x4^3*z^12 - 2*x1^22*x2^21*x3^14*x4^3*z^12 - 4*x1^23*x2^19*x3^15*x4^3*z^12 + x1^22*x2^20*x3^15*x4^3*z^12 - x1^24*x2^17*x3^16*x4^3*z^12 + 2*x1^23*x2^18*x3^16*x4^3*z^12 + 3*x1^22*x2^19*x3^16*x4^3*z^12 - x1^21*x2^20*x3^16*x4^3*z^12 - x1^22*x2^18*x3^17*x4^3*z^12 - x1^21*x2^19*x3^17*x4^3*z^12 + x1^20*x2^20*x3^17*x4^3*z^12 + x1^21*x2^18*x3^18*x4^3*z^12 + x1^31*x2^21*x3^4*x4^4*z^12 + x1^30*x2^22*x3^4*x4^4*z^12 - 2*x1^30*x2^21*x3^5*x4^4*z^12 - x1^28*x2^23*x3^5*x4^4*z^12 + 2*x1^30*x2^20*x3^6*x4^4*z^12 + 2*x1^29*x2^21*x3^6*x4^4*z^12 + 2*x1^28*x2^22*x3^6*x4^4*z^12 + 2*x1^27*x2^23*x3^6*x4^4*z^12 - x1^26*x2^24*x3^6*x4^4*z^12 + 2*x1^32*x2^17*x3^7*x4^4*z^12 + x1^31*x2^18*x3^7*x4^4*z^12 - 4*x1^29*x2^20*x3^7*x4^4*z^12 - 3*x1^27*x2^22*x3^7*x4^4*z^12 - 3*x1^26*x2^23*x3^7*x4^4*z^12 - x1^25*x2^24*x3^7*x4^4*z^12 - x1^32*x2^16*x3^8*x4^4*z^12 - x1^31*x2^17*x3^8*x4^4*z^12 - x1^30*x2^18*x3^8*x4^4*z^12 + 2*x1^29*x2^19*x3^8*x4^4*z^12 + x1^28*x2^20*x3^8*x4^4*z^12 + 3*x1^27*x2^21*x3^8*x4^4*z^12 + 5*x1^26*x2^22*x3^8*x4^4*z^12 - x1^24*x2^24*x3^8*x4^4*z^12 + 4*x1^31*x2^16*x3^9*x4^4*z^12 + x1^30*x2^17*x3^9*x4^4*z^12 + x1^29*x2^18*x3^9*x4^4*z^12 - 5*x1^28*x2^19*x3^9*x4^4*z^12 - 5*x1^26*x2^21*x3^9*x4^4*z^12 - 3*x1^25*x2^22*x3^9*x4^4*z^12 - x1^24*x2^23*x3^9*x4^4*z^12 - 2*x1^31*x2^15*x3^10*x4^4*z^12 - 3*x1^30*x2^16*x3^10*x4^4*z^12 - x1^29*x2^17*x3^10*x4^4*z^12 + x1^28*x2^18*x3^10*x4^4*z^12 + 2*x1^26*x2^20*x3^10*x4^4*z^12 + 6*x1^25*x2^21*x3^10*x4^4*z^12 + 5*x1^30*x2^15*x3^11*x4^4*z^12 + 2*x1^29*x2^16*x3^11*x4^4*z^12 + 4*x1^28*x2^17*x3^11*x4^4*z^12 - 4*x1^27*x2^18*x3^11*x4^4*z^12 - 6*x1^25*x2^20*x3^11*x4^4*z^12 - 2*x1^24*x2^21*x3^11*x4^4*z^12 - 2*x1^30*x2^14*x3^12*x4^4*z^12 - 3*x1^29*x2^15*x3^12*x4^4*z^12 - 3*x1^28*x2^16*x3^12*x4^4*z^12 - 2*x1^27*x2^17*x3^12*x4^4*z^12 - x1^26*x2^18*x3^12*x4^4*z^12 + 2*x1^25*x2^19*x3^12*x4^4*z^12 + 6*x1^24*x2^20*x3^12*x4^4*z^12 + 2*x1^29*x2^14*x3^13*x4^4*z^12 + x1^28*x2^15*x3^13*x4^4*z^12 + 2*x1^27*x2^16*x3^13*x4^4*z^12 - 3*x1^26*x2^17*x3^13*x4^4*z^12 + 2*x1^25*x2^18*x3^13*x4^4*z^12 - 5*x1^24*x2^19*x3^13*x4^4*z^12 - 2*x1^23*x2^20*x3^13*x4^4*z^12 - x1^27*x2^15*x3^14*x4^4*z^12 + 5*x1^23*x2^19*x3^14*x4^4*z^12 + 2*x1^26*x2^15*x3^15*x4^4*z^12 - x1^25*x2^16*x3^15*x4^4*z^12 + 2*x1^24*x2^17*x3^15*x4^4*z^12 - 3*x1^23*x2^18*x3^15*x4^4*z^12 - x1^22*x2^19*x3^15*x4^4*z^12 + 2*x1^24*x2^16*x3^16*x4^4*z^12 + 3*x1^22*x2^18*x3^16*x4^4*z^12 - x1^21*x2^18*x3^17*x4^4*z^12 - x1^19*x2^19*x3^18*x4^4*z^12 + x1^27*x2^23*x3^5*x4^5*z^12 + x1^29*x2^20*x3^6*x4^5*z^12 + 2*x1^27*x2^22*x3^6*x4^5*z^12 + 2*x1^26*x2^23*x3^6*x4^5*z^12 + x1^25*x2^24*x3^6*x4^5*z^12 - 2*x1^29*x2^19*x3^7*x4^5*z^12 - x1^28*x2^20*x3^7*x4^5*z^12 - 2*x1^26*x2^22*x3^7*x4^5*z^12 + x1^25*x2^23*x3^7*x4^5*z^12 - x1^24*x2^24*x3^7*x4^5*z^12 + x1^29*x2^18*x3^8*x4^5*z^12 + 5*x1^28*x2^19*x3^8*x4^5*z^12 - x1^27*x2^20*x3^8*x4^5*z^12 + 2*x1^26*x2^21*x3^8*x4^5*z^12 + 2*x1^25*x2^22*x3^8*x4^5*z^12 + x1^30*x2^16*x3^9*x4^5*z^12 - 2*x1^28*x2^18*x3^9*x4^5*z^12 - x1^27*x2^19*x3^9*x4^5*z^12 - 4*x1^25*x2^21*x3^9*x4^5*z^12 + x1^24*x2^22*x3^9*x4^5*z^12 + x1^23*x2^23*x3^9*x4^5*z^12 - 2*x1^30*x2^15*x3^10*x4^5*z^12 + 5*x1^27*x2^18*x3^10*x4^5*z^12 - x1^26*x2^19*x3^10*x4^5*z^12 + 4*x1^25*x2^20*x3^10*x4^5*z^12 + 3*x1^24*x2^21*x3^10*x4^5*z^12 - 2*x1^23*x2^22*x3^10*x4^5*z^12 + 2*x1^29*x2^15*x3^11*x4^5*z^12 + x1^28*x2^16*x3^11*x4^5*z^12 - x1^27*x2^17*x3^11*x4^5*z^12 - 3*x1^26*x2^18*x3^11*x4^5*z^12 - x1^25*x2^19*x3^11*x4^5*z^12 - 5*x1^24*x2^20*x3^11*x4^5*z^12 + x1^23*x2^21*x3^11*x4^5*z^12 - x1^29*x2^14*x3^12*x4^5*z^12 - x1^27*x2^16*x3^12*x4^5*z^12 + 4*x1^26*x2^17*x3^12*x4^5*z^12 - x1^25*x2^18*x3^12*x4^5*z^12 + 6*x1^24*x2^19*x3^12*x4^5*z^12 + 2*x1^23*x2^20*x3^12*x4^5*z^12 - 2*x1^22*x2^21*x3^12*x4^5*z^12 - 2*x1^25*x2^17*x3^13*x4^5*z^12 - x1^24*x2^18*x3^13*x4^5*z^12 - 6*x1^23*x2^19*x3^13*x4^5*z^12 + 2*x1^22*x2^20*x3^13*x4^5*z^12 + x1^21*x2^21*x3^13*x4^5*z^12 + 2*x1^25*x2^16*x3^14*x4^5*z^12 - 3*x1^24*x2^17*x3^14*x4^5*z^12 + 3*x1^23*x2^18*x3^14*x4^5*z^12 - 2*x1^21*x2^20*x3^14*x4^5*z^12 + x1^24*x2^16*x3^15*x4^5*z^12 + 2*x1^23*x2^17*x3^15*x4^5*z^12 - 4*x1^22*x2^18*x3^15*x4^5*z^12 + 2*x1^21*x2^19*x3^15*x4^5*z^12 + x1^20*x2^20*x3^15*x4^5*z^12 - x1^23*x2^16*x3^16*x4^5*z^12 + x1^22*x2^17*x3^16*x4^5*z^12 + 2*x1^21*x2^18*x3^16*x4^5*z^12 - 2*x1^20*x2^19*x3^16*x4^5*z^12 - x1^21*x2^17*x3^17*x4^5*z^12 + x1^19*x2^19*x3^17*x4^5*z^12 - x1^29*x2^19*x3^6*x4^6*z^12 - x1^28*x2^20*x3^6*x4^6*z^12 - 2*x1^26*x2^22*x3^6*x4^6*z^12 - x1^25*x2^23*x3^6*x4^6*z^12 - x1^28*x2^19*x3^7*x4^6*z^12 + x1^26*x2^21*x3^7*x4^6*z^12 + x1^24*x2^23*x3^7*x4^6*z^12 + x1^28*x2^18*x3^8*x4^6*z^12 - x1^27*x2^19*x3^8*x4^6*z^12 - 4*x1^26*x2^20*x3^8*x4^6*z^12 - x1^24*x2^22*x3^8*x4^6*z^12 - 2*x1^30*x2^15*x3^9*x4^6*z^12 - 3*x1^27*x2^18*x3^9*x4^6*z^12 + x1^26*x2^19*x3^9*x4^6*z^12 + x1^25*x2^20*x3^9*x4^6*z^12 + 3*x1^23*x2^22*x3^9*x4^6*z^12 - x1^29*x2^15*x3^10*x4^6*z^12 + 2*x1^28*x2^16*x3^10*x4^6*z^12 + x1^27*x2^17*x3^10*x4^6*z^12 - 4*x1^25*x2^19*x3^10*x4^6*z^12 - 2*x1^23*x2^21*x3^10*x4^6*z^12 - x1^22*x2^22*x3^10*x4^6*z^12 - x1^29*x2^14*x3^11*x4^6*z^12 - 3*x1^28*x2^15*x3^11*x4^6*z^12 - x1^27*x2^16*x3^11*x4^6*z^12 - x1^26*x2^17*x3^11*x4^6*z^12 + 2*x1^25*x2^18*x3^11*x4^6*z^12 - x1^24*x2^19*x3^11*x4^6*z^12 - x1^23*x2^20*x3^11*x4^6*z^12 + 5*x1^22*x2^21*x3^11*x4^6*z^12 + 4*x1^27*x2^15*x3^12*x4^6*z^12 - x1^26*x2^16*x3^12*x4^6*z^12 + 3*x1^25*x2^17*x3^12*x4^6*z^12 + x1^23*x2^19*x3^12*x4^6*z^12 - 3*x1^22*x2^20*x3^12*x4^6*z^12 - x1^21*x2^21*x3^12*x4^6*z^12 - 3*x1^26*x2^15*x3^13*x4^6*z^12 + 3*x1^24*x2^17*x3^13*x4^6*z^12 + 4*x1^21*x2^20*x3^13*x4^6*z^12 + 2*x1^25*x2^15*x3^14*x4^6*z^12 + 2*x1^24*x2^16*x3^14*x4^6*z^12 - 2*x1^23*x2^17*x3^14*x4^6*z^12 + x1^22*x2^18*x3^14*x4^6*z^12 - 3*x1^21*x2^19*x3^14*x4^6*z^12 - 2*x1^20*x2^20*x3^14*x4^6*z^12 - 2*x1^24*x2^15*x3^15*x4^6*z^12 - x1^23*x2^16*x3^15*x4^6*z^12 + 2*x1^21*x2^18*x3^15*x4^6*z^12 + 3*x1^20*x2^19*x3^15*x4^6*z^12 - 2*x1^22*x2^16*x3^16*x4^6*z^12 + x1^21*x2^17*x3^16*x4^6*z^12 - 3*x1^20*x2^18*x3^16*x4^6*z^12 - x1^19*x2^19*x3^16*x4^6*z^12 - 2*x1^20*x2^17*x3^17*x4^6*z^12 + 3*x1^19*x2^18*x3^17*x4^6*z^12 - 2*x1^18*x2^18*x3^18*x4^6*z^12 + x1^28*x2^18*x3^7*x4^7*z^12 + x1^27*x2^19*x3^7*x4^7*z^12 - x1^26*x2^20*x3^7*x4^7*z^12 + x1^25*x2^21*x3^7*x4^7*z^12 - x1^25*x2^20*x3^8*x4^7*z^12 - x1^24*x2^21*x3^8*x4^7*z^12 - x1^23*x2^22*x3^8*x4^7*z^12 + x1^27*x2^17*x3^9*x4^7*z^12 + x1^26*x2^18*x3^9*x4^7*z^12 + 2*x1^24*x2^20*x3^9*x4^7*z^12 - x1^23*x2^21*x3^9*x4^7*z^12 + x1^29*x2^14*x3^10*x4^7*z^12 + x1^28*x2^15*x3^10*x4^7*z^12 - x1^27*x2^16*x3^10*x4^7*z^12 - 3*x1^24*x2^19*x3^10*x4^7*z^12 - x1^27*x2^15*x3^11*x4^7*z^12 + x1^26*x2^16*x3^11*x4^7*z^12 + 3*x1^24*x2^18*x3^11*x4^7*z^12 + 2*x1^23*x2^19*x3^11*x4^7*z^12 - x1^22*x2^20*x3^11*x4^7*z^12 + x1^26*x2^15*x3^12*x4^7*z^12 - 2*x1^24*x2^17*x3^12*x4^7*z^12 - 2*x1^23*x2^18*x3^12*x4^7*z^12 - x1^22*x2^19*x3^12*x4^7*z^12 - x1^21*x2^20*x3^12*x4^7*z^12 - x1^25*x2^15*x3^13*x4^7*z^12 + 2*x1^21*x2^19*x3^13*x4^7*z^12 + x1^24*x2^15*x3^14*x4^7*z^12 - x1^23*x2^16*x3^14*x4^7*z^12 - 2*x1^21*x2^18*x3^14*x4^7*z^12 - 2*x1^20*x2^19*x3^14*x4^7*z^12 + 2*x1^22*x2^16*x3^15*x4^7*z^12 - x1^21*x2^17*x3^15*x4^7*z^12 + 2*x1^20*x2^18*x3^15*x4^7*z^12 + x1^19*x2^19*x3^15*x4^7*z^12 + x1^20*x2^17*x3^16*x4^7*z^12 - 2*x1^19*x2^18*x3^16*x4^7*z^12 + x1^18*x2^18*x3^17*x4^7*z^12 + x1^26*x2^18*x3^8*x4^8*z^12 + 2*x1^25*x2^19*x3^8*x4^8*z^12 + 2*x1^24*x2^20*x3^8*x4^8*z^12 + x1^23*x2^21*x3^8*x4^8*z^12 - x1^26*x2^17*x3^9*x4^8*z^12 + x1^24*x2^19*x3^9*x4^8*z^12 + x1^23*x2^20*x3^9*x4^8*z^12 + x1^25*x2^17*x3^10*x4^8*z^12 + 2*x1^24*x2^18*x3^10*x4^8*z^12 + 2*x1^22*x2^20*x3^10*x4^8*z^12 + 2*x1^26*x2^15*x3^11*x4^8*z^12 - x1^23*x2^18*x3^11*x4^8*z^12 + 2*x1^22*x2^19*x3^11*x4^8*z^12 - 2*x1^21*x2^20*x3^11*x4^8*z^12 + x1^26*x2^14*x3^12*x4^8*z^12 + 2*x1^24*x2^16*x3^12*x4^8*z^12 + x1^22*x2^18*x3^12*x4^8*z^12 + 2*x1^20*x2^20*x3^12*x4^8*z^12 - x1^25*x2^14*x3^13*x4^8*z^12 - x1^24*x2^15*x3^13*x4^8*z^12 + x1^23*x2^16*x3^13*x4^8*z^12 - 2*x1^22*x2^17*x3^13*x4^8*z^12 - x1^21*x2^18*x3^13*x4^8*z^12 + x1^24*x2^14*x3^14*x4^8*z^12 + x1^23*x2^15*x3^14*x4^8*z^12 + 3*x1^21*x2^17*x3^14*x4^8*z^12 - x1^19*x2^19*x3^14*x4^8*z^12 - x1^21*x2^16*x3^15*x4^8*z^12 - x1^19*x2^18*x3^15*x4^8*z^12 + x1^20*x2^16*x3^16*x4^8*z^12 - x1^25*x2^17*x3^9*x4^9*z^12 - 2*x1^24*x2^18*x3^9*x4^9*z^12 - 2*x1^23*x2^19*x3^9*x4^9*z^12 - x1^22*x2^20*x3^9*x4^9*z^12 + x1^24*x2^17*x3^10*x4^9*z^12 + x1^23*x2^18*x3^10*x4^9*z^12 - x1^22*x2^19*x3^10*x4^9*z^12 + 2*x1^21*x2^20*x3^10*x4^9*z^12 - x1^25*x2^15*x3^11*x4^9*z^12 - x1^24*x2^16*x3^11*x4^9*z^12 - x1^23*x2^17*x3^11*x4^9*z^12 - 2*x1^22*x2^18*x3^11*x4^9*z^12 - x1^21*x2^19*x3^11*x4^9*z^12 + x1^24*x2^15*x3^12*x4^9*z^12 - x1^23*x2^16*x3^12*x4^9*z^12 + x1^22*x2^17*x3^12*x4^9*z^12 + x1^20*x2^19*x3^12*x4^9*z^12 + x1^24*x2^14*x3^13*x4^9*z^12 - 3*x1^21*x2^17*x3^13*x4^9*z^12 - x1^19*x2^19*x3^13*x4^9*z^12 - x1^22*x2^15*x3^14*x4^9*z^12 + 2*x1^21*x2^16*x3^14*x4^9*z^12 + x1^20*x2^17*x3^14*x4^9*z^12 + x1^19*x2^18*x3^14*x4^9*z^12 + 2*x1^21*x2^15*x3^15*x4^9*z^12 - 3*x1^20*x2^16*x3^15*x4^9*z^12 + x1^18*x2^18*x3^15*x4^9*z^12 + x1^19*x2^16*x3^16*x4^9*z^12 + x1^24*x2^16*x3^10*x4^10*z^12 + x1^24*x2^15*x3^11*x4^10*z^12 + x1^23*x2^16*x3^11*x4^10*z^12 - x1^22*x2^17*x3^11*x4^10*z^12 - 2*x1^20*x2^19*x3^11*x4^10*z^12 + x1^22*x2^16*x3^12*x4^10*z^12 + x1^21*x2^17*x3^12*x4^10*z^12 - x1^20*x2^17*x3^13*x4^10*z^12 - 2*x1^19*x2^18*x3^13*x4^10*z^12 + 2*x1^20*x2^16*x3^14*x4^10*z^12 - x1^19*x2^17*x3^14*x4^10*z^12 - x1^18*x2^18*x3^14*x4^10*z^12 + x1^21*x2^17*x3^11*x4^11*z^12 + 2*x1^21*x2^16*x3^12*x4^11*z^12 + x1^20*x2^17*x3^12*x4^11*z^12 + x1^19*x2^18*x3^12*x4^11*z^12 - x1^18*x2^18*x3^13*x4^11*z^12 + x1^18*x2^17*x3^14*x4^11*z^12 + x1^17*x2^17*x3^14*x4^12*z^12 - x1^32*x2^19*x3^4*z^11 + x1^30*x2^21*x3^4*z^11 + x1^31*x2^19*x3^5*z^11 - x1^30*x2^20*x3^5*z^11 - 2*x1^31*x2^18*x3^6*z^11 - x1^30*x2^19*x3^6*z^11 - x1^29*x2^20*x3^6*z^11 + x1^27*x2^22*x3^6*z^11 + x1^31*x2^17*x3^7*z^11 + 2*x1^30*x2^18*x3^7*z^11 + x1^28*x2^20*x3^7*z^11 - x1^27*x2^21*x3^7*z^11 - x1^26*x2^22*x3^7*z^11 - x1^25*x2^23*x3^7*z^11 - 2*x1^30*x2^17*x3^8*z^11 - x1^29*x2^18*x3^8*z^11 - x1^28*x2^19*x3^8*z^11 + x1^26*x2^21*x3^8*z^11 + x1^25*x2^22*x3^8*z^11 + x1^24*x2^23*x3^8*z^11 + x1^30*x2^16*x3^9*z^11 + 2*x1^29*x2^17*x3^9*z^11 + x1^27*x2^19*x3^9*z^11 - 2*x1^26*x2^20*x3^9*z^11 - 2*x1^24*x2^22*x3^9*z^11 - 2*x1^29*x2^16*x3^10*z^11 + 2*x1^23*x2^22*x3^10*z^11 + 2*x1^28*x2^16*x3^11*z^11 + 2*x1^26*x2^18*x3^11*z^11 - x1^25*x2^19*x3^11*z^11 - 2*x1^23*x2^21*x3^11*z^11 - x1^22*x2^22*x3^11*z^11 - x1^28*x2^15*x3^12*z^11 - x1^27*x2^16*x3^12*z^11 - x1^26*x2^17*x3^12*z^11 + x1^24*x2^19*x3^12*z^11 + x1^23*x2^20*x3^12*z^11 + 2*x1^22*x2^21*x3^12*z^11 + x1^27*x2^15*x3^13*z^11 + x1^26*x2^16*x3^13*z^11 + x1^25*x2^17*x3^13*z^11 - x1^24*x2^18*x3^13*z^11 - 2*x1^22*x2^20*x3^13*z^11 - x1^21*x2^21*x3^13*z^11 - x1^25*x2^16*x3^14*z^11 + x1^23*x2^18*x3^14*z^11 + x1^22*x2^19*x3^14*z^11 + 2*x1^21*x2^20*x3^14*z^11 + x1^22*x2^18*x3^15*z^11 - x1^21*x2^19*x3^15*z^11 - x1^22*x2^17*x3^16*z^11 + x1^20*x2^19*x3^16*z^11 - x1^29*x2^22*x3^3*x4*z^11 - x1^28*x2^23*x3^3*x4*z^11 - x1^31*x2^19*x3^4*x4*z^11 - x1^30*x2^20*x3^4*x4*z^11 + 2*x1^28*x2^22*x3^4*x4*z^11 + 2*x1^27*x2^23*x3^4*x4*z^11 + x1^26*x2^24*x3^4*x4*z^11 + 2*x1^31*x2^18*x3^5*x4*z^11 + x1^30*x2^19*x3^5*x4*z^11 - x1^29*x2^20*x3^5*x4*z^11 - 2*x1^27*x2^22*x3^5*x4*z^11 - 2*x1^26*x2^23*x3^5*x4*z^11 - x1^25*x2^24*x3^5*x4*z^11 - 4*x1^30*x2^18*x3^6*x4*z^11 + 3*x1^27*x2^21*x3^6*x4*z^11 + 2*x1^26*x2^22*x3^6*x4*z^11 + 4*x1^25*x2^23*x3^6*x4*z^11 + 4*x1^30*x2^17*x3^7*x4*z^11 + 2*x1^29*x2^18*x3^7*x4*z^11 - x1^27*x2^20*x3^7*x4*z^11 - 2*x1^26*x2^21*x3^7*x4*z^11 - 3*x1^25*x2^22*x3^7*x4*z^11 - 4*x1^24*x2^23*x3^7*x4*z^11 - x1^30*x2^16*x3^8*x4*z^11 - 5*x1^29*x2^17*x3^8*x4*z^11 - x1^28*x2^18*x3^8*x4*z^11 - 3*x1^27*x2^19*x3^8*x4*z^11 + 4*x1^26*x2^20*x3^8*x4*z^11 + 6*x1^24*x2^22*x3^8*x4*z^11 + x1^23*x2^23*x3^8*x4*z^11 + 6*x1^29*x2^16*x3^9*x4*z^11 + 2*x1^28*x2^17*x3^9*x4*z^11 + 2*x1^27*x2^18*x3^9*x4*z^11 - 2*x1^25*x2^20*x3^9*x4*z^11 - 2*x1^24*x2^21*x3^9*x4*z^11 - 6*x1^23*x2^22*x3^9*x4*z^11 - x1^29*x2^15*x3^10*x4*z^11 - 6*x1^28*x2^16*x3^10*x4*z^11 - 4*x1^26*x2^18*x3^10*x4*z^11 + 4*x1^25*x2^19*x3^10*x4*z^11 + 6*x1^23*x2^21*x3^10*x4*z^11 + 2*x1^22*x2^22*x3^10*x4*z^11 + 4*x1^28*x2^15*x3^11*x4*z^11 + x1^27*x2^16*x3^11*x4*z^11 + x1^26*x2^17*x3^11*x4*z^11 - 2*x1^24*x2^19*x3^11*x4*z^11 - 2*x1^23*x2^20*x3^11*x4*z^11 - 6*x1^22*x2^21*x3^11*x4*z^11 - 4*x1^27*x2^15*x3^12*x4*z^11 - 3*x1^25*x2^17*x3^12*x4*z^11 + 3*x1^24*x2^18*x3^12*x4*z^11 + 6*x1^22*x2^20*x3^12*x4*z^11 + 2*x1^21*x2^21*x3^12*x4*z^11 + x1^26*x2^15*x3^13*x4*z^11 - x1^24*x2^17*x3^13*x4*z^11 - 2*x1^23*x2^18*x3^13*x4*z^11 - 2*x1^22*x2^19*x3^13*x4*z^11 - 6*x1^21*x2^20*x3^13*x4*z^11 - x1^25*x2^15*x3^14*x4*z^11 + 3*x1^23*x2^17*x3^14*x4*z^11 + 5*x1^21*x2^19*x3^14*x4*z^11 + 2*x1^20*x2^20*x3^14*x4*z^11 + x1^24*x2^15*x3^15*x4*z^11 - x1^23*x2^16*x3^15*x4*z^11 - x1^22*x2^17*x3^15*x4*z^11 - x1^21*x2^18*x3^15*x4*z^11 - 5*x1^20*x2^19*x3^15*x4*z^11 + 2*x1^22*x2^16*x3^16*x4*z^11 - x1^21*x2^17*x3^16*x4*z^11 + 2*x1^20*x2^18*x3^16*x4*z^11 + 2*x1^19*x2^19*x3^16*x4*z^11 - 2*x1^19*x2^18*x3^17*x4*z^11 + x1^28*x2^21*x3^4*x4^2*z^11 - x1^28*x2^20*x3^5*x4^2*z^11 - 3*x1^27*x2^21*x3^5*x4^2*z^11 - x1^26*x2^22*x3^5*x4^2*z^11 - 2*x1^25*x2^23*x3^5*x4^2*z^11 - x1^30*x2^17*x3^6*x4^2*z^11 + x1^27*x2^20*x3^6*x4^2*z^11 + 2*x1^26*x2^21*x3^6*x4^2*z^11 + 3*x1^24*x2^23*x3^6*x4^2*z^11 + 3*x1^29*x2^17*x3^7*x4^2*z^11 + x1^28*x2^18*x3^7*x4^2*z^11 - x1^27*x2^19*x3^7*x4^2*z^11 - 4*x1^26*x2^20*x3^7*x4^2*z^11 + x1^25*x2^21*x3^7*x4^2*z^11 - 5*x1^24*x2^22*x3^7*x4^2*z^11 - x1^23*x2^23*x3^7*x4^2*z^11 - 2*x1^29*x2^16*x3^8*x4^2*z^11 - 2*x1^28*x2^17*x3^8*x4^2*z^11 + 2*x1^26*x2^19*x3^8*x4^2*z^11 + 2*x1^25*x2^20*x3^8*x4^2*z^11 + 2*x1^24*x2^21*x3^8*x4^2*z^11 + 6*x1^23*x2^22*x3^8*x4^2*z^11 + 5*x1^28*x2^16*x3^9*x4^2*z^11 + x1^27*x2^17*x3^9*x4^2*z^11 + x1^26*x2^18*x3^9*x4^2*z^11 - 4*x1^25*x2^19*x3^9*x4^2*z^11 - 6*x1^23*x2^21*x3^9*x4^2*z^11 - 2*x1^22*x2^22*x3^9*x4^2*z^11 - 3*x1^28*x2^15*x3^10*x4^2*z^11 - 2*x1^27*x2^16*x3^10*x4^2*z^11 - x1^26*x2^17*x3^10*x4^2*z^11 + 2*x1^24*x2^19*x3^10*x4^2*z^11 + 2*x1^23*x2^20*x3^10*x4^2*z^11 + 6*x1^22*x2^21*x3^10*x4^2*z^11 + 4*x1^27*x2^15*x3^11*x4^2*z^11 + 2*x1^26*x2^16*x3^11*x4^2*z^11 + 3*x1^25*x2^17*x3^11*x4^2*z^11 - 4*x1^24*x2^18*x3^11*x4^2*z^11 - 6*x1^22*x2^20*x3^11*x4^2*z^11 - 2*x1^21*x2^21*x3^11*x4^2*z^11 - x1^27*x2^14*x3^12*x4^2*z^11 - 2*x1^26*x2^15*x3^12*x4^2*z^11 - 3*x1^25*x2^16*x3^12*x4^2*z^11 + x1^23*x2^18*x3^12*x4^2*z^11 + 2*x1^22*x2^19*x3^12*x4^2*z^11 + 6*x1^21*x2^20*x3^12*x4^2*z^11 + x1^26*x2^14*x3^13*x4^2*z^11 + 2*x1^25*x2^15*x3^13*x4^2*z^11 + 3*x1^24*x2^16*x3^13*x4^2*z^11 - 2*x1^23*x2^17*x3^13*x4^2*z^11 + x1^22*x2^18*x3^13*x4^2*z^11 - 6*x1^21*x2^19*x3^13*x4^2*z^11 - 2*x1^20*x2^20*x3^13*x4^2*z^11 - 2*x1^24*x2^15*x3^14*x4^2*z^11 - x1^23*x2^16*x3^14*x4^2*z^11 + x1^22*x2^17*x3^14*x4^2*z^11 + x1^21*x2^18*x3^14*x4^2*z^11 + 6*x1^20*x2^19*x3^14*x4^2*z^11 + x1^23*x2^15*x3^15*x4^2*z^11 + x1^21*x2^17*x3^15*x4^2*z^11 - 2*x1^20*x2^18*x3^15*x4^2*z^11 - 2*x1^19*x2^19*x3^15*x4^2*z^11 - x1^20*x2^17*x3^16*x4^2*z^11 + 2*x1^19*x2^18*x3^16*x4^2*z^11 + x1^19*x2^17*x3^17*x4^2*z^11 - x1^18*x2^18*x3^17*x4^2*z^11 + x1^28*x2^21*x3^3*x4^3*z^11 - x1^29*x2^19*x3^4*x4^3*z^11 + x1^28*x2^19*x3^5*x4^3*z^11 - x1^27*x2^20*x3^5*x4^3*z^11 + x1^25*x2^22*x3^5*x4^3*z^11 + x1^30*x2^16*x3^6*x4^3*z^11 + 2*x1^29*x2^17*x3^6*x4^3*z^11 - x1^27*x2^19*x3^6*x4^3*z^11 - x1^26*x2^20*x3^6*x4^3*z^11 - x1^25*x2^21*x3^6*x4^3*z^11 + x1^24*x2^22*x3^6*x4^3*z^11 - 2*x1^30*x2^15*x3^7*x4^3*z^11 - x1^29*x2^16*x3^7*x4^3*z^11 + x1^28*x2^17*x3^7*x4^3*z^11 + 3*x1^27*x2^18*x3^7*x4^3*z^11 - x1^26*x2^19*x3^7*x4^3*z^11 + x1^24*x2^21*x3^7*x4^3*z^11 - 2*x1^23*x2^22*x3^7*x4^3*z^11 + x1^30*x2^14*x3^8*x4^3*z^11 - x1^28*x2^16*x3^8*x4^3*z^11 - x1^27*x2^17*x3^8*x4^3*z^11 + x1^26*x2^18*x3^8*x4^3*z^11 - 3*x1^24*x2^20*x3^8*x4^3*z^11 + x1^23*x2^21*x3^8*x4^3*z^11 + x1^22*x2^22*x3^8*x4^3*z^11 - x1^30*x2^13*x3^9*x4^3*z^11 - 2*x1^29*x2^14*x3^9*x4^3*z^11 - 2*x1^28*x2^15*x3^9*x4^3*z^11 - x1^27*x2^16*x3^9*x4^3*z^11 + 2*x1^26*x2^17*x3^9*x4^3*z^11 - x1^25*x2^18*x3^9*x4^3*z^11 + x1^24*x2^19*x3^9*x4^3*z^11 - x1^23*x2^20*x3^9*x4^3*z^11 - 2*x1^22*x2^21*x3^9*x4^3*z^11 + x1^29*x2^13*x3^10*x4^3*z^11 - x1^27*x2^15*x3^10*x4^3*z^11 - x1^26*x2^16*x3^10*x4^3*z^11 + x1^25*x2^17*x3^10*x4^3*z^11 + 2*x1^24*x2^18*x3^10*x4^3*z^11 - 2*x1^23*x2^19*x3^10*x4^3*z^11 + 2*x1^22*x2^20*x3^10*x4^3*z^11 - x1^28*x2^13*x3^11*x4^3*z^11 - x1^27*x2^14*x3^11*x4^3*z^11 - 4*x1^26*x2^15*x3^11*x4^3*z^11 + 2*x1^23*x2^18*x3^11*x4^3*z^11 + x1^22*x2^19*x3^11*x4^3*z^11 - 2*x1^21*x2^20*x3^11*x4^3*z^11 + x1^27*x2^13*x3^12*x4^3*z^11 + x1^26*x2^14*x3^12*x4^3*z^11 + x1^25*x2^15*x3^12*x4^3*z^11 - 2*x1^24*x2^16*x3^12*x4^3*z^11 - x1^23*x2^17*x3^12*x4^3*z^11 - 2*x1^22*x2^18*x3^12*x4^3*z^11 + 2*x1^21*x2^19*x3^12*x4^3*z^11 + x1^20*x2^20*x3^12*x4^3*z^11 + x1^25*x2^14*x3^13*x4^3*z^11 + x1^24*x2^15*x3^13*x4^3*z^11 + 2*x1^22*x2^17*x3^13*x4^3*z^11 + x1^21*x2^18*x3^13*x4^3*z^11 - 2*x1^20*x2^19*x3^13*x4^3*z^11 + x1^23*x2^15*x3^14*x4^3*z^11 + x1^22*x2^16*x3^14*x4^3*z^11 - 3*x1^21*x2^17*x3^14*x4^3*z^11 + x1^19*x2^19*x3^14*x4^3*z^11 + x1^20*x2^17*x3^15*x4^3*z^11 - x1^18*x2^18*x3^16*x4^3*z^11 - x1^28*x2^19*x3^4*x4^4*z^11 - 2*x1^27*x2^20*x3^4*x4^4*z^11 - 2*x1^26*x2^21*x3^4*x4^4*z^11 + 2*x1^28*x2^18*x3^5*x4^4*z^11 + x1^27*x2^19*x3^5*x4^4*z^11 + x1^26*x2^20*x3^5*x4^4*z^11 + 2*x1^25*x2^21*x3^5*x4^4*z^11 + x1^24*x2^22*x3^5*x4^4*z^11 - 4*x1^27*x2^18*x3^6*x4^4*z^11 - 3*x1^25*x2^20*x3^6*x4^4*z^11 - 3*x1^24*x2^21*x3^6*x4^4*z^11 - x1^29*x2^15*x3^7*x4^4*z^11 - 2*x1^28*x2^16*x3^7*x4^4*z^11 + 2*x1^27*x2^17*x3^7*x4^4*z^11 + 2*x1^26*x2^18*x3^7*x4^4*z^11 + x1^25*x2^19*x3^7*x4^4*z^11 + 4*x1^24*x2^20*x3^7*x4^4*z^11 + x1^23*x2^21*x3^7*x4^4*z^11 + x1^22*x2^22*x3^7*x4^4*z^11 + 3*x1^29*x2^14*x3^8*x4^4*z^11 + x1^28*x2^15*x3^8*x4^4*z^11 - 5*x1^26*x2^17*x3^8*x4^4*z^11 + x1^25*x2^18*x3^8*x4^4*z^11 - 5*x1^24*x2^19*x3^8*x4^4*z^11 - 4*x1^23*x2^20*x3^8*x4^4*z^11 - 2*x1^29*x2^13*x3^9*x4^4*z^11 - 2*x1^28*x2^14*x3^9*x4^4*z^11 - x1^27*x2^15*x3^9*x4^4*z^11 + x1^26*x2^16*x3^9*x4^4*z^11 + 2*x1^25*x2^17*x3^9*x4^4*z^11 + 3*x1^24*x2^18*x3^9*x4^4*z^11 + 5*x1^23*x2^19*x3^9*x4^4*z^11 + x1^22*x2^20*x3^9*x4^4*z^11 + 3*x1^28*x2^13*x3^10*x4^4*z^11 + x1^27*x2^14*x3^10*x4^4*z^11 + 2*x1^26*x2^15*x3^10*x4^4*z^11 - 3*x1^25*x2^16*x3^10*x4^4*z^11 + x1^24*x2^17*x3^10*x4^4*z^11 - 6*x1^23*x2^18*x3^10*x4^4*z^11 - 2*x1^22*x2^19*x3^10*x4^4*z^11 - 2*x1^27*x2^13*x3^11*x4^4*z^11 - 2*x1^26*x2^14*x3^11*x4^4*z^11 - x1^25*x2^15*x3^11*x4^4*z^11 - x1^24*x2^16*x3^11*x4^4*z^11 + x1^23*x2^17*x3^11*x4^4*z^11 + 6*x1^22*x2^18*x3^11*x4^4*z^11 + 2*x1^26*x2^13*x3^12*x4^4*z^11 + x1^25*x2^14*x3^12*x4^4*z^11 - x1^24*x2^15*x3^12*x4^4*z^11 + 2*x1^23*x2^16*x3^12*x4^4*z^11 - 4*x1^22*x2^17*x3^12*x4^4*z^11 - 2*x1^21*x2^18*x3^12*x4^4*z^11 - 2*x1^25*x2^13*x3^13*x4^4*z^11 + 4*x1^21*x2^17*x3^13*x4^4*z^11 - x1^20*x2^18*x3^13*x4^4*z^11 - 2*x1^23*x2^14*x3^14*x4^4*z^11 + x1^22*x2^15*x3^14*x4^4*z^11 - 2*x1^21*x2^16*x3^14*x4^4*z^11 - x1^20*x2^17*x3^14*x4^4*z^11 + x1^19*x2^18*x3^14*x4^4*z^11 - 2*x1^21*x2^15*x3^15*x4^4*z^11 + 3*x1^20*x2^16*x3^15*x4^4*z^11 - x1^19*x2^17*x3^15*x4^4*z^11 - 2*x1^19*x2^16*x3^16*x4^4*z^11 + x1^18*x2^17*x3^16*x4^4*z^11 - x1^26*x2^19*x3^5*x4^5*z^11 - x1^24*x2^21*x3^5*x4^5*z^11 - x1^23*x2^22*x3^5*x4^5*z^11 - x1^27*x2^17*x3^6*x4^5*z^11 - x1^23*x2^21*x3^6*x4^5*z^11 - x1^22*x2^22*x3^6*x4^5*z^11 + 3*x1^26*x2^17*x3^7*x4^5*z^11 + 2*x1^24*x2^19*x3^7*x4^5*z^11 + 2*x1^23*x2^20*x3^7*x4^5*z^11 - x1^22*x2^21*x3^7*x4^5*z^11 - x1^27*x2^15*x3^8*x4^5*z^11 - x1^26*x2^16*x3^8*x4^5*z^11 - 3*x1^25*x2^17*x3^8*x4^5*z^11 - 3*x1^23*x2^19*x3^8*x4^5*z^11 + x1^22*x2^20*x3^8*x4^5*z^11 + 5*x1^25*x2^16*x3^9*x4^5*z^11 - x1^24*x2^17*x3^9*x4^5*z^11 + 3*x1^23*x2^18*x3^9*x4^5*z^11 + 2*x1^22*x2^19*x3^9*x4^5*z^11 - 2*x1^21*x2^20*x3^9*x4^5*z^11 - x1^25*x2^15*x3^10*x4^5*z^11 - 2*x1^24*x2^16*x3^10*x4^5*z^11 - 4*x1^22*x2^18*x3^10*x4^5*z^11 + x1^20*x2^20*x3^10*x4^5*z^11 + x1^26*x2^13*x3^11*x4^5*z^11 - x1^25*x2^14*x3^11*x4^5*z^11 + 3*x1^24*x2^15*x3^11*x4^5*z^11 - 2*x1^23*x2^16*x3^11*x4^5*z^11 + 7*x1^22*x2^17*x3^11*x4^5*z^11 + 2*x1^21*x2^18*x3^11*x4^5*z^11 - 2*x1^20*x2^19*x3^11*x4^5*z^11 - x1^23*x2^15*x3^12*x4^5*z^11 - 7*x1^21*x2^17*x3^12*x4^5*z^11 + 2*x1^20*x2^18*x3^12*x4^5*z^11 + x1^23*x2^14*x3^13*x4^5*z^11 - 2*x1^22*x2^15*x3^13*x4^5*z^11 + 3*x1^21*x2^16*x3^13*x4^5*z^11 + 3*x1^20*x2^17*x3^13*x4^5*z^11 - 2*x1^19*x2^18*x3^13*x4^5*z^11 - x1^22*x2^14*x3^14*x4^5*z^11 + x1^21*x2^15*x3^14*x4^5*z^11 - 3*x1^20*x2^16*x3^14*x4^5*z^11 + 2*x1^19*x2^17*x3^14*x4^5*z^11 + x1^18*x2^18*x3^14*x4^5*z^11 - x1^20*x2^15*x3^15*x4^5*z^11 - 2*x1^18*x2^17*x3^15*x4^5*z^11 + x1^26*x2^17*x3^6*x4^6*z^11 + 2*x1^25*x2^18*x3^6*x4^6*z^11 + x1^24*x2^19*x3^6*x4^6*z^11 + 2*x1^22*x2^21*x3^6*x4^6*z^11 - x1^26*x2^16*x3^7*x4^6*z^11 - x1^21*x2^21*x3^7*x4^6*z^11 - x1^25*x2^16*x3^8*x4^6*z^11 + 2*x1^23*x2^18*x3^8*x4^6*z^11 + x1^22*x2^19*x3^8*x4^6*z^11 + 3*x1^21*x2^20*x3^8*x4^6*z^11 + x1^27*x2^13*x3^9*x4^6*z^11 + 2*x1^26*x2^14*x3^9*x4^6*z^11 - x1^25*x2^15*x3^9*x4^6*z^11 + x1^24*x2^16*x3^9*x4^6*z^11 - 2*x1^23*x2^17*x3^9*x4^6*z^11 - x1^21*x2^19*x3^9*x4^6*z^11 - x1^20*x2^20*x3^9*x4^6*z^11 - x1^26*x2^13*x3^10*x4^6*z^11 - x1^24*x2^15*x3^10*x4^6*z^11 + 2*x1^21*x2^18*x3^10*x4^6*z^11 + 4*x1^20*x2^19*x3^10*x4^6*z^11 + 3*x1^25*x2^13*x3^11*x4^6*z^11 + x1^24*x2^14*x3^11*x4^6*z^11 - 4*x1^22*x2^16*x3^11*x4^6*z^11 + x1^21*x2^17*x3^11*x4^6*z^11 - 3*x1^20*x2^18*x3^11*x4^6*z^11 - x1^19*x2^19*x3^11*x4^6*z^11 - 2*x1^24*x2^13*x3^12*x4^6*z^11 - x1^23*x2^14*x3^12*x4^6*z^11 + 2*x1^22*x2^15*x3^12*x4^6*z^11 - 2*x1^20*x2^17*x3^12*x4^6*z^11 + 4*x1^19*x2^18*x3^12*x4^6*z^11 + x1^22*x2^14*x3^13*x4^6*z^11 - x1^21*x2^15*x3^13*x4^6*z^11 - 4*x1^19*x2^17*x3^13*x4^6*z^11 - x1^18*x2^18*x3^13*x4^6*z^11 + x1^19*x2^16*x3^14*x4^6*z^11 + 4*x1^18*x2^17*x3^14*x4^6*z^11 + 2*x1^19*x2^15*x3^15*x4^6*z^11 - 2*x1^18*x2^16*x3^15*x4^6*z^11 - x1^17*x2^17*x3^15*x4^6*z^11 + 2*x1^17*x2^16*x3^16*x4^6*z^11 - x1^25*x2^16*x3^7*x4^7*z^11 - x1^24*x2^17*x3^7*x4^7*z^11 - x1^23*x2^18*x3^7*x4^7*z^11 + 2*x1^25*x2^15*x3^8*x4^7*z^11 - x1^23*x2^17*x3^8*x4^7*z^11 + x1^22*x2^18*x3^8*x4^7*z^11 - x1^21*x2^19*x3^8*x4^7*z^11 + x1^20*x2^20*x3^8*x4^7*z^11 - 2*x1^24*x2^15*x3^9*x4^7*z^11 - x1^23*x2^16*x3^9*x4^7*z^11 - 2*x1^22*x2^17*x3^9*x4^7*z^11 - 2*x1^21*x2^18*x3^9*x4^7*z^11 - x1^25*x2^13*x3^10*x4^7*z^11 + x1^23*x2^15*x3^10*x4^7*z^11 + 2*x1^22*x2^16*x3^10*x4^7*z^11 + 3*x1^21*x2^17*x3^10*x4^7*z^11 + x1^20*x2^18*x3^10*x4^7*z^11 + x1^25*x2^12*x3^11*x4^7*z^11 - x1^23*x2^14*x3^11*x4^7*z^11 - x1^22*x2^15*x3^11*x4^7*z^11 - x1^21*x2^16*x3^11*x4^7*z^11 - 3*x1^20*x2^17*x3^11*x4^7*z^11 - x1^19*x2^18*x3^11*x4^7*z^11 + x1^23*x2^13*x3^12*x4^7*z^11 + x1^21*x2^15*x3^12*x4^7*z^11 + x1^20*x2^16*x3^12*x4^7*z^11 + 3*x1^19*x2^17*x3^12*x4^7*z^11 + x1^22*x2^13*x3^13*x4^7*z^11 - 2*x1^21*x2^14*x3^13*x4^7*z^11 + x1^19*x2^16*x3^13*x4^7*z^11 - 3*x1^18*x2^17*x3^13*x4^7*z^11 - x1^19*x2^15*x3^14*x4^7*z^11 + x1^18*x2^16*x3^14*x4^7*z^11 + x1^17*x2^17*x3^14*x4^7*z^11 - x1^17*x2^16*x3^15*x4^7*z^11 - x1^24*x2^15*x3^8*x4^8*z^11 - x1^22*x2^17*x3^8*x4^8*z^11 - x1^21*x2^18*x3^8*x4^8*z^11 - 2*x1^20*x2^19*x3^8*x4^8*z^11 + x1^23*x2^15*x3^9*x4^8*z^11 + x1^22*x2^16*x3^9*x4^8*z^11 + x1^21*x2^17*x3^9*x4^8*z^11 - x1^19*x2^19*x3^9*x4^8*z^11 - x1^21*x2^16*x3^10*x4^8*z^11 + x1^20*x2^17*x3^10*x4^8*z^11 - 2*x1^19*x2^18*x3^10*x4^8*z^11 - x1^22*x2^14*x3^11*x4^8*z^11 - x1^21*x2^15*x3^11*x4^8*z^11 + 2*x1^20*x2^16*x3^11*x4^8*z^11 + x1^21*x2^14*x3^12*x4^8*z^11 - 2*x1^20*x2^15*x3^12*x4^8*z^11 + x1^21*x2^13*x3^13*x4^8*z^11 + x1^20*x2^14*x3^13*x4^8*z^11 + 2*x1^19*x2^15*x3^13*x4^8*z^11 - x1^18*x2^16*x3^13*x4^8*z^11 + x1^17*x2^17*x3^13*x4^8*z^11 - x1^18*x2^15*x3^14*x4^8*z^11 + x1^22*x2^15*x3^9*x4^9*z^11 + x1^21*x2^16*x3^9*x4^9*z^11 + x1^20*x2^17*x3^9*x4^9*z^11 + 2*x1^19*x2^18*x3^9*x4^9*z^11 - 2*x1^20*x2^16*x3^10*x4^9*z^11 + x1^22*x2^13*x3^11*x4^9*z^11 + x1^21*x2^14*x3^11*x4^9*z^11 + 2*x1^20*x2^15*x3^11*x4^9*z^11 + x1^19*x2^16*x3^11*x4^9*z^11 + x1^18*x2^17*x3^11*x4^9*z^11 + x1^20*x2^14*x3^12*x4^9*z^11 - 2*x1^19*x2^15*x3^12*x4^9*z^11 - x1^20*x2^13*x3^13*x4^9*z^11 + 2*x1^18*x2^15*x3^13*x4^9*z^11 - x1^18*x2^14*x3^14*x4^9*z^11 - x1^21*x2^14*x3^10*x4^10*z^11 - x1^20*x2^15*x3^10*x4^10*z^11 - x1^20*x2^14*x3^11*x4^10*z^11 + x1^17*x2^17*x3^11*x4^10*z^11 - x1^18*x2^15*x3^12*x4^10*z^11 + x1^16*x2^16*x3^13*x4^10*z^11 - x1^17*x2^14*x3^14*x4^10*z^11 - x1^18*x2^15*x3^11*x4^11*z^11 - x1^17*x2^15*x3^12*x4^11*z^11 - x1^16*x2^16*x3^12*x4^11*z^11 - x1^30*x2^17*x3^3*z^10 + x1^28*x2^19*x3^3*z^10 + x1^26*x2^21*x3^3*z^10 + x1^29*x2^17*x3^4*z^10 + x1^28*x2^18*x3^4*z^10 - x1^27*x2^19*x3^4*z^10 - 2*x1^26*x2^20*x3^4*z^10 - x1^25*x2^21*x3^4*z^10 - x1^29*x2^16*x3^5*z^10 - x1^28*x2^17*x3^5*z^10 + x1^27*x2^18*x3^5*z^10 + x1^26*x2^19*x3^5*z^10 + x1^24*x2^21*x3^5*z^10 + 2*x1^28*x2^16*x3^6*z^10 + 2*x1^26*x2^18*x3^6*z^10 - x1^25*x2^19*x3^6*z^10 - 2*x1^23*x2^21*x3^6*z^10 - 2*x1^28*x2^15*x3^7*z^10 - x1^27*x2^16*x3^7*z^10 - x1^26*x2^17*x3^7*z^10 + x1^24*x2^19*x3^7*z^10 + x1^23*x2^20*x3^7*z^10 + 2*x1^22*x2^21*x3^7*z^10 + x1^28*x2^14*x3^8*z^10 + 2*x1^27*x2^15*x3^8*z^10 + x1^25*x2^17*x3^8*z^10 - x1^24*x2^18*x3^8*z^10 - 2*x1^22*x2^20*x3^8*z^10 - x1^21*x2^21*x3^8*z^10 - 2*x1^27*x2^14*x3^9*z^10 - x1^26*x2^15*x3^9*z^10 - x1^25*x2^16*x3^9*z^10 + x1^23*x2^18*x3^9*z^10 + x1^22*x2^19*x3^9*z^10 + 2*x1^21*x2^20*x3^9*z^10 + 2*x1^26*x2^14*x3^10*z^10 + x1^24*x2^16*x3^10*z^10 - 2*x1^23*x2^17*x3^10*z^10 - 2*x1^21*x2^19*x3^10*z^10 - x1^25*x2^14*x3^11*z^10 + 2*x1^20*x2^19*x3^11*z^10 + x1^23*x2^15*x3^12*z^10 - x1^22*x2^16*x3^12*z^10 - 2*x1^20*x2^18*x3^12*z^10 - x1^19*x2^19*x3^12*z^10 + x1^20*x2^17*x3^13*z^10 + 2*x1^19*x2^18*x3^13*z^10 - x1^21*x2^15*x3^14*z^10 + x1^20*x2^16*x3^14*z^10 - 2*x1^19*x2^17*x3^14*z^10 - x1^18*x2^18*x3^14*z^10 + x1^20*x2^15*x3^15*z^10 - x1^27*x2^20*x3^2*x4*z^10 - x1^26*x2^21*x3^2*x4*z^10 - x1^27*x2^19*x3^3*x4*z^10 + x1^26*x2^20*x3^3*x4*z^10 + x1^24*x2^22*x3^3*x4*z^10 + x1^29*x2^16*x3^4*x4*z^10 - x1^26*x2^19*x3^4*x4*z^10 - 2*x1^25*x2^20*x3^4*x4*z^10 - 2*x1^24*x2^21*x3^4*x4*z^10 - 2*x1^23*x2^22*x3^4*x4*z^10 - 3*x1^28*x2^16*x3^5*x4*z^10 - x1^27*x2^17*x3^5*x4*z^10 + x1^26*x2^18*x3^5*x4*z^10 + 4*x1^25*x2^19*x3^5*x4*z^10 + 4*x1^23*x2^21*x3^5*x4*z^10 + x1^22*x2^22*x3^5*x4*z^10 + 2*x1^28*x2^15*x3^6*x4*z^10 + 2*x1^27*x2^16*x3^6*x4*z^10 - 2*x1^25*x2^18*x3^6*x4*z^10 - 2*x1^24*x2^19*x3^6*x4*z^10 - 2*x1^23*x2^20*x3^6*x4*z^10 - 5*x1^22*x2^21*x3^6*x4*z^10 - 5*x1^27*x2^15*x3^7*x4*z^10 - x1^26*x2^16*x3^7*x4*z^10 - x1^25*x2^17*x3^7*x4*z^10 + 4*x1^24*x2^18*x3^7*x4*z^10 + 6*x1^22*x2^20*x3^7*x4*z^10 + 2*x1^21*x2^21*x3^7*x4*z^10 + 4*x1^27*x2^14*x3^8*x4*z^10 + x1^26*x2^15*x3^8*x4*z^10 + x1^25*x2^16*x3^8*x4*z^10 - 2*x1^23*x2^18*x3^8*x4*z^10 - 2*x1^22*x2^19*x3^8*x4*z^10 - 6*x1^21*x2^20*x3^8*x4*z^10 - 2*x1^27*x2^13*x3^9*x4*z^10 - 5*x1^26*x2^14*x3^9*x4*z^10 - 4*x1^24*x2^16*x3^9*x4*z^10 + 4*x1^23*x2^17*x3^9*x4*z^10 + 6*x1^21*x2^19*x3^9*x4*z^10 + 2*x1^20*x2^20*x3^9*x4*z^10 + 3*x1^26*x2^13*x3^10*x4*z^10 + 2*x1^25*x2^14*x3^10*x4*z^10 + x1^24*x2^15*x3^10*x4*z^10 - 2*x1^22*x2^17*x3^10*x4*z^10 - 2*x1^21*x2^18*x3^10*x4*z^10 - 6*x1^20*x2^19*x3^10*x4*z^10 - 3*x1^25*x2^13*x3^11*x4*z^10 - x1^24*x2^14*x3^11*x4*z^10 - 2*x1^23*x2^15*x3^11*x4*z^10 + 5*x1^22*x2^16*x3^11*x4*z^10 + 6*x1^20*x2^18*x3^11*x4*z^10 + 2*x1^19*x2^19*x3^11*x4*z^10 + 2*x1^24*x2^13*x3^12*x4*z^10 + x1^23*x2^14*x3^12*x4*z^10 - 2*x1^21*x2^16*x3^12*x4*z^10 - x1^20*x2^17*x3^12*x4*z^10 - 6*x1^19*x2^18*x3^12*x4*z^10 + 3*x1^21*x2^15*x3^13*x4*z^10 + 5*x1^19*x2^17*x3^13*x4*z^10 + 2*x1^18*x2^18*x3^13*x4*z^10 - x1^20*x2^15*x3^14*x4*z^10 - x1^19*x2^16*x3^14*x4*z^10 - 5*x1^18*x2^17*x3^14*x4*z^10 - x1^19*x2^15*x3^15*x4*z^10 + 2*x1^18*x2^16*x3^15*x4*z^10 + x1^17*x2^17*x3^15*x4*z^10 - x1^17*x2^16*x3^16*x4*z^10 + x1^26*x2^20*x3^2*x4^2*z^10 + x1^26*x2^19*x3^3*x4^2*z^10 + x1^25*x2^20*x3^3*x4^2*z^10 - 2*x1^25*x2^19*x3^4*x4^2*z^10 + x1^24*x2^20*x3^4*x4^2*z^10 - x1^23*x2^21*x3^4*x4^2*z^10 + 2*x1^25*x2^18*x3^5*x4^2*z^10 + 2*x1^24*x2^19*x3^5*x4^2*z^10 + x1^23*x2^20*x3^5*x4^2*z^10 + 3*x1^22*x2^21*x3^5*x4^2*z^10 + 2*x1^27*x2^15*x3^6*x4^2*z^10 - x1^25*x2^17*x3^6*x4^2*z^10 - 3*x1^24*x2^18*x3^6*x4^2*z^10 - 4*x1^22*x2^20*x3^6*x4^2*z^10 - x1^21*x2^21*x3^6*x4^2*z^10 - x1^27*x2^14*x3^7*x4^2*z^10 - x1^26*x2^15*x3^7*x4^2*z^10 - x1^25*x2^16*x3^7*x4^2*z^10 + 2*x1^24*x2^17*x3^7*x4^2*z^10 + 3*x1^23*x2^18*x3^7*x4^2*z^10 + x1^22*x2^19*x3^7*x4^2*z^10 + 5*x1^21*x2^20*x3^7*x4^2*z^10 + 4*x1^26*x2^14*x3^8*x4^2*z^10 - 4*x1^23*x2^17*x3^8*x4^2*z^10 - 6*x1^21*x2^19*x3^8*x4^2*z^10 - 2*x1^20*x2^20*x3^8*x4^2*z^10 - x1^26*x2^13*x3^9*x4^2*z^10 - 2*x1^25*x2^14*x3^9*x4^2*z^10 + x1^23*x2^16*x3^9*x4^2*z^10 + 2*x1^22*x2^17*x3^9*x4^2*z^10 + 2*x1^21*x2^18*x3^9*x4^2*z^10 + 6*x1^20*x2^19*x3^9*x4^2*z^10 + 2*x1^25*x2^13*x3^10*x4^2*z^10 + 2*x1^24*x2^14*x3^10*x4^2*z^10 + x1^23*x2^15*x3^10*x4^2*z^10 - 3*x1^22*x2^16*x3^10*x4^2*z^10 - 6*x1^20*x2^18*x3^10*x4^2*z^10 - 2*x1^19*x2^19*x3^10*x4^2*z^10 - x1^24*x2^13*x3^11*x4^2*z^10 - 2*x1^23*x2^14*x3^11*x4^2*z^10 - x1^22*x2^15*x3^11*x4^2*z^10 + 2*x1^20*x2^17*x3^11*x4^2*z^10 + 6*x1^19*x2^18*x3^11*x4^2*z^10 + x1^23*x2^13*x3^12*x4^2*z^10 + 2*x1^22*x2^14*x3^12*x4^2*z^10 - x1^21*x2^15*x3^12*x4^2*z^10 + 2*x1^20*x2^16*x3^12*x4^2*z^10 - 5*x1^19*x2^17*x3^12*x4^2*z^10 - 2*x1^18*x2^18*x3^12*x4^2*z^10 - x1^22*x2^13*x3^13*x4^2*z^10 - x1^21*x2^14*x3^13*x4^2*z^10 - x1^20*x2^15*x3^13*x4^2*z^10 - x1^19*x2^16*x3^13*x4^2*z^10 + 5*x1^18*x2^17*x3^13*x4^2*z^10 + 2*x1^19*x2^15*x3^14*x4^2*z^10 - x1^18*x2^16*x3^14*x4^2*z^10 - 2*x1^17*x2^17*x3^14*x4^2*z^10 - x1^18*x2^15*x3^15*x4^2*z^10 + x1^17*x2^16*x3^15*x4^2*z^10 - x1^27*x2^17*x3^3*x4^3*z^10 - x1^25*x2^19*x3^3*x4^3*z^10 - x1^24*x2^20*x3^3*x4^3*z^10 + x1^26*x2^17*x3^4*x4^3*z^10 + x1^24*x2^19*x3^4*x4^3*z^10 - x1^26*x2^16*x3^5*x4^3*z^10 - x1^25*x2^17*x3^5*x4^3*z^10 - x1^28*x2^13*x3^6*x4^3*z^10 - x1^26*x2^15*x3^6*x4^3*z^10 + x1^25*x2^16*x3^6*x4^3*z^10 - 2*x1^24*x2^17*x3^6*x4^3*z^10 + 2*x1^23*x2^18*x3^6*x4^3*z^10 + 3*x1^22*x2^19*x3^6*x4^3*z^10 - x1^21*x2^20*x3^6*x4^3*z^10 + x1^27*x2^13*x3^7*x4^3*z^10 + x1^26*x2^14*x3^7*x4^3*z^10 + x1^25*x2^15*x3^7*x4^3*z^10 - 2*x1^24*x2^16*x3^7*x4^3*z^10 - x1^23*x2^17*x3^7*x4^3*z^10 - x1^22*x2^18*x3^7*x4^3*z^10 + x1^21*x2^19*x3^7*x4^3*z^10 + x1^20*x2^20*x3^7*x4^3*z^10 - 2*x1^27*x2^12*x3^8*x4^3*z^10 - x1^26*x2^13*x3^8*x4^3*z^10 + x1^25*x2^14*x3^8*x4^3*z^10 + 3*x1^24*x2^15*x3^8*x4^3*z^10 - x1^23*x2^16*x3^8*x4^3*z^10 + x1^21*x2^18*x3^8*x4^3*z^10 - x1^20*x2^19*x3^8*x4^3*z^10 + x1^27*x2^11*x3^9*x4^3*z^10 + x1^26*x2^12*x3^9*x4^3*z^10 + 2*x1^25*x2^13*x3^9*x4^3*z^10 + x1^24*x2^14*x3^9*x4^3*z^10 + 2*x1^23*x2^15*x3^9*x4^3*z^10 - 2*x1^21*x2^17*x3^9*x4^3*z^10 + 2*x1^20*x2^18*x3^9*x4^3*z^10 + x1^19*x2^19*x3^9*x4^3*z^10 - x1^26*x2^11*x3^10*x4^3*z^10 - x1^24*x2^13*x3^10*x4^3*z^10 + 2*x1^23*x2^14*x3^10*x4^3*z^10 - x1^22*x2^15*x3^10*x4^3*z^10 + 2*x1^21*x2^16*x3^10*x4^3*z^10 - x1^20*x2^17*x3^10*x4^3*z^10 - 2*x1^19*x2^18*x3^10*x4^3*z^10 + x1^23*x2^13*x3^11*x4^3*z^10 + 2*x1^22*x2^14*x3^11*x4^3*z^10 + 2*x1^21*x2^15*x3^11*x4^3*z^10 - 3*x1^20*x2^16*x3^11*x4^3*z^10 + 2*x1^19*x2^17*x3^11*x4^3*z^10 - x1^23*x2^12*x3^12*x4^3*z^10 - x1^22*x2^13*x3^12*x4^3*z^10 - 2*x1^21*x2^14*x3^12*x4^3*z^10 + x1^20*x2^15*x3^12*x4^3*z^10 + 2*x1^19*x2^16*x3^12*x4^3*z^10 - 2*x1^18*x2^17*x3^12*x4^3*z^10 - x1^21*x2^13*x3^13*x4^3*z^10 - x1^19*x2^15*x3^13*x4^3*z^10 + x1^17*x2^17*x3^13*x4^3*z^10 - x1^19*x2^14*x3^14*x4^3*z^10 - x1^18*x2^15*x3^14*x4^3*z^10 + x1^26*x2^16*x3^4*x4^4*z^10 + x1^24*x2^18*x3^4*x4^4*z^10 + 2*x1^23*x2^19*x3^4*x4^4*z^10 + x1^22*x2^20*x3^4*x4^4*z^10 - 3*x1^25*x2^16*x3^5*x4^4*z^10 - x1^24*x2^17*x3^5*x4^4*z^10 - 3*x1^23*x2^18*x3^5*x4^4*z^10 - x1^22*x2^19*x3^5*x4^4*z^10 - x1^21*x2^20*x3^5*x4^4*z^10 + 2*x1^25*x2^15*x3^6*x4^4*z^10 + 2*x1^24*x2^16*x3^6*x4^4*z^10 + x1^23*x2^17*x3^6*x4^4*z^10 + 3*x1^22*x2^18*x3^6*x4^4*z^10 + x1^21*x2^19*x3^6*x4^4*z^10 + x1^20*x2^20*x3^6*x4^4*z^10 + x1^27*x2^12*x3^7*x4^4*z^10 + x1^25*x2^14*x3^7*x4^4*z^10 - 4*x1^24*x2^15*x3^7*x4^4*z^10 - 4*x1^22*x2^17*x3^7*x4^4*z^10 - 3*x1^21*x2^18*x3^7*x4^4*z^10 - 2*x1^26*x2^12*x3^8*x4^4*z^10 - 2*x1^25*x2^13*x3^8*x4^4*z^10 + 3*x1^23*x2^15*x3^8*x4^4*z^10 + x1^22*x2^16*x3^8*x4^4*z^10 + 4*x1^21*x2^17*x3^8*x4^4*z^10 + 2*x1^20*x2^18*x3^8*x4^4*z^10 + 2*x1^26*x2^11*x3^9*x4^4*z^10 + x1^25*x2^12*x3^9*x4^4*z^10 - 4*x1^23*x2^14*x3^9*x4^4*z^10 - 7*x1^21*x2^16*x3^9*x4^4*z^10 - 3*x1^20*x2^17*x3^9*x4^4*z^10 - x1^25*x2^11*x3^10*x4^4*z^10 - x1^24*x2^12*x3^10*x4^4*z^10 + x1^23*x2^13*x3^10*x4^4*z^10 + x1^22*x2^14*x3^10*x4^4*z^10 + x1^21*x2^15*x3^10*x4^4*z^10 + 6*x1^20*x2^16*x3^10*x4^4*z^10 + x1^23*x2^12*x3^11*x4^4*z^10 - 2*x1^22*x2^13*x3^11*x4^4*z^10 + 3*x1^21*x2^14*x3^11*x4^4*z^10 - 5*x1^20*x2^15*x3^11*x4^4*z^10 - x1^19*x2^16*x3^11*x4^4*z^10 + x1^21*x2^13*x3^12*x4^4*z^10 + x1^20*x2^14*x3^12*x4^4*z^10 + 5*x1^19*x2^15*x3^12*x4^4*z^10 - x1^18*x2^16*x3^12*x4^4*z^10 + 2*x1^20*x2^13*x3^13*x4^4*z^10 - 2*x1^19*x2^14*x3^13*x4^4*z^10 - x1^18*x2^15*x3^13*x4^4*z^10 + x1^17*x2^16*x3^13*x4^4*z^10 + 2*x1^18*x2^14*x3^14*x4^4*z^10 - x1^16*x2^16*x3^14*x4^4*z^10 + 2*x1^23*x2^17*x3^5*x4^5*z^10 + x1^20*x2^20*x3^5*x4^5*z^10 + 2*x1^24*x2^15*x3^6*x4^5*z^10 + x1^22*x2^17*x3^6*x4^5*z^10 - x1^20*x2^19*x3^6*x4^5*z^10 - x1^24*x2^14*x3^7*x4^5*z^10 - x1^23*x2^15*x3^7*x4^5*z^10 - 2*x1^21*x2^17*x3^7*x4^5*z^10 - x1^20*x2^18*x3^7*x4^5*z^10 + x1^25*x2^12*x3^8*x4^5*z^10 + x1^24*x2^13*x3^8*x4^5*z^10 + 4*x1^23*x2^14*x3^8*x4^5*z^10 + 3*x1^21*x2^16*x3^8*x4^5*z^10 + x1^20*x2^17*x3^8*x4^5*z^10 - x1^19*x2^18*x3^8*x4^5*z^10 - 2*x1^24*x2^12*x3^9*x4^5*z^10 + x1^23*x2^13*x3^9*x4^5*z^10 - 2*x1^22*x2^14*x3^9*x4^5*z^10 - 5*x1^20*x2^16*x3^9*x4^5*z^10 + x1^18*x2^18*x3^9*x4^5*z^10 + x1^24*x2^11*x3^10*x4^5*z^10 + x1^23*x2^12*x3^10*x4^5*z^10 + x1^22*x2^13*x3^10*x4^5*z^10 - x1^21*x2^14*x3^10*x4^5*z^10 + 5*x1^20*x2^15*x3^10*x4^5*z^10 + x1^19*x2^16*x3^10*x4^5*z^10 - 2*x1^18*x2^17*x3^10*x4^5*z^10 - x1^22*x2^12*x3^11*x4^5*z^10 - x1^21*x2^13*x3^11*x4^5*z^10 + 2*x1^20*x2^14*x3^11*x4^5*z^10 - 6*x1^19*x2^15*x3^11*x4^5*z^10 - x1^18*x2^16*x3^11*x4^5*z^10 + x1^17*x2^17*x3^11*x4^5*z^10 + x1^21*x2^12*x3^12*x4^5*z^10 - 2*x1^20*x2^13*x3^12*x4^5*z^10 + x1^19*x2^14*x3^12*x4^5*z^10 + 2*x1^18*x2^15*x3^12*x4^5*z^10 - x1^17*x2^16*x3^12*x4^5*z^10 + x1^19*x2^13*x3^13*x4^5*z^10 - x1^18*x2^14*x3^13*x4^5*z^10 + x1^17*x2^15*x3^13*x4^5*z^10 + x1^17*x2^14*x3^14*x4^5*z^10 - x1^16*x2^15*x3^14*x4^5*z^10 + x1^15*x2^15*x3^15*x4^5*z^10 - x1^24*x2^14*x3^6*x4^6*z^10 - 2*x1^22*x2^16*x3^6*x4^6*z^10 - 2*x1^21*x2^17*x3^6*x4^6*z^10 - x1^19*x2^19*x3^6*x4^6*z^10 + 2*x1^22*x2^15*x3^7*x4^6*z^10 + x1^21*x2^16*x3^7*x4^6*z^10 + x1^19*x2^18*x3^7*x4^6*z^10 - x1^23*x2^13*x3^8*x4^6*z^10 - x1^21*x2^15*x3^8*x4^6*z^10 + x1^20*x2^16*x3^8*x4^6*z^10 - x1^19*x2^17*x3^8*x4^6*z^10 - 2*x1^18*x2^18*x3^8*x4^6*z^10 - x1^23*x2^12*x3^9*x4^6*z^10 + x1^21*x2^14*x3^9*x4^6*z^10 + 2*x1^20*x2^15*x3^9*x4^6*z^10 + x1^19*x2^16*x3^9*x4^6*z^10 + 2*x1^18*x2^17*x3^9*x4^6*z^10 + x1^23*x2^11*x3^10*x4^6*z^10 + x1^22*x2^12*x3^10*x4^6*z^10 - x1^21*x2^13*x3^10*x4^6*z^10 - 3*x1^20*x2^14*x3^10*x4^6*z^10 + 3*x1^19*x2^15*x3^10*x4^6*z^10 - 2*x1^18*x2^16*x3^10*x4^6*z^10 - 2*x1^17*x2^17*x3^10*x4^6*z^10 + 2*x1^20*x2^13*x3^11*x4^6*z^10 + 3*x1^18*x2^15*x3^11*x4^6*z^10 + 4*x1^17*x2^16*x3^11*x4^6*z^10 + 2*x1^20*x2^12*x3^12*x4^6*z^10 - 2*x1^19*x2^13*x3^12*x4^6*z^10 + x1^18*x2^14*x3^12*x4^6*z^10 - 3*x1^17*x2^15*x3^12*x4^6*z^10 - x1^16*x2^16*x3^12*x4^6*z^10 + x1^18*x2^13*x3^13*x4^6*z^10 - x1^17*x2^14*x3^13*x4^6*z^10 + 4*x1^16*x2^15*x3^13*x4^6*z^10 - 2*x1^15*x2^15*x3^14*x4^6*z^10 + x1^23*x2^13*x3^7*x4^7*z^10 + 2*x1^20*x2^16*x3^7*x4^7*z^10 - x1^22*x2^13*x3^8*x4^7*z^10 - 2*x1^21*x2^14*x3^8*x4^7*z^10 - 2*x1^20*x2^15*x3^8*x4^7*z^10 - x1^19*x2^16*x3^8*x4^7*z^10 + x1^18*x2^17*x3^8*x4^7*z^10 + x1^21*x2^13*x3^9*x4^7*z^10 + 2*x1^20*x2^14*x3^9*x4^7*z^10 + x1^19*x2^15*x3^9*x4^7*z^10 + x1^18*x2^16*x3^9*x4^7*z^10 + x1^17*x2^17*x3^9*x4^7*z^10 + x1^21*x2^12*x3^10*x4^7*z^10 - x1^20*x2^13*x3^10*x4^7*z^10 - x1^19*x2^14*x3^10*x4^7*z^10 - 2*x1^18*x2^15*x3^10*x4^7*z^10 - 2*x1^17*x2^16*x3^10*x4^7*z^10 - x1^21*x2^11*x3^11*x4^7*z^10 + x1^20*x2^12*x3^11*x4^7*z^10 + 3*x1^19*x2^13*x3^11*x4^7*z^10 - x1^18*x2^14*x3^11*x4^7*z^10 + 2*x1^17*x2^15*x3^11*x4^7*z^10 + x1^16*x2^16*x3^11*x4^7*z^10 - x1^19*x2^12*x3^12*x4^7*z^10 + x1^17*x2^14*x3^12*x4^7*z^10 - 2*x1^16*x2^15*x3^12*x4^7*z^10 + x1^15*x2^15*x3^13*x4^7*z^10 + x1^21*x2^13*x3^8*x4^8*z^10 + 2*x1^20*x2^14*x3^8*x4^8*z^10 + x1^19*x2^15*x3^8*x4^8*z^10 + x1^17*x2^17*x3^8*x4^8*z^10 + x1^20*x2^13*x3^9*x4^8*z^10 - x1^18*x2^15*x3^9*x4^8*z^10 - x1^17*x2^16*x3^9*x4^8*z^10 - x1^19*x2^13*x3^10*x4^8*z^10 + 2*x1^18*x2^14*x3^10*x4^8*z^10 - x1^16*x2^15*x3^11*x4^8*z^10 - x1^18*x2^12*x3^12*x4^8*z^10 + 2*x1^17*x2^13*x3^12*x4^8*z^10 - x1^16*x2^14*x3^12*x4^8*z^10 - x1^16*x2^13*x3^13*x4^8*z^10 - x1^19*x2^13*x3^9*x4^9*z^10 - x1^18*x2^14*x3^9*x4^9*z^10 - x1^16*x2^16*x3^9*x4^9*z^10 + x1^17*x2^14*x3^10*x4^9*z^10 + x1^16*x2^15*x3^10*x4^9*z^10 - 2*x1^17*x2^13*x3^11*x4^9*z^10 - x1^16*x2^14*x3^11*x4^9*z^10 + x1^16*x2^13*x3^12*x4^9*z^10 + x1^17*x2^13*x3^10*x4^10*z^10 + x1^25*x2^18*x3^2*z^9 + x1^24*x2^19*x3^2*z^9 + 2*x1^27*x2^15*x3^3*z^9 - x1^25*x2^17*x3^3*z^9 - x1^24*x2^18*x3^3*z^9 - 2*x1^23*x2^19*x3^3*z^9 - x1^22*x2^20*x3^3*z^9 - x1^27*x2^14*x3^4*z^9 + x1^23*x2^18*x3^4*z^9 + 2*x1^22*x2^19*x3^4*z^9 + x1^21*x2^20*x3^4*z^9 + x1^26*x2^14*x3^5*z^9 + x1^25*x2^15*x3^5*z^9 - 2*x1^23*x2^17*x3^5*z^9 - 2*x1^21*x2^19*x3^5*z^9 - 2*x1^26*x2^13*x3^6*z^9 + 2*x1^20*x2^19*x3^6*z^9 + 2*x1^25*x2^13*x3^7*z^9 + 2*x1^23*x2^15*x3^7*z^9 - x1^22*x2^16*x3^7*z^9 - 2*x1^20*x2^18*x3^7*z^9 - x1^19*x2^19*x3^7*z^9 - 2*x1^25*x2^12*x3^8*z^9 - x1^24*x2^13*x3^8*z^9 - x1^23*x2^14*x3^8*z^9 + x1^21*x2^16*x3^8*z^9 + x1^20*x2^17*x3^8*z^9 + 2*x1^19*x2^18*x3^8*z^9 + 2*x1^24*x2^12*x3^9*z^9 + x1^22*x2^14*x3^9*z^9 - x1^21*x2^15*x3^9*z^9 - 2*x1^19*x2^17*x3^9*z^9 - x1^18*x2^18*x3^9*z^9 + x1^20*x2^15*x3^10*z^9 + x1^19*x2^16*x3^10*z^9 + 2*x1^18*x2^17*x3^10*z^9 - x1^20*x2^14*x3^11*z^9 - 2*x1^18*x2^16*x3^11*z^9 + x1^20*x2^13*x3^12*z^9 + 2*x1^17*x2^16*x3^12*z^9 - x1^19*x2^13*x3^13*z^9 - x1^17*x2^15*x3^13*z^9 - x1^16*x2^16*x3^13*z^9 + x1^16*x2^15*x3^14*z^9 - x1^25*x2^18*x3*x4*z^9 + x1^24*x2^18*x3^2*x4*z^9 + x1^23*x2^19*x3^2*x4*z^9 + x1^22*x2^20*x3^2*x4*z^9 - 2*x1^24*x2^17*x3^3*x4*z^9 - x1^23*x2^18*x3^3*x4*z^9 + x1^22*x2^19*x3^3*x4*z^9 - x1^21*x2^20*x3^3*x4*z^9 - 2*x1^26*x2^14*x3^4*x4*z^9 + x1^24*x2^16*x3^4*x4*z^9 + 3*x1^23*x2^17*x3^4*x4*z^9 + 4*x1^21*x2^19*x3^4*x4*z^9 + x1^20*x2^20*x3^4*x4*z^9 + x1^26*x2^13*x3^5*x4*z^9 + x1^25*x2^14*x3^5*x4*z^9 + x1^24*x2^15*x3^5*x4*z^9 - 2*x1^23*x2^16*x3^5*x4*z^9 - 3*x1^22*x2^17*x3^5*x4*z^9 - x1^21*x2^18*x3^5*x4*z^9 - 5*x1^20*x2^19*x3^5*x4*z^9 - 4*x1^25*x2^13*x3^6*x4*z^9 + 4*x1^22*x2^16*x3^6*x4*z^9 + 6*x1^20*x2^18*x3^6*x4*z^9 + 2*x1^19*x2^19*x3^6*x4*z^9 + 2*x1^25*x2^12*x3^7*x4*z^9 + x1^24*x2^13*x3^7*x4*z^9 + x1^23*x2^14*x3^7*x4*z^9 - x1^22*x2^15*x3^7*x4*z^9 - 2*x1^21*x2^16*x3^7*x4*z^9 - 2*x1^20*x2^17*x3^7*x4*z^9 - 6*x1^19*x2^18*x3^7*x4*z^9 - 5*x1^24*x2^12*x3^8*x4*z^9 - x1^22*x2^14*x3^8*x4*z^9 + 4*x1^21*x2^15*x3^8*x4*z^9 + 6*x1^19*x2^17*x3^8*x4*z^9 + 2*x1^18*x2^18*x3^8*x4*z^9 + 2*x1^24*x2^11*x3^9*x4*z^9 + 3*x1^23*x2^12*x3^9*x4*z^9 + 2*x1^22*x2^13*x3^9*x4*z^9 - x1^21*x2^14*x3^9*x4*z^9 - 2*x1^20*x2^15*x3^9*x4*z^9 - 2*x1^19*x2^16*x3^9*x4*z^9 - 6*x1^18*x2^17*x3^9*x4*z^9 - 2*x1^23*x2^11*x3^10*x4*z^9 - 2*x1^22*x2^12*x3^10*x4*z^9 - 2*x1^21*x2^13*x3^10*x4*z^9 + 4*x1^20*x2^14*x3^10*x4*z^9 + 6*x1^18*x2^16*x3^10*x4*z^9 + 2*x1^17*x2^17*x3^10*x4*z^9 + 2*x1^21*x2^12*x3^11*x4*z^9 - x1^19*x2^14*x3^11*x4*z^9 - 2*x1^18*x2^15*x3^11*x4*z^9 - 6*x1^17*x2^16*x3^11*x4*z^9 - 2*x1^20*x2^12*x3^12*x4*z^9 + x1^19*x2^13*x3^12*x4*z^9 - x1^18*x2^14*x3^12*x4*z^9 + 4*x1^17*x2^15*x3^12*x4*z^9 + 2*x1^16*x2^16*x3^12*x4*z^9 - x1^18*x2^13*x3^13*x4*z^9 - 4*x1^16*x2^15*x3^13*x4*z^9 + 2*x1^15*x2^15*x3^14*x4*z^9 - x1^23*x2^18*x3^2*x4^2*z^9 - x1^22*x2^19*x3^2*x4^2*z^9 - x1^23*x2^17*x3^3*x4^2*z^9 - x1^22*x2^18*x3^3*x4^2*z^9 - x1^21*x2^19*x3^3*x4^2*z^9 + 2*x1^23*x2^16*x3^4*x4^2*z^9 + x1^22*x2^17*x3^4*x4^2*z^9 - x1^21*x2^18*x3^4*x4^2*z^9 + 2*x1^20*x2^19*x3^4*x4^2*z^9 - 4*x1^22*x2^16*x3^5*x4^2*z^9 - 2*x1^20*x2^18*x3^5*x4^2*z^9 - 2*x1^19*x2^19*x3^5*x4^2*z^9 - x1^24*x2^13*x3^6*x4^2*z^9 - x1^23*x2^14*x3^6*x4^2*z^9 + 3*x1^22*x2^15*x3^6*x4^2*z^9 + 2*x1^21*x2^16*x3^6*x4^2*z^9 + 5*x1^19*x2^18*x3^6*x4^2*z^9 + 2*x1^24*x2^12*x3^7*x4^2*z^9 - 3*x1^21*x2^15*x3^7*x4^2*z^9 - x1^20*x2^16*x3^7*x4^2*z^9 - 5*x1^19*x2^17*x3^7*x4^2*z^9 - 2*x1^18*x2^18*x3^7*x4^2*z^9 - x1^23*x2^12*x3^8*x4^2*z^9 - x1^22*x2^13*x3^8*x4^2*z^9 + 2*x1^21*x2^14*x3^8*x4^2*z^9 + 2*x1^20*x2^15*x3^8*x4^2*z^9 + 2*x1^19*x2^16*x3^8*x4^2*z^9 + 6*x1^18*x2^17*x3^8*x4^2*z^9 + x1^23*x2^11*x3^9*x4^2*z^9 + x1^22*x2^12*x3^9*x4^2*z^9 - 3*x1^20*x2^14*x3^9*x4^2*z^9 - 6*x1^18*x2^16*x3^9*x4^2*z^9 - 2*x1^17*x2^17*x3^9*x4^2*z^9 - x1^21*x2^12*x3^10*x4^2*z^9 + x1^20*x2^13*x3^10*x4^2*z^9 + x1^18*x2^15*x3^10*x4^2*z^9 + 6*x1^17*x2^16*x3^10*x4^2*z^9 - x1^19*x2^13*x3^11*x4^2*z^9 + 2*x1^18*x2^14*x3^11*x4^2*z^9 - 3*x1^17*x2^15*x3^11*x4^2*z^9 - 2*x1^16*x2^16*x3^11*x4^2*z^9 - x1^18*x2^13*x3^12*x4^2*z^9 - x1^17*x2^14*x3^12*x4^2*z^9 + 3*x1^16*x2^15*x3^12*x4^2*z^9 + x1^17*x2^13*x3^13*x4^2*z^9 - x1^15*x2^15*x3^13*x4^2*z^9 + 2*x1^24*x2^15*x3^3*x4^3*z^9 + x1^23*x2^16*x3^3*x4^3*z^9 + x1^22*x2^17*x3^3*x4^3*z^9 + x1^21*x2^18*x3^3*x4^3*z^9 - x1^24*x2^14*x3^4*x4^3*z^9 + x1^22*x2^16*x3^4*x4^3*z^9 - x1^21*x2^17*x3^4*x4^3*z^9 - x1^20*x2^18*x3^4*x4^3*z^9 + x1^23*x2^14*x3^5*x4^3*z^9 + 2*x1^21*x2^16*x3^5*x4^3*z^9 + x1^20*x2^17*x3^5*x4^3*z^9 - x1^19*x2^18*x3^5*x4^3*z^9 + x1^26*x2^10*x3^6*x4^3*z^9 + x1^25*x2^11*x3^6*x4^3*z^9 + x1^24*x2^12*x3^6*x4^3*z^9 - x1^23*x2^13*x3^6*x4^3*z^9 - x1^22*x2^14*x3^6*x4^3*z^9 + 2*x1^21*x2^15*x3^6*x4^3*z^9 - x1^20*x2^16*x3^6*x4^3*z^9 - x1^19*x2^17*x3^6*x4^3*z^9 - x1^25*x2^10*x3^7*x4^3*z^9 + x1^24*x2^11*x3^7*x4^3*z^9 - 2*x1^21*x2^14*x3^7*x4^3*z^9 + 3*x1^20*x2^15*x3^7*x4^3*z^9 + 2*x1^19*x2^16*x3^7*x4^3*z^9 - 2*x1^18*x2^17*x3^7*x4^3*z^9 + x1^24*x2^10*x3^8*x4^3*z^9 + 2*x1^22*x2^12*x3^8*x4^3*z^9 - x1^21*x2^13*x3^8*x4^3*z^9 - 2*x1^20*x2^14*x3^8*x4^3*z^9 - x1^19*x2^15*x3^8*x4^3*z^9 + x1^18*x2^16*x3^8*x4^3*z^9 + x1^17*x2^17*x3^8*x4^3*z^9 - x1^23*x2^10*x3^9*x4^3*z^9 - 3*x1^20*x2^13*x3^9*x4^3*z^9 + x1^19*x2^14*x3^9*x4^3*z^9 - 2*x1^17*x2^16*x3^9*x4^3*z^9 + x1^22*x2^10*x3^10*x4^3*z^9 + x1^20*x2^12*x3^10*x4^3*z^9 + x1^19*x2^13*x3^10*x4^3*z^9 - 2*x1^18*x2^14*x3^10*x4^3*z^9 + x1^17*x2^15*x3^10*x4^3*z^9 + x1^16*x2^16*x3^10*x4^3*z^9 + x1^20*x2^11*x3^11*x4^3*z^9 - x1^19*x2^12*x3^11*x4^3*z^9 - x1^16*x2^15*x3^11*x4^3*z^9 + 2*x1^18*x2^12*x3^12*x4^3*z^9 + x1^16*x2^14*x3^12*x4^3*z^9 + x1^16*x2^13*x3^13*x4^3*z^9 - x1^15*x2^14*x3^13*x4^3*z^9 + x1^14*x2^14*x3^14*x4^3*z^9 - 2*x1^23*x2^14*x3^4*x4^4*z^9 - x1^22*x2^15*x3^4*x4^4*z^9 - x1^21*x2^16*x3^4*x4^4*z^9 - x1^19*x2^18*x3^4*x4^4*z^9 + x1^23*x2^13*x3^5*x4^4*z^9 + x1^22*x2^14*x3^5*x4^4*z^9 + x1^21*x2^15*x3^5*x4^4*z^9 + 3*x1^20*x2^16*x3^5*x4^4*z^9 + 2*x1^19*x2^17*x3^5*x4^4*z^9 - 4*x1^22*x2^13*x3^6*x4^4*z^9 - x1^21*x2^14*x3^6*x4^4*z^9 - 3*x1^20*x2^15*x3^6*x4^4*z^9 - x1^19*x2^16*x3^6*x4^4*z^9 - x1^18*x2^17*x3^6*x4^4*z^9 - x1^23*x2^11*x3^7*x4^4*z^9 + x1^22*x2^12*x3^7*x4^4*z^9 + 3*x1^21*x2^13*x3^7*x4^4*z^9 + 5*x1^19*x2^15*x3^7*x4^4*z^9 + 2*x1^18*x2^16*x3^7*x4^4*z^9 + x1^22*x2^11*x3^8*x4^4*z^9 - 3*x1^21*x2^12*x3^8*x4^4*z^9 + x1^20*x2^13*x3^8*x4^4*z^9 - 6*x1^19*x2^14*x3^8*x4^4*z^9 - 2*x1^18*x2^15*x3^8*x4^4*z^9 - x1^22*x2^10*x3^9*x4^4*z^9 + x1^21*x2^11*x3^9*x4^4*z^9 + 2*x1^20*x2^12*x3^9*x4^4*z^9 + 3*x1^19*x2^13*x3^9*x4^4*z^9 + 7*x1^18*x2^14*x3^9*x4^4*z^9 + 2*x1^17*x2^15*x3^9*x4^4*z^9 + x1^21*x2^10*x3^10*x4^4*z^9 - 2*x1^20*x2^11*x3^10*x4^4*z^9 + x1^19*x2^12*x3^10*x4^4*z^9 - 5*x1^18*x2^13*x3^10*x4^4*z^9 - 2*x1^17*x2^14*x3^10*x4^4*z^9 + x1^19*x2^11*x3^11*x4^4*z^9 + 5*x1^17*x2^13*x3^11*x4^4*z^9 - x1^17*x2^12*x3^12*x4^4*z^9 - 3*x1^16*x2^13*x3^12*x4^4*z^9 - x1^20*x2^15*x3^5*x4^5*z^9 - x1^19*x2^16*x3^5*x4^5*z^9 - x1^18*x2^17*x3^5*x4^5*z^9 - x1^21*x2^13*x3^6*x4^5*z^9 - 2*x1^19*x2^15*x3^6*x4^5*z^9 - x1^18*x2^16*x3^6*x4^5*z^9 + x1^17*x2^17*x3^6*x4^5*z^9 + 2*x1^21*x2^12*x3^7*x4^5*z^9 + x1^19*x2^14*x3^7*x4^5*z^9 - x1^22*x2^10*x3^8*x4^5*z^9 - 2*x1^20*x2^12*x3^8*x4^5*z^9 - 5*x1^18*x2^14*x3^8*x4^5*z^9 + x1^17*x2^15*x3^8*x4^5*z^9 + 2*x1^21*x2^10*x3^9*x4^5*z^9 + 2*x1^20*x2^11*x3^9*x4^5*z^9 - x1^19*x2^12*x3^9*x4^5*z^9 + 2*x1^18*x2^13*x3^9*x4^5*z^9 + 3*x1^17*x2^14*x3^9*x4^5*z^9 - x1^16*x2^15*x3^9*x4^5*z^9 - x1^20*x2^10*x3^10*x4^5*z^9 + 2*x1^18*x2^12*x3^10*x4^5*z^9 - 4*x1^17*x2^13*x3^10*x4^5*z^9 + x1^15*x2^15*x3^10*x4^5*z^9 - x1^16*x2^12*x3^12*x4^5*z^9 + x1^15*x2^13*x3^12*x4^5*z^9 - x1^14*x2^13*x3^13*x4^5*z^9 + 2*x1^21*x2^12*x3^6*x4^6*z^9 + x1^19*x2^14*x3^6*x4^6*z^9 + 2*x1^18*x2^15*x3^6*x4^6*z^9 + x1^17*x2^16*x3^6*x4^6*z^9 + x1^20*x2^12*x3^7*x4^6*z^9 - 2*x1^19*x2^13*x3^7*x4^6*z^9 - x1^17*x2^15*x3^7*x4^6*z^9 - x1^16*x2^16*x3^7*x4^6*z^9 + x1^20*x2^11*x3^8*x4^6*z^9 + 3*x1^19*x2^12*x3^8*x4^6*z^9 + 2*x1^18*x2^13*x3^8*x4^6*z^9 - x1^17*x2^14*x3^8*x4^6*z^9 + 2*x1^16*x2^15*x3^8*x4^6*z^9 - x1^20*x2^10*x3^9*x4^6*z^9 - x1^19*x2^11*x3^9*x4^6*z^9 - 3*x1^18*x2^12*x3^9*x4^6*z^9 - 3*x1^16*x2^14*x3^9*x4^6*z^9 - x1^15*x2^15*x3^9*x4^6*z^9 - x1^19*x2^10*x3^10*x4^6*z^9 + x1^18*x2^11*x3^10*x4^6*z^9 + x1^17*x2^12*x3^10*x4^6*z^9 + 3*x1^15*x2^14*x3^10*x4^6*z^9 - 2*x1^17*x2^11*x3^11*x4^6*z^9 + 2*x1^16*x2^12*x3^11*x4^6*z^9 - 3*x1^15*x2^13*x3^11*x4^6*z^9 - 2*x1^14*x2^14*x3^11*x4^6*z^9 - x1^15*x2^12*x3^12*x4^6*z^9 + 3*x1^14*x2^13*x3^12*x4^6*z^9 - x1^13*x2^13*x3^13*x4^6*z^9 - x1^20*x2^11*x3^7*x4^7*z^9 - x1^19*x2^12*x3^7*x4^7*z^9 - x1^17*x2^14*x3^7*x4^7*z^9 + x1^18*x2^12*x3^8*x4^7*z^9 + x1^17*x2^13*x3^8*x4^7*z^9 + 2*x1^16*x2^14*x3^8*x4^7*z^9 - x1^17*x2^12*x3^9*x4^7*z^9 - x1^15*x2^14*x3^9*x4^7*z^9 + x1^17*x2^11*x3^10*x4^7*z^9 - x1^16*x2^12*x3^10*x4^7*z^9 + x1^15*x2^13*x3^10*x4^7*z^9 + x1^14*x2^14*x3^10*x4^7*z^9 - x1^16*x2^11*x3^11*x4^7*z^9 - x1^15*x2^12*x3^11*x4^7*z^9 - x1^14*x2^13*x3^11*x4^7*z^9 - 2*x1^17*x2^12*x3^8*x4^8*z^9 - 2*x1^16*x2^13*x3^8*x4^8*z^9 - x1^15*x2^14*x3^8*x4^8*z^9 - x1^17*x2^11*x3^9*x4^8*z^9 - x1^15*x2^13*x3^9*x4^8*z^9 + x1^14*x2^14*x3^9*x4^8*z^9 - x1^15*x2^12*x3^10*x4^8*z^9 + x1^15*x2^12*x3^9*x4^9*z^9 - x1^13*x2^13*x3^10*x4^9*z^9 + x1^14*x2^11*x3^11*x4^9*z^9 + x1^23*x2^16*x3*z^8 + x1^22*x2^17*x3*z^8 - x1^22*x2^16*x3^2*z^8 - x1^20*x2^18*x3^2*z^8 - x1^24*x2^13*x3^3*z^8 - x1^23*x2^14*x3^3*z^8 + x1^22*x2^15*x3^3*z^8 + x1^21*x2^16*x3^3*z^8 + x1^20*x2^17*x3^3*z^8 + 2*x1^19*x2^18*x3^3*z^8 + 2*x1^24*x2^12*x3^4*z^8 - x1^22*x2^14*x3^4*z^8 - x1^21*x2^15*x3^4*z^8 - 2*x1^19*x2^17*x3^4*z^8 - x1^18*x2^18*x3^4*z^8 - x1^24*x2^11*x3^5*z^8 + x1^20*x2^15*x3^5*z^8 + x1^19*x2^16*x3^5*z^8 + 2*x1^18*x2^17*x3^5*z^8 + x1^24*x2^10*x3^6*z^8 + 2*x1^23*x2^11*x3^6*z^8 + x1^21*x2^13*x3^6*z^8 - 2*x1^20*x2^14*x3^6*z^8 - 2*x1^18*x2^16*x3^6*z^8 - x1^23*x2^10*x3^7*z^8 + 2*x1^17*x2^16*x3^7*z^8 + x1^22*x2^10*x3^8*z^8 + 2*x1^20*x2^12*x3^8*z^8 - x1^19*x2^13*x3^8*z^8 - 2*x1^17*x2^15*x3^8*z^8 - x1^16*x2^16*x3^8*z^8 - x1^21*x2^10*x3^9*z^8 + x1^18*x2^13*x3^9*z^8 + x1^17*x2^14*x3^9*z^8 + 2*x1^16*x2^15*x3^9*z^8 - x1^19*x2^11*x3^10*z^8 - 2*x1^18*x2^12*x3^10*z^8 - 2*x1^16*x2^14*x3^10*z^8 - x1^15*x2^15*x3^10*z^8 + x1^17*x2^12*x3^11*z^8 + 2*x1^15*x2^14*x3^11*z^8 - x1^15*x2^13*x3^12*z^8 + x1^14*x2^13*x3^13*z^8 + x1^22*x2^16*x3*x4*z^8 - 2*x1^22*x2^15*x3^2*x4*z^8 - x1^21*x2^16*x3^2*x4*z^8 - 2*x1^19*x2^18*x3^2*x4*z^8 + 4*x1^21*x2^15*x3^3*x4*z^8 + 2*x1^19*x2^17*x3^3*x4*z^8 + x1^23*x2^12*x3^4*x4*z^8 + x1^22*x2^13*x3^4*x4*z^8 - 3*x1^21*x2^14*x3^4*x4*z^8 - 2*x1^20*x2^15*x3^4*x4*z^8 - 5*x1^18*x2^17*x3^4*x4*z^8 - 3*x1^23*x2^11*x3^5*x4*z^8 + x1^22*x2^12*x3^5*x4*z^8 + 3*x1^20*x2^14*x3^5*x4*z^8 + x1^19*x2^15*x3^5*x4*z^8 + 5*x1^18*x2^16*x3^5*x4*z^8 + 2*x1^17*x2^17*x3^5*x4*z^8 + 2*x1^23*x2^10*x3^6*x4*z^8 + 2*x1^22*x2^11*x3^6*x4*z^8 - x1^21*x2^12*x3^6*x4*z^8 - 2*x1^20*x2^13*x3^6*x4*z^8 - 2*x1^19*x2^14*x3^6*x4*z^8 - 2*x1^18*x2^15*x3^6*x4*z^8 - 6*x1^17*x2^16*x3^6*x4*z^8 - 3*x1^22*x2^10*x3^7*x4*z^8 + 4*x1^19*x2^13*x3^7*x4*z^8 + 6*x1^17*x2^15*x3^7*x4*z^8 + 2*x1^16*x2^16*x3^7*x4*z^8 + 2*x1^21*x2^10*x3^8*x4*z^8 + x1^20*x2^11*x3^8*x4*z^8 - x1^19*x2^12*x3^8*x4*z^8 - 2*x1^18*x2^13*x3^8*x4*z^8 - 2*x1^17*x2^14*x3^8*x4*z^8 - 6*x1^16*x2^15*x3^8*x4*z^8 - 2*x1^20*x2^10*x3^9*x4*z^8 - 2*x1^19*x2^11*x3^9*x4*z^8 + 2*x1^18*x2^12*x3^9*x4*z^8 + 6*x1^16*x2^14*x3^9*x4*z^8 + 2*x1^15*x2^15*x3^9*x4*z^8 + 2*x1^19*x2^10*x3^10*x4*z^8 + x1^18*x2^11*x3^10*x4*z^8 - x1^16*x2^13*x3^10*x4*z^8 - 6*x1^15*x2^14*x3^10*x4*z^8 + x1^17*x2^11*x3^11*x4*z^8 - 2*x1^16*x2^12*x3^11*x4*z^8 + 3*x1^15*x2^13*x3^11*x4*z^8 + 2*x1^14*x2^14*x3^11*x4*z^8 + 2*x1^15*x2^12*x3^12*x4*z^8 - 3*x1^14*x2^13*x3^12*x4*z^8 + x1^13*x2^13*x3^13*x4*z^8 + x1^19*x2^17*x3^2*x4^2*z^8 + x1^21*x2^14*x3^3*x4^2*z^8 + 2*x1^18*x2^17*x3^3*x4^2*z^8 - 3*x1^20*x2^14*x3^4*x4^2*z^8 - x1^19*x2^15*x3^4*x4^2*z^8 - x1^18*x2^16*x3^4*x4^2*z^8 - x1^17*x2^17*x3^4*x4^2*z^8 + 2*x1^20*x2^13*x3^5*x4^2*z^8 + 2*x1^19*x2^14*x3^5*x4^2*z^8 + 4*x1^17*x2^16*x3^5*x4^2*z^8 - 4*x1^19*x2^13*x3^6*x4^2*z^8 - x1^18*x2^14*x3^6*x4^2*z^8 - 3*x1^17*x2^15*x3^6*x4^2*z^8 - 2*x1^16*x2^16*x3^6*x4^2*z^8 - x1^21*x2^10*x3^7*x4^2*z^8 + 3*x1^19*x2^12*x3^7*x4^2*z^8 - x1^18*x2^13*x3^7*x4^2*z^8 + x1^17*x2^14*x3^7*x4^2*z^8 + 6*x1^16*x2^15*x3^7*x4^2*z^8 - 2*x1^18*x2^12*x3^8*x4^2*z^8 + 2*x1^17*x2^13*x3^8*x4^2*z^8 - 6*x1^16*x2^14*x3^8*x4^2*z^8 - 2*x1^15*x2^15*x3^8*x4^2*z^8 + x1^18*x2^11*x3^9*x4^2*z^8 + x1^17*x2^12*x3^9*x4^2*z^8 + 6*x1^15*x2^14*x3^9*x4^2*z^8 - x1^17*x2^11*x3^10*x4^2*z^8 + x1^16*x2^12*x3^10*x4^2*z^8 - 2*x1^15*x2^13*x3^10*x4^2*z^8 - 2*x1^14*x2^14*x3^10*x4^2*z^8 + x1^16*x2^11*x3^11*x4^2*z^8 + 2*x1^14*x2^13*x3^11*x4^2*z^8 - x1^21*x2^13*x3^3*x4^3*z^8 - 2*x1^20*x2^14*x3^3*x4^3*z^8 - 2*x1^19*x2^15*x3^3*x4^3*z^8 - x1^18*x2^16*x3^3*x4^3*z^8 + 2*x1^21*x2^12*x3^4*x4^3*z^8 + x1^20*x2^13*x3^4*x4^3*z^8 - x1^18*x2^15*x3^4*x4^3*z^8 - x1^21*x2^11*x3^5*x4^3*z^8 + x1^19*x2^13*x3^5*x4^3*z^8 - x1^18*x2^14*x3^5*x4^3*z^8 - x1^17*x2^15*x3^5*x4^3*z^8 - x1^23*x2^8*x3^6*x4^3*z^8 - x1^21*x2^10*x3^6*x4^3*z^8 + x1^20*x2^11*x3^6*x4^3*z^8 + x1^19*x2^12*x3^6*x4^3*z^8 + 2*x1^18*x2^13*x3^6*x4^3*z^8 - x1^16*x2^15*x3^6*x4^3*z^8 + x1^22*x2^8*x3^7*x4^3*z^8 - x1^20*x2^10*x3^7*x4^3*z^8 + 3*x1^18*x2^12*x3^7*x4^3*z^8 - 2*x1^17*x2^13*x3^7*x4^3*z^8 + x1^20*x2^9*x3^8*x4^3*z^8 + x1^19*x2^10*x3^8*x4^3*z^8 - 2*x1^18*x2^11*x3^8*x4^3*z^8 + 3*x1^17*x2^12*x3^8*x4^3*z^8 + 3*x1^16*x2^13*x3^8*x4^3*z^8 - 2*x1^15*x2^14*x3^8*x4^3*z^8 - x1^18*x2^10*x3^9*x4^3*z^8 - x1^17*x2^11*x3^9*x4^3*z^8 - 2*x1^16*x2^12*x3^9*x4^3*z^8 + x1^15*x2^13*x3^9*x4^3*z^8 + x1^14*x2^14*x3^9*x4^3*z^8 - x1^17*x2^10*x3^10*x4^3*z^8 - x1^15*x2^12*x3^10*x4^3*z^8 - x1^14*x2^13*x3^10*x4^3*z^8 - x1^15*x2^11*x3^11*x4^3*z^8 + x1^14*x2^12*x3^11*x4^3*z^8 - x1^13*x2^12*x3^12*x4^3*z^8 + x1^20*x2^12*x3^4*x4^4*z^8 + 2*x1^19*x2^13*x3^4*x4^4*z^8 + 2*x1^18*x2^14*x3^4*x4^4*z^8 + x1^17*x2^15*x3^4*x4^4*z^8 - 3*x1^20*x2^11*x3^5*x4^4*z^8 - x1^19*x2^12*x3^5*x4^4*z^8 - x1^18*x2^13*x3^5*x4^4*z^8 - 2*x1^16*x2^15*x3^5*x4^4*z^8 + 2*x1^20*x2^10*x3^6*x4^4*z^8 + 2*x1^19*x2^11*x3^6*x4^4*z^8 + x1^18*x2^12*x3^6*x4^4*z^8 + 5*x1^17*x2^13*x3^6*x4^4*z^8 + x1^16*x2^14*x3^6*x4^4*z^8 - x1^21*x2^8*x3^7*x4^4*z^8 - 2*x1^19*x2^10*x3^7*x4^4*z^8 - x1^18*x2^11*x3^7*x4^4*z^8 - 4*x1^17*x2^12*x3^7*x4^4*z^8 - 2*x1^16*x2^13*x3^7*x4^4*z^8 - x1^15*x2^14*x3^7*x4^4*z^8 + 2*x1^18*x2^10*x3^8*x4^4*z^8 + 8*x1^16*x2^12*x3^8*x4^4*z^8 + 2*x1^15*x2^13*x3^8*x4^4*z^8 - x1^18*x2^9*x3^9*x4^4*z^8 + x1^17*x2^10*x3^9*x4^4*z^8 - 3*x1^16*x2^11*x3^9*x4^4*z^8 - 4*x1^15*x2^12*x3^9*x4^4*z^8 - x1^14*x2^13*x3^9*x4^4*z^8 + 4*x1^15*x2^11*x3^10*x4^4*z^8 + x1^14*x2^12*x3^10*x4^4*z^8 - x1^14*x2^11*x3^11*x4^4*z^8 - x1^13*x2^12*x3^11*x4^4*z^8 + x1^12*x2^12*x3^12*x4^4*z^8 + x1^18*x2^12*x3^5*x4^5*z^8 - x1^17*x2^13*x3^5*x4^5*z^8 + x1^16*x2^14*x3^5*x4^5*z^8 + x1^15*x2^15*x3^5*x4^5*z^8 + x1^15*x2^14*x3^6*x4^5*z^8 + x1^17*x2^11*x3^7*x4^5*z^8 - 3*x1^16*x2^12*x3^7*x4^5*z^8 + x1^14*x2^14*x3^7*x4^5*z^8 - x1^17*x2^10*x3^8*x4^5*z^8 + x1^16*x2^11*x3^8*x4^5*z^8 + x1^15*x2^12*x3^8*x4^5*z^8 - 2*x1^17*x2^9*x3^9*x4^5*z^8 - 3*x1^15*x2^11*x3^9*x4^5*z^8 + x1^14*x2^12*x3^9*x4^5*z^8 - x1^15*x2^10*x3^10*x4^5*z^8 - x1^13*x2^12*x3^10*x4^5*z^8 + x1^12*x2^12*x3^11*x4^5*z^8 - x1^18*x2^10*x3^6*x4^6*z^8 - 2*x1^17*x2^11*x3^6*x4^6*z^8 - x1^15*x2^13*x3^6*x4^6*z^8 - x1^14*x2^14*x3^6*x4^6*z^8 + x1^17*x2^10*x3^7*x4^6*z^8 + x1^14*x2^13*x3^7*x4^6*z^8 - x1^17*x2^9*x3^8*x4^6*z^8 - 4*x1^16*x2^10*x3^8*x4^6*z^8 - x1^15*x2^11*x3^8*x4^6*z^8 - 2*x1^14*x2^12*x3^8*x4^6*z^8 - x1^13*x2^13*x3^8*x4^6*z^8 + x1^16*x2^9*x3^9*x4^6*z^8 + 2*x1^15*x2^10*x3^9*x4^6*z^8 + 3*x1^13*x2^12*x3^9*x4^6*z^8 + x1^14*x2^10*x3^10*x4^6*z^8 - x1^13*x2^11*x3^10*x4^6*z^8 - 2*x1^12*x2^12*x3^10*x4^6*z^8 + x1^12*x2^11*x3^11*x4^6*z^8 + x1^16*x2^10*x3^7*x4^7*z^8 - x1^16*x2^9*x3^8*x4^7*z^8 + x1^14*x2^11*x3^8*x4^7*z^8 - x1^13*x2^12*x3^8*x4^7*z^8 + x1^11*x2^11*x3^11*x4^7*z^8 + x1^13*x2^11*x3^8*x4^8*z^8 + x1^12*x2^12*x3^8*x4^8*z^8 - x1^13*x2^10*x3^9*x4^8*z^8 + x1^12*x2^11*x3^9*x4^8*z^8 + x1^21*x2^14*z^7 - x1^20*x2^14*x3*z^7 - x1^19*x2^15*x3*z^7 - x1^18*x2^16*x3*z^7 + x1^20*x2^13*x3^2*z^7 + x1^19*x2^14*x3^2*z^7 - x1^18*x2^15*x3^2*z^7 + x1^17*x2^16*x3^2*z^7 + x1^22*x2^10*x3^3*z^7 - x1^21*x2^11*x3^3*z^7 + x1^20*x2^12*x3^3*z^7 - x1^19*x2^13*x3^3*z^7 - 2*x1^17*x2^15*x3^3*z^7 - x1^16*x2^16*x3^3*z^7 - x1^21*x2^10*x3^4*z^7 - x1^20*x2^11*x3^4*z^7 + x1^19*x2^12*x3^4*z^7 + x1^18*x2^13*x3^4*z^7 + x1^17*x2^14*x3^4*z^7 + 2*x1^16*x2^15*x3^4*z^7 + 2*x1^21*x2^9*x3^5*z^7 - x1^19*x2^11*x3^5*z^7 - x1^18*x2^12*x3^5*z^7 - 2*x1^16*x2^14*x3^5*z^7 - x1^15*x2^15*x3^5*z^7 - x1^21*x2^8*x3^6*z^7 - x1^20*x2^9*x3^6*z^7 - x1^19*x2^10*x3^6*z^7 + x1^17*x2^12*x3^6*z^7 + x1^16*x2^13*x3^6*z^7 + 2*x1^15*x2^14*x3^6*z^7 + x1^20*x2^8*x3^7*z^7 - 2*x1^17*x2^11*x3^7*z^7 - 2*x1^15*x2^13*x3^7*z^7 + x1^17*x2^10*x3^8*z^7 + 2*x1^14*x2^13*x3^8*z^7 + x1^17*x2^9*x3^9*z^7 - x1^16*x2^10*x3^9*z^7 - 2*x1^14*x2^12*x3^9*z^7 - x1^13*x2^13*x3^9*z^7 + x1^15*x2^10*x3^10*z^7 + x1^14*x2^11*x3^10*z^7 + 2*x1^13*x2^12*x3^10*z^7 - x1^12*x2^12*x3^11*z^7 - x1^20*x2^13*x3*x4*z^7 - x1^17*x2^16*x3*x4*z^7 + 3*x1^19*x2^13*x3^2*x4*z^7 + x1^18*x2^14*x3^2*x4*z^7 + x1^17*x2^15*x3^2*x4*z^7 + x1^16*x2^16*x3^2*x4*z^7 - 2*x1^19*x2^12*x3^3*x4*z^7 - 2*x1^18*x2^13*x3^3*x4*z^7 - 4*x1^16*x2^15*x3^3*x4*z^7 - x1^21*x2^9*x3^4*x4*z^7 + x1^20*x2^10*x3^4*x4*z^7 - x1^19*x2^11*x3^4*x4*z^7 + 4*x1^18*x2^12*x3^4*x4*z^7 + x1^17*x2^13*x3^4*x4*z^7 + 3*x1^16*x2^14*x3^4*x4*z^7 + 2*x1^15*x2^15*x3^4*x4*z^7 + 2*x1^20*x2^9*x3^5*x4*z^7 + x1^19*x2^10*x3^5*x4*z^7 - 3*x1^18*x2^11*x3^5*x4*z^7 - x1^17*x2^12*x3^5*x4*z^7 - x1^16*x2^13*x3^5*x4*z^7 - 6*x1^15*x2^14*x3^5*x4*z^7 - 2*x1^20*x2^8*x3^6*x4*z^7 - 2*x1^19*x2^9*x3^6*x4*z^7 + 4*x1^17*x2^11*x3^6*x4*z^7 + 6*x1^15*x2^13*x3^6*x4*z^7 + 2*x1^14*x2^14*x3^6*x4*z^7 + x1^19*x2^8*x3^7*x4*z^7 + x1^18*x2^9*x3^7*x4*z^7 - 2*x1^17*x2^10*x3^7*x4*z^7 - x1^16*x2^11*x3^7*x4*z^7 - 2*x1^15*x2^12*x3^7*x4*z^7 - 6*x1^14*x2^13*x3^7*x4*z^7 - x1^17*x2^9*x3^8*x4*z^7 + 2*x1^16*x2^10*x3^8*x4*z^7 - x1^15*x2^11*x3^8*x4*z^7 + 5*x1^14*x2^12*x3^8*x4*z^7 + 2*x1^13*x2^13*x3^8*x4*z^7 + x1^15*x2^10*x3^9*x4*z^7 - 5*x1^13*x2^12*x3^9*x4*z^7 - 2*x1^14*x2^10*x3^10*x4*z^7 + x1^13*x2^11*x3^10*x4*z^7 + 2*x1^12*x2^12*x3^10*x4*z^7 - x1^12*x2^11*x3^11*x4*z^7 - 2*x1^18*x2^12*x3^3*x4^2*z^7 - x1^15*x2^15*x3^3*x4^2*z^7 + x1^18*x2^11*x3^4*x4^2*z^7 + x1^17*x2^12*x3^4*x4^2*z^7 + x1^16*x2^13*x3^4*x4^2*z^7 + 3*x1^15*x2^14*x3^4*x4^2*z^7 + x1^20*x2^8*x3^5*x4^2*z^7 - x1^19*x2^9*x3^5*x4^2*z^7 - x1^18*x2^10*x3^5*x4^2*z^7 - 3*x1^17*x2^11*x3^5*x4^2*z^7 - 2*x1^15*x2^13*x3^5*x4^2*z^7 - 2*x1^14*x2^14*x3^5*x4^2*z^7 + 2*x1^17*x2^10*x3^6*x4^2*z^7 + x1^15*x2^12*x3^6*x4^2*z^7 + 5*x1^14*x2^13*x3^6*x4^2*z^7 - 3*x1^16*x2^10*x3^7*x4^2*z^7 + 2*x1^15*x2^11*x3^7*x4^2*z^7 - 2*x1^14*x2^12*x3^7*x4^2*z^7 - 2*x1^13*x2^13*x3^7*x4^2*z^7 + 2*x1^16*x2^9*x3^8*x4^2*z^7 - x1^14*x2^11*x3^8*x4^2*z^7 + 4*x1^13*x2^12*x3^8*x4^2*z^7 - x1^15*x2^9*x3^9*x4^2*z^7 - x1^13*x2^11*x3^9*x4^2*z^7 - 2*x1^12*x2^12*x3^9*x4^2*z^7 + x1^12*x2^11*x3^10*x4^2*z^7 - x1^11*x2^11*x3^11*x4^2*z^7 + x1^19*x2^10*x3^3*x4^3*z^7 + x1^17*x2^12*x3^3*x4^3*z^7 + x1^16*x2^13*x3^3*x4^3*z^7 + 2*x1^15*x2^14*x3^3*x4^3*z^7 - x1^18*x2^10*x3^4*x4^3*z^7 - x1^17*x2^11*x3^4*x4^3*z^7 - 3*x1^16*x2^12*x3^4*x4^3*z^7 + x1^14*x2^14*x3^4*x4^3*z^7 + 2*x1^18*x2^9*x3^5*x4^3*z^7 + x1^17*x2^10*x3^5*x4^3*z^7 - x1^15*x2^12*x3^5*x4^3*z^7 - x1^18*x2^8*x3^6*x4^3*z^7 - 4*x1^15*x2^11*x3^6*x4^3*z^7 - x1^14*x2^12*x3^6*x4^3*z^7 + x1^13*x2^13*x3^6*x4^3*z^7 - x1^18*x2^7*x3^7*x4^3*z^7 + x1^17*x2^8*x3^7*x4^3*z^7 + x1^15*x2^10*x3^7*x4^3*z^7 - x1^14*x2^11*x3^7*x4^3*z^7 - x1^13*x2^12*x3^7*x4^3*z^7 - 2*x1^16*x2^8*x3^8*x4^3*z^7 - 2*x1^14*x2^10*x3^8*x4^3*z^7 - x1^13*x2^11*x3^8*x4^3*z^7 + x1^14*x2^9*x3^9*x4^3*z^7 + 2*x1^13*x2^10*x3^9*x4^3*z^7 + x1^11*x2^11*x3^10*x4^3*z^7 - x1^18*x2^9*x3^4*x4^4*z^7 - x1^16*x2^11*x3^4*x4^4*z^7 - x1^15*x2^12*x3^4*x4^4*z^7 - 2*x1^14*x2^13*x3^4*x4^4*z^7 + 2*x1^17*x2^9*x3^5*x4^4*z^7 + 2*x1^16*x2^10*x3^5*x4^4*z^7 + 4*x1^15*x2^11*x3^5*x4^4*z^7 - 2*x1^17*x2^8*x3^6*x4^4*z^7 - x1^16*x2^9*x3^6*x4^4*z^7 - 3*x1^15*x2^10*x3^6*x4^4*z^7 - 2*x1^14*x2^11*x3^6*x4^4*z^7 - 2*x1^13*x2^12*x3^6*x4^4*z^7 + x1^17*x2^7*x3^7*x4^4*z^7 - x1^15*x2^9*x3^7*x4^4*z^7 + 4*x1^14*x2^10*x3^7*x4^4*z^7 + x1^13*x2^11*x3^7*x4^4*z^7 - x1^14*x2^9*x3^8*x4^4*z^7 - 3*x1^13*x2^10*x3^8*x4^4*z^7 - 3*x1^12*x2^11*x3^8*x4^4*z^7 + x1^13*x2^9*x3^9*x4^4*z^7 + x1^12*x2^10*x3^9*x4^4*z^7 - x1^11*x2^10*x3^10*x4^4*z^7 - x1^16*x2^9*x3^5*x4^5*z^7 - x1^15*x2^10*x3^5*x4^5*z^7 + x1^14*x2^11*x3^5*x4^5*z^7 - x1^13*x2^12*x3^5*x4^5*z^7 + 2*x1^15*x2^9*x3^6*x4^5*z^7 - x1^14*x2^10*x3^6*x4^5*z^7 + x1^13*x2^11*x3^6*x4^5*z^7 + x1^13*x2^10*x3^7*x4^5*z^7 + x1^14*x2^8*x3^8*x4^5*z^7 - x1^13*x2^9*x3^8*x4^5*z^7 + 2*x1^12*x2^10*x3^8*x4^5*z^7 + 2*x1^12*x2^9*x3^9*x4^5*z^7 - x1^11*x2^10*x3^9*x4^5*z^7 + x1^10*x2^10*x3^10*x4^5*z^7 + x1^14*x2^9*x3^6*x4^6*z^7 + x1^12*x2^11*x3^6*x4^6*z^7 - 2*x1^14*x2^8*x3^7*x4^6*z^7 - x1^12*x2^10*x3^7*x4^6*z^7 + x1^13*x2^8*x3^8*x4^6*z^7 + 3*x1^11*x2^10*x3^8*x4^6*z^7 - x1^11*x2^9*x3^9*x4^6*z^7 - x1^10*x2^10*x3^9*x4^6*z^7 + x1^12*x2^8*x3^8*x4^7*z^7 - x1^10*x2^10*x3^8*x4^7*z^7 - 2*x1^18*x2^12*z^6 + x1^18*x2^11*x3*z^6 + 2*x1^15*x2^14*x3*z^6 - x1^17*x2^11*x3^2*z^6 - x1^16*x2^12*x3^2*z^6 - x1^15*x2^13*x3^2*z^6 - x1^20*x2^7*x3^3*z^6 - x1^19*x2^8*x3^3*z^6 + x1^18*x2^9*x3^3*z^6 + x1^17*x2^10*x3^3*z^6 + 2*x1^14*x2^13*x3^3*z^6 + x1^19*x2^7*x3^4*z^6 - x1^18*x2^8*x3^4*z^6 + x1^17*x2^9*x3^4*z^6 - x1^16*x2^10*x3^4*z^6 - 2*x1^14*x2^12*x3^4*z^6 - x1^13*x2^13*x3^4*z^6 - x1^18*x2^7*x3^5*z^6 + x1^16*x2^9*x3^5*z^6 + x1^15*x2^10*x3^5*z^6 + x1^14*x2^11*x3^5*z^6 + 2*x1^13*x2^12*x3^5*z^6 + x1^17*x2^7*x3^6*z^6 - x1^15*x2^9*x3^6*z^6 - 2*x1^13*x2^11*x3^6*z^6 - x1^12*x2^12*x3^6*z^6 - x1^16*x2^7*x3^7*z^6 + x1^14*x2^9*x3^7*z^6 + x1^13*x2^10*x3^7*z^6 + 2*x1^12*x2^11*x3^7*z^6 - x1^14*x2^8*x3^8*z^6 - 2*x1^12*x2^10*x3^8*z^6 - x1^12*x2^9*x3^9*z^6 + 2*x1^11*x2^10*x3^9*z^6 - x1^10*x2^10*x3^10*z^6 + 2*x1^17*x2^11*x3*x4*z^6 + x1^14*x2^14*x3*x4*z^6 - x1^17*x2^10*x3^2*x4*z^6 - x1^16*x2^11*x3^2*x4*z^6 - x1^15*x2^12*x3^2*x4*z^6 - 3*x1^14*x2^13*x3^2*x4*z^6 + 4*x1^16*x2^10*x3^3*x4*z^6 + 2*x1^14*x2^12*x3^3*x4*z^6 + 2*x1^13*x2^13*x3^3*x4*z^6 + x1^18*x2^7*x3^4*x4*z^6 - 2*x1^16*x2^9*x3^4*x4*z^6 - x1^14*x2^11*x3^4*x4*z^6 - 5*x1^13*x2^12*x3^4*x4*z^6 - x1^16*x2^8*x3^5*x4*z^6 + 3*x1^15*x2^9*x3^5*x4*z^6 - x1^14*x2^10*x3^5*x4*z^6 + 4*x1^13*x2^11*x3^5*x4*z^6 + 2*x1^12*x2^12*x3^5*x4*z^6 + x1^16*x2^7*x3^6*x4*z^6 - x1^15*x2^8*x3^6*x4*z^6 - x1^13*x2^10*x3^6*x4*z^6 - 6*x1^12*x2^11*x3^6*x4*z^6 - x1^15*x2^7*x3^7*x4*z^6 + x1^14*x2^8*x3^7*x4*z^6 - 2*x1^13*x2^9*x3^7*x4*z^6 + 4*x1^12*x2^10*x3^7*x4*z^6 + 2*x1^11*x2^11*x3^7*x4*z^6 - x1^13*x2^8*x3^8*x4*z^6 + x1^12*x2^9*x3^8*x4*z^6 - 4*x1^11*x2^10*x3^8*x4*z^6 + x1^10*x2^10*x3^9*x4*z^6 + x1^15*x2^10*x3^3*x4^2*z^6 + x1^14*x2^11*x3^3*x4^2*z^6 + x1^13*x2^12*x3^3*x4^2*z^6 - x1^16*x2^8*x3^4*x4^2*z^6 - 3*x1^15*x2^9*x3^4*x4^2*z^6 + x1^14*x2^10*x3^4*x4^2*z^6 - x1^13*x2^11*x3^4*x4^2*z^6 - 2*x1^12*x2^12*x3^4*x4^2*z^6 - x1^17*x2^6*x3^5*x4^2*z^6 - x1^16*x2^7*x3^5*x4^2*z^6 + x1^15*x2^8*x3^5*x4^2*z^6 + x1^14*x2^9*x3^5*x4^2*z^6 - x1^13*x2^10*x3^5*x4^2*z^6 + 4*x1^12*x2^11*x3^5*x4^2*z^6 - x1^15*x2^7*x3^6*x4^2*z^6 - 2*x1^14*x2^8*x3^6*x4^2*z^6 + x1^13*x2^9*x3^6*x4^2*z^6 - x1^12*x2^10*x3^6*x4^2*z^6 - 2*x1^11*x2^11*x3^6*x4^2*z^6 + x1^13*x2^8*x3^7*x4^2*z^6 + 2*x1^11*x2^10*x3^7*x4^2*z^6 - x1^11*x2^9*x3^8*x4^2*z^6 + x1^10*x2^9*x3^9*x4^2*z^6 - x1^17*x2^7*x3^3*x4^3*z^6 - x1^16*x2^8*x3^3*x4^3*z^6 - x1^15*x2^9*x3^3*x4^3*z^6 - x1^14*x2^10*x3^3*x4^3*z^6 - x1^12*x2^12*x3^3*x4^3*z^6 + x1^16*x2^7*x3^4*x4^3*z^6 - x1^15*x2^8*x3^4*x4^3*z^6 + 2*x1^13*x2^10*x3^4*x4^3*z^6 + x1^12*x2^11*x3^4*x4^3*z^6 - x1^15*x2^7*x3^5*x4^3*z^6 - 3*x1^13*x2^9*x3^5*x4^3*z^6 + x1^11*x2^11*x3^5*x4^3*z^6 + x1^15*x2^6*x3^6*x4^3*z^6 + x1^14*x2^7*x3^6*x4^3*z^6 + x1^13*x2^8*x3^6*x4^3*z^6 + 2*x1^12*x2^9*x3^6*x4^3*z^6 - 3*x1^12*x2^8*x3^7*x4^3*z^6 + x1^10*x2^10*x3^7*x4^3*z^6 + x1^11*x2^8*x3^8*x4^3*z^6 - x1^9*x2^9*x3^9*x4^3*z^6 + x1^14*x2^8*x3^4*x4^4*z^6 + x1^13*x2^9*x3^4*x4^4*z^6 + x1^11*x2^11*x3^4*x4^4*z^6 - x1^13*x2^8*x3^5*x4^4*z^6 - 3*x1^12*x2^9*x3^5*x4^4*z^6 - 2*x1^11*x2^10*x3^5*x4^4*z^6 + 2*x1^12*x2^8*x3^6*x4^4*z^6 + x1^11*x2^9*x3^6*x4^4*z^6 + x1^10*x2^10*x3^6*x4^4*z^6 + x1^9*x2^9*x3^8*x4^4*z^6 + x1^13*x2^7*x3^5*x4^5*z^6 + x1^11*x2^9*x3^5*x4^5*z^6 - x1^12*x2^7*x3^6*x4^5*z^6 - x1^10*x2^9*x3^6*x4^5*z^6 - x1^9*x2^8*x3^8*x4^5*z^6 + x1^9*x2^8*x3^7*x4^6*z^6 - x1^8*x2^8*x3^8*x4^6*z^6 + x1^15*x2^10*z^5 + x1^14*x2^11*z^5 + x1^13*x2^12*z^5 - 2*x1^15*x2^9*x3*z^5 - x1^12*x2^12*x3*z^5 + x1^15*x2^8*x3^2*z^5 + 2*x1^12*x2^11*x3^2*z^5 + x1^17*x2^5*x3^3*z^5 + x1^16*x2^6*x3^3*z^5 - 2*x1^14*x2^8*x3^3*z^5 - 2*x1^12*x2^10*x3^3*z^5 - x1^16*x2^5*x3^4*z^5 + x1^14*x2^7*x3^4*z^5 + 2*x1^11*x2^10*x3^4*z^5 + x1^14*x2^6*x3^5*z^5 - x1^13*x2^7*x3^5*z^5 - 2*x1^11*x2^9*x3^5*z^5 - x1^10*x2^10*x3^5*z^5 + x1^11*x2^8*x3^6*z^5 + 2*x1^10*x2^9*x3^6*z^5 + x1^11*x2^7*x3^7*z^5 - x1^10*x2^8*x3^7*z^5 - x1^9*x2^9*x3^7*z^5 + x1^9*x2^8*x3^8*z^5 - x1^14*x2^9*x3*x4*z^5 - x1^13*x2^10*x3*x4*z^5 - x1^12*x2^11*x3*x4*z^5 + 3*x1^14*x2^8*x3^2*x4*z^5 - x1^13*x2^9*x3^2*x4*z^5 + x1^12*x2^10*x3^2*x4*z^5 + 2*x1^11*x2^11*x3^2*x4*z^5 - x1^14*x2^7*x3^3*x4*z^5 - x1^13*x2^8*x3^3*x4*z^5 - 4*x1^11*x2^10*x3^3*x4*z^5 + 3*x1^13*x2^7*x3^4*x4*z^5 - 3*x1^12*x2^8*x3^4*x4*z^5 + 2*x1^11*x2^9*x3^4*x4*z^5 + 2*x1^10*x2^10*x3^4*x4*z^5 - x1^13*x2^6*x3^5*x4*z^5 - 4*x1^10*x2^9*x3^5*x4*z^5 + x1^12*x2^6*x3^6*x4*z^5 - 2*x1^11*x2^7*x3^6*x4*z^5 + 2*x1^10*x2^8*x3^6*x4*z^5 + 2*x1^9*x2^9*x3^6*x4*z^5 + x1^10*x2^7*x3^7*x4*z^5 - 2*x1^9*x2^8*x3^7*x4*z^5 + x1^8*x2^8*x3^8*x4*z^5 - x1^13*x2^7*x3^3*x4^2*z^5 + x1^12*x2^8*x3^3*x4^2*z^5 - x1^11*x2^9*x3^3*x4^2*z^5 - x1^10*x2^10*x3^3*x4^2*z^5 + 2*x1^12*x2^7*x3^4*x4^2*z^5 + 2*x1^10*x2^9*x3^4*x4^2*z^5 + x1^13*x2^5*x3^5*x4^2*z^5 - x1^12*x2^6*x3^5*x4^2*z^5 + x1^11*x2^7*x3^5*x4^2*z^5 - 2*x1^9*x2^9*x3^5*x4^2*z^5 + x1^11*x2^6*x3^6*x4^2*z^5 + x1^10*x2^7*x3^6*x4^2*z^5 + x1^9*x2^8*x3^6*x4^2*z^5 - x1^8*x2^8*x3^7*x4^2*z^5 + x1^14*x2^5*x3^3*x4^3*z^5 + 2*x1^12*x2^7*x3^3*x4^3*z^5 + x1^11*x2^8*x3^3*x4^3*z^5 + x1^10*x2^9*x3^3*x4^3*z^5 - x1^13*x2^5*x3^4*x4^3*z^5 + x1^10*x2^8*x3^4*x4^3*z^5 - x1^9*x2^9*x3^4*x4^3*z^5 + x1^10*x2^7*x3^5*x4^3*z^5 + x1^9*x2^8*x3^5*x4^3*z^5 - x1^10*x2^6*x3^6*x4^3*z^5 - x1^9*x2^7*x3^6*x4^3*z^5 + x1^8*x2^8*x3^6*x4^3*z^5 + x1^8*x2^7*x3^7*x4^3*z^5 + x1^12*x2^5*x3^4*x4^4*z^5 - x1^10*x2^7*x3^4*x4^4*z^5 - x1^9*x2^8*x3^4*x4^4*z^5 - x1^10*x2^6*x3^5*x4^4*z^5 - x1^9*x2^7*x3^5*x4^4*z^5 + 2*x1^8*x2^8*x3^5*x4^4*z^5 - x1^8*x2^7*x3^6*x4^4*z^5 - x1^7*x2^7*x3^7*x4^4*z^5 + x1^9*x2^6*x3^5*x4^5*z^5 - x1^8*x2^7*x3^5*x4^5*z^5 + x1^8*x2^6*x3^6*x4^5*z^5 - x1^13*x2^7*z^4 + x1^12*x2^8*z^4 - x1^11*x2^9*z^4 - x1^10*x2^10*z^4 + x1^12*x2^7*x3*z^4 + x1^11*x2^8*x3*z^4 + x1^10*x2^9*x3*z^4 - 2*x1^12*x2^6*x3^2*z^4 - x1^9*x2^9*x3^2*z^4 - x1^13*x2^4*x3^3*z^4 + x1^10*x2^7*x3^3*z^4 + 2*x1^9*x2^8*x3^3*z^4 + x1^12*x2^4*x3^4*z^4 + x1^10*x2^6*x3^4*z^4 - 2*x1^9*x2^7*x3^4*z^4 - x1^9*x2^6*x3^5*z^4 + 2*x1^8*x2^7*x3^5*z^4 - x1^7*x2^7*x3^6*z^4 + x1^12*x2^6*x3*x4*z^4 - x1^11*x2^7*x3*x4*z^4 + x1^10*x2^8*x3*x4*z^4 + x1^9*x2^9*x3*x4*z^4 - x1^11*x2^6*x3^2*x4*z^4 - 2*x1^9*x2^8*x3^2*x4*z^4 + x1^12*x2^4*x3^3*x4*z^4 + 3*x1^11*x2^5*x3^3*x4*z^4 + x1^9*x2^7*x3^3*x4*z^4 + 2*x1^8*x2^8*x3^3*x4*z^4 - x1^10*x2^5*x3^4*x4*z^4 + x1^9*x2^6*x3^4*x4*z^4 - 2*x1^8*x2^7*x3^4*x4*z^4 - x1^9*x2^5*x3^5*x4*z^4 + x1^8*x2^6*x3^5*x4*z^4 + x1^7*x2^7*x3^5*x4*z^4 - x1^7*x2^6*x3^6*x4*z^4 - x1^11*x2^5*x3^2*x4^2*z^4 - x1^9*x2^6*x3^3*x4^2*z^4 + x1^8*x2^7*x3^3*x4^2*z^4 - x1^9*x2^5*x3^4*x4^2*z^4 - x1^8*x2^6*x3^4*x4^2*z^4 - x1^7*x2^7*x3^4*x4^2*z^4 - x1^8*x2^5*x3^5*x4^2*z^4 + x1^7*x2^6*x3^5*x4^2*z^4 - x1^6*x2^6*x3^6*x4^2*z^4 - x1^8*x2^6*x3^3*x4^3*z^4 - x1^7*x2^7*x3^3*x4^3*z^4 + x1^9*x2^4*x3^4*x4^3*z^4 + x1^8*x2^5*x3^4*x4^3*z^4 - x1^7*x2^6*x3^4*x4^3*z^4 + x1^7*x2^5*x3^5*x4^3*z^4 - x1^6*x2^6*x3^5*x4^3*z^4 - x1^8*x2^4*x3^4*x4^4*z^4 + x1^6*x2^6*x3^4*x4^4*z^4 + x1^6*x2^5*x3^5*x4^4*z^4 + x1^11*x2^4*z^3 + x1^10*x2^5*z^3 - x1^9*x2^6*z^3 + x1^8*x2^7*z^3 - x1^10*x2^4*x3*z^3 + x1^9*x2^5*x3*z^3 - x1^8*x2^6*x3*z^3 - x1^7*x2^7*x3*z^3 - x1^10*x2^3*x3^2*z^3 + x1^8*x2^5*x3^2*z^3 + x1^7*x2^6*x3^2*z^3 + 2*x1^8*x2^4*x3^3*z^3 - x1^7*x2^5*x3^3*z^3 - x1^6*x2^6*x3^3*z^3 - x1^7*x2^4*x3^4*z^3 + x1^6*x2^5*x3^4*z^3 - x1^9*x2^4*x3*x4*z^3 + x1^8*x2^5*x3*x4*z^3 - x1^7*x2^6*x3*x4*z^3 + x1^9*x2^3*x3^2*x4*z^3 - x1^8*x2^4*x3^2*x4*z^3 + x1^7*x2^5*x3^2*x4*z^3 + x1^6*x2^6*x3^2*x4*z^3 - x1^8*x2^3*x3^3*x4*z^3 - x1^7*x2^4*x3^3*x4*z^3 - 3*x1^6*x2^5*x3^3*x4*z^3 + x1^5*x2^5*x3^4*x4*z^3 + x1^8*x2^3*x3^2*x4^2*z^3 + x1^7*x2^4*x3^2*x4^2*z^3 + x1^5*x2^4*x3^4*x4^2*z^3 - x1^6*x2^3*x3^3*x4^3*z^3 + x1^5*x2^4*x3^3*x4^3*z^3 - x1^8*x2^2*z^2 - x1^7*x2^3*z^2 - x1^6*x2^4*z^2 - x1^6*x2^3*x3*z^2 + x1^5*x2^4*x3*z^2 + x1^6*x2^2*x3^2*z^2 - x1^4*x2^4*x3^2*z^2 + x1^6*x2^2*x3*x4*z^2 + x1^5*x2^3*x3*x4*z^2 + x1^5*x2^2*x3^2*x4*z^2 - x1^4*x2^3*x3^2*x4*z^2 + 2*x1^3*x2^3*x3^3*x4*z^2 - x1^4*x2^2*x3^2*x4^2*z^2 + 2*x1^3*x2^2*z - x1^2*x2*x3*x4*z - 1)) == ((-1) * (x1^82*x2^65*x3^41*x4^17*z^41 + x1^80*x2^64*x3^40*x4^16*z^40 - 2*x1^79*x2^63*x3^41*x4^17*z^40 + x1^78*x2^63*x3^39*x4^15*z^39 - 2*x1^79*x2^62*x3^38*x4^16*z^39 + x1^78*x2^62*x3^39*x4^16*z^39 - x1^77*x2^63*x3^39*x4^16*z^39 - x1^77*x2^62*x3^40*x4^16*z^39 - x1^76*x2^63*x3^40*x4^16*z^39 + x1^78*x2^61*x3^39*x4^17*z^39 - x1^76*x2^63*x3^39*x4^17*z^39 - x1^77*x2^61*x3^40*x4^17*z^39 + x1^76*x2^62*x3^40*x4^17*z^39 + x1^76*x2^61*x3^41*x4^17*z^39 + x1^75*x2^62*x3^41*x4^17*z^39 + x1^74*x2^63*x3^41*x4^17*z^39 - x1^77*x2^61*x3^38*x4^14*z^38 + x1^76*x2^62*x3^38*x4^14*z^38 - x1^77*x2^61*x3^37*x4^15*z^38 - x1^75*x2^61*x3^39*x4^15*z^38 - x1^74*x2^62*x3^39*x4^15*z^38 - x1^77*x2^60*x3^37*x4^16*z^38 + 3*x1^76*x2^60*x3^38*x4^16*z^38 + x1^75*x2^61*x3^38*x4^16*z^38 + x1^74*x2^62*x3^38*x4^16*z^38 - x1^76*x2^59*x3^39*x4^16*z^38 - x1^75*x2^60*x3^39*x4^16*z^38 + x1^74*x2^61*x3^39*x4^16*z^38 - x1^73*x2^62*x3^39*x4^16*z^38 + x1^75*x2^59*x3^40*x4^16*z^38 - x1^74*x2^60*x3^40*x4^16*z^38 + x1^73*x2^61*x3^40*x4^16*z^38 - x1^76*x2^60*x3^37*x4^17*z^38 + x1^75*x2^61*x3^37*x4^17*z^38 + x1^76*x2^59*x3^38*x4^17*z^38 + x1^75*x2^60*x3^38*x4^17*z^38 - 2*x1^74*x2^61*x3^38*x4^17*z^38 - x1^75*x2^59*x3^39*x4^17*z^38 - x1^74*x2^60*x3^39*x4^17*z^38 + x1^72*x2^62*x3^39*x4^17*z^38 + x1^75*x2^58*x3^40*x4^17*z^38 + x1^74*x2^59*x3^40*x4^17*z^38 - x1^73*x2^60*x3^40*x4^17*z^38 + x1^72*x2^61*x3^40*x4^17*z^38 - x1^74*x2^58*x3^41*x4^17*z^38 + x1^73*x2^59*x3^41*x4^17*z^38 - x1^72*x2^60*x3^41*x4^17*z^38 - x1^71*x2^61*x3^41*x4^17*z^38 - x1^76*x2^60*x3^36*x4^13*z^37 - x1^76*x2^59*x3^37*x4^13*z^37 + x1^74*x2^61*x3^37*x4^13*z^37 + x1^76*x2^59*x3^36*x4^14*z^37 - x1^75*x2^60*x3^36*x4^14*z^37 + x1^75*x2^59*x3^37*x4^14*z^37 - x1^74*x2^60*x3^37*x4^14*z^37 - x1^73*x2^61*x3^37*x4^14*z^37 + x1^75*x2^58*x3^38*x4^14*z^37 + x1^74*x2^59*x3^38*x4^14*z^37 + x1^76*x2^59*x3^35*x4^15*z^37 - x1^75*x2^59*x3^36*x4^15*z^37 + x1^74*x2^60*x3^36*x4^15*z^37 + x1^75*x2^58*x3^37*x4^15*z^37 + x1^74*x2^59*x3^37*x4^15*z^37 + x1^73*x2^60*x3^37*x4^15*z^37 - x1^74*x2^58*x3^38*x4^15*z^37 + x1^73*x2^59*x3^38*x4^15*z^37 + x1^71*x2^60*x3^39*x4^15*z^37 + x1^75*x2^59*x3^35*x4^16*z^37 - x1^75*x2^58*x3^36*x4^16*z^37 - x1^74*x2^59*x3^36*x4^16*z^37 + x1^73*x2^60*x3^36*x4^16*z^37 + 2*x1^74*x2^58*x3^37*x4^16*z^37 - x1^73*x2^59*x3^37*x4^16*z^37 + x1^72*x2^60*x3^37*x4^16*z^37 - 2*x1^74*x2^57*x3^38*x4^16*z^37 - x1^73*x2^58*x3^38*x4^16*z^37 - 3*x1^71*x2^60*x3^38*x4^16*z^37 - x1^70*x2^61*x3^38*x4^16*z^37 + 2*x1^73*x2^57*x3^39*x4^16*z^37 + x1^71*x2^59*x3^39*x4^16*z^37 - x1^73*x2^56*x3^40*x4^16*z^37 - x1^72*x2^57*x3^40*x4^16*z^37 + x1^71*x2^58*x3^40*x4^16*z^37 - x1^70*x2^59*x3^40*x4^16*z^37 + x1^75*x2^58*x3^35*x4^17*z^37 - 2*x1^74*x2^58*x3^36*x4^17*z^37 + x1^73*x2^59*x3^36*x4^17*z^37 + 2*x1^73*x2^58*x3^37*x4^17*z^37 - x1^72*x2^59*x3^37*x4^17*z^37 - x1^70*x2^61*x3^37*x4^17*z^37 - 2*x1^73*x2^57*x3^38*x4^17*z^37 - x1^72*x2^58*x3^38*x4^17*z^37 + x1^69*x2^61*x3^38*x4^17*z^37 + x1^73*x2^56*x3^39*x4^17*z^37 + 2*x1^70*x2^59*x3^39*x4^17*z^37 - x1^72*x2^56*x3^40*x4^17*z^37 - x1^71*x2^57*x3^40*x4^17*z^37 - x1^70*x2^58*x3^40*x4^17*z^37 + x1^72*x2^55*x3^41*x4^17*z^37 + x1^71*x2^56*x3^41*x4^17*z^37 - x1^70*x2^57*x3^41*x4^17*z^37 + x1^69*x2^58*x3^41*x4^17*z^37 - x1^74*x2^59*x3^35*x4^12*z^36 + x1^74*x2^58*x3^36*x4^12*z^36 - x1^73*x2^59*x3^36*x4^12*z^36 + x1^75*x2^58*x3^34*x4^13*z^36 + x1^74*x2^58*x3^35*x4^13*z^36 - 2*x1^74*x2^57*x3^36*x4^13*z^36 + x1^73*x2^58*x3^36*x4^13*z^36 + x1^72*x2^59*x3^36*x4^13*z^36 + x1^73*x2^57*x3^37*x4^13*z^36 + x1^72*x2^58*x3^37*x4^13*z^36 - x1^70*x2^60*x3^37*x4^13*z^36 - x1^74*x2^58*x3^34*x4^14*z^36 - x1^74*x2^57*x3^35*x4^14*z^36 + x1^73*x2^58*x3^35*x4^14*z^36 + x1^72*x2^59*x3^35*x4^14*z^36 - x1^73*x2^57*x3^36*x4^14*z^36 - x1^72*x2^58*x3^36*x4^14*z^36 + x1^73*x2^56*x3^37*x4^14*z^36 - x1^72*x2^57*x3^37*x4^14*z^36 + x1^69*x2^60*x3^37*x4^14*z^36 - x1^72*x2^56*x3^38*x4^14*z^36 - x1^71*x2^57*x3^38*x4^14*z^36 - 2*x1^70*x2^58*x3^38*x4^14*z^36 - x1^68*x2^60*x3^38*x4^14*z^36 + x1^74*x2^57*x3^34*x4^15*z^36 - x1^73*x2^57*x3^35*x4^15*z^36 - x1^72*x2^58*x3^35*x4^15*z^36 - x1^71*x2^59*x3^35*x4^15*z^36 + 2*x1^73*x2^56*x3^36*x4^15*z^36 - x1^71*x2^58*x3^36*x4^15*z^36 + x1^70*x2^59*x3^36*x4^15*z^36 - x1^69*x2^60*x3^36*x4^15*z^36 - 2*x1^72*x2^56*x3^37*x4^15*z^36 - 2*x1^70*x2^58*x3^37*x4^15*z^36 + x1^72*x2^55*x3^38*x4^15*z^36 + x1^71*x2^56*x3^38*x4^15*z^36 - x1^70*x2^57*x3^38*x4^15*z^36 + x1^69*x2^58*x3^38*x4^15*z^36 - x1^74*x2^57*x3^33*x4^16*z^36 + 2*x1^73*x2^57*x3^34*x4^16*z^36 - x1^72*x2^58*x3^34*x4^16*z^36 - 2*x1^73*x2^56*x3^35*x4^16*z^36 - 2*x1^72*x2^57*x3^35*x4^16*z^36 + 2*x1^71*x2^58*x3^35*x4^16*z^36 - x1^70*x2^59*x3^35*x4^16*z^36 + 4*x1^72*x2^56*x3^36*x4^16*z^36 + x1^69*x2^59*x3^36*x4^16*z^36 - 2*x1^72*x2^55*x3^37*x4^16*z^36 - 2*x1^71*x2^56*x3^37*x4^16*z^36 + 3*x1^70*x2^57*x3^37*x4^16*z^36 - 3*x1^69*x2^58*x3^37*x4^16*z^36 + 4*x1^71*x2^55*x3^38*x4^16*z^36 + x1^69*x2^57*x3^38*x4^16*z^36 + x1^68*x2^58*x3^38*x4^16*z^36 - 2*x1^71*x2^54*x3^39*x4^16*z^36 - x1^70*x2^55*x3^39*x4^16*z^36 + x1^69*x2^56*x3^39*x4^16*z^36 - 3*x1^68*x2^57*x3^39*x4^16*z^36 + x1^70*x2^54*x3^40*x4^16*z^36 + x1^69*x2^55*x3^40*x4^16*z^36 + x1^68*x2^56*x3^40*x4^16*z^36 - x1^73*x2^57*x3^33*x4^17*z^36 + x1^73*x2^56*x3^34*x4^17*z^36 + x1^72*x2^57*x3^34*x4^17*z^36 - x1^71*x2^58*x3^34*x4^17*z^36 - 2*x1^72*x2^56*x3^35*x4^17*z^36 - x1^71*x2^57*x3^35*x4^17*z^36 + x1^72*x2^55*x3^36*x4^17*z^36 + 2*x1^71*x2^56*x3^36*x4^17*z^36 + x1^69*x2^58*x3^36*x4^17*z^36 - x1^68*x2^59*x3^36*x4^17*z^36 - 2*x1^71*x2^55*x3^37*x4^17*z^36 - x1^68*x2^58*x3^37*x4^17*z^36 + x1^66*x2^60*x3^37*x4^17*z^36 + 2*x1^70*x2^55*x3^38*x4^17*z^36 + 2*x1^68*x2^57*x3^38*x4^17*z^36 - x1^66*x2^59*x3^38*x4^17*z^36 - x1^65*x2^60*x3^38*x4^17*z^36 - 2*x1^70*x2^54*x3^39*x4^17*z^36 - x1^67*x2^57*x3^39*x4^17*z^36 + x1^70*x2^53*x3^40*x4^17*z^36 + 2*x1^67*x2^56*x3^40*x4^17*z^36 - x1^69*x2^53*x3^41*x4^17*z^36 - x1^68*x2^54*x3^41*x4^17*z^36 - x1^67*x2^55*x3^41*x4^17*z^36 + x1^74*x2^57*x3^33*x4^11*z^35 - x1^73*x2^57*x3^34*x4^11*z^35 + x1^73*x2^57*x3^33*x4^12*z^35 + x1^72*x2^56*x3^35*x4^12*z^35 + x1^70*x2^58*x3^35*x4^12*z^35 - x1^71*x2^56*x3^36*x4^12*z^35 - x1^69*x2^58*x3^36*x4^12*z^35 - x1^73*x2^56*x3^33*x4^13*z^35 - x1^72*x2^55*x3^35*x4^13*z^35 - x1^71*x2^56*x3^35*x4^13*z^35 - 2*x1^70*x2^57*x3^35*x4^13*z^35 + 2*x1^71*x2^55*x3^36*x4^13*z^35 + 3*x1^70*x2^56*x3^36*x4^13*z^35 + x1^69*x2^57*x3^36*x4^13*z^35 - x1^71*x2^54*x3^37*x4^13*z^35 - x1^69*x2^56*x3^37*x4^13*z^35 - x1^68*x2^57*x3^37*x4^13*z^35 + x1^73*x2^56*x3^32*x4^14*z^35 - x1^71*x2^57*x3^33*x4^14*z^35 - x1^72*x2^55*x3^34*x4^14*z^35 + 3*x1^70*x2^57*x3^34*x4^14*z^35 - 2*x1^70*x2^56*x3^35*x4^14*z^35 - x1^69*x2^57*x3^35*x4^14*z^35 - x1^68*x2^58*x3^35*x4^14*z^35 - x1^67*x2^59*x3^35*x4^14*z^35 - x1^71*x2^54*x3^36*x4^14*z^35 + 3*x1^69*x2^56*x3^36*x4^14*z^35 + x1^67*x2^58*x3^36*x4^14*z^35 - x1^70*x2^54*x3^37*x4^14*z^35 - 2*x1^69*x2^55*x3^37*x4^14*z^35 + x1^67*x2^57*x3^37*x4^14*z^35 - x1^66*x2^58*x3^37*x4^14*z^35 + x1^70*x2^53*x3^38*x4^14*z^35 + x1^68*x2^55*x3^38*x4^14*z^35 + x1^67*x2^56*x3^38*x4^14*z^35 + x1^66*x2^57*x3^38*x4^14*z^35 + x1^65*x2^58*x3^38*x4^14*z^35 - x1^72*x2^56*x3^32*x4^15*z^35 + x1^71*x2^56*x3^33*x4^15*z^35 - 2*x1^71*x2^55*x3^34*x4^15*z^35 - x1^69*x2^57*x3^34*x4^15*z^35 + 2*x1^71*x2^54*x3^35*x4^15*z^35 + x1^70*x2^55*x3^35*x4^15*z^35 - x1^69*x2^56*x3^35*x4^15*z^35 + 2*x1^68*x2^57*x3^35*x4^15*z^35 + x1^67*x2^58*x3^35*x4^15*z^35 - 4*x1^70*x2^54*x3^36*x4^15*z^35 + x1^69*x2^55*x3^36*x4^15*z^35 - x1^68*x2^56*x3^36*x4^15*z^35 - x1^67*x2^57*x3^36*x4^15*z^35 + x1^66*x2^58*x3^36*x4^15*z^35 + x1^65*x2^59*x3^36*x4^15*z^35 + 2*x1^70*x2^53*x3^37*x4^15*z^35 + x1^69*x2^54*x3^37*x4^15*z^35 - x1^68*x2^55*x3^37*x4^15*z^35 + 3*x1^67*x2^56*x3^37*x4^15*z^35 + x1^66*x2^57*x3^37*x4^15*z^35 - x1^69*x2^53*x3^38*x4^15*z^35 - x1^68*x2^54*x3^38*x4^15*z^35 - x1^67*x2^55*x3^38*x4^15*z^35 - x1^72*x2^55*x3^32*x4^16*z^35 + 4*x1^71*x2^55*x3^33*x4^16*z^35 - x1^70*x2^56*x3^33*x4^16*z^35 + x1^69*x2^57*x3^33*x4^16*z^35 - 2*x1^71*x2^54*x3^34*x4^16*z^35 - 4*x1^70*x2^55*x3^34*x4^16*z^35 + 2*x1^69*x2^56*x3^34*x4^16*z^35 - x1^68*x2^57*x3^34*x4^16*z^35 + x1^67*x2^58*x3^34*x4^16*z^35 + 6*x1^70*x2^54*x3^35*x4^16*z^35 + x1^69*x2^55*x3^35*x4^16*z^35 + x1^67*x2^57*x3^35*x4^16*z^35 - x1^66*x2^58*x3^35*x4^16*z^35 - 2*x1^70*x2^53*x3^36*x4^16*z^35 - 4*x1^69*x2^54*x3^36*x4^16*z^35 + x1^68*x2^55*x3^36*x4^16*z^35 - 3*x1^67*x2^56*x3^36*x4^16*z^35 + x1^66*x2^57*x3^36*x4^16*z^35 + 5*x1^69*x2^53*x3^37*x4^16*z^35 + x1^68*x2^54*x3^37*x4^16*z^35 + 2*x1^66*x2^56*x3^37*x4^16*z^35 - x1^64*x2^58*x3^37*x4^16*z^35 - 2*x1^69*x2^52*x3^38*x4^16*z^35 - 2*x1^68*x2^53*x3^38*x4^16*z^35 - 4*x1^66*x2^55*x3^38*x4^16*z^35 + 3*x1^68*x2^52*x3^39*x4^16*z^35 + x1^67*x2^53*x3^39*x4^16*z^35 + x1^66*x2^54*x3^39*x4^16*z^35 + x1^65*x2^55*x3^39*x4^16*z^35 - x1^68*x2^51*x3^40*x4^16*z^35 - 2*x1^65*x2^54*x3^40*x4^16*z^35 + x1^72*x2^55*x3^31*x4^17*z^35 - 2*x1^71*x2^55*x3^32*x4^17*z^35 + x1^70*x2^56*x3^32*x4^17*z^35 + 2*x1^70*x2^55*x3^33*x4^17*z^35 + x1^68*x2^57*x3^33*x4^17*z^35 - 2*x1^70*x2^54*x3^34*x4^17*z^35 - x1^69*x2^55*x3^34*x4^17*z^35 - x1^68*x2^56*x3^34*x4^17*z^35 + x1^66*x2^58*x3^34*x4^17*z^35 + x1^70*x2^53*x3^35*x4^17*z^35 + 2*x1^69*x2^54*x3^35*x4^17*z^35 + x1^67*x2^56*x3^35*x4^17*z^35 - x1^65*x2^58*x3^35*x4^17*z^35 - 2*x1^69*x2^53*x3^36*x4^17*z^35 - x1^68*x2^54*x3^36*x4^17*z^35 - x1^67*x2^55*x3^36*x4^17*z^35 - x1^66*x2^56*x3^36*x4^17*z^35 + x1^64*x2^58*x3^36*x4^17*z^35 + x1^69*x2^52*x3^37*x4^17*z^35 + 2*x1^68*x2^53*x3^37*x4^17*z^35 + x1^66*x2^55*x3^37*x4^17*z^35 - x1^65*x2^56*x3^37*x4^17*z^35 + x1^64*x2^57*x3^37*x4^17*z^35 - x1^63*x2^58*x3^37*x4^17*z^35 - 2*x1^68*x2^52*x3^38*x4^17*z^35 - x1^65*x2^55*x3^38*x4^17*z^35 - x1^64*x2^56*x3^38*x4^17*z^35 + x1^63*x2^57*x3^38*x4^17*z^35 + x1^62*x2^58*x3^38*x4^17*z^35 + x1^67*x2^52*x3^39*x4^17*z^35 + x1^66*x2^53*x3^39*x4^17*z^35 + x1^65*x2^54*x3^39*x4^17*z^35 - 2*x1^67*x2^51*x3^40*x4^17*z^35 - x1^64*x2^54*x3^40*x4^17*z^35 + 2*x1^64*x2^53*x3^41*x4^17*z^35 + x1^72*x2^55*x3^33*x4^10*z^34 - x1^70*x2^57*x3^33*x4^10*z^34 + x1^72*x2^55*x3^32*x4^11*z^34 + x1^71*x2^56*x3^32*x4^11*z^34 - 3*x1^71*x2^55*x3^33*x4^11*z^34 - x1^69*x2^57*x3^33*x4^11*z^34 + x1^70*x2^55*x3^34*x4^11*z^34 + 2*x1^68*x2^57*x3^34*x4^11*z^34 - x1^70*x2^54*x3^35*x4^11*z^34 - x1^68*x2^56*x3^35*x4^11*z^34 - x1^72*x2^55*x3^31*x4^12*z^34 + x1^71*x2^55*x3^32*x4^12*z^34 - 2*x1^70*x2^56*x3^32*x4^12*z^34 - 2*x1^70*x2^55*x3^33*x4^12*z^34 + x1^69*x2^56*x3^33*x4^12*z^34 - x1^68*x2^57*x3^33*x4^12*z^34 - x1^69*x2^55*x3^34*x4^12*z^34 - x1^69*x2^54*x3^35*x4^12*z^34 + x1^68*x2^55*x3^35*x4^12*z^34 - 2*x1^67*x2^56*x3^35*x4^12*z^34 + x1^69*x2^53*x3^36*x4^12*z^34 - x1^68*x2^54*x3^36*x4^12*z^34 + x1^67*x2^55*x3^36*x4^12*z^34 + x1^66*x2^56*x3^36*x4^12*z^34 + x1^71*x2^55*x3^31*x4^13*z^34 - x1^70*x2^55*x3^32*x4^13*z^34 - x1^69*x2^56*x3^32*x4^13*z^34 + 3*x1^70*x2^54*x3^33*x4^13*z^34 + 3*x1^69*x2^55*x3^33*x4^13*z^34 + x1^68*x2^56*x3^33*x4^13*z^34 - x1^69*x2^54*x3^34*x4^13*z^34 - 4*x1^68*x2^55*x3^34*x4^13*z^34 + x1^67*x2^56*x3^34*x4^13*z^34 - x1^65*x2^58*x3^34*x4^13*z^34 + 2*x1^69*x2^53*x3^35*x4^13*z^34 + 2*x1^68*x2^54*x3^35*x4^13*z^34 + 3*x1^67*x2^55*x3^35*x4^13*z^34 + x1^66*x2^56*x3^35*x4^13*z^34 + 2*x1^65*x2^57*x3^35*x4^13*z^34 - 4*x1^67*x2^54*x3^36*x4^13*z^34 - 2*x1^66*x2^55*x3^36*x4^13*z^34 - 2*x1^65*x2^56*x3^36*x4^13*z^34 + 2*x1^68*x2^52*x3^37*x4^13*z^34 + x1^67*x2^53*x3^37*x4^13*z^34 + x1^66*x2^54*x3^37*x4^13*z^34 + x1^64*x2^56*x3^37*x4^13*z^34 - x1^71*x2^54*x3^31*x4^14*z^34 - 2*x1^69*x2^55*x3^32*x4^14*z^34 - x1^68*x2^56*x3^32*x4^14*z^34 + x1^69*x2^54*x3^33*x4^14*z^34 + 2*x1^68*x2^55*x3^33*x4^14*z^34 + 2*x1^66*x2^57*x3^33*x4^14*z^34 + x1^69*x2^53*x3^34*x4^14*z^34 + x1^68*x2^54*x3^34*x4^14*z^34 - x1^67*x2^55*x3^34*x4^14*z^34 - x1^65*x2^57*x3^34*x4^14*z^34 + x1^64*x2^58*x3^34*x4^14*z^34 - x1^69*x2^52*x3^35*x4^14*z^34 + x1^68*x2^53*x3^35*x4^14*z^34 + 4*x1^67*x2^54*x3^35*x4^14*z^34 + x1^64*x2^57*x3^35*x4^14*z^34 + x1^67*x2^53*x3^36*x4^14*z^34 - x1^65*x2^55*x3^36*x4^14*z^34 - 2*x1^64*x2^56*x3^36*x4^14*z^34 - x1^68*x2^51*x3^37*x4^14*z^34 + 3*x1^66*x2^53*x3^37*x4^14*z^34 + x1^65*x2^54*x3^37*x4^14*z^34 + x1^64*x2^55*x3^37*x4^14*z^34 - 2*x1^67*x2^51*x3^38*x4^14*z^34 - x1^66*x2^52*x3^38*x4^14*z^34 - x1^65*x2^53*x3^38*x4^14*z^34 - x1^63*x2^55*x3^38*x4^14*z^34 + x1^71*x2^54*x3^30*x4^15*z^34 - x1^70*x2^54*x3^31*x4^15*z^34 + 2*x1^70*x2^53*x3^32*x4^15*z^34 + x1^69*x2^54*x3^32*x4^15*z^34 + x1^67*x2^56*x3^32*x4^15*z^34 - 4*x1^69*x2^53*x3^33*x4^15*z^34 + x1^68*x2^54*x3^33*x4^15*z^34 - 2*x1^66*x2^56*x3^33*x4^15*z^34 + 2*x1^69*x2^52*x3^34*x4^15*z^34 + 2*x1^68*x2^53*x3^34*x4^15*z^34 - 2*x1^67*x2^54*x3^34*x4^15*z^34 + 3*x1^66*x2^55*x3^34*x4^15*z^34 - 5*x1^68*x2^52*x3^35*x4^15*z^34 - x1^67*x2^53*x3^35*x4^15*z^34 - 2*x1^65*x2^55*x3^35*x4^15*z^34 + 2*x1^68*x2^51*x3^36*x4^15*z^34 + 2*x1^67*x2^52*x3^36*x4^15*z^34 + 3*x1^65*x2^54*x3^36*x4^15*z^34 + x1^64*x2^55*x3^36*x4^15*z^34 + x1^63*x2^56*x3^36*x4^15*z^34 - x1^62*x2^57*x3^36*x4^15*z^34 - 3*x1^67*x2^51*x3^37*x4^15*z^34 - x1^66*x2^52*x3^37*x4^15*z^34 - x1^65*x2^53*x3^37*x4^15*z^34 - x1^64*x2^54*x3^37*x4^15*z^34 + x1^67*x2^50*x3^38*x4^15*z^34 + 2*x1^64*x2^53*x3^38*x4^15*z^34 + x1^70*x2^54*x3^30*x4^16*z^34 - 2*x1^70*x2^53*x3^31*x4^16*z^34 - x1^69*x2^54*x3^31*x4^16*z^34 + 2*x1^68*x2^55*x3^31*x4^16*z^34 + 5*x1^69*x2^53*x3^32*x4^16*z^34 - x1^67*x2^55*x3^32*x4^16*z^34 - 2*x1^69*x2^52*x3^33*x4^16*z^34 - 5*x1^68*x2^53*x3^33*x4^16*z^34 + x1^67*x2^54*x3^33*x4^16*z^34 - 2*x1^66*x2^55*x3^33*x4^16*z^34 + x1^65*x2^56*x3^33*x4^16*z^34 + 6*x1^68*x2^52*x3^34*x4^16*z^34 + 2*x1^67*x2^53*x3^34*x4^16*z^34 + x1^66*x2^54*x3^34*x4^16*z^34 + 2*x1^65*x2^55*x3^34*x4^16*z^34 - x1^64*x2^56*x3^34*x4^16*z^34 - x1^63*x2^57*x3^34*x4^16*z^34 - 2*x1^68*x2^51*x3^35*x4^16*z^34 - 6*x1^67*x2^52*x3^35*x4^16*z^34 - 4*x1^65*x2^54*x3^35*x4^16*z^34 + 2*x1^63*x2^56*x3^35*x4^16*z^34 + 2*x1^62*x2^57*x3^35*x4^16*z^34 + 6*x1^67*x2^51*x3^36*x4^16*z^34 + x1^66*x2^52*x3^36*x4^16*z^34 + x1^65*x2^53*x3^36*x4^16*z^34 + 3*x1^64*x2^54*x3^36*x4^16*z^34 - x1^63*x2^55*x3^36*x4^16*z^34 - 2*x1^62*x2^56*x3^36*x4^16*z^34 - 2*x1^67*x2^50*x3^37*x4^16*z^34 - 3*x1^66*x2^51*x3^37*x4^16*z^34 - x1^65*x2^52*x3^37*x4^16*z^34 - 4*x1^64*x2^53*x3^37*x4^16*z^34 + x1^63*x2^54*x3^37*x4^16*z^34 - x1^62*x2^55*x3^37*x4^16*z^34 + x1^61*x2^56*x3^37*x4^16*z^34 + 4*x1^66*x2^50*x3^38*x4^16*z^34 + 2*x1^64*x2^52*x3^38*x4^16*z^34 + 2*x1^63*x2^53*x3^38*x4^16*z^34 - x1^66*x2^49*x3^39*x4^16*z^34 - x1^65*x2^50*x3^39*x4^16*z^34 - x1^64*x2^51*x3^39*x4^16*z^34 - 3*x1^63*x2^52*x3^39*x4^16*z^34 + x1^65*x2^49*x3^40*x4^16*z^34 + x1^62*x2^52*x3^40*x4^16*z^34 + x1^70*x2^53*x3^30*x4^17*z^34 - 2*x1^69*x2^53*x3^31*x4^17*z^34 - x1^68*x2^54*x3^31*x4^17*z^34 - x1^67*x2^55*x3^31*x4^17*z^34 + x1^69*x2^52*x3^32*x4^17*z^34 + 2*x1^68*x2^53*x3^32*x4^17*z^34 + x1^66*x2^55*x3^32*x4^17*z^34 - x1^65*x2^56*x3^32*x4^17*z^34 - 2*x1^68*x2^52*x3^33*x4^17*z^34 - x1^65*x2^55*x3^33*x4^17*z^34 + 2*x1^67*x2^52*x3^34*x4^17*z^34 + 2*x1^65*x2^54*x3^34*x4^17*z^34 - x1^62*x2^57*x3^34*x4^17*z^34 - 2*x1^67*x2^51*x3^35*x4^17*z^34 - x1^66*x2^52*x3^35*x4^17*z^34 - x1^65*x2^53*x3^35*x4^17*z^34 + x1^63*x2^55*x3^35*x4^17*z^34 + x1^62*x2^56*x3^35*x4^17*z^34 + x1^61*x2^57*x3^35*x4^17*z^34 + x1^67*x2^50*x3^36*x4^17*z^34 + 2*x1^66*x2^51*x3^36*x4^17*z^34 + x1^64*x2^53*x3^36*x4^17*z^34 + x1^63*x2^54*x3^36*x4^17*z^34 - 2*x1^61*x2^56*x3^36*x4^17*z^34 - 2*x1^66*x2^50*x3^37*x4^17*z^34 - x1^65*x2^51*x3^37*x4^17*z^34 - x1^64*x2^52*x3^37*x4^17*z^34 - x1^63*x2^53*x3^37*x4^17*z^34 + x1^62*x2^54*x3^37*x4^17*z^34 + x1^61*x2^55*x3^37*x4^17*z^34 + x1^66*x2^49*x3^38*x4^17*z^34 + 2*x1^65*x2^50*x3^38*x4^17*z^34 + x1^63*x2^52*x3^38*x4^17*z^34 - x1^62*x2^53*x3^38*x4^17*z^34 + x1^61*x2^54*x3^38*x4^17*z^34 - x1^60*x2^55*x3^38*x4^17*z^34 - x1^65*x2^49*x3^39*x4^17*z^34 + x1^64*x2^50*x3^39*x4^17*z^34 - x1^63*x2^51*x3^39*x4^17*z^34 - x1^62*x2^52*x3^39*x4^17*z^34 + x1^64*x2^49*x3^40*x4^17*z^34 + x1^63*x2^50*x3^40*x4^17*z^34 + x1^62*x2^51*x3^40*x4^17*z^34 - x1^61*x2^51*x3^41*x4^17*z^34 - x1^70*x2^54*x3^32*x4^9*z^33 + x1^69*x2^55*x3^32*x4^9*z^33 - x1^70*x2^53*x3^33*x4^9*z^33 - x1^69*x2^54*x3^33*x4^9*z^33 - x1^71*x2^54*x3^30*x4^10*z^33 + x1^69*x2^53*x3^33*x4^10*z^33 - x1^68*x2^54*x3^33*x4^10*z^33 + x1^66*x2^56*x3^33*x4^10*z^33 - x1^66*x2^55*x3^34*x4^10*z^33 - x1^70*x2^54*x3^30*x4^11*z^33 + 2*x1^70*x2^53*x3^31*x4^11*z^33 + x1^69*x2^54*x3^31*x4^11*z^33 - x1^68*x2^55*x3^31*x4^11*z^33 - 3*x1^69*x2^53*x3^32*x4^11*z^33 - 2*x1^67*x2^55*x3^32*x4^11*z^33 - x1^66*x2^56*x3^32*x4^11*z^33 + x1^69*x2^52*x3^33*x4^11*z^33 + 2*x1^68*x2^53*x3^33*x4^11*z^33 + x1^67*x2^54*x3^33*x4^11*z^33 + 4*x1^66*x2^55*x3^33*x4^11*z^33 + x1^65*x2^56*x3^33*x4^11*z^33 - x1^68*x2^52*x3^34*x4^11*z^33 - x1^65*x2^55*x3^34*x4^11*z^33 + x1^68*x2^51*x3^35*x4^11*z^33 + x1^67*x2^52*x3^35*x4^11*z^33 + 2*x1^65*x2^54*x3^35*x4^11*z^33 + x1^64*x2^55*x3^35*x4^11*z^33 - x1^70*x2^53*x3^30*x4^12*z^33 + x1^69*x2^53*x3^31*x4^12*z^33 + x1^67*x2^55*x3^31*x4^12*z^33 - x1^68*x2^53*x3^32*x4^12*z^33 + 3*x1^67*x2^54*x3^32*x4^12*z^33 + 2*x1^65*x2^56*x3^32*x4^12*z^33 - x1^67*x2^53*x3^33*x4^12*z^33 - x1^66*x2^54*x3^33*x4^12*z^33 + x1^65*x2^55*x3^33*x4^12*z^33 - x1^68*x2^51*x3^34*x4^12*z^33 + 3*x1^66*x2^53*x3^34*x4^12*z^33 - x1^65*x2^54*x3^34*x4^12*z^33 - x1^67*x2^51*x3^35*x4^12*z^33 - x1^67*x2^50*x3^36*x4^12*z^33 - x1^66*x2^51*x3^36*x4^12*z^33 + x1^65*x2^52*x3^36*x4^12*z^33 - x1^64*x2^53*x3^36*x4^12*z^33 - x1^70*x2^53*x3^29*x4^13*z^33 + x1^69*x2^53*x3^30*x4^13*z^33 + x1^68*x2^54*x3^30*x4^13*z^33 - x1^68*x2^53*x3^31*x4^13*z^33 - 4*x1^67*x2^54*x3^31*x4^13*z^33 + x1^68*x2^52*x3^32*x4^13*z^33 + 4*x1^67*x2^53*x3^32*x4^13*z^33 + 3*x1^66*x2^54*x3^32*x4^13*z^33 - x1^65*x2^55*x3^32*x4^13*z^33 + x1^64*x2^56*x3^32*x4^13*z^33 - 2*x1^67*x2^52*x3^33*x4^13*z^33 - 8*x1^66*x2^53*x3^33*x4^13*z^33 - 2*x1^64*x2^55*x3^33*x4^13*z^33 + x1^67*x2^51*x3^34*x4^13*z^33 + 2*x1^66*x2^52*x3^34*x4^13*z^33 + 4*x1^65*x2^53*x3^34*x4^13*z^33 + x1^64*x2^54*x3^34*x4^13*z^33 + 2*x1^63*x2^55*x3^34*x4^13*z^33 + x1^61*x2^57*x3^34*x4^13*z^33 - x1^66*x2^51*x3^35*x4^13*z^33 - 5*x1^65*x2^52*x3^35*x4^13*z^33 - x1^64*x2^53*x3^35*x4^13*z^33 - 2*x1^63*x2^54*x3^35*x4^13*z^33 - 2*x1^62*x2^55*x3^35*x4^13*z^33 + 2*x1^66*x2^50*x3^36*x4^13*z^33 + x1^64*x2^52*x3^36*x4^13*z^33 + x1^63*x2^53*x3^36*x4^13*z^33 + 3*x1^62*x2^54*x3^36*x4^13*z^33 - x1^65*x2^50*x3^37*x4^13*z^33 - 2*x1^64*x2^51*x3^37*x4^13*z^33 - 2*x1^63*x2^52*x3^37*x4^13*z^33 - x1^62*x2^53*x3^37*x4^13*z^33 + x1^69*x2^53*x3^29*x4^14*z^33 - x1^68*x2^53*x3^30*x4^14*z^33 + x1^67*x2^54*x3^30*x4^14*z^33 + x1^68*x2^52*x3^31*x4^14*z^33 + x1^67*x2^53*x3^31*x4^14*z^33 + x1^65*x2^55*x3^31*x4^14*z^33 - x1^68*x2^51*x3^32*x4^14*z^33 - x1^67*x2^52*x3^32*x4^14*z^33 + 2*x1^66*x2^53*x3^32*x4^14*z^33 + x1^65*x2^54*x3^32*x4^14*z^33 + x1^64*x2^55*x3^32*x4^14*z^33 + 2*x1^67*x2^51*x3^33*x4^14*z^33 - 3*x1^66*x2^52*x3^33*x4^14*z^33 - 3*x1^65*x2^53*x3^33*x4^14*z^33 + 2*x1^64*x2^54*x3^33*x4^14*z^33 - x1^63*x2^55*x3^33*x4^14*z^33 - x1^62*x2^56*x3^33*x4^14*z^33 + 2*x1^65*x2^52*x3^34*x4^14*z^33 - 3*x1^64*x2^53*x3^34*x4^14*z^33 + x1^62*x2^55*x3^34*x4^14*z^33 - x1^60*x2^57*x3^34*x4^14*z^33 + x1^66*x2^50*x3^35*x4^14*z^33 - 2*x1^64*x2^52*x3^35*x4^14*z^33 - x1^63*x2^53*x3^35*x4^14*z^33 - x1^62*x2^54*x3^35*x4^14*z^33 + x1^61*x2^55*x3^35*x4^14*z^33 + x1^59*x2^57*x3^35*x4^14*z^33 + x1^65*x2^50*x3^36*x4^14*z^33 + x1^64*x2^51*x3^36*x4^14*z^33 - x1^63*x2^52*x3^36*x4^14*z^33 + x1^61*x2^54*x3^36*x4^14*z^33 + x1^64*x2^50*x3^37*x4^14*z^33 - x1^62*x2^52*x3^37*x4^14*z^33 - 2*x1^61*x2^53*x3^37*x4^14*z^33 + x1^64*x2^49*x3^38*x4^14*z^33 + 2*x1^63*x2^50*x3^38*x4^14*z^33 + 2*x1^62*x2^51*x3^38*x4^14*z^33 + x1^61*x2^52*x3^38*x4^14*z^33 - 2*x1^68*x2^52*x3^30*x4^15*z^33 - x1^66*x2^54*x3^30*x4^15*z^33 + 2*x1^68*x2^51*x3^31*x4^15*z^33 + 2*x1^67*x2^52*x3^31*x4^15*z^33 - x1^66*x2^53*x3^31*x4^15*z^33 + x1^65*x2^54*x3^31*x4^15*z^33 - 6*x1^67*x2^51*x3^32*x4^15*z^33 - x1^65*x2^53*x3^32*x4^15*z^33 - x1^64*x2^54*x3^32*x4^15*z^33 + 2*x1^67*x2^50*x3^33*x4^15*z^33 + 6*x1^66*x2^51*x3^33*x4^15*z^33 - 2*x1^65*x2^52*x3^33*x4^15*z^33 + 2*x1^64*x2^53*x3^33*x4^15*z^33 - 6*x1^66*x2^50*x3^34*x4^15*z^33 - x1^65*x2^51*x3^34*x4^15*z^33 + x1^64*x2^52*x3^34*x4^15*z^33 - 3*x1^63*x2^53*x3^34*x4^15*z^33 + x1^61*x2^55*x3^34*x4^15*z^33 + 2*x1^66*x2^49*x3^35*x4^15*z^33 + 3*x1^65*x2^50*x3^35*x4^15*z^33 + x1^64*x2^51*x3^35*x4^15*z^33 + 4*x1^63*x2^52*x3^35*x4^15*z^33 - 4*x1^65*x2^49*x3^36*x4^15*z^33 - 2*x1^63*x2^51*x3^36*x4^15*z^33 - 2*x1^62*x2^52*x3^36*x4^15*z^33 + x1^65*x2^48*x3^37*x4^15*z^33 + x1^64*x2^49*x3^37*x4^15*z^33 + x1^63*x2^50*x3^37*x4^15*z^33 + 3*x1^62*x2^51*x3^37*x4^15*z^33 - 2*x1^64*x2^48*x3^38*x4^15*z^33 - x1^61*x2^51*x3^38*x4^15*z^33 - x1^63*x2^48*x3^39*x4^15*z^33 - x1^69*x2^52*x3^28*x4^16*z^33 + 3*x1^68*x2^52*x3^29*x4^16*z^33 - 2*x1^67*x2^53*x3^29*x4^16*z^33 - 2*x1^68*x2^51*x3^30*x4^16*z^33 - 3*x1^67*x2^52*x3^30*x4^16*z^33 + 2*x1^66*x2^53*x3^30*x4^16*z^33 - x1^65*x2^54*x3^30*x4^16*z^33 + 6*x1^67*x2^51*x3^31*x4^16*z^33 + x1^66*x2^52*x3^31*x4^16*z^33 - x1^64*x2^54*x3^31*x4^16*z^33 - 2*x1^63*x2^55*x3^31*x4^16*z^33 - 2*x1^67*x2^50*x3^32*x4^16*z^33 - 6*x1^66*x2^51*x3^32*x4^16*z^33 - 2*x1^64*x2^53*x3^32*x4^16*z^33 + 2*x1^63*x2^54*x3^32*x4^16*z^33 + 2*x1^62*x2^55*x3^32*x4^16*z^33 + 6*x1^66*x2^50*x3^33*x4^16*z^33 + 2*x1^65*x2^51*x3^33*x4^16*z^33 + 2*x1^64*x2^52*x3^33*x4^16*z^33 + x1^63*x2^53*x3^33*x4^16*z^33 - x1^62*x2^54*x3^33*x4^16*z^33 - 2*x1^61*x2^55*x3^33*x4^16*z^33 - 2*x1^66*x2^49*x3^34*x4^16*z^33 - 6*x1^65*x2^50*x3^34*x4^16*z^33 - 4*x1^63*x2^52*x3^34*x4^16*z^33 + 3*x1^60*x2^55*x3^34*x4^16*z^33 + 6*x1^65*x2^49*x3^35*x4^16*z^33 + 2*x1^64*x2^50*x3^35*x4^16*z^33 + 2*x1^63*x2^51*x3^35*x4^16*z^33 + 2*x1^62*x2^52*x3^35*x4^16*z^33 + x1^61*x2^53*x3^35*x4^16*z^33 - 2*x1^60*x2^54*x3^35*x4^16*z^33 - 2*x1^59*x2^55*x3^35*x4^16*z^33 - 2*x1^65*x2^48*x3^36*x4^16*z^33 - 5*x1^64*x2^49*x3^36*x4^16*z^33 - x1^63*x2^50*x3^36*x4^16*z^33 - 3*x1^62*x2^51*x3^36*x4^16*z^33 - x1^60*x2^53*x3^36*x4^16*z^33 + 3*x1^59*x2^54*x3^36*x4^16*z^33 + 5*x1^64*x2^48*x3^37*x4^16*z^33 + 2*x1^62*x2^50*x3^37*x4^16*z^33 + 3*x1^61*x2^51*x3^37*x4^16*z^33 - x1^60*x2^52*x3^37*x4^16*z^33 - x1^59*x2^53*x3^37*x4^16*z^33 - 2*x1^63*x2^48*x3^38*x4^16*z^33 - 4*x1^61*x2^50*x3^38*x4^16*z^33 + 2*x1^63*x2^47*x3^39*x4^16*z^33 + x1^61*x2^49*x3^39*x4^16*z^33 + 2*x1^60*x2^50*x3^39*x4^16*z^33 - x1^60*x2^49*x3^40*x4^16*z^33 - x1^68*x2^52*x3^28*x4^17*z^33 + x1^67*x2^52*x3^29*x4^17*z^33 - 2*x1^67*x2^51*x3^30*x4^17*z^33 - x1^65*x2^53*x3^30*x4^17*z^33 + x1^67*x2^50*x3^31*x4^17*z^33 + 2*x1^66*x2^51*x3^31*x4^17*z^33 + 2*x1^64*x2^53*x3^31*x4^17*z^33 + x1^63*x2^54*x3^31*x4^17*z^33 - 2*x1^66*x2^50*x3^32*x4^17*z^33 - x1^65*x2^51*x3^32*x4^17*z^33 - x1^64*x2^52*x3^32*x4^17*z^33 + x1^61*x2^55*x3^32*x4^17*z^33 + x1^66*x2^49*x3^33*x4^17*z^33 + 2*x1^65*x2^50*x3^33*x4^17*z^33 + x1^63*x2^52*x3^33*x4^17*z^33 - 2*x1^62*x2^53*x3^33*x4^17*z^33 - x1^60*x2^55*x3^33*x4^17*z^33 - 2*x1^65*x2^49*x3^34*x4^17*z^33 + x1^59*x2^55*x3^34*x4^17*z^33 + 2*x1^64*x2^49*x3^35*x4^17*z^33 + 2*x1^62*x2^51*x3^35*x4^17*z^33 - x1^61*x2^52*x3^35*x4^17*z^33 - 2*x1^59*x2^54*x3^35*x4^17*z^33 - x1^58*x2^55*x3^35*x4^17*z^33 - 2*x1^64*x2^48*x3^36*x4^17*z^33 - x1^63*x2^49*x3^36*x4^17*z^33 - x1^62*x2^50*x3^36*x4^17*z^33 + x1^58*x2^54*x3^36*x4^17*z^33 + x1^64*x2^47*x3^37*x4^17*z^33 + 2*x1^63*x2^48*x3^37*x4^17*z^33 + x1^61*x2^50*x3^37*x4^17*z^33 + x1^60*x2^51*x3^37*x4^17*z^33 - 2*x1^58*x2^53*x3^37*x4^17*z^33 - 2*x1^63*x2^47*x3^38*x4^17*z^33 - x1^62*x2^48*x3^38*x4^17*z^33 - x1^61*x2^49*x3^38*x4^17*z^33 - x1^60*x2^50*x3^38*x4^17*z^33 + x1^59*x2^51*x3^38*x4^17*z^33 + x1^58*x2^52*x3^38*x4^17*z^33 + x1^62*x2^47*x3^39*x4^17*z^33 + x1^60*x2^49*x3^39*x4^17*z^33 - x1^60*x2^48*x3^40*x4^17*z^33 - x1^59*x2^49*x3^40*x4^17*z^33 - x1^68*x2^54*x3^30*x4^8*z^32 + x1^69*x2^52*x3^31*x4^8*z^32 - x1^67*x2^53*x3^32*x4^8*z^32 + x1^67*x2^53*x3^31*x4^9*z^32 - x1^68*x2^51*x3^32*x4^9*z^32 + x1^67*x2^52*x3^32*x4^9*z^32 + x1^65*x2^54*x3^32*x4^9*z^32 + x1^67*x2^51*x3^33*x4^9*z^32 + 2*x1^66*x2^52*x3^33*x4^9*z^32 + 2*x1^65*x2^53*x3^33*x4^9*z^32 + x1^68*x2^52*x3^30*x4^10*z^32 + x1^67*x2^53*x3^30*x4^10*z^32 + x1^66*x2^54*x3^30*x4^10*z^32 - x1^68*x2^51*x3^31*x4^10*z^32 - x1^67*x2^52*x3^31*x4^10*z^32 + x1^66*x2^53*x3^31*x4^10*z^32 - x1^65*x2^54*x3^31*x4^10*z^32 + x1^67*x2^51*x3^32*x4^10*z^32 + x1^65*x2^53*x3^32*x4^10*z^32 - 2*x1^66*x2^51*x3^33*x4^10*z^32 - x1^65*x2^52*x3^33*x4^10*z^32 - x1^64*x2^53*x3^33*x4^10*z^32 + x1^65*x2^51*x3^34*x4^10*z^32 + x1^63*x2^53*x3^34*x4^10*z^32 + x1^62*x2^54*x3^34*x4^10*z^32 + x1^69*x2^52*x3^28*x4^11*z^32 - 3*x1^68*x2^52*x3^29*x4^11*z^32 + x1^67*x2^53*x3^29*x4^11*z^32 + 2*x1^68*x2^51*x3^30*x4^11*z^32 + 3*x1^67*x2^52*x3^30*x4^11*z^32 - 2*x1^66*x2^53*x3^30*x4^11*z^32 + 2*x1^65*x2^54*x3^30*x4^11*z^32 - 3*x1^67*x2^51*x3^31*x4^11*z^32 - x1^65*x2^53*x3^31*x4^11*z^32 - x1^64*x2^54*x3^31*x4^11*z^32 + x1^63*x2^55*x3^31*x4^11*z^32 + x1^67*x2^50*x3^32*x4^11*z^32 + 3*x1^66*x2^51*x3^32*x4^11*z^32 + 3*x1^64*x2^53*x3^32*x4^11*z^32 + x1^63*x2^54*x3^32*x4^11*z^32 + x1^62*x2^55*x3^32*x4^11*z^32 - 2*x1^66*x2^50*x3^33*x4^11*z^32 + x1^65*x2^51*x3^33*x4^11*z^32 - 2*x1^64*x2^52*x3^33*x4^11*z^32 - 3*x1^63*x2^53*x3^33*x4^11*z^32 - x1^62*x2^54*x3^33*x4^11*z^32 + x1^66*x2^49*x3^34*x4^11*z^32 + x1^65*x2^50*x3^34*x4^11*z^32 + 2*x1^63*x2^52*x3^34*x4^11*z^32 - x1^62*x2^53*x3^34*x4^11*z^32 - x1^65*x2^49*x3^35*x4^11*z^32 - 2*x1^64*x2^50*x3^35*x4^11*z^32 - x1^63*x2^51*x3^35*x4^11*z^32 - 2*x1^61*x2^53*x3^35*x4^11*z^32 + x1^68*x2^52*x3^28*x4^12*z^32 - x1^67*x2^52*x3^29*x4^12*z^32 + x1^66*x2^53*x3^29*x4^12*z^32 - x1^67*x2^50*x3^31*x4^12*z^32 + 4*x1^65*x2^52*x3^31*x4^12*z^32 - 2*x1^64*x2^53*x3^31*x4^12*z^32 + x1^62*x2^55*x3^31*x4^12*z^32 + x1^66*x2^50*x3^32*x4^12*z^32 - 3*x1^65*x2^51*x3^32*x4^12*z^32 - 2*x1^64*x2^52*x3^32*x4^12*z^32 + x1^63*x2^53*x3^32*x4^12*z^32 - 2*x1^62*x2^54*x3^32*x4^12*z^32 - 2*x1^61*x2^55*x3^32*x4^12*z^32 - x1^65*x2^50*x3^33*x4^12*z^32 + 5*x1^64*x2^51*x3^33*x4^12*z^32 + 2*x1^62*x2^53*x3^33*x4^12*z^32 + x1^60*x2^55*x3^33*x4^12*z^32 - x1^63*x2^51*x3^34*x4^12*z^32 - 2*x1^61*x2^53*x3^34*x4^12*z^32 - x1^65*x2^48*x3^35*x4^12*z^32 + x1^64*x2^49*x3^35*x4^12*z^32 + 2*x1^63*x2^50*x3^35*x4^12*z^32 + x1^61*x2^52*x3^35*x4^12*z^32 + x1^64*x2^48*x3^36*x4^12*z^32 + x1^63*x2^49*x3^36*x4^12*z^32 + x1^62*x2^50*x3^36*x4^12*z^32 + 3*x1^66*x2^52*x3^29*x4^13*z^32 + x1^65*x2^53*x3^29*x4^13*z^32 - 5*x1^65*x2^52*x3^30*x4^13*z^32 - x1^63*x2^54*x3^30*x4^13*z^32 + 2*x1^65*x2^51*x3^31*x4^13*z^32 + 5*x1^64*x2^52*x3^31*x4^13*z^32 - x1^63*x2^53*x3^31*x4^13*z^32 + 2*x1^62*x2^54*x3^31*x4^13*z^32 - x1^61*x2^55*x3^31*x4^13*z^32 - 2*x1^65*x2^50*x3^32*x4^13*z^32 - 7*x1^64*x2^51*x3^32*x4^13*z^32 - 3*x1^63*x2^52*x3^32*x4^13*z^32 - 2*x1^62*x2^53*x3^32*x4^13*z^32 - x1^61*x2^54*x3^32*x4^13*z^32 + x1^60*x2^55*x3^32*x4^13*z^32 + 2*x1^64*x2^50*x3^33*x4^13*z^32 + 6*x1^63*x2^51*x3^33*x4^13*z^32 - x1^62*x2^52*x3^33*x4^13*z^32 + 3*x1^61*x2^53*x3^33*x4^13*z^32 - x1^60*x2^54*x3^33*x4^13*z^32 - 2*x1^64*x2^49*x3^34*x4^13*z^32 - 5*x1^63*x2^50*x3^34*x4^13*z^32 - 3*x1^61*x2^52*x3^34*x4^13*z^32 - x1^60*x2^53*x3^34*x4^13*z^32 + x1^59*x2^54*x3^34*x4^13*z^32 + x1^64*x2^48*x3^35*x4^13*z^32 + x1^63*x2^49*x3^35*x4^13*z^32 + 3*x1^62*x2^50*x3^35*x4^13*z^32 + x1^61*x2^51*x3^35*x4^13*z^32 + 4*x1^60*x2^52*x3^35*x4^13*z^32 - 2*x1^63*x2^48*x3^36*x4^13*z^32 - 3*x1^62*x2^49*x3^36*x4^13*z^32 - x1^61*x2^50*x3^36*x4^13*z^32 - x1^60*x2^51*x3^36*x4^13*z^32 - x1^59*x2^52*x3^36*x4^13*z^32 + x1^63*x2^47*x3^37*x4^13*z^32 + x1^61*x2^49*x3^37*x4^13*z^32 + x1^60*x2^50*x3^37*x4^13*z^32 + 2*x1^59*x2^51*x3^37*x4^13*z^32 - x1^68*x2^51*x3^27*x4^14*z^32 + x1^67*x2^51*x3^28*x4^14*z^32 - x1^66*x2^52*x3^28*x4^14*z^32 - x1^66*x2^51*x3^29*x4^14*z^32 - 2*x1^64*x2^53*x3^29*x4^14*z^32 + x1^66*x2^50*x3^30*x4^14*z^32 + x1^63*x2^53*x3^30*x4^14*z^32 - x1^62*x2^54*x3^30*x4^14*z^32 - x1^66*x2^49*x3^31*x4^14*z^32 - x1^65*x2^50*x3^31*x4^14*z^32 + 2*x1^64*x2^51*x3^31*x4^14*z^32 - x1^63*x2^52*x3^31*x4^14*z^32 - x1^62*x2^53*x3^31*x4^14*z^32 - x1^60*x2^55*x3^31*x4^14*z^32 + 2*x1^65*x2^49*x3^32*x4^14*z^32 - x1^63*x2^51*x3^32*x4^14*z^32 + 3*x1^62*x2^52*x3^32*x4^14*z^32 + x1^59*x2^55*x3^32*x4^14*z^32 - x1^65*x2^48*x3^33*x4^14*z^32 - x1^64*x2^49*x3^33*x4^14*z^32 + x1^63*x2^50*x3^33*x4^14*z^32 + 2*x1^62*x2^51*x3^33*x4^14*z^32 + x1^61*x2^52*x3^33*x4^14*z^32 - 2*x1^60*x2^53*x3^33*x4^14*z^32 - x1^58*x2^55*x3^33*x4^14*z^32 + 2*x1^64*x2^48*x3^34*x4^14*z^32 - 2*x1^63*x2^49*x3^34*x4^14*z^32 - 3*x1^62*x2^50*x3^34*x4^14*z^32 + 2*x1^61*x2^51*x3^34*x4^14*z^32 - x1^58*x2^54*x3^34*x4^14*z^32 + x1^57*x2^55*x3^34*x4^14*z^32 + x1^63*x2^48*x3^35*x4^14*z^32 + x1^62*x2^49*x3^35*x4^14*z^32 - 2*x1^61*x2^50*x3^35*x4^14*z^32 + x1^60*x2^51*x3^35*x4^14*z^32 + x1^59*x2^52*x3^35*x4^14*z^32 - x1^58*x2^53*x3^35*x4^14*z^32 - x1^57*x2^54*x3^35*x4^14*z^32 - x1^56*x2^55*x3^35*x4^14*z^32 + x1^63*x2^47*x3^36*x4^14*z^32 - x1^62*x2^48*x3^36*x4^14*z^32 - 2*x1^61*x2^49*x3^36*x4^14*z^32 - x1^59*x2^51*x3^36*x4^14*z^32 + x1^62*x2^47*x3^37*x4^14*z^32 + x1^61*x2^48*x3^37*x4^14*z^32 - x1^60*x2^49*x3^37*x4^14*z^32 + x1^58*x2^51*x3^37*x4^14*z^32 - x1^61*x2^47*x3^38*x4^14*z^32 - x1^60*x2^48*x3^38*x4^14*z^32 - x1^59*x2^49*x3^38*x4^14*z^32 - 2*x1^58*x2^50*x3^38*x4^14*z^32 + x1^67*x2^50*x3^28*x4^15*z^32 - x1^65*x2^52*x3^28*x4^15*z^32 - 3*x1^66*x2^50*x3^29*x4^15*z^32 + x1^65*x2^51*x3^29*x4^15*z^32 + x1^64*x2^52*x3^29*x4^15*z^32 + 2*x1^66*x2^49*x3^30*x4^15*z^32 + 3*x1^65*x2^50*x3^30*x4^15*z^32 - 2*x1^64*x2^51*x3^30*x4^15*z^32 + x1^63*x2^52*x3^30*x4^15*z^32 - 6*x1^65*x2^49*x3^31*x4^15*z^32 - x1^64*x2^50*x3^31*x4^15*z^32 - x1^62*x2^52*x3^31*x4^15*z^32 + x1^61*x2^53*x3^31*x4^15*z^32 + 2*x1^65*x2^48*x3^32*x4^15*z^32 + 6*x1^64*x2^49*x3^32*x4^15*z^32 + 3*x1^62*x2^51*x3^32*x4^15*z^32 - x1^60*x2^53*x3^32*x4^15*z^32 - x1^59*x2^54*x3^32*x4^15*z^32 - 6*x1^64*x2^48*x3^33*x4^15*z^32 - 2*x1^63*x2^49*x3^33*x4^15*z^32 - 2*x1^62*x2^50*x3^33*x4^15*z^32 - 2*x1^61*x2^51*x3^33*x4^15*z^32 + x1^60*x2^52*x3^33*x4^15*z^32 + x1^59*x2^53*x3^33*x4^15*z^32 + 2*x1^64*x2^47*x3^34*x4^15*z^32 + 5*x1^63*x2^48*x3^34*x4^15*z^32 + x1^62*x2^49*x3^34*x4^15*z^32 + 3*x1^61*x2^50*x3^34*x4^15*z^32 - 2*x1^58*x2^53*x3^34*x4^15*z^32 - 5*x1^63*x2^47*x3^35*x4^15*z^32 - 2*x1^61*x2^49*x3^35*x4^15*z^32 - 3*x1^60*x2^50*x3^35*x4^15*z^32 + x1^59*x2^51*x3^35*x4^15*z^32 + x1^58*x2^52*x3^35*x4^15*z^32 + 2*x1^63*x2^46*x3^36*x4^15*z^32 + 2*x1^62*x2^47*x3^36*x4^15*z^32 + 4*x1^60*x2^49*x3^36*x4^15*z^32 - 2*x1^62*x2^46*x3^37*x4^15*z^32 + x1^61*x2^47*x3^37*x4^15*z^32 - x1^60*x2^48*x3^37*x4^15*z^32 - 2*x1^59*x2^49*x3^37*x4^15*z^32 + x1^61*x2^46*x3^38*x4^15*z^32 + x1^60*x2^47*x3^38*x4^15*z^32 + x1^59*x2^48*x3^38*x4^15*z^32 + x1^60*x2^46*x3^39*x4^15*z^32 + x1^59*x2^47*x3^39*x4^15*z^32 - 2*x1^67*x2^50*x3^27*x4^16*z^32 + 4*x1^66*x2^50*x3^28*x4^16*z^32 + x1^64*x2^52*x3^28*x4^16*z^32 - 2*x1^66*x2^49*x3^29*x4^16*z^32 - 4*x1^65*x2^50*x3^29*x4^16*z^32 + x1^64*x2^51*x3^29*x4^16*z^32 - x1^63*x2^52*x3^29*x4^16*z^32 + 2*x1^62*x2^53*x3^29*x4^16*z^32 + 6*x1^65*x2^49*x3^30*x4^16*z^32 + 2*x1^64*x2^50*x3^30*x4^16*z^32 + x1^63*x2^51*x3^30*x4^16*z^32 - 2*x1^61*x2^53*x3^30*x4^16*z^32 - 2*x1^65*x2^48*x3^31*x4^16*z^32 - 6*x1^64*x2^49*x3^31*x4^16*z^32 - 4*x1^62*x2^51*x3^31*x4^16*z^32 + 2*x1^61*x2^52*x3^31*x4^16*z^32 + 2*x1^60*x2^53*x3^31*x4^16*z^32 + 2*x1^59*x2^54*x3^31*x4^16*z^32 + 6*x1^64*x2^48*x3^32*x4^16*z^32 + 2*x1^63*x2^49*x3^32*x4^16*z^32 + 2*x1^62*x2^50*x3^32*x4^16*z^32 + x1^61*x2^51*x3^32*x4^16*z^32 - 2*x1^60*x2^52*x3^32*x4^16*z^32 - 3*x1^59*x2^53*x3^32*x4^16*z^32 - 2*x1^58*x2^54*x3^32*x4^16*z^32 - 2*x1^64*x2^47*x3^33*x4^16*z^32 - 6*x1^63*x2^48*x3^33*x4^16*z^32 - 4*x1^61*x2^50*x3^33*x4^16*z^32 + x1^60*x2^51*x3^33*x4^16*z^32 + 5*x1^58*x2^53*x3^33*x4^16*z^32 + 6*x1^63*x2^47*x3^34*x4^16*z^32 + 2*x1^62*x2^48*x3^34*x4^16*z^32 + 2*x1^61*x2^49*x3^34*x4^16*z^32 + x1^60*x2^50*x3^34*x4^16*z^32 - x1^59*x2^51*x3^34*x4^16*z^32 - x1^58*x2^52*x3^34*x4^16*z^32 - 2*x1^57*x2^53*x3^34*x4^16*z^32 - 2*x1^63*x2^46*x3^35*x4^16*z^32 - 6*x1^62*x2^47*x3^35*x4^16*z^32 - 4*x1^60*x2^49*x3^35*x4^16*z^32 + 4*x1^57*x2^52*x3^35*x4^16*z^32 + 5*x1^62*x2^46*x3^36*x4^16*z^32 + x1^61*x2^47*x3^36*x4^16*z^32 + 3*x1^60*x2^48*x3^36*x4^16*z^32 + 2*x1^59*x2^49*x3^36*x4^16*z^32 - x1^58*x2^50*x3^36*x4^16*z^32 - x1^57*x2^51*x3^36*x4^16*z^32 - x1^56*x2^52*x3^36*x4^16*z^32 - x1^62*x2^45*x3^37*x4^16*z^32 - 4*x1^61*x2^46*x3^37*x4^16*z^32 - 3*x1^59*x2^48*x3^37*x4^16*z^32 - x1^58*x2^49*x3^37*x4^16*z^32 + 2*x1^56*x2^51*x3^37*x4^16*z^32 + x1^61*x2^45*x3^38*x4^16*z^32 - x1^60*x2^46*x3^38*x4^16*z^32 + x1^59*x2^47*x3^38*x4^16*z^32 + 2*x1^58*x2^48*x3^38*x4^16*z^32 - x1^60*x2^45*x3^39*x4^16*z^32 - x1^59*x2^46*x3^39*x4^16*z^32 - x1^58*x2^47*x3^39*x4^16*z^32 + x1^57*x2^47*x3^40*x4^16*z^32 - x1^66*x2^50*x3^27*x4^17*z^32 + x1^66*x2^49*x3^28*x4^17*z^32 + x1^65*x2^50*x3^28*x4^17*z^32 + x1^63*x2^52*x3^28*x4^17*z^32 - 2*x1^65*x2^49*x3^29*x4^17*z^32 - x1^62*x2^52*x3^29*x4^17*z^32 + 2*x1^64*x2^49*x3^30*x4^17*z^32 + x1^62*x2^51*x3^30*x4^17*z^32 - 2*x1^64*x2^48*x3^31*x4^17*z^32 - x1^63*x2^49*x3^31*x4^17*z^32 - x1^62*x2^50*x3^31*x4^17*z^32 + x1^64*x2^47*x3^32*x4^17*z^32 + 2*x1^63*x2^48*x3^32*x4^17*z^32 + x1^61*x2^50*x3^32*x4^17*z^32 - x1^60*x2^51*x3^32*x4^17*z^32 - 2*x1^58*x2^53*x3^32*x4^17*z^32 - 2*x1^63*x2^47*x3^33*x4^17*z^32 - x1^62*x2^48*x3^33*x4^17*z^32 - x1^61*x2^49*x3^33*x4^17*z^32 + x1^59*x2^51*x3^33*x4^17*z^32 + x1^58*x2^52*x3^33*x4^17*z^32 + 2*x1^57*x2^53*x3^33*x4^17*z^32 + x1^63*x2^46*x3^34*x4^17*z^32 + 2*x1^62*x2^47*x3^34*x4^17*z^32 + x1^60*x2^49*x3^34*x4^17*z^32 - 2*x1^59*x2^50*x3^34*x4^17*z^32 - 2*x1^57*x2^52*x3^34*x4^17*z^32 - 2*x1^62*x2^46*x3^35*x4^17*z^32 + 2*x1^56*x2^52*x3^35*x4^17*z^32 + 2*x1^61*x2^46*x3^36*x4^17*z^32 + 2*x1^59*x2^48*x3^36*x4^17*z^32 - x1^57*x2^50*x3^36*x4^17*z^32 - x1^56*x2^51*x3^36*x4^17*z^32 - x1^61*x2^45*x3^37*x4^17*z^32 - 2*x1^60*x2^46*x3^37*x4^17*z^32 - x1^59*x2^47*x3^37*x4^17*z^32 + x1^55*x2^51*x3^37*x4^17*z^32 + x1^60*x2^45*x3^38*x4^17*z^32 + 2*x1^59*x2^46*x3^38*x4^17*z^32 + x1^58*x2^47*x3^38*x4^17*z^32 + x1^57*x2^48*x3^38*x4^17*z^32 - 2*x1^55*x2^50*x3^38*x4^17*z^32 - x1^58*x2^46*x3^39*x4^17*z^32 - x1^57*x2^47*x3^39*x4^17*z^32 - x1^65*x2^52*x3^31*x4^7*z^31 - x1^66*x2^52*x3^29*x4^8*z^31 + x1^66*x2^51*x3^30*x4^8*z^31 + 2*x1^65*x2^52*x3^30*x4^8*z^31 - x1^66*x2^50*x3^31*x4^8*z^31 - x1^65*x2^51*x3^31*x4^8*z^31 + x1^66*x2^49*x3^32*x4^8*z^31 + x1^64*x2^51*x3^32*x4^8*z^31 + x1^63*x2^52*x3^32*x4^8*z^31 + x1^66*x2^52*x3^28*x4^9*z^31 + x1^66*x2^51*x3^29*x4^9*z^31 - 2*x1^65*x2^52*x3^29*x4^9*z^31 + x1^64*x2^53*x3^29*x4^9*z^31 + x1^66*x2^50*x3^30*x4^9*z^31 - 2*x1^64*x2^51*x3^31*x4^9*z^31 + x1^63*x2^52*x3^31*x4^9*z^31 + x1^65*x2^49*x3^32*x4^9*z^31 + x1^64*x2^50*x3^32*x4^9*z^31 - x1^62*x2^52*x3^32*x4^9*z^31 - x1^65*x2^48*x3^33*x4^9*z^31 - x1^63*x2^50*x3^33*x4^9*z^31 - 2*x1^62*x2^51*x3^33*x4^9*z^31 - x1^61*x2^52*x3^33*x4^9*z^31 - x1^67*x2^50*x3^28*x4^10*z^31 + 2*x1^66*x2^50*x3^29*x4^10*z^31 - x1^65*x2^51*x3^29*x4^10*z^31 + x1^63*x2^53*x3^29*x4^10*z^31 - x1^66*x2^49*x3^30*x4^10*z^31 - 2*x1^65*x2^50*x3^30*x4^10*z^31 + x1^64*x2^51*x3^30*x4^10*z^31 - 3*x1^63*x2^52*x3^30*x4^10*z^31 - x1^62*x2^53*x3^30*x4^10*z^31 + x1^61*x2^54*x3^30*x4^10*z^31 + 2*x1^65*x2^49*x3^31*x4^10*z^31 + 2*x1^64*x2^50*x3^31*x4^10*z^31 + x1^63*x2^51*x3^31*x4^10*z^31 + x1^62*x2^52*x3^31*x4^10*z^31 - x1^61*x2^53*x3^31*x4^10*z^31 - x1^65*x2^48*x3^32*x4^10*z^31 - x1^64*x2^49*x3^32*x4^10*z^31 - x1^63*x2^50*x3^32*x4^10*z^31 - 2*x1^62*x2^51*x3^32*x4^10*z^31 - x1^61*x2^52*x3^32*x4^10*z^31 - x1^64*x2^48*x3^33*x4^10*z^31 + x1^63*x2^49*x3^33*x4^10*z^31 + 2*x1^62*x2^50*x3^33*x4^10*z^31 + 2*x1^61*x2^51*x3^33*x4^10*z^31 + x1^60*x2^52*x3^33*x4^10*z^31 - 2*x1^62*x2^49*x3^34*x4^10*z^31 - x1^59*x2^52*x3^34*x4^10*z^31 + 2*x1^67*x2^50*x3^27*x4^11*z^31 - 4*x1^66*x2^50*x3^28*x4^11*z^31 + x1^65*x2^51*x3^28*x4^11*z^31 - x1^64*x2^52*x3^28*x4^11*z^31 + x1^66*x2^49*x3^29*x4^11*z^31 + 3*x1^65*x2^50*x3^29*x4^11*z^31 - x1^64*x2^51*x3^29*x4^11*z^31 + 2*x1^63*x2^52*x3^29*x4^11*z^31 - 2*x1^62*x2^53*x3^29*x4^11*z^31 - 4*x1^65*x2^49*x3^30*x4^11*z^31 - 3*x1^64*x2^50*x3^30*x4^11*z^31 - 2*x1^62*x2^52*x3^30*x4^11*z^31 + 2*x1^65*x2^48*x3^31*x4^11*z^31 + 2*x1^64*x2^49*x3^31*x4^11*z^31 - 3*x1^63*x2^50*x3^31*x4^11*z^31 + 3*x1^62*x2^51*x3^31*x4^11*z^31 + x1^61*x2^52*x3^31*x4^11*z^31 - x1^60*x2^53*x3^31*x4^11*z^31 - x1^59*x2^54*x3^31*x4^11*z^31 - 2*x1^64*x2^48*x3^32*x4^11*z^31 - x1^63*x2^49*x3^32*x4^11*z^31 - 2*x1^62*x2^50*x3^32*x4^11*z^31 - x1^61*x2^51*x3^32*x4^11*z^31 + x1^59*x2^53*x3^32*x4^11*z^31 + 2*x1^64*x2^47*x3^33*x4^11*z^31 + x1^63*x2^48*x3^33*x4^11*z^31 - x1^62*x2^49*x3^33*x4^11*z^31 + x1^61*x2^50*x3^33*x4^11*z^31 + x1^59*x2^52*x3^33*x4^11*z^31 - x1^63*x2^47*x3^34*x4^11*z^31 - x1^61*x2^49*x3^34*x4^11*z^31 - 2*x1^60*x2^50*x3^34*x4^11*z^31 + x1^63*x2^46*x3^35*x4^11*z^31 + 2*x1^61*x2^48*x3^35*x4^11*z^31 + 2*x1^60*x2^49*x3^35*x4^11*z^31 + x1^58*x2^51*x3^35*x4^11*z^31 - x1^67*x2^50*x3^26*x4^12*z^31 + x1^66*x2^50*x3^27*x4^12*z^31 - x1^65*x2^51*x3^27*x4^12*z^31 - x1^65*x2^50*x3^28*x4^12*z^31 + x1^64*x2^51*x3^28*x4^12*z^31 - x1^63*x2^52*x3^28*x4^12*z^31 + x1^65*x2^49*x3^29*x4^12*z^31 - 2*x1^64*x2^50*x3^29*x4^12*z^31 - x1^63*x2^51*x3^29*x4^12*z^31 + 2*x1^62*x2^52*x3^29*x4^12*z^31 - x1^61*x2^53*x3^29*x4^12*z^31 - x1^65*x2^48*x3^30*x4^12*z^31 + x1^64*x2^49*x3^30*x4^12*z^31 + 6*x1^63*x2^50*x3^30*x4^12*z^31 - 2*x1^62*x2^51*x3^30*x4^12*z^31 + x1^61*x2^52*x3^30*x4^12*z^31 + x1^60*x2^53*x3^30*x4^12*z^31 + 2*x1^64*x2^48*x3^31*x4^12*z^31 - x1^63*x2^49*x3^31*x4^12*z^31 - 5*x1^62*x2^50*x3^31*x4^12*z^31 + x1^61*x2^51*x3^31*x4^12*z^31 - x1^60*x2^52*x3^31*x4^12*z^31 - x1^59*x2^53*x3^31*x4^12*z^31 - x1^58*x2^54*x3^31*x4^12*z^31 - x1^64*x2^47*x3^32*x4^12*z^31 + 5*x1^62*x2^49*x3^32*x4^12*z^31 + 2*x1^60*x2^51*x3^32*x4^12*z^31 - x1^59*x2^52*x3^32*x4^12*z^31 + 2*x1^58*x2^53*x3^32*x4^12*z^31 + x1^63*x2^47*x3^33*x4^12*z^31 - x1^62*x2^48*x3^33*x4^12*z^31 - 3*x1^61*x2^49*x3^33*x4^12*z^31 - 4*x1^59*x2^51*x3^33*x4^12*z^31 - x1^58*x2^52*x3^33*x4^12*z^31 - x1^57*x2^53*x3^33*x4^12*z^31 + x1^62*x2^47*x3^34*x4^12*z^31 + 2*x1^61*x2^48*x3^34*x4^12*z^31 + x1^59*x2^50*x3^34*x4^12*z^31 + x1^58*x2^51*x3^34*x4^12*z^31 + x1^62*x2^46*x3^35*x4^12*z^31 - x1^60*x2^48*x3^35*x4^12*z^31 - 2*x1^58*x2^50*x3^35*x4^12*z^31 - x1^62*x2^45*x3^36*x4^12*z^31 - 2*x1^59*x2^48*x3^36*x4^12*z^31 + x1^66*x2^49*x3^27*x4^13*z^31 - 2*x1^64*x2^51*x3^27*x4^13*z^31 - x1^65*x2^49*x3^28*x4^13*z^31 + x1^64*x2^50*x3^28*x4^13*z^31 + 2*x1^63*x2^51*x3^28*x4^13*z^31 - 2*x1^62*x2^52*x3^28*x4^13*z^31 + x1^64*x2^49*x3^29*x4^13*z^31 - 5*x1^63*x2^50*x3^29*x4^13*z^31 - x1^62*x2^51*x3^29*x4^13*z^31 - x1^61*x2^52*x3^29*x4^13*z^31 + x1^63*x2^49*x3^30*x4^13*z^31 + 5*x1^62*x2^50*x3^30*x4^13*z^31 - 3*x1^61*x2^51*x3^30*x4^13*z^31 + 2*x1^60*x2^52*x3^30*x4^13*z^31 - x1^59*x2^53*x3^30*x4^13*z^31 - 6*x1^62*x2^49*x3^31*x4^13*z^31 - x1^61*x2^50*x3^31*x4^13*z^31 - x1^60*x2^51*x3^31*x4^13*z^31 - x1^59*x2^52*x3^31*x4^13*z^31 + x1^58*x2^53*x3^31*x4^13*z^31 + x1^57*x2^54*x3^31*x4^13*z^31 + 3*x1^62*x2^48*x3^32*x4^13*z^31 + 7*x1^61*x2^49*x3^32*x4^13*z^31 + 4*x1^59*x2^51*x3^32*x4^13*z^31 - x1^57*x2^53*x3^32*x4^13*z^31 - 2*x1^56*x2^54*x3^32*x4^13*z^31 - 2*x1^62*x2^47*x3^33*x4^13*z^31 - 4*x1^61*x2^48*x3^33*x4^13*z^31 - x1^60*x2^49*x3^33*x4^13*z^31 - 3*x1^59*x2^50*x3^33*x4^13*z^31 + 2*x1^57*x2^52*x3^33*x4^13*z^31 + 2*x1^56*x2^53*x3^33*x4^13*z^31 + 3*x1^61*x2^47*x3^34*x4^13*z^31 + 4*x1^60*x2^48*x3^34*x4^13*z^31 + 4*x1^58*x2^50*x3^34*x4^13*z^31 - x1^57*x2^51*x3^34*x4^13*z^31 - x1^55*x2^53*x3^34*x4^13*z^31 - x1^62*x2^45*x3^35*x4^13*z^31 - x1^61*x2^46*x3^35*x4^13*z^31 - 3*x1^60*x2^47*x3^35*x4^13*z^31 - x1^59*x2^48*x3^35*x4^13*z^31 - 2*x1^58*x2^49*x3^35*x4^13*z^31 - 2*x1^57*x2^50*x3^35*x4^13*z^31 + x1^61*x2^45*x3^36*x4^13*z^31 + x1^60*x2^46*x3^36*x4^13*z^31 + 3*x1^59*x2^47*x3^36*x4^13*z^31 + x1^58*x2^48*x3^36*x4^13*z^31 + 3*x1^57*x2^49*x3^36*x4^13*z^31 - x1^60*x2^45*x3^37*x4^13*z^31 - 2*x1^59*x2^46*x3^37*x4^13*z^31 - x1^58*x2^47*x3^37*x4^13*z^31 - x1^56*x2^49*x3^37*x4^13*z^31 + x1^64*x2^50*x3^27*x4^14*z^31 + x1^63*x2^51*x3^27*x4^14*z^31 - x1^65*x2^48*x3^28*x4^14*z^31 + x1^63*x2^50*x3^28*x4^14*z^31 + x1^61*x2^52*x3^28*x4^14*z^31 + 2*x1^64*x2^48*x3^29*x4^14*z^31 - 2*x1^63*x2^49*x3^29*x4^14*z^31 - x1^62*x2^50*x3^29*x4^14*z^31 + 2*x1^61*x2^51*x3^29*x4^14*z^31 + x1^60*x2^52*x3^29*x4^14*z^31 + x1^59*x2^53*x3^29*x4^14*z^31 - 2*x1^63*x2^48*x3^30*x4^14*z^31 + 3*x1^62*x2^49*x3^30*x4^14*z^31 - 2*x1^61*x2^50*x3^30*x4^14*z^31 - 2*x1^60*x2^51*x3^30*x4^14*z^31 - x1^59*x2^52*x3^30*x4^14*z^31 + 2*x1^63*x2^47*x3^31*x4^14*z^31 + x1^62*x2^48*x3^31*x4^14*z^31 - 2*x1^61*x2^49*x3^31*x4^14*z^31 + x1^60*x2^50*x3^31*x4^14*z^31 - 2*x1^59*x2^51*x3^31*x4^14*z^31 + x1^58*x2^52*x3^31*x4^14*z^31 + x1^56*x2^54*x3^31*x4^14*z^31 - x1^63*x2^46*x3^32*x4^14*z^31 - 2*x1^62*x2^47*x3^32*x4^14*z^31 + 2*x1^61*x2^48*x3^32*x4^14*z^31 - 2*x1^59*x2^50*x3^32*x4^14*z^31 - x1^58*x2^51*x3^32*x4^14*z^31 - 2*x1^57*x2^52*x3^32*x4^14*z^31 - x1^56*x2^53*x3^32*x4^14*z^31 - x1^55*x2^54*x3^32*x4^14*z^31 + x1^62*x2^46*x3^33*x4^14*z^31 - x1^61*x2^47*x3^33*x4^14*z^31 + x1^59*x2^49*x3^33*x4^14*z^31 - 3*x1^58*x2^50*x3^33*x4^14*z^31 - x1^57*x2^51*x3^33*x4^14*z^31 + x1^56*x2^52*x3^33*x4^14*z^31 + 2*x1^55*x2^53*x3^33*x4^14*z^31 - x1^62*x2^45*x3^34*x4^14*z^31 - x1^61*x2^46*x3^34*x4^14*z^31 + x1^60*x2^47*x3^34*x4^14*z^31 + x1^59*x2^48*x3^34*x4^14*z^31 + 2*x1^58*x2^49*x3^34*x4^14*z^31 - x1^57*x2^50*x3^34*x4^14*z^31 - x1^56*x2^51*x3^34*x4^14*z^31 - x1^55*x2^52*x3^34*x4^14*z^31 + x1^61*x2^45*x3^35*x4^14*z^31 - 3*x1^60*x2^46*x3^35*x4^14*z^31 - 2*x1^59*x2^47*x3^35*x4^14*z^31 + 2*x1^58*x2^48*x3^35*x4^14*z^31 - x1^57*x2^49*x3^35*x4^14*z^31 + x1^56*x2^50*x3^35*x4^14*z^31 + x1^54*x2^52*x3^35*x4^14*z^31 + x1^57*x2^48*x3^36*x4^14*z^31 + x1^56*x2^49*x3^36*x4^14*z^31 - x1^58*x2^46*x3^37*x4^14*z^31 - x1^56*x2^48*x3^37*x4^14*z^31 + x1^58*x2^45*x3^38*x4^14*z^31 + x1^57*x2^46*x3^38*x4^14*z^31 + x1^55*x2^48*x3^38*x4^14*z^31 - x1^65*x2^49*x3^26*x4^15*z^31 + x1^64*x2^50*x3^26*x4^15*z^31 + 2*x1^65*x2^48*x3^27*x4^15*z^31 + x1^64*x2^49*x3^27*x4^15*z^31 - 2*x1^63*x2^50*x3^27*x4^15*z^31 - 5*x1^64*x2^48*x3^28*x4^15*z^31 + x1^63*x2^49*x3^28*x4^15*z^31 + x1^62*x2^50*x3^28*x4^15*z^31 + x1^61*x2^51*x3^28*x4^15*z^31 + x1^60*x2^52*x3^28*x4^15*z^31 + 2*x1^64*x2^47*x3^29*x4^15*z^31 + 5*x1^63*x2^48*x3^29*x4^15*z^31 - 2*x1^62*x2^49*x3^29*x4^15*z^31 + x1^61*x2^50*x3^29*x4^15*z^31 - 2*x1^60*x2^51*x3^29*x4^15*z^31 - x1^59*x2^52*x3^29*x4^15*z^31 - 6*x1^63*x2^47*x3^30*x4^15*z^31 - 2*x1^62*x2^48*x3^30*x4^15*z^31 + x1^60*x2^50*x3^30*x4^15*z^31 + 2*x1^59*x2^51*x3^30*x4^15*z^31 + x1^58*x2^52*x3^30*x4^15*z^31 + 2*x1^63*x2^46*x3^31*x4^15*z^31 + 6*x1^62*x2^47*x3^31*x4^15*z^31 + 3*x1^60*x2^49*x3^31*x4^15*z^31 - x1^59*x2^50*x3^31*x4^15*z^31 - 2*x1^58*x2^51*x3^31*x4^15*z^31 - 2*x1^57*x2^52*x3^31*x4^15*z^31 - 6*x1^62*x2^46*x3^32*x4^15*z^31 - 2*x1^61*x2^47*x3^32*x4^15*z^31 - 2*x1^60*x2^48*x3^32*x4^15*z^31 - x1^59*x2^49*x3^32*x4^15*z^31 + 2*x1^57*x2^51*x3^32*x4^15*z^31 + x1^56*x2^52*x3^32*x4^15*z^31 + 2*x1^62*x2^45*x3^33*x4^15*z^31 + 6*x1^61*x2^46*x3^33*x4^15*z^31 + 4*x1^59*x2^48*x3^33*x4^15*z^31 - 4*x1^56*x2^51*x3^33*x4^15*z^31 - 5*x1^61*x2^45*x3^34*x4^15*z^31 - x1^60*x2^46*x3^34*x4^15*z^31 - 3*x1^59*x2^47*x3^34*x4^15*z^31 - 2*x1^58*x2^48*x3^34*x4^15*z^31 + x1^57*x2^49*x3^34*x4^15*z^31 + x1^56*x2^50*x3^34*x4^15*z^31 + x1^55*x2^51*x3^34*x4^15*z^31 + x1^61*x2^44*x3^35*x4^15*z^31 + 4*x1^60*x2^45*x3^35*x4^15*z^31 + 3*x1^58*x2^47*x3^35*x4^15*z^31 + x1^57*x2^48*x3^35*x4^15*z^31 - 2*x1^55*x2^50*x3^35*x4^15*z^31 - 3*x1^60*x2^44*x3^36*x4^15*z^31 - x1^59*x2^45*x3^36*x4^15*z^31 - 2*x1^58*x2^46*x3^36*x4^15*z^31 - 2*x1^57*x2^47*x3^36*x4^15*z^31 + x1^59*x2^44*x3^37*x4^15*z^31 - x1^58*x2^45*x3^37*x4^15*z^31 + 2*x1^57*x2^46*x3^37*x4^15*z^31 - x1^57*x2^45*x3^38*x4^15*z^31 - x1^56*x2^46*x3^38*x4^15*z^31 - x1^56*x2^45*x3^39*x4^15*z^31 + x1^65*x2^49*x3^25*x4^16*z^31 - x1^65*x2^48*x3^26*x4^16*z^31 - 2*x1^64*x2^49*x3^26*x4^16*z^31 + x1^63*x2^50*x3^26*x4^16*z^31 + 5*x1^64*x2^48*x3^27*x4^16*z^31 + x1^63*x2^49*x3^27*x4^16*z^31 + x1^62*x2^50*x3^27*x4^16*z^31 - 2*x1^64*x2^47*x3^28*x4^16*z^31 - 5*x1^63*x2^48*x3^28*x4^16*z^31 - 3*x1^61*x2^50*x3^28*x4^16*z^31 + 6*x1^63*x2^47*x3^29*x4^16*z^31 + x1^62*x2^48*x3^29*x4^16*z^31 + 2*x1^61*x2^49*x3^29*x4^16*z^31 - x1^59*x2^51*x3^29*x4^16*z^31 - 2*x1^58*x2^52*x3^29*x4^16*z^31 - 2*x1^63*x2^46*x3^30*x4^16*z^31 - 6*x1^62*x2^47*x3^30*x4^16*z^31 - 5*x1^60*x2^49*x3^30*x4^16*z^31 + 2*x1^59*x2^50*x3^30*x4^16*z^31 + x1^58*x2^51*x3^30*x4^16*z^31 + 3*x1^57*x2^52*x3^30*x4^16*z^31 + 6*x1^62*x2^46*x3^31*x4^16*z^31 + 2*x1^61*x2^47*x3^31*x4^16*z^31 + 2*x1^60*x2^48*x3^31*x4^16*z^31 - x1^58*x2^50*x3^31*x4^16*z^31 - 2*x1^57*x2^51*x3^31*x4^16*z^31 - 3*x1^56*x2^52*x3^31*x4^16*z^31 - 2*x1^62*x2^45*x3^32*x4^16*z^31 - 6*x1^61*x2^46*x3^32*x4^16*z^31 - 4*x1^59*x2^48*x3^32*x4^16*z^31 + 4*x1^58*x2^49*x3^32*x4^16*z^31 + 5*x1^56*x2^51*x3^32*x4^16*z^31 + 2*x1^55*x2^52*x3^32*x4^16*z^31 + 6*x1^61*x2^45*x3^33*x4^16*z^31 + 2*x1^60*x2^46*x3^33*x4^16*z^31 + 2*x1^59*x2^47*x3^33*x4^16*z^31 - x1^57*x2^49*x3^33*x4^16*z^31 - x1^56*x2^50*x3^33*x4^16*z^31 - 4*x1^55*x2^51*x3^33*x4^16*z^31 - 2*x1^61*x2^44*x3^34*x4^16*z^31 - 6*x1^60*x2^45*x3^34*x4^16*z^31 - 4*x1^58*x2^47*x3^34*x4^16*z^31 + x1^57*x2^48*x3^34*x4^16*z^31 + x1^56*x2^49*x3^34*x4^16*z^31 + 5*x1^55*x2^50*x3^34*x4^16*z^31 + 5*x1^60*x2^44*x3^35*x4^16*z^31 + 2*x1^59*x2^45*x3^35*x4^16*z^31 + 2*x1^58*x2^46*x3^35*x4^16*z^31 + 2*x1^57*x2^47*x3^35*x4^16*z^31 - 2*x1^55*x2^49*x3^35*x4^16*z^31 - 2*x1^54*x2^50*x3^35*x4^16*z^31 - x1^60*x2^43*x3^36*x4^16*z^31 - 4*x1^59*x2^44*x3^36*x4^16*z^31 - 4*x1^57*x2^46*x3^36*x4^16*z^31 - x1^56*x2^47*x3^36*x4^16*z^31 + x1^55*x2^48*x3^36*x4^16*z^31 + 3*x1^54*x2^49*x3^36*x4^16*z^31 + 2*x1^59*x2^43*x3^37*x4^16*z^31 + 2*x1^58*x2^44*x3^37*x4^16*z^31 + 2*x1^57*x2^45*x3^37*x4^16*z^31 + x1^56*x2^46*x3^37*x4^16*z^31 - x1^53*x2^49*x3^37*x4^16*z^31 - x1^58*x2^43*x3^38*x4^16*z^31 - x1^56*x2^45*x3^38*x4^16*z^31 + x1^55*x2^46*x3^38*x4^16*z^31 + x1^56*x2^44*x3^39*x4^16*z^31 + x1^55*x2^45*x3^39*x4^16*z^31 - x1^62*x2^50*x3^26*x4^17*z^31 + x1^64*x2^47*x3^27*x4^17*z^31 + 2*x1^63*x2^48*x3^27*x4^17*z^31 - x1^62*x2^49*x3^27*x4^17*z^31 + x1^61*x2^50*x3^27*x4^17*z^31 - 2*x1^63*x2^47*x3^28*x4^17*z^31 - x1^62*x2^48*x3^28*x4^17*z^31 + x1^63*x2^46*x3^29*x4^17*z^31 + 2*x1^62*x2^47*x3^29*x4^17*z^31 + x1^60*x2^49*x3^29*x4^17*z^31 - x1^59*x2^50*x3^29*x4^17*z^31 - 2*x1^62*x2^46*x3^30*x4^17*z^31 + x1^57*x2^51*x3^30*x4^17*z^31 + 2*x1^61*x2^46*x3^31*x4^17*z^31 + 2*x1^59*x2^48*x3^31*x4^17*z^31 - x1^58*x2^49*x3^31*x4^17*z^31 - 2*x1^56*x2^51*x3^31*x4^17*z^31 - 2*x1^61*x2^45*x3^32*x4^17*z^31 - x1^60*x2^46*x3^32*x4^17*z^31 - x1^59*x2^47*x3^32*x4^17*z^31 + x1^57*x2^49*x3^32*x4^17*z^31 + x1^56*x2^50*x3^32*x4^17*z^31 + 2*x1^55*x2^51*x3^32*x4^17*z^31 + x1^61*x2^44*x3^33*x4^17*z^31 + 2*x1^60*x2^45*x3^33*x4^17*z^31 + x1^58*x2^47*x3^33*x4^17*z^31 - x1^57*x2^48*x3^33*x4^17*z^31 - 2*x1^55*x2^50*x3^33*x4^17*z^31 - x1^54*x2^51*x3^33*x4^17*z^31 - 2*x1^60*x2^44*x3^34*x4^17*z^31 - x1^59*x2^45*x3^34*x4^17*z^31 - x1^58*x2^46*x3^34*x4^17*z^31 + x1^56*x2^48*x3^34*x4^17*z^31 + x1^55*x2^49*x3^34*x4^17*z^31 + 2*x1^54*x2^50*x3^34*x4^17*z^31 + 2*x1^59*x2^44*x3^35*x4^17*z^31 + x1^57*x2^46*x3^35*x4^17*z^31 - 2*x1^56*x2^47*x3^35*x4^17*z^31 - 2*x1^54*x2^49*x3^35*x4^17*z^31 - x1^58*x2^44*x3^36*x4^17*z^31 - x1^56*x2^46*x3^36*x4^17*z^31 - x1^55*x2^47*x3^36*x4^17*z^31 + x1^54*x2^48*x3^36*x4^17*z^31 + x1^53*x2^49*x3^36*x4^17*z^31 + x1^57*x2^44*x3^37*x4^17*z^31 + 2*x1^56*x2^45*x3^37*x4^17*z^31 + x1^55*x2^46*x3^37*x4^17*z^31 - x1^54*x2^47*x3^37*x4^17*z^31 - x1^53*x2^48*x3^37*x4^17*z^31 - x1^56*x2^44*x3^38*x4^17*z^31 - x1^54*x2^46*x3^38*x4^17*z^31 + x1^52*x2^48*x3^38*x4^17*z^31 + x1^66*x2^49*x3^29*x4^6*z^30 + x1^65*x2^50*x3^29*x4^6*z^30 + x1^64*x2^50*x3^30*x4^6*z^30 + x1^65*x2^51*x3^27*x4^7*z^30 - x1^66*x2^49*x3^28*x4^7*z^30 + x1^64*x2^50*x3^29*x4^7*z^30 - x1^65*x2^48*x3^30*x4^7*z^30 + x1^62*x2^51*x3^30*x4^7*z^30 + x1^62*x2^50*x3^31*x4^7*z^30 + x1^61*x2^51*x3^31*x4^7*z^30 + x1^64*x2^51*x3^27*x4^8*z^30 - 2*x1^64*x2^50*x3^28*x4^8*z^30 + x1^62*x2^52*x3^28*x4^8*z^30 + 2*x1^63*x2^50*x3^29*x4^8*z^30 - x1^62*x2^51*x3^29*x4^8*z^30 - x1^64*x2^48*x3^30*x4^8*z^30 - x1^63*x2^49*x3^30*x4^8*z^30 - 2*x1^62*x2^50*x3^30*x4^8*z^30 - x1^61*x2^51*x3^30*x4^8*z^30 - x1^60*x2^52*x3^30*x4^8*z^30 + 2*x1^62*x2^49*x3^31*x4^8*z^30 - 2*x1^63*x2^47*x3^32*x4^8*z^30 - x1^62*x2^48*x3^32*x4^8*z^30 - x1^61*x2^49*x3^32*x4^8*z^30 - x1^60*x2^50*x3^32*x4^8*z^30 + x1^64*x2^50*x3^27*x4^9*z^30 - x1^65*x2^48*x3^28*x4^9*z^30 + x1^64*x2^49*x3^28*x4^9*z^30 - 2*x1^63*x2^50*x3^28*x4^9*z^30 - x1^62*x2^51*x3^28*x4^9*z^30 - x1^61*x2^52*x3^28*x4^9*z^30 + 2*x1^62*x2^50*x3^29*x4^9*z^30 - x1^61*x2^51*x3^29*x4^9*z^30 - 2*x1^62*x2^49*x3^30*x4^9*z^30 + x1^61*x2^50*x3^30*x4^9*z^30 + x1^60*x2^51*x3^30*x4^9*z^30 + 2*x1^63*x2^47*x3^31*x4^9*z^30 - x1^62*x2^48*x3^31*x4^9*z^30 + x1^61*x2^49*x3^31*x4^9*z^30 + x1^63*x2^46*x3^32*x4^9*z^30 - x1^61*x2^48*x3^32*x4^9*z^30 - x1^60*x2^49*x3^32*x4^9*z^30 - x1^59*x2^50*x3^32*x4^9*z^30 + 2*x1^62*x2^46*x3^33*x4^9*z^30 + x1^61*x2^47*x3^33*x4^9*z^30 + x1^60*x2^48*x3^33*x4^9*z^30 + x1^58*x2^50*x3^33*x4^9*z^30 + x1^65*x2^49*x3^26*x4^10*z^30 - x1^65*x2^48*x3^27*x4^10*z^30 - x1^64*x2^49*x3^27*x4^10*z^30 + x1^63*x2^50*x3^27*x4^10*z^30 + 3*x1^64*x2^48*x3^28*x4^10*z^30 - x1^63*x2^49*x3^28*x4^10*z^30 + 2*x1^61*x2^51*x3^28*x4^10*z^30 - x1^60*x2^52*x3^28*x4^10*z^30 - 3*x1^63*x2^48*x3^29*x4^10*z^30 - x1^62*x2^49*x3^29*x4^10*z^30 - x1^61*x2^50*x3^29*x4^10*z^30 - x1^59*x2^52*x3^29*x4^10*z^30 + x1^63*x2^47*x3^30*x4^10*z^30 + 3*x1^62*x2^48*x3^30*x4^10*z^30 + x1^61*x2^49*x3^30*x4^10*z^30 + x1^60*x2^50*x3^30*x4^10*z^30 + x1^59*x2^51*x3^30*x4^10*z^30 - x1^57*x2^53*x3^30*x4^10*z^30 - x1^62*x2^47*x3^31*x4^10*z^30 - 3*x1^61*x2^48*x3^31*x4^10*z^30 - 2*x1^60*x2^49*x3^31*x4^10*z^30 - x1^59*x2^50*x3^31*x4^10*z^30 + x1^57*x2^52*x3^31*x4^10*z^30 + 2*x1^61*x2^47*x3^32*x4^10*z^30 + 2*x1^60*x2^48*x3^32*x4^10*z^30 + x1^59*x2^49*x3^32*x4^10*z^30 + 2*x1^58*x2^50*x3^32*x4^10*z^30 - x1^62*x2^45*x3^33*x4^10*z^30 + x1^61*x2^46*x3^33*x4^10*z^30 - x1^60*x2^47*x3^33*x4^10*z^30 + x1^59*x2^48*x3^33*x4^10*z^30 - 2*x1^57*x2^50*x3^33*x4^10*z^30 + x1^59*x2^47*x3^34*x4^10*z^30 + x1^58*x2^48*x3^34*x4^10*z^30 + x1^57*x2^49*x3^34*x4^10*z^30 - 2*x1^65*x2^49*x3^25*x4^11*z^30 + x1^65*x2^48*x3^26*x4^11*z^30 + 2*x1^64*x2^49*x3^26*x4^11*z^30 - 2*x1^63*x2^50*x3^26*x4^11*z^30 - 4*x1^64*x2^48*x3^27*x4^11*z^30 - x1^63*x2^49*x3^27*x4^11*z^30 + x1^64*x2^47*x3^28*x4^11*z^30 + 4*x1^63*x2^48*x3^28*x4^11*z^30 + x1^61*x2^50*x3^28*x4^11*z^30 - x1^60*x2^51*x3^28*x4^11*z^30 - 4*x1^63*x2^47*x3^29*x4^11*z^30 + 2*x1^62*x2^48*x3^29*x4^11*z^30 - 2*x1^60*x2^50*x3^29*x4^11*z^30 + x1^59*x2^51*x3^29*x4^11*z^30 + 2*x1^58*x2^52*x3^29*x4^11*z^30 + x1^63*x2^46*x3^30*x4^11*z^30 + 3*x1^62*x2^47*x3^30*x4^11*z^30 - x1^61*x2^48*x3^30*x4^11*z^30 + 4*x1^60*x2^49*x3^30*x4^11*z^30 - x1^58*x2^51*x3^30*x4^11*z^30 - 3*x1^57*x2^52*x3^30*x4^11*z^30 - 4*x1^62*x2^46*x3^31*x4^11*z^30 - 2*x1^61*x2^47*x3^31*x4^11*z^30 + x1^58*x2^50*x3^31*x4^11*z^30 + x1^56*x2^52*x3^31*x4^11*z^30 + x1^62*x2^45*x3^32*x4^11*z^30 + x1^61*x2^46*x3^32*x4^11*z^30 + 2*x1^59*x2^48*x3^32*x4^11*z^30 - x1^58*x2^49*x3^32*x4^11*z^30 + x1^57*x2^50*x3^32*x4^11*z^30 - 2*x1^56*x2^51*x3^32*x4^11*z^30 - x1^55*x2^52*x3^32*x4^11*z^30 - 3*x1^61*x2^45*x3^33*x4^11*z^30 - x1^60*x2^46*x3^33*x4^11*z^30 - 2*x1^59*x2^47*x3^33*x4^11*z^30 + x1^57*x2^49*x3^33*x4^11*z^30 + x1^61*x2^44*x3^34*x4^11*z^30 + x1^56*x2^49*x3^34*x4^11*z^30 - 2*x1^60*x2^44*x3^35*x4^11*z^30 - x1^58*x2^46*x3^35*x4^11*z^30 - 2*x1^57*x2^47*x3^35*x4^11*z^30 - x1^56*x2^48*x3^35*x4^11*z^30 + 2*x1^64*x2^48*x3^26*x4^12*z^30 + x1^62*x2^50*x3^26*x4^12*z^30 - x1^64*x2^47*x3^27*x4^12*z^30 - 2*x1^63*x2^48*x3^27*x4^12*z^30 + 3*x1^62*x2^49*x3^27*x4^12*z^30 - x1^61*x2^50*x3^27*x4^12*z^30 + x1^60*x2^51*x3^27*x4^12*z^30 + 2*x1^63*x2^47*x3^28*x4^12*z^30 - 3*x1^62*x2^48*x3^28*x4^12*z^30 - 3*x1^61*x2^49*x3^28*x4^12*z^30 + 2*x1^60*x2^50*x3^28*x4^12*z^30 - x1^59*x2^51*x3^28*x4^12*z^30 - 2*x1^62*x2^47*x3^29*x4^12*z^30 + 7*x1^61*x2^48*x3^29*x4^12*z^30 + x1^59*x2^50*x3^29*x4^12*z^30 + 2*x1^62*x2^46*x3^30*x4^12*z^30 - 2*x1^61*x2^47*x3^30*x4^12*z^30 - 7*x1^60*x2^48*x3^30*x4^12*z^30 + 2*x1^59*x2^49*x3^30*x4^12*z^30 - 3*x1^58*x2^50*x3^30*x4^12*z^30 + x1^57*x2^51*x3^30*x4^12*z^30 - x1^56*x2^52*x3^30*x4^12*z^30 - x1^62*x2^45*x3^31*x4^12*z^30 + 4*x1^60*x2^47*x3^31*x4^12*z^30 + 2*x1^58*x2^49*x3^31*x4^12*z^30 + x1^57*x2^50*x3^31*x4^12*z^30 + 2*x1^61*x2^45*x3^32*x4^12*z^30 - 2*x1^60*x2^46*x3^32*x4^12*z^30 - 3*x1^59*x2^47*x3^32*x4^12*z^30 + x1^58*x2^48*x3^32*x4^12*z^30 - 5*x1^57*x2^49*x3^32*x4^12*z^30 - x1^60*x2^45*x3^33*x4^12*z^30 + 3*x1^59*x2^46*x3^33*x4^12*z^30 + 3*x1^57*x2^48*x3^33*x4^12*z^30 + x1^56*x2^49*x3^33*x4^12*z^30 + x1^55*x2^50*x3^33*x4^12*z^30 + x1^60*x2^44*x3^34*x4^12*z^30 - 2*x1^59*x2^45*x3^34*x4^12*z^30 - 2*x1^58*x2^46*x3^34*x4^12*z^30 - 3*x1^56*x2^48*x3^34*x4^12*z^30 + x1^60*x2^43*x3^35*x4^12*z^30 + x1^59*x2^44*x3^35*x4^12*z^30 + x1^55*x2^48*x3^35*x4^12*z^30 + x1^59*x2^43*x3^36*x4^12*z^30 + x1^58*x2^44*x3^36*x4^12*z^30 + x1^56*x2^46*x3^36*x4^12*z^30 - x1^64*x2^48*x3^25*x4^13*z^30 + 2*x1^63*x2^49*x3^25*x4^13*z^30 + x1^63*x2^48*x3^26*x4^13*z^30 - 3*x1^62*x2^49*x3^26*x4^13*z^30 + 2*x1^61*x2^50*x3^26*x4^13*z^30 - x1^63*x2^47*x3^27*x4^13*z^30 + x1^62*x2^48*x3^27*x4^13*z^30 + 2*x1^61*x2^49*x3^27*x4^13*z^30 - x1^60*x2^50*x3^27*x4^13*z^30 + 2*x1^59*x2^51*x3^27*x4^13*z^30 + x1^62*x2^47*x3^28*x4^13*z^30 - 4*x1^61*x2^48*x3^28*x4^13*z^30 + 2*x1^57*x2^52*x3^28*x4^13*z^30 + 2*x1^61*x2^47*x3^29*x4^13*z^30 + 4*x1^60*x2^48*x3^29*x4^13*z^30 - 2*x1^59*x2^49*x3^29*x4^13*z^30 + x1^58*x2^50*x3^29*x4^13*z^30 - x1^57*x2^51*x3^29*x4^13*z^30 - 2*x1^56*x2^52*x3^29*x4^13*z^30 - 6*x1^60*x2^47*x3^30*x4^13*z^30 - x1^59*x2^48*x3^30*x4^13*z^30 + x1^58*x2^49*x3^30*x4^13*z^30 + x1^57*x2^50*x3^30*x4^13*z^30 + 2*x1^56*x2^51*x3^30*x4^13*z^30 + 2*x1^55*x2^52*x3^30*x4^13*z^30 + 2*x1^60*x2^46*x3^31*x4^13*z^30 + 6*x1^59*x2^47*x3^31*x4^13*z^30 - x1^58*x2^48*x3^31*x4^13*z^30 + 3*x1^57*x2^49*x3^31*x4^13*z^30 - 2*x1^56*x2^50*x3^31*x4^13*z^30 - x1^55*x2^51*x3^31*x4^13*z^30 - 3*x1^54*x2^52*x3^31*x4^13*z^30 - x1^60*x2^45*x3^32*x4^13*z^30 - 5*x1^59*x2^46*x3^32*x4^13*z^30 - 3*x1^58*x2^47*x3^32*x4^13*z^30 - 2*x1^57*x2^48*x3^32*x4^13*z^30 - x1^56*x2^49*x3^32*x4^13*z^30 + x1^55*x2^50*x3^32*x4^13*z^30 + 2*x1^54*x2^51*x3^32*x4^13*z^30 + 2*x1^53*x2^52*x3^32*x4^13*z^30 + 4*x1^59*x2^45*x3^33*x4^13*z^30 + 5*x1^58*x2^46*x3^33*x4^13*z^30 - x1^57*x2^47*x3^33*x4^13*z^30 + 5*x1^56*x2^48*x3^33*x4^13*z^30 - x1^54*x2^50*x3^33*x4^13*z^30 - 3*x1^53*x2^51*x3^33*x4^13*z^30 - x1^60*x2^43*x3^34*x4^13*z^30 - x1^59*x2^44*x3^34*x4^13*z^30 - 4*x1^58*x2^45*x3^34*x4^13*z^30 - x1^57*x2^46*x3^34*x4^13*z^30 - 2*x1^56*x2^47*x3^34*x4^13*z^30 - 2*x1^55*x2^48*x3^34*x4^13*z^30 + 2*x1^54*x2^49*x3^34*x4^13*z^30 + x1^53*x2^50*x3^34*x4^13*z^30 + 3*x1^58*x2^44*x3^35*x4^13*z^30 + 3*x1^57*x2^45*x3^35*x4^13*z^30 + 4*x1^55*x2^47*x3^35*x4^13*z^30 - x1^58*x2^43*x3^36*x4^13*z^30 - 2*x1^57*x2^44*x3^36*x4^13*z^30 - x1^56*x2^45*x3^36*x4^13*z^30 - x1^55*x2^46*x3^36*x4^13*z^30 - 2*x1^54*x2^47*x3^36*x4^13*z^30 + 2*x1^56*x2^44*x3^37*x4^13*z^30 + 2*x1^55*x2^45*x3^37*x4^13*z^30 + x1^54*x2^46*x3^37*x4^13*z^30 + x1^64*x2^47*x3^25*x4^14*z^30 - x1^62*x2^48*x3^26*x4^14*z^30 - x1^63*x2^46*x3^27*x4^14*z^30 + 3*x1^61*x2^48*x3^27*x4^14*z^30 - x1^60*x2^49*x3^27*x4^14*z^30 - x1^59*x2^50*x3^27*x4^14*z^30 + 2*x1^62*x2^46*x3^28*x4^14*z^30 - x1^61*x2^47*x3^28*x4^14*z^30 - 2*x1^60*x2^48*x3^28*x4^14*z^30 - x1^58*x2^50*x3^28*x4^14*z^30 - x1^57*x2^51*x3^28*x4^14*z^30 - x1^62*x2^45*x3^29*x4^14*z^30 - 2*x1^61*x2^46*x3^29*x4^14*z^30 + 2*x1^60*x2^47*x3^29*x4^14*z^30 + x1^59*x2^48*x3^29*x4^14*z^30 + 2*x1^58*x2^49*x3^29*x4^14*z^30 - x1^57*x2^50*x3^29*x4^14*z^30 - x1^56*x2^51*x3^29*x4^14*z^30 - x1^55*x2^52*x3^29*x4^14*z^30 + 2*x1^61*x2^45*x3^30*x4^14*z^30 - x1^60*x2^46*x3^30*x4^14*z^30 - 2*x1^59*x2^47*x3^30*x4^14*z^30 + 4*x1^56*x2^50*x3^30*x4^14*z^30 + x1^55*x2^51*x3^30*x4^14*z^30 + x1^54*x2^52*x3^30*x4^14*z^30 - 2*x1^60*x2^45*x3^31*x4^14*z^30 + 2*x1^59*x2^46*x3^31*x4^14*z^30 - 2*x1^58*x2^47*x3^31*x4^14*z^30 - x1^57*x2^48*x3^31*x4^14*z^30 + x1^56*x2^49*x3^31*x4^14*z^30 + x1^55*x2^50*x3^31*x4^14*z^30 - x1^53*x2^52*x3^31*x4^14*z^30 + 2*x1^60*x2^44*x3^32*x4^14*z^30 + x1^59*x2^45*x3^32*x4^14*z^30 - x1^58*x2^46*x3^32*x4^14*z^30 + x1^57*x2^47*x3^32*x4^14*z^30 - 2*x1^56*x2^48*x3^32*x4^14*z^30 + x1^55*x2^49*x3^32*x4^14*z^30 + 2*x1^54*x2^50*x3^32*x4^14*z^30 + 2*x1^53*x2^51*x3^32*x4^14*z^30 + x1^52*x2^52*x3^32*x4^14*z^30 - x1^60*x2^43*x3^33*x4^14*z^30 - x1^59*x2^44*x3^33*x4^14*z^30 + 3*x1^58*x2^45*x3^33*x4^14*z^30 - x1^56*x2^47*x3^33*x4^14*z^30 + x1^55*x2^48*x3^33*x4^14*z^30 + x1^54*x2^49*x3^33*x4^14*z^30 - x1^52*x2^51*x3^33*x4^14*z^30 + 2*x1^59*x2^43*x3^34*x4^14*z^30 - x1^58*x2^44*x3^34*x4^14*z^30 + x1^56*x2^46*x3^34*x4^14*z^30 - 3*x1^55*x2^47*x3^34*x4^14*z^30 - x1^54*x2^48*x3^34*x4^14*z^30 + x1^53*x2^49*x3^34*x4^14*z^30 + 2*x1^52*x2^50*x3^34*x4^14*z^30 - x1^58*x2^43*x3^35*x4^14*z^30 + x1^57*x2^44*x3^35*x4^14*z^30 + x1^56*x2^45*x3^35*x4^14*z^30 + x1^55*x2^46*x3^35*x4^14*z^30 - 2*x1^53*x2^48*x3^35*x4^14*z^30 - x1^52*x2^49*x3^35*x4^14*z^30 - x1^57*x2^43*x3^36*x4^14*z^30 + x1^55*x2^45*x3^36*x4^14*z^30 - x1^54*x2^46*x3^36*x4^14*z^30 + x1^53*x2^46*x3^37*x4^14*z^30 - x1^54*x2^44*x3^38*x4^14*z^30 + x1^64*x2^47*x3^24*x4^15*z^30 - x1^63*x2^48*x3^24*x4^15*z^30 - 2*x1^63*x2^47*x3^25*x4^15*z^30 + x1^62*x2^48*x3^25*x4^15*z^30 + 2*x1^63*x2^46*x3^26*x4^15*z^30 + 2*x1^62*x2^47*x3^26*x4^15*z^30 - x1^61*x2^48*x3^26*x4^15*z^30 - x1^59*x2^50*x3^26*x4^15*z^30 - 6*x1^62*x2^46*x3^27*x4^15*z^30 - x1^61*x2^47*x3^27*x4^15*z^30 - x1^60*x2^48*x3^27*x4^15*z^30 + x1^59*x2^49*x3^27*x4^15*z^30 + 2*x1^58*x2^50*x3^27*x4^15*z^30 + 2*x1^62*x2^45*x3^28*x4^15*z^30 + 6*x1^61*x2^46*x3^28*x4^15*z^30 - x1^60*x2^47*x3^28*x4^15*z^30 + 2*x1^59*x2^48*x3^28*x4^15*z^30 - 3*x1^58*x2^49*x3^28*x4^15*z^30 - 2*x1^57*x2^50*x3^28*x4^15*z^30 - x1^56*x2^51*x3^28*x4^15*z^30 - 6*x1^61*x2^45*x3^29*x4^15*z^30 - 2*x1^60*x2^46*x3^29*x4^15*z^30 - x1^59*x2^47*x3^29*x4^15*z^30 + 3*x1^57*x2^49*x3^29*x4^15*z^30 + 2*x1^56*x2^50*x3^29*x4^15*z^30 + x1^55*x2^51*x3^29*x4^15*z^30 + 2*x1^61*x2^44*x3^30*x4^15*z^30 + 6*x1^60*x2^45*x3^30*x4^15*z^30 + 4*x1^58*x2^47*x3^30*x4^15*z^30 - 3*x1^57*x2^48*x3^30*x4^15*z^30 - 2*x1^56*x2^49*x3^30*x4^15*z^30 - 4*x1^55*x2^50*x3^30*x4^15*z^30 - 6*x1^60*x2^44*x3^31*x4^15*z^30 - 2*x1^59*x2^45*x3^31*x4^15*z^30 - 2*x1^58*x2^46*x3^31*x4^15*z^30 + x1^56*x2^48*x3^31*x4^15*z^30 + 2*x1^55*x2^49*x3^31*x4^15*z^30 + 3*x1^54*x2^50*x3^31*x4^15*z^30 + 2*x1^60*x2^43*x3^32*x4^15*z^30 + 6*x1^59*x2^44*x3^32*x4^15*z^30 + 4*x1^57*x2^46*x3^32*x4^15*z^30 - x1^56*x2^47*x3^32*x4^15*z^30 - x1^55*x2^48*x3^32*x4^15*z^30 - 5*x1^54*x2^49*x3^32*x4^15*z^30 - 6*x1^59*x2^43*x3^33*x4^15*z^30 - 2*x1^58*x2^44*x3^33*x4^15*z^30 - 2*x1^57*x2^45*x3^33*x4^15*z^30 - 2*x1^56*x2^46*x3^33*x4^15*z^30 + 2*x1^54*x2^48*x3^33*x4^15*z^30 + 2*x1^53*x2^49*x3^33*x4^15*z^30 + x1^59*x2^42*x3^34*x4^15*z^30 + 5*x1^58*x2^43*x3^34*x4^15*z^30 - x1^57*x2^44*x3^34*x4^15*z^30 + 4*x1^56*x2^45*x3^34*x4^15*z^30 + x1^55*x2^46*x3^34*x4^15*z^30 - x1^54*x2^47*x3^34*x4^15*z^30 - 3*x1^53*x2^48*x3^34*x4^15*z^30 - 3*x1^58*x2^42*x3^35*x4^15*z^30 - 2*x1^56*x2^44*x3^35*x4^15*z^30 - x1^55*x2^45*x3^35*x4^15*z^30 + x1^52*x2^48*x3^35*x4^15*z^30 + 2*x1^57*x2^42*x3^36*x4^15*z^30 + x1^56*x2^43*x3^36*x4^15*z^30 + 3*x1^55*x2^44*x3^36*x4^15*z^30 + x1^54*x2^45*x3^36*x4^15*z^30 - x1^54*x2^44*x3^37*x4^15*z^30 + 2*x1^63*x2^47*x3^24*x4^16*z^30 - 2*x1^63*x2^46*x3^25*x4^16*z^30 - 2*x1^62*x2^47*x3^25*x4^16*z^30 + x1^61*x2^48*x3^25*x4^16*z^30 - 2*x1^60*x2^49*x3^25*x4^16*z^30 + 5*x1^62*x2^46*x3^26*x4^16*z^30 + x1^61*x2^47*x3^26*x4^16*z^30 + x1^60*x2^48*x3^26*x4^16*z^30 + x1^59*x2^49*x3^26*x4^16*z^30 - x1^58*x2^50*x3^26*x4^16*z^30 - 2*x1^62*x2^45*x3^27*x4^16*z^30 - 5*x1^61*x2^46*x3^27*x4^16*z^30 - 3*x1^59*x2^48*x3^27*x4^16*z^30 + x1^57*x2^50*x3^27*x4^16*z^30 + 6*x1^61*x2^45*x3^28*x4^16*z^30 + 2*x1^60*x2^46*x3^28*x4^16*z^30 + 2*x1^59*x2^47*x3^28*x4^16*z^30 + x1^58*x2^48*x3^28*x4^16*z^30 - x1^56*x2^50*x3^28*x4^16*z^30 - 2*x1^61*x2^44*x3^29*x4^16*z^30 - 6*x1^60*x2^45*x3^29*x4^16*z^30 - 3*x1^58*x2^47*x3^29*x4^16*z^30 + 3*x1^57*x2^48*x3^29*x4^16*z^30 + 4*x1^55*x2^50*x3^29*x4^16*z^30 + 6*x1^60*x2^44*x3^30*x4^16*z^30 + 2*x1^59*x2^45*x3^30*x4^16*z^30 + 2*x1^58*x2^46*x3^30*x4^16*z^30 - x1^56*x2^48*x3^30*x4^16*z^30 - x1^55*x2^49*x3^30*x4^16*z^30 - 4*x1^54*x2^50*x3^30*x4^16*z^30 - 2*x1^60*x2^43*x3^31*x4^16*z^30 - 6*x1^59*x2^44*x3^31*x4^16*z^30 - 4*x1^57*x2^46*x3^31*x4^16*z^30 + 4*x1^56*x2^47*x3^31*x4^16*z^30 + 6*x1^54*x2^49*x3^31*x4^16*z^30 + x1^53*x2^50*x3^31*x4^16*z^30 + 6*x1^59*x2^43*x3^32*x4^16*z^30 + 2*x1^58*x2^44*x3^32*x4^16*z^30 + 2*x1^57*x2^45*x3^32*x4^16*z^30 - 2*x1^55*x2^47*x3^32*x4^16*z^30 - 2*x1^54*x2^48*x3^32*x4^16*z^30 - 6*x1^53*x2^49*x3^32*x4^16*z^30 - x1^59*x2^42*x3^33*x4^16*z^30 - 6*x1^58*x2^43*x3^33*x4^16*z^30 - 4*x1^56*x2^45*x3^33*x4^16*z^30 + 3*x1^55*x2^46*x3^33*x4^16*z^30 + x1^54*x2^47*x3^33*x4^16*z^30 + 5*x1^53*x2^48*x3^33*x4^16*z^30 + x1^52*x2^49*x3^33*x4^16*z^30 + 4*x1^58*x2^42*x3^34*x4^16*z^30 + 3*x1^57*x2^43*x3^34*x4^16*z^30 + 2*x1^56*x2^44*x3^34*x4^16*z^30 + x1^55*x2^45*x3^34*x4^16*z^30 - 2*x1^53*x2^47*x3^34*x4^16*z^30 - 4*x1^52*x2^48*x3^34*x4^16*z^30 - 4*x1^57*x2^42*x3^35*x4^16*z^30 - 2*x1^56*x2^43*x3^35*x4^16*z^30 - 3*x1^55*x2^44*x3^35*x4^16*z^30 + 4*x1^52*x2^47*x3^35*x4^16*z^30 + x1^57*x2^41*x3^36*x4^16*z^30 + 2*x1^56*x2^42*x3^36*x4^16*z^30 + 2*x1^55*x2^43*x3^36*x4^16*z^30 + x1^53*x2^45*x3^36*x4^16*z^30 - x1^52*x2^46*x3^36*x4^16*z^30 - 2*x1^51*x2^47*x3^36*x4^16*z^30 - x1^56*x2^41*x3^37*x4^16*z^30 - 2*x1^55*x2^42*x3^37*x4^16*z^30 - 2*x1^54*x2^43*x3^37*x4^16*z^30 + x1^52*x2^45*x3^37*x4^16*z^30 + x1^51*x2^46*x3^37*x4^16*z^30 + x1^54*x2^42*x3^38*x4^16*z^30 + x1^53*x2^43*x3^38*x4^16*z^30 - x1^62*x2^46*x3^25*x4^17*z^30 + x1^60*x2^48*x3^25*x4^17*z^30 + x1^61*x2^46*x3^26*x4^17*z^30 - x1^60*x2^47*x3^26*x4^17*z^30 - 2*x1^61*x2^45*x3^27*x4^17*z^30 - x1^60*x2^46*x3^27*x4^17*z^30 - x1^59*x2^47*x3^27*x4^17*z^30 + x1^57*x2^49*x3^27*x4^17*z^30 + x1^61*x2^44*x3^28*x4^17*z^30 + 2*x1^60*x2^45*x3^28*x4^17*z^30 + x1^58*x2^47*x3^28*x4^17*z^30 - x1^57*x2^48*x3^28*x4^17*z^30 - x1^56*x2^49*x3^28*x4^17*z^30 - x1^55*x2^50*x3^28*x4^17*z^30 - 2*x1^60*x2^44*x3^29*x4^17*z^30 - x1^59*x2^45*x3^29*x4^17*z^30 - x1^58*x2^46*x3^29*x4^17*z^30 + x1^56*x2^48*x3^29*x4^17*z^30 + x1^55*x2^49*x3^29*x4^17*z^30 + x1^54*x2^50*x3^29*x4^17*z^30 + x1^60*x2^43*x3^30*x4^17*z^30 + 2*x1^59*x2^44*x3^30*x4^17*z^30 + x1^57*x2^46*x3^30*x4^17*z^30 - 2*x1^56*x2^47*x3^30*x4^17*z^30 - 2*x1^54*x2^49*x3^30*x4^17*z^30 - 2*x1^59*x2^43*x3^31*x4^17*z^30 + 2*x1^53*x2^49*x3^31*x4^17*z^30 + 2*x1^58*x2^43*x3^32*x4^17*z^30 + 2*x1^56*x2^45*x3^32*x4^17*z^30 - x1^55*x2^46*x3^32*x4^17*z^30 - 2*x1^53*x2^48*x3^32*x4^17*z^30 - x1^52*x2^49*x3^32*x4^17*z^30 - x1^58*x2^42*x3^33*x4^17*z^30 - x1^57*x2^43*x3^33*x4^17*z^30 - x1^56*x2^44*x3^33*x4^17*z^30 + x1^54*x2^46*x3^33*x4^17*z^30 + x1^53*x2^47*x3^33*x4^17*z^30 + 2*x1^52*x2^48*x3^33*x4^17*z^30 + x1^57*x2^42*x3^34*x4^17*z^30 + x1^56*x2^43*x3^34*x4^17*z^30 + x1^55*x2^44*x3^34*x4^17*z^30 - x1^54*x2^45*x3^34*x4^17*z^30 - 2*x1^52*x2^47*x3^34*x4^17*z^30 - x1^51*x2^48*x3^34*x4^17*z^30 - x1^55*x2^43*x3^35*x4^17*z^30 + x1^53*x2^45*x3^35*x4^17*z^30 + x1^52*x2^46*x3^35*x4^17*z^30 + 2*x1^51*x2^47*x3^35*x4^17*z^30 + x1^52*x2^45*x3^36*x4^17*z^30 - x1^51*x2^46*x3^36*x4^17*z^30 - x1^52*x2^44*x3^37*x4^17*z^30 + x1^50*x2^46*x3^37*x4^17*z^30 - x1^65*x2^48*x3^27*x4^5*z^29 - x1^64*x2^48*x3^27*x4^6*z^29 + x1^64*x2^47*x3^28*x4^6*z^29 - x1^63*x2^47*x3^29*x4^6*z^29 - x1^62*x2^48*x3^29*x4^6*z^29 - 2*x1^61*x2^49*x3^29*x4^6*z^29 - x1^61*x2^48*x3^30*x4^6*z^29 + x1^64*x2^47*x3^27*x4^7*z^29 + x1^63*x2^48*x3^27*x4^7*z^29 - 2*x1^62*x2^49*x3^27*x4^7*z^29 + 2*x1^63*x2^47*x3^28*x4^7*z^29 + x1^62*x2^48*x3^28*x4^7*z^29 - x1^61*x2^48*x3^29*x4^7*z^29 - x1^60*x2^49*x3^29*x4^7*z^29 + 2*x1^62*x2^46*x3^30*x4^7*z^29 + x1^60*x2^48*x3^30*x4^7*z^29 - x1^59*x2^49*x3^30*x4^7*z^29 - x1^58*x2^50*x3^30*x4^7*z^29 - x1^58*x2^49*x3^31*x4^7*z^29 - x1^63*x2^49*x3^25*x4^8*z^29 - x1^64*x2^47*x3^26*x4^8*z^29 + 3*x1^62*x2^49*x3^26*x4^8*z^29 - 2*x1^61*x2^50*x3^26*x4^8*z^29 - x1^63*x2^47*x3^27*x4^8*z^29 - x1^62*x2^48*x3^27*x4^8*z^29 - 2*x1^61*x2^49*x3^27*x4^8*z^29 + x1^60*x2^50*x3^27*x4^8*z^29 + x1^63*x2^46*x3^28*x4^8*z^29 + 3*x1^61*x2^48*x3^28*x4^8*z^29 - x1^58*x2^51*x3^28*x4^8*z^29 - x1^62*x2^46*x3^29*x4^8*z^29 - x1^60*x2^48*x3^29*x4^8*z^29 + x1^59*x2^49*x3^29*x4^8*z^29 - x1^58*x2^50*x3^29*x4^8*z^29 + x1^61*x2^46*x3^30*x4^8*z^29 + 2*x1^60*x2^47*x3^30*x4^8*z^29 + x1^59*x2^48*x3^30*x4^8*z^29 + x1^58*x2^49*x3^30*x4^8*z^29 + x1^57*x2^50*x3^30*x4^8*z^29 - 2*x1^61*x2^45*x3^31*x4^8*z^29 + x1^60*x2^46*x3^31*x4^8*z^29 - x1^59*x2^47*x3^31*x4^8*z^29 - x1^58*x2^48*x3^31*x4^8*z^29 + x1^60*x2^45*x3^32*x4^8*z^29 + 2*x1^59*x2^46*x3^32*x4^8*z^29 + 2*x1^58*x2^47*x3^32*x4^8*z^29 + x1^57*x2^48*x3^32*x4^8*z^29 - x1^62*x2^49*x3^25*x4^9*z^29 + x1^63*x2^47*x3^26*x4^9*z^29 + x1^61*x2^49*x3^26*x4^9*z^29 + x1^63*x2^46*x3^27*x4^9*z^29 - 3*x1^61*x2^48*x3^27*x4^9*z^29 - x1^59*x2^50*x3^27*x4^9*z^29 - x1^58*x2^51*x3^27*x4^9*z^29 + x1^61*x2^47*x3^28*x4^9*z^29 + 2*x1^60*x2^48*x3^28*x4^9*z^29 - x1^59*x2^49*x3^28*x4^9*z^29 + x1^58*x2^50*x3^28*x4^9*z^29 + x1^57*x2^51*x3^28*x4^9*z^29 - 2*x1^62*x2^45*x3^29*x4^9*z^29 - x1^60*x2^47*x3^29*x4^9*z^29 - 2*x1^58*x2^49*x3^29*x4^9*z^29 - x1^56*x2^51*x3^29*x4^9*z^29 + 2*x1^61*x2^45*x3^30*x4^9*z^29 - 2*x1^60*x2^46*x3^30*x4^9*z^29 + x1^59*x2^47*x3^30*x4^9*z^29 - 2*x1^56*x2^50*x3^30*x4^9*z^29 - 2*x1^60*x2^45*x3^31*x4^9*z^29 - 2*x1^58*x2^47*x3^31*x4^9*z^29 - x1^57*x2^48*x3^31*x4^9*z^29 - x1^59*x2^45*x3^32*x4^9*z^29 - x1^58*x2^46*x3^32*x4^9*z^29 + x1^56*x2^48*x3^32*x4^9*z^29 - x1^59*x2^44*x3^33*x4^9*z^29 - 2*x1^58*x2^45*x3^33*x4^9*z^29 - 2*x1^57*x2^46*x3^33*x4^9*z^29 - x1^56*x2^47*x3^33*x4^9*z^29 - x1^64*x2^47*x3^24*x4^10*z^29 + 2*x1^63*x2^47*x3^25*x4^10*z^29 - x1^62*x2^48*x3^25*x4^10*z^29 - x1^63*x2^46*x3^26*x4^10*z^29 - 2*x1^62*x2^47*x3^26*x4^10*z^29 + x1^61*x2^48*x3^26*x4^10*z^29 - 2*x1^60*x2^49*x3^26*x4^10*z^29 + 2*x1^62*x2^46*x3^27*x4^10*z^29 + 2*x1^61*x2^47*x3^27*x4^10*z^29 + x1^59*x2^49*x3^27*x4^10*z^29 - x1^58*x2^50*x3^27*x4^10*z^29 - 2*x1^61*x2^46*x3^28*x4^10*z^29 + x1^57*x2^50*x3^28*x4^10*z^29 + x1^61*x2^45*x3^29*x4^10*z^29 + x1^60*x2^46*x3^29*x4^10*z^29 + 2*x1^59*x2^47*x3^29*x4^10*z^29 + 2*x1^58*x2^48*x3^29*x4^10*z^29 - x1^56*x2^50*x3^29*x4^10*z^29 + x1^60*x2^45*x3^30*x4^10*z^29 - 2*x1^59*x2^46*x3^30*x4^10*z^29 - 3*x1^58*x2^47*x3^30*x4^10*z^29 - x1^56*x2^49*x3^30*x4^10*z^29 + x1^55*x2^50*x3^30*x4^10*z^29 + 3*x1^58*x2^46*x3^31*x4^10*z^29 + x1^55*x2^49*x3^31*x4^10*z^29 - x1^54*x2^50*x3^31*x4^10*z^29 - x1^53*x2^51*x3^31*x4^10*z^29 + x1^59*x2^44*x3^32*x4^10*z^29 - 2*x1^58*x2^45*x3^32*x4^10*z^29 - x1^56*x2^47*x3^32*x4^10*z^29 - x1^55*x2^48*x3^32*x4^10*z^29 + x1^59*x2^43*x3^33*x4^10*z^29 + x1^58*x2^44*x3^33*x4^10*z^29 + x1^57*x2^45*x3^33*x4^10*z^29 - x1^57*x2^44*x3^34*x4^10*z^29 + x1^56*x2^45*x3^34*x4^10*z^29 - x1^55*x2^46*x3^34*x4^10*z^29 - x1^54*x2^47*x3^34*x4^10*z^29 + 2*x1^64*x2^47*x3^23*x4^11*z^29 - 3*x1^63*x2^47*x3^24*x4^11*z^29 + 2*x1^62*x2^48*x3^24*x4^11*z^29 + x1^63*x2^46*x3^25*x4^11*z^29 + 3*x1^62*x2^47*x3^25*x4^11*z^29 - x1^61*x2^48*x3^25*x4^11*z^29 + 2*x1^60*x2^49*x3^25*x4^11*z^29 - 3*x1^62*x2^46*x3^26*x4^11*z^29 - 2*x1^61*x2^47*x3^26*x4^11*z^29 + x1^59*x2^49*x3^26*x4^11*z^29 + 2*x1^58*x2^50*x3^26*x4^11*z^29 + 2*x1^62*x2^45*x3^27*x4^11*z^29 + 3*x1^61*x2^46*x3^27*x4^11*z^29 - x1^60*x2^47*x3^27*x4^11*z^29 + 2*x1^59*x2^48*x3^27*x4^11*z^29 - 2*x1^58*x2^49*x3^27*x4^11*z^29 - 2*x1^57*x2^50*x3^27*x4^11*z^29 - 4*x1^61*x2^45*x3^28*x4^11*z^29 - 3*x1^58*x2^48*x3^28*x4^11*z^29 + 3*x1^56*x2^50*x3^28*x4^11*z^29 + x1^61*x2^44*x3^29*x4^11*z^29 + 3*x1^60*x2^45*x3^29*x4^11*z^29 - x1^59*x2^46*x3^29*x4^11*z^29 - 3*x1^57*x2^48*x3^29*x4^11*z^29 + x1^56*x2^49*x3^29*x4^11*z^29 - 4*x1^55*x2^50*x3^29*x4^11*z^29 - 5*x1^60*x2^44*x3^30*x4^11*z^29 + x1^59*x2^45*x3^30*x4^11*z^29 + x1^58*x2^46*x3^30*x4^11*z^29 - 2*x1^57*x2^47*x3^30*x4^11*z^29 + x1^56*x2^48*x3^30*x4^11*z^29 + x1^55*x2^49*x3^30*x4^11*z^29 + 3*x1^54*x2^50*x3^30*x4^11*z^29 + x1^53*x2^51*x3^30*x4^11*z^29 + x1^60*x2^43*x3^31*x4^11*z^29 + 2*x1^59*x2^44*x3^31*x4^11*z^29 + 4*x1^57*x2^46*x3^31*x4^11*z^29 - x1^55*x2^48*x3^31*x4^11*z^29 - 2*x1^54*x2^49*x3^31*x4^11*z^29 + x1^53*x2^50*x3^31*x4^11*z^29 - 3*x1^59*x2^43*x3^32*x4^11*z^29 - x1^57*x2^45*x3^32*x4^11*z^29 - x1^56*x2^46*x3^32*x4^11*z^29 + 3*x1^55*x2^47*x3^32*x4^11*z^29 + 2*x1^52*x2^50*x3^32*x4^11*z^29 + x1^58*x2^43*x3^33*x4^11*z^29 + 4*x1^56*x2^45*x3^33*x4^11*z^29 + x1^55*x2^46*x3^33*x4^11*z^29 - x1^54*x2^47*x3^33*x4^11*z^29 - x1^58*x2^42*x3^34*x4^11*z^29 - x1^56*x2^44*x3^34*x4^11*z^29 + x1^54*x2^46*x3^34*x4^11*z^29 + x1^57*x2^42*x3^35*x4^11*z^29 + 2*x1^56*x2^43*x3^35*x4^11*z^29 + x1^54*x2^45*x3^35*x4^11*z^29 + x1^53*x2^46*x3^35*x4^11*z^29 - x1^63*x2^46*x3^24*x4^12*z^29 + x1^61*x2^48*x3^24*x4^12*z^29 + 2*x1^62*x2^46*x3^25*x4^12*z^29 - 2*x1^61*x2^47*x3^25*x4^12*z^29 - x1^60*x2^48*x3^25*x4^12*z^29 + x1^59*x2^49*x3^25*x4^12*z^29 - x1^62*x2^45*x3^26*x4^12*z^29 - 2*x1^61*x2^46*x3^26*x4^12*z^29 + 4*x1^60*x2^47*x3^26*x4^12*z^29 - 2*x1^59*x2^48*x3^26*x4^12*z^29 - x1^58*x2^49*x3^26*x4^12*z^29 + 2*x1^61*x2^45*x3^27*x4^12*z^29 - 3*x1^59*x2^47*x3^27*x4^12*z^29 + 3*x1^58*x2^48*x3^27*x4^12*z^29 - 2*x1^57*x2^49*x3^27*x4^12*z^29 - x1^61*x2^44*x3^28*x4^12*z^29 - 2*x1^60*x2^45*x3^28*x4^12*z^29 + 6*x1^59*x2^46*x3^28*x4^12*z^29 + x1^58*x2^47*x3^28*x4^12*z^29 + 2*x1^57*x2^48*x3^28*x4^12*z^29 + 2*x1^60*x2^44*x3^29*x4^12*z^29 - 2*x1^59*x2^45*x3^29*x4^12*z^29 - 6*x1^58*x2^46*x3^29*x4^12*z^29 + x1^57*x2^47*x3^29*x4^12*z^29 - 4*x1^56*x2^48*x3^29*x4^12*z^29 + x1^55*x2^49*x3^29*x4^12*z^29 + x1^53*x2^51*x3^29*x4^12*z^29 - x1^59*x2^44*x3^30*x4^12*z^29 + 5*x1^58*x2^45*x3^30*x4^12*z^29 + x1^57*x2^46*x3^30*x4^12*z^29 + 3*x1^56*x2^47*x3^30*x4^12*z^29 + x1^55*x2^48*x3^30*x4^12*z^29 - x1^54*x2^49*x3^30*x4^12*z^29 - 2*x1^53*x2^50*x3^30*x4^12*z^29 + 2*x1^59*x2^43*x3^31*x4^12*z^29 - 3*x1^58*x2^44*x3^31*x4^12*z^29 - 4*x1^57*x2^45*x3^31*x4^12*z^29 + x1^56*x2^46*x3^31*x4^12*z^29 - 5*x1^55*x2^47*x3^31*x4^12*z^29 + 2*x1^52*x2^50*x3^31*x4^12*z^29 - x1^59*x2^42*x3^32*x4^12*z^29 - x1^58*x2^43*x3^32*x4^12*z^29 + 4*x1^57*x2^44*x3^32*x4^12*z^29 + x1^55*x2^46*x3^32*x4^12*z^29 + 2*x1^54*x2^47*x3^32*x4^12*z^29 - x1^52*x2^49*x3^32*x4^12*z^29 - 2*x1^57*x2^43*x3^33*x4^12*z^29 - 2*x1^56*x2^44*x3^33*x4^12*z^29 + x1^55*x2^45*x3^33*x4^12*z^29 - 5*x1^54*x2^46*x3^33*x4^12*z^29 - x1^53*x2^47*x3^33*x4^12*z^29 + x1^58*x2^41*x3^34*x4^12*z^29 - x1^57*x2^42*x3^34*x4^12*z^29 + 2*x1^56*x2^43*x3^34*x4^12*z^29 + x1^54*x2^45*x3^34*x4^12*z^29 + 2*x1^53*x2^46*x3^34*x4^12*z^29 - x1^57*x2^41*x3^35*x4^12*z^29 - 2*x1^56*x2^42*x3^35*x4^12*z^29 - 2*x1^55*x2^43*x3^35*x4^12*z^29 - x1^53*x2^45*x3^35*x4^12*z^29 - x1^55*x2^42*x3^36*x4^12*z^29 + x1^63*x2^46*x3^23*x4^13*z^29 + x1^61*x2^47*x3^24*x4^13*z^29 - 3*x1^60*x2^47*x3^25*x4^13*z^29 - 2*x1^58*x2^49*x3^25*x4^13*z^29 + x1^60*x2^46*x3^26*x4^13*z^29 + 3*x1^59*x2^47*x3^26*x4^13*z^29 - 2*x1^58*x2^48*x3^26*x4^13*z^29 + x1^57*x2^49*x3^26*x4^13*z^29 - 2*x1^56*x2^50*x3^26*x4^13*z^29 - 5*x1^59*x2^46*x3^27*x4^13*z^29 + x1^55*x2^50*x3^27*x4^13*z^29 + 2*x1^59*x2^45*x3^28*x4^13*z^29 + 5*x1^58*x2^46*x3^28*x4^13*z^29 - 2*x1^57*x2^47*x3^28*x4^13*z^29 + 3*x1^56*x2^48*x3^28*x4^13*z^29 - 2*x1^55*x2^49*x3^28*x4^13*z^29 - x1^54*x2^50*x3^28*x4^13*z^29 - 2*x1^53*x2^51*x3^28*x4^13*z^29 - 6*x1^58*x2^45*x3^29*x4^13*z^29 - 2*x1^57*x2^46*x3^29*x4^13*z^29 + x1^56*x2^47*x3^29*x4^13*z^29 + 2*x1^55*x2^48*x3^29*x4^13*z^29 + 3*x1^54*x2^49*x3^29*x4^13*z^29 + 3*x1^53*x2^50*x3^29*x4^13*z^29 + 2*x1^52*x2^51*x3^29*x4^13*z^29 + 2*x1^58*x2^44*x3^30*x4^13*z^29 + 6*x1^57*x2^45*x3^30*x4^13*z^29 + 4*x1^55*x2^47*x3^30*x4^13*z^29 - 4*x1^54*x2^48*x3^30*x4^13*z^29 - 2*x1^53*x2^49*x3^30*x4^13*z^29 - 5*x1^52*x2^50*x3^30*x4^13*z^29 - 6*x1^57*x2^44*x3^31*x4^13*z^29 - 2*x1^56*x2^45*x3^31*x4^13*z^29 - x1^54*x2^47*x3^31*x4^13*z^29 + x1^53*x2^48*x3^31*x4^13*z^29 + 3*x1^52*x2^49*x3^31*x4^13*z^29 + 2*x1^51*x2^50*x3^31*x4^13*z^29 + x1^58*x2^42*x3^32*x4^13*z^29 + 3*x1^57*x2^43*x3^32*x4^13*z^29 + 5*x1^56*x2^44*x3^32*x4^13*z^29 + 5*x1^54*x2^46*x3^32*x4^13*z^29 - x1^53*x2^47*x3^32*x4^13*z^29 - x1^52*x2^48*x3^32*x4^13*z^29 - 4*x1^51*x2^49*x3^32*x4^13*z^29 + x1^58*x2^41*x3^33*x4^13*z^29 - 5*x1^56*x2^43*x3^33*x4^13*z^29 - 3*x1^55*x2^44*x3^33*x4^13*z^29 - x1^54*x2^45*x3^33*x4^13*z^29 - 2*x1^53*x2^46*x3^33*x4^13*z^29 + x1^52*x2^47*x3^33*x4^13*z^29 + x1^51*x2^48*x3^33*x4^13*z^29 + x1^50*x2^49*x3^33*x4^13*z^29 + x1^57*x2^41*x3^34*x4^13*z^29 + 3*x1^56*x2^42*x3^34*x4^13*z^29 + 3*x1^55*x2^43*x3^34*x4^13*z^29 + 4*x1^53*x2^45*x3^34*x4^13*z^29 - x1^51*x2^47*x3^34*x4^13*z^29 - 2*x1^50*x2^48*x3^34*x4^13*z^29 + x1^56*x2^41*x3^35*x4^13*z^29 - 2*x1^55*x2^42*x3^35*x4^13*z^29 - 2*x1^54*x2^43*x3^35*x4^13*z^29 - 2*x1^53*x2^44*x3^35*x4^13*z^29 - 2*x1^52*x2^45*x3^35*x4^13*z^29 + x1^54*x2^42*x3^36*x4^13*z^29 + 2*x1^52*x2^44*x3^36*x4^13*z^29 - x1^52*x2^43*x3^37*x4^13*z^29 - x1^51*x2^44*x3^37*x4^13*z^29 - x1^61*x2^47*x3^23*x4^14*z^29 - x1^62*x2^45*x3^24*x4^14*z^29 + x1^61*x2^46*x3^24*x4^14*z^29 + x1^60*x2^47*x3^24*x4^14*z^29 + x1^61*x2^45*x3^25*x4^14*z^29 - 3*x1^60*x2^46*x3^25*x4^14*z^29 - 2*x1^59*x2^47*x3^25*x4^14*z^29 + x1^58*x2^48*x3^25*x4^14*z^29 - x1^60*x2^45*x3^26*x4^14*z^29 + 4*x1^59*x2^46*x3^26*x4^14*z^29 + 2*x1^60*x2^44*x3^27*x4^14*z^29 + x1^59*x2^45*x3^27*x4^14*z^29 - 3*x1^58*x2^46*x3^27*x4^14*z^29 - x1^57*x2^47*x3^27*x4^14*z^29 - 2*x1^56*x2^48*x3^27*x4^14*z^29 + x1^55*x2^49*x3^27*x4^14*z^29 - x1^60*x2^43*x3^28*x4^14*z^29 - 2*x1^59*x2^44*x3^28*x4^14*z^29 + 2*x1^58*x2^45*x3^28*x4^14*z^29 + x1^57*x2^46*x3^28*x4^14*z^29 + 2*x1^56*x2^47*x3^28*x4^14*z^29 + x1^55*x2^48*x3^28*x4^14*z^29 + 2*x1^59*x2^43*x3^29*x4^14*z^29 - x1^57*x2^45*x3^29*x4^14*z^29 - 2*x1^55*x2^47*x3^29*x4^14*z^29 - x1^54*x2^48*x3^29*x4^14*z^29 + 2*x1^52*x2^50*x3^29*x4^14*z^29 - x1^59*x2^42*x3^30*x4^14*z^29 - 2*x1^58*x2^43*x3^30*x4^14*z^29 + 2*x1^57*x2^44*x3^30*x4^14*z^29 + 2*x1^55*x2^46*x3^30*x4^14*z^29 + x1^54*x2^47*x3^30*x4^14*z^29 - 2*x1^52*x2^49*x3^30*x4^14*z^29 - 2*x1^51*x2^50*x3^30*x4^14*z^29 + 2*x1^58*x2^42*x3^31*x4^14*z^29 - x1^57*x2^43*x3^31*x4^14*z^29 - 2*x1^56*x2^44*x3^31*x4^14*z^29 + 3*x1^53*x2^47*x3^31*x4^14*z^29 - x1^52*x2^48*x3^31*x4^14*z^29 + 2*x1^51*x2^49*x3^31*x4^14*z^29 - x1^58*x2^41*x3^32*x4^14*z^29 - 2*x1^57*x2^42*x3^32*x4^14*z^29 + 2*x1^56*x2^43*x3^32*x4^14*z^29 - 2*x1^55*x2^44*x3^32*x4^14*z^29 - 2*x1^54*x2^45*x3^32*x4^14*z^29 + x1^53*x2^46*x3^32*x4^14*z^29 + x1^52*x2^47*x3^32*x4^14*z^29 - x1^51*x2^48*x3^32*x4^14*z^29 - 2*x1^50*x2^49*x3^32*x4^14*z^29 + x1^57*x2^41*x3^33*x4^14*z^29 + 2*x1^56*x2^42*x3^33*x4^14*z^29 - x1^55*x2^43*x3^33*x4^14*z^29 - x1^54*x2^44*x3^33*x4^14*z^29 - x1^53*x2^45*x3^33*x4^14*z^29 + x1^52*x2^46*x3^33*x4^14*z^29 + x1^50*x2^48*x3^33*x4^14*z^29 - x1^56*x2^41*x3^34*x4^14*z^29 - x1^53*x2^44*x3^34*x4^14*z^29 + x1^52*x2^45*x3^34*x4^14*z^29 + x1^51*x2^46*x3^34*x4^14*z^29 - x1^49*x2^48*x3^34*x4^14*z^29 + x1^55*x2^41*x3^35*x4^14*z^29 + x1^54*x2^42*x3^35*x4^14*z^29 + x1^53*x2^43*x3^35*x4^14*z^29 - x1^52*x2^44*x3^35*x4^14*z^29 + x1^50*x2^46*x3^35*x4^14*z^29 + 2*x1^49*x2^47*x3^35*x4^14*z^29 + x1^53*x2^42*x3^36*x4^14*z^29 + x1^62*x2^45*x3^23*x4^15*z^29 + x1^61*x2^46*x3^23*x4^15*z^29 - x1^60*x2^47*x3^23*x4^15*z^29 - 4*x1^61*x2^45*x3^24*x4^15*z^29 + x1^59*x2^47*x3^24*x4^15*z^29 + 2*x1^61*x2^44*x3^25*x4^15*z^29 + 4*x1^60*x2^45*x3^25*x4^15*z^29 - x1^59*x2^46*x3^25*x4^15*z^29 + x1^58*x2^47*x3^25*x4^15*z^29 - x1^57*x2^48*x3^25*x4^15*z^29 - 6*x1^60*x2^44*x3^26*x4^15*z^29 - x1^59*x2^45*x3^26*x4^15*z^29 - x1^58*x2^46*x3^26*x4^15*z^29 + x1^56*x2^48*x3^26*x4^15*z^29 + x1^55*x2^49*x3^26*x4^15*z^29 + 2*x1^60*x2^43*x3^27*x4^15*z^29 + 6*x1^59*x2^44*x3^27*x4^15*z^29 + 5*x1^57*x2^46*x3^27*x4^15*z^29 - x1^56*x2^47*x3^27*x4^15*z^29 - x1^55*x2^48*x3^27*x4^15*z^29 - 2*x1^54*x2^49*x3^27*x4^15*z^29 - 6*x1^59*x2^43*x3^28*x4^15*z^29 - 2*x1^58*x2^44*x3^28*x4^15*z^29 - 2*x1^57*x2^45*x3^28*x4^15*z^29 + 2*x1^55*x2^47*x3^28*x4^15*z^29 + 3*x1^54*x2^48*x3^28*x4^15*z^29 + 2*x1^53*x2^49*x3^28*x4^15*z^29 + 2*x1^59*x2^42*x3^29*x4^15*z^29 + 6*x1^58*x2^43*x3^29*x4^15*z^29 + 4*x1^56*x2^45*x3^29*x4^15*z^29 - 4*x1^55*x2^46*x3^29*x4^15*z^29 - x1^54*x2^47*x3^29*x4^15*z^29 - 5*x1^53*x2^48*x3^29*x4^15*z^29 - x1^52*x2^49*x3^29*x4^15*z^29 - 6*x1^58*x2^42*x3^30*x4^15*z^29 - 2*x1^57*x2^43*x3^30*x4^15*z^29 - 2*x1^56*x2^44*x3^30*x4^15*z^29 + 2*x1^54*x2^46*x3^30*x4^15*z^29 + 2*x1^53*x2^47*x3^30*x4^15*z^29 + 5*x1^52*x2^48*x3^30*x4^15*z^29 + 2*x1^58*x2^41*x3^31*x4^15*z^29 + 6*x1^57*x2^42*x3^31*x4^15*z^29 + 4*x1^55*x2^44*x3^31*x4^15*z^29 - 3*x1^54*x2^45*x3^31*x4^15*z^29 - x1^53*x2^46*x3^31*x4^15*z^29 - 5*x1^52*x2^47*x3^31*x4^15*z^29 - x1^51*x2^48*x3^31*x4^15*z^29 - 5*x1^57*x2^41*x3^32*x4^15*z^29 - 2*x1^56*x2^42*x3^32*x4^15*z^29 - 2*x1^55*x2^43*x3^32*x4^15*z^29 - x1^54*x2^44*x3^32*x4^15*z^29 + 2*x1^52*x2^46*x3^32*x4^15*z^29 + 4*x1^51*x2^47*x3^32*x4^15*z^29 + 2*x1^57*x2^40*x3^33*x4^15*z^29 + 5*x1^56*x2^41*x3^33*x4^15*z^29 + 4*x1^54*x2^43*x3^33*x4^15*z^29 - 4*x1^51*x2^46*x3^33*x4^15*z^29 - 3*x1^56*x2^40*x3^34*x4^15*z^29 - 2*x1^55*x2^41*x3^34*x4^15*z^29 - x1^54*x2^42*x3^34*x4^15*z^29 - x1^53*x2^43*x3^34*x4^15*z^29 - x1^52*x2^44*x3^34*x4^15*z^29 + x1^51*x2^45*x3^34*x4^15*z^29 + 2*x1^50*x2^46*x3^34*x4^15*z^29 + 2*x1^55*x2^40*x3^35*x4^15*z^29 + 3*x1^53*x2^42*x3^35*x4^15*z^29 - x1^52*x2^43*x3^35*x4^15*z^29 - x1^51*x2^44*x3^35*x4^15*z^29 - x1^50*x2^45*x3^35*x4^15*z^29 - x1^54*x2^40*x3^36*x4^15*z^29 - x1^53*x2^41*x3^36*x4^15*z^29 - x1^52*x2^42*x3^36*x4^15*z^29 - x1^51*x2^43*x3^36*x4^15*z^29 - x1^50*x2^44*x3^36*x4^15*z^29 - x1^51*x2^42*x3^37*x4^15*z^29 + x1^61*x2^45*x3^23*x4^16*z^29 + x1^59*x2^47*x3^23*x4^16*z^29 - x1^61*x2^44*x3^24*x4^16*z^29 - 3*x1^60*x2^45*x3^24*x4^16*z^29 - x1^58*x2^47*x3^24*x4^16*z^29 + x1^57*x2^48*x3^24*x4^16*z^29 + 5*x1^60*x2^44*x3^25*x4^16*z^29 + 2*x1^59*x2^45*x3^25*x4^16*z^29 + x1^57*x2^47*x3^25*x4^16*z^29 - 2*x1^60*x2^43*x3^26*x4^16*z^29 - 6*x1^59*x2^44*x3^26*x4^16*z^29 + x1^58*x2^45*x3^26*x4^16*z^29 - 4*x1^57*x2^46*x3^26*x4^16*z^29 + x1^56*x2^47*x3^26*x4^16*z^29 + x1^54*x2^49*x3^26*x4^16*z^29 + 6*x1^59*x2^43*x3^27*x4^16*z^29 + 2*x1^58*x2^44*x3^27*x4^16*z^29 + x1^57*x2^45*x3^27*x4^16*z^29 - x1^55*x2^47*x3^27*x4^16*z^29 - 2*x1^54*x2^48*x3^27*x4^16*z^29 - x1^53*x2^49*x3^27*x4^16*z^29 - 2*x1^59*x2^42*x3^28*x4^16*z^29 - 6*x1^58*x2^43*x3^28*x4^16*z^29 - 4*x1^56*x2^45*x3^28*x4^16*z^29 + 3*x1^55*x2^46*x3^28*x4^16*z^29 + 4*x1^53*x2^48*x3^28*x4^16*z^29 + 6*x1^58*x2^42*x3^29*x4^16*z^29 + 2*x1^57*x2^43*x3^29*x4^16*z^29 + 2*x1^56*x2^44*x3^29*x4^16*z^29 - 2*x1^54*x2^46*x3^29*x4^16*z^29 - 3*x1^53*x2^47*x3^29*x4^16*z^29 - 4*x1^52*x2^48*x3^29*x4^16*z^29 - 2*x1^58*x2^41*x3^30*x4^16*z^29 - 6*x1^57*x2^42*x3^30*x4^16*z^29 - 4*x1^55*x2^44*x3^30*x4^16*z^29 + 4*x1^54*x2^45*x3^30*x4^16*z^29 + 6*x1^52*x2^47*x3^30*x4^16*z^29 + x1^51*x2^48*x3^30*x4^16*z^29 + 6*x1^57*x2^41*x3^31*x4^16*z^29 + 2*x1^56*x2^42*x3^31*x4^16*z^29 + 2*x1^55*x2^43*x3^31*x4^16*z^29 - 2*x1^53*x2^45*x3^31*x4^16*z^29 - 2*x1^52*x2^46*x3^31*x4^16*z^29 - 6*x1^51*x2^47*x3^31*x4^16*z^29 - x1^57*x2^40*x3^32*x4^16*z^29 - 6*x1^56*x2^41*x3^32*x4^16*z^29 - 4*x1^54*x2^43*x3^32*x4^16*z^29 + 4*x1^53*x2^44*x3^32*x4^16*z^29 + 6*x1^51*x2^46*x3^32*x4^16*z^29 + 2*x1^50*x2^47*x3^32*x4^16*z^29 + 2*x1^56*x2^40*x3^33*x4^16*z^29 + 3*x1^55*x2^41*x3^33*x4^16*z^29 + x1^54*x2^42*x3^33*x4^16*z^29 + x1^53*x2^43*x3^33*x4^16*z^29 - x1^52*x2^44*x3^33*x4^16*z^29 - 3*x1^51*x2^45*x3^33*x4^16*z^29 - 5*x1^50*x2^46*x3^33*x4^16*z^29 - 2*x1^55*x2^40*x3^34*x4^16*z^29 - 2*x1^54*x2^41*x3^34*x4^16*z^29 - 3*x1^53*x2^42*x3^34*x4^16*z^29 + x1^52*x2^43*x3^34*x4^16*z^29 + 4*x1^50*x2^45*x3^34*x4^16*z^29 + x1^49*x2^46*x3^34*x4^16*z^29 + x1^54*x2^40*x3^35*x4^16*z^29 + 2*x1^53*x2^41*x3^35*x4^16*z^29 + x1^52*x2^42*x3^35*x4^16*z^29 + x1^51*x2^43*x3^35*x4^16*z^29 - 2*x1^50*x2^44*x3^35*x4^16*z^29 - 2*x1^49*x2^45*x3^35*x4^16*z^29 - x1^53*x2^40*x3^36*x4^16*z^29 - 2*x1^52*x2^41*x3^36*x4^16*z^29 + 2*x1^49*x2^44*x3^36*x4^16*z^29 + x1^52*x2^40*x3^37*x4^16*z^29 + x1^50*x2^42*x3^37*x4^16*z^29 - x1^48*x2^44*x3^37*x4^16*z^29 - x1^60*x2^44*x3^24*x4^17*z^29 + x1^58*x2^46*x3^24*x4^17*z^29 + x1^56*x2^48*x3^24*x4^17*z^29 + x1^59*x2^44*x3^25*x4^17*z^29 + x1^58*x2^45*x3^25*x4^17*z^29 - x1^57*x2^46*x3^25*x4^17*z^29 - 2*x1^56*x2^47*x3^25*x4^17*z^29 - x1^55*x2^48*x3^25*x4^17*z^29 - x1^59*x2^43*x3^26*x4^17*z^29 - x1^58*x2^44*x3^26*x4^17*z^29 + x1^57*x2^45*x3^26*x4^17*z^29 + x1^56*x2^46*x3^26*x4^17*z^29 + x1^54*x2^48*x3^26*x4^17*z^29 + 2*x1^58*x2^43*x3^27*x4^17*z^29 + 2*x1^56*x2^45*x3^27*x4^17*z^29 - x1^55*x2^46*x3^27*x4^17*z^29 - 2*x1^53*x2^48*x3^27*x4^17*z^29 - 2*x1^58*x2^42*x3^28*x4^17*z^29 - x1^57*x2^43*x3^28*x4^17*z^29 - x1^56*x2^44*x3^28*x4^17*z^29 + x1^54*x2^46*x3^28*x4^17*z^29 + x1^53*x2^47*x3^28*x4^17*z^29 + 2*x1^52*x2^48*x3^28*x4^17*z^29 + x1^58*x2^41*x3^29*x4^17*z^29 + 2*x1^57*x2^42*x3^29*x4^17*z^29 + x1^55*x2^44*x3^29*x4^17*z^29 - x1^54*x2^45*x3^29*x4^17*z^29 - 2*x1^52*x2^47*x3^29*x4^17*z^29 - x1^51*x2^48*x3^29*x4^17*z^29 - 2*x1^57*x2^41*x3^30*x4^17*z^29 - x1^56*x2^42*x3^30*x4^17*z^29 - x1^55*x2^43*x3^30*x4^17*z^29 + x1^53*x2^45*x3^30*x4^17*z^29 + x1^52*x2^46*x3^30*x4^17*z^29 + 2*x1^51*x2^47*x3^30*x4^17*z^29 + 2*x1^56*x2^41*x3^31*x4^17*z^29 + x1^54*x2^43*x3^31*x4^17*z^29 - 2*x1^53*x2^44*x3^31*x4^17*z^29 - 2*x1^51*x2^46*x3^31*x4^17*z^29 - x1^55*x2^41*x3^32*x4^17*z^29 + 2*x1^50*x2^46*x3^32*x4^17*z^29 + x1^53*x2^42*x3^33*x4^17*z^29 - x1^52*x2^43*x3^33*x4^17*z^29 - 2*x1^50*x2^45*x3^33*x4^17*z^29 - x1^49*x2^46*x3^33*x4^17*z^29 + x1^50*x2^44*x3^34*x4^17*z^29 + 2*x1^49*x2^45*x3^34*x4^17*z^29 - x1^51*x2^42*x3^35*x4^17*z^29 + x1^50*x2^43*x3^35*x4^17*z^29 - 2*x1^49*x2^44*x3^35*x4^17*z^29 - x1^48*x2^45*x3^35*x4^17*z^29 + x1^50*x2^42*x3^36*x4^17*z^29 + x1^61*x2^48*x3^27*x4^4*z^28 - x1^63*x2^46*x3^26*x4^5*z^28 - x1^62*x2^47*x3^26*x4^5*z^28 + 2*x1^62*x2^46*x3^27*x4^5*z^28 + x1^60*x2^48*x3^27*x4^5*z^28 - x1^62*x2^45*x3^28*x4^5*z^28 + x1^60*x2^47*x3^28*x4^5*z^28 - x1^59*x2^48*x3^28*x4^5*z^28 - x1^60*x2^46*x3^29*x4^5*z^28 + x1^63*x2^46*x3^25*x4^6*z^28 - 2*x1^62*x2^46*x3^26*x4^6*z^28 + x1^61*x2^47*x3^26*x4^6*z^28 + x1^62*x2^45*x3^27*x4^6*z^28 + x1^61*x2^46*x3^27*x4^6*z^28 - x1^60*x2^47*x3^27*x4^6*z^28 + x1^59*x2^48*x3^27*x4^6*z^28 - 2*x1^61*x2^45*x3^28*x4^6*z^28 - x1^59*x2^47*x3^28*x4^6*z^28 + x1^61*x2^44*x3^29*x4^6*z^28 - x1^59*x2^46*x3^29*x4^6*z^28 + 2*x1^58*x2^47*x3^29*x4^6*z^28 + x1^57*x2^48*x3^29*x4^6*z^28 + x1^59*x2^45*x3^30*x4^6*z^28 + x1^56*x2^48*x3^30*x4^6*z^28 - x1^62*x2^46*x3^25*x4^7*z^28 + x1^61*x2^47*x3^25*x4^7*z^28 - x1^59*x2^49*x3^25*x4^7*z^28 - x1^60*x2^47*x3^26*x4^7*z^28 + x1^59*x2^48*x3^26*x4^7*z^28 - 2*x1^60*x2^46*x3^27*x4^7*z^28 - x1^59*x2^47*x3^27*x4^7*z^28 + x1^57*x2^49*x3^27*x4^7*z^28 - x1^60*x2^45*x3^28*x4^7*z^28 - x1^59*x2^46*x3^28*x4^7*z^28 - x1^58*x2^47*x3^28*x4^7*z^28 + x1^56*x2^49*x3^28*x4^7*z^28 + 2*x1^60*x2^44*x3^29*x4^7*z^28 - x1^59*x2^45*x3^29*x4^7*z^28 + x1^57*x2^47*x3^29*x4^7*z^28 - x1^59*x2^44*x3^30*x4^7*z^28 - 2*x1^57*x2^46*x3^30*x4^7*z^28 - x1^56*x2^47*x3^30*x4^7*z^28 + x1^55*x2^48*x3^30*x4^7*z^28 - 2*x1^61*x2^47*x3^24*x4^8*z^28 + x1^62*x2^45*x3^25*x4^8*z^28 + 4*x1^60*x2^47*x3^25*x4^8*z^28 + x1^58*x2^49*x3^25*x4^8*z^28 + x1^61*x2^45*x3^26*x4^8*z^28 - 4*x1^59*x2^47*x3^26*x4^8*z^28 + 2*x1^58*x2^48*x3^26*x4^8*z^28 + x1^56*x2^50*x3^26*x4^8*z^28 + x1^60*x2^45*x3^27*x4^8*z^28 + 4*x1^59*x2^46*x3^27*x4^8*z^28 + x1^58*x2^47*x3^27*x4^8*z^28 - x1^55*x2^50*x3^27*x4^8*z^28 - 2*x1^60*x2^44*x3^28*x4^8*z^28 - 2*x1^59*x2^45*x3^28*x4^8*z^28 - 3*x1^58*x2^46*x3^28*x4^8*z^28 - 2*x1^56*x2^48*x3^28*x4^8*z^28 + 2*x1^59*x2^44*x3^29*x4^8*z^28 + x1^58*x2^45*x3^29*x4^8*z^28 + x1^56*x2^47*x3^29*x4^8*z^28 - x1^54*x2^49*x3^29*x4^8*z^28 - x1^59*x2^43*x3^30*x4^8*z^28 - x1^58*x2^44*x3^30*x4^8*z^28 - 2*x1^57*x2^45*x3^30*x4^8*z^28 - 2*x1^55*x2^47*x3^30*x4^8*z^28 + 2*x1^58*x2^43*x3^31*x4^8*z^28 + x1^57*x2^44*x3^31*x4^8*z^28 + x1^56*x2^45*x3^31*x4^8*z^28 + x1^55*x2^46*x3^31*x4^8*z^28 - x1^58*x2^42*x3^32*x4^8*z^28 - x1^56*x2^44*x3^32*x4^8*z^28 - x1^55*x2^45*x3^32*x4^8*z^28 - 2*x1^54*x2^46*x3^32*x4^8*z^28 + x1^61*x2^47*x3^23*x4^9*z^28 - x1^62*x2^45*x3^24*x4^9*z^28 - x1^60*x2^47*x3^24*x4^9*z^28 + x1^60*x2^46*x3^25*x4^9*z^28 + x1^59*x2^47*x3^25*x4^9*z^28 + x1^57*x2^49*x3^25*x4^9*z^28 - x1^61*x2^44*x3^26*x4^9*z^28 - 3*x1^59*x2^46*x3^26*x4^9*z^28 - x1^58*x2^47*x3^26*x4^9*z^28 - x1^57*x2^48*x3^26*x4^9*z^28 - x1^56*x2^49*x3^26*x4^9*z^28 + x1^60*x2^44*x3^27*x4^9*z^28 - 2*x1^59*x2^45*x3^27*x4^9*z^28 + 2*x1^58*x2^46*x3^27*x4^9*z^28 + x1^56*x2^48*x3^27*x4^9*z^28 + x1^55*x2^49*x3^27*x4^9*z^28 + x1^54*x2^50*x3^27*x4^9*z^28 - x1^60*x2^43*x3^28*x4^9*z^28 - x1^59*x2^44*x3^28*x4^9*z^28 + x1^58*x2^45*x3^28*x4^9*z^28 - 3*x1^57*x2^46*x3^28*x4^9*z^28 - x1^56*x2^47*x3^28*x4^9*z^28 + x1^55*x2^48*x3^28*x4^9*z^28 - x1^54*x2^49*x3^28*x4^9*z^28 + 2*x1^59*x2^43*x3^29*x4^9*z^28 + 2*x1^58*x2^44*x3^29*x4^9*z^28 + 2*x1^57*x2^45*x3^29*x4^9*z^28 + x1^55*x2^47*x3^29*x4^9*z^28 + x1^53*x2^49*x3^29*x4^9*z^28 - x1^59*x2^42*x3^30*x4^9*z^28 - 3*x1^58*x2^43*x3^30*x4^9*z^28 - x1^55*x2^46*x3^30*x4^9*z^28 - x1^54*x2^47*x3^30*x4^9*z^28 + 2*x1^53*x2^48*x3^30*x4^9*z^28 + x1^52*x2^49*x3^30*x4^9*z^28 + x1^56*x2^44*x3^31*x4^9*z^28 + x1^55*x2^45*x3^31*x4^9*z^28 + x1^54*x2^46*x3^31*x4^9*z^28 - x1^57*x2^42*x3^32*x4^9*z^28 + x1^54*x2^45*x3^32*x4^9*z^28 + x1^57*x2^41*x3^33*x4^9*z^28 + x1^55*x2^43*x3^33*x4^9*z^28 + x1^54*x2^44*x3^33*x4^9*z^28 + 2*x1^53*x2^45*x3^33*x4^9*z^28 + x1^62*x2^46*x3^22*x4^10*z^28 - x1^62*x2^45*x3^23*x4^10*z^28 - x1^61*x2^46*x3^23*x4^10*z^28 + x1^60*x2^47*x3^23*x4^10*z^28 + 3*x1^61*x2^45*x3^24*x4^10*z^28 + x1^58*x2^48*x3^24*x4^10*z^28 - 3*x1^60*x2^45*x3^25*x4^10*z^28 + x1^60*x2^44*x3^26*x4^10*z^28 + x1^59*x2^45*x3^26*x4^10*z^28 + x1^58*x2^46*x3^26*x4^10*z^28 + x1^55*x2^49*x3^26*x4^10*z^28 - x1^59*x2^44*x3^27*x4^10*z^28 - x1^58*x2^45*x3^27*x4^10*z^28 - 2*x1^57*x2^46*x3^27*x4^10*z^28 + x1^54*x2^49*x3^27*x4^10*z^28 + x1^57*x2^45*x3^28*x4^10*z^28 - x1^56*x2^46*x3^28*x4^10*z^28 - x1^55*x2^47*x3^28*x4^10*z^28 - 2*x1^54*x2^48*x3^28*x4^10*z^28 - x1^53*x2^49*x3^28*x4^10*z^28 + x1^52*x2^50*x3^28*x4^10*z^28 - 2*x1^57*x2^44*x3^29*x4^10*z^28 - 2*x1^54*x2^47*x3^29*x4^10*z^28 + 2*x1^53*x2^48*x3^29*x4^10*z^28 + x1^52*x2^49*x3^29*x4^10*z^28 + x1^58*x2^42*x3^30*x4^10*z^28 - x1^57*x2^43*x3^30*x4^10*z^28 + x1^56*x2^44*x3^30*x4^10*z^28 + x1^53*x2^47*x3^30*x4^10*z^28 - 2*x1^52*x2^48*x3^30*x4^10*z^28 - x1^51*x2^49*x3^30*x4^10*z^28 - x1^55*x2^44*x3^31*x4^10*z^28 - x1^54*x2^45*x3^31*x4^10*z^28 + x1^53*x2^46*x3^31*x4^10*z^28 + x1^50*x2^49*x3^31*x4^10*z^28 + x1^57*x2^41*x3^32*x4^10*z^28 + x1^53*x2^45*x3^32*x4^10*z^28 - x1^55*x2^42*x3^33*x4^10*z^28 - 2*x1^53*x2^44*x3^33*x4^10*z^28 + x1^55*x2^41*x3^34*x4^10*z^28 + x1^54*x2^42*x3^34*x4^10*z^28 - x1^53*x2^43*x3^34*x4^10*z^28 + x1^52*x2^44*x3^34*x4^10*z^28 + x1^62*x2^45*x3^22*x4^11*z^28 - 4*x1^61*x2^45*x3^23*x4^11*z^28 - x1^60*x2^46*x3^23*x4^11*z^28 - 2*x1^59*x2^47*x3^23*x4^11*z^28 + x1^61*x2^44*x3^24*x4^11*z^28 + 4*x1^60*x2^45*x3^24*x4^11*z^28 - x1^59*x2^46*x3^24*x4^11*z^28 - 2*x1^57*x2^48*x3^24*x4^11*z^28 - 4*x1^60*x2^44*x3^25*x4^11*z^28 + x1^59*x2^45*x3^25*x4^11*z^28 + x1^56*x2^48*x3^25*x4^11*z^28 + x1^60*x2^43*x3^26*x4^11*z^28 + 4*x1^59*x2^44*x3^26*x4^11*z^28 - 3*x1^58*x2^45*x3^26*x4^11*z^28 + 4*x1^57*x2^46*x3^26*x4^11*z^28 - 2*x1^55*x2^48*x3^26*x4^11*z^28 - 3*x1^54*x2^49*x3^26*x4^11*z^28 - 4*x1^59*x2^43*x3^27*x4^11*z^28 - 2*x1^58*x2^44*x3^27*x4^11*z^28 + x1^57*x2^45*x3^27*x4^11*z^28 + 3*x1^55*x2^47*x3^27*x4^11*z^28 + 2*x1^54*x2^48*x3^27*x4^11*z^28 + 3*x1^53*x2^49*x3^27*x4^11*z^28 + 2*x1^59*x2^42*x3^28*x4^11*z^28 + 4*x1^58*x2^43*x3^28*x4^11*z^28 - 2*x1^57*x2^44*x3^28*x4^11*z^28 + x1^56*x2^45*x3^28*x4^11*z^28 - x1^55*x2^46*x3^28*x4^11*z^28 + x1^54*x2^47*x3^28*x4^11*z^28 - 2*x1^53*x2^48*x3^28*x4^11*z^28 - x1^52*x2^49*x3^28*x4^11*z^28 - 3*x1^58*x2^42*x3^29*x4^11*z^28 + x1^57*x2^43*x3^29*x4^11*z^28 - x1^55*x2^45*x3^29*x4^11*z^28 + 3*x1^54*x2^46*x3^29*x4^11*z^28 + 3*x1^53*x2^47*x3^29*x4^11*z^28 + 2*x1^52*x2^48*x3^29*x4^11*z^28 + x1^58*x2^41*x3^30*x4^11*z^28 + 5*x1^57*x2^42*x3^30*x4^11*z^28 + x1^55*x2^44*x3^30*x4^11*z^28 - 2*x1^54*x2^45*x3^30*x4^11*z^28 - x1^52*x2^47*x3^30*x4^11*z^28 - x1^50*x2^49*x3^30*x4^11*z^28 - 4*x1^57*x2^41*x3^31*x4^11*z^28 + 2*x1^56*x2^42*x3^31*x4^11*z^28 - 2*x1^54*x2^44*x3^31*x4^11*z^28 + x1^53*x2^45*x3^31*x4^11*z^28 + 2*x1^51*x2^47*x3^31*x4^11*z^28 - x1^57*x2^40*x3^32*x4^11*z^28 + 2*x1^56*x2^41*x3^32*x4^11*z^28 + 2*x1^54*x2^43*x3^32*x4^11*z^28 + x1^53*x2^44*x3^32*x4^11*z^28 - 2*x1^52*x2^45*x3^32*x4^11*z^28 - 2*x1^51*x2^46*x3^32*x4^11*z^28 - x1^49*x2^48*x3^32*x4^11*z^28 + x1^54*x2^42*x3^33*x4^11*z^28 - x1^53*x2^43*x3^33*x4^11*z^28 - x1^52*x2^44*x3^33*x4^11*z^28 + x1^52*x2^43*x3^34*x4^11*z^28 - x1^51*x2^44*x3^34*x4^11*z^28 - x1^53*x2^41*x3^35*x4^11*z^28 - x1^51*x2^43*x3^35*x4^11*z^28 + x1^61*x2^45*x3^22*x4^12*z^28 - x1^60*x2^46*x3^22*x4^12*z^28 - x1^61*x2^44*x3^23*x4^12*z^28 - x1^60*x2^45*x3^23*x4^12*z^28 + 3*x1^59*x2^46*x3^23*x4^12*z^28 - x1^58*x2^47*x3^23*x4^12*z^28 + 3*x1^60*x2^44*x3^24*x4^12*z^28 - 2*x1^59*x2^45*x3^24*x4^12*z^28 - 2*x1^58*x2^46*x3^24*x4^12*z^28 + x1^57*x2^47*x3^24*x4^12*z^28 - x1^56*x2^48*x3^24*x4^12*z^28 - 3*x1^59*x2^44*x3^25*x4^12*z^28 + 5*x1^58*x2^45*x3^25*x4^12*z^28 - x1^57*x2^46*x3^25*x4^12*z^28 + x1^56*x2^47*x3^25*x4^12*z^28 - x1^54*x2^49*x3^25*x4^12*z^28 + 2*x1^59*x2^43*x3^26*x4^12*z^28 - x1^58*x2^44*x3^26*x4^12*z^28 - 4*x1^57*x2^45*x3^26*x4^12*z^28 + x1^56*x2^46*x3^26*x4^12*z^28 - 2*x1^55*x2^47*x3^26*x4^12*z^28 + x1^54*x2^48*x3^26*x4^12*z^28 + x1^53*x2^49*x3^26*x4^12*z^28 - x1^59*x2^42*x3^27*x4^12*z^28 - 2*x1^58*x2^43*x3^27*x4^12*z^28 + 6*x1^57*x2^44*x3^27*x4^12*z^28 - x1^55*x2^46*x3^27*x4^12*z^28 - x1^52*x2^49*x3^27*x4^12*z^28 + 2*x1^58*x2^42*x3^28*x4^12*z^28 - x1^57*x2^43*x3^28*x4^12*z^28 - 5*x1^56*x2^44*x3^28*x4^12*z^28 - 4*x1^54*x2^46*x3^28*x4^12*z^28 + 2*x1^53*x2^47*x3^28*x4^12*z^28 + x1^52*x2^48*x3^28*x4^12*z^28 + 2*x1^51*x2^49*x3^28*x4^12*z^28 - x1^58*x2^41*x3^29*x4^12*z^28 - 2*x1^57*x2^42*x3^29*x4^12*z^28 + 6*x1^56*x2^43*x3^29*x4^12*z^28 + x1^55*x2^44*x3^29*x4^12*z^28 + 2*x1^54*x2^45*x3^29*x4^12*z^28 + x1^53*x2^46*x3^29*x4^12*z^28 + x1^52*x2^47*x3^29*x4^12*z^28 - 2*x1^51*x2^48*x3^29*x4^12*z^28 - x1^50*x2^49*x3^29*x4^12*z^28 + x1^57*x2^41*x3^30*x4^12*z^28 - 3*x1^56*x2^42*x3^30*x4^12*z^28 - 5*x1^55*x2^43*x3^30*x4^12*z^28 - 5*x1^53*x2^45*x3^30*x4^12*z^28 + x1^52*x2^46*x3^30*x4^12*z^28 - x1^51*x2^47*x3^30*x4^12*z^28 + 5*x1^50*x2^48*x3^30*x4^12*z^28 + x1^57*x2^40*x3^31*x4^12*z^28 - 2*x1^56*x2^41*x3^31*x4^12*z^28 + 5*x1^55*x2^42*x3^31*x4^12*z^28 + x1^54*x2^43*x3^31*x4^12*z^28 + x1^53*x2^44*x3^31*x4^12*z^28 + 3*x1^52*x2^45*x3^31*x4^12*z^28 - x1^50*x2^47*x3^31*x4^12*z^28 - x1^49*x2^48*x3^31*x4^12*z^28 - 2*x1^54*x2^42*x3^32*x4^12*z^28 - 4*x1^52*x2^44*x3^32*x4^12*z^28 + 2*x1^49*x2^47*x3^32*x4^12*z^28 + 3*x1^54*x2^41*x3^33*x4^12*z^28 + 2*x1^53*x2^42*x3^33*x4^12*z^28 + x1^52*x2^43*x3^33*x4^12*z^28 + 2*x1^51*x2^44*x3^33*x4^12*z^28 + 2*x1^50*x2^45*x3^33*x4^12*z^28 - x1^55*x2^39*x3^34*x4^12*z^28 - x1^54*x2^40*x3^34*x4^12*z^28 - x1^53*x2^41*x3^34*x4^12*z^28 - 2*x1^51*x2^43*x3^34*x4^12*z^28 + x1^53*x2^40*x3^35*x4^12*z^28 + 2*x1^52*x2^41*x3^35*x4^12*z^28 + x1^51*x2^42*x3^35*x4^12*z^28 + x1^50*x2^43*x3^35*x4^12*z^28 + x1^60*x2^45*x3^22*x4^13*z^28 - 2*x1^59*x2^46*x3^22*x4^13*z^28 - x1^57*x2^47*x3^23*x4^13*z^28 - 4*x1^58*x2^45*x3^24*x4^13*z^28 + 3*x1^58*x2^44*x3^25*x4^13*z^28 + 4*x1^57*x2^45*x3^25*x4^13*z^28 + 2*x1^55*x2^47*x3^25*x4^13*z^28 + x1^54*x2^48*x3^25*x4^13*z^28 - 6*x1^57*x2^44*x3^26*x4^13*z^28 - x1^55*x2^46*x3^26*x4^13*z^28 + x1^54*x2^47*x3^26*x4^13*z^28 + x1^53*x2^48*x3^26*x4^13*z^28 + 2*x1^52*x2^49*x3^26*x4^13*z^28 + 2*x1^57*x2^43*x3^27*x4^13*z^28 + 6*x1^56*x2^44*x3^27*x4^13*z^28 + 3*x1^54*x2^46*x3^27*x4^13*z^28 - 3*x1^53*x2^47*x3^27*x4^13*z^28 - x1^52*x2^48*x3^27*x4^13*z^28 - 3*x1^51*x2^49*x3^27*x4^13*z^28 - 6*x1^56*x2^43*x3^28*x4^13*z^28 - 2*x1^55*x2^44*x3^28*x4^13*z^28 - 2*x1^54*x2^45*x3^28*x4^13*z^28 + x1^53*x2^46*x3^28*x4^13*z^28 + 2*x1^52*x2^47*x3^28*x4^13*z^28 + x1^51*x2^48*x3^28*x4^13*z^28 + 3*x1^50*x2^49*x3^28*x4^13*z^28 + 2*x1^56*x2^42*x3^29*x4^13*z^28 + 6*x1^55*x2^43*x3^29*x4^13*z^28 + 3*x1^53*x2^45*x3^29*x4^13*z^28 - 5*x1^52*x2^46*x3^29*x4^13*z^28 - 3*x1^51*x2^47*x3^29*x4^13*z^28 - 5*x1^50*x2^48*x3^29*x4^13*z^28 - 2*x1^49*x2^49*x3^29*x4^13*z^28 - 6*x1^55*x2^42*x3^30*x4^13*z^28 - 2*x1^54*x2^43*x3^30*x4^13*z^28 - 2*x1^52*x2^45*x3^30*x4^13*z^28 + x1^51*x2^46*x3^30*x4^13*z^28 + 3*x1^50*x2^47*x3^30*x4^13*z^28 + 4*x1^49*x2^48*x3^30*x4^13*z^28 - x1^56*x2^40*x3^31*x4^13*z^28 + 2*x1^55*x2^41*x3^31*x4^13*z^28 + 6*x1^54*x2^42*x3^31*x4^13*z^28 + 3*x1^52*x2^44*x3^31*x4^13*z^28 - 2*x1^51*x2^45*x3^31*x4^13*z^28 - 2*x1^50*x2^46*x3^31*x4^13*z^28 - 5*x1^49*x2^47*x3^31*x4^13*z^28 - 6*x1^54*x2^41*x3^32*x4^13*z^28 - 2*x1^53*x2^42*x3^32*x4^13*z^28 - x1^52*x2^43*x3^32*x4^13*z^28 - 3*x1^51*x2^44*x3^32*x4^13*z^28 + x1^50*x2^45*x3^32*x4^13*z^28 + 2*x1^49*x2^46*x3^32*x4^13*z^28 + 2*x1^48*x2^47*x3^32*x4^13*z^28 - x1^55*x2^39*x3^33*x4^13*z^28 - x1^54*x2^40*x3^33*x4^13*z^28 + 2*x1^53*x2^41*x3^33*x4^13*z^28 + x1^52*x2^42*x3^33*x4^13*z^28 + 4*x1^51*x2^43*x3^33*x4^13*z^28 - x1^50*x2^44*x3^33*x4^13*z^28 - x1^49*x2^45*x3^33*x4^13*z^28 - 3*x1^48*x2^46*x3^33*x4^13*z^28 - x1^53*x2^40*x3^34*x4^13*z^28 - 2*x1^52*x2^41*x3^34*x4^13*z^28 - 2*x1^51*x2^42*x3^34*x4^13*z^28 - x1^50*x2^43*x3^34*x4^13*z^28 + x1^49*x2^44*x3^34*x4^13*z^28 + x1^47*x2^46*x3^34*x4^13*z^28 - x1^52*x2^40*x3^35*x4^13*z^28 + 2*x1^50*x2^42*x3^35*x4^13*z^28 - x1^49*x2^42*x3^36*x4^13*z^28 + x1^60*x2^45*x3^21*x4^14*z^28 - x1^60*x2^44*x3^22*x4^14*z^28 + x1^58*x2^46*x3^22*x4^14*z^28 - 2*x1^60*x2^43*x3^23*x4^14*z^28 - x1^59*x2^44*x3^23*x4^14*z^28 + 3*x1^58*x2^45*x3^23*x4^14*z^28 + 2*x1^59*x2^43*x3^24*x4^14*z^28 - x1^58*x2^44*x3^24*x4^14*z^28 - 2*x1^57*x2^45*x3^24*x4^14*z^28 - x1^55*x2^47*x3^24*x4^14*z^28 - x1^59*x2^42*x3^25*x4^14*z^28 - 2*x1^58*x2^43*x3^25*x4^14*z^28 + 3*x1^57*x2^44*x3^25*x4^14*z^28 + 2*x1^56*x2^45*x3^25*x4^14*z^28 + 2*x1^55*x2^46*x3^25*x4^14*z^28 - x1^53*x2^48*x3^25*x4^14*z^28 + 2*x1^58*x2^42*x3^26*x4^14*z^28 - x1^57*x2^43*x3^26*x4^14*z^28 - 3*x1^56*x2^44*x3^26*x4^14*z^28 - x1^55*x2^45*x3^26*x4^14*z^28 - 2*x1^54*x2^46*x3^26*x4^14*z^28 + x1^53*x2^47*x3^26*x4^14*z^28 + x1^52*x2^48*x3^26*x4^14*z^28 - 2*x1^57*x2^42*x3^27*x4^14*z^28 + 2*x1^56*x2^43*x3^27*x4^14*z^28 - 2*x1^55*x2^44*x3^27*x4^14*z^28 + x1^54*x2^45*x3^27*x4^14*z^28 + 2*x1^53*x2^46*x3^27*x4^14*z^28 + x1^52*x2^47*x3^27*x4^14*z^28 - x1^51*x2^48*x3^27*x4^14*z^28 + 2*x1^57*x2^41*x3^28*x4^14*z^28 + x1^56*x2^42*x3^28*x4^14*z^28 - x1^55*x2^43*x3^28*x4^14*z^28 - 3*x1^53*x2^45*x3^28*x4^14*z^28 - x1^52*x2^46*x3^28*x4^14*z^28 - x1^51*x2^47*x3^28*x4^14*z^28 + 2*x1^50*x2^48*x3^28*x4^14*z^28 - x1^57*x2^40*x3^29*x4^14*z^28 - 2*x1^56*x2^41*x3^29*x4^14*z^28 + 2*x1^55*x2^42*x3^29*x4^14*z^28 + 2*x1^53*x2^44*x3^29*x4^14*z^28 + x1^51*x2^46*x3^29*x4^14*z^28 - 2*x1^49*x2^48*x3^29*x4^14*z^28 + 2*x1^56*x2^40*x3^30*x4^14*z^28 - x1^54*x2^42*x3^30*x4^14*z^28 + x1^53*x2^43*x3^30*x4^14*z^28 - x1^52*x2^44*x3^30*x4^14*z^28 - x1^51*x2^45*x3^30*x4^14*z^28 - x1^50*x2^46*x3^30*x4^14*z^28 + 2*x1^49*x2^47*x3^30*x4^14*z^28 + x1^48*x2^48*x3^30*x4^14*z^28 - 2*x1^55*x2^40*x3^31*x4^14*z^28 + 2*x1^54*x2^41*x3^31*x4^14*z^28 + 2*x1^52*x2^43*x3^31*x4^14*z^28 + x1^51*x2^44*x3^31*x4^14*z^28 - 2*x1^49*x2^46*x3^31*x4^14*z^28 - 2*x1^48*x2^47*x3^31*x4^14*z^28 + 2*x1^55*x2^39*x3^32*x4^14*z^28 + x1^54*x2^40*x3^32*x4^14*z^28 - x1^53*x2^41*x3^32*x4^14*z^28 + x1^52*x2^42*x3^32*x4^14*z^28 + x1^51*x2^43*x3^32*x4^14*z^28 + 2*x1^50*x2^44*x3^32*x4^14*z^28 + 2*x1^48*x2^46*x3^32*x4^14*z^28 - x1^54*x2^39*x3^33*x4^14*z^28 - 3*x1^52*x2^41*x3^33*x4^14*z^28 - x1^51*x2^42*x3^33*x4^14*z^28 + 2*x1^50*x2^43*x3^33*x4^14*z^28 - x1^48*x2^45*x3^33*x4^14*z^28 - x1^47*x2^46*x3^33*x4^14*z^28 - x1^51*x2^41*x3^34*x4^14*z^28 + x1^47*x2^45*x3^34*x4^14*z^28 - x1^52*x2^39*x3^35*x4^14*z^28 - x1^51*x2^40*x3^35*x4^14*z^28 - x1^48*x2^43*x3^35*x4^14*z^28 - x1^46*x2^45*x3^35*x4^14*z^28 - x1^60*x2^44*x3^21*x4^15*z^28 + x1^60*x2^43*x3^22*x4^15*z^28 + x1^59*x2^44*x3^22*x4^15*z^28 - x1^58*x2^45*x3^22*x4^15*z^28 + x1^57*x2^46*x3^22*x4^15*z^28 - 4*x1^59*x2^43*x3^23*x4^15*z^28 - x1^58*x2^44*x3^23*x4^15*z^28 - x1^57*x2^45*x3^23*x4^15*z^28 + x1^55*x2^47*x3^23*x4^15*z^28 + 2*x1^59*x2^42*x3^24*x4^15*z^28 + 5*x1^58*x2^43*x3^24*x4^15*z^28 + 2*x1^56*x2^45*x3^24*x4^15*z^28 - x1^54*x2^47*x3^24*x4^15*z^28 - 6*x1^58*x2^42*x3^25*x4^15*z^28 - 2*x1^57*x2^43*x3^25*x4^15*z^28 - 2*x1^56*x2^44*x3^25*x4^15*z^28 - x1^55*x2^45*x3^25*x4^15*z^28 + x1^54*x2^46*x3^25*x4^15*z^28 + x1^53*x2^47*x3^25*x4^15*z^28 + 2*x1^58*x2^41*x3^26*x4^15*z^28 + 6*x1^57*x2^42*x3^26*x4^15*z^28 + 3*x1^55*x2^44*x3^26*x4^15*z^28 - 2*x1^54*x2^45*x3^26*x4^15*z^28 - x1^53*x2^46*x3^26*x4^15*z^28 - 2*x1^52*x2^47*x3^26*x4^15*z^28 - 6*x1^57*x2^41*x3^27*x4^15*z^28 - 2*x1^56*x2^42*x3^27*x4^15*z^28 - 2*x1^55*x2^43*x3^27*x4^15*z^28 + x1^52*x2^46*x3^27*x4^15*z^28 + 2*x1^51*x2^47*x3^27*x4^15*z^28 + 2*x1^57*x2^40*x3^28*x4^15*z^28 + 6*x1^56*x2^41*x3^28*x4^15*z^28 + 4*x1^54*x2^43*x3^28*x4^15*z^28 - 4*x1^53*x2^44*x3^28*x4^15*z^28 - 6*x1^51*x2^46*x3^28*x4^15*z^28 - x1^50*x2^47*x3^28*x4^15*z^28 - 6*x1^56*x2^40*x3^29*x4^15*z^28 - 2*x1^55*x2^41*x3^29*x4^15*z^28 - 2*x1^54*x2^42*x3^29*x4^15*z^28 + 2*x1^52*x2^44*x3^29*x4^15*z^28 + 2*x1^51*x2^45*x3^29*x4^15*z^28 + 6*x1^50*x2^46*x3^29*x4^15*z^28 + x1^56*x2^39*x3^30*x4^15*z^28 + 6*x1^55*x2^40*x3^30*x4^15*z^28 + 4*x1^53*x2^42*x3^30*x4^15*z^28 - 4*x1^52*x2^43*x3^30*x4^15*z^28 - 6*x1^50*x2^45*x3^30*x4^15*z^28 - 2*x1^49*x2^46*x3^30*x4^15*z^28 - 5*x1^55*x2^39*x3^31*x4^15*z^28 - 2*x1^54*x2^40*x3^31*x4^15*z^28 - 2*x1^53*x2^41*x3^31*x4^15*z^28 - x1^52*x2^42*x3^31*x4^15*z^28 + x1^51*x2^43*x3^31*x4^15*z^28 + 3*x1^50*x2^44*x3^31*x4^15*z^28 + 5*x1^49*x2^45*x3^31*x4^15*z^28 + 5*x1^54*x2^39*x3^32*x4^15*z^28 + 3*x1^52*x2^41*x3^32*x4^15*z^28 - 2*x1^51*x2^42*x3^32*x4^15*z^28 - 4*x1^49*x2^44*x3^32*x4^15*z^28 - x1^48*x2^45*x3^32*x4^15*z^28 - 2*x1^54*x2^38*x3^33*x4^15*z^28 - 3*x1^53*x2^39*x3^33*x4^15*z^28 - 2*x1^52*x2^40*x3^33*x4^15*z^28 + 2*x1^49*x2^43*x3^33*x4^15*z^28 + 2*x1^48*x2^44*x3^33*x4^15*z^28 + 2*x1^53*x2^38*x3^34*x4^15*z^28 + 2*x1^52*x2^39*x3^34*x4^15*z^28 + 2*x1^51*x2^40*x3^34*x4^15*z^28 - x1^50*x2^41*x3^34*x4^15*z^28 - 2*x1^48*x2^43*x3^34*x4^15*z^28 - x1^51*x2^39*x3^35*x4^15*z^28 - x1^49*x2^41*x3^35*x4^15*z^28 + x1^48*x2^42*x3^35*x4^15*z^28 + x1^47*x2^43*x3^35*x4^15*z^28 + x1^50*x2^39*x3^36*x4^15*z^28 - x1^49*x2^40*x3^36*x4^15*z^28 - x1^48*x2^41*x3^36*x4^15*z^28 + x1^47*x2^42*x3^36*x4^15*z^28 + 2*x1^59*x2^43*x3^22*x4^16*z^28 - x1^57*x2^45*x3^22*x4^16*z^28 - 3*x1^58*x2^43*x3^23*x4^16*z^28 + x1^57*x2^44*x3^23*x4^16*z^28 + 4*x1^58*x2^42*x3^24*x4^16*z^28 + x1^57*x2^43*x3^24*x4^16*z^28 - x1^54*x2^46*x3^24*x4^16*z^28 - x1^53*x2^47*x3^24*x4^16*z^28 - x1^58*x2^41*x3^25*x4^16*z^28 - 5*x1^57*x2^42*x3^25*x4^16*z^28 - x1^56*x2^43*x3^25*x4^16*z^28 - 2*x1^55*x2^44*x3^25*x4^16*z^28 + 3*x1^54*x2^45*x3^25*x4^16*z^28 + x1^53*x2^46*x3^25*x4^16*z^28 + 2*x1^52*x2^47*x3^25*x4^16*z^28 + 6*x1^57*x2^41*x3^26*x4^16*z^28 + 2*x1^56*x2^42*x3^26*x4^16*z^28 + 2*x1^55*x2^43*x3^26*x4^16*z^28 - 2*x1^53*x2^45*x3^26*x4^16*z^28 - x1^52*x2^46*x3^26*x4^16*z^28 - 2*x1^51*x2^47*x3^26*x4^16*z^28 - 2*x1^57*x2^40*x3^27*x4^16*z^28 - 6*x1^56*x2^41*x3^27*x4^16*z^28 - 4*x1^54*x2^43*x3^27*x4^16*z^28 + 4*x1^53*x2^44*x3^27*x4^16*z^28 + x1^52*x2^45*x3^27*x4^16*z^28 + 5*x1^51*x2^46*x3^27*x4^16*z^28 + x1^50*x2^47*x3^27*x4^16*z^28 + 6*x1^56*x2^40*x3^28*x4^16*z^28 + 2*x1^55*x2^41*x3^28*x4^16*z^28 + 2*x1^54*x2^42*x3^28*x4^16*z^28 - 2*x1^52*x2^44*x3^28*x4^16*z^28 - 2*x1^51*x2^45*x3^28*x4^16*z^28 - 5*x1^50*x2^46*x3^28*x4^16*z^28 - 2*x1^56*x2^39*x3^29*x4^16*z^28 - 6*x1^55*x2^40*x3^29*x4^16*z^28 - 4*x1^53*x2^42*x3^29*x4^16*z^28 + 4*x1^52*x2^43*x3^29*x4^16*z^28 + 6*x1^50*x2^45*x3^29*x4^16*z^28 + 2*x1^49*x2^46*x3^29*x4^16*z^28 + 5*x1^55*x2^39*x3^30*x4^16*z^28 + 2*x1^54*x2^40*x3^30*x4^16*z^28 + 2*x1^53*x2^41*x3^30*x4^16*z^28 - 2*x1^51*x2^43*x3^30*x4^16*z^28 - 2*x1^50*x2^44*x3^30*x4^16*z^28 - 6*x1^49*x2^45*x3^30*x4^16*z^28 - 5*x1^54*x2^39*x3^31*x4^16*z^28 - 2*x1^53*x2^40*x3^31*x4^16*z^28 - 4*x1^52*x2^41*x3^31*x4^16*z^28 + 4*x1^51*x2^42*x3^31*x4^16*z^28 + 6*x1^49*x2^44*x3^31*x4^16*z^28 + 2*x1^48*x2^45*x3^31*x4^16*z^28 + x1^54*x2^38*x3^32*x4^16*z^28 + 2*x1^53*x2^39*x3^32*x4^16*z^28 + 3*x1^52*x2^40*x3^32*x4^16*z^28 - 2*x1^50*x2^42*x3^32*x4^16*z^28 - 2*x1^49*x2^43*x3^32*x4^16*z^28 - 6*x1^48*x2^44*x3^32*x4^16*z^28 - x1^53*x2^38*x3^33*x4^16*z^28 - x1^52*x2^39*x3^33*x4^16*z^28 - 2*x1^51*x2^40*x3^33*x4^16*z^28 + 2*x1^50*x2^41*x3^33*x4^16*z^28 - x1^49*x2^42*x3^33*x4^16*z^28 + 5*x1^48*x2^43*x3^33*x4^16*z^28 + 2*x1^47*x2^44*x3^33*x4^16*z^28 + x1^51*x2^39*x3^34*x4^16*z^28 - 2*x1^48*x2^42*x3^34*x4^16*z^28 - 3*x1^47*x2^43*x3^34*x4^16*z^28 - 2*x1^48*x2^41*x3^35*x4^16*z^28 + 2*x1^47*x2^42*x3^35*x4^16*z^28 + x1^48*x2^40*x3^36*x4^16*z^28 - x1^46*x2^42*x3^36*x4^16*z^28 + x1^55*x2^45*x3^23*x4^17*z^28 + x1^54*x2^46*x3^23*x4^17*z^28 + 2*x1^57*x2^42*x3^24*x4^17*z^28 - x1^55*x2^44*x3^24*x4^17*z^28 - x1^54*x2^45*x3^24*x4^17*z^28 - 2*x1^53*x2^46*x3^24*x4^17*z^28 - x1^52*x2^47*x3^24*x4^17*z^28 - x1^57*x2^41*x3^25*x4^17*z^28 + x1^53*x2^45*x3^25*x4^17*z^28 + 2*x1^52*x2^46*x3^25*x4^17*z^28 + x1^51*x2^47*x3^25*x4^17*z^28 + x1^56*x2^41*x3^26*x4^17*z^28 + x1^55*x2^42*x3^26*x4^17*z^28 - 2*x1^53*x2^44*x3^26*x4^17*z^28 - 2*x1^51*x2^46*x3^26*x4^17*z^28 - 2*x1^56*x2^40*x3^27*x4^17*z^28 + 2*x1^50*x2^46*x3^27*x4^17*z^28 + 2*x1^55*x2^40*x3^28*x4^17*z^28 + 2*x1^53*x2^42*x3^28*x4^17*z^28 - x1^52*x2^43*x3^28*x4^17*z^28 - 2*x1^50*x2^45*x3^28*x4^17*z^28 - x1^49*x2^46*x3^28*x4^17*z^28 - 2*x1^55*x2^39*x3^29*x4^17*z^28 - x1^54*x2^40*x3^29*x4^17*z^28 - x1^53*x2^41*x3^29*x4^17*z^28 + x1^51*x2^43*x3^29*x4^17*z^28 + x1^50*x2^44*x3^29*x4^17*z^28 + 2*x1^49*x2^45*x3^29*x4^17*z^28 + 2*x1^54*x2^39*x3^30*x4^17*z^28 + x1^52*x2^41*x3^30*x4^17*z^28 - x1^51*x2^42*x3^30*x4^17*z^28 - 2*x1^49*x2^44*x3^30*x4^17*z^28 - x1^48*x2^45*x3^30*x4^17*z^28 + x1^50*x2^42*x3^31*x4^17*z^28 + x1^49*x2^43*x3^31*x4^17*z^28 + 2*x1^48*x2^44*x3^31*x4^17*z^28 - x1^50*x2^41*x3^32*x4^17*z^28 - 2*x1^48*x2^43*x3^32*x4^17*z^28 + x1^50*x2^40*x3^33*x4^17*z^28 + 2*x1^47*x2^43*x3^33*x4^17*z^28 - x1^49*x2^40*x3^34*x4^17*z^28 - x1^47*x2^42*x3^34*x4^17*z^28 - x1^46*x2^43*x3^34*x4^17*z^28 + x1^46*x2^42*x3^35*x4^17*z^28 - x1^60*x2^46*x3^26*x4^3*z^27 + x1^62*x2^45*x3^24*x4^4*z^27 + x1^61*x2^44*x3^26*x4^4*z^27 + x1^59*x2^45*x3^27*x4^4*z^27 - x1^58*x2^46*x3^27*x4^4*z^27 - x1^57*x2^47*x3^27*x4^4*z^27 + x1^61*x2^45*x3^24*x4^5*z^27 - 2*x1^61*x2^44*x3^25*x4^5*z^27 + x1^59*x2^46*x3^25*x4^5*z^27 + 3*x1^60*x2^44*x3^26*x4^5*z^27 + 2*x1^58*x2^46*x3^26*x4^5*z^27 + x1^57*x2^47*x3^26*x4^5*z^27 - x1^60*x2^43*x3^27*x4^5*z^27 - x1^59*x2^44*x3^27*x4^5*z^27 + x1^58*x2^45*x3^27*x4^5*z^27 - 3*x1^57*x2^46*x3^27*x4^5*z^27 - x1^56*x2^47*x3^27*x4^5*z^27 + 2*x1^59*x2^43*x3^28*x4^5*z^27 - x1^55*x2^47*x3^28*x4^5*z^27 + 2*x1^57*x2^44*x3^29*x4^5*z^27 + x1^61*x2^44*x3^24*x4^6*z^27 - 2*x1^60*x2^44*x3^25*x4^6*z^27 - x1^58*x2^46*x3^25*x4^6*z^27 + x1^59*x2^44*x3^26*x4^6*z^27 - 2*x1^56*x2^47*x3^26*x4^6*z^27 - 3*x1^59*x2^43*x3^27*x4^6*z^27 + x1^58*x2^44*x3^27*x4^6*z^27 + x1^57*x2^45*x3^27*x4^6*z^27 - x1^56*x2^46*x3^27*x4^6*z^27 + x1^58*x2^43*x3^28*x4^6*z^27 - x1^57*x2^44*x3^28*x4^6*z^27 + 3*x1^56*x2^45*x3^28*x4^6*z^27 + x1^55*x2^46*x3^28*x4^6*z^27 - 2*x1^58*x2^42*x3^29*x4^6*z^27 - x1^57*x2^43*x3^29*x4^6*z^27 + x1^55*x2^45*x3^29*x4^6*z^27 - 2*x1^56*x2^43*x3^30*x4^6*z^27 - x1^53*x2^46*x3^30*x4^6*z^27 + x1^61*x2^44*x3^23*x4^7*z^27 - x1^59*x2^46*x3^23*x4^7*z^27 + x1^58*x2^47*x3^23*x4^7*z^27 - x1^60*x2^44*x3^24*x4^7*z^27 + x1^59*x2^45*x3^24*x4^7*z^27 + x1^58*x2^46*x3^24*x4^7*z^27 - 2*x1^57*x2^47*x3^24*x4^7*z^27 + x1^57*x2^46*x3^25*x4^7*z^27 + x1^55*x2^48*x3^25*x4^7*z^27 - x1^58*x2^44*x3^26*x4^7*z^27 + x1^57*x2^45*x3^26*x4^7*z^27 - x1^56*x2^46*x3^26*x4^7*z^27 - 2*x1^58*x2^43*x3^27*x4^7*z^27 + x1^57*x2^44*x3^27*x4^7*z^27 + x1^56*x2^45*x3^27*x4^7*z^27 + x1^55*x2^46*x3^27*x4^7*z^27 - x1^53*x2^48*x3^27*x4^7*z^27 + x1^58*x2^42*x3^28*x4^7*z^27 - x1^57*x2^43*x3^28*x4^7*z^27 + x1^54*x2^46*x3^28*x4^7*z^27 - x1^53*x2^47*x3^28*x4^7*z^27 - x1^52*x2^48*x3^28*x4^7*z^27 - 2*x1^57*x2^42*x3^29*x4^7*z^27 - 2*x1^55*x2^44*x3^29*x4^7*z^27 + x1^57*x2^41*x3^30*x4^7*z^27 + x1^53*x2^45*x3^30*x4^7*z^27 + 2*x1^59*x2^46*x3^22*x4^8*z^27 - x1^60*x2^44*x3^23*x4^8*z^27 - x1^59*x2^45*x3^23*x4^8*z^27 - 2*x1^58*x2^46*x3^23*x4^8*z^27 + x1^57*x2^47*x3^23*x4^8*z^27 + x1^59*x2^44*x3^24*x4^8*z^27 + 4*x1^58*x2^45*x3^24*x4^8*z^27 + x1^57*x2^46*x3^24*x4^8*z^27 + x1^56*x2^47*x3^24*x4^8*z^27 - x1^59*x2^43*x3^25*x4^8*z^27 - 3*x1^58*x2^44*x3^25*x4^8*z^27 - 5*x1^57*x2^45*x3^25*x4^8*z^27 - 2*x1^55*x2^47*x3^25*x4^8*z^27 + x1^59*x2^42*x3^26*x4^8*z^27 + 3*x1^57*x2^44*x3^26*x4^8*z^27 + x1^56*x2^45*x3^26*x4^8*z^27 + x1^55*x2^46*x3^26*x4^8*z^27 + x1^54*x2^47*x3^26*x4^8*z^27 - x1^52*x2^49*x3^26*x4^8*z^27 - x1^58*x2^42*x3^27*x4^8*z^27 - 4*x1^56*x2^44*x3^27*x4^8*z^27 - x1^55*x2^45*x3^27*x4^8*z^27 - 3*x1^54*x2^46*x3^27*x4^8*z^27 + x1^53*x2^47*x3^27*x4^8*z^27 + x1^52*x2^48*x3^27*x4^8*z^27 + x1^51*x2^49*x3^27*x4^8*z^27 + 3*x1^57*x2^42*x3^28*x4^8*z^27 + 3*x1^56*x2^43*x3^28*x4^8*z^27 + 3*x1^55*x2^44*x3^28*x4^8*z^27 + 3*x1^54*x2^45*x3^28*x4^8*z^27 - 2*x1^56*x2^42*x3^29*x4^8*z^27 - 2*x1^55*x2^43*x3^29*x4^8*z^27 + x1^54*x2^44*x3^29*x4^8*z^27 - 2*x1^53*x2^45*x3^29*x4^8*z^27 + x1^52*x2^46*x3^29*x4^8*z^27 + x1^51*x2^47*x3^29*x4^8*z^27 + x1^57*x2^40*x3^30*x4^8*z^27 + x1^56*x2^41*x3^30*x4^8*z^27 + 2*x1^55*x2^42*x3^30*x4^8*z^27 + 2*x1^54*x2^43*x3^30*x4^8*z^27 + x1^53*x2^44*x3^30*x4^8*z^27 + x1^52*x2^45*x3^30*x4^8*z^27 - x1^55*x2^41*x3^31*x4^8*z^27 - 2*x1^54*x2^42*x3^31*x4^8*z^27 - 2*x1^52*x2^44*x3^31*x4^8*z^27 + x1^56*x2^39*x3^32*x4^8*z^27 + x1^55*x2^40*x3^32*x4^8*z^27 + x1^54*x2^41*x3^32*x4^8*z^27 + x1^53*x2^42*x3^32*x4^8*z^27 + x1^51*x2^44*x3^32*x4^8*z^27 + x1^60*x2^43*x3^23*x4^9*z^27 - 2*x1^58*x2^45*x3^23*x4^9*z^27 - x1^56*x2^47*x3^23*x4^9*z^27 + 2*x1^58*x2^44*x3^24*x4^9*z^27 + 3*x1^57*x2^45*x3^24*x4^9*z^27 - x1^56*x2^46*x3^24*x4^9*z^27 + x1^55*x2^47*x3^24*x4^9*z^27 - 2*x1^59*x2^42*x3^25*x4^9*z^27 - 2*x1^57*x2^44*x3^25*x4^9*z^27 - x1^55*x2^46*x3^25*x4^9*z^27 + 4*x1^58*x2^42*x3^26*x4^9*z^27 - x1^57*x2^43*x3^26*x4^9*z^27 + 3*x1^56*x2^44*x3^26*x4^9*z^27 + 3*x1^54*x2^46*x3^26*x4^9*z^27 - x1^53*x2^47*x3^26*x4^9*z^27 + x1^52*x2^48*x3^26*x4^9*z^27 - x1^58*x2^41*x3^27*x4^9*z^27 - 3*x1^57*x2^42*x3^27*x4^9*z^27 + x1^56*x2^43*x3^27*x4^9*z^27 - x1^55*x2^44*x3^27*x4^9*z^27 + x1^54*x2^45*x3^27*x4^9*z^27 + 3*x1^57*x2^41*x3^28*x4^9*z^27 + 2*x1^56*x2^42*x3^28*x4^9*z^27 - x1^55*x2^43*x3^28*x4^9*z^27 + 2*x1^54*x2^44*x3^28*x4^9*z^27 + 3*x1^53*x2^45*x3^28*x4^9*z^27 - x1^57*x2^40*x3^29*x4^9*z^27 - x1^56*x2^41*x3^29*x4^9*z^27 - x1^55*x2^42*x3^29*x4^9*z^27 - 4*x1^54*x2^43*x3^29*x4^9*z^27 - 2*x1^53*x2^44*x3^29*x4^9*z^27 + x1^52*x2^45*x3^29*x4^9*z^27 + x1^50*x2^47*x3^29*x4^9*z^27 + x1^56*x2^40*x3^30*x4^9*z^27 + 2*x1^55*x2^41*x3^30*x4^9*z^27 + 3*x1^54*x2^42*x3^30*x4^9*z^27 - x1^51*x2^45*x3^30*x4^9*z^27 - x1^49*x2^47*x3^30*x4^9*z^27 - 2*x1^56*x2^39*x3^31*x4^9*z^27 + x1^54*x2^41*x3^31*x4^9*z^27 - x1^53*x2^42*x3^31*x4^9*z^27 + x1^52*x2^43*x3^31*x4^9*z^27 - x1^51*x2^44*x3^31*x4^9*z^27 - x1^54*x2^40*x3^32*x4^9*z^27 + x1^53*x2^41*x3^32*x4^9*z^27 + x1^52*x2^42*x3^32*x4^9*z^27 - x1^53*x2^40*x3^33*x4^9*z^27 - x1^52*x2^41*x3^33*x4^9*z^27 - x1^50*x2^43*x3^33*x4^9*z^27 - x1^61*x2^44*x3^20*x4^10*z^27 + 2*x1^60*x2^44*x3^21*x4^10*z^27 - x1^59*x2^45*x3^21*x4^10*z^27 - x1^60*x2^43*x3^22*x4^10*z^27 - 2*x1^59*x2^44*x3^22*x4^10*z^27 + x1^58*x2^45*x3^22*x4^10*z^27 - 2*x1^57*x2^46*x3^22*x4^10*z^27 + 2*x1^59*x2^43*x3^23*x4^10*z^27 + 2*x1^58*x2^44*x3^23*x4^10*z^27 - x1^55*x2^47*x3^23*x4^10*z^27 - 2*x1^58*x2^43*x3^24*x4^10*z^27 - x1^57*x2^44*x3^24*x4^10*z^27 - x1^56*x2^45*x3^24*x4^10*z^27 + x1^55*x2^46*x3^24*x4^10*z^27 - x1^53*x2^48*x3^24*x4^10*z^27 + x1^57*x2^43*x3^25*x4^10*z^27 + x1^56*x2^44*x3^25*x4^10*z^27 + x1^54*x2^46*x3^25*x4^10*z^27 - x1^53*x2^47*x3^25*x4^10*z^27 + x1^52*x2^48*x3^25*x4^10*z^27 - x1^55*x2^44*x3^26*x4^10*z^27 + 2*x1^54*x2^45*x3^26*x4^10*z^27 + x1^53*x2^46*x3^26*x4^10*z^27 + 2*x1^52*x2^47*x3^26*x4^10*z^27 - x1^51*x2^48*x3^26*x4^10*z^27 - x1^53*x2^45*x3^27*x4^10*z^27 - 2*x1^51*x2^47*x3^27*x4^10*z^27 + x1^53*x2^44*x3^28*x4^10*z^27 + 3*x1^51*x2^46*x3^28*x4^10*z^27 + x1^50*x2^47*x3^28*x4^10*z^27 - x1^55*x2^41*x3^29*x4^10*z^27 + 2*x1^54*x2^42*x3^29*x4^10*z^27 - 2*x1^52*x2^44*x3^29*x4^10*z^27 - 2*x1^49*x2^47*x3^29*x4^10*z^27 + x1^56*x2^39*x3^30*x4^10*z^27 - x1^55*x2^40*x3^30*x4^10*z^27 - x1^54*x2^41*x3^30*x4^10*z^27 - x1^52*x2^43*x3^30*x4^10*z^27 + x1^51*x2^44*x3^30*x4^10*z^27 - x1^50*x2^45*x3^30*x4^10*z^27 + 2*x1^48*x2^47*x3^30*x4^10*z^27 + x1^53*x2^41*x3^31*x4^10*z^27 - x1^52*x2^42*x3^31*x4^10*z^27 + x1^51*x2^43*x3^31*x4^10*z^27 - x1^49*x2^45*x3^31*x4^10*z^27 - x1^48*x2^46*x3^31*x4^10*z^27 + x1^55*x2^38*x3^32*x4^10*z^27 - x1^54*x2^39*x3^32*x4^10*z^27 - 2*x1^53*x2^40*x3^32*x4^10*z^27 - x1^51*x2^42*x3^32*x4^10*z^27 - x1^54*x2^38*x3^33*x4^10*z^27 + x1^53*x2^39*x3^33*x4^10*z^27 - 2*x1^51*x2^41*x3^33*x4^10*z^27 + 2*x1^50*x2^42*x3^33*x4^10*z^27 - x1^52*x2^39*x3^34*x4^10*z^27 - x1^50*x2^41*x3^34*x4^10*z^27 - x1^60*x2^44*x3^20*x4^11*z^27 + x1^60*x2^43*x3^21*x4^11*z^27 + x1^59*x2^44*x3^21*x4^11*z^27 - x1^58*x2^45*x3^21*x4^11*z^27 - 3*x1^59*x2^43*x3^22*x4^11*z^27 + x1^56*x2^46*x3^22*x4^11*z^27 + 2*x1^59*x2^42*x3^23*x4^11*z^27 + 3*x1^58*x2^43*x3^23*x4^11*z^27 - x1^57*x2^44*x3^23*x4^11*z^27 + 4*x1^56*x2^45*x3^23*x4^11*z^27 - 4*x1^58*x2^42*x3^24*x4^11*z^27 - x1^55*x2^45*x3^24*x4^11*z^27 + 2*x1^54*x2^46*x3^24*x4^11*z^27 + 3*x1^53*x2^47*x3^24*x4^11*z^27 + x1^58*x2^41*x3^25*x4^11*z^27 + 4*x1^57*x2^42*x3^25*x4^11*z^27 - 2*x1^56*x2^43*x3^25*x4^11*z^27 + x1^55*x2^44*x3^25*x4^11*z^27 - 4*x1^54*x2^45*x3^25*x4^11*z^27 - x1^53*x2^46*x3^25*x4^11*z^27 - 3*x1^52*x2^47*x3^25*x4^11*z^27 - 4*x1^57*x2^41*x3^26*x4^11*z^27 + x1^55*x2^43*x3^26*x4^11*z^27 + x1^53*x2^45*x3^26*x4^11*z^27 - 2*x1^52*x2^46*x3^26*x4^11*z^27 + 3*x1^51*x2^47*x3^26*x4^11*z^27 + x1^50*x2^48*x3^26*x4^11*z^27 + x1^57*x2^40*x3^27*x4^11*z^27 + 4*x1^56*x2^41*x3^27*x4^11*z^27 - 2*x1^55*x2^42*x3^27*x4^11*z^27 + 3*x1^54*x2^43*x3^27*x4^11*z^27 - x1^53*x2^44*x3^27*x4^11*z^27 - 2*x1^52*x2^45*x3^27*x4^11*z^27 - 4*x1^51*x2^46*x3^27*x4^11*z^27 - x1^50*x2^47*x3^27*x4^11*z^27 - x1^49*x2^48*x3^27*x4^11*z^27 - 4*x1^56*x2^40*x3^28*x4^11*z^27 - 2*x1^55*x2^41*x3^28*x4^11*z^27 + 4*x1^52*x2^44*x3^28*x4^11*z^27 + x1^51*x2^45*x3^28*x4^11*z^27 + x1^50*x2^46*x3^28*x4^11*z^27 - x1^49*x2^47*x3^28*x4^11*z^27 + 2*x1^56*x2^39*x3^29*x4^11*z^27 + 3*x1^55*x2^40*x3^29*x4^11*z^27 - 3*x1^54*x2^41*x3^29*x4^11*z^27 + x1^53*x2^42*x3^29*x4^11*z^27 - x1^52*x2^43*x3^29*x4^11*z^27 - 2*x1^51*x2^44*x3^29*x4^11*z^27 - 3*x1^50*x2^45*x3^29*x4^11*z^27 - x1^48*x2^47*x3^29*x4^11*z^27 - 2*x1^55*x2^39*x3^30*x4^11*z^27 - 2*x1^53*x2^41*x3^30*x4^11*z^27 - 2*x1^52*x2^42*x3^30*x4^11*z^27 + 2*x1^51*x2^43*x3^30*x4^11*z^27 + 3*x1^50*x2^44*x3^30*x4^11*z^27 - x1^48*x2^46*x3^30*x4^11*z^27 - x1^55*x2^38*x3^31*x4^11*z^27 + 3*x1^54*x2^39*x3^31*x4^11*z^27 + x1^52*x2^41*x3^31*x4^11*z^27 - x1^51*x2^42*x3^31*x4^11*z^27 - x1^50*x2^43*x3^31*x4^11*z^27 - x1^47*x2^46*x3^31*x4^11*z^27 + 2*x1^54*x2^38*x3^32*x4^11*z^27 + x1^53*x2^39*x3^32*x4^11*z^27 + x1^51*x2^41*x3^32*x4^11*z^27 + 2*x1^48*x2^44*x3^32*x4^11*z^27 + x1^47*x2^45*x3^32*x4^11*z^27 - x1^51*x2^40*x3^33*x4^11*z^27 - 2*x1^50*x2^41*x3^33*x4^11*z^27 + x1^50*x2^40*x3^34*x4^11*z^27 - x1^49*x2^41*x3^34*x4^11*z^27 - x1^60*x2^43*x3^20*x4^12*z^27 + x1^59*x2^44*x3^20*x4^12*z^27 + x1^59*x2^43*x3^21*x4^12*z^27 - x1^58*x2^44*x3^21*x4^12*z^27 - x1^59*x2^42*x3^22*x4^12*z^27 - 2*x1^58*x2^43*x3^22*x4^12*z^27 + 3*x1^57*x2^44*x3^22*x4^12*z^27 - x1^56*x2^45*x3^22*x4^12*z^27 + x1^55*x2^46*x3^22*x4^12*z^27 + 2*x1^58*x2^42*x3^23*x4^12*z^27 - 2*x1^56*x2^44*x3^23*x4^12*z^27 - 2*x1^54*x2^46*x3^23*x4^12*z^27 + x1^53*x2^47*x3^23*x4^12*z^27 - x1^58*x2^41*x3^24*x4^12*z^27 - 2*x1^57*x2^42*x3^24*x4^12*z^27 + 5*x1^56*x2^43*x3^24*x4^12*z^27 + 3*x1^54*x2^45*x3^24*x4^12*z^27 + x1^53*x2^46*x3^24*x4^12*z^27 - x1^52*x2^47*x3^24*x4^12*z^27 + 2*x1^57*x2^41*x3^25*x4^12*z^27 - 2*x1^56*x2^42*x3^25*x4^12*z^27 - 5*x1^55*x2^43*x3^25*x4^12*z^27 + 2*x1^54*x2^44*x3^25*x4^12*z^27 - 3*x1^53*x2^45*x3^25*x4^12*z^27 + x1^52*x2^46*x3^25*x4^12*z^27 + x1^51*x2^47*x3^25*x4^12*z^27 + x1^50*x2^48*x3^25*x4^12*z^27 - 2*x1^56*x2^41*x3^26*x4^12*z^27 + 6*x1^55*x2^42*x3^26*x4^12*z^27 + 2*x1^53*x2^44*x3^26*x4^12*z^27 - 2*x1^50*x2^47*x3^26*x4^12*z^27 - x1^49*x2^48*x3^26*x4^12*z^27 + 2*x1^56*x2^40*x3^27*x4^12*z^27 - x1^55*x2^41*x3^27*x4^12*z^27 - 5*x1^54*x2^42*x3^27*x4^12*z^27 - 4*x1^52*x2^44*x3^27*x4^12*z^27 + 4*x1^51*x2^45*x3^27*x4^12*z^27 + 4*x1^49*x2^47*x3^27*x4^12*z^27 - x1^56*x2^39*x3^28*x4^12*z^27 - 2*x1^55*x2^40*x3^28*x4^12*z^27 + 6*x1^54*x2^41*x3^28*x4^12*z^27 + x1^53*x2^42*x3^28*x4^12*z^27 + x1^52*x2^43*x3^28*x4^12*z^27 + 2*x1^51*x2^44*x3^28*x4^12*z^27 + x1^50*x2^45*x3^28*x4^12*z^27 - 3*x1^49*x2^46*x3^28*x4^12*z^27 - 3*x1^48*x2^47*x3^28*x4^12*z^27 + 2*x1^55*x2^39*x3^29*x4^12*z^27 - x1^54*x2^40*x3^29*x4^12*z^27 - 5*x1^53*x2^41*x3^29*x4^12*z^27 - 4*x1^51*x2^43*x3^29*x4^12*z^27 + x1^50*x2^44*x3^29*x4^12*z^27 + 5*x1^48*x2^46*x3^29*x4^12*z^27 - x1^54*x2^39*x3^30*x4^12*z^27 + 7*x1^53*x2^40*x3^30*x4^12*z^27 + x1^52*x2^41*x3^30*x4^12*z^27 + 3*x1^51*x2^42*x3^30*x4^12*z^27 + 3*x1^50*x2^43*x3^30*x4^12*z^27 + x1^49*x2^44*x3^30*x4^12*z^27 - 2*x1^48*x2^45*x3^30*x4^12*z^27 - 2*x1^47*x2^46*x3^30*x4^12*z^27 - x1^54*x2^38*x3^31*x4^12*z^27 - 2*x1^53*x2^39*x3^31*x4^12*z^27 - 4*x1^52*x2^40*x3^31*x4^12*z^27 - 3*x1^50*x2^42*x3^31*x4^12*z^27 + 3*x1^47*x2^45*x3^31*x4^12*z^27 + 2*x1^52*x2^39*x3^32*x4^12*z^27 - x1^51*x2^40*x3^32*x4^12*z^27 + 3*x1^49*x2^42*x3^32*x4^12*z^27 - x1^46*x2^45*x3^32*x4^12*z^27 - 2*x1^50*x2^40*x3^33*x4^12*z^27 - 4*x1^49*x2^41*x3^33*x4^12*z^27 - x1^47*x2^43*x3^33*x4^12*z^27 + x1^51*x2^38*x3^34*x4^12*z^27 + x1^50*x2^39*x3^34*x4^12*z^27 + 2*x1^49*x2^40*x3^34*x4^12*z^27 + x1^48*x2^41*x3^34*x4^12*z^27 - x1^48*x2^40*x3^35*x4^12*z^27 - x1^59*x2^43*x3^20*x4^13*z^27 + x1^58*x2^44*x3^20*x4^13*z^27 + 2*x1^59*x2^42*x3^21*x4^13*z^27 - x1^58*x2^43*x3^21*x4^13*z^27 - 2*x1^57*x2^44*x3^21*x4^13*z^27 + 2*x1^56*x2^45*x3^21*x4^13*z^27 + x1^57*x2^43*x3^22*x4^13*z^27 + 2*x1^56*x2^44*x3^22*x4^13*z^27 - x1^55*x2^45*x3^22*x4^13*z^27 + x1^54*x2^46*x3^22*x4^13*z^27 - x1^57*x2^42*x3^23*x4^13*z^27 - 5*x1^56*x2^43*x3^23*x4^13*z^27 + x1^54*x2^45*x3^23*x4^13*z^27 + x1^52*x2^47*x3^23*x4^13*z^27 + 2*x1^56*x2^42*x3^24*x4^13*z^27 + 5*x1^55*x2^43*x3^24*x4^13*z^27 - x1^54*x2^44*x3^24*x4^13*z^27 + 2*x1^53*x2^45*x3^24*x4^13*z^27 - x1^52*x2^46*x3^24*x4^13*z^27 - x1^51*x2^47*x3^24*x4^13*z^27 - 6*x1^55*x2^42*x3^25*x4^13*z^27 - 3*x1^54*x2^43*x3^25*x4^13*z^27 - 2*x1^53*x2^44*x3^25*x4^13*z^27 + x1^50*x2^47*x3^25*x4^13*z^27 + 2*x1^55*x2^41*x3^26*x4^13*z^27 + 6*x1^54*x2^42*x3^26*x4^13*z^27 + 3*x1^52*x2^44*x3^26*x4^13*z^27 - 4*x1^51*x2^45*x3^26*x4^13*z^27 - 4*x1^49*x2^47*x3^26*x4^13*z^27 - 6*x1^54*x2^41*x3^27*x4^13*z^27 - 2*x1^53*x2^42*x3^27*x4^13*z^27 - 2*x1^52*x2^43*x3^27*x4^13*z^27 + x1^50*x2^45*x3^27*x4^13*z^27 + 2*x1^49*x2^46*x3^27*x4^13*z^27 + 4*x1^48*x2^47*x3^27*x4^13*z^27 + 2*x1^54*x2^40*x3^28*x4^13*z^27 + 6*x1^53*x2^41*x3^28*x4^13*z^27 + 4*x1^51*x2^43*x3^28*x4^13*z^27 - 4*x1^50*x2^44*x3^28*x4^13*z^27 - 6*x1^48*x2^46*x3^28*x4^13*z^27 - x1^47*x2^47*x3^28*x4^13*z^27 + x1^55*x2^38*x3^29*x4^13*z^27 - 6*x1^53*x2^40*x3^29*x4^13*z^27 - 2*x1^52*x2^41*x3^29*x4^13*z^27 - x1^51*x2^42*x3^29*x4^13*z^27 - x1^50*x2^43*x3^29*x4^13*z^27 + 3*x1^49*x2^44*x3^29*x4^13*z^27 + 3*x1^48*x2^45*x3^29*x4^13*z^27 + 6*x1^47*x2^46*x3^29*x4^13*z^27 - x1^54*x2^38*x3^30*x4^13*z^27 + x1^53*x2^39*x3^30*x4^13*z^27 + 6*x1^52*x2^40*x3^30*x4^13*z^27 + 2*x1^50*x2^42*x3^30*x4^13*z^27 - 3*x1^49*x2^43*x3^30*x4^13*z^27 - x1^48*x2^44*x3^30*x4^13*z^27 - 5*x1^47*x2^45*x3^30*x4^13*z^27 - x1^46*x2^46*x3^30*x4^13*z^27 + x1^53*x2^38*x3^31*x4^13*z^27 - 2*x1^52*x2^39*x3^31*x4^13*z^27 - 2*x1^51*x2^40*x3^31*x4^13*z^27 - x1^50*x2^41*x3^31*x4^13*z^27 - 2*x1^49*x2^42*x3^31*x4^13*z^27 + x1^48*x2^43*x3^31*x4^13*z^27 + 3*x1^47*x2^44*x3^31*x4^13*z^27 + 4*x1^46*x2^45*x3^31*x4^13*z^27 - 2*x1^52*x2^38*x3^32*x4^13*z^27 + 3*x1^51*x2^39*x3^32*x4^13*z^27 + 2*x1^50*x2^40*x3^32*x4^13*z^27 + 2*x1^49*x2^41*x3^32*x4^13*z^27 - x1^48*x2^42*x3^32*x4^13*z^27 - 4*x1^46*x2^44*x3^32*x4^13*z^27 + x1^51*x2^38*x3^33*x4^13*z^27 - 3*x1^48*x2^41*x3^33*x4^13*z^27 + x1^47*x2^42*x3^33*x4^13*z^27 + x1^46*x2^43*x3^33*x4^13*z^27 + 2*x1^45*x2^44*x3^33*x4^13*z^27 - x1^50*x2^38*x3^34*x4^13*z^27 - x1^49*x2^39*x3^34*x4^13*z^27 + 2*x1^48*x2^40*x3^34*x4^13*z^27 - x1^47*x2^41*x3^34*x4^13*z^27 - 2*x1^46*x2^42*x3^34*x4^13*z^27 - x1^45*x2^43*x3^34*x4^13*z^27 + 2*x1^58*x2^42*x3^21*x4^14*z^27 - x1^57*x2^43*x3^21*x4^14*z^27 - 2*x1^56*x2^44*x3^21*x4^14*z^27 + x1^55*x2^45*x3^21*x4^14*z^27 - x1^58*x2^41*x3^22*x4^14*z^27 + 3*x1^56*x2^43*x3^22*x4^14*z^27 - x1^54*x2^45*x3^22*x4^14*z^27 + 3*x1^57*x2^41*x3^23*x4^14*z^27 + 2*x1^56*x2^42*x3^23*x4^14*z^27 - x1^53*x2^45*x3^23*x4^14*z^27 + x1^52*x2^46*x3^23*x4^14*z^27 - x1^57*x2^40*x3^24*x4^14*z^27 - 2*x1^56*x2^41*x3^24*x4^14*z^27 + 2*x1^55*x2^42*x3^24*x4^14*z^27 + x1^53*x2^44*x3^24*x4^14*z^27 + x1^52*x2^45*x3^24*x4^14*z^27 - x1^51*x2^46*x3^24*x4^14*z^27 + 2*x1^56*x2^40*x3^25*x4^14*z^27 - x1^54*x2^42*x3^25*x4^14*z^27 - 3*x1^52*x2^44*x3^25*x4^14*z^27 - x1^51*x2^45*x3^25*x4^14*z^27 + x1^50*x2^46*x3^25*x4^14*z^27 + x1^49*x2^47*x3^25*x4^14*z^27 - x1^56*x2^39*x3^26*x4^14*z^27 - 2*x1^55*x2^40*x3^26*x4^14*z^27 + 2*x1^54*x2^41*x3^26*x4^14*z^27 + 3*x1^52*x2^43*x3^26*x4^14*z^27 + x1^51*x2^44*x3^26*x4^14*z^27 + x1^50*x2^45*x3^26*x4^14*z^27 - 2*x1^49*x2^46*x3^26*x4^14*z^27 - x1^48*x2^47*x3^26*x4^14*z^27 + 2*x1^55*x2^39*x3^27*x4^14*z^27 - x1^54*x2^40*x3^27*x4^14*z^27 - 2*x1^53*x2^41*x3^27*x4^14*z^27 - x1^51*x2^43*x3^27*x4^14*z^27 + 2*x1^50*x2^44*x3^27*x4^14*z^27 - 2*x1^49*x2^45*x3^27*x4^14*z^27 + 2*x1^48*x2^46*x3^27*x4^14*z^27 - 2*x1^54*x2^39*x3^28*x4^14*z^27 + 2*x1^53*x2^40*x3^28*x4^14*z^27 - 2*x1^52*x2^41*x3^28*x4^14*z^27 + x1^51*x2^42*x3^28*x4^14*z^27 + 2*x1^49*x2^44*x3^28*x4^14*z^27 + x1^48*x2^45*x3^28*x4^14*z^27 - 2*x1^47*x2^46*x3^28*x4^14*z^27 + x1^54*x2^38*x3^29*x4^14*z^27 + x1^53*x2^39*x3^29*x4^14*z^27 - x1^52*x2^40*x3^29*x4^14*z^27 - 3*x1^50*x2^42*x3^29*x4^14*z^27 - 2*x1^48*x2^44*x3^29*x4^14*z^27 + 2*x1^47*x2^45*x3^29*x4^14*z^27 + x1^46*x2^46*x3^29*x4^14*z^27 - x1^54*x2^37*x3^30*x4^14*z^27 - x1^53*x2^38*x3^30*x4^14*z^27 + x1^52*x2^39*x3^30*x4^14*z^27 + x1^50*x2^41*x3^30*x4^14*z^27 - x1^49*x2^42*x3^30*x4^14*z^27 - 2*x1^46*x2^45*x3^30*x4^14*z^27 + x1^53*x2^37*x3^31*x4^14*z^27 - x1^52*x2^38*x3^31*x4^14*z^27 - x1^51*x2^39*x3^31*x4^14*z^27 - x1^49*x2^41*x3^31*x4^14*z^27 - x1^48*x2^42*x3^31*x4^14*z^27 - x1^47*x2^43*x3^31*x4^14*z^27 + 2*x1^46*x2^44*x3^31*x4^14*z^27 + x1^45*x2^45*x3^31*x4^14*z^27 - x1^52*x2^37*x3^32*x4^14*z^27 - x1^50*x2^39*x3^32*x4^14*z^27 - x1^47*x2^42*x3^32*x4^14*z^27 - 2*x1^46*x2^43*x3^32*x4^14*z^27 - 2*x1^45*x2^44*x3^32*x4^14*z^27 + x1^51*x2^37*x3^33*x4^14*z^27 + x1^48*x2^40*x3^33*x4^14*z^27 + x1^47*x2^41*x3^33*x4^14*z^27 - x1^46*x2^42*x3^33*x4^14*z^27 + x1^45*x2^43*x3^33*x4^14*z^27 + x1^48*x2^39*x3^34*x4^14*z^27 + x1^47*x2^40*x3^34*x4^14*z^27 - x1^44*x2^43*x3^34*x4^14*z^27 + x1^48*x2^38*x3^35*x4^14*z^27 + x1^45*x2^41*x3^35*x4^14*z^27 - x1^58*x2^42*x3^20*x4^15*z^27 + 2*x1^57*x2^42*x3^21*x4^15*z^27 + x1^56*x2^43*x3^21*x4^15*z^27 - 4*x1^57*x2^41*x3^22*x4^15*z^27 - x1^56*x2^42*x3^22*x4^15*z^27 - x1^54*x2^44*x3^22*x4^15*z^27 + x1^57*x2^40*x3^23*x4^15*z^27 + 5*x1^56*x2^41*x3^23*x4^15*z^27 + 3*x1^54*x2^43*x3^23*x4^15*z^27 - x1^53*x2^44*x3^23*x4^15*z^27 - x1^51*x2^46*x3^23*x4^15*z^27 - 6*x1^56*x2^40*x3^24*x4^15*z^27 - 2*x1^55*x2^41*x3^24*x4^15*z^27 - x1^54*x2^42*x3^24*x4^15*z^27 - x1^53*x2^43*x3^24*x4^15*z^27 + x1^52*x2^44*x3^24*x4^15*z^27 + x1^51*x2^45*x3^24*x4^15*z^27 + x1^50*x2^46*x3^24*x4^15*z^27 + 2*x1^56*x2^39*x3^25*x4^15*z^27 + 6*x1^55*x2^40*x3^25*x4^15*z^27 + 4*x1^53*x2^42*x3^25*x4^15*z^27 - 2*x1^52*x2^43*x3^25*x4^15*z^27 - 3*x1^50*x2^45*x3^25*x4^15*z^27 - 6*x1^55*x2^39*x3^26*x4^15*z^27 - 2*x1^54*x2^40*x3^26*x4^15*z^27 - 2*x1^53*x2^41*x3^26*x4^15*z^27 + 2*x1^51*x2^43*x3^26*x4^15*z^27 + 3*x1^50*x2^44*x3^26*x4^15*z^27 + 3*x1^49*x2^45*x3^26*x4^15*z^27 + 2*x1^55*x2^38*x3^27*x4^15*z^27 + 6*x1^54*x2^39*x3^27*x4^15*z^27 + 4*x1^52*x2^41*x3^27*x4^15*z^27 - 4*x1^51*x2^42*x3^27*x4^15*z^27 - 6*x1^49*x2^44*x3^27*x4^15*z^27 - 6*x1^54*x2^38*x3^28*x4^15*z^27 - 2*x1^53*x2^39*x3^28*x4^15*z^27 - 2*x1^52*x2^40*x3^28*x4^15*z^27 + 2*x1^50*x2^42*x3^28*x4^15*z^27 + 2*x1^49*x2^43*x3^28*x4^15*z^27 + 6*x1^48*x2^44*x3^28*x4^15*z^27 + 2*x1^54*x2^37*x3^29*x4^15*z^27 + 6*x1^53*x2^38*x3^29*x4^15*z^27 + 4*x1^51*x2^40*x3^29*x4^15*z^27 - 4*x1^50*x2^41*x3^29*x4^15*z^27 - 6*x1^48*x2^43*x3^29*x4^15*z^27 - 2*x1^47*x2^44*x3^29*x4^15*z^27 - 4*x1^53*x2^37*x3^30*x4^15*z^27 - 2*x1^52*x2^38*x3^30*x4^15*z^27 - x1^51*x2^39*x3^30*x4^15*z^27 + 2*x1^49*x2^41*x3^30*x4^15*z^27 + 2*x1^48*x2^42*x3^30*x4^15*z^27 + 6*x1^47*x2^43*x3^30*x4^15*z^27 + 4*x1^52*x2^37*x3^31*x4^15*z^27 + x1^51*x2^38*x3^31*x4^15*z^27 + 3*x1^50*x2^39*x3^31*x4^15*z^27 - 3*x1^49*x2^40*x3^31*x4^15*z^27 + x1^48*x2^41*x3^31*x4^15*z^27 - 5*x1^47*x2^42*x3^31*x4^15*z^27 - 2*x1^46*x2^43*x3^31*x4^15*z^27 - 2*x1^51*x2^37*x3^32*x4^15*z^27 - x1^50*x2^38*x3^32*x4^15*z^27 - x1^49*x2^39*x3^32*x4^15*z^27 + x1^48*x2^40*x3^32*x4^15*z^27 + 3*x1^47*x2^41*x3^32*x4^15*z^27 + 3*x1^46*x2^42*x3^32*x4^15*z^27 + 2*x1^50*x2^37*x3^33*x4^15*z^27 + 2*x1^49*x2^38*x3^33*x4^15*z^27 - 4*x1^46*x2^41*x3^33*x4^15*z^27 - 2*x1^49*x2^37*x3^34*x4^15*z^27 + x1^46*x2^40*x3^34*x4^15*z^27 + x1^45*x2^41*x3^34*x4^15*z^27 - x1^47*x2^38*x3^35*x4^15*z^27 + x1^46*x2^39*x3^35*x4^15*z^27 - x1^45*x2^40*x3^35*x4^15*z^27 + x1^57*x2^41*x3^21*x4^16*z^27 - x1^55*x2^43*x3^21*x4^16*z^27 - x1^54*x2^44*x3^21*x4^16*z^27 - x1^53*x2^45*x3^21*x4^16*z^27 - 3*x1^56*x2^41*x3^22*x4^16*z^27 - x1^55*x2^42*x3^22*x4^16*z^27 + 2*x1^54*x2^43*x3^22*x4^16*z^27 + 2*x1^53*x2^44*x3^22*x4^16*z^27 + x1^52*x2^45*x3^22*x4^16*z^27 + 2*x1^56*x2^40*x3^23*x4^16*z^27 + 2*x1^55*x2^41*x3^23*x4^16*z^27 - x1^54*x2^42*x3^23*x4^16*z^27 - 2*x1^53*x2^43*x3^23*x4^16*z^27 - x1^52*x2^44*x3^23*x4^16*z^27 - x1^51*x2^45*x3^23*x4^16*z^27 - 5*x1^55*x2^40*x3^24*x4^16*z^27 - x1^54*x2^41*x3^24*x4^16*z^27 + 4*x1^52*x2^43*x3^24*x4^16*z^27 + x1^51*x2^44*x3^24*x4^16*z^27 + 4*x1^50*x2^45*x3^24*x4^16*z^27 + 5*x1^55*x2^39*x3^25*x4^16*z^27 + x1^54*x2^40*x3^25*x4^16*z^27 + x1^53*x2^41*x3^25*x4^16*z^27 - 2*x1^51*x2^43*x3^25*x4^16*z^27 - 3*x1^50*x2^44*x3^25*x4^16*z^27 - 4*x1^49*x2^45*x3^25*x4^16*z^27 - 2*x1^55*x2^38*x3^26*x4^16*z^27 - 6*x1^54*x2^39*x3^26*x4^16*z^27 - 4*x1^52*x2^41*x3^26*x4^16*z^27 + 4*x1^51*x2^42*x3^26*x4^16*z^27 + 6*x1^49*x2^44*x3^26*x4^16*z^27 + x1^48*x2^45*x3^26*x4^16*z^27 + 6*x1^54*x2^38*x3^27*x4^16*z^27 + 2*x1^53*x2^39*x3^27*x4^16*z^27 + 2*x1^52*x2^40*x3^27*x4^16*z^27 - 2*x1^50*x2^42*x3^27*x4^16*z^27 - 2*x1^49*x2^43*x3^27*x4^16*z^27 - 6*x1^48*x2^44*x3^27*x4^16*z^27 - x1^54*x2^37*x3^28*x4^16*z^27 - 6*x1^53*x2^38*x3^28*x4^16*z^27 - 4*x1^51*x2^40*x3^28*x4^16*z^27 + 4*x1^50*x2^41*x3^28*x4^16*z^27 + 6*x1^48*x2^43*x3^28*x4^16*z^27 + 2*x1^47*x2^44*x3^28*x4^16*z^27 + 3*x1^53*x2^37*x3^29*x4^16*z^27 + 4*x1^52*x2^38*x3^29*x4^16*z^27 + 2*x1^51*x2^39*x3^29*x4^16*z^27 - 2*x1^49*x2^41*x3^29*x4^16*z^27 - 2*x1^48*x2^42*x3^29*x4^16*z^27 - 6*x1^47*x2^43*x3^29*x4^16*z^27 - 3*x1^52*x2^37*x3^30*x4^16*z^27 - 2*x1^51*x2^38*x3^30*x4^16*z^27 - 3*x1^50*x2^39*x3^30*x4^16*z^27 + 4*x1^49*x2^40*x3^30*x4^16*z^27 + 6*x1^47*x2^42*x3^30*x4^16*z^27 + 2*x1^46*x2^43*x3^30*x4^16*z^27 + x1^51*x2^37*x3^31*x4^16*z^27 + 2*x1^50*x2^38*x3^31*x4^16*z^27 + x1^49*x2^39*x3^31*x4^16*z^27 - 2*x1^47*x2^41*x3^31*x4^16*z^27 - 6*x1^46*x2^42*x3^31*x4^16*z^27 - 2*x1^49*x2^38*x3^32*x4^16*z^27 + 2*x1^48*x2^39*x3^32*x4^16*z^27 - 2*x1^47*x2^40*x3^32*x4^16*z^27 + 6*x1^46*x2^41*x3^32*x4^16*z^27 + 2*x1^45*x2^42*x3^32*x4^16*z^27 - x1^47*x2^39*x3^33*x4^16*z^27 - 4*x1^45*x2^41*x3^33*x4^16*z^27 + x1^47*x2^38*x3^34*x4^16*z^27 - x1^46*x2^39*x3^34*x4^16*z^27 + 2*x1^45*x2^40*x3^34*x4^16*z^27 + x1^44*x2^41*x3^34*x4^16*z^27 - x1^46*x2^38*x3^35*x4^16*z^27 + x1^53*x2^43*x3^22*x4^17*z^27 + x1^52*x2^44*x3^22*x4^17*z^27 - x1^52*x2^43*x3^23*x4^17*z^27 - x1^50*x2^45*x3^23*x4^17*z^27 - x1^54*x2^40*x3^24*x4^17*z^27 - x1^53*x2^41*x3^24*x4^17*z^27 + x1^52*x2^42*x3^24*x4^17*z^27 + x1^51*x2^43*x3^24*x4^17*z^27 + x1^50*x2^44*x3^24*x4^17*z^27 + 2*x1^49*x2^45*x3^24*x4^17*z^27 + 2*x1^54*x2^39*x3^25*x4^17*z^27 - x1^52*x2^41*x3^25*x4^17*z^27 - x1^51*x2^42*x3^25*x4^17*z^27 - 2*x1^49*x2^44*x3^25*x4^17*z^27 - x1^48*x2^45*x3^25*x4^17*z^27 - x1^54*x2^38*x3^26*x4^17*z^27 + x1^50*x2^42*x3^26*x4^17*z^27 + x1^49*x2^43*x3^26*x4^17*z^27 + 2*x1^48*x2^44*x3^26*x4^17*z^27 + x1^54*x2^37*x3^27*x4^17*z^27 + 2*x1^53*x2^38*x3^27*x4^17*z^27 + x1^51*x2^40*x3^27*x4^17*z^27 - 2*x1^50*x2^41*x3^27*x4^17*z^27 - 2*x1^48*x2^43*x3^27*x4^17*z^27 - x1^53*x2^37*x3^28*x4^17*z^27 + 2*x1^47*x2^43*x3^28*x4^17*z^27 + x1^52*x2^37*x3^29*x4^17*z^27 + 2*x1^50*x2^39*x3^29*x4^17*z^27 - x1^49*x2^40*x3^29*x4^17*z^27 - 2*x1^47*x2^42*x3^29*x4^17*z^27 - x1^46*x2^43*x3^29*x4^17*z^27 - x1^51*x2^37*x3^30*x4^17*z^27 + x1^48*x2^40*x3^30*x4^17*z^27 + x1^47*x2^41*x3^30*x4^17*z^27 + 2*x1^46*x2^42*x3^30*x4^17*z^27 - x1^49*x2^38*x3^31*x4^17*z^27 - 2*x1^48*x2^39*x3^31*x4^17*z^27 - 2*x1^46*x2^41*x3^31*x4^17*z^27 - x1^45*x2^42*x3^31*x4^17*z^27 + x1^47*x2^39*x3^32*x4^17*z^27 + 2*x1^45*x2^41*x3^32*x4^17*z^27 - x1^45*x2^40*x3^33*x4^17*z^27 + x1^44*x2^40*x3^34*x4^17*z^27 + x1^59*x2^45*x3^24*x4^2*z^26 - x1^58*x2^44*x3^25*x4^3*z^26 + x1^57*x2^44*x3^26*x4^3*z^26 - 2*x1^59*x2^43*x3^24*x4^4*z^26 - x1^58*x2^44*x3^24*x4^4*z^26 - x1^57*x2^45*x3^24*x4^4*z^26 + x1^59*x2^42*x3^25*x4^4*z^26 - x1^57*x2^44*x3^25*x4^4*z^26 + x1^56*x2^45*x3^25*x4^4*z^26 - 2*x1^58*x2^42*x3^26*x4^4*z^26 - x1^56*x2^44*x3^26*x4^4*z^26 - 2*x1^56*x2^43*x3^27*x4^4*z^26 + x1^54*x2^45*x3^27*x4^4*z^26 - x1^60*x2^43*x3^22*x4^5*z^26 + 3*x1^59*x2^43*x3^23*x4^5*z^26 - x1^58*x2^44*x3^23*x4^5*z^26 - 2*x1^59*x2^42*x3^24*x4^5*z^26 - 2*x1^58*x2^43*x3^24*x4^5*z^26 + 2*x1^57*x2^44*x3^24*x4^5*z^26 - 2*x1^56*x2^45*x3^24*x4^5*z^26 + 5*x1^58*x2^42*x3^25*x4^5*z^26 - 2*x1^57*x2^43*x3^25*x4^5*z^26 + x1^55*x2^45*x3^25*x4^5*z^26 - x1^54*x2^46*x3^25*x4^5*z^26 - 2*x1^58*x2^41*x3^26*x4^5*z^26 - 2*x1^57*x2^42*x3^26*x4^5*z^26 + 2*x1^56*x2^43*x3^26*x4^5*z^26 - 4*x1^55*x2^44*x3^26*x4^5*z^26 - x1^54*x2^45*x3^26*x4^5*z^26 - x1^53*x2^46*x3^26*x4^5*z^26 + 3*x1^57*x2^41*x3^27*x4^5*z^26 - x1^56*x2^42*x3^27*x4^5*z^26 + x1^54*x2^44*x3^27*x4^5*z^26 - x1^56*x2^41*x3^28*x4^5*z^26 + x1^55*x2^42*x3^28*x4^5*z^26 - 3*x1^54*x2^43*x3^28*x4^5*z^26 + x1^52*x2^45*x3^28*x4^5*z^26 - x1^54*x2^42*x3^29*x4^5*z^26 - x1^53*x2^43*x3^29*x4^5*z^26 - x1^52*x2^44*x3^29*x4^5*z^26 - x1^59*x2^43*x3^22*x4^6*z^26 + x1^58*x2^43*x3^23*x4^6*z^26 - x1^57*x2^44*x3^23*x4^6*z^26 - x1^58*x2^42*x3^24*x4^6*z^26 - x1^57*x2^43*x3^24*x4^6*z^26 + x1^55*x2^45*x3^24*x4^6*z^26 + x1^58*x2^41*x3^25*x4^6*z^26 + 2*x1^57*x2^42*x3^25*x4^6*z^26 - 2*x1^56*x2^43*x3^25*x4^6*z^26 + x1^55*x2^44*x3^25*x4^6*z^26 - x1^53*x2^46*x3^25*x4^6*z^26 - 2*x1^57*x2^41*x3^26*x4^6*z^26 + 4*x1^56*x2^42*x3^26*x4^6*z^26 + x1^55*x2^43*x3^26*x4^6*z^26 - 2*x1^54*x2^44*x3^26*x4^6*z^26 + x1^53*x2^45*x3^26*x4^6*z^26 + 3*x1^52*x2^46*x3^26*x4^6*z^26 + x1^57*x2^40*x3^27*x4^6*z^26 + 2*x1^56*x2^41*x3^27*x4^6*z^26 - 2*x1^55*x2^42*x3^27*x4^6*z^26 + 2*x1^54*x2^43*x3^27*x4^6*z^26 - x1^53*x2^44*x3^27*x4^6*z^26 - x1^56*x2^40*x3^28*x4^6*z^26 + x1^55*x2^41*x3^28*x4^6*z^26 + x1^54*x2^42*x3^28*x4^6*z^26 - x1^53*x2^43*x3^28*x4^6*z^26 + x1^55*x2^40*x3^29*x4^6*z^26 + x1^54*x2^41*x3^29*x4^6*z^26 + 3*x1^53*x2^42*x3^29*x4^6*z^26 - x1^51*x2^44*x3^29*x4^6*z^26 + x1^53*x2^41*x3^30*x4^6*z^26 + x1^52*x2^42*x3^30*x4^6*z^26 + x1^51*x2^43*x3^30*x4^6*z^26 + x1^58*x2^44*x3^21*x4^7*z^26 - 2*x1^57*x2^44*x3^22*x4^7*z^26 + x1^56*x2^45*x3^22*x4^7*z^26 - x1^57*x2^43*x3^23*x4^7*z^26 + x1^56*x2^44*x3^23*x4^7*z^26 - x1^55*x2^45*x3^23*x4^7*z^26 - x1^53*x2^47*x3^23*x4^7*z^26 - x1^56*x2^43*x3^24*x4^7*z^26 - x1^55*x2^44*x3^24*x4^7*z^26 - x1^54*x2^45*x3^24*x4^7*z^26 + x1^52*x2^47*x3^24*x4^7*z^26 + x1^56*x2^42*x3^25*x4^7*z^26 + x1^55*x2^43*x3^25*x4^7*z^26 - 2*x1^54*x2^44*x3^25*x4^7*z^26 - 2*x1^56*x2^41*x3^26*x4^7*z^26 + 2*x1^55*x2^42*x3^26*x4^7*z^26 + x1^54*x2^43*x3^26*x4^7*z^26 + x1^52*x2^45*x3^26*x4^7*z^26 + x1^51*x2^46*x3^26*x4^7*z^26 + x1^55*x2^41*x3^27*x4^7*z^26 + x1^53*x2^43*x3^27*x4^7*z^26 + x1^52*x2^44*x3^27*x4^7*z^26 - 2*x1^51*x2^45*x3^27*x4^7*z^26 + x1^50*x2^46*x3^27*x4^7*z^26 + x1^49*x2^47*x3^27*x4^7*z^26 - x1^56*x2^39*x3^28*x4^7*z^26 - x1^55*x2^40*x3^28*x4^7*z^26 + x1^54*x2^41*x3^28*x4^7*z^26 - x1^53*x2^42*x3^28*x4^7*z^26 + x1^52*x2^43*x3^28*x4^7*z^26 - x1^51*x2^44*x3^28*x4^7*z^26 + x1^49*x2^46*x3^28*x4^7*z^26 + x1^53*x2^41*x3^29*x4^7*z^26 + x1^52*x2^42*x3^29*x4^7*z^26 + x1^51*x2^43*x3^29*x4^7*z^26 - x1^55*x2^38*x3^30*x4^7*z^26 - x1^54*x2^39*x3^30*x4^7*z^26 - 2*x1^52*x2^41*x3^30*x4^7*z^26 + x1^51*x2^42*x3^30*x4^7*z^26 - 2*x1^58*x2^44*x3^20*x4^8*z^26 + x1^59*x2^42*x3^21*x4^8*z^26 + 3*x1^57*x2^44*x3^21*x4^8*z^26 - x1^56*x2^45*x3^21*x4^8*z^26 - x1^58*x2^42*x3^22*x4^8*z^26 - x1^57*x2^43*x3^22*x4^8*z^26 - 3*x1^56*x2^44*x3^22*x4^8*z^26 - 2*x1^54*x2^46*x3^22*x4^8*z^26 + x1^57*x2^42*x3^23*x4^8*z^26 + 5*x1^56*x2^43*x3^23*x4^8*z^26 + 3*x1^55*x2^44*x3^23*x4^8*z^26 + x1^54*x2^45*x3^23*x4^8*z^26 + x1^53*x2^46*x3^23*x4^8*z^26 - x1^52*x2^47*x3^23*x4^8*z^26 - x1^57*x2^41*x3^24*x4^8*z^26 - 2*x1^56*x2^42*x3^24*x4^8*z^26 - 5*x1^55*x2^43*x3^24*x4^8*z^26 - 2*x1^53*x2^45*x3^24*x4^8*z^26 + x1^51*x2^47*x3^24*x4^8*z^26 + x1^57*x2^40*x3^25*x4^8*z^26 + x1^56*x2^41*x3^25*x4^8*z^26 + 4*x1^55*x2^42*x3^25*x4^8*z^26 + 3*x1^54*x2^43*x3^25*x4^8*z^26 + 3*x1^53*x2^44*x3^25*x4^8*z^26 + x1^52*x2^45*x3^25*x4^8*z^26 - x1^50*x2^47*x3^25*x4^8*z^26 - x1^56*x2^40*x3^26*x4^8*z^26 - 2*x1^55*x2^41*x3^26*x4^8*z^26 - 4*x1^54*x2^42*x3^26*x4^8*z^26 - x1^53*x2^43*x3^26*x4^8*z^26 - 3*x1^52*x2^44*x3^26*x4^8*z^26 - x1^50*x2^46*x3^26*x4^8*z^26 + 2*x1^49*x2^47*x3^26*x4^8*z^26 + x1^56*x2^39*x3^27*x4^8*z^26 + 2*x1^55*x2^40*x3^27*x4^8*z^26 + 3*x1^54*x2^41*x3^27*x4^8*z^26 + x1^53*x2^42*x3^27*x4^8*z^26 + x1^52*x2^43*x3^27*x4^8*z^26 + 3*x1^51*x2^44*x3^27*x4^8*z^26 - x1^50*x2^45*x3^27*x4^8*z^26 - x1^49*x2^46*x3^27*x4^8*z^26 - x1^48*x2^47*x3^27*x4^8*z^26 - x1^55*x2^39*x3^28*x4^8*z^26 - 2*x1^54*x2^40*x3^28*x4^8*z^26 - 3*x1^53*x2^41*x3^28*x4^8*z^26 - x1^52*x2^42*x3^28*x4^8*z^26 - 5*x1^51*x2^43*x3^28*x4^8*z^26 + x1^50*x2^44*x3^28*x4^8*z^26 + x1^49*x2^45*x3^28*x4^8*z^26 + x1^54*x2^39*x3^29*x4^8*z^26 + 2*x1^53*x2^40*x3^29*x4^8*z^26 + x1^52*x2^41*x3^29*x4^8*z^26 + x1^50*x2^43*x3^29*x4^8*z^26 - 2*x1^49*x2^44*x3^29*x4^8*z^26 - x1^48*x2^45*x3^29*x4^8*z^26 - x1^54*x2^38*x3^30*x4^8*z^26 - x1^53*x2^39*x3^30*x4^8*z^26 - 2*x1^52*x2^40*x3^30*x4^8*z^26 - x1^51*x2^41*x3^30*x4^8*z^26 - 3*x1^50*x2^42*x3^30*x4^8*z^26 + x1^54*x2^37*x3^31*x4^8*z^26 + 2*x1^51*x2^40*x3^31*x4^8*z^26 + x1^49*x2^42*x3^31*x4^8*z^26 - x1^53*x2^37*x3^32*x4^8*z^26 - 2*x1^51*x2^39*x3^32*x4^8*z^26 - x1^50*x2^40*x3^32*x4^8*z^26 - x1^49*x2^41*x3^32*x4^8*z^26 - x1^58*x2^42*x3^21*x4^9*z^26 + x1^57*x2^43*x3^21*x4^9*z^26 - x1^55*x2^45*x3^21*x4^9*z^26 + x1^57*x2^42*x3^22*x4^9*z^26 - 2*x1^56*x2^43*x3^22*x4^9*z^26 + x1^55*x2^44*x3^22*x4^9*z^26 + x1^54*x2^45*x3^22*x4^9*z^26 + x1^57*x2^41*x3^23*x4^9*z^26 - x1^56*x2^42*x3^23*x4^9*z^26 + x1^55*x2^43*x3^23*x4^9*z^26 - x1^54*x2^44*x3^23*x4^9*z^26 + x1^53*x2^45*x3^23*x4^9*z^26 - x1^57*x2^40*x3^24*x4^9*z^26 - x1^56*x2^41*x3^24*x4^9*z^26 - 2*x1^54*x2^43*x3^24*x4^9*z^26 - 2*x1^53*x2^44*x3^24*x4^9*z^26 + x1^52*x2^45*x3^24*x4^9*z^26 + x1^51*x2^46*x3^24*x4^9*z^26 + 4*x1^56*x2^40*x3^25*x4^9*z^26 + 2*x1^55*x2^41*x3^25*x4^9*z^26 + 2*x1^54*x2^42*x3^25*x4^9*z^26 + x1^53*x2^43*x3^25*x4^9*z^26 + x1^52*x2^44*x3^25*x4^9*z^26 - x1^51*x2^45*x3^25*x4^9*z^26 - x1^49*x2^47*x3^25*x4^9*z^26 - 2*x1^56*x2^39*x3^26*x4^9*z^26 - 4*x1^55*x2^40*x3^26*x4^9*z^26 - 2*x1^54*x2^41*x3^26*x4^9*z^26 - x1^53*x2^42*x3^26*x4^9*z^26 + x1^50*x2^45*x3^26*x4^9*z^26 + x1^49*x2^46*x3^26*x4^9*z^26 + 3*x1^55*x2^39*x3^27*x4^9*z^26 + x1^54*x2^40*x3^27*x4^9*z^26 + 2*x1^53*x2^41*x3^27*x4^9*z^26 + x1^52*x2^42*x3^27*x4^9*z^26 + x1^51*x2^43*x3^27*x4^9*z^26 - x1^50*x2^44*x3^27*x4^9*z^26 - 2*x1^48*x2^46*x3^27*x4^9*z^26 - x1^55*x2^38*x3^28*x4^9*z^26 - 2*x1^54*x2^39*x3^28*x4^9*z^26 - 2*x1^53*x2^40*x3^28*x4^9*z^26 - 3*x1^52*x2^41*x3^28*x4^9*z^26 + x1^51*x2^42*x3^28*x4^9*z^26 - x1^50*x2^43*x3^28*x4^9*z^26 + x1^49*x2^44*x3^28*x4^9*z^26 + 3*x1^54*x2^38*x3^29*x4^9*z^26 + x1^53*x2^39*x3^29*x4^9*z^26 + x1^52*x2^40*x3^29*x4^9*z^26 + 2*x1^51*x2^41*x3^29*x4^9*z^26 + 2*x1^50*x2^42*x3^29*x4^9*z^26 + x1^49*x2^43*x3^29*x4^9*z^26 - x1^47*x2^45*x3^29*x4^9*z^26 - x1^54*x2^37*x3^30*x4^9*z^26 - x1^53*x2^38*x3^30*x4^9*z^26 - 3*x1^51*x2^40*x3^30*x4^9*z^26 + 2*x1^48*x2^43*x3^30*x4^9*z^26 + x1^47*x2^44*x3^30*x4^9*z^26 + 2*x1^53*x2^37*x3^31*x4^9*z^26 + x1^51*x2^39*x3^31*x4^9*z^26 + x1^50*x2^40*x3^31*x4^9*z^26 - x1^49*x2^41*x3^31*x4^9*z^26 + x1^51*x2^38*x3^32*x4^9*z^26 - x1^48*x2^41*x3^32*x4^9*z^26 - x1^51*x2^37*x3^33*x4^9*z^26 + x1^49*x2^39*x3^33*x4^9*z^26 + x1^48*x2^40*x3^33*x4^9*z^26 - x1^59*x2^42*x3^19*x4^10*z^26 + 2*x1^58*x2^42*x3^20*x4^10*z^26 + x1^56*x2^44*x3^20*x4^10*z^26 - 2*x1^57*x2^42*x3^21*x4^10*z^26 + x1^54*x2^45*x3^21*x4^10*z^26 + 2*x1^57*x2^41*x3^22*x4^10*z^26 + x1^56*x2^42*x3^22*x4^10*z^26 + x1^55*x2^43*x3^22*x4^10*z^26 + x1^52*x2^46*x3^22*x4^10*z^26 - 2*x1^56*x2^41*x3^23*x4^10*z^26 - 2*x1^54*x2^43*x3^23*x4^10*z^26 + x1^52*x2^45*x3^23*x4^10*z^26 + x1^51*x2^46*x3^23*x4^10*z^26 - x1^52*x2^44*x3^24*x4^10*z^26 - 2*x1^51*x2^45*x3^24*x4^10*z^26 - x1^50*x2^46*x3^24*x4^10*z^26 + x1^49*x2^47*x3^24*x4^10*z^26 + x1^52*x2^43*x3^25*x4^10*z^26 - x1^51*x2^44*x3^25*x4^10*z^26 + x1^50*x2^45*x3^25*x4^10*z^26 - x1^48*x2^47*x3^25*x4^10*z^26 - 2*x1^52*x2^42*x3^26*x4^10*z^26 - 2*x1^50*x2^44*x3^26*x4^10*z^26 - 3*x1^49*x2^45*x3^26*x4^10*z^26 + x1^50*x2^43*x3^27*x4^10*z^26 + 2*x1^49*x2^44*x3^27*x4^10*z^26 + x1^48*x2^45*x3^27*x4^10*z^26 + 2*x1^47*x2^46*x3^27*x4^10*z^26 - 2*x1^49*x2^43*x3^28*x4^10*z^26 - x1^48*x2^44*x3^28*x4^10*z^26 - x1^46*x2^46*x3^28*x4^10*z^26 - x1^53*x2^38*x3^29*x4^10*z^26 - x1^50*x2^41*x3^29*x4^10*z^26 + x1^49*x2^42*x3^29*x4^10*z^26 + x1^47*x2^44*x3^29*x4^10*z^26 + x1^46*x2^45*x3^29*x4^10*z^26 - 2*x1^53*x2^37*x3^30*x4^10*z^26 - x1^52*x2^38*x3^30*x4^10*z^26 + x1^51*x2^39*x3^30*x4^10*z^26 - x1^50*x2^40*x3^30*x4^10*z^26 + x1^49*x2^41*x3^30*x4^10*z^26 - x1^48*x2^42*x3^30*x4^10*z^26 + x1^53*x2^36*x3^31*x4^10*z^26 + x1^52*x2^37*x3^31*x4^10*z^26 - 2*x1^51*x2^38*x3^31*x4^10*z^26 + x1^49*x2^40*x3^31*x4^10*z^26 - x1^47*x2^42*x3^31*x4^10*z^26 + x1^46*x2^43*x3^31*x4^10*z^26 + x1^45*x2^44*x3^31*x4^10*z^26 - 2*x1^52*x2^36*x3^32*x4^10*z^26 + 2*x1^48*x2^40*x3^32*x4^10*z^26 + x1^51*x2^36*x3^33*x4^10*z^26 - x1^48*x2^38*x3^34*x4^10*z^26 + x1^47*x2^39*x3^34*x4^10*z^26 - x1^58*x2^42*x3^19*x4^11*z^26 + x1^57*x2^43*x3^19*x4^11*z^26 + x1^58*x2^41*x3^20*x4^11*z^26 + 2*x1^57*x2^42*x3^20*x4^11*z^26 - x1^56*x2^43*x3^20*x4^11*z^26 - 4*x1^57*x2^41*x3^21*x4^11*z^26 + x1^56*x2^42*x3^21*x4^11*z^26 + x1^53*x2^45*x3^21*x4^11*z^26 + x1^57*x2^40*x3^22*x4^11*z^26 + 4*x1^56*x2^41*x3^22*x4^11*z^26 - 3*x1^55*x2^42*x3^22*x4^11*z^26 + 2*x1^54*x2^43*x3^22*x4^11*z^26 - x1^53*x2^44*x3^22*x4^11*z^26 - 2*x1^52*x2^45*x3^22*x4^11*z^26 - x1^51*x2^46*x3^22*x4^11*z^26 - 4*x1^56*x2^40*x3^23*x4^11*z^26 - 2*x1^55*x2^41*x3^23*x4^11*z^26 + x1^54*x2^42*x3^23*x4^11*z^26 + x1^52*x2^44*x3^23*x4^11*z^26 + x1^51*x2^45*x3^23*x4^11*z^26 + x1^50*x2^46*x3^23*x4^11*z^26 + 2*x1^56*x2^39*x3^24*x4^11*z^26 + 4*x1^55*x2^40*x3^24*x4^11*z^26 - 2*x1^54*x2^41*x3^24*x4^11*z^26 + x1^53*x2^42*x3^24*x4^11*z^26 - 3*x1^52*x2^43*x3^24*x4^11*z^26 - 3*x1^50*x2^45*x3^24*x4^11*z^26 - x1^49*x2^46*x3^24*x4^11*z^26 - 4*x1^55*x2^39*x3^25*x4^11*z^26 + x1^53*x2^41*x3^25*x4^11*z^26 + 2*x1^51*x2^43*x3^25*x4^11*z^26 + 2*x1^50*x2^44*x3^25*x4^11*z^26 + 3*x1^49*x2^45*x3^25*x4^11*z^26 + x1^55*x2^38*x3^26*x4^11*z^26 + 4*x1^54*x2^39*x3^26*x4^11*z^26 - 2*x1^53*x2^40*x3^26*x4^11*z^26 + 2*x1^52*x2^41*x3^26*x4^11*z^26 - 3*x1^51*x2^42*x3^26*x4^11*z^26 - x1^50*x2^43*x3^26*x4^11*z^26 - 2*x1^49*x2^44*x3^26*x4^11*z^26 + x1^48*x2^45*x3^26*x4^11*z^26 - 4*x1^54*x2^38*x3^27*x4^11*z^26 + x1^52*x2^40*x3^27*x4^11*z^26 + x1^51*x2^41*x3^27*x4^11*z^26 + x1^50*x2^42*x3^27*x4^11*z^26 - x1^49*x2^43*x3^27*x4^11*z^26 + 4*x1^48*x2^44*x3^27*x4^11*z^26 - 2*x1^47*x2^45*x3^27*x4^11*z^26 + x1^46*x2^46*x3^27*x4^11*z^26 + 4*x1^53*x2^38*x3^28*x4^11*z^26 - 2*x1^52*x2^39*x3^28*x4^11*z^26 + 3*x1^51*x2^40*x3^28*x4^11*z^26 - 2*x1^49*x2^42*x3^28*x4^11*z^26 - 5*x1^48*x2^43*x3^28*x4^11*z^26 - x1^47*x2^44*x3^28*x4^11*z^26 + x1^46*x2^45*x3^28*x4^11*z^26 - 3*x1^52*x2^38*x3^29*x4^11*z^26 + x1^50*x2^40*x3^29*x4^11*z^26 + 2*x1^49*x2^41*x3^29*x4^11*z^26 + 2*x1^48*x2^42*x3^29*x4^11*z^26 + 2*x1^47*x2^43*x3^29*x4^11*z^26 - x1^46*x2^44*x3^29*x4^11*z^26 + x1^45*x2^45*x3^29*x4^11*z^26 + x1^52*x2^37*x3^30*x4^11*z^26 - x1^51*x2^38*x3^30*x4^11*z^26 - 4*x1^47*x2^42*x3^30*x4^11*z^26 - x1^46*x2^43*x3^30*x4^11*z^26 + x1^45*x2^44*x3^30*x4^11*z^26 + x1^52*x2^36*x3^31*x4^11*z^26 - x1^50*x2^38*x3^31*x4^11*z^26 + 2*x1^47*x2^41*x3^31*x4^11*z^26 - x1^45*x2^43*x3^31*x4^11*z^26 - x1^51*x2^36*x3^32*x4^11*z^26 - 2*x1^50*x2^37*x3^32*x4^11*z^26 - x1^49*x2^38*x3^32*x4^11*z^26 - 3*x1^48*x2^39*x3^32*x4^11*z^26 - 2*x1^47*x2^40*x3^32*x4^11*z^26 - x1^45*x2^42*x3^32*x4^11*z^26 - x1^44*x2^43*x3^32*x4^11*z^26 + x1^47*x2^39*x3^33*x4^11*z^26 - x1^58*x2^41*x3^19*x4^12*z^26 - x1^57*x2^42*x3^19*x4^12*z^26 + 2*x1^56*x2^43*x3^19*x4^12*z^26 + 2*x1^57*x2^41*x3^20*x4^12*z^26 - 2*x1^55*x2^43*x3^20*x4^12*z^26 + x1^54*x2^44*x3^20*x4^12*z^26 - 2*x1^56*x2^41*x3^21*x4^12*z^26 + 3*x1^55*x2^42*x3^21*x4^12*z^26 + x1^54*x2^43*x3^21*x4^12*z^26 + x1^53*x2^44*x3^21*x4^12*z^26 + 2*x1^56*x2^40*x3^22*x4^12*z^26 - x1^55*x2^41*x3^22*x4^12*z^26 - 2*x1^54*x2^42*x3^22*x4^12*z^26 + x1^53*x2^43*x3^22*x4^12*z^26 - 2*x1^52*x2^44*x3^22*x4^12*z^26 - x1^56*x2^39*x3^23*x4^12*z^26 - 2*x1^55*x2^40*x3^23*x4^12*z^26 + 6*x1^54*x2^41*x3^23*x4^12*z^26 - 2*x1^53*x2^42*x3^23*x4^12*z^26 + x1^52*x2^43*x3^23*x4^12*z^26 + x1^51*x2^44*x3^23*x4^12*z^26 + x1^50*x2^45*x3^23*x4^12*z^26 - x1^49*x2^46*x3^23*x4^12*z^26 + 2*x1^55*x2^39*x3^24*x4^12*z^26 - x1^54*x2^40*x3^24*x4^12*z^26 - 5*x1^53*x2^41*x3^24*x4^12*z^26 - 5*x1^51*x2^43*x3^24*x4^12*z^26 - x1^50*x2^44*x3^24*x4^12*z^26 - x1^49*x2^45*x3^24*x4^12*z^26 + 2*x1^48*x2^46*x3^24*x4^12*z^26 - x1^55*x2^38*x3^25*x4^12*z^26 - 2*x1^54*x2^39*x3^25*x4^12*z^26 + 6*x1^53*x2^40*x3^25*x4^12*z^26 + x1^52*x2^41*x3^25*x4^12*z^26 + 4*x1^51*x2^42*x3^25*x4^12*z^26 - x1^50*x2^43*x3^25*x4^12*z^26 - 2*x1^48*x2^45*x3^25*x4^12*z^26 - 2*x1^47*x2^46*x3^25*x4^12*z^26 + 2*x1^54*x2^38*x3^26*x4^12*z^26 - 2*x1^53*x2^39*x3^26*x4^12*z^26 - 6*x1^52*x2^40*x3^26*x4^12*z^26 - 4*x1^50*x2^42*x3^26*x4^12*z^26 + 4*x1^49*x2^43*x3^26*x4^12*z^26 - x1^48*x2^44*x3^26*x4^12*z^26 + 5*x1^47*x2^45*x3^26*x4^12*z^26 + x1^46*x2^46*x3^26*x4^12*z^26 - 2*x1^53*x2^38*x3^27*x4^12*z^26 + 6*x1^52*x2^39*x3^27*x4^12*z^26 + 2*x1^50*x2^41*x3^27*x4^12*z^26 + x1^49*x2^42*x3^27*x4^12*z^26 - x1^48*x2^43*x3^27*x4^12*z^26 - 2*x1^47*x2^44*x3^27*x4^12*z^26 - 5*x1^46*x2^45*x3^27*x4^12*z^26 + x1^53*x2^37*x3^28*x4^12*z^26 - x1^52*x2^38*x3^28*x4^12*z^26 - 5*x1^51*x2^39*x3^28*x4^12*z^26 - 3*x1^49*x2^41*x3^28*x4^12*z^26 + 2*x1^48*x2^42*x3^28*x4^12*z^26 - x1^47*x2^43*x3^28*x4^12*z^26 + 5*x1^46*x2^44*x3^28*x4^12*z^26 + x1^45*x2^45*x3^28*x4^12*z^26 - x1^53*x2^36*x3^29*x4^12*z^26 - x1^52*x2^37*x3^29*x4^12*z^26 + 5*x1^51*x2^38*x3^29*x4^12*z^26 + x1^50*x2^39*x3^29*x4^12*z^26 + 2*x1^49*x2^40*x3^29*x4^12*z^26 + 2*x1^48*x2^41*x3^29*x4^12*z^26 + x1^47*x2^42*x3^29*x4^12*z^26 - 2*x1^46*x2^43*x3^29*x4^12*z^26 - 4*x1^45*x2^44*x3^29*x4^12*z^26 + x1^52*x2^36*x3^30*x4^12*z^26 - x1^51*x2^37*x3^30*x4^12*z^26 - 5*x1^50*x2^38*x3^30*x4^12*z^26 - x1^49*x2^39*x3^30*x4^12*z^26 - 4*x1^48*x2^40*x3^30*x4^12*z^26 - 2*x1^46*x2^42*x3^30*x4^12*z^26 + 4*x1^45*x2^43*x3^30*x4^12*z^26 + 3*x1^50*x2^37*x3^31*x4^12*z^26 + x1^49*x2^38*x3^31*x4^12*z^26 + 3*x1^48*x2^39*x3^31*x4^12*z^26 + x1^47*x2^40*x3^31*x4^12*z^26 - x1^45*x2^42*x3^31*x4^12*z^26 - 2*x1^44*x2^43*x3^31*x4^12*z^26 + x1^50*x2^36*x3^32*x4^12*z^26 - x1^49*x2^37*x3^32*x4^12*z^26 - x1^47*x2^39*x3^32*x4^12*z^26 + 2*x1^46*x2^40*x3^32*x4^12*z^26 + x1^44*x2^42*x3^32*x4^12*z^26 - x1^48*x2^37*x3^33*x4^12*z^26 + 3*x1^46*x2^39*x3^33*x4^12*z^26 - x1^46*x2^38*x3^34*x4^12*z^26 - x1^55*x2^43*x3^19*x4^13*z^26 + x1^57*x2^40*x3^20*x4^13*z^26 + x1^56*x2^41*x3^20*x4^13*z^26 - 2*x1^55*x2^42*x3^20*x4^13*z^26 - x1^53*x2^44*x3^20*x4^13*z^26 - 2*x1^56*x2^40*x3^21*x4^13*z^26 + 3*x1^54*x2^42*x3^21*x4^13*z^26 - 2*x1^53*x2^43*x3^21*x4^13*z^26 - x1^51*x2^45*x3^21*x4^13*z^26 + x1^56*x2^39*x3^22*x4^13*z^26 - 6*x1^54*x2^41*x3^22*x4^13*z^26 + x1^51*x2^44*x3^22*x4^13*z^26 + x1^50*x2^45*x3^22*x4^13*z^26 + 2*x1^54*x2^40*x3^23*x4^13*z^26 + 7*x1^53*x2^41*x3^23*x4^13*z^26 + 3*x1^51*x2^43*x3^23*x4^13*z^26 - 2*x1^50*x2^44*x3^23*x4^13*z^26 - x1^49*x2^45*x3^23*x4^13*z^26 - x1^48*x2^46*x3^23*x4^13*z^26 - 6*x1^53*x2^40*x3^24*x4^13*z^26 - 2*x1^52*x2^41*x3^24*x4^13*z^26 - x1^51*x2^42*x3^24*x4^13*z^26 + 2*x1^49*x2^44*x3^24*x4^13*z^26 + 2*x1^48*x2^45*x3^24*x4^13*z^26 + x1^47*x2^46*x3^24*x4^13*z^26 + 2*x1^53*x2^39*x3^25*x4^13*z^26 + 6*x1^52*x2^40*x3^25*x4^13*z^26 + 4*x1^50*x2^42*x3^25*x4^13*z^26 - 2*x1^49*x2^43*x3^25*x4^13*z^26 - x1^48*x2^44*x3^25*x4^13*z^26 - 4*x1^47*x2^45*x3^25*x4^13*z^26 - 6*x1^52*x2^39*x3^26*x4^13*z^26 - 2*x1^51*x2^40*x3^26*x4^13*z^26 - 2*x1^50*x2^41*x3^26*x4^13*z^26 + 2*x1^48*x2^43*x3^26*x4^13*z^26 + 3*x1^47*x2^44*x3^26*x4^13*z^26 + 4*x1^46*x2^45*x3^26*x4^13*z^26 + 2*x1^52*x2^38*x3^27*x4^13*z^26 + 6*x1^51*x2^39*x3^27*x4^13*z^26 + 4*x1^49*x2^41*x3^27*x4^13*z^26 - 4*x1^48*x2^42*x3^27*x4^13*z^26 - 6*x1^46*x2^44*x3^27*x4^13*z^26 - x1^45*x2^45*x3^27*x4^13*z^26 - 6*x1^51*x2^38*x3^28*x4^13*z^26 - 2*x1^50*x2^39*x3^28*x4^13*z^26 - 2*x1^49*x2^40*x3^28*x4^13*z^26 + 2*x1^47*x2^42*x3^28*x4^13*z^26 + 2*x1^46*x2^43*x3^28*x4^13*z^26 + 6*x1^45*x2^44*x3^28*x4^13*z^26 - x1^51*x2^37*x3^29*x4^13*z^26 + 5*x1^50*x2^38*x3^29*x4^13*z^26 - x1^49*x2^39*x3^29*x4^13*z^26 + 3*x1^48*x2^40*x3^29*x4^13*z^26 - 3*x1^47*x2^41*x3^29*x4^13*z^26 - x1^46*x2^42*x3^29*x4^13*z^26 - 6*x1^45*x2^43*x3^29*x4^13*z^26 - 2*x1^44*x2^44*x3^29*x4^13*z^26 - x1^50*x2^37*x3^30*x4^13*z^26 - 2*x1^49*x2^38*x3^30*x4^13*z^26 - x1^48*x2^39*x3^30*x4^13*z^26 - x1^47*x2^40*x3^30*x4^13*z^26 + 3*x1^46*x2^41*x3^30*x4^13*z^26 + 3*x1^45*x2^42*x3^30*x4^13*z^26 + 5*x1^44*x2^43*x3^30*x4^13*z^26 + x1^49*x2^37*x3^31*x4^13*z^26 + x1^47*x2^39*x3^31*x4^13*z^26 - x1^45*x2^41*x3^31*x4^13*z^26 - 4*x1^44*x2^42*x3^31*x4^13*z^26 - x1^43*x2^43*x3^31*x4^13*z^26 + x1^49*x2^36*x3^32*x4^13*z^26 + 2*x1^48*x2^37*x3^32*x4^13*z^26 - x1^47*x2^38*x3^32*x4^13*z^26 - x1^46*x2^39*x3^32*x4^13*z^26 + x1^45*x2^40*x3^32*x4^13*z^26 + 2*x1^44*x2^41*x3^32*x4^13*z^26 + 2*x1^43*x2^42*x3^32*x4^13*z^26 + x1^47*x2^37*x3^33*x4^13*z^26 - 2*x1^43*x2^41*x3^33*x4^13*z^26 + x1^46*x2^37*x3^34*x4^13*z^26 - x1^45*x2^38*x3^34*x4^13*z^26 + x1^43*x2^40*x3^34*x4^13*z^26 + x1^42*x2^41*x3^34*x4^13*z^26 + 2*x1^56*x2^40*x3^20*x4^14*z^26 + x1^55*x2^41*x3^20*x4^14*z^26 - x1^54*x2^42*x3^20*x4^14*z^26 + x1^53*x2^43*x3^20*x4^14*z^26 - 2*x1^55*x2^40*x3^21*x4^14*z^26 - x1^54*x2^41*x3^21*x4^14*z^26 + x1^53*x2^42*x3^21*x4^14*z^26 - x1^50*x2^45*x3^21*x4^14*z^26 + 3*x1^55*x2^39*x3^22*x4^14*z^26 + x1^54*x2^40*x3^22*x4^14*z^26 - 2*x1^53*x2^41*x3^22*x4^14*z^26 - x1^52*x2^42*x3^22*x4^14*z^26 + x1^50*x2^44*x3^22*x4^14*z^26 + x1^49*x2^45*x3^22*x4^14*z^26 - x1^55*x2^38*x3^23*x4^14*z^26 - 2*x1^54*x2^39*x3^23*x4^14*z^26 + x1^53*x2^40*x3^23*x4^14*z^26 - 3*x1^52*x2^41*x3^23*x4^14*z^26 - x1^51*x2^42*x3^23*x4^14*z^26 - x1^48*x2^45*x3^23*x4^14*z^26 + 2*x1^54*x2^38*x3^24*x4^14*z^26 + x1^53*x2^39*x3^24*x4^14*z^26 - x1^52*x2^40*x3^24*x4^14*z^26 - 2*x1^50*x2^42*x3^24*x4^14*z^26 + 2*x1^47*x2^45*x3^24*x4^14*z^26 - x1^54*x2^37*x3^25*x4^14*z^26 - 2*x1^53*x2^38*x3^25*x4^14*z^26 + 2*x1^52*x2^39*x3^25*x4^14*z^26 + 2*x1^50*x2^41*x3^25*x4^14*z^26 + x1^48*x2^43*x3^25*x4^14*z^26 - 2*x1^46*x2^45*x3^25*x4^14*z^26 + 2*x1^53*x2^37*x3^26*x4^14*z^26 - x1^51*x2^39*x3^26*x4^14*z^26 - 2*x1^49*x2^41*x3^26*x4^14*z^26 - 2*x1^47*x2^43*x3^26*x4^14*z^26 + 2*x1^46*x2^44*x3^26*x4^14*z^26 + x1^45*x2^45*x3^26*x4^14*z^26 - 2*x1^52*x2^37*x3^27*x4^14*z^26 + 2*x1^51*x2^38*x3^27*x4^14*z^26 + 3*x1^49*x2^40*x3^27*x4^14*z^26 + x1^47*x2^42*x3^27*x4^14*z^26 - x1^46*x2^43*x3^27*x4^14*z^26 - 2*x1^45*x2^44*x3^27*x4^14*z^26 + x1^52*x2^36*x3^28*x4^14*z^26 - 2*x1^50*x2^38*x3^28*x4^14*z^26 - x1^48*x2^40*x3^28*x4^14*z^26 + 2*x1^47*x2^41*x3^28*x4^14*z^26 - 2*x1^46*x2^42*x3^28*x4^14*z^26 + 2*x1^45*x2^43*x3^28*x4^14*z^26 - x1^51*x2^36*x3^29*x4^14*z^26 + 2*x1^50*x2^37*x3^29*x4^14*z^26 + x1^48*x2^39*x3^29*x4^14*z^26 + 2*x1^46*x2^41*x3^29*x4^14*z^26 + x1^45*x2^42*x3^29*x4^14*z^26 - 2*x1^44*x2^43*x3^29*x4^14*z^26 + x1^51*x2^35*x3^30*x4^14*z^26 + x1^50*x2^36*x3^30*x4^14*z^26 - x1^49*x2^37*x3^30*x4^14*z^26 - 2*x1^47*x2^39*x3^30*x4^14*z^26 + 2*x1^44*x2^42*x3^30*x4^14*z^26 + x1^43*x2^43*x3^30*x4^14*z^26 - x1^50*x2^35*x3^31*x4^14*z^26 + x1^48*x2^37*x3^31*x4^14*z^26 + 2*x1^47*x2^38*x3^31*x4^14*z^26 - x1^46*x2^39*x3^31*x4^14*z^26 + x1^45*x2^40*x3^31*x4^14*z^26 - 2*x1^43*x2^42*x3^31*x4^14*z^26 - x1^47*x2^37*x3^32*x4^14*z^26 - x1^46*x2^38*x3^32*x4^14*z^26 - x1^45*x2^39*x3^32*x4^14*z^26 + 2*x1^43*x2^41*x3^32*x4^14*z^26 + x1^42*x2^42*x3^32*x4^14*z^26 - x1^47*x2^36*x3^33*x4^14*z^26 + x1^46*x2^37*x3^33*x4^14*z^26 + x1^45*x2^38*x3^33*x4^14*z^26 - x1^44*x2^39*x3^33*x4^14*z^26 - x1^56*x2^40*x3^19*x4^15*z^26 + 2*x1^55*x2^40*x3^20*x4^15*z^26 - 2*x1^55*x2^39*x3^21*x4^15*z^26 - x1^54*x2^40*x3^21*x4^15*z^26 + x1^52*x2^42*x3^21*x4^15*z^26 - x1^51*x2^43*x3^21*x4^15*z^26 + 5*x1^54*x2^39*x3^22*x4^15*z^26 + x1^53*x2^40*x3^22*x4^15*z^26 + x1^52*x2^41*x3^22*x4^15*z^26 - 2*x1^51*x2^42*x3^22*x4^15*z^26 - x1^49*x2^44*x3^22*x4^15*z^26 - 5*x1^54*x2^38*x3^23*x4^15*z^26 - x1^53*x2^39*x3^23*x4^15*z^26 - x1^52*x2^40*x3^23*x4^15*z^26 + x1^50*x2^42*x3^23*x4^15*z^26 + x1^49*x2^43*x3^23*x4^15*z^26 + x1^48*x2^44*x3^23*x4^15*z^26 + 2*x1^54*x2^37*x3^24*x4^15*z^26 + 6*x1^53*x2^38*x3^24*x4^15*z^26 + 4*x1^51*x2^40*x3^24*x4^15*z^26 - 4*x1^50*x2^41*x3^24*x4^15*z^26 - x1^49*x2^42*x3^24*x4^15*z^26 - 4*x1^48*x2^43*x3^24*x4^15*z^26 - x1^47*x2^44*x3^24*x4^15*z^26 - 6*x1^53*x2^37*x3^25*x4^15*z^26 - 2*x1^52*x2^38*x3^25*x4^15*z^26 - 2*x1^51*x2^39*x3^25*x4^15*z^26 + 2*x1^49*x2^41*x3^25*x4^15*z^26 + 2*x1^48*x2^42*x3^25*x4^15*z^26 + 4*x1^47*x2^43*x3^25*x4^15*z^26 + 2*x1^53*x2^36*x3^26*x4^15*z^26 + 6*x1^52*x2^37*x3^26*x4^15*z^26 + 4*x1^50*x2^39*x3^26*x4^15*z^26 - 4*x1^49*x2^40*x3^26*x4^15*z^26 - 6*x1^47*x2^42*x3^26*x4^15*z^26 - 2*x1^46*x2^43*x3^26*x4^15*z^26 - 6*x1^52*x2^36*x3^27*x4^15*z^26 - 2*x1^51*x2^37*x3^27*x4^15*z^26 - 2*x1^50*x2^38*x3^27*x4^15*z^26 + 2*x1^48*x2^40*x3^27*x4^15*z^26 + 2*x1^47*x2^41*x3^27*x4^15*z^26 + 6*x1^46*x2^42*x3^27*x4^15*z^26 + x1^52*x2^35*x3^28*x4^15*z^26 + 6*x1^51*x2^36*x3^28*x4^15*z^26 + x1^50*x2^37*x3^28*x4^15*z^26 + 4*x1^49*x2^38*x3^28*x4^15*z^26 - 4*x1^48*x2^39*x3^28*x4^15*z^26 - 6*x1^46*x2^41*x3^28*x4^15*z^26 - 2*x1^45*x2^42*x3^28*x4^15*z^26 - 3*x1^51*x2^35*x3^29*x4^15*z^26 - 3*x1^50*x2^36*x3^29*x4^15*z^26 - 3*x1^49*x2^37*x3^29*x4^15*z^26 + 2*x1^47*x2^39*x3^29*x4^15*z^26 + 2*x1^46*x2^40*x3^29*x4^15*z^26 + 6*x1^45*x2^41*x3^29*x4^15*z^26 + 3*x1^50*x2^35*x3^30*x4^15*z^26 + 2*x1^49*x2^36*x3^30*x4^15*z^26 + 3*x1^48*x2^37*x3^30*x4^15*z^26 - 4*x1^47*x2^38*x3^30*x4^15*z^26 - 6*x1^45*x2^40*x3^30*x4^15*z^26 - 2*x1^44*x2^41*x3^30*x4^15*z^26 - x1^49*x2^35*x3^31*x4^15*z^26 - 2*x1^48*x2^36*x3^31*x4^15*z^26 + x1^46*x2^38*x3^31*x4^15*z^26 + 2*x1^45*x2^39*x3^31*x4^15*z^26 + 4*x1^44*x2^40*x3^31*x4^15*z^26 + x1^47*x2^36*x3^32*x4^15*z^26 - x1^46*x2^37*x3^32*x4^15*z^26 + 2*x1^45*x2^38*x3^32*x4^15*z^26 - 4*x1^44*x2^39*x3^32*x4^15*z^26 - x1^43*x2^40*x3^32*x4^15*z^26 - x1^45*x2^37*x3^33*x4^15*z^26 + x1^44*x2^38*x3^33*x4^15*z^26 + 2*x1^43*x2^39*x3^33*x4^15*z^26 + x1^44*x2^37*x3^34*x4^15*z^26 - x1^43*x2^38*x3^34*x4^15*z^26 - 2*x1^52*x2^42*x3^20*x4^16*z^26 - x1^51*x2^43*x3^20*x4^16*z^26 - 2*x1^54*x2^39*x3^21*x4^16*z^26 + x1^52*x2^41*x3^21*x4^16*z^26 + 2*x1^51*x2^42*x3^21*x4^16*z^26 + 2*x1^50*x2^43*x3^21*x4^16*z^26 + x1^49*x2^44*x3^21*x4^16*z^26 + x1^54*x2^38*x3^22*x4^16*z^26 + x1^53*x2^39*x3^22*x4^16*z^26 + x1^52*x2^40*x3^22*x4^16*z^26 - 3*x1^51*x2^41*x3^22*x4^16*z^26 - 3*x1^50*x2^42*x3^22*x4^16*z^26 - 2*x1^49*x2^43*x3^22*x4^16*z^26 - x1^48*x2^44*x3^22*x4^16*z^26 - 4*x1^53*x2^38*x3^23*x4^16*z^26 + 4*x1^50*x2^41*x3^23*x4^16*z^26 + 2*x1^49*x2^42*x3^23*x4^16*z^26 + 5*x1^48*x2^43*x3^23*x4^16*z^26 + 3*x1^53*x2^37*x3^24*x4^16*z^26 + x1^52*x2^38*x3^24*x4^16*z^26 + x1^51*x2^39*x3^24*x4^16*z^26 - x1^50*x2^40*x3^24*x4^16*z^26 - 2*x1^49*x2^41*x3^24*x4^16*z^26 - 3*x1^48*x2^42*x3^24*x4^16*z^26 - 5*x1^47*x2^43*x3^24*x4^16*z^26 - x1^53*x2^36*x3^25*x4^16*z^26 - 6*x1^52*x2^37*x3^25*x4^16*z^26 - 2*x1^50*x2^39*x3^25*x4^16*z^26 + 4*x1^49*x2^40*x3^25*x4^16*z^26 + 6*x1^47*x2^42*x3^25*x4^16*z^26 + 2*x1^46*x2^43*x3^25*x4^16*z^26 + 6*x1^52*x2^36*x3^26*x4^16*z^26 + 2*x1^51*x2^37*x3^26*x4^16*z^26 + 2*x1^50*x2^38*x3^26*x4^16*z^26 - 2*x1^48*x2^40*x3^26*x4^16*z^26 - 2*x1^47*x2^41*x3^26*x4^16*z^26 - 6*x1^46*x2^42*x3^26*x4^16*z^26 - 6*x1^51*x2^36*x3^27*x4^16*z^26 - 4*x1^49*x2^38*x3^27*x4^16*z^26 + 4*x1^48*x2^39*x3^27*x4^16*z^26 + 6*x1^46*x2^41*x3^27*x4^16*z^26 + 2*x1^45*x2^42*x3^27*x4^16*z^26 + 2*x1^51*x2^35*x3^28*x4^16*z^26 + 2*x1^50*x2^36*x3^28*x4^16*z^26 + x1^49*x2^37*x3^28*x4^16*z^26 - 2*x1^47*x2^39*x3^28*x4^16*z^26 - 2*x1^46*x2^40*x3^28*x4^16*z^26 - 6*x1^45*x2^41*x3^28*x4^16*z^26 - 2*x1^50*x2^35*x3^29*x4^16*z^26 - x1^49*x2^36*x3^29*x4^16*z^26 - 3*x1^48*x2^37*x3^29*x4^16*z^26 + 2*x1^47*x2^38*x3^29*x4^16*z^26 + 6*x1^45*x2^40*x3^29*x4^16*z^26 + 2*x1^44*x2^41*x3^29*x4^16*z^26 + x1^48*x2^36*x3^30*x4^16*z^26 - 2*x1^45*x2^39*x3^30*x4^16*z^26 - 6*x1^44*x2^40*x3^30*x4^16*z^26 + 2*x1^46*x2^37*x3^31*x4^16*z^26 - 2*x1^45*x2^38*x3^31*x4^16*z^26 + 4*x1^44*x2^39*x3^31*x4^16*z^26 + 2*x1^43*x2^40*x3^31*x4^16*z^26 - x1^46*x2^36*x3^32*x4^16*z^26 + x1^44*x2^38*x3^32*x4^16*z^26 - 4*x1^43*x2^39*x3^32*x4^16*z^26 + x1^45*x2^36*x3^33*x4^16*z^26 + x1^43*x2^38*x3^33*x4^16*z^26 + x1^42*x2^39*x3^33*x4^16*z^26 - x1^42*x2^38*x3^34*x4^16*z^26 + x1^51*x2^41*x3^21*x4^17*z^26 - x1^50*x2^41*x3^22*x4^17*z^26 - x1^49*x2^42*x3^22*x4^17*z^26 - x1^48*x2^43*x3^22*x4^17*z^26 + x1^50*x2^40*x3^23*x4^17*z^26 + x1^49*x2^41*x3^23*x4^17*z^26 - x1^48*x2^42*x3^23*x4^17*z^26 + x1^47*x2^43*x3^23*x4^17*z^26 + x1^52*x2^37*x3^24*x4^17*z^26 - x1^51*x2^38*x3^24*x4^17*z^26 + x1^50*x2^39*x3^24*x4^17*z^26 - x1^49*x2^40*x3^24*x4^17*z^26 - 2*x1^47*x2^42*x3^24*x4^17*z^26 - x1^46*x2^43*x3^24*x4^17*z^26 - x1^51*x2^37*x3^25*x4^17*z^26 - x1^50*x2^38*x3^25*x4^17*z^26 + x1^49*x2^39*x3^25*x4^17*z^26 + x1^48*x2^40*x3^25*x4^17*z^26 + x1^47*x2^41*x3^25*x4^17*z^26 + 2*x1^46*x2^42*x3^25*x4^17*z^26 + 2*x1^51*x2^36*x3^26*x4^17*z^26 - x1^49*x2^38*x3^26*x4^17*z^26 - x1^48*x2^39*x3^26*x4^17*z^26 - 2*x1^46*x2^41*x3^26*x4^17*z^26 - x1^45*x2^42*x3^26*x4^17*z^26 - x1^51*x2^35*x3^27*x4^17*z^26 - x1^50*x2^36*x3^27*x4^17*z^26 - x1^49*x2^37*x3^27*x4^17*z^26 + x1^47*x2^39*x3^27*x4^17*z^26 + x1^46*x2^40*x3^27*x4^17*z^26 + 2*x1^45*x2^41*x3^27*x4^17*z^26 + x1^50*x2^35*x3^28*x4^17*z^26 - 2*x1^47*x2^38*x3^28*x4^17*z^26 - 2*x1^45*x2^40*x3^28*x4^17*z^26 + x1^47*x2^37*x3^29*x4^17*z^26 + 2*x1^44*x2^40*x3^29*x4^17*z^26 + x1^47*x2^36*x3^30*x4^17*z^26 - x1^46*x2^37*x3^30*x4^17*z^26 - 2*x1^44*x2^39*x3^30*x4^17*z^26 - x1^43*x2^40*x3^30*x4^17*z^26 + x1^45*x2^37*x3^31*x4^17*z^26 + x1^44*x2^38*x3^31*x4^17*z^26 + 2*x1^43*x2^39*x3^31*x4^17*z^26 - x1^42*x2^39*x3^32*x4^17*z^26 + x1^57*x2^43*x3^23*x4^2*z^25 - 2*x1^56*x2^43*x3^24*x4^2*z^25 + x1^56*x2^42*x3^25*x4^2*z^25 - x1^57*x2^43*x3^22*x4^3*z^25 - x1^58*x2^41*x3^23*x4^3*z^25 + 2*x1^56*x2^43*x3^23*x4^3*z^25 - x1^55*x2^44*x3^23*x4^3*z^25 - x1^56*x2^42*x3^24*x4^3*z^25 + 2*x1^55*x2^42*x3^25*x4^3*z^25 - x1^55*x2^41*x3^26*x4^3*z^25 - x1^52*x2^44*x3^26*x4^3*z^25 + x1^58*x2^41*x3^22*x4^4*z^25 - 2*x1^57*x2^41*x3^23*x4^4*z^25 + x1^56*x2^42*x3^23*x4^4*z^25 - x1^54*x2^44*x3^23*x4^4*z^25 + 2*x1^57*x2^40*x3^24*x4^4*z^25 + x1^56*x2^41*x3^24*x4^4*z^25 - x1^55*x2^42*x3^24*x4^4*z^25 + 3*x1^54*x2^43*x3^24*x4^4*z^25 + x1^53*x2^44*x3^24*x4^4*z^25 - x1^52*x2^45*x3^24*x4^4*z^25 - 3*x1^56*x2^40*x3^25*x4^4*z^25 + x1^55*x2^41*x3^25*x4^4*z^25 - x1^53*x2^43*x3^25*x4^4*z^25 + x1^52*x2^44*x3^25*x4^4*z^25 + x1^55*x2^40*x3^26*x4^4*z^25 - x1^54*x2^41*x3^26*x4^4*z^25 + 3*x1^53*x2^42*x3^26*x4^4*z^25 + x1^52*x2^43*x3^26*x4^4*z^25 + x1^53*x2^41*x3^27*x4^4*z^25 + x1^52*x2^42*x3^27*x4^4*z^25 + x1^51*x2^43*x3^27*x4^4*z^25 - 2*x1^58*x2^41*x3^21*x4^5*z^25 + 4*x1^57*x2^41*x3^22*x4^5*z^25 - x1^56*x2^42*x3^22*x4^5*z^25 + x1^55*x2^43*x3^22*x4^5*z^25 - 2*x1^57*x2^40*x3^23*x4^5*z^25 - 4*x1^56*x2^41*x3^23*x4^5*z^25 + x1^55*x2^42*x3^23*x4^5*z^25 - 2*x1^54*x2^43*x3^23*x4^5*z^25 + 2*x1^53*x2^44*x3^23*x4^5*z^25 + 5*x1^56*x2^40*x3^24*x4^5*z^25 + x1^55*x2^41*x3^24*x4^5*z^25 - x1^54*x2^42*x3^24*x4^5*z^25 + 2*x1^53*x2^43*x3^24*x4^5*z^25 - 2*x1^56*x2^39*x3^25*x4^5*z^25 - 4*x1^55*x2^40*x3^25*x4^5*z^25 + 3*x1^54*x2^41*x3^25*x4^5*z^25 - 2*x1^53*x2^42*x3^25*x4^5*z^25 + x1^51*x2^44*x3^25*x4^5*z^25 + 4*x1^55*x2^39*x3^26*x4^5*z^25 + 2*x1^52*x2^42*x3^26*x4^5*z^25 - x1^55*x2^38*x3^27*x4^5*z^25 - 2*x1^54*x2^39*x3^27*x4^5*z^25 + x1^53*x2^40*x3^27*x4^5*z^25 - 3*x1^52*x2^41*x3^27*x4^5*z^25 + x1^51*x2^42*x3^27*x4^5*z^25 + x1^54*x2^38*x3^28*x4^5*z^25 - x1^53*x2^39*x3^28*x4^5*z^25 - x1^52*x2^40*x3^28*x4^5*z^25 + x1^51*x2^41*x3^28*x4^5*z^25 + x1^52*x2^39*x3^29*x4^5*z^25 - x1^51*x2^40*x3^29*x4^5*z^25 + x1^50*x2^41*x3^29*x4^5*z^25 + x1^49*x2^42*x3^29*x4^5*z^25 + x1^58*x2^41*x3^20*x4^6*z^25 - x1^57*x2^41*x3^21*x4^6*z^25 + x1^56*x2^42*x3^21*x4^6*z^25 + x1^56*x2^41*x3^22*x4^6*z^25 - x1^56*x2^40*x3^23*x4^6*z^25 + x1^55*x2^41*x3^23*x4^6*z^25 + x1^53*x2^43*x3^23*x4^6*z^25 + x1^52*x2^44*x3^23*x4^6*z^25 + x1^56*x2^39*x3^24*x4^6*z^25 + 2*x1^55*x2^40*x3^24*x4^6*z^25 - 4*x1^54*x2^41*x3^24*x4^6*z^25 + 3*x1^53*x2^42*x3^24*x4^6*z^25 - 2*x1^51*x2^44*x3^24*x4^6*z^25 - x1^55*x2^39*x3^25*x4^6*z^25 - x1^54*x2^40*x3^25*x4^6*z^25 + 2*x1^53*x2^41*x3^25*x4^6*z^25 + 2*x1^50*x2^44*x3^25*x4^6*z^25 + x1^49*x2^45*x3^25*x4^6*z^25 + x1^55*x2^38*x3^26*x4^6*z^25 + 3*x1^54*x2^39*x3^26*x4^6*z^25 - 3*x1^53*x2^40*x3^26*x4^6*z^25 - 2*x1^52*x2^41*x3^26*x4^6*z^25 - x1^51*x2^42*x3^26*x4^6*z^25 + 2*x1^50*x2^43*x3^26*x4^6*z^25 - 2*x1^49*x2^44*x3^26*x4^6*z^25 - x1^48*x2^45*x3^26*x4^6*z^25 - 2*x1^54*x2^38*x3^27*x4^6*z^25 + x1^53*x2^39*x3^27*x4^6*z^25 + x1^52*x2^40*x3^27*x4^6*z^25 - 2*x1^51*x2^41*x3^27*x4^6*z^25 + x1^50*x2^42*x3^27*x4^6*z^25 + x1^49*x2^43*x3^27*x4^6*z^25 - x1^47*x2^45*x3^27*x4^6*z^25 + x1^54*x2^37*x3^28*x4^6*z^25 + x1^53*x2^38*x3^28*x4^6*z^25 - x1^52*x2^39*x3^28*x4^6*z^25 + x1^51*x2^40*x3^28*x4^6*z^25 - 2*x1^50*x2^41*x3^28*x4^6*z^25 - x1^53*x2^37*x3^29*x4^6*z^25 + x1^52*x2^38*x3^29*x4^6*z^25 - 2*x1^50*x2^40*x3^29*x4^6*z^25 - x1^49*x2^41*x3^29*x4^6*z^25 - x1^51*x2^38*x3^30*x4^6*z^25 + x1^50*x2^39*x3^30*x4^6*z^25 - x1^49*x2^40*x3^30*x4^6*z^25 - x1^48*x2^41*x3^30*x4^6*z^25 - x1^56*x2^43*x3^19*x4^7*z^25 + x1^56*x2^42*x3^20*x4^7*z^25 + x1^55*x2^43*x3^20*x4^7*z^25 - x1^54*x2^44*x3^20*x4^7*z^25 - x1^57*x2^40*x3^21*x4^7*z^25 - 2*x1^55*x2^42*x3^21*x4^7*z^25 - x1^54*x2^43*x3^21*x4^7*z^25 + x1^56*x2^40*x3^22*x4^7*z^25 + x1^55*x2^41*x3^22*x4^7*z^25 + 2*x1^54*x2^42*x3^22*x4^7*z^25 + x1^53*x2^43*x3^22*x4^7*z^25 + x1^52*x2^44*x3^22*x4^7*z^25 - x1^51*x2^45*x3^22*x4^7*z^25 - x1^55*x2^40*x3^23*x4^7*z^25 - x1^54*x2^41*x3^23*x4^7*z^25 + x1^52*x2^43*x3^23*x4^7*z^25 - x1^51*x2^44*x3^23*x4^7*z^25 + x1^49*x2^46*x3^23*x4^7*z^25 + x1^54*x2^40*x3^24*x4^7*z^25 + x1^53*x2^41*x3^24*x4^7*z^25 + 3*x1^51*x2^43*x3^24*x4^7*z^25 + 2*x1^50*x2^44*x3^24*x4^7*z^25 - x1^48*x2^46*x3^24*x4^7*z^25 - x1^54*x2^39*x3^25*x4^7*z^25 + x1^53*x2^40*x3^25*x4^7*z^25 - 2*x1^52*x2^41*x3^25*x4^7*z^25 + x1^50*x2^43*x3^25*x4^7*z^25 + x1^49*x2^44*x3^25*x4^7*z^25 + 2*x1^53*x2^39*x3^26*x4^7*z^25 - x1^51*x2^41*x3^26*x4^7*z^25 + x1^50*x2^42*x3^26*x4^7*z^25 - x1^49*x2^43*x3^26*x4^7*z^25 - x1^48*x2^44*x3^26*x4^7*z^25 - x1^53*x2^38*x3^27*x4^7*z^25 + x1^50*x2^41*x3^27*x4^7*z^25 - 2*x1^49*x2^42*x3^27*x4^7*z^25 + 2*x1^48*x2^43*x3^27*x4^7*z^25 + x1^47*x2^44*x3^27*x4^7*z^25 - x1^46*x2^45*x3^27*x4^7*z^25 + x1^53*x2^37*x3^28*x4^7*z^25 + x1^52*x2^38*x3^28*x4^7*z^25 + x1^51*x2^39*x3^28*x4^7*z^25 - 2*x1^53*x2^36*x3^29*x4^7*z^25 + x1^51*x2^38*x3^29*x4^7*z^25 - x1^50*x2^39*x3^29*x4^7*z^25 + x1^49*x2^40*x3^29*x4^7*z^25 - x1^48*x2^41*x3^29*x4^7*z^25 + x1^52*x2^36*x3^30*x4^7*z^25 + x1^50*x2^38*x3^30*x4^7*z^25 + 2*x1^49*x2^39*x3^30*x4^7*z^25 + x1^50*x2^37*x3^31*x4^7*z^25 - x1^56*x2^42*x3^19*x4^8*z^25 + 4*x1^55*x2^42*x3^20*x4^8*z^25 + x1^54*x2^43*x3^20*x4^8*z^25 + 2*x1^53*x2^44*x3^20*x4^8*z^25 - 3*x1^55*x2^41*x3^21*x4^8*z^25 - 5*x1^54*x2^42*x3^21*x4^8*z^25 + x1^53*x2^43*x3^21*x4^8*z^25 - 2*x1^52*x2^44*x3^21*x4^8*z^25 + x1^51*x2^45*x3^21*x4^8*z^25 + x1^56*x2^39*x3^22*x4^8*z^25 + 5*x1^54*x2^41*x3^22*x4^8*z^25 + x1^53*x2^42*x3^22*x4^8*z^25 + x1^51*x2^44*x3^22*x4^8*z^25 - 2*x1^55*x2^39*x3^23*x4^8*z^25 - 5*x1^53*x2^41*x3^23*x4^8*z^25 - 2*x1^52*x2^42*x3^23*x4^8*z^25 - 5*x1^51*x2^43*x3^23*x4^8*z^25 + x1^50*x2^44*x3^23*x4^8*z^25 + x1^48*x2^46*x3^23*x4^8*z^25 + 2*x1^54*x2^39*x3^24*x4^8*z^25 + 3*x1^53*x2^40*x3^24*x4^8*z^25 + 4*x1^52*x2^41*x3^24*x4^8*z^25 + x1^51*x2^42*x3^24*x4^8*z^25 + x1^50*x2^43*x3^24*x4^8*z^25 - x1^49*x2^44*x3^24*x4^8*z^25 - 2*x1^48*x2^45*x3^24*x4^8*z^25 - x1^47*x2^46*x3^24*x4^8*z^25 - 2*x1^54*x2^38*x3^25*x4^8*z^25 - 3*x1^53*x2^39*x3^25*x4^8*z^25 - 4*x1^52*x2^40*x3^25*x4^8*z^25 - 3*x1^50*x2^42*x3^25*x4^8*z^25 + x1^48*x2^44*x3^25*x4^8*z^25 + 3*x1^47*x2^45*x3^25*x4^8*z^25 + x1^54*x2^37*x3^26*x4^8*z^25 + 3*x1^52*x2^39*x3^26*x4^8*z^25 + 2*x1^51*x2^40*x3^26*x4^8*z^25 + x1^50*x2^41*x3^26*x4^8*z^25 + x1^49*x2^42*x3^26*x4^8*z^25 - x1^47*x2^44*x3^26*x4^8*z^25 - x1^46*x2^45*x3^26*x4^8*z^25 - 2*x1^53*x2^37*x3^27*x4^8*z^25 - 3*x1^52*x2^38*x3^27*x4^8*z^25 - 5*x1^51*x2^39*x3^27*x4^8*z^25 - x1^50*x2^40*x3^27*x4^8*z^25 - 2*x1^49*x2^41*x3^27*x4^8*z^25 + x1^48*x2^42*x3^27*x4^8*z^25 + 2*x1^46*x2^44*x3^27*x4^8*z^25 + x1^53*x2^36*x3^28*x4^8*z^25 + x1^52*x2^37*x3^28*x4^8*z^25 + 3*x1^51*x2^38*x3^28*x4^8*z^25 + 2*x1^50*x2^39*x3^28*x4^8*z^25 - x1^49*x2^40*x3^28*x4^8*z^25 + 3*x1^48*x2^41*x3^28*x4^8*z^25 - x1^47*x2^42*x3^28*x4^8*z^25 - x1^46*x2^43*x3^28*x4^8*z^25 - x1^52*x2^36*x3^29*x4^8*z^25 - 2*x1^50*x2^38*x3^29*x4^8*z^25 - x1^49*x2^39*x3^29*x4^8*z^25 - x1^48*x2^40*x3^29*x4^8*z^25 + x1^46*x2^42*x3^29*x4^8*z^25 + 2*x1^45*x2^43*x3^29*x4^8*z^25 + x1^52*x2^35*x3^30*x4^8*z^25 - x1^51*x2^36*x3^30*x4^8*z^25 + x1^50*x2^37*x3^30*x4^8*z^25 + x1^49*x2^38*x3^30*x4^8*z^25 + x1^48*x2^39*x3^30*x4^8*z^25 + 2*x1^47*x2^40*x3^30*x4^8*z^25 - x1^51*x2^35*x3^31*x4^8*z^25 - x1^50*x2^36*x3^31*x4^8*z^25 - x1^49*x2^37*x3^31*x4^8*z^25 - x1^48*x2^38*x3^31*x4^8*z^25 - x1^47*x2^39*x3^31*x4^8*z^25 + x1^47*x2^38*x3^32*x4^8*z^25 + x1^46*x2^39*x3^32*x4^8*z^25 + x1^57*x2^40*x3^19*x4^9*z^25 - x1^55*x2^42*x3^19*x4^9*z^25 + x1^54*x2^43*x3^19*x4^9*z^25 - x1^56*x2^40*x3^20*x4^9*z^25 + x1^55*x2^41*x3^20*x4^9*z^25 + x1^54*x2^42*x3^20*x4^9*z^25 - 2*x1^53*x2^43*x3^20*x4^9*z^25 - 2*x1^56*x2^39*x3^21*x4^9*z^25 + x1^55*x2^40*x3^21*x4^9*z^25 - x1^54*x2^41*x3^21*x4^9*z^25 + x1^53*x2^42*x3^21*x4^9*z^25 + x1^52*x2^43*x3^21*x4^9*z^25 + x1^51*x2^44*x3^21*x4^9*z^25 + x1^50*x2^45*x3^21*x4^9*z^25 + 3*x1^55*x2^39*x3^22*x4^9*z^25 - 2*x1^54*x2^40*x3^22*x4^9*z^25 + x1^53*x2^41*x3^22*x4^9*z^25 - x1^52*x2^42*x3^22*x4^9*z^25 - 2*x1^50*x2^44*x3^22*x4^9*z^25 - x1^49*x2^45*x3^22*x4^9*z^25 - x1^55*x2^38*x3^23*x4^9*z^25 - 3*x1^54*x2^39*x3^23*x4^9*z^25 - x1^52*x2^41*x3^23*x4^9*z^25 + x1^51*x2^42*x3^23*x4^9*z^25 + x1^50*x2^43*x3^23*x4^9*z^25 + x1^49*x2^44*x3^23*x4^9*z^25 + x1^48*x2^45*x3^23*x4^9*z^25 + 4*x1^54*x2^38*x3^24*x4^9*z^25 + 2*x1^53*x2^39*x3^24*x4^9*z^25 + x1^52*x2^40*x3^24*x4^9*z^25 + x1^50*x2^42*x3^24*x4^9*z^25 - x1^49*x2^43*x3^24*x4^9*z^25 - x1^48*x2^44*x3^24*x4^9*z^25 - 2*x1^47*x2^45*x3^24*x4^9*z^25 - x1^54*x2^37*x3^25*x4^9*z^25 - 4*x1^53*x2^38*x3^25*x4^9*z^25 - 2*x1^52*x2^39*x3^25*x4^9*z^25 - 3*x1^51*x2^40*x3^25*x4^9*z^25 - x1^50*x2^41*x3^25*x4^9*z^25 + x1^49*x2^42*x3^25*x4^9*z^25 + x1^47*x2^44*x3^25*x4^9*z^25 + x1^46*x2^45*x3^25*x4^9*z^25 + 4*x1^53*x2^37*x3^26*x4^9*z^25 + 3*x1^52*x2^38*x3^26*x4^9*z^25 + 4*x1^51*x2^39*x3^26*x4^9*z^25 + 2*x1^50*x2^40*x3^26*x4^9*z^25 + x1^49*x2^41*x3^26*x4^9*z^25 - 4*x1^48*x2^42*x3^26*x4^9*z^25 + x1^47*x2^43*x3^26*x4^9*z^25 - 3*x1^46*x2^44*x3^26*x4^9*z^25 - 2*x1^53*x2^36*x3^27*x4^9*z^25 - 2*x1^52*x2^37*x3^27*x4^9*z^25 - x1^51*x2^38*x3^27*x4^9*z^25 - 2*x1^50*x2^39*x3^27*x4^9*z^25 + 2*x1^49*x2^40*x3^27*x4^9*z^25 - x1^48*x2^41*x3^27*x4^9*z^25 + 2*x1^47*x2^42*x3^27*x4^9*z^25 + x1^45*x2^44*x3^27*x4^9*z^25 + 4*x1^52*x2^36*x3^28*x4^9*z^25 - x1^51*x2^37*x3^28*x4^9*z^25 + 2*x1^50*x2^38*x3^28*x4^9*z^25 + 3*x1^49*x2^39*x3^28*x4^9*z^25 - x1^46*x2^42*x3^28*x4^9*z^25 - x1^45*x2^43*x3^28*x4^9*z^25 - x1^52*x2^35*x3^29*x4^9*z^25 - 2*x1^51*x2^36*x3^29*x4^9*z^25 - 4*x1^49*x2^38*x3^29*x4^9*z^25 - x1^47*x2^40*x3^29*x4^9*z^25 - x1^45*x2^42*x3^29*x4^9*z^25 + 2*x1^51*x2^35*x3^30*x4^9*z^25 + x1^50*x2^36*x3^30*x4^9*z^25 + x1^49*x2^37*x3^30*x4^9*z^25 + x1^48*x2^38*x3^30*x4^9*z^25 - x1^45*x2^41*x3^30*x4^9*z^25 - 2*x1^44*x2^42*x3^30*x4^9*z^25 + x1^49*x2^36*x3^31*x4^9*z^25 - 2*x1^48*x2^37*x3^31*x4^9*z^25 - x1^47*x2^38*x3^31*x4^9*z^25 - x1^46*x2^38*x3^32*x4^9*z^25 + x1^47*x2^36*x3^33*x4^9*z^25 - x1^45*x2^38*x3^33*x4^9*z^25 + x1^57*x2^41*x3^17*x4^10*z^25 - x1^57*x2^40*x3^18*x4^10*z^25 - x1^56*x2^41*x3^18*x4^10*z^25 + 2*x1^55*x2^42*x3^18*x4^10*z^25 + x1^56*x2^40*x3^19*x4^10*z^25 + x1^55*x2^41*x3^19*x4^10*z^25 - x1^53*x2^43*x3^19*x4^10*z^25 - x1^56*x2^39*x3^20*x4^10*z^25 - x1^55*x2^40*x3^20*x4^10*z^25 - x1^53*x2^42*x3^20*x4^10*z^25 + x1^52*x2^43*x3^20*x4^10*z^25 + x1^55*x2^39*x3^21*x4^10*z^25 - x1^54*x2^40*x3^21*x4^10*z^25 - x1^52*x2^42*x3^21*x4^10*z^25 - x1^51*x2^43*x3^21*x4^10*z^25 - 2*x1^50*x2^44*x3^21*x4^10*z^25 - x1^54*x2^39*x3^22*x4^10*z^25 - 2*x1^52*x2^41*x3^22*x4^10*z^25 + 2*x1^51*x2^42*x3^22*x4^10*z^25 + 2*x1^49*x2^44*x3^22*x4^10*z^25 - x1^48*x2^45*x3^22*x4^10*z^25 + x1^53*x2^39*x3^23*x4^10*z^25 - x1^50*x2^42*x3^23*x4^10*z^25 - 2*x1^48*x2^44*x3^23*x4^10*z^25 + x1^51*x2^40*x3^24*x4^10*z^25 + 2*x1^50*x2^41*x3^24*x4^10*z^25 + 3*x1^48*x2^43*x3^24*x4^10*z^25 + 2*x1^47*x2^44*x3^24*x4^10*z^25 - x1^49*x2^41*x3^25*x4^10*z^25 - x1^47*x2^43*x3^25*x4^10*z^25 - x1^46*x2^44*x3^25*x4^10*z^25 + x1^45*x2^45*x3^25*x4^10*z^25 + 2*x1^48*x2^41*x3^26*x4^10*z^25 + 2*x1^47*x2^42*x3^26*x4^10*z^25 + 2*x1^45*x2^44*x3^26*x4^10*z^25 - x1^49*x2^39*x3^27*x4^10*z^25 + x1^48*x2^40*x3^27*x4^10*z^25 - x1^47*x2^41*x3^27*x4^10*z^25 - 2*x1^44*x2^44*x3^27*x4^10*z^25 + x1^52*x2^35*x3^28*x4^10*z^25 + x1^46*x2^41*x3^28*x4^10*z^25 + x1^45*x2^42*x3^28*x4^10*z^25 + x1^44*x2^43*x3^28*x4^10*z^25 - 3*x1^51*x2^35*x3^29*x4^10*z^25 + 2*x1^50*x2^36*x3^29*x4^10*z^25 + 2*x1^49*x2^37*x3^29*x4^10*z^25 - 2*x1^48*x2^38*x3^29*x4^10*z^25 + x1^47*x2^39*x3^29*x4^10*z^25 - x1^44*x2^42*x3^29*x4^10*z^25 + x1^51*x2^34*x3^30*x4^10*z^25 + 2*x1^50*x2^35*x3^30*x4^10*z^25 - x1^49*x2^36*x3^30*x4^10*z^25 + 2*x1^48*x2^37*x3^30*x4^10*z^25 + 2*x1^47*x2^38*x3^30*x4^10*z^25 - x1^46*x2^39*x3^30*x4^10*z^25 + 2*x1^44*x2^41*x3^30*x4^10*z^25 - 2*x1^50*x2^34*x3^31*x4^10*z^25 - 2*x1^49*x2^35*x3^31*x4^10*z^25 + x1^46*x2^38*x3^31*x4^10*z^25 - x1^45*x2^39*x3^31*x4^10*z^25 + x1^44*x2^40*x3^31*x4^10*z^25 - x1^43*x2^41*x3^31*x4^10*z^25 + x1^49*x2^34*x3^32*x4^10*z^25 + x1^48*x2^35*x3^32*x4^10*z^25 + 2*x1^47*x2^36*x3^32*x4^10*z^25 + x1^46*x2^37*x3^32*x4^10*z^25 - x1^45*x2^38*x3^32*x4^10*z^25 - x1^47*x2^35*x3^33*x4^10*z^25 - x1^46*x2^36*x3^33*x4^10*z^25 + x1^45*x2^37*x3^33*x4^10*z^25 - x1^56*x2^41*x3^17*x4^11*z^25 - x1^56*x2^40*x3^18*x4^11*z^25 + 2*x1^56*x2^39*x3^19*x4^11*z^25 + 3*x1^55*x2^40*x3^19*x4^11*z^25 - 2*x1^54*x2^41*x3^19*x4^11*z^25 - x1^52*x2^43*x3^19*x4^11*z^25 - 3*x1^55*x2^39*x3^20*x4^11*z^25 + 2*x1^53*x2^41*x3^20*x4^11*z^25 + 2*x1^51*x2^43*x3^20*x4^11*z^25 + x1^50*x2^44*x3^20*x4^11*z^25 + x1^55*x2^38*x3^21*x4^11*z^25 + 4*x1^54*x2^39*x3^21*x4^11*z^25 - 2*x1^53*x2^40*x3^21*x4^11*z^25 + x1^52*x2^41*x3^21*x4^11*z^25 - 3*x1^51*x2^42*x3^21*x4^11*z^25 - x1^50*x2^43*x3^21*x4^11*z^25 - x1^49*x2^44*x3^21*x4^11*z^25 - 4*x1^54*x2^38*x3^22*x4^11*z^25 + x1^52*x2^40*x3^22*x4^11*z^25 + 2*x1^50*x2^42*x3^22*x4^11*z^25 + x1^49*x2^43*x3^22*x4^11*z^25 + x1^48*x2^44*x3^22*x4^11*z^25 + x1^47*x2^45*x3^22*x4^11*z^25 + x1^54*x2^37*x3^23*x4^11*z^25 + 4*x1^53*x2^38*x3^23*x4^11*z^25 - 2*x1^52*x2^39*x3^23*x4^11*z^25 + 3*x1^51*x2^40*x3^23*x4^11*z^25 - 2*x1^50*x2^41*x3^23*x4^11*z^25 - x1^49*x2^42*x3^23*x4^11*z^25 - 4*x1^48*x2^43*x3^23*x4^11*z^25 - x1^46*x2^45*x3^23*x4^11*z^25 - 4*x1^53*x2^37*x3^24*x4^11*z^25 - 2*x1^52*x2^38*x3^24*x4^11*z^25 + 4*x1^49*x2^41*x3^24*x4^11*z^25 + x1^48*x2^42*x3^24*x4^11*z^25 + 3*x1^47*x2^43*x3^24*x4^11*z^25 - x1^46*x2^44*x3^24*x4^11*z^25 + 2*x1^53*x2^36*x3^25*x4^11*z^25 + 4*x1^52*x2^37*x3^25*x4^11*z^25 - 2*x1^51*x2^38*x3^25*x4^11*z^25 + x1^50*x2^39*x3^25*x4^11*z^25 - 4*x1^49*x2^40*x3^25*x4^11*z^25 - 3*x1^47*x2^42*x3^25*x4^11*z^25 + x1^45*x2^44*x3^25*x4^11*z^25 - 4*x1^52*x2^36*x3^26*x4^11*z^25 + x1^50*x2^38*x3^26*x4^11*z^25 - x1^49*x2^39*x3^26*x4^11*z^25 + x1^48*x2^40*x3^26*x4^11*z^25 + x1^47*x2^41*x3^26*x4^11*z^25 + 3*x1^46*x2^42*x3^26*x4^11*z^25 - 2*x1^45*x2^43*x3^26*x4^11*z^25 - x1^44*x2^44*x3^26*x4^11*z^25 - x1^52*x2^35*x3^27*x4^11*z^25 + 4*x1^51*x2^36*x3^27*x4^11*z^25 - 2*x1^50*x2^37*x3^27*x4^11*z^25 + 2*x1^49*x2^38*x3^27*x4^11*z^25 - 5*x1^48*x2^39*x3^27*x4^11*z^25 - 2*x1^47*x2^40*x3^27*x4^11*z^25 - 3*x1^46*x2^41*x3^27*x4^11*z^25 + x1^45*x2^42*x3^27*x4^11*z^25 + x1^44*x2^43*x3^27*x4^11*z^25 - x1^50*x2^36*x3^28*x4^11*z^25 + 2*x1^49*x2^37*x3^28*x4^11*z^25 + 3*x1^45*x2^41*x3^28*x4^11*z^25 - x1^44*x2^42*x3^28*x4^11*z^25 + x1^43*x2^43*x3^28*x4^11*z^25 - x1^50*x2^35*x3^29*x4^11*z^25 - x1^49*x2^36*x3^29*x4^11*z^25 + 2*x1^48*x2^37*x3^29*x4^11*z^25 - 2*x1^46*x2^39*x3^29*x4^11*z^25 - 3*x1^45*x2^40*x3^29*x4^11*z^25 - x1^44*x2^41*x3^29*x4^11*z^25 - x1^48*x2^36*x3^30*x4^11*z^25 + 2*x1^47*x2^37*x3^30*x4^11*z^25 + 2*x1^46*x2^38*x3^30*x4^11*z^25 + x1^44*x2^40*x3^30*x4^11*z^25 + x1^43*x2^41*x3^30*x4^11*z^25 - 2*x1^46*x2^37*x3^31*x4^11*z^25 + x1^45*x2^38*x3^31*x4^11*z^25 - x1^44*x2^39*x3^31*x4^11*z^25 - x1^43*x2^40*x3^31*x4^11*z^25 + x1^42*x2^41*x3^31*x4^11*z^25 + x1^47*x2^35*x3^32*x4^11*z^25 + x1^45*x2^37*x3^32*x4^11*z^25 + x1^44*x2^38*x3^32*x4^11*z^25 + x1^42*x2^40*x3^32*x4^11*z^25 + x1^56*x2^40*x3^17*x4^12*z^25 - x1^56*x2^39*x3^18*x4^12*z^25 + x1^54*x2^41*x3^18*x4^12*z^25 - x1^53*x2^42*x3^18*x4^12*z^25 + 2*x1^55*x2^39*x3^19*x4^12*z^25 + x1^54*x2^40*x3^19*x4^12*z^25 - x1^53*x2^41*x3^19*x4^12*z^25 - 2*x1^51*x2^43*x3^19*x4^12*z^25 - x1^55*x2^38*x3^20*x4^12*z^25 - x1^54*x2^39*x3^20*x4^12*z^25 + 2*x1^53*x2^40*x3^20*x4^12*z^25 + x1^51*x2^42*x3^20*x4^12*z^25 + x1^50*x2^43*x3^20*x4^12*z^25 - x1^49*x2^44*x3^20*x4^12*z^25 + x1^54*x2^38*x3^21*x4^12*z^25 - x1^53*x2^39*x3^21*x4^12*z^25 - 4*x1^52*x2^40*x3^21*x4^12*z^25 + x1^51*x2^41*x3^21*x4^12*z^25 - 2*x1^50*x2^42*x3^21*x4^12*z^25 - x1^49*x2^43*x3^21*x4^12*z^25 + x1^48*x2^44*x3^21*x4^12*z^25 - 2*x1^53*x2^38*x3^22*x4^12*z^25 + 6*x1^52*x2^39*x3^22*x4^12*z^25 + 2*x1^50*x2^41*x3^22*x4^12*z^25 + x1^48*x2^43*x3^22*x4^12*z^25 - x1^47*x2^44*x3^22*x4^12*z^25 + 2*x1^53*x2^37*x3^23*x4^12*z^25 - x1^52*x2^38*x3^23*x4^12*z^25 - 5*x1^51*x2^39*x3^23*x4^12*z^25 - 3*x1^49*x2^41*x3^23*x4^12*z^25 + 3*x1^48*x2^42*x3^23*x4^12*z^25 - x1^47*x2^43*x3^23*x4^12*z^25 + 2*x1^46*x2^44*x3^23*x4^12*z^25 - x1^53*x2^36*x3^24*x4^12*z^25 - 2*x1^52*x2^37*x3^24*x4^12*z^25 + 6*x1^51*x2^38*x3^24*x4^12*z^25 + x1^50*x2^39*x3^24*x4^12*z^25 + 3*x1^49*x2^40*x3^24*x4^12*z^25 + 2*x1^47*x2^42*x3^24*x4^12*z^25 - 2*x1^45*x2^44*x3^24*x4^12*z^25 + 2*x1^52*x2^36*x3^25*x4^12*z^25 - x1^51*x2^37*x3^25*x4^12*z^25 - 5*x1^50*x2^38*x3^25*x4^12*z^25 - 5*x1^48*x2^40*x3^25*x4^12*z^25 + 3*x1^47*x2^41*x3^25*x4^12*z^25 - 2*x1^46*x2^42*x3^25*x4^12*z^25 + 6*x1^45*x2^43*x3^25*x4^12*z^25 + x1^44*x2^44*x3^25*x4^12*z^25 - 2*x1^51*x2^36*x3^26*x4^12*z^25 + 6*x1^50*x2^37*x3^26*x4^12*z^25 + x1^49*x2^38*x3^26*x4^12*z^25 + 4*x1^48*x2^39*x3^26*x4^12*z^25 - 2*x1^45*x2^42*x3^26*x4^12*z^25 - 6*x1^44*x2^43*x3^26*x4^12*z^25 + x1^51*x2^35*x3^27*x4^12*z^25 - x1^50*x2^36*x3^27*x4^12*z^25 - 6*x1^49*x2^37*x3^27*x4^12*z^25 + x1^48*x2^38*x3^27*x4^12*z^25 - 3*x1^47*x2^39*x3^27*x4^12*z^25 + 3*x1^46*x2^40*x3^27*x4^12*z^25 - x1^45*x2^41*x3^27*x4^12*z^25 + 6*x1^44*x2^42*x3^27*x4^12*z^25 + 2*x1^43*x2^43*x3^27*x4^12*z^25 - x1^50*x2^35*x3^28*x4^12*z^25 + 5*x1^49*x2^36*x3^28*x4^12*z^25 + x1^48*x2^37*x3^28*x4^12*z^25 + 3*x1^47*x2^38*x3^28*x4^12*z^25 + x1^46*x2^39*x3^28*x4^12*z^25 - x1^45*x2^40*x3^28*x4^12*z^25 - 2*x1^44*x2^41*x3^28*x4^12*z^25 - 5*x1^43*x2^42*x3^28*x4^12*z^25 + x1^50*x2^34*x3^29*x4^12*z^25 + x1^49*x2^35*x3^29*x4^12*z^25 - 4*x1^48*x2^36*x3^29*x4^12*z^25 - x1^47*x2^37*x3^29*x4^12*z^25 - 3*x1^46*x2^38*x3^29*x4^12*z^25 - x1^44*x2^40*x3^29*x4^12*z^25 + 4*x1^43*x2^41*x3^29*x4^12*z^25 + x1^42*x2^42*x3^29*x4^12*z^25 - x1^49*x2^34*x3^30*x4^12*z^25 + 2*x1^47*x2^36*x3^30*x4^12*z^25 + 3*x1^46*x2^37*x3^30*x4^12*z^25 + x1^45*x2^38*x3^30*x4^12*z^25 - x1^43*x2^40*x3^30*x4^12*z^25 - 2*x1^42*x2^41*x3^30*x4^12*z^25 - 3*x1^45*x2^37*x3^31*x4^12*z^25 + x1^44*x2^38*x3^31*x4^12*z^25 + x1^43*x2^39*x3^31*x4^12*z^25 + 2*x1^42*x2^40*x3^31*x4^12*z^25 - x1^46*x2^35*x3^32*x4^12*z^25 + x1^45*x2^36*x3^32*x4^12*z^25 + 2*x1^44*x2^37*x3^32*x4^12*z^25 - 2*x1^43*x2^38*x3^32*x4^12*z^25 - x1^42*x2^39*x3^32*x4^12*z^25 - x1^41*x2^40*x3^32*x4^12*z^25 + x1^44*x2^36*x3^33*x4^12*z^25 - 2*x1^55*x2^39*x3^18*x4^13*z^25 + x1^53*x2^41*x3^18*x4^13*z^25 - x1^52*x2^42*x3^18*x4^13*z^25 - x1^53*x2^40*x3^19*x4^13*z^25 + x1^51*x2^42*x3^19*x4^13*z^25 + x1^50*x2^43*x3^19*x4^13*z^25 - 2*x1^54*x2^38*x3^20*x4^13*z^25 + 2*x1^52*x2^40*x3^20*x4^13*z^25 - 2*x1^51*x2^41*x3^20*x4^13*z^25 + x1^50*x2^42*x3^20*x4^13*z^25 - 2*x1^52*x2^39*x3^21*x4^13*z^25 - x1^51*x2^40*x3^21*x4^13*z^25 + 2*x1^48*x2^43*x3^21*x4^13*z^25 + x1^47*x2^44*x3^21*x4^13*z^25 - 2*x1^53*x2^37*x3^22*x4^13*z^25 + x1^52*x2^38*x3^22*x4^13*z^25 + 5*x1^51*x2^39*x3^22*x4^13*z^25 + 3*x1^49*x2^41*x3^22*x4^13*z^25 - 4*x1^48*x2^42*x3^22*x4^13*z^25 - 2*x1^47*x2^43*x3^22*x4^13*z^25 - 2*x1^46*x2^44*x3^22*x4^13*z^25 - 6*x1^51*x2^38*x3^23*x4^13*z^25 - 2*x1^50*x2^39*x3^23*x4^13*z^25 - 2*x1^49*x2^40*x3^23*x4^13*z^25 - x1^48*x2^41*x3^23*x4^13*z^25 + 2*x1^47*x2^42*x3^23*x4^13*z^25 + 2*x1^46*x2^43*x3^23*x4^13*z^25 + 2*x1^45*x2^44*x3^23*x4^13*z^25 + 2*x1^51*x2^37*x3^24*x4^13*z^25 + 6*x1^50*x2^38*x3^24*x4^13*z^25 + 4*x1^48*x2^40*x3^24*x4^13*z^25 - 4*x1^47*x2^41*x3^24*x4^13*z^25 - x1^46*x2^42*x3^24*x4^13*z^25 - 5*x1^45*x2^43*x3^24*x4^13*z^25 - x1^44*x2^44*x3^24*x4^13*z^25 - 6*x1^50*x2^37*x3^25*x4^13*z^25 - 2*x1^49*x2^38*x3^25*x4^13*z^25 - 2*x1^48*x2^39*x3^25*x4^13*z^25 + 2*x1^46*x2^41*x3^25*x4^13*z^25 + 2*x1^45*x2^42*x3^25*x4^13*z^25 + 5*x1^44*x2^43*x3^25*x4^13*z^25 + 2*x1^50*x2^36*x3^26*x4^13*z^25 + 6*x1^49*x2^37*x3^26*x4^13*z^25 + 4*x1^47*x2^39*x3^26*x4^13*z^25 - 4*x1^46*x2^40*x3^26*x4^13*z^25 - 6*x1^44*x2^42*x3^26*x4^13*z^25 - 2*x1^43*x2^43*x3^26*x4^13*z^25 - x1^51*x2^34*x3^27*x4^13*z^25 - 4*x1^49*x2^36*x3^27*x4^13*z^25 - 2*x1^48*x2^37*x3^27*x4^13*z^25 - 2*x1^47*x2^38*x3^27*x4^13*z^25 + 2*x1^45*x2^40*x3^27*x4^13*z^25 + 2*x1^44*x2^41*x3^27*x4^13*z^25 + 6*x1^43*x2^42*x3^27*x4^13*z^25 + x1^50*x2^34*x3^28*x4^13*z^25 + x1^49*x2^35*x3^28*x4^13*z^25 + 4*x1^48*x2^36*x3^28*x4^13*z^25 + 2*x1^47*x2^37*x3^28*x4^13*z^25 + 4*x1^46*x2^38*x3^28*x4^13*z^25 - 4*x1^45*x2^39*x3^28*x4^13*z^25 - 6*x1^43*x2^41*x3^28*x4^13*z^25 - 2*x1^42*x2^42*x3^28*x4^13*z^25 - x1^49*x2^34*x3^29*x4^13*z^25 - x1^48*x2^35*x3^29*x4^13*z^25 - x1^47*x2^36*x3^29*x4^13*z^25 + x1^45*x2^38*x3^29*x4^13*z^25 + 3*x1^44*x2^39*x3^29*x4^13*z^25 + 2*x1^43*x2^40*x3^29*x4^13*z^25 + 6*x1^42*x2^41*x3^29*x4^13*z^25 + x1^48*x2^34*x3^30*x4^13*z^25 + x1^47*x2^35*x3^30*x4^13*z^25 + x1^46*x2^36*x3^30*x4^13*z^25 + 2*x1^45*x2^37*x3^30*x4^13*z^25 - x1^43*x2^39*x3^30*x4^13*z^25 - 5*x1^42*x2^40*x3^30*x4^13*z^25 - 2*x1^41*x2^41*x3^30*x4^13*z^25 - x1^46*x2^35*x3^31*x4^13*z^25 + x1^43*x2^38*x3^31*x4^13*z^25 + 2*x1^42*x2^39*x3^31*x4^13*z^25 + 3*x1^41*x2^40*x3^31*x4^13*z^25 - x1^45*x2^35*x3^32*x4^13*z^25 - 2*x1^41*x2^39*x3^32*x4^13*z^25 - x1^43*x2^36*x3^33*x4^13*z^25 + x1^40*x2^39*x3^33*x4^13*z^25 - x1^54*x2^39*x3^18*x4^14*z^25 - x1^51*x2^42*x3^18*x4^14*z^25 + x1^54*x2^38*x3^19*x4^14*z^25 - x1^50*x2^42*x3^19*x4^14*z^25 - 3*x1^53*x2^38*x3^20*x4^14*z^25 - 2*x1^52*x2^39*x3^20*x4^14*z^25 + x1^50*x2^41*x3^20*x4^14*z^25 - x1^48*x2^43*x3^20*x4^14*z^25 + 2*x1^53*x2^37*x3^21*x4^14*z^25 - x1^52*x2^38*x3^21*x4^14*z^25 - x1^51*x2^39*x3^21*x4^14*z^25 + 2*x1^50*x2^40*x3^21*x4^14*z^25 + x1^47*x2^43*x3^21*x4^14*z^25 + x1^46*x2^44*x3^21*x4^14*z^25 - 2*x1^52*x2^37*x3^22*x4^14*z^25 - x1^51*x2^38*x3^22*x4^14*z^25 - x1^50*x2^39*x3^22*x4^14*z^25 + x1^49*x2^40*x3^22*x4^14*z^25 - 2*x1^46*x2^43*x3^22*x4^14*z^25 - x1^45*x2^44*x3^22*x4^14*z^25 + 4*x1^52*x2^36*x3^23*x4^14*z^25 - x1^50*x2^38*x3^23*x4^14*z^25 + 2*x1^47*x2^41*x3^23*x4^14*z^25 - x1^46*x2^42*x3^23*x4^14*z^25 + 2*x1^45*x2^43*x3^23*x4^14*z^25 - 2*x1^51*x2^36*x3^24*x4^14*z^25 + 2*x1^50*x2^37*x3^24*x4^14*z^25 - 2*x1^49*x2^38*x3^24*x4^14*z^25 + x1^48*x2^39*x3^24*x4^14*z^25 + 2*x1^46*x2^41*x3^24*x4^14*z^25 - 2*x1^44*x2^43*x3^24*x4^14*z^25 + 2*x1^51*x2^35*x3^25*x4^14*z^25 + x1^50*x2^36*x3^25*x4^14*z^25 - x1^49*x2^37*x3^25*x4^14*z^25 - 3*x1^47*x2^39*x3^25*x4^14*z^25 - 2*x1^45*x2^41*x3^25*x4^14*z^25 + 2*x1^44*x2^42*x3^25*x4^14*z^25 + x1^43*x2^43*x3^25*x4^14*z^25 - x1^51*x2^34*x3^26*x4^14*z^25 - 2*x1^50*x2^35*x3^26*x4^14*z^25 + 2*x1^49*x2^36*x3^26*x4^14*z^25 + 2*x1^47*x2^38*x3^26*x4^14*z^25 + x1^45*x2^40*x3^26*x4^14*z^25 - 2*x1^43*x2^42*x3^26*x4^14*z^25 + 2*x1^50*x2^34*x3^27*x4^14*z^25 - x1^49*x2^35*x3^27*x4^14*z^25 - 2*x1^48*x2^36*x3^27*x4^14*z^25 - 2*x1^46*x2^38*x3^27*x4^14*z^25 - 2*x1^44*x2^40*x3^27*x4^14*z^25 + 2*x1^43*x2^41*x3^27*x4^14*z^25 + x1^42*x2^42*x3^27*x4^14*z^25 - 2*x1^49*x2^34*x3^28*x4^14*z^25 + x1^48*x2^35*x3^28*x4^14*z^25 + 2*x1^46*x2^37*x3^28*x4^14*z^25 + x1^44*x2^39*x3^28*x4^14*z^25 - x1^43*x2^40*x3^28*x4^14*z^25 - 2*x1^42*x2^41*x3^28*x4^14*z^25 + x1^48*x2^34*x3^29*x4^14*z^25 - x1^47*x2^35*x3^29*x4^14*z^25 - 2*x1^46*x2^36*x3^29*x4^14*z^25 - 2*x1^45*x2^37*x3^29*x4^14*z^25 + x1^44*x2^38*x3^29*x4^14*z^25 - 2*x1^43*x2^39*x3^29*x4^14*z^25 + 2*x1^42*x2^40*x3^29*x4^14*z^25 - x1^47*x2^34*x3^30*x4^14*z^25 + x1^45*x2^36*x3^30*x4^14*z^25 + x1^44*x2^37*x3^30*x4^14*z^25 + 2*x1^43*x2^38*x3^30*x4^14*z^25 - 2*x1^41*x2^40*x3^30*x4^14*z^25 + x1^46*x2^34*x3^31*x4^14*z^25 - x1^44*x2^36*x3^31*x4^14*z^25 - x1^43*x2^37*x3^31*x4^14*z^25 + x1^41*x2^39*x3^31*x4^14*z^25 + x1^40*x2^40*x3^31*x4^14*z^25 + x1^44*x2^35*x3^32*x4^14*z^25 + 2*x1^42*x2^37*x3^32*x4^14*z^25 - x1^40*x2^39*x3^32*x4^14*z^25 + x1^51*x2^41*x3^18*x4^15*z^25 + 2*x1^53*x2^38*x3^19*x4^15*z^25 - x1^51*x2^40*x3^19*x4^15*z^25 - x1^53*x2^37*x3^20*x4^15*z^25 - x1^52*x2^38*x3^20*x4^15*z^25 + 2*x1^50*x2^40*x3^20*x4^15*z^25 + x1^49*x2^41*x3^20*x4^15*z^25 + 4*x1^52*x2^37*x3^21*x4^15*z^25 - x1^50*x2^39*x3^21*x4^15*z^25 - 4*x1^49*x2^40*x3^21*x4^15*z^25 - 2*x1^48*x2^41*x3^21*x4^15*z^25 - x1^47*x2^42*x3^21*x4^15*z^25 - 3*x1^52*x2^36*x3^22*x4^15*z^25 - x1^51*x2^37*x3^22*x4^15*z^25 - x1^50*x2^38*x3^22*x4^15*z^25 + x1^49*x2^39*x3^22*x4^15*z^25 + x1^48*x2^40*x3^22*x4^15*z^25 + 2*x1^47*x2^41*x3^22*x4^15*z^25 + 2*x1^46*x2^42*x3^22*x4^15*z^25 + x1^52*x2^35*x3^23*x4^15*z^25 + 6*x1^51*x2^36*x3^23*x4^15*z^25 + 2*x1^49*x2^38*x3^23*x4^15*z^25 - 4*x1^48*x2^39*x3^23*x4^15*z^25 - 6*x1^46*x2^41*x3^23*x4^15*z^25 - 6*x1^51*x2^35*x3^24*x4^15*z^25 - 2*x1^50*x2^36*x3^24*x4^15*z^25 - 2*x1^49*x2^37*x3^24*x4^15*z^25 + 2*x1^47*x2^39*x3^24*x4^15*z^25 + 2*x1^46*x2^40*x3^24*x4^15*z^25 + 6*x1^45*x2^41*x3^24*x4^15*z^25 + 2*x1^51*x2^34*x3^25*x4^15*z^25 + 6*x1^50*x2^35*x3^25*x4^15*z^25 + 4*x1^48*x2^37*x3^25*x4^15*z^25 - 4*x1^47*x2^38*x3^25*x4^15*z^25 - 6*x1^45*x2^40*x3^25*x4^15*z^25 - 2*x1^44*x2^41*x3^25*x4^15*z^25 - 5*x1^50*x2^34*x3^26*x4^15*z^25 - 3*x1^49*x2^35*x3^26*x4^15*z^25 - 2*x1^48*x2^36*x3^26*x4^15*z^25 + 2*x1^46*x2^38*x3^26*x4^15*z^25 + 2*x1^45*x2^39*x3^26*x4^15*z^25 + 6*x1^44*x2^40*x3^26*x4^15*z^25 + 5*x1^49*x2^34*x3^27*x4^15*z^25 + 2*x1^48*x2^35*x3^27*x4^15*z^25 + 4*x1^47*x2^36*x3^27*x4^15*z^25 - 4*x1^46*x2^37*x3^27*x4^15*z^25 - 6*x1^44*x2^39*x3^27*x4^15*z^25 - 2*x1^43*x2^40*x3^27*x4^15*z^25 - x1^49*x2^33*x3^28*x4^15*z^25 - 3*x1^48*x2^34*x3^28*x4^15*z^25 - 3*x1^47*x2^35*x3^28*x4^15*z^25 - x1^46*x2^36*x3^28*x4^15*z^25 + x1^45*x2^37*x3^28*x4^15*z^25 + 2*x1^44*x2^38*x3^28*x4^15*z^25 + 6*x1^43*x2^39*x3^28*x4^15*z^25 + x1^48*x2^33*x3^29*x4^15*z^25 + 2*x1^47*x2^34*x3^29*x4^15*z^25 + 3*x1^46*x2^35*x3^29*x4^15*z^25 - 2*x1^45*x2^36*x3^29*x4^15*z^25 + x1^44*x2^37*x3^29*x4^15*z^25 - 6*x1^43*x2^38*x3^29*x4^15*z^25 - 2*x1^42*x2^39*x3^29*x4^15*z^25 - 2*x1^46*x2^34*x3^30*x4^15*z^25 - x1^45*x2^35*x3^30*x4^15*z^25 + x1^43*x2^37*x3^30*x4^15*z^25 + 6*x1^42*x2^38*x3^30*x4^15*z^25 + x1^45*x2^34*x3^31*x4^15*z^25 - x1^44*x2^35*x3^31*x4^15*z^25 + 2*x1^43*x2^36*x3^31*x4^15*z^25 - 4*x1^42*x2^37*x3^31*x4^15*z^25 - x1^41*x2^38*x3^31*x4^15*z^25 + x1^43*x2^35*x3^32*x4^15*z^25 - x1^42*x2^36*x3^32*x4^15*z^25 + 2*x1^41*x2^37*x3^32*x4^15*z^25 - 2*x1^50*x2^40*x3^19*x4^16*z^25 - x1^49*x2^41*x3^19*x4^16*z^25 + 3*x1^49*x2^40*x3^20*x4^16*z^25 + x1^47*x2^42*x3^20*x4^16*z^25 + x1^51*x2^37*x3^21*x4^16*z^25 + x1^50*x2^38*x3^21*x4^16*z^25 - 3*x1^49*x2^39*x3^21*x4^16*z^25 - 2*x1^48*x2^40*x3^21*x4^16*z^25 - x1^47*x2^41*x3^21*x4^16*z^25 - 3*x1^46*x2^42*x3^21*x4^16*z^25 - 3*x1^51*x2^36*x3^22*x4^16*z^25 + x1^50*x2^37*x3^22*x4^16*z^25 + 3*x1^48*x2^39*x3^22*x4^16*z^25 + 2*x1^47*x2^40*x3^22*x4^16*z^25 + 5*x1^46*x2^41*x3^22*x4^16*z^25 + x1^45*x2^42*x3^22*x4^16*z^25 + 2*x1^51*x2^35*x3^23*x4^16*z^25 + 2*x1^50*x2^36*x3^23*x4^16*z^25 - 2*x1^48*x2^38*x3^23*x4^16*z^25 - 2*x1^47*x2^39*x3^23*x4^16*z^25 - 2*x1^46*x2^40*x3^23*x4^16*z^25 - 6*x1^45*x2^41*x3^23*x4^16*z^25 - x1^51*x2^34*x3^24*x4^16*z^25 - 5*x1^50*x2^35*x3^24*x4^16*z^25 + x1^49*x2^36*x3^24*x4^16*z^25 - x1^48*x2^37*x3^24*x4^16*z^25 + 4*x1^47*x2^38*x3^24*x4^16*z^25 + 6*x1^45*x2^40*x3^24*x4^16*z^25 + 2*x1^44*x2^41*x3^24*x4^16*z^25 + 3*x1^50*x2^34*x3^25*x4^16*z^25 + 2*x1^49*x2^35*x3^25*x4^16*z^25 + 2*x1^48*x2^36*x3^25*x4^16*z^25 - x1^47*x2^37*x3^25*x4^16*z^25 - 2*x1^46*x2^38*x3^25*x4^16*z^25 - 2*x1^45*x2^39*x3^25*x4^16*z^25 - 6*x1^44*x2^40*x3^25*x4^16*z^25 - x1^50*x2^33*x3^26*x4^16*z^25 - 4*x1^49*x2^34*x3^26*x4^16*z^25 - x1^48*x2^35*x3^26*x4^16*z^25 - 4*x1^47*x2^36*x3^26*x4^16*z^25 + 4*x1^46*x2^37*x3^26*x4^16*z^25 + 6*x1^44*x2^39*x3^26*x4^16*z^25 + 2*x1^43*x2^40*x3^26*x4^16*z^25 + x1^49*x2^33*x3^27*x4^16*z^25 + x1^48*x2^34*x3^27*x4^16*z^25 + x1^47*x2^35*x3^27*x4^16*z^25 - 2*x1^45*x2^37*x3^27*x4^16*z^25 - 2*x1^44*x2^38*x3^27*x4^16*z^25 - 6*x1^43*x2^39*x3^27*x4^16*z^25 - x1^48*x2^33*x3^28*x4^16*z^25 - x1^46*x2^35*x3^28*x4^16*z^25 + 4*x1^45*x2^36*x3^28*x4^16*z^25 + 6*x1^43*x2^38*x3^28*x4^16*z^25 + 2*x1^42*x2^39*x3^28*x4^16*z^25 + x1^47*x2^33*x3^29*x4^16*z^25 - x1^44*x2^36*x3^29*x4^16*z^25 - 6*x1^42*x2^38*x3^29*x4^16*z^25 + x1^45*x2^34*x3^30*x4^16*z^25 + 2*x1^44*x2^35*x3^30*x4^16*z^25 - x1^43*x2^36*x3^30*x4^16*z^25 + 3*x1^42*x2^37*x3^30*x4^16*z^25 + 2*x1^41*x2^38*x3^30*x4^16*z^25 - x1^43*x2^35*x3^31*x4^16*z^25 - 3*x1^41*x2^37*x3^31*x4^16*z^25 + x1^41*x2^36*x3^32*x4^16*z^25 - x1^40*x2^36*x3^33*x4^16*z^25 - 2*x1^48*x2^39*x3^21*x4^17*z^25 + x1^48*x2^38*x3^22*x4^17*z^25 + 2*x1^45*x2^41*x3^22*x4^17*z^25 - x1^47*x2^38*x3^23*x4^17*z^25 - x1^46*x2^39*x3^23*x4^17*z^25 - x1^45*x2^40*x3^23*x4^17*z^25 - x1^50*x2^34*x3^24*x4^17*z^25 - x1^49*x2^35*x3^24*x4^17*z^25 + x1^48*x2^36*x3^24*x4^17*z^25 + x1^47*x2^37*x3^24*x4^17*z^25 + 2*x1^44*x2^40*x3^24*x4^17*z^25 + x1^49*x2^34*x3^25*x4^17*z^25 - x1^48*x2^35*x3^25*x4^17*z^25 + x1^47*x2^36*x3^25*x4^17*z^25 - x1^46*x2^37*x3^25*x4^17*z^25 - 2*x1^44*x2^39*x3^25*x4^17*z^25 - x1^43*x2^40*x3^25*x4^17*z^25 - x1^48*x2^34*x3^26*x4^17*z^25 + x1^46*x2^36*x3^26*x4^17*z^25 + x1^45*x2^37*x3^26*x4^17*z^25 + x1^44*x2^38*x3^26*x4^17*z^25 + 2*x1^43*x2^39*x3^26*x4^17*z^25 + x1^47*x2^34*x3^27*x4^17*z^25 - x1^45*x2^36*x3^27*x4^17*z^25 - 2*x1^43*x2^38*x3^27*x4^17*z^25 - x1^42*x2^39*x3^27*x4^17*z^25 - x1^46*x2^34*x3^28*x4^17*z^25 + x1^44*x2^36*x3^28*x4^17*z^25 + x1^43*x2^37*x3^28*x4^17*z^25 + 2*x1^42*x2^38*x3^28*x4^17*z^25 - x1^44*x2^35*x3^29*x4^17*z^25 - 2*x1^42*x2^37*x3^29*x4^17*z^25 - x1^42*x2^36*x3^30*x4^17*z^25 + 2*x1^41*x2^37*x3^30*x4^17*z^25 - x1^40*x2^37*x3^31*x4^17*z^25 - x1^56*x2^42*x3^21*x4*z^24 - x1^55*x2^41*x3^23*x4*z^24 - x1^55*x2^42*x3^21*x4^2*z^24 + 2*x1^55*x2^41*x3^22*x4^2*z^24 - x1^53*x2^43*x3^22*x4^2*z^24 - 3*x1^54*x2^41*x3^23*x4^2*z^24 + x1^53*x2^42*x3^23*x4^2*z^24 + x1^54*x2^40*x3^24*x4^2*z^24 + x1^53*x2^41*x3^24*x4^2*z^24 + x1^52*x2^42*x3^24*x4^2*z^24 + x1^51*x2^43*x3^24*x4^2*z^24 - 2*x1^53*x2^40*x3^25*x4^2*z^24 - x1^55*x2^41*x3^21*x4^3*z^24 + 2*x1^54*x2^41*x3^22*x4^3*z^24 + x1^53*x2^42*x3^22*x4^3*z^24 + x1^52*x2^43*x3^22*x4^3*z^24 + 2*x1^55*x2^39*x3^23*x4^3*z^24 - 2*x1^53*x2^41*x3^23*x4^3*z^24 + x1^52*x2^42*x3^23*x4^3*z^24 + 3*x1^53*x2^40*x3^24*x4^3*z^24 - x1^52*x2^41*x3^24*x4^3*z^24 - x1^52*x2^40*x3^25*x4^3*z^24 - x1^51*x2^41*x3^25*x4^3*z^24 - x1^50*x2^42*x3^25*x4^3*z^24 + 2*x1^52*x2^39*x3^26*x4^3*z^24 + x1^49*x2^42*x3^26*x4^3*z^24 - x1^56*x2^40*x3^20*x4^4*z^24 + 2*x1^56*x2^39*x3^21*x4^4*z^24 + x1^55*x2^40*x3^21*x4^4*z^24 - x1^54*x2^41*x3^21*x4^4*z^24 - 4*x1^55*x2^39*x3^22*x4^4*z^24 + 2*x1^54*x2^40*x3^22*x4^4*z^24 - 2*x1^52*x2^42*x3^22*x4^4*z^24 + x1^51*x2^43*x3^22*x4^4*z^24 + 2*x1^55*x2^38*x3^23*x4^4*z^24 + 3*x1^54*x2^39*x3^23*x4^4*z^24 - 2*x1^53*x2^40*x3^23*x4^4*z^24 + 2*x1^52*x2^41*x3^23*x4^4*z^24 + x1^50*x2^43*x3^23*x4^4*z^24 - 4*x1^54*x2^38*x3^24*x4^4*z^24 + x1^52*x2^40*x3^24*x4^4*z^24 - 2*x1^51*x2^41*x3^24*x4^4*z^24 - x1^50*x2^42*x3^24*x4^4*z^24 + x1^49*x2^43*x3^24*x4^4*z^24 + x1^48*x2^44*x3^24*x4^4*z^24 + x1^54*x2^37*x3^25*x4^4*z^24 + 2*x1^53*x2^38*x3^25*x4^4*z^24 - x1^52*x2^39*x3^25*x4^4*z^24 + 3*x1^51*x2^40*x3^25*x4^4*z^24 - x1^53*x2^37*x3^26*x4^4*z^24 + x1^52*x2^38*x3^26*x4^4*z^24 + x1^51*x2^39*x3^26*x4^4*z^24 - x1^50*x2^40*x3^26*x4^4*z^24 - x1^51*x2^38*x3^27*x4^4*z^24 + x1^50*x2^39*x3^27*x4^4*z^24 - x1^49*x2^40*x3^27*x4^4*z^24 - x1^48*x2^41*x3^27*x4^4*z^24 + 2*x1^56*x2^40*x3^19*x4^5*z^24 - x1^56*x2^39*x3^20*x4^5*z^24 - 2*x1^55*x2^40*x3^20*x4^5*z^24 + 2*x1^54*x2^41*x3^20*x4^5*z^24 + 5*x1^55*x2^39*x3^21*x4^5*z^24 + x1^54*x2^40*x3^21*x4^5*z^24 - 2*x1^55*x2^38*x3^22*x4^5*z^24 - 5*x1^54*x2^39*x3^22*x4^5*z^24 + x1^53*x2^40*x3^22*x4^5*z^24 - x1^52*x2^41*x3^22*x4^5*z^24 + 6*x1^54*x2^38*x3^23*x4^5*z^24 + x1^52*x2^40*x3^23*x4^5*z^24 + 3*x1^51*x2^41*x3^23*x4^5*z^24 - x1^50*x2^42*x3^23*x4^5*z^24 - 2*x1^49*x2^43*x3^23*x4^5*z^24 - 2*x1^54*x2^37*x3^24*x4^5*z^24 - 5*x1^53*x2^38*x3^24*x4^5*z^24 + 2*x1^52*x2^39*x3^24*x4^5*z^24 - 4*x1^51*x2^40*x3^24*x4^5*z^24 + x1^50*x2^41*x3^24*x4^5*z^24 + x1^49*x2^42*x3^24*x4^5*z^24 + 2*x1^48*x2^43*x3^24*x4^5*z^24 + 4*x1^53*x2^37*x3^25*x4^5*z^24 + x1^52*x2^38*x3^25*x4^5*z^24 + x1^50*x2^40*x3^25*x4^5*z^24 - x1^49*x2^41*x3^25*x4^5*z^24 - 2*x1^48*x2^42*x3^25*x4^5*z^24 + x1^46*x2^44*x3^25*x4^5*z^24 - 2*x1^53*x2^36*x3^26*x4^5*z^24 - 2*x1^52*x2^37*x3^26*x4^5*z^24 - 3*x1^50*x2^39*x3^26*x4^5*z^24 + x1^49*x2^40*x3^26*x4^5*z^24 - x1^48*x2^41*x3^26*x4^5*z^24 + 2*x1^52*x2^36*x3^27*x4^5*z^24 - x1^51*x2^37*x3^27*x4^5*z^24 + x1^50*x2^38*x3^27*x4^5*z^24 + 2*x1^49*x2^39*x3^27*x4^5*z^24 - x1^48*x2^40*x3^27*x4^5*z^24 - x1^52*x2^35*x3^28*x4^5*z^24 - x1^51*x2^36*x3^28*x4^5*z^24 + x1^50*x2^37*x3^28*x4^5*z^24 - x1^49*x2^38*x3^28*x4^5*z^24 + 2*x1^48*x2^39*x3^28*x4^5*z^24 - x1^50*x2^36*x3^29*x4^5*z^24 - x1^49*x2^37*x3^29*x4^5*z^24 + x1^48*x2^38*x3^29*x4^5*z^24 - x1^47*x2^39*x3^29*x4^5*z^24 - 2*x1^55*x2^39*x3^20*x4^6*z^24 - x1^54*x2^40*x3^20*x4^6*z^24 - x1^53*x2^41*x3^20*x4^6*z^24 + x1^55*x2^38*x3^21*x4^6*z^24 + 2*x1^54*x2^39*x3^21*x4^6*z^24 - x1^53*x2^40*x3^21*x4^6*z^24 - x1^51*x2^42*x3^21*x4^6*z^24 - 2*x1^54*x2^38*x3^22*x4^6*z^24 + 3*x1^53*x2^39*x3^22*x4^6*z^24 + x1^52*x2^40*x3^22*x4^6*z^24 + x1^50*x2^42*x3^22*x4^6*z^24 + x1^49*x2^43*x3^22*x4^6*z^24 + 2*x1^53*x2^38*x3^23*x4^6*z^24 - 5*x1^52*x2^39*x3^23*x4^6*z^24 + x1^51*x2^40*x3^23*x4^6*z^24 - x1^50*x2^41*x3^23*x4^6*z^24 - x1^49*x2^42*x3^23*x4^6*z^24 - 2*x1^48*x2^43*x3^23*x4^6*z^24 - 3*x1^53*x2^37*x3^24*x4^6*z^24 - x1^52*x2^38*x3^24*x4^6*z^24 + 4*x1^51*x2^39*x3^24*x4^6*z^24 - 2*x1^50*x2^40*x3^24*x4^6*z^24 - x1^48*x2^42*x3^24*x4^6*z^24 + x1^53*x2^36*x3^25*x4^6*z^24 + x1^52*x2^37*x3^25*x4^6*z^24 - 4*x1^51*x2^38*x3^25*x4^6*z^24 + 2*x1^50*x2^39*x3^25*x4^6*z^24 - x1^49*x2^40*x3^25*x4^6*z^24 - x1^48*x2^41*x3^25*x4^6*z^24 - 2*x1^47*x2^42*x3^25*x4^6*z^24 - x1^46*x2^43*x3^25*x4^6*z^24 - x1^52*x2^36*x3^26*x4^6*z^24 - x1^51*x2^37*x3^26*x4^6*z^24 - x1^50*x2^38*x3^26*x4^6*z^24 + 4*x1^48*x2^40*x3^26*x4^6*z^24 + x1^47*x2^41*x3^26*x4^6*z^24 - x1^46*x2^42*x3^26*x4^6*z^24 + 2*x1^52*x2^35*x3^27*x4^6*z^24 + x1^51*x2^36*x3^27*x4^6*z^24 - 2*x1^50*x2^37*x3^27*x4^6*z^24 + x1^49*x2^38*x3^27*x4^6*z^24 - 2*x1^48*x2^39*x3^27*x4^6*z^24 + x1^47*x2^40*x3^27*x4^6*z^24 + x1^44*x2^43*x3^27*x4^6*z^24 - x1^51*x2^35*x3^28*x4^6*z^24 + x1^50*x2^36*x3^28*x4^6*z^24 - x1^49*x2^37*x3^28*x4^6*z^24 - 2*x1^48*x2^38*x3^28*x4^6*z^24 + 2*x1^47*x2^39*x3^28*x4^6*z^24 - x1^47*x2^38*x3^29*x4^6*z^24 + x1^46*x2^39*x3^29*x4^6*z^24 + x1^48*x2^36*x3^30*x4^6*z^24 - x1^47*x2^37*x3^30*x4^6*z^24 + x1^46*x2^38*x3^30*x4^6*z^24 + x1^55*x2^41*x3^17*x4^7*z^24 - 2*x1^54*x2^41*x3^18*x4^7*z^24 + x1^53*x2^42*x3^18*x4^7*z^24 + x1^55*x2^39*x3^19*x4^7*z^24 + 2*x1^53*x2^41*x3^19*x4^7*z^24 + x1^51*x2^43*x3^19*x4^7*z^24 - x1^54*x2^39*x3^20*x4^7*z^24 - 2*x1^53*x2^40*x3^20*x4^7*z^24 - 2*x1^52*x2^41*x3^20*x4^7*z^24 - x1^51*x2^42*x3^20*x4^7*z^24 + x1^49*x2^44*x3^20*x4^7*z^24 + x1^54*x2^38*x3^21*x4^7*z^24 + x1^53*x2^39*x3^21*x4^7*z^24 + 3*x1^52*x2^40*x3^21*x4^7*z^24 + x1^51*x2^41*x3^21*x4^7*z^24 + x1^50*x2^42*x3^21*x4^7*z^24 - x1^48*x2^44*x3^21*x4^7*z^24 - x1^53*x2^38*x3^22*x4^7*z^24 - 2*x1^52*x2^39*x3^22*x4^7*z^24 - 2*x1^51*x2^40*x3^22*x4^7*z^24 - x1^50*x2^41*x3^22*x4^7*z^24 - 2*x1^49*x2^42*x3^22*x4^7*z^24 - x1^48*x2^43*x3^22*x4^7*z^24 + x1^47*x2^44*x3^22*x4^7*z^24 + 2*x1^51*x2^39*x3^23*x4^7*z^24 + x1^50*x2^40*x3^23*x4^7*z^24 + x1^49*x2^41*x3^23*x4^7*z^24 - x1^48*x2^42*x3^23*x4^7*z^24 + x1^47*x2^43*x3^23*x4^7*z^24 - x1^46*x2^44*x3^23*x4^7*z^24 - x1^50*x2^39*x3^24*x4^7*z^24 + x1^49*x2^40*x3^24*x4^7*z^24 - x1^47*x2^42*x3^24*x4^7*z^24 - x1^46*x2^43*x3^24*x4^7*z^24 + x1^52*x2^36*x3^25*x4^7*z^24 + x1^51*x2^37*x3^25*x4^7*z^24 - x1^50*x2^38*x3^25*x4^7*z^24 + 2*x1^48*x2^40*x3^25*x4^7*z^24 - x1^46*x2^42*x3^25*x4^7*z^24 - x1^45*x2^43*x3^25*x4^7*z^24 - x1^49*x2^38*x3^26*x4^7*z^24 + x1^48*x2^39*x3^26*x4^7*z^24 - x1^47*x2^40*x3^26*x4^7*z^24 + 2*x1^46*x2^41*x3^26*x4^7*z^24 + x1^51*x2^35*x3^27*x4^7*z^24 + x1^48*x2^38*x3^27*x4^7*z^24 - x1^47*x2^39*x3^27*x4^7*z^24 - x1^44*x2^42*x3^27*x4^7*z^24 - 2*x1^48*x2^37*x3^28*x4^7*z^24 + 2*x1^50*x2^34*x3^29*x4^7*z^24 + x1^48*x2^36*x3^29*x4^7*z^24 + x1^47*x2^37*x3^29*x4^7*z^24 - x1^46*x2^38*x3^29*x4^7*z^24 - x1^47*x2^36*x3^30*x4^7*z^24 - x1^45*x2^38*x3^30*x4^7*z^24 - x1^47*x2^35*x3^31*x4^7*z^24 - x1^46*x2^36*x3^31*x4^7*z^24 + x1^54*x2^41*x3^17*x4^8*z^24 - x1^54*x2^40*x3^18*x4^8*z^24 - x1^53*x2^41*x3^18*x4^8*z^24 + x1^52*x2^42*x3^18*x4^8*z^24 - x1^55*x2^38*x3^19*x4^8*z^24 + 4*x1^53*x2^40*x3^19*x4^8*z^24 - x1^52*x2^41*x3^19*x4^8*z^24 - x1^53*x2^39*x3^20*x4^8*z^24 - 4*x1^52*x2^40*x3^20*x4^8*z^24 + x1^51*x2^41*x3^20*x4^8*z^24 - 3*x1^50*x2^42*x3^20*x4^8*z^24 - x1^49*x2^43*x3^20*x4^8*z^24 + x1^54*x2^37*x3^21*x4^8*z^24 + 3*x1^52*x2^39*x3^21*x4^8*z^24 + 2*x1^51*x2^40*x3^21*x4^8*z^24 + 2*x1^50*x2^41*x3^21*x4^8*z^24 - x1^48*x2^43*x3^21*x4^8*z^24 - x1^47*x2^44*x3^21*x4^8*z^24 - 2*x1^53*x2^37*x3^22*x4^8*z^24 - x1^52*x2^38*x3^22*x4^8*z^24 - 4*x1^51*x2^39*x3^22*x4^8*z^24 - 2*x1^49*x2^41*x3^22*x4^8*z^24 + 4*x1^48*x2^42*x3^22*x4^8*z^24 + x1^47*x2^43*x3^22*x4^8*z^24 + 2*x1^46*x2^44*x3^22*x4^8*z^24 + x1^53*x2^36*x3^23*x4^8*z^24 + 2*x1^52*x2^37*x3^23*x4^8*z^24 + 4*x1^51*x2^38*x3^23*x4^8*z^24 + x1^50*x2^39*x3^23*x4^8*z^24 + x1^49*x2^40*x3^23*x4^8*z^24 + 2*x1^48*x2^41*x3^23*x4^8*z^24 - x1^47*x2^42*x3^23*x4^8*z^24 - x1^46*x2^43*x3^23*x4^8*z^24 - 2*x1^45*x2^44*x3^23*x4^8*z^24 - 2*x1^52*x2^36*x3^24*x4^8*z^24 - x1^51*x2^37*x3^24*x4^8*z^24 - 4*x1^50*x2^38*x3^24*x4^8*z^24 - x1^49*x2^39*x3^24*x4^8*z^24 - 3*x1^48*x2^40*x3^24*x4^8*z^24 + x1^47*x2^41*x3^24*x4^8*z^24 + x1^46*x2^42*x3^24*x4^8*z^24 + 4*x1^45*x2^43*x3^24*x4^8*z^24 + x1^44*x2^44*x3^24*x4^8*z^24 + 2*x1^51*x2^36*x3^25*x4^8*z^24 + 4*x1^50*x2^37*x3^25*x4^8*z^24 + 4*x1^49*x2^38*x3^25*x4^8*z^24 - 2*x1^46*x2^41*x3^25*x4^8*z^24 - 2*x1^45*x2^42*x3^25*x4^8*z^24 - 2*x1^44*x2^43*x3^25*x4^8*z^24 - 2*x1^51*x2^35*x3^26*x4^8*z^24 - 2*x1^50*x2^36*x3^26*x4^8*z^24 - 3*x1^49*x2^37*x3^26*x4^8*z^24 - x1^47*x2^39*x3^26*x4^8*z^24 + x1^46*x2^40*x3^26*x4^8*z^24 + x1^45*x2^41*x3^26*x4^8*z^24 + 3*x1^44*x2^42*x3^26*x4^8*z^24 + x1^51*x2^34*x3^27*x4^8*z^24 + x1^50*x2^35*x3^27*x4^8*z^24 + 3*x1^49*x2^36*x3^27*x4^8*z^24 + 3*x1^48*x2^37*x3^27*x4^8*z^24 + 2*x1^47*x2^38*x3^27*x4^8*z^24 + 2*x1^46*x2^39*x3^27*x4^8*z^24 - 2*x1^45*x2^40*x3^27*x4^8*z^24 - x1^44*x2^41*x3^27*x4^8*z^24 - x1^43*x2^42*x3^27*x4^8*z^24 - 3*x1^50*x2^34*x3^28*x4^8*z^24 - 3*x1^48*x2^36*x3^28*x4^8*z^24 - 3*x1^47*x2^37*x3^28*x4^8*z^24 - x1^46*x2^38*x3^28*x4^8*z^24 + x1^45*x2^39*x3^28*x4^8*z^24 + 2*x1^43*x2^41*x3^28*x4^8*z^24 + x1^49*x2^34*x3^29*x4^8*z^24 + x1^47*x2^36*x3^29*x4^8*z^24 + x1^46*x2^37*x3^29*x4^8*z^24 + x1^45*x2^38*x3^29*x4^8*z^24 - x1^44*x2^39*x3^29*x4^8*z^24 - x1^42*x2^41*x3^29*x4^8*z^24 - x1^49*x2^33*x3^30*x4^8*z^24 - x1^45*x2^37*x3^30*x4^8*z^24 + x1^47*x2^34*x3^31*x4^8*z^24 + x1^46*x2^35*x3^31*x4^8*z^24 + x1^45*x2^36*x3^31*x4^8*z^24 + x1^44*x2^37*x3^31*x4^8*z^24 + x1^45*x2^35*x3^32*x4^8*z^24 - x1^44*x2^36*x3^32*x4^8*z^24 + x1^55*x2^39*x3^17*x4^9*z^24 + x1^55*x2^38*x3^18*x4^9*z^24 + x1^54*x2^39*x3^18*x4^9*z^24 - 2*x1^53*x2^40*x3^18*x4^9*z^24 + x1^52*x2^41*x3^18*x4^9*z^24 - x1^53*x2^39*x3^19*x4^9*z^24 + x1^52*x2^40*x3^19*x4^9*z^24 - x1^51*x2^41*x3^19*x4^9*z^24 - x1^49*x2^43*x3^19*x4^9*z^24 - x1^54*x2^37*x3^20*x4^9*z^24 - x1^52*x2^39*x3^20*x4^9*z^24 - x1^51*x2^40*x3^20*x4^9*z^24 + x1^49*x2^42*x3^20*x4^9*z^24 + 2*x1^48*x2^43*x3^20*x4^9*z^24 + 4*x1^53*x2^37*x3^21*x4^9*z^24 + 2*x1^52*x2^38*x3^21*x4^9*z^24 + 3*x1^51*x2^39*x3^21*x4^9*z^24 - x1^50*x2^40*x3^21*x4^9*z^24 - x1^49*x2^41*x3^21*x4^9*z^24 - 2*x1^48*x2^42*x3^21*x4^9*z^24 - 2*x1^47*x2^43*x3^21*x4^9*z^24 - x1^46*x2^44*x3^21*x4^9*z^24 - 2*x1^53*x2^36*x3^22*x4^9*z^24 - 4*x1^52*x2^37*x3^22*x4^9*z^24 - x1^51*x2^38*x3^22*x4^9*z^24 + 2*x1^49*x2^40*x3^22*x4^9*z^24 + x1^48*x2^41*x3^22*x4^9*z^24 + x1^47*x2^42*x3^22*x4^9*z^24 + 2*x1^46*x2^43*x3^22*x4^9*z^24 + x1^45*x2^44*x3^22*x4^9*z^24 + 4*x1^52*x2^36*x3^23*x4^9*z^24 + x1^51*x2^37*x3^23*x4^9*z^24 + 2*x1^50*x2^38*x3^23*x4^9*z^24 - 2*x1^47*x2^41*x3^23*x4^9*z^24 - 3*x1^45*x2^43*x3^23*x4^9*z^24 - x1^52*x2^35*x3^24*x4^9*z^24 - 4*x1^51*x2^36*x3^24*x4^9*z^24 - 2*x1^50*x2^37*x3^24*x4^9*z^24 - 4*x1^49*x2^38*x3^24*x4^9*z^24 + x1^47*x2^40*x3^24*x4^9*z^24 + x1^46*x2^41*x3^24*x4^9*z^24 + 2*x1^44*x2^43*x3^24*x4^9*z^24 + 4*x1^51*x2^35*x3^25*x4^9*z^24 + 2*x1^50*x2^36*x3^25*x4^9*z^24 + 3*x1^49*x2^37*x3^25*x4^9*z^24 + x1^48*x2^38*x3^25*x4^9*z^24 - x1^46*x2^40*x3^25*x4^9*z^24 - 2*x1^45*x2^41*x3^25*x4^9*z^24 - 2*x1^44*x2^42*x3^25*x4^9*z^24 - x1^51*x2^34*x3^26*x4^9*z^24 - 4*x1^50*x2^35*x3^26*x4^9*z^24 - 2*x1^49*x2^36*x3^26*x4^9*z^24 - 4*x1^48*x2^37*x3^26*x4^9*z^24 - 2*x1^47*x2^38*x3^26*x4^9*z^24 - x1^46*x2^39*x3^26*x4^9*z^24 + 3*x1^45*x2^40*x3^26*x4^9*z^24 + x1^44*x2^41*x3^26*x4^9*z^24 + x1^43*x2^42*x3^26*x4^9*z^24 + 4*x1^50*x2^34*x3^27*x4^9*z^24 + x1^49*x2^35*x3^27*x4^9*z^24 + 2*x1^48*x2^36*x3^27*x4^9*z^24 + x1^47*x2^37*x3^27*x4^9*z^24 - x1^46*x2^38*x3^27*x4^9*z^24 - 3*x1^45*x2^39*x3^27*x4^9*z^24 - 2*x1^44*x2^40*x3^27*x4^9*z^24 - x1^43*x2^41*x3^27*x4^9*z^24 - x1^50*x2^33*x3^28*x4^9*z^24 - 4*x1^49*x2^34*x3^28*x4^9*z^24 + x1^48*x2^35*x3^28*x4^9*z^24 - x1^46*x2^37*x3^28*x4^9*z^24 - x1^45*x2^38*x3^28*x4^9*z^24 + x1^44*x2^39*x3^28*x4^9*z^24 - x1^43*x2^40*x3^28*x4^9*z^24 + x1^42*x2^41*x3^28*x4^9*z^24 + 3*x1^49*x2^33*x3^29*x4^9*z^24 - x1^48*x2^34*x3^29*x4^9*z^24 + 3*x1^46*x2^36*x3^29*x4^9*z^24 + x1^45*x2^37*x3^29*x4^9*z^24 - x1^43*x2^39*x3^29*x4^9*z^24 - 2*x1^48*x2^33*x3^30*x4^9*z^24 - 3*x1^46*x2^35*x3^30*x4^9*z^24 - x1^45*x2^36*x3^30*x4^9*z^24 + x1^43*x2^38*x3^30*x4^9*z^24 + x1^41*x2^40*x3^30*x4^9*z^24 - x1^46*x2^34*x3^31*x4^9*z^24 + x1^43*x2^36*x3^32*x4^9*z^24 + x1^55*x2^39*x3^16*x4^10*z^24 - x1^54*x2^40*x3^16*x4^10*z^24 - x1^54*x2^39*x3^17*x4^10*z^24 + x1^53*x2^39*x3^18*x4^10*z^24 - 2*x1^51*x2^41*x3^18*x4^10*z^24 - x1^50*x2^42*x3^18*x4^10*z^24 - x1^53*x2^38*x3^19*x4^10*z^24 + x1^52*x2^39*x3^19*x4^10*z^24 + x1^51*x2^40*x3^19*x4^10*z^24 + x1^50*x2^41*x3^19*x4^10*z^24 + 2*x1^49*x2^42*x3^19*x4^10*z^24 + x1^48*x2^43*x3^19*x4^10*z^24 + x1^53*x2^37*x3^20*x4^10*z^24 + x1^52*x2^38*x3^20*x4^10*z^24 - x1^50*x2^40*x3^20*x4^10*z^24 - x1^49*x2^41*x3^20*x4^10*z^24 - 2*x1^48*x2^42*x3^20*x4^10*z^24 - x1^47*x2^43*x3^20*x4^10*z^24 - x1^52*x2^37*x3^21*x4^10*z^24 + x1^50*x2^39*x3^21*x4^10*z^24 + 3*x1^49*x2^40*x3^21*x4^10*z^24 + x1^48*x2^41*x3^21*x4^10*z^24 + 3*x1^47*x2^42*x3^21*x4^10*z^24 + x1^46*x2^43*x3^21*x4^10*z^24 - x1^49*x2^39*x3^22*x4^10*z^24 - x1^47*x2^41*x3^22*x4^10*z^24 - 3*x1^46*x2^42*x3^22*x4^10*z^24 - x1^49*x2^38*x3^23*x4^10*z^24 + x1^48*x2^39*x3^23*x4^10*z^24 + 2*x1^46*x2^41*x3^23*x4^10*z^24 + x1^45*x2^42*x3^23*x4^10*z^24 - x1^47*x2^39*x3^24*x4^10*z^24 - x1^46*x2^40*x3^24*x4^10*z^24 - 2*x1^45*x2^41*x3^24*x4^10*z^24 - x1^43*x2^43*x3^24*x4^10*z^24 + x1^44*x2^41*x3^25*x4^10*z^24 - x1^50*x2^34*x3^26*x4^10*z^24 + x1^46*x2^38*x3^26*x4^10*z^24 - 2*x1^45*x2^39*x3^26*x4^10*z^24 - x1^43*x2^41*x3^26*x4^10*z^24 - x1^42*x2^42*x3^26*x4^10*z^24 + x1^50*x2^33*x3^27*x4^10*z^24 + x1^49*x2^34*x3^27*x4^10*z^24 - x1^48*x2^35*x3^27*x4^10*z^24 + x1^46*x2^37*x3^27*x4^10*z^24 + x1^45*x2^38*x3^27*x4^10*z^24 + x1^44*x2^39*x3^27*x4^10*z^24 - 3*x1^49*x2^33*x3^28*x4^10*z^24 + x1^48*x2^34*x3^28*x4^10*z^24 - 2*x1^46*x2^36*x3^28*x4^10*z^24 + x1^45*x2^37*x3^28*x4^10*z^24 - x1^44*x2^38*x3^28*x4^10*z^24 + x1^43*x2^39*x3^28*x4^10*z^24 - x1^42*x2^40*x3^28*x4^10*z^24 - x1^41*x2^41*x3^28*x4^10*z^24 + 4*x1^48*x2^33*x3^29*x4^10*z^24 + x1^47*x2^34*x3^29*x4^10*z^24 - x1^46*x2^35*x3^29*x4^10*z^24 + x1^45*x2^36*x3^29*x4^10*z^24 + x1^44*x2^37*x3^29*x4^10*z^24 + x1^42*x2^39*x3^29*x4^10*z^24 - x1^48*x2^32*x3^30*x4^10*z^24 - 2*x1^47*x2^33*x3^30*x4^10*z^24 - x1^44*x2^36*x3^30*x4^10*z^24 + 2*x1^42*x2^38*x3^30*x4^10*z^24 - 2*x1^41*x2^39*x3^30*x4^10*z^24 + x1^47*x2^32*x3^31*x4^10*z^24 + 2*x1^46*x2^33*x3^31*x4^10*z^24 + 2*x1^45*x2^34*x3^31*x4^10*z^24 + x1^44*x2^35*x3^31*x4^10*z^24 - x1^43*x2^36*x3^31*x4^10*z^24 + x1^41*x2^38*x3^31*x4^10*z^24 - x1^45*x2^33*x3^32*x4^10*z^24 - x1^44*x2^34*x3^32*x4^10*z^24 - x1^43*x2^35*x3^32*x4^10*z^24 - 2*x1^54*x2^38*x3^17*x4^11*z^24 + x1^53*x2^39*x3^17*x4^11*z^24 + 2*x1^52*x2^40*x3^17*x4^11*z^24 + 3*x1^53*x2^38*x3^18*x4^11*z^24 - 2*x1^52*x2^39*x3^18*x4^11*z^24 - x1^51*x2^40*x3^18*x4^11*z^24 - 3*x1^53*x2^37*x3^19*x4^11*z^24 - 3*x1^52*x2^38*x3^19*x4^11*z^24 - x1^51*x2^39*x3^19*x4^11*z^24 + x1^49*x2^41*x3^19*x4^11*z^24 + x1^53*x2^36*x3^20*x4^11*z^24 + 3*x1^52*x2^37*x3^20*x4^11*z^24 - 2*x1^51*x2^38*x3^20*x4^11*z^24 - 4*x1^49*x2^40*x3^20*x4^11*z^24 - 2*x1^48*x2^41*x3^20*x4^11*z^24 - 2*x1^47*x2^42*x3^20*x4^11*z^24 - x1^46*x2^43*x3^20*x4^11*z^24 - 4*x1^52*x2^36*x3^21*x4^11*z^24 + x1^50*x2^38*x3^21*x4^11*z^24 + 3*x1^48*x2^40*x3^21*x4^11*z^24 + 3*x1^47*x2^41*x3^21*x4^11*z^24 + 2*x1^46*x2^42*x3^21*x4^11*z^24 + x1^52*x2^35*x3^22*x4^11*z^24 + 4*x1^51*x2^36*x3^22*x4^11*z^24 - 2*x1^50*x2^37*x3^22*x4^11*z^24 + 2*x1^49*x2^38*x3^22*x4^11*z^24 - 4*x1^48*x2^39*x3^22*x4^11*z^24 - 3*x1^46*x2^41*x3^22*x4^11*z^24 + x1^45*x2^42*x3^22*x4^11*z^24 - 4*x1^51*x2^35*x3^23*x4^11*z^24 + x1^49*x2^37*x3^23*x4^11*z^24 + 2*x1^47*x2^39*x3^23*x4^11*z^24 - x1^46*x2^40*x3^23*x4^11*z^24 + 4*x1^45*x2^41*x3^23*x4^11*z^24 - 2*x1^44*x2^42*x3^23*x4^11*z^24 + x1^43*x2^43*x3^23*x4^11*z^24 + x1^51*x2^34*x3^24*x4^11*z^24 + 4*x1^50*x2^35*x3^24*x4^11*z^24 - 2*x1^49*x2^36*x3^24*x4^11*z^24 + 3*x1^48*x2^37*x3^24*x4^11*z^24 - 2*x1^47*x2^38*x3^24*x4^11*z^24 - 4*x1^45*x2^40*x3^24*x4^11*z^24 - 2*x1^44*x2^41*x3^24*x4^11*z^24 + 2*x1^43*x2^42*x3^24*x4^11*z^24 - 2*x1^50*x2^34*x3^25*x4^11*z^24 - 2*x1^49*x2^35*x3^25*x4^11*z^24 + 4*x1^46*x2^38*x3^25*x4^11*z^24 + x1^45*x2^39*x3^25*x4^11*z^24 + 4*x1^44*x2^40*x3^25*x4^11*z^24 - 2*x1^43*x2^41*x3^25*x4^11*z^24 - x1^42*x2^42*x3^25*x4^11*z^24 + x1^50*x2^33*x3^26*x4^11*z^24 + 2*x1^49*x2^34*x3^26*x4^11*z^24 + x1^47*x2^36*x3^26*x4^11*z^24 - 3*x1^46*x2^37*x3^26*x4^11*z^24 + x1^45*x2^38*x3^26*x4^11*z^24 - 2*x1^44*x2^39*x3^26*x4^11*z^24 + 2*x1^42*x2^41*x3^26*x4^11*z^24 + x1^48*x2^34*x3^27*x4^11*z^24 + x1^47*x2^35*x3^27*x4^11*z^24 - x1^46*x2^36*x3^27*x4^11*z^24 + 2*x1^45*x2^37*x3^27*x4^11*z^24 + 3*x1^44*x2^38*x3^27*x4^11*z^24 + 5*x1^43*x2^39*x3^27*x4^11*z^24 - x1^42*x2^40*x3^27*x4^11*z^24 - x1^41*x2^41*x3^27*x4^11*z^24 - 2*x1^47*x2^34*x3^28*x4^11*z^24 + x1^46*x2^35*x3^28*x4^11*z^24 - 3*x1^45*x2^36*x3^28*x4^11*z^24 - x1^44*x2^37*x3^28*x4^11*z^24 - 2*x1^43*x2^38*x3^28*x4^11*z^24 + x1^42*x2^39*x3^28*x4^11*z^24 + x1^41*x2^40*x3^28*x4^11*z^24 - x1^47*x2^33*x3^29*x4^11*z^24 + 2*x1^46*x2^34*x3^29*x4^11*z^24 + 2*x1^42*x2^38*x3^29*x4^11*z^24 + x1^40*x2^40*x3^29*x4^11*z^24 + x1^45*x2^34*x3^30*x4^11*z^24 - x1^43*x2^36*x3^30*x4^11*z^24 + x1^41*x2^38*x3^30*x4^11*z^24 - x1^44*x2^34*x3^31*x4^11*z^24 + x1^43*x2^35*x3^31*x4^11*z^24 - x1^41*x2^37*x3^31*x4^11*z^24 + x1^40*x2^38*x3^31*x4^11*z^24 + x1^54*x2^38*x3^16*x4^12*z^24 - x1^53*x2^38*x3^17*x4^12*z^24 - x1^52*x2^39*x3^17*x4^12*z^24 + 2*x1^53*x2^37*x3^18*x4^12*z^24 - x1^49*x2^41*x3^18*x4^12*z^24 + x1^48*x2^42*x3^18*x4^12*z^24 - 2*x1^52*x2^37*x3^19*x4^12*z^24 + 3*x1^51*x2^38*x3^19*x4^12*z^24 - x1^50*x2^39*x3^19*x4^12*z^24 + x1^49*x2^40*x3^19*x4^12*z^24 + x1^48*x2^41*x3^19*x4^12*z^24 + x1^47*x2^42*x3^19*x4^12*z^24 + 3*x1^52*x2^36*x3^20*x4^12*z^24 - x1^51*x2^37*x3^20*x4^12*z^24 - 5*x1^50*x2^38*x3^20*x4^12*z^24 + 2*x1^49*x2^39*x3^20*x4^12*z^24 - 3*x1^48*x2^40*x3^20*x4^12*z^24 - x1^46*x2^42*x3^20*x4^12*z^24 + x1^45*x2^43*x3^20*x4^12*z^24 - x1^51*x2^36*x3^21*x4^12*z^24 + 5*x1^50*x2^37*x3^21*x4^12*z^24 + 2*x1^49*x2^38*x3^21*x4^12*z^24 + 2*x1^48*x2^39*x3^21*x4^12*z^24 - x1^47*x2^40*x3^21*x4^12*z^24 - x1^45*x2^42*x3^21*x4^12*z^24 - x1^44*x2^43*x3^21*x4^12*z^24 + 2*x1^51*x2^35*x3^22*x4^12*z^24 - 2*x1^50*x2^36*x3^22*x4^12*z^24 - 6*x1^49*x2^37*x3^22*x4^12*z^24 - 4*x1^47*x2^39*x3^22*x4^12*z^24 + 2*x1^46*x2^40*x3^22*x4^12*z^24 - x1^45*x2^41*x3^22*x4^12*z^24 + 3*x1^44*x2^42*x3^22*x4^12*z^24 - 2*x1^50*x2^35*x3^23*x4^12*z^24 + 6*x1^49*x2^36*x3^23*x4^12*z^24 + 3*x1^47*x2^38*x3^23*x4^12*z^24 - 3*x1^44*x2^41*x3^23*x4^12*z^24 - 3*x1^43*x2^42*x3^23*x4^12*z^24 + 2*x1^50*x2^34*x3^24*x4^12*z^24 - x1^49*x2^35*x3^24*x4^12*z^24 - 5*x1^48*x2^36*x3^24*x4^12*z^24 - 5*x1^46*x2^38*x3^24*x4^12*z^24 + 3*x1^45*x2^39*x3^24*x4^12*z^24 - 2*x1^44*x2^40*x3^24*x4^12*z^24 + 6*x1^43*x2^41*x3^24*x4^12*z^24 - x1^50*x2^33*x3^25*x4^12*z^24 - 2*x1^49*x2^34*x3^25*x4^12*z^24 + 6*x1^48*x2^35*x3^25*x4^12*z^24 + x1^47*x2^36*x3^25*x4^12*z^24 + 3*x1^46*x2^37*x3^25*x4^12*z^24 - x1^43*x2^40*x3^25*x4^12*z^24 - 6*x1^42*x2^41*x3^25*x4^12*z^24 - x1^48*x2^34*x3^26*x4^12*z^24 - 6*x1^47*x2^35*x3^26*x4^12*z^24 - 5*x1^45*x2^37*x3^26*x4^12*z^24 + 3*x1^44*x2^38*x3^26*x4^12*z^24 - 2*x1^43*x2^39*x3^26*x4^12*z^24 + 6*x1^42*x2^40*x3^26*x4^12*z^24 + 2*x1^41*x2^41*x3^26*x4^12*z^24 + 3*x1^47*x2^34*x3^27*x4^12*z^24 + 2*x1^46*x2^35*x3^27*x4^12*z^24 + 2*x1^45*x2^36*x3^27*x4^12*z^24 - x1^44*x2^37*x3^27*x4^12*z^24 - x1^43*x2^38*x3^27*x4^12*z^24 - 2*x1^42*x2^39*x3^27*x4^12*z^24 - 6*x1^41*x2^40*x3^27*x4^12*z^24 + x1^47*x2^33*x3^28*x4^12*z^24 - 3*x1^46*x2^34*x3^28*x4^12*z^24 - 3*x1^44*x2^36*x3^28*x4^12*z^24 + x1^43*x2^37*x3^28*x4^12*z^24 - x1^42*x2^38*x3^28*x4^12*z^24 + 5*x1^41*x2^39*x3^28*x4^12*z^24 + 2*x1^40*x2^40*x3^28*x4^12*z^24 - x1^46*x2^33*x3^29*x4^12*z^24 + x1^44*x2^35*x3^29*x4^12*z^24 + x1^43*x2^36*x3^29*x4^12*z^24 - 2*x1^41*x2^38*x3^29*x4^12*z^24 - 3*x1^40*x2^39*x3^29*x4^12*z^24 + x1^45*x2^33*x3^30*x4^12*z^24 - x1^44*x2^34*x3^30*x4^12*z^24 - 2*x1^43*x2^35*x3^30*x4^12*z^24 - 2*x1^42*x2^36*x3^30*x4^12*z^24 + 4*x1^40*x2^38*x3^30*x4^12*z^24 - 2*x1^40*x2^37*x3^31*x4^12*z^24 - x1^39*x2^38*x3^31*x4^12*z^24 + x1^39*x2^37*x3^32*x4^12*z^24 - x1^53*x2^37*x3^17*x4^13*z^24 - x1^52*x2^38*x3^17*x4^13*z^24 + x1^51*x2^39*x3^17*x4^13*z^24 + 2*x1^52*x2^37*x3^18*x4^13*z^24 + x1^51*x2^38*x3^18*x4^13*z^24 - 2*x1^50*x2^39*x3^18*x4^13*z^24 + x1^49*x2^40*x3^18*x4^13*z^24 + x1^47*x2^42*x3^18*x4^13*z^24 - x1^52*x2^36*x3^19*x4^13*z^24 + 3*x1^50*x2^38*x3^19*x4^13*z^24 - x1^49*x2^39*x3^19*x4^13*z^24 - 2*x1^47*x2^41*x3^19*x4^13*z^24 - 2*x1^46*x2^42*x3^19*x4^13*z^24 + x1^51*x2^36*x3^20*x4^13*z^24 - x1^50*x2^37*x3^20*x4^13*z^24 - x1^49*x2^38*x3^20*x4^13*z^24 + x1^48*x2^39*x3^20*x4^13*z^24 + 2*x1^47*x2^40*x3^20*x4^13*z^24 + x1^46*x2^41*x3^20*x4^13*z^24 + x1^45*x2^42*x3^20*x4^13*z^24 - 2*x1^51*x2^35*x3^21*x4^13*z^24 + 2*x1^50*x2^36*x3^21*x4^13*z^24 + 5*x1^49*x2^37*x3^21*x4^13*z^24 - x1^48*x2^38*x3^21*x4^13*z^24 + x1^47*x2^39*x3^21*x4^13*z^24 - 2*x1^46*x2^40*x3^21*x4^13*z^24 - x1^45*x2^41*x3^21*x4^13*z^24 - 4*x1^44*x2^42*x3^21*x4^13*z^24 + x1^50*x2^35*x3^22*x4^13*z^24 - 4*x1^49*x2^36*x3^22*x4^13*z^24 - x1^47*x2^38*x3^22*x4^13*z^24 + 2*x1^45*x2^40*x3^22*x4^13*z^24 + 3*x1^44*x2^41*x3^22*x4^13*z^24 + 4*x1^43*x2^42*x3^22*x4^13*z^24 + 2*x1^49*x2^35*x3^23*x4^13*z^24 + 6*x1^48*x2^36*x3^23*x4^13*z^24 + 4*x1^46*x2^38*x3^23*x4^13*z^24 - 4*x1^45*x2^39*x3^23*x4^13*z^24 - 6*x1^43*x2^41*x3^23*x4^13*z^24 - x1^42*x2^42*x3^23*x4^13*z^24 - 6*x1^48*x2^35*x3^24*x4^13*z^24 - 2*x1^47*x2^36*x3^24*x4^13*z^24 - 2*x1^46*x2^37*x3^24*x4^13*z^24 + 2*x1^44*x2^39*x3^24*x4^13*z^24 + 2*x1^43*x2^40*x3^24*x4^13*z^24 + 6*x1^42*x2^41*x3^24*x4^13*z^24 + x1^49*x2^33*x3^25*x4^13*z^24 + 6*x1^47*x2^35*x3^25*x4^13*z^24 + 4*x1^45*x2^37*x3^25*x4^13*z^24 - 4*x1^44*x2^38*x3^25*x4^13*z^24 - 6*x1^42*x2^40*x3^25*x4^13*z^24 - 2*x1^41*x2^41*x3^25*x4^13*z^24 - x1^48*x2^33*x3^26*x4^13*z^24 - 3*x1^47*x2^34*x3^26*x4^13*z^24 - 4*x1^46*x2^35*x3^26*x4^13*z^24 - 2*x1^45*x2^36*x3^26*x4^13*z^24 + 2*x1^43*x2^38*x3^26*x4^13*z^24 + 2*x1^42*x2^39*x3^26*x4^13*z^24 + 6*x1^41*x2^40*x3^26*x4^13*z^24 + x1^48*x2^32*x3^27*x4^13*z^24 + x1^47*x2^33*x3^27*x4^13*z^24 + 4*x1^46*x2^34*x3^27*x4^13*z^24 + x1^45*x2^35*x3^27*x4^13*z^24 + 2*x1^44*x2^36*x3^27*x4^13*z^24 - 4*x1^43*x2^37*x3^27*x4^13*z^24 - 6*x1^41*x2^39*x3^27*x4^13*z^24 - 2*x1^40*x2^40*x3^27*x4^13*z^24 - x1^47*x2^32*x3^28*x4^13*z^24 - 2*x1^46*x2^33*x3^28*x4^13*z^24 - 2*x1^45*x2^34*x3^28*x4^13*z^24 - 2*x1^44*x2^35*x3^28*x4^13*z^24 + 2*x1^41*x2^38*x3^28*x4^13*z^24 + 6*x1^40*x2^39*x3^28*x4^13*z^24 + 2*x1^45*x2^33*x3^29*x4^13*z^24 + x1^44*x2^34*x3^29*x4^13*z^24 + 3*x1^43*x2^35*x3^29*x4^13*z^24 - 3*x1^42*x2^36*x3^29*x4^13*z^24 - 6*x1^40*x2^38*x3^29*x4^13*z^24 - 2*x1^39*x2^39*x3^29*x4^13*z^24 - x1^44*x2^33*x3^30*x4^13*z^24 - x1^43*x2^34*x3^30*x4^13*z^24 + 4*x1^39*x2^38*x3^30*x4^13*z^24 + x1^42*x2^34*x3^31*x4^13*z^24 + x1^41*x2^35*x3^31*x4^13*z^24 - 2*x1^39*x2^37*x3^31*x4^13*z^24 - x1^38*x2^38*x3^31*x4^13*z^24 + x1^51*x2^37*x3^18*x4^14*z^24 + x1^50*x2^38*x3^18*x4^14*z^24 - 2*x1^49*x2^39*x3^18*x4^14*z^24 + x1^47*x2^41*x3^18*x4^14*z^24 - 2*x1^51*x2^36*x3^19*x4^14*z^24 - x1^48*x2^39*x3^19*x4^14*z^24 - x1^47*x2^40*x3^19*x4^14*z^24 + x1^51*x2^35*x3^20*x4^14*z^24 + x1^50*x2^36*x3^20*x4^14*z^24 + x1^49*x2^37*x3^20*x4^14*z^24 + x1^48*x2^38*x3^20*x4^14*z^24 - x1^47*x2^39*x3^20*x4^14*z^24 - x1^46*x2^40*x3^20*x4^14*z^24 + x1^44*x2^42*x3^20*x4^14*z^24 - x1^51*x2^34*x3^21*x4^14*z^24 - 3*x1^50*x2^35*x3^21*x4^14*z^24 + x1^49*x2^36*x3^21*x4^14*z^24 + x1^48*x2^37*x3^21*x4^14*z^24 + 2*x1^47*x2^38*x3^21*x4^14*z^24 - x1^46*x2^39*x3^21*x4^14*z^24 - 2*x1^43*x2^42*x3^21*x4^14*z^24 + 2*x1^50*x2^34*x3^22*x4^14*z^24 - x1^49*x2^35*x3^22*x4^14*z^24 - x1^48*x2^36*x3^22*x4^14*z^24 + x1^47*x2^37*x3^22*x4^14*z^24 - x1^44*x2^40*x3^22*x4^14*z^24 + 2*x1^43*x2^41*x3^22*x4^14*z^24 + x1^42*x2^42*x3^22*x4^14*z^24 - x1^50*x2^33*x3^23*x4^14*z^24 - 3*x1^49*x2^34*x3^23*x4^14*z^24 - 2*x1^47*x2^36*x3^23*x4^14*z^24 + 2*x1^46*x2^37*x3^23*x4^14*z^24 + x1^44*x2^39*x3^23*x4^14*z^24 - x1^43*x2^40*x3^23*x4^14*z^24 - 2*x1^42*x2^41*x3^23*x4^14*z^24 + 2*x1^49*x2^33*x3^24*x4^14*z^24 - x1^48*x2^34*x3^24*x4^14*z^24 - 2*x1^47*x2^35*x3^24*x4^14*z^24 - x1^45*x2^37*x3^24*x4^14*z^24 + 2*x1^44*x2^38*x3^24*x4^14*z^24 - 2*x1^43*x2^39*x3^24*x4^14*z^24 + 2*x1^42*x2^40*x3^24*x4^14*z^24 - x1^49*x2^32*x3^25*x4^14*z^24 - 2*x1^48*x2^33*x3^25*x4^14*z^24 + 2*x1^47*x2^34*x3^25*x4^14*z^24 - 2*x1^46*x2^35*x3^25*x4^14*z^24 + x1^45*x2^36*x3^25*x4^14*z^24 + 2*x1^43*x2^38*x3^25*x4^14*z^24 + x1^42*x2^39*x3^25*x4^14*z^24 - 2*x1^41*x2^40*x3^25*x4^14*z^24 + 2*x1^48*x2^32*x3^26*x4^14*z^24 + 2*x1^47*x2^33*x3^26*x4^14*z^24 - x1^46*x2^34*x3^26*x4^14*z^24 - 3*x1^44*x2^36*x3^26*x4^14*z^24 - 2*x1^42*x2^38*x3^26*x4^14*z^24 + 2*x1^41*x2^39*x3^26*x4^14*z^24 + x1^40*x2^40*x3^26*x4^14*z^24 - 2*x1^47*x2^32*x3^27*x4^14*z^24 - x1^46*x2^33*x3^27*x4^14*z^24 + x1^45*x2^34*x3^27*x4^14*z^24 + 3*x1^44*x2^35*x3^27*x4^14*z^24 + x1^42*x2^37*x3^27*x4^14*z^24 - 2*x1^40*x2^39*x3^27*x4^14*z^24 + x1^46*x2^32*x3^28*x4^14*z^24 + x1^45*x2^33*x3^28*x4^14*z^24 - x1^43*x2^35*x3^28*x4^14*z^24 + x1^42*x2^36*x3^28*x4^14*z^24 - 2*x1^41*x2^37*x3^28*x4^14*z^24 + 2*x1^40*x2^38*x3^28*x4^14*z^24 + x1^39*x2^39*x3^28*x4^14*z^24 - x1^44*x2^33*x3^29*x4^14*z^24 + x1^43*x2^34*x3^29*x4^14*z^24 + x1^42*x2^35*x3^29*x4^14*z^24 + 2*x1^41*x2^36*x3^29*x4^14*z^24 - 2*x1^39*x2^38*x3^29*x4^14*z^24 - x1^41*x2^35*x3^30*x4^14*z^24 - 2*x1^40*x2^36*x3^30*x4^14*z^24 + x1^39*x2^37*x3^30*x4^14*z^24 - x1^41*x2^34*x3^31*x4^14*z^24 + x1^40*x2^35*x3^31*x4^14*z^24 - x1^38*x2^37*x3^31*x4^14*z^24 - x1^39*x2^35*x3^32*x4^14*z^24 + x1^49*x2^39*x3^17*x4^15*z^24 - 2*x1^48*x2^39*x3^18*x4^15*z^24 - x1^50*x2^36*x3^19*x4^15*z^24 - x1^49*x2^37*x3^19*x4^15*z^24 + 4*x1^48*x2^38*x3^19*x4^15*z^24 + 2*x1^47*x2^39*x3^19*x4^15*z^24 + 3*x1^50*x2^35*x3^20*x4^15*z^24 - x1^49*x2^36*x3^20*x4^15*z^24 - 3*x1^47*x2^38*x3^20*x4^15*z^24 - 2*x1^46*x2^39*x3^20*x4^15*z^24 - 3*x1^45*x2^40*x3^20*x4^15*z^24 - 2*x1^50*x2^34*x3^21*x4^15*z^24 - 2*x1^49*x2^35*x3^21*x4^15*z^24 + 2*x1^47*x2^37*x3^21*x4^15*z^24 + 2*x1^46*x2^38*x3^21*x4^15*z^24 + 4*x1^45*x2^39*x3^21*x4^15*z^24 + 3*x1^44*x2^40*x3^21*x4^15*z^24 + x1^50*x2^33*x3^22*x4^15*z^24 + 5*x1^49*x2^34*x3^22*x4^15*z^24 - x1^48*x2^35*x3^22*x4^15*z^24 + x1^47*x2^36*x3^22*x4^15*z^24 - 4*x1^46*x2^37*x3^22*x4^15*z^24 - 6*x1^44*x2^39*x3^22*x4^15*z^24 - x1^43*x2^40*x3^22*x4^15*z^24 - 4*x1^49*x2^33*x3^23*x4^15*z^24 - 2*x1^48*x2^34*x3^23*x4^15*z^24 - 2*x1^47*x2^35*x3^23*x4^15*z^24 + x1^46*x2^36*x3^23*x4^15*z^24 + 2*x1^45*x2^37*x3^23*x4^15*z^24 + 2*x1^44*x2^38*x3^23*x4^15*z^24 + 6*x1^43*x2^39*x3^23*x4^15*z^24 + x1^49*x2^32*x3^24*x4^15*z^24 + 7*x1^48*x2^33*x3^24*x4^15*z^24 + 4*x1^46*x2^35*x3^24*x4^15*z^24 - 4*x1^45*x2^36*x3^24*x4^15*z^24 - 6*x1^43*x2^38*x3^24*x4^15*z^24 - 2*x1^42*x2^39*x3^24*x4^15*z^24 - 4*x1^48*x2^32*x3^25*x4^15*z^24 - 3*x1^47*x2^33*x3^25*x4^15*z^24 - 2*x1^46*x2^34*x3^25*x4^15*z^24 + 2*x1^44*x2^36*x3^25*x4^15*z^24 + 2*x1^43*x2^37*x3^25*x4^15*z^24 + 6*x1^42*x2^38*x3^25*x4^15*z^24 + 4*x1^47*x2^32*x3^26*x4^15*z^24 + x1^46*x2^33*x3^26*x4^15*z^24 + 4*x1^45*x2^34*x3^26*x4^15*z^24 - 3*x1^44*x2^35*x3^26*x4^15*z^24 - 6*x1^42*x2^37*x3^26*x4^15*z^24 - 2*x1^41*x2^38*x3^26*x4^15*z^24 - x1^46*x2^32*x3^27*x4^15*z^24 - x1^45*x2^33*x3^27*x4^15*z^24 - x1^44*x2^34*x3^27*x4^15*z^24 + 2*x1^42*x2^36*x3^27*x4^15*z^24 + 6*x1^41*x2^37*x3^27*x4^15*z^24 + x1^45*x2^32*x3^28*x4^15*z^24 + 2*x1^44*x2^33*x3^28*x4^15*z^24 - x1^43*x2^34*x3^28*x4^15*z^24 + 2*x1^42*x2^35*x3^28*x4^15*z^24 - 5*x1^41*x2^36*x3^28*x4^15*z^24 - 2*x1^40*x2^37*x3^28*x4^15*z^24 - x1^44*x2^32*x3^29*x4^15*z^24 - x1^42*x2^34*x3^29*x4^15*z^24 + 5*x1^40*x2^36*x3^29*x4^15*z^24 - x1^42*x2^33*x3^30*x4^15*z^24 + 2*x1^41*x2^34*x3^30*x4^15*z^24 - 2*x1^40*x2^35*x3^30*x4^15*z^24 - 2*x1^39*x2^36*x3^30*x4^15*z^24 - x1^40*x2^34*x3^31*x4^15*z^24 + 2*x1^39*x2^35*x3^31*x4^15*z^24 - x1^48*x2^38*x3^18*x4^16*z^24 + 3*x1^47*x2^38*x3^19*x4^16*z^24 + x1^46*x2^39*x3^19*x4^16*z^24 + x1^45*x2^40*x3^19*x4^16*z^24 - 2*x1^47*x2^37*x3^20*x4^16*z^24 - 2*x1^46*x2^38*x3^20*x4^16*z^24 - 2*x1^44*x2^40*x3^20*x4^16*z^24 - x1^49*x2^34*x3^21*x4^16*z^24 + x1^48*x2^35*x3^21*x4^16*z^24 - x1^47*x2^36*x3^21*x4^16*z^24 + 4*x1^46*x2^37*x3^21*x4^16*z^24 + x1^45*x2^38*x3^21*x4^16*z^24 + 3*x1^44*x2^39*x3^21*x4^16*z^24 + 2*x1^43*x2^40*x3^21*x4^16*z^24 + x1^49*x2^33*x3^22*x4^16*z^24 + 2*x1^48*x2^34*x3^22*x4^16*z^24 - 3*x1^46*x2^36*x3^22*x4^16*z^24 - x1^45*x2^37*x3^22*x4^16*z^24 - x1^44*x2^38*x3^22*x4^16*z^24 - 6*x1^43*x2^39*x3^22*x4^16*z^24 - 4*x1^48*x2^33*x3^23*x4^16*z^24 + 4*x1^45*x2^36*x3^23*x4^16*z^24 + 6*x1^43*x2^38*x3^23*x4^16*z^24 + 2*x1^42*x2^39*x3^23*x4^16*z^24 + x1^48*x2^32*x3^24*x4^16*z^24 + 3*x1^47*x2^33*x3^24*x4^16*z^24 + x1^46*x2^34*x3^24*x4^16*z^24 - 2*x1^45*x2^35*x3^24*x4^16*z^24 - 2*x1^44*x2^36*x3^24*x4^16*z^24 - 2*x1^43*x2^37*x3^24*x4^16*z^24 - 6*x1^42*x2^38*x3^24*x4^16*z^24 - 3*x1^47*x2^32*x3^25*x4^16*z^24 - x1^45*x2^34*x3^25*x4^16*z^24 + 4*x1^44*x2^35*x3^25*x4^16*z^24 + 6*x1^42*x2^37*x3^25*x4^16*z^24 + 2*x1^41*x2^38*x3^25*x4^16*z^24 + x1^47*x2^31*x3^26*x4^16*z^24 + x1^46*x2^32*x3^26*x4^16*z^24 + x1^45*x2^33*x3^26*x4^16*z^24 - x1^44*x2^34*x3^26*x4^16*z^24 - x1^43*x2^35*x3^26*x4^16*z^24 - 2*x1^42*x2^36*x3^26*x4^16*z^24 - 6*x1^41*x2^37*x3^26*x4^16*z^24 - x1^46*x2^31*x3^27*x4^16*z^24 - x1^44*x2^33*x3^27*x4^16*z^24 + 4*x1^43*x2^34*x3^27*x4^16*z^24 - x1^42*x2^35*x3^27*x4^16*z^24 + 6*x1^41*x2^36*x3^27*x4^16*z^24 + 2*x1^40*x2^37*x3^27*x4^16*z^24 - x1^43*x2^33*x3^28*x4^16*z^24 - 2*x1^42*x2^34*x3^28*x4^16*z^24 - x1^41*x2^35*x3^28*x4^16*z^24 - 6*x1^40*x2^36*x3^28*x4^16*z^24 - x1^43*x2^32*x3^29*x4^16*z^24 + x1^42*x2^33*x3^29*x4^16*z^24 + 2*x1^40*x2^35*x3^29*x4^16*z^24 + 2*x1^39*x2^36*x3^29*x4^16*z^24 - x1^41*x2^33*x3^30*x4^16*z^24 - x1^40*x2^34*x3^30*x4^16*z^24 - 2*x1^39*x2^35*x3^30*x4^16*z^24 + x1^38*x2^35*x3^31*x4^16*z^24 + x1^45*x2^37*x3^21*x4^17*z^24 + x1^44*x2^38*x3^21*x4^17*z^24 + x1^43*x2^39*x3^21*x4^17*z^24 - 2*x1^45*x2^36*x3^22*x4^17*z^24 - x1^42*x2^39*x3^22*x4^17*z^24 + x1^45*x2^35*x3^23*x4^17*z^24 + 2*x1^42*x2^38*x3^23*x4^17*z^24 + x1^47*x2^32*x3^24*x4^17*z^24 + x1^46*x2^33*x3^24*x4^17*z^24 - 2*x1^44*x2^35*x3^24*x4^17*z^24 - 2*x1^42*x2^37*x3^24*x4^17*z^24 - x1^46*x2^32*x3^25*x4^17*z^24 + x1^44*x2^34*x3^25*x4^17*z^24 + 2*x1^41*x2^37*x3^25*x4^17*z^24 + x1^44*x2^33*x3^26*x4^17*z^24 - x1^43*x2^34*x3^26*x4^17*z^24 - 2*x1^41*x2^36*x3^26*x4^17*z^24 - x1^40*x2^37*x3^26*x4^17*z^24 + x1^41*x2^35*x3^27*x4^17*z^24 + 2*x1^40*x2^36*x3^27*x4^17*z^24 + x1^41*x2^34*x3^28*x4^17*z^24 - x1^40*x2^35*x3^28*x4^17*z^24 - x1^39*x2^36*x3^28*x4^17*z^24 + x1^39*x2^35*x3^29*x4^17*z^24 + 2*x1^53*x2^40*x3^21*x4*z^23 - x1^53*x2^39*x3^22*x4*z^23 + 2*x1^52*x2^39*x3^23*x4*z^23 + x1^54*x2^40*x3^19*x4^2*z^23 - 3*x1^53*x2^40*x3^20*x4^2*z^23 + 2*x1^52*x2^41*x3^20*x4^2*z^23 + 2*x1^53*x2^39*x3^21*x4^2*z^23 + 2*x1^52*x2^40*x3^21*x4^2*z^23 - x1^51*x2^41*x3^21*x4^2*z^23 - 5*x1^52*x2^39*x3^22*x4^2*z^23 + x1^49*x2^42*x3^22*x4^2*z^23 + 2*x1^52*x2^38*x3^23*x4^2*z^23 + 2*x1^51*x2^39*x3^23*x4^2*z^23 + 2*x1^49*x2^41*x3^23*x4^2*z^23 - 3*x1^51*x2^38*x3^24*x4^2*z^23 + x1^50*x2^39*x3^24*x4^2*z^23 - x1^49*x2^40*x3^24*x4^2*z^23 - x1^48*x2^41*x3^24*x4^2*z^23 + x1^50*x2^38*x3^25*x4^2*z^23 + x1^49*x2^39*x3^25*x4^2*z^23 + x1^48*x2^40*x3^25*x4^2*z^23 + x1^53*x2^40*x3^19*x4^3*z^23 - x1^52*x2^40*x3^20*x4^3*z^23 - x1^54*x2^37*x3^21*x4^3*z^23 + 3*x1^52*x2^39*x3^21*x4^3*z^23 + x1^50*x2^41*x3^21*x4^3*z^23 + x1^49*x2^42*x3^21*x4^3*z^23 + x1^53*x2^37*x3^22*x4^3*z^23 - 2*x1^52*x2^38*x3^22*x4^3*z^23 - 2*x1^51*x2^39*x3^22*x4^3*z^23 + x1^50*x2^40*x3^22*x4^3*z^23 - x1^49*x2^41*x3^22*x4^3*z^23 - x1^48*x2^42*x3^22*x4^3*z^23 - x1^52*x2^37*x3^23*x4^3*z^23 + x1^51*x2^38*x3^23*x4^3*z^23 - x1^50*x2^39*x3^23*x4^3*z^23 + x1^49*x2^40*x3^23*x4^3*z^23 - x1^51*x2^37*x3^24*x4^3*z^23 - 2*x1^50*x2^38*x3^24*x4^3*z^23 - 2*x1^48*x2^40*x3^24*x4^3*z^23 + x1^50*x2^37*x3^25*x4^3*z^23 - x1^49*x2^38*x3^25*x4^3*z^23 + x1^48*x2^39*x3^25*x4^3*z^23 + x1^47*x2^40*x3^25*x4^3*z^23 - x1^49*x2^37*x3^26*x4^3*z^23 - x1^48*x2^38*x3^26*x4^3*z^23 - x1^47*x2^39*x3^26*x4^3*z^23 + x1^55*x2^38*x3^18*x4^4*z^23 - 2*x1^54*x2^38*x3^19*x4^4*z^23 + x1^53*x2^39*x3^19*x4^4*z^23 + 2*x1^54*x2^37*x3^20*x4^4*z^23 + 2*x1^53*x2^38*x3^20*x4^4*z^23 - x1^52*x2^39*x3^20*x4^4*z^23 + 2*x1^51*x2^40*x3^20*x4^4*z^23 - 6*x1^53*x2^37*x3^21*x4^4*z^23 - x1^52*x2^38*x3^21*x4^4*z^23 - x1^51*x2^39*x3^21*x4^4*z^23 + x1^49*x2^41*x3^21*x4^4*z^23 + 2*x1^53*x2^36*x3^22*x4^4*z^23 + 5*x1^52*x2^37*x3^22*x4^4*z^23 - 3*x1^51*x2^38*x3^22*x4^4*z^23 + x1^50*x2^39*x3^22*x4^4*z^23 - x1^48*x2^41*x3^22*x4^4*z^23 - 4*x1^52*x2^36*x3^23*x4^4*z^23 - x1^51*x2^37*x3^23*x4^4*z^23 - 3*x1^49*x2^39*x3^23*x4^4*z^23 + x1^48*x2^40*x3^23*x4^4*z^23 + 2*x1^52*x2^35*x3^24*x4^4*z^23 + 2*x1^51*x2^36*x3^24*x4^4*z^23 + 3*x1^49*x2^38*x3^24*x4^4*z^23 - x1^48*x2^39*x3^24*x4^4*z^23 + x1^47*x2^40*x3^24*x4^4*z^23 + x1^46*x2^41*x3^24*x4^4*z^23 - x1^45*x2^42*x3^24*x4^4*z^23 - 2*x1^51*x2^35*x3^25*x4^4*z^23 + x1^50*x2^36*x3^25*x4^4*z^23 - x1^49*x2^37*x3^25*x4^4*z^23 - 2*x1^48*x2^38*x3^25*x4^4*z^23 + x1^47*x2^39*x3^25*x4^4*z^23 + x1^51*x2^34*x3^26*x4^4*z^23 + x1^50*x2^35*x3^26*x4^4*z^23 - x1^49*x2^36*x3^26*x4^4*z^23 + x1^48*x2^37*x3^26*x4^4*z^23 - 2*x1^47*x2^38*x3^26*x4^4*z^23 + x1^49*x2^35*x3^27*x4^4*z^23 + x1^48*x2^36*x3^27*x4^4*z^23 - x1^47*x2^37*x3^27*x4^4*z^23 + x1^46*x2^38*x3^27*x4^4*z^23 - 2*x1^55*x2^38*x3^17*x4^5*z^23 + 3*x1^54*x2^38*x3^18*x4^5*z^23 - 2*x1^53*x2^39*x3^18*x4^5*z^23 - 2*x1^54*x2^37*x3^19*x4^5*z^23 - 3*x1^53*x2^38*x3^19*x4^5*z^23 + x1^52*x2^39*x3^19*x4^5*z^23 - 2*x1^51*x2^40*x3^19*x4^5*z^23 + 5*x1^53*x2^37*x3^20*x4^5*z^23 + x1^52*x2^38*x3^20*x4^5*z^23 - x1^50*x2^40*x3^20*x4^5*z^23 - 2*x1^49*x2^41*x3^20*x4^5*z^23 - 2*x1^53*x2^36*x3^21*x4^5*z^23 - 5*x1^52*x2^37*x3^21*x4^5*z^23 + x1^51*x2^38*x3^21*x4^5*z^23 - 3*x1^50*x2^39*x3^21*x4^5*z^23 + x1^49*x2^40*x3^21*x4^5*z^23 + 2*x1^48*x2^41*x3^21*x4^5*z^23 + 6*x1^52*x2^36*x3^22*x4^5*z^23 + x1^51*x2^37*x3^22*x4^5*z^23 + x1^50*x2^38*x3^22*x4^5*z^23 + 2*x1^49*x2^39*x3^22*x4^5*z^23 - 2*x1^47*x2^41*x3^22*x4^5*z^23 + x1^46*x2^42*x3^22*x4^5*z^23 - 2*x1^52*x2^35*x3^23*x4^5*z^23 - 6*x1^51*x2^36*x3^23*x4^5*z^23 + x1^50*x2^37*x3^23*x4^5*z^23 - 2*x1^49*x2^38*x3^23*x4^5*z^23 - x1^47*x2^40*x3^23*x4^5*z^23 + 4*x1^46*x2^41*x3^23*x4^5*z^23 + 5*x1^51*x2^35*x3^24*x4^5*z^23 + x1^50*x2^36*x3^24*x4^5*z^23 + 3*x1^48*x2^38*x3^24*x4^5*z^23 - x1^47*x2^39*x3^24*x4^5*z^23 - x1^46*x2^40*x3^24*x4^5*z^23 - 2*x1^45*x2^41*x3^24*x4^5*z^23 - 2*x1^51*x2^34*x3^25*x4^5*z^23 - 3*x1^50*x2^35*x3^25*x4^5*z^23 + 2*x1^49*x2^36*x3^25*x4^5*z^23 - 4*x1^48*x2^37*x3^25*x4^5*z^23 + x1^47*x2^38*x3^25*x4^5*z^23 + 3*x1^45*x2^40*x3^25*x4^5*z^23 - x1^43*x2^42*x3^25*x4^5*z^23 + 4*x1^50*x2^34*x3^26*x4^5*z^23 + x1^48*x2^36*x3^26*x4^5*z^23 + 3*x1^47*x2^37*x3^26*x4^5*z^23 - x1^46*x2^38*x3^26*x4^5*z^23 + x1^44*x2^40*x3^26*x4^5*z^23 + x1^43*x2^41*x3^26*x4^5*z^23 - x1^50*x2^33*x3^27*x4^5*z^23 - x1^49*x2^34*x3^27*x4^5*z^23 + 2*x1^48*x2^35*x3^27*x4^5*z^23 - 2*x1^47*x2^36*x3^27*x4^5*z^23 + x1^46*x2^37*x3^27*x4^5*z^23 + x1^49*x2^33*x3^28*x4^5*z^23 - x1^48*x2^34*x3^28*x4^5*z^23 + x1^47*x2^35*x3^28*x4^5*z^23 + 2*x1^46*x2^36*x3^28*x4^5*z^23 - 2*x1^45*x2^37*x3^28*x4^5*z^23 + x1^47*x2^34*x3^29*x4^5*z^23 + x1^46*x2^35*x3^29*x4^5*z^23 + x1^45*x2^36*x3^29*x4^5*z^23 + x1^54*x2^37*x3^18*x4^6*z^23 - 2*x1^53*x2^37*x3^19*x4^6*z^23 + x1^52*x2^38*x3^19*x4^6*z^23 + x1^53*x2^36*x3^20*x4^6*z^23 + 2*x1^52*x2^37*x3^20*x4^6*z^23 - 2*x1^51*x2^38*x3^20*x4^6*z^23 + 3*x1^50*x2^39*x3^20*x4^6*z^23 + x1^49*x2^40*x3^20*x4^6*z^23 - 2*x1^52*x2^36*x3^21*x4^6*z^23 - x1^51*x2^37*x3^21*x4^6*z^23 + x1^50*x2^38*x3^21*x4^6*z^23 - 2*x1^49*x2^39*x3^21*x4^6*z^23 + x1^48*x2^40*x3^21*x4^6*z^23 + x1^47*x2^41*x3^21*x4^6*z^23 + x1^52*x2^35*x3^22*x4^6*z^23 + 2*x1^51*x2^36*x3^22*x4^6*z^23 - 4*x1^50*x2^37*x3^22*x4^6*z^23 - x1^49*x2^38*x3^22*x4^6*z^23 - 4*x1^48*x2^39*x3^22*x4^6*z^23 - x1^46*x2^41*x3^22*x4^6*z^23 - x1^45*x2^42*x3^22*x4^6*z^23 - 2*x1^51*x2^35*x3^23*x4^6*z^23 + 2*x1^50*x2^36*x3^23*x4^6*z^23 + 4*x1^49*x2^37*x3^23*x4^6*z^23 + x1^47*x2^39*x3^23*x4^6*z^23 - x1^46*x2^40*x3^23*x4^6*z^23 + x1^45*x2^41*x3^23*x4^6*z^23 + x1^44*x2^42*x3^23*x4^6*z^23 + 3*x1^50*x2^35*x3^24*x4^6*z^23 - 2*x1^49*x2^36*x3^24*x4^6*z^23 + x1^48*x2^37*x3^24*x4^6*z^23 - 2*x1^47*x2^38*x3^24*x4^6*z^23 - x1^45*x2^40*x3^24*x4^6*z^23 + 2*x1^44*x2^41*x3^24*x4^6*z^23 + x1^43*x2^42*x3^24*x4^6*z^23 - 3*x1^50*x2^34*x3^25*x4^6*z^23 + 3*x1^48*x2^36*x3^25*x4^6*z^23 - x1^47*x2^37*x3^25*x4^6*z^23 + 3*x1^46*x2^38*x3^25*x4^6*z^23 - x1^45*x2^39*x3^25*x4^6*z^23 + x1^44*x2^40*x3^25*x4^6*z^23 + x1^50*x2^33*x3^26*x4^6*z^23 + x1^49*x2^34*x3^26*x4^6*z^23 - 4*x1^48*x2^35*x3^26*x4^6*z^23 + 3*x1^47*x2^36*x3^26*x4^6*z^23 - 2*x1^45*x2^38*x3^26*x4^6*z^23 - 3*x1^44*x2^39*x3^26*x4^6*z^23 + x1^42*x2^41*x3^26*x4^6*z^23 - 2*x1^49*x2^33*x3^27*x4^6*z^23 - 2*x1^46*x2^36*x3^27*x4^6*z^23 + 3*x1^45*x2^37*x3^27*x4^6*z^23 - x1^43*x2^39*x3^27*x4^6*z^23 - x1^42*x2^40*x3^27*x4^6*z^23 - 2*x1^47*x2^34*x3^28*x4^6*z^23 + x1^47*x2^33*x3^29*x4^6*z^23 - x1^45*x2^35*x3^29*x4^6*z^23 + x1^44*x2^36*x3^29*x4^6*z^23 - x1^45*x2^34*x3^30*x4^6*z^23 - x1^44*x2^35*x3^30*x4^6*z^23 + x1^53*x2^39*x3^16*x4^7*z^23 - x1^54*x2^37*x3^17*x4^7*z^23 - 2*x1^52*x2^39*x3^17*x4^7*z^23 - x1^51*x2^40*x3^17*x4^7*z^23 - x1^50*x2^41*x3^17*x4^7*z^23 + x1^52*x2^38*x3^18*x4^7*z^23 + 2*x1^51*x2^39*x3^18*x4^7*z^23 + x1^49*x2^41*x3^18*x4^7*z^23 - x1^48*x2^42*x3^18*x4^7*z^23 - 3*x1^51*x2^38*x3^19*x4^7*z^23 - x1^50*x2^39*x3^19*x4^7*z^23 - x1^48*x2^41*x3^19*x4^7*z^23 + x1^51*x2^37*x3^20*x4^7*z^23 + 3*x1^50*x2^38*x3^20*x4^7*z^23 + x1^49*x2^39*x3^20*x4^7*z^23 + 3*x1^48*x2^40*x3^20*x4^7*z^23 + x1^47*x2^41*x3^20*x4^7*z^23 - x1^45*x2^43*x3^20*x4^7*z^23 - x1^50*x2^37*x3^21*x4^7*z^23 - 2*x1^49*x2^38*x3^21*x4^7*z^23 - x1^48*x2^39*x3^21*x4^7*z^23 - x1^47*x2^40*x3^21*x4^7*z^23 + x1^45*x2^42*x3^21*x4^7*z^23 + x1^44*x2^43*x3^21*x4^7*z^23 + x1^49*x2^37*x3^22*x4^7*z^23 + 2*x1^48*x2^38*x3^22*x4^7*z^23 - x1^44*x2^42*x3^22*x4^7*z^23 + x1^47*x2^38*x3^23*x4^7*z^23 - 3*x1^46*x2^39*x3^23*x4^7*z^23 + x1^44*x2^41*x3^23*x4^7*z^23 - x1^46*x2^38*x3^24*x4^7*z^23 - 2*x1^44*x2^40*x3^24*x4^7*z^23 - 2*x1^43*x2^41*x3^24*x4^7*z^23 + x1^42*x2^42*x3^24*x4^7*z^23 - x1^49*x2^34*x3^25*x4^7*z^23 - x1^48*x2^35*x3^25*x4^7*z^23 + x1^46*x2^37*x3^25*x4^7*z^23 - x1^45*x2^38*x3^25*x4^7*z^23 + x1^44*x2^39*x3^25*x4^7*z^23 - x1^43*x2^40*x3^25*x4^7*z^23 + x1^42*x2^41*x3^25*x4^7*z^23 + 2*x1^49*x2^33*x3^26*x4^7*z^23 - x1^47*x2^35*x3^26*x4^7*z^23 + x1^46*x2^36*x3^26*x4^7*z^23 - x1^45*x2^37*x3^26*x4^7*z^23 - x1^43*x2^39*x3^26*x4^7*z^23 - x1^42*x2^40*x3^26*x4^7*z^23 - x1^49*x2^32*x3^27*x4^7*z^23 - 2*x1^48*x2^33*x3^27*x4^7*z^23 + 2*x1^47*x2^34*x3^27*x4^7*z^23 - x1^45*x2^36*x3^27*x4^7*z^23 + 2*x1^43*x2^38*x3^27*x4^7*z^23 - x1^42*x2^39*x3^27*x4^7*z^23 - x1^47*x2^33*x3^28*x4^7*z^23 + x1^45*x2^35*x3^28*x4^7*z^23 - x1^44*x2^35*x3^29*x4^7*z^23 + x1^44*x2^34*x3^30*x4^7*z^23 + x1^43*x2^34*x3^31*x4^7*z^23 - x1^54*x2^37*x3^16*x4^8*z^23 + 2*x1^52*x2^39*x3^16*x4^8*z^23 - x1^51*x2^40*x3^16*x4^8*z^23 + x1^53*x2^37*x3^17*x4^8*z^23 - x1^52*x2^38*x3^17*x4^8*z^23 - 2*x1^51*x2^39*x3^17*x4^8*z^23 + 2*x1^50*x2^40*x3^17*x4^8*z^23 - x1^49*x2^41*x3^17*x4^8*z^23 + x1^53*x2^36*x3^18*x4^8*z^23 - x1^52*x2^37*x3^18*x4^8*z^23 + 4*x1^51*x2^38*x3^18*x4^8*z^23 - x1^49*x2^40*x3^18*x4^8*z^23 - x1^47*x2^42*x3^18*x4^8*z^23 - x1^52*x2^36*x3^19*x4^8*z^23 + x1^51*x2^37*x3^19*x4^8*z^23 - 3*x1^50*x2^38*x3^19*x4^8*z^23 + x1^49*x2^39*x3^19*x4^8*z^23 - x1^48*x2^40*x3^19*x4^8*z^23 + 2*x1^47*x2^41*x3^19*x4^8*z^23 + x1^46*x2^42*x3^19*x4^8*z^23 + x1^51*x2^36*x3^20*x4^8*z^23 + 2*x1^50*x2^37*x3^20*x4^8*z^23 + 2*x1^49*x2^38*x3^20*x4^8*z^23 - x1^47*x2^40*x3^20*x4^8*z^23 - x1^46*x2^41*x3^20*x4^8*z^23 - x1^45*x2^42*x3^20*x4^8*z^23 - 2*x1^51*x2^35*x3^21*x4^8*z^23 - 3*x1^50*x2^36*x3^21*x4^8*z^23 - 3*x1^49*x2^37*x3^21*x4^8*z^23 + x1^48*x2^38*x3^21*x4^8*z^23 - x1^47*x2^39*x3^21*x4^8*z^23 + 3*x1^46*x2^40*x3^21*x4^8*z^23 + x1^45*x2^41*x3^21*x4^8*z^23 + 4*x1^44*x2^42*x3^21*x4^8*z^23 + x1^51*x2^34*x3^22*x4^8*z^23 + 2*x1^50*x2^35*x3^22*x4^8*z^23 + 4*x1^49*x2^36*x3^22*x4^8*z^23 + x1^48*x2^37*x3^22*x4^8*z^23 - x1^46*x2^39*x3^22*x4^8*z^23 - 2*x1^45*x2^40*x3^22*x4^8*z^23 - 3*x1^44*x2^41*x3^22*x4^8*z^23 - 4*x1^43*x2^42*x3^22*x4^8*z^23 - 2*x1^50*x2^34*x3^23*x4^8*z^23 - 2*x1^49*x2^35*x3^23*x4^8*z^23 - 5*x1^48*x2^36*x3^23*x4^8*z^23 - x1^47*x2^37*x3^23*x4^8*z^23 - 2*x1^46*x2^38*x3^23*x4^8*z^23 + 4*x1^45*x2^39*x3^23*x4^8*z^23 - x1^44*x2^40*x3^23*x4^8*z^23 + 4*x1^43*x2^41*x3^23*x4^8*z^23 + x1^42*x2^42*x3^23*x4^8*z^23 + x1^50*x2^33*x3^24*x4^8*z^23 + 2*x1^49*x2^34*x3^24*x4^8*z^23 + 4*x1^48*x2^35*x3^24*x4^8*z^23 + 2*x1^47*x2^36*x3^24*x4^8*z^23 - x1^46*x2^37*x3^24*x4^8*z^23 + x1^45*x2^38*x3^24*x4^8*z^23 - 2*x1^44*x2^39*x3^24*x4^8*z^23 - x1^43*x2^40*x3^24*x4^8*z^23 - 4*x1^42*x2^41*x3^24*x4^8*z^23 - 2*x1^49*x2^33*x3^25*x4^8*z^23 - 4*x1^47*x2^35*x3^25*x4^8*z^23 - x1^46*x2^36*x3^25*x4^8*z^23 - 2*x1^45*x2^37*x3^25*x4^8*z^23 + x1^44*x2^38*x3^25*x4^8*z^23 + 3*x1^43*x2^39*x3^25*x4^8*z^23 + 4*x1^42*x2^40*x3^25*x4^8*z^23 + x1^49*x2^32*x3^26*x4^8*z^23 + x1^48*x2^33*x3^26*x4^8*z^23 + x1^47*x2^34*x3^26*x4^8*z^23 + 5*x1^46*x2^35*x3^26*x4^8*z^23 + x1^45*x2^36*x3^26*x4^8*z^23 - x1^43*x2^38*x3^26*x4^8*z^23 - x1^42*x2^39*x3^26*x4^8*z^23 - 3*x1^41*x2^40*x3^26*x4^8*z^23 - 2*x1^48*x2^32*x3^27*x4^8*z^23 - x1^47*x2^33*x3^27*x4^8*z^23 - 2*x1^46*x2^34*x3^27*x4^8*z^23 - x1^45*x2^35*x3^27*x4^8*z^23 - 2*x1^44*x2^36*x3^27*x4^8*z^23 + x1^42*x2^38*x3^27*x4^8*z^23 + 3*x1^41*x2^39*x3^27*x4^8*z^23 + 2*x1^47*x2^32*x3^28*x4^8*z^23 + x1^46*x2^33*x3^28*x4^8*z^23 + x1^45*x2^34*x3^28*x4^8*z^23 + 3*x1^44*x2^35*x3^28*x4^8*z^23 + 3*x1^43*x2^36*x3^28*x4^8*z^23 - 2*x1^42*x2^37*x3^28*x4^8*z^23 - x1^40*x2^39*x3^28*x4^8*z^23 - x1^46*x2^32*x3^29*x4^8*z^23 - x1^45*x2^33*x3^29*x4^8*z^23 + x1^42*x2^36*x3^29*x4^8*z^23 + x1^41*x2^37*x3^29*x4^8*z^23 + x1^40*x2^38*x3^29*x4^8*z^23 + x1^45*x2^32*x3^30*x4^8*z^23 + x1^43*x2^34*x3^30*x4^8*z^23 - x1^42*x2^34*x3^31*x4^8*z^23 - x1^52*x2^39*x3^15*x4^9*z^23 + x1^51*x2^39*x3^16*x4^9*z^23 - x1^50*x2^40*x3^16*x4^9*z^23 - x1^52*x2^37*x3^17*x4^9*z^23 - 2*x1^51*x2^38*x3^17*x4^9*z^23 - 2*x1^50*x2^39*x3^17*x4^9*z^23 + x1^52*x2^36*x3^18*x4^9*z^23 - x1^51*x2^37*x3^18*x4^9*z^23 + x1^49*x2^39*x3^18*x4^9*z^23 + x1^48*x2^40*x3^18*x4^9*z^23 - x1^47*x2^41*x3^18*x4^9*z^23 - x1^52*x2^35*x3^19*x4^9*z^23 - 2*x1^51*x2^36*x3^19*x4^9*z^23 - x1^50*x2^37*x3^19*x4^9*z^23 + x1^46*x2^41*x3^19*x4^9*z^23 + x1^45*x2^42*x3^19*x4^9*z^23 + 4*x1^51*x2^35*x3^20*x4^9*z^23 + x1^50*x2^36*x3^20*x4^9*z^23 + 2*x1^49*x2^37*x3^20*x4^9*z^23 + 2*x1^47*x2^39*x3^20*x4^9*z^23 - x1^45*x2^41*x3^20*x4^9*z^23 - 2*x1^44*x2^42*x3^20*x4^9*z^23 - x1^51*x2^34*x3^21*x4^9*z^23 - 4*x1^50*x2^35*x3^21*x4^9*z^23 - 2*x1^49*x2^36*x3^21*x4^9*z^23 - 5*x1^48*x2^37*x3^21*x4^9*z^23 - 2*x1^47*x2^38*x3^21*x4^9*z^23 + x1^46*x2^39*x3^21*x4^9*z^23 + x1^45*x2^40*x3^21*x4^9*z^23 + 2*x1^44*x2^41*x3^21*x4^9*z^23 + 2*x1^43*x2^42*x3^21*x4^9*z^23 + 4*x1^50*x2^34*x3^22*x4^9*z^23 + 3*x1^49*x2^35*x3^22*x4^9*z^23 + 4*x1^48*x2^36*x3^22*x4^9*z^23 - x1^46*x2^38*x3^22*x4^9*z^23 - 5*x1^45*x2^39*x3^22*x4^9*z^23 - 2*x1^43*x2^41*x3^22*x4^9*z^23 - x1^42*x2^42*x3^22*x4^9*z^23 - 2*x1^50*x2^33*x3^23*x4^9*z^23 - 4*x1^49*x2^34*x3^23*x4^9*z^23 - 2*x1^48*x2^35*x3^23*x4^9*z^23 - 2*x1^47*x2^36*x3^23*x4^9*z^23 + 3*x1^46*x2^37*x3^23*x4^9*z^23 + x1^45*x2^38*x3^23*x4^9*z^23 + 3*x1^44*x2^39*x3^23*x4^9*z^23 + 2*x1^42*x2^41*x3^23*x4^9*z^23 + 4*x1^49*x2^33*x3^24*x4^9*z^23 + x1^48*x2^34*x3^24*x4^9*z^23 + 3*x1^47*x2^35*x3^24*x4^9*z^23 + x1^46*x2^36*x3^24*x4^9*z^23 + x1^45*x2^37*x3^24*x4^9*z^23 - x1^44*x2^38*x3^24*x4^9*z^23 - 2*x1^43*x2^39*x3^24*x4^9*z^23 - x1^42*x2^40*x3^24*x4^9*z^23 - x1^49*x2^32*x3^25*x4^9*z^23 - 4*x1^48*x2^33*x3^25*x4^9*z^23 - x1^47*x2^34*x3^25*x4^9*z^23 - 4*x1^46*x2^35*x3^25*x4^9*z^23 + 3*x1^43*x2^38*x3^25*x4^9*z^23 + x1^42*x2^39*x3^25*x4^9*z^23 + 2*x1^41*x2^40*x3^25*x4^9*z^23 + 3*x1^48*x2^32*x3^26*x4^9*z^23 + 2*x1^47*x2^33*x3^26*x4^9*z^23 + 2*x1^46*x2^34*x3^26*x4^9*z^23 + x1^45*x2^35*x3^26*x4^9*z^23 + x1^44*x2^36*x3^26*x4^9*z^23 - 2*x1^42*x2^38*x3^26*x4^9*z^23 - x1^41*x2^39*x3^26*x4^9*z^23 - 2*x1^48*x2^31*x3^27*x4^9*z^23 - 3*x1^47*x2^32*x3^27*x4^9*z^23 + x1^46*x2^33*x3^27*x4^9*z^23 - 3*x1^45*x2^34*x3^27*x4^9*z^23 + x1^43*x2^36*x3^27*x4^9*z^23 + 3*x1^42*x2^37*x3^27*x4^9*z^23 + x1^41*x2^38*x3^27*x4^9*z^23 + 2*x1^40*x2^39*x3^27*x4^9*z^23 + 3*x1^47*x2^31*x3^28*x4^9*z^23 + x1^46*x2^32*x3^28*x4^9*z^23 + x1^45*x2^33*x3^28*x4^9*z^23 - 2*x1^42*x2^36*x3^28*x4^9*z^23 - x1^41*x2^37*x3^28*x4^9*z^23 + x1^40*x2^38*x3^28*x4^9*z^23 - 2*x1^46*x2^31*x3^29*x4^9*z^23 - x1^44*x2^33*x3^29*x4^9*z^23 + x1^43*x2^34*x3^29*x4^9*z^23 - x1^42*x2^35*x3^29*x4^9*z^23 + x1^39*x2^38*x3^29*x4^9*z^23 + x1^45*x2^31*x3^30*x4^9*z^23 + x1^44*x2^32*x3^30*x4^9*z^23 + x1^43*x2^33*x3^30*x4^9*z^23 + x1^41*x2^35*x3^30*x4^9*z^23 - x1^40*x2^36*x3^30*x4^9*z^23 - x1^39*x2^37*x3^30*x4^9*z^23 + x1^42*x2^33*x3^31*x4^9*z^23 + x1^41*x2^34*x3^31*x4^9*z^23 + x1^53*x2^37*x3^15*x4^10*z^23 - x1^52*x2^37*x3^16*x4^10*z^23 + x1^50*x2^39*x3^16*x4^10*z^23 + x1^49*x2^40*x3^16*x4^10*z^23 + x1^52*x2^36*x3^17*x4^10*z^23 - x1^51*x2^37*x3^17*x4^10*z^23 - 2*x1^50*x2^38*x3^17*x4^10*z^23 - x1^48*x2^40*x3^17*x4^10*z^23 - x1^47*x2^41*x3^17*x4^10*z^23 + 2*x1^50*x2^37*x3^18*x4^10*z^23 + x1^48*x2^39*x3^18*x4^10*z^23 + x1^47*x2^40*x3^18*x4^10*z^23 + 2*x1^46*x2^41*x3^18*x4^10*z^23 - x1^49*x2^37*x3^19*x4^10*z^23 - 2*x1^48*x2^38*x3^19*x4^10*z^23 - 3*x1^47*x2^39*x3^19*x4^10*z^23 - 2*x1^46*x2^40*x3^19*x4^10*z^23 - 2*x1^45*x2^41*x3^19*x4^10*z^23 - x1^44*x2^42*x3^19*x4^10*z^23 - x1^49*x2^36*x3^20*x4^10*z^23 + x1^47*x2^38*x3^20*x4^10*z^23 + x1^46*x2^39*x3^20*x4^10*z^23 + 3*x1^45*x2^40*x3^20*x4^10*z^23 + x1^44*x2^41*x3^20*x4^10*z^23 + x1^43*x2^42*x3^20*x4^10*z^23 + x1^48*x2^36*x3^21*x4^10*z^23 - x1^46*x2^38*x3^21*x4^10*z^23 - 2*x1^45*x2^39*x3^21*x4^10*z^23 - 2*x1^44*x2^40*x3^21*x4^10*z^23 - x1^43*x2^41*x3^21*x4^10*z^23 + x1^46*x2^37*x3^22*x4^10*z^23 + 2*x1^44*x2^39*x3^22*x4^10*z^23 + x1^42*x2^41*x3^22*x4^10*z^23 + x1^44*x2^38*x3^23*x4^10*z^23 - 2*x1^43*x2^39*x3^23*x4^10*z^23 + x1^49*x2^32*x3^24*x4^10*z^23 + x1^42*x2^39*x3^24*x4^10*z^23 - 2*x1^48*x2^32*x3^25*x4^10*z^23 + x1^47*x2^33*x3^25*x4^10*z^23 + x1^48*x2^31*x3^26*x4^10*z^23 + 2*x1^47*x2^32*x3^26*x4^10*z^23 - x1^46*x2^33*x3^26*x4^10*z^23 + 2*x1^45*x2^34*x3^26*x4^10*z^23 + x1^44*x2^35*x3^26*x4^10*z^23 + x1^40*x2^39*x3^26*x4^10*z^23 - 2*x1^47*x2^31*x3^27*x4^10*z^23 - 2*x1^46*x2^32*x3^27*x4^10*z^23 + x1^43*x2^35*x3^27*x4^10*z^23 - 2*x1^42*x2^36*x3^27*x4^10*z^23 - 2*x1^40*x2^38*x3^27*x4^10*z^23 + 2*x1^46*x2^31*x3^28*x4^10*z^23 + x1^41*x2^36*x3^28*x4^10*z^23 - x1^40*x2^37*x3^28*x4^10*z^23 + x1^39*x2^38*x3^28*x4^10*z^23 - x1^45*x2^31*x3^29*x4^10*z^23 - 2*x1^44*x2^32*x3^29*x4^10*z^23 - x1^43*x2^33*x3^29*x4^10*z^23 + x1^41*x2^35*x3^29*x4^10*z^23 - 2*x1^39*x2^37*x3^29*x4^10*z^23 + x1^44*x2^31*x3^30*x4^10*z^23 + 2*x1^43*x2^32*x3^30*x4^10*z^23 - x1^43*x2^31*x3^31*x4^10*z^23 - x1^41*x2^33*x3^31*x4^10*z^23 + x1^39*x2^35*x3^31*x4^10*z^23 - x1^38*x2^36*x3^31*x4^10*z^23 - 2*x1^52*x2^36*x3^16*x4^11*z^23 - x1^51*x2^37*x3^16*x4^11*z^23 + x1^50*x2^38*x3^16*x4^11*z^23 + 2*x1^51*x2^36*x3^17*x4^11*z^23 - x1^49*x2^38*x3^17*x4^11*z^23 - 2*x1^48*x2^39*x3^17*x4^11*z^23 - x1^47*x2^40*x3^17*x4^11*z^23 + x1^46*x2^41*x3^17*x4^11*z^23 - 2*x1^51*x2^35*x3^18*x4^11*z^23 - x1^50*x2^36*x3^18*x4^11*z^23 + x1^49*x2^37*x3^18*x4^11*z^23 + 2*x1^48*x2^38*x3^18*x4^11*z^23 + x1^47*x2^39*x3^18*x4^11*z^23 - x1^45*x2^41*x3^18*x4^11*z^23 + 3*x1^50*x2^35*x3^19*x4^11*z^23 - 2*x1^49*x2^36*x3^19*x4^11*z^23 + 2*x1^48*x2^37*x3^19*x4^11*z^23 - 3*x1^45*x2^40*x3^19*x4^11*z^23 + x1^44*x2^41*x3^19*x4^11*z^23 - 3*x1^50*x2^34*x3^20*x4^11*z^23 + x1^48*x2^36*x3^20*x4^11*z^23 + 5*x1^46*x2^38*x3^20*x4^11*z^23 + x1^45*x2^39*x3^20*x4^11*z^23 + 3*x1^44*x2^40*x3^20*x4^11*z^23 + 2*x1^50*x2^33*x3^21*x4^11*z^23 + 4*x1^49*x2^34*x3^21*x4^11*z^23 - 2*x1^48*x2^35*x3^21*x4^11*z^23 + x1^47*x2^36*x3^21*x4^11*z^23 - 4*x1^46*x2^37*x3^21*x4^11*z^23 - 4*x1^44*x2^39*x3^21*x4^11*z^23 - x1^43*x2^40*x3^21*x4^11*z^23 - 4*x1^49*x2^33*x3^22*x4^11*z^23 + x1^47*x2^35*x3^22*x4^11*z^23 + 2*x1^45*x2^37*x3^22*x4^11*z^23 + 4*x1^43*x2^39*x3^22*x4^11*z^23 - 2*x1^42*x2^40*x3^22*x4^11*z^23 - x1^41*x2^41*x3^22*x4^11*z^23 - x1^49*x2^32*x3^23*x4^11*z^23 + 4*x1^48*x2^33*x3^23*x4^11*z^23 - 2*x1^47*x2^34*x3^23*x4^11*z^23 + 2*x1^46*x2^35*x3^23*x4^11*z^23 - 4*x1^45*x2^36*x3^23*x4^11*z^23 - 3*x1^43*x2^38*x3^23*x4^11*z^23 + 2*x1^41*x2^40*x3^23*x4^11*z^23 - x1^48*x2^32*x3^24*x4^11*z^23 - 2*x1^47*x2^33*x3^24*x4^11*z^23 + x1^46*x2^34*x3^24*x4^11*z^23 + 2*x1^44*x2^36*x3^24*x4^11*z^23 - x1^43*x2^37*x3^24*x4^11*z^23 + 4*x1^42*x2^38*x3^24*x4^11*z^23 - 2*x1^41*x2^39*x3^24*x4^11*z^23 + x1^47*x2^32*x3^25*x4^11*z^23 - x1^46*x2^33*x3^25*x4^11*z^23 + x1^45*x2^34*x3^25*x4^11*z^23 - 2*x1^44*x2^35*x3^25*x4^11*z^23 - 4*x1^42*x2^37*x3^25*x4^11*z^23 - 2*x1^41*x2^38*x3^25*x4^11*z^23 + 2*x1^40*x2^39*x3^25*x4^11*z^23 - x1^47*x2^31*x3^26*x4^11*z^23 - 2*x1^44*x2^34*x3^26*x4^11*z^23 + 2*x1^43*x2^35*x3^26*x4^11*z^23 + x1^42*x2^36*x3^26*x4^11*z^23 + 2*x1^41*x2^37*x3^26*x4^11*z^23 - 2*x1^40*x2^38*x3^26*x4^11*z^23 - x1^39*x2^39*x3^26*x4^11*z^23 + x1^46*x2^31*x3^27*x4^11*z^23 - x1^45*x2^32*x3^27*x4^11*z^23 - x1^44*x2^33*x3^27*x4^11*z^23 - 2*x1^43*x2^34*x3^27*x4^11*z^23 + 3*x1^42*x2^35*x3^27*x4^11*z^23 - 3*x1^41*x2^36*x3^27*x4^11*z^23 - x1^40*x2^37*x3^27*x4^11*z^23 + x1^44*x2^32*x3^28*x4^11*z^23 + x1^43*x2^33*x3^28*x4^11*z^23 + 2*x1^42*x2^34*x3^28*x4^11*z^23 + 3*x1^40*x2^36*x3^28*x4^11*z^23 - x1^39*x2^37*x3^28*x4^11*z^23 - x1^38*x2^38*x3^28*x4^11*z^23 + x1^43*x2^32*x3^29*x4^11*z^23 - x1^41*x2^34*x3^29*x4^11*z^23 + 2*x1^40*x2^35*x3^29*x4^11*z^23 + x1^39*x2^36*x3^29*x4^11*z^23 - x1^38*x2^37*x3^29*x4^11*z^23 - x1^41*x2^33*x3^30*x4^11*z^23 - x1^38*x2^36*x3^30*x4^11*z^23 + x1^52*x2^36*x3^15*x4^12*z^23 - x1^51*x2^36*x3^16*x4^12*z^23 - x1^50*x2^37*x3^16*x4^12*z^23 - x1^47*x2^40*x3^16*x4^12*z^23 + x1^51*x2^35*x3^17*x4^12*z^23 - x1^49*x2^37*x3^17*x4^12*z^23 - x1^50*x2^35*x3^18*x4^12*z^23 + x1^49*x2^36*x3^18*x4^12*z^23 - x1^48*x2^37*x3^18*x4^12*z^23 - x1^44*x2^41*x3^18*x4^12*z^23 + 2*x1^50*x2^34*x3^19*x4^12*z^23 - x1^49*x2^35*x3^19*x4^12*z^23 - 4*x1^48*x2^36*x3^19*x4^12*z^23 - 2*x1^46*x2^38*x3^19*x4^12*z^23 + 2*x1^45*x2^39*x3^19*x4^12*z^23 - x1^44*x2^40*x3^19*x4^12*z^23 + x1^43*x2^41*x3^19*x4^12*z^23 - 3*x1^49*x2^34*x3^20*x4^12*z^23 + 4*x1^48*x2^35*x3^20*x4^12*z^23 + x1^47*x2^36*x3^20*x4^12*z^23 + 2*x1^46*x2^37*x3^20*x4^12*z^23 - x1^42*x2^41*x3^20*x4^12*z^23 + x1^49*x2^33*x3^21*x4^12*z^23 - 2*x1^48*x2^34*x3^21*x4^12*z^23 - 6*x1^47*x2^35*x3^21*x4^12*z^23 - 5*x1^45*x2^37*x3^21*x4^12*z^23 + 3*x1^44*x2^38*x3^21*x4^12*z^23 + 4*x1^42*x2^40*x3^21*x4^12*z^23 + x1^41*x2^41*x3^21*x4^12*z^23 - x1^49*x2^32*x3^22*x4^12*z^23 - 2*x1^48*x2^33*x3^22*x4^12*z^23 + 6*x1^47*x2^34*x3^22*x4^12*z^23 + x1^46*x2^35*x3^22*x4^12*z^23 + 4*x1^45*x2^36*x3^22*x4^12*z^23 - 2*x1^42*x2^39*x3^22*x4^12*z^23 - 4*x1^41*x2^40*x3^22*x4^12*z^23 + 2*x1^48*x2^32*x3^23*x4^12*z^23 - 2*x1^47*x2^33*x3^23*x4^12*z^23 - 6*x1^46*x2^34*x3^23*x4^12*z^23 - 4*x1^44*x2^36*x3^23*x4^12*z^23 + 4*x1^43*x2^37*x3^23*x4^12*z^23 - 2*x1^42*x2^38*x3^23*x4^12*z^23 + 6*x1^41*x2^39*x3^23*x4^12*z^23 + 2*x1^40*x2^40*x3^23*x4^12*z^23 + x1^48*x2^31*x3^24*x4^12*z^23 - 2*x1^47*x2^32*x3^24*x4^12*z^23 + 5*x1^46*x2^33*x3^24*x4^12*z^23 + 3*x1^44*x2^35*x3^24*x4^12*z^23 - x1^41*x2^38*x3^24*x4^12*z^23 - 6*x1^40*x2^39*x3^24*x4^12*z^23 + x1^46*x2^32*x3^25*x4^12*z^23 - 4*x1^45*x2^33*x3^25*x4^12*z^23 - x1^44*x2^34*x3^25*x4^12*z^23 - 5*x1^43*x2^35*x3^25*x4^12*z^23 + 3*x1^42*x2^36*x3^25*x4^12*z^23 - 2*x1^41*x2^37*x3^25*x4^12*z^23 + 6*x1^40*x2^38*x3^25*x4^12*z^23 + 2*x1^39*x2^39*x3^25*x4^12*z^23 + 2*x1^45*x2^32*x3^26*x4^12*z^23 + 3*x1^44*x2^33*x3^26*x4^12*z^23 + 4*x1^43*x2^34*x3^26*x4^12*z^23 - x1^40*x2^37*x3^26*x4^12*z^23 - 6*x1^39*x2^38*x3^26*x4^12*z^23 - x1^45*x2^31*x3^27*x4^12*z^23 - 2*x1^44*x2^32*x3^27*x4^12*z^23 - 2*x1^43*x2^33*x3^27*x4^12*z^23 - 3*x1^42*x2^34*x3^27*x4^12*z^23 + 3*x1^41*x2^35*x3^27*x4^12*z^23 + 6*x1^39*x2^37*x3^27*x4^12*z^23 + 2*x1^38*x2^38*x3^27*x4^12*z^23 - x1^41*x2^34*x3^28*x4^12*z^23 - 2*x1^39*x2^36*x3^28*x4^12*z^23 - 4*x1^38*x2^37*x3^28*x4^12*z^23 - x1^39*x2^35*x3^29*x4^12*z^23 + 4*x1^38*x2^36*x3^29*x4^12*z^23 + x1^37*x2^37*x3^29*x4^12*z^23 + 2*x1^39*x2^34*x3^30*x4^12*z^23 + x1^38*x2^35*x3^30*x4^12*z^23 - 2*x1^37*x2^36*x3^30*x4^12*z^23 + x1^37*x2^35*x3^31*x4^12*z^23 + x1^48*x2^38*x3^16*x4^13*z^23 + x1^47*x2^39*x3^16*x4^13*z^23 + 2*x1^50*x2^35*x3^17*x4^13*z^23 + x1^49*x2^36*x3^17*x4^13*z^23 + x1^45*x2^40*x3^17*x4^13*z^23 + x1^47*x2^37*x3^18*x4^13*z^23 + x1^46*x2^38*x3^18*x4^13*z^23 - x1^45*x2^39*x3^18*x4^13*z^23 - 2*x1^44*x2^40*x3^18*x4^13*z^23 - x1^43*x2^41*x3^18*x4^13*z^23 + 2*x1^49*x2^34*x3^19*x4^13*z^23 - x1^48*x2^35*x3^19*x4^13*z^23 - 2*x1^47*x2^36*x3^19*x4^13*z^23 + x1^46*x2^37*x3^19*x4^13*z^23 + x1^44*x2^39*x3^19*x4^13*z^23 + 3*x1^43*x2^40*x3^19*x4^13*z^23 + x1^42*x2^41*x3^19*x4^13*z^23 - x1^49*x2^33*x3^20*x4^13*z^23 + x1^48*x2^34*x3^20*x4^13*z^23 + 3*x1^47*x2^35*x3^20*x4^13*z^23 - x1^46*x2^36*x3^20*x4^13*z^23 - 4*x1^44*x2^38*x3^20*x4^13*z^23 - x1^43*x2^39*x3^20*x4^13*z^23 - 5*x1^42*x2^40*x3^20*x4^13*z^23 + 2*x1^48*x2^33*x3^21*x4^13*z^23 - 3*x1^47*x2^34*x3^21*x4^13*z^23 - x1^46*x2^35*x3^21*x4^13*z^23 - x1^45*x2^36*x3^21*x4^13*z^23 + x1^44*x2^37*x3^21*x4^13*z^23 + 2*x1^43*x2^38*x3^21*x4^13*z^23 + 2*x1^42*x2^39*x3^21*x4^13*z^23 + 5*x1^41*x2^40*x3^21*x4^13*z^23 - x1^48*x2^32*x3^22*x4^13*z^23 + 2*x1^47*x2^33*x3^22*x4^13*z^23 + 5*x1^46*x2^34*x3^22*x4^13*z^23 - x1^45*x2^35*x3^22*x4^13*z^23 + 2*x1^44*x2^36*x3^22*x4^13*z^23 - 4*x1^43*x2^37*x3^22*x4^13*z^23 - 6*x1^41*x2^39*x3^22*x4^13*z^23 - 2*x1^40*x2^40*x3^22*x4^13*z^23 - x1^48*x2^31*x3^23*x4^13*z^23 - 6*x1^46*x2^33*x3^23*x4^13*z^23 - 2*x1^45*x2^34*x3^23*x4^13*z^23 - 2*x1^44*x2^35*x3^23*x4^13*z^23 + 2*x1^42*x2^37*x3^23*x4^13*z^23 + 2*x1^41*x2^38*x3^23*x4^13*z^23 + 6*x1^40*x2^39*x3^23*x4^13*z^23 + x1^46*x2^32*x3^24*x4^13*z^23 + 6*x1^45*x2^33*x3^24*x4^13*z^23 + 4*x1^43*x2^35*x3^24*x4^13*z^23 - 4*x1^42*x2^36*x3^24*x4^13*z^23 - 6*x1^40*x2^38*x3^24*x4^13*z^23 - 2*x1^39*x2^39*x3^24*x4^13*z^23 - 3*x1^45*x2^32*x3^25*x4^13*z^23 - 2*x1^44*x2^33*x3^25*x4^13*z^23 + 2*x1^41*x2^36*x3^25*x4^13*z^23 + 2*x1^40*x2^37*x3^25*x4^13*z^23 + 6*x1^39*x2^38*x3^25*x4^13*z^23 + x1^45*x2^31*x3^26*x4^13*z^23 + 3*x1^44*x2^32*x3^26*x4^13*z^23 + 2*x1^43*x2^33*x3^26*x4^13*z^23 + 3*x1^42*x2^34*x3^26*x4^13*z^23 - 2*x1^41*x2^35*x3^26*x4^13*z^23 - 6*x1^39*x2^37*x3^26*x4^13*z^23 - 2*x1^38*x2^38*x3^26*x4^13*z^23 - x1^44*x2^31*x3^27*x4^13*z^23 - 2*x1^43*x2^32*x3^27*x4^13*z^23 - 2*x1^42*x2^33*x3^27*x4^13*z^23 + x1^40*x2^35*x3^27*x4^13*z^23 + 2*x1^39*x2^36*x3^27*x4^13*z^23 + 6*x1^38*x2^37*x3^27*x4^13*z^23 + x1^43*x2^31*x3^28*x4^13*z^23 + 2*x1^42*x2^32*x3^28*x4^13*z^23 + x1^41*x2^33*x3^28*x4^13*z^23 - 2*x1^40*x2^34*x3^28*x4^13*z^23 + x1^39*x2^35*x3^28*x4^13*z^23 - 4*x1^38*x2^36*x3^28*x4^13*z^23 - 2*x1^37*x2^37*x3^28*x4^13*z^23 - x1^41*x2^32*x3^29*x4^13*z^23 - 2*x1^40*x2^33*x3^29*x4^13*z^23 - x1^39*x2^34*x3^29*x4^13*z^23 + 4*x1^37*x2^36*x3^29*x4^13*z^23 - x1^37*x2^35*x3^30*x4^13*z^23 - x1^36*x2^36*x3^30*x4^13*z^23 - x1^37*x2^34*x3^31*x4^13*z^23 + x1^36*x2^35*x3^31*x4^13*z^23 - 2*x1^47*x2^37*x3^17*x4^14*z^23 - x1^46*x2^38*x3^17*x4^14*z^23 - x1^49*x2^34*x3^18*x4^14*z^23 + x1^48*x2^35*x3^18*x4^14*z^23 - x1^47*x2^36*x3^18*x4^14*z^23 + x1^46*x2^37*x3^18*x4^14*z^23 + 2*x1^45*x2^38*x3^18*x4^14*z^23 + x1^48*x2^34*x3^19*x4^14*z^23 + x1^47*x2^35*x3^19*x4^14*z^23 - 2*x1^46*x2^36*x3^19*x4^14*z^23 + 3*x1^44*x2^38*x3^19*x4^14*z^23 + x1^43*x2^39*x3^19*x4^14*z^23 + x1^42*x2^40*x3^19*x4^14*z^23 - 3*x1^48*x2^33*x3^20*x4^14*z^23 + x1^47*x2^34*x3^20*x4^14*z^23 - x1^44*x2^37*x3^20*x4^14*z^23 + 3*x1^43*x2^38*x3^20*x4^14*z^23 + x1^42*x2^39*x3^20*x4^14*z^23 - x1^41*x2^40*x3^20*x4^14*z^23 + x1^48*x2^32*x3^21*x4^14*z^23 + 2*x1^47*x2^33*x3^21*x4^14*z^23 + x1^46*x2^34*x3^21*x4^14*z^23 - 2*x1^44*x2^36*x3^21*x4^14*z^23 - 2*x1^42*x2^38*x3^21*x4^14*z^23 + 2*x1^41*x2^39*x3^21*x4^14*z^23 + x1^40*x2^40*x3^21*x4^14*z^23 - x1^48*x2^31*x3^22*x4^14*z^23 - 3*x1^47*x2^32*x3^22*x4^14*z^23 + x1^46*x2^33*x3^22*x4^14*z^23 + x1^45*x2^34*x3^22*x4^14*z^23 + 2*x1^44*x2^35*x3^22*x4^14*z^23 - x1^43*x2^36*x3^22*x4^14*z^23 + x1^42*x2^37*x3^22*x4^14*z^23 - 2*x1^40*x2^39*x3^22*x4^14*z^23 + 3*x1^47*x2^31*x3^23*x4^14*z^23 + x1^46*x2^32*x3^23*x4^14*z^23 + x1^44*x2^34*x3^23*x4^14*z^23 - 2*x1^41*x2^37*x3^23*x4^14*z^23 + 2*x1^40*x2^38*x3^23*x4^14*z^23 + x1^39*x2^39*x3^23*x4^14*z^23 - 3*x1^46*x2^31*x3^24*x4^14*z^23 + x1^45*x2^32*x3^24*x4^14*z^23 + 3*x1^43*x2^34*x3^24*x4^14*z^23 + x1^41*x2^36*x3^24*x4^14*z^23 - x1^40*x2^37*x3^24*x4^14*z^23 - 2*x1^39*x2^38*x3^24*x4^14*z^23 + x1^46*x2^30*x3^25*x4^14*z^23 + 2*x1^45*x2^31*x3^25*x4^14*z^23 - x1^43*x2^33*x3^25*x4^14*z^23 - x1^42*x2^34*x3^25*x4^14*z^23 + 2*x1^41*x2^35*x3^25*x4^14*z^23 - 2*x1^40*x2^36*x3^25*x4^14*z^23 + 2*x1^39*x2^37*x3^25*x4^14*z^23 - x1^45*x2^30*x3^26*x4^14*z^23 - 2*x1^44*x2^31*x3^26*x4^14*z^23 - 2*x1^43*x2^32*x3^26*x4^14*z^23 + x1^42*x2^33*x3^26*x4^14*z^23 + 2*x1^40*x2^35*x3^26*x4^14*z^23 + x1^39*x2^36*x3^26*x4^14*z^23 - 2*x1^38*x2^37*x3^26*x4^14*z^23 + 2*x1^43*x2^31*x3^27*x4^14*z^23 + x1^42*x2^32*x3^27*x4^14*z^23 - x1^41*x2^33*x3^27*x4^14*z^23 - x1^40*x2^34*x3^27*x4^14*z^23 - 2*x1^39*x2^35*x3^27*x4^14*z^23 + 2*x1^38*x2^36*x3^27*x4^14*z^23 + x1^37*x2^37*x3^27*x4^14*z^23 - x1^42*x2^31*x3^28*x4^14*z^23 - x1^40*x2^33*x3^28*x4^14*z^23 + x1^39*x2^34*x3^28*x4^14*z^23 - 2*x1^37*x2^36*x3^28*x4^14*z^23 + x1^39*x2^33*x3^29*x4^14*z^23 - 2*x1^38*x2^34*x3^29*x4^14*z^23 + x1^36*x2^36*x3^29*x4^14*z^23 + x1^47*x2^37*x3^16*x4^15*z^23 - 2*x1^46*x2^37*x3^17*x4^15*z^23 + 2*x1^46*x2^36*x3^18*x4^15*z^23 + 2*x1^45*x2^37*x3^18*x4^15*z^23 + x1^44*x2^38*x3^18*x4^15*z^23 + x1^43*x2^39*x3^18*x4^15*z^23 + x1^48*x2^33*x3^19*x4^15*z^23 - x1^47*x2^34*x3^19*x4^15*z^23 + x1^46*x2^35*x3^19*x4^15*z^23 - 4*x1^45*x2^36*x3^19*x4^15*z^23 - 2*x1^44*x2^37*x3^19*x4^15*z^23 - 3*x1^43*x2^38*x3^19*x4^15*z^23 - x1^48*x2^32*x3^20*x4^15*z^23 - 2*x1^47*x2^33*x3^20*x4^15*z^23 + 3*x1^45*x2^35*x3^20*x4^15*z^23 + x1^44*x2^36*x3^20*x4^15*z^23 + x1^43*x2^37*x3^20*x4^15*z^23 + 5*x1^42*x2^38*x3^20*x4^15*z^23 + 4*x1^47*x2^32*x3^21*x4^15*z^23 - 4*x1^44*x2^35*x3^21*x4^15*z^23 - 6*x1^42*x2^37*x3^21*x4^15*z^23 - 2*x1^41*x2^38*x3^21*x4^15*z^23 - 3*x1^47*x2^31*x3^22*x4^15*z^23 - 3*x1^46*x2^32*x3^22*x4^15*z^23 - x1^45*x2^33*x3^22*x4^15*z^23 + 2*x1^44*x2^34*x3^22*x4^15*z^23 + 2*x1^43*x2^35*x3^22*x4^15*z^23 + 2*x1^42*x2^36*x3^22*x4^15*z^23 + 6*x1^41*x2^37*x3^22*x4^15*z^23 + x1^47*x2^30*x3^23*x4^15*z^23 + 4*x1^46*x2^31*x3^23*x4^15*z^23 + 3*x1^44*x2^33*x3^23*x4^15*z^23 - 4*x1^43*x2^34*x3^23*x4^15*z^23 - 6*x1^41*x2^36*x3^23*x4^15*z^23 - 2*x1^40*x2^37*x3^23*x4^15*z^23 - 2*x1^46*x2^30*x3^24*x4^15*z^23 - 2*x1^45*x2^31*x3^24*x4^15*z^23 - 3*x1^44*x2^32*x3^24*x4^15*z^23 + 2*x1^42*x2^34*x3^24*x4^15*z^23 + 2*x1^41*x2^35*x3^24*x4^15*z^23 + 6*x1^40*x2^36*x3^24*x4^15*z^23 + 2*x1^45*x2^30*x3^25*x4^15*z^23 + x1^44*x2^31*x3^25*x4^15*z^23 + 3*x1^43*x2^32*x3^25*x4^15*z^23 - 3*x1^42*x2^33*x3^25*x4^15*z^23 - 6*x1^40*x2^35*x3^25*x4^15*z^23 - 2*x1^39*x2^36*x3^25*x4^15*z^23 - x1^44*x2^30*x3^26*x4^15*z^23 - x1^43*x2^31*x3^26*x4^15*z^23 + x1^41*x2^33*x3^26*x4^15*z^23 + x1^40*x2^34*x3^26*x4^15*z^23 + 6*x1^39*x2^35*x3^26*x4^15*z^23 - x1^42*x2^31*x3^27*x4^15*z^23 - 3*x1^41*x2^32*x3^27*x4^15*z^23 + x1^40*x2^33*x3^27*x4^15*z^23 - 4*x1^39*x2^34*x3^27*x4^15*z^23 - 2*x1^38*x2^35*x3^27*x4^15*z^23 - x1^39*x2^33*x3^28*x4^15*z^23 + 4*x1^38*x2^34*x3^28*x4^15*z^23 + x1^39*x2^32*x3^29*x4^15*z^23 - x1^38*x2^33*x3^29*x4^15*z^23 - x1^37*x2^34*x3^29*x4^15*z^23 + x1^37*x2^33*x3^30*x4^15*z^23 + 2*x1^45*x2^36*x3^18*x4^16*z^23 - x1^45*x2^35*x3^19*x4^16*z^23 - x1^44*x2^36*x3^19*x4^16*z^23 - x1^43*x2^37*x3^19*x4^16*z^23 - 3*x1^42*x2^38*x3^19*x4^16*z^23 + 4*x1^44*x2^35*x3^20*x4^16*z^23 + 2*x1^42*x2^37*x3^20*x4^16*z^23 + x1^41*x2^38*x3^20*x4^16*z^23 + x1^47*x2^31*x3^21*x4^16*z^23 + x1^46*x2^32*x3^21*x4^16*z^23 - x1^45*x2^33*x3^21*x4^16*z^23 - 2*x1^44*x2^34*x3^21*x4^16*z^23 - x1^43*x2^35*x3^21*x4^16*z^23 - x1^42*x2^36*x3^21*x4^16*z^23 - 5*x1^41*x2^37*x3^21*x4^16*z^23 - 2*x1^46*x2^31*x3^22*x4^16*z^23 - x1^44*x2^33*x3^22*x4^16*z^23 + 5*x1^43*x2^34*x3^22*x4^16*z^23 + 4*x1^41*x2^36*x3^22*x4^16*z^23 + 2*x1^40*x2^37*x3^22*x4^16*z^23 + x1^46*x2^30*x3^23*x4^16*z^23 + 2*x1^45*x2^31*x3^23*x4^16*z^23 - x1^44*x2^32*x3^23*x4^16*z^23 - 2*x1^43*x2^33*x3^23*x4^16*z^23 - 2*x1^42*x2^34*x3^23*x4^16*z^23 - 2*x1^41*x2^35*x3^23*x4^16*z^23 - 6*x1^40*x2^36*x3^23*x4^16*z^23 - x1^45*x2^30*x3^24*x4^16*z^23 - x1^43*x2^32*x3^24*x4^16*z^23 + 3*x1^42*x2^33*x3^24*x4^16*z^23 + 6*x1^40*x2^35*x3^24*x4^16*z^23 + 2*x1^39*x2^36*x3^24*x4^16*z^23 + x1^44*x2^30*x3^25*x4^16*z^23 + x1^43*x2^31*x3^25*x4^16*z^23 - 2*x1^42*x2^32*x3^25*x4^16*z^23 - 2*x1^41*x2^33*x3^25*x4^16*z^23 - 2*x1^40*x2^34*x3^25*x4^16*z^23 - 6*x1^39*x2^35*x3^25*x4^16*z^23 - x1^43*x2^30*x3^26*x4^16*z^23 + 3*x1^41*x2^32*x3^26*x4^16*z^23 + 5*x1^39*x2^34*x3^26*x4^16*z^23 + 2*x1^38*x2^35*x3^26*x4^16*z^23 + x1^42*x2^30*x3^27*x4^16*z^23 - x1^40*x2^32*x3^27*x4^16*z^23 - 5*x1^38*x2^34*x3^27*x4^16*z^23 + x1^40*x2^31*x3^28*x4^16*z^23 + 2*x1^38*x2^33*x3^28*x4^16*z^23 + 2*x1^37*x2^34*x3^28*x4^16*z^23 + x1^38*x2^32*x3^29*x4^16*z^23 - 2*x1^37*x2^33*x3^29*x4^16*z^23 + x1^36*x2^33*x3^30*x4^16*z^23 - x1^43*x2^34*x3^21*x4^17*z^23 + x1^42*x2^35*x3^21*x4^17*z^23 - x1^41*x2^36*x3^21*x4^17*z^23 - x1^40*x2^37*x3^21*x4^17*z^23 + x1^42*x2^34*x3^22*x4^17*z^23 + x1^41*x2^35*x3^22*x4^17*z^23 + x1^40*x2^36*x3^22*x4^17*z^23 - 2*x1^42*x2^33*x3^23*x4^17*z^23 - x1^39*x2^36*x3^23*x4^17*z^23 - x1^43*x2^31*x3^24*x4^17*z^23 + x1^40*x2^34*x3^24*x4^17*z^23 + 2*x1^39*x2^35*x3^24*x4^17*z^23 + x1^42*x2^31*x3^25*x4^17*z^23 + x1^40*x2^33*x3^25*x4^17*z^23 - 2*x1^39*x2^34*x3^25*x4^17*z^23 - x1^39*x2^33*x3^26*x4^17*z^23 + 2*x1^38*x2^34*x3^26*x4^17*z^23 - x1^37*x2^34*x3^27*x4^17*z^23 + x1^52*x2^38*x3^20*z^22 - x1^52*x2^38*x3^19*x4*z^22 + x1^50*x2^40*x3^19*x4*z^22 + 2*x1^51*x2^38*x3^20*x4*z^22 - x1^50*x2^39*x3^20*x4*z^22 - 2*x1^51*x2^37*x3^21*x4*z^22 - x1^50*x2^38*x3^21*x4*z^22 - x1^48*x2^40*x3^21*x4*z^22 + 3*x1^50*x2^37*x3^22*x4*z^22 - x1^49*x2^38*x3^22*x4*z^22 - x1^49*x2^37*x3^23*x4*z^22 - x1^48*x2^38*x3^23*x4*z^22 - x1^47*x2^39*x3^23*x4*z^22 + 2*x1^52*x2^38*x3^18*x4^2*z^22 - 4*x1^51*x2^38*x3^19*x4^2*z^22 - x1^49*x2^40*x3^19*x4^2*z^22 + 2*x1^51*x2^37*x3^20*x4^2*z^22 + 5*x1^50*x2^38*x3^20*x4^2*z^22 - 2*x1^49*x2^39*x3^20*x4^2*z^22 - x1^47*x2^41*x3^20*x4^2*z^22 - 5*x1^50*x2^37*x3^21*x4^2*z^22 - x1^49*x2^38*x3^21*x4^2*z^22 - 2*x1^48*x2^39*x3^21*x4^2*z^22 + x1^46*x2^41*x3^21*x4^2*z^22 + 2*x1^50*x2^36*x3^22*x4^2*z^22 + 4*x1^49*x2^37*x3^22*x4^2*z^22 + 3*x1^47*x2^39*x3^22*x4^2*z^22 - 4*x1^49*x2^36*x3^23*x4^2*z^22 - 2*x1^47*x2^38*x3^23*x4^2*z^22 - x1^46*x2^39*x3^23*x4^2*z^22 + x1^49*x2^35*x3^24*x4^2*z^22 + 2*x1^48*x2^36*x3^24*x4^2*z^22 + 2*x1^46*x2^38*x3^24*x4^2*z^22 - x1^48*x2^35*x3^25*x4^2*z^22 + x1^47*x2^36*x3^25*x4^2*z^22 - x1^46*x2^37*x3^25*x4^2*z^22 - x1^45*x2^38*x3^25*x4^2*z^22 - x1^52*x2^38*x3^17*x4^3*z^22 + x1^51*x2^38*x3^18*x4^3*z^22 + x1^52*x2^36*x3^19*x4^3*z^22 - x1^51*x2^37*x3^19*x4^3*z^22 - x1^50*x2^38*x3^19*x4^3*z^22 - x1^48*x2^40*x3^19*x4^3*z^22 - x1^52*x2^35*x3^20*x4^3*z^22 - x1^51*x2^36*x3^20*x4^3*z^22 + 3*x1^50*x2^37*x3^20*x4^3*z^22 + x1^47*x2^40*x3^20*x4^3*z^22 + x1^51*x2^35*x3^21*x4^3*z^22 - 3*x1^49*x2^37*x3^21*x4^3*z^22 + x1^48*x2^38*x3^21*x4^3*z^22 - x1^47*x2^39*x3^21*x4^3*z^22 - x1^46*x2^40*x3^21*x4^3*z^22 - x1^45*x2^41*x3^21*x4^3*z^22 - x1^51*x2^34*x3^22*x4^3*z^22 - x1^50*x2^35*x3^22*x4^3*z^22 + 2*x1^49*x2^36*x3^22*x4^3*z^22 + x1^48*x2^37*x3^22*x4^3*z^22 + x1^47*x2^38*x3^22*x4^3*z^22 - x1^46*x2^39*x3^22*x4^3*z^22 + x1^50*x2^34*x3^23*x4^3*z^22 - 2*x1^49*x2^35*x3^23*x4^3*z^22 - 2*x1^48*x2^36*x3^23*x4^3*z^22 + 2*x1^47*x2^37*x3^23*x4^3*z^22 - x1^46*x2^38*x3^23*x4^3*z^22 - x1^45*x2^39*x3^23*x4^3*z^22 + x1^43*x2^41*x3^23*x4^3*z^22 + 2*x1^48*x2^35*x3^24*x4^3*z^22 + 2*x1^46*x2^37*x3^24*x4^3*z^22 + x1^45*x2^38*x3^24*x4^3*z^22 - x1^48*x2^34*x3^25*x4^3*z^22 - x1^47*x2^35*x3^25*x4^3*z^22 + x1^46*x2^36*x3^25*x4^3*z^22 - x1^45*x2^37*x3^25*x4^3*z^22 + x1^47*x2^34*x3^26*x4^3*z^22 - x1^46*x2^35*x3^26*x4^3*z^22 + x1^45*x2^36*x3^26*x4^3*z^22 + x1^44*x2^37*x3^26*x4^3*z^22 - x1^53*x2^37*x3^16*x4^4*z^22 + x1^53*x2^36*x3^17*x4^4*z^22 + x1^52*x2^37*x3^17*x4^4*z^22 - x1^51*x2^38*x3^17*x4^4*z^22 - 4*x1^52*x2^36*x3^18*x4^4*z^22 - x1^49*x2^39*x3^18*x4^4*z^22 + 2*x1^52*x2^35*x3^19*x4^4*z^22 + 4*x1^51*x2^36*x3^19*x4^4*z^22 - 2*x1^50*x2^37*x3^19*x4^4*z^22 - 6*x1^51*x2^35*x3^20*x4^4*z^22 - x1^50*x2^36*x3^20*x4^4*z^22 - x1^48*x2^38*x3^20*x4^4*z^22 - x1^46*x2^40*x3^20*x4^4*z^22 + 2*x1^51*x2^34*x3^21*x4^4*z^22 + 6*x1^50*x2^35*x3^21*x4^4*z^22 + 4*x1^48*x2^37*x3^21*x4^4*z^22 + x1^47*x2^38*x3^21*x4^4*z^22 - x1^46*x2^39*x3^21*x4^4*z^22 - 2*x1^45*x2^40*x3^21*x4^4*z^22 - 5*x1^50*x2^34*x3^22*x4^4*z^22 - 2*x1^49*x2^35*x3^22*x4^4*z^22 - x1^48*x2^36*x3^22*x4^4*z^22 - x1^47*x2^37*x3^22*x4^4*z^22 + x1^46*x2^38*x3^22*x4^4*z^22 + x1^45*x2^39*x3^22*x4^4*z^22 - x1^43*x2^41*x3^22*x4^4*z^22 + 2*x1^50*x2^33*x3^23*x4^4*z^22 + 3*x1^49*x2^34*x3^23*x4^4*z^22 - x1^48*x2^35*x3^23*x4^4*z^22 + 3*x1^47*x2^36*x3^23*x4^4*z^22 - 2*x1^44*x2^39*x3^23*x4^4*z^22 - 4*x1^49*x2^33*x3^24*x4^4*z^22 - x1^47*x2^35*x3^24*x4^4*z^22 - 3*x1^46*x2^36*x3^24*x4^4*z^22 + x1^45*x2^37*x3^24*x4^4*z^22 - x1^43*x2^39*x3^24*x4^4*z^22 - x1^42*x2^40*x3^24*x4^4*z^22 + 2*x1^49*x2^32*x3^25*x4^4*z^22 + x1^48*x2^33*x3^25*x4^4*z^22 - 2*x1^47*x2^34*x3^25*x4^4*z^22 + 2*x1^46*x2^35*x3^25*x4^4*z^22 - x1^45*x2^36*x3^25*x4^4*z^22 - x1^48*x2^32*x3^26*x4^4*z^22 + x1^47*x2^33*x3^26*x4^4*z^22 - x1^46*x2^34*x3^26*x4^4*z^22 - 2*x1^45*x2^35*x3^26*x4^4*z^22 + 2*x1^44*x2^36*x3^26*x4^4*z^22 - 2*x1^46*x2^33*x3^27*x4^4*z^22 - x1^45*x2^34*x3^27*x4^4*z^22 - x1^44*x2^35*x3^27*x4^4*z^22 - x1^53*x2^36*x3^16*x4^5*z^22 + 4*x1^52*x2^36*x3^17*x4^5*z^22 + x1^51*x2^37*x3^17*x4^5*z^22 + 2*x1^50*x2^38*x3^17*x4^5*z^22 - 2*x1^52*x2^35*x3^18*x4^5*z^22 - 4*x1^51*x2^36*x3^18*x4^5*z^22 + x1^50*x2^37*x3^18*x4^5*z^22 + 2*x1^48*x2^39*x3^18*x4^5*z^22 + 6*x1^51*x2^35*x3^19*x4^5*z^22 + x1^49*x2^37*x3^19*x4^5*z^22 - x1^47*x2^39*x3^19*x4^5*z^22 - 2*x1^51*x2^34*x3^20*x4^5*z^22 - 6*x1^50*x2^35*x3^20*x4^5*z^22 + 3*x1^49*x2^36*x3^20*x4^5*z^22 - 5*x1^48*x2^37*x3^20*x4^5*z^22 + 2*x1^47*x2^38*x3^20*x4^5*z^22 + 2*x1^46*x2^39*x3^20*x4^5*z^22 + 3*x1^45*x2^40*x3^20*x4^5*z^22 + 6*x1^50*x2^34*x3^21*x4^5*z^22 + 2*x1^49*x2^35*x3^21*x4^5*z^22 - x1^48*x2^36*x3^21*x4^5*z^22 + 2*x1^47*x2^37*x3^21*x4^5*z^22 - 2*x1^46*x2^38*x3^21*x4^5*z^22 - x1^45*x2^39*x3^21*x4^5*z^22 - 2*x1^44*x2^40*x3^21*x4^5*z^22 - 2*x1^50*x2^33*x3^22*x4^5*z^22 - 6*x1^49*x2^34*x3^22*x4^5*z^22 + 2*x1^48*x2^35*x3^22*x4^5*z^22 - 3*x1^47*x2^36*x3^22*x4^5*z^22 + x1^46*x2^37*x3^22*x4^5*z^22 + 3*x1^44*x2^39*x3^22*x4^5*z^22 - x1^42*x2^41*x3^22*x4^5*z^22 + 6*x1^49*x2^33*x3^23*x4^5*z^22 + x1^48*x2^34*x3^23*x4^5*z^22 + 2*x1^47*x2^35*x3^23*x4^5*z^22 + 2*x1^46*x2^36*x3^23*x4^5*z^22 - 2*x1^45*x2^37*x3^23*x4^5*z^22 - x1^44*x2^38*x3^23*x4^5*z^22 - x1^43*x2^39*x3^23*x4^5*z^22 - 2*x1^49*x2^32*x3^24*x4^5*z^22 - 4*x1^48*x2^33*x3^24*x4^5*z^22 + 2*x1^47*x2^34*x3^24*x4^5*z^22 - 3*x1^46*x2^35*x3^24*x4^5*z^22 + x1^45*x2^36*x3^24*x4^5*z^22 + 3*x1^43*x2^38*x3^24*x4^5*z^22 - x1^42*x2^39*x3^24*x4^5*z^22 + 5*x1^48*x2^32*x3^25*x4^5*z^22 + 3*x1^45*x2^35*x3^25*x4^5*z^22 - 2*x1^44*x2^36*x3^25*x4^5*z^22 + x1^43*x2^37*x3^25*x4^5*z^22 - x1^42*x2^38*x3^25*x4^5*z^22 - x1^48*x2^31*x3^26*x4^5*z^22 - x1^47*x2^32*x3^26*x4^5*z^22 + x1^46*x2^33*x3^26*x4^5*z^22 - 3*x1^45*x2^34*x3^26*x4^5*z^22 - x1^44*x2^35*x3^26*x4^5*z^22 + x1^42*x2^37*x3^26*x4^5*z^22 - x1^41*x2^38*x3^26*x4^5*z^22 - x1^40*x2^39*x3^26*x4^5*z^22 + x1^47*x2^31*x3^27*x4^5*z^22 + x1^44*x2^34*x3^27*x4^5*z^22 - 2*x1^43*x2^35*x3^27*x4^5*z^22 + x1^45*x2^32*x3^28*x4^5*z^22 + x1^44*x2^33*x3^28*x4^5*z^22 - 2*x1^42*x2^34*x3^29*x4^5*z^22 - x1^52*x2^36*x3^16*x4^6*z^22 + x1^52*x2^35*x3^17*x4^6*z^22 + x1^51*x2^36*x3^17*x4^6*z^22 - x1^50*x2^37*x3^17*x4^6*z^22 - 3*x1^51*x2^35*x3^18*x4^6*z^22 + 2*x1^50*x2^36*x3^18*x4^6*z^22 - x1^48*x2^38*x3^18*x4^6*z^22 + 3*x1^50*x2^35*x3^19*x4^6*z^22 - 3*x1^49*x2^36*x3^19*x4^6*z^22 + 2*x1^48*x2^37*x3^19*x4^6*z^22 - 2*x1^50*x2^34*x3^20*x4^6*z^22 + 2*x1^48*x2^36*x3^20*x4^6*z^22 - x1^47*x2^37*x3^20*x4^6*z^22 + x1^46*x2^38*x3^20*x4^6*z^22 - x1^45*x2^39*x3^20*x4^6*z^22 + x1^50*x2^33*x3^21*x4^6*z^22 + 2*x1^49*x2^34*x3^21*x4^6*z^22 - 4*x1^48*x2^35*x3^21*x4^6*z^22 + x1^47*x2^36*x3^21*x4^6*z^22 - x1^46*x2^37*x3^21*x4^6*z^22 - x1^44*x2^39*x3^21*x4^6*z^22 - 2*x1^49*x2^33*x3^22*x4^6*z^22 + 3*x1^47*x2^35*x3^22*x4^6*z^22 - x1^46*x2^36*x3^22*x4^6*z^22 + 4*x1^45*x2^37*x3^22*x4^6*z^22 + x1^44*x2^38*x3^22*x4^6*z^22 + x1^49*x2^32*x3^23*x4^6*z^22 + 2*x1^48*x2^33*x3^23*x4^6*z^22 - 4*x1^47*x2^34*x3^23*x4^6*z^22 - x1^46*x2^35*x3^23*x4^6*z^22 - 5*x1^45*x2^36*x3^23*x4^6*z^22 - x1^44*x2^37*x3^23*x4^6*z^22 + 2*x1^42*x2^39*x3^23*x4^6*z^22 - 3*x1^48*x2^32*x3^24*x4^6*z^22 + 2*x1^47*x2^33*x3^24*x4^6*z^22 + 2*x1^46*x2^34*x3^24*x4^6*z^22 - 2*x1^45*x2^35*x3^24*x4^6*z^22 + 3*x1^44*x2^36*x3^24*x4^6*z^22 - x1^43*x2^37*x3^24*x4^6*z^22 + 2*x1^42*x2^38*x3^24*x4^6*z^22 - 2*x1^41*x2^39*x3^24*x4^6*z^22 - x1^40*x2^40*x3^24*x4^6*z^22 - x1^48*x2^31*x3^25*x4^6*z^22 + 2*x1^47*x2^32*x3^25*x4^6*z^22 - 2*x1^46*x2^33*x3^25*x4^6*z^22 + x1^45*x2^34*x3^25*x4^6*z^22 - x1^44*x2^35*x3^25*x4^6*z^22 - 2*x1^43*x2^36*x3^25*x4^6*z^22 - x1^42*x2^37*x3^25*x4^6*z^22 + 2*x1^41*x2^38*x3^25*x4^6*z^22 - x1^47*x2^31*x3^26*x4^6*z^22 - x1^46*x2^32*x3^26*x4^6*z^22 + 2*x1^45*x2^33*x3^26*x4^6*z^22 + 2*x1^43*x2^35*x3^26*x4^6*z^22 - x1^42*x2^36*x3^26*x4^6*z^22 + 2*x1^41*x2^37*x3^26*x4^6*z^22 + x1^40*x2^38*x3^26*x4^6*z^22 - x1^45*x2^32*x3^27*x4^6*z^22 + x1^44*x2^33*x3^27*x4^6*z^22 + x1^43*x2^34*x3^27*x4^6*z^22 - x1^42*x2^35*x3^27*x4^6*z^22 - x1^41*x2^36*x3^27*x4^6*z^22 + x1^40*x2^37*x3^27*x4^6*z^22 + x1^39*x2^38*x3^27*x4^6*z^22 + x1^44*x2^32*x3^28*x4^6*z^22 + x1^43*x2^33*x3^28*x4^6*z^22 + 2*x1^42*x2^34*x3^28*x4^6*z^22 - x1^43*x2^32*x3^29*x4^6*z^22 + x1^41*x2^33*x3^30*x4^6*z^22 - x1^51*x2^38*x3^14*x4^7*z^22 + x1^52*x2^36*x3^15*x4^7*z^22 + x1^50*x2^38*x3^15*x4^7*z^22 + x1^52*x2^35*x3^16*x4^7*z^22 - x1^51*x2^36*x3^16*x4^7*z^22 - x1^50*x2^37*x3^16*x4^7*z^22 - x1^48*x2^39*x3^16*x4^7*z^22 + 2*x1^50*x2^36*x3^17*x4^7*z^22 + 3*x1^49*x2^37*x3^17*x4^7*z^22 + 2*x1^47*x2^39*x3^17*x4^7*z^22 + x1^46*x2^40*x3^17*x4^7*z^22 - 2*x1^49*x2^36*x3^18*x4^7*z^22 - x1^47*x2^38*x3^18*x4^7*z^22 + x1^44*x2^41*x3^18*x4^7*z^22 - x1^49*x2^35*x3^19*x4^7*z^22 + 2*x1^48*x2^36*x3^19*x4^7*z^22 + 2*x1^46*x2^38*x3^19*x4^7*z^22 - 2*x1^45*x2^39*x3^19*x4^7*z^22 - x1^43*x2^41*x3^19*x4^7*z^22 - 2*x1^47*x2^36*x3^20*x4^7*z^22 - x1^46*x2^37*x3^20*x4^7*z^22 - x1^45*x2^38*x3^20*x4^7*z^22 - x1^43*x2^40*x3^20*x4^7*z^22 + x1^42*x2^41*x3^20*x4^7*z^22 + x1^45*x2^37*x3^21*x4^7*z^22 - x1^44*x2^38*x3^21*x4^7*z^22 - x1^42*x2^40*x3^21*x4^7*z^22 - x1^41*x2^41*x3^21*x4^7*z^22 + x1^45*x2^36*x3^22*x4^7*z^22 - 2*x1^44*x2^37*x3^22*x4^7*z^22 + x1^43*x2^38*x3^22*x4^7*z^22 + 2*x1^42*x2^39*x3^22*x4^7*z^22 + x1^41*x2^40*x3^22*x4^7*z^22 - 2*x1^44*x2^36*x3^23*x4^7*z^22 - x1^41*x2^39*x3^23*x4^7*z^22 + x1^44*x2^35*x3^24*x4^7*z^22 + 2*x1^40*x2^39*x3^24*x4^7*z^22 + 2*x1^47*x2^31*x3^25*x4^7*z^22 - x1^46*x2^32*x3^25*x4^7*z^22 + 2*x1^44*x2^34*x3^25*x4^7*z^22 - x1^43*x2^35*x3^25*x4^7*z^22 - x1^42*x2^36*x3^25*x4^7*z^22 - 2*x1^46*x2^31*x3^26*x4^7*z^22 - x1^43*x2^34*x3^26*x4^7*z^22 + x1^41*x2^36*x3^26*x4^7*z^22 - x1^40*x2^37*x3^26*x4^7*z^22 + x1^39*x2^38*x3^26*x4^7*z^22 + x1^46*x2^30*x3^27*x4^7*z^22 + 2*x1^45*x2^31*x3^27*x4^7*z^22 + x1^44*x2^32*x3^27*x4^7*z^22 - x1^43*x2^33*x3^27*x4^7*z^22 - x1^42*x2^34*x3^27*x4^7*z^22 - 2*x1^40*x2^36*x3^27*x4^7*z^22 - x1^41*x2^34*x3^28*x4^7*z^22 - x1^42*x2^32*x3^29*x4^7*z^22 - x1^52*x2^35*x3^15*x4^8*z^22 - x1^51*x2^36*x3^15*x4^8*z^22 + 2*x1^50*x2^37*x3^15*x4^8*z^22 - x1^49*x2^38*x3^15*x4^8*z^22 + x1^48*x2^39*x3^15*x4^8*z^22 + x1^51*x2^35*x3^16*x4^8*z^22 - 4*x1^49*x2^37*x3^16*x4^8*z^22 + 2*x1^48*x2^38*x3^16*x4^8*z^22 - x1^47*x2^39*x3^16*x4^8*z^22 + x1^46*x2^40*x3^16*x4^8*z^22 + 2*x1^49*x2^36*x3^17*x4^8*z^22 - x1^46*x2^39*x3^17*x4^8*z^22 - 2*x1^45*x2^40*x3^17*x4^8*z^22 - 2*x1^50*x2^34*x3^18*x4^8*z^22 - x1^49*x2^35*x3^18*x4^8*z^22 - 4*x1^48*x2^36*x3^18*x4^8*z^22 + x1^47*x2^37*x3^18*x4^8*z^22 - x1^46*x2^38*x3^18*x4^8*z^22 + 3*x1^45*x2^39*x3^18*x4^8*z^22 + 2*x1^44*x2^40*x3^18*x4^8*z^22 + x1^43*x2^41*x3^18*x4^8*z^22 + x1^50*x2^33*x3^19*x4^8*z^22 + 2*x1^49*x2^34*x3^19*x4^8*z^22 + 3*x1^48*x2^35*x3^19*x4^8*z^22 - x1^46*x2^37*x3^19*x4^8*z^22 - x1^45*x2^38*x3^19*x4^8*z^22 - 2*x1^44*x2^39*x3^19*x4^8*z^22 - 3*x1^43*x2^40*x3^19*x4^8*z^22 - x1^42*x2^41*x3^19*x4^8*z^22 - 2*x1^49*x2^33*x3^20*x4^8*z^22 - x1^48*x2^34*x3^20*x4^8*z^22 - 3*x1^47*x2^35*x3^20*x4^8*z^22 + x1^46*x2^36*x3^20*x4^8*z^22 - 2*x1^45*x2^37*x3^20*x4^8*z^22 + 3*x1^44*x2^38*x3^20*x4^8*z^22 + x1^43*x2^39*x3^20*x4^8*z^22 + 5*x1^42*x2^40*x3^20*x4^8*z^22 + 2*x1^48*x2^33*x3^21*x4^8*z^22 + 4*x1^47*x2^34*x3^21*x4^8*z^22 + 4*x1^46*x2^35*x3^21*x4^8*z^22 + x1^45*x2^36*x3^21*x4^8*z^22 - 2*x1^44*x2^37*x3^21*x4^8*z^22 - 3*x1^43*x2^38*x3^21*x4^8*z^22 - 2*x1^42*x2^39*x3^21*x4^8*z^22 - 5*x1^41*x2^40*x3^21*x4^8*z^22 - 2*x1^48*x2^32*x3^22*x4^8*z^22 - 3*x1^47*x2^33*x3^22*x4^8*z^22 - 5*x1^46*x2^34*x3^22*x4^8*z^22 - x1^44*x2^36*x3^22*x4^8*z^22 + 5*x1^43*x2^37*x3^22*x4^8*z^22 + x1^42*x2^38*x3^22*x4^8*z^22 + 4*x1^41*x2^39*x3^22*x4^8*z^22 + 2*x1^40*x2^40*x3^22*x4^8*z^22 + x1^48*x2^31*x3^23*x4^8*z^22 + 2*x1^47*x2^32*x3^23*x4^8*z^22 + 4*x1^46*x2^33*x3^23*x4^8*z^22 + 2*x1^45*x2^34*x3^23*x4^8*z^22 + 2*x1^44*x2^35*x3^23*x4^8*z^22 + x1^43*x2^36*x3^23*x4^8*z^22 - 2*x1^42*x2^37*x3^23*x4^8*z^22 - 2*x1^41*x2^38*x3^23*x4^8*z^22 - 3*x1^40*x2^39*x3^23*x4^8*z^22 - 2*x1^47*x2^31*x3^24*x4^8*z^22 - 5*x1^45*x2^33*x3^24*x4^8*z^22 - x1^43*x2^35*x3^24*x4^8*z^22 + 4*x1^42*x2^36*x3^24*x4^8*z^22 + 2*x1^41*x2^37*x3^24*x4^8*z^22 + 3*x1^40*x2^38*x3^24*x4^8*z^22 + x1^39*x2^39*x3^24*x4^8*z^22 + 2*x1^46*x2^31*x3^25*x4^8*z^22 + 2*x1^44*x2^33*x3^25*x4^8*z^22 - x1^43*x2^34*x3^25*x4^8*z^22 + x1^42*x2^35*x3^25*x4^8*z^22 - 2*x1^41*x2^36*x3^25*x4^8*z^22 - x1^40*x2^37*x3^25*x4^8*z^22 - 4*x1^39*x2^38*x3^25*x4^8*z^22 - 2*x1^46*x2^30*x3^26*x4^8*z^22 + x1^44*x2^32*x3^26*x4^8*z^22 - 2*x1^43*x2^33*x3^26*x4^8*z^22 - 3*x1^42*x2^34*x3^26*x4^8*z^22 + 2*x1^40*x2^36*x3^26*x4^8*z^22 + x1^39*x2^37*x3^26*x4^8*z^22 + x1^38*x2^38*x3^26*x4^8*z^22 + 2*x1^45*x2^30*x3^27*x4^8*z^22 + x1^44*x2^31*x3^27*x4^8*z^22 + 2*x1^43*x2^32*x3^27*x4^8*z^22 + 2*x1^42*x2^33*x3^27*x4^8*z^22 - x1^39*x2^36*x3^27*x4^8*z^22 - 2*x1^38*x2^37*x3^27*x4^8*z^22 - x1^43*x2^31*x3^28*x4^8*z^22 + x1^42*x2^32*x3^28*x4^8*z^22 + x1^39*x2^35*x3^28*x4^8*z^22 + x1^38*x2^36*x3^28*x4^8*z^22 + x1^43*x2^30*x3^29*x4^8*z^22 + x1^42*x2^31*x3^29*x4^8*z^22 - x1^38*x2^35*x3^29*x4^8*z^22 - x1^37*x2^36*x3^29*x4^8*z^22 - x1^40*x2^32*x3^30*x4^8*z^22 - x1^51*x2^36*x3^14*x4^9*z^22 + 2*x1^49*x2^37*x3^15*x4^9*z^22 + x1^47*x2^39*x3^15*x4^9*z^22 - x1^51*x2^34*x3^16*x4^9*z^22 - x1^50*x2^35*x3^16*x4^9*z^22 - 3*x1^48*x2^37*x3^16*x4^9*z^22 - x1^47*x2^38*x3^16*x4^9*z^22 + x1^45*x2^40*x3^16*x4^9*z^22 + 2*x1^50*x2^34*x3^17*x4^9*z^22 + 3*x1^48*x2^36*x3^17*x4^9*z^22 + 2*x1^47*x2^37*x3^17*x4^9*z^22 + x1^46*x2^38*x3^17*x4^9*z^22 - x1^44*x2^40*x3^17*x4^9*z^22 - x1^50*x2^33*x3^18*x4^9*z^22 - 3*x1^49*x2^34*x3^18*x4^9*z^22 - x1^48*x2^35*x3^18*x4^9*z^22 - x1^46*x2^37*x3^18*x4^9*z^22 - x1^45*x2^38*x3^18*x4^9*z^22 - x1^44*x2^39*x3^18*x4^9*z^22 + x1^43*x2^40*x3^18*x4^9*z^22 + 4*x1^49*x2^33*x3^19*x4^9*z^22 + x1^48*x2^34*x3^19*x4^9*z^22 + 3*x1^47*x2^35*x3^19*x4^9*z^22 + x1^46*x2^36*x3^19*x4^9*z^22 + x1^45*x2^37*x3^19*x4^9*z^22 - x1^44*x2^38*x3^19*x4^9*z^22 + x1^43*x2^39*x3^19*x4^9*z^22 - 2*x1^42*x2^40*x3^19*x4^9*z^22 - x1^49*x2^32*x3^20*x4^9*z^22 - 4*x1^48*x2^33*x3^20*x4^9*z^22 - 2*x1^47*x2^34*x3^20*x4^9*z^22 - 3*x1^46*x2^35*x3^20*x4^9*z^22 + x1^44*x2^37*x3^20*x4^9*z^22 - x1^42*x2^39*x3^20*x4^9*z^22 + 2*x1^41*x2^40*x3^20*x4^9*z^22 + 4*x1^48*x2^32*x3^21*x4^9*z^22 + 2*x1^47*x2^33*x3^21*x4^9*z^22 + 3*x1^46*x2^34*x3^21*x4^9*z^22 + 2*x1^44*x2^36*x3^21*x4^9*z^22 - x1^43*x2^37*x3^21*x4^9*z^22 - 2*x1^41*x2^39*x3^21*x4^9*z^22 - x1^40*x2^40*x3^21*x4^9*z^22 - x1^48*x2^31*x3^22*x4^9*z^22 - 4*x1^47*x2^32*x3^22*x4^9*z^22 - 2*x1^46*x2^33*x3^22*x4^9*z^22 - 4*x1^45*x2^34*x3^22*x4^9*z^22 + x1^44*x2^35*x3^22*x4^9*z^22 + 5*x1^42*x2^37*x3^22*x4^9*z^22 + 2*x1^41*x2^38*x3^22*x4^9*z^22 + 2*x1^40*x2^39*x3^22*x4^9*z^22 + 4*x1^47*x2^31*x3^23*x4^9*z^22 + 2*x1^46*x2^32*x3^23*x4^9*z^22 + 4*x1^45*x2^33*x3^23*x4^9*z^22 - x1^43*x2^35*x3^23*x4^9*z^22 - 4*x1^42*x2^36*x3^23*x4^9*z^22 - 4*x1^41*x2^37*x3^23*x4^9*z^22 - 2*x1^40*x2^38*x3^23*x4^9*z^22 - x1^47*x2^30*x3^24*x4^9*z^22 - 4*x1^46*x2^31*x3^24*x4^9*z^22 - x1^45*x2^32*x3^24*x4^9*z^22 - 2*x1^44*x2^33*x3^24*x4^9*z^22 + x1^43*x2^34*x3^24*x4^9*z^22 - x1^42*x2^35*x3^24*x4^9*z^22 + 3*x1^41*x2^36*x3^24*x4^9*z^22 + x1^40*x2^37*x3^24*x4^9*z^22 + x1^39*x2^38*x3^24*x4^9*z^22 + 4*x1^46*x2^30*x3^25*x4^9*z^22 + 2*x1^44*x2^32*x3^25*x4^9*z^22 + x1^42*x2^34*x3^25*x4^9*z^22 - x1^41*x2^35*x3^25*x4^9*z^22 - 3*x1^40*x2^36*x3^25*x4^9*z^22 - x1^38*x2^38*x3^25*x4^9*z^22 - 4*x1^45*x2^30*x3^26*x4^9*z^22 + x1^44*x2^31*x3^26*x4^9*z^22 - 3*x1^43*x2^32*x3^26*x4^9*z^22 + x1^41*x2^34*x3^26*x4^9*z^22 + 3*x1^40*x2^35*x3^26*x4^9*z^22 + x1^39*x2^36*x3^26*x4^9*z^22 + x1^38*x2^37*x3^26*x4^9*z^22 + 2*x1^45*x2^29*x3^27*x4^9*z^22 + 3*x1^44*x2^30*x3^27*x4^9*z^22 - 2*x1^40*x2^34*x3^27*x4^9*z^22 - 3*x1^39*x2^35*x3^27*x4^9*z^22 - x1^37*x2^37*x3^27*x4^9*z^22 - 2*x1^44*x2^29*x3^28*x4^9*z^22 - 2*x1^43*x2^30*x3^28*x4^9*z^22 - x1^42*x2^31*x3^28*x4^9*z^22 + 2*x1^39*x2^34*x3^28*x4^9*z^22 + x1^38*x2^35*x3^28*x4^9*z^22 + x1^42*x2^30*x3^29*x4^9*z^22 - x1^39*x2^33*x3^29*x4^9*z^22 + x1^37*x2^35*x3^29*x4^9*z^22 - x1^41*x2^30*x3^30*x4^9*z^22 + x1^40*x2^31*x3^30*x4^9*z^22 + x1^39*x2^32*x3^30*x4^9*z^22 - x1^38*x2^33*x3^30*x4^9*z^22 + x1^36*x2^35*x3^30*x4^9*z^22 - x1^48*x2^38*x3^14*x4^10*z^22 - x1^49*x2^36*x3^15*x4^10*z^22 - x1^46*x2^39*x3^15*x4^10*z^22 - x1^45*x2^39*x3^16*x4^10*z^22 + x1^50*x2^33*x3^17*x4^10*z^22 - x1^49*x2^34*x3^17*x4^10*z^22 + 2*x1^47*x2^36*x3^17*x4^10*z^22 + 3*x1^46*x2^37*x3^17*x4^10*z^22 + x1^45*x2^38*x3^17*x4^10*z^22 + x1^44*x2^39*x3^17*x4^10*z^22 + x1^43*x2^40*x3^17*x4^10*z^22 + x1^49*x2^33*x3^18*x4^10*z^22 - x1^47*x2^35*x3^18*x4^10*z^22 - 2*x1^46*x2^36*x3^18*x4^10*z^22 - x1^45*x2^37*x3^18*x4^10*z^22 - x1^43*x2^39*x3^18*x4^10*z^22 - x1^42*x2^40*x3^18*x4^10*z^22 + 2*x1^47*x2^34*x3^19*x4^10*z^22 - x1^46*x2^35*x3^19*x4^10*z^22 + x1^45*x2^36*x3^19*x4^10*z^22 + 2*x1^44*x2^37*x3^19*x4^10*z^22 + 2*x1^43*x2^38*x3^19*x4^10*z^22 + x1^42*x2^39*x3^19*x4^10*z^22 + x1^41*x2^40*x3^19*x4^10*z^22 - x1^43*x2^37*x3^20*x4^10*z^22 - 2*x1^42*x2^38*x3^20*x4^10*z^22 - x1^40*x2^40*x3^20*x4^10*z^22 - x1^43*x2^36*x3^21*x4^10*z^22 + x1^42*x2^37*x3^21*x4^10*z^22 + x1^41*x2^38*x3^21*x4^10*z^22 - x1^47*x2^31*x3^22*x4^10*z^22 - x1^41*x2^37*x3^22*x4^10*z^22 + x1^47*x2^30*x3^23*x4^10*z^22 + x1^46*x2^31*x3^23*x4^10*z^22 - x1^45*x2^32*x3^23*x4^10*z^22 - 3*x1^46*x2^30*x3^24*x4^10*z^22 - x1^43*x2^33*x3^24*x4^10*z^22 + 3*x1^45*x2^30*x3^25*x4^10*z^22 - x1^45*x2^29*x3^26*x4^10*z^22 - x1^44*x2^30*x3^26*x4^10*z^22 - x1^43*x2^31*x3^26*x4^10*z^22 + x1^42*x2^32*x3^26*x4^10*z^22 - x1^41*x2^33*x3^26*x4^10*z^22 - 3*x1^40*x2^34*x3^26*x4^10*z^22 - x1^38*x2^36*x3^26*x4^10*z^22 + x1^44*x2^29*x3^27*x4^10*z^22 + x1^43*x2^30*x3^27*x4^10*z^22 + 2*x1^42*x2^31*x3^27*x4^10*z^22 - x1^39*x2^34*x3^27*x4^10*z^22 - x1^38*x2^35*x3^27*x4^10*z^22 + 2*x1^37*x2^36*x3^27*x4^10*z^22 - x1^42*x2^30*x3^28*x4^10*z^22 + x1^41*x2^31*x3^28*x4^10*z^22 + x1^40*x2^32*x3^28*x4^10*z^22 + x1^39*x2^33*x3^28*x4^10*z^22 - 2*x1^37*x2^35*x3^28*x4^10*z^22 + x1^39*x2^32*x3^29*x4^10*z^22 - 2*x1^38*x2^33*x3^29*x4^10*z^22 - x1^37*x2^34*x3^29*x4^10*z^22 + x1^36*x2^35*x3^29*x4^10*z^22 - x1^39*x2^31*x3^30*x4^10*z^22 + x1^37*x2^33*x3^30*x4^10*z^22 - x1^36*x2^34*x3^30*x4^10*z^22 + x1^50*x2^35*x3^14*x4^11*z^22 + x1^47*x2^38*x3^14*x4^11*z^22 - x1^48*x2^36*x3^15*x4^11*z^22 + 2*x1^47*x2^37*x3^15*x4^11*z^22 + x1^46*x2^38*x3^15*x4^11*z^22 - x1^45*x2^39*x3^15*x4^11*z^22 + 2*x1^49*x2^34*x3^16*x4^11*z^22 + 2*x1^48*x2^35*x3^16*x4^11*z^22 - 3*x1^46*x2^37*x3^16*x4^11*z^22 + x1^44*x2^39*x3^16*x4^11*z^22 - 2*x1^49*x2^33*x3^17*x4^11*z^22 + x1^48*x2^34*x3^17*x4^11*z^22 + 2*x1^47*x2^35*x3^17*x4^11*z^22 - x1^46*x2^36*x3^17*x4^11*z^22 + x1^45*x2^37*x3^17*x4^11*z^22 + x1^44*x2^38*x3^17*x4^11*z^22 - x1^42*x2^40*x3^17*x4^11*z^22 - x1^49*x2^32*x3^18*x4^11*z^22 + 3*x1^48*x2^33*x3^18*x4^11*z^22 - x1^47*x2^34*x3^18*x4^11*z^22 + x1^46*x2^35*x3^18*x4^11*z^22 - 3*x1^45*x2^36*x3^18*x4^11*z^22 - 3*x1^43*x2^38*x3^18*x4^11*z^22 + x1^42*x2^39*x3^18*x4^11*z^22 + x1^41*x2^40*x3^18*x4^11*z^22 - 3*x1^48*x2^32*x3^19*x4^11*z^22 + 2*x1^46*x2^34*x3^19*x4^11*z^22 + 2*x1^45*x2^35*x3^19*x4^11*z^22 + 2*x1^44*x2^36*x3^19*x4^11*z^22 - x1^43*x2^37*x3^19*x4^11*z^22 + 4*x1^42*x2^38*x3^19*x4^11*z^22 - x1^41*x2^39*x3^19*x4^11*z^22 + 4*x1^47*x2^32*x3^20*x4^11*z^22 - 4*x1^46*x2^33*x3^20*x4^11*z^22 + x1^45*x2^34*x3^20*x4^11*z^22 - 2*x1^44*x2^35*x3^20*x4^11*z^22 - x1^43*x2^36*x3^20*x4^11*z^22 - 4*x1^42*x2^37*x3^20*x4^11*z^22 - x1^41*x2^38*x3^20*x4^11*z^22 + x1^40*x2^39*x3^20*x4^11*z^22 - 4*x1^47*x2^31*x3^21*x4^11*z^22 - 2*x1^46*x2^32*x3^21*x4^11*z^22 + 4*x1^43*x2^35*x3^21*x4^11*z^22 + x1^42*x2^36*x3^21*x4^11*z^22 + 4*x1^41*x2^37*x3^21*x4^11*z^22 - 2*x1^40*x2^38*x3^21*x4^11*z^22 + x1^47*x2^30*x3^22*x4^11*z^22 + 4*x1^46*x2^31*x3^22*x4^11*z^22 - 2*x1^45*x2^32*x3^22*x4^11*z^22 + x1^44*x2^33*x3^22*x4^11*z^22 - 4*x1^43*x2^34*x3^22*x4^11*z^22 - 3*x1^41*x2^36*x3^22*x4^11*z^22 + 2*x1^39*x2^38*x3^22*x4^11*z^22 + x1^45*x2^31*x3^23*x4^11*z^22 + 3*x1^44*x2^32*x3^23*x4^11*z^22 + 2*x1^42*x2^34*x3^23*x4^11*z^22 + 4*x1^40*x2^36*x3^23*x4^11*z^22 - 2*x1^39*x2^37*x3^23*x4^11*z^22 - x1^38*x2^38*x3^23*x4^11*z^22 - x1^44*x2^31*x3^24*x4^11*z^22 + 2*x1^43*x2^32*x3^24*x4^11*z^22 - 2*x1^42*x2^33*x3^24*x4^11*z^22 - 3*x1^40*x2^35*x3^24*x4^11*z^22 + 2*x1^38*x2^37*x3^24*x4^11*z^22 - x1^44*x2^30*x3^25*x4^11*z^22 + x1^43*x2^31*x3^25*x4^11*z^22 + x1^41*x2^33*x3^25*x4^11*z^22 - x1^40*x2^34*x3^25*x4^11*z^22 + 4*x1^39*x2^35*x3^25*x4^11*z^22 - 2*x1^38*x2^36*x3^25*x4^11*z^22 + x1^43*x2^30*x3^26*x4^11*z^22 - x1^42*x2^31*x3^26*x4^11*z^22 + 2*x1^40*x2^33*x3^26*x4^11*z^22 - x1^39*x2^34*x3^26*x4^11*z^22 - x1^38*x2^35*x3^26*x4^11*z^22 + 2*x1^37*x2^36*x3^26*x4^11*z^22 - x1^42*x2^30*x3^27*x4^11*z^22 + 2*x1^40*x2^32*x3^27*x4^11*z^22 - 2*x1^37*x2^35*x3^27*x4^11*z^22 - x1^40*x2^31*x3^28*x4^11*z^22 - x1^38*x2^33*x3^28*x4^11*z^22 + x1^37*x2^34*x3^28*x4^11*z^22 - x1^38*x2^32*x3^29*x4^11*z^22 - x1^36*x2^34*x3^29*x4^11*z^22 - x1^47*x2^37*x3^14*x4^12*z^22 - 2*x1^49*x2^34*x3^15*x4^12*z^22 + x1^47*x2^36*x3^15*x4^12*z^22 - x1^46*x2^37*x3^15*x4^12*z^22 - x1^45*x2^37*x3^16*x4^12*z^22 + x1^44*x2^38*x3^16*x4^12*z^22 + x1^43*x2^39*x3^16*x4^12*z^22 - 2*x1^48*x2^33*x3^17*x4^12*z^22 + x1^47*x2^34*x3^17*x4^12*z^22 + 2*x1^46*x2^35*x3^17*x4^12*z^22 + x1^45*x2^36*x3^17*x4^12*z^22 + x1^44*x2^37*x3^17*x4^12*z^22 - x1^43*x2^38*x3^17*x4^12*z^22 + x1^48*x2^32*x3^18*x4^12*z^22 - x1^47*x2^33*x3^18*x4^12*z^22 - 3*x1^46*x2^34*x3^18*x4^12*z^22 + x1^44*x2^36*x3^18*x4^12*z^22 + 3*x1^43*x2^37*x3^18*x4^12*z^22 + 2*x1^41*x2^39*x3^18*x4^12*z^22 - 3*x1^47*x2^32*x3^19*x4^12*z^22 + 4*x1^46*x2^33*x3^19*x4^12*z^22 + 2*x1^44*x2^35*x3^19*x4^12*z^22 - x1^43*x2^36*x3^19*x4^12*z^22 + x1^42*x2^37*x3^19*x4^12*z^22 - 2*x1^41*x2^38*x3^19*x4^12*z^22 - 2*x1^40*x2^39*x3^19*x4^12*z^22 + x1^47*x2^31*x3^20*x4^12*z^22 - x1^46*x2^32*x3^20*x4^12*z^22 - 4*x1^45*x2^33*x3^20*x4^12*z^22 - 3*x1^43*x2^35*x3^20*x4^12*z^22 + 3*x1^42*x2^36*x3^20*x4^12*z^22 - 2*x1^41*x2^37*x3^20*x4^12*z^22 + 6*x1^40*x2^38*x3^20*x4^12*z^22 - 2*x1^46*x2^31*x3^21*x4^12*z^22 + 6*x1^45*x2^32*x3^21*x4^12*z^22 + 3*x1^44*x2^33*x3^21*x4^12*z^22 + 3*x1^43*x2^34*x3^21*x4^12*z^22 - x1^40*x2^37*x3^21*x4^12*z^22 - 6*x1^39*x2^38*x3^21*x4^12*z^22 + x1^46*x2^30*x3^22*x4^12*z^22 - 5*x1^44*x2^32*x3^22*x4^12*z^22 - 5*x1^42*x2^34*x3^22*x4^12*z^22 + 3*x1^41*x2^35*x3^22*x4^12*z^22 - 2*x1^40*x2^36*x3^22*x4^12*z^22 + 6*x1^39*x2^37*x3^22*x4^12*z^22 + 2*x1^38*x2^38*x3^22*x4^12*z^22 - x1^45*x2^30*x3^23*x4^12*z^22 + 3*x1^44*x2^31*x3^23*x4^12*z^22 + 2*x1^43*x2^32*x3^23*x4^12*z^22 + 4*x1^42*x2^33*x3^23*x4^12*z^22 - 2*x1^39*x2^36*x3^23*x4^12*z^22 - 6*x1^38*x2^37*x3^23*x4^12*z^22 - x1^45*x2^29*x3^24*x4^12*z^22 - 4*x1^43*x2^31*x3^24*x4^12*z^22 - x1^42*x2^32*x3^24*x4^12*z^22 - 3*x1^41*x2^33*x3^24*x4^12*z^22 + 4*x1^40*x2^34*x3^24*x4^12*z^22 - 2*x1^39*x2^35*x3^24*x4^12*z^22 + 6*x1^38*x2^36*x3^24*x4^12*z^22 + 2*x1^37*x2^37*x3^24*x4^12*z^22 + x1^44*x2^29*x3^25*x4^12*z^22 + x1^43*x2^30*x3^25*x4^12*z^22 + x1^42*x2^31*x3^25*x4^12*z^22 + 2*x1^41*x2^32*x3^25*x4^12*z^22 + x1^39*x2^34*x3^25*x4^12*z^22 - x1^38*x2^35*x3^25*x4^12*z^22 - 6*x1^37*x2^36*x3^25*x4^12*z^22 - x1^42*x2^30*x3^26*x4^12*z^22 - x1^41*x2^31*x3^26*x4^12*z^22 - 3*x1^40*x2^32*x3^26*x4^12*z^22 + 2*x1^39*x2^33*x3^26*x4^12*z^22 - 3*x1^38*x2^34*x3^26*x4^12*z^22 + 6*x1^37*x2^35*x3^26*x4^12*z^22 + 2*x1^36*x2^36*x3^26*x4^12*z^22 + x1^41*x2^30*x3^27*x4^12*z^22 + 2*x1^40*x2^31*x3^27*x4^12*z^22 - x1^37*x2^34*x3^27*x4^12*z^22 - 6*x1^36*x2^35*x3^27*x4^12*z^22 - x1^39*x2^31*x3^28*x4^12*z^22 + x1^38*x2^32*x3^28*x4^12*z^22 - x1^37*x2^33*x3^28*x4^12*z^22 + 4*x1^36*x2^34*x3^28*x4^12*z^22 + x1^35*x2^35*x3^28*x4^12*z^22 - 2*x1^35*x2^34*x3^29*x4^12*z^22 - x1^35*x2^33*x3^30*x4^12*z^22 + 2*x1^46*x2^36*x3^15*x4^13*z^22 + x1^45*x2^37*x3^15*x4^13*z^22 - x1^47*x2^33*x3^17*x4^13*z^22 - x1^46*x2^34*x3^17*x4^13*z^22 + x1^44*x2^36*x3^17*x4^13*z^22 - x1^43*x2^37*x3^17*x4^13*z^22 - x1^41*x2^39*x3^17*x4^13*z^22 + 2*x1^47*x2^32*x3^18*x4^13*z^22 - 2*x1^46*x2^33*x3^18*x4^13*z^22 - x1^45*x2^34*x3^18*x4^13*z^22 + x1^44*x2^35*x3^18*x4^13*z^22 + x1^41*x2^38*x3^18*x4^13*z^22 + 3*x1^40*x2^39*x3^18*x4^13*z^22 - x1^46*x2^32*x3^19*x4^13*z^22 + 2*x1^45*x2^33*x3^19*x4^13*z^22 - 3*x1^42*x2^36*x3^19*x4^13*z^22 - 5*x1^40*x2^38*x3^19*x4^13*z^22 - x1^39*x2^39*x3^19*x4^13*z^22 + x1^47*x2^30*x3^20*x4^13*z^22 + x1^46*x2^31*x3^20*x4^13*z^22 - 3*x1^45*x2^32*x3^20*x4^13*z^22 - x1^44*x2^33*x3^20*x4^13*z^22 + 2*x1^42*x2^35*x3^20*x4^13*z^22 + 2*x1^41*x2^36*x3^20*x4^13*z^22 + 2*x1^40*x2^37*x3^20*x4^13*z^22 + 6*x1^39*x2^38*x3^20*x4^13*z^22 + x1^45*x2^31*x3^21*x4^13*z^22 + 3*x1^44*x2^32*x3^21*x4^13*z^22 - x1^43*x2^33*x3^21*x4^13*z^22 + x1^42*x2^34*x3^21*x4^13*z^22 - 4*x1^41*x2^35*x3^21*x4^13*z^22 - 6*x1^39*x2^37*x3^21*x4^13*z^22 - 2*x1^38*x2^38*x3^21*x4^13*z^22 + x1^46*x2^29*x3^22*x4^13*z^22 - 3*x1^44*x2^31*x3^22*x4^13*z^22 - x1^43*x2^32*x3^22*x4^13*z^22 - 2*x1^42*x2^33*x3^22*x4^13*z^22 + x1^41*x2^34*x3^22*x4^13*z^22 + 2*x1^40*x2^35*x3^22*x4^13*z^22 + 2*x1^39*x2^36*x3^22*x4^13*z^22 + 6*x1^38*x2^37*x3^22*x4^13*z^22 + 2*x1^44*x2^30*x3^23*x4^13*z^22 + 6*x1^43*x2^31*x3^23*x4^13*z^22 + x1^42*x2^32*x3^23*x4^13*z^22 + 4*x1^41*x2^33*x3^23*x4^13*z^22 - 4*x1^40*x2^34*x3^23*x4^13*z^22 - 6*x1^38*x2^36*x3^23*x4^13*z^22 - 2*x1^37*x2^37*x3^23*x4^13*z^22 - 2*x1^43*x2^30*x3^24*x4^13*z^22 - 2*x1^42*x2^31*x3^24*x4^13*z^22 - 2*x1^41*x2^32*x3^24*x4^13*z^22 + 2*x1^39*x2^34*x3^24*x4^13*z^22 + 2*x1^38*x2^35*x3^24*x4^13*z^22 + 6*x1^37*x2^36*x3^24*x4^13*z^22 - x1^43*x2^29*x3^25*x4^13*z^22 + 2*x1^42*x2^30*x3^25*x4^13*z^22 + 2*x1^40*x2^32*x3^25*x4^13*z^22 - 5*x1^39*x2^33*x3^25*x4^13*z^22 - 6*x1^37*x2^35*x3^25*x4^13*z^22 - 2*x1^36*x2^36*x3^25*x4^13*z^22 - 2*x1^41*x2^30*x3^26*x4^13*z^22 - x1^40*x2^31*x3^26*x4^13*z^22 - x1^39*x2^32*x3^26*x4^13*z^22 + x1^38*x2^33*x3^26*x4^13*z^22 + 6*x1^36*x2^35*x3^26*x4^13*z^22 + x1^39*x2^31*x3^27*x4^13*z^22 - x1^38*x2^32*x3^27*x4^13*z^22 + x1^37*x2^33*x3^27*x4^13*z^22 - 3*x1^36*x2^34*x3^27*x4^13*z^22 - 2*x1^35*x2^35*x3^27*x4^13*z^22 - x1^38*x2^31*x3^28*x4^13*z^22 + x1^36*x2^33*x3^28*x4^13*z^22 + 3*x1^35*x2^34*x3^28*x4^13*z^22 + 2*x1^36*x2^32*x3^29*x4^13*z^22 - x1^35*x2^33*x3^29*x4^13*z^22 + x1^34*x2^33*x3^30*x4^13*z^22 + x1^45*x2^36*x3^15*x4^14*z^22 - x1^45*x2^35*x3^16*x4^14*z^22 + 3*x1^44*x2^35*x3^17*x4^14*z^22 + x1^43*x2^36*x3^17*x4^14*z^22 + x1^42*x2^37*x3^17*x4^14*z^22 + x1^47*x2^31*x3^18*x4^14*z^22 + x1^46*x2^32*x3^18*x4^14*z^22 - x1^45*x2^33*x3^18*x4^14*z^22 - x1^44*x2^34*x3^18*x4^14*z^22 + x1^43*x2^35*x3^18*x4^14*z^22 - x1^41*x2^37*x3^18*x4^14*z^22 - x1^46*x2^31*x3^19*x4^14*z^22 + x1^45*x2^32*x3^19*x4^14*z^22 - x1^44*x2^33*x3^19*x4^14*z^22 + x1^43*x2^34*x3^19*x4^14*z^22 + x1^42*x2^35*x3^19*x4^14*z^22 + 2*x1^41*x2^36*x3^19*x4^14*z^22 - x1^40*x2^37*x3^19*x4^14*z^22 - 2*x1^39*x2^38*x3^19*x4^14*z^22 + x1^46*x2^30*x3^20*x4^14*z^22 + 2*x1^45*x2^31*x3^20*x4^14*z^22 - 2*x1^43*x2^33*x3^20*x4^14*z^22 + x1^41*x2^35*x3^20*x4^14*z^22 - 2*x1^40*x2^36*x3^20*x4^14*z^22 + x1^39*x2^37*x3^20*x4^14*z^22 - x1^46*x2^29*x3^21*x4^14*z^22 - 2*x1^45*x2^30*x3^21*x4^14*z^22 + 2*x1^44*x2^31*x3^21*x4^14*z^22 - 2*x1^43*x2^32*x3^21*x4^14*z^22 - x1^41*x2^34*x3^21*x4^14*z^22 + 2*x1^40*x2^35*x3^21*x4^14*z^22 + x1^39*x2^36*x3^21*x4^14*z^22 - 2*x1^38*x2^37*x3^21*x4^14*z^22 + 2*x1^45*x2^29*x3^22*x4^14*z^22 + 2*x1^44*x2^30*x3^22*x4^14*z^22 - 2*x1^41*x2^33*x3^22*x4^14*z^22 - 2*x1^39*x2^35*x3^22*x4^14*z^22 + 2*x1^38*x2^36*x3^22*x4^14*z^22 + x1^37*x2^37*x3^22*x4^14*z^22 - 2*x1^44*x2^29*x3^23*x4^14*z^22 - 2*x1^43*x2^30*x3^23*x4^14*z^22 - x1^42*x2^31*x3^23*x4^14*z^22 + 2*x1^41*x2^32*x3^23*x4^14*z^22 - x1^40*x2^33*x3^23*x4^14*z^22 + x1^39*x2^34*x3^23*x4^14*z^22 - 2*x1^37*x2^36*x3^23*x4^14*z^22 + x1^43*x2^29*x3^24*x4^14*z^22 + x1^42*x2^30*x3^24*x4^14*z^22 - x1^40*x2^32*x3^24*x4^14*z^22 - 2*x1^38*x2^34*x3^24*x4^14*z^22 + 2*x1^37*x2^35*x3^24*x4^14*z^22 + x1^36*x2^36*x3^24*x4^14*z^22 - x1^42*x2^29*x3^25*x4^14*z^22 - 2*x1^41*x2^30*x3^25*x4^14*z^22 + x1^40*x2^31*x3^25*x4^14*z^22 + 2*x1^38*x2^33*x3^25*x4^14*z^22 - x1^37*x2^34*x3^25*x4^14*z^22 - 2*x1^36*x2^35*x3^25*x4^14*z^22 + x1^41*x2^29*x3^26*x4^14*z^22 + x1^40*x2^30*x3^26*x4^14*z^22 + x1^39*x2^31*x3^26*x4^14*z^22 + x1^38*x2^32*x3^26*x4^14*z^22 - 3*x1^37*x2^33*x3^26*x4^14*z^22 + 2*x1^36*x2^34*x3^26*x4^14*z^22 - 2*x1^38*x2^31*x3^27*x4^14*z^22 + x1^37*x2^32*x3^27*x4^14*z^22 + 2*x1^36*x2^33*x3^27*x4^14*z^22 - 2*x1^35*x2^34*x3^27*x4^14*z^22 + x1^37*x2^31*x3^28*x4^14*z^22 - x1^36*x2^32*x3^28*x4^14*z^22 + x1^34*x2^34*x3^28*x4^14*z^22 - 2*x1^44*x2^35*x3^16*x4^15*z^22 + x1^44*x2^34*x3^17*x4^15*z^22 + x1^43*x2^35*x3^17*x4^15*z^22 + x1^42*x2^36*x3^17*x4^15*z^22 + x1^41*x2^37*x3^17*x4^15*z^22 - 4*x1^43*x2^34*x3^18*x4^15*z^22 - 2*x1^41*x2^36*x3^18*x4^15*z^22 - x1^40*x2^37*x3^18*x4^15*z^22 - x1^46*x2^30*x3^19*x4^15*z^22 - x1^45*x2^31*x3^19*x4^15*z^22 + x1^44*x2^32*x3^19*x4^15*z^22 + 2*x1^43*x2^33*x3^19*x4^15*z^22 + x1^42*x2^34*x3^19*x4^15*z^22 + x1^41*x2^35*x3^19*x4^15*z^22 + 5*x1^40*x2^36*x3^19*x4^15*z^22 + 2*x1^45*x2^30*x3^20*x4^15*z^22 + x1^43*x2^32*x3^20*x4^15*z^22 - 5*x1^42*x2^33*x3^20*x4^15*z^22 - 4*x1^40*x2^35*x3^20*x4^15*z^22 - 2*x1^39*x2^36*x3^20*x4^15*z^22 - x1^45*x2^29*x3^21*x4^15*z^22 - 2*x1^44*x2^30*x3^21*x4^15*z^22 + 2*x1^42*x2^32*x3^21*x4^15*z^22 + 2*x1^41*x2^33*x3^21*x4^15*z^22 + 2*x1^40*x2^34*x3^21*x4^15*z^22 + 6*x1^39*x2^35*x3^21*x4^15*z^22 + 3*x1^44*x2^29*x3^22*x4^15*z^22 + x1^43*x2^30*x3^22*x4^15*z^22 + x1^42*x2^31*x3^22*x4^15*z^22 - 4*x1^41*x2^32*x3^22*x4^15*z^22 - 6*x1^39*x2^34*x3^22*x4^15*z^22 - 2*x1^38*x2^35*x3^22*x4^15*z^22 - x1^44*x2^28*x3^23*x4^15*z^22 - 2*x1^43*x2^29*x3^23*x4^15*z^22 + x1^41*x2^31*x3^23*x4^15*z^22 + x1^40*x2^32*x3^23*x4^15*z^22 + 2*x1^39*x2^33*x3^23*x4^15*z^22 + 6*x1^38*x2^34*x3^23*x4^15*z^22 + x1^43*x2^28*x3^24*x4^15*z^22 + 2*x1^41*x2^30*x3^24*x4^15*z^22 - 4*x1^40*x2^31*x3^24*x4^15*z^22 + x1^39*x2^32*x3^24*x4^15*z^22 - 6*x1^38*x2^33*x3^24*x4^15*z^22 - 2*x1^37*x2^34*x3^24*x4^15*z^22 + x1^40*x2^30*x3^25*x4^15*z^22 + x1^39*x2^31*x3^25*x4^15*z^22 + x1^38*x2^32*x3^25*x4^15*z^22 + 6*x1^37*x2^33*x3^25*x4^15*z^22 + x1^40*x2^29*x3^26*x4^15*z^22 - 2*x1^39*x2^30*x3^26*x4^15*z^22 + x1^38*x2^31*x3^26*x4^15*z^22 - 4*x1^37*x2^32*x3^26*x4^15*z^22 - 2*x1^36*x2^33*x3^26*x4^15*z^22 + 2*x1^38*x2^30*x3^27*x4^15*z^22 + x1^37*x2^31*x3^27*x4^15*z^22 + 4*x1^36*x2^32*x3^27*x4^15*z^22 - x1^35*x2^32*x3^28*x4^15*z^22 - x1^42*x2^34*x3^18*x4^16*z^22 - x1^41*x2^35*x3^18*x4^16*z^22 - x1^40*x2^36*x3^18*x4^16*z^22 + 3*x1^42*x2^33*x3^19*x4^16*z^22 - x1^41*x2^34*x3^19*x4^16*z^22 + x1^40*x2^35*x3^19*x4^16*z^22 + 2*x1^39*x2^36*x3^19*x4^16*z^22 - 2*x1^42*x2^32*x3^20*x4^16*z^22 - 2*x1^41*x2^33*x3^20*x4^16*z^22 - 4*x1^39*x2^35*x3^20*x4^16*z^22 - x1^44*x2^29*x3^21*x4^16*z^22 - x1^43*x2^30*x3^21*x4^16*z^22 + 5*x1^41*x2^32*x3^21*x4^16*z^22 - x1^40*x2^33*x3^21*x4^16*z^22 + 3*x1^39*x2^34*x3^21*x4^16*z^22 + 2*x1^38*x2^35*x3^21*x4^16*z^22 + x1^43*x2^29*x3^22*x4^16*z^22 - x1^41*x2^31*x3^22*x4^16*z^22 - 2*x1^40*x2^32*x3^22*x4^16*z^22 - 2*x1^39*x2^33*x3^22*x4^16*z^22 - 5*x1^38*x2^34*x3^22*x4^16*z^22 - x1^43*x2^28*x3^23*x4^16*z^22 - x1^42*x2^29*x3^23*x4^16*z^22 + 4*x1^40*x2^31*x3^23*x4^16*z^22 + 6*x1^38*x2^33*x3^23*x4^16*z^22 + 2*x1^37*x2^34*x3^23*x4^16*z^22 + x1^42*x2^28*x3^24*x4^16*z^22 - x1^40*x2^30*x3^24*x4^16*z^22 - x1^39*x2^31*x3^24*x4^16*z^22 - x1^38*x2^32*x3^24*x4^16*z^22 - 6*x1^37*x2^33*x3^24*x4^16*z^22 - x1^40*x2^29*x3^25*x4^16*z^22 + x1^39*x2^30*x3^25*x4^16*z^22 - x1^38*x2^31*x3^25*x4^16*z^22 + 4*x1^37*x2^32*x3^25*x4^16*z^22 + 2*x1^36*x2^33*x3^25*x4^16*z^22 - x1^37*x2^31*x3^26*x4^16*z^22 - 4*x1^36*x2^32*x3^26*x4^16*z^22 - x1^37*x2^30*x3^27*x4^16*z^22 + x1^36*x2^31*x3^27*x4^16*z^22 + x1^35*x2^32*x3^27*x4^16*z^22 - x1^35*x2^31*x3^28*x4^16*z^22 + x1^41*x2^31*x3^21*x4^17*z^22 + x1^40*x2^32*x3^21*x4^17*z^22 - x1^39*x2^33*x3^21*x4^17*z^22 + x1^38*x2^34*x3^21*x4^17*z^22 - x1^40*x2^31*x3^22*x4^17*z^22 + x1^39*x2^32*x3^22*x4^17*z^22 - x1^38*x2^33*x3^22*x4^17*z^22 - x1^37*x2^34*x3^22*x4^17*z^22 - x1^40*x2^30*x3^23*x4^17*z^22 + x1^38*x2^32*x3^23*x4^17*z^22 + x1^37*x2^33*x3^23*x4^17*z^22 + 2*x1^38*x2^31*x3^24*x4^17*z^22 - x1^37*x2^32*x3^24*x4^17*z^22 - x1^36*x2^33*x3^24*x4^17*z^22 - x1^37*x2^31*x3^25*x4^17*z^22 + x1^36*x2^32*x3^25*x4^17*z^22 - 2*x1^49*x2^36*x3^20*z^21 + x1^50*x2^37*x3^17*x4*z^21 - x1^49*x2^38*x3^17*x4*z^21 - 2*x1^50*x2^36*x3^18*x4*z^21 - x1^49*x2^37*x3^18*x4*z^21 + 2*x1^48*x2^38*x3^18*x4*z^21 + 4*x1^49*x2^36*x3^19*x4*z^21 - x1^48*x2^37*x3^19*x4*z^21 - x1^46*x2^39*x3^19*x4*z^21 - 2*x1^49*x2^35*x3^20*x4*z^21 - 3*x1^48*x2^36*x3^20*x4*z^21 + 2*x1^47*x2^37*x3^20*x4*z^21 - x1^46*x2^38*x3^20*x4*z^21 + 4*x1^48*x2^35*x3^21*x4*z^21 + x1^46*x2^37*x3^21*x4*z^21 - x1^48*x2^34*x3^22*x4*z^21 - 2*x1^47*x2^35*x3^22*x4*z^21 - 2*x1^45*x2^37*x3^22*x4*z^21 + x1^47*x2^34*x3^23*x4*z^21 - x1^46*x2^35*x3^23*x4*z^21 + x1^45*x2^36*x3^23*x4*z^21 + x1^44*x2^37*x3^23*x4*z^21 - 2*x1^50*x2^37*x3^16*x4^2*z^21 + x1^50*x2^36*x3^17*x4^2*z^21 + 2*x1^49*x2^37*x3^17*x4^2*z^21 - x1^48*x2^38*x3^17*x4^2*z^21 - 5*x1^49*x2^36*x3^18*x4^2*z^21 - x1^48*x2^37*x3^18*x4^2*z^21 - x1^47*x2^38*x3^18*x4^2*z^21 + 2*x1^49*x2^35*x3^19*x4^2*z^21 + 5*x1^48*x2^36*x3^19*x4^2*z^21 + 2*x1^46*x2^38*x3^19*x4^2*z^21 - 6*x1^48*x2^35*x3^20*x4^2*z^21 - 2*x1^47*x2^36*x3^20*x4^2*z^21 - 3*x1^46*x2^37*x3^20*x4^2*z^21 - x1^45*x2^38*x3^20*x4^2*z^21 + x1^43*x2^40*x3^20*x4^2*z^21 + 2*x1^48*x2^34*x3^21*x4^2*z^21 + 5*x1^47*x2^35*x3^21*x4^2*z^21 - x1^46*x2^36*x3^21*x4^2*z^21 + 4*x1^45*x2^37*x3^21*x4^2*z^21 + x1^44*x2^38*x3^21*x4^2*z^21 - x1^43*x2^39*x3^21*x4^2*z^21 - x1^42*x2^40*x3^21*x4^2*z^21 - 4*x1^47*x2^34*x3^22*x4^2*z^21 - x1^46*x2^35*x3^22*x4^2*z^21 - 3*x1^45*x2^36*x3^22*x4^2*z^21 - x1^44*x2^37*x3^22*x4^2*z^21 + 2*x1^47*x2^33*x3^23*x4^2*z^21 + 2*x1^46*x2^34*x3^23*x4^2*z^21 + 4*x1^44*x2^36*x3^23*x4^2*z^21 - 2*x1^46*x2^33*x3^24*x4^2*z^21 - 2*x1^44*x2^35*x3^24*x4^2*z^21 - x1^43*x2^36*x3^24*x4^2*z^21 + x1^46*x2^32*x3^25*x4^2*z^21 + x1^45*x2^33*x3^25*x4^2*z^21 - x1^44*x2^34*x3^25*x4^2*z^21 + x1^43*x2^35*x3^25*x4^2*z^21 - x1^51*x2^34*x3^17*x4^3*z^21 + 2*x1^49*x2^36*x3^17*x4^3*z^21 + x1^47*x2^38*x3^17*x4^3*z^21 + 2*x1^50*x2^34*x3^18*x4^3*z^21 - 2*x1^49*x2^35*x3^18*x4^3*z^21 - 2*x1^48*x2^36*x3^18*x4^3*z^21 + x1^47*x2^37*x3^18*x4^3*z^21 - x1^46*x2^38*x3^18*x4^3*z^21 - 2*x1^49*x2^34*x3^19*x4^3*z^21 + 3*x1^48*x2^35*x3^19*x4^3*z^21 - x1^47*x2^36*x3^19*x4^3*z^21 + 2*x1^49*x2^33*x3^20*x4^3*z^21 + x1^48*x2^34*x3^20*x4^3*z^21 - 2*x1^47*x2^35*x3^20*x4^3*z^21 - 2*x1^45*x2^37*x3^20*x4^3*z^21 + 2*x1^44*x2^38*x3^20*x4^3*z^21 - x1^49*x2^32*x3^21*x4^3*z^21 + 3*x1^47*x2^34*x3^21*x4^3*z^21 + x1^44*x2^37*x3^21*x4^3*z^21 + x1^48*x2^32*x3^22*x4^3*z^21 + x1^45*x2^35*x3^22*x4^3*z^21 - 3*x1^44*x2^36*x3^22*x4^3*z^21 + x1^42*x2^38*x3^22*x4^3*z^21 + x1^41*x2^39*x3^22*x4^3*z^21 - x1^48*x2^31*x3^23*x4^3*z^21 - x1^47*x2^32*x3^23*x4^3*z^21 + 2*x1^46*x2^33*x3^23*x4^3*z^21 + x1^45*x2^34*x3^23*x4^3*z^21 + 2*x1^44*x2^35*x3^23*x4^3*z^21 - x1^43*x2^36*x3^23*x4^3*z^21 - x1^40*x2^39*x3^23*x4^3*z^21 - 2*x1^46*x2^32*x3^24*x4^3*z^21 - x1^45*x2^33*x3^24*x4^3*z^21 + x1^44*x2^34*x3^24*x4^3*z^21 - 3*x1^43*x2^35*x3^24*x4^3*z^21 + x1^45*x2^32*x3^25*x4^3*z^21 + x1^44*x2^33*x3^25*x4^3*z^21 + x1^43*x2^34*x3^25*x4^3*z^21 - x1^44*x2^32*x3^26*x4^3*z^21 + x1^43*x2^33*x3^26*x4^3*z^21 - x1^42*x2^34*x3^26*x4^3*z^21 + x1^52*x2^35*x3^14*x4^4*z^21 - 2*x1^51*x2^35*x3^15*x4^4*z^21 + x1^50*x2^36*x3^15*x4^4*z^21 + 2*x1^51*x2^34*x3^16*x4^4*z^21 + 2*x1^50*x2^35*x3^16*x4^4*z^21 - x1^49*x2^36*x3^16*x4^4*z^21 + 2*x1^48*x2^37*x3^16*x4^4*z^21 - 5*x1^50*x2^34*x3^17*x4^4*z^21 + x1^46*x2^38*x3^17*x4^4*z^21 + 2*x1^50*x2^33*x3^18*x4^4*z^21 + 5*x1^49*x2^34*x3^18*x4^4*z^21 - x1^48*x2^35*x3^18*x4^4*z^21 + 2*x1^47*x2^36*x3^18*x4^4*z^21 - x1^46*x2^37*x3^18*x4^4*z^21 + x1^44*x2^39*x3^18*x4^4*z^21 - 6*x1^49*x2^33*x3^19*x4^4*z^21 - 2*x1^48*x2^34*x3^19*x4^4*z^21 - x1^47*x2^35*x3^19*x4^4*z^21 + x1^45*x2^37*x3^19*x4^4*z^21 + x1^44*x2^38*x3^19*x4^4*z^21 - x1^43*x2^39*x3^19*x4^4*z^21 + 2*x1^49*x2^32*x3^20*x4^4*z^21 + 6*x1^48*x2^33*x3^20*x4^4*z^21 + 3*x1^46*x2^35*x3^20*x4^4*z^21 - 2*x1^45*x2^36*x3^20*x4^4*z^21 - 2*x1^44*x2^37*x3^20*x4^4*z^21 - x1^43*x2^38*x3^20*x4^4*z^21 + x1^42*x2^39*x3^20*x4^4*z^21 - 6*x1^48*x2^32*x3^21*x4^4*z^21 - 2*x1^47*x2^33*x3^21*x4^4*z^21 - 2*x1^46*x2^34*x3^21*x4^4*z^21 - 2*x1^45*x2^35*x3^21*x4^4*z^21 - x1^44*x2^36*x3^21*x4^4*z^21 + x1^41*x2^39*x3^21*x4^4*z^21 + 2*x1^48*x2^31*x3^22*x4^4*z^21 + 4*x1^47*x2^32*x3^22*x4^4*z^21 - x1^46*x2^33*x3^22*x4^4*z^21 + 4*x1^45*x2^34*x3^22*x4^4*z^21 - 4*x1^42*x2^37*x3^22*x4^4*z^21 + x1^41*x2^38*x3^22*x4^4*z^21 + x1^40*x2^39*x3^22*x4^4*z^21 - 5*x1^47*x2^31*x3^23*x4^4*z^21 - x1^46*x2^32*x3^23*x4^4*z^21 - x1^45*x2^33*x3^23*x4^4*z^21 - 2*x1^44*x2^34*x3^23*x4^4*z^21 + x1^43*x2^35*x3^23*x4^4*z^21 - x1^42*x2^36*x3^23*x4^4*z^21 + x1^41*x2^37*x3^23*x4^4*z^21 + x1^47*x2^30*x3^24*x4^4*z^21 + 2*x1^46*x2^31*x3^24*x4^4*z^21 - x1^45*x2^32*x3^24*x4^4*z^21 + 3*x1^44*x2^33*x3^24*x4^4*z^21 + x1^43*x2^34*x3^24*x4^4*z^21 - x1^41*x2^36*x3^24*x4^4*z^21 + x1^40*x2^37*x3^24*x4^4*z^21 + x1^39*x2^38*x3^24*x4^4*z^21 - 3*x1^46*x2^30*x3^25*x4^4*z^21 - x1^44*x2^32*x3^25*x4^4*z^21 - x1^43*x2^33*x3^25*x4^4*z^21 + 2*x1^42*x2^34*x3^25*x4^4*z^21 - 2*x1^44*x2^31*x3^26*x4^4*z^21 + x1^43*x2^31*x3^27*x4^4*z^21 + x1^42*x2^32*x3^27*x4^4*z^21 + 2*x1^41*x2^33*x3^27*x4^4*z^21 + x1^51*x2^35*x3^14*x4^5*z^21 - x1^51*x2^34*x3^15*x4^5*z^21 - x1^50*x2^35*x3^15*x4^5*z^21 + x1^49*x2^36*x3^15*x4^5*z^21 + 4*x1^50*x2^34*x3^16*x4^5*z^21 - x1^47*x2^37*x3^16*x4^5*z^21 - 2*x1^50*x2^33*x3^17*x4^5*z^21 - 4*x1^49*x2^34*x3^17*x4^5*z^21 + x1^48*x2^35*x3^17*x4^5*z^21 - 4*x1^47*x2^36*x3^17*x4^5*z^21 + 6*x1^49*x2^33*x3^18*x4^5*z^21 + x1^47*x2^35*x3^18*x4^5*z^21 + x1^46*x2^36*x3^18*x4^5*z^21 - 2*x1^45*x2^37*x3^18*x4^5*z^21 - 3*x1^44*x2^38*x3^18*x4^5*z^21 - 2*x1^49*x2^32*x3^19*x4^5*z^21 - 6*x1^48*x2^33*x3^19*x4^5*z^21 + 2*x1^47*x2^34*x3^19*x4^5*z^21 - 3*x1^46*x2^35*x3^19*x4^5*z^21 + 3*x1^45*x2^36*x3^19*x4^5*z^21 + x1^44*x2^37*x3^19*x4^5*z^21 + 3*x1^43*x2^38*x3^19*x4^5*z^21 + 6*x1^48*x2^32*x3^20*x4^5*z^21 + x1^47*x2^33*x3^20*x4^5*z^21 - 2*x1^44*x2^36*x3^20*x4^5*z^21 - 4*x1^42*x2^38*x3^20*x4^5*z^21 - x1^41*x2^39*x3^20*x4^5*z^21 - 2*x1^48*x2^31*x3^21*x4^5*z^21 - 6*x1^47*x2^32*x3^21*x4^5*z^21 + 2*x1^46*x2^33*x3^21*x4^5*z^21 - 4*x1^45*x2^34*x3^21*x4^5*z^21 + 3*x1^44*x2^35*x3^21*x4^5*z^21 + x1^43*x2^36*x3^21*x4^5*z^21 + 3*x1^42*x2^37*x3^21*x4^5*z^21 + 6*x1^47*x2^31*x3^22*x4^5*z^21 + 2*x1^46*x2^32*x3^22*x4^5*z^21 + 2*x1^44*x2^34*x3^22*x4^5*z^21 - 3*x1^43*x2^35*x3^22*x4^5*z^21 - x1^42*x2^36*x3^22*x4^5*z^21 - 2*x1^41*x2^37*x3^22*x4^5*z^21 + x1^39*x2^39*x3^22*x4^5*z^21 - 2*x1^47*x2^30*x3^23*x4^5*z^21 - 6*x1^46*x2^31*x3^23*x4^5*z^21 + 2*x1^45*x2^32*x3^23*x4^5*z^21 - 4*x1^44*x2^33*x3^23*x4^5*z^21 + x1^42*x2^35*x3^23*x4^5*z^21 + 3*x1^41*x2^36*x3^23*x4^5*z^21 - x1^40*x2^37*x3^23*x4^5*z^21 - x1^39*x2^38*x3^23*x4^5*z^21 + 5*x1^46*x2^30*x3^24*x4^5*z^21 + 2*x1^43*x2^33*x3^24*x4^5*z^21 - 2*x1^42*x2^34*x3^24*x4^5*z^21 - x1^41*x2^35*x3^24*x4^5*z^21 - 2*x1^40*x2^36*x3^24*x4^5*z^21 + x1^39*x2^37*x3^24*x4^5*z^21 - 3*x1^45*x2^30*x3^25*x4^5*z^21 + x1^44*x2^31*x3^25*x4^5*z^21 - 3*x1^43*x2^32*x3^25*x4^5*z^21 + x1^41*x2^34*x3^25*x4^5*z^21 + x1^40*x2^35*x3^25*x4^5*z^21 - 2*x1^39*x2^36*x3^25*x4^5*z^21 + x1^45*x2^29*x3^26*x4^5*z^21 - x1^43*x2^31*x3^26*x4^5*z^21 + x1^40*x2^34*x3^26*x4^5*z^21 - x1^39*x2^35*x3^26*x4^5*z^21 + x1^38*x2^36*x3^26*x4^5*z^21 + x1^41*x2^32*x3^27*x4^5*z^21 + x1^40*x2^33*x3^27*x4^5*z^21 - x1^41*x2^31*x3^28*x4^5*z^21 - x1^40*x2^32*x3^28*x4^5*z^21 + x1^39*x2^32*x3^29*x4^5*z^21 - x1^50*x2^35*x3^14*x4^6*z^21 - 2*x1^50*x2^34*x3^15*x4^6*z^21 + x1^50*x2^33*x3^16*x4^6*z^21 + 2*x1^49*x2^34*x3^16*x4^6*z^21 - x1^48*x2^35*x3^16*x4^6*z^21 + 2*x1^47*x2^36*x3^16*x4^6*z^21 - 2*x1^49*x2^33*x3^17*x4^6*z^21 - x1^48*x2^34*x3^17*x4^6*z^21 + x1^45*x2^37*x3^17*x4^6*z^21 + x1^49*x2^32*x3^18*x4^6*z^21 + 2*x1^48*x2^33*x3^18*x4^6*z^21 - 3*x1^47*x2^34*x3^18*x4^6*z^21 - 3*x1^45*x2^36*x3^18*x4^6*z^21 + x1^43*x2^38*x3^18*x4^6*z^21 - 2*x1^48*x2^32*x3^19*x4^6*z^21 + 2*x1^47*x2^33*x3^19*x4^6*z^21 + 3*x1^46*x2^34*x3^19*x4^6*z^21 - 2*x1^45*x2^35*x3^19*x4^6*z^21 + x1^44*x2^36*x3^19*x4^6*z^21 - x1^43*x2^37*x3^19*x4^6*z^21 - x1^42*x2^38*x3^19*x4^6*z^21 + 2*x1^47*x2^32*x3^20*x4^6*z^21 - 4*x1^46*x2^33*x3^20*x4^6*z^21 + x1^45*x2^34*x3^20*x4^6*z^21 - x1^44*x2^35*x3^20*x4^6*z^21 - x1^42*x2^37*x3^20*x4^6*z^21 + x1^41*x2^38*x3^20*x4^6*z^21 - 2*x1^47*x2^31*x3^21*x4^6*z^21 + 3*x1^45*x2^33*x3^21*x4^6*z^21 + x1^44*x2^34*x3^21*x4^6*z^21 + 5*x1^43*x2^35*x3^21*x4^6*z^21 - 2*x1^42*x2^36*x3^21*x4^6*z^21 + 3*x1^41*x2^37*x3^21*x4^6*z^21 - x1^40*x2^38*x3^21*x4^6*z^21 + x1^47*x2^30*x3^22*x4^6*z^21 + 2*x1^46*x2^31*x3^22*x4^6*z^21 - 4*x1^45*x2^32*x3^22*x4^6*z^21 - x1^43*x2^34*x3^22*x4^6*z^21 - 2*x1^41*x2^36*x3^22*x4^6*z^21 + x1^40*x2^37*x3^22*x4^6*z^21 + x1^39*x2^38*x3^22*x4^6*z^21 - 2*x1^46*x2^30*x3^23*x4^6*z^21 + 3*x1^44*x2^32*x3^23*x4^6*z^21 - x1^43*x2^33*x3^23*x4^6*z^21 + 5*x1^42*x2^34*x3^23*x4^6*z^21 + 3*x1^41*x2^35*x3^23*x4^6*z^21 + 2*x1^40*x2^36*x3^23*x4^6*z^21 - 3*x1^39*x2^37*x3^23*x4^6*z^21 - x1^38*x2^38*x3^23*x4^6*z^21 + 4*x1^45*x2^30*x3^24*x4^6*z^21 - 4*x1^44*x2^31*x3^24*x4^6*z^21 - x1^43*x2^32*x3^24*x4^6*z^21 - x1^42*x2^33*x3^24*x4^6*z^21 - x1^41*x2^34*x3^24*x4^6*z^21 - x1^40*x2^35*x3^24*x4^6*z^21 + 2*x1^39*x2^36*x3^24*x4^6*z^21 + x1^38*x2^37*x3^24*x4^6*z^21 + 3*x1^44*x2^30*x3^25*x4^6*z^21 + 3*x1^43*x2^31*x3^25*x4^6*z^21 - x1^42*x2^32*x3^25*x4^6*z^21 + 2*x1^41*x2^33*x3^25*x4^6*z^21 + x1^40*x2^34*x3^25*x4^6*z^21 + 2*x1^39*x2^35*x3^25*x4^6*z^21 - 2*x1^38*x2^36*x3^25*x4^6*z^21 + x1^44*x2^29*x3^26*x4^6*z^21 - x1^43*x2^30*x3^26*x4^6*z^21 + 2*x1^42*x2^31*x3^26*x4^6*z^21 - 2*x1^40*x2^33*x3^26*x4^6*z^21 + x1^38*x2^35*x3^26*x4^6*z^21 - x1^37*x2^36*x3^26*x4^6*z^21 + x1^41*x2^31*x3^27*x4^6*z^21 + x1^40*x2^32*x3^27*x4^6*z^21 - x1^39*x2^33*x3^27*x4^6*z^21 + x1^38*x2^34*x3^27*x4^6*z^21 - x1^37*x2^35*x3^27*x4^6*z^21 - x1^40*x2^31*x3^28*x4^6*z^21 - 2*x1^39*x2^32*x3^28*x4^6*z^21 - x1^49*x2^36*x3^13*x4^7*z^21 + x1^48*x2^36*x3^14*x4^7*z^21 + x1^46*x2^38*x3^14*x4^7*z^21 + x1^50*x2^33*x3^15*x4^7*z^21 - 3*x1^48*x2^35*x3^15*x4^7*z^21 - 2*x1^47*x2^36*x3^15*x4^7*z^21 - x1^45*x2^38*x3^15*x4^7*z^21 - x1^49*x2^33*x3^16*x4^7*z^21 + x1^47*x2^35*x3^16*x4^7*z^21 - x1^46*x2^36*x3^16*x4^7*z^21 + x1^45*x2^37*x3^16*x4^7*z^21 + x1^48*x2^33*x3^17*x4^7*z^21 - x1^47*x2^34*x3^17*x4^7*z^21 - 2*x1^46*x2^35*x3^17*x4^7*z^21 - x1^45*x2^36*x3^17*x4^7*z^21 + x1^46*x2^34*x3^18*x4^7*z^21 + x1^45*x2^35*x3^18*x4^7*z^21 - 2*x1^43*x2^37*x3^18*x4^7*z^21 - 2*x1^41*x2^39*x3^18*x4^7*z^21 + x1^45*x2^34*x3^19*x4^7*z^21 + x1^42*x2^37*x3^19*x4^7*z^21 + x1^41*x2^38*x3^19*x4^7*z^21 + 2*x1^40*x2^39*x3^19*x4^7*z^21 + x1^43*x2^35*x3^20*x4^7*z^21 - 2*x1^40*x2^38*x3^20*x4^7*z^21 + x1^41*x2^36*x3^21*x4^7*z^21 + 2*x1^39*x2^38*x3^21*x4^7*z^21 - x1^43*x2^33*x3^22*x4^7*z^21 - x1^42*x2^34*x3^22*x4^7*z^21 + x1^41*x2^35*x3^22*x4^7*z^21 - x1^40*x2^36*x3^22*x4^7*z^21 - x1^38*x2^38*x3^22*x4^7*z^21 - x1^46*x2^29*x3^23*x4^7*z^21 + x1^44*x2^31*x3^23*x4^7*z^21 - x1^43*x2^32*x3^23*x4^7*z^21 + x1^40*x2^35*x3^23*x4^7*z^21 + x1^39*x2^36*x3^23*x4^7*z^21 + x1^38*x2^37*x3^23*x4^7*z^21 + x1^45*x2^29*x3^24*x4^7*z^21 - x1^44*x2^30*x3^24*x4^7*z^21 - x1^43*x2^31*x3^24*x4^7*z^21 + x1^42*x2^32*x3^24*x4^7*z^21 - x1^41*x2^33*x3^24*x4^7*z^21 - 2*x1^39*x2^35*x3^24*x4^7*z^21 + x1^38*x2^36*x3^24*x4^7*z^21 - 2*x1^44*x2^29*x3^25*x4^7*z^21 - x1^41*x2^32*x3^25*x4^7*z^21 - x1^40*x2^33*x3^25*x4^7*z^21 + 2*x1^39*x2^34*x3^25*x4^7*z^21 + x1^43*x2^29*x3^26*x4^7*z^21 - x1^39*x2^33*x3^26*x4^7*z^21 - x1^38*x2^34*x3^26*x4^7*z^21 + x1^37*x2^35*x3^26*x4^7*z^21 - x1^42*x2^29*x3^27*x4^7*z^21 - x1^41*x2^30*x3^27*x4^7*z^21 - x1^40*x2^31*x3^27*x4^7*z^21 + x1^39*x2^32*x3^27*x4^7*z^21 + x1^38*x2^33*x3^27*x4^7*z^21 + x1^36*x2^35*x3^27*x4^7*z^21 + x1^41*x2^29*x3^28*x4^7*z^21 + x1^40*x2^30*x3^28*x4^7*z^21 - x1^39*x2^31*x3^28*x4^7*z^21 + x1^38*x2^32*x3^28*x4^7*z^21 + x1^37*x2^33*x3^28*x4^7*z^21 + x1^38*x2^31*x3^29*x4^7*z^21 + x1^50*x2^34*x3^13*x4^8*z^21 - x1^48*x2^36*x3^13*x4^8*z^21 + x1^47*x2^37*x3^13*x4^8*z^21 + x1^50*x2^33*x3^14*x4^8*z^21 - x1^49*x2^34*x3^14*x4^8*z^21 + 2*x1^48*x2^35*x3^14*x4^8*z^21 + x1^47*x2^36*x3^14*x4^8*z^21 - x1^46*x2^37*x3^14*x4^8*z^21 + x1^48*x2^34*x3^15*x4^8*z^21 - x1^47*x2^35*x3^15*x4^8*z^21 + x1^46*x2^36*x3^15*x4^8*z^21 - x1^45*x2^37*x3^15*x4^8*z^21 + x1^44*x2^38*x3^15*x4^8*z^21 + x1^48*x2^33*x3^16*x4^8*z^21 + x1^47*x2^34*x3^16*x4^8*z^21 + 2*x1^46*x2^35*x3^16*x4^8*z^21 + x1^45*x2^36*x3^16*x4^8*z^21 - 2*x1^43*x2^38*x3^16*x4^8*z^21 - x1^42*x2^39*x3^16*x4^8*z^21 - 2*x1^47*x2^33*x3^17*x4^8*z^21 - 3*x1^46*x2^34*x3^17*x4^8*z^21 + x1^45*x2^35*x3^17*x4^8*z^21 - 2*x1^44*x2^36*x3^17*x4^8*z^21 + 2*x1^43*x2^37*x3^17*x4^8*z^21 + 2*x1^42*x2^38*x3^17*x4^8*z^21 + 3*x1^41*x2^39*x3^17*x4^8*z^21 + x1^48*x2^31*x3^18*x4^8*z^21 + 2*x1^47*x2^32*x3^18*x4^8*z^21 + 4*x1^46*x2^33*x3^18*x4^8*z^21 + 2*x1^45*x2^34*x3^18*x4^8*z^21 + x1^44*x2^35*x3^18*x4^8*z^21 - x1^43*x2^36*x3^18*x4^8*z^21 - 2*x1^42*x2^37*x3^18*x4^8*z^21 - 3*x1^41*x2^38*x3^18*x4^8*z^21 - 3*x1^40*x2^39*x3^18*x4^8*z^21 - 2*x1^47*x2^31*x3^19*x4^8*z^21 - 2*x1^46*x2^32*x3^19*x4^8*z^21 - 5*x1^45*x2^33*x3^19*x4^8*z^21 - x1^43*x2^35*x3^19*x4^8*z^21 + 5*x1^42*x2^36*x3^19*x4^8*z^21 + 5*x1^40*x2^38*x3^19*x4^8*z^21 + x1^39*x2^39*x3^19*x4^8*z^21 + x1^47*x2^30*x3^20*x4^8*z^21 + 2*x1^46*x2^31*x3^20*x4^8*z^21 + 4*x1^45*x2^32*x3^20*x4^8*z^21 + 2*x1^44*x2^33*x3^20*x4^8*z^21 - x1^43*x2^34*x3^20*x4^8*z^21 - x1^42*x2^35*x3^20*x4^8*z^21 - 3*x1^41*x2^36*x3^20*x4^8*z^21 - 5*x1^39*x2^38*x3^20*x4^8*z^21 - 2*x1^46*x2^30*x3^21*x4^8*z^21 - x1^45*x2^31*x3^21*x4^8*z^21 - 4*x1^44*x2^32*x3^21*x4^8*z^21 - 3*x1^42*x2^34*x3^21*x4^8*z^21 + 2*x1^41*x2^35*x3^21*x4^8*z^21 + 2*x1^40*x2^36*x3^21*x4^8*z^21 + 4*x1^39*x2^37*x3^21*x4^8*z^21 + 2*x1^38*x2^38*x3^21*x4^8*z^21 + 2*x1^45*x2^30*x3^22*x4^8*z^21 + 2*x1^44*x2^31*x3^22*x4^8*z^21 + 4*x1^43*x2^32*x3^22*x4^8*z^21 + x1^42*x2^33*x3^22*x4^8*z^21 - 4*x1^40*x2^35*x3^22*x4^8*z^21 - 3*x1^39*x2^36*x3^22*x4^8*z^21 - 4*x1^38*x2^37*x3^22*x4^8*z^21 - x1^45*x2^29*x3^23*x4^8*z^21 - 2*x1^44*x2^30*x3^23*x4^8*z^21 - 3*x1^43*x2^31*x3^23*x4^8*z^21 - x1^42*x2^32*x3^23*x4^8*z^21 - 2*x1^41*x2^33*x3^23*x4^8*z^21 + 2*x1^40*x2^34*x3^23*x4^8*z^21 + 3*x1^38*x2^36*x3^23*x4^8*z^21 + x1^37*x2^37*x3^23*x4^8*z^21 + x1^45*x2^28*x3^24*x4^8*z^21 + x1^44*x2^29*x3^24*x4^8*z^21 + x1^42*x2^31*x3^24*x4^8*z^21 - 4*x1^39*x2^34*x3^24*x4^8*z^21 - 3*x1^38*x2^35*x3^24*x4^8*z^21 - 4*x1^37*x2^36*x3^24*x4^8*z^21 - x1^44*x2^28*x3^25*x4^8*z^21 + x1^43*x2^29*x3^25*x4^8*z^21 + x1^41*x2^31*x3^25*x4^8*z^21 + 2*x1^39*x2^33*x3^25*x4^8*z^21 + 2*x1^38*x2^34*x3^25*x4^8*z^21 + 2*x1^37*x2^35*x3^25*x4^8*z^21 + x1^36*x2^36*x3^25*x4^8*z^21 + x1^43*x2^28*x3^26*x4^8*z^21 - 2*x1^40*x2^31*x3^26*x4^8*z^21 - x1^38*x2^33*x3^26*x4^8*z^21 - x1^37*x2^34*x3^26*x4^8*z^21 - 2*x1^36*x2^35*x3^26*x4^8*z^21 - x1^42*x2^28*x3^27*x4^8*z^21 - x1^41*x2^29*x3^27*x4^8*z^21 - x1^39*x2^31*x3^27*x4^8*z^21 + x1^36*x2^34*x3^27*x4^8*z^21 - x1^39*x2^30*x3^28*x4^8*z^21 - x1^38*x2^31*x3^28*x4^8*z^21 - x1^37*x2^32*x3^28*x4^8*z^21 - x1^36*x2^33*x3^28*x4^8*z^21 - x1^35*x2^34*x3^28*x4^8*z^21 - x1^39*x2^29*x3^29*x4^8*z^21 - x1^36*x2^32*x3^29*x4^8*z^21 + x1^35*x2^33*x3^29*x4^8*z^21 - x1^49*x2^34*x3^13*x4^9*z^21 + x1^48*x2^34*x3^14*x4^9*z^21 + x1^46*x2^36*x3^14*x4^9*z^21 + x1^45*x2^37*x3^14*x4^9*z^21 - x1^44*x2^38*x3^14*x4^9*z^21 - x1^49*x2^32*x3^15*x4^9*z^21 - x1^47*x2^34*x3^15*x4^9*z^21 - 2*x1^46*x2^35*x3^15*x4^9*z^21 - x1^45*x2^36*x3^15*x4^9*z^21 - x1^44*x2^37*x3^15*x4^9*z^21 + 2*x1^48*x2^32*x3^16*x4^9*z^21 + 2*x1^47*x2^33*x3^16*x4^9*z^21 + 3*x1^46*x2^34*x3^16*x4^9*z^21 + x1^45*x2^35*x3^16*x4^9*z^21 + 3*x1^44*x2^36*x3^16*x4^9*z^21 + x1^43*x2^37*x3^16*x4^9*z^21 - x1^41*x2^39*x3^16*x4^9*z^21 - 2*x1^48*x2^31*x3^17*x4^9*z^21 - 4*x1^47*x2^32*x3^17*x4^9*z^21 - 3*x1^45*x2^34*x3^17*x4^9*z^21 - x1^43*x2^36*x3^17*x4^9*z^21 + x1^41*x2^38*x3^17*x4^9*z^21 + x1^40*x2^39*x3^17*x4^9*z^21 + 2*x1^47*x2^31*x3^18*x4^9*z^21 + 2*x1^46*x2^32*x3^18*x4^9*z^21 + 2*x1^45*x2^33*x3^18*x4^9*z^21 + x1^44*x2^34*x3^18*x4^9*z^21 - x1^43*x2^35*x3^18*x4^9*z^21 - x1^42*x2^36*x3^18*x4^9*z^21 + x1^41*x2^37*x3^18*x4^9*z^21 - x1^40*x2^38*x3^18*x4^9*z^21 - 2*x1^47*x2^30*x3^19*x4^9*z^21 - 4*x1^46*x2^31*x3^19*x4^9*z^21 - 2*x1^45*x2^32*x3^19*x4^9*z^21 - 2*x1^44*x2^33*x3^19*x4^9*z^21 + x1^43*x2^34*x3^19*x4^9*z^21 + x1^41*x2^36*x3^19*x4^9*z^21 + x1^39*x2^38*x3^19*x4^9*z^21 + 4*x1^46*x2^30*x3^20*x4^9*z^21 + x1^45*x2^31*x3^20*x4^9*z^21 + 3*x1^44*x2^32*x3^20*x4^9*z^21 + x1^42*x2^34*x3^20*x4^9*z^21 - 3*x1^41*x2^35*x3^20*x4^9*z^21 - x1^40*x2^36*x3^20*x4^9*z^21 - 2*x1^39*x2^37*x3^20*x4^9*z^21 - x1^46*x2^29*x3^21*x4^9*z^21 - 4*x1^45*x2^30*x3^21*x4^9*z^21 - 2*x1^44*x2^31*x3^21*x4^9*z^21 - 4*x1^43*x2^32*x3^21*x4^9*z^21 + 2*x1^42*x2^33*x3^21*x4^9*z^21 + 5*x1^40*x2^35*x3^21*x4^9*z^21 + 2*x1^38*x2^37*x3^21*x4^9*z^21 + 4*x1^45*x2^29*x3^22*x4^9*z^21 + 2*x1^44*x2^30*x3^22*x4^9*z^21 + 3*x1^43*x2^31*x3^22*x4^9*z^21 - 2*x1^40*x2^34*x3^22*x4^9*z^21 - 4*x1^39*x2^35*x3^22*x4^9*z^21 - 2*x1^38*x2^36*x3^22*x4^9*z^21 - x1^37*x2^37*x3^22*x4^9*z^21 - 2*x1^45*x2^28*x3^23*x4^9*z^21 - 4*x1^44*x2^29*x3^23*x4^9*z^21 - 4*x1^42*x2^31*x3^23*x4^9*z^21 + 2*x1^41*x2^32*x3^23*x4^9*z^21 + 5*x1^39*x2^34*x3^23*x4^9*z^21 + 3*x1^38*x2^35*x3^23*x4^9*z^21 + 2*x1^37*x2^36*x3^23*x4^9*z^21 + 4*x1^44*x2^28*x3^24*x4^9*z^21 + x1^43*x2^29*x3^24*x4^9*z^21 + x1^42*x2^30*x3^24*x4^9*z^21 + x1^41*x2^31*x3^24*x4^9*z^21 - x1^40*x2^32*x3^24*x4^9*z^21 - 2*x1^39*x2^33*x3^24*x4^9*z^21 - 2*x1^38*x2^34*x3^24*x4^9*z^21 - x1^37*x2^35*x3^24*x4^9*z^21 - 4*x1^43*x2^28*x3^25*x4^9*z^21 - 2*x1^41*x2^30*x3^25*x4^9*z^21 + 3*x1^40*x2^31*x3^25*x4^9*z^21 - x1^39*x2^32*x3^25*x4^9*z^21 + 3*x1^38*x2^33*x3^25*x4^9*z^21 + 2*x1^37*x2^34*x3^25*x4^9*z^21 + 2*x1^42*x2^28*x3^26*x4^9*z^21 + x1^40*x2^30*x3^26*x4^9*z^21 - 2*x1^37*x2^33*x3^26*x4^9*z^21 - x1^36*x2^34*x3^26*x4^9*z^21 - x1^35*x2^35*x3^26*x4^9*z^21 - 2*x1^41*x2^28*x3^27*x4^9*z^21 - 2*x1^40*x2^29*x3^27*x4^9*z^21 + x1^38*x2^31*x3^27*x4^9*z^21 + 3*x1^37*x2^32*x3^27*x4^9*z^21 + x1^36*x2^33*x3^27*x4^9*z^21 + x1^35*x2^34*x3^27*x4^9*z^21 + 2*x1^40*x2^28*x3^28*x4^9*z^21 - x1^37*x2^31*x3^28*x4^9*z^21 + x1^38*x2^29*x3^29*x4^9*z^21 - x1^37*x2^30*x3^29*x4^9*z^21 + x1^36*x2^31*x3^29*x4^9*z^21 - x1^34*x2^33*x3^29*x4^9*z^21 + x1^48*x2^34*x3^13*x4^10*z^21 - x1^46*x2^36*x3^13*x4^10*z^21 + x1^45*x2^37*x3^13*x4^10*z^21 + x1^47*x2^34*x3^14*x4^10*z^21 + x1^44*x2^37*x3^14*x4^10*z^21 - x1^47*x2^33*x3^15*x4^10*z^21 + x1^43*x2^37*x3^15*x4^10*z^21 + x1^42*x2^38*x3^15*x4^10*z^21 + x1^46*x2^33*x3^16*x4^10*z^21 - x1^43*x2^36*x3^16*x4^10*z^21 - 2*x1^46*x2^32*x3^17*x4^10*z^21 - 2*x1^43*x2^35*x3^17*x4^10*z^21 - x1^42*x2^36*x3^17*x4^10*z^21 - x1^41*x2^37*x3^17*x4^10*z^21 + x1^47*x2^30*x3^18*x4^10*z^21 - x1^46*x2^31*x3^18*x4^10*z^21 + x1^45*x2^32*x3^18*x4^10*z^21 - 2*x1^44*x2^33*x3^18*x4^10*z^21 + x1^42*x2^35*x3^18*x4^10*z^21 + x1^41*x2^36*x3^18*x4^10*z^21 - x1^40*x2^37*x3^18*x4^10*z^21 - x1^44*x2^32*x3^19*x4^10*z^21 - x1^43*x2^33*x3^19*x4^10*z^21 - x1^42*x2^34*x3^19*x4^10*z^21 + x1^41*x2^35*x3^19*x4^10*z^21 - 2*x1^40*x2^36*x3^19*x4^10*z^21 + x1^46*x2^29*x3^20*x4^10*z^21 + x1^39*x2^36*x3^20*x4^10*z^21 - 2*x1^45*x2^29*x3^21*x4^10*z^21 + x1^44*x2^30*x3^21*x4^10*z^21 + x1^45*x2^28*x3^22*x4^10*z^21 + 2*x1^44*x2^29*x3^22*x4^10*z^21 - x1^43*x2^30*x3^22*x4^10*z^21 + 2*x1^42*x2^31*x3^22*x4^10*z^21 - 2*x1^44*x2^28*x3^23*x4^10*z^21 - 2*x1^43*x2^29*x3^23*x4^10*z^21 + x1^40*x2^32*x3^23*x4^10*z^21 + 2*x1^43*x2^28*x3^24*x4^10*z^21 + x1^42*x2^29*x3^24*x4^10*z^21 + x1^41*x2^30*x3^24*x4^10*z^21 - x1^40*x2^31*x3^24*x4^10*z^21 + x1^38*x2^33*x3^24*x4^10*z^21 - x1^42*x2^28*x3^25*x4^10*z^21 - x1^41*x2^29*x3^25*x4^10*z^21 - x1^39*x2^31*x3^25*x4^10*z^21 + x1^38*x2^32*x3^25*x4^10*z^21 - x1^37*x2^33*x3^25*x4^10*z^21 + x1^40*x2^29*x3^26*x4^10*z^21 - 3*x1^39*x2^30*x3^26*x4^10*z^21 - x1^38*x2^31*x3^26*x4^10*z^21 - x1^37*x2^32*x3^26*x4^10*z^21 + x1^36*x2^33*x3^26*x4^10*z^21 + 2*x1^35*x2^34*x3^26*x4^10*z^21 - x1^37*x2^31*x3^27*x4^10*z^21 + x1^36*x2^32*x3^27*x4^10*z^21 - x1^38*x2^29*x3^28*x4^10*z^21 + x1^37*x2^30*x3^28*x4^10*z^21 - 3*x1^36*x2^31*x3^28*x4^10*z^21 - x1^35*x2^32*x3^28*x4^10*z^21 + 2*x1^34*x2^33*x3^28*x4^10*z^21 + x1^37*x2^29*x3^29*x4^10*z^21 + x1^34*x2^32*x3^29*x4^10*z^21 - x1^47*x2^33*x3^14*x4^11*z^21 - x1^46*x2^34*x3^14*x4^11*z^21 + 2*x1^45*x2^35*x3^14*x4^11*z^21 - x1^43*x2^37*x3^14*x4^11*z^21 + x1^47*x2^32*x3^15*x4^11*z^21 - x1^46*x2^33*x3^15*x4^11*z^21 + x1^45*x2^34*x3^15*x4^11*z^21 - x1^44*x2^35*x3^15*x4^11*z^21 + x1^41*x2^38*x3^15*x4^11*z^21 + 2*x1^43*x2^35*x3^16*x4^11*z^21 + 2*x1^42*x2^36*x3^16*x4^11*z^21 + 2*x1^41*x2^37*x3^16*x4^11*z^21 - x1^40*x2^38*x3^16*x4^11*z^21 + x1^47*x2^30*x3^17*x4^11*z^21 + 3*x1^46*x2^31*x3^17*x4^11*z^21 - 2*x1^45*x2^32*x3^17*x4^11*z^21 - x1^44*x2^33*x3^17*x4^11*z^21 - 3*x1^43*x2^34*x3^17*x4^11*z^21 - x1^41*x2^36*x3^17*x4^11*z^21 + x1^39*x2^38*x3^17*x4^11*z^21 - x1^46*x2^30*x3^18*x4^11*z^21 + x1^45*x2^31*x3^18*x4^11*z^21 + 3*x1^44*x2^32*x3^18*x4^11*z^21 + x1^42*x2^34*x3^18*x4^11*z^21 + 3*x1^40*x2^36*x3^18*x4^11*z^21 - x1^39*x2^37*x3^18*x4^11*z^21 - x1^38*x2^38*x3^18*x4^11*z^21 + 4*x1^45*x2^30*x3^19*x4^11*z^21 - 2*x1^44*x2^31*x3^19*x4^11*z^21 + 2*x1^43*x2^32*x3^19*x4^11*z^21 - 4*x1^42*x2^33*x3^19*x4^11*z^21 - x1^41*x2^34*x3^19*x4^11*z^21 - 3*x1^40*x2^35*x3^19*x4^11*z^21 + x1^38*x2^37*x3^19*x4^11*z^21 - x1^45*x2^29*x3^20*x4^11*z^21 + 2*x1^43*x2^31*x3^20*x4^11*z^21 + 2*x1^42*x2^32*x3^20*x4^11*z^21 + 3*x1^41*x2^33*x3^20*x4^11*z^21 - x1^40*x2^34*x3^20*x4^11*z^21 + 4*x1^39*x2^35*x3^20*x4^11*z^21 - 2*x1^38*x2^36*x3^20*x4^11*z^21 + 3*x1^44*x2^29*x3^21*x4^11*z^21 - x1^43*x2^30*x3^21*x4^11*z^21 + 3*x1^42*x2^31*x3^21*x4^11*z^21 - 2*x1^41*x2^32*x3^21*x4^11*z^21 - 4*x1^39*x2^34*x3^21*x4^11*z^21 - 2*x1^38*x2^35*x3^21*x4^11*z^21 + 2*x1^37*x2^36*x3^21*x4^11*z^21 - x1^44*x2^28*x3^22*x4^11*z^21 - 2*x1^43*x2^29*x3^22*x4^11*z^21 - x1^41*x2^31*x3^22*x4^11*z^21 + 4*x1^40*x2^32*x3^22*x4^11*z^21 + x1^39*x2^33*x3^22*x4^11*z^21 + 4*x1^38*x2^34*x3^22*x4^11*z^21 - 2*x1^37*x2^35*x3^22*x4^11*z^21 - x1^36*x2^36*x3^22*x4^11*z^21 + x1^43*x2^28*x3^23*x4^11*z^21 - x1^42*x2^29*x3^23*x4^11*z^21 - 3*x1^41*x2^30*x3^23*x4^11*z^21 - 4*x1^40*x2^31*x3^23*x4^11*z^21 - 3*x1^38*x2^33*x3^23*x4^11*z^21 + 2*x1^36*x2^35*x3^23*x4^11*z^21 + x1^41*x2^29*x3^24*x4^11*z^21 + x1^40*x2^30*x3^24*x4^11*z^21 - 3*x1^38*x2^32*x3^24*x4^11*z^21 + 4*x1^37*x2^33*x3^24*x4^11*z^21 - 2*x1^36*x2^34*x3^24*x4^11*z^21 - x1^35*x2^35*x3^24*x4^11*z^21 + x1^40*x2^29*x3^25*x4^11*z^21 + x1^38*x2^31*x3^25*x4^11*z^21 + 2*x1^35*x2^34*x3^25*x4^11*z^21 - 3*x1^35*x2^33*x3^26*x4^11*z^21 + x1^37*x2^30*x3^27*x4^11*z^21 - 2*x1^36*x2^31*x3^27*x4^11*z^21 - x1^35*x2^32*x3^27*x4^11*z^21 + 2*x1^34*x2^33*x3^27*x4^11*z^21 + x1^35*x2^31*x3^28*x4^11*z^21 - x1^34*x2^32*x3^28*x4^11*z^21 - x1^45*x2^35*x3^13*x4^12*z^21 + x1^44*x2^35*x3^14*x4^12*z^21 + x1^46*x2^32*x3^15*x4^12*z^21 + x1^45*x2^33*x3^15*x4^12*z^21 - 2*x1^44*x2^34*x3^15*x4^12*z^21 - x1^43*x2^35*x3^15*x4^12*z^21 + x1^41*x2^37*x3^15*x4^12*z^21 - 2*x1^46*x2^31*x3^16*x4^12*z^21 + 2*x1^45*x2^32*x3^16*x4^12*z^21 + x1^44*x2^33*x3^16*x4^12*z^21 - x1^40*x2^37*x3^16*x4^12*z^21 + x1^45*x2^31*x3^17*x4^12*z^21 - 2*x1^44*x2^32*x3^17*x4^12*z^21 - x1^43*x2^33*x3^17*x4^12*z^21 + x1^41*x2^35*x3^17*x4^12*z^21 + 3*x1^39*x2^37*x3^17*x4^12*z^21 - x1^46*x2^29*x3^18*x4^12*z^21 - x1^45*x2^30*x3^18*x4^12*z^21 + 3*x1^44*x2^31*x3^18*x4^12*z^21 + x1^43*x2^32*x3^18*x4^12*z^21 + x1^42*x2^33*x3^18*x4^12*z^21 - x1^41*x2^34*x3^18*x4^12*z^21 - x1^40*x2^35*x3^18*x4^12*z^21 - 3*x1^39*x2^36*x3^18*x4^12*z^21 - 3*x1^38*x2^37*x3^18*x4^12*z^21 + x1^45*x2^29*x3^19*x4^12*z^21 - 4*x1^43*x2^31*x3^19*x4^12*z^21 - x1^41*x2^33*x3^19*x4^12*z^21 + 4*x1^40*x2^34*x3^19*x4^12*z^21 - 2*x1^39*x2^35*x3^19*x4^12*z^21 + 6*x1^38*x2^36*x3^19*x4^12*z^21 + x1^37*x2^37*x3^19*x4^12*z^21 - 3*x1^44*x2^29*x3^20*x4^12*z^21 + 6*x1^43*x2^30*x3^20*x4^12*z^21 + 3*x1^41*x2^32*x3^20*x4^12*z^21 - x1^40*x2^33*x3^20*x4^12*z^21 - x1^38*x2^35*x3^20*x4^12*z^21 - 6*x1^37*x2^36*x3^20*x4^12*z^21 - x1^44*x2^28*x3^21*x4^12*z^21 - 5*x1^42*x2^30*x3^21*x4^12*z^21 - x1^41*x2^31*x3^21*x4^12*z^21 - 5*x1^40*x2^32*x3^21*x4^12*z^21 + 3*x1^39*x2^33*x3^21*x4^12*z^21 - 2*x1^38*x2^34*x3^21*x4^12*z^21 + 6*x1^37*x2^35*x3^21*x4^12*z^21 + 2*x1^36*x2^36*x3^21*x4^12*z^21 + 3*x1^42*x2^29*x3^22*x4^12*z^21 + 2*x1^41*x2^30*x3^22*x4^12*z^21 + 2*x1^40*x2^31*x3^22*x4^12*z^21 - x1^37*x2^34*x3^22*x4^12*z^21 - 6*x1^36*x2^35*x3^22*x4^12*z^21 - x1^42*x2^28*x3^23*x4^12*z^21 - 3*x1^41*x2^29*x3^23*x4^12*z^21 - 3*x1^39*x2^31*x3^23*x4^12*z^21 + 2*x1^38*x2^32*x3^23*x4^12*z^21 - 2*x1^37*x2^33*x3^23*x4^12*z^21 + 6*x1^36*x2^34*x3^23*x4^12*z^21 + 2*x1^35*x2^35*x3^23*x4^12*z^21 + x1^41*x2^28*x3^24*x4^12*z^21 + x1^40*x2^29*x3^24*x4^12*z^21 + x1^39*x2^30*x3^24*x4^12*z^21 - x1^38*x2^31*x3^24*x4^12*z^21 + x1^37*x2^32*x3^24*x4^12*z^21 - 2*x1^36*x2^33*x3^24*x4^12*z^21 - 6*x1^35*x2^34*x3^24*x4^12*z^21 - x1^40*x2^28*x3^25*x4^12*z^21 - 2*x1^39*x2^29*x3^25*x4^12*z^21 - x1^38*x2^30*x3^25*x4^12*z^21 + 3*x1^37*x2^31*x3^25*x4^12*z^21 - 3*x1^36*x2^32*x3^25*x4^12*z^21 + 5*x1^35*x2^33*x3^25*x4^12*z^21 + 2*x1^34*x2^34*x3^25*x4^12*z^21 + x1^38*x2^29*x3^26*x4^12*z^21 + x1^35*x2^32*x3^26*x4^12*z^21 - 5*x1^34*x2^33*x3^26*x4^12*z^21 - x1^35*x2^31*x3^27*x4^12*z^21 + 2*x1^34*x2^32*x3^27*x4^12*z^21 + 2*x1^33*x2^33*x3^27*x4^12*z^21 + x1^34*x2^31*x3^28*x4^12*z^21 - 2*x1^33*x2^32*x3^28*x4^12*z^21 + x1^44*x2^34*x3^14*x4^13*z^21 + x1^43*x2^35*x3^14*x4^13*z^21 - 2*x1^43*x2^34*x3^15*x4^13*z^21 - x1^42*x2^35*x3^15*x4^13*z^21 - x1^41*x2^36*x3^15*x4^13*z^21 + x1^43*x2^33*x3^16*x4^13*z^21 - 2*x1^41*x2^35*x3^16*x4^13*z^21 - x1^39*x2^37*x3^16*x4^13*z^21 + x1^45*x2^30*x3^17*x4^13*z^21 - x1^44*x2^31*x3^17*x4^13*z^21 + x1^43*x2^32*x3^17*x4^13*z^21 + x1^41*x2^34*x3^17*x4^13*z^21 - x1^39*x2^36*x3^17*x4^13*z^21 + 2*x1^38*x2^37*x3^17*x4^13*z^21 - 2*x1^44*x2^30*x3^18*x4^13*z^21 + x1^43*x2^31*x3^18*x4^13*z^21 + x1^42*x2^32*x3^18*x4^13*z^21 - x1^41*x2^33*x3^18*x4^13*z^21 - 4*x1^40*x2^34*x3^18*x4^13*z^21 + x1^39*x2^35*x3^18*x4^13*z^21 - 3*x1^38*x2^36*x3^18*x4^13*z^21 - 2*x1^37*x2^37*x3^18*x4^13*z^21 + x1^44*x2^29*x3^19*x4^13*z^21 - 2*x1^43*x2^30*x3^19*x4^13*z^21 - x1^42*x2^31*x3^19*x4^13*z^21 + 2*x1^40*x2^33*x3^19*x4^13*z^21 + x1^38*x2^35*x3^19*x4^13*z^21 + 6*x1^37*x2^36*x3^19*x4^13*z^21 - 2*x1^43*x2^29*x3^20*x4^13*z^21 + 3*x1^42*x2^30*x3^20*x4^13*z^21 - 4*x1^39*x2^33*x3^20*x4^13*z^21 - 6*x1^37*x2^35*x3^20*x4^13*z^21 - 2*x1^36*x2^36*x3^20*x4^13*z^21 + x1^43*x2^28*x3^21*x4^13*z^21 - 3*x1^42*x2^29*x3^21*x4^13*z^21 - 3*x1^41*x2^30*x3^21*x4^13*z^21 - x1^40*x2^31*x3^21*x4^13*z^21 + 2*x1^39*x2^32*x3^21*x4^13*z^21 + 2*x1^38*x2^33*x3^21*x4^13*z^21 + 2*x1^37*x2^34*x3^21*x4^13*z^21 + 6*x1^36*x2^35*x3^21*x4^13*z^21 - x1^43*x2^27*x3^22*x4^13*z^21 + 2*x1^41*x2^29*x3^22*x4^13*z^21 + 2*x1^39*x2^31*x3^22*x4^13*z^21 - 4*x1^38*x2^32*x3^22*x4^13*z^21 - 6*x1^36*x2^34*x3^22*x4^13*z^21 - 2*x1^35*x2^35*x3^22*x4^13*z^21 + x1^42*x2^27*x3^23*x4^13*z^21 - x1^41*x2^28*x3^23*x4^13*z^21 - 2*x1^40*x2^29*x3^23*x4^13*z^21 - 3*x1^39*x2^30*x3^23*x4^13*z^21 + x1^37*x2^32*x3^23*x4^13*z^21 + 2*x1^36*x2^33*x3^23*x4^13*z^21 + 6*x1^35*x2^34*x3^23*x4^13*z^21 + x1^40*x2^28*x3^24*x4^13*z^21 + x1^39*x2^29*x3^24*x4^13*z^21 + 2*x1^38*x2^30*x3^24*x4^13*z^21 - 3*x1^37*x2^31*x3^24*x4^13*z^21 + x1^36*x2^32*x3^24*x4^13*z^21 - 6*x1^35*x2^33*x3^24*x4^13*z^21 - 2*x1^34*x2^34*x3^24*x4^13*z^21 + x1^39*x2^28*x3^25*x4^13*z^21 + 2*x1^36*x2^31*x3^25*x4^13*z^21 + x1^35*x2^32*x3^25*x4^13*z^21 + 6*x1^34*x2^33*x3^25*x4^13*z^21 + x1^37*x2^29*x3^26*x4^13*z^21 - 2*x1^34*x2^32*x3^26*x4^13*z^21 - 2*x1^33*x2^33*x3^26*x4^13*z^21 + x1^35*x2^30*x3^27*x4^13*z^21 + 2*x1^33*x2^32*x3^27*x4^13*z^21 - x1^32*x2^32*x3^28*x4^13*z^21 - x1^42*x2^34*x3^15*x4^14*z^21 - x1^41*x2^35*x3^15*x4^14*z^21 + 2*x1^42*x2^33*x3^16*x4^14*z^21 - x1^42*x2^32*x3^17*x4^14*z^21 - x1^41*x2^33*x3^17*x4^14*z^21 - x1^40*x2^34*x3^17*x4^14*z^21 - 3*x1^39*x2^35*x3^17*x4^14*z^21 - x1^44*x2^29*x3^18*x4^14*z^21 - x1^43*x2^30*x3^18*x4^14*z^21 + 3*x1^41*x2^32*x3^18*x4^14*z^21 - x1^40*x2^33*x3^18*x4^14*z^21 - x1^37*x2^36*x3^18*x4^14*z^21 + x1^44*x2^28*x3^19*x4^14*z^21 + x1^43*x2^29*x3^19*x4^14*z^21 - x1^42*x2^30*x3^19*x4^14*z^21 - x1^41*x2^31*x3^19*x4^14*z^21 + x1^40*x2^32*x3^19*x4^14*z^21 + x1^39*x2^33*x3^19*x4^14*z^21 - 3*x1^38*x2^34*x3^19*x4^14*z^21 + x1^36*x2^36*x3^19*x4^14*z^21 - 2*x1^43*x2^28*x3^20*x4^14*z^21 - x1^41*x2^30*x3^20*x4^14*z^21 + 2*x1^40*x2^31*x3^20*x4^14*z^21 + 3*x1^38*x2^33*x3^20*x4^14*z^21 - 2*x1^36*x2^35*x3^20*x4^14*z^21 + x1^43*x2^27*x3^21*x4^14*z^21 + 2*x1^42*x2^28*x3^21*x4^14*z^21 - 2*x1^40*x2^30*x3^21*x4^14*z^21 + 2*x1^38*x2^32*x3^21*x4^14*z^21 - 2*x1^37*x2^33*x3^21*x4^14*z^21 + 2*x1^36*x2^34*x3^21*x4^14*z^21 - x1^42*x2^27*x3^22*x4^14*z^21 - x1^41*x2^28*x3^22*x4^14*z^21 - 2*x1^40*x2^29*x3^22*x4^14*z^21 + x1^39*x2^30*x3^22*x4^14*z^21 - x1^38*x2^31*x3^22*x4^14*z^21 + 2*x1^37*x2^32*x3^22*x4^14*z^21 + x1^36*x2^33*x3^22*x4^14*z^21 - 2*x1^35*x2^34*x3^22*x4^14*z^21 + x1^40*x2^28*x3^23*x4^14*z^21 + x1^39*x2^29*x3^23*x4^14*z^21 - x1^38*x2^30*x3^23*x4^14*z^21 - x1^37*x2^31*x3^23*x4^14*z^21 - 2*x1^36*x2^32*x3^23*x4^14*z^21 + 2*x1^35*x2^33*x3^23*x4^14*z^21 + x1^34*x2^34*x3^23*x4^14*z^21 + x1^38*x2^29*x3^24*x4^14*z^21 - x1^37*x2^30*x3^24*x4^14*z^21 + 2*x1^36*x2^31*x3^24*x4^14*z^21 - 2*x1^34*x2^33*x3^24*x4^14*z^21 + x1^37*x2^29*x3^25*x4^14*z^21 + x1^36*x2^30*x3^25*x4^14*z^21 - 3*x1^35*x2^31*x3^25*x4^14*z^21 + x1^34*x2^32*x3^25*x4^14*z^21 + x1^33*x2^33*x3^25*x4^14*z^21 - x1^36*x2^29*x3^26*x4^14*z^21 + x1^34*x2^31*x3^26*x4^14*z^21 - x1^33*x2^32*x3^26*x4^14*z^21 + x1^41*x2^33*x3^16*x4^15*z^21 + x1^40*x2^34*x3^16*x4^15*z^21 + x1^39*x2^35*x3^16*x4^15*z^21 - 3*x1^41*x2^32*x3^17*x4^15*z^21 + x1^40*x2^33*x3^17*x4^15*z^21 - x1^39*x2^34*x3^17*x4^15*z^21 - x1^38*x2^35*x3^17*x4^15*z^21 + 2*x1^41*x2^31*x3^18*x4^15*z^21 + 2*x1^40*x2^32*x3^18*x4^15*z^21 + 4*x1^38*x2^34*x3^18*x4^15*z^21 + x1^43*x2^28*x3^19*x4^15*z^21 + x1^42*x2^29*x3^19*x4^15*z^21 - 5*x1^40*x2^31*x3^19*x4^15*z^21 + x1^39*x2^32*x3^19*x4^15*z^21 - 3*x1^38*x2^33*x3^19*x4^15*z^21 - 2*x1^37*x2^34*x3^19*x4^15*z^21 - x1^43*x2^27*x3^20*x4^15*z^21 - x1^42*x2^28*x3^20*x4^15*z^21 + x1^41*x2^29*x3^20*x4^15*z^21 + x1^40*x2^30*x3^20*x4^15*z^21 + 2*x1^39*x2^31*x3^20*x4^15*z^21 + 2*x1^38*x2^32*x3^20*x4^15*z^21 + 5*x1^37*x2^33*x3^20*x4^15*z^21 + 2*x1^42*x2^27*x3^21*x4^15*z^21 - x1^41*x2^28*x3^21*x4^15*z^21 - 4*x1^39*x2^30*x3^21*x4^15*z^21 - 6*x1^37*x2^32*x3^21*x4^15*z^21 - 2*x1^36*x2^33*x3^21*x4^15*z^21 - x1^41*x2^27*x3^22*x4^15*z^21 + 2*x1^39*x2^29*x3^22*x4^15*z^21 + 2*x1^38*x2^30*x3^22*x4^15*z^21 + 2*x1^37*x2^31*x3^22*x4^15*z^21 + 6*x1^36*x2^32*x3^22*x4^15*z^21 + x1^40*x2^27*x3^23*x4^15*z^21 - 3*x1^38*x2^29*x3^23*x4^15*z^21 - 5*x1^36*x2^31*x3^23*x4^15*z^21 - 2*x1^35*x2^32*x3^23*x4^15*z^21 - x1^39*x2^27*x3^24*x4^15*z^21 + x1^38*x2^28*x3^24*x4^15*z^21 + x1^37*x2^29*x3^24*x4^15*z^21 + x1^36*x2^30*x3^24*x4^15*z^21 + 5*x1^35*x2^31*x3^24*x4^15*z^21 - 2*x1^37*x2^28*x3^25*x4^15*z^21 - 3*x1^35*x2^30*x3^25*x4^15*z^21 - 2*x1^34*x2^31*x3^25*x4^15*z^21 - x1^35*x2^29*x3^26*x4^15*z^21 + 3*x1^34*x2^30*x3^26*x4^15*z^21 - 2*x1^33*x2^30*x3^27*x4^15*z^21 + x1^40*x2^31*x3^18*x4^16*z^21 - x1^39*x2^32*x3^18*x4^16*z^21 + x1^38*x2^33*x3^18*x4^16*z^21 + x1^37*x2^34*x3^18*x4^16*z^21 - x1^40*x2^30*x3^19*x4^16*z^21 - 2*x1^39*x2^31*x3^19*x4^16*z^21 - 2*x1^37*x2^33*x3^19*x4^16*z^21 + 4*x1^39*x2^30*x3^20*x4^16*z^21 + 2*x1^37*x2^32*x3^20*x4^16*z^21 + 2*x1^36*x2^33*x3^20*x4^16*z^21 + x1^40*x2^28*x3^21*x4^16*z^21 - 2*x1^38*x2^30*x3^21*x4^16*z^21 - x1^37*x2^31*x3^21*x4^16*z^21 - 4*x1^36*x2^32*x3^21*x4^16*z^21 - x1^39*x2^28*x3^22*x4^16*z^21 + 2*x1^38*x2^29*x3^22*x4^16*z^21 - 2*x1^37*x2^30*x3^22*x4^16*z^21 + 5*x1^36*x2^31*x3^22*x4^16*z^21 + 2*x1^35*x2^32*x3^22*x4^16*z^21 + x1^39*x2^27*x3^23*x4^16*z^21 - x1^37*x2^29*x3^23*x4^16*z^21 - x1^36*x2^30*x3^23*x4^16*z^21 - 6*x1^35*x2^31*x3^23*x4^16*z^21 - x1^38*x2^27*x3^24*x4^16*z^21 - x1^36*x2^29*x3^24*x4^16*z^21 + 2*x1^35*x2^30*x3^24*x4^16*z^21 + 2*x1^34*x2^31*x3^24*x4^16*z^21 + x1^35*x2^29*x3^25*x4^16*z^21 - 2*x1^34*x2^30*x3^25*x4^16*z^21 + x1^33*x2^30*x3^26*x4^16*z^21 - x1^38*x2^29*x3^21*x4^17*z^21 - x1^37*x2^30*x3^21*x4^17*z^21 - x1^36*x2^31*x3^21*x4^17*z^21 - x1^36*x2^30*x3^22*x4^17*z^21 + x1^35*x2^31*x3^22*x4^17*z^21 + x1^36*x2^29*x3^23*x4^17*z^21 - x1^34*x2^31*x3^23*x4^17*z^21 + x1^48*x2^34*x3^18*z^20 - x1^46*x2^36*x3^18*z^20 - x1^47*x2^34*x3^19*z^20 + x1^46*x2^35*x3^19*z^20 + x1^46*x2^34*x3^20*z^20 + x1^45*x2^35*x3^20*z^20 + x1^44*x2^36*x3^20*z^20 - x1^49*x2^35*x3^15*x4*z^20 + 2*x1^48*x2^35*x3^16*x4*z^20 - x1^47*x2^36*x3^16*x4*z^20 - 2*x1^48*x2^34*x3^17*x4*z^20 - 2*x1^47*x2^35*x3^17*x4*z^20 + x1^46*x2^36*x3^17*x4*z^20 + x1^44*x2^38*x3^17*x4*z^20 + 6*x1^47*x2^34*x3^18*x4*z^20 + x1^46*x2^35*x3^18*x4*z^20 + x1^45*x2^36*x3^18*x4*z^20 - x1^43*x2^38*x3^18*x4*z^20 - 2*x1^47*x2^33*x3^19*x4*z^20 - 5*x1^46*x2^34*x3^19*x4*z^20 + 2*x1^45*x2^35*x3^19*x4*z^20 - 2*x1^44*x2^36*x3^19*x4*z^20 + x1^43*x2^37*x3^19*x4*z^20 + 4*x1^46*x2^33*x3^20*x4*z^20 + x1^45*x2^34*x3^20*x4*z^20 + 2*x1^44*x2^35*x3^20*x4*z^20 - x1^42*x2^37*x3^20*x4*z^20 - 2*x1^46*x2^32*x3^21*x4*z^20 - 2*x1^45*x2^33*x3^21*x4*z^20 - 4*x1^43*x2^35*x3^21*x4*z^20 + 2*x1^45*x2^32*x3^22*x4*z^20 + 2*x1^43*x2^34*x3^22*x4*z^20 + x1^42*x2^35*x3^22*x4*z^20 - x1^45*x2^31*x3^23*x4*z^20 - x1^44*x2^32*x3^23*x4*z^20 + x1^43*x2^33*x3^23*x4*z^20 - x1^42*x2^34*x3^23*x4*z^20 + 2*x1^49*x2^35*x3^14*x4^2*z^20 - 3*x1^48*x2^35*x3^15*x4^2*z^20 + x1^47*x2^36*x3^15*x4^2*z^20 + 2*x1^48*x2^34*x3^16*x4^2*z^20 + 3*x1^47*x2^35*x3^16*x4^2*z^20 + 2*x1^45*x2^37*x3^16*x4^2*z^20 - 5*x1^47*x2^34*x3^17*x4^2*z^20 - x1^46*x2^35*x3^17*x4^2*z^20 - x1^45*x2^36*x3^17*x4^2*z^20 - x1^44*x2^37*x3^17*x4^2*z^20 + x1^43*x2^38*x3^17*x4^2*z^20 + 2*x1^47*x2^33*x3^18*x4^2*z^20 + 5*x1^46*x2^34*x3^18*x4^2*z^20 + 3*x1^44*x2^36*x3^18*x4^2*z^20 - x1^42*x2^38*x3^18*x4^2*z^20 - 6*x1^46*x2^33*x3^19*x4^2*z^20 - 2*x1^45*x2^34*x3^19*x4^2*z^20 - 2*x1^44*x2^35*x3^19*x4^2*z^20 - 2*x1^43*x2^36*x3^19*x4^2*z^20 + x1^41*x2^38*x3^19*x4^2*z^20 + 2*x1^46*x2^32*x3^20*x4^2*z^20 + 6*x1^45*x2^33*x3^20*x4^2*z^20 + 4*x1^43*x2^35*x3^20*x4^2*z^20 + x1^41*x2^37*x3^20*x4^2*z^20 - 2*x1^40*x2^38*x3^20*x4^2*z^20 - 5*x1^45*x2^32*x3^21*x4^2*z^20 - 2*x1^44*x2^33*x3^21*x4^2*z^20 - 2*x1^43*x2^34*x3^21*x4^2*z^20 - x1^42*x2^35*x3^21*x4^2*z^20 - x1^41*x2^36*x3^21*x4^2*z^20 + x1^40*x2^37*x3^21*x4^2*z^20 + x1^39*x2^38*x3^21*x4^2*z^20 + 2*x1^45*x2^31*x3^22*x4^2*z^20 + 3*x1^44*x2^32*x3^22*x4^2*z^20 - x1^43*x2^33*x3^22*x4^2*z^20 + 5*x1^42*x2^34*x3^22*x4^2*z^20 - x1^40*x2^36*x3^22*x4^2*z^20 - x1^39*x2^37*x3^22*x4^2*z^20 - 4*x1^44*x2^31*x3^23*x4^2*z^20 - 2*x1^42*x2^33*x3^23*x4^2*z^20 - 2*x1^41*x2^34*x3^23*x4^2*z^20 + x1^44*x2^30*x3^24*x4^2*z^20 + x1^43*x2^31*x3^24*x4^2*z^20 - x1^42*x2^32*x3^24*x4^2*z^20 + 3*x1^41*x2^33*x3^24*x4^2*z^20 - x1^43*x2^30*x3^25*x4^2*z^20 - x1^42*x2^31*x3^25*x4^2*z^20 - x1^41*x2^32*x3^25*x4^2*z^20 + x1^49*x2^33*x3^15*x4^3*z^20 - x1^48*x2^34*x3^15*x4^3*z^20 + x1^46*x2^36*x3^15*x4^3*z^20 - x1^49*x2^32*x3^16*x4^3*z^20 - x1^48*x2^33*x3^16*x4^3*z^20 + 3*x1^47*x2^34*x3^16*x4^3*z^20 - x1^46*x2^35*x3^16*x4^3*z^20 - x1^45*x2^36*x3^16*x4^3*z^20 + 2*x1^48*x2^32*x3^17*x4^3*z^20 - 2*x1^46*x2^34*x3^17*x4^3*z^20 + x1^45*x2^35*x3^17*x4^3*z^20 - x1^44*x2^36*x3^17*x4^3*z^20 - x1^48*x2^31*x3^18*x4^3*z^20 - 2*x1^47*x2^32*x3^18*x4^3*z^20 + 2*x1^46*x2^33*x3^18*x4^3*z^20 + x1^45*x2^34*x3^18*x4^3*z^20 + x1^44*x2^35*x3^18*x4^3*z^20 - x1^43*x2^36*x3^18*x4^3*z^20 - x1^42*x2^37*x3^18*x4^3*z^20 + 2*x1^47*x2^31*x3^19*x4^3*z^20 - x1^46*x2^32*x3^19*x4^3*z^20 - 2*x1^45*x2^33*x3^19*x4^3*z^20 + x1^44*x2^34*x3^19*x4^3*z^20 - x1^43*x2^35*x3^19*x4^3*z^20 + 2*x1^42*x2^36*x3^19*x4^3*z^20 + x1^41*x2^37*x3^19*x4^3*z^20 + x1^40*x2^38*x3^19*x4^3*z^20 - 2*x1^46*x2^31*x3^20*x4^3*z^20 + 2*x1^45*x2^32*x3^20*x4^3*z^20 - 2*x1^44*x2^33*x3^20*x4^3*z^20 + 2*x1^42*x2^35*x3^20*x4^3*z^20 - 2*x1^40*x2^37*x3^20*x4^3*z^20 - x1^39*x2^38*x3^20*x4^3*z^20 + 2*x1^46*x2^30*x3^21*x4^3*z^20 - 3*x1^44*x2^32*x3^21*x4^3*z^20 - 2*x1^42*x2^34*x3^21*x4^3*z^20 + x1^41*x2^35*x3^21*x4^3*z^20 + 2*x1^39*x2^37*x3^21*x4^3*z^20 - x1^46*x2^29*x3^22*x4^3*z^20 + 3*x1^44*x2^31*x3^22*x4^3*z^20 - x1^43*x2^32*x3^22*x4^3*z^20 - x1^42*x2^33*x3^22*x4^3*z^20 + x1^41*x2^34*x3^22*x4^3*z^20 + x1^40*x2^35*x3^22*x4^3*z^20 - x1^39*x2^36*x3^22*x4^3*z^20 - x1^38*x2^37*x3^22*x4^3*z^20 + x1^45*x2^29*x3^23*x4^3*z^20 + x1^42*x2^32*x3^23*x4^3*z^20 - 3*x1^41*x2^33*x3^23*x4^3*z^20 + x1^39*x2^35*x3^23*x4^3*z^20 + x1^38*x2^36*x3^23*x4^3*z^20 + 3*x1^43*x2^30*x3^24*x4^3*z^20 + x1^42*x2^31*x3^24*x4^3*z^20 + x1^41*x2^32*x3^24*x4^3*z^20 + x1^40*x2^33*x3^24*x4^3*z^20 - 2*x1^40*x2^32*x3^25*x4^3*z^20 + x1^41*x2^30*x3^26*x4^3*z^20 + x1^40*x2^31*x3^26*x4^3*z^20 + x1^50*x2^33*x3^13*x4^4*z^20 - 2*x1^49*x2^33*x3^14*x4^4*z^20 - x1^47*x2^35*x3^14*x4^4*z^20 + 2*x1^49*x2^32*x3^15*x4^4*z^20 + 2*x1^48*x2^33*x3^15*x4^4*z^20 - x1^45*x2^36*x3^15*x4^4*z^20 - 6*x1^48*x2^32*x3^16*x4^4*z^20 - x1^47*x2^33*x3^16*x4^4*z^20 - 2*x1^46*x2^34*x3^16*x4^4*z^20 - x1^43*x2^37*x3^16*x4^4*z^20 + 2*x1^48*x2^31*x3^17*x4^4*z^20 + 6*x1^47*x2^32*x3^17*x4^4*z^20 - x1^46*x2^33*x3^17*x4^4*z^20 + 3*x1^45*x2^34*x3^17*x4^4*z^20 - 2*x1^44*x2^35*x3^17*x4^4*z^20 - x1^43*x2^36*x3^17*x4^4*z^20 - x1^42*x2^37*x3^17*x4^4*z^20 - 6*x1^47*x2^31*x3^18*x4^4*z^20 - 2*x1^46*x2^32*x3^18*x4^4*z^20 - x1^45*x2^33*x3^18*x4^4*z^20 + 3*x1^43*x2^35*x3^18*x4^4*z^20 + 2*x1^42*x2^36*x3^18*x4^4*z^20 + x1^41*x2^37*x3^18*x4^4*z^20 - x1^40*x2^38*x3^18*x4^4*z^20 + 2*x1^47*x2^30*x3^19*x4^4*z^20 + 6*x1^46*x2^31*x3^19*x4^4*z^20 + 4*x1^44*x2^33*x3^19*x4^4*z^20 - 2*x1^43*x2^34*x3^19*x4^4*z^20 - 2*x1^41*x2^36*x3^19*x4^4*z^20 + x1^39*x2^38*x3^19*x4^4*z^20 - 6*x1^46*x2^30*x3^20*x4^4*z^20 - 2*x1^45*x2^31*x3^20*x4^4*z^20 - 2*x1^44*x2^32*x3^20*x4^4*z^20 - 2*x1^43*x2^33*x3^20*x4^4*z^20 + x1^42*x2^34*x3^20*x4^4*z^20 + 3*x1^41*x2^35*x3^20*x4^4*z^20 + 3*x1^40*x2^36*x3^20*x4^4*z^20 - x1^39*x2^37*x3^20*x4^4*z^20 + 2*x1^46*x2^29*x3^21*x4^4*z^20 + 6*x1^45*x2^30*x3^21*x4^4*z^20 + 4*x1^43*x2^32*x3^21*x4^4*z^20 - 3*x1^40*x2^35*x3^21*x4^4*z^20 + 2*x1^39*x2^36*x3^21*x4^4*z^20 - 6*x1^45*x2^29*x3^22*x4^4*z^20 - x1^44*x2^30*x3^22*x4^4*z^20 - 2*x1^42*x2^32*x3^22*x4^4*z^20 + x1^40*x2^34*x3^22*x4^4*z^20 + 2*x1^39*x2^35*x3^22*x4^4*z^20 - x1^38*x2^36*x3^22*x4^4*z^20 + 2*x1^45*x2^28*x3^23*x4^4*z^20 + 3*x1^44*x2^29*x3^23*x4^4*z^20 - x1^43*x2^30*x3^23*x4^4*z^20 + 4*x1^42*x2^31*x3^23*x4^4*z^20 + x1^41*x2^32*x3^23*x4^4*z^20 - x1^40*x2^33*x3^23*x4^4*z^20 - x1^39*x2^34*x3^23*x4^4*z^20 + 2*x1^38*x2^35*x3^23*x4^4*z^20 - 2*x1^44*x2^28*x3^24*x4^4*z^20 + x1^43*x2^29*x3^24*x4^4*z^20 - x1^41*x2^31*x3^24*x4^4*z^20 - x1^39*x2^33*x3^24*x4^4*z^20 + x1^38*x2^34*x3^24*x4^4*z^20 - x1^37*x2^35*x3^24*x4^4*z^20 + x1^43*x2^28*x3^25*x4^4*z^20 + 2*x1^41*x2^30*x3^25*x4^4*z^20 - x1^39*x2^32*x3^25*x4^4*z^20 + x1^41*x2^29*x3^26*x4^4*z^20 + x1^40*x2^30*x3^26*x4^4*z^20 + 2*x1^39*x2^31*x3^26*x4^4*z^20 - x1^39*x2^30*x3^27*x4^4*z^20 - x1^38*x2^31*x3^27*x4^4*z^20 + 2*x1^49*x2^33*x3^13*x4^5*z^20 - x1^48*x2^34*x3^13*x4^5*z^20 - 2*x1^49*x2^32*x3^14*x4^5*z^20 - 2*x1^48*x2^33*x3^14*x4^5*z^20 + x1^47*x2^34*x3^14*x4^5*z^20 + 5*x1^48*x2^32*x3^15*x4^5*z^20 - x1^47*x2^33*x3^15*x4^5*z^20 - x1^44*x2^36*x3^15*x4^5*z^20 - 2*x1^48*x2^31*x3^16*x4^5*z^20 - 5*x1^47*x2^32*x3^16*x4^5*z^20 + 3*x1^46*x2^33*x3^16*x4^5*z^20 - 3*x1^45*x2^34*x3^16*x4^5*z^20 + x1^44*x2^35*x3^16*x4^5*z^20 + 2*x1^43*x2^36*x3^16*x4^5*z^20 + x1^42*x2^37*x3^16*x4^5*z^20 + 6*x1^47*x2^31*x3^17*x4^5*z^20 + 2*x1^46*x2^32*x3^17*x4^5*z^20 - x1^45*x2^33*x3^17*x4^5*z^20 + x1^44*x2^34*x3^17*x4^5*z^20 - x1^43*x2^35*x3^17*x4^5*z^20 - x1^42*x2^36*x3^17*x4^5*z^20 - x1^41*x2^37*x3^17*x4^5*z^20 - 2*x1^47*x2^30*x3^18*x4^5*z^20 - 6*x1^46*x2^31*x3^18*x4^5*z^20 + 2*x1^45*x2^32*x3^18*x4^5*z^20 - 2*x1^44*x2^33*x3^18*x4^5*z^20 + 3*x1^43*x2^34*x3^18*x4^5*z^20 + 3*x1^41*x2^36*x3^18*x4^5*z^20 + x1^40*x2^37*x3^18*x4^5*z^20 + 6*x1^46*x2^30*x3^19*x4^5*z^20 + x1^45*x2^31*x3^19*x4^5*z^20 - 2*x1^42*x2^34*x3^19*x4^5*z^20 - 2*x1^41*x2^35*x3^19*x4^5*z^20 - 3*x1^40*x2^36*x3^19*x4^5*z^20 - 2*x1^46*x2^29*x3^20*x4^5*z^20 - 6*x1^45*x2^30*x3^20*x4^5*z^20 + 2*x1^44*x2^31*x3^20*x4^5*z^20 - 3*x1^43*x2^32*x3^20*x4^5*z^20 + 5*x1^42*x2^33*x3^20*x4^5*z^20 + x1^41*x2^34*x3^20*x4^5*z^20 + 5*x1^40*x2^35*x3^20*x4^5*z^20 + x1^38*x2^37*x3^20*x4^5*z^20 + 6*x1^45*x2^29*x3^21*x4^5*z^20 + x1^44*x2^30*x3^21*x4^5*z^20 + x1^42*x2^32*x3^21*x4^5*z^20 - 3*x1^41*x2^33*x3^21*x4^5*z^20 - 6*x1^39*x2^35*x3^21*x4^5*z^20 + 3*x1^38*x2^36*x3^21*x4^5*z^20 - x1^45*x2^28*x3^22*x4^5*z^20 - 6*x1^44*x2^29*x3^22*x4^5*z^20 + 2*x1^43*x2^30*x3^22*x4^5*z^20 - 4*x1^42*x2^31*x3^22*x4^5*z^20 + x1^41*x2^32*x3^22*x4^5*z^20 + 4*x1^39*x2^34*x3^22*x4^5*z^20 - x1^37*x2^36*x3^22*x4^5*z^20 + 3*x1^44*x2^28*x3^23*x4^5*z^20 + 3*x1^43*x2^29*x3^23*x4^5*z^20 + x1^42*x2^30*x3^23*x4^5*z^20 + x1^41*x2^31*x3^23*x4^5*z^20 - x1^40*x2^32*x3^23*x4^5*z^20 - x1^39*x2^33*x3^23*x4^5*z^20 - 3*x1^38*x2^34*x3^23*x4^5*z^20 + x1^37*x2^35*x3^23*x4^5*z^20 + x1^36*x2^36*x3^23*x4^5*z^20 - 3*x1^43*x2^28*x3^24*x4^5*z^20 - x1^41*x2^30*x3^24*x4^5*z^20 + x1^39*x2^32*x3^24*x4^5*z^20 + 2*x1^38*x2^33*x3^24*x4^5*z^20 - x1^37*x2^34*x3^24*x4^5*z^20 + x1^42*x2^28*x3^25*x4^5*z^20 - x1^38*x2^32*x3^25*x4^5*z^20 - 2*x1^37*x2^33*x3^25*x4^5*z^20 + 2*x1^36*x2^34*x3^25*x4^5*z^20 - x1^41*x2^28*x3^26*x4^5*z^20 + x1^39*x2^30*x3^26*x4^5*z^20 + 2*x1^38*x2^31*x3^26*x4^5*z^20 - x1^37*x2^32*x3^26*x4^5*z^20 - x1^36*x2^33*x3^26*x4^5*z^20 - x1^38*x2^30*x3^27*x4^5*z^20 + x1^37*x2^30*x3^28*x4^5*z^20 + x1^48*x2^33*x3^13*x4^6*z^20 - x1^47*x2^34*x3^13*x4^6*z^20 - 2*x1^48*x2^32*x3^14*x4^6*z^20 + x1^47*x2^33*x3^14*x4^6*z^20 + 2*x1^46*x2^34*x3^14*x4^6*z^20 - x1^45*x2^35*x3^14*x4^6*z^20 + 3*x1^47*x2^32*x3^15*x4^6*z^20 - 2*x1^47*x2^31*x3^16*x4^6*z^20 - x1^44*x2^34*x3^16*x4^6*z^20 - x1^42*x2^36*x3^16*x4^6*z^20 + x1^47*x2^30*x3^17*x4^6*z^20 + 2*x1^46*x2^31*x3^17*x4^6*z^20 - 4*x1^45*x2^32*x3^17*x4^6*z^20 + 3*x1^44*x2^33*x3^17*x4^6*z^20 - x1^42*x2^35*x3^17*x4^6*z^20 - x1^41*x2^36*x3^17*x4^6*z^20 - 2*x1^46*x2^30*x3^18*x4^6*z^20 + 3*x1^44*x2^32*x3^18*x4^6*z^20 + 4*x1^42*x2^34*x3^18*x4^6*z^20 + 3*x1^41*x2^35*x3^18*x4^6*z^20 + x1^40*x2^36*x3^18*x4^6*z^20 - x1^39*x2^37*x3^18*x4^6*z^20 + x1^46*x2^29*x3^19*x4^6*z^20 + 2*x1^45*x2^30*x3^19*x4^6*z^20 - 4*x1^44*x2^31*x3^19*x4^6*z^20 - x1^43*x2^32*x3^19*x4^6*z^20 - 4*x1^42*x2^33*x3^19*x4^6*z^20 + x1^41*x2^34*x3^19*x4^6*z^20 + 2*x1^39*x2^36*x3^19*x4^6*z^20 + x1^38*x2^37*x3^19*x4^6*z^20 - 2*x1^45*x2^29*x3^20*x4^6*z^20 + 2*x1^44*x2^30*x3^20*x4^6*z^20 + 4*x1^43*x2^31*x3^20*x4^6*z^20 + 2*x1^41*x2^33*x3^20*x4^6*z^20 - 3*x1^40*x2^34*x3^20*x4^6*z^20 + x1^39*x2^35*x3^20*x4^6*z^20 - 3*x1^38*x2^36*x3^20*x4^6*z^20 + 2*x1^44*x2^29*x3^21*x4^6*z^20 - 4*x1^43*x2^30*x3^21*x4^6*z^20 + x1^42*x2^31*x3^21*x4^6*z^20 - 3*x1^41*x2^32*x3^21*x4^6*z^20 - 2*x1^40*x2^33*x3^21*x4^6*z^20 - 2*x1^39*x2^34*x3^21*x4^6*z^20 + x1^37*x2^36*x3^21*x4^6*z^20 - x1^44*x2^28*x3^22*x4^6*z^20 + 3*x1^42*x2^30*x3^22*x4^6*z^20 + x1^41*x2^31*x3^22*x4^6*z^20 + 4*x1^40*x2^32*x3^22*x4^6*z^20 - 2*x1^39*x2^33*x3^22*x4^6*z^20 + 2*x1^38*x2^34*x3^22*x4^6*z^20 - 4*x1^37*x2^35*x3^22*x4^6*z^20 + x1^44*x2^27*x3^23*x4^6*z^20 + x1^43*x2^28*x3^23*x4^6*z^20 - 3*x1^42*x2^29*x3^23*x4^6*z^20 - x1^40*x2^31*x3^23*x4^6*z^20 - 3*x1^38*x2^33*x3^23*x4^6*z^20 - x1^37*x2^34*x3^23*x4^6*z^20 + x1^36*x2^35*x3^23*x4^6*z^20 - x1^43*x2^27*x3^24*x4^6*z^20 + x1^41*x2^29*x3^24*x4^6*z^20 + 3*x1^39*x2^31*x3^24*x4^6*z^20 + x1^38*x2^32*x3^24*x4^6*z^20 + 2*x1^37*x2^33*x3^24*x4^6*z^20 - 3*x1^36*x2^34*x3^24*x4^6*z^20 - x1^35*x2^35*x3^24*x4^6*z^20 + x1^42*x2^27*x3^25*x4^6*z^20 - 2*x1^41*x2^28*x3^25*x4^6*z^20 - 2*x1^40*x2^29*x3^25*x4^6*z^20 - 2*x1^39*x2^30*x3^25*x4^6*z^20 - x1^41*x2^27*x3^26*x4^6*z^20 + x1^38*x2^30*x3^26*x4^6*z^20 - x1^37*x2^31*x3^26*x4^6*z^20 + x1^36*x2^32*x3^26*x4^6*z^20 - x1^35*x2^33*x3^26*x4^6*z^20 + x1^39*x2^28*x3^27*x4^6*z^20 - 2*x1^37*x2^30*x3^27*x4^6*z^20 + x1^36*x2^31*x3^27*x4^6*z^20 + x1^35*x2^32*x3^27*x4^6*z^20 - x1^48*x2^33*x3^12*x4^7*z^20 - x1^45*x2^36*x3^12*x4^7*z^20 - 2*x1^48*x2^32*x3^13*x4^7*z^20 + x1^47*x2^33*x3^13*x4^7*z^20 + 3*x1^46*x2^34*x3^13*x4^7*z^20 - x1^45*x2^35*x3^13*x4^7*z^20 + x1^44*x2^36*x3^13*x4^7*z^20 - x1^46*x2^33*x3^14*x4^7*z^20 + x1^45*x2^34*x3^14*x4^7*z^20 - 2*x1^47*x2^31*x3^15*x4^7*z^20 - x1^46*x2^32*x3^15*x4^7*z^20 + x1^45*x2^33*x3^15*x4^7*z^20 + x1^44*x2^34*x3^15*x4^7*z^20 + 3*x1^43*x2^35*x3^15*x4^7*z^20 - x1^42*x2^36*x3^15*x4^7*z^20 + x1^45*x2^32*x3^16*x4^7*z^20 - x1^44*x2^33*x3^16*x4^7*z^20 + x1^43*x2^34*x3^16*x4^7*z^20 + x1^41*x2^36*x3^16*x4^7*z^20 + x1^40*x2^37*x3^16*x4^7*z^20 - x1^44*x2^32*x3^17*x4^7*z^20 + x1^42*x2^34*x3^17*x4^7*z^20 - x1^41*x2^35*x3^17*x4^7*z^20 - x1^40*x2^36*x3^17*x4^7*z^20 - 2*x1^39*x2^37*x3^17*x4^7*z^20 - x1^42*x2^33*x3^18*x4^7*z^20 + 2*x1^39*x2^36*x3^18*x4^7*z^20 + 2*x1^38*x2^37*x3^18*x4^7*z^20 - 2*x1^40*x2^34*x3^19*x4^7*z^20 + x1^39*x2^35*x3^19*x4^7*z^20 - 2*x1^38*x2^36*x3^19*x4^7*z^20 - x1^37*x2^37*x3^19*x4^7*z^20 - x1^38*x2^35*x3^20*x4^7*z^20 + 2*x1^37*x2^36*x3^20*x4^7*z^20 - x1^43*x2^29*x3^21*x4^7*z^20 - x1^36*x2^36*x3^21*x4^7*z^20 + 2*x1^42*x2^29*x3^22*x4^7*z^20 - x1^41*x2^30*x3^22*x4^7*z^20 + x1^40*x2^31*x3^22*x4^7*z^20 + x1^39*x2^32*x3^22*x4^7*z^20 + x1^38*x2^33*x3^22*x4^7*z^20 + x1^42*x2^28*x3^23*x4^7*z^20 - x1^41*x2^29*x3^23*x4^7*z^20 - x1^40*x2^30*x3^23*x4^7*z^20 + 2*x1^38*x2^32*x3^23*x4^7*z^20 - x1^37*x2^33*x3^23*x4^7*z^20 + x1^36*x2^34*x3^23*x4^7*z^20 - x1^35*x2^35*x3^23*x4^7*z^20 + x1^41*x2^28*x3^24*x4^7*z^20 + x1^40*x2^29*x3^24*x4^7*z^20 + 2*x1^39*x2^30*x3^24*x4^7*z^20 + 2*x1^36*x2^33*x3^24*x4^7*z^20 + x1^39*x2^29*x3^25*x4^7*z^20 - x1^36*x2^32*x3^25*x4^7*z^20 - x1^40*x2^27*x3^26*x4^7*z^20 - x1^39*x2^28*x3^26*x4^7*z^20 + x1^35*x2^32*x3^26*x4^7*z^20 - x1^38*x2^28*x3^27*x4^7*z^20 - x1^35*x2^31*x3^27*x4^7*z^20 - x1^37*x2^28*x3^28*x4^7*z^20 + x1^36*x2^29*x3^28*x4^7*z^20 - x1^34*x2^31*x3^28*x4^7*z^20 + x1^48*x2^32*x3^12*x4^8*z^20 - x1^46*x2^34*x3^12*x4^8*z^20 + x1^45*x2^35*x3^12*x4^8*z^20 - x1^44*x2^36*x3^12*x4^8*z^20 + x1^45*x2^34*x3^13*x4^8*z^20 - 2*x1^42*x2^37*x3^13*x4^8*z^20 - x1^47*x2^31*x3^14*x4^8*z^20 - x1^46*x2^32*x3^14*x4^8*z^20 - 3*x1^45*x2^33*x3^14*x4^8*z^20 - x1^44*x2^34*x3^14*x4^8*z^20 + 2*x1^42*x2^36*x3^14*x4^8*z^20 + 2*x1^41*x2^37*x3^14*x4^8*z^20 + x1^47*x2^30*x3^15*x4^8*z^20 + x1^46*x2^31*x3^15*x4^8*z^20 + 2*x1^45*x2^32*x3^15*x4^8*z^20 - x1^42*x2^35*x3^15*x4^8*z^20 - 2*x1^40*x2^37*x3^15*x4^8*z^20 - 2*x1^45*x2^31*x3^16*x4^8*z^20 - 3*x1^44*x2^32*x3^16*x4^8*z^20 + x1^43*x2^33*x3^16*x4^8*z^20 - 3*x1^42*x2^34*x3^16*x4^8*z^20 + 2*x1^41*x2^35*x3^16*x4^8*z^20 + 4*x1^39*x2^37*x3^16*x4^8*z^20 + x1^45*x2^30*x3^17*x4^8*z^20 + 2*x1^44*x2^31*x3^17*x4^8*z^20 + 2*x1^43*x2^32*x3^17*x4^8*z^20 + x1^42*x2^33*x3^17*x4^8*z^20 - x1^41*x2^34*x3^17*x4^8*z^20 - x1^40*x2^35*x3^17*x4^8*z^20 - x1^39*x2^36*x3^17*x4^8*z^20 - 4*x1^38*x2^37*x3^17*x4^8*z^20 - 2*x1^45*x2^29*x3^18*x4^8*z^20 - 3*x1^44*x2^30*x3^18*x4^8*z^20 - 5*x1^43*x2^31*x3^18*x4^8*z^20 - 2*x1^41*x2^33*x3^18*x4^8*z^20 + 4*x1^40*x2^34*x3^18*x4^8*z^20 + 4*x1^38*x2^36*x3^18*x4^8*z^20 + 2*x1^37*x2^37*x3^18*x4^8*z^20 + x1^45*x2^28*x3^19*x4^8*z^20 + 2*x1^44*x2^29*x3^19*x4^8*z^20 + 4*x1^43*x2^30*x3^19*x4^8*z^20 + 2*x1^42*x2^31*x3^19*x4^8*z^20 - 3*x1^39*x2^34*x3^19*x4^8*z^20 - 2*x1^38*x2^35*x3^19*x4^8*z^20 - 4*x1^37*x2^36*x3^19*x4^8*z^20 - 2*x1^44*x2^28*x3^20*x4^8*z^20 - 5*x1^42*x2^30*x3^20*x4^8*z^20 - 2*x1^40*x2^32*x3^20*x4^8*z^20 + 4*x1^39*x2^33*x3^20*x4^8*z^20 + 2*x1^38*x2^34*x3^20*x4^8*z^20 + 4*x1^37*x2^35*x3^20*x4^8*z^20 + x1^36*x2^36*x3^20*x4^8*z^20 + 2*x1^43*x2^28*x3^21*x4^8*z^20 + x1^42*x2^29*x3^21*x4^8*z^20 + 3*x1^41*x2^30*x3^21*x4^8*z^20 - x1^40*x2^31*x3^21*x4^8*z^20 - 3*x1^38*x2^33*x3^21*x4^8*z^20 - x1^37*x2^34*x3^21*x4^8*z^20 - 4*x1^36*x2^35*x3^21*x4^8*z^20 - x1^43*x2^27*x3^22*x4^8*z^20 - x1^41*x2^29*x3^22*x4^8*z^20 - x1^39*x2^31*x3^22*x4^8*z^20 + 2*x1^38*x2^32*x3^22*x4^8*z^20 + 2*x1^37*x2^33*x3^22*x4^8*z^20 + 4*x1^36*x2^34*x3^22*x4^8*z^20 + 2*x1^35*x2^35*x3^22*x4^8*z^20 + x1^42*x2^27*x3^23*x4^8*z^20 - x1^41*x2^28*x3^23*x4^8*z^20 + x1^40*x2^29*x3^23*x4^8*z^20 + x1^39*x2^30*x3^23*x4^8*z^20 - x1^38*x2^31*x3^23*x4^8*z^20 - 2*x1^37*x2^32*x3^23*x4^8*z^20 - 2*x1^36*x2^33*x3^23*x4^8*z^20 - 2*x1^35*x2^34*x3^23*x4^8*z^20 - x1^42*x2^26*x3^24*x4^8*z^20 - x1^41*x2^27*x3^24*x4^8*z^20 + x1^39*x2^29*x3^24*x4^8*z^20 + 3*x1^37*x2^31*x3^24*x4^8*z^20 + 4*x1^35*x2^33*x3^24*x4^8*z^20 + 2*x1^34*x2^34*x3^24*x4^8*z^20 + x1^41*x2^26*x3^25*x4^8*z^20 - x1^39*x2^28*x3^25*x4^8*z^20 - 3*x1^38*x2^29*x3^25*x4^8*z^20 - x1^37*x2^30*x3^25*x4^8*z^20 - 3*x1^36*x2^31*x3^25*x4^8*z^20 - 2*x1^35*x2^32*x3^25*x4^8*z^20 - 2*x1^34*x2^33*x3^25*x4^8*z^20 + x1^38*x2^28*x3^26*x4^8*z^20 + x1^37*x2^29*x3^26*x4^8*z^20 + 2*x1^36*x2^30*x3^26*x4^8*z^20 + x1^35*x2^31*x3^26*x4^8*z^20 + x1^33*x2^33*x3^26*x4^8*z^20 + x1^38*x2^27*x3^27*x4^8*z^20 - x1^37*x2^28*x3^27*x4^8*z^20 - x1^36*x2^29*x3^27*x4^8*z^20 - x1^34*x2^31*x3^27*x4^8*z^20 + x1^33*x2^31*x3^28*x4^8*z^20 - x1^47*x2^32*x3^12*x4^9*z^20 + x1^46*x2^33*x3^12*x4^9*z^20 + x1^43*x2^36*x3^12*x4^9*z^20 + x1^47*x2^31*x3^13*x4^9*z^20 + x1^46*x2^32*x3^13*x4^9*z^20 + x1^45*x2^33*x3^13*x4^9*z^20 + x1^44*x2^34*x3^13*x4^9*z^20 + x1^43*x2^35*x3^13*x4^9*z^20 - x1^46*x2^31*x3^14*x4^9*z^20 + x1^43*x2^34*x3^14*x4^9*z^20 + x1^41*x2^36*x3^14*x4^9*z^20 + x1^40*x2^37*x3^14*x4^9*z^20 + 2*x1^46*x2^30*x3^15*x4^9*z^20 + x1^45*x2^31*x3^15*x4^9*z^20 + x1^44*x2^32*x3^15*x4^9*z^20 + 2*x1^42*x2^34*x3^15*x4^9*z^20 - x1^39*x2^37*x3^15*x4^9*z^20 - x1^46*x2^29*x3^16*x4^9*z^20 - 2*x1^45*x2^30*x3^16*x4^9*z^20 - 2*x1^44*x2^31*x3^16*x4^9*z^20 - 2*x1^43*x2^32*x3^16*x4^9*z^20 - x1^41*x2^34*x3^16*x4^9*z^20 - x1^39*x2^36*x3^16*x4^9*z^20 + x1^38*x2^37*x3^16*x4^9*z^20 + 4*x1^45*x2^29*x3^17*x4^9*z^20 + 3*x1^44*x2^30*x3^17*x4^9*z^20 + 4*x1^43*x2^31*x3^17*x4^9*z^20 - x1^40*x2^34*x3^17*x4^9*z^20 - x1^38*x2^36*x3^17*x4^9*z^20 - x1^37*x2^37*x3^17*x4^9*z^20 - x1^45*x2^28*x3^18*x4^9*z^20 - 3*x1^44*x2^29*x3^18*x4^9*z^20 - 2*x1^42*x2^31*x3^18*x4^9*z^20 + 2*x1^41*x2^32*x3^18*x4^9*z^20 + x1^40*x2^33*x3^18*x4^9*z^20 + 3*x1^39*x2^34*x3^18*x4^9*z^20 + 2*x1^38*x2^35*x3^18*x4^9*z^20 + x1^37*x2^36*x3^18*x4^9*z^20 + 4*x1^44*x2^28*x3^19*x4^9*z^20 + 3*x1^43*x2^29*x3^19*x4^9*z^20 + 4*x1^42*x2^30*x3^19*x4^9*z^20 - x1^40*x2^32*x3^19*x4^9*z^20 - 4*x1^39*x2^33*x3^19*x4^9*z^20 - 2*x1^38*x2^34*x3^19*x4^9*z^20 - 2*x1^37*x2^35*x3^19*x4^9*z^20 - 2*x1^44*x2^27*x3^20*x4^9*z^20 - 4*x1^43*x2^28*x3^20*x4^9*z^20 - 2*x1^42*x2^29*x3^20*x4^9*z^20 - 2*x1^41*x2^30*x3^20*x4^9*z^20 + 3*x1^40*x2^31*x3^20*x4^9*z^20 + 4*x1^38*x2^33*x3^20*x4^9*z^20 + x1^37*x2^34*x3^20*x4^9*z^20 + 2*x1^36*x2^35*x3^20*x4^9*z^20 + 5*x1^43*x2^27*x3^21*x4^9*z^20 + 3*x1^41*x2^29*x3^21*x4^9*z^20 + x1^40*x2^30*x3^21*x4^9*z^20 + x1^39*x2^31*x3^21*x4^9*z^20 - 2*x1^38*x2^32*x3^21*x4^9*z^20 - 4*x1^37*x2^33*x3^21*x4^9*z^20 - 2*x1^36*x2^34*x3^21*x4^9*z^20 - x1^35*x2^35*x3^21*x4^9*z^20 - x1^43*x2^26*x3^22*x4^9*z^20 - 5*x1^42*x2^27*x3^22*x4^9*z^20 - 5*x1^40*x2^29*x3^22*x4^9*z^20 + x1^39*x2^30*x3^22*x4^9*z^20 + 5*x1^37*x2^32*x3^22*x4^9*z^20 + 2*x1^36*x2^33*x3^22*x4^9*z^20 + 2*x1^35*x2^34*x3^22*x4^9*z^20 + 3*x1^42*x2^26*x3^23*x4^9*z^20 + 3*x1^41*x2^27*x3^23*x4^9*z^20 + 2*x1^40*x2^28*x3^23*x4^9*z^20 + x1^39*x2^29*x3^23*x4^9*z^20 - x1^38*x2^30*x3^23*x4^9*z^20 - 2*x1^37*x2^31*x3^23*x4^9*z^20 - 4*x1^36*x2^32*x3^23*x4^9*z^20 - 2*x1^35*x2^33*x3^23*x4^9*z^20 - x1^34*x2^34*x3^23*x4^9*z^20 - 3*x1^41*x2^26*x3^24*x4^9*z^20 - 2*x1^40*x2^27*x3^24*x4^9*z^20 - 2*x1^39*x2^28*x3^24*x4^9*z^20 + 2*x1^38*x2^29*x3^24*x4^9*z^20 - x1^37*x2^30*x3^24*x4^9*z^20 + 3*x1^36*x2^31*x3^24*x4^9*z^20 + 2*x1^35*x2^32*x3^24*x4^9*z^20 + x1^40*x2^26*x3^25*x4^9*z^20 + 2*x1^39*x2^27*x3^25*x4^9*z^20 - x1^37*x2^29*x3^25*x4^9*z^20 - 2*x1^36*x2^30*x3^25*x4^9*z^20 - x1^35*x2^31*x3^25*x4^9*z^20 - x1^34*x2^32*x3^25*x4^9*z^20 - x1^38*x2^27*x3^26*x4^9*z^20 + x1^37*x2^28*x3^26*x4^9*z^20 - x1^36*x2^29*x3^26*x4^9*z^20 + x1^35*x2^30*x3^26*x4^9*z^20 - x1^34*x2^31*x3^26*x4^9*z^20 + x1^36*x2^28*x3^27*x4^9*z^20 - x1^35*x2^29*x3^27*x4^9*z^20 - 2*x1^34*x2^30*x3^27*x4^9*z^20 + x1^33*x2^31*x3^27*x4^9*z^20 - x1^32*x2^32*x3^27*x4^9*z^20 - x1^35*x2^28*x3^28*x4^9*z^20 + x1^34*x2^29*x3^28*x4^9*z^20 - x1^32*x2^31*x3^28*x4^9*z^20 - x1^44*x2^34*x3^12*x4^10*z^20 - x1^45*x2^32*x3^13*x4^10*z^20 - x1^44*x2^33*x3^13*x4^10*z^20 + x1^43*x2^34*x3^13*x4^10*z^20 - x1^42*x2^35*x3^13*x4^10*z^20 - x1^41*x2^36*x3^13*x4^10*z^20 - x1^46*x2^30*x3^14*x4^10*z^20 - x1^44*x2^32*x3^14*x4^10*z^20 - x1^43*x2^33*x3^14*x4^10*z^20 + x1^42*x2^34*x3^14*x4^10*z^20 + x1^41*x2^35*x3^14*x4^10*z^20 + x1^40*x2^36*x3^14*x4^10*z^20 - x1^45*x2^30*x3^15*x4^10*z^20 + x1^44*x2^31*x3^15*x4^10*z^20 + x1^43*x2^32*x3^15*x4^10*z^20 - x1^39*x2^36*x3^15*x4^10*z^20 - 2*x1^43*x2^31*x3^16*x4^10*z^20 + x1^42*x2^32*x3^16*x4^10*z^20 + x1^41*x2^33*x3^16*x4^10*z^20 + 2*x1^38*x2^36*x3^16*x4^10*z^20 - x1^44*x2^29*x3^17*x4^10*z^20 + 2*x1^43*x2^30*x3^17*x4^10*z^20 + x1^41*x2^32*x3^17*x4^10*z^20 - x1^40*x2^33*x3^17*x4^10*z^20 + x1^39*x2^34*x3^17*x4^10*z^20 + x1^38*x2^35*x3^17*x4^10*z^20 - x1^37*x2^36*x3^17*x4^10*z^20 - x1^44*x2^28*x3^18*x4^10*z^20 - x1^43*x2^29*x3^18*x4^10*z^20 - x1^42*x2^30*x3^18*x4^10*z^20 + x1^39*x2^33*x3^18*x4^10*z^20 - x1^38*x2^34*x3^18*x4^10*z^20 + x1^36*x2^36*x3^18*x4^10*z^20 + x1^44*x2^27*x3^19*x4^10*z^20 + x1^42*x2^29*x3^19*x4^10*z^20 - x1^41*x2^30*x3^19*x4^10*z^20 + x1^40*x2^31*x3^19*x4^10*z^20 + x1^39*x2^32*x3^19*x4^10*z^20 - 2*x1^43*x2^27*x3^20*x4^10*z^20 - x1^41*x2^29*x3^20*x4^10*z^20 + 2*x1^42*x2^27*x3^21*x4^10*z^20 - x1^39*x2^30*x3^21*x4^10*z^20 - 2*x1^42*x2^26*x3^22*x4^10*z^20 - x1^41*x2^27*x3^22*x4^10*z^20 - x1^40*x2^28*x3^22*x4^10*z^20 - x1^37*x2^31*x3^22*x4^10*z^20 + 2*x1^41*x2^26*x3^23*x4^10*z^20 + 2*x1^39*x2^28*x3^23*x4^10*z^20 - x1^37*x2^30*x3^23*x4^10*z^20 - x1^36*x2^31*x3^23*x4^10*z^20 + x1^37*x2^29*x3^24*x4^10*z^20 + 2*x1^36*x2^30*x3^24*x4^10*z^20 + x1^35*x2^31*x3^24*x4^10*z^20 - x1^34*x2^32*x3^24*x4^10*z^20 - x1^37*x2^28*x3^25*x4^10*z^20 + x1^36*x2^29*x3^25*x4^10*z^20 - x1^35*x2^30*x3^25*x4^10*z^20 + x1^33*x2^32*x3^25*x4^10*z^20 + x1^37*x2^27*x3^26*x4^10*z^20 + 2*x1^35*x2^29*x3^26*x4^10*z^20 + 3*x1^34*x2^30*x3^26*x4^10*z^20 - x1^32*x2^32*x3^26*x4^10*z^20 - x1^36*x2^27*x3^27*x4^10*z^20 - x1^34*x2^29*x3^27*x4^10*z^20 + x1^33*x2^29*x3^28*x4^10*z^20 + 2*x1^43*x2^33*x3^13*x4^11*z^20 + x1^42*x2^34*x3^13*x4^11*z^20 + x1^45*x2^30*x3^14*x4^11*z^20 - x1^44*x2^31*x3^14*x4^11*z^20 + x1^43*x2^32*x3^14*x4^11*z^20 - x1^42*x2^33*x3^14*x4^11*z^20 - x1^41*x2^34*x3^14*x4^11*z^20 + x1^45*x2^29*x3^15*x4^11*z^20 - x1^44*x2^30*x3^15*x4^11*z^20 + 2*x1^42*x2^32*x3^15*x4^11*z^20 - 2*x1^40*x2^34*x3^15*x4^11*z^20 + x1^39*x2^35*x3^15*x4^11*z^20 - x1^38*x2^36*x3^15*x4^11*z^20 + 2*x1^44*x2^29*x3^16*x4^11*z^20 - 2*x1^43*x2^30*x3^16*x4^11*z^20 - x1^42*x2^31*x3^16*x4^11*z^20 - 2*x1^41*x2^32*x3^16*x4^11*z^20 + x1^40*x2^33*x3^16*x4^11*z^20 - 3*x1^39*x2^34*x3^16*x4^11*z^20 - 3*x1^38*x2^35*x3^16*x4^11*z^20 - x1^44*x2^28*x3^17*x4^11*z^20 - 2*x1^43*x2^29*x3^17*x4^11*z^20 + x1^42*x2^30*x3^17*x4^11*z^20 + x1^41*x2^31*x3^17*x4^11*z^20 + x1^40*x2^32*x3^17*x4^11*z^20 + x1^39*x2^33*x3^17*x4^11*z^20 + 4*x1^38*x2^34*x3^17*x4^11*z^20 - 4*x1^37*x2^35*x3^17*x4^11*z^20 + x1^44*x2^27*x3^18*x4^11*z^20 + 3*x1^43*x2^28*x3^18*x4^11*z^20 - 2*x1^42*x2^29*x3^18*x4^11*z^20 - 3*x1^41*x2^30*x3^18*x4^11*z^20 - 5*x1^40*x2^31*x3^18*x4^11*z^20 + x1^39*x2^32*x3^18*x4^11*z^20 - 3*x1^38*x2^33*x3^18*x4^11*z^20 + 2*x1^36*x2^35*x3^18*x4^11*z^20 - x1^43*x2^27*x3^19*x4^11*z^20 - x1^42*x2^28*x3^19*x4^11*z^20 + 2*x1^41*x2^29*x3^19*x4^11*z^20 + x1^39*x2^31*x3^19*x4^11*z^20 + 4*x1^37*x2^33*x3^19*x4^11*z^20 - 2*x1^36*x2^34*x3^19*x4^11*z^20 - x1^35*x2^35*x3^19*x4^11*z^20 + x1^42*x2^27*x3^20*x4^11*z^20 - 3*x1^41*x2^28*x3^20*x4^11*z^20 + 2*x1^40*x2^29*x3^20*x4^11*z^20 - 5*x1^39*x2^30*x3^20*x4^11*z^20 - x1^38*x2^31*x3^20*x4^11*z^20 - 3*x1^37*x2^32*x3^20*x4^11*z^20 + 2*x1^35*x2^34*x3^20*x4^11*z^20 - x1^41*x2^27*x3^21*x4^11*z^20 + x1^40*x2^28*x3^21*x4^11*z^20 + x1^38*x2^30*x3^21*x4^11*z^20 - x1^37*x2^31*x3^21*x4^11*z^20 + 4*x1^36*x2^32*x3^21*x4^11*z^20 - 2*x1^35*x2^33*x3^21*x4^11*z^20 + x1^40*x2^27*x3^22*x4^11*z^20 + x1^39*x2^28*x3^22*x4^11*z^20 - x1^38*x2^29*x3^22*x4^11*z^20 + 2*x1^37*x2^30*x3^22*x4^11*z^20 - 3*x1^36*x2^31*x3^22*x4^11*z^20 - 2*x1^35*x2^32*x3^22*x4^11*z^20 + 2*x1^34*x2^33*x3^22*x4^11*z^20 - x1^39*x2^27*x3^23*x4^11*z^20 + 3*x1^37*x2^29*x3^23*x4^11*z^20 + 3*x1^35*x2^31*x3^23*x4^11*z^20 - 2*x1^34*x2^32*x3^23*x4^11*z^20 - x1^33*x2^33*x3^23*x4^11*z^20 - x1^37*x2^28*x3^24*x4^11*z^20 + x1^34*x2^31*x3^24*x4^11*z^20 + 2*x1^33*x2^32*x3^24*x4^11*z^20 - 2*x1^35*x2^29*x3^25*x4^11*z^20 + x1^34*x2^30*x3^25*x4^11*z^20 - 2*x1^33*x2^31*x3^25*x4^11*z^20 - x1^32*x2^32*x3^25*x4^11*z^20 + 2*x1^32*x2^31*x3^26*x4^11*z^20 + x1^32*x2^30*x3^27*x4^11*z^20 - x1^43*x2^33*x3^12*x4^12*z^20 + x1^42*x2^33*x3^13*x4^12*z^20 + x1^41*x2^34*x3^13*x4^12*z^20 - x1^42*x2^32*x3^14*x4^12*z^20 - x1^41*x2^33*x3^14*x4^12*z^20 - x1^44*x2^29*x3^15*x4^12*z^20 + x1^43*x2^30*x3^15*x4^12*z^20 - x1^42*x2^31*x3^15*x4^12*z^20 + x1^39*x2^34*x3^15*x4^12*z^20 - x1^37*x2^36*x3^15*x4^12*z^20 + 2*x1^43*x2^29*x3^16*x4^12*z^20 - x1^42*x2^30*x3^16*x4^12*z^20 - x1^41*x2^31*x3^16*x4^12*z^20 + 3*x1^39*x2^33*x3^16*x4^12*z^20 - x1^38*x2^34*x3^16*x4^12*z^20 + 3*x1^37*x2^35*x3^16*x4^12*z^20 - x1^43*x2^28*x3^17*x4^12*z^20 + 2*x1^42*x2^29*x3^17*x4^12*z^20 + x1^41*x2^30*x3^17*x4^12*z^20 + 2*x1^40*x2^31*x3^17*x4^12*z^20 - 2*x1^39*x2^32*x3^17*x4^12*z^20 - 5*x1^36*x2^35*x3^17*x4^12*z^20 + x1^43*x2^27*x3^18*x4^12*z^20 + x1^42*x2^28*x3^18*x4^12*z^20 - 3*x1^41*x2^29*x3^18*x4^12*z^20 - x1^40*x2^30*x3^18*x4^12*z^20 + 4*x1^38*x2^32*x3^18*x4^12*z^20 - 2*x1^37*x2^33*x3^18*x4^12*z^20 + 6*x1^36*x2^34*x3^18*x4^12*z^20 + 2*x1^35*x2^35*x3^18*x4^12*z^20 - x1^43*x2^26*x3^19*x4^12*z^20 + 2*x1^41*x2^28*x3^19*x4^12*z^20 + x1^40*x2^29*x3^19*x4^12*z^20 + 3*x1^39*x2^30*x3^19*x4^12*z^20 - 2*x1^38*x2^31*x3^19*x4^12*z^20 - 2*x1^36*x2^33*x3^19*x4^12*z^20 - 6*x1^35*x2^34*x3^19*x4^12*z^20 - 3*x1^40*x2^28*x3^20*x4^12*z^20 - x1^39*x2^29*x3^20*x4^12*z^20 - 3*x1^38*x2^30*x3^20*x4^12*z^20 + 4*x1^37*x2^31*x3^20*x4^12*z^20 - 2*x1^36*x2^32*x3^20*x4^12*z^20 + 6*x1^35*x2^33*x3^20*x4^12*z^20 + 2*x1^34*x2^34*x3^20*x4^12*z^20 + x1^41*x2^26*x3^21*x4^12*z^20 + 4*x1^40*x2^27*x3^21*x4^12*z^20 + 2*x1^38*x2^29*x3^21*x4^12*z^20 - x1^35*x2^32*x3^21*x4^12*z^20 - 6*x1^34*x2^33*x3^21*x4^12*z^20 - 3*x1^39*x2^27*x3^22*x4^12*z^20 - x1^38*x2^28*x3^22*x4^12*z^20 - 3*x1^37*x2^29*x3^22*x4^12*z^20 + 3*x1^36*x2^30*x3^22*x4^12*z^20 - 2*x1^35*x2^31*x3^22*x4^12*z^20 + 6*x1^34*x2^32*x3^22*x4^12*z^20 + 2*x1^33*x2^33*x3^22*x4^12*z^20 + 3*x1^38*x2^27*x3^23*x4^12*z^20 + 2*x1^37*x2^28*x3^23*x4^12*z^20 - x1^36*x2^29*x3^23*x4^12*z^20 - x1^35*x2^30*x3^23*x4^12*z^20 - 6*x1^33*x2^32*x3^23*x4^12*z^20 + 4*x1^35*x2^29*x3^24*x4^12*z^20 - x1^34*x2^30*x3^24*x4^12*z^20 + 4*x1^33*x2^31*x3^24*x4^12*z^20 + 2*x1^32*x2^32*x3^24*x4^12*z^20 + x1^35*x2^28*x3^25*x4^12*z^20 - 4*x1^32*x2^31*x3^25*x4^12*z^20 - x1^33*x2^29*x3^26*x4^12*z^20 + x1^32*x2^30*x3^26*x4^12*z^20 + x1^31*x2^31*x3^26*x4^12*z^20 - x1^31*x2^30*x3^27*x4^12*z^20 - 2*x1^41*x2^32*x3^14*x4^13*z^20 - x1^40*x2^33*x3^14*x4^13*z^20 - x1^39*x2^34*x3^14*x4^13*z^20 + 2*x1^38*x2^34*x3^15*x4^13*z^20 - 2*x1^40*x2^31*x3^16*x4^13*z^20 + x1^39*x2^32*x3^16*x4^13*z^20 + x1^38*x2^33*x3^16*x4^13*z^20 + 3*x1^36*x2^35*x3^16*x4^13*z^20 - x1^43*x2^27*x3^17*x4^13*z^20 - x1^42*x2^28*x3^17*x4^13*z^20 + x1^41*x2^29*x3^17*x4^13*z^20 - x1^39*x2^31*x3^17*x4^13*z^20 - 3*x1^38*x2^32*x3^17*x4^13*z^20 + x1^37*x2^33*x3^17*x4^13*z^20 - 2*x1^36*x2^34*x3^17*x4^13*z^20 - x1^35*x2^35*x3^17*x4^13*z^20 - x1^41*x2^28*x3^18*x4^13*z^20 + x1^40*x2^29*x3^18*x4^13*z^20 - x1^39*x2^30*x3^18*x4^13*z^20 + 2*x1^38*x2^31*x3^18*x4^13*z^20 + x1^37*x2^32*x3^18*x4^13*z^20 + x1^36*x2^33*x3^18*x4^13*z^20 + 5*x1^35*x2^34*x3^18*x4^13*z^20 - x1^42*x2^26*x3^19*x4^13*z^20 - x1^41*x2^27*x3^19*x4^13*z^20 + 4*x1^40*x2^28*x3^19*x4^13*z^20 - 4*x1^37*x2^31*x3^19*x4^13*z^20 + x1^36*x2^32*x3^19*x4^13*z^20 - 4*x1^35*x2^33*x3^19*x4^13*z^20 - 2*x1^34*x2^34*x3^19*x4^13*z^20 - x1^41*x2^26*x3^20*x4^13*z^20 + x1^38*x2^29*x3^20*x4^13*z^20 + 2*x1^37*x2^30*x3^20*x4^13*z^20 + 2*x1^36*x2^31*x3^20*x4^13*z^20 + 2*x1^35*x2^32*x3^20*x4^13*z^20 + 6*x1^34*x2^33*x3^20*x4^13*z^20 - x1^40*x2^26*x3^21*x4^13*z^20 + x1^39*x2^27*x3^21*x4^13*z^20 + 2*x1^38*x2^28*x3^21*x4^13*z^20 + 2*x1^37*x2^29*x3^21*x4^13*z^20 - 3*x1^36*x2^30*x3^21*x4^13*z^20 - 6*x1^34*x2^32*x3^21*x4^13*z^20 - 2*x1^33*x2^33*x3^21*x4^13*z^20 + x1^39*x2^26*x3^22*x4^13*z^20 - x1^38*x2^27*x3^22*x4^13*z^20 - x1^37*x2^28*x3^22*x4^13*z^20 + x1^35*x2^30*x3^22*x4^13*z^20 + 2*x1^34*x2^31*x3^22*x4^13*z^20 + 6*x1^33*x2^32*x3^22*x4^13*z^20 - x1^38*x2^26*x3^23*x4^13*z^20 + x1^36*x2^28*x3^23*x4^13*z^20 - 2*x1^35*x2^29*x3^23*x4^13*z^20 + x1^34*x2^30*x3^23*x4^13*z^20 - 5*x1^33*x2^31*x3^23*x4^13*z^20 - 2*x1^32*x2^32*x3^23*x4^13*z^20 - x1^36*x2^27*x3^24*x4^13*z^20 + 5*x1^32*x2^31*x3^24*x4^13*z^20 - 2*x1^34*x2^28*x3^25*x4^13*z^20 + x1^33*x2^29*x3^25*x4^13*z^20 - 2*x1^32*x2^30*x3^25*x4^13*z^20 - 2*x1^31*x2^31*x3^25*x4^13*z^20 - x1^32*x2^29*x3^26*x4^13*z^20 + 2*x1^31*x2^30*x3^26*x4^13*z^20 - x1^30*x2^30*x3^27*x4^13*z^20 + x1^40*x2^31*x3^15*x4^14*z^20 - x1^39*x2^32*x3^15*x4^14*z^20 + x1^38*x2^33*x3^15*x4^14*z^20 - x1^39*x2^31*x3^16*x4^14*z^20 - x1^38*x2^32*x3^16*x4^14*z^20 - x1^37*x2^33*x3^16*x4^14*z^20 + 3*x1^39*x2^30*x3^17*x4^14*z^20 - x1^38*x2^31*x3^17*x4^14*z^20 + x1^37*x2^32*x3^17*x4^14*z^20 + 2*x1^36*x2^33*x3^17*x4^14*z^20 + x1^42*x2^26*x3^18*x4^14*z^20 + x1^39*x2^29*x3^18*x4^14*z^20 - 2*x1^38*x2^30*x3^18*x4^14*z^20 - x1^37*x2^31*x3^18*x4^14*z^20 - 2*x1^36*x2^32*x3^18*x4^14*z^20 + x1^35*x2^33*x3^18*x4^14*z^20 + x1^34*x2^34*x3^18*x4^14*z^20 - x1^41*x2^26*x3^19*x4^14*z^20 - x1^40*x2^27*x3^19*x4^14*z^20 + 3*x1^38*x2^29*x3^19*x4^14*z^20 - x1^37*x2^30*x3^19*x4^14*z^20 - x1^34*x2^33*x3^19*x4^14*z^20 - x1^38*x2^28*x3^20*x4^14*z^20 - 3*x1^35*x2^31*x3^20*x4^14*z^20 + x1^33*x2^33*x3^20*x4^14*z^20 - 2*x1^38*x2^27*x3^21*x4^14*z^20 + 2*x1^37*x2^28*x3^21*x4^14*z^20 + 2*x1^35*x2^30*x3^21*x4^14*z^20 - x1^34*x2^31*x3^21*x4^14*z^20 - 2*x1^33*x2^32*x3^21*x4^14*z^20 + x1^35*x2^29*x3^22*x4^14*z^20 - 3*x1^34*x2^30*x3^22*x4^14*z^20 + 2*x1^33*x2^31*x3^22*x4^14*z^20 + x1^36*x2^27*x3^23*x4^14*z^20 - x1^35*x2^28*x3^23*x4^14*z^20 + 2*x1^34*x2^29*x3^23*x4^14*z^20 + 2*x1^33*x2^30*x3^23*x4^14*z^20 - 2*x1^32*x2^31*x3^23*x4^14*z^20 - x1^35*x2^27*x3^24*x4^14*z^20 - 2*x1^33*x2^29*x3^24*x4^14*z^20 + x1^31*x2^31*x3^24*x4^14*z^20 - x1^39*x2^30*x3^16*x4^15*z^20 + x1^38*x2^31*x3^16*x4^15*z^20 - x1^37*x2^32*x3^16*x4^15*z^20 - x1^36*x2^33*x3^16*x4^15*z^20 + x1^39*x2^29*x3^17*x4^15*z^20 + 2*x1^38*x2^30*x3^17*x4^15*z^20 + 2*x1^36*x2^32*x3^17*x4^15*z^20 - 4*x1^38*x2^29*x3^18*x4^15*z^20 - 2*x1^36*x2^31*x3^18*x4^15*z^20 - 2*x1^35*x2^32*x3^18*x4^15*z^20 + x1^38*x2^28*x3^19*x4^15*z^20 + 3*x1^37*x2^29*x3^19*x4^15*z^20 + x1^36*x2^30*x3^19*x4^15*z^20 + 4*x1^35*x2^31*x3^19*x4^15*z^20 + x1^40*x2^25*x3^20*x4^15*z^20 + x1^39*x2^26*x3^20*x4^15*z^20 - x1^38*x2^27*x3^20*x4^15*z^20 - 4*x1^37*x2^28*x3^20*x4^15*z^20 + x1^36*x2^29*x3^20*x4^15*z^20 - 5*x1^35*x2^30*x3^20*x4^15*z^20 - 2*x1^34*x2^31*x3^20*x4^15*z^20 - x1^39*x2^25*x3^21*x4^15*z^20 + x1^37*x2^27*x3^21*x4^15*z^20 + 3*x1^36*x2^28*x3^21*x4^15*z^20 + 2*x1^35*x2^29*x3^21*x4^15*z^20 + 6*x1^34*x2^30*x3^21*x4^15*z^20 - 2*x1^36*x2^27*x3^22*x4^15*z^20 - 5*x1^34*x2^29*x3^22*x4^15*z^20 - 2*x1^33*x2^30*x3^22*x4^15*z^20 + x1^35*x2^27*x3^23*x4^15*z^20 + x1^34*x2^28*x3^23*x4^15*z^20 + 5*x1^33*x2^29*x3^23*x4^15*z^20 + x1^34*x2^27*x3^24*x4^15*z^20 - 2*x1^33*x2^28*x3^24*x4^15*z^20 - x1^32*x2^29*x3^24*x4^15*z^20 + 2*x1^32*x2^28*x3^25*x4^15*z^20 - x1^38*x2^28*x3^18*x4^16*z^20 - x1^37*x2^29*x3^18*x4^16*z^20 + x1^36*x2^30*x3^18*x4^16*z^20 - x1^35*x2^31*x3^18*x4^16*z^20 + 2*x1^37*x2^28*x3^19*x4^16*z^20 + 2*x1^35*x2^30*x3^19*x4^16*z^20 + x1^34*x2^31*x3^19*x4^16*z^20 - x1^36*x2^28*x3^20*x4^16*z^20 - 4*x1^34*x2^30*x3^20*x4^16*z^20 + x1^36*x2^27*x3^21*x4^16*z^20 - 2*x1^35*x2^28*x3^21*x4^16*z^20 + 3*x1^34*x2^29*x3^21*x4^16*z^20 + 2*x1^33*x2^30*x3^21*x4^16*z^20 + x1^36*x2^26*x3^22*x4^16*z^20 + x1^34*x2^28*x3^22*x4^16*z^20 - 4*x1^33*x2^29*x3^22*x4^16*z^20 - 2*x1^34*x2^27*x3^23*x4^16*z^20 + x1^33*x2^28*x3^23*x4^16*z^20 + 2*x1^32*x2^29*x3^23*x4^16*z^20 + x1^33*x2^27*x3^24*x4^16*z^20 - x1^32*x2^28*x3^24*x4^16*z^20 + 2*x1^33*x2^29*x3^21*x4^17*z^20 - x1^46*x2^33*x3^16*z^19 + x1^45*x2^34*x3^16*z^19 + x1^46*x2^32*x3^17*z^19 + x1^45*x2^33*x3^17*z^19 - 2*x1^44*x2^34*x3^17*z^19 - x1^45*x2^32*x3^18*z^19 - x1^44*x2^33*x3^18*z^19 + x1^42*x2^35*x3^18*z^19 + x1^45*x2^31*x3^19*z^19 + x1^44*x2^32*x3^19*z^19 - x1^43*x2^33*x3^19*z^19 + x1^42*x2^34*x3^19*z^19 - x1^44*x2^31*x3^20*z^19 + x1^43*x2^32*x3^20*z^19 - x1^42*x2^33*x3^20*z^19 - x1^41*x2^34*x3^20*z^19 + x1^47*x2^34*x3^13*x4*z^19 - x1^47*x2^33*x3^14*x4*z^19 - x1^46*x2^34*x3^14*x4*z^19 + x1^45*x2^35*x3^14*x4*z^19 + 4*x1^46*x2^33*x3^15*x4*z^19 + x1^45*x2^34*x3^15*x4*z^19 - 2*x1^46*x2^32*x3^16*x4*z^19 - 4*x1^45*x2^33*x3^16*x4*z^19 + x1^44*x2^34*x3^16*x4*z^19 - x1^43*x2^35*x3^16*x4*z^19 + x1^42*x2^36*x3^16*x4*z^19 + 6*x1^45*x2^32*x3^17*x4*z^19 + x1^44*x2^33*x3^17*x4*z^19 + x1^43*x2^34*x3^17*x4*z^19 + x1^42*x2^35*x3^17*x4*z^19 - x1^40*x2^37*x3^17*x4*z^19 - 2*x1^45*x2^31*x3^18*x4*z^19 - 6*x1^44*x2^32*x3^18*x4*z^19 - 4*x1^42*x2^34*x3^18*x4*z^19 + x1^40*x2^36*x3^18*x4*z^19 + x1^39*x2^37*x3^18*x4*z^19 + 5*x1^44*x2^31*x3^19*x4*z^19 + 2*x1^43*x2^32*x3^19*x4*z^19 + 2*x1^42*x2^33*x3^19*x4*z^19 + x1^41*x2^34*x3^19*x4*z^19 - x1^39*x2^36*x3^19*x4*z^19 - 2*x1^44*x2^30*x3^20*x4*z^19 - 3*x1^43*x2^31*x3^20*x4*z^19 + x1^42*x2^32*x3^20*x4*z^19 - 5*x1^41*x2^33*x3^20*x4*z^19 + x1^39*x2^35*x3^20*x4*z^19 + x1^38*x2^36*x3^20*x4*z^19 + 4*x1^43*x2^30*x3^21*x4*z^19 + 2*x1^41*x2^32*x3^21*x4*z^19 + 2*x1^40*x2^33*x3^21*x4*z^19 - 2*x1^43*x2^29*x3^22*x4*z^19 - x1^42*x2^30*x3^22*x4*z^19 + x1^41*x2^31*x3^22*x4*z^19 - 3*x1^40*x2^32*x3^22*x4*z^19 + x1^42*x2^29*x3^23*x4*z^19 + x1^41*x2^30*x3^23*x4*z^19 + x1^40*x2^31*x3^23*x4*z^19 + x1^47*x2^33*x3^13*x4^2*z^19 - 4*x1^46*x2^33*x3^14*x4^2*z^19 - x1^45*x2^34*x3^14*x4^2*z^19 - 2*x1^44*x2^35*x3^14*x4^2*z^19 + 2*x1^46*x2^32*x3^15*x4^2*z^19 + 4*x1^45*x2^33*x3^15*x4^2*z^19 - x1^44*x2^34*x3^15*x4^2*z^19 + 2*x1^43*x2^35*x3^15*x4^2*z^19 - x1^42*x2^36*x3^15*x4^2*z^19 - 6*x1^45*x2^32*x3^16*x4^2*z^19 - x1^44*x2^33*x3^16*x4^2*z^19 - x1^43*x2^34*x3^16*x4^2*z^19 - x1^42*x2^35*x3^16*x4^2*z^19 + 2*x1^45*x2^31*x3^17*x4^2*z^19 + 6*x1^44*x2^32*x3^17*x4^2*z^19 - x1^43*x2^33*x3^17*x4^2*z^19 + 4*x1^42*x2^34*x3^17*x4^2*z^19 - 2*x1^41*x2^35*x3^17*x4^2*z^19 - x1^39*x2^37*x3^17*x4^2*z^19 - 6*x1^44*x2^31*x3^18*x4^2*z^19 - 2*x1^43*x2^32*x3^18*x4^2*z^19 - x1^42*x2^33*x3^18*x4^2*z^19 - x1^41*x2^34*x3^18*x4^2*z^19 + 2*x1^39*x2^36*x3^18*x4^2*z^19 + x1^38*x2^37*x3^18*x4^2*z^19 + 2*x1^44*x2^30*x3^19*x4^2*z^19 + 6*x1^43*x2^31*x3^19*x4^2*z^19 + 4*x1^41*x2^33*x3^19*x4^2*z^19 - x1^40*x2^34*x3^19*x4^2*z^19 - x1^39*x2^35*x3^19*x4^2*z^19 - 3*x1^38*x2^36*x3^19*x4^2*z^19 - 6*x1^43*x2^30*x3^20*x4^2*z^19 - 2*x1^42*x2^31*x3^20*x4^2*z^19 - 2*x1^41*x2^32*x3^20*x4^2*z^19 - 2*x1^40*x2^33*x3^20*x4^2*z^19 + 2*x1^38*x2^35*x3^20*x4^2*z^19 + x1^37*x2^36*x3^20*x4^2*z^19 + 2*x1^43*x2^29*x3^21*x4^2*z^19 + 4*x1^42*x2^30*x3^21*x4^2*z^19 + 5*x1^40*x2^32*x3^21*x4^2*z^19 - x1^39*x2^33*x3^21*x4^2*z^19 - 2*x1^37*x2^35*x3^21*x4^2*z^19 - 5*x1^42*x2^29*x3^22*x4^2*z^19 - x1^41*x2^30*x3^22*x4^2*z^19 - x1^40*x2^31*x3^22*x4^2*z^19 - 2*x1^39*x2^32*x3^22*x4^2*z^19 - x1^38*x2^33*x3^22*x4^2*z^19 + x1^37*x2^34*x3^22*x4^2*z^19 + x1^36*x2^35*x3^22*x4^2*z^19 + x1^42*x2^28*x3^23*x4^2*z^19 + 2*x1^41*x2^29*x3^23*x4^2*z^19 + 4*x1^39*x2^31*x3^23*x4^2*z^19 - x1^41*x2^28*x3^24*x4^2*z^19 - x1^40*x2^29*x3^24*x4^2*z^19 - x1^39*x2^30*x3^24*x4^2*z^19 - x1^38*x2^31*x3^24*x4^2*z^19 + 2*x1^38*x2^30*x3^25*x4^2*z^19 - x1^48*x2^31*x3^13*x4^3*z^19 + x1^46*x2^33*x3^13*x4^3*z^19 - x1^45*x2^34*x3^13*x4^3*z^19 + 2*x1^47*x2^31*x3^14*x4^3*z^19 - 2*x1^46*x2^32*x3^14*x4^3*z^19 - x1^45*x2^33*x3^14*x4^3*z^19 + 2*x1^44*x2^34*x3^14*x4^3*z^19 - 2*x1^46*x2^31*x3^15*x4^3*z^19 + 3*x1^45*x2^32*x3^15*x4^3*z^19 - x1^44*x2^33*x3^15*x4^3*z^19 - x1^43*x2^34*x3^15*x4^3*z^19 - x1^42*x2^35*x3^15*x4^3*z^19 - x1^41*x2^36*x3^15*x4^3*z^19 + 2*x1^46*x2^30*x3^16*x4^3*z^19 + x1^45*x2^31*x3^16*x4^3*z^19 - 2*x1^44*x2^32*x3^16*x4^3*z^19 - x1^42*x2^34*x3^16*x4^3*z^19 + 2*x1^41*x2^35*x3^16*x4^3*z^19 + x1^40*x2^36*x3^16*x4^3*z^19 - x1^46*x2^29*x3^17*x4^3*z^19 - 2*x1^45*x2^30*x3^17*x4^3*z^19 + 2*x1^44*x2^31*x3^17*x4^3*z^19 + x1^42*x2^33*x3^17*x4^3*z^19 - x1^40*x2^35*x3^17*x4^3*z^19 - x1^39*x2^36*x3^17*x4^3*z^19 + 2*x1^45*x2^29*x3^18*x4^3*z^19 - x1^43*x2^31*x3^18*x4^3*z^19 + x1^42*x2^32*x3^18*x4^3*z^19 - 2*x1^41*x2^33*x3^18*x4^3*z^19 + x1^40*x2^34*x3^18*x4^3*z^19 + 2*x1^39*x2^35*x3^18*x4^3*z^19 + 2*x1^38*x2^36*x3^18*x4^3*z^19 - x1^45*x2^28*x3^19*x4^3*z^19 - 2*x1^44*x2^29*x3^19*x4^3*z^19 + 2*x1^43*x2^30*x3^19*x4^3*z^19 + 2*x1^41*x2^32*x3^19*x4^3*z^19 - 2*x1^38*x2^35*x3^19*x4^3*z^19 - 2*x1^37*x2^36*x3^19*x4^3*z^19 + 2*x1^44*x2^28*x3^20*x4^3*z^19 - x1^43*x2^29*x3^20*x4^3*z^19 - 2*x1^42*x2^30*x3^20*x4^3*z^19 + x1^41*x2^31*x3^20*x4^3*z^19 + 2*x1^39*x2^33*x3^20*x4^3*z^19 - 2*x1^38*x2^34*x3^20*x4^3*z^19 + 2*x1^37*x2^35*x3^20*x4^3*z^19 + x1^36*x2^36*x3^20*x4^3*z^19 - x1^43*x2^28*x3^21*x4^3*z^19 + 2*x1^42*x2^29*x3^21*x4^3*z^19 - x1^41*x2^30*x3^21*x4^3*z^19 + 2*x1^39*x2^32*x3^21*x4^3*z^19 - 2*x1^37*x2^34*x3^21*x4^3*z^19 - x1^36*x2^35*x3^21*x4^3*z^19 + 2*x1^43*x2^27*x3^22*x4^3*z^19 + x1^42*x2^28*x3^22*x4^3*z^19 - 2*x1^41*x2^29*x3^22*x4^3*z^19 - x1^40*x2^30*x3^22*x4^3*z^19 - x1^39*x2^31*x3^22*x4^3*z^19 + x1^38*x2^32*x3^22*x4^3*z^19 - x1^37*x2^33*x3^22*x4^3*z^19 + x1^36*x2^34*x3^22*x4^3*z^19 + x1^41*x2^28*x3^23*x4^3*z^19 - x1^39*x2^30*x3^23*x4^3*z^19 + x1^38*x2^31*x3^23*x4^3*z^19 + x1^37*x2^32*x3^23*x4^3*z^19 - x1^36*x2^33*x3^23*x4^3*z^19 - x1^35*x2^34*x3^23*x4^3*z^19 - x1^40*x2^28*x3^24*x4^3*z^19 - x1^39*x2^29*x3^24*x4^3*z^19 - 3*x1^38*x2^30*x3^24*x4^3*z^19 + x1^37*x2^30*x3^25*x4^3*z^19 - x1^37*x2^29*x3^26*x4^3*z^19 - x1^48*x2^32*x3^11*x4^4*z^19 + x1^47*x2^32*x3^12*x4^4*z^19 - 2*x1^46*x2^33*x3^12*x4^4*z^19 - 3*x1^47*x2^31*x3^13*x4^4*z^19 - x1^46*x2^32*x3^13*x4^4*z^19 + x1^44*x2^34*x3^13*x4^4*z^19 + 2*x1^47*x2^30*x3^14*x4^4*z^19 + 3*x1^46*x2^31*x3^14*x4^4*z^19 - x1^45*x2^32*x3^14*x4^4*z^19 + x1^44*x2^33*x3^14*x4^4*z^19 - x1^43*x2^34*x3^14*x4^4*z^19 - 6*x1^46*x2^30*x3^15*x4^4*z^19 - x1^44*x2^32*x3^15*x4^4*z^19 + x1^43*x2^33*x3^15*x4^4*z^19 + x1^42*x2^34*x3^15*x4^4*z^19 + 2*x1^41*x2^35*x3^15*x4^4*z^19 + 2*x1^46*x2^29*x3^16*x4^4*z^19 + 6*x1^45*x2^30*x3^16*x4^4*z^19 + 5*x1^43*x2^32*x3^16*x4^4*z^19 - 2*x1^42*x2^33*x3^16*x4^4*z^19 - 2*x1^40*x2^35*x3^16*x4^4*z^19 + x1^39*x2^36*x3^16*x4^4*z^19 - 6*x1^45*x2^29*x3^17*x4^4*z^19 - 2*x1^44*x2^30*x3^17*x4^4*z^19 - 2*x1^43*x2^31*x3^17*x4^4*z^19 + 2*x1^41*x2^33*x3^17*x4^4*z^19 + 2*x1^40*x2^34*x3^17*x4^4*z^19 + 2*x1^39*x2^35*x3^17*x4^4*z^19 + 2*x1^45*x2^28*x3^18*x4^4*z^19 + 6*x1^44*x2^29*x3^18*x4^4*z^19 + 4*x1^42*x2^31*x3^18*x4^4*z^19 - 4*x1^41*x2^32*x3^18*x4^4*z^19 - x1^40*x2^33*x3^18*x4^4*z^19 - 6*x1^39*x2^34*x3^18*x4^4*z^19 - 2*x1^38*x2^35*x3^18*x4^4*z^19 - 6*x1^44*x2^28*x3^19*x4^4*z^19 - 2*x1^43*x2^29*x3^19*x4^4*z^19 - 2*x1^42*x2^30*x3^19*x4^4*z^19 - x1^41*x2^31*x3^19*x4^4*z^19 + 2*x1^40*x2^32*x3^19*x4^4*z^19 + x1^39*x2^33*x3^19*x4^4*z^19 + 3*x1^38*x2^34*x3^19*x4^4*z^19 - x1^36*x2^36*x3^19*x4^4*z^19 + 2*x1^44*x2^27*x3^20*x4^4*z^19 + 6*x1^43*x2^28*x3^20*x4^4*z^19 + 4*x1^41*x2^30*x3^20*x4^4*z^19 - x1^40*x2^31*x3^20*x4^4*z^19 + x1^39*x2^32*x3^20*x4^4*z^19 - 3*x1^38*x2^33*x3^20*x4^4*z^19 - x1^37*x2^34*x3^20*x4^4*z^19 - 6*x1^43*x2^27*x3^21*x4^4*z^19 - 2*x1^42*x2^28*x3^21*x4^4*z^19 - 2*x1^41*x2^29*x3^21*x4^4*z^19 - 2*x1^40*x2^30*x3^21*x4^4*z^19 + x1^38*x2^32*x3^21*x4^4*z^19 + 3*x1^37*x2^33*x3^21*x4^4*z^19 - x1^36*x2^34*x3^21*x4^4*z^19 - x1^35*x2^35*x3^21*x4^4*z^19 + x1^43*x2^26*x3^22*x4^4*z^19 + 5*x1^42*x2^27*x3^22*x4^4*z^19 + 3*x1^40*x2^29*x3^22*x4^4*z^19 - 2*x1^37*x2^32*x3^22*x4^4*z^19 + x1^36*x2^33*x3^22*x4^4*z^19 - 3*x1^42*x2^26*x3^23*x4^4*z^19 - x1^41*x2^27*x3^23*x4^4*z^19 - x1^38*x2^30*x3^23*x4^4*z^19 + x1^37*x2^31*x3^23*x4^4*z^19 + 2*x1^36*x2^32*x3^23*x4^4*z^19 - 2*x1^35*x2^33*x3^23*x4^4*z^19 + x1^41*x2^26*x3^24*x4^4*z^19 + x1^39*x2^28*x3^24*x4^4*z^19 - x1^38*x2^29*x3^24*x4^4*z^19 + x1^36*x2^31*x3^24*x4^4*z^19 + x1^35*x2^32*x3^24*x4^4*z^19 - x1^37*x2^28*x3^26*x4^4*z^19 - 2*x1^36*x2^29*x3^26*x4^4*z^19 + x1^47*x2^32*x3^11*x4^5*z^19 + 2*x1^47*x2^31*x3^12*x4^5*z^19 - x1^47*x2^30*x3^13*x4^5*z^19 - 4*x1^46*x2^31*x3^13*x4^5*z^19 + x1^45*x2^32*x3^13*x4^5*z^19 - x1^44*x2^33*x3^13*x4^5*z^19 + x1^43*x2^34*x3^13*x4^5*z^19 + 6*x1^46*x2^30*x3^14*x4^5*z^19 + x1^45*x2^31*x3^14*x4^5*z^19 - 2*x1^42*x2^34*x3^14*x4^5*z^19 - x1^41*x2^35*x3^14*x4^5*z^19 - 2*x1^46*x2^29*x3^15*x4^5*z^19 - 6*x1^45*x2^30*x3^15*x4^5*z^19 + 3*x1^44*x2^31*x3^15*x4^5*z^19 - 2*x1^43*x2^32*x3^15*x4^5*z^19 + 3*x1^42*x2^33*x3^15*x4^5*z^19 + x1^41*x2^34*x3^15*x4^5*z^19 + x1^40*x2^35*x3^15*x4^5*z^19 + 6*x1^45*x2^29*x3^16*x4^5*z^19 + x1^44*x2^30*x3^16*x4^5*z^19 - x1^43*x2^31*x3^16*x4^5*z^19 - 2*x1^41*x2^33*x3^16*x4^5*z^19 - x1^40*x2^34*x3^16*x4^5*z^19 - x1^39*x2^35*x3^16*x4^5*z^19 - x1^38*x2^36*x3^16*x4^5*z^19 - 2*x1^45*x2^28*x3^17*x4^5*z^19 - 6*x1^44*x2^29*x3^17*x4^5*z^19 + 2*x1^43*x2^30*x3^17*x4^5*z^19 - 4*x1^42*x2^31*x3^17*x4^5*z^19 + 3*x1^41*x2^32*x3^17*x4^5*z^19 + x1^40*x2^33*x3^17*x4^5*z^19 + 4*x1^39*x2^34*x3^17*x4^5*z^19 + x1^37*x2^36*x3^17*x4^5*z^19 + 6*x1^44*x2^28*x3^18*x4^5*z^19 + 2*x1^43*x2^29*x3^18*x4^5*z^19 - 4*x1^40*x2^32*x3^18*x4^5*z^19 - 2*x1^39*x2^33*x3^18*x4^5*z^19 - 3*x1^38*x2^34*x3^18*x4^5*z^19 + x1^37*x2^35*x3^18*x4^5*z^19 - 2*x1^44*x2^27*x3^19*x4^5*z^19 - 6*x1^43*x2^28*x3^19*x4^5*z^19 + 2*x1^42*x2^29*x3^19*x4^5*z^19 - 3*x1^41*x2^30*x3^19*x4^5*z^19 + 5*x1^40*x2^31*x3^19*x4^5*z^19 + 5*x1^38*x2^33*x3^19*x4^5*z^19 - x1^36*x2^35*x3^19*x4^5*z^19 + 6*x1^43*x2^27*x3^20*x4^5*z^19 + x1^42*x2^28*x3^20*x4^5*z^19 - 3*x1^39*x2^31*x3^20*x4^5*z^19 - 3*x1^38*x2^32*x3^20*x4^5*z^19 - 6*x1^37*x2^33*x3^20*x4^5*z^19 + 2*x1^36*x2^34*x3^20*x4^5*z^19 - 6*x1^42*x2^27*x3^21*x4^5*z^19 + 2*x1^41*x2^28*x3^21*x4^5*z^19 - 3*x1^40*x2^29*x3^21*x4^5*z^19 + 3*x1^39*x2^30*x3^21*x4^5*z^19 + 4*x1^37*x2^32*x3^21*x4^5*z^19 + x1^36*x2^33*x3^21*x4^5*z^19 - x1^35*x2^34*x3^21*x4^5*z^19 + 2*x1^42*x2^26*x3^22*x4^5*z^19 + 2*x1^41*x2^27*x3^22*x4^5*z^19 - x1^40*x2^28*x3^22*x4^5*z^19 + x1^39*x2^29*x3^22*x4^5*z^19 - 2*x1^38*x2^30*x3^22*x4^5*z^19 - 4*x1^36*x2^32*x3^22*x4^5*z^19 + 3*x1^35*x2^33*x3^22*x4^5*z^19 - 2*x1^41*x2^26*x3^23*x4^5*z^19 - 3*x1^39*x2^28*x3^23*x4^5*z^19 - x1^38*x2^29*x3^23*x4^5*z^19 + 3*x1^36*x2^31*x3^23*x4^5*z^19 + x1^35*x2^32*x3^23*x4^5*z^19 - x1^34*x2^33*x3^23*x4^5*z^19 + x1^39*x2^27*x3^24*x4^5*z^19 - x1^38*x2^28*x3^24*x4^5*z^19 - x1^37*x2^29*x3^24*x4^5*z^19 - 2*x1^36*x2^30*x3^24*x4^5*z^19 - x1^35*x2^31*x3^24*x4^5*z^19 + 2*x1^34*x2^32*x3^24*x4^5*z^19 - x1^39*x2^26*x3^25*x4^5*z^19 - x1^38*x2^27*x3^25*x4^5*z^19 + x1^37*x2^28*x3^25*x4^5*z^19 + x1^36*x2^28*x3^26*x4^5*z^19 - x1^35*x2^29*x3^26*x4^5*z^19 + 2*x1^33*x2^31*x3^26*x4^5*z^19 + x1^35*x2^28*x3^27*x4^5*z^19 + x1^46*x2^31*x3^12*x4^6*z^19 + x1^44*x2^33*x3^12*x4^6*z^19 - x1^45*x2^31*x3^13*x4^6*z^19 + x1^44*x2^32*x3^13*x4^6*z^19 + x1^42*x2^34*x3^13*x4^6*z^19 + 2*x1^45*x2^30*x3^14*x4^6*z^19 - 2*x1^42*x2^33*x3^14*x4^6*z^19 + x1^40*x2^35*x3^14*x4^6*z^19 - 2*x1^45*x2^29*x3^15*x4^6*z^19 + x1^44*x2^30*x3^15*x4^6*z^19 + x1^43*x2^31*x3^15*x4^6*z^19 - 2*x1^42*x2^32*x3^15*x4^6*z^19 + x1^40*x2^34*x3^15*x4^6*z^19 - x1^39*x2^35*x3^15*x4^6*z^19 + 2*x1^44*x2^29*x3^16*x4^6*z^19 - 4*x1^43*x2^30*x3^16*x4^6*z^19 + x1^42*x2^31*x3^16*x4^6*z^19 - x1^41*x2^32*x3^16*x4^6*z^19 - x1^39*x2^34*x3^16*x4^6*z^19 + x1^38*x2^35*x3^16*x4^6*z^19 - 2*x1^44*x2^28*x3^17*x4^6*z^19 + 3*x1^42*x2^30*x3^17*x4^6*z^19 + 2*x1^40*x2^32*x3^17*x4^6*z^19 - 2*x1^39*x2^33*x3^17*x4^6*z^19 + x1^38*x2^34*x3^17*x4^6*z^19 + x1^44*x2^27*x3^18*x4^6*z^19 + 2*x1^43*x2^28*x3^18*x4^6*z^19 - 4*x1^42*x2^29*x3^18*x4^6*z^19 - 2*x1^40*x2^31*x3^18*x4^6*z^19 - 3*x1^38*x2^33*x3^18*x4^6*z^19 - x1^37*x2^34*x3^18*x4^6*z^19 - 2*x1^43*x2^27*x3^19*x4^6*z^19 + 3*x1^41*x2^29*x3^19*x4^6*z^19 + 4*x1^39*x2^31*x3^19*x4^6*z^19 - x1^38*x2^32*x3^19*x4^6*z^19 + 2*x1^37*x2^33*x3^19*x4^6*z^19 - 4*x1^36*x2^34*x3^19*x4^6*z^19 - x1^35*x2^35*x3^19*x4^6*z^19 + 2*x1^42*x2^27*x3^20*x4^6*z^19 - 4*x1^41*x2^28*x3^20*x4^6*z^19 - x1^40*x2^29*x3^20*x4^6*z^19 - 4*x1^39*x2^30*x3^20*x4^6*z^19 + 2*x1^36*x2^33*x3^20*x4^6*z^19 + 4*x1^35*x2^34*x3^20*x4^6*z^19 - x1^42*x2^26*x3^21*x4^6*z^19 + x1^41*x2^27*x3^21*x4^6*z^19 + 4*x1^40*x2^28*x3^21*x4^6*z^19 + x1^39*x2^29*x3^21*x4^6*z^19 + 2*x1^38*x2^30*x3^21*x4^6*z^19 - x1^37*x2^31*x3^21*x4^6*z^19 + 4*x1^36*x2^32*x3^21*x4^6*z^19 - 4*x1^35*x2^33*x3^21*x4^6*z^19 + x1^41*x2^26*x3^22*x4^6*z^19 - 4*x1^40*x2^27*x3^22*x4^6*z^19 + x1^39*x2^28*x3^22*x4^6*z^19 - 2*x1^38*x2^29*x3^22*x4^6*z^19 - 2*x1^37*x2^30*x3^22*x4^6*z^19 - 2*x1^36*x2^31*x3^22*x4^6*z^19 + 3*x1^34*x2^33*x3^22*x4^6*z^19 - x1^41*x2^25*x3^23*x4^6*z^19 - x1^40*x2^26*x3^23*x4^6*z^19 + 3*x1^39*x2^27*x3^23*x4^6*z^19 + 3*x1^37*x2^29*x3^23*x4^6*z^19 - x1^36*x2^30*x3^23*x4^6*z^19 + x1^35*x2^31*x3^23*x4^6*z^19 - 3*x1^34*x2^32*x3^23*x4^6*z^19 + x1^33*x2^33*x3^23*x4^6*z^19 + x1^40*x2^25*x3^24*x4^6*z^19 - x1^38*x2^27*x3^24*x4^6*z^19 - x1^37*x2^28*x3^24*x4^6*z^19 + x1^36*x2^29*x3^24*x4^6*z^19 - 2*x1^35*x2^30*x3^24*x4^6*z^19 - x1^34*x2^31*x3^24*x4^6*z^19 + 2*x1^33*x2^32*x3^24*x4^6*z^19 - x1^38*x2^26*x3^25*x4^6*z^19 + 3*x1^36*x2^28*x3^25*x4^6*z^19 - 2*x1^34*x2^30*x3^25*x4^6*z^19 - 2*x1^33*x2^31*x3^25*x4^6*z^19 + x1^37*x2^26*x3^26*x4^6*z^19 - x1^36*x2^27*x3^26*x4^6*z^19 - 2*x1^35*x2^28*x3^26*x4^6*z^19 + x1^34*x2^29*x3^26*x4^6*z^19 - x1^35*x2^27*x3^27*x4^6*z^19 - x1^32*x2^30*x3^27*x4^6*z^19 + x1^46*x2^31*x3^11*x4^7*z^19 - x1^45*x2^32*x3^11*x4^7*z^19 + x1^43*x2^34*x3^11*x4^7*z^19 - x1^46*x2^30*x3^12*x4^7*z^19 + x1^45*x2^31*x3^12*x4^7*z^19 + 2*x1^44*x2^32*x3^12*x4^7*z^19 - x1^43*x2^33*x3^12*x4^7*z^19 + 2*x1^45*x2^30*x3^13*x4^7*z^19 - x1^43*x2^32*x3^13*x4^7*z^19 - x1^42*x2^33*x3^13*x4^7*z^19 - x1^41*x2^34*x3^13*x4^7*z^19 + x1^40*x2^35*x3^13*x4^7*z^19 - 2*x1^45*x2^29*x3^14*x4^7*z^19 + x1^44*x2^30*x3^14*x4^7*z^19 + x1^43*x2^31*x3^14*x4^7*z^19 - 2*x1^40*x2^34*x3^14*x4^7*z^19 - x1^39*x2^35*x3^14*x4^7*z^19 - x1^38*x2^36*x3^14*x4^7*z^19 + x1^44*x2^29*x3^15*x4^7*z^19 + 3*x1^42*x2^31*x3^15*x4^7*z^19 + x1^41*x2^32*x3^15*x4^7*z^19 - x1^40*x2^33*x3^15*x4^7*z^19 + x1^39*x2^34*x3^15*x4^7*z^19 + x1^38*x2^35*x3^15*x4^7*z^19 + x1^37*x2^36*x3^15*x4^7*z^19 - 3*x1^37*x2^35*x3^16*x4^7*z^19 + x1^39*x2^32*x3^17*x4^7*z^19 + 3*x1^36*x2^35*x3^17*x4^7*z^19 + x1^37*x2^33*x3^18*x4^7*z^19 - x1^36*x2^34*x3^18*x4^7*z^19 - x1^35*x2^35*x3^18*x4^7*z^19 + x1^41*x2^28*x3^19*x4^7*z^19 + x1^35*x2^34*x3^19*x4^7*z^19 - x1^41*x2^27*x3^20*x4^7*z^19 - x1^40*x2^28*x3^20*x4^7*z^19 + x1^39*x2^29*x3^20*x4^7*z^19 + x1^42*x2^25*x3^21*x4^7*z^19 + 2*x1^40*x2^27*x3^21*x4^7*z^19 + x1^39*x2^28*x3^21*x4^7*z^19 - x1^41*x2^25*x3^22*x4^7*z^19 - x1^40*x2^26*x3^22*x4^7*z^19 - 2*x1^39*x2^27*x3^22*x4^7*z^19 - 2*x1^38*x2^28*x3^22*x4^7*z^19 - x1^37*x2^29*x3^22*x4^7*z^19 + x1^36*x2^30*x3^22*x4^7*z^19 - 2*x1^35*x2^31*x3^22*x4^7*z^19 + x1^40*x2^25*x3^23*x4^7*z^19 + x1^39*x2^26*x3^23*x4^7*z^19 + x1^37*x2^28*x3^23*x4^7*z^19 + 2*x1^36*x2^29*x3^23*x4^7*z^19 + x1^35*x2^30*x3^23*x4^7*z^19 - x1^33*x2^32*x3^23*x4^7*z^19 - x1^39*x2^25*x3^24*x4^7*z^19 - x1^38*x2^26*x3^24*x4^7*z^19 - x1^37*x2^27*x3^24*x4^7*z^19 - 3*x1^36*x2^28*x3^24*x4^7*z^19 - 2*x1^35*x2^29*x3^24*x4^7*z^19 + x1^33*x2^31*x3^24*x4^7*z^19 - x1^32*x2^32*x3^24*x4^7*z^19 + x1^37*x2^26*x3^25*x4^7*z^19 + x1^36*x2^26*x3^26*x4^7*z^19 + x1^33*x2^29*x3^26*x4^7*z^19 + x1^34*x2^27*x3^27*x4^7*z^19 - x1^46*x2^30*x3^11*x4^8*z^19 + x1^44*x2^32*x3^11*x4^8*z^19 - x1^43*x2^33*x3^11*x4^8*z^19 - x1^42*x2^34*x3^11*x4^8*z^19 + x1^41*x2^35*x3^11*x4^8*z^19 - x1^45*x2^30*x3^12*x4^8*z^19 + x1^43*x2^32*x3^12*x4^8*z^19 - x1^40*x2^35*x3^12*x4^8*z^19 - x1^44*x2^30*x3^13*x4^8*z^19 - 2*x1^43*x2^31*x3^13*x4^8*z^19 + x1^40*x2^34*x3^13*x4^8*z^19 + 2*x1^39*x2^35*x3^13*x4^8*z^19 + 2*x1^38*x2^36*x3^13*x4^8*z^19 + x1^45*x2^28*x3^14*x4^8*z^19 + 3*x1^43*x2^30*x3^14*x4^8*z^19 + 2*x1^42*x2^31*x3^14*x4^8*z^19 - 3*x1^38*x2^35*x3^14*x4^8*z^19 - 2*x1^37*x2^36*x3^14*x4^8*z^19 - x1^44*x2^28*x3^15*x4^8*z^19 - x1^43*x2^29*x3^15*x4^8*z^19 - 3*x1^42*x2^30*x3^15*x4^8*z^19 - x1^41*x2^31*x3^15*x4^8*z^19 + 3*x1^39*x2^33*x3^15*x4^8*z^19 - x1^38*x2^34*x3^15*x4^8*z^19 + 4*x1^37*x2^35*x3^15*x4^8*z^19 + x1^44*x2^27*x3^16*x4^8*z^19 + 3*x1^42*x2^29*x3^16*x4^8*z^19 + x1^41*x2^30*x3^16*x4^8*z^19 - x1^40*x2^31*x3^16*x4^8*z^19 - 2*x1^38*x2^33*x3^16*x4^8*z^19 - 4*x1^36*x2^35*x3^16*x4^8*z^19 - x1^43*x2^27*x3^17*x4^8*z^19 - x1^42*x2^28*x3^17*x4^8*z^19 - 3*x1^41*x2^29*x3^17*x4^8*z^19 + x1^40*x2^30*x3^17*x4^8*z^19 - x1^39*x2^31*x3^17*x4^8*z^19 + 2*x1^38*x2^32*x3^17*x4^8*z^19 + x1^37*x2^33*x3^17*x4^8*z^19 + 4*x1^36*x2^34*x3^17*x4^8*z^19 + x1^35*x2^35*x3^17*x4^8*z^19 + 2*x1^42*x2^27*x3^18*x4^8*z^19 + 4*x1^41*x2^28*x3^18*x4^8*z^19 + 4*x1^40*x2^29*x3^18*x4^8*z^19 + x1^39*x2^30*x3^18*x4^8*z^19 - 4*x1^37*x2^32*x3^18*x4^8*z^19 - 2*x1^36*x2^33*x3^18*x4^8*z^19 - 4*x1^35*x2^34*x3^18*x4^8*z^19 - 2*x1^42*x2^26*x3^19*x4^8*z^19 - 2*x1^41*x2^27*x3^19*x4^8*z^19 - 5*x1^40*x2^28*x3^19*x4^8*z^19 - x1^38*x2^30*x3^19*x4^8*z^19 + 4*x1^37*x2^31*x3^19*x4^8*z^19 + 2*x1^36*x2^32*x3^19*x4^8*z^19 + 4*x1^35*x2^33*x3^19*x4^8*z^19 + x1^34*x2^34*x3^19*x4^8*z^19 + x1^42*x2^25*x3^20*x4^8*z^19 + 2*x1^41*x2^26*x3^20*x4^8*z^19 + x1^39*x2^28*x3^20*x4^8*z^19 - 2*x1^38*x2^29*x3^20*x4^8*z^19 - 3*x1^36*x2^31*x3^20*x4^8*z^19 - 2*x1^35*x2^32*x3^20*x4^8*z^19 - 4*x1^34*x2^33*x3^20*x4^8*z^19 - 2*x1^41*x2^25*x3^21*x4^8*z^19 + x1^40*x2^26*x3^21*x4^8*z^19 - x1^38*x2^28*x3^21*x4^8*z^19 + 3*x1^36*x2^30*x3^21*x4^8*z^19 + 2*x1^35*x2^31*x3^21*x4^8*z^19 + 4*x1^34*x2^32*x3^21*x4^8*z^19 + x1^33*x2^33*x3^21*x4^8*z^19 + 2*x1^40*x2^25*x3^22*x4^8*z^19 - x1^39*x2^26*x3^22*x4^8*z^19 + x1^38*x2^27*x3^22*x4^8*z^19 - x1^37*x2^28*x3^22*x4^8*z^19 - x1^36*x2^29*x3^22*x4^8*z^19 - 3*x1^35*x2^30*x3^22*x4^8*z^19 - x1^34*x2^31*x3^22*x4^8*z^19 - 4*x1^33*x2^32*x3^22*x4^8*z^19 - x1^39*x2^25*x3^23*x4^8*z^19 + x1^38*x2^26*x3^23*x4^8*z^19 + x1^37*x2^27*x3^23*x4^8*z^19 + x1^36*x2^28*x3^23*x4^8*z^19 + x1^34*x2^30*x3^23*x4^8*z^19 + 3*x1^33*x2^31*x3^23*x4^8*z^19 + x1^32*x2^32*x3^23*x4^8*z^19 + x1^38*x2^25*x3^24*x4^8*z^19 - x1^36*x2^27*x3^24*x4^8*z^19 - 2*x1^35*x2^28*x3^24*x4^8*z^19 - 3*x1^34*x2^29*x3^24*x4^8*z^19 - 2*x1^32*x2^31*x3^24*x4^8*z^19 - x1^37*x2^25*x3^25*x4^8*z^19 + x1^35*x2^27*x3^25*x4^8*z^19 + 3*x1^34*x2^28*x3^25*x4^8*z^19 + x1^32*x2^30*x3^25*x4^8*z^19 + x1^31*x2^31*x3^25*x4^8*z^19 - x1^35*x2^26*x3^26*x4^8*z^19 - 2*x1^33*x2^28*x3^26*x4^8*z^19 + x1^31*x2^29*x3^27*x4^8*z^19 + x1^42*x2^33*x3^11*x4^9*z^19 + x1^45*x2^29*x3^12*x4^9*z^19 + x1^44*x2^30*x3^12*x4^9*z^19 - x1^40*x2^34*x3^12*x4^9*z^19 - x1^39*x2^35*x3^12*x4^9*z^19 - x1^44*x2^29*x3^13*x4^9*z^19 - x1^43*x2^30*x3^13*x4^9*z^19 - x1^40*x2^33*x3^13*x4^9*z^19 + x1^39*x2^34*x3^13*x4^9*z^19 + 2*x1^44*x2^28*x3^14*x4^9*z^19 + x1^43*x2^29*x3^14*x4^9*z^19 - 2*x1^40*x2^32*x3^14*x4^9*z^19 - 2*x1^39*x2^33*x3^14*x4^9*z^19 - x1^38*x2^34*x3^14*x4^9*z^19 - 2*x1^37*x2^35*x3^14*x4^9*z^19 - x1^44*x2^27*x3^15*x4^9*z^19 - x1^43*x2^28*x3^15*x4^9*z^19 - 2*x1^42*x2^29*x3^15*x4^9*z^19 + 3*x1^40*x2^31*x3^15*x4^9*z^19 + 2*x1^39*x2^32*x3^15*x4^9*z^19 - x1^38*x2^33*x3^15*x4^9*z^19 + 2*x1^36*x2^35*x3^15*x4^9*z^19 + 4*x1^43*x2^27*x3^16*x4^9*z^19 + x1^42*x2^28*x3^16*x4^9*z^19 + 2*x1^41*x2^29*x3^16*x4^9*z^19 - x1^40*x2^30*x3^16*x4^9*z^19 + x1^39*x2^31*x3^16*x4^9*z^19 - 2*x1^38*x2^32*x3^16*x4^9*z^19 - 2*x1^36*x2^34*x3^16*x4^9*z^19 - x1^43*x2^26*x3^17*x4^9*z^19 - 3*x1^42*x2^27*x3^17*x4^9*z^19 - 2*x1^41*x2^28*x3^17*x4^9*z^19 - 4*x1^40*x2^29*x3^17*x4^9*z^19 + x1^39*x2^30*x3^17*x4^9*z^19 + 5*x1^37*x2^32*x3^17*x4^9*z^19 + 2*x1^35*x2^34*x3^17*x4^9*z^19 + 3*x1^42*x2^26*x3^18*x4^9*z^19 + 2*x1^41*x2^27*x3^18*x4^9*z^19 + 2*x1^40*x2^28*x3^18*x4^9*z^19 - x1^39*x2^29*x3^18*x4^9*z^19 - 2*x1^38*x2^30*x3^18*x4^9*z^19 - 2*x1^37*x2^31*x3^18*x4^9*z^19 - 4*x1^36*x2^32*x3^18*x4^9*z^19 - 2*x1^35*x2^33*x3^18*x4^9*z^19 - x1^34*x2^34*x3^18*x4^9*z^19 - 2*x1^42*x2^25*x3^19*x4^9*z^19 - 4*x1^41*x2^26*x3^19*x4^9*z^19 - x1^40*x2^27*x3^19*x4^9*z^19 - 5*x1^39*x2^28*x3^19*x4^9*z^19 + x1^38*x2^29*x3^19*x4^9*z^19 + 5*x1^36*x2^31*x3^19*x4^9*z^19 + 3*x1^35*x2^32*x3^19*x4^9*z^19 + 2*x1^34*x2^33*x3^19*x4^9*z^19 + 5*x1^41*x2^25*x3^20*x4^9*z^19 + 2*x1^40*x2^26*x3^20*x4^9*z^19 + 3*x1^39*x2^27*x3^20*x4^9*z^19 + 2*x1^38*x2^28*x3^20*x4^9*z^19 - x1^37*x2^29*x3^20*x4^9*z^19 - 4*x1^36*x2^30*x3^20*x4^9*z^19 - 4*x1^35*x2^31*x3^20*x4^9*z^19 - 2*x1^34*x2^32*x3^20*x4^9*z^19 - 5*x1^40*x2^25*x3^21*x4^9*z^19 - x1^39*x2^26*x3^21*x4^9*z^19 - 3*x1^38*x2^27*x3^21*x4^9*z^19 + 2*x1^37*x2^28*x3^21*x4^9*z^19 - x1^36*x2^29*x3^21*x4^9*z^19 + 3*x1^35*x2^30*x3^21*x4^9*z^19 + x1^34*x2^31*x3^21*x4^9*z^19 + 2*x1^33*x2^32*x3^21*x4^9*z^19 + x1^40*x2^24*x3^22*x4^9*z^19 + 3*x1^39*x2^25*x3^22*x4^9*z^19 + 2*x1^38*x2^26*x3^22*x4^9*z^19 + x1^37*x2^27*x3^22*x4^9*z^19 + x1^36*x2^28*x3^22*x4^9*z^19 - 3*x1^34*x2^30*x3^22*x4^9*z^19 - 2*x1^33*x2^31*x3^22*x4^9*z^19 - x1^32*x2^32*x3^22*x4^9*z^19 - x1^39*x2^24*x3^23*x4^9*z^19 - 2*x1^38*x2^25*x3^23*x4^9*z^19 - 3*x1^37*x2^26*x3^23*x4^9*z^19 + x1^36*x2^27*x3^23*x4^9*z^19 - x1^35*x2^28*x3^23*x4^9*z^19 + 4*x1^34*x2^29*x3^23*x4^9*z^19 + x1^33*x2^30*x3^23*x4^9*z^19 + 2*x1^32*x2^31*x3^23*x4^9*z^19 + 2*x1^37*x2^25*x3^24*x4^9*z^19 + x1^36*x2^26*x3^24*x4^9*z^19 - 2*x1^33*x2^29*x3^24*x4^9*z^19 - x1^36*x2^25*x3^25*x4^9*z^19 + x1^35*x2^26*x3^25*x4^9*z^19 - 2*x1^34*x2^27*x3^25*x4^9*z^19 + 4*x1^33*x2^28*x3^25*x4^9*z^19 - x1^31*x2^30*x3^25*x4^9*z^19 - x1^34*x2^26*x3^26*x4^9*z^19 + x1^33*x2^27*x3^26*x4^9*z^19 - 2*x1^32*x2^28*x3^26*x4^9*z^19 + x1^31*x2^29*x3^26*x4^9*z^19 + x1^30*x2^30*x3^26*x4^9*z^19 + x1^40*x2^33*x3^12*x4^10*z^19 + x1^41*x2^31*x3^13*x4^10*z^19 + 2*x1^42*x2^29*x3^14*x4^10*z^19 + x1^40*x2^31*x3^14*x4^10*z^19 + x1^39*x2^32*x3^14*x4^10*z^19 - x1^38*x2^33*x3^14*x4^10*z^19 - 2*x1^37*x2^34*x3^14*x4^10*z^19 - x1^36*x2^35*x3^14*x4^10*z^19 - x1^43*x2^27*x3^15*x4^10*z^19 + x1^42*x2^28*x3^15*x4^10*z^19 - x1^41*x2^29*x3^15*x4^10*z^19 + x1^39*x2^31*x3^15*x4^10*z^19 + 2*x1^36*x2^34*x3^15*x4^10*z^19 + x1^40*x2^29*x3^16*x4^10*z^19 + x1^39*x2^30*x3^16*x4^10*z^19 - 2*x1^38*x2^31*x3^16*x4^10*z^19 + x1^36*x2^33*x3^16*x4^10*z^19 - 2*x1^35*x2^34*x3^16*x4^10*z^19 - 2*x1^42*x2^26*x3^17*x4^10*z^19 - x1^38*x2^30*x3^17*x4^10*z^19 + x1^41*x2^26*x3^18*x4^10*z^19 + 2*x1^38*x2^29*x3^18*x4^10*z^19 - x1^41*x2^25*x3^19*x4^10*z^19 - 2*x1^40*x2^26*x3^19*x4^10*z^19 - x1^39*x2^27*x3^19*x4^10*z^19 + 2*x1^38*x2^28*x3^19*x4^10*z^19 - x1^37*x2^29*x3^19*x4^10*z^19 + x1^41*x2^24*x3^20*x4^10*z^19 + x1^40*x2^25*x3^20*x4^10*z^19 + x1^38*x2^27*x3^20*x4^10*z^19 - x1^37*x2^28*x3^20*x4^10*z^19 - x1^40*x2^24*x3^21*x4^10*z^19 + x1^39*x2^25*x3^21*x4^10*z^19 + x1^37*x2^27*x3^21*x4^10*z^19 + x1^36*x2^28*x3^21*x4^10*z^19 + 2*x1^35*x2^29*x3^21*x4^10*z^19 + x1^39*x2^24*x3^22*x4^10*z^19 + 2*x1^37*x2^26*x3^22*x4^10*z^19 - 2*x1^36*x2^27*x3^22*x4^10*z^19 - 2*x1^34*x2^29*x3^22*x4^10*z^19 + x1^33*x2^30*x3^22*x4^10*z^19 - x1^38*x2^24*x3^23*x4^10*z^19 + x1^35*x2^27*x3^23*x4^10*z^19 + 2*x1^33*x2^29*x3^23*x4^10*z^19 - x1^36*x2^25*x3^24*x4^10*z^19 - 2*x1^35*x2^26*x3^24*x4^10*z^19 - 3*x1^33*x2^28*x3^24*x4^10*z^19 - 2*x1^32*x2^29*x3^24*x4^10*z^19 + x1^34*x2^26*x3^25*x4^10*z^19 + x1^32*x2^28*x3^25*x4^10*z^19 + x1^31*x2^29*x3^25*x4^10*z^19 - x1^30*x2^30*x3^25*x4^10*z^19 - x1^32*x2^27*x3^26*x4^10*z^19 - x1^30*x2^29*x3^26*x4^10*z^19 + x1^31*x2^27*x3^27*x4^10*z^19 - x1^41*x2^32*x3^11*x4^11*z^19 + x1^39*x2^33*x3^12*x4^11*z^19 - 2*x1^40*x2^31*x3^13*x4^11*z^19 - x1^39*x2^32*x3^13*x4^11*z^19 - x1^38*x2^33*x3^13*x4^11*z^19 - x1^43*x2^27*x3^14*x4^11*z^19 - x1^42*x2^28*x3^14*x4^11*z^19 + x1^41*x2^29*x3^14*x4^11*z^19 + x1^40*x2^30*x3^14*x4^11*z^19 - x1^39*x2^31*x3^14*x4^11*z^19 - x1^38*x2^32*x3^14*x4^11*z^19 + x1^37*x2^33*x3^14*x4^11*z^19 - x1^42*x2^27*x3^15*x4^11*z^19 - 2*x1^41*x2^28*x3^15*x4^11*z^19 + x1^40*x2^29*x3^15*x4^11*z^19 - 2*x1^39*x2^30*x3^15*x4^11*z^19 - 2*x1^37*x2^32*x3^15*x4^11*z^19 + x1^36*x2^33*x3^15*x4^11*z^19 + x1^35*x2^34*x3^15*x4^11*z^19 - 2*x1^41*x2^27*x3^16*x4^11*z^19 + x1^40*x2^28*x3^16*x4^11*z^19 + 2*x1^39*x2^29*x3^16*x4^11*z^19 + x1^38*x2^30*x3^16*x4^11*z^19 - x1^37*x2^31*x3^16*x4^11*z^19 + 2*x1^36*x2^32*x3^16*x4^11*z^19 - 2*x1^35*x2^33*x3^16*x4^11*z^19 + x1^34*x2^34*x3^16*x4^11*z^19 + x1^42*x2^25*x3^17*x4^11*z^19 + 2*x1^41*x2^26*x3^17*x4^11*z^19 - 2*x1^40*x2^27*x3^17*x4^11*z^19 + x1^39*x2^28*x3^17*x4^11*z^19 - 3*x1^38*x2^29*x3^17*x4^11*z^19 + 2*x1^37*x2^30*x3^17*x4^11*z^19 - 2*x1^36*x2^31*x3^17*x4^11*z^19 - 2*x1^35*x2^32*x3^17*x4^11*z^19 + 3*x1^34*x2^33*x3^17*x4^11*z^19 - 2*x1^40*x2^26*x3^18*x4^11*z^19 + x1^38*x2^28*x3^18*x4^11*z^19 + 5*x1^37*x2^29*x3^18*x4^11*z^19 + x1^36*x2^30*x3^18*x4^11*z^19 + 4*x1^35*x2^31*x3^18*x4^11*z^19 - 2*x1^34*x2^32*x3^18*x4^11*z^19 - x1^33*x2^33*x3^18*x4^11*z^19 - x1^38*x2^27*x3^19*x4^11*z^19 - 4*x1^37*x2^28*x3^19*x4^11*z^19 + x1^36*x2^29*x3^19*x4^11*z^19 - 3*x1^35*x2^30*x3^19*x4^11*z^19 + 2*x1^33*x2^32*x3^19*x4^11*z^19 + x1^38*x2^26*x3^20*x4^11*z^19 + x1^36*x2^28*x3^20*x4^11*z^19 - x1^35*x2^29*x3^20*x4^11*z^19 + 4*x1^34*x2^30*x3^20*x4^11*z^19 - 2*x1^33*x2^31*x3^20*x4^11*z^19 - x1^32*x2^32*x3^20*x4^11*z^19 + x1^37*x2^26*x3^21*x4^11*z^19 - x1^36*x2^27*x3^21*x4^11*z^19 + x1^35*x2^28*x3^21*x4^11*z^19 - 2*x1^34*x2^29*x3^21*x4^11*z^19 + 2*x1^32*x2^31*x3^21*x4^11*z^19 - 2*x1^34*x2^28*x3^22*x4^11*z^19 + 3*x1^33*x2^29*x3^22*x4^11*z^19 - 3*x1^32*x2^30*x3^22*x4^11*z^19 + x1^34*x2^27*x3^23*x4^11*z^19 - 2*x1^32*x2^29*x3^23*x4^11*z^19 + 3*x1^31*x2^30*x3^23*x4^11*z^19 + x1^32*x2^28*x3^24*x4^11*z^19 - x1^31*x2^29*x3^24*x4^11*z^19 - x1^30*x2^30*x3^24*x4^11*z^19 + x1^30*x2^29*x3^25*x4^11*z^19 + 2*x1^40*x2^31*x3^12*x4^12*z^19 - x1^38*x2^32*x3^13*x4^12*z^19 - x1^37*x2^33*x3^13*x4^12*z^19 + 2*x1^39*x2^30*x3^14*x4^12*z^19 - x1^38*x2^31*x3^14*x4^12*z^19 - x1^35*x2^34*x3^14*x4^12*z^19 + x1^42*x2^26*x3^15*x4^12*z^19 + x1^41*x2^27*x3^15*x4^12*z^19 - x1^40*x2^28*x3^15*x4^12*z^19 + x1^38*x2^30*x3^15*x4^12*z^19 + 3*x1^37*x2^31*x3^15*x4^12*z^19 - x1^36*x2^32*x3^15*x4^12*z^19 + x1^35*x2^33*x3^15*x4^12*z^19 + x1^34*x2^34*x3^15*x4^12*z^19 + x1^40*x2^27*x3^16*x4^12*z^19 - x1^39*x2^28*x3^16*x4^12*z^19 + 2*x1^38*x2^29*x3^16*x4^12*z^19 - 3*x1^37*x2^30*x3^16*x4^12*z^19 - 5*x1^34*x2^33*x3^16*x4^12*z^19 + x1^40*x2^26*x3^17*x4^12*z^19 - 3*x1^39*x2^27*x3^17*x4^12*z^19 - x1^37*x2^29*x3^17*x4^12*z^19 + 3*x1^36*x2^30*x3^17*x4^12*z^19 - 2*x1^35*x2^31*x3^17*x4^12*z^19 + 4*x1^34*x2^32*x3^17*x4^12*z^19 + 2*x1^33*x2^33*x3^17*x4^12*z^19 + x1^39*x2^26*x3^18*x4^12*z^19 + x1^38*x2^27*x3^18*x4^12*z^19 + 2*x1^37*x2^28*x3^18*x4^12*z^19 - 2*x1^36*x2^29*x3^18*x4^12*z^19 - 2*x1^35*x2^30*x3^18*x4^12*z^19 - x1^34*x2^31*x3^18*x4^12*z^19 - 6*x1^33*x2^32*x3^18*x4^12*z^19 + x1^40*x2^24*x3^19*x4^12*z^19 - 3*x1^38*x2^26*x3^19*x4^12*z^19 - x1^36*x2^28*x3^19*x4^12*z^19 + 3*x1^35*x2^29*x3^19*x4^12*z^19 - 2*x1^34*x2^30*x3^19*x4^12*z^19 + 6*x1^33*x2^31*x3^19*x4^12*z^19 + 2*x1^32*x2^32*x3^19*x4^12*z^19 + 2*x1^38*x2^25*x3^20*x4^12*z^19 + x1^37*x2^26*x3^20*x4^12*z^19 + 2*x1^36*x2^27*x3^20*x4^12*z^19 - 2*x1^35*x2^28*x3^20*x4^12*z^19 + x1^34*x2^29*x3^20*x4^12*z^19 - 2*x1^33*x2^30*x3^20*x4^12*z^19 - 6*x1^32*x2^31*x3^20*x4^12*z^19 - 3*x1^37*x2^25*x3^21*x4^12*z^19 - 2*x1^36*x2^26*x3^21*x4^12*z^19 - 2*x1^35*x2^27*x3^21*x4^12*z^19 + 5*x1^34*x2^28*x3^21*x4^12*z^19 - 3*x1^33*x2^29*x3^21*x4^12*z^19 + 6*x1^32*x2^30*x3^21*x4^12*z^19 + 2*x1^31*x2^31*x3^21*x4^12*z^19 + x1^35*x2^26*x3^22*x4^12*z^19 - x1^33*x2^28*x3^22*x4^12*z^19 - 6*x1^31*x2^30*x3^22*x4^12*z^19 - 2*x1^34*x2^26*x3^23*x4^12*z^19 - x1^32*x2^28*x3^23*x4^12*z^19 + 4*x1^31*x2^29*x3^23*x4^12*z^19 + 2*x1^30*x2^30*x3^23*x4^12*z^19 - 2*x1^32*x2^27*x3^24*x4^12*z^19 - x1^31*x2^28*x3^24*x4^12*z^19 - 4*x1^30*x2^29*x3^24*x4^12*z^19 + x1^29*x2^29*x3^25*x4^12*z^19 + x1^38*x2^30*x3^14*x4^13*z^19 + x1^37*x2^31*x3^14*x4^13*z^19 + 2*x1^36*x2^32*x3^14*x4^13*z^19 - 2*x1^38*x2^29*x3^15*x4^13*z^19 + 2*x1^37*x2^30*x3^15*x4^13*z^19 + x1^36*x2^31*x3^15*x4^13*z^19 - x1^35*x2^32*x3^15*x4^13*z^19 + x1^34*x2^33*x3^15*x4^13*z^19 + x1^37*x2^29*x3^16*x4^13*z^19 - 2*x1^36*x2^30*x3^16*x4^13*z^19 + 2*x1^35*x2^31*x3^16*x4^13*z^19 - x1^34*x2^32*x3^16*x4^13*z^19 - 2*x1^33*x2^33*x3^16*x4^13*z^19 + x1^40*x2^25*x3^17*x4^13*z^19 + x1^39*x2^26*x3^17*x4^13*z^19 - x1^37*x2^28*x3^17*x4^13*z^19 + 3*x1^36*x2^29*x3^17*x4^13*z^19 + x1^35*x2^30*x3^17*x4^13*z^19 + 4*x1^33*x2^32*x3^17*x4^13*z^19 + 2*x1^38*x2^26*x3^18*x4^13*z^19 - 3*x1^35*x2^29*x3^18*x4^13*z^19 + x1^34*x2^30*x3^18*x4^13*z^19 - 3*x1^33*x2^31*x3^18*x4^13*z^19 - 2*x1^32*x2^32*x3^18*x4^13*z^19 + x1^39*x2^24*x3^19*x4^13*z^19 - x1^37*x2^26*x3^19*x4^13*z^19 - x1^36*x2^27*x3^19*x4^13*z^19 + x1^35*x2^28*x3^19*x4^13*z^19 + x1^34*x2^29*x3^19*x4^13*z^19 + 2*x1^33*x2^30*x3^19*x4^13*z^19 + 5*x1^32*x2^31*x3^19*x4^13*z^19 + 2*x1^37*x2^25*x3^20*x4^13*z^19 + x1^36*x2^26*x3^20*x4^13*z^19 - x1^35*x2^27*x3^20*x4^13*z^19 - 4*x1^34*x2^28*x3^20*x4^13*z^19 - 6*x1^32*x2^30*x3^20*x4^13*z^19 - 2*x1^31*x2^31*x3^20*x4^13*z^19 + x1^36*x2^25*x3^21*x4^13*z^19 + x1^34*x2^27*x3^21*x4^13*z^19 + x1^32*x2^29*x3^21*x4^13*z^19 + 6*x1^31*x2^30*x3^21*x4^13*z^19 + 2*x1^32*x2^28*x3^22*x4^13*z^19 - 4*x1^31*x2^29*x3^22*x4^13*z^19 - 2*x1^30*x2^30*x3^22*x4^13*z^19 + x1^33*x2^26*x3^23*x4^13*z^19 + 4*x1^30*x2^29*x3^23*x4^13*z^19 + x1^31*x2^27*x3^24*x4^13*z^19 - x1^30*x2^28*x3^24*x4^13*z^19 - x1^29*x2^29*x3^24*x4^13*z^19 + x1^29*x2^28*x3^25*x4^13*z^19 - x1^38*x2^28*x3^15*x4^14*z^19 - x1^37*x2^29*x3^15*x4^14*z^19 + x1^36*x2^30*x3^15*x4^14*z^19 - x1^35*x2^31*x3^15*x4^14*z^19 + x1^37*x2^28*x3^16*x4^14*z^19 - x1^36*x2^29*x3^16*x4^14*z^19 + x1^35*x2^30*x3^16*x4^14*z^19 + x1^34*x2^31*x3^16*x4^14*z^19 - x1^37*x2^27*x3^17*x4^14*z^19 - 2*x1^36*x2^28*x3^17*x4^14*z^19 - 2*x1^34*x2^30*x3^17*x4^14*z^19 - x1^39*x2^24*x3^18*x4^14*z^19 + x1^37*x2^26*x3^18*x4^14*z^19 + x1^36*x2^27*x3^18*x4^14*z^19 - 2*x1^35*x2^28*x3^18*x4^14*z^19 + 2*x1^34*x2^29*x3^18*x4^14*z^19 + 2*x1^33*x2^30*x3^18*x4^14*z^19 - x1^32*x2^31*x3^18*x4^14*z^19 + x1^36*x2^26*x3^19*x4^14*z^19 - x1^35*x2^27*x3^19*x4^14*z^19 - x1^34*x2^28*x3^19*x4^14*z^19 - 2*x1^33*x2^29*x3^19*x4^14*z^19 + x1^32*x2^30*x3^19*x4^14*z^19 + x1^31*x2^31*x3^19*x4^14*z^19 + x1^37*x2^24*x3^20*x4^14*z^19 + x1^36*x2^25*x3^20*x4^14*z^19 + x1^35*x2^26*x3^20*x4^14*z^19 - x1^34*x2^27*x3^20*x4^14*z^19 + 3*x1^33*x2^28*x3^20*x4^14*z^19 - x1^31*x2^30*x3^20*x4^14*z^19 - x1^35*x2^25*x3^21*x4^14*z^19 - 3*x1^32*x2^28*x3^21*x4^14*z^19 + x1^31*x2^29*x3^21*x4^14*z^19 + x1^30*x2^30*x3^21*x4^14*z^19 + x1^34*x2^25*x3^22*x4^14*z^19 + x1^32*x2^27*x3^22*x4^14*z^19 + x1^31*x2^28*x3^22*x4^14*z^19 - x1^30*x2^29*x3^22*x4^14*z^19 - x1^31*x2^27*x3^23*x4^14*z^19 + x1^30*x2^27*x3^24*x4^14*z^19 + x1^37*x2^27*x3^16*x4^15*z^19 + x1^36*x2^28*x3^16*x4^15*z^19 - x1^35*x2^29*x3^16*x4^15*z^19 + x1^34*x2^30*x3^16*x4^15*z^19 - 2*x1^36*x2^27*x3^17*x4^15*z^19 - 2*x1^34*x2^29*x3^17*x4^15*z^19 - x1^33*x2^30*x3^17*x4^15*z^19 + x1^36*x2^26*x3^18*x4^15*z^19 + 2*x1^35*x2^27*x3^18*x4^15*z^19 + 4*x1^33*x2^29*x3^18*x4^15*z^19 - 3*x1^35*x2^26*x3^19*x4^15*z^19 - 4*x1^33*x2^28*x3^19*x4^15*z^19 - 2*x1^32*x2^29*x3^19*x4^15*z^19 - x1^36*x2^24*x3^20*x4^15*z^19 + 2*x1^34*x2^26*x3^20*x4^15*z^19 + x1^33*x2^27*x3^20*x4^15*z^19 + 5*x1^32*x2^28*x3^20*x4^15*z^19 + x1^35*x2^24*x3^21*x4^15*z^19 + 2*x1^33*x2^26*x3^21*x4^15*z^19 - 5*x1^32*x2^27*x3^21*x4^15*z^19 - 2*x1^31*x2^28*x3^21*x4^15*z^19 + x1^33*x2^25*x3^22*x4^15*z^19 + 4*x1^31*x2^27*x3^22*x4^15*z^19 - 2*x1^30*x2^27*x3^23*x4^15*z^19 + x1^35*x2^26*x3^18*x4^16*z^19 + x1^34*x2^27*x3^18*x4^16*z^19 + x1^33*x2^28*x3^18*x4^16*z^19 + x1^33*x2^27*x3^19*x4^16*z^19 - 3*x1^32*x2^28*x3^19*x4^16*z^19 + x1^34*x2^25*x3^20*x4^16*z^19 + x1^32*x2^27*x3^20*x4^16*z^19 + 2*x1^31*x2^28*x3^20*x4^16*z^19 + x1^32*x2^26*x3^21*x4^16*z^19 - 2*x1^31*x2^27*x3^21*x4^16*z^19 - x1^32*x2^25*x3^22*x4^16*z^19 + x1^30*x2^27*x3^22*x4^16*z^19 - x1^30*x2^27*x3^21*x4^17*z^19 + x1^45*x2^31*x3^14*z^18 - 2*x1^44*x2^31*x3^15*z^18 + x1^43*x2^32*x3^15*z^18 + 2*x1^43*x2^31*x3^16*z^18 - x1^42*x2^32*x3^16*z^18 - x1^40*x2^34*x3^16*z^18 - 2*x1^43*x2^30*x3^17*z^18 - x1^42*x2^31*x3^17*z^18 + x1^39*x2^34*x3^17*z^18 + x1^43*x2^29*x3^18*z^18 + 2*x1^40*x2^32*x3^18*z^18 - x1^42*x2^29*x3^19*z^18 - x1^41*x2^30*x3^19*z^18 - x1^40*x2^31*x3^19*z^18 + x1^42*x2^28*x3^20*z^18 + x1^41*x2^29*x3^20*z^18 - x1^40*x2^30*x3^20*z^18 + x1^39*x2^31*x3^20*z^18 - x1^46*x2^32*x3^11*x4*z^18 + 2*x1^45*x2^32*x3^12*x4*z^18 - x1^44*x2^33*x3^12*x4*z^18 - 2*x1^45*x2^31*x3^13*x4*z^18 - 2*x1^44*x2^32*x3^13*x4*z^18 - x1^42*x2^34*x3^13*x4*z^18 + 5*x1^44*x2^31*x3^14*x4*z^18 + x1^42*x2^33*x3^14*x4*z^18 - x1^40*x2^35*x3^14*x4*z^18 - 2*x1^44*x2^30*x3^15*x4*z^18 - 5*x1^43*x2^31*x3^15*x4*z^18 - 3*x1^41*x2^33*x3^15*x4*z^18 + x1^39*x2^35*x3^15*x4*z^18 + 6*x1^43*x2^30*x3^16*x4*z^18 + 2*x1^42*x2^31*x3^16*x4*z^18 + 2*x1^41*x2^32*x3^16*x4*z^18 + 2*x1^40*x2^33*x3^16*x4*z^18 - x1^39*x2^34*x3^16*x4*z^18 - x1^38*x2^35*x3^16*x4*z^18 - 2*x1^43*x2^29*x3^17*x4*z^18 - 6*x1^42*x2^30*x3^17*x4*z^18 - 3*x1^40*x2^32*x3^17*x4*z^18 + x1^39*x2^33*x3^17*x4*z^18 + x1^37*x2^35*x3^17*x4*z^18 + 6*x1^42*x2^29*x3^18*x4*z^18 + 2*x1^41*x2^30*x3^18*x4*z^18 + 2*x1^40*x2^31*x3^18*x4*z^18 + 2*x1^39*x2^32*x3^18*x4*z^18 + x1^38*x2^33*x3^18*x4*z^18 - 2*x1^37*x2^34*x3^18*x4*z^18 - x1^36*x2^35*x3^18*x4*z^18 - 2*x1^42*x2^28*x3^19*x4*z^18 - 4*x1^41*x2^29*x3^19*x4*z^18 - 5*x1^39*x2^31*x3^19*x4*z^18 + x1^38*x2^32*x3^19*x4*z^18 + 2*x1^36*x2^34*x3^19*x4*z^18 + 5*x1^41*x2^28*x3^20*x4*z^18 + x1^40*x2^29*x3^20*x4*z^18 + x1^39*x2^30*x3^20*x4*z^18 + 2*x1^38*x2^31*x3^20*x4*z^18 + x1^37*x2^32*x3^20*x4*z^18 - x1^36*x2^33*x3^20*x4*z^18 - x1^35*x2^34*x3^20*x4*z^18 - x1^41*x2^27*x3^21*x4*z^18 - 2*x1^40*x2^28*x3^21*x4*z^18 - 4*x1^38*x2^30*x3^21*x4*z^18 + 3*x1^40*x2^27*x3^22*x4*z^18 + x1^39*x2^28*x3^22*x4*z^18 + x1^38*x2^29*x3^22*x4*z^18 + x1^37*x2^30*x3^22*x4*z^18 - 2*x1^37*x2^29*x3^23*x4*z^18 - x1^45*x2^32*x3^11*x4^2*z^18 + x1^45*x2^31*x3^12*x4^2*z^18 + x1^44*x2^32*x3^12*x4^2*z^18 - x1^43*x2^33*x3^12*x4^2*z^18 - 4*x1^44*x2^31*x3^13*x4^2*z^18 + x1^43*x2^32*x3^13*x4^2*z^18 + 2*x1^44*x2^30*x3^14*x4^2*z^18 + 4*x1^43*x2^31*x3^14*x4^2*z^18 - x1^42*x2^32*x3^14*x4^2*z^18 + 3*x1^41*x2^33*x3^14*x4^2*z^18 + x1^40*x2^34*x3^14*x4^2*z^18 - 6*x1^43*x2^30*x3^15*x4^2*z^18 - x1^42*x2^31*x3^15*x4^2*z^18 - x1^41*x2^32*x3^15*x4^2*z^18 + x1^39*x2^34*x3^15*x4^2*z^18 + x1^38*x2^35*x3^15*x4^2*z^18 + 2*x1^43*x2^29*x3^16*x4^2*z^18 + 6*x1^42*x2^30*x3^16*x4^2*z^18 + 3*x1^40*x2^32*x3^16*x4^2*z^18 - 3*x1^39*x2^33*x3^16*x4^2*z^18 - x1^38*x2^34*x3^16*x4^2*z^18 - 2*x1^37*x2^35*x3^16*x4^2*z^18 - 6*x1^42*x2^29*x3^17*x4^2*z^18 - 2*x1^41*x2^30*x3^17*x4^2*z^18 - 2*x1^40*x2^31*x3^17*x4^2*z^18 + 3*x1^38*x2^33*x3^17*x4^2*z^18 + 2*x1^37*x2^34*x3^17*x4^2*z^18 + 2*x1^36*x2^35*x3^17*x4^2*z^18 + 2*x1^42*x2^28*x3^18*x4^2*z^18 + 6*x1^41*x2^29*x3^18*x4^2*z^18 + 4*x1^39*x2^31*x3^18*x4^2*z^18 - 3*x1^38*x2^32*x3^18*x4^2*z^18 - 4*x1^36*x2^34*x3^18*x4^2*z^18 - x1^35*x2^35*x3^18*x4^2*z^18 - 6*x1^41*x2^28*x3^19*x4^2*z^18 - 2*x1^40*x2^29*x3^19*x4^2*z^18 - 2*x1^39*x2^30*x3^19*x4^2*z^18 - 2*x1^38*x2^31*x3^19*x4^2*z^18 + x1^37*x2^32*x3^19*x4^2*z^18 + 3*x1^36*x2^33*x3^19*x4^2*z^18 + 3*x1^35*x2^34*x3^19*x4^2*z^18 + 2*x1^41*x2^27*x3^20*x4^2*z^18 + 6*x1^40*x2^28*x3^20*x4^2*z^18 + 4*x1^38*x2^30*x3^20*x4^2*z^18 - 4*x1^35*x2^33*x3^20*x4^2*z^18 - 5*x1^40*x2^27*x3^21*x4^2*z^18 - x1^39*x2^28*x3^21*x4^2*z^18 - x1^38*x2^29*x3^21*x4^2*z^18 - 3*x1^37*x2^30*x3^21*x4^2*z^18 + 2*x1^35*x2^32*x3^21*x4^2*z^18 + x1^34*x2^33*x3^21*x4^2*z^18 + 3*x1^39*x2^27*x3^22*x4^2*z^18 + 2*x1^38*x2^28*x3^22*x4^2*z^18 + 4*x1^37*x2^29*x3^22*x4^2*z^18 - x1^36*x2^30*x3^22*x4^2*z^18 + x1^35*x2^31*x3^22*x4^2*z^18 - x1^34*x2^32*x3^22*x4^2*z^18 - x1^39*x2^26*x3^23*x4^2*z^18 - x1^38*x2^27*x3^23*x4^2*z^18 - 2*x1^37*x2^28*x3^23*x4^2*z^18 - 2*x1^36*x2^29*x3^23*x4^2*z^18 + 2*x1^36*x2^28*x3^24*x4^2*z^18 - x1^35*x2^28*x3^25*x4^2*z^18 - x1^46*x2^29*x3^12*x4^3*z^18 + 2*x1^44*x2^31*x3^12*x4^3*z^18 - x1^43*x2^32*x3^12*x4^3*z^18 + 2*x1^45*x2^29*x3^13*x4^3*z^18 - x1^43*x2^31*x3^13*x4^3*z^18 + x1^42*x2^32*x3^13*x4^3*z^18 + x1^40*x2^34*x3^13*x4^3*z^18 - x1^45*x2^28*x3^14*x4^3*z^18 - 2*x1^44*x2^29*x3^14*x4^3*z^18 + 2*x1^43*x2^30*x3^14*x4^3*z^18 + x1^42*x2^31*x3^14*x4^3*z^18 + x1^41*x2^32*x3^14*x4^3*z^18 - x1^40*x2^33*x3^14*x4^3*z^18 - 2*x1^39*x2^34*x3^14*x4^3*z^18 + 2*x1^44*x2^28*x3^15*x4^3*z^18 - x1^43*x2^29*x3^15*x4^3*z^18 - 2*x1^42*x2^30*x3^15*x4^3*z^18 - x1^40*x2^32*x3^15*x4^3*z^18 + 2*x1^39*x2^33*x3^15*x4^3*z^18 + 2*x1^38*x2^34*x3^15*x4^3*z^18 + x1^37*x2^35*x3^15*x4^3*z^18 - 2*x1^43*x2^28*x3^16*x4^3*z^18 + 2*x1^42*x2^29*x3^16*x4^3*z^18 - 2*x1^41*x2^30*x3^16*x4^3*z^18 + x1^40*x2^31*x3^16*x4^3*z^18 + x1^39*x2^32*x3^16*x4^3*z^18 - 2*x1^37*x2^34*x3^16*x4^3*z^18 - x1^36*x2^35*x3^16*x4^3*z^18 + 2*x1^43*x2^27*x3^17*x4^3*z^18 + x1^42*x2^28*x3^17*x4^3*z^18 - x1^41*x2^29*x3^17*x4^3*z^18 - 3*x1^39*x2^31*x3^17*x4^3*z^18 - x1^37*x2^33*x3^17*x4^3*z^18 + 3*x1^36*x2^34*x3^17*x4^3*z^18 - x1^43*x2^26*x3^18*x4^3*z^18 - 2*x1^42*x2^27*x3^18*x4^3*z^18 + 2*x1^41*x2^28*x3^18*x4^3*z^18 - x1^38*x2^31*x3^18*x4^3*z^18 - x1^36*x2^33*x3^18*x4^3*z^18 - 3*x1^35*x2^34*x3^18*x4^3*z^18 + 2*x1^42*x2^26*x3^19*x4^3*z^18 - x1^40*x2^28*x3^19*x4^3*z^18 + x1^39*x2^29*x3^19*x4^3*z^18 - 2*x1^38*x2^30*x3^19*x4^3*z^18 - x1^37*x2^31*x3^19*x4^3*z^18 - x1^36*x2^32*x3^19*x4^3*z^18 + 3*x1^35*x2^33*x3^19*x4^3*z^18 + x1^34*x2^34*x3^19*x4^3*z^18 - x1^42*x2^25*x3^20*x4^3*z^18 - 2*x1^41*x2^26*x3^20*x4^3*z^18 + 2*x1^40*x2^27*x3^20*x4^3*z^18 + 2*x1^38*x2^29*x3^20*x4^3*z^18 - x1^36*x2^31*x3^20*x4^3*z^18 - 2*x1^35*x2^32*x3^20*x4^3*z^18 - x1^34*x2^33*x3^20*x4^3*z^18 + x1^41*x2^25*x3^21*x4^3*z^18 - x1^40*x2^26*x3^21*x4^3*z^18 - 3*x1^39*x2^27*x3^21*x4^3*z^18 + x1^38*x2^28*x3^21*x4^3*z^18 - x1^35*x2^31*x3^21*x4^3*z^18 + 3*x1^34*x2^32*x3^21*x4^3*z^18 - x1^40*x2^25*x3^22*x4^3*z^18 - x1^39*x2^26*x3^22*x4^3*z^18 - 3*x1^38*x2^27*x3^22*x4^3*z^18 + 2*x1^36*x2^29*x3^22*x4^3*z^18 - x1^35*x2^30*x3^22*x4^3*z^18 - x1^34*x2^31*x3^22*x4^3*z^18 - 2*x1^37*x2^27*x3^23*x4^3*z^18 - x1^36*x2^28*x3^23*x4^3*z^18 + x1^35*x2^29*x3^23*x4^3*z^18 - x1^34*x2^30*x3^23*x4^3*z^18 + x1^33*x2^31*x3^23*x4^3*z^18 + x1^36*x2^27*x3^24*x4^3*z^18 + 2*x1^35*x2^28*x3^24*x4^3*z^18 - x1^46*x2^30*x3^10*x4^4*z^18 + x1^45*x2^31*x3^10*x4^4*z^18 + x1^46*x2^29*x3^11*x4^4*z^18 + x1^45*x2^30*x3^11*x4^4*z^18 - 4*x1^45*x2^29*x3^12*x4^4*z^18 + x1^43*x2^31*x3^12*x4^4*z^18 + 2*x1^42*x2^32*x3^12*x4^4*z^18 + x1^41*x2^33*x3^12*x4^4*z^18 + 2*x1^45*x2^28*x3^13*x4^4*z^18 + 4*x1^44*x2^29*x3^13*x4^4*z^18 - x1^43*x2^30*x3^13*x4^4*z^18 + 2*x1^42*x2^31*x3^13*x4^4*z^18 - x1^41*x2^32*x3^13*x4^4*z^18 - 2*x1^40*x2^33*x3^13*x4^4*z^18 - x1^39*x2^34*x3^13*x4^4*z^18 - 6*x1^44*x2^28*x3^14*x4^4*z^18 - 2*x1^43*x2^29*x3^14*x4^4*z^18 - x1^42*x2^30*x3^14*x4^4*z^18 + 2*x1^40*x2^32*x3^14*x4^4*z^18 + 2*x1^39*x2^33*x3^14*x4^4*z^18 + x1^38*x2^34*x3^14*x4^4*z^18 + 2*x1^44*x2^27*x3^15*x4^4*z^18 + 6*x1^43*x2^28*x3^15*x4^4*z^18 + 2*x1^41*x2^30*x3^15*x4^4*z^18 - 3*x1^40*x2^31*x3^15*x4^4*z^18 - 2*x1^39*x2^32*x3^15*x4^4*z^18 - 3*x1^38*x2^33*x3^15*x4^4*z^18 - x1^37*x2^34*x3^15*x4^4*z^18 - 6*x1^43*x2^27*x3^16*x4^4*z^18 - 2*x1^42*x2^28*x3^16*x4^4*z^18 - 2*x1^41*x2^29*x3^16*x4^4*z^18 + 2*x1^38*x2^32*x3^16*x4^4*z^18 + 3*x1^37*x2^33*x3^16*x4^4*z^18 + 2*x1^43*x2^26*x3^17*x4^4*z^18 + 6*x1^42*x2^27*x3^17*x4^4*z^18 + 4*x1^40*x2^29*x3^17*x4^4*z^18 - 4*x1^39*x2^30*x3^17*x4^4*z^18 - 6*x1^37*x2^32*x3^17*x4^4*z^18 - x1^36*x2^33*x3^17*x4^4*z^18 - 6*x1^42*x2^26*x3^18*x4^4*z^18 - 2*x1^41*x2^27*x3^18*x4^4*z^18 - 2*x1^40*x2^28*x3^18*x4^4*z^18 + 2*x1^38*x2^30*x3^18*x4^4*z^18 + 2*x1^37*x2^31*x3^18*x4^4*z^18 + 6*x1^36*x2^32*x3^18*x4^4*z^18 + x1^34*x2^34*x3^18*x4^4*z^18 + 2*x1^42*x2^25*x3^19*x4^4*z^18 + 6*x1^41*x2^26*x3^19*x4^4*z^18 + 4*x1^39*x2^28*x3^19*x4^4*z^18 - 2*x1^38*x2^29*x3^19*x4^4*z^18 + x1^37*x2^30*x3^19*x4^4*z^18 - 5*x1^36*x2^31*x3^19*x4^4*z^18 - 2*x1^35*x2^32*x3^19*x4^4*z^18 + x1^34*x2^33*x3^19*x4^4*z^18 - 5*x1^41*x2^25*x3^20*x4^4*z^18 - 2*x1^40*x2^26*x3^20*x4^4*z^18 - 2*x1^39*x2^27*x3^20*x4^4*z^18 - x1^38*x2^28*x3^20*x4^4*z^18 + x1^37*x2^29*x3^20*x4^4*z^18 + x1^36*x2^30*x3^20*x4^4*z^18 + 3*x1^35*x2^31*x3^20*x4^4*z^18 - 2*x1^34*x2^32*x3^20*x4^4*z^18 + 5*x1^40*x2^25*x3^21*x4^4*z^18 + x1^39*x2^26*x3^21*x4^4*z^18 + 4*x1^38*x2^27*x3^21*x4^4*z^18 + x1^36*x2^29*x3^21*x4^4*z^18 - 3*x1^35*x2^30*x3^21*x4^4*z^18 - x1^34*x2^31*x3^21*x4^4*z^18 + x1^33*x2^32*x3^21*x4^4*z^18 - x1^40*x2^24*x3^22*x4^4*z^18 - 3*x1^39*x2^25*x3^22*x4^4*z^18 - x1^38*x2^26*x3^22*x4^4*z^18 - x1^36*x2^28*x3^22*x4^4*z^18 + 2*x1^35*x2^29*x3^22*x4^4*z^18 + x1^34*x2^30*x3^22*x4^4*z^18 - 2*x1^33*x2^31*x3^22*x4^4*z^18 + x1^39*x2^24*x3^23*x4^4*z^18 + 2*x1^38*x2^25*x3^23*x4^4*z^18 + x1^37*x2^26*x3^23*x4^4*z^18 - x1^36*x2^27*x3^23*x4^4*z^18 - x1^35*x2^28*x3^23*x4^4*z^18 - x1^37*x2^25*x3^24*x4^4*z^18 - x1^33*x2^29*x3^24*x4^4*z^18 - 2*x1^32*x2^30*x3^24*x4^4*z^18 - x1^35*x2^26*x3^25*x4^4*z^18 - x1^34*x2^27*x3^25*x4^4*z^18 - x1^45*x2^30*x3^10*x4^5*z^18 + 2*x1^45*x2^29*x3^11*x4^5*z^18 - x1^44*x2^30*x3^11*x4^5*z^18 - 2*x1^43*x2^31*x3^11*x4^5*z^18 - x1^45*x2^28*x3^12*x4^5*z^18 - 4*x1^44*x2^29*x3^12*x4^5*z^18 + x1^43*x2^30*x3^12*x4^5*z^18 + 4*x1^44*x2^28*x3^13*x4^5*z^18 + 2*x1^43*x2^29*x3^13*x4^5*z^18 + x1^41*x2^31*x3^13*x4^5*z^18 - 2*x1^44*x2^27*x3^14*x4^5*z^18 - 6*x1^43*x2^28*x3^14*x4^5*z^18 - 3*x1^41*x2^30*x3^14*x4^5*z^18 + 3*x1^40*x2^31*x3^14*x4^5*z^18 + 2*x1^39*x2^32*x3^14*x4^5*z^18 + 2*x1^38*x2^33*x3^14*x4^5*z^18 + x1^37*x2^34*x3^14*x4^5*z^18 + 6*x1^43*x2^27*x3^15*x4^5*z^18 + x1^42*x2^28*x3^15*x4^5*z^18 - 4*x1^39*x2^31*x3^15*x4^5*z^18 - 3*x1^38*x2^32*x3^15*x4^5*z^18 - 2*x1^37*x2^33*x3^15*x4^5*z^18 - 2*x1^43*x2^26*x3^16*x4^5*z^18 - 6*x1^42*x2^27*x3^16*x4^5*z^18 + 2*x1^41*x2^28*x3^16*x4^5*z^18 - 3*x1^40*x2^29*x3^16*x4^5*z^18 + 5*x1^39*x2^30*x3^16*x4^5*z^18 + x1^38*x2^31*x3^16*x4^5*z^18 + 4*x1^37*x2^32*x3^16*x4^5*z^18 - x1^36*x2^33*x3^16*x4^5*z^18 + 6*x1^42*x2^26*x3^17*x4^5*z^18 + x1^41*x2^27*x3^17*x4^5*z^18 - 3*x1^38*x2^30*x3^17*x4^5*z^18 - 5*x1^36*x2^32*x3^17*x4^5*z^18 + 2*x1^35*x2^33*x3^17*x4^5*z^18 - x1^34*x2^34*x3^17*x4^5*z^18 - 2*x1^42*x2^25*x3^18*x4^5*z^18 - 6*x1^41*x2^26*x3^18*x4^5*z^18 + 2*x1^40*x2^27*x3^18*x4^5*z^18 - 4*x1^39*x2^28*x3^18*x4^5*z^18 + 4*x1^38*x2^29*x3^18*x4^5*z^18 + 6*x1^36*x2^31*x3^18*x4^5*z^18 + 2*x1^35*x2^32*x3^18*x4^5*z^18 - 2*x1^34*x2^33*x3^18*x4^5*z^18 + 4*x1^41*x2^25*x3^19*x4^5*z^18 + 2*x1^40*x2^26*x3^19*x4^5*z^18 - 4*x1^37*x2^29*x3^19*x4^5*z^18 - x1^36*x2^30*x3^19*x4^5*z^18 - 6*x1^35*x2^31*x3^19*x4^5*z^18 + 2*x1^34*x2^32*x3^19*x4^5*z^18 + x1^33*x2^33*x3^19*x4^5*z^18 - x1^41*x2^24*x3^20*x4^5*z^18 - 4*x1^40*x2^25*x3^20*x4^5*z^18 - 3*x1^38*x2^27*x3^20*x4^5*z^18 + 5*x1^37*x2^28*x3^20*x4^5*z^18 + 6*x1^35*x2^30*x3^20*x4^5*z^18 + 2*x1^34*x2^31*x3^20*x4^5*z^18 - x1^33*x2^32*x3^20*x4^5*z^18 + x1^40*x2^24*x3^21*x4^5*z^18 - 2*x1^36*x2^28*x3^21*x4^5*z^18 - x1^35*x2^29*x3^21*x4^5*z^18 - 4*x1^34*x2^30*x3^21*x4^5*z^18 + 3*x1^33*x2^31*x3^21*x4^5*z^18 - x1^39*x2^24*x3^22*x4^5*z^18 + x1^38*x2^25*x3^22*x4^5*z^18 - 2*x1^37*x2^26*x3^22*x4^5*z^18 + 2*x1^36*x2^27*x3^22*x4^5*z^18 + 4*x1^34*x2^29*x3^22*x4^5*z^18 + x1^33*x2^30*x3^22*x4^5*z^18 - 2*x1^32*x2^31*x3^22*x4^5*z^18 + x1^38*x2^24*x3^23*x4^5*z^18 + x1^34*x2^28*x3^23*x4^5*z^18 - x1^33*x2^29*x3^23*x4^5*z^18 + x1^32*x2^30*x3^23*x4^5*z^18 + x1^33*x2^28*x3^24*x4^5*z^18 - x1^31*x2^30*x3^24*x4^5*z^18 + x1^35*x2^25*x3^25*x4^5*z^18 + x1^32*x2^28*x3^25*x4^5*z^18 + x1^31*x2^29*x3^25*x4^5*z^18 - x1^30*x2^29*x3^26*x4^5*z^18 + x1^44*x2^29*x3^11*x4^6*z^18 + x1^41*x2^32*x3^11*x4^6*z^18 + x1^44*x2^28*x3^12*x4^6*z^18 - x1^43*x2^29*x3^12*x4^6*z^18 - 2*x1^42*x2^30*x3^12*x4^6*z^18 + x1^41*x2^31*x3^12*x4^6*z^18 - x1^39*x2^33*x3^12*x4^6*z^18 + x1^44*x2^27*x3^13*x4^6*z^18 + x1^43*x2^28*x3^13*x4^6*z^18 - 3*x1^42*x2^29*x3^13*x4^6*z^18 - 2*x1^40*x2^31*x3^13*x4^6*z^18 - x1^39*x2^32*x3^13*x4^6*z^18 - x1^38*x2^33*x3^13*x4^6*z^18 + x1^42*x2^28*x3^14*x4^6*z^18 + 3*x1^41*x2^29*x3^14*x4^6*z^18 - 3*x1^40*x2^30*x3^14*x4^6*z^18 + 2*x1^39*x2^31*x3^14*x4^6*z^18 + x1^38*x2^32*x3^14*x4^6*z^18 + x1^37*x2^33*x3^14*x4^6*z^18 - x1^36*x2^34*x3^14*x4^6*z^18 + x1^43*x2^26*x3^15*x4^6*z^18 + 2*x1^42*x2^27*x3^15*x4^6*z^18 - 2*x1^41*x2^28*x3^15*x4^6*z^18 - x1^40*x2^29*x3^15*x4^6*z^18 - 2*x1^39*x2^30*x3^15*x4^6*z^18 + 2*x1^38*x2^31*x3^15*x4^6*z^18 + x1^35*x2^34*x3^15*x4^6*z^18 - 2*x1^42*x2^26*x3^16*x4^6*z^18 + 2*x1^41*x2^27*x3^16*x4^6*z^18 + 4*x1^40*x2^28*x3^16*x4^6*z^18 + 2*x1^38*x2^30*x3^16*x4^6*z^18 - x1^37*x2^31*x3^16*x4^6*z^18 + x1^36*x2^32*x3^16*x4^6*z^18 - x1^35*x2^33*x3^16*x4^6*z^18 + 2*x1^41*x2^26*x3^17*x4^6*z^18 - 4*x1^40*x2^27*x3^17*x4^6*z^18 + x1^39*x2^28*x3^17*x4^6*z^18 - 2*x1^38*x2^29*x3^17*x4^6*z^18 - x1^36*x2^31*x3^17*x4^6*z^18 + 2*x1^35*x2^32*x3^17*x4^6*z^18 + x1^34*x2^33*x3^17*x4^6*z^18 - 2*x1^41*x2^25*x3^18*x4^6*z^18 + 3*x1^39*x2^27*x3^18*x4^6*z^18 + 4*x1^37*x2^29*x3^18*x4^6*z^18 - 2*x1^36*x2^30*x3^18*x4^6*z^18 + 2*x1^35*x2^31*x3^18*x4^6*z^18 - 4*x1^34*x2^32*x3^18*x4^6*z^18 + x1^33*x2^33*x3^18*x4^6*z^18 + x1^41*x2^24*x3^19*x4^6*z^18 + 2*x1^40*x2^25*x3^19*x4^6*z^18 - 4*x1^39*x2^26*x3^19*x4^6*z^18 - 2*x1^37*x2^28*x3^19*x4^6*z^18 - x1^35*x2^30*x3^19*x4^6*z^18 + 4*x1^33*x2^32*x3^19*x4^6*z^18 + x1^39*x2^25*x3^20*x4^6*z^18 + 4*x1^38*x2^26*x3^20*x4^6*z^18 + 4*x1^36*x2^28*x3^20*x4^6*z^18 - x1^35*x2^29*x3^20*x4^6*z^18 + 2*x1^34*x2^30*x3^20*x4^6*z^18 - 4*x1^33*x2^31*x3^20*x4^6*z^18 - 2*x1^32*x2^32*x3^20*x4^6*z^18 - 3*x1^38*x2^25*x3^21*x4^6*z^18 - x1^37*x2^26*x3^21*x4^6*z^18 - 5*x1^36*x2^27*x3^21*x4^6*z^18 - x1^34*x2^29*x3^21*x4^6*z^18 + 3*x1^32*x2^31*x3^21*x4^6*z^18 - x1^38*x2^24*x3^22*x4^6*z^18 + 3*x1^37*x2^25*x3^22*x4^6*z^18 - 2*x1^34*x2^28*x3^22*x4^6*z^18 + 2*x1^33*x2^29*x3^22*x4^6*z^18 - 3*x1^32*x2^30*x3^22*x4^6*z^18 + x1^37*x2^24*x3^23*x4^6*z^18 - x1^35*x2^26*x3^23*x4^6*z^18 - 2*x1^34*x2^27*x3^23*x4^6*z^18 - x1^33*x2^28*x3^23*x4^6*z^18 + x1^32*x2^29*x3^23*x4^6*z^18 + 2*x1^31*x2^30*x3^23*x4^6*z^18 - x1^36*x2^24*x3^24*x4^6*z^18 + x1^35*x2^25*x3^24*x4^6*z^18 + 2*x1^34*x2^26*x3^24*x4^6*z^18 + x1^33*x2^27*x3^24*x4^6*z^18 - 2*x1^31*x2^29*x3^24*x4^6*z^18 - x1^32*x2^27*x3^25*x4^6*z^18 + x1^31*x2^28*x3^25*x4^6*z^18 + 2*x1^30*x2^29*x3^25*x4^6*z^18 + x1^44*x2^29*x3^10*x4^7*z^18 - x1^43*x2^30*x3^10*x4^7*z^18 + x1^41*x2^32*x3^10*x4^7*z^18 + x1^39*x2^34*x3^10*x4^7*z^18 - 2*x1^39*x2^33*x3^11*x4^7*z^18 - x1^38*x2^34*x3^11*x4^7*z^18 + 2*x1^43*x2^28*x3^12*x4^7*z^18 - x1^41*x2^30*x3^12*x4^7*z^18 + x1^39*x2^32*x3^12*x4^7*z^18 + 2*x1^38*x2^33*x3^12*x4^7*z^18 + x1^37*x2^34*x3^12*x4^7*z^18 - x1^43*x2^27*x3^13*x4^7*z^18 + x1^42*x2^28*x3^13*x4^7*z^18 - x1^41*x2^29*x3^13*x4^7*z^18 - 2*x1^36*x2^34*x3^13*x4^7*z^18 + 2*x1^42*x2^27*x3^14*x4^7*z^18 + 2*x1^40*x2^29*x3^14*x4^7*z^18 - x1^39*x2^30*x3^14*x4^7*z^18 + 2*x1^36*x2^33*x3^14*x4^7*z^18 + 2*x1^35*x2^34*x3^14*x4^7*z^18 - x1^42*x2^26*x3^15*x4^7*z^18 - x1^38*x2^30*x3^15*x4^7*z^18 - 2*x1^37*x2^31*x3^15*x4^7*z^18 + x1^36*x2^32*x3^15*x4^7*z^18 - 2*x1^35*x2^33*x3^15*x4^7*z^18 - x1^34*x2^34*x3^15*x4^7*z^18 - x1^35*x2^32*x3^16*x4^7*z^18 + 2*x1^34*x2^33*x3^16*x4^7*z^18 - x1^40*x2^26*x3^17*x4^7*z^18 - x1^33*x2^33*x3^17*x4^7*z^18 + 2*x1^39*x2^26*x3^18*x4^7*z^18 - x1^38*x2^27*x3^18*x4^7*z^18 - x1^40*x2^24*x3^19*x4^7*z^18 - 2*x1^38*x2^26*x3^19*x4^7*z^18 - x1^36*x2^28*x3^19*x4^7*z^18 + x1^39*x2^24*x3^20*x4^7*z^18 + 2*x1^38*x2^25*x3^20*x4^7*z^18 + 2*x1^37*x2^26*x3^20*x4^7*z^18 + x1^36*x2^27*x3^20*x4^7*z^18 - x1^34*x2^29*x3^20*x4^7*z^18 - x1^39*x2^23*x3^21*x4^7*z^18 - x1^38*x2^24*x3^21*x4^7*z^18 - 3*x1^37*x2^25*x3^21*x4^7*z^18 - x1^36*x2^26*x3^21*x4^7*z^18 - x1^35*x2^27*x3^21*x4^7*z^18 + x1^33*x2^29*x3^21*x4^7*z^18 + x1^38*x2^23*x3^22*x4^7*z^18 + 2*x1^37*x2^24*x3^22*x4^7*z^18 + 2*x1^36*x2^25*x3^22*x4^7*z^18 + 3*x1^35*x2^26*x3^22*x4^7*z^18 + 2*x1^34*x2^27*x3^22*x4^7*z^18 + x1^33*x2^28*x3^22*x4^7*z^18 - 2*x1^36*x2^24*x3^23*x4^7*z^18 - x1^35*x2^25*x3^23*x4^7*z^18 - x1^34*x2^26*x3^23*x4^7*z^18 - 2*x1^32*x2^28*x3^23*x4^7*z^18 + x1^35*x2^24*x3^24*x4^7*z^18 + x1^34*x2^25*x3^24*x4^7*z^18 + 2*x1^32*x2^27*x3^24*x4^7*z^18 + x1^31*x2^28*x3^24*x4^7*z^18 - x1^30*x2^29*x3^24*x4^7*z^18 - x1^33*x2^25*x3^25*x4^7*z^18 - x1^32*x2^26*x3^25*x4^7*z^18 + x1^30*x2^28*x3^25*x4^7*z^18 - x1^29*x2^28*x3^26*x4^7*z^18 - x1^41*x2^31*x3^10*x4^8*z^18 - x1^40*x2^32*x3^10*x4^8*z^18 + x1^43*x2^28*x3^11*x4^8*z^18 + x1^42*x2^29*x3^11*x4^8*z^18 - x1^41*x2^30*x3^11*x4^8*z^18 - x1^39*x2^32*x3^11*x4^8*z^18 - x1^38*x2^33*x3^11*x4^8*z^18 - x1^37*x2^34*x3^11*x4^8*z^18 - x1^43*x2^27*x3^12*x4^8*z^18 + x1^40*x2^30*x3^12*x4^8*z^18 - x1^39*x2^31*x3^12*x4^8*z^18 + x1^38*x2^32*x3^12*x4^8*z^18 + 2*x1^36*x2^34*x3^12*x4^8*z^18 - x1^42*x2^27*x3^13*x4^8*z^18 + x1^41*x2^28*x3^13*x4^8*z^18 + 2*x1^40*x2^29*x3^13*x4^8*z^18 - x1^37*x2^32*x3^13*x4^8*z^18 - x1^36*x2^33*x3^13*x4^8*z^18 - 3*x1^35*x2^34*x3^13*x4^8*z^18 - 2*x1^42*x2^26*x3^14*x4^8*z^18 - x1^41*x2^27*x3^14*x4^8*z^18 - 3*x1^40*x2^28*x3^14*x4^8*z^18 - x1^39*x2^29*x3^14*x4^8*z^18 + 3*x1^37*x2^31*x3^14*x4^8*z^18 - x1^36*x2^32*x3^14*x4^8*z^18 + 3*x1^35*x2^33*x3^14*x4^8*z^18 + 2*x1^34*x2^34*x3^14*x4^8*z^18 + x1^41*x2^26*x3^15*x4^8*z^18 + 2*x1^40*x2^27*x3^15*x4^8*z^18 - x1^38*x2^29*x3^15*x4^8*z^18 - x1^37*x2^30*x3^15*x4^8*z^18 - 2*x1^36*x2^31*x3^15*x4^8*z^18 - 2*x1^35*x2^32*x3^15*x4^8*z^18 - 3*x1^34*x2^33*x3^15*x4^8*z^18 - 2*x1^41*x2^25*x3^16*x4^8*z^18 - x1^40*x2^26*x3^16*x4^8*z^18 - 3*x1^39*x2^27*x3^16*x4^8*z^18 + 4*x1^36*x2^30*x3^16*x4^8*z^18 + x1^35*x2^31*x3^16*x4^8*z^18 + 4*x1^34*x2^32*x3^16*x4^8*z^18 + x1^33*x2^33*x3^16*x4^8*z^18 + x1^40*x2^25*x3^17*x4^8*z^18 + 2*x1^39*x2^26*x3^17*x4^8*z^18 + x1^38*x2^27*x3^17*x4^8*z^18 - x1^37*x2^28*x3^17*x4^8*z^18 - x1^36*x2^29*x3^17*x4^8*z^18 - 3*x1^35*x2^30*x3^17*x4^8*z^18 - x1^34*x2^31*x3^17*x4^8*z^18 - 4*x1^33*x2^32*x3^17*x4^8*z^18 - 2*x1^40*x2^24*x3^18*x4^8*z^18 - 3*x1^38*x2^26*x3^18*x4^8*z^18 - x1^37*x2^27*x3^18*x4^8*z^18 - 3*x1^36*x2^28*x3^18*x4^8*z^18 + 2*x1^35*x2^29*x3^18*x4^8*z^18 + 2*x1^34*x2^30*x3^18*x4^8*z^18 + 4*x1^33*x2^31*x3^18*x4^8*z^18 + 2*x1^32*x2^32*x3^18*x4^8*z^18 + x1^40*x2^23*x3^19*x4^8*z^18 + 2*x1^39*x2^24*x3^19*x4^8*z^18 + 5*x1^37*x2^26*x3^19*x4^8*z^18 + x1^36*x2^27*x3^19*x4^8*z^18 - 4*x1^34*x2^29*x3^19*x4^8*z^18 - 3*x1^33*x2^30*x3^19*x4^8*z^18 - 4*x1^32*x2^31*x3^19*x4^8*z^18 - 2*x1^39*x2^23*x3^20*x4^8*z^18 - 2*x1^38*x2^24*x3^20*x4^8*z^18 - x1^37*x2^25*x3^20*x4^8*z^18 - x1^36*x2^26*x3^20*x4^8*z^18 + 2*x1^35*x2^27*x3^20*x4^8*z^18 + 5*x1^34*x2^28*x3^20*x4^8*z^18 + 2*x1^33*x2^29*x3^20*x4^8*z^18 + 4*x1^32*x2^30*x3^20*x4^8*z^18 + x1^31*x2^31*x3^20*x4^8*z^18 + 2*x1^38*x2^23*x3^21*x4^8*z^18 + x1^37*x2^24*x3^21*x4^8*z^18 - 2*x1^35*x2^26*x3^21*x4^8*z^18 - 2*x1^33*x2^28*x3^21*x4^8*z^18 - x1^32*x2^29*x3^21*x4^8*z^18 - 4*x1^31*x2^30*x3^21*x4^8*z^18 - x1^37*x2^23*x3^22*x4^8*z^18 - x1^36*x2^24*x3^22*x4^8*z^18 + x1^32*x2^28*x3^22*x4^8*z^18 + 2*x1^31*x2^29*x3^22*x4^8*z^18 + x1^30*x2^30*x3^22*x4^8*z^18 + x1^35*x2^24*x3^23*x4^8*z^18 - x1^34*x2^25*x3^23*x4^8*z^18 - x1^33*x2^26*x3^23*x4^8*z^18 + x1^31*x2^28*x3^23*x4^8*z^18 - x1^30*x2^29*x3^23*x4^8*z^18 + 2*x1^32*x2^26*x3^24*x4^8*z^18 + 2*x1^31*x2^27*x3^24*x4^8*z^18 + x1^30*x2^28*x3^24*x4^8*z^18 + x1^32*x2^25*x3^25*x4^8*z^18 - x1^31*x2^26*x3^25*x4^8*z^18 + x1^30*x2^26*x3^26*x4^8*z^18 + x1^40*x2^31*x3^10*x4^9*z^18 - x1^39*x2^31*x3^11*x4^9*z^18 - x1^37*x2^33*x3^11*x4^9*z^18 - x1^42*x2^27*x3^12*x4^9*z^18 - x1^41*x2^28*x3^12*x4^9*z^18 - x1^40*x2^29*x3^12*x4^9*z^18 + x1^37*x2^32*x3^12*x4^9*z^18 + x1^36*x2^33*x3^12*x4^9*z^18 + x1^42*x2^26*x3^13*x4^9*z^18 + 2*x1^40*x2^28*x3^13*x4^9*z^18 - 3*x1^39*x2^29*x3^13*x4^9*z^18 - 3*x1^38*x2^30*x3^13*x4^9*z^18 - x1^37*x2^31*x3^13*x4^9*z^18 - x1^36*x2^32*x3^13*x4^9*z^18 - 2*x1^35*x2^33*x3^13*x4^9*z^18 - 3*x1^41*x2^26*x3^14*x4^9*z^18 - x1^40*x2^27*x3^14*x4^9*z^18 + 2*x1^38*x2^29*x3^14*x4^9*z^18 + x1^37*x2^30*x3^14*x4^9*z^18 + 2*x1^36*x2^31*x3^14*x4^9*z^18 + x1^35*x2^32*x3^14*x4^9*z^18 + 2*x1^34*x2^33*x3^14*x4^9*z^18 + 3*x1^41*x2^25*x3^15*x4^9*z^18 + x1^40*x2^26*x3^15*x4^9*z^18 + x1^39*x2^27*x3^15*x4^9*z^18 - x1^37*x2^29*x3^15*x4^9*z^18 - 5*x1^36*x2^30*x3^15*x4^9*z^18 - x1^35*x2^31*x3^15*x4^9*z^18 - x1^34*x2^32*x3^15*x4^9*z^18 - x1^33*x2^33*x3^15*x4^9*z^18 - 4*x1^40*x2^25*x3^16*x4^9*z^18 - 3*x1^39*x2^26*x3^16*x4^9*z^18 - x1^38*x2^27*x3^16*x4^9*z^18 + 2*x1^37*x2^28*x3^16*x4^9*z^18 + x1^36*x2^29*x3^16*x4^9*z^18 + 4*x1^35*x2^30*x3^16*x4^9*z^18 + 2*x1^33*x2^32*x3^16*x4^9*z^18 + 4*x1^40*x2^24*x3^17*x4^9*z^18 + x1^39*x2^25*x3^17*x4^9*z^18 + 2*x1^38*x2^26*x3^17*x4^9*z^18 - x1^37*x2^27*x3^17*x4^9*z^18 + x1^36*x2^28*x3^17*x4^9*z^18 - 2*x1^35*x2^29*x3^17*x4^9*z^18 - 4*x1^34*x2^30*x3^17*x4^9*z^18 - 2*x1^33*x2^31*x3^17*x4^9*z^18 - x1^32*x2^32*x3^17*x4^9*z^18 - x1^40*x2^23*x3^18*x4^9*z^18 - 4*x1^39*x2^24*x3^18*x4^9*z^18 + x1^38*x2^25*x3^18*x4^9*z^18 - 4*x1^37*x2^26*x3^18*x4^9*z^18 + 2*x1^36*x2^27*x3^18*x4^9*z^18 + x1^35*x2^28*x3^18*x4^9*z^18 + 5*x1^34*x2^29*x3^18*x4^9*z^18 + 2*x1^33*x2^30*x3^18*x4^9*z^18 + 2*x1^32*x2^31*x3^18*x4^9*z^18 + 4*x1^39*x2^23*x3^19*x4^9*z^18 + 3*x1^38*x2^24*x3^19*x4^9*z^18 + 2*x1^37*x2^25*x3^19*x4^9*z^18 + x1^36*x2^26*x3^19*x4^9*z^18 - x1^34*x2^28*x3^19*x4^9*z^18 - 4*x1^33*x2^29*x3^19*x4^9*z^18 - 2*x1^32*x2^30*x3^19*x4^9*z^18 - x1^31*x2^31*x3^19*x4^9*z^18 - 4*x1^38*x2^23*x3^20*x4^9*z^18 - x1^37*x2^24*x3^20*x4^9*z^18 - 3*x1^36*x2^25*x3^20*x4^9*z^18 + x1^35*x2^26*x3^20*x4^9*z^18 - x1^34*x2^27*x3^20*x4^9*z^18 + 3*x1^33*x2^28*x3^20*x4^9*z^18 + 3*x1^32*x2^29*x3^20*x4^9*z^18 + 2*x1^31*x2^30*x3^20*x4^9*z^18 + x1^37*x2^23*x3^21*x4^9*z^18 + x1^36*x2^24*x3^21*x4^9*z^18 + x1^35*x2^25*x3^21*x4^9*z^18 - 2*x1^33*x2^27*x3^21*x4^9*z^18 - 2*x1^32*x2^28*x3^21*x4^9*z^18 - x1^31*x2^29*x3^21*x4^9*z^18 - x1^36*x2^23*x3^22*x4^9*z^18 - 2*x1^35*x2^24*x3^22*x4^9*z^18 + x1^34*x2^25*x3^22*x4^9*z^18 - x1^33*x2^26*x3^22*x4^9*z^18 + 3*x1^32*x2^27*x3^22*x4^9*z^18 - x1^31*x2^28*x3^22*x4^9*z^18 + x1^30*x2^29*x3^22*x4^9*z^18 + x1^35*x2^23*x3^23*x4^9*z^18 + x1^33*x2^25*x3^23*x4^9*z^18 - 4*x1^31*x2^27*x3^23*x4^9*z^18 + x1^30*x2^28*x3^23*x4^9*z^18 - x1^29*x2^29*x3^23*x4^9*z^18 + x1^33*x2^24*x3^24*x4^9*z^18 - 2*x1^32*x2^25*x3^24*x4^9*z^18 + 2*x1^31*x2^26*x3^24*x4^9*z^18 + x1^30*x2^27*x3^24*x4^9*z^18 - x1^29*x2^28*x3^24*x4^9*z^18 + x1^31*x2^25*x3^25*x4^9*z^18 - 2*x1^30*x2^26*x3^25*x4^9*z^18 + x1^28*x2^28*x3^25*x4^9*z^18 - x1^39*x2^31*x3^10*x4^10*z^18 - x1^38*x2^31*x3^11*x4^10*z^18 + x1^38*x2^30*x3^12*x4^10*z^18 - x1^37*x2^30*x3^13*x4^10*z^18 + x1^35*x2^32*x3^13*x4^10*z^18 + x1^40*x2^26*x3^14*x4^10*z^18 - 2*x1^39*x2^27*x3^14*x4^10*z^18 + x1^37*x2^29*x3^14*x4^10*z^18 - 2*x1^35*x2^31*x3^14*x4^10*z^18 + 2*x1^34*x2^32*x3^14*x4^10*z^18 + x1^33*x2^33*x3^14*x4^10*z^18 + x1^40*x2^25*x3^15*x4^10*z^18 + x1^39*x2^26*x3^15*x4^10*z^18 + x1^37*x2^28*x3^15*x4^10*z^18 - x1^36*x2^29*x3^15*x4^10*z^18 + x1^35*x2^30*x3^15*x4^10*z^18 - 2*x1^33*x2^32*x3^15*x4^10*z^18 - x1^40*x2^24*x3^16*x4^10*z^18 + x1^39*x2^25*x3^16*x4^10*z^18 - x1^38*x2^26*x3^16*x4^10*z^18 + x1^37*x2^27*x3^16*x4^10*z^18 - x1^36*x2^28*x3^16*x4^10*z^18 + x1^34*x2^30*x3^16*x4^10*z^18 + x1^33*x2^31*x3^16*x4^10*z^18 - x1^40*x2^23*x3^17*x4^10*z^18 + 2*x1^39*x2^24*x3^17*x4^10*z^18 + 2*x1^38*x2^25*x3^17*x4^10*z^18 + x1^36*x2^27*x3^17*x4^10*z^18 - x1^38*x2^24*x3^18*x4^10*z^18 + 3*x1^36*x2^26*x3^18*x4^10*z^18 - x1^35*x2^27*x3^18*x4^10*z^18 + x1^38*x2^23*x3^19*x4^10*z^18 - 2*x1^34*x2^27*x3^19*x4^10*z^18 - x1^33*x2^28*x3^19*x4^10*z^18 - x1^38*x2^22*x3^20*x4^10*z^18 - x1^37*x2^23*x3^20*x4^10*z^18 + x1^35*x2^25*x3^20*x4^10*z^18 + x1^34*x2^26*x3^20*x4^10*z^18 + 2*x1^33*x2^27*x3^20*x4^10*z^18 + x1^32*x2^28*x3^20*x4^10*z^18 + x1^37*x2^22*x3^21*x4^10*z^18 - x1^35*x2^24*x3^21*x4^10*z^18 - 3*x1^34*x2^25*x3^21*x4^10*z^18 - x1^33*x2^26*x3^21*x4^10*z^18 - 3*x1^32*x2^27*x3^21*x4^10*z^18 - x1^31*x2^28*x3^21*x4^10*z^18 + x1^34*x2^24*x3^22*x4^10*z^18 + x1^32*x2^26*x3^22*x4^10*z^18 + 3*x1^31*x2^27*x3^22*x4^10*z^18 + x1^34*x2^23*x3^23*x4^10*z^18 - x1^33*x2^24*x3^23*x4^10*z^18 - 2*x1^31*x2^26*x3^23*x4^10*z^18 - x1^30*x2^27*x3^23*x4^10*z^18 + x1^32*x2^24*x3^24*x4^10*z^18 + x1^31*x2^25*x3^24*x4^10*z^18 + 2*x1^30*x2^26*x3^24*x4^10*z^18 + x1^28*x2^28*x3^24*x4^10*z^18 - x1^29*x2^26*x3^25*x4^10*z^18 + x1^38*x2^30*x3^11*x4^11*z^18 + x1^37*x2^31*x3^11*x4^11*z^18 - x1^38*x2^29*x3^12*x4^11*z^18 + x1^37*x2^30*x3^12*x4^11*z^18 - x1^35*x2^32*x3^12*x4^11*z^18 + 2*x1^35*x2^31*x3^13*x4^11*z^18 + x1^40*x2^25*x3^14*x4^11*z^18 + x1^39*x2^26*x3^14*x4^11*z^18 - 3*x1^37*x2^28*x3^14*x4^11*z^18 + 2*x1^36*x2^29*x3^14*x4^11*z^18 + 2*x1^33*x2^32*x3^14*x4^11*z^18 - x1^40*x2^24*x3^15*x4^11*z^18 + 3*x1^38*x2^26*x3^15*x4^11*z^18 + 2*x1^37*x2^27*x3^15*x4^11*z^18 - 3*x1^35*x2^29*x3^15*x4^11*z^18 + 4*x1^34*x2^30*x3^15*x4^11*z^18 - x1^33*x2^31*x3^15*x4^11*z^18 - x1^32*x2^32*x3^15*x4^11*z^18 - x1^38*x2^25*x3^16*x4^11*z^18 + x1^37*x2^26*x3^16*x4^11*z^18 - 3*x1^36*x2^27*x3^16*x4^11*z^18 + x1^35*x2^28*x3^16*x4^11*z^18 - 3*x1^34*x2^29*x3^16*x4^11*z^18 + 3*x1^32*x2^31*x3^16*x4^11*z^18 - x1^39*x2^23*x3^17*x4^11*z^18 - 2*x1^38*x2^24*x3^17*x4^11*z^18 + x1^37*x2^25*x3^17*x4^11*z^18 + 2*x1^35*x2^27*x3^17*x4^11*z^18 - x1^34*x2^28*x3^17*x4^11*z^18 + 2*x1^33*x2^29*x3^17*x4^11*z^18 - 3*x1^32*x2^30*x3^17*x4^11*z^18 - x1^38*x2^23*x3^18*x4^11*z^18 - x1^37*x2^24*x3^18*x4^11*z^18 + x1^36*x2^25*x3^18*x4^11*z^18 - x1^35*x2^26*x3^18*x4^11*z^18 - 4*x1^33*x2^28*x3^18*x4^11*z^18 - 2*x1^32*x2^29*x3^18*x4^11*z^18 + 2*x1^31*x2^30*x3^18*x4^11*z^18 + x1^37*x2^23*x3^19*x4^11*z^18 + x1^36*x2^24*x3^19*x4^11*z^18 + 4*x1^34*x2^26*x3^19*x4^11*z^18 + x1^33*x2^27*x3^19*x4^11*z^18 + 4*x1^32*x2^28*x3^19*x4^11*z^18 - 2*x1^31*x2^29*x3^19*x4^11*z^18 - x1^30*x2^30*x3^19*x4^11*z^18 - x1^35*x2^24*x3^20*x4^11*z^18 - x1^34*x2^25*x3^20*x4^11*z^18 + 2*x1^33*x2^26*x3^20*x4^11*z^18 - x1^32*x2^27*x3^20*x4^11*z^18 + x1^31*x2^28*x3^20*x4^11*z^18 + 2*x1^30*x2^29*x3^20*x4^11*z^18 - x1^33*x2^25*x3^21*x4^11*z^18 - 3*x1^32*x2^26*x3^21*x4^11*z^18 + 2*x1^31*x2^27*x3^21*x4^11*z^18 - 2*x1^30*x2^28*x3^21*x4^11*z^18 - x1^29*x2^29*x3^21*x4^11*z^18 - x1^30*x2^27*x3^22*x4^11*z^18 + 2*x1^29*x2^28*x3^22*x4^11*z^18 - x1^28*x2^28*x3^23*x4^11*z^18 - x1^37*x2^29*x3^12*x4^12*z^18 - x1^36*x2^30*x3^12*x4^12*z^18 - x1^35*x2^31*x3^12*x4^12*z^18 + 2*x1^37*x2^28*x3^13*x4^12*z^18 - 2*x1^36*x2^29*x3^13*x4^12*z^18 - x1^35*x2^30*x3^13*x4^12*z^18 + x1^34*x2^31*x3^13*x4^12*z^18 - x1^33*x2^32*x3^13*x4^12*z^18 - x1^36*x2^28*x3^14*x4^12*z^18 + 2*x1^35*x2^29*x3^14*x4^12*z^18 - 2*x1^34*x2^30*x3^14*x4^12*z^18 + x1^33*x2^31*x3^14*x4^12*z^18 + x1^32*x2^32*x3^14*x4^12*z^18 - x1^39*x2^24*x3^15*x4^12*z^18 - x1^38*x2^25*x3^15*x4^12*z^18 + x1^36*x2^27*x3^15*x4^12*z^18 - 3*x1^35*x2^28*x3^15*x4^12*z^18 - x1^34*x2^29*x3^15*x4^12*z^18 - 4*x1^32*x2^31*x3^15*x4^12*z^18 + x1^39*x2^23*x3^16*x4^12*z^18 - 3*x1^37*x2^25*x3^16*x4^12*z^18 - x1^35*x2^27*x3^16*x4^12*z^18 + 4*x1^34*x2^28*x3^16*x4^12*z^18 - 2*x1^33*x2^29*x3^16*x4^12*z^18 + 3*x1^32*x2^30*x3^16*x4^12*z^18 + 2*x1^31*x2^31*x3^16*x4^12*z^18 + 2*x1^37*x2^24*x3^17*x4^12*z^18 + x1^36*x2^25*x3^17*x4^12*z^18 + x1^35*x2^26*x3^17*x4^12*z^18 - 3*x1^34*x2^27*x3^17*x4^12*z^18 - x1^32*x2^29*x3^17*x4^12*z^18 - 5*x1^31*x2^30*x3^17*x4^12*z^18 - 4*x1^36*x2^24*x3^18*x4^12*z^18 + x1^35*x2^25*x3^18*x4^12*z^18 + 2*x1^33*x2^27*x3^18*x4^12*z^18 - x1^32*x2^28*x3^18*x4^12*z^18 + 6*x1^31*x2^29*x3^18*x4^12*z^18 + 2*x1^30*x2^30*x3^18*x4^12*z^18 - x1^36*x2^23*x3^19*x4^12*z^18 + 2*x1^35*x2^24*x3^19*x4^12*z^18 - 2*x1^33*x2^26*x3^19*x4^12*z^18 - x1^32*x2^27*x3^19*x4^12*z^18 - x1^31*x2^28*x3^19*x4^12*z^18 - 6*x1^30*x2^29*x3^19*x4^12*z^18 - 2*x1^34*x2^24*x3^20*x4^12*z^18 - x1^33*x2^25*x3^20*x4^12*z^18 + 3*x1^32*x2^26*x3^20*x4^12*z^18 - x1^31*x2^27*x3^20*x4^12*z^18 + 5*x1^30*x2^28*x3^20*x4^12*z^18 + 2*x1^29*x2^29*x3^20*x4^12*z^18 + 2*x1^33*x2^24*x3^21*x4^12*z^18 + x1^32*x2^25*x3^21*x4^12*z^18 - x1^30*x2^27*x3^21*x4^12*z^18 - 5*x1^29*x2^28*x3^21*x4^12*z^18 + 2*x1^31*x2^25*x3^22*x4^12*z^18 - x1^30*x2^26*x3^22*x4^12*z^18 + 3*x1^29*x2^27*x3^22*x4^12*z^18 + 2*x1^28*x2^28*x3^22*x4^12*z^18 + 2*x1^29*x2^26*x3^23*x4^12*z^18 - 3*x1^28*x2^27*x3^23*x4^12*z^18 + 2*x1^27*x2^27*x3^24*x4^12*z^18 - x1^36*x2^27*x3^14*x4^13*z^18 + x1^35*x2^28*x3^14*x4^13*z^18 - x1^34*x2^29*x3^14*x4^13*z^18 - x1^33*x2^30*x3^14*x4^13*z^18 + 2*x1^35*x2^27*x3^15*x4^13*z^18 - x1^34*x2^28*x3^15*x4^13*z^18 + x1^33*x2^29*x3^15*x4^13*z^18 - x1^32*x2^30*x3^15*x4^13*z^18 - x1^31*x2^31*x3^15*x4^13*z^18 - x1^35*x2^26*x3^16*x4^13*z^18 + 2*x1^34*x2^27*x3^16*x4^13*z^18 + x1^33*x2^28*x3^16*x4^13*z^18 - x1^32*x2^29*x3^16*x4^13*z^18 + 2*x1^31*x2^30*x3^16*x4^13*z^18 + x1^37*x2^23*x3^17*x4^13*z^18 - x1^36*x2^24*x3^17*x4^13*z^18 - x1^35*x2^25*x3^17*x4^13*z^18 + x1^34*x2^26*x3^17*x4^13*z^18 - 3*x1^33*x2^27*x3^17*x4^13*z^18 - 2*x1^31*x2^29*x3^17*x4^13*z^18 - 2*x1^30*x2^30*x3^17*x4^13*z^18 - x1^34*x2^25*x3^18*x4^13*z^18 + 3*x1^33*x2^26*x3^18*x4^13*z^18 + x1^31*x2^28*x3^18*x4^13*z^18 + 4*x1^30*x2^29*x3^18*x4^13*z^18 + x1^34*x2^24*x3^19*x4^13*z^18 - x1^32*x2^26*x3^19*x4^13*z^18 + 3*x1^31*x2^27*x3^19*x4^13*z^18 - 5*x1^30*x2^28*x3^19*x4^13*z^18 - 2*x1^29*x2^29*x3^19*x4^13*z^18 - x1^33*x2^24*x3^20*x4^13*z^18 + x1^31*x2^26*x3^20*x4^13*z^18 + x1^30*x2^27*x3^20*x4^13*z^18 + 6*x1^29*x2^28*x3^20*x4^13*z^18 - 2*x1^31*x2^25*x3^21*x4^13*z^18 + x1^30*x2^26*x3^21*x4^13*z^18 - 2*x1^29*x2^27*x3^21*x4^13*z^18 - 2*x1^28*x2^28*x3^21*x4^13*z^18 - x1^29*x2^26*x3^22*x4^13*z^18 + 2*x1^28*x2^27*x3^22*x4^13*z^18 - x1^27*x2^27*x3^23*x4^13*z^18 + x1^35*x2^26*x3^15*x4^14*z^18 + x1^34*x2^27*x3^15*x4^14*z^18 + x1^33*x2^28*x3^15*x4^14*z^18 - x1^35*x2^25*x3^16*x4^14*z^18 - x1^34*x2^26*x3^16*x4^14*z^18 + x1^33*x2^27*x3^16*x4^14*z^18 - x1^32*x2^28*x3^16*x4^14*z^18 + 2*x1^34*x2^25*x3^17*x4^14*z^18 + 2*x1^32*x2^27*x3^17*x4^14*z^18 + x1^31*x2^28*x3^17*x4^14*z^18 - x1^33*x2^25*x3^18*x4^14*z^18 + x1^32*x2^26*x3^18*x4^14*z^18 - x1^31*x2^27*x3^18*x4^14*z^18 + x1^30*x2^28*x3^18*x4^14*z^18 + x1^34*x2^23*x3^19*x4^14*z^18 + x1^33*x2^24*x3^19*x4^14*z^18 - x1^32*x2^25*x3^19*x4^14*z^18 + 2*x1^31*x2^26*x3^19*x4^14*z^18 + 2*x1^30*x2^27*x3^19*x4^14*z^18 - x1^29*x2^28*x3^19*x4^14*z^18 - x1^33*x2^23*x3^20*x4^14*z^18 - x1^32*x2^24*x3^20*x4^14*z^18 - 3*x1^30*x2^26*x3^20*x4^14*z^18 + x1^28*x2^28*x3^20*x4^14*z^18 + x1^30*x2^25*x3^21*x4^14*z^18 - x1^29*x2^25*x3^22*x4^14*z^18 - x1^34*x2^25*x3^16*x4^15*z^18 - x1^33*x2^26*x3^16*x4^15*z^18 - x1^32*x2^27*x3^16*x4^15*z^18 + x1^34*x2^24*x3^17*x4^15*z^18 + x1^33*x2^25*x3^17*x4^15*z^18 - x1^32*x2^26*x3^17*x4^15*z^18 + 3*x1^31*x2^27*x3^17*x4^15*z^18 - 2*x1^33*x2^24*x3^18*x4^15*z^18 - 2*x1^31*x2^26*x3^18*x4^15*z^18 - 2*x1^30*x2^27*x3^18*x4^15*z^18 - x1^33*x2^23*x3^19*x4^15*z^18 + 5*x1^30*x2^26*x3^19*x4^15*z^18 + x1^31*x2^24*x3^20*x4^15*z^18 - 2*x1^30*x2^25*x3^20*x4^15*z^18 - 2*x1^29*x2^26*x3^20*x4^15*z^18 - 2*x1^30*x2^24*x3^21*x4^15*z^18 + 3*x1^29*x2^25*x3^21*x4^15*z^18 - x1^28*x2^25*x3^22*x4^15*z^18 - 2*x1^30*x2^26*x3^18*x4^16*z^18 + x1^29*x2^26*x3^19*x4^16*z^18 - 2*x1^29*x2^25*x3^20*x4^16*z^18 - x1^43*x2^30*x3^12*z^17 + x1^43*x2^29*x3^13*z^17 + x1^42*x2^30*x3^13*z^17 - x1^41*x2^31*x3^13*z^17 - 2*x1^42*x2^29*x3^14*z^17 - x1^41*x2^30*x3^14*z^17 + x1^42*x2^28*x3^15*z^17 + 2*x1^41*x2^29*x3^15*z^17 + x1^39*x2^31*x3^15*z^17 - x1^38*x2^32*x3^15*z^17 - 2*x1^41*x2^28*x3^16*z^17 - x1^38*x2^31*x3^16*z^17 + x1^36*x2^33*x3^16*z^17 + 2*x1^40*x2^28*x3^17*z^17 + 2*x1^38*x2^30*x3^17*z^17 - x1^36*x2^32*x3^17*z^17 - x1^35*x2^33*x3^17*z^17 - 2*x1^40*x2^27*x3^18*z^17 - x1^37*x2^30*x3^18*z^17 + x1^40*x2^26*x3^19*z^17 + 2*x1^37*x2^29*x3^19*z^17 - x1^39*x2^26*x3^20*z^17 - x1^38*x2^27*x3^20*z^17 - x1^37*x2^28*x3^20*z^17 - x1^44*x2^30*x3^10*x4*z^17 + 2*x1^43*x2^30*x3^11*x4*z^17 + x1^42*x2^31*x3^11*x4*z^17 + x1^41*x2^32*x3^11*x4*z^17 - 2*x1^43*x2^29*x3^12*x4*z^17 - 2*x1^42*x2^30*x3^12*x4*z^17 - x1^40*x2^32*x3^12*x4*z^17 + x1^39*x2^33*x3^12*x4*z^17 + 6*x1^42*x2^29*x3^13*x4*z^17 + x1^41*x2^30*x3^13*x4*z^17 + 2*x1^40*x2^31*x3^13*x4*z^17 + x1^39*x2^32*x3^13*x4*z^17 - 2*x1^42*x2^28*x3^14*x4*z^17 - 6*x1^41*x2^29*x3^14*x4*z^17 + x1^40*x2^30*x3^14*x4*z^17 - 4*x1^39*x2^31*x3^14*x4*z^17 + x1^38*x2^32*x3^14*x4*z^17 + x1^36*x2^34*x3^14*x4*z^17 + 6*x1^41*x2^28*x3^15*x4*z^17 + 2*x1^40*x2^29*x3^15*x4*z^17 + x1^39*x2^30*x3^15*x4*z^17 + x1^38*x2^31*x3^15*x4*z^17 - x1^37*x2^32*x3^15*x4*z^17 - x1^36*x2^33*x3^15*x4*z^17 - x1^35*x2^34*x3^15*x4*z^17 - 2*x1^41*x2^27*x3^16*x4*z^17 - 6*x1^40*x2^28*x3^16*x4*z^17 - 4*x1^38*x2^30*x3^16*x4*z^17 + x1^37*x2^31*x3^16*x4*z^17 + 3*x1^35*x2^33*x3^16*x4*z^17 + 6*x1^40*x2^27*x3^17*x4*z^17 + 2*x1^39*x2^28*x3^17*x4*z^17 + 2*x1^38*x2^29*x3^17*x4*z^17 + 2*x1^37*x2^30*x3^17*x4*z^17 - x1^36*x2^31*x3^17*x4*z^17 - 3*x1^35*x2^32*x3^17*x4*z^17 - x1^34*x2^33*x3^17*x4*z^17 - 2*x1^40*x2^26*x3^18*x4*z^17 - 6*x1^39*x2^27*x3^18*x4*z^17 - 4*x1^37*x2^29*x3^18*x4*z^17 + 4*x1^34*x2^32*x3^18*x4*z^17 + 6*x1^39*x2^26*x3^19*x4*z^17 + x1^38*x2^27*x3^19*x4*z^17 + x1^37*x2^28*x3^19*x4*z^17 + 3*x1^36*x2^29*x3^19*x4*z^17 - 2*x1^34*x2^31*x3^19*x4*z^17 - x1^33*x2^32*x3^19*x4*z^17 - 2*x1^39*x2^25*x3^20*x4*z^17 - 3*x1^38*x2^26*x3^20*x4*z^17 - x1^37*x2^27*x3^20*x4*z^17 - 4*x1^36*x2^28*x3^20*x4*z^17 + x1^35*x2^29*x3^20*x4*z^17 - x1^34*x2^30*x3^20*x4*z^17 + x1^33*x2^31*x3^20*x4*z^17 + 2*x1^38*x2^25*x3^21*x4*z^17 + 2*x1^36*x2^27*x3^21*x4*z^17 + 2*x1^35*x2^28*x3^21*x4*z^17 - x1^37*x2^25*x3^22*x4*z^17 - x1^36*x2^26*x3^22*x4*z^17 - 3*x1^35*x2^27*x3^22*x4*z^17 + x1^34*x2^27*x3^23*x4*z^17 - 2*x1^43*x2^30*x3^10*x4^2*z^17 + x1^42*x2^31*x3^10*x4^2*z^17 + 2*x1^43*x2^29*x3^11*x4^2*z^17 + 2*x1^42*x2^30*x3^11*x4^2*z^17 - 2*x1^41*x2^31*x3^11*x4^2*z^17 + x1^40*x2^32*x3^11*x4^2*z^17 - 5*x1^42*x2^29*x3^12*x4^2*z^17 + x1^40*x2^31*x3^12*x4^2*z^17 + x1^38*x2^33*x3^12*x4^2*z^17 + 2*x1^42*x2^28*x3^13*x4^2*z^17 + 5*x1^41*x2^29*x3^13*x4^2*z^17 - 2*x1^40*x2^30*x3^13*x4^2*z^17 + x1^39*x2^31*x3^13*x4^2*z^17 - 2*x1^38*x2^32*x3^13*x4^2*z^17 - x1^37*x2^33*x3^13*x4^2*z^17 - 6*x1^41*x2^28*x3^14*x4^2*z^17 - 2*x1^40*x2^29*x3^14*x4^2*z^17 + x1^38*x2^31*x3^14*x4^2*z^17 + x1^37*x2^32*x3^14*x4^2*z^17 + x1^36*x2^33*x3^14*x4^2*z^17 + 2*x1^41*x2^27*x3^15*x4^2*z^17 + 6*x1^40*x2^28*x3^15*x4^2*z^17 + 3*x1^38*x2^30*x3^15*x4^2*z^17 - 4*x1^37*x2^31*x3^15*x4^2*z^17 - x1^36*x2^32*x3^15*x4^2*z^17 - 4*x1^35*x2^33*x3^15*x4^2*z^17 - 6*x1^40*x2^27*x3^16*x4^2*z^17 - 2*x1^39*x2^28*x3^16*x4^2*z^17 - 2*x1^38*x2^29*x3^16*x4^2*z^17 + 2*x1^36*x2^31*x3^16*x4^2*z^17 + 3*x1^35*x2^32*x3^16*x4^2*z^17 + 4*x1^34*x2^33*x3^16*x4^2*z^17 + 2*x1^40*x2^26*x3^17*x4^2*z^17 + 6*x1^39*x2^27*x3^17*x4^2*z^17 + 4*x1^37*x2^29*x3^17*x4^2*z^17 - 4*x1^36*x2^30*x3^17*x4^2*z^17 - 7*x1^34*x2^32*x3^17*x4^2*z^17 - x1^33*x2^33*x3^17*x4^2*z^17 - 6*x1^39*x2^26*x3^18*x4^2*z^17 - 2*x1^38*x2^27*x3^18*x4^2*z^17 - 2*x1^37*x2^28*x3^18*x4^2*z^17 - x1^36*x2^29*x3^18*x4^2*z^17 + 2*x1^35*x2^30*x3^18*x4^2*z^17 + 2*x1^34*x2^31*x3^18*x4^2*z^17 + 4*x1^33*x2^32*x3^18*x4^2*z^17 + x1^39*x2^25*x3^19*x4^2*z^17 + 6*x1^38*x2^26*x3^19*x4^2*z^17 + 4*x1^36*x2^28*x3^19*x4^2*z^17 - x1^35*x2^29*x3^19*x4^2*z^17 + x1^34*x2^30*x3^19*x4^2*z^17 - 5*x1^33*x2^31*x3^19*x4^2*z^17 - x1^32*x2^32*x3^19*x4^2*z^17 - 3*x1^38*x2^25*x3^20*x4^2*z^17 - 4*x1^37*x2^26*x3^20*x4^2*z^17 - 2*x1^36*x2^27*x3^20*x4^2*z^17 - 2*x1^35*x2^28*x3^20*x4^2*z^17 + 2*x1^33*x2^30*x3^20*x4^2*z^17 + 2*x1^32*x2^31*x3^20*x4^2*z^17 + 3*x1^37*x2^25*x3^21*x4^2*z^17 + 2*x1^36*x2^26*x3^21*x4^2*z^17 + 3*x1^35*x2^27*x3^21*x4^2*z^17 + x1^33*x2^29*x3^21*x4^2*z^17 - 3*x1^32*x2^30*x3^21*x4^2*z^17 - 2*x1^35*x2^26*x3^22*x4^2*z^17 - 4*x1^34*x2^27*x3^22*x4^2*z^17 + x1^33*x2^28*x3^22*x4^2*z^17 + x1^32*x2^29*x3^22*x4^2*z^17 + 2*x1^34*x2^26*x3^23*x4^2*z^17 - x1^33*x2^26*x3^24*x4^2*z^17 + x1^43*x2^30*x3^9*x4^3*z^17 + x1^44*x2^28*x3^10*x4^3*z^17 - x1^42*x2^30*x3^10*x4^3*z^17 + x1^41*x2^31*x3^10*x4^3*z^17 - x1^43*x2^28*x3^11*x4^3*z^17 + 2*x1^42*x2^29*x3^11*x4^3*z^17 + x1^41*x2^30*x3^11*x4^3*z^17 + 2*x1^43*x2^27*x3^12*x4^3*z^17 - 2*x1^41*x2^29*x3^12*x4^3*z^17 - x1^40*x2^30*x3^12*x4^3*z^17 - x1^39*x2^31*x3^12*x4^3*z^17 + x1^38*x2^32*x3^12*x4^3*z^17 - x1^43*x2^26*x3^13*x4^3*z^17 - 2*x1^42*x2^27*x3^13*x4^3*z^17 + 2*x1^41*x2^28*x3^13*x4^3*z^17 - x1^40*x2^29*x3^13*x4^3*z^17 + x1^39*x2^30*x3^13*x4^3*z^17 - x1^37*x2^32*x3^13*x4^3*z^17 - x1^36*x2^33*x3^13*x4^3*z^17 + 2*x1^42*x2^26*x3^14*x4^3*z^17 - x1^40*x2^28*x3^14*x4^3*z^17 - 3*x1^38*x2^30*x3^14*x4^3*z^17 - x1^37*x2^31*x3^14*x4^3*z^17 + x1^36*x2^32*x3^14*x4^3*z^17 + 2*x1^35*x2^33*x3^14*x4^3*z^17 - x1^42*x2^25*x3^15*x4^3*z^17 - 2*x1^41*x2^26*x3^15*x4^3*z^17 + 2*x1^40*x2^27*x3^15*x4^3*z^17 + 3*x1^38*x2^29*x3^15*x4^3*z^17 + x1^36*x2^31*x3^15*x4^3*z^17 - 2*x1^35*x2^32*x3^15*x4^3*z^17 - 2*x1^34*x2^33*x3^15*x4^3*z^17 + 2*x1^41*x2^25*x3^16*x4^3*z^17 - x1^40*x2^26*x3^16*x4^3*z^17 - 2*x1^39*x2^27*x3^16*x4^3*z^17 - x1^37*x2^29*x3^16*x4^3*z^17 + 2*x1^36*x2^30*x3^16*x4^3*z^17 - 2*x1^35*x2^31*x3^16*x4^3*z^17 + 2*x1^34*x2^32*x3^16*x4^3*z^17 + x1^33*x2^33*x3^16*x4^3*z^17 - 2*x1^40*x2^25*x3^17*x4^3*z^17 + 2*x1^39*x2^26*x3^17*x4^3*z^17 - 2*x1^38*x2^27*x3^17*x4^3*z^17 + x1^37*x2^28*x3^17*x4^3*z^17 + 2*x1^35*x2^30*x3^17*x4^3*z^17 + x1^34*x2^31*x3^17*x4^3*z^17 - 2*x1^33*x2^32*x3^17*x4^3*z^17 + 2*x1^40*x2^24*x3^18*x4^3*z^17 + x1^39*x2^25*x3^18*x4^3*z^17 - x1^38*x2^26*x3^18*x4^3*z^17 - 2*x1^36*x2^28*x3^18*x4^3*z^17 + 2*x1^35*x2^29*x3^18*x4^3*z^17 + 3*x1^33*x2^31*x3^18*x4^3*z^17 + x1^32*x2^32*x3^18*x4^3*z^17 - x1^40*x2^23*x3^19*x4^3*z^17 - 2*x1^39*x2^24*x3^19*x4^3*z^17 + x1^38*x2^25*x3^19*x4^3*z^17 - x1^35*x2^28*x3^19*x4^3*z^17 + x1^34*x2^29*x3^19*x4^3*z^17 + x1^33*x2^30*x3^19*x4^3*z^17 - 2*x1^32*x2^31*x3^19*x4^3*z^17 + 2*x1^39*x2^23*x3^20*x4^3*z^17 + x1^36*x2^26*x3^20*x4^3*z^17 - 2*x1^35*x2^27*x3^20*x4^3*z^17 - x1^34*x2^28*x3^20*x4^3*z^17 - x1^33*x2^29*x3^20*x4^3*z^17 + 3*x1^32*x2^30*x3^20*x4^3*z^17 + x1^31*x2^31*x3^20*x4^3*z^17 - x1^38*x2^23*x3^21*x4^3*z^17 + x1^36*x2^25*x3^21*x4^3*z^17 + x1^35*x2^26*x3^21*x4^3*z^17 - x1^34*x2^27*x3^21*x4^3*z^17 - x1^33*x2^28*x3^21*x4^3*z^17 - x1^32*x2^29*x3^21*x4^3*z^17 - x1^31*x2^30*x3^21*x4^3*z^17 + x1^35*x2^25*x3^22*x4^3*z^17 + x1^34*x2^26*x3^22*x4^3*z^17 + 2*x1^31*x2^29*x3^22*x4^3*z^17 - x1^35*x2^24*x3^23*x4^3*z^17 + 2*x1^33*x2^26*x3^23*x4^3*z^17 - x1^32*x2^27*x3^23*x4^3*z^17 - x1^31*x2^28*x3^23*x4^3*z^17 + x1^44*x2^27*x3^10*x4^4*z^17 + 2*x1^43*x2^28*x3^10*x4^4*z^17 - x1^41*x2^30*x3^10*x4^4*z^17 - x1^40*x2^31*x3^10*x4^4*z^17 - 4*x1^43*x2^27*x3^11*x4^4*z^17 + x1^39*x2^31*x3^11*x4^4*z^17 + x1^38*x2^32*x3^11*x4^4*z^17 + 2*x1^43*x2^26*x3^12*x4^4*z^17 + 6*x1^42*x2^27*x3^12*x4^4*z^17 + 3*x1^40*x2^29*x3^12*x4^4*z^17 - 3*x1^39*x2^30*x3^12*x4^4*z^17 - x1^38*x2^31*x3^12*x4^4*z^17 - 2*x1^37*x2^32*x3^12*x4^4*z^17 - 6*x1^42*x2^26*x3^13*x4^4*z^17 - 2*x1^41*x2^27*x3^13*x4^4*z^17 + 2*x1^38*x2^30*x3^13*x4^4*z^17 + 2*x1^37*x2^31*x3^13*x4^4*z^17 + 2*x1^36*x2^32*x3^13*x4^4*z^17 + x1^35*x2^33*x3^13*x4^4*z^17 + 2*x1^42*x2^25*x3^14*x4^4*z^17 + 6*x1^41*x2^26*x3^14*x4^4*z^17 + 4*x1^39*x2^28*x3^14*x4^4*z^17 - 2*x1^38*x2^29*x3^14*x4^4*z^17 - x1^37*x2^30*x3^14*x4^4*z^17 - 4*x1^36*x2^31*x3^14*x4^4*z^17 - x1^35*x2^32*x3^14*x4^4*z^17 - x1^34*x2^33*x3^14*x4^4*z^17 - 6*x1^41*x2^25*x3^15*x4^4*z^17 - 2*x1^40*x2^26*x3^15*x4^4*z^17 - 2*x1^39*x2^27*x3^15*x4^4*z^17 + 2*x1^37*x2^29*x3^15*x4^4*z^17 + 4*x1^36*x2^30*x3^15*x4^4*z^17 + 3*x1^35*x2^31*x3^15*x4^4*z^17 + x1^34*x2^32*x3^15*x4^4*z^17 + 2*x1^41*x2^24*x3^16*x4^4*z^17 + 6*x1^40*x2^25*x3^16*x4^4*z^17 + 4*x1^38*x2^27*x3^16*x4^4*z^17 - 4*x1^37*x2^28*x3^16*x4^4*z^17 - 6*x1^35*x2^30*x3^16*x4^4*z^17 - x1^33*x2^32*x3^16*x4^4*z^17 - 6*x1^40*x2^24*x3^17*x4^4*z^17 - 2*x1^39*x2^25*x3^17*x4^4*z^17 - 2*x1^38*x2^26*x3^17*x4^4*z^17 + 2*x1^36*x2^28*x3^17*x4^4*z^17 + 2*x1^35*x2^29*x3^17*x4^4*z^17 + 6*x1^34*x2^30*x3^17*x4^4*z^17 + x1^40*x2^23*x3^18*x4^4*z^17 + 6*x1^39*x2^24*x3^18*x4^4*z^17 + 4*x1^37*x2^26*x3^18*x4^4*z^17 - 4*x1^36*x2^27*x3^18*x4^4*z^17 - 6*x1^34*x2^29*x3^18*x4^4*z^17 - 2*x1^33*x2^30*x3^18*x4^4*z^17 - 4*x1^39*x2^23*x3^19*x4^4*z^17 - 3*x1^38*x2^24*x3^19*x4^4*z^17 - 2*x1^37*x2^25*x3^19*x4^4*z^17 + x1^35*x2^27*x3^19*x4^4*z^17 + 4*x1^33*x2^29*x3^19*x4^4*z^17 - x1^32*x2^30*x3^19*x4^4*z^17 + 4*x1^38*x2^23*x3^20*x4^4*z^17 + x1^37*x2^24*x3^20*x4^4*z^17 + 2*x1^36*x2^25*x3^20*x4^4*z^17 - x1^35*x2^26*x3^20*x4^4*z^17 + x1^34*x2^27*x3^20*x4^4*z^17 - 5*x1^33*x2^28*x3^20*x4^4*z^17 - 2*x1^32*x2^29*x3^20*x4^4*z^17 + 2*x1^31*x2^30*x3^20*x4^4*z^17 - x1^37*x2^23*x3^21*x4^4*z^17 - x1^36*x2^24*x3^21*x4^4*z^17 - 2*x1^35*x2^25*x3^21*x4^4*z^17 - x1^34*x2^26*x3^21*x4^4*z^17 + x1^33*x2^27*x3^21*x4^4*z^17 + x1^32*x2^28*x3^21*x4^4*z^17 - x1^31*x2^29*x3^21*x4^4*z^17 + 2*x1^36*x2^23*x3^22*x4^4*z^17 + 2*x1^35*x2^24*x3^22*x4^4*z^17 + x1^33*x2^26*x3^22*x4^4*z^17 - 3*x1^32*x2^27*x3^22*x4^4*z^17 + x1^30*x2^29*x3^22*x4^4*z^17 - x1^35*x2^23*x3^23*x4^4*z^17 - x1^33*x2^25*x3^23*x4^4*z^17 + 2*x1^32*x2^26*x3^23*x4^4*z^17 - x1^31*x2^27*x3^23*x4^4*z^17 - 2*x1^30*x2^28*x3^23*x4^4*z^17 - x1^31*x2^26*x3^24*x4^4*z^17 + x1^30*x2^27*x3^24*x4^4*z^17 + x1^29*x2^28*x3^24*x4^4*z^17 - x1^43*x2^28*x3^9*x4^5*z^17 + x1^43*x2^27*x3^10*x4^5*z^17 + 2*x1^42*x2^28*x3^10*x4^5*z^17 - 4*x1^42*x2^27*x3^11*x4^5*z^17 + 2*x1^40*x2^29*x3^11*x4^5*z^17 + 2*x1^39*x2^30*x3^11*x4^5*z^17 + x1^38*x2^31*x3^11*x4^5*z^17 - x1^37*x2^32*x3^11*x4^5*z^17 + 3*x1^42*x2^26*x3^12*x4^5*z^17 + 2*x1^41*x2^27*x3^12*x4^5*z^17 - x1^39*x2^29*x3^12*x4^5*z^17 - x1^38*x2^30*x3^12*x4^5*z^17 + x1^36*x2^32*x3^12*x4^5*z^17 - 2*x1^42*x2^25*x3^13*x4^5*z^17 - 5*x1^41*x2^26*x3^13*x4^5*z^17 + x1^40*x2^27*x3^13*x4^5*z^17 - x1^39*x2^28*x3^13*x4^5*z^17 + 3*x1^38*x2^29*x3^13*x4^5*z^17 + 3*x1^36*x2^31*x3^13*x4^5*z^17 - x1^35*x2^32*x3^13*x4^5*z^17 + 6*x1^41*x2^25*x3^14*x4^5*z^17 + 2*x1^40*x2^26*x3^14*x4^5*z^17 + x1^39*x2^27*x3^14*x4^5*z^17 + x1^38*x2^28*x3^14*x4^5*z^17 - 2*x1^37*x2^29*x3^14*x4^5*z^17 - 2*x1^36*x2^30*x3^14*x4^5*z^17 - 3*x1^35*x2^31*x3^14*x4^5*z^17 - 2*x1^41*x2^24*x3^15*x4^5*z^17 - 6*x1^40*x2^25*x3^15*x4^5*z^17 + 2*x1^39*x2^26*x3^15*x4^5*z^17 - 3*x1^38*x2^27*x3^15*x4^5*z^17 + 5*x1^37*x2^28*x3^15*x4^5*z^17 + 6*x1^35*x2^30*x3^15*x4^5*z^17 + x1^34*x2^31*x3^15*x4^5*z^17 + 6*x1^40*x2^24*x3^16*x4^5*z^17 + x1^39*x2^25*x3^16*x4^5*z^17 - 3*x1^36*x2^28*x3^16*x4^5*z^17 - x1^35*x2^29*x3^16*x4^5*z^17 - 6*x1^34*x2^30*x3^16*x4^5*z^17 + 2*x1^33*x2^31*x3^16*x4^5*z^17 + x1^32*x2^32*x3^16*x4^5*z^17 - 6*x1^39*x2^24*x3^17*x4^5*z^17 + 2*x1^38*x2^25*x3^17*x4^5*z^17 - 3*x1^37*x2^26*x3^17*x4^5*z^17 + 5*x1^36*x2^27*x3^17*x4^5*z^17 + 5*x1^34*x2^29*x3^17*x4^5*z^17 + x1^33*x2^30*x3^17*x4^5*z^17 - 2*x1^32*x2^31*x3^17*x4^5*z^17 + 3*x1^39*x2^23*x3^18*x4^5*z^17 + 3*x1^38*x2^24*x3^18*x4^5*z^17 - 3*x1^35*x2^27*x3^18*x4^5*z^17 - 6*x1^33*x2^29*x3^18*x4^5*z^17 + 2*x1^32*x2^30*x3^18*x4^5*z^17 - 3*x1^38*x2^23*x3^19*x4^5*z^17 + x1^37*x2^24*x3^19*x4^5*z^17 - 2*x1^36*x2^25*x3^19*x4^5*z^17 + 4*x1^35*x2^26*x3^19*x4^5*z^17 + 6*x1^33*x2^28*x3^19*x4^5*z^17 + 2*x1^32*x2^29*x3^19*x4^5*z^17 - 2*x1^31*x2^30*x3^19*x4^5*z^17 + x1^38*x2^22*x3^20*x4^5*z^17 + x1^37*x2^23*x3^20*x4^5*z^17 + x1^35*x2^25*x3^20*x4^5*z^17 - 2*x1^34*x2^26*x3^20*x4^5*z^17 - 2*x1^33*x2^27*x3^20*x4^5*z^17 - 5*x1^32*x2^28*x3^20*x4^5*z^17 + x1^31*x2^29*x3^20*x4^5*z^17 - x1^37*x2^22*x3^21*x4^5*z^17 + x1^36*x2^23*x3^21*x4^5*z^17 + 3*x1^34*x2^25*x3^21*x4^5*z^17 - 2*x1^33*x2^26*x3^21*x4^5*z^17 + 5*x1^32*x2^27*x3^21*x4^5*z^17 + x1^31*x2^28*x3^21*x4^5*z^17 - 3*x1^30*x2^29*x3^21*x4^5*z^17 - x1^35*x2^23*x3^22*x4^5*z^17 - x1^34*x2^24*x3^22*x4^5*z^17 - x1^33*x2^25*x3^22*x4^5*z^17 + x1^32*x2^26*x3^22*x4^5*z^17 - 3*x1^31*x2^27*x3^22*x4^5*z^17 + 2*x1^30*x2^28*x3^22*x4^5*z^17 - x1^34*x2^23*x3^23*x4^5*z^17 + x1^33*x2^24*x3^23*x4^5*z^17 - 2*x1^29*x2^28*x3^23*x4^5*z^17 + x1^30*x2^26*x3^24*x4^5*z^17 + x1^29*x2^27*x3^24*x4^5*z^17 - x1^28*x2^27*x3^25*x4^5*z^17 - x1^42*x2^27*x3^10*x4^6*z^17 + x1^41*x2^28*x3^10*x4^6*z^17 - x1^39*x2^30*x3^10*x4^6*z^17 + x1^38*x2^31*x3^10*x4^6*z^17 - x1^41*x2^27*x3^11*x4^6*z^17 + x1^39*x2^29*x3^11*x4^6*z^17 - x1^37*x2^31*x3^11*x4^6*z^17 - x1^42*x2^25*x3^12*x4^6*z^17 - 2*x1^40*x2^27*x3^12*x4^6*z^17 - 2*x1^36*x2^31*x3^12*x4^6*z^17 + x1^35*x2^32*x3^12*x4^6*z^17 - x1^41*x2^25*x3^13*x4^6*z^17 - x1^40*x2^26*x3^13*x4^6*z^17 + 2*x1^39*x2^27*x3^13*x4^6*z^17 + x1^38*x2^28*x3^13*x4^6*z^17 + 3*x1^37*x2^29*x3^13*x4^6*z^17 - x1^36*x2^30*x3^13*x4^6*z^17 + 2*x1^35*x2^31*x3^13*x4^6*z^17 + x1^41*x2^24*x3^14*x4^6*z^17 + x1^40*x2^25*x3^14*x4^6*z^17 - 5*x1^39*x2^26*x3^14*x4^6*z^17 - 3*x1^38*x2^27*x3^14*x4^6*z^17 - 2*x1^37*x2^28*x3^14*x4^6*z^17 + x1^36*x2^29*x3^14*x4^6*z^17 - x1^35*x2^30*x3^14*x4^6*z^17 - x1^34*x2^31*x3^14*x4^6*z^17 - 2*x1^40*x2^24*x3^15*x4^6*z^17 + 2*x1^38*x2^26*x3^15*x4^6*z^17 - x1^37*x2^27*x3^15*x4^6*z^17 + 3*x1^36*x2^28*x3^15*x4^6*z^17 - x1^35*x2^29*x3^15*x4^6*z^17 - 2*x1^33*x2^31*x3^15*x4^6*z^17 - x1^32*x2^32*x3^15*x4^6*z^17 + x1^40*x2^23*x3^16*x4^6*z^17 + 2*x1^39*x2^24*x3^16*x4^6*z^17 - 4*x1^38*x2^25*x3^16*x4^6*z^17 - x1^37*x2^26*x3^16*x4^6*z^17 - 4*x1^36*x2^27*x3^16*x4^6*z^17 + 2*x1^33*x2^30*x3^16*x4^6*z^17 + 2*x1^32*x2^31*x3^16*x4^6*z^17 - 2*x1^39*x2^23*x3^17*x4^6*z^17 + 2*x1^38*x2^24*x3^17*x4^6*z^17 + 4*x1^37*x2^25*x3^17*x4^6*z^17 + 2*x1^35*x2^27*x3^17*x4^6*z^17 - 3*x1^34*x2^28*x3^17*x4^6*z^17 + 2*x1^33*x2^29*x3^17*x4^6*z^17 - 4*x1^32*x2^30*x3^17*x4^6*z^17 - x1^31*x2^31*x3^17*x4^6*z^17 - x1^39*x2^22*x3^18*x4^6*z^17 + 2*x1^38*x2^23*x3^18*x4^6*z^17 - 4*x1^37*x2^24*x3^18*x4^6*z^17 + x1^36*x2^25*x3^18*x4^6*z^17 - 2*x1^35*x2^26*x3^18*x4^6*z^17 - x1^33*x2^28*x3^18*x4^6*z^17 + 4*x1^31*x2^30*x3^18*x4^6*z^17 - x1^37*x2^23*x3^19*x4^6*z^17 + 3*x1^36*x2^24*x3^19*x4^6*z^17 + 4*x1^34*x2^26*x3^19*x4^6*z^17 - 2*x1^33*x2^27*x3^19*x4^6*z^17 + 2*x1^32*x2^28*x3^19*x4^6*z^17 - 4*x1^31*x2^29*x3^19*x4^6*z^17 - x1^30*x2^30*x3^19*x4^6*z^17 - 2*x1^36*x2^23*x3^20*x4^6*z^17 - 3*x1^35*x2^24*x3^20*x4^6*z^17 - 3*x1^34*x2^25*x3^20*x4^6*z^17 - x1^32*x2^27*x3^20*x4^6*z^17 + 4*x1^30*x2^29*x3^20*x4^6*z^17 + x1^36*x2^22*x3^21*x4^6*z^17 + 2*x1^35*x2^23*x3^21*x4^6*z^17 + 2*x1^34*x2^24*x3^21*x4^6*z^17 + 4*x1^33*x2^25*x3^21*x4^6*z^17 + 2*x1^31*x2^27*x3^21*x4^6*z^17 - 3*x1^30*x2^28*x3^21*x4^6*z^17 - x1^29*x2^29*x3^21*x4^6*z^17 - x1^33*x2^24*x3^22*x4^6*z^17 + x1^31*x2^26*x3^22*x4^6*z^17 + 3*x1^30*x2^27*x3^22*x4^6*z^17 + 3*x1^29*x2^28*x3^22*x4^6*z^17 + x1^31*x2^25*x3^23*x4^6*z^17 + 2*x1^30*x2^26*x3^23*x4^6*z^17 - 3*x1^29*x2^27*x3^23*x4^6*z^17 - 2*x1^30*x2^25*x3^24*x4^6*z^17 - x1^29*x2^26*x3^24*x4^6*z^17 + 2*x1^28*x2^27*x3^24*x4^6*z^17 + x1^39*x2^30*x3^9*x4^7*z^17 + x1^38*x2^31*x3^9*x4^7*z^17 + x1^37*x2^32*x3^9*x4^7*z^17 - x1^41*x2^27*x3^10*x4^7*z^17 + x1^39*x2^29*x3^10*x4^7*z^17 - x1^38*x2^30*x3^10*x4^7*z^17 - 2*x1^37*x2^31*x3^10*x4^7*z^17 - 2*x1^36*x2^32*x3^10*x4^7*z^17 - x1^35*x2^33*x3^10*x4^7*z^17 + 2*x1^41*x2^26*x3^11*x4^7*z^17 - 2*x1^40*x2^27*x3^11*x4^7*z^17 + x1^38*x2^29*x3^11*x4^7*z^17 + 2*x1^35*x2^32*x3^11*x4^7*z^17 + x1^34*x2^33*x3^11*x4^7*z^17 - x1^40*x2^26*x3^12*x4^7*z^17 - x1^38*x2^28*x3^12*x4^7*z^17 - x1^37*x2^29*x3^12*x4^7*z^17 + x1^36*x2^30*x3^12*x4^7*z^17 - x1^35*x2^31*x3^12*x4^7*z^17 - 4*x1^34*x2^32*x3^12*x4^7*z^17 + x1^41*x2^24*x3^13*x4^7*z^17 + x1^40*x2^25*x3^13*x4^7*z^17 - x1^39*x2^26*x3^13*x4^7*z^17 + x1^38*x2^27*x3^13*x4^7*z^17 - x1^37*x2^28*x3^13*x4^7*z^17 + 2*x1^36*x2^29*x3^13*x4^7*z^17 - x1^34*x2^31*x3^13*x4^7*z^17 + 3*x1^33*x2^32*x3^13*x4^7*z^17 - x1^38*x2^26*x3^14*x4^7*z^17 - x1^37*x2^27*x3^14*x4^7*z^17 - x1^36*x2^28*x3^14*x4^7*z^17 + x1^34*x2^30*x3^14*x4^7*z^17 - x1^33*x2^31*x3^14*x4^7*z^17 - x1^32*x2^32*x3^14*x4^7*z^17 + x1^40*x2^23*x3^15*x4^7*z^17 + x1^39*x2^24*x3^15*x4^7*z^17 + 2*x1^37*x2^26*x3^15*x4^7*z^17 - x1^36*x2^27*x3^15*x4^7*z^17 + x1^32*x2^31*x3^15*x4^7*z^17 - x1^38*x2^24*x3^16*x4^7*z^17 + x1^39*x2^22*x3^17*x4^7*z^17 + 2*x1^37*x2^24*x3^17*x4^7*z^17 + x1^36*x2^25*x3^17*x4^7*z^17 + x1^35*x2^26*x3^17*x4^7*z^17 - x1^37*x2^23*x3^18*x4^7*z^17 - 2*x1^36*x2^24*x3^18*x4^7*z^17 - x1^34*x2^26*x3^18*x4^7*z^17 + x1^33*x2^27*x3^18*x4^7*z^17 + 3*x1^36*x2^23*x3^19*x4^7*z^17 + x1^35*x2^24*x3^19*x4^7*z^17 + x1^33*x2^26*x3^19*x4^7*z^17 - x1^36*x2^22*x3^20*x4^7*z^17 - 3*x1^35*x2^23*x3^20*x4^7*z^17 - x1^34*x2^24*x3^20*x4^7*z^17 - 3*x1^33*x2^25*x3^20*x4^7*z^17 - x1^32*x2^26*x3^20*x4^7*z^17 + x1^30*x2^28*x3^20*x4^7*z^17 + x1^35*x2^22*x3^21*x4^7*z^17 + 2*x1^34*x2^23*x3^21*x4^7*z^17 + x1^33*x2^24*x3^21*x4^7*z^17 + x1^32*x2^25*x3^21*x4^7*z^17 - x1^30*x2^27*x3^21*x4^7*z^17 - x1^29*x2^28*x3^21*x4^7*z^17 - x1^34*x2^22*x3^22*x4^7*z^17 - 2*x1^33*x2^23*x3^22*x4^7*z^17 - x1^32*x2^24*x3^22*x4^7*z^17 - x1^31*x2^25*x3^22*x4^7*z^17 - x1^30*x2^26*x3^22*x4^7*z^17 + x1^29*x2^27*x3^22*x4^7*z^17 + x1^32*x2^23*x3^23*x4^7*z^17 + 2*x1^31*x2^24*x3^23*x4^7*z^17 - x1^29*x2^26*x3^23*x4^7*z^17 + x1^28*x2^26*x3^24*x4^7*z^17 + x1^28*x2^25*x3^25*x4^7*z^17 - x1^27*x2^26*x3^25*x4^7*z^17 - x1^39*x2^29*x3^9*x4^8*z^17 + x1^36*x2^31*x3^10*x4^8*z^17 - x1^41*x2^25*x3^11*x4^8*z^17 - x1^39*x2^27*x3^11*x4^8*z^17 + x1^37*x2^29*x3^11*x4^8*z^17 + 3*x1^36*x2^30*x3^11*x4^8*z^17 + 2*x1^34*x2^32*x3^11*x4^8*z^17 + x1^39*x2^26*x3^12*x4^8*z^17 - x1^37*x2^28*x3^12*x4^8*z^17 - 3*x1^36*x2^29*x3^12*x4^8*z^17 + x1^34*x2^31*x3^12*x4^8*z^17 - 3*x1^33*x2^32*x3^12*x4^8*z^17 - x1^40*x2^24*x3^13*x4^8*z^17 + x1^39*x2^25*x3^13*x4^8*z^17 - x1^38*x2^26*x3^13*x4^8*z^17 + x1^37*x2^27*x3^13*x4^8*z^17 + x1^36*x2^28*x3^13*x4^8*z^17 - x1^34*x2^30*x3^13*x4^8*z^17 + 4*x1^33*x2^31*x3^13*x4^8*z^17 + x1^32*x2^32*x3^13*x4^8*z^17 + x1^39*x2^24*x3^14*x4^8*z^17 + 2*x1^38*x2^25*x3^14*x4^8*z^17 + 3*x1^37*x2^26*x3^14*x4^8*z^17 + x1^36*x2^27*x3^14*x4^8*z^17 - x1^35*x2^28*x3^14*x4^8*z^17 - 2*x1^34*x2^29*x3^14*x4^8*z^17 - x1^33*x2^30*x3^14*x4^8*z^17 - 4*x1^32*x2^31*x3^14*x4^8*z^17 - x1^39*x2^23*x3^15*x4^8*z^17 - x1^38*x2^24*x3^15*x4^8*z^17 - 3*x1^37*x2^25*x3^15*x4^8*z^17 + x1^36*x2^26*x3^15*x4^8*z^17 + 2*x1^35*x2^27*x3^15*x4^8*z^17 + 4*x1^34*x2^28*x3^15*x4^8*z^17 + 2*x1^33*x2^29*x3^15*x4^8*z^17 + 4*x1^32*x2^30*x3^15*x4^8*z^17 + x1^31*x2^31*x3^15*x4^8*z^17 + 2*x1^38*x2^23*x3^16*x4^8*z^17 + 2*x1^37*x2^24*x3^16*x4^8*z^17 + x1^36*x2^25*x3^16*x4^8*z^17 - x1^34*x2^27*x3^16*x4^8*z^17 - 3*x1^33*x2^28*x3^16*x4^8*z^17 - 2*x1^32*x2^29*x3^16*x4^8*z^17 - 4*x1^31*x2^30*x3^16*x4^8*z^17 - 2*x1^38*x2^22*x3^17*x4^8*z^17 - x1^36*x2^24*x3^17*x4^8*z^17 - x1^35*x2^25*x3^17*x4^8*z^17 + 4*x1^33*x2^27*x3^17*x4^8*z^17 + 2*x1^32*x2^28*x3^17*x4^8*z^17 + 4*x1^31*x2^29*x3^17*x4^8*z^17 + x1^30*x2^30*x3^17*x4^8*z^17 + 3*x1^37*x2^22*x3^18*x4^8*z^17 + 2*x1^35*x2^24*x3^18*x4^8*z^17 - 2*x1^32*x2^27*x3^18*x4^8*z^17 - x1^31*x2^28*x3^18*x4^8*z^17 - 4*x1^30*x2^29*x3^18*x4^8*z^17 - x1^37*x2^21*x3^19*x4^8*z^17 - 2*x1^36*x2^22*x3^19*x4^8*z^17 - x1^35*x2^23*x3^19*x4^8*z^17 - x1^34*x2^24*x3^19*x4^8*z^17 - 2*x1^33*x2^25*x3^19*x4^8*z^17 + x1^31*x2^27*x3^19*x4^8*z^17 + 4*x1^30*x2^28*x3^19*x4^8*z^17 + 2*x1^29*x2^29*x3^19*x4^8*z^17 + x1^36*x2^21*x3^20*x4^8*z^17 + 2*x1^35*x2^22*x3^20*x4^8*z^17 + 2*x1^34*x2^23*x3^20*x4^8*z^17 + x1^33*x2^24*x3^20*x4^8*z^17 + x1^32*x2^25*x3^20*x4^8*z^17 - 3*x1^31*x2^26*x3^20*x4^8*z^17 - 2*x1^30*x2^27*x3^20*x4^8*z^17 - 4*x1^29*x2^28*x3^20*x4^8*z^17 - 2*x1^34*x2^22*x3^21*x4^8*z^17 - x1^33*x2^23*x3^21*x4^8*z^17 + x1^31*x2^25*x3^21*x4^8*z^17 + x1^30*x2^26*x3^21*x4^8*z^17 + x1^28*x2^28*x3^21*x4^8*z^17 + x1^33*x2^22*x3^22*x4^8*z^17 + x1^31*x2^24*x3^22*x4^8*z^17 - x1^30*x2^25*x3^22*x4^8*z^17 + x1^29*x2^26*x3^22*x4^8*z^17 - x1^30*x2^24*x3^23*x4^8*z^17 + 2*x1^29*x2^25*x3^23*x4^8*z^17 - x1^28*x2^26*x3^23*x4^8*z^17 - x1^27*x2^27*x3^23*x4^8*z^17 - x1^27*x2^26*x3^24*x4^8*z^17 + x1^38*x2^29*x3^9*x4^9*z^17 - x1^37*x2^30*x3^9*x4^9*z^17 - x1^38*x2^28*x3^10*x4^9*z^17 - x1^37*x2^29*x3^10*x4^9*z^17 - x1^36*x2^30*x3^10*x4^9*z^17 - x1^35*x2^31*x3^10*x4^9*z^17 + x1^37*x2^28*x3^11*x4^9*z^17 + x1^33*x2^32*x3^11*x4^9*z^17 + x1^40*x2^24*x3^12*x4^9*z^17 + x1^38*x2^26*x3^12*x4^9*z^17 - x1^37*x2^27*x3^12*x4^9*z^17 - x1^36*x2^28*x3^12*x4^9*z^17 - x1^35*x2^29*x3^12*x4^9*z^17 - x1^33*x2^31*x3^12*x4^9*z^17 - 2*x1^39*x2^24*x3^13*x4^9*z^17 - x1^37*x2^26*x3^13*x4^9*z^17 + x1^36*x2^27*x3^13*x4^9*z^17 + 3*x1^35*x2^28*x3^13*x4^9*z^17 + 3*x1^34*x2^29*x3^13*x4^9*z^17 + 3*x1^32*x2^31*x3^13*x4^9*z^17 + x1^39*x2^23*x3^14*x4^9*z^17 + x1^38*x2^24*x3^14*x4^9*z^17 + 2*x1^37*x2^25*x3^14*x4^9*z^17 - 2*x1^36*x2^26*x3^14*x4^9*z^17 - 2*x1^35*x2^27*x3^14*x4^9*z^17 - 3*x1^34*x2^28*x3^14*x4^9*z^17 - 3*x1^33*x2^29*x3^14*x4^9*z^17 - x1^32*x2^30*x3^14*x4^9*z^17 - x1^31*x2^31*x3^14*x4^9*z^17 - 3*x1^38*x2^23*x3^15*x4^9*z^17 - x1^37*x2^24*x3^15*x4^9*z^17 - x1^36*x2^25*x3^15*x4^9*z^17 + x1^35*x2^26*x3^15*x4^9*z^17 + 3*x1^33*x2^28*x3^15*x4^9*z^17 + 2*x1^32*x2^29*x3^15*x4^9*z^17 + 2*x1^31*x2^30*x3^15*x4^9*z^17 + 2*x1^38*x2^22*x3^16*x4^9*z^17 + 2*x1^37*x2^23*x3^16*x4^9*z^17 + 2*x1^36*x2^24*x3^16*x4^9*z^17 - 4*x1^33*x2^27*x3^16*x4^9*z^17 - 4*x1^32*x2^28*x3^16*x4^9*z^17 - 2*x1^31*x2^29*x3^16*x4^9*z^17 - x1^38*x2^21*x3^17*x4^9*z^17 - 4*x1^37*x2^22*x3^17*x4^9*z^17 - x1^36*x2^23*x3^17*x4^9*z^17 - x1^35*x2^24*x3^17*x4^9*z^17 + 3*x1^34*x2^25*x3^17*x4^9*z^17 + x1^33*x2^26*x3^17*x4^9*z^17 + 4*x1^32*x2^27*x3^17*x4^9*z^17 + x1^31*x2^28*x3^17*x4^9*z^17 + 2*x1^30*x2^29*x3^17*x4^9*z^17 + 2*x1^37*x2^21*x3^18*x4^9*z^17 + x1^36*x2^22*x3^18*x4^9*z^17 + x1^35*x2^23*x3^18*x4^9*z^17 - 2*x1^34*x2^24*x3^18*x4^9*z^17 - x1^33*x2^25*x3^18*x4^9*z^17 - x1^32*x2^26*x3^18*x4^9*z^17 - 4*x1^31*x2^27*x3^18*x4^9*z^17 - 2*x1^30*x2^28*x3^18*x4^9*z^17 - x1^29*x2^29*x3^18*x4^9*z^17 - 2*x1^36*x2^21*x3^19*x4^9*z^17 - x1^35*x2^22*x3^19*x4^9*z^17 - 4*x1^34*x2^23*x3^19*x4^9*z^17 + 2*x1^33*x2^24*x3^19*x4^9*z^17 + 4*x1^31*x2^26*x3^19*x4^9*z^17 + x1^30*x2^27*x3^19*x4^9*z^17 + 2*x1^29*x2^28*x3^19*x4^9*z^17 + x1^35*x2^21*x3^20*x4^9*z^17 + x1^34*x2^22*x3^20*x4^9*z^17 - 2*x1^32*x2^24*x3^20*x4^9*z^17 - 2*x1^31*x2^25*x3^20*x4^9*z^17 - 3*x1^30*x2^26*x3^20*x4^9*z^17 - x1^28*x2^28*x3^20*x4^9*z^17 + x1^33*x2^22*x3^21*x4^9*z^17 + 3*x1^32*x2^23*x3^21*x4^9*z^17 - x1^31*x2^24*x3^21*x4^9*z^17 + 4*x1^30*x2^25*x3^21*x4^9*z^17 + x1^29*x2^26*x3^21*x4^9*z^17 + x1^30*x2^24*x3^22*x4^9*z^17 - 4*x1^29*x2^25*x3^22*x4^9*z^17 + x1^27*x2^27*x3^22*x4^9*z^17 - x1^30*x2^23*x3^23*x4^9*z^17 + x1^29*x2^24*x3^23*x4^9*z^17 + x1^28*x2^25*x3^23*x4^9*z^17 - x1^28*x2^24*x3^24*x4^9*z^17 + x1^36*x2^29*x3^10*x4^10*z^17 + x1^35*x2^30*x3^10*x4^10*z^17 + x1^37*x2^27*x3^11*x4^10*z^17 + x1^35*x2^29*x3^11*x4^10*z^17 + x1^36*x2^27*x3^12*x4^10*z^17 - x1^35*x2^28*x3^12*x4^10*z^17 - x1^34*x2^29*x3^12*x4^10*z^17 - 2*x1^32*x2^31*x3^12*x4^10*z^17 + 2*x1^34*x2^28*x3^13*x4^10*z^17 + x1^38*x2^23*x3^14*x4^10*z^17 + x1^35*x2^26*x3^14*x4^10*z^17 - x1^34*x2^27*x3^14*x4^10*z^17 - x1^31*x2^30*x3^14*x4^10*z^17 - 2*x1^36*x2^24*x3^15*x4^10*z^17 + x1^35*x2^25*x3^15*x4^10*z^17 - x1^34*x2^26*x3^15*x4^10*z^17 + x1^33*x2^27*x3^15*x4^10*z^17 + x1^37*x2^22*x3^16*x4^10*z^17 + x1^36*x2^23*x3^16*x4^10*z^17 - 2*x1^34*x2^25*x3^16*x4^10*z^17 + x1^32*x2^27*x3^16*x4^10*z^17 - x1^31*x2^28*x3^16*x4^10*z^17 - x1^30*x2^29*x3^16*x4^10*z^17 + x1^36*x2^22*x3^17*x4^10*z^17 + x1^35*x2^23*x3^17*x4^10*z^17 - x1^33*x2^25*x3^17*x4^10*z^17 + x1^32*x2^26*x3^17*x4^10*z^17 + x1^36*x2^21*x3^18*x4^10*z^17 - x1^35*x2^22*x3^18*x4^10*z^17 + x1^34*x2^23*x3^18*x4^10*z^17 - x1^33*x2^24*x3^18*x4^10*z^17 - x1^32*x2^25*x3^18*x4^10*z^17 - 2*x1^31*x2^26*x3^18*x4^10*z^17 - x1^35*x2^21*x3^19*x4^10*z^17 + x1^34*x2^22*x3^19*x4^10*z^17 + 2*x1^33*x2^23*x3^19*x4^10*z^17 + x1^32*x2^24*x3^19*x4^10*z^17 + 2*x1^31*x2^25*x3^19*x4^10*z^17 + 2*x1^30*x2^26*x3^19*x4^10*z^17 + x1^29*x2^27*x3^19*x4^10*z^17 + x1^34*x2^21*x3^20*x4^10*z^17 - x1^32*x2^23*x3^20*x4^10*z^17 - x1^31*x2^24*x3^20*x4^10*z^17 - 3*x1^30*x2^25*x3^20*x4^10*z^17 - x1^29*x2^26*x3^20*x4^10*z^17 - x1^28*x2^27*x3^20*x4^10*z^17 - x1^33*x2^21*x3^21*x4^10*z^17 + x1^31*x2^23*x3^21*x4^10*z^17 + 2*x1^30*x2^24*x3^21*x4^10*z^17 + 2*x1^29*x2^25*x3^21*x4^10*z^17 + x1^28*x2^26*x3^21*x4^10*z^17 - x1^31*x2^22*x3^22*x4^10*z^17 - 2*x1^29*x2^24*x3^22*x4^10*z^17 - x1^27*x2^26*x3^22*x4^10*z^17 - x1^29*x2^23*x3^23*x4^10*z^17 + 2*x1^28*x2^24*x3^23*x4^10*z^17 - x1^27*x2^24*x3^24*x4^10*z^17 - x1^36*x2^27*x3^11*x4^11*z^17 + x1^35*x2^28*x3^11*x4^11*z^17 - x1^34*x2^29*x3^11*x4^11*z^17 - x1^36*x2^26*x3^12*x4^11*z^17 + x1^35*x2^27*x3^12*x4^11*z^17 - 2*x1^35*x2^26*x3^13*x4^11*z^17 + 2*x1^34*x2^27*x3^13*x4^11*z^17 + x1^33*x2^28*x3^13*x4^11*z^17 - x1^32*x2^29*x3^13*x4^11*z^17 + x1^31*x2^30*x3^13*x4^11*z^17 - x1^38*x2^22*x3^14*x4^11*z^17 - x1^35*x2^25*x3^14*x4^11*z^17 + 2*x1^34*x2^26*x3^14*x4^11*z^17 - x1^33*x2^27*x3^14*x4^11*z^17 + 2*x1^32*x2^28*x3^14*x4^11*z^17 - x1^31*x2^29*x3^14*x4^11*z^17 - 2*x1^30*x2^30*x3^14*x4^11*z^17 + x1^36*x2^23*x3^15*x4^11*z^17 - x1^35*x2^24*x3^15*x4^11*z^17 - 4*x1^34*x2^25*x3^15*x4^11*z^17 + x1^32*x2^27*x3^15*x4^11*z^17 + x1^31*x2^28*x3^15*x4^11*z^17 + x1^30*x2^29*x3^15*x4^11*z^17 + x1^36*x2^22*x3^16*x4^11*z^17 + 2*x1^35*x2^23*x3^16*x4^11*z^17 + x1^34*x2^24*x3^16*x4^11*z^17 + x1^33*x2^25*x3^16*x4^11*z^17 - 2*x1^32*x2^26*x3^16*x4^11*z^17 + 4*x1^31*x2^27*x3^16*x4^11*z^17 - x1^30*x2^28*x3^16*x4^11*z^17 - x1^29*x2^29*x3^16*x4^11*z^17 + x1^34*x2^23*x3^17*x4^11*z^17 + 2*x1^32*x2^25*x3^17*x4^11*z^17 - 4*x1^31*x2^26*x3^17*x4^11*z^17 + x1^30*x2^27*x3^17*x4^11*z^17 + 3*x1^29*x2^28*x3^17*x4^11*z^17 + 2*x1^34*x2^22*x3^18*x4^11*z^17 + x1^33*x2^23*x3^18*x4^11*z^17 + x1^32*x2^24*x3^18*x4^11*z^17 - x1^31*x2^25*x3^18*x4^11*z^17 + 5*x1^30*x2^26*x3^18*x4^11*z^17 - 2*x1^29*x2^27*x3^18*x4^11*z^17 - x1^33*x2^22*x3^19*x4^11*z^17 - x1^32*x2^23*x3^19*x4^11*z^17 - x1^30*x2^25*x3^19*x4^11*z^17 - 3*x1^29*x2^26*x3^19*x4^11*z^17 + 2*x1^28*x2^27*x3^19*x4^11*z^17 + x1^31*x2^23*x3^20*x4^11*z^17 + x1^29*x2^25*x3^20*x4^11*z^17 - 2*x1^28*x2^26*x3^20*x4^11*z^17 - x1^27*x2^27*x3^20*x4^11*z^17 + x1^29*x2^24*x3^21*x4^11*z^17 + x1^28*x2^25*x3^21*x4^11*z^17 + 2*x1^27*x2^26*x3^21*x4^11*z^17 + x1^35*x2^26*x3^12*x4^12*z^17 - x1^34*x2^27*x3^12*x4^12*z^17 + x1^33*x2^28*x3^12*x4^12*z^17 + x1^32*x2^29*x3^12*x4^12*z^17 - 2*x1^34*x2^26*x3^13*x4^12*z^17 + x1^33*x2^27*x3^13*x4^12*z^17 - x1^32*x2^28*x3^13*x4^12*z^17 + x1^31*x2^29*x3^13*x4^12*z^17 + x1^30*x2^30*x3^13*x4^12*z^17 + x1^34*x2^25*x3^14*x4^12*z^17 - 2*x1^33*x2^26*x3^14*x4^12*z^17 - x1^32*x2^27*x3^14*x4^12*z^17 + x1^31*x2^28*x3^14*x4^12*z^17 - 2*x1^30*x2^29*x3^14*x4^12*z^17 + x1^34*x2^24*x3^15*x4^12*z^17 - x1^33*x2^25*x3^15*x4^12*z^17 + 3*x1^32*x2^26*x3^15*x4^12*z^17 + 2*x1^30*x2^28*x3^15*x4^12*z^17 + 2*x1^29*x2^29*x3^15*x4^12*z^17 - x1^36*x2^21*x3^16*x4^12*z^17 + 2*x1^34*x2^23*x3^16*x4^12*z^17 + x1^33*x2^24*x3^16*x4^12*z^17 - x1^32*x2^25*x3^16*x4^12*z^17 - x1^30*x2^27*x3^16*x4^12*z^17 - 4*x1^29*x2^28*x3^16*x4^12*z^17 - 2*x1^34*x2^22*x3^17*x4^12*z^17 - x1^33*x2^23*x3^17*x4^12*z^17 - x1^32*x2^24*x3^17*x4^12*z^17 + 4*x1^31*x2^25*x3^17*x4^12*z^17 - 2*x1^30*x2^26*x3^17*x4^12*z^17 + 5*x1^29*x2^27*x3^17*x4^12*z^17 + 2*x1^28*x2^28*x3^17*x4^12*z^17 + 2*x1^33*x2^22*x3^18*x4^12*z^17 + x1^32*x2^23*x3^18*x4^12*z^17 - 3*x1^31*x2^24*x3^18*x4^12*z^17 - x1^30*x2^25*x3^18*x4^12*z^17 - 6*x1^28*x2^27*x3^18*x4^12*z^17 + x1^30*x2^24*x3^19*x4^12*z^17 - x1^29*x2^25*x3^19*x4^12*z^17 + 5*x1^28*x2^26*x3^19*x4^12*z^17 + 2*x1^27*x2^27*x3^19*x4^12*z^17 - x1^28*x2^25*x3^20*x4^12*z^17 - 5*x1^27*x2^26*x3^20*x4^12*z^17 - 2*x1^28*x2^24*x3^21*x4^12*z^17 + 2*x1^27*x2^25*x3^21*x4^12*z^17 + x1^26*x2^26*x3^21*x4^12*z^17 - 2*x1^26*x2^25*x3^22*x4^12*z^17 + x1^34*x2^24*x3^14*x4^13*z^17 + x1^33*x2^25*x3^14*x4^13*z^17 - x1^32*x2^26*x3^14*x4^13*z^17 + x1^31*x2^27*x3^14*x4^13*z^17 + x1^32*x2^25*x3^15*x4^13*z^17 - x1^31*x2^26*x3^15*x4^13*z^17 - x1^30*x2^27*x3^15*x4^13*z^17 + x1^29*x2^28*x3^15*x4^13*z^17 - 3*x1^31*x2^25*x3^16*x4^13*z^17 + x1^30*x2^26*x3^16*x4^13*z^17 - 2*x1^29*x2^27*x3^16*x4^13*z^17 - x1^28*x2^28*x3^16*x4^13*z^17 - x1^34*x2^21*x3^17*x4^13*z^17 - x1^33*x2^22*x3^17*x4^13*z^17 + x1^32*x2^23*x3^17*x4^13*z^17 + 2*x1^31*x2^24*x3^17*x4^13*z^17 - x1^30*x2^25*x3^17*x4^13*z^17 + 4*x1^28*x2^27*x3^17*x4^13*z^17 - x1^32*x2^22*x3^18*x4^13*z^17 - 2*x1^30*x2^24*x3^18*x4^13*z^17 + 2*x1^29*x2^25*x3^18*x4^13*z^17 - 3*x1^28*x2^26*x3^18*x4^13*z^17 - 2*x1^27*x2^27*x3^18*x4^13*z^17 - x1^31*x2^22*x3^19*x4^13*z^17 + 2*x1^30*x2^23*x3^19*x4^13*z^17 - 2*x1^28*x2^25*x3^19*x4^13*z^17 + 4*x1^27*x2^26*x3^19*x4^13*z^17 + x1^28*x2^24*x3^20*x4^13*z^17 - x1^27*x2^25*x3^20*x4^13*z^17 - 2*x1^26*x2^26*x3^20*x4^13*z^17 + x1^26*x2^25*x3^21*x4^13*z^17 - x1^33*x2^23*x3^15*x4^14*z^17 - 2*x1^30*x2^26*x3^15*x4^14*z^17 + x1^32*x2^23*x3^16*x4^14*z^17 + x1^31*x2^24*x3^16*x4^14*z^17 + x1^30*x2^25*x3^16*x4^14*z^17 + x1^30*x2^24*x3^17*x4^14*z^17 - 3*x1^29*x2^25*x3^17*x4^14*z^17 - x1^30*x2^23*x3^18*x4^14*z^17 + 2*x1^29*x2^24*x3^18*x4^14*z^17 - 2*x1^27*x2^26*x3^18*x4^14*z^17 - x1^30*x2^22*x3^19*x4^14*z^17 - x1^29*x2^23*x3^19*x4^14*z^17 - 2*x1^28*x2^24*x3^19*x4^14*z^17 + x1^27*x2^24*x3^20*x4^14*z^17 + 2*x1^29*x2^25*x3^16*x4^15*z^17 - x1^31*x2^22*x3^17*x4^15*z^17 - x1^30*x2^23*x3^17*x4^15*z^17 - x1^29*x2^24*x3^17*x4^15*z^17 - x1^28*x2^25*x3^17*x4^15*z^17 - x1^29*x2^23*x3^18*x4^15*z^17 + 3*x1^28*x2^24*x3^18*x4^15*z^17 + x1^29*x2^22*x3^19*x4^15*z^17 - 2*x1^27*x2^24*x3^19*x4^15*z^17 + x1^27*x2^23*x3^20*x4^15*z^17 + x1^27*x2^24*x3^18*x4^16*z^17 + x1^26*x2^23*x3^20*x4^16*z^17 + x1^42*x2^28*x3^10*z^16 - 2*x1^41*x2^28*x3^11*z^16 + x1^40*x2^29*x3^11*z^16 + 2*x1^40*x2^28*x3^12*z^16 + x1^38*x2^30*x3^12*z^16 - 2*x1^40*x2^27*x3^13*z^16 - x1^39*x2^28*x3^13*z^16 - x1^38*x2^29*x3^13*z^16 + x1^36*x2^31*x3^13*z^16 + x1^40*x2^26*x3^14*z^16 + 2*x1^39*x2^27*x3^14*z^16 + x1^37*x2^29*x3^14*z^16 - x1^35*x2^31*x3^14*z^16 - 2*x1^39*x2^26*x3^15*z^16 - x1^38*x2^27*x3^15*z^16 - x1^37*x2^28*x3^15*z^16 - x1^36*x2^29*x3^15*z^16 + x1^34*x2^31*x3^15*z^16 + x1^39*x2^25*x3^16*z^16 + 2*x1^38*x2^26*x3^16*z^16 + x1^36*x2^28*x3^16*z^16 - x1^35*x2^29*x3^16*z^16 + x1^34*x2^30*x3^16*z^16 - x1^33*x2^31*x3^16*z^16 - 2*x1^38*x2^25*x3^17*z^16 - x1^35*x2^28*x3^17*z^16 - x1^34*x2^29*x3^17*z^16 + x1^33*x2^30*x3^17*z^16 + x1^32*x2^31*x3^17*z^16 + x1^37*x2^25*x3^18*z^16 + x1^36*x2^26*x3^18*z^16 + x1^35*x2^27*x3^18*z^16 - 2*x1^37*x2^24*x3^19*z^16 - x1^34*x2^27*x3^19*z^16 + 2*x1^34*x2^26*x3^20*z^16 + x1^42*x2^29*x3^8*x4*z^16 - x1^41*x2^29*x3^9*x4*z^16 + 3*x1^41*x2^28*x3^10*x4*z^16 + x1^39*x2^30*x3^10*x4*z^16 - 2*x1^41*x2^27*x3^11*x4*z^16 - 3*x1^40*x2^28*x3^11*x4*z^16 + x1^39*x2^29*x3^11*x4*z^16 - 2*x1^38*x2^30*x3^11*x4*z^16 - x1^37*x2^31*x3^11*x4*z^16 + 6*x1^40*x2^27*x3^12*x4*z^16 + x1^38*x2^29*x3^12*x4*z^16 - x1^35*x2^32*x3^12*x4*z^16 - 2*x1^40*x2^26*x3^13*x4*z^16 - 6*x1^39*x2^27*x3^13*x4*z^16 - 4*x1^37*x2^29*x3^13*x4*z^16 + x1^36*x2^30*x3^13*x4*z^16 + x1^34*x2^32*x3^13*x4*z^16 + 6*x1^39*x2^26*x3^14*x4*z^16 + 2*x1^38*x2^27*x3^14*x4*z^16 + 2*x1^37*x2^28*x3^14*x4*z^16 - x1^35*x2^30*x3^14*x4*z^16 - x1^34*x2^31*x3^14*x4*z^16 - x1^33*x2^32*x3^14*x4*z^16 - 2*x1^39*x2^25*x3^15*x4*z^16 - 6*x1^38*x2^26*x3^15*x4*z^16 - 4*x1^36*x2^28*x3^15*x4*z^16 + 4*x1^35*x2^29*x3^15*x4*z^16 + x1^34*x2^30*x3^15*x4*z^16 + 4*x1^33*x2^31*x3^15*x4*z^16 + x1^32*x2^32*x3^15*x4*z^16 + 6*x1^38*x2^25*x3^16*x4*z^16 + 2*x1^37*x2^26*x3^16*x4*z^16 + 2*x1^36*x2^27*x3^16*x4*z^16 + x1^35*x2^28*x3^16*x4*z^16 - 2*x1^34*x2^29*x3^16*x4*z^16 - 2*x1^33*x2^30*x3^16*x4*z^16 - 3*x1^32*x2^31*x3^16*x4*z^16 - 2*x1^38*x2^24*x3^17*x4*z^16 - 6*x1^37*x2^25*x3^17*x4*z^16 - 4*x1^35*x2^27*x3^17*x4*z^16 + x1^34*x2^28*x3^17*x4*z^16 - x1^33*x2^29*x3^17*x4*z^16 + 5*x1^32*x2^30*x3^17*x4*z^16 + x1^31*x2^31*x3^17*x4*z^16 + 6*x1^37*x2^24*x3^18*x4*z^16 + 2*x1^36*x2^25*x3^18*x4*z^16 + 2*x1^35*x2^26*x3^18*x4*z^16 + 2*x1^34*x2^27*x3^18*x4*z^16 - 2*x1^32*x2^29*x3^18*x4*z^16 - 2*x1^31*x2^30*x3^18*x4*z^16 - x1^37*x2^23*x3^19*x4*z^16 - 5*x1^36*x2^24*x3^19*x4*z^16 - 2*x1^35*x2^25*x3^19*x4*z^16 - 3*x1^34*x2^26*x3^19*x4*z^16 - x1^32*x2^28*x3^19*x4*z^16 + 3*x1^31*x2^29*x3^19*x4*z^16 + 3*x1^36*x2^23*x3^20*x4*z^16 + x1^35*x2^24*x3^20*x4*z^16 + 2*x1^34*x2^25*x3^20*x4*z^16 + 3*x1^33*x2^26*x3^20*x4*z^16 - x1^32*x2^27*x3^20*x4*z^16 - x1^31*x2^28*x3^20*x4*z^16 - x1^35*x2^23*x3^21*x4*z^16 - 3*x1^33*x2^25*x3^21*x4*z^16 + x1^33*x2^24*x3^22*x4*z^16 + 2*x1^32*x2^25*x3^22*x4*z^16 - 2*x1^41*x2^28*x3^9*x4^2*z^16 + x1^40*x2^29*x3^9*x4^2*z^16 - x1^39*x2^30*x3^9*x4^2*z^16 + x1^41*x2^27*x3^10*x4^2*z^16 + 4*x1^40*x2^28*x3^10*x4^2*z^16 - 2*x1^39*x2^29*x3^10*x4^2*z^16 + x1^38*x2^30*x3^10*x4^2*z^16 - x1^37*x2^31*x3^10*x4^2*z^16 - 6*x1^40*x2^27*x3^11*x4^2*z^16 - x1^39*x2^28*x3^11*x4^2*z^16 + x1^37*x2^30*x3^11*x4^2*z^16 + 2*x1^36*x2^31*x3^11*x4^2*z^16 + 2*x1^40*x2^26*x3^12*x4^2*z^16 + 6*x1^39*x2^27*x3^12*x4^2*z^16 - x1^38*x2^28*x3^12*x4^2*z^16 + 2*x1^37*x2^29*x3^12*x4^2*z^16 - 3*x1^36*x2^30*x3^12*x4^2*z^16 - 2*x1^35*x2^31*x3^12*x4^2*z^16 - x1^34*x2^32*x3^12*x4^2*z^16 - 6*x1^39*x2^26*x3^13*x4^2*z^16 - 2*x1^38*x2^27*x3^13*x4^2*z^16 - x1^37*x2^28*x3^13*x4^2*z^16 + x1^36*x2^29*x3^13*x4^2*z^16 + 3*x1^35*x2^30*x3^13*x4^2*z^16 + 3*x1^34*x2^31*x3^13*x4^2*z^16 + x1^33*x2^32*x3^13*x4^2*z^16 + 2*x1^39*x2^25*x3^14*x4^2*z^16 + 6*x1^38*x2^26*x3^14*x4^2*z^16 + 4*x1^36*x2^28*x3^14*x4^2*z^16 - 4*x1^35*x2^29*x3^14*x4^2*z^16 - 2*x1^34*x2^30*x3^14*x4^2*z^16 - 5*x1^33*x2^31*x3^14*x4^2*z^16 - 6*x1^38*x2^25*x3^15*x4^2*z^16 - 2*x1^37*x2^26*x3^15*x4^2*z^16 - 2*x1^36*x2^27*x3^15*x4^2*z^16 + 2*x1^34*x2^29*x3^15*x4^2*z^16 + 3*x1^33*x2^30*x3^15*x4^2*z^16 + 5*x1^32*x2^31*x3^15*x4^2*z^16 + 2*x1^38*x2^24*x3^16*x4^2*z^16 + 6*x1^37*x2^25*x3^16*x4^2*z^16 + 4*x1^35*x2^27*x3^16*x4^2*z^16 - 4*x1^34*x2^28*x3^16*x4^2*z^16 - 6*x1^32*x2^30*x3^16*x4^2*z^16 - 2*x1^31*x2^31*x3^16*x4^2*z^16 - 6*x1^37*x2^24*x3^17*x4^2*z^16 - 2*x1^36*x2^25*x3^17*x4^2*z^16 - 2*x1^35*x2^26*x3^17*x4^2*z^16 + 2*x1^33*x2^28*x3^17*x4^2*z^16 + 2*x1^32*x2^29*x3^17*x4^2*z^16 + 6*x1^31*x2^30*x3^17*x4^2*z^16 + 6*x1^36*x2^24*x3^18*x4^2*z^16 + 4*x1^34*x2^26*x3^18*x4^2*z^16 - 2*x1^33*x2^27*x3^18*x4^2*z^16 - 6*x1^31*x2^29*x3^18*x4^2*z^16 - x1^30*x2^30*x3^18*x4^2*z^16 - 2*x1^36*x2^23*x3^19*x4^2*z^16 - 2*x1^35*x2^24*x3^19*x4^2*z^16 - x1^34*x2^25*x3^19*x4^2*z^16 - x1^33*x2^26*x3^19*x4^2*z^16 + x1^32*x2^27*x3^19*x4^2*z^16 + x1^31*x2^28*x3^19*x4^2*z^16 + 3*x1^30*x2^29*x3^19*x4^2*z^16 + x1^35*x2^23*x3^20*x4^2*z^16 + 2*x1^34*x2^24*x3^20*x4^2*z^16 + 4*x1^33*x2^25*x3^20*x4^2*z^16 + x1^32*x2^26*x3^20*x4^2*z^16 - 4*x1^30*x2^28*x3^20*x4^2*z^16 - x1^33*x2^24*x3^21*x4^2*z^16 - 2*x1^32*x2^25*x3^21*x4^2*z^16 + x1^30*x2^27*x3^21*x4^2*z^16 + x1^29*x2^28*x3^21*x4^2*z^16 + x1^31*x2^25*x3^22*x4^2*z^16 - 2*x1^29*x2^27*x3^22*x4^2*z^16 - x1^31*x2^24*x3^23*x4^2*z^16 + x1^42*x2^26*x3^9*x4^3*z^16 - 2*x1^40*x2^28*x3^9*x4^3*z^16 - x1^38*x2^30*x3^9*x4^3*z^16 - x1^42*x2^25*x3^10*x4^3*z^16 - x1^41*x2^26*x3^10*x4^3*z^16 + x1^39*x2^28*x3^10*x4^3*z^16 + x1^38*x2^29*x3^10*x4^3*z^16 - x1^36*x2^31*x3^10*x4^3*z^16 + 2*x1^41*x2^25*x3^11*x4^3*z^16 - 2*x1^39*x2^27*x3^11*x4^3*z^16 - x1^38*x2^28*x3^11*x4^3*z^16 - x1^37*x2^29*x3^11*x4^3*z^16 + x1^35*x2^31*x3^11*x4^3*z^16 - 2*x1^40*x2^25*x3^12*x4^3*z^16 + 2*x1^39*x2^26*x3^12*x4^3*z^16 - x1^38*x2^27*x3^12*x4^3*z^16 + 2*x1^37*x2^28*x3^12*x4^3*z^16 + 2*x1^36*x2^29*x3^12*x4^3*z^16 + x1^35*x2^30*x3^12*x4^3*z^16 - x1^34*x2^31*x3^12*x4^3*z^16 + 2*x1^40*x2^24*x3^13*x4^3*z^16 + x1^39*x2^25*x3^13*x4^3*z^16 - x1^38*x2^26*x3^13*x4^3*z^16 - 2*x1^36*x2^28*x3^13*x4^3*z^16 - x1^34*x2^30*x3^13*x4^3*z^16 + 2*x1^33*x2^31*x3^13*x4^3*z^16 - x1^40*x2^23*x3^14*x4^3*z^16 - 2*x1^39*x2^24*x3^14*x4^3*z^16 + 2*x1^38*x2^25*x3^14*x4^3*z^16 + 2*x1^36*x2^27*x3^14*x4^3*z^16 + 2*x1^34*x2^29*x3^14*x4^3*z^16 + x1^33*x2^30*x3^14*x4^3*z^16 - 2*x1^32*x2^31*x3^14*x4^3*z^16 + 2*x1^39*x2^23*x3^15*x4^3*z^16 - x1^37*x2^25*x3^15*x4^3*z^16 - 2*x1^35*x2^27*x3^15*x4^3*z^16 - 2*x1^33*x2^29*x3^15*x4^3*z^16 + 2*x1^32*x2^30*x3^15*x4^3*z^16 + x1^31*x2^31*x3^15*x4^3*z^16 - x1^39*x2^22*x3^16*x4^3*z^16 - 2*x1^38*x2^23*x3^16*x4^3*z^16 + 2*x1^37*x2^24*x3^16*x4^3*z^16 + 3*x1^35*x2^26*x3^16*x4^3*z^16 + x1^33*x2^28*x3^16*x4^3*z^16 - x1^32*x2^29*x3^16*x4^3*z^16 - 2*x1^31*x2^30*x3^16*x4^3*z^16 + 2*x1^38*x2^22*x3^17*x4^3*z^16 - 2*x1^36*x2^24*x3^17*x4^3*z^16 - x1^34*x2^26*x3^17*x4^3*z^16 + 2*x1^33*x2^27*x3^17*x4^3*z^16 - 2*x1^32*x2^28*x3^17*x4^3*z^16 + 2*x1^31*x2^29*x3^17*x4^3*z^16 - 2*x1^37*x2^22*x3^18*x4^3*z^16 + x1^36*x2^23*x3^18*x4^3*z^16 - 2*x1^35*x2^24*x3^18*x4^3*z^16 + x1^32*x2^27*x3^18*x4^3*z^16 - 4*x1^30*x2^29*x3^18*x4^3*z^16 + x1^37*x2^21*x3^19*x4^3*z^16 + 2*x1^36*x2^22*x3^19*x4^3*z^16 - x1^33*x2^25*x3^19*x4^3*z^16 + x1^32*x2^26*x3^19*x4^3*z^16 + x1^31*x2^27*x3^19*x4^3*z^16 + 2*x1^30*x2^28*x3^19*x4^3*z^16 - x1^36*x2^21*x3^20*x4^3*z^16 - x1^35*x2^22*x3^20*x4^3*z^16 - 2*x1^32*x2^25*x3^20*x4^3*z^16 + x1^31*x2^26*x3^20*x4^3*z^16 + x1^30*x2^27*x3^20*x4^3*z^16 - 2*x1^29*x2^28*x3^20*x4^3*z^16 + x1^34*x2^22*x3^21*x4^3*z^16 - x1^32*x2^24*x3^21*x4^3*z^16 + 2*x1^30*x2^26*x3^21*x4^3*z^16 + 3*x1^29*x2^27*x3^21*x4^3*z^16 + x1^32*x2^23*x3^22*x4^3*z^16 - x1^28*x2^27*x3^22*x4^3*z^16 + x1^31*x2^23*x3^23*x4^3*z^16 + x1^28*x2^26*x3^23*x4^3*z^16 - x1^42*x2^26*x3^8*x4^4*z^16 + x1^39*x2^29*x3^8*x4^4*z^16 + 2*x1^41*x2^26*x3^9*x4^4*z^16 + x1^37*x2^30*x3^9*x4^4*z^16 - 3*x1^41*x2^25*x3^10*x4^4*z^16 - 2*x1^40*x2^26*x3^10*x4^4*z^16 - x1^39*x2^27*x3^10*x4^4*z^16 + x1^36*x2^30*x3^10*x4^4*z^16 + 2*x1^41*x2^24*x3^11*x4^4*z^16 + 5*x1^40*x2^25*x3^11*x4^4*z^16 + x1^39*x2^26*x3^11*x4^4*z^16 - 2*x1^37*x2^28*x3^11*x4^4*z^16 - x1^36*x2^29*x3^11*x4^4*z^16 - x1^35*x2^30*x3^11*x4^4*z^16 - x1^34*x2^31*x3^11*x4^4*z^16 - 6*x1^40*x2^24*x3^12*x4^4*z^16 - 2*x1^39*x2^25*x3^12*x4^4*z^16 - 3*x1^38*x2^26*x3^12*x4^4*z^16 - x1^37*x2^27*x3^12*x4^4*z^16 + x1^35*x2^29*x3^12*x4^4*z^16 + x1^34*x2^30*x3^12*x4^4*z^16 + x1^33*x2^31*x3^12*x4^4*z^16 + 2*x1^40*x2^23*x3^13*x4^4*z^16 + 6*x1^39*x2^24*x3^13*x4^4*z^16 + 4*x1^37*x2^26*x3^13*x4^4*z^16 - 4*x1^36*x2^27*x3^13*x4^4*z^16 - 2*x1^35*x2^28*x3^13*x4^4*z^16 - 4*x1^34*x2^29*x3^13*x4^4*z^16 - x1^33*x2^30*x3^13*x4^4*z^16 - x1^32*x2^31*x3^13*x4^4*z^16 - 6*x1^39*x2^23*x3^14*x4^4*z^16 - 2*x1^38*x2^24*x3^14*x4^4*z^16 - 2*x1^37*x2^25*x3^14*x4^4*z^16 + 2*x1^35*x2^27*x3^14*x4^4*z^16 + 2*x1^34*x2^28*x3^14*x4^4*z^16 + 4*x1^33*x2^29*x3^14*x4^4*z^16 + x1^31*x2^31*x3^14*x4^4*z^16 + 2*x1^39*x2^22*x3^15*x4^4*z^16 + 6*x1^38*x2^23*x3^15*x4^4*z^16 + 4*x1^36*x2^25*x3^15*x4^4*z^16 - 4*x1^35*x2^26*x3^15*x4^4*z^16 - 6*x1^33*x2^28*x3^15*x4^4*z^16 - 2*x1^32*x2^29*x3^15*x4^4*z^16 - 5*x1^38*x2^22*x3^16*x4^4*z^16 - 2*x1^37*x2^23*x3^16*x4^4*z^16 - 2*x1^36*x2^24*x3^16*x4^4*z^16 + 2*x1^34*x2^26*x3^16*x4^4*z^16 + 2*x1^33*x2^27*x3^16*x4^4*z^16 + 6*x1^32*x2^28*x3^16*x4^4*z^16 + x1^38*x2^21*x3^17*x4^4*z^16 + 5*x1^37*x2^22*x3^17*x4^4*z^16 + x1^36*x2^23*x3^17*x4^4*z^16 + 4*x1^35*x2^24*x3^17*x4^4*z^16 - 4*x1^34*x2^25*x3^17*x4^4*z^16 - 6*x1^32*x2^27*x3^17*x4^4*z^16 - 2*x1^31*x2^28*x3^17*x4^4*z^16 - 2*x1^37*x2^21*x3^18*x4^4*z^16 - 2*x1^36*x2^22*x3^18*x4^4*z^16 - 2*x1^35*x2^23*x3^18*x4^4*z^16 + x1^34*x2^24*x3^18*x4^4*z^16 + 2*x1^33*x2^25*x3^18*x4^4*z^16 + 2*x1^32*x2^26*x3^18*x4^4*z^16 + 6*x1^31*x2^27*x3^18*x4^4*z^16 + 2*x1^36*x2^21*x3^19*x4^4*z^16 + 2*x1^35*x2^22*x3^19*x4^4*z^16 + 4*x1^34*x2^23*x3^19*x4^4*z^16 - 3*x1^33*x2^24*x3^19*x4^4*z^16 - 5*x1^31*x2^26*x3^19*x4^4*z^16 - x1^30*x2^27*x3^19*x4^4*z^16 + 2*x1^29*x2^28*x3^19*x4^4*z^16 - x1^35*x2^21*x3^20*x4^4*z^16 - 2*x1^34*x2^22*x3^20*x4^4*z^16 + x1^31*x2^25*x3^20*x4^4*z^16 + 2*x1^30*x2^26*x3^20*x4^4*z^16 - x1^32*x2^23*x3^21*x4^4*z^16 + 2*x1^31*x2^24*x3^21*x4^4*z^16 - 2*x1^30*x2^25*x3^21*x4^4*z^16 + 2*x1^28*x2^27*x3^21*x4^4*z^16 - x1^32*x2^22*x3^22*x4^4*z^16 - x1^31*x2^23*x3^22*x4^4*z^16 + x1^29*x2^25*x3^22*x4^4*z^16 + x1^30*x2^23*x3^23*x4^4*z^16 - x1^29*x2^24*x3^23*x4^4*z^16 + 2*x1^27*x2^26*x3^23*x4^4*z^16 - x1^38*x2^29*x3^8*x4^5*z^16 + x1^41*x2^25*x3^9*x4^5*z^16 + x1^40*x2^26*x3^9*x4^5*z^16 + 2*x1^39*x2^27*x3^9*x4^5*z^16 - 2*x1^38*x2^28*x3^9*x4^5*z^16 - x1^37*x2^29*x3^9*x4^5*z^16 + x1^36*x2^30*x3^9*x4^5*z^16 - 2*x1^40*x2^25*x3^10*x4^5*z^16 - x1^39*x2^26*x3^10*x4^5*z^16 - x1^38*x2^27*x3^10*x4^5*z^16 + 3*x1^37*x2^28*x3^10*x4^5*z^16 + 2*x1^40*x2^24*x3^11*x4^5*z^16 + x1^39*x2^25*x3^11*x4^5*z^16 - x1^37*x2^27*x3^11*x4^5*z^16 - 3*x1^36*x2^28*x3^11*x4^5*z^16 - 2*x1^35*x2^29*x3^11*x4^5*z^16 + x1^33*x2^31*x3^11*x4^5*z^16 - x1^40*x2^23*x3^12*x4^5*z^16 - 4*x1^39*x2^24*x3^12*x4^5*z^16 + x1^38*x2^25*x3^12*x4^5*z^16 + 3*x1^36*x2^27*x3^12*x4^5*z^16 + x1^35*x2^28*x3^12*x4^5*z^16 + 4*x1^34*x2^29*x3^12*x4^5*z^16 - x1^33*x2^30*x3^12*x4^5*z^16 - x1^32*x2^31*x3^12*x4^5*z^16 + 5*x1^39*x2^23*x3^13*x4^5*z^16 + 2*x1^38*x2^24*x3^13*x4^5*z^16 + x1^37*x2^25*x3^13*x4^5*z^16 - x1^36*x2^26*x3^13*x4^5*z^16 - 3*x1^35*x2^27*x3^13*x4^5*z^16 - x1^34*x2^28*x3^13*x4^5*z^16 - 5*x1^33*x2^29*x3^13*x4^5*z^16 + x1^32*x2^30*x3^13*x4^5*z^16 - 2*x1^39*x2^22*x3^14*x4^5*z^16 - 6*x1^38*x2^23*x3^14*x4^5*z^16 + x1^37*x2^24*x3^14*x4^5*z^16 - 3*x1^36*x2^25*x3^14*x4^5*z^16 + 3*x1^35*x2^26*x3^14*x4^5*z^16 - x1^34*x2^27*x3^14*x4^5*z^16 + 6*x1^33*x2^28*x3^14*x4^5*z^16 + x1^32*x2^29*x3^14*x4^5*z^16 - x1^31*x2^30*x3^14*x4^5*z^16 + 6*x1^38*x2^22*x3^15*x4^5*z^16 + 2*x1^37*x2^23*x3^15*x4^5*z^16 - 4*x1^34*x2^26*x3^15*x4^5*z^16 - x1^33*x2^27*x3^15*x4^5*z^16 - 6*x1^32*x2^28*x3^15*x4^5*z^16 + 2*x1^31*x2^29*x3^15*x4^5*z^16 - x1^38*x2^21*x3^16*x4^5*z^16 - 6*x1^37*x2^22*x3^16*x4^5*z^16 + 2*x1^36*x2^23*x3^16*x4^5*z^16 - 3*x1^35*x2^24*x3^16*x4^5*z^16 + 5*x1^34*x2^25*x3^16*x4^5*z^16 + 5*x1^32*x2^27*x3^16*x4^5*z^16 + x1^31*x2^28*x3^16*x4^5*z^16 - 2*x1^30*x2^29*x3^16*x4^5*z^16 + 2*x1^37*x2^21*x3^17*x4^5*z^16 - 2*x1^35*x2^23*x3^17*x4^5*z^16 - 3*x1^33*x2^25*x3^17*x4^5*z^16 - x1^32*x2^26*x3^17*x4^5*z^16 - 6*x1^31*x2^27*x3^17*x4^5*z^16 + 2*x1^30*x2^28*x3^17*x4^5*z^16 + x1^29*x2^29*x3^17*x4^5*z^16 - 2*x1^36*x2^21*x3^18*x4^5*z^16 + x1^35*x2^22*x3^18*x4^5*z^16 - 3*x1^34*x2^23*x3^18*x4^5*z^16 + 3*x1^33*x2^24*x3^18*x4^5*z^16 + 5*x1^31*x2^26*x3^18*x4^5*z^16 + x1^30*x2^27*x3^18*x4^5*z^16 - 2*x1^29*x2^28*x3^18*x4^5*z^16 + x1^35*x2^21*x3^19*x4^5*z^16 - x1^34*x2^22*x3^19*x4^5*z^16 - 2*x1^32*x2^24*x3^19*x4^5*z^16 - 6*x1^30*x2^26*x3^19*x4^5*z^16 + 2*x1^29*x2^27*x3^19*x4^5*z^16 - x1^34*x2^21*x3^20*x4^5*z^16 + x1^33*x2^22*x3^20*x4^5*z^16 + 2*x1^32*x2^23*x3^20*x4^5*z^16 - x1^31*x2^24*x3^20*x4^5*z^16 + 4*x1^30*x2^25*x3^20*x4^5*z^16 + x1^29*x2^26*x3^20*x4^5*z^16 - x1^28*x2^27*x3^20*x4^5*z^16 + x1^33*x2^21*x3^21*x4^5*z^16 - x1^32*x2^22*x3^21*x4^5*z^16 - x1^31*x2^23*x3^21*x4^5*z^16 - 2*x1^29*x2^25*x3^21*x4^5*z^16 + x1^28*x2^26*x3^21*x4^5*z^16 + x1^27*x2^27*x3^21*x4^5*z^16 + 2*x1^31*x2^22*x3^22*x4^5*z^16 + x1^29*x2^24*x3^22*x4^5*z^16 - x1^28*x2^25*x3^22*x4^5*z^16 - 2*x1^27*x2^26*x3^22*x4^5*z^16 + x1^29*x2^23*x3^23*x4^5*z^16 - x1^28*x2^24*x3^23*x4^5*z^16 + x1^26*x2^26*x3^23*x4^5*z^16 - x1^26*x2^25*x3^24*x4^5*z^16 - x1^40*x2^25*x3^9*x4^6*z^16 - x1^38*x2^27*x3^9*x4^6*z^16 - x1^37*x2^28*x3^9*x4^6*z^16 - x1^35*x2^30*x3^9*x4^6*z^16 - x1^40*x2^24*x3^10*x4^6*z^16 + x1^39*x2^25*x3^10*x4^6*z^16 + x1^38*x2^26*x3^10*x4^6*z^16 - x1^37*x2^27*x3^10*x4^6*z^16 + 2*x1^36*x2^28*x3^10*x4^6*z^16 - x1^39*x2^24*x3^11*x4^6*z^16 - x1^38*x2^25*x3^11*x4^6*z^16 - 2*x1^36*x2^27*x3^11*x4^6*z^16 - 2*x1^35*x2^28*x3^11*x4^6*z^16 + x1^34*x2^29*x3^11*x4^6*z^16 + x1^38*x2^24*x3^12*x4^6*z^16 + 3*x1^37*x2^25*x3^12*x4^6*z^16 + 2*x1^36*x2^26*x3^12*x4^6*z^16 - 2*x1^34*x2^28*x3^12*x4^6*z^16 + x1^33*x2^29*x3^12*x4^6*z^16 + x1^32*x2^30*x3^12*x4^6*z^16 - x1^39*x2^22*x3^13*x4^6*z^16 + x1^38*x2^23*x3^13*x4^6*z^16 - 3*x1^37*x2^24*x3^13*x4^6*z^16 - 2*x1^33*x2^28*x3^13*x4^6*z^16 + x1^32*x2^29*x3^13*x4^6*z^16 - x1^38*x2^22*x3^14*x4^6*z^16 - x1^37*x2^23*x3^14*x4^6*z^16 + 3*x1^36*x2^24*x3^14*x4^6*z^16 + 2*x1^35*x2^25*x3^14*x4^6*z^16 + 5*x1^34*x2^26*x3^14*x4^6*z^16 - 2*x1^33*x2^27*x3^14*x4^6*z^16 + 2*x1^32*x2^28*x3^14*x4^6*z^16 - 4*x1^31*x2^29*x3^14*x4^6*z^16 + x1^30*x2^30*x3^14*x4^6*z^16 + x1^38*x2^21*x3^15*x4^6*z^16 + 2*x1^37*x2^22*x3^15*x4^6*z^16 - 3*x1^36*x2^23*x3^15*x4^6*z^16 - x1^35*x2^24*x3^15*x4^6*z^16 - x1^34*x2^25*x3^15*x4^6*z^16 + x1^33*x2^26*x3^15*x4^6*z^16 - x1^32*x2^27*x3^15*x4^6*z^16 + 4*x1^30*x2^29*x3^15*x4^6*z^16 - x1^37*x2^21*x3^16*x4^6*z^16 + 3*x1^35*x2^23*x3^16*x4^6*z^16 + 4*x1^33*x2^25*x3^16*x4^6*z^16 - x1^32*x2^26*x3^16*x4^6*z^16 + 2*x1^31*x2^27*x3^16*x4^6*z^16 - 4*x1^30*x2^28*x3^16*x4^6*z^16 - 2*x1^29*x2^29*x3^16*x4^6*z^16 + x1^36*x2^21*x3^17*x4^6*z^16 - 3*x1^35*x2^22*x3^17*x4^6*z^16 - x1^34*x2^23*x3^17*x4^6*z^16 - 4*x1^33*x2^24*x3^17*x4^6*z^16 + 2*x1^30*x2^27*x3^17*x4^6*z^16 + 4*x1^29*x2^28*x3^17*x4^6*z^16 + x1^36*x2^20*x3^18*x4^6*z^16 + 4*x1^34*x2^22*x3^18*x4^6*z^16 + x1^33*x2^23*x3^18*x4^6*z^16 + 2*x1^32*x2^24*x3^18*x4^6*z^16 - 3*x1^31*x2^25*x3^18*x4^6*z^16 + 2*x1^30*x2^26*x3^18*x4^6*z^16 - 4*x1^29*x2^27*x3^18*x4^6*z^16 - x1^28*x2^28*x3^18*x4^6*z^16 - x1^35*x2^20*x3^19*x4^6*z^16 - x1^34*x2^21*x3^19*x4^6*z^16 - x1^33*x2^22*x3^19*x4^6*z^16 - 2*x1^32*x2^23*x3^19*x4^6*z^16 - x1^30*x2^25*x3^19*x4^6*z^16 + 4*x1^28*x2^27*x3^19*x4^6*z^16 + x1^33*x2^21*x3^20*x4^6*z^16 + x1^32*x2^22*x3^20*x4^6*z^16 + 3*x1^31*x2^23*x3^20*x4^6*z^16 - x1^30*x2^24*x3^20*x4^6*z^16 + 2*x1^29*x2^25*x3^20*x4^6*z^16 - 4*x1^28*x2^26*x3^20*x4^6*z^16 - x1^27*x2^27*x3^20*x4^6*z^16 - x1^32*x2^21*x3^21*x4^6*z^16 - 2*x1^31*x2^22*x3^21*x4^6*z^16 - 2*x1^29*x2^24*x3^21*x4^6*z^16 + 3*x1^27*x2^26*x3^21*x4^6*z^16 + x1^30*x2^22*x3^22*x4^6*z^16 + 2*x1^28*x2^24*x3^22*x4^6*z^16 - 3*x1^27*x2^25*x3^22*x4^6*z^16 - 2*x1^26*x2^26*x3^22*x4^6*z^16 + x1^26*x2^25*x3^23*x4^6*z^16 + x1^26*x2^24*x3^24*x4^6*z^16 - x1^37*x2^28*x3^8*x4^7*z^16 + x1^36*x2^29*x3^8*x4^7*z^16 + x1^35*x2^30*x3^8*x4^7*z^16 + x1^37*x2^27*x3^9*x4^7*z^16 - x1^36*x2^28*x3^9*x4^7*z^16 - 2*x1^35*x2^29*x3^9*x4^7*z^16 - x1^34*x2^30*x3^9*x4^7*z^16 - x1^33*x2^31*x3^9*x4^7*z^16 + x1^39*x2^24*x3^10*x4^7*z^16 - x1^38*x2^25*x3^10*x4^7*z^16 + x1^37*x2^26*x3^10*x4^7*z^16 - x1^36*x2^27*x3^10*x4^7*z^16 + 2*x1^33*x2^30*x3^10*x4^7*z^16 + 2*x1^32*x2^31*x3^10*x4^7*z^16 - 2*x1^38*x2^24*x3^11*x4^7*z^16 + x1^36*x2^26*x3^11*x4^7*z^16 - 2*x1^35*x2^27*x3^11*x4^7*z^16 - 2*x1^34*x2^28*x3^11*x4^7*z^16 + x1^33*x2^29*x3^11*x4^7*z^16 - 2*x1^32*x2^30*x3^11*x4^7*z^16 - x1^31*x2^31*x3^11*x4^7*z^16 + x1^38*x2^23*x3^12*x4^7*z^16 - x1^35*x2^26*x3^12*x4^7*z^16 + 2*x1^34*x2^27*x3^12*x4^7*z^16 - 2*x1^33*x2^28*x3^12*x4^7*z^16 - 2*x1^32*x2^29*x3^12*x4^7*z^16 + 3*x1^31*x2^30*x3^12*x4^7*z^16 - x1^38*x2^22*x3^13*x4^7*z^16 - x1^37*x2^23*x3^13*x4^7*z^16 - x1^36*x2^24*x3^13*x4^7*z^16 - x1^30*x2^30*x3^13*x4^7*z^16 + 2*x1^38*x2^21*x3^14*x4^7*z^16 + x1^35*x2^24*x3^14*x4^7*z^16 - x1^34*x2^25*x3^14*x4^7*z^16 + x1^33*x2^26*x3^14*x4^7*z^16 - 2*x1^37*x2^21*x3^15*x4^7*z^16 - 2*x1^35*x2^23*x3^15*x4^7*z^16 - 2*x1^34*x2^24*x3^15*x4^7*z^16 - x1^37*x2^20*x3^16*x4^7*z^16 + x1^36*x2^21*x3^16*x4^7*z^16 + x1^35*x2^22*x3^16*x4^7*z^16 + x1^33*x2^24*x3^16*x4^7*z^16 - 2*x1^35*x2^21*x3^17*x4^7*z^16 - 3*x1^34*x2^22*x3^17*x4^7*z^16 - 2*x1^32*x2^24*x3^17*x4^7*z^16 - x1^31*x2^25*x3^17*x4^7*z^16 + 2*x1^34*x2^21*x3^18*x4^7*z^16 + x1^32*x2^23*x3^18*x4^7*z^16 - x1^29*x2^26*x3^18*x4^7*z^16 + x1^34*x2^20*x3^19*x4^7*z^16 - 2*x1^33*x2^21*x3^19*x4^7*z^16 - 2*x1^31*x2^23*x3^19*x4^7*z^16 + 2*x1^30*x2^24*x3^19*x4^7*z^16 + x1^28*x2^26*x3^19*x4^7*z^16 + 2*x1^32*x2^21*x3^20*x4^7*z^16 + x1^31*x2^22*x3^20*x4^7*z^16 + x1^30*x2^23*x3^20*x4^7*z^16 + x1^28*x2^25*x3^20*x4^7*z^16 - x1^27*x2^26*x3^20*x4^7*z^16 - x1^30*x2^22*x3^21*x4^7*z^16 + x1^29*x2^23*x3^21*x4^7*z^16 + x1^27*x2^25*x3^21*x4^7*z^16 + x1^26*x2^26*x3^21*x4^7*z^16 + x1^29*x2^22*x3^22*x4^7*z^16 - x1^27*x2^24*x3^22*x4^7*z^16 - x1^26*x2^25*x3^22*x4^7*z^16 - 2*x1^27*x2^23*x3^23*x4^7*z^16 + x1^26*x2^24*x3^23*x4^7*z^16 + x1^25*x2^25*x3^23*x4^7*z^16 - x1^25*x2^24*x3^24*x4^7*z^16 + x1^37*x2^27*x3^8*x4^8*z^16 - x1^35*x2^29*x3^8*x4^8*z^16 + x1^36*x2^27*x3^9*x4^8*z^16 + x1^35*x2^27*x3^10*x4^8*z^16 + 2*x1^34*x2^28*x3^10*x4^8*z^16 - x1^33*x2^29*x3^10*x4^8*z^16 + 2*x1^38*x2^23*x3^11*x4^8*z^16 + x1^37*x2^24*x3^11*x4^8*z^16 - x1^34*x2^27*x3^11*x4^8*z^16 - x1^33*x2^28*x3^11*x4^8*z^16 - x1^32*x2^29*x3^11*x4^8*z^16 - 2*x1^31*x2^30*x3^11*x4^8*z^16 + x1^37*x2^23*x3^12*x4^8*z^16 + x1^35*x2^25*x3^12*x4^8*z^16 + 4*x1^33*x2^27*x3^12*x4^8*z^16 + 2*x1^31*x2^29*x3^12*x4^8*z^16 + x1^30*x2^30*x3^12*x4^8*z^16 + x1^37*x2^22*x3^13*x4^8*z^16 + x1^36*x2^23*x3^13*x4^8*z^16 - 3*x1^33*x2^26*x3^13*x4^8*z^16 - 2*x1^32*x2^27*x3^13*x4^8*z^16 + x1^31*x2^28*x3^13*x4^8*z^16 - 4*x1^30*x2^29*x3^13*x4^8*z^16 - x1^37*x2^21*x3^14*x4^8*z^16 - 2*x1^35*x2^23*x3^14*x4^8*z^16 + x1^34*x2^24*x3^14*x4^8*z^16 - 2*x1^33*x2^25*x3^14*x4^8*z^16 + 2*x1^32*x2^26*x3^14*x4^8*z^16 + x1^31*x2^27*x3^14*x4^8*z^16 + 2*x1^30*x2^28*x3^14*x4^8*z^16 + 2*x1^29*x2^29*x3^14*x4^8*z^16 + 3*x1^36*x2^21*x3^15*x4^8*z^16 - x1^35*x2^22*x3^15*x4^8*z^16 + 4*x1^34*x2^23*x3^15*x4^8*z^16 - x1^33*x2^24*x3^15*x4^8*z^16 - 2*x1^32*x2^25*x3^15*x4^8*z^16 - 4*x1^31*x2^26*x3^15*x4^8*z^16 - 3*x1^30*x2^27*x3^15*x4^8*z^16 - 4*x1^29*x2^28*x3^15*x4^8*z^16 - x1^36*x2^20*x3^16*x4^8*z^16 - x1^35*x2^21*x3^16*x4^8*z^16 - x1^33*x2^23*x3^16*x4^8*z^16 + x1^32*x2^24*x3^16*x4^8*z^16 + 3*x1^31*x2^25*x3^16*x4^8*z^16 + 2*x1^30*x2^26*x3^16*x4^8*z^16 + 4*x1^29*x2^27*x3^16*x4^8*z^16 + x1^28*x2^28*x3^16*x4^8*z^16 + 2*x1^35*x2^20*x3^17*x4^8*z^16 + x1^34*x2^21*x3^17*x4^8*z^16 + x1^33*x2^22*x3^17*x4^8*z^16 - x1^32*x2^23*x3^17*x4^8*z^16 - x1^30*x2^25*x3^17*x4^8*z^16 - 2*x1^29*x2^26*x3^17*x4^8*z^16 - 4*x1^28*x2^27*x3^17*x4^8*z^16 - x1^34*x2^20*x3^18*x4^8*z^16 - x1^33*x2^21*x3^18*x4^8*z^16 - x1^32*x2^22*x3^18*x4^8*z^16 - x1^31*x2^23*x3^18*x4^8*z^16 + x1^30*x2^24*x3^18*x4^8*z^16 + 3*x1^28*x2^26*x3^18*x4^8*z^16 + x1^27*x2^27*x3^18*x4^8*z^16 + x1^33*x2^20*x3^19*x4^8*z^16 + 2*x1^32*x2^21*x3^19*x4^8*z^16 + x1^30*x2^23*x3^19*x4^8*z^16 - x1^29*x2^24*x3^19*x4^8*z^16 + 2*x1^28*x2^25*x3^19*x4^8*z^16 - 3*x1^27*x2^26*x3^19*x4^8*z^16 - x1^32*x2^20*x3^20*x4^8*z^16 - x1^31*x2^21*x3^20*x4^8*z^16 - x1^30*x2^22*x3^20*x4^8*z^16 - x1^29*x2^23*x3^20*x4^8*z^16 + x1^28*x2^24*x3^20*x4^8*z^16 - x1^27*x2^25*x3^20*x4^8*z^16 + 2*x1^26*x2^26*x3^20*x4^8*z^16 + 2*x1^29*x2^22*x3^21*x4^8*z^16 - x1^28*x2^23*x3^21*x4^8*z^16 - x1^27*x2^24*x3^21*x4^8*z^16 + x1^26*x2^25*x3^21*x4^8*z^16 - x1^28*x2^22*x3^22*x4^8*z^16 + x1^27*x2^23*x3^22*x4^8*z^16 - x1^25*x2^25*x3^22*x4^8*z^16 - x1^36*x2^26*x3^9*x4^9*z^16 - x1^35*x2^27*x3^9*x4^9*z^16 + x1^35*x2^26*x3^10*x4^9*z^16 + x1^34*x2^27*x3^10*x4^9*z^16 + x1^33*x2^28*x3^10*x4^9*z^16 + x1^32*x2^29*x3^10*x4^9*z^16 + x1^31*x2^30*x3^10*x4^9*z^16 - 2*x1^35*x2^25*x3^11*x4^9*z^16 - x1^34*x2^26*x3^11*x4^9*z^16 - x1^33*x2^27*x3^11*x4^9*z^16 - x1^32*x2^28*x3^11*x4^9*z^16 + x1^31*x2^29*x3^11*x4^9*z^16 - x1^30*x2^30*x3^11*x4^9*z^16 - 2*x1^37*x2^22*x3^12*x4^9*z^16 - x1^36*x2^23*x3^12*x4^9*z^16 + x1^34*x2^25*x3^12*x4^9*z^16 + x1^33*x2^26*x3^12*x4^9*z^16 + 2*x1^32*x2^27*x3^12*x4^9*z^16 + x1^30*x2^29*x3^12*x4^9*z^16 + x1^36*x2^22*x3^13*x4^9*z^16 + x1^35*x2^23*x3^13*x4^9*z^16 - 3*x1^34*x2^24*x3^13*x4^9*z^16 + x1^33*x2^25*x3^13*x4^9*z^16 - 2*x1^32*x2^26*x3^13*x4^9*z^16 - 3*x1^31*x2^27*x3^13*x4^9*z^16 - x1^30*x2^28*x3^13*x4^9*z^16 - x1^29*x2^29*x3^13*x4^9*z^16 - 2*x1^36*x2^21*x3^14*x4^9*z^16 - x1^34*x2^23*x3^14*x4^9*z^16 + 2*x1^33*x2^24*x3^14*x4^9*z^16 + x1^32*x2^25*x3^14*x4^9*z^16 + 5*x1^31*x2^26*x3^14*x4^9*z^16 + 3*x1^30*x2^27*x3^14*x4^9*z^16 + 2*x1^29*x2^28*x3^14*x4^9*z^16 + x1^36*x2^20*x3^15*x4^9*z^16 + x1^35*x2^21*x3^15*x4^9*z^16 - x1^33*x2^23*x3^15*x4^9*z^16 - x1^32*x2^24*x3^15*x4^9*z^16 - 2*x1^31*x2^25*x3^15*x4^9*z^16 - 3*x1^30*x2^26*x3^15*x4^9*z^16 - x1^28*x2^28*x3^15*x4^9*z^16 - 3*x1^35*x2^20*x3^16*x4^9*z^16 - x1^34*x2^21*x3^16*x4^9*z^16 + 3*x1^32*x2^23*x3^16*x4^9*z^16 + 4*x1^30*x2^25*x3^16*x4^9*z^16 + 3*x1^29*x2^26*x3^16*x4^9*z^16 + 2*x1^28*x2^27*x3^16*x4^9*z^16 + x1^35*x2^19*x3^17*x4^9*z^16 + 2*x1^34*x2^20*x3^17*x4^9*z^16 + x1^33*x2^21*x3^17*x4^9*z^16 - x1^32*x2^22*x3^17*x4^9*z^16 - x1^31*x2^23*x3^17*x4^9*z^16 - 4*x1^30*x2^24*x3^17*x4^9*z^16 - 3*x1^29*x2^25*x3^17*x4^9*z^16 - 2*x1^28*x2^26*x3^17*x4^9*z^16 - x1^34*x2^19*x3^18*x4^9*z^16 - x1^32*x2^21*x3^18*x4^9*z^16 + 5*x1^31*x2^22*x3^18*x4^9*z^16 + 2*x1^30*x2^23*x3^18*x4^9*z^16 + 5*x1^29*x2^24*x3^18*x4^9*z^16 + 2*x1^27*x2^26*x3^18*x4^9*z^16 - x1^31*x2^21*x3^19*x4^9*z^16 - x1^29*x2^23*x3^19*x4^9*z^16 - 5*x1^28*x2^24*x3^19*x4^9*z^16 - x1^26*x2^26*x3^19*x4^9*z^16 - x1^31*x2^20*x3^20*x4^9*z^16 + 2*x1^30*x2^21*x3^20*x4^9*z^16 - x1^29*x2^22*x3^20*x4^9*z^16 + 5*x1^28*x2^23*x3^20*x4^9*z^16 + 3*x1^27*x2^24*x3^20*x4^9*z^16 - 2*x1^29*x2^21*x3^21*x4^9*z^16 - x1^28*x2^22*x3^21*x4^9*z^16 - 4*x1^27*x2^23*x3^21*x4^9*z^16 + x1^26*x2^23*x3^22*x4^9*z^16 - x1^32*x2^28*x3^10*x4^10*z^16 - 2*x1^33*x2^26*x3^11*x4^10*z^16 - x1^32*x2^27*x3^11*x4^10*z^16 - x1^30*x2^29*x3^11*x4^10*z^16 + x1^34*x2^24*x3^12*x4^10*z^16 - x1^33*x2^25*x3^12*x4^10*z^16 + x1^32*x2^26*x3^12*x4^10*z^16 - x1^31*x2^27*x3^12*x4^10*z^16 + 2*x1^29*x2^29*x3^12*x4^10*z^16 - x1^31*x2^26*x3^13*x4^10*z^16 - x1^30*x2^27*x3^13*x4^10*z^16 - x1^29*x2^28*x3^13*x4^10*z^16 + x1^36*x2^20*x3^14*x4^10*z^16 - x1^35*x2^21*x3^14*x4^10*z^16 - 2*x1^34*x2^22*x3^14*x4^10*z^16 + 2*x1^33*x2^23*x3^14*x4^10*z^16 - x1^32*x2^24*x3^14*x4^10*z^16 + x1^29*x2^27*x3^14*x4^10*z^16 + x1^34*x2^21*x3^15*x4^10*z^16 + x1^33*x2^22*x3^15*x4^10*z^16 - x1^32*x2^23*x3^15*x4^10*z^16 + x1^31*x2^24*x3^15*x4^10*z^16 - 2*x1^29*x2^26*x3^15*x4^10*z^16 - x1^33*x2^21*x3^16*x4^10*z^16 - x1^32*x2^22*x3^16*x4^10*z^16 + 2*x1^30*x2^24*x3^16*x4^10*z^16 - x1^29*x2^25*x3^16*x4^10*z^16 + x1^28*x2^26*x3^16*x4^10*z^16 + x1^34*x2^19*x3^17*x4^10*z^16 - 2*x1^32*x2^21*x3^17*x4^10*z^16 - 3*x1^31*x2^22*x3^17*x4^10*z^16 - x1^29*x2^24*x3^17*x4^10*z^16 - x1^28*x2^25*x3^17*x4^10*z^16 - x1^33*x2^19*x3^18*x4^10*z^16 + x1^31*x2^21*x3^18*x4^10*z^16 - x1^30*x2^22*x3^18*x4^10*z^16 + x1^28*x2^24*x3^18*x4^10*z^16 + x1^27*x2^25*x3^18*x4^10*z^16 + x1^31*x2^20*x3^19*x4^10*z^16 - x1^30*x2^21*x3^19*x4^10*z^16 - x1^29*x2^22*x3^19*x4^10*z^16 - 2*x1^28*x2^23*x3^19*x4^10*z^16 - x1^27*x2^24*x3^19*x4^10*z^16 - x1^26*x2^25*x3^19*x4^10*z^16 + x1^28*x2^22*x3^20*x4^10*z^16 + 2*x1^27*x2^23*x3^20*x4^10*z^16 + x1^25*x2^25*x3^20*x4^10*z^16 + x1^28*x2^21*x3^21*x4^10*z^16 - x1^27*x2^22*x3^21*x4^10*z^16 - x1^26*x2^23*x3^21*x4^10*z^16 + x1^26*x2^22*x3^22*x4^10*z^16 + x1^34*x2^24*x3^11*x4^11*z^16 + x1^33*x2^25*x3^11*x4^11*z^16 - x1^32*x2^26*x3^11*x4^11*z^16 + x1^31*x2^27*x3^11*x4^11*z^16 + x1^33*x2^24*x3^12*x4^11*z^16 + 2*x1^32*x2^25*x3^12*x4^11*z^16 - x1^30*x2^27*x3^12*x4^11*z^16 + x1^29*x2^28*x3^12*x4^11*z^16 + 2*x1^32*x2^24*x3^13*x4^11*z^16 - x1^31*x2^25*x3^13*x4^11*z^16 + x1^30*x2^26*x3^13*x4^11*z^16 - x1^29*x2^27*x3^13*x4^11*z^16 - x1^28*x2^28*x3^13*x4^11*z^16 + x1^35*x2^20*x3^14*x4^11*z^16 - x1^33*x2^22*x3^14*x4^11*z^16 - x1^32*x2^23*x3^14*x4^11*z^16 + 2*x1^31*x2^24*x3^14*x4^11*z^16 - x1^30*x2^25*x3^14*x4^11*z^16 - x1^29*x2^26*x3^14*x4^11*z^16 + 2*x1^28*x2^27*x3^14*x4^11*z^16 + x1^34*x2^20*x3^15*x4^11*z^16 + 2*x1^33*x2^21*x3^15*x4^11*z^16 - 2*x1^32*x2^22*x3^15*x4^11*z^16 + x1^31*x2^23*x3^15*x4^11*z^16 + 2*x1^30*x2^24*x3^15*x4^11*z^16 + 3*x1^29*x2^25*x3^15*x4^11*z^16 - 3*x1^28*x2^26*x3^15*x4^11*z^16 - x1^27*x2^27*x3^15*x4^11*z^16 - x1^33*x2^20*x3^16*x4^11*z^16 - 2*x1^32*x2^21*x3^16*x4^11*z^16 - 2*x1^29*x2^24*x3^16*x4^11*z^16 + x1^28*x2^25*x3^16*x4^11*z^16 + x1^27*x2^26*x3^16*x4^11*z^16 + 2*x1^31*x2^21*x3^17*x4^11*z^16 - 3*x1^29*x2^23*x3^17*x4^11*z^16 + 4*x1^28*x2^24*x3^17*x4^11*z^16 - 2*x1^27*x2^25*x3^17*x4^11*z^16 - x1^26*x2^26*x3^17*x4^11*z^16 - x1^30*x2^21*x3^18*x4^11*z^16 - x1^29*x2^22*x3^18*x4^11*z^16 - x1^27*x2^24*x3^18*x4^11*z^16 + x1^26*x2^25*x3^18*x4^11*z^16 - x1^26*x2^24*x3^19*x4^11*z^16 - x1^26*x2^23*x3^20*x4^11*z^16 + x1^25*x2^24*x3^20*x4^11*z^16 - x1^24*x2^24*x3^21*x4^11*z^16 - x1^33*x2^23*x3^12*x4^12*z^16 - x1^32*x2^24*x3^12*x4^12*z^16 + x1^31*x2^25*x3^12*x4^12*z^16 - x1^30*x2^26*x3^12*x4^12*z^16 - x1^31*x2^24*x3^13*x4^12*z^16 + x1^30*x2^25*x3^13*x4^12*z^16 + x1^29*x2^26*x3^13*x4^12*z^16 - x1^28*x2^27*x3^13*x4^12*z^16 - x1^31*x2^23*x3^14*x4^12*z^16 + 3*x1^30*x2^24*x3^14*x4^12*z^16 - x1^29*x2^25*x3^14*x4^12*z^16 + 2*x1^28*x2^26*x3^14*x4^12*z^16 + x1^27*x2^27*x3^14*x4^12*z^16 - 2*x1^30*x2^23*x3^15*x4^12*z^16 - 4*x1^27*x2^26*x3^15*x4^12*z^16 - x1^31*x2^21*x3^16*x4^12*z^16 + 2*x1^29*x2^23*x3^16*x4^12*z^16 - 3*x1^28*x2^24*x3^16*x4^12*z^16 + 4*x1^27*x2^25*x3^16*x4^12*z^16 + 2*x1^26*x2^26*x3^16*x4^12*z^16 - 2*x1^29*x2^22*x3^17*x4^12*z^16 + x1^28*x2^23*x3^17*x4^12*z^16 - x1^27*x2^24*x3^17*x4^12*z^16 - 5*x1^26*x2^25*x3^17*x4^12*z^16 - 2*x1^29*x2^21*x3^18*x4^12*z^16 + 2*x1^28*x2^22*x3^18*x4^12*z^16 - x1^27*x2^23*x3^18*x4^12*z^16 + 4*x1^26*x2^24*x3^18*x4^12*z^16 + 2*x1^25*x2^25*x3^18*x4^12*z^16 - x1^27*x2^22*x3^19*x4^12*z^16 + x1^26*x2^23*x3^19*x4^12*z^16 - 4*x1^25*x2^24*x3^19*x4^12*z^16 + 2*x1^24*x2^24*x3^20*x4^12*z^16 - x1^31*x2^22*x3^14*x4^13*z^16 - x1^30*x2^23*x3^14*x4^13*z^16 - x1^29*x2^24*x3^14*x4^13*z^16 - x1^30*x2^22*x3^15*x4^13*z^16 - 3*x1^29*x2^23*x3^15*x4^13*z^16 + x1^28*x2^24*x3^15*x4^13*z^16 - x1^27*x2^25*x3^15*x4^13*z^16 - x1^30*x2^21*x3^16*x4^13*z^16 + x1^29*x2^22*x3^16*x4^13*z^16 - x1^27*x2^24*x3^16*x4^13*z^16 + 3*x1^26*x2^25*x3^16*x4^13*z^16 + x1^30*x2^20*x3^17*x4^13*z^16 - x1^29*x2^21*x3^17*x4^13*z^16 - 3*x1^28*x2^22*x3^17*x4^13*z^16 + x1^27*x2^23*x3^17*x4^13*z^16 - x1^26*x2^24*x3^17*x4^13*z^16 - 2*x1^25*x2^25*x3^17*x4^13*z^16 + x1^28*x2^21*x3^18*x4^13*z^16 - x1^26*x2^23*x3^18*x4^13*z^16 + 2*x1^25*x2^24*x3^18*x4^13*z^16 - x1^24*x2^24*x3^19*x4^13*z^16 + x1^30*x2^21*x3^15*x4^14*z^16 + x1^27*x2^24*x3^15*x4^14*z^16 - 2*x1^27*x2^23*x3^16*x4^14*z^16 + x1^26*x2^23*x3^17*x4^14*z^16 + x1^27*x2^21*x3^18*x4^14*z^16 - 2*x1^26*x2^22*x3^18*x4^14*z^16 + x1^24*x2^24*x3^18*x4^14*z^16 + x1^25*x2^22*x3^19*x4^14*z^16 - x1^26*x2^23*x3^16*x4^15*z^16 + 2*x1^26*x2^22*x3^17*x4^15*z^16 - x1^25*x2^22*x3^18*x4^15*z^16 + x1^40*x2^26*x3^9*z^15 - 2*x1^39*x2^26*x3^10*z^15 - x1^38*x2^27*x3^10*z^15 - x1^37*x2^28*x3^10*z^15 + x1^39*x2^25*x3^11*z^15 + 2*x1^38*x2^26*x3^11*z^15 + x1^36*x2^28*x3^11*z^15 - x1^35*x2^29*x3^11*z^15 - 2*x1^38*x2^25*x3^12*z^15 - x1^35*x2^28*x3^12*z^15 + 2*x1^37*x2^25*x3^13*z^15 + 2*x1^35*x2^27*x3^13*z^15 - x1^32*x2^30*x3^13*z^15 - 2*x1^37*x2^24*x3^14*z^15 - x1^36*x2^25*x3^14*z^15 - x1^35*x2^26*x3^14*z^15 + x1^33*x2^28*x3^14*z^15 + x1^32*x2^29*x3^14*z^15 + x1^31*x2^30*x3^14*z^15 + x1^37*x2^23*x3^15*z^15 + 2*x1^36*x2^24*x3^15*z^15 + x1^34*x2^26*x3^15*z^15 + x1^33*x2^27*x3^15*z^15 - 2*x1^31*x2^29*x3^15*z^15 - 2*x1^36*x2^23*x3^16*z^15 - x1^35*x2^24*x3^16*z^15 - x1^34*x2^25*x3^16*z^15 - x1^33*x2^26*x3^16*z^15 + x1^32*x2^27*x3^16*z^15 + x1^31*x2^28*x3^16*z^15 + x1^36*x2^22*x3^17*z^15 + 2*x1^35*x2^23*x3^17*z^15 + x1^33*x2^25*x3^17*z^15 - x1^32*x2^26*x3^17*z^15 + x1^31*x2^27*x3^17*z^15 - x1^30*x2^28*x3^17*z^15 - x1^35*x2^22*x3^18*z^15 + x1^34*x2^23*x3^18*z^15 - x1^33*x2^24*x3^18*z^15 - x1^32*x2^25*x3^18*z^15 + x1^34*x2^22*x3^19*z^15 + x1^33*x2^23*x3^19*z^15 + x1^32*x2^24*x3^19*z^15 - x1^31*x2^24*x3^20*z^15 + x1^40*x2^27*x3^7*x4*z^15 - x1^40*x2^26*x3^8*x4*z^15 - x1^39*x2^27*x3^8*x4*z^15 - x1^37*x2^29*x3^8*x4*z^15 + 4*x1^39*x2^26*x3^9*x4*z^15 - x1^38*x2^27*x3^9*x4*z^15 + x1^36*x2^29*x3^9*x4*z^15 - 2*x1^39*x2^25*x3^10*x4*z^15 - 4*x1^38*x2^26*x3^10*x4*z^15 + 2*x1^37*x2^27*x3^10*x4*z^15 - 2*x1^36*x2^28*x3^10*x4*z^15 + 6*x1^38*x2^25*x3^11*x4*z^15 + 2*x1^37*x2^26*x3^11*x4*z^15 - x1^34*x2^29*x3^11*x4*z^15 - 2*x1^38*x2^24*x3^12*x4*z^15 - 6*x1^37*x2^25*x3^12*x4*z^15 - 2*x1^35*x2^27*x3^12*x4*z^15 + 3*x1^34*x2^28*x3^12*x4*z^15 + x1^33*x2^29*x3^12*x4*z^15 + 2*x1^32*x2^30*x3^12*x4*z^15 + 6*x1^37*x2^24*x3^13*x4*z^15 + 2*x1^36*x2^25*x3^13*x4*z^15 + 2*x1^35*x2^26*x3^13*x4*z^15 - x1^33*x2^28*x3^13*x4*z^15 - 2*x1^32*x2^29*x3^13*x4*z^15 - 2*x1^31*x2^30*x3^13*x4*z^15 - 2*x1^37*x2^23*x3^14*x4*z^15 - 6*x1^36*x2^24*x3^14*x4*z^15 - 4*x1^34*x2^26*x3^14*x4*z^15 + 4*x1^33*x2^27*x3^14*x4*z^15 + 6*x1^31*x2^29*x3^14*x4*z^15 + 6*x1^36*x2^23*x3^15*x4*z^15 + 2*x1^35*x2^24*x3^15*x4*z^15 + 2*x1^34*x2^25*x3^15*x4*z^15 - 2*x1^32*x2^27*x3^15*x4*z^15 - 2*x1^31*x2^28*x3^15*x4*z^15 - 6*x1^30*x2^29*x3^15*x4*z^15 - 2*x1^36*x2^22*x3^16*x4*z^15 - 6*x1^35*x2^23*x3^16*x4*z^15 - 4*x1^33*x2^25*x3^16*x4*z^15 + 2*x1^32*x2^26*x3^16*x4*z^15 + 6*x1^30*x2^28*x3^16*x4*z^15 + x1^29*x2^29*x3^16*x4*z^15 + 5*x1^35*x2^22*x3^17*x4*z^15 + 3*x1^34*x2^23*x3^17*x4*z^15 + 2*x1^33*x2^24*x3^17*x4*z^15 + x1^32*x2^25*x3^17*x4*z^15 - x1^31*x2^26*x3^17*x4*z^15 - x1^30*x2^27*x3^17*x4*z^15 - 3*x1^29*x2^28*x3^17*x4*z^15 - 5*x1^34*x2^22*x3^18*x4*z^15 - 2*x1^33*x2^23*x3^18*x4*z^15 - 4*x1^32*x2^24*x3^18*x4*z^15 + 4*x1^29*x2^27*x3^18*x4*z^15 + x1^34*x2^21*x3^19*x4*z^15 + 2*x1^33*x2^22*x3^19*x4*z^15 + 3*x1^32*x2^23*x3^19*x4*z^15 + 3*x1^31*x2^24*x3^19*x4*z^15 - x1^30*x2^25*x3^19*x4*z^15 - x1^29*x2^26*x3^19*x4*z^15 - x1^28*x2^27*x3^19*x4*z^15 - x1^33*x2^21*x3^20*x4*z^15 - 2*x1^32*x2^22*x3^20*x4*z^15 - 2*x1^31*x2^23*x3^20*x4*z^15 - x1^30*x2^24*x3^20*x4*z^15 + 2*x1^28*x2^26*x3^20*x4*z^15 + x1^31*x2^22*x3^21*x4*z^15 + 2*x1^30*x2^23*x3^21*x4*z^15 + x1^39*x2^27*x3^7*x4^2*z^15 - x1^38*x2^28*x3^7*x4^2*z^15 - 2*x1^39*x2^26*x3^8*x4^2*z^15 - x1^38*x2^27*x3^8*x4^2*z^15 + x1^37*x2^28*x3^8*x4^2*z^15 + x1^39*x2^25*x3^9*x4^2*z^15 + 4*x1^38*x2^26*x3^9*x4^2*z^15 - 2*x1^37*x2^27*x3^9*x4^2*z^15 + x1^36*x2^28*x3^9*x4^2*z^15 - x1^35*x2^29*x3^9*x4^2*z^15 - 4*x1^38*x2^25*x3^10*x4^2*z^15 - 2*x1^37*x2^26*x3^10*x4^2*z^15 - x1^36*x2^27*x3^10*x4^2*z^15 + 2*x1^34*x2^29*x3^10*x4^2*z^15 + x1^33*x2^30*x3^10*x4^2*z^15 + 2*x1^38*x2^24*x3^11*x4^2*z^15 + 6*x1^37*x2^25*x3^11*x4^2*z^15 + 4*x1^35*x2^27*x3^11*x4^2*z^15 - 3*x1^34*x2^28*x3^11*x4^2*z^15 - 2*x1^33*x2^29*x3^11*x4^2*z^15 - 3*x1^32*x2^30*x3^11*x4^2*z^15 - 6*x1^37*x2^24*x3^12*x4^2*z^15 - 2*x1^36*x2^25*x3^12*x4^2*z^15 - 2*x1^35*x2^26*x3^12*x4^2*z^15 + 3*x1^33*x2^28*x3^12*x4^2*z^15 + 3*x1^32*x2^29*x3^12*x4^2*z^15 + 3*x1^31*x2^30*x3^12*x4^2*z^15 + 2*x1^37*x2^23*x3^13*x4^2*z^15 + 6*x1^36*x2^24*x3^13*x4^2*z^15 + 4*x1^34*x2^26*x3^13*x4^2*z^15 - 4*x1^33*x2^27*x3^13*x4^2*z^15 - x1^32*x2^28*x3^13*x4^2*z^15 - 6*x1^31*x2^29*x3^13*x4^2*z^15 - x1^30*x2^30*x3^13*x4^2*z^15 - 6*x1^36*x2^23*x3^14*x4^2*z^15 - 2*x1^35*x2^24*x3^14*x4^2*z^15 - 2*x1^34*x2^25*x3^14*x4^2*z^15 + 2*x1^32*x2^27*x3^14*x4^2*z^15 + 2*x1^31*x2^28*x3^14*x4^2*z^15 + 6*x1^30*x2^29*x3^14*x4^2*z^15 + 2*x1^36*x2^22*x3^15*x4^2*z^15 + 6*x1^35*x2^23*x3^15*x4^2*z^15 + 4*x1^33*x2^25*x3^15*x4^2*z^15 - 4*x1^32*x2^26*x3^15*x4^2*z^15 - 6*x1^30*x2^28*x3^15*x4^2*z^15 - 2*x1^29*x2^29*x3^15*x4^2*z^15 - 4*x1^35*x2^22*x3^16*x4^2*z^15 - 2*x1^34*x2^23*x3^16*x4^2*z^15 - 2*x1^33*x2^24*x3^16*x4^2*z^15 + 2*x1^31*x2^26*x3^16*x4^2*z^15 + 2*x1^30*x2^27*x3^16*x4^2*z^15 + 6*x1^29*x2^28*x3^16*x4^2*z^15 + x1^35*x2^21*x3^17*x4^2*z^15 + 4*x1^34*x2^22*x3^17*x4^2*z^15 + x1^33*x2^23*x3^17*x4^2*z^15 + 4*x1^32*x2^24*x3^17*x4^2*z^15 - 4*x1^31*x2^25*x3^17*x4^2*z^15 - 6*x1^29*x2^27*x3^17*x4^2*z^15 - 2*x1^28*x2^28*x3^17*x4^2*z^15 - x1^34*x2^21*x3^18*x4^2*z^15 - x1^33*x2^22*x3^18*x4^2*z^15 - x1^32*x2^23*x3^18*x4^2*z^15 + x1^30*x2^25*x3^18*x4^2*z^15 + x1^29*x2^26*x3^18*x4^2*z^15 + 5*x1^28*x2^27*x3^18*x4^2*z^15 + x1^33*x2^21*x3^19*x4^2*z^15 + 2*x1^31*x2^23*x3^19*x4^2*z^15 - x1^30*x2^24*x3^19*x4^2*z^15 - x1^29*x2^25*x3^19*x4^2*z^15 - 5*x1^28*x2^26*x3^19*x4^2*z^15 + x1^31*x2^22*x3^20*x4^2*z^15 - x1^30*x2^23*x3^20*x4^2*z^15 + x1^28*x2^25*x3^20*x4^2*z^15 + 2*x1^27*x2^26*x3^20*x4^2*z^15 - 2*x1^27*x2^25*x3^21*x4^2*z^15 + x1^26*x2^25*x3^22*x4^2*z^15 + x1^38*x2^26*x3^8*x4^3*z^15 - x1^37*x2^27*x3^8*x4^3*z^15 - x1^36*x2^28*x3^8*x4^3*z^15 + x1^35*x2^29*x3^8*x4^3*z^15 - x1^40*x2^23*x3^9*x4^3*z^15 - 2*x1^39*x2^24*x3^9*x4^3*z^15 + x1^37*x2^26*x3^9*x4^3*z^15 + x1^36*x2^27*x3^9*x4^3*z^15 + x1^35*x2^28*x3^9*x4^3*z^15 + 2*x1^39*x2^23*x3^10*x4^3*z^15 - x1^37*x2^25*x3^10*x4^3*z^15 + x1^36*x2^26*x3^10*x4^3*z^15 - 2*x1^35*x2^27*x3^10*x4^3*z^15 - x1^34*x2^28*x3^10*x4^3*z^15 + x1^32*x2^30*x3^10*x4^3*z^15 - x1^39*x2^22*x3^11*x4^3*z^15 - 2*x1^38*x2^23*x3^11*x4^3*z^15 + 2*x1^35*x2^26*x3^11*x4^3*z^15 + x1^33*x2^28*x3^11*x4^3*z^15 - x1^32*x2^29*x3^11*x4^3*z^15 - x1^31*x2^30*x3^11*x4^3*z^15 + 2*x1^38*x2^22*x3^12*x4^3*z^15 - x1^37*x2^23*x3^12*x4^3*z^15 - 2*x1^36*x2^24*x3^12*x4^3*z^15 - x1^34*x2^26*x3^12*x4^3*z^15 - 2*x1^32*x2^28*x3^12*x4^3*z^15 + x1^31*x2^29*x3^12*x4^3*z^15 - 2*x1^37*x2^22*x3^13*x4^3*z^15 + 2*x1^36*x2^23*x3^13*x4^3*z^15 - 2*x1^35*x2^24*x3^13*x4^3*z^15 + x1^34*x2^25*x3^13*x4^3*z^15 + 2*x1^32*x2^27*x3^13*x4^3*z^15 - x1^30*x2^29*x3^13*x4^3*z^15 + 2*x1^37*x2^21*x3^14*x4^3*z^15 + x1^36*x2^22*x3^14*x4^3*z^15 - x1^35*x2^23*x3^14*x4^3*z^15 - 3*x1^33*x2^25*x3^14*x4^3*z^15 - 2*x1^31*x2^27*x3^14*x4^3*z^15 + 2*x1^30*x2^28*x3^14*x4^3*z^15 - x1^37*x2^20*x3^15*x4^3*z^15 - 2*x1^36*x2^21*x3^15*x4^3*z^15 + 2*x1^35*x2^22*x3^15*x4^3*z^15 + 2*x1^33*x2^24*x3^15*x4^3*z^15 + x1^31*x2^26*x3^15*x4^3*z^15 - 2*x1^29*x2^28*x3^15*x4^3*z^15 + 2*x1^36*x2^20*x3^16*x4^3*z^15 - x1^34*x2^22*x3^16*x4^3*z^15 - 2*x1^32*x2^24*x3^16*x4^3*z^15 - 2*x1^30*x2^26*x3^16*x4^3*z^15 + 2*x1^29*x2^27*x3^16*x4^3*z^15 + x1^28*x2^28*x3^16*x4^3*z^15 - 2*x1^35*x2^20*x3^17*x4^3*z^15 + 2*x1^32*x2^23*x3^17*x4^3*z^15 + x1^30*x2^25*x3^17*x4^3*z^15 - x1^29*x2^26*x3^17*x4^3*z^15 - 2*x1^28*x2^27*x3^17*x4^3*z^15 + x1^34*x2^20*x3^18*x4^3*z^15 + x1^31*x2^23*x3^18*x4^3*z^15 + 3*x1^30*x2^24*x3^18*x4^3*z^15 - x1^29*x2^25*x3^18*x4^3*z^15 + 2*x1^28*x2^26*x3^18*x4^3*z^15 + x1^27*x2^27*x3^18*x4^3*z^15 - x1^33*x2^20*x3^19*x4^3*z^15 - x1^32*x2^21*x3^19*x4^3*z^15 + x1^30*x2^23*x3^19*x4^3*z^15 + 2*x1^29*x2^24*x3^19*x4^3*z^15 - x1^28*x2^25*x3^19*x4^3*z^15 - 3*x1^27*x2^26*x3^19*x4^3*z^15 + x1^32*x2^20*x3^20*x4^3*z^15 - x1^29*x2^23*x3^20*x4^3*z^15 + x1^28*x2^24*x3^20*x4^3*z^15 + 2*x1^27*x2^25*x3^20*x4^3*z^15 - x1^29*x2^22*x3^21*x4^3*z^15 + x1^28*x2^23*x3^21*x4^3*z^15 - x1^27*x2^24*x3^21*x4^3*z^15 - 2*x1^26*x2^25*x3^21*x4^3*z^15 - x1^40*x2^24*x3^7*x4^4*z^15 - x1^39*x2^25*x3^7*x4^4*z^15 + x1^37*x2^27*x3^7*x4^4*z^15 - x1^36*x2^28*x3^7*x4^4*z^15 + 2*x1^39*x2^24*x3^8*x4^4*z^15 - x1^35*x2^28*x3^8*x4^4*z^15 - 2*x1^39*x2^23*x3^9*x4^4*z^15 - 2*x1^38*x2^24*x3^9*x4^4*z^15 - x1^37*x2^25*x3^9*x4^4*z^15 + x1^36*x2^26*x3^9*x4^4*z^15 + x1^35*x2^27*x3^9*x4^4*z^15 - 2*x1^34*x2^28*x3^9*x4^4*z^15 - x1^33*x2^29*x3^9*x4^4*z^15 + x1^39*x2^22*x3^10*x4^4*z^15 + 4*x1^38*x2^23*x3^10*x4^4*z^15 + x1^37*x2^24*x3^10*x4^4*z^15 - x1^35*x2^26*x3^10*x4^4*z^15 - x1^33*x2^28*x3^10*x4^4*z^15 - 5*x1^38*x2^22*x3^11*x4^4*z^15 - 3*x1^37*x2^23*x3^11*x4^4*z^15 - 3*x1^36*x2^24*x3^11*x4^4*z^15 + x1^35*x2^25*x3^11*x4^4*z^15 + x1^34*x2^26*x3^11*x4^4*z^15 + 2*x1^33*x2^27*x3^11*x4^4*z^15 + x1^32*x2^28*x3^11*x4^4*z^15 + 2*x1^38*x2^21*x3^12*x4^4*z^15 + 6*x1^37*x2^22*x3^12*x4^4*z^15 + x1^36*x2^23*x3^12*x4^4*z^15 + 3*x1^35*x2^24*x3^12*x4^4*z^15 - 3*x1^34*x2^25*x3^12*x4^4*z^15 + x1^33*x2^26*x3^12*x4^4*z^15 - 5*x1^32*x2^27*x3^12*x4^4*z^15 + x1^31*x2^28*x3^12*x4^4*z^15 - 6*x1^37*x2^21*x3^13*x4^4*z^15 - 2*x1^36*x2^22*x3^13*x4^4*z^15 - 2*x1^35*x2^23*x3^13*x4^4*z^15 + 2*x1^33*x2^25*x3^13*x4^4*z^15 + 2*x1^32*x2^26*x3^13*x4^4*z^15 + 6*x1^31*x2^27*x3^13*x4^4*z^15 + x1^37*x2^20*x3^14*x4^4*z^15 + 6*x1^36*x2^21*x3^14*x4^4*z^15 + 4*x1^34*x2^23*x3^14*x4^4*z^15 - 4*x1^33*x2^24*x3^14*x4^4*z^15 - 6*x1^31*x2^26*x3^14*x4^4*z^15 - 2*x1^30*x2^27*x3^14*x4^4*z^15 - 4*x1^36*x2^20*x3^15*x4^4*z^15 - 3*x1^35*x2^21*x3^15*x4^4*z^15 - 2*x1^34*x2^22*x3^15*x4^4*z^15 + 2*x1^32*x2^24*x3^15*x4^4*z^15 + 2*x1^31*x2^25*x3^15*x4^4*z^15 + 6*x1^30*x2^26*x3^15*x4^4*z^15 + 4*x1^35*x2^20*x3^16*x4^4*z^15 + x1^34*x2^21*x3^16*x4^4*z^15 + 2*x1^33*x2^22*x3^16*x4^4*z^15 - 4*x1^32*x2^23*x3^16*x4^4*z^15 - 6*x1^30*x2^25*x3^16*x4^4*z^15 - 2*x1^29*x2^26*x3^16*x4^4*z^15 - x1^35*x2^19*x3^17*x4^4*z^15 - 2*x1^34*x2^20*x3^17*x4^4*z^15 - 2*x1^33*x2^21*x3^17*x4^4*z^15 + x1^31*x2^23*x3^17*x4^4*z^15 + 2*x1^30*x2^24*x3^17*x4^4*z^15 + 6*x1^29*x2^25*x3^17*x4^4*z^15 + x1^34*x2^19*x3^18*x4^4*z^15 + x1^33*x2^20*x3^18*x4^4*z^15 + 2*x1^32*x2^21*x3^18*x4^4*z^15 - 3*x1^31*x2^22*x3^18*x4^4*z^15 - 7*x1^29*x2^24*x3^18*x4^4*z^15 - 2*x1^28*x2^25*x3^18*x4^4*z^15 - x1^32*x2^20*x3^19*x4^4*z^15 - x1^31*x2^21*x3^19*x4^4*z^15 + 6*x1^28*x2^24*x3^19*x4^4*z^15 - x1^26*x2^26*x3^19*x4^4*z^15 + x1^31*x2^20*x3^20*x4^4*z^15 + 2*x1^29*x2^22*x3^20*x4^4*z^15 - 3*x1^28*x2^23*x3^20*x4^4*z^15 + 2*x1^26*x2^25*x3^20*x4^4*z^15 + x1^29*x2^21*x3^21*x4^4*z^15 + 2*x1^27*x2^23*x3^21*x4^4*z^15 - x1^26*x2^24*x3^21*x4^4*z^15 - x1^25*x2^25*x3^21*x4^4*z^15 + x1^27*x2^22*x3^22*x4^4*z^15 + x1^36*x2^27*x3^7*x4^5*z^15 - 3*x1^36*x2^26*x3^8*x4^5*z^15 + x1^34*x2^28*x3^8*x4^5*z^15 - x1^38*x2^23*x3^9*x4^5*z^15 - 2*x1^36*x2^25*x3^9*x4^5*z^15 + x1^35*x2^26*x3^9*x4^5*z^15 + x1^33*x2^28*x3^9*x4^5*z^15 - x1^32*x2^29*x3^9*x4^5*z^15 + 2*x1^38*x2^22*x3^10*x4^5*z^15 + x1^37*x2^23*x3^10*x4^5*z^15 - x1^35*x2^25*x3^10*x4^5*z^15 - 3*x1^34*x2^26*x3^10*x4^5*z^15 - x1^33*x2^27*x3^10*x4^5*z^15 - 3*x1^32*x2^28*x3^10*x4^5*z^15 - 4*x1^37*x2^22*x3^11*x4^5*z^15 + 2*x1^36*x2^23*x3^11*x4^5*z^15 + 4*x1^34*x2^25*x3^11*x4^5*z^15 + x1^33*x2^26*x3^11*x4^5*z^15 + 5*x1^32*x2^27*x3^11*x4^5*z^15 + x1^31*x2^28*x3^11*x4^5*z^15 - x1^30*x2^29*x3^11*x4^5*z^15 + 4*x1^37*x2^21*x3^12*x4^5*z^15 + 2*x1^36*x2^22*x3^12*x4^5*z^15 - x1^35*x2^23*x3^12*x4^5*z^15 - 2*x1^34*x2^24*x3^12*x4^5*z^15 - 2*x1^33*x2^25*x3^12*x4^5*z^15 - x1^32*x2^26*x3^12*x4^5*z^15 - 5*x1^31*x2^27*x3^12*x4^5*z^15 + x1^30*x2^28*x3^12*x4^5*z^15 + x1^29*x2^29*x3^12*x4^5*z^15 - x1^37*x2^20*x3^13*x4^5*z^15 - 5*x1^36*x2^21*x3^13*x4^5*z^15 + x1^35*x2^22*x3^13*x4^5*z^15 - 2*x1^34*x2^23*x3^13*x4^5*z^15 + 3*x1^33*x2^24*x3^13*x4^5*z^15 + 5*x1^31*x2^26*x3^13*x4^5*z^15 + x1^30*x2^27*x3^13*x4^5*z^15 - x1^29*x2^28*x3^13*x4^5*z^15 + 5*x1^36*x2^20*x3^14*x4^5*z^15 + 2*x1^35*x2^21*x3^14*x4^5*z^15 + x1^34*x2^22*x3^14*x4^5*z^15 - x1^33*x2^23*x3^14*x4^5*z^15 - 2*x1^32*x2^24*x3^14*x4^5*z^15 - 6*x1^30*x2^26*x3^14*x4^5*z^15 + 2*x1^29*x2^27*x3^14*x4^5*z^15 - x1^36*x2^19*x3^15*x4^5*z^15 - 5*x1^35*x2^20*x3^15*x4^5*z^15 + x1^34*x2^21*x3^15*x4^5*z^15 - 4*x1^33*x2^22*x3^15*x4^5*z^15 + 4*x1^32*x2^23*x3^15*x4^5*z^15 + 6*x1^30*x2^25*x3^15*x4^5*z^15 + 2*x1^29*x2^26*x3^15*x4^5*z^15 - 2*x1^28*x2^27*x3^15*x4^5*z^15 + 2*x1^35*x2^19*x3^16*x4^5*z^15 + 2*x1^34*x2^20*x3^16*x4^5*z^15 + x1^32*x2^22*x3^16*x4^5*z^15 - 4*x1^31*x2^23*x3^16*x4^5*z^15 - x1^30*x2^24*x3^16*x4^5*z^15 - 6*x1^29*x2^25*x3^16*x4^5*z^15 + 2*x1^28*x2^26*x3^16*x4^5*z^15 + x1^27*x2^27*x3^16*x4^5*z^15 - 2*x1^34*x2^19*x3^17*x4^5*z^15 + x1^33*x2^20*x3^17*x4^5*z^15 + x1^32*x2^21*x3^17*x4^5*z^15 + 5*x1^31*x2^22*x3^17*x4^5*z^15 + 5*x1^29*x2^24*x3^17*x4^5*z^15 + x1^28*x2^25*x3^17*x4^5*z^15 - 2*x1^27*x2^26*x3^17*x4^5*z^15 + x1^33*x2^19*x3^18*x4^5*z^15 - x1^32*x2^20*x3^18*x4^5*z^15 - x1^31*x2^21*x3^18*x4^5*z^15 - x1^30*x2^22*x3^18*x4^5*z^15 + 2*x1^29*x2^23*x3^18*x4^5*z^15 - 6*x1^28*x2^24*x3^18*x4^5*z^15 + 2*x1^27*x2^25*x3^18*x4^5*z^15 + x1^26*x2^26*x3^18*x4^5*z^15 + 2*x1^30*x2^21*x3^19*x4^5*z^15 - x1^29*x2^22*x3^19*x4^5*z^15 + 2*x1^28*x2^23*x3^19*x4^5*z^15 + x1^27*x2^24*x3^19*x4^5*z^15 - 2*x1^26*x2^25*x3^19*x4^5*z^15 - x1^29*x2^21*x3^20*x4^5*z^15 - x1^28*x2^22*x3^20*x4^5*z^15 - 3*x1^27*x2^23*x3^20*x4^5*z^15 + 2*x1^26*x2^24*x3^20*x4^5*z^15 - x1^28*x2^21*x3^21*x4^5*z^15 + 2*x1^27*x2^22*x3^21*x4^5*z^15 - 2*x1^25*x2^24*x3^21*x4^5*z^15 - 2*x1^26*x2^22*x3^22*x4^5*z^15 + x1^25*x2^23*x3^22*x4^5*z^15 + x1^24*x2^24*x3^22*x4^5*z^15 - x1^35*x2^26*x3^8*x4^6*z^15 + x1^38*x2^22*x3^9*x4^6*z^15 + x1^37*x2^23*x3^9*x4^6*z^15 + 2*x1^35*x2^25*x3^9*x4^6*z^15 + 3*x1^34*x2^26*x3^9*x4^6*z^15 + x1^33*x2^27*x3^9*x4^6*z^15 + 2*x1^32*x2^28*x3^9*x4^6*z^15 + x1^31*x2^29*x3^9*x4^6*z^15 + x1^37*x2^22*x3^10*x4^6*z^15 - 2*x1^35*x2^24*x3^10*x4^6*z^15 + x1^32*x2^27*x3^10*x4^6*z^15 - x1^30*x2^29*x3^10*x4^6*z^15 - x1^37*x2^21*x3^11*x4^6*z^15 + x1^36*x2^22*x3^11*x4^6*z^15 + 4*x1^35*x2^23*x3^11*x4^6*z^15 + x1^31*x2^27*x3^11*x4^6*z^15 - x1^30*x2^28*x3^11*x4^6*z^15 - x1^37*x2^20*x3^12*x4^6*z^15 + x1^36*x2^21*x3^12*x4^6*z^15 - 2*x1^35*x2^22*x3^12*x4^6*z^15 - 2*x1^34*x2^23*x3^12*x4^6*z^15 - 2*x1^33*x2^24*x3^12*x4^6*z^15 - x1^32*x2^25*x3^12*x4^6*z^15 + 3*x1^30*x2^27*x3^12*x4^6*z^15 - x1^36*x2^20*x3^13*x4^6*z^15 + x1^35*x2^21*x3^13*x4^6*z^15 + 5*x1^34*x2^22*x3^13*x4^6*z^15 + 2*x1^33*x2^23*x3^13*x4^6*z^15 - 3*x1^31*x2^25*x3^13*x4^6*z^15 + 2*x1^30*x2^26*x3^13*x4^6*z^15 - 4*x1^29*x2^27*x3^13*x4^6*z^15 - x1^36*x2^19*x3^14*x4^6*z^15 + 2*x1^35*x2^20*x3^14*x4^6*z^15 - 4*x1^34*x2^21*x3^14*x4^6*z^15 + x1^33*x2^22*x3^14*x4^6*z^15 - x1^32*x2^23*x3^14*x4^6*z^15 - x1^31*x2^24*x3^14*x4^6*z^15 - x1^30*x2^25*x3^14*x4^6*z^15 + 4*x1^28*x2^27*x3^14*x4^6*z^15 - x1^34*x2^20*x3^15*x4^6*z^15 + 2*x1^33*x2^21*x3^15*x4^6*z^15 + x1^32*x2^22*x3^15*x4^6*z^15 + 3*x1^31*x2^23*x3^15*x4^6*z^15 - 2*x1^30*x2^24*x3^15*x4^6*z^15 + 2*x1^29*x2^25*x3^15*x4^6*z^15 - 4*x1^28*x2^26*x3^15*x4^6*z^15 - x1^27*x2^27*x3^15*x4^6*z^15 - 3*x1^33*x2^20*x3^16*x4^6*z^15 - 2*x1^32*x2^21*x3^16*x4^6*z^15 - 2*x1^31*x2^22*x3^16*x4^6*z^15 - x1^29*x2^24*x3^16*x4^6*z^15 + 4*x1^27*x2^26*x3^16*x4^6*z^15 + x1^33*x2^19*x3^17*x4^6*z^15 + 3*x1^32*x2^20*x3^17*x4^6*z^15 + 3*x1^30*x2^22*x3^17*x4^6*z^15 - x1^29*x2^23*x3^17*x4^6*z^15 + 2*x1^28*x2^24*x3^17*x4^6*z^15 - 4*x1^27*x2^25*x3^17*x4^6*z^15 - 2*x1^26*x2^26*x3^17*x4^6*z^15 - x1^32*x2^19*x3^18*x4^6*z^15 - x1^31*x2^20*x3^18*x4^6*z^15 - x1^30*x2^21*x3^18*x4^6*z^15 - x1^28*x2^23*x3^18*x4^6*z^15 + 2*x1^27*x2^24*x3^18*x4^6*z^15 + 4*x1^26*x2^25*x3^18*x4^6*z^15 + x1^31*x2^19*x3^19*x4^6*z^15 + 2*x1^30*x2^20*x3^19*x4^6*z^15 + x1^29*x2^21*x3^19*x4^6*z^15 - 2*x1^28*x2^22*x3^19*x4^6*z^15 + 3*x1^27*x2^23*x3^19*x4^6*z^15 - 4*x1^26*x2^24*x3^19*x4^6*z^15 - x1^25*x2^25*x3^19*x4^6*z^15 - x1^29*x2^20*x3^20*x4^6*z^15 - x1^26*x2^23*x3^20*x4^6*z^15 + 4*x1^25*x2^24*x3^20*x4^6*z^15 + x1^26*x2^22*x3^21*x4^6*z^15 - 2*x1^25*x2^23*x3^21*x4^6*z^15 - x1^24*x2^24*x3^21*x4^6*z^15 - x1^25*x2^22*x3^22*x4^6*z^15 + x1^24*x2^23*x3^22*x4^6*z^15 - x1^35*x2^26*x3^7*x4^7*z^15 + x1^34*x2^27*x3^7*x4^7*z^15 - x1^31*x2^29*x3^8*x4^7*z^15 - 2*x1^34*x2^25*x3^9*x4^7*z^15 + 2*x1^30*x2^29*x3^9*x4^7*z^15 - x1^37*x2^21*x3^10*x4^7*z^15 - x1^36*x2^22*x3^10*x4^7*z^15 + x1^35*x2^23*x3^10*x4^7*z^15 - x1^33*x2^25*x3^10*x4^7*z^15 + 2*x1^31*x2^27*x3^10*x4^7*z^15 - x1^30*x2^28*x3^10*x4^7*z^15 - x1^29*x2^29*x3^10*x4^7*z^15 + x1^34*x2^23*x3^11*x4^7*z^15 - x1^33*x2^24*x3^11*x4^7*z^15 + x1^32*x2^25*x3^11*x4^7*z^15 - x1^31*x2^26*x3^11*x4^7*z^15 + x1^30*x2^27*x3^11*x4^7*z^15 + 2*x1^29*x2^28*x3^11*x4^7*z^15 - x1^36*x2^20*x3^12*x4^7*z^15 - x1^35*x2^21*x3^12*x4^7*z^15 - x1^33*x2^23*x3^12*x4^7*z^15 + x1^32*x2^24*x3^12*x4^7*z^15 + x1^29*x2^27*x3^12*x4^7*z^15 + x1^36*x2^19*x3^13*x4^7*z^15 + x1^34*x2^21*x3^13*x4^7*z^15 + 2*x1^33*x2^22*x3^13*x4^7*z^15 - 2*x1^35*x2^19*x3^14*x4^7*z^15 - x1^34*x2^20*x3^14*x4^7*z^15 - 2*x1^33*x2^21*x3^14*x4^7*z^15 - x1^32*x2^22*x3^14*x4^7*z^15 + 3*x1^33*x2^20*x3^15*x4^7*z^15 + 2*x1^32*x2^21*x3^15*x4^7*z^15 + 2*x1^30*x2^23*x3^15*x4^7*z^15 + x1^34*x2^18*x3^16*x4^7*z^15 - x1^32*x2^20*x3^16*x4^7*z^15 + x1^31*x2^21*x3^16*x4^7*z^15 - x1^30*x2^22*x3^16*x4^7*z^15 - x1^33*x2^18*x3^17*x4^7*z^15 + x1^32*x2^19*x3^17*x4^7*z^15 + 2*x1^31*x2^20*x3^17*x4^7*z^15 + x1^30*x2^21*x3^17*x4^7*z^15 - x1^31*x2^19*x3^18*x4^7*z^15 - x1^30*x2^20*x3^18*x4^7*z^15 + 2*x1^28*x2^22*x3^18*x4^7*z^15 + 2*x1^26*x2^24*x3^18*x4^7*z^15 - x1^30*x2^19*x3^19*x4^7*z^15 - x1^27*x2^22*x3^19*x4^7*z^15 - x1^26*x2^23*x3^19*x4^7*z^15 - 2*x1^25*x2^24*x3^19*x4^7*z^15 - x1^28*x2^20*x3^20*x4^7*z^15 + 2*x1^25*x2^23*x3^20*x4^7*z^15 - x1^26*x2^21*x3^21*x4^7*z^15 - 2*x1^24*x2^23*x3^21*x4^7*z^15 + x1^23*x2^23*x3^22*x4^7*z^15 - x1^34*x2^25*x3^8*x4^8*z^15 - x1^33*x2^26*x3^8*x4^8*z^15 + x1^31*x2^28*x3^8*x4^8*z^15 + x1^34*x2^24*x3^9*x4^8*z^15 - x1^31*x2^27*x3^9*x4^8*z^15 + x1^33*x2^24*x3^10*x4^8*z^15 - x1^32*x2^25*x3^10*x4^8*z^15 - x1^31*x2^26*x3^10*x4^8*z^15 - 2*x1^29*x2^28*x3^10*x4^8*z^15 - x1^35*x2^21*x3^11*x4^8*z^15 - 2*x1^34*x2^22*x3^11*x4^8*z^15 + 3*x1^31*x2^25*x3^11*x4^8*z^15 + x1^29*x2^27*x3^11*x4^8*z^15 + x1^28*x2^28*x3^11*x4^8*z^15 + x1^35*x2^20*x3^12*x4^8*z^15 - x1^33*x2^22*x3^12*x4^8*z^15 - 2*x1^32*x2^23*x3^12*x4^8*z^15 - 2*x1^31*x2^24*x3^12*x4^8*z^15 - x1^30*x2^25*x3^12*x4^8*z^15 - x1^29*x2^26*x3^12*x4^8*z^15 - 3*x1^28*x2^27*x3^12*x4^8*z^15 - x1^33*x2^21*x3^13*x4^8*z^15 + x1^32*x2^22*x3^13*x4^8*z^15 - x1^31*x2^23*x3^13*x4^8*z^15 + 3*x1^30*x2^24*x3^13*x4^8*z^15 + 2*x1^29*x2^25*x3^13*x4^8*z^15 + 2*x1^28*x2^26*x3^13*x4^8*z^15 + x1^27*x2^27*x3^13*x4^8*z^15 + 2*x1^34*x2^19*x3^14*x4^8*z^15 + x1^32*x2^21*x3^14*x4^8*z^15 - x1^31*x2^22*x3^14*x4^8*z^15 - x1^30*x2^23*x3^14*x4^8*z^15 - 2*x1^29*x2^24*x3^14*x4^8*z^15 - x1^28*x2^25*x3^14*x4^8*z^15 - 3*x1^27*x2^26*x3^14*x4^8*z^15 - x1^33*x2^19*x3^15*x4^8*z^15 - x1^32*x2^20*x3^15*x4^8*z^15 + x1^29*x2^23*x3^15*x4^8*z^15 + 2*x1^28*x2^24*x3^15*x4^8*z^15 + 4*x1^27*x2^25*x3^15*x4^8*z^15 + 2*x1^26*x2^26*x3^15*x4^8*z^15 + x1^33*x2^18*x3^16*x4^8*z^15 + x1^31*x2^20*x3^16*x4^8*z^15 - x1^30*x2^21*x3^16*x4^8*z^15 - x1^29*x2^22*x3^16*x4^8*z^15 - 2*x1^28*x2^23*x3^16*x4^8*z^15 - 2*x1^27*x2^24*x3^16*x4^8*z^15 - 4*x1^26*x2^25*x3^16*x4^8*z^15 - x1^31*x2^19*x3^17*x4^8*z^15 - x1^30*x2^20*x3^17*x4^8*z^15 + 2*x1^29*x2^21*x3^17*x4^8*z^15 + 2*x1^28*x2^22*x3^17*x4^8*z^15 + x1^26*x2^24*x3^17*x4^8*z^15 + x1^25*x2^25*x3^17*x4^8*z^15 - x1^29*x2^20*x3^18*x4^8*z^15 + x1^28*x2^21*x3^18*x4^8*z^15 - x1^27*x2^22*x3^18*x4^8*z^15 + x1^26*x2^23*x3^18*x4^8*z^15 - x1^25*x2^24*x3^18*x4^8*z^15 - x1^28*x2^20*x3^19*x4^8*z^15 - x1^27*x2^21*x3^19*x4^8*z^15 + 2*x1^26*x2^22*x3^19*x4^8*z^15 - x1^25*x2^23*x3^19*x4^8*z^15 + x1^27*x2^20*x3^20*x4^8*z^15 - x1^25*x2^22*x3^20*x4^8*z^15 + x1^24*x2^23*x3^20*x4^8*z^15 + x1^33*x2^24*x3^9*x4^9*z^15 + x1^32*x2^25*x3^9*x4^9*z^15 + 2*x1^31*x2^26*x3^9*x4^9*z^15 + x1^29*x2^28*x3^9*x4^9*z^15 - x1^33*x2^23*x3^10*x4^9*z^15 - 2*x1^31*x2^25*x3^10*x4^9*z^15 - x1^28*x2^28*x3^10*x4^9*z^15 + 3*x1^32*x2^23*x3^11*x4^9*z^15 + x1^31*x2^24*x3^11*x4^9*z^15 + 2*x1^30*x2^25*x3^11*x4^9*z^15 + x1^29*x2^26*x3^11*x4^9*z^15 + x1^28*x2^27*x3^11*x4^9*z^15 + x1^34*x2^20*x3^12*x4^9*z^15 + 2*x1^33*x2^21*x3^12*x4^9*z^15 - x1^32*x2^22*x3^12*x4^9*z^15 - x1^30*x2^24*x3^12*x4^9*z^15 - 2*x1^29*x2^25*x3^12*x4^9*z^15 - x1^28*x2^26*x3^12*x4^9*z^15 - x1^33*x2^20*x3^13*x4^9*z^15 - x1^32*x2^21*x3^13*x4^9*z^15 + 5*x1^31*x2^22*x3^13*x4^9*z^15 + x1^30*x2^23*x3^13*x4^9*z^15 + 3*x1^29*x2^24*x3^13*x4^9*z^15 + 2*x1^28*x2^25*x3^13*x4^9*z^15 + x1^27*x2^26*x3^13*x4^9*z^15 + x1^34*x2^18*x3^14*x4^9*z^15 + x1^33*x2^19*x3^14*x4^9*z^15 + x1^32*x2^20*x3^14*x4^9*z^15 - 3*x1^31*x2^21*x3^14*x4^9*z^15 - x1^30*x2^22*x3^14*x4^9*z^15 - x1^29*x2^23*x3^14*x4^9*z^15 - 3*x1^28*x2^24*x3^14*x4^9*z^15 - 2*x1^27*x2^25*x3^14*x4^9*z^15 - x1^26*x2^26*x3^14*x4^9*z^15 - 2*x1^33*x2^18*x3^15*x4^9*z^15 + x1^32*x2^19*x3^15*x4^9*z^15 + 3*x1^30*x2^21*x3^15*x4^9*z^15 + x1^29*x2^22*x3^15*x4^9*z^15 + 4*x1^28*x2^23*x3^15*x4^9*z^15 + 2*x1^27*x2^24*x3^15*x4^9*z^15 + x1^26*x2^25*x3^15*x4^9*z^15 + x1^32*x2^18*x3^16*x4^9*z^15 - x1^30*x2^20*x3^16*x4^9*z^15 - 3*x1^29*x2^21*x3^16*x4^9*z^15 - 3*x1^28*x2^22*x3^16*x4^9*z^15 - 4*x1^27*x2^23*x3^16*x4^9*z^15 - x1^26*x2^24*x3^16*x4^9*z^15 - x1^25*x2^25*x3^16*x4^9*z^15 - x1^31*x2^18*x3^17*x4^9*z^15 + 2*x1^29*x2^20*x3^17*x4^9*z^15 + 5*x1^27*x2^22*x3^17*x4^9*z^15 + 2*x1^26*x2^23*x3^17*x4^9*z^15 + x1^25*x2^24*x3^17*x4^9*z^15 + x1^30*x2^18*x3^18*x4^9*z^15 - x1^29*x2^19*x3^18*x4^9*z^15 - x1^28*x2^20*x3^18*x4^9*z^15 - 3*x1^27*x2^21*x3^18*x4^9*z^15 - 5*x1^26*x2^22*x3^18*x4^9*z^15 - x1^25*x2^23*x3^18*x4^9*z^15 + 2*x1^28*x2^19*x3^19*x4^9*z^15 + 3*x1^26*x2^21*x3^19*x4^9*z^15 + x1^25*x2^22*x3^19*x4^9*z^15 + x1^24*x2^23*x3^19*x4^9*z^15 + x1^26*x2^20*x3^20*x4^9*z^15 - 3*x1^25*x2^21*x3^20*x4^9*z^15 - x1^23*x2^23*x3^20*x4^9*z^15 + 2*x1^24*x2^21*x3^21*x4^9*z^15 - x1^31*x2^23*x3^11*x4^10*z^15 + 2*x1^30*x2^24*x3^11*x4^10*z^15 + x1^28*x2^26*x3^11*x4^10*z^15 + x1^27*x2^27*x3^11*x4^10*z^15 - x1^31*x2^22*x3^12*x4^10*z^15 - x1^30*x2^23*x3^12*x4^10*z^15 - x1^29*x2^24*x3^12*x4^10*z^15 - x1^33*x2^19*x3^13*x4^10*z^15 + x1^31*x2^21*x3^13*x4^10*z^15 - x1^30*x2^22*x3^13*x4^10*z^15 + x1^29*x2^23*x3^13*x4^10*z^15 - x1^28*x2^24*x3^13*x4^10*z^15 + x1^27*x2^25*x3^13*x4^10*z^15 + x1^26*x2^26*x3^13*x4^10*z^15 - x1^33*x2^18*x3^14*x4^10*z^15 - x1^32*x2^19*x3^14*x4^10*z^15 + 2*x1^31*x2^20*x3^14*x4^10*z^15 - x1^30*x2^21*x3^14*x4^10*z^15 - x1^29*x2^22*x3^14*x4^10*z^15 - x1^27*x2^24*x3^14*x4^10*z^15 - x1^31*x2^19*x3^15*x4^10*z^15 - x1^30*x2^20*x3^15*x4^10*z^15 - x1^28*x2^22*x3^15*x4^10*z^15 - 2*x1^27*x2^23*x3^15*x4^10*z^15 + 2*x1^26*x2^24*x3^15*x4^10*z^15 + 2*x1^28*x2^21*x3^16*x4^10*z^15 - x1^27*x2^22*x3^16*x4^10*z^15 - x1^26*x2^23*x3^16*x4^10*z^15 - x1^30*x2^18*x3^17*x4^10*z^15 + x1^28*x2^20*x3^17*x4^10*z^15 + x1^27*x2^21*x3^17*x4^10*z^15 + x1^26*x2^22*x3^17*x4^10*z^15 + x1^29*x2^18*x3^18*x4^10*z^15 + x1^27*x2^20*x3^18*x4^10*z^15 - x1^26*x2^21*x3^18*x4^10*z^15 + x1^25*x2^22*x3^18*x4^10*z^15 - x1^26*x2^20*x3^19*x4^10*z^15 + 2*x1^25*x2^21*x3^19*x4^10*z^15 - x1^24*x2^21*x3^20*x4^10*z^15 - x1^31*x2^22*x3^11*x4^11*z^15 - x1^30*x2^23*x3^11*x4^11*z^15 - x1^29*x2^24*x3^11*x4^11*z^15 + x1^31*x2^21*x3^12*x4^11*z^15 - 3*x1^29*x2^23*x3^12*x4^11*z^15 - x1^28*x2^24*x3^12*x4^11*z^15 - x1^27*x2^25*x3^12*x4^11*z^15 + x1^29*x2^22*x3^13*x4^11*z^15 - x1^28*x2^23*x3^13*x4^11*z^15 - x1^27*x2^24*x3^13*x4^11*z^15 + x1^26*x2^25*x3^13*x4^11*z^15 + x1^29*x2^21*x3^14*x4^11*z^15 - 2*x1^28*x2^22*x3^14*x4^11*z^15 + 2*x1^27*x2^23*x3^14*x4^11*z^15 - 2*x1^26*x2^24*x3^14*x4^11*z^15 - x1^25*x2^25*x3^14*x4^11*z^15 - 3*x1^30*x2^19*x3^15*x4^11*z^15 - x1^29*x2^20*x3^15*x4^11*z^15 + 2*x1^28*x2^21*x3^15*x4^11*z^15 - x1^27*x2^22*x3^15*x4^11*z^15 - 4*x1^26*x2^23*x3^15*x4^11*z^15 + 2*x1^25*x2^24*x3^15*x4^11*z^15 + x1^29*x2^19*x3^16*x4^11*z^15 - x1^27*x2^21*x3^16*x4^11*z^15 + 2*x1^26*x2^22*x3^16*x4^11*z^15 - 2*x1^25*x2^23*x3^16*x4^11*z^15 - x1^24*x2^24*x3^16*x4^11*z^15 - x1^27*x2^20*x3^17*x4^11*z^15 + x1^25*x2^22*x3^17*x4^11*z^15 + x1^24*x2^23*x3^17*x4^11*z^15 + x1^25*x2^21*x3^18*x4^11*z^15 - x1^24*x2^22*x3^18*x4^11*z^15 + x1^23*x2^22*x3^19*x4^11*z^15 + x1^30*x2^21*x3^12*x4^12*z^15 + x1^29*x2^22*x3^12*x4^12*z^15 + x1^28*x2^23*x3^12*x4^12*z^15 - x1^30*x2^20*x3^13*x4^12*z^15 + 3*x1^28*x2^22*x3^13*x4^12*z^15 - x1^27*x2^23*x3^13*x4^12*z^15 + x1^26*x2^24*x3^13*x4^12*z^15 - x1^28*x2^21*x3^14*x4^12*z^15 + x1^26*x2^23*x3^14*x4^12*z^15 - 3*x1^25*x2^24*x3^14*x4^12*z^15 + x1^29*x2^19*x3^15*x4^12*z^15 + x1^28*x2^20*x3^15*x4^12*z^15 + 4*x1^27*x2^21*x3^15*x4^12*z^15 - 2*x1^26*x2^22*x3^15*x4^12*z^15 + 2*x1^25*x2^23*x3^15*x4^12*z^15 + 2*x1^24*x2^24*x3^15*x4^12*z^15 + x1^28*x2^19*x3^16*x4^12*z^15 - x1^27*x2^20*x3^16*x4^12*z^15 + 2*x1^25*x2^22*x3^16*x4^12*z^15 - 5*x1^24*x2^23*x3^16*x4^12*z^15 + 2*x1^26*x2^20*x3^17*x4^12*z^15 - 2*x1^25*x2^21*x3^17*x4^12*z^15 + 2*x1^24*x2^22*x3^17*x4^12*z^15 + 2*x1^23*x2^23*x3^17*x4^12*z^15 + x1^24*x2^21*x3^18*x4^12*z^15 - 3*x1^23*x2^22*x3^18*x4^12*z^15 + x1^22*x2^22*x3^19*x4^12*z^15 - x1^28*x2^20*x3^14*x4^13*z^15 + 2*x1^26*x2^22*x3^14*x4^13*z^15 + x1^26*x2^21*x3^15*x4^13*z^15 + 2*x1^24*x2^23*x3^15*x4^13*z^15 - x1^26*x2^20*x3^16*x4^13*z^15 + x1^25*x2^21*x3^16*x4^13*z^15 - x1^23*x2^23*x3^16*x4^13*z^15 + x1^25*x2^20*x3^17*x4^13*z^15 + x1^24*x2^21*x3^17*x4^13*z^15 + 2*x1^23*x2^22*x3^17*x4^13*z^15 - x1^25*x2^21*x3^15*x4^14*z^15 + x1^24*x2^21*x3^16*x4^14*z^15 - x1^23*x2^20*x3^17*x4^15*z^15 - x1^38*x2^25*x3^7*z^14 + x1^37*x2^25*x3^8*z^14 - 2*x1^37*x2^24*x3^9*z^14 - x1^35*x2^26*x3^9*z^14 + x1^37*x2^23*x3^10*z^14 + 2*x1^36*x2^24*x3^10*z^14 + 2*x1^34*x2^26*x3^10*z^14 + x1^33*x2^27*x3^10*z^14 - 2*x1^36*x2^23*x3^11*z^14 - x1^35*x2^24*x3^11*z^14 - x1^34*x2^25*x3^11*z^14 + x1^31*x2^28*x3^11*z^14 + x1^36*x2^22*x3^12*z^14 + 2*x1^35*x2^23*x3^12*z^14 + x1^33*x2^25*x3^12*z^14 - 2*x1^32*x2^26*x3^12*z^14 - x1^30*x2^28*x3^12*z^14 - 2*x1^35*x2^22*x3^13*z^14 + x1^29*x2^28*x3^13*z^14 + 2*x1^34*x2^22*x3^14*z^14 + 2*x1^32*x2^24*x3^14*z^14 - x1^31*x2^25*x3^14*z^14 - 2*x1^29*x2^27*x3^14*z^14 - x1^28*x2^28*x3^14*z^14 - 2*x1^34*x2^21*x3^15*z^14 - x1^33*x2^22*x3^15*z^14 - x1^32*x2^23*x3^15*z^14 + x1^28*x2^27*x3^15*z^14 + x1^34*x2^20*x3^16*z^14 + 2*x1^33*x2^21*x3^16*z^14 + x1^31*x2^23*x3^16*z^14 + x1^30*x2^24*x3^16*z^14 - 2*x1^28*x2^26*x3^16*z^14 - 2*x1^33*x2^20*x3^17*z^14 - x1^32*x2^21*x3^17*z^14 - x1^31*x2^22*x3^17*z^14 - x1^30*x2^23*x3^17*z^14 + x1^29*x2^24*x3^17*z^14 + x1^28*x2^25*x3^17*z^14 + x1^32*x2^20*x3^18*z^14 + x1^30*x2^22*x3^18*z^14 - x1^30*x2^21*x3^19*z^14 - x1^29*x2^22*x3^19*z^14 + x1^36*x2^27*x3^6*x4*z^14 - x1^38*x2^24*x3^7*x4*z^14 - 2*x1^37*x2^25*x3^7*x4*z^14 + x1^36*x2^26*x3^7*x4*z^14 - x1^35*x2^27*x3^7*x4*z^14 + 4*x1^37*x2^24*x3^8*x4*z^14 + x1^35*x2^26*x3^8*x4*z^14 - 2*x1^37*x2^23*x3^9*x4*z^14 - 6*x1^36*x2^24*x3^9*x4*z^14 + 2*x1^35*x2^25*x3^9*x4*z^14 - 2*x1^34*x2^26*x3^9*x4*z^14 + 2*x1^33*x2^27*x3^9*x4*z^14 + 6*x1^36*x2^23*x3^10*x4*z^14 + 2*x1^35*x2^24*x3^10*x4*z^14 - x1^33*x2^26*x3^10*x4*z^14 - 2*x1^32*x2^27*x3^10*x4*z^14 - x1^31*x2^28*x3^10*x4*z^14 - 2*x1^36*x2^22*x3^11*x4*z^14 - 6*x1^35*x2^23*x3^11*x4*z^14 - 4*x1^33*x2^25*x3^11*x4*z^14 + 3*x1^32*x2^26*x3^11*x4*z^14 + 2*x1^31*x2^27*x3^11*x4*z^14 + 3*x1^30*x2^28*x3^11*x4*z^14 + 6*x1^35*x2^22*x3^12*x4*z^14 + 2*x1^34*x2^23*x3^12*x4*z^14 + 2*x1^33*x2^24*x3^12*x4*z^14 - 2*x1^31*x2^26*x3^12*x4*z^14 - 4*x1^30*x2^27*x3^12*x4*z^14 - 3*x1^29*x2^28*x3^12*x4*z^14 - 2*x1^35*x2^21*x3^13*x4*z^14 - 6*x1^34*x2^22*x3^13*x4*z^14 - 4*x1^32*x2^24*x3^13*x4*z^14 + 4*x1^31*x2^25*x3^13*x4*z^14 + 6*x1^29*x2^27*x3^13*x4*z^14 + x1^28*x2^28*x3^13*x4*z^14 + 6*x1^34*x2^21*x3^14*x4*z^14 + 2*x1^33*x2^22*x3^14*x4*z^14 + 2*x1^32*x2^23*x3^14*x4*z^14 - 2*x1^30*x2^25*x3^14*x4*z^14 - 2*x1^29*x2^26*x3^14*x4*z^14 - 6*x1^28*x2^27*x3^14*x4*z^14 - x1^34*x2^20*x3^15*x4*z^14 - 6*x1^33*x2^21*x3^15*x4*z^14 - 4*x1^31*x2^23*x3^15*x4*z^14 + 4*x1^30*x2^24*x3^15*x4*z^14 + 6*x1^28*x2^26*x3^15*x4*z^14 + 2*x1^27*x2^27*x3^15*x4*z^14 + 4*x1^33*x2^20*x3^16*x4*z^14 + 3*x1^32*x2^21*x3^16*x4*z^14 + 2*x1^31*x2^22*x3^16*x4*z^14 - x1^29*x2^24*x3^16*x4*z^14 - x1^28*x2^25*x3^16*x4*z^14 - 5*x1^27*x2^26*x3^16*x4*z^14 - 4*x1^32*x2^20*x3^17*x4*z^14 - x1^31*x2^21*x3^17*x4*z^14 - 4*x1^30*x2^22*x3^17*x4*z^14 + x1^28*x2^24*x3^17*x4*z^14 + 5*x1^27*x2^25*x3^17*x4*z^14 + x1^31*x2^20*x3^18*x4*z^14 + x1^30*x2^21*x3^18*x4*z^14 + 2*x1^29*x2^22*x3^18*x4*z^14 + x1^28*x2^23*x3^18*x4*z^14 - 2*x1^27*x2^24*x3^18*x4*z^14 - 2*x1^26*x2^25*x3^18*x4*z^14 - x1^30*x2^20*x3^19*x4*z^14 - 2*x1^29*x2^21*x3^19*x4*z^14 - 2*x1^28*x2^22*x3^19*x4*z^14 + x1^27*x2^23*x3^19*x4*z^14 + 3*x1^26*x2^24*x3^19*x4*z^14 + x1^29*x2^20*x3^20*x4*z^14 + x1^28*x2^21*x3^20*x4*z^14 + x1^27*x2^22*x3^20*x4*z^14 - x1^25*x2^24*x3^20*x4*z^14 + x1^37*x2^25*x3^6*x4^2*z^14 - x1^36*x2^26*x3^6*x4^2*z^14 + x1^35*x2^27*x3^6*x4^2*z^14 - x1^37*x2^24*x3^7*x4^2*z^14 - x1^36*x2^25*x3^7*x4^2*z^14 + 2*x1^33*x2^28*x3^7*x4^2*z^14 + 4*x1^36*x2^24*x3^8*x4^2*z^14 - 2*x1^33*x2^27*x3^8*x4^2*z^14 - 2*x1^32*x2^28*x3^8*x4^2*z^14 - 3*x1^36*x2^23*x3^9*x4^2*z^14 - 3*x1^35*x2^24*x3^9*x4^2*z^14 - x1^34*x2^25*x3^9*x4^2*z^14 + x1^33*x2^26*x3^9*x4^2*z^14 + x1^32*x2^27*x3^9*x4^2*z^14 + 2*x1^31*x2^28*x3^9*x4^2*z^14 + 2*x1^36*x2^22*x3^10*x4^2*z^14 + 5*x1^35*x2^23*x3^10*x4^2*z^14 - x1^34*x2^24*x3^10*x4^2*z^14 + 3*x1^33*x2^25*x3^10*x4^2*z^14 - 3*x1^32*x2^26*x3^10*x4^2*z^14 - x1^31*x2^27*x3^10*x4^2*z^14 - 4*x1^30*x2^28*x3^10*x4^2*z^14 - 6*x1^35*x2^22*x3^11*x4^2*z^14 - 2*x1^34*x2^23*x3^11*x4^2*z^14 - 2*x1^33*x2^24*x3^11*x4^2*z^14 + x1^31*x2^26*x3^11*x4^2*z^14 + 2*x1^30*x2^27*x3^11*x4^2*z^14 + 4*x1^29*x2^28*x3^11*x4^2*z^14 + 2*x1^35*x2^21*x3^12*x4^2*z^14 + 6*x1^34*x2^22*x3^12*x4^2*z^14 + 4*x1^32*x2^24*x3^12*x4^2*z^14 - 4*x1^31*x2^25*x3^12*x4^2*z^14 - 6*x1^29*x2^27*x3^12*x4^2*z^14 - 2*x1^28*x2^28*x3^12*x4^2*z^14 - 6*x1^34*x2^21*x3^13*x4^2*z^14 - 2*x1^33*x2^22*x3^13*x4^2*z^14 - 2*x1^32*x2^23*x3^13*x4^2*z^14 + 2*x1^30*x2^25*x3^13*x4^2*z^14 + 2*x1^29*x2^26*x3^13*x4^2*z^14 + 6*x1^28*x2^27*x3^13*x4^2*z^14 + 6*x1^33*x2^21*x3^14*x4^2*z^14 + 4*x1^31*x2^23*x3^14*x4^2*z^14 - 4*x1^30*x2^24*x3^14*x4^2*z^14 - 6*x1^28*x2^26*x3^14*x4^2*z^14 - 2*x1^27*x2^27*x3^14*x4^2*z^14 - 3*x1^33*x2^20*x3^15*x4^2*z^14 - 3*x1^32*x2^21*x3^15*x4^2*z^14 - 2*x1^31*x2^22*x3^15*x4^2*z^14 + 2*x1^29*x2^24*x3^15*x4^2*z^14 + 2*x1^28*x2^25*x3^15*x4^2*z^14 + 6*x1^27*x2^26*x3^15*x4^2*z^14 + 3*x1^32*x2^20*x3^16*x4^2*z^14 + 2*x1^30*x2^22*x3^16*x4^2*z^14 - 4*x1^29*x2^23*x3^16*x4^2*z^14 - 6*x1^27*x2^25*x3^16*x4^2*z^14 - 2*x1^26*x2^26*x3^16*x4^2*z^14 - x1^32*x2^19*x3^17*x4^2*z^14 - x1^31*x2^20*x3^17*x4^2*z^14 - x1^30*x2^21*x3^17*x4^2*z^14 + x1^29*x2^22*x3^17*x4^2*z^14 + x1^28*x2^23*x3^17*x4^2*z^14 + 2*x1^27*x2^24*x3^17*x4^2*z^14 + 6*x1^26*x2^25*x3^17*x4^2*z^14 + x1^31*x2^19*x3^18*x4^2*z^14 + x1^29*x2^21*x3^18*x4^2*z^14 - 3*x1^28*x2^22*x3^18*x4^2*z^14 - 5*x1^26*x2^24*x3^18*x4^2*z^14 - x1^25*x2^25*x3^18*x4^2*z^14 + x1^28*x2^21*x3^19*x4^2*z^14 + x1^26*x2^23*x3^19*x4^2*z^14 + 4*x1^25*x2^24*x3^19*x4^2*z^14 - x1^26*x2^22*x3^20*x4^2*z^14 - 2*x1^25*x2^23*x3^20*x4^2*z^14 + x1^24*x2^23*x3^21*x4^2*z^14 - x1^37*x2^24*x3^6*x4^3*z^14 - x1^34*x2^27*x3^6*x4^3*z^14 + x1^38*x2^22*x3^7*x4^3*z^14 - x1^35*x2^25*x3^7*x4^3*z^14 - x1^34*x2^26*x3^7*x4^3*z^14 - x1^37*x2^22*x3^8*x4^3*z^14 + x1^36*x2^23*x3^8*x4^3*z^14 - x1^35*x2^24*x3^8*x4^3*z^14 - x1^34*x2^25*x3^8*x4^3*z^14 - x1^31*x2^28*x3^8*x4^3*z^14 + 2*x1^37*x2^21*x3^9*x4^3*z^14 + 2*x1^36*x2^22*x3^9*x4^3*z^14 + x1^35*x2^23*x3^9*x4^3*z^14 + x1^32*x2^26*x3^9*x4^3*z^14 + x1^30*x2^28*x3^9*x4^3*z^14 - x1^37*x2^20*x3^10*x4^3*z^14 - 2*x1^36*x2^21*x3^10*x4^3*z^14 + x1^35*x2^22*x3^10*x4^3*z^14 + x1^34*x2^23*x3^10*x4^3*z^14 + x1^33*x2^24*x3^10*x4^3*z^14 + x1^31*x2^26*x3^10*x4^3*z^14 + x1^30*x2^27*x3^10*x4^3*z^14 - x1^29*x2^28*x3^10*x4^3*z^14 + 2*x1^36*x2^20*x3^11*x4^3*z^14 + x1^33*x2^23*x3^11*x4^3*z^14 - x1^32*x2^24*x3^11*x4^3*z^14 - x1^30*x2^26*x3^11*x4^3*z^14 + x1^29*x2^27*x3^11*x4^3*z^14 + x1^28*x2^28*x3^11*x4^3*z^14 - x1^36*x2^19*x3^12*x4^3*z^14 - 2*x1^35*x2^20*x3^12*x4^3*z^14 + 2*x1^34*x2^21*x3^12*x4^3*z^14 + 3*x1^32*x2^23*x3^12*x4^3*z^14 + x1^30*x2^25*x3^12*x4^3*z^14 - x1^29*x2^26*x3^12*x4^3*z^14 - x1^28*x2^27*x3^12*x4^3*z^14 + 2*x1^35*x2^19*x3^13*x4^3*z^14 - x1^34*x2^20*x3^13*x4^3*z^14 - 2*x1^33*x2^21*x3^13*x4^3*z^14 - x1^31*x2^23*x3^13*x4^3*z^14 + 2*x1^30*x2^24*x3^13*x4^3*z^14 - 2*x1^29*x2^25*x3^13*x4^3*z^14 + 2*x1^28*x2^26*x3^13*x4^3*z^14 - 2*x1^34*x2^19*x3^14*x4^3*z^14 + 2*x1^33*x2^20*x3^14*x4^3*z^14 - 2*x1^32*x2^21*x3^14*x4^3*z^14 + x1^31*x2^22*x3^14*x4^3*z^14 + 2*x1^29*x2^24*x3^14*x4^3*z^14 + x1^28*x2^25*x3^14*x4^3*z^14 - 2*x1^27*x2^26*x3^14*x4^3*z^14 + x1^34*x2^18*x3^15*x4^3*z^14 + 2*x1^33*x2^19*x3^15*x4^3*z^14 - x1^32*x2^20*x3^15*x4^3*z^14 - x1^31*x2^21*x3^15*x4^3*z^14 - 3*x1^30*x2^22*x3^15*x4^3*z^14 - 2*x1^28*x2^24*x3^15*x4^3*z^14 + 2*x1^27*x2^25*x3^15*x4^3*z^14 + x1^26*x2^26*x3^15*x4^3*z^14 - x1^33*x2^18*x3^16*x4^3*z^14 - x1^32*x2^19*x3^16*x4^3*z^14 + x1^31*x2^20*x3^16*x4^3*z^14 + 3*x1^30*x2^21*x3^16*x4^3*z^14 + x1^28*x2^23*x3^16*x4^3*z^14 - 2*x1^26*x2^25*x3^16*x4^3*z^14 + x1^31*x2^19*x3^17*x4^3*z^14 - x1^30*x2^20*x3^17*x4^3*z^14 - x1^29*x2^21*x3^17*x4^3*z^14 - 2*x1^27*x2^23*x3^17*x4^3*z^14 + 2*x1^26*x2^24*x3^17*x4^3*z^14 + x1^25*x2^25*x3^17*x4^3*z^14 - x1^30*x2^19*x3^18*x4^3*z^14 + x1^29*x2^20*x3^18*x4^3*z^14 - 2*x1^26*x2^23*x3^18*x4^3*z^14 - 3*x1^25*x2^24*x3^18*x4^3*z^14 + x1^28*x2^20*x3^19*x4^3*z^14 - 3*x1^26*x2^22*x3^19*x4^3*z^14 + x1^24*x2^24*x3^19*x4^3*z^14 - x1^27*x2^20*x3^20*x4^3*z^14 + 2*x1^26*x2^21*x3^20*x4^3*z^14 + x1^25*x2^22*x3^20*x4^3*z^14 - 2*x1^24*x2^23*x3^20*x4^3*z^14 + x1^37*x2^22*x3^7*x4^4*z^14 + 2*x1^36*x2^23*x3^7*x4^4*z^14 + x1^35*x2^24*x3^7*x4^4*z^14 - 2*x1^34*x2^25*x3^7*x4^4*z^14 + x1^33*x2^26*x3^7*x4^4*z^14 + x1^32*x2^27*x3^7*x4^4*z^14 - 2*x1^37*x2^21*x3^8*x4^4*z^14 - x1^36*x2^22*x3^8*x4^4*z^14 - x1^35*x2^23*x3^8*x4^4*z^14 + 3*x1^34*x2^24*x3^8*x4^4*z^14 - x1^31*x2^27*x3^8*x4^4*z^14 + 4*x1^36*x2^21*x3^9*x4^4*z^14 + x1^34*x2^23*x3^9*x4^4*z^14 - 2*x1^33*x2^24*x3^9*x4^4*z^14 - 2*x1^32*x2^25*x3^9*x4^4*z^14 - 3*x1^31*x2^26*x3^9*x4^4*z^14 + 2*x1^30*x2^27*x3^9*x4^4*z^14 - 4*x1^36*x2^20*x3^10*x4^4*z^14 - 3*x1^35*x2^21*x3^10*x4^4*z^14 - x1^34*x2^22*x3^10*x4^4*z^14 + 2*x1^33*x2^23*x3^10*x4^4*z^14 + x1^32*x2^24*x3^10*x4^4*z^14 + 2*x1^31*x2^25*x3^10*x4^4*z^14 + 2*x1^30*x2^26*x3^10*x4^4*z^14 - x1^29*x2^27*x3^10*x4^4*z^14 + x1^36*x2^19*x3^11*x4^4*z^14 + 5*x1^35*x2^20*x3^11*x4^4*z^14 + x1^34*x2^21*x3^11*x4^4*z^14 + 3*x1^33*x2^22*x3^11*x4^4*z^14 - 2*x1^32*x2^23*x3^11*x4^4*z^14 - 6*x1^30*x2^25*x3^11*x4^4*z^14 - x1^29*x2^26*x3^11*x4^4*z^14 + x1^28*x2^27*x3^11*x4^4*z^14 - 6*x1^35*x2^19*x3^12*x4^4*z^14 - 3*x1^34*x2^20*x3^12*x4^4*z^14 - 3*x1^33*x2^21*x3^12*x4^4*z^14 + x1^32*x2^22*x3^12*x4^4*z^14 + x1^31*x2^23*x3^12*x4^4*z^14 + 2*x1^30*x2^24*x3^12*x4^4*z^14 + 6*x1^29*x2^25*x3^12*x4^4*z^14 - x1^27*x2^27*x3^12*x4^4*z^14 + x1^35*x2^18*x3^13*x4^4*z^14 + 6*x1^34*x2^19*x3^13*x4^4*z^14 + 4*x1^32*x2^21*x3^13*x4^4*z^14 - 4*x1^31*x2^22*x3^13*x4^4*z^14 - 6*x1^29*x2^24*x3^13*x4^4*z^14 - 2*x1^28*x2^25*x3^13*x4^4*z^14 - 4*x1^34*x2^18*x3^14*x4^4*z^14 - 2*x1^33*x2^19*x3^14*x4^4*z^14 - x1^32*x2^20*x3^14*x4^4*z^14 + 2*x1^30*x2^22*x3^14*x4^4*z^14 + 2*x1^29*x2^23*x3^14*x4^4*z^14 + 6*x1^28*x2^24*x3^14*x4^4*z^14 + 4*x1^33*x2^18*x3^15*x4^4*z^14 + 4*x1^31*x2^20*x3^15*x4^4*z^14 - 3*x1^30*x2^21*x3^15*x4^4*z^14 - 6*x1^28*x2^23*x3^15*x4^4*z^14 - 2*x1^27*x2^24*x3^15*x4^4*z^14 - x1^32*x2^18*x3^16*x4^4*z^14 + 2*x1^29*x2^21*x3^16*x4^4*z^14 + 3*x1^28*x2^22*x3^16*x4^4*z^14 + 6*x1^27*x2^23*x3^16*x4^4*z^14 + x1^31*x2^18*x3^17*x4^4*z^14 + x1^30*x2^19*x3^17*x4^4*z^14 - 2*x1^29*x2^20*x3^17*x4^4*z^14 + x1^28*x2^21*x3^17*x4^4*z^14 - 5*x1^27*x2^22*x3^17*x4^4*z^14 - 2*x1^26*x2^23*x3^17*x4^4*z^14 - x1^30*x2^18*x3^18*x4^4*z^14 - x1^28*x2^20*x3^18*x4^4*z^14 + 5*x1^26*x2^22*x3^18*x4^4*z^14 + x1^25*x2^23*x3^18*x4^4*z^14 - x1^28*x2^19*x3^19*x4^4*z^14 + x1^27*x2^20*x3^19*x4^4*z^14 - 2*x1^26*x2^21*x3^19*x4^4*z^14 - x1^25*x2^22*x3^19*x4^4*z^14 - 2*x1^26*x2^20*x3^20*x4^4*z^14 + 2*x1^25*x2^21*x3^20*x4^4*z^14 + x1^24*x2^22*x3^20*x4^4*z^14 - 2*x1^23*x2^23*x3^20*x4^4*z^14 - x1^24*x2^21*x3^21*x4^4*z^14 + x1^23*x2^22*x3^21*x4^4*z^14 + x1^34*x2^25*x3^6*x4^5*z^14 - x1^34*x2^24*x3^7*x4^5*z^14 - 2*x1^33*x2^25*x3^7*x4^5*z^14 - x1^32*x2^26*x3^7*x4^5*z^14 - x1^31*x2^27*x3^7*x4^5*z^14 + x1^35*x2^22*x3^8*x4^5*z^14 + 4*x1^33*x2^24*x3^8*x4^5*z^14 + 2*x1^32*x2^25*x3^8*x4^5*z^14 + x1^36*x2^20*x3^9*x4^5*z^14 - 3*x1^33*x2^23*x3^9*x4^5*z^14 + x1^31*x2^25*x3^9*x4^5*z^14 - 2*x1^30*x2^26*x3^9*x4^5*z^14 - 3*x1^35*x2^20*x3^10*x4^5*z^14 + 3*x1^32*x2^23*x3^10*x4^5*z^14 + 4*x1^30*x2^25*x3^10*x4^5*z^14 + 2*x1^29*x2^26*x3^10*x4^5*z^14 + x1^28*x2^27*x3^10*x4^5*z^14 + 2*x1^35*x2^19*x3^11*x4^5*z^14 + 2*x1^34*x2^20*x3^11*x4^5*z^14 - x1^33*x2^21*x3^11*x4^5*z^14 - 3*x1^32*x2^22*x3^11*x4^5*z^14 - 3*x1^31*x2^23*x3^11*x4^5*z^14 - x1^30*x2^24*x3^11*x4^5*z^14 - 7*x1^29*x2^25*x3^11*x4^5*z^14 + x1^28*x2^26*x3^11*x4^5*z^14 - 5*x1^34*x2^19*x3^12*x4^5*z^14 - x1^32*x2^21*x3^12*x4^5*z^14 + 4*x1^31*x2^22*x3^12*x4^5*z^14 + 5*x1^29*x2^24*x3^12*x4^5*z^14 + x1^28*x2^25*x3^12*x4^5*z^14 - 2*x1^27*x2^26*x3^12*x4^5*z^14 + 3*x1^34*x2^18*x3^13*x4^5*z^14 + 3*x1^33*x2^19*x3^13*x4^5*z^14 - x1^32*x2^20*x3^13*x4^5*z^14 - 2*x1^31*x2^21*x3^13*x4^5*z^14 - x1^30*x2^22*x3^13*x4^5*z^14 - x1^29*x2^23*x3^13*x4^5*z^14 - 6*x1^28*x2^24*x3^13*x4^5*z^14 + 2*x1^27*x2^25*x3^13*x4^5*z^14 + x1^26*x2^26*x3^13*x4^5*z^14 - 4*x1^33*x2^18*x3^14*x4^5*z^14 - 4*x1^31*x2^20*x3^14*x4^5*z^14 + 4*x1^30*x2^21*x3^14*x4^5*z^14 + 5*x1^28*x2^23*x3^14*x4^5*z^14 + x1^27*x2^24*x3^14*x4^5*z^14 - 2*x1^26*x2^25*x3^14*x4^5*z^14 + x1^33*x2^17*x3^15*x4^5*z^14 + 2*x1^32*x2^18*x3^15*x4^5*z^14 - 2*x1^29*x2^21*x3^15*x4^5*z^14 - 6*x1^27*x2^23*x3^15*x4^5*z^14 + 2*x1^26*x2^24*x3^15*x4^5*z^14 - x1^32*x2^17*x3^16*x4^5*z^14 - x1^31*x2^18*x3^16*x4^5*z^14 - x1^30*x2^19*x3^16*x4^5*z^14 + 3*x1^29*x2^20*x3^16*x4^5*z^14 - 2*x1^28*x2^21*x3^16*x4^5*z^14 + 5*x1^27*x2^22*x3^16*x4^5*z^14 + 2*x1^26*x2^23*x3^16*x4^5*z^14 - 2*x1^25*x2^24*x3^16*x4^5*z^14 + x1^30*x2^18*x3^17*x4^5*z^14 - x1^29*x2^19*x3^17*x4^5*z^14 - 3*x1^28*x2^20*x3^17*x4^5*z^14 - 5*x1^26*x2^22*x3^17*x4^5*z^14 + 2*x1^25*x2^23*x3^17*x4^5*z^14 + x1^24*x2^24*x3^17*x4^5*z^14 - x1^29*x2^18*x3^18*x4^5*z^14 + 2*x1^28*x2^19*x3^18*x4^5*z^14 + 2*x1^26*x2^21*x3^18*x4^5*z^14 - 2*x1^24*x2^23*x3^18*x4^5*z^14 - x1^27*x2^19*x3^19*x4^5*z^14 + x1^26*x2^20*x3^19*x4^5*z^14 - 3*x1^25*x2^21*x3^19*x4^5*z^14 + 2*x1^24*x2^22*x3^19*x4^5*z^14 + x1^23*x2^23*x3^19*x4^5*z^14 + x1^24*x2^21*x3^20*x4^5*z^14 - x1^23*x2^22*x3^20*x4^5*z^14 - x1^23*x2^21*x3^21*x4^5*z^14 + x1^22*x2^22*x3^21*x4^5*z^14 + x1^33*x2^24*x3^7*x4^6*z^14 - x1^32*x2^25*x3^7*x4^6*z^14 + 2*x1^32*x2^24*x3^8*x4^6*z^14 + x1^31*x2^25*x3^8*x4^6*z^14 - x1^35*x2^20*x3^9*x4^6*z^14 - 2*x1^34*x2^21*x3^9*x4^6*z^14 - x1^31*x2^24*x3^9*x4^6*z^14 - x1^29*x2^26*x3^9*x4^6*z^14 - 2*x1^28*x2^27*x3^9*x4^6*z^14 + x1^35*x2^19*x3^10*x4^6*z^14 + x1^32*x2^22*x3^10*x4^6*z^14 + x1^31*x2^23*x3^10*x4^6*z^14 - x1^30*x2^24*x3^10*x4^6*z^14 - 3*x1^28*x2^26*x3^10*x4^6*z^14 + x1^27*x2^27*x3^10*x4^6*z^14 + x1^34*x2^19*x3^11*x4^6*z^14 - 3*x1^32*x2^21*x3^11*x4^6*z^14 - 2*x1^31*x2^22*x3^11*x4^6*z^14 + 2*x1^30*x2^23*x3^11*x4^6*z^14 + 2*x1^29*x2^24*x3^11*x4^6*z^14 + 2*x1^27*x2^26*x3^11*x4^6*z^14 + x1^34*x2^18*x3^12*x4^6*z^14 + 3*x1^32*x2^20*x3^12*x4^6*z^14 + 2*x1^31*x2^21*x3^12*x4^6*z^14 + x1^30*x2^22*x3^12*x4^6*z^14 - x1^29*x2^23*x3^12*x4^6*z^14 + 3*x1^28*x2^24*x3^12*x4^6*z^14 - 3*x1^27*x2^25*x3^12*x4^6*z^14 - 2*x1^26*x2^26*x3^12*x4^6*z^14 + x1^33*x2^18*x3^13*x4^6*z^14 - x1^32*x2^19*x3^13*x4^6*z^14 - x1^31*x2^20*x3^13*x4^6*z^14 - 4*x1^30*x2^21*x3^13*x4^6*z^14 + 2*x1^27*x2^24*x3^13*x4^6*z^14 + 4*x1^26*x2^25*x3^13*x4^6*z^14 + x1^33*x2^17*x3^14*x4^6*z^14 + x1^32*x2^18*x3^14*x4^6*z^14 + 4*x1^31*x2^19*x3^14*x4^6*z^14 + 2*x1^30*x2^20*x3^14*x4^6*z^14 + x1^29*x2^21*x3^14*x4^6*z^14 - 3*x1^28*x2^22*x3^14*x4^6*z^14 + 2*x1^27*x2^23*x3^14*x4^6*z^14 - 4*x1^26*x2^24*x3^14*x4^6*z^14 - x1^25*x2^25*x3^14*x4^6*z^14 - x1^32*x2^17*x3^15*x4^6*z^14 - 3*x1^31*x2^18*x3^15*x4^6*z^14 + 2*x1^30*x2^19*x3^15*x4^6*z^14 - x1^29*x2^20*x3^15*x4^6*z^14 - x1^27*x2^22*x3^15*x4^6*z^14 + 4*x1^25*x2^24*x3^15*x4^6*z^14 + 3*x1^30*x2^18*x3^16*x4^6*z^14 + x1^29*x2^19*x3^16*x4^6*z^14 + 4*x1^28*x2^20*x3^16*x4^6*z^14 - x1^27*x2^21*x3^16*x4^6*z^14 + 2*x1^26*x2^22*x3^16*x4^6*z^14 - 4*x1^25*x2^23*x3^16*x4^6*z^14 - x1^24*x2^24*x3^16*x4^6*z^14 - 3*x1^29*x2^18*x3^17*x4^6*z^14 - 2*x1^28*x2^19*x3^17*x4^6*z^14 + x1^27*x2^20*x3^17*x4^6*z^14 + 4*x1^24*x2^23*x3^17*x4^6*z^14 - 4*x1^26*x2^20*x3^18*x4^6*z^14 + x1^25*x2^21*x3^18*x4^6*z^14 - 3*x1^24*x2^22*x3^18*x4^6*z^14 - 2*x1^23*x2^23*x3^18*x4^6*z^14 - x1^26*x2^19*x3^19*x4^6*z^14 + 3*x1^23*x2^22*x3^19*x4^6*z^14 + x1^24*x2^20*x3^20*x4^6*z^14 - x1^23*x2^21*x3^20*x4^6*z^14 - x1^22*x2^22*x3^20*x4^6*z^14 + x1^22*x2^21*x3^21*x4^6*z^14 + x1^32*x2^24*x3^7*x4^7*z^14 + x1^30*x2^26*x3^7*x4^7*z^14 - 2*x1^32*x2^23*x3^8*x4^7*z^14 + 2*x1^31*x2^24*x3^8*x4^7*z^14 - x1^29*x2^26*x3^8*x4^7*z^14 + x1^28*x2^27*x3^8*x4^7*z^14 + x1^31*x2^23*x3^9*x4^7*z^14 + 2*x1^29*x2^25*x3^9*x4^7*z^14 + x1^28*x2^26*x3^9*x4^7*z^14 - x1^27*x2^27*x3^9*x4^7*z^14 + x1^34*x2^19*x3^10*x4^7*z^14 + x1^33*x2^20*x3^10*x4^7*z^14 - x1^31*x2^22*x3^10*x4^7*z^14 + x1^30*x2^23*x3^10*x4^7*z^14 - x1^29*x2^24*x3^10*x4^7*z^14 - 2*x1^34*x2^18*x3^11*x4^7*z^14 + x1^32*x2^20*x3^11*x4^7*z^14 - x1^31*x2^21*x3^11*x4^7*z^14 + x1^30*x2^22*x3^11*x4^7*z^14 + x1^28*x2^24*x3^11*x4^7*z^14 + x1^27*x2^25*x3^11*x4^7*z^14 - x1^26*x2^26*x3^11*x4^7*z^14 + 2*x1^33*x2^18*x3^12*x4^7*z^14 + 2*x1^30*x2^21*x3^12*x4^7*z^14 - 2*x1^28*x2^23*x3^12*x4^7*z^14 + x1^27*x2^24*x3^12*x4^7*z^14 - x1^32*x2^18*x3^13*x4^7*z^14 - 3*x1^31*x2^19*x3^13*x4^7*z^14 - x1^29*x2^21*x3^13*x4^7*z^14 + 2*x1^31*x2^18*x3^14*x4^7*z^14 + x1^29*x2^20*x3^14*x4^7*z^14 + x1^31*x2^17*x3^15*x4^7*z^14 - 2*x1^30*x2^18*x3^15*x4^7*z^14 - x1^29*x2^19*x3^15*x4^7*z^14 - 2*x1^28*x2^20*x3^15*x4^7*z^14 + x1^27*x2^21*x3^15*x4^7*z^14 - x1^30*x2^17*x3^16*x4^7*z^14 + x1^29*x2^18*x3^16*x4^7*z^14 - x1^28*x2^19*x3^16*x4^7*z^14 - x1^26*x2^21*x3^16*x4^7*z^14 - x1^25*x2^22*x3^16*x4^7*z^14 + x1^29*x2^17*x3^17*x4^7*z^14 - x1^27*x2^19*x3^17*x4^7*z^14 + x1^26*x2^20*x3^17*x4^7*z^14 + x1^25*x2^21*x3^17*x4^7*z^14 + 2*x1^24*x2^22*x3^17*x4^7*z^14 + x1^27*x2^18*x3^18*x4^7*z^14 - 2*x1^24*x2^21*x3^18*x4^7*z^14 - 2*x1^23*x2^22*x3^18*x4^7*z^14 + 2*x1^25*x2^19*x3^19*x4^7*z^14 - x1^24*x2^20*x3^19*x4^7*z^14 + 2*x1^23*x2^21*x3^19*x4^7*z^14 + x1^22*x2^22*x3^19*x4^7*z^14 + x1^23*x2^20*x3^20*x4^7*z^14 - 2*x1^22*x2^21*x3^20*x4^7*z^14 + x1^21*x2^21*x3^21*x4^7*z^14 + x1^32*x2^22*x3^8*x4^8*z^14 + x1^30*x2^24*x3^8*x4^8*z^14 + x1^29*x2^25*x3^8*x4^8*z^14 - x1^30*x2^23*x3^9*x4^8*z^14 - x1^29*x2^24*x3^9*x4^8*z^14 + x1^28*x2^25*x3^9*x4^8*z^14 + x1^31*x2^21*x3^10*x4^8*z^14 - x1^30*x2^22*x3^10*x4^8*z^14 + x1^29*x2^23*x3^10*x4^8*z^14 - x1^28*x2^24*x3^10*x4^8*z^14 + 2*x1^26*x2^26*x3^10*x4^8*z^14 + x1^33*x2^18*x3^11*x4^8*z^14 + x1^31*x2^20*x3^11*x4^8*z^14 - 3*x1^28*x2^23*x3^11*x4^8*z^14 - 2*x1^27*x2^24*x3^11*x4^8*z^14 - x1^26*x2^25*x3^11*x4^8*z^14 - x1^32*x2^18*x3^12*x4^8*z^14 - x1^30*x2^20*x3^12*x4^8*z^14 + 2*x1^29*x2^21*x3^12*x4^8*z^14 + 4*x1^28*x2^22*x3^12*x4^8*z^14 + x1^27*x2^23*x3^12*x4^8*z^14 + x1^26*x2^24*x3^12*x4^8*z^14 + x1^25*x2^25*x3^12*x4^8*z^14 - 3*x1^29*x2^20*x3^13*x4^8*z^14 - 2*x1^28*x2^21*x3^13*x4^8*z^14 + x1^27*x2^22*x3^13*x4^8*z^14 - 2*x1^26*x2^23*x3^13*x4^8*z^14 - 3*x1^25*x2^24*x3^13*x4^8*z^14 - x1^28*x2^20*x3^14*x4^8*z^14 + x1^27*x2^21*x3^14*x4^8*z^14 - x1^26*x2^22*x3^14*x4^8*z^14 + 3*x1^25*x2^23*x3^14*x4^8*z^14 + x1^24*x2^24*x3^14*x4^8*z^14 - x1^30*x2^17*x3^15*x4^8*z^14 + x1^29*x2^18*x3^15*x4^8*z^14 - 3*x1^28*x2^19*x3^15*x4^8*z^14 - 3*x1^26*x2^21*x3^15*x4^8*z^14 + x1^25*x2^22*x3^15*x4^8*z^14 - 4*x1^24*x2^23*x3^15*x4^8*z^14 + x1^27*x2^19*x3^16*x4^8*z^14 + 2*x1^25*x2^21*x3^16*x4^8*z^14 + 2*x1^23*x2^23*x3^16*x4^8*z^14 - x1^27*x2^18*x3^17*x4^8*z^14 + x1^26*x2^19*x3^17*x4^8*z^14 - 3*x1^25*x2^20*x3^17*x4^8*z^14 - 2*x1^24*x2^21*x3^17*x4^8*z^14 + x1^26*x2^18*x3^18*x4^8*z^14 + 2*x1^24*x2^20*x3^18*x4^8*z^14 - x1^22*x2^22*x3^18*x4^8*z^14 - x1^31*x2^21*x3^9*x4^9*z^14 - x1^29*x2^23*x3^9*x4^9*z^14 - x1^28*x2^24*x3^9*x4^9*z^14 - x1^27*x2^25*x3^9*x4^9*z^14 - x1^26*x2^26*x3^9*x4^9*z^14 + 2*x1^30*x2^21*x3^10*x4^9*z^14 + 2*x1^28*x2^23*x3^10*x4^9*z^14 + x1^27*x2^24*x3^10*x4^9*z^14 - x1^30*x2^20*x3^11*x4^9*z^14 - x1^29*x2^21*x3^11*x4^9*z^14 - 2*x1^28*x2^22*x3^11*x4^9*z^14 - 2*x1^27*x2^23*x3^11*x4^9*z^14 - x1^26*x2^24*x3^11*x4^9*z^14 - x1^25*x2^25*x3^11*x4^9*z^14 - x1^31*x2^18*x3^12*x4^9*z^14 - x1^30*x2^19*x3^12*x4^9*z^14 + 2*x1^29*x2^20*x3^12*x4^9*z^14 - x1^28*x2^21*x3^12*x4^9*z^14 + 2*x1^27*x2^22*x3^12*x4^9*z^14 + 2*x1^26*x2^23*x3^12*x4^9*z^14 - 3*x1^28*x2^20*x3^13*x4^9*z^14 - 3*x1^27*x2^21*x3^13*x4^9*z^14 - 3*x1^26*x2^22*x3^13*x4^9*z^14 - 3*x1^25*x2^23*x3^13*x4^9*z^14 - x1^31*x2^16*x3^14*x4^9*z^14 - x1^30*x2^17*x3^14*x4^9*z^14 - x1^29*x2^18*x3^14*x4^9*z^14 + 3*x1^28*x2^19*x3^14*x4^9*z^14 + x1^27*x2^20*x3^14*x4^9*z^14 + 4*x1^26*x2^21*x3^14*x4^9*z^14 + x1^24*x2^23*x3^14*x4^9*z^14 + x1^30*x2^16*x3^15*x4^9*z^14 - x1^28*x2^18*x3^15*x4^9*z^14 - x1^27*x2^19*x3^15*x4^9*z^14 - x1^26*x2^20*x3^15*x4^9*z^14 - 3*x1^25*x2^21*x3^15*x4^9*z^14 - x1^23*x2^23*x3^15*x4^9*z^14 + 2*x1^27*x2^18*x3^16*x4^9*z^14 + 5*x1^25*x2^20*x3^16*x4^9*z^14 + 3*x1^24*x2^21*x3^16*x4^9*z^14 + x1^23*x2^22*x3^16*x4^9*z^14 - x1^26*x2^18*x3^17*x4^9*z^14 - x1^25*x2^19*x3^17*x4^9*z^14 - 4*x1^24*x2^20*x3^17*x4^9*z^14 - x1^23*x2^21*x3^17*x4^9*z^14 - x1^25*x2^18*x3^18*x4^9*z^14 + 2*x1^24*x2^19*x3^18*x4^9*z^14 + x1^23*x2^20*x3^18*x4^9*z^14 + x1^22*x2^21*x3^18*x4^9*z^14 - 2*x1^23*x2^19*x3^19*x4^9*z^14 - x1^29*x2^20*x3^11*x4^10*z^14 - x1^25*x2^24*x3^11*x4^10*z^14 + 2*x1^27*x2^21*x3^12*x4^10*z^14 + 2*x1^25*x2^23*x3^12*x4^10*z^14 + x1^30*x2^17*x3^13*x4^10*z^14 + x1^29*x2^18*x3^13*x4^10*z^14 - x1^28*x2^19*x3^13*x4^10*z^14 + x1^25*x2^22*x3^13*x4^10*z^14 - x1^24*x2^23*x3^13*x4^10*z^14 + x1^29*x2^17*x3^14*x4^10*z^14 - x1^27*x2^19*x3^14*x4^10*z^14 - x1^26*x2^20*x3^14*x4^10*z^14 - x1^25*x2^21*x3^14*x4^10*z^14 + 2*x1^24*x2^22*x3^14*x4^10*z^14 + x1^26*x2^19*x3^15*x4^10*z^14 - x1^25*x2^20*x3^15*x4^10*z^14 + x1^24*x2^21*x3^15*x4^10*z^14 - x1^27*x2^17*x3^16*x4^10*z^14 - x1^25*x2^19*x3^16*x4^10*z^14 + 2*x1^25*x2^18*x3^17*x4^10*z^14 - x1^24*x2^19*x3^17*x4^10*z^14 - x1^23*x2^20*x3^17*x4^10*z^14 + x1^22*x2^21*x3^17*x4^10*z^14 - x1^24*x2^18*x3^18*x4^10*z^14 + x1^23*x2^19*x3^18*x4^10*z^14 - x1^21*x2^21*x3^18*x4^10*z^14 + x1^29*x2^19*x3^11*x4^11*z^14 + 2*x1^26*x2^22*x3^11*x4^11*z^14 - x1^27*x2^20*x3^12*x4^11*z^14 + x1^25*x2^22*x3^12*x4^11*z^14 + 2*x1^24*x2^23*x3^12*x4^11*z^14 - x1^27*x2^19*x3^13*x4^11*z^14 - 3*x1^26*x2^20*x3^13*x4^11*z^14 + x1^25*x2^21*x3^13*x4^11*z^14 - x1^24*x2^22*x3^13*x4^11*z^14 + x1^26*x2^19*x3^14*x4^11*z^14 - x1^25*x2^20*x3^14*x4^11*z^14 - x1^24*x2^21*x3^14*x4^11*z^14 + 3*x1^23*x2^22*x3^14*x4^11*z^14 + 2*x1^26*x2^18*x3^15*x4^11*z^14 - x1^23*x2^21*x3^15*x4^11*z^14 + x1^24*x2^19*x3^16*x4^11*z^14 + 2*x1^22*x2^21*x3^16*x4^11*z^14 - x1^21*x2^21*x3^17*x4^11*z^14 - 2*x1^25*x2^21*x3^12*x4^12*z^14 + x1^27*x2^18*x3^13*x4^12*z^14 - 2*x1^23*x2^22*x3^13*x4^12*z^14 + x1^26*x2^18*x3^14*x4^12*z^14 + 3*x1^25*x2^19*x3^14*x4^12*z^14 - x1^24*x2^20*x3^14*x4^12*z^14 + x1^23*x2^21*x3^14*x4^12*z^14 + x1^22*x2^22*x3^14*x4^12*z^14 - x1^25*x2^18*x3^15*x4^12*z^14 - 2*x1^24*x2^19*x3^15*x4^12*z^14 - 3*x1^22*x2^21*x3^15*x4^12*z^14 - x1^23*x2^19*x3^16*x4^12*z^14 + 2*x1^21*x2^21*x3^16*x4^12*z^14 - x1^21*x2^20*x3^17*x4^12*z^14 + x1^25*x2^18*x3^14*x4^13*z^14 + x1^24*x2^19*x3^14*x4^13*z^14 - x1^23*x2^20*x3^14*x4^13*z^14 - x1^21*x2^21*x3^15*x4^13*z^14 - x1^20*x2^20*x3^17*x4^13*z^14 + x1^22*x2^19*x3^15*x4^14*z^14 - x1^36*x2^23*x3^6*z^13 + x1^36*x2^22*x3^7*z^13 + x1^35*x2^23*x3^7*z^13 + x1^33*x2^25*x3^7*z^13 - 2*x1^35*x2^22*x3^8*z^13 - x1^32*x2^25*x3^8*z^13 + 2*x1^34*x2^22*x3^9*z^13 + x1^32*x2^24*x3^9*z^13 - 2*x1^34*x2^21*x3^10*z^13 - x1^33*x2^22*x3^10*z^13 - x1^32*x2^23*x3^10*z^13 + x1^34*x2^20*x3^11*z^13 + 2*x1^33*x2^21*x3^11*z^13 + x1^31*x2^23*x3^11*z^13 - x1^30*x2^24*x3^11*z^13 - 2*x1^28*x2^26*x3^11*z^13 - 2*x1^33*x2^20*x3^12*z^13 - x1^32*x2^21*x3^12*z^13 - x1^31*x2^22*x3^12*z^13 + x1^29*x2^24*x3^12*z^13 + x1^28*x2^25*x3^12*z^13 + 2*x1^27*x2^26*x3^12*z^13 + x1^33*x2^19*x3^13*z^13 + 2*x1^32*x2^20*x3^13*z^13 + x1^30*x2^22*x3^13*z^13 - 2*x1^29*x2^23*x3^13*z^13 - 2*x1^27*x2^25*x3^13*z^13 - 2*x1^32*x2^19*x3^14*z^13 + 2*x1^26*x2^25*x3^14*z^13 + 2*x1^31*x2^19*x3^15*z^13 + 2*x1^29*x2^21*x3^15*z^13 - x1^27*x2^23*x3^15*z^13 - x1^26*x2^24*x3^15*z^13 - x1^31*x2^18*x3^16*z^13 - 2*x1^30*x2^19*x3^16*z^13 - x1^29*x2^20*x3^16*z^13 + x1^25*x2^24*x3^16*z^13 + x1^30*x2^18*x3^17*z^13 + 2*x1^29*x2^19*x3^17*z^13 + x1^28*x2^20*x3^17*z^13 + x1^27*x2^21*x3^17*z^13 - 2*x1^25*x2^23*x3^17*z^13 - x1^28*x2^19*x3^18*z^13 - x1^27*x2^20*x3^18*z^13 + x1^36*x2^23*x3^5*x4*z^13 - x1^34*x2^25*x3^5*x4*z^13 - 2*x1^35*x2^23*x3^6*x4*z^13 + 2*x1^34*x2^24*x3^6*x4*z^13 + 3*x1^35*x2^22*x3^7*x4*z^13 + 2*x1^34*x2^23*x3^7*x4*z^13 - x1^31*x2^26*x3^7*x4*z^13 - 2*x1^35*x2^21*x3^8*x4*z^13 - 5*x1^34*x2^22*x3^8*x4*z^13 + x1^33*x2^23*x3^8*x4*z^13 - 2*x1^32*x2^24*x3^8*x4*z^13 + 2*x1^31*x2^25*x3^8*x4*z^13 + x1^30*x2^26*x3^8*x4*z^13 + x1^29*x2^27*x3^8*x4*z^13 + 6*x1^34*x2^21*x3^9*x4*z^13 + 2*x1^33*x2^22*x3^9*x4*z^13 + 2*x1^32*x2^23*x3^9*x4*z^13 - 3*x1^30*x2^25*x3^9*x4*z^13 - 2*x1^29*x2^26*x3^9*x4*z^13 - x1^28*x2^27*x3^9*x4*z^13 - 2*x1^34*x2^20*x3^10*x4*z^13 - 6*x1^33*x2^21*x3^10*x4*z^13 - 4*x1^31*x2^23*x3^10*x4*z^13 + 4*x1^30*x2^24*x3^10*x4*z^13 + 2*x1^29*x2^25*x3^10*x4*z^13 + 5*x1^28*x2^26*x3^10*x4*z^13 + 6*x1^33*x2^20*x3^11*x4*z^13 + 2*x1^32*x2^21*x3^11*x4*z^13 + 2*x1^31*x2^22*x3^11*x4*z^13 - 2*x1^29*x2^24*x3^11*x4*z^13 - 2*x1^28*x2^25*x3^11*x4*z^13 - 5*x1^27*x2^26*x3^11*x4*z^13 - 2*x1^33*x2^19*x3^12*x4*z^13 - 6*x1^32*x2^20*x3^12*x4*z^13 - 4*x1^30*x2^22*x3^12*x4*z^13 + 4*x1^29*x2^23*x3^12*x4*z^13 + 6*x1^27*x2^25*x3^12*x4*z^13 + 2*x1^26*x2^26*x3^12*x4*z^13 + 5*x1^32*x2^19*x3^13*x4*z^13 + 2*x1^31*x2^20*x3^13*x4*z^13 + 2*x1^30*x2^21*x3^13*x4*z^13 - 2*x1^28*x2^23*x3^13*x4*z^13 - 2*x1^27*x2^24*x3^13*x4*z^13 - 6*x1^26*x2^25*x3^13*x4*z^13 - x1^32*x2^18*x3^14*x4*z^13 - 5*x1^31*x2^19*x3^14*x4*z^13 - x1^30*x2^20*x3^14*x4*z^13 - 4*x1^29*x2^21*x3^14*x4*z^13 + 4*x1^28*x2^22*x3^14*x4*z^13 + 6*x1^26*x2^24*x3^14*x4*z^13 + 2*x1^25*x2^25*x3^14*x4*z^13 + 2*x1^31*x2^18*x3^15*x4*z^13 + x1^30*x2^19*x3^15*x4*z^13 + 2*x1^29*x2^20*x3^15*x4*z^13 - 2*x1^27*x2^22*x3^15*x4*z^13 - 2*x1^26*x2^23*x3^15*x4*z^13 - 6*x1^25*x2^24*x3^15*x4*z^13 - 2*x1^30*x2^18*x3^16*x4*z^13 - x1^29*x2^19*x3^16*x4*z^13 - 3*x1^28*x2^20*x3^16*x4*z^13 + 2*x1^27*x2^21*x3^16*x4*z^13 + x1^26*x2^22*x3^16*x4*z^13 + 5*x1^25*x2^23*x3^16*x4*z^13 + x1^24*x2^24*x3^16*x4*z^13 + x1^29*x2^18*x3^17*x4*z^13 + x1^28*x2^19*x3^17*x4*z^13 - x1^25*x2^22*x3^17*x4*z^13 - 4*x1^24*x2^23*x3^17*x4*z^13 - x1^25*x2^21*x3^18*x4*z^13 + 3*x1^24*x2^22*x3^18*x4*z^13 + x1^25*x2^20*x3^19*x4*z^13 - 2*x1^23*x2^22*x3^19*x4*z^13 - x1^35*x2^23*x3^5*x4^2*z^13 + x1^34*x2^24*x3^5*x4^2*z^13 + x1^33*x2^25*x3^5*x4^2*z^13 - x1^32*x2^26*x3^5*x4^2*z^13 - x1^35*x2^22*x3^6*x4^2*z^13 - x1^34*x2^23*x3^6*x4^2*z^13 + x1^33*x2^24*x3^6*x4^2*z^13 + x1^31*x2^26*x3^6*x4^2*z^13 + 2*x1^34*x2^22*x3^7*x4^2*z^13 + x1^32*x2^24*x3^7*x4^2*z^13 - 2*x1^31*x2^25*x3^7*x4^2*z^13 - 2*x1^30*x2^26*x3^7*x4^2*z^13 - 2*x1^29*x2^27*x3^7*x4^2*z^13 - 2*x1^34*x2^21*x3^8*x4^2*z^13 - 2*x1^33*x2^22*x3^8*x4^2*z^13 + 2*x1^30*x2^25*x3^8*x4^2*z^13 + 3*x1^29*x2^26*x3^8*x4^2*z^13 + 2*x1^28*x2^27*x3^8*x4^2*z^13 + x1^34*x2^20*x3^9*x4^2*z^13 + 4*x1^33*x2^21*x3^9*x4^2*z^13 + 2*x1^31*x2^23*x3^9*x4^2*z^13 - 3*x1^30*x2^24*x3^9*x4^2*z^13 - 5*x1^28*x2^26*x3^9*x4^2*z^13 - 5*x1^33*x2^20*x3^10*x4^2*z^13 - 3*x1^32*x2^21*x3^10*x4^2*z^13 - x1^31*x2^22*x3^10*x4^2*z^13 + x1^30*x2^23*x3^10*x4^2*z^13 + 2*x1^29*x2^24*x3^10*x4^2*z^13 + 2*x1^28*x2^25*x3^10*x4^2*z^13 + 5*x1^27*x2^26*x3^10*x4^2*z^13 + 2*x1^33*x2^19*x3^11*x4^2*z^13 + 6*x1^32*x2^20*x3^11*x4^2*z^13 + 4*x1^30*x2^22*x3^11*x4^2*z^13 - 4*x1^29*x2^23*x3^11*x4^2*z^13 - 6*x1^27*x2^25*x3^11*x4^2*z^13 - x1^26*x2^26*x3^11*x4^2*z^13 - 6*x1^32*x2^19*x3^12*x4^2*z^13 - 2*x1^31*x2^20*x3^12*x4^2*z^13 - 2*x1^30*x2^21*x3^12*x4^2*z^13 + 2*x1^28*x2^23*x3^12*x4^2*z^13 + 2*x1^27*x2^24*x3^12*x4^2*z^13 + 6*x1^26*x2^25*x3^12*x4^2*z^13 + x1^32*x2^18*x3^13*x4^2*z^13 + 6*x1^31*x2^19*x3^13*x4^2*z^13 + 4*x1^29*x2^21*x3^13*x4^2*z^13 - 4*x1^28*x2^22*x3^13*x4^2*z^13 - 6*x1^26*x2^24*x3^13*x4^2*z^13 - 2*x1^25*x2^25*x3^13*x4^2*z^13 - 2*x1^31*x2^18*x3^14*x4^2*z^13 - x1^30*x2^19*x3^14*x4^2*z^13 + 2*x1^27*x2^22*x3^14*x4^2*z^13 + 2*x1^26*x2^23*x3^14*x4^2*z^13 + 6*x1^25*x2^24*x3^14*x4^2*z^13 + 2*x1^30*x2^18*x3^15*x4^2*z^13 + x1^29*x2^19*x3^15*x4^2*z^13 + 2*x1^28*x2^20*x3^15*x4^2*z^13 - 3*x1^27*x2^21*x3^15*x4^2*z^13 - 6*x1^25*x2^23*x3^15*x4^2*z^13 - 2*x1^24*x2^24*x3^15*x4^2*z^13 - x1^29*x2^18*x3^16*x4^2*z^13 - x1^28*x2^19*x3^16*x4^2*z^13 + x1^27*x2^20*x3^16*x4^2*z^13 + 2*x1^26*x2^21*x3^16*x4^2*z^13 + 2*x1^25*x2^22*x3^16*x4^2*z^13 + 6*x1^24*x2^23*x3^16*x4^2*z^13 + x1^28*x2^18*x3^17*x4^2*z^13 - 2*x1^26*x2^20*x3^17*x4^2*z^13 - 5*x1^24*x2^22*x3^17*x4^2*z^13 - 2*x1^23*x2^23*x3^17*x4^2*z^13 - x1^27*x2^18*x3^18*x4^2*z^13 + x1^25*x2^20*x3^18*x4^2*z^13 + x1^24*x2^21*x3^18*x4^2*z^13 + 4*x1^23*x2^22*x3^18*x4^2*z^13 - x1^25*x2^19*x3^19*x4^2*z^13 + x1^24*x2^20*x3^19*x4^2*z^13 - x1^23*x2^21*x3^19*x4^2*z^13 - x1^22*x2^22*x3^19*x4^2*z^13 + x1^22*x2^21*x3^20*x4^2*z^13 + x1^36*x2^20*x3^6*x4^3*z^13 + x1^34*x2^22*x3^6*x4^3*z^13 + x1^31*x2^25*x3^6*x4^3*z^13 + x1^30*x2^26*x3^6*x4^3*z^13 - x1^35*x2^20*x3^7*x4^3*z^13 + x1^31*x2^24*x3^7*x4^3*z^13 + x1^35*x2^19*x3^8*x4^3*z^13 + x1^34*x2^20*x3^8*x4^3*z^13 - 2*x1^32*x2^22*x3^8*x4^3*z^13 + x1^31*x2^23*x3^8*x4^3*z^13 + 3*x1^30*x2^24*x3^8*x4^3*z^13 + x1^28*x2^26*x3^8*x4^3*z^13 - 2*x1^34*x2^19*x3^9*x4^3*z^13 - 2*x1^32*x2^21*x3^9*x4^3*z^13 - x1^31*x2^22*x3^9*x4^3*z^13 - x1^30*x2^23*x3^9*x4^3*z^13 + x1^29*x2^24*x3^9*x4^3*z^13 - x1^28*x2^25*x3^9*x4^3*z^13 - 2*x1^27*x2^26*x3^9*x4^3*z^13 + 2*x1^34*x2^18*x3^10*x4^3*z^13 + 2*x1^33*x2^19*x3^10*x4^3*z^13 - x1^31*x2^21*x3^10*x4^3*z^13 - 2*x1^30*x2^22*x3^10*x4^3*z^13 - 2*x1^28*x2^24*x3^10*x4^3*z^13 + 2*x1^27*x2^25*x3^10*x4^3*z^13 - x1^34*x2^17*x3^11*x4^3*z^13 - 2*x1^33*x2^18*x3^11*x4^3*z^13 + x1^32*x2^19*x3^11*x4^3*z^13 + x1^31*x2^20*x3^11*x4^3*z^13 + x1^30*x2^21*x3^11*x4^3*z^13 - x1^29*x2^22*x3^11*x4^3*z^13 + x1^28*x2^23*x3^11*x4^3*z^13 - 2*x1^26*x2^25*x3^11*x4^3*z^13 + 2*x1^33*x2^17*x3^12*x4^3*z^13 - x1^31*x2^19*x3^12*x4^3*z^13 - 2*x1^29*x2^21*x3^12*x4^3*z^13 - 2*x1^27*x2^23*x3^12*x4^3*z^13 + 2*x1^26*x2^24*x3^12*x4^3*z^13 + x1^25*x2^25*x3^12*x4^3*z^13 - 2*x1^32*x2^17*x3^13*x4^3*z^13 + x1^31*x2^18*x3^13*x4^3*z^13 + x1^30*x2^19*x3^13*x4^3*z^13 + 3*x1^29*x2^20*x3^13*x4^3*z^13 + x1^27*x2^22*x3^13*x4^3*z^13 - x1^26*x2^23*x3^13*x4^3*z^13 - 2*x1^25*x2^24*x3^13*x4^3*z^13 + x1^31*x2^17*x3^14*x4^3*z^13 - x1^30*x2^18*x3^14*x4^3*z^13 - 2*x1^29*x2^19*x3^14*x4^3*z^13 - x1^28*x2^20*x3^14*x4^3*z^13 + 2*x1^27*x2^21*x3^14*x4^3*z^13 - 2*x1^26*x2^22*x3^14*x4^3*z^13 + 2*x1^25*x2^23*x3^14*x4^3*z^13 - x1^30*x2^17*x3^15*x4^3*z^13 - x1^29*x2^18*x3^15*x4^3*z^13 + 2*x1^28*x2^19*x3^15*x4^3*z^13 + x1^27*x2^20*x3^15*x4^3*z^13 + 3*x1^26*x2^21*x3^15*x4^3*z^13 + x1^25*x2^22*x3^15*x4^3*z^13 - 2*x1^24*x2^23*x3^15*x4^3*z^13 + x1^29*x2^17*x3^16*x4^3*z^13 - 2*x1^27*x2^19*x3^16*x4^3*z^13 - 2*x1^26*x2^20*x3^16*x4^3*z^13 - 3*x1^25*x2^21*x3^16*x4^3*z^13 + 2*x1^24*x2^22*x3^16*x4^3*z^13 + x1^23*x2^23*x3^16*x4^3*z^13 + x1^27*x2^18*x3^17*x4^3*z^13 + 2*x1^25*x2^20*x3^17*x4^3*z^13 + x1^24*x2^21*x3^17*x4^3*z^13 - 2*x1^23*x2^22*x3^17*x4^3*z^13 - 3*x1^24*x2^20*x3^18*x4^3*z^13 + x1^23*x2^21*x3^18*x4^3*z^13 + 2*x1^22*x2^22*x3^18*x4^3*z^13 - x1^24*x2^19*x3^19*x4^3*z^13 + x1^22*x2^21*x3^19*x4^3*z^13 - x1^22*x2^20*x3^20*x4^3*z^13 + x1^33*x2^23*x3^5*x4^4*z^13 - 2*x1^32*x2^23*x3^6*x4^4*z^13 + x1^30*x2^25*x3^6*x4^4*z^13 - x1^35*x2^19*x3^7*x4^4*z^13 - x1^33*x2^21*x3^7*x4^4*z^13 + x1^32*x2^22*x3^7*x4^4*z^13 + 2*x1^31*x2^23*x3^7*x4^4*z^13 + 2*x1^30*x2^24*x3^7*x4^4*z^13 + x1^29*x2^25*x3^7*x4^4*z^13 + 3*x1^34*x2^19*x3^8*x4^4*z^13 + x1^33*x2^20*x3^8*x4^4*z^13 + x1^32*x2^21*x3^8*x4^4*z^13 - 4*x1^31*x2^22*x3^8*x4^4*z^13 - x1^30*x2^23*x3^8*x4^4*z^13 - 2*x1^29*x2^24*x3^8*x4^4*z^13 + x1^28*x2^25*x3^8*x4^4*z^13 + x1^27*x2^26*x3^8*x4^4*z^13 - 2*x1^34*x2^18*x3^9*x4^4*z^13 - 2*x1^33*x2^19*x3^9*x4^4*z^13 - x1^32*x2^20*x3^9*x4^4*z^13 + 3*x1^31*x2^21*x3^9*x4^4*z^13 + x1^30*x2^22*x3^9*x4^4*z^13 + 2*x1^29*x2^23*x3^9*x4^4*z^13 + 6*x1^28*x2^24*x3^9*x4^4*z^13 + 5*x1^33*x2^18*x3^10*x4^4*z^13 + 2*x1^32*x2^19*x3^10*x4^4*z^13 + 2*x1^31*x2^20*x3^10*x4^4*z^13 - 3*x1^30*x2^21*x3^10*x4^4*z^13 - 6*x1^28*x2^23*x3^10*x4^4*z^13 - 2*x1^27*x2^24*x3^10*x4^4*z^13 + x1^26*x2^25*x3^10*x4^4*z^13 - 4*x1^33*x2^17*x3^11*x4^4*z^13 - 3*x1^32*x2^18*x3^11*x4^4*z^13 - x1^31*x2^19*x3^11*x4^4*z^13 + 2*x1^30*x2^20*x3^11*x4^4*z^13 + 2*x1^28*x2^22*x3^11*x4^4*z^13 + 6*x1^27*x2^23*x3^11*x4^4*z^13 + 2*x1^33*x2^16*x3^12*x4^4*z^13 + 5*x1^32*x2^17*x3^12*x4^4*z^13 + 3*x1^31*x2^18*x3^12*x4^4*z^13 + 5*x1^30*x2^19*x3^12*x4^4*z^13 - 3*x1^29*x2^20*x3^12*x4^4*z^13 - 6*x1^27*x2^22*x3^12*x4^4*z^13 - 2*x1^26*x2^23*x3^12*x4^4*z^13 - 3*x1^32*x2^16*x3^13*x4^4*z^13 - x1^31*x2^17*x3^13*x4^4*z^13 - 2*x1^30*x2^18*x3^13*x4^4*z^13 - x1^29*x2^19*x3^13*x4^4*z^13 + 2*x1^28*x2^20*x3^13*x4^4*z^13 + 2*x1^27*x2^21*x3^13*x4^4*z^13 + 6*x1^26*x2^22*x3^13*x4^4*z^13 + 3*x1^31*x2^16*x3^14*x4^4*z^13 + x1^30*x2^17*x3^14*x4^4*z^13 + 3*x1^29*x2^18*x3^14*x4^4*z^13 - 3*x1^28*x2^19*x3^14*x4^4*z^13 - 6*x1^26*x2^21*x3^14*x4^4*z^13 - 2*x1^25*x2^22*x3^14*x4^4*z^13 - 2*x1^30*x2^16*x3^15*x4^4*z^13 - x1^29*x2^17*x3^15*x4^4*z^13 - x1^28*x2^18*x3^15*x4^4*z^13 + x1^27*x2^19*x3^15*x4^4*z^13 + 6*x1^25*x2^21*x3^15*x4^4*z^13 - x1^28*x2^17*x3^16*x4^4*z^13 - 2*x1^27*x2^18*x3^16*x4^4*z^13 - 4*x1^25*x2^20*x3^16*x4^4*z^13 - 3*x1^24*x2^21*x3^16*x4^4*z^13 + 4*x1^24*x2^20*x3^17*x4^4*z^13 + x1^25*x2^18*x3^18*x4^4*z^13 + 2*x1^23*x2^19*x3^19*x4^4*z^13 - x1^22*x2^20*x3^19*x4^4*z^13 - x1^32*x2^22*x3^6*x4^5*z^13 - x1^31*x2^23*x3^6*x4^5*z^13 - 2*x1^30*x2^24*x3^6*x4^5*z^13 - x1^29*x2^25*x3^6*x4^5*z^13 + 2*x1^31*x2^22*x3^7*x4^5*z^13 + x1^29*x2^24*x3^7*x4^5*z^13 + x1^28*x2^25*x3^7*x4^5*z^13 + x1^27*x2^26*x3^7*x4^5*z^13 - 2*x1^32*x2^20*x3^8*x4^5*z^13 - 2*x1^31*x2^21*x3^8*x4^5*z^13 - x1^30*x2^22*x3^8*x4^5*z^13 - 2*x1^29*x2^23*x3^8*x4^5*z^13 - 3*x1^28*x2^24*x3^8*x4^5*z^13 - 2*x1^33*x2^18*x3^9*x4^5*z^13 + 4*x1^30*x2^21*x3^9*x4^5*z^13 + 2*x1^28*x2^23*x3^9*x4^5*z^13 + x1^33*x2^17*x3^10*x4^5*z^13 + x1^32*x2^18*x3^10*x4^5*z^13 - 3*x1^30*x2^20*x3^10*x4^5*z^13 - x1^29*x2^21*x3^10*x4^5*z^13 - x1^28*x2^22*x3^10*x4^5*z^13 - 5*x1^27*x2^23*x3^10*x4^5*z^13 + 2*x1^26*x2^24*x3^10*x4^5*z^13 - x1^25*x2^25*x3^10*x4^5*z^13 - 5*x1^32*x2^17*x3^11*x4^5*z^13 + x1^31*x2^18*x3^11*x4^5*z^13 - x1^30*x2^19*x3^11*x4^5*z^13 + 5*x1^29*x2^20*x3^11*x4^5*z^13 + 5*x1^27*x2^22*x3^11*x4^5*z^13 + 3*x1^26*x2^23*x3^11*x4^5*z^13 - x1^25*x2^24*x3^11*x4^5*z^13 + x1^32*x2^16*x3^12*x4^5*z^13 + 2*x1^31*x2^17*x3^12*x4^5*z^13 - x1^30*x2^18*x3^12*x4^5*z^13 - x1^29*x2^19*x3^12*x4^5*z^13 - 2*x1^28*x2^20*x3^12*x4^5*z^13 - x1^27*x2^21*x3^12*x4^5*z^13 - 6*x1^26*x2^22*x3^12*x4^5*z^13 + 2*x1^25*x2^23*x3^12*x4^5*z^13 + x1^24*x2^24*x3^12*x4^5*z^13 - 2*x1^31*x2^16*x3^13*x4^5*z^13 - x1^30*x2^17*x3^13*x4^5*z^13 - 2*x1^29*x2^18*x3^13*x4^5*z^13 + 4*x1^28*x2^19*x3^13*x4^5*z^13 + 5*x1^26*x2^21*x3^13*x4^5*z^13 + x1^25*x2^22*x3^13*x4^5*z^13 - 2*x1^24*x2^23*x3^13*x4^5*z^13 + x1^30*x2^16*x3^14*x4^5*z^13 + x1^27*x2^19*x3^14*x4^5*z^13 - 6*x1^25*x2^21*x3^14*x4^5*z^13 + 2*x1^24*x2^22*x3^14*x4^5*z^13 + x1^23*x2^23*x3^14*x4^5*z^13 - x1^29*x2^16*x3^15*x4^5*z^13 - x1^28*x2^17*x3^15*x4^5*z^13 + 2*x1^27*x2^18*x3^15*x4^5*z^13 - x1^26*x2^19*x3^15*x4^5*z^13 + 4*x1^25*x2^20*x3^15*x4^5*z^13 + x1^24*x2^21*x3^15*x4^5*z^13 - 2*x1^23*x2^22*x3^15*x4^5*z^13 + x1^28*x2^16*x3^16*x4^5*z^13 - x1^26*x2^18*x3^16*x4^5*z^13 + x1^25*x2^19*x3^16*x4^5*z^13 - 5*x1^24*x2^20*x3^16*x4^5*z^13 + 3*x1^23*x2^21*x3^16*x4^5*z^13 + x1^26*x2^17*x3^17*x4^5*z^13 - x1^25*x2^18*x3^17*x4^5*z^13 + 2*x1^24*x2^19*x3^17*x4^5*z^13 + 2*x1^23*x2^20*x3^17*x4^5*z^13 - 3*x1^22*x2^21*x3^17*x4^5*z^13 + x1^24*x2^18*x3^18*x4^5*z^13 - 3*x1^23*x2^19*x3^18*x4^5*z^13 + x1^22*x2^20*x3^18*x4^5*z^13 + x1^21*x2^21*x3^18*x4^5*z^13 + x1^22*x2^19*x3^19*x4^5*z^13 - x1^21*x2^20*x3^19*x4^5*z^13 + x1^31*x2^22*x3^6*x4^6*z^13 + x1^29*x2^24*x3^6*x4^6*z^13 + x1^31*x2^21*x3^7*x4^6*z^13 - x1^30*x2^22*x3^7*x4^6*z^13 + x1^30*x2^21*x3^8*x4^6*z^13 + x1^29*x2^22*x3^8*x4^6*z^13 - x1^28*x2^23*x3^8*x4^6*z^13 + x1^33*x2^17*x3^9*x4^6*z^13 + 2*x1^31*x2^19*x3^9*x4^6*z^13 + 2*x1^30*x2^20*x3^9*x4^6*z^13 - x1^29*x2^21*x3^9*x4^6*z^13 - 2*x1^28*x2^22*x3^9*x4^6*z^13 - 2*x1^26*x2^24*x3^9*x4^6*z^13 + x1^25*x2^25*x3^9*x4^6*z^13 - 2*x1^31*x2^18*x3^10*x4^6*z^13 - x1^29*x2^20*x3^10*x4^6*z^13 + 2*x1^28*x2^21*x3^10*x4^6*z^13 - 2*x1^26*x2^23*x3^10*x4^6*z^13 + 4*x1^25*x2^24*x3^10*x4^6*z^13 + x1^32*x2^16*x3^11*x4^6*z^13 + x1^30*x2^18*x3^11*x4^6*z^13 + 2*x1^28*x2^20*x3^11*x4^6*z^13 - x1^27*x2^21*x3^11*x4^6*z^13 - 5*x1^25*x2^23*x3^11*x4^6*z^13 - x1^24*x2^24*x3^11*x4^6*z^13 - 2*x1^30*x2^17*x3^12*x4^6*z^13 - 3*x1^29*x2^18*x3^12*x4^6*z^13 - 3*x1^28*x2^19*x3^12*x4^6*z^13 + x1^27*x2^20*x3^12*x4^6*z^13 - x1^25*x2^22*x3^12*x4^6*z^13 + 3*x1^24*x2^23*x3^12*x4^6*z^13 + x1^30*x2^16*x3^13*x4^6*z^13 + 2*x1^29*x2^17*x3^13*x4^6*z^13 - x1^28*x2^18*x3^13*x4^6*z^13 + x1^27*x2^19*x3^13*x4^6*z^13 - x1^26*x2^20*x3^13*x4^6*z^13 + 2*x1^25*x2^21*x3^13*x4^6*z^13 - 4*x1^24*x2^22*x3^13*x4^6*z^13 - 2*x1^23*x2^23*x3^13*x4^6*z^13 - 3*x1^29*x2^16*x3^14*x4^6*z^13 - 2*x1^28*x2^17*x3^14*x4^6*z^13 - 3*x1^27*x2^18*x3^14*x4^6*z^13 - x1^25*x2^20*x3^14*x4^6*z^13 + 2*x1^24*x2^21*x3^14*x4^6*z^13 + 4*x1^23*x2^22*x3^14*x4^6*z^13 + 3*x1^28*x2^16*x3^15*x4^6*z^13 + 2*x1^27*x2^17*x3^15*x4^6*z^13 - 4*x1^25*x2^19*x3^15*x4^6*z^13 + 3*x1^24*x2^20*x3^15*x4^6*z^13 - 4*x1^23*x2^21*x3^15*x4^6*z^13 - x1^22*x2^22*x3^15*x4^6*z^13 - x1^26*x2^17*x3^16*x4^6*z^13 - x1^23*x2^20*x3^16*x4^6*z^13 + 4*x1^22*x2^21*x3^16*x4^6*z^13 + 2*x1^25*x2^17*x3^17*x4^6*z^13 + x1^23*x2^19*x3^17*x4^6*z^13 - 4*x1^22*x2^20*x3^17*x4^6*z^13 - x1^21*x2^21*x3^17*x4^6*z^13 + 2*x1^23*x2^18*x3^18*x4^6*z^13 + x1^22*x2^19*x3^18*x4^6*z^13 + 4*x1^21*x2^20*x3^18*x4^6*z^13 - x1^20*x2^20*x3^19*x4^6*z^13 - x1^30*x2^21*x3^7*x4^7*z^13 + x1^29*x2^22*x3^7*x4^7*z^13 - x1^28*x2^23*x3^7*x4^7*z^13 - x1^27*x2^24*x3^7*x4^7*z^13 + 2*x1^29*x2^21*x3^8*x4^7*z^13 + x1^27*x2^23*x3^8*x4^7*z^13 - x1^29*x2^20*x3^9*x4^7*z^13 - x1^25*x2^24*x3^9*x4^7*z^13 - x1^32*x2^16*x3^10*x4^7*z^13 - x1^29*x2^19*x3^10*x4^7*z^13 + x1^28*x2^20*x3^10*x4^7*z^13 + x1^27*x2^21*x3^10*x4^7*z^13 + x1^31*x2^16*x3^11*x4^7*z^13 + 2*x1^30*x2^17*x3^11*x4^7*z^13 - x1^29*x2^18*x3^11*x4^7*z^13 - x1^26*x2^21*x3^11*x4^7*z^13 + x1^25*x2^22*x3^11*x4^7*z^13 - x1^24*x2^23*x3^11*x4^7*z^13 - x1^30*x2^16*x3^12*x4^7*z^13 - 2*x1^29*x2^17*x3^12*x4^7*z^13 + 2*x1^28*x2^18*x3^12*x4^7*z^13 + 2*x1^25*x2^21*x3^12*x4^7*z^13 - x1^30*x2^15*x3^13*x4^7*z^13 + x1^29*x2^16*x3^13*x4^7*z^13 + 2*x1^28*x2^17*x3^13*x4^7*z^13 + x1^27*x2^18*x3^13*x4^7*z^13 + x1^26*x2^19*x3^13*x4^7*z^13 - x1^25*x2^20*x3^13*x4^7*z^13 - x1^28*x2^16*x3^14*x4^7*z^13 + 2*x1^25*x2^19*x3^14*x4^7*z^13 + x1^24*x2^20*x3^14*x4^7*z^13 + x1^23*x2^21*x3^14*x4^7*z^13 - x1^27*x2^16*x3^15*x4^7*z^13 - x1^24*x2^19*x3^15*x4^7*z^13 - x1^23*x2^20*x3^15*x4^7*z^13 - x1^22*x2^21*x3^15*x4^7*z^13 + 3*x1^22*x2^20*x3^16*x4^7*z^13 - x1^24*x2^17*x3^17*x4^7*z^13 - 3*x1^21*x2^20*x3^17*x4^7*z^13 - x1^22*x2^18*x3^18*x4^7*z^13 + x1^21*x2^19*x3^18*x4^7*z^13 + x1^20*x2^20*x3^18*x4^7*z^13 - x1^20*x2^19*x3^19*x4^7*z^13 - 2*x1^29*x2^20*x3^8*x4^8*z^13 - x1^28*x2^21*x3^8*x4^8*z^13 - x1^27*x2^22*x3^8*x4^8*z^13 - x1^25*x2^24*x3^8*x4^8*z^13 - x1^28*x2^20*x3^9*x4^8*z^13 + x1^25*x2^23*x3^9*x4^8*z^13 - x1^28*x2^19*x3^10*x4^8*z^13 - x1^27*x2^20*x3^10*x4^8*z^13 - x1^26*x2^21*x3^10*x4^8*z^13 - x1^30*x2^16*x3^11*x4^8*z^13 - 2*x1^29*x2^17*x3^11*x4^8*z^13 + x1^28*x2^18*x3^11*x4^8*z^13 + x1^27*x2^19*x3^11*x4^8*z^13 + 3*x1^24*x2^22*x3^11*x4^8*z^13 + x1^23*x2^23*x3^11*x4^8*z^13 - x1^29*x2^16*x3^12*x4^8*z^13 - x1^27*x2^18*x3^12*x4^8*z^13 - 2*x1^25*x2^20*x3^12*x4^8*z^13 - 2*x1^24*x2^21*x3^12*x4^8*z^13 - 2*x1^23*x2^22*x3^12*x4^8*z^13 + x1^28*x2^16*x3^13*x4^8*z^13 - x1^27*x2^17*x3^13*x4^8*z^13 + x1^26*x2^18*x3^13*x4^8*z^13 + 3*x1^25*x2^19*x3^13*x4^8*z^13 - x1^24*x2^20*x3^13*x4^8*z^13 + x1^23*x2^21*x3^13*x4^8*z^13 + x1^22*x2^22*x3^13*x4^8*z^13 - x1^28*x2^15*x3^14*x4^8*z^13 - x1^27*x2^16*x3^14*x4^8*z^13 - x1^26*x2^17*x3^14*x4^8*z^13 - 2*x1^24*x2^19*x3^14*x4^8*z^13 + 2*x1^23*x2^20*x3^14*x4^8*z^13 - x1^22*x2^21*x3^14*x4^8*z^13 + x1^26*x2^16*x3^15*x4^8*z^13 + x1^25*x2^17*x3^15*x4^8*z^13 + x1^24*x2^18*x3^15*x4^8*z^13 + 3*x1^23*x2^19*x3^15*x4^8*z^13 + x1^21*x2^21*x3^15*x4^8*z^13 - x1^25*x2^16*x3^16*x4^8*z^13 - x1^23*x2^18*x3^16*x4^8*z^13 - x1^22*x2^19*x3^16*x4^8*z^13 + x1^22*x2^18*x3^17*x4^8*z^13 + x1^20*x2^20*x3^17*x4^8*z^13 - x1^21*x2^18*x3^18*x4^8*z^13 + 2*x1^28*x2^19*x3^9*x4^9*z^13 + x1^27*x2^20*x3^9*x4^9*z^13 + x1^26*x2^21*x3^9*x4^9*z^13 + x1^24*x2^23*x3^9*x4^9*z^13 - x1^27*x2^19*x3^10*x4^9*z^13 - x1^26*x2^20*x3^10*x4^9*z^13 - x1^25*x2^21*x3^10*x4^9*z^13 - 2*x1^24*x2^22*x3^10*x4^9*z^13 + 2*x1^27*x2^18*x3^11*x4^9*z^13 + 2*x1^25*x2^20*x3^11*x4^9*z^13 + x1^24*x2^21*x3^11*x4^9*z^13 + x1^23*x2^22*x3^11*x4^9*z^13 + x1^28*x2^16*x3^12*x4^9*z^13 - x1^26*x2^18*x3^12*x4^9*z^13 - x1^24*x2^20*x3^12*x4^9*z^13 - 2*x1^23*x2^21*x3^12*x4^9*z^13 + 2*x1^26*x2^17*x3^13*x4^9*z^13 + 3*x1^24*x2^19*x3^13*x4^9*z^13 + 2*x1^23*x2^20*x3^13*x4^9*z^13 + 2*x1^22*x2^21*x3^13*x4^9*z^13 + x1^27*x2^15*x3^14*x4^9*z^13 - x1^24*x2^18*x3^14*x4^9*z^13 - 4*x1^23*x2^19*x3^14*x4^9*z^13 - x1^22*x2^20*x3^14*x4^9*z^13 - x1^26*x2^15*x3^15*x4^9*z^13 - 2*x1^24*x2^17*x3^15*x4^9*z^13 + 4*x1^23*x2^18*x3^15*x4^9*z^13 - x1^21*x2^20*x3^15*x4^9*z^13 - x1^24*x2^16*x3^16*x4^9*z^13 - 4*x1^22*x2^18*x3^16*x4^9*z^13 - x1^20*x2^20*x3^16*x4^9*z^13 + 2*x1^21*x2^18*x3^17*x4^9*z^13 - x1^27*x2^17*x3^11*x4^10*z^13 + x1^26*x2^18*x3^11*x4^10*z^13 + 2*x1^25*x2^19*x3^11*x4^10*z^13 + x1^23*x2^21*x3^11*x4^10*z^13 - x1^25*x2^18*x3^12*x4^10*z^13 + x1^23*x2^20*x3^12*x4^10*z^13 - 2*x1^22*x2^21*x3^12*x4^10*z^13 - x1^26*x2^16*x3^13*x4^10*z^13 + x1^24*x2^18*x3^13*x4^10*z^13 + x1^23*x2^19*x3^13*x4^10*z^13 + x1^22*x2^20*x3^13*x4^10*z^13 - x1^25*x2^16*x3^14*x4^10*z^13 + x1^23*x2^18*x3^14*x4^10*z^13 + 2*x1^22*x2^19*x3^14*x4^10*z^13 - x1^23*x2^17*x3^15*x4^10*z^13 + x1^22*x2^18*x3^15*x4^10*z^13 + x1^23*x2^16*x3^16*x4^10*z^13 - x1^21*x2^18*x3^16*x4^10*z^13 + x1^20*x2^19*x3^16*x4^10*z^13 - x1^26*x2^17*x3^11*x4^11*z^13 - x1^23*x2^20*x3^11*x4^11*z^13 - x1^25*x2^17*x3^12*x4^11*z^13 - 2*x1^24*x2^18*x3^12*x4^11*z^13 + x1^23*x2^19*x3^12*x4^11*z^13 - x1^21*x2^21*x3^12*x4^11*z^13 + x1^23*x2^18*x3^13*x4^11*z^13 + 2*x1^21*x2^20*x3^13*x4^11*z^13 - x1^23*x2^17*x3^14*x4^11*z^13 + x1^22*x2^18*x3^14*x4^11*z^13 - x1^21*x2^19*x3^14*x4^11*z^13 - x1^20*x2^20*x3^14*x4^11*z^13 - x1^21*x2^18*x3^15*x4^11*z^13 + 2*x1^20*x2^19*x3^15*x4^11*z^13 - x1^19*x2^19*x3^16*x4^11*z^13 + x1^22*x2^19*x3^12*x4^12*z^13 + x1^23*x2^17*x3^13*x4^12*z^13 - x1^22*x2^18*x3^13*x4^12*z^13 + x1^20*x2^20*x3^13*x4^12*z^13 - x1^22*x2^17*x3^14*x4^12*z^13 - 2*x1^20*x2^19*x3^14*x4^12*z^13 + x1^20*x2^18*x3^15*x4^12*z^13 + x1^19*x2^19*x3^15*x4^12*z^13 - x1^21*x2^17*x3^14*x4^13*z^13 - x1^32*x2^23*x3^5*z^12 + x1^34*x2^20*x3^6*z^12 + 2*x1^33*x2^21*x3^6*z^12 - x1^32*x2^22*x3^6*z^12 + x1^31*x2^23*x3^6*z^12 - 2*x1^33*x2^20*x3^7*z^12 - x1^32*x2^21*x3^7*z^12 + x1^33*x2^19*x3^8*z^12 + 2*x1^32*x2^20*x3^8*z^12 + x1^30*x2^22*x3^8*z^12 - x1^29*x2^23*x3^8*z^12 - 2*x1^32*x2^19*x3^9*z^12 + x1^27*x2^24*x3^9*z^12 + 2*x1^31*x2^19*x3^10*z^12 + 2*x1^29*x2^21*x3^10*z^12 - x1^28*x2^22*x3^10*z^12 - 2*x1^26*x2^24*x3^10*z^12 - 2*x1^31*x2^18*x3^11*z^12 - x1^30*x2^19*x3^11*z^12 - x1^29*x2^20*x3^11*z^12 + x1^27*x2^22*x3^11*z^12 + x1^26*x2^23*x3^11*z^12 + 2*x1^25*x2^24*x3^11*z^12 + x1^31*x2^17*x3^12*z^12 + 2*x1^30*x2^18*x3^12*z^12 + x1^28*x2^20*x3^12*z^12 - x1^27*x2^21*x3^12*z^12 - 2*x1^25*x2^23*x3^12*z^12 - x1^24*x2^24*x3^12*z^12 - 2*x1^30*x2^17*x3^13*z^12 - x1^29*x2^18*x3^13*z^12 - x1^28*x2^19*x3^13*z^12 + x1^26*x2^21*x3^13*z^12 + x1^25*x2^22*x3^13*z^12 + 2*x1^24*x2^23*x3^13*z^12 + 2*x1^29*x2^17*x3^14*z^12 + x1^27*x2^19*x3^14*z^12 - 2*x1^26*x2^20*x3^14*z^12 - 2*x1^24*x2^22*x3^14*z^12 - x1^28*x2^17*x3^15*z^12 - x1^26*x2^19*x3^15*z^12 - x1^25*x2^20*x3^15*z^12 + x1^24*x2^21*x3^15*z^12 + x1^23*x2^22*x3^15*z^12 + x1^27*x2^17*x3^16*z^12 + 2*x1^26*x2^18*x3^16*z^12 + x1^25*x2^19*x3^16*z^12 - x1^24*x2^20*x3^16*z^12 - x1^23*x2^21*x3^16*z^12 - x1^26*x2^17*x3^17*z^12 - x1^24*x2^19*x3^17*z^12 + x1^22*x2^21*x3^17*z^12 + x1^34*x2^21*x3^4*x4*z^12 - x1^32*x2^23*x3^4*x4*z^12 - x1^30*x2^25*x3^4*x4*z^12 - 2*x1^33*x2^21*x3^5*x4*z^12 + 2*x1^30*x2^24*x3^5*x4*z^12 + x1^29*x2^25*x3^5*x4*z^12 + 2*x1^33*x2^20*x3^6*x4*z^12 + 2*x1^32*x2^21*x3^6*x4*z^12 - x1^31*x2^22*x3^6*x4*z^12 - x1^30*x2^23*x3^6*x4*z^12 - 2*x1^29*x2^24*x3^6*x4*z^12 - x1^28*x2^25*x3^6*x4*z^12 - x1^33*x2^19*x3^7*x4*z^12 - 4*x1^32*x2^20*x3^7*x4*z^12 - x1^30*x2^22*x3^7*x4*z^12 + 3*x1^29*x2^23*x3^7*x4*z^12 + 2*x1^28*x2^24*x3^7*x4*z^12 + 2*x1^27*x2^25*x3^7*x4*z^12 + 5*x1^32*x2^19*x3^8*x4*z^12 + 3*x1^31*x2^20*x3^8*x4*z^12 + x1^30*x2^21*x3^8*x4*z^12 - x1^29*x2^22*x3^8*x4*z^12 - x1^28*x2^23*x3^8*x4*z^12 - 3*x1^27*x2^24*x3^8*x4*z^12 - 2*x1^26*x2^25*x3^8*x4*z^12 - 2*x1^32*x2^18*x3^9*x4*z^12 - 6*x1^31*x2^19*x3^9*x4*z^12 - 4*x1^29*x2^21*x3^9*x4*z^12 + 4*x1^28*x2^22*x3^9*x4*z^12 + 6*x1^26*x2^24*x3^9*x4*z^12 + x1^25*x2^25*x3^9*x4*z^12 + 6*x1^31*x2^18*x3^10*x4*z^12 + 2*x1^30*x2^19*x3^10*x4*z^12 + 2*x1^29*x2^20*x3^10*x4*z^12 - 2*x1^27*x2^22*x3^10*x4*z^12 - 2*x1^26*x2^23*x3^10*x4*z^12 - 6*x1^25*x2^24*x3^10*x4*z^12 - x1^31*x2^17*x3^11*x4*z^12 - 6*x1^30*x2^18*x3^11*x4*z^12 - 4*x1^28*x2^20*x3^11*x4*z^12 + 4*x1^27*x2^21*x3^11*x4*z^12 + 6*x1^25*x2^23*x3^11*x4*z^12 + 2*x1^24*x2^24*x3^11*x4*z^12 + 4*x1^30*x2^17*x3^12*x4*z^12 + 3*x1^29*x2^18*x3^12*x4*z^12 + 2*x1^28*x2^19*x3^12*x4*z^12 - 2*x1^26*x2^21*x3^12*x4*z^12 - 2*x1^25*x2^22*x3^12*x4*z^12 - 6*x1^24*x2^23*x3^12*x4*z^12 - 4*x1^29*x2^17*x3^13*x4*z^12 - 3*x1^27*x2^19*x3^13*x4*z^12 + 4*x1^26*x2^20*x3^13*x4*z^12 + 6*x1^24*x2^22*x3^13*x4*z^12 + 2*x1^23*x2^23*x3^13*x4*z^12 + x1^29*x2^16*x3^14*x4*z^12 + 2*x1^28*x2^17*x3^14*x4*z^12 + x1^27*x2^18*x3^14*x4*z^12 - x1^25*x2^20*x3^14*x4*z^12 - 2*x1^24*x2^21*x3^14*x4*z^12 - 6*x1^23*x2^22*x3^14*x4*z^12 - x1^28*x2^16*x3^15*x4*z^12 - x1^26*x2^18*x3^15*x4*z^12 + 4*x1^25*x2^19*x3^15*x4*z^12 - x1^24*x2^20*x3^15*x4*z^12 + 6*x1^23*x2^21*x3^15*x4*z^12 + 2*x1^22*x2^22*x3^15*x4*z^12 - x1^25*x2^18*x3^16*x4*z^12 - 2*x1^23*x2^20*x3^16*x4*z^12 - 5*x1^22*x2^21*x3^16*x4*z^12 - x1^25*x2^17*x3^17*x4*z^12 + x1^24*x2^18*x3^17*x4*z^12 + 3*x1^22*x2^20*x3^17*x4*z^12 + x1^21*x2^21*x3^17*x4*z^12 - x1^23*x2^18*x3^18*x4*z^12 - x1^21*x2^20*x3^18*x4*z^12 + x1^31*x2^23*x3^4*x4^2*z^12 + x1^32*x2^21*x3^5*x4^2*z^12 + x1^31*x2^22*x3^5*x4^2*z^12 + x1^30*x2^23*x3^5*x4^2*z^12 + x1^29*x2^24*x3^5*x4^2*z^12 + x1^28*x2^25*x3^5*x4^2*z^12 + x1^32*x2^20*x3^6*x4^2*z^12 + x1^31*x2^21*x3^6*x4^2*z^12 + x1^30*x2^22*x3^6*x4^2*z^12 - 3*x1^29*x2^23*x3^6*x4^2*z^12 - 2*x1^27*x2^25*x3^6*x4^2*z^12 - 2*x1^32*x2^19*x3^7*x4^2*z^12 - x1^31*x2^20*x3^7*x4^2*z^12 + x1^30*x2^21*x3^7*x4^2*z^12 + x1^29*x2^22*x3^7*x4^2*z^12 + x1^28*x2^23*x3^7*x4^2*z^12 + 2*x1^27*x2^24*x3^7*x4^2*z^12 + 3*x1^26*x2^25*x3^7*x4^2*z^12 + 4*x1^31*x2^19*x3^8*x4^2*z^12 - 4*x1^28*x2^22*x3^8*x4^2*z^12 - 5*x1^26*x2^24*x3^8*x4^2*z^12 - 2*x1^25*x2^25*x3^8*x4^2*z^12 - 4*x1^31*x2^18*x3^9*x4^2*z^12 - 2*x1^30*x2^19*x3^9*x4^2*z^12 + x1^28*x2^21*x3^9*x4^2*z^12 + 2*x1^27*x2^22*x3^9*x4^2*z^12 + 2*x1^26*x2^23*x3^9*x4^2*z^12 + 5*x1^25*x2^24*x3^9*x4^2*z^12 + x1^31*x2^17*x3^10*x4^2*z^12 + 5*x1^30*x2^18*x3^10*x4^2*z^12 + x1^29*x2^19*x3^10*x4^2*z^12 + 3*x1^28*x2^20*x3^10*x4^2*z^12 - 4*x1^27*x2^21*x3^10*x4^2*z^12 - 6*x1^25*x2^23*x3^10*x4^2*z^12 - 2*x1^24*x2^24*x3^10*x4^2*z^12 - 5*x1^30*x2^17*x3^11*x4^2*z^12 - 2*x1^29*x2^18*x3^11*x4^2*z^12 - 2*x1^28*x2^19*x3^11*x4^2*z^12 + 2*x1^26*x2^21*x3^11*x4^2*z^12 + 2*x1^25*x2^22*x3^11*x4^2*z^12 + 6*x1^24*x2^23*x3^11*x4^2*z^12 + x1^30*x2^16*x3^12*x4^2*z^12 + 5*x1^29*x2^17*x3^12*x4^2*z^12 + x1^28*x2^18*x3^12*x4^2*z^12 + 4*x1^27*x2^19*x3^12*x4^2*z^12 - 4*x1^26*x2^20*x3^12*x4^2*z^12 - 6*x1^24*x2^22*x3^12*x4^2*z^12 - 2*x1^23*x2^23*x3^12*x4^2*z^12 - 2*x1^29*x2^16*x3^13*x4^2*z^12 - 3*x1^28*x2^17*x3^13*x4^2*z^12 - 2*x1^27*x2^18*x3^13*x4^2*z^12 + 2*x1^25*x2^20*x3^13*x4^2*z^12 + 2*x1^24*x2^21*x3^13*x4^2*z^12 + 6*x1^23*x2^22*x3^13*x4^2*z^12 + 2*x1^28*x2^16*x3^14*x4^2*z^12 + x1^27*x2^17*x3^14*x4^2*z^12 + x1^26*x2^18*x3^14*x4^2*z^12 - 5*x1^25*x2^19*x3^14*x4^2*z^12 - 6*x1^23*x2^21*x3^14*x4^2*z^12 - 2*x1^22*x2^22*x3^14*x4^2*z^12 - x1^27*x2^16*x3^15*x4^2*z^12 - x1^26*x2^17*x3^15*x4^2*z^12 + x1^24*x2^19*x3^15*x4^2*z^12 + x1^23*x2^20*x3^15*x4^2*z^12 + 6*x1^22*x2^21*x3^15*x4^2*z^12 + x1^25*x2^17*x3^16*x4^2*z^12 - x1^24*x2^18*x3^16*x4^2*z^12 + x1^23*x2^19*x3^16*x4^2*z^12 - 4*x1^22*x2^20*x3^16*x4^2*z^12 - 2*x1^21*x2^21*x3^16*x4^2*z^12 - x1^23*x2^18*x3^17*x4^2*z^12 + 4*x1^21*x2^20*x3^17*x4^2*z^12 + x1^22*x2^18*x3^18*x4^2*z^12 - x1^21*x2^19*x3^18*x4^2*z^12 - x1^20*x2^20*x3^18*x4^2*z^12 - x1^29*x2^23*x3^5*x4^3*z^12 - 2*x1^33*x2^18*x3^6*x4^3*z^12 - x1^32*x2^19*x3^6*x4^3*z^12 + x1^30*x2^21*x3^6*x4^3*z^12 - x1^29*x2^22*x3^6*x4^3*z^12 - x1^28*x2^23*x3^6*x4^3*z^12 - x1^27*x2^24*x3^6*x4^3*z^12 + x1^33*x2^17*x3^7*x4^3*z^12 - x1^31*x2^19*x3^7*x4^3*z^12 - x1^30*x2^20*x3^7*x4^3*z^12 + x1^29*x2^21*x3^7*x4^3*z^12 + x1^26*x2^24*x3^7*x4^3*z^12 - x1^32*x2^17*x3^8*x4^3*z^12 - x1^30*x2^19*x3^8*x4^3*z^12 + x1^29*x2^20*x3^8*x4^3*z^12 + x1^28*x2^21*x3^8*x4^3*z^12 + x1^27*x2^22*x3^8*x4^3*z^12 - 2*x1^26*x2^23*x3^8*x4^3*z^12 - x1^25*x2^24*x3^8*x4^3*z^12 + 2*x1^32*x2^16*x3^9*x4^3*z^12 + x1^31*x2^17*x3^9*x4^3*z^12 - x1^30*x2^18*x3^9*x4^3*z^12 - x1^29*x2^19*x3^9*x4^3*z^12 + 2*x1^28*x2^20*x3^9*x4^3*z^12 + 2*x1^27*x2^21*x3^9*x4^3*z^12 - 2*x1^26*x2^22*x3^9*x4^3*z^12 + 2*x1^25*x2^23*x3^9*x4^3*z^12 + x1^24*x2^24*x3^9*x4^3*z^12 - 2*x1^31*x2^16*x3^10*x4^3*z^12 + x1^30*x2^17*x3^10*x4^3*z^12 - 3*x1^29*x2^18*x3^10*x4^3*z^12 + 2*x1^26*x2^21*x3^10*x4^3*z^12 + x1^25*x2^22*x3^10*x4^3*z^12 - 2*x1^24*x2^23*x3^10*x4^3*z^12 + 2*x1^31*x2^15*x3^11*x4^3*z^12 + 2*x1^30*x2^16*x3^11*x4^3*z^12 - x1^28*x2^18*x3^11*x4^3*z^12 - 2*x1^27*x2^19*x3^11*x4^3*z^12 - 2*x1^25*x2^21*x3^11*x4^3*z^12 + 2*x1^24*x2^22*x3^11*x4^3*z^12 + x1^23*x2^23*x3^11*x4^3*z^12 - 2*x1^30*x2^15*x3^12*x4^3*z^12 + x1^28*x2^17*x3^12*x4^3*z^12 + 2*x1^27*x2^18*x3^12*x4^3*z^12 + x1^25*x2^20*x3^12*x4^3*z^12 - 2*x1^23*x2^22*x3^12*x4^3*z^12 - x1^27*x2^17*x3^13*x4^3*z^12 - 2*x1^26*x2^18*x3^13*x4^3*z^12 - x1^25*x2^19*x3^13*x4^3*z^12 - 2*x1^24*x2^20*x3^13*x4^3*z^12 + 2*x1^23*x2^21*x3^13*x4^3*z^12 + x1^22*x2^22*x3^13*x4^3*z^12 - x1^27*x2^16*x3^14*x4^3*z^12 + 2*x1^26*x2^17*x3^14*x4^3*z^12 + x1^25*x2^18*x3^14*x4^3*z^12 + 3*x1^24*x2^19*x3^14*x4^3*z^12 - x1^23*x2^20*x3^14*x4^3*z^12 - 2*x1^22*x2^21*x3^14*x4^3*z^12 - 4*x1^23*x2^19*x3^15*x4^3*z^12 + x1^22*x2^20*x3^15*x4^3*z^12 - x1^24*x2^17*x3^16*x4^3*z^12 + 2*x1^23*x2^18*x3^16*x4^3*z^12 + 3*x1^22*x2^19*x3^16*x4^3*z^12 - x1^21*x2^20*x3^16*x4^3*z^12 - x1^22*x2^18*x3^17*x4^3*z^12 - x1^21*x2^19*x3^17*x4^3*z^12 + x1^20*x2^20*x3^17*x4^3*z^12 + x1^21*x2^18*x3^18*x4^3*z^12 + x1^31*x2^21*x3^4*x4^4*z^12 + x1^30*x2^22*x3^4*x4^4*z^12 - 2*x1^30*x2^21*x3^5*x4^4*z^12 - x1^28*x2^23*x3^5*x4^4*z^12 + 2*x1^30*x2^20*x3^6*x4^4*z^12 + 2*x1^29*x2^21*x3^6*x4^4*z^12 + 2*x1^28*x2^22*x3^6*x4^4*z^12 + 2*x1^27*x2^23*x3^6*x4^4*z^12 - x1^26*x2^24*x3^6*x4^4*z^12 + 2*x1^32*x2^17*x3^7*x4^4*z^12 + x1^31*x2^18*x3^7*x4^4*z^12 - 4*x1^29*x2^20*x3^7*x4^4*z^12 - 3*x1^27*x2^22*x3^7*x4^4*z^12 - 3*x1^26*x2^23*x3^7*x4^4*z^12 - x1^25*x2^24*x3^7*x4^4*z^12 - x1^32*x2^16*x3^8*x4^4*z^12 - x1^31*x2^17*x3^8*x4^4*z^12 - x1^30*x2^18*x3^8*x4^4*z^12 + 2*x1^29*x2^19*x3^8*x4^4*z^12 + x1^28*x2^20*x3^8*x4^4*z^12 + 3*x1^27*x2^21*x3^8*x4^4*z^12 + 5*x1^26*x2^22*x3^8*x4^4*z^12 - x1^24*x2^24*x3^8*x4^4*z^12 + 4*x1^31*x2^16*x3^9*x4^4*z^12 + x1^30*x2^17*x3^9*x4^4*z^12 + x1^29*x2^18*x3^9*x4^4*z^12 - 5*x1^28*x2^19*x3^9*x4^4*z^12 - 5*x1^26*x2^21*x3^9*x4^4*z^12 - 3*x1^25*x2^22*x3^9*x4^4*z^12 - x1^24*x2^23*x3^9*x4^4*z^12 - 2*x1^31*x2^15*x3^10*x4^4*z^12 - 3*x1^30*x2^16*x3^10*x4^4*z^12 - x1^29*x2^17*x3^10*x4^4*z^12 + x1^28*x2^18*x3^10*x4^4*z^12 + 2*x1^26*x2^20*x3^10*x4^4*z^12 + 6*x1^25*x2^21*x3^10*x4^4*z^12 + 5*x1^30*x2^15*x3^11*x4^4*z^12 + 2*x1^29*x2^16*x3^11*x4^4*z^12 + 4*x1^28*x2^17*x3^11*x4^4*z^12 - 4*x1^27*x2^18*x3^11*x4^4*z^12 - 6*x1^25*x2^20*x3^11*x4^4*z^12 - 2*x1^24*x2^21*x3^11*x4^4*z^12 - 2*x1^30*x2^14*x3^12*x4^4*z^12 - 3*x1^29*x2^15*x3^12*x4^4*z^12 - 3*x1^28*x2^16*x3^12*x4^4*z^12 - 2*x1^27*x2^17*x3^12*x4^4*z^12 - x1^26*x2^18*x3^12*x4^4*z^12 + 2*x1^25*x2^19*x3^12*x4^4*z^12 + 6*x1^24*x2^20*x3^12*x4^4*z^12 + 2*x1^29*x2^14*x3^13*x4^4*z^12 + x1^28*x2^15*x3^13*x4^4*z^12 + 2*x1^27*x2^16*x3^13*x4^4*z^12 - 3*x1^26*x2^17*x3^13*x4^4*z^12 + 2*x1^25*x2^18*x3^13*x4^4*z^12 - 5*x1^24*x2^19*x3^13*x4^4*z^12 - 2*x1^23*x2^20*x3^13*x4^4*z^12 - x1^27*x2^15*x3^14*x4^4*z^12 + 5*x1^23*x2^19*x3^14*x4^4*z^12 + 2*x1^26*x2^15*x3^15*x4^4*z^12 - x1^25*x2^16*x3^15*x4^4*z^12 + 2*x1^24*x2^17*x3^15*x4^4*z^12 - 3*x1^23*x2^18*x3^15*x4^4*z^12 - x1^22*x2^19*x3^15*x4^4*z^12 + 2*x1^24*x2^16*x3^16*x4^4*z^12 + 3*x1^22*x2^18*x3^16*x4^4*z^12 - x1^21*x2^18*x3^17*x4^4*z^12 - x1^19*x2^19*x3^18*x4^4*z^12 + x1^27*x2^23*x3^5*x4^5*z^12 + x1^29*x2^20*x3^6*x4^5*z^12 + 2*x1^27*x2^22*x3^6*x4^5*z^12 + 2*x1^26*x2^23*x3^6*x4^5*z^12 + x1^25*x2^24*x3^6*x4^5*z^12 - 2*x1^29*x2^19*x3^7*x4^5*z^12 - x1^28*x2^20*x3^7*x4^5*z^12 - 2*x1^26*x2^22*x3^7*x4^5*z^12 + x1^25*x2^23*x3^7*x4^5*z^12 - x1^24*x2^24*x3^7*x4^5*z^12 + x1^29*x2^18*x3^8*x4^5*z^12 + 5*x1^28*x2^19*x3^8*x4^5*z^12 - x1^27*x2^20*x3^8*x4^5*z^12 + 2*x1^26*x2^21*x3^8*x4^5*z^12 + 2*x1^25*x2^22*x3^8*x4^5*z^12 + x1^30*x2^16*x3^9*x4^5*z^12 - 2*x1^28*x2^18*x3^9*x4^5*z^12 - x1^27*x2^19*x3^9*x4^5*z^12 - 4*x1^25*x2^21*x3^9*x4^5*z^12 + x1^24*x2^22*x3^9*x4^5*z^12 + x1^23*x2^23*x3^9*x4^5*z^12 - 2*x1^30*x2^15*x3^10*x4^5*z^12 + 5*x1^27*x2^18*x3^10*x4^5*z^12 - x1^26*x2^19*x3^10*x4^5*z^12 + 4*x1^25*x2^20*x3^10*x4^5*z^12 + 3*x1^24*x2^21*x3^10*x4^5*z^12 - 2*x1^23*x2^22*x3^10*x4^5*z^12 + 2*x1^29*x2^15*x3^11*x4^5*z^12 + x1^28*x2^16*x3^11*x4^5*z^12 - x1^27*x2^17*x3^11*x4^5*z^12 - 3*x1^26*x2^18*x3^11*x4^5*z^12 - x1^25*x2^19*x3^11*x4^5*z^12 - 5*x1^24*x2^20*x3^11*x4^5*z^12 + x1^23*x2^21*x3^11*x4^5*z^12 - x1^29*x2^14*x3^12*x4^5*z^12 - x1^27*x2^16*x3^12*x4^5*z^12 + 4*x1^26*x2^17*x3^12*x4^5*z^12 - x1^25*x2^18*x3^12*x4^5*z^12 + 6*x1^24*x2^19*x3^12*x4^5*z^12 + 2*x1^23*x2^20*x3^12*x4^5*z^12 - 2*x1^22*x2^21*x3^12*x4^5*z^12 - 2*x1^25*x2^17*x3^13*x4^5*z^12 - x1^24*x2^18*x3^13*x4^5*z^12 - 6*x1^23*x2^19*x3^13*x4^5*z^12 + 2*x1^22*x2^20*x3^13*x4^5*z^12 + x1^21*x2^21*x3^13*x4^5*z^12 + 2*x1^25*x2^16*x3^14*x4^5*z^12 - 3*x1^24*x2^17*x3^14*x4^5*z^12 + 3*x1^23*x2^18*x3^14*x4^5*z^12 - 2*x1^21*x2^20*x3^14*x4^5*z^12 + x1^24*x2^16*x3^15*x4^5*z^12 + 2*x1^23*x2^17*x3^15*x4^5*z^12 - 4*x1^22*x2^18*x3^15*x4^5*z^12 + 2*x1^21*x2^19*x3^15*x4^5*z^12 + x1^20*x2^20*x3^15*x4^5*z^12 - x1^23*x2^16*x3^16*x4^5*z^12 + x1^22*x2^17*x3^16*x4^5*z^12 + 2*x1^21*x2^18*x3^16*x4^5*z^12 - 2*x1^20*x2^19*x3^16*x4^5*z^12 - x1^21*x2^17*x3^17*x4^5*z^12 + x1^19*x2^19*x3^17*x4^5*z^12 - x1^29*x2^19*x3^6*x4^6*z^12 - x1^28*x2^20*x3^6*x4^6*z^12 - 2*x1^26*x2^22*x3^6*x4^6*z^12 - x1^25*x2^23*x3^6*x4^6*z^12 - x1^28*x2^19*x3^7*x4^6*z^12 + x1^26*x2^21*x3^7*x4^6*z^12 + x1^24*x2^23*x3^7*x4^6*z^12 + x1^28*x2^18*x3^8*x4^6*z^12 - x1^27*x2^19*x3^8*x4^6*z^12 - 4*x1^26*x2^20*x3^8*x4^6*z^12 - x1^24*x2^22*x3^8*x4^6*z^12 - 2*x1^30*x2^15*x3^9*x4^6*z^12 - 3*x1^27*x2^18*x3^9*x4^6*z^12 + x1^26*x2^19*x3^9*x4^6*z^12 + x1^25*x2^20*x3^9*x4^6*z^12 + 3*x1^23*x2^22*x3^9*x4^6*z^12 - x1^29*x2^15*x3^10*x4^6*z^12 + 2*x1^28*x2^16*x3^10*x4^6*z^12 + x1^27*x2^17*x3^10*x4^6*z^12 - 4*x1^25*x2^19*x3^10*x4^6*z^12 - 2*x1^23*x2^21*x3^10*x4^6*z^12 - x1^22*x2^22*x3^10*x4^6*z^12 - x1^29*x2^14*x3^11*x4^6*z^12 - 3*x1^28*x2^15*x3^11*x4^6*z^12 - x1^27*x2^16*x3^11*x4^6*z^12 - x1^26*x2^17*x3^11*x4^6*z^12 + 2*x1^25*x2^18*x3^11*x4^6*z^12 - x1^24*x2^19*x3^11*x4^6*z^12 - x1^23*x2^20*x3^11*x4^6*z^12 + 5*x1^22*x2^21*x3^11*x4^6*z^12 + 4*x1^27*x2^15*x3^12*x4^6*z^12 - x1^26*x2^16*x3^12*x4^6*z^12 + 3*x1^25*x2^17*x3^12*x4^6*z^12 + x1^23*x2^19*x3^12*x4^6*z^12 - 3*x1^22*x2^20*x3^12*x4^6*z^12 - x1^21*x2^21*x3^12*x4^6*z^12 - 3*x1^26*x2^15*x3^13*x4^6*z^12 + 3*x1^24*x2^17*x3^13*x4^6*z^12 + 4*x1^21*x2^20*x3^13*x4^6*z^12 + 2*x1^25*x2^15*x3^14*x4^6*z^12 + 2*x1^24*x2^16*x3^14*x4^6*z^12 - 2*x1^23*x2^17*x3^14*x4^6*z^12 + x1^22*x2^18*x3^14*x4^6*z^12 - 3*x1^21*x2^19*x3^14*x4^6*z^12 - 2*x1^20*x2^20*x3^14*x4^6*z^12 - 2*x1^24*x2^15*x3^15*x4^6*z^12 - x1^23*x2^16*x3^15*x4^6*z^12 + 2*x1^21*x2^18*x3^15*x4^6*z^12 + 3*x1^20*x2^19*x3^15*x4^6*z^12 - 2*x1^22*x2^16*x3^16*x4^6*z^12 + x1^21*x2^17*x3^16*x4^6*z^12 - 3*x1^20*x2^18*x3^16*x4^6*z^12 - x1^19*x2^19*x3^16*x4^6*z^12 - 2*x1^20*x2^17*x3^17*x4^6*z^12 + 3*x1^19*x2^18*x3^17*x4^6*z^12 - 2*x1^18*x2^18*x3^18*x4^6*z^12 + x1^28*x2^18*x3^7*x4^7*z^12 + x1^27*x2^19*x3^7*x4^7*z^12 - x1^26*x2^20*x3^7*x4^7*z^12 + x1^25*x2^21*x3^7*x4^7*z^12 - x1^25*x2^20*x3^8*x4^7*z^12 - x1^24*x2^21*x3^8*x4^7*z^12 - x1^23*x2^22*x3^8*x4^7*z^12 + x1^27*x2^17*x3^9*x4^7*z^12 + x1^26*x2^18*x3^9*x4^7*z^12 + 2*x1^24*x2^20*x3^9*x4^7*z^12 - x1^23*x2^21*x3^9*x4^7*z^12 + x1^29*x2^14*x3^10*x4^7*z^12 + x1^28*x2^15*x3^10*x4^7*z^12 - x1^27*x2^16*x3^10*x4^7*z^12 - 3*x1^24*x2^19*x3^10*x4^7*z^12 - x1^27*x2^15*x3^11*x4^7*z^12 + x1^26*x2^16*x3^11*x4^7*z^12 + 3*x1^24*x2^18*x3^11*x4^7*z^12 + 2*x1^23*x2^19*x3^11*x4^7*z^12 - x1^22*x2^20*x3^11*x4^7*z^12 + x1^26*x2^15*x3^12*x4^7*z^12 - 2*x1^24*x2^17*x3^12*x4^7*z^12 - 2*x1^23*x2^18*x3^12*x4^7*z^12 - x1^22*x2^19*x3^12*x4^7*z^12 - x1^21*x2^20*x3^12*x4^7*z^12 - x1^25*x2^15*x3^13*x4^7*z^12 + 2*x1^21*x2^19*x3^13*x4^7*z^12 + x1^24*x2^15*x3^14*x4^7*z^12 - x1^23*x2^16*x3^14*x4^7*z^12 - 2*x1^21*x2^18*x3^14*x4^7*z^12 - 2*x1^20*x2^19*x3^14*x4^7*z^12 + 2*x1^22*x2^16*x3^15*x4^7*z^12 - x1^21*x2^17*x3^15*x4^7*z^12 + 2*x1^20*x2^18*x3^15*x4^7*z^12 + x1^19*x2^19*x3^15*x4^7*z^12 + x1^20*x2^17*x3^16*x4^7*z^12 - 2*x1^19*x2^18*x3^16*x4^7*z^12 + x1^18*x2^18*x3^17*x4^7*z^12 + x1^26*x2^18*x3^8*x4^8*z^12 + 2*x1^25*x2^19*x3^8*x4^8*z^12 + 2*x1^24*x2^20*x3^8*x4^8*z^12 + x1^23*x2^21*x3^8*x4^8*z^12 - x1^26*x2^17*x3^9*x4^8*z^12 + x1^24*x2^19*x3^9*x4^8*z^12 + x1^23*x2^20*x3^9*x4^8*z^12 + x1^25*x2^17*x3^10*x4^8*z^12 + 2*x1^24*x2^18*x3^10*x4^8*z^12 + 2*x1^22*x2^20*x3^10*x4^8*z^12 + 2*x1^26*x2^15*x3^11*x4^8*z^12 - x1^23*x2^18*x3^11*x4^8*z^12 + 2*x1^22*x2^19*x3^11*x4^8*z^12 - 2*x1^21*x2^20*x3^11*x4^8*z^12 + x1^26*x2^14*x3^12*x4^8*z^12 + 2*x1^24*x2^16*x3^12*x4^8*z^12 + x1^22*x2^18*x3^12*x4^8*z^12 + 2*x1^20*x2^20*x3^12*x4^8*z^12 - x1^25*x2^14*x3^13*x4^8*z^12 - x1^24*x2^15*x3^13*x4^8*z^12 + x1^23*x2^16*x3^13*x4^8*z^12 - 2*x1^22*x2^17*x3^13*x4^8*z^12 - x1^21*x2^18*x3^13*x4^8*z^12 + x1^24*x2^14*x3^14*x4^8*z^12 + x1^23*x2^15*x3^14*x4^8*z^12 + 3*x1^21*x2^17*x3^14*x4^8*z^12 - x1^19*x2^19*x3^14*x4^8*z^12 - x1^21*x2^16*x3^15*x4^8*z^12 - x1^19*x2^18*x3^15*x4^8*z^12 + x1^20*x2^16*x3^16*x4^8*z^12 - x1^25*x2^17*x3^9*x4^9*z^12 - 2*x1^24*x2^18*x3^9*x4^9*z^12 - 2*x1^23*x2^19*x3^9*x4^9*z^12 - x1^22*x2^20*x3^9*x4^9*z^12 + x1^24*x2^17*x3^10*x4^9*z^12 + x1^23*x2^18*x3^10*x4^9*z^12 - x1^22*x2^19*x3^10*x4^9*z^12 + 2*x1^21*x2^20*x3^10*x4^9*z^12 - x1^25*x2^15*x3^11*x4^9*z^12 - x1^24*x2^16*x3^11*x4^9*z^12 - x1^23*x2^17*x3^11*x4^9*z^12 - 2*x1^22*x2^18*x3^11*x4^9*z^12 - x1^21*x2^19*x3^11*x4^9*z^12 + x1^24*x2^15*x3^12*x4^9*z^12 - x1^23*x2^16*x3^12*x4^9*z^12 + x1^22*x2^17*x3^12*x4^9*z^12 + x1^20*x2^19*x3^12*x4^9*z^12 + x1^24*x2^14*x3^13*x4^9*z^12 - 3*x1^21*x2^17*x3^13*x4^9*z^12 - x1^19*x2^19*x3^13*x4^9*z^12 - x1^22*x2^15*x3^14*x4^9*z^12 + 2*x1^21*x2^16*x3^14*x4^9*z^12 + x1^20*x2^17*x3^14*x4^9*z^12 + x1^19*x2^18*x3^14*x4^9*z^12 + 2*x1^21*x2^15*x3^15*x4^9*z^12 - 3*x1^20*x2^16*x3^15*x4^9*z^12 + x1^18*x2^18*x3^15*x4^9*z^12 + x1^19*x2^16*x3^16*x4^9*z^12 + x1^24*x2^16*x3^10*x4^10*z^12 + x1^24*x2^15*x3^11*x4^10*z^12 + x1^23*x2^16*x3^11*x4^10*z^12 - x1^22*x2^17*x3^11*x4^10*z^12 - 2*x1^20*x2^19*x3^11*x4^10*z^12 + x1^22*x2^16*x3^12*x4^10*z^12 + x1^21*x2^17*x3^12*x4^10*z^12 - x1^20*x2^17*x3^13*x4^10*z^12 - 2*x1^19*x2^18*x3^13*x4^10*z^12 + 2*x1^20*x2^16*x3^14*x4^10*z^12 - x1^19*x2^17*x3^14*x4^10*z^12 - x1^18*x2^18*x3^14*x4^10*z^12 + x1^21*x2^17*x3^11*x4^11*z^12 + 2*x1^21*x2^16*x3^12*x4^11*z^12 + x1^20*x2^17*x3^12*x4^11*z^12 + x1^19*x2^18*x3^12*x4^11*z^12 - x1^18*x2^18*x3^13*x4^11*z^12 + x1^18*x2^17*x3^14*x4^11*z^12 + x1^17*x2^17*x3^14*x4^12*z^12 - x1^32*x2^19*x3^4*z^11 + x1^30*x2^21*x3^4*z^11 + x1^31*x2^19*x3^5*z^11 - x1^30*x2^20*x3^5*z^11 - 2*x1^31*x2^18*x3^6*z^11 - x1^30*x2^19*x3^6*z^11 - x1^29*x2^20*x3^6*z^11 + x1^27*x2^22*x3^6*z^11 + x1^31*x2^17*x3^7*z^11 + 2*x1^30*x2^18*x3^7*z^11 + x1^28*x2^20*x3^7*z^11 - x1^27*x2^21*x3^7*z^11 - x1^26*x2^22*x3^7*z^11 - x1^25*x2^23*x3^7*z^11 - 2*x1^30*x2^17*x3^8*z^11 - x1^29*x2^18*x3^8*z^11 - x1^28*x2^19*x3^8*z^11 + x1^26*x2^21*x3^8*z^11 + x1^25*x2^22*x3^8*z^11 + x1^24*x2^23*x3^8*z^11 + x1^30*x2^16*x3^9*z^11 + 2*x1^29*x2^17*x3^9*z^11 + x1^27*x2^19*x3^9*z^11 - 2*x1^26*x2^20*x3^9*z^11 - 2*x1^24*x2^22*x3^9*z^11 - 2*x1^29*x2^16*x3^10*z^11 + 2*x1^23*x2^22*x3^10*z^11 + 2*x1^28*x2^16*x3^11*z^11 + 2*x1^26*x2^18*x3^11*z^11 - x1^25*x2^19*x3^11*z^11 - 2*x1^23*x2^21*x3^11*z^11 - x1^22*x2^22*x3^11*z^11 - x1^28*x2^15*x3^12*z^11 - x1^27*x2^16*x3^12*z^11 - x1^26*x2^17*x3^12*z^11 + x1^24*x2^19*x3^12*z^11 + x1^23*x2^20*x3^12*z^11 + 2*x1^22*x2^21*x3^12*z^11 + x1^27*x2^15*x3^13*z^11 + x1^26*x2^16*x3^13*z^11 + x1^25*x2^17*x3^13*z^11 - x1^24*x2^18*x3^13*z^11 - 2*x1^22*x2^20*x3^13*z^11 - x1^21*x2^21*x3^13*z^11 - x1^25*x2^16*x3^14*z^11 + x1^23*x2^18*x3^14*z^11 + x1^22*x2^19*x3^14*z^11 + 2*x1^21*x2^20*x3^14*z^11 + x1^22*x2^18*x3^15*z^11 - x1^21*x2^19*x3^15*z^11 - x1^22*x2^17*x3^16*z^11 + x1^20*x2^19*x3^16*z^11 - x1^29*x2^22*x3^3*x4*z^11 - x1^28*x2^23*x3^3*x4*z^11 - x1^31*x2^19*x3^4*x4*z^11 - x1^30*x2^20*x3^4*x4*z^11 + 2*x1^28*x2^22*x3^4*x4*z^11 + 2*x1^27*x2^23*x3^4*x4*z^11 + x1^26*x2^24*x3^4*x4*z^11 + 2*x1^31*x2^18*x3^5*x4*z^11 + x1^30*x2^19*x3^5*x4*z^11 - x1^29*x2^20*x3^5*x4*z^11 - 2*x1^27*x2^22*x3^5*x4*z^11 - 2*x1^26*x2^23*x3^5*x4*z^11 - x1^25*x2^24*x3^5*x4*z^11 - 4*x1^30*x2^18*x3^6*x4*z^11 + 3*x1^27*x2^21*x3^6*x4*z^11 + 2*x1^26*x2^22*x3^6*x4*z^11 + 4*x1^25*x2^23*x3^6*x4*z^11 + 4*x1^30*x2^17*x3^7*x4*z^11 + 2*x1^29*x2^18*x3^7*x4*z^11 - x1^27*x2^20*x3^7*x4*z^11 - 2*x1^26*x2^21*x3^7*x4*z^11 - 3*x1^25*x2^22*x3^7*x4*z^11 - 4*x1^24*x2^23*x3^7*x4*z^11 - x1^30*x2^16*x3^8*x4*z^11 - 5*x1^29*x2^17*x3^8*x4*z^11 - x1^28*x2^18*x3^8*x4*z^11 - 3*x1^27*x2^19*x3^8*x4*z^11 + 4*x1^26*x2^20*x3^8*x4*z^11 + 6*x1^24*x2^22*x3^8*x4*z^11 + x1^23*x2^23*x3^8*x4*z^11 + 6*x1^29*x2^16*x3^9*x4*z^11 + 2*x1^28*x2^17*x3^9*x4*z^11 + 2*x1^27*x2^18*x3^9*x4*z^11 - 2*x1^25*x2^20*x3^9*x4*z^11 - 2*x1^24*x2^21*x3^9*x4*z^11 - 6*x1^23*x2^22*x3^9*x4*z^11 - x1^29*x2^15*x3^10*x4*z^11 - 6*x1^28*x2^16*x3^10*x4*z^11 - 4*x1^26*x2^18*x3^10*x4*z^11 + 4*x1^25*x2^19*x3^10*x4*z^11 + 6*x1^23*x2^21*x3^10*x4*z^11 + 2*x1^22*x2^22*x3^10*x4*z^11 + 4*x1^28*x2^15*x3^11*x4*z^11 + x1^27*x2^16*x3^11*x4*z^11 + x1^26*x2^17*x3^11*x4*z^11 - 2*x1^24*x2^19*x3^11*x4*z^11 - 2*x1^23*x2^20*x3^11*x4*z^11 - 6*x1^22*x2^21*x3^11*x4*z^11 - 4*x1^27*x2^15*x3^12*x4*z^11 - 3*x1^25*x2^17*x3^12*x4*z^11 + 3*x1^24*x2^18*x3^12*x4*z^11 + 6*x1^22*x2^20*x3^12*x4*z^11 + 2*x1^21*x2^21*x3^12*x4*z^11 + x1^26*x2^15*x3^13*x4*z^11 - x1^24*x2^17*x3^13*x4*z^11 - 2*x1^23*x2^18*x3^13*x4*z^11 - 2*x1^22*x2^19*x3^13*x4*z^11 - 6*x1^21*x2^20*x3^13*x4*z^11 - x1^25*x2^15*x3^14*x4*z^11 + 3*x1^23*x2^17*x3^14*x4*z^11 + 5*x1^21*x2^19*x3^14*x4*z^11 + 2*x1^20*x2^20*x3^14*x4*z^11 + x1^24*x2^15*x3^15*x4*z^11 - x1^23*x2^16*x3^15*x4*z^11 - x1^22*x2^17*x3^15*x4*z^11 - x1^21*x2^18*x3^15*x4*z^11 - 5*x1^20*x2^19*x3^15*x4*z^11 + 2*x1^22*x2^16*x3^16*x4*z^11 - x1^21*x2^17*x3^16*x4*z^11 + 2*x1^20*x2^18*x3^16*x4*z^11 + 2*x1^19*x2^19*x3^16*x4*z^11 - 2*x1^19*x2^18*x3^17*x4*z^11 + x1^28*x2^21*x3^4*x4^2*z^11 - x1^28*x2^20*x3^5*x4^2*z^11 - 3*x1^27*x2^21*x3^5*x4^2*z^11 - x1^26*x2^22*x3^5*x4^2*z^11 - 2*x1^25*x2^23*x3^5*x4^2*z^11 - x1^30*x2^17*x3^6*x4^2*z^11 + x1^27*x2^20*x3^6*x4^2*z^11 + 2*x1^26*x2^21*x3^6*x4^2*z^11 + 3*x1^24*x2^23*x3^6*x4^2*z^11 + 3*x1^29*x2^17*x3^7*x4^2*z^11 + x1^28*x2^18*x3^7*x4^2*z^11 - x1^27*x2^19*x3^7*x4^2*z^11 - 4*x1^26*x2^20*x3^7*x4^2*z^11 + x1^25*x2^21*x3^7*x4^2*z^11 - 5*x1^24*x2^22*x3^7*x4^2*z^11 - x1^23*x2^23*x3^7*x4^2*z^11 - 2*x1^29*x2^16*x3^8*x4^2*z^11 - 2*x1^28*x2^17*x3^8*x4^2*z^11 + 2*x1^26*x2^19*x3^8*x4^2*z^11 + 2*x1^25*x2^20*x3^8*x4^2*z^11 + 2*x1^24*x2^21*x3^8*x4^2*z^11 + 6*x1^23*x2^22*x3^8*x4^2*z^11 + 5*x1^28*x2^16*x3^9*x4^2*z^11 + x1^27*x2^17*x3^9*x4^2*z^11 + x1^26*x2^18*x3^9*x4^2*z^11 - 4*x1^25*x2^19*x3^9*x4^2*z^11 - 6*x1^23*x2^21*x3^9*x4^2*z^11 - 2*x1^22*x2^22*x3^9*x4^2*z^11 - 3*x1^28*x2^15*x3^10*x4^2*z^11 - 2*x1^27*x2^16*x3^10*x4^2*z^11 - x1^26*x2^17*x3^10*x4^2*z^11 + 2*x1^24*x2^19*x3^10*x4^2*z^11 + 2*x1^23*x2^20*x3^10*x4^2*z^11 + 6*x1^22*x2^21*x3^10*x4^2*z^11 + 4*x1^27*x2^15*x3^11*x4^2*z^11 + 2*x1^26*x2^16*x3^11*x4^2*z^11 + 3*x1^25*x2^17*x3^11*x4^2*z^11 - 4*x1^24*x2^18*x3^11*x4^2*z^11 - 6*x1^22*x2^20*x3^11*x4^2*z^11 - 2*x1^21*x2^21*x3^11*x4^2*z^11 - x1^27*x2^14*x3^12*x4^2*z^11 - 2*x1^26*x2^15*x3^12*x4^2*z^11 - 3*x1^25*x2^16*x3^12*x4^2*z^11 + x1^23*x2^18*x3^12*x4^2*z^11 + 2*x1^22*x2^19*x3^12*x4^2*z^11 + 6*x1^21*x2^20*x3^12*x4^2*z^11 + x1^26*x2^14*x3^13*x4^2*z^11 + 2*x1^25*x2^15*x3^13*x4^2*z^11 + 3*x1^24*x2^16*x3^13*x4^2*z^11 - 2*x1^23*x2^17*x3^13*x4^2*z^11 + x1^22*x2^18*x3^13*x4^2*z^11 - 6*x1^21*x2^19*x3^13*x4^2*z^11 - 2*x1^20*x2^20*x3^13*x4^2*z^11 - 2*x1^24*x2^15*x3^14*x4^2*z^11 - x1^23*x2^16*x3^14*x4^2*z^11 + x1^22*x2^17*x3^14*x4^2*z^11 + x1^21*x2^18*x3^14*x4^2*z^11 + 6*x1^20*x2^19*x3^14*x4^2*z^11 + x1^23*x2^15*x3^15*x4^2*z^11 + x1^21*x2^17*x3^15*x4^2*z^11 - 2*x1^20*x2^18*x3^15*x4^2*z^11 - 2*x1^19*x2^19*x3^15*x4^2*z^11 - x1^20*x2^17*x3^16*x4^2*z^11 + 2*x1^19*x2^18*x3^16*x4^2*z^11 + x1^19*x2^17*x3^17*x4^2*z^11 - x1^18*x2^18*x3^17*x4^2*z^11 + x1^28*x2^21*x3^3*x4^3*z^11 - x1^29*x2^19*x3^4*x4^3*z^11 + x1^28*x2^19*x3^5*x4^3*z^11 - x1^27*x2^20*x3^5*x4^3*z^11 + x1^25*x2^22*x3^5*x4^3*z^11 + x1^30*x2^16*x3^6*x4^3*z^11 + 2*x1^29*x2^17*x3^6*x4^3*z^11 - x1^27*x2^19*x3^6*x4^3*z^11 - x1^26*x2^20*x3^6*x4^3*z^11 - x1^25*x2^21*x3^6*x4^3*z^11 + x1^24*x2^22*x3^6*x4^3*z^11 - 2*x1^30*x2^15*x3^7*x4^3*z^11 - x1^29*x2^16*x3^7*x4^3*z^11 + x1^28*x2^17*x3^7*x4^3*z^11 + 3*x1^27*x2^18*x3^7*x4^3*z^11 - x1^26*x2^19*x3^7*x4^3*z^11 + x1^24*x2^21*x3^7*x4^3*z^11 - 2*x1^23*x2^22*x3^7*x4^3*z^11 + x1^30*x2^14*x3^8*x4^3*z^11 - x1^28*x2^16*x3^8*x4^3*z^11 - x1^27*x2^17*x3^8*x4^3*z^11 + x1^26*x2^18*x3^8*x4^3*z^11 - 3*x1^24*x2^20*x3^8*x4^3*z^11 + x1^23*x2^21*x3^8*x4^3*z^11 + x1^22*x2^22*x3^8*x4^3*z^11 - x1^30*x2^13*x3^9*x4^3*z^11 - 2*x1^29*x2^14*x3^9*x4^3*z^11 - 2*x1^28*x2^15*x3^9*x4^3*z^11 - x1^27*x2^16*x3^9*x4^3*z^11 + 2*x1^26*x2^17*x3^9*x4^3*z^11 - x1^25*x2^18*x3^9*x4^3*z^11 + x1^24*x2^19*x3^9*x4^3*z^11 - x1^23*x2^20*x3^9*x4^3*z^11 - 2*x1^22*x2^21*x3^9*x4^3*z^11 + x1^29*x2^13*x3^10*x4^3*z^11 - x1^27*x2^15*x3^10*x4^3*z^11 - x1^26*x2^16*x3^10*x4^3*z^11 + x1^25*x2^17*x3^10*x4^3*z^11 + 2*x1^24*x2^18*x3^10*x4^3*z^11 - 2*x1^23*x2^19*x3^10*x4^3*z^11 + 2*x1^22*x2^20*x3^10*x4^3*z^11 - x1^28*x2^13*x3^11*x4^3*z^11 - x1^27*x2^14*x3^11*x4^3*z^11 - 4*x1^26*x2^15*x3^11*x4^3*z^11 + 2*x1^23*x2^18*x3^11*x4^3*z^11 + x1^22*x2^19*x3^11*x4^3*z^11 - 2*x1^21*x2^20*x3^11*x4^3*z^11 + x1^27*x2^13*x3^12*x4^3*z^11 + x1^26*x2^14*x3^12*x4^3*z^11 + x1^25*x2^15*x3^12*x4^3*z^11 - 2*x1^24*x2^16*x3^12*x4^3*z^11 - x1^23*x2^17*x3^12*x4^3*z^11 - 2*x1^22*x2^18*x3^12*x4^3*z^11 + 2*x1^21*x2^19*x3^12*x4^3*z^11 + x1^20*x2^20*x3^12*x4^3*z^11 + x1^25*x2^14*x3^13*x4^3*z^11 + x1^24*x2^15*x3^13*x4^3*z^11 + 2*x1^22*x2^17*x3^13*x4^3*z^11 + x1^21*x2^18*x3^13*x4^3*z^11 - 2*x1^20*x2^19*x3^13*x4^3*z^11 + x1^23*x2^15*x3^14*x4^3*z^11 + x1^22*x2^16*x3^14*x4^3*z^11 - 3*x1^21*x2^17*x3^14*x4^3*z^11 + x1^19*x2^19*x3^14*x4^3*z^11 + x1^20*x2^17*x3^15*x4^3*z^11 - x1^18*x2^18*x3^16*x4^3*z^11 - x1^28*x2^19*x3^4*x4^4*z^11 - 2*x1^27*x2^20*x3^4*x4^4*z^11 - 2*x1^26*x2^21*x3^4*x4^4*z^11 + 2*x1^28*x2^18*x3^5*x4^4*z^11 + x1^27*x2^19*x3^5*x4^4*z^11 + x1^26*x2^20*x3^5*x4^4*z^11 + 2*x1^25*x2^21*x3^5*x4^4*z^11 + x1^24*x2^22*x3^5*x4^4*z^11 - 4*x1^27*x2^18*x3^6*x4^4*z^11 - 3*x1^25*x2^20*x3^6*x4^4*z^11 - 3*x1^24*x2^21*x3^6*x4^4*z^11 - x1^29*x2^15*x3^7*x4^4*z^11 - 2*x1^28*x2^16*x3^7*x4^4*z^11 + 2*x1^27*x2^17*x3^7*x4^4*z^11 + 2*x1^26*x2^18*x3^7*x4^4*z^11 + x1^25*x2^19*x3^7*x4^4*z^11 + 4*x1^24*x2^20*x3^7*x4^4*z^11 + x1^23*x2^21*x3^7*x4^4*z^11 + x1^22*x2^22*x3^7*x4^4*z^11 + 3*x1^29*x2^14*x3^8*x4^4*z^11 + x1^28*x2^15*x3^8*x4^4*z^11 - 5*x1^26*x2^17*x3^8*x4^4*z^11 + x1^25*x2^18*x3^8*x4^4*z^11 - 5*x1^24*x2^19*x3^8*x4^4*z^11 - 4*x1^23*x2^20*x3^8*x4^4*z^11 - 2*x1^29*x2^13*x3^9*x4^4*z^11 - 2*x1^28*x2^14*x3^9*x4^4*z^11 - x1^27*x2^15*x3^9*x4^4*z^11 + x1^26*x2^16*x3^9*x4^4*z^11 + 2*x1^25*x2^17*x3^9*x4^4*z^11 + 3*x1^24*x2^18*x3^9*x4^4*z^11 + 5*x1^23*x2^19*x3^9*x4^4*z^11 + x1^22*x2^20*x3^9*x4^4*z^11 + 3*x1^28*x2^13*x3^10*x4^4*z^11 + x1^27*x2^14*x3^10*x4^4*z^11 + 2*x1^26*x2^15*x3^10*x4^4*z^11 - 3*x1^25*x2^16*x3^10*x4^4*z^11 + x1^24*x2^17*x3^10*x4^4*z^11 - 6*x1^23*x2^18*x3^10*x4^4*z^11 - 2*x1^22*x2^19*x3^10*x4^4*z^11 - 2*x1^27*x2^13*x3^11*x4^4*z^11 - 2*x1^26*x2^14*x3^11*x4^4*z^11 - x1^25*x2^15*x3^11*x4^4*z^11 - x1^24*x2^16*x3^11*x4^4*z^11 + x1^23*x2^17*x3^11*x4^4*z^11 + 6*x1^22*x2^18*x3^11*x4^4*z^11 + 2*x1^26*x2^13*x3^12*x4^4*z^11 + x1^25*x2^14*x3^12*x4^4*z^11 - x1^24*x2^15*x3^12*x4^4*z^11 + 2*x1^23*x2^16*x3^12*x4^4*z^11 - 4*x1^22*x2^17*x3^12*x4^4*z^11 - 2*x1^21*x2^18*x3^12*x4^4*z^11 - 2*x1^25*x2^13*x3^13*x4^4*z^11 + 4*x1^21*x2^17*x3^13*x4^4*z^11 - x1^20*x2^18*x3^13*x4^4*z^11 - 2*x1^23*x2^14*x3^14*x4^4*z^11 + x1^22*x2^15*x3^14*x4^4*z^11 - 2*x1^21*x2^16*x3^14*x4^4*z^11 - x1^20*x2^17*x3^14*x4^4*z^11 + x1^19*x2^18*x3^14*x4^4*z^11 - 2*x1^21*x2^15*x3^15*x4^4*z^11 + 3*x1^20*x2^16*x3^15*x4^4*z^11 - x1^19*x2^17*x3^15*x4^4*z^11 - 2*x1^19*x2^16*x3^16*x4^4*z^11 + x1^18*x2^17*x3^16*x4^4*z^11 - x1^26*x2^19*x3^5*x4^5*z^11 - x1^24*x2^21*x3^5*x4^5*z^11 - x1^23*x2^22*x3^5*x4^5*z^11 - x1^27*x2^17*x3^6*x4^5*z^11 - x1^23*x2^21*x3^6*x4^5*z^11 - x1^22*x2^22*x3^6*x4^5*z^11 + 3*x1^26*x2^17*x3^7*x4^5*z^11 + 2*x1^24*x2^19*x3^7*x4^5*z^11 + 2*x1^23*x2^20*x3^7*x4^5*z^11 - x1^22*x2^21*x3^7*x4^5*z^11 - x1^27*x2^15*x3^8*x4^5*z^11 - x1^26*x2^16*x3^8*x4^5*z^11 - 3*x1^25*x2^17*x3^8*x4^5*z^11 - 3*x1^23*x2^19*x3^8*x4^5*z^11 + x1^22*x2^20*x3^8*x4^5*z^11 + 5*x1^25*x2^16*x3^9*x4^5*z^11 - x1^24*x2^17*x3^9*x4^5*z^11 + 3*x1^23*x2^18*x3^9*x4^5*z^11 + 2*x1^22*x2^19*x3^9*x4^5*z^11 - 2*x1^21*x2^20*x3^9*x4^5*z^11 - x1^25*x2^15*x3^10*x4^5*z^11 - 2*x1^24*x2^16*x3^10*x4^5*z^11 - 4*x1^22*x2^18*x3^10*x4^5*z^11 + x1^20*x2^20*x3^10*x4^5*z^11 + x1^26*x2^13*x3^11*x4^5*z^11 - x1^25*x2^14*x3^11*x4^5*z^11 + 3*x1^24*x2^15*x3^11*x4^5*z^11 - 2*x1^23*x2^16*x3^11*x4^5*z^11 + 7*x1^22*x2^17*x3^11*x4^5*z^11 + 2*x1^21*x2^18*x3^11*x4^5*z^11 - 2*x1^20*x2^19*x3^11*x4^5*z^11 - x1^23*x2^15*x3^12*x4^5*z^11 - 7*x1^21*x2^17*x3^12*x4^5*z^11 + 2*x1^20*x2^18*x3^12*x4^5*z^11 + x1^23*x2^14*x3^13*x4^5*z^11 - 2*x1^22*x2^15*x3^13*x4^5*z^11 + 3*x1^21*x2^16*x3^13*x4^5*z^11 + 3*x1^20*x2^17*x3^13*x4^5*z^11 - 2*x1^19*x2^18*x3^13*x4^5*z^11 - x1^22*x2^14*x3^14*x4^5*z^11 + x1^21*x2^15*x3^14*x4^5*z^11 - 3*x1^20*x2^16*x3^14*x4^5*z^11 + 2*x1^19*x2^17*x3^14*x4^5*z^11 + x1^18*x2^18*x3^14*x4^5*z^11 - x1^20*x2^15*x3^15*x4^5*z^11 - 2*x1^18*x2^17*x3^15*x4^5*z^11 + x1^26*x2^17*x3^6*x4^6*z^11 + 2*x1^25*x2^18*x3^6*x4^6*z^11 + x1^24*x2^19*x3^6*x4^6*z^11 + 2*x1^22*x2^21*x3^6*x4^6*z^11 - x1^26*x2^16*x3^7*x4^6*z^11 - x1^21*x2^21*x3^7*x4^6*z^11 - x1^25*x2^16*x3^8*x4^6*z^11 + 2*x1^23*x2^18*x3^8*x4^6*z^11 + x1^22*x2^19*x3^8*x4^6*z^11 + 3*x1^21*x2^20*x3^8*x4^6*z^11 + x1^27*x2^13*x3^9*x4^6*z^11 + 2*x1^26*x2^14*x3^9*x4^6*z^11 - x1^25*x2^15*x3^9*x4^6*z^11 + x1^24*x2^16*x3^9*x4^6*z^11 - 2*x1^23*x2^17*x3^9*x4^6*z^11 - x1^21*x2^19*x3^9*x4^6*z^11 - x1^20*x2^20*x3^9*x4^6*z^11 - x1^26*x2^13*x3^10*x4^6*z^11 - x1^24*x2^15*x3^10*x4^6*z^11 + 2*x1^21*x2^18*x3^10*x4^6*z^11 + 4*x1^20*x2^19*x3^10*x4^6*z^11 + 3*x1^25*x2^13*x3^11*x4^6*z^11 + x1^24*x2^14*x3^11*x4^6*z^11 - 4*x1^22*x2^16*x3^11*x4^6*z^11 + x1^21*x2^17*x3^11*x4^6*z^11 - 3*x1^20*x2^18*x3^11*x4^6*z^11 - x1^19*x2^19*x3^11*x4^6*z^11 - 2*x1^24*x2^13*x3^12*x4^6*z^11 - x1^23*x2^14*x3^12*x4^6*z^11 + 2*x1^22*x2^15*x3^12*x4^6*z^11 - 2*x1^20*x2^17*x3^12*x4^6*z^11 + 4*x1^19*x2^18*x3^12*x4^6*z^11 + x1^22*x2^14*x3^13*x4^6*z^11 - x1^21*x2^15*x3^13*x4^6*z^11 - 4*x1^19*x2^17*x3^13*x4^6*z^11 - x1^18*x2^18*x3^13*x4^6*z^11 + x1^19*x2^16*x3^14*x4^6*z^11 + 4*x1^18*x2^17*x3^14*x4^6*z^11 + 2*x1^19*x2^15*x3^15*x4^6*z^11 - 2*x1^18*x2^16*x3^15*x4^6*z^11 - x1^17*x2^17*x3^15*x4^6*z^11 + 2*x1^17*x2^16*x3^16*x4^6*z^11 - x1^25*x2^16*x3^7*x4^7*z^11 - x1^24*x2^17*x3^7*x4^7*z^11 - x1^23*x2^18*x3^7*x4^7*z^11 + 2*x1^25*x2^15*x3^8*x4^7*z^11 - x1^23*x2^17*x3^8*x4^7*z^11 + x1^22*x2^18*x3^8*x4^7*z^11 - x1^21*x2^19*x3^8*x4^7*z^11 + x1^20*x2^20*x3^8*x4^7*z^11 - 2*x1^24*x2^15*x3^9*x4^7*z^11 - x1^23*x2^16*x3^9*x4^7*z^11 - 2*x1^22*x2^17*x3^9*x4^7*z^11 - 2*x1^21*x2^18*x3^9*x4^7*z^11 - x1^25*x2^13*x3^10*x4^7*z^11 + x1^23*x2^15*x3^10*x4^7*z^11 + 2*x1^22*x2^16*x3^10*x4^7*z^11 + 3*x1^21*x2^17*x3^10*x4^7*z^11 + x1^20*x2^18*x3^10*x4^7*z^11 + x1^25*x2^12*x3^11*x4^7*z^11 - x1^23*x2^14*x3^11*x4^7*z^11 - x1^22*x2^15*x3^11*x4^7*z^11 - x1^21*x2^16*x3^11*x4^7*z^11 - 3*x1^20*x2^17*x3^11*x4^7*z^11 - x1^19*x2^18*x3^11*x4^7*z^11 + x1^23*x2^13*x3^12*x4^7*z^11 + x1^21*x2^15*x3^12*x4^7*z^11 + x1^20*x2^16*x3^12*x4^7*z^11 + 3*x1^19*x2^17*x3^12*x4^7*z^11 + x1^22*x2^13*x3^13*x4^7*z^11 - 2*x1^21*x2^14*x3^13*x4^7*z^11 + x1^19*x2^16*x3^13*x4^7*z^11 - 3*x1^18*x2^17*x3^13*x4^7*z^11 - x1^19*x2^15*x3^14*x4^7*z^11 + x1^18*x2^16*x3^14*x4^7*z^11 + x1^17*x2^17*x3^14*x4^7*z^11 - x1^17*x2^16*x3^15*x4^7*z^11 - x1^24*x2^15*x3^8*x4^8*z^11 - x1^22*x2^17*x3^8*x4^8*z^11 - x1^21*x2^18*x3^8*x4^8*z^11 - 2*x1^20*x2^19*x3^8*x4^8*z^11 + x1^23*x2^15*x3^9*x4^8*z^11 + x1^22*x2^16*x3^9*x4^8*z^11 + x1^21*x2^17*x3^9*x4^8*z^11 - x1^19*x2^19*x3^9*x4^8*z^11 - x1^21*x2^16*x3^10*x4^8*z^11 + x1^20*x2^17*x3^10*x4^8*z^11 - 2*x1^19*x2^18*x3^10*x4^8*z^11 - x1^22*x2^14*x3^11*x4^8*z^11 - x1^21*x2^15*x3^11*x4^8*z^11 + 2*x1^20*x2^16*x3^11*x4^8*z^11 + x1^21*x2^14*x3^12*x4^8*z^11 - 2*x1^20*x2^15*x3^12*x4^8*z^11 + x1^21*x2^13*x3^13*x4^8*z^11 + x1^20*x2^14*x3^13*x4^8*z^11 + 2*x1^19*x2^15*x3^13*x4^8*z^11 - x1^18*x2^16*x3^13*x4^8*z^11 + x1^17*x2^17*x3^13*x4^8*z^11 - x1^18*x2^15*x3^14*x4^8*z^11 + x1^22*x2^15*x3^9*x4^9*z^11 + x1^21*x2^16*x3^9*x4^9*z^11 + x1^20*x2^17*x3^9*x4^9*z^11 + 2*x1^19*x2^18*x3^9*x4^9*z^11 - 2*x1^20*x2^16*x3^10*x4^9*z^11 + x1^22*x2^13*x3^11*x4^9*z^11 + x1^21*x2^14*x3^11*x4^9*z^11 + 2*x1^20*x2^15*x3^11*x4^9*z^11 + x1^19*x2^16*x3^11*x4^9*z^11 + x1^18*x2^17*x3^11*x4^9*z^11 + x1^20*x2^14*x3^12*x4^9*z^11 - 2*x1^19*x2^15*x3^12*x4^9*z^11 - x1^20*x2^13*x3^13*x4^9*z^11 + 2*x1^18*x2^15*x3^13*x4^9*z^11 - x1^18*x2^14*x3^14*x4^9*z^11 - x1^21*x2^14*x3^10*x4^10*z^11 - x1^20*x2^15*x3^10*x4^10*z^11 - x1^20*x2^14*x3^11*x4^10*z^11 + x1^17*x2^17*x3^11*x4^10*z^11 - x1^18*x2^15*x3^12*x4^10*z^11 + x1^16*x2^16*x3^13*x4^10*z^11 - x1^17*x2^14*x3^14*x4^10*z^11 - x1^18*x2^15*x3^11*x4^11*z^11 - x1^17*x2^15*x3^12*x4^11*z^11 - x1^16*x2^16*x3^12*x4^11*z^11 - x1^30*x2^17*x3^3*z^10 + x1^28*x2^19*x3^3*z^10 + x1^26*x2^21*x3^3*z^10 + x1^29*x2^17*x3^4*z^10 + x1^28*x2^18*x3^4*z^10 - x1^27*x2^19*x3^4*z^10 - 2*x1^26*x2^20*x3^4*z^10 - x1^25*x2^21*x3^4*z^10 - x1^29*x2^16*x3^5*z^10 - x1^28*x2^17*x3^5*z^10 + x1^27*x2^18*x3^5*z^10 + x1^26*x2^19*x3^5*z^10 + x1^24*x2^21*x3^5*z^10 + 2*x1^28*x2^16*x3^6*z^10 + 2*x1^26*x2^18*x3^6*z^10 - x1^25*x2^19*x3^6*z^10 - 2*x1^23*x2^21*x3^6*z^10 - 2*x1^28*x2^15*x3^7*z^10 - x1^27*x2^16*x3^7*z^10 - x1^26*x2^17*x3^7*z^10 + x1^24*x2^19*x3^7*z^10 + x1^23*x2^20*x3^7*z^10 + 2*x1^22*x2^21*x3^7*z^10 + x1^28*x2^14*x3^8*z^10 + 2*x1^27*x2^15*x3^8*z^10 + x1^25*x2^17*x3^8*z^10 - x1^24*x2^18*x3^8*z^10 - 2*x1^22*x2^20*x3^8*z^10 - x1^21*x2^21*x3^8*z^10 - 2*x1^27*x2^14*x3^9*z^10 - x1^26*x2^15*x3^9*z^10 - x1^25*x2^16*x3^9*z^10 + x1^23*x2^18*x3^9*z^10 + x1^22*x2^19*x3^9*z^10 + 2*x1^21*x2^20*x3^9*z^10 + 2*x1^26*x2^14*x3^10*z^10 + x1^24*x2^16*x3^10*z^10 - 2*x1^23*x2^17*x3^10*z^10 - 2*x1^21*x2^19*x3^10*z^10 - x1^25*x2^14*x3^11*z^10 + 2*x1^20*x2^19*x3^11*z^10 + x1^23*x2^15*x3^12*z^10 - x1^22*x2^16*x3^12*z^10 - 2*x1^20*x2^18*x3^12*z^10 - x1^19*x2^19*x3^12*z^10 + x1^20*x2^17*x3^13*z^10 + 2*x1^19*x2^18*x3^13*z^10 - x1^21*x2^15*x3^14*z^10 + x1^20*x2^16*x3^14*z^10 - 2*x1^19*x2^17*x3^14*z^10 - x1^18*x2^18*x3^14*z^10 + x1^20*x2^15*x3^15*z^10 - x1^27*x2^20*x3^2*x4*z^10 - x1^26*x2^21*x3^2*x4*z^10 - x1^27*x2^19*x3^3*x4*z^10 + x1^26*x2^20*x3^3*x4*z^10 + x1^24*x2^22*x3^3*x4*z^10 + x1^29*x2^16*x3^4*x4*z^10 - x1^26*x2^19*x3^4*x4*z^10 - 2*x1^25*x2^20*x3^4*x4*z^10 - 2*x1^24*x2^21*x3^4*x4*z^10 - 2*x1^23*x2^22*x3^4*x4*z^10 - 3*x1^28*x2^16*x3^5*x4*z^10 - x1^27*x2^17*x3^5*x4*z^10 + x1^26*x2^18*x3^5*x4*z^10 + 4*x1^25*x2^19*x3^5*x4*z^10 + 4*x1^23*x2^21*x3^5*x4*z^10 + x1^22*x2^22*x3^5*x4*z^10 + 2*x1^28*x2^15*x3^6*x4*z^10 + 2*x1^27*x2^16*x3^6*x4*z^10 - 2*x1^25*x2^18*x3^6*x4*z^10 - 2*x1^24*x2^19*x3^6*x4*z^10 - 2*x1^23*x2^20*x3^6*x4*z^10 - 5*x1^22*x2^21*x3^6*x4*z^10 - 5*x1^27*x2^15*x3^7*x4*z^10 - x1^26*x2^16*x3^7*x4*z^10 - x1^25*x2^17*x3^7*x4*z^10 + 4*x1^24*x2^18*x3^7*x4*z^10 + 6*x1^22*x2^20*x3^7*x4*z^10 + 2*x1^21*x2^21*x3^7*x4*z^10 + 4*x1^27*x2^14*x3^8*x4*z^10 + x1^26*x2^15*x3^8*x4*z^10 + x1^25*x2^16*x3^8*x4*z^10 - 2*x1^23*x2^18*x3^8*x4*z^10 - 2*x1^22*x2^19*x3^8*x4*z^10 - 6*x1^21*x2^20*x3^8*x4*z^10 - 2*x1^27*x2^13*x3^9*x4*z^10 - 5*x1^26*x2^14*x3^9*x4*z^10 - 4*x1^24*x2^16*x3^9*x4*z^10 + 4*x1^23*x2^17*x3^9*x4*z^10 + 6*x1^21*x2^19*x3^9*x4*z^10 + 2*x1^20*x2^20*x3^9*x4*z^10 + 3*x1^26*x2^13*x3^10*x4*z^10 + 2*x1^25*x2^14*x3^10*x4*z^10 + x1^24*x2^15*x3^10*x4*z^10 - 2*x1^22*x2^17*x3^10*x4*z^10 - 2*x1^21*x2^18*x3^10*x4*z^10 - 6*x1^20*x2^19*x3^10*x4*z^10 - 3*x1^25*x2^13*x3^11*x4*z^10 - x1^24*x2^14*x3^11*x4*z^10 - 2*x1^23*x2^15*x3^11*x4*z^10 + 5*x1^22*x2^16*x3^11*x4*z^10 + 6*x1^20*x2^18*x3^11*x4*z^10 + 2*x1^19*x2^19*x3^11*x4*z^10 + 2*x1^24*x2^13*x3^12*x4*z^10 + x1^23*x2^14*x3^12*x4*z^10 - 2*x1^21*x2^16*x3^12*x4*z^10 - x1^20*x2^17*x3^12*x4*z^10 - 6*x1^19*x2^18*x3^12*x4*z^10 + 3*x1^21*x2^15*x3^13*x4*z^10 + 5*x1^19*x2^17*x3^13*x4*z^10 + 2*x1^18*x2^18*x3^13*x4*z^10 - x1^20*x2^15*x3^14*x4*z^10 - x1^19*x2^16*x3^14*x4*z^10 - 5*x1^18*x2^17*x3^14*x4*z^10 - x1^19*x2^15*x3^15*x4*z^10 + 2*x1^18*x2^16*x3^15*x4*z^10 + x1^17*x2^17*x3^15*x4*z^10 - x1^17*x2^16*x3^16*x4*z^10 + x1^26*x2^20*x3^2*x4^2*z^10 + x1^26*x2^19*x3^3*x4^2*z^10 + x1^25*x2^20*x3^3*x4^2*z^10 - 2*x1^25*x2^19*x3^4*x4^2*z^10 + x1^24*x2^20*x3^4*x4^2*z^10 - x1^23*x2^21*x3^4*x4^2*z^10 + 2*x1^25*x2^18*x3^5*x4^2*z^10 + 2*x1^24*x2^19*x3^5*x4^2*z^10 + x1^23*x2^20*x3^5*x4^2*z^10 + 3*x1^22*x2^21*x3^5*x4^2*z^10 + 2*x1^27*x2^15*x3^6*x4^2*z^10 - x1^25*x2^17*x3^6*x4^2*z^10 - 3*x1^24*x2^18*x3^6*x4^2*z^10 - 4*x1^22*x2^20*x3^6*x4^2*z^10 - x1^21*x2^21*x3^6*x4^2*z^10 - x1^27*x2^14*x3^7*x4^2*z^10 - x1^26*x2^15*x3^7*x4^2*z^10 - x1^25*x2^16*x3^7*x4^2*z^10 + 2*x1^24*x2^17*x3^7*x4^2*z^10 + 3*x1^23*x2^18*x3^7*x4^2*z^10 + x1^22*x2^19*x3^7*x4^2*z^10 + 5*x1^21*x2^20*x3^7*x4^2*z^10 + 4*x1^26*x2^14*x3^8*x4^2*z^10 - 4*x1^23*x2^17*x3^8*x4^2*z^10 - 6*x1^21*x2^19*x3^8*x4^2*z^10 - 2*x1^20*x2^20*x3^8*x4^2*z^10 - x1^26*x2^13*x3^9*x4^2*z^10 - 2*x1^25*x2^14*x3^9*x4^2*z^10 + x1^23*x2^16*x3^9*x4^2*z^10 + 2*x1^22*x2^17*x3^9*x4^2*z^10 + 2*x1^21*x2^18*x3^9*x4^2*z^10 + 6*x1^20*x2^19*x3^9*x4^2*z^10 + 2*x1^25*x2^13*x3^10*x4^2*z^10 + 2*x1^24*x2^14*x3^10*x4^2*z^10 + x1^23*x2^15*x3^10*x4^2*z^10 - 3*x1^22*x2^16*x3^10*x4^2*z^10 - 6*x1^20*x2^18*x3^10*x4^2*z^10 - 2*x1^19*x2^19*x3^10*x4^2*z^10 - x1^24*x2^13*x3^11*x4^2*z^10 - 2*x1^23*x2^14*x3^11*x4^2*z^10 - x1^22*x2^15*x3^11*x4^2*z^10 + 2*x1^20*x2^17*x3^11*x4^2*z^10 + 6*x1^19*x2^18*x3^11*x4^2*z^10 + x1^23*x2^13*x3^12*x4^2*z^10 + 2*x1^22*x2^14*x3^12*x4^2*z^10 - x1^21*x2^15*x3^12*x4^2*z^10 + 2*x1^20*x2^16*x3^12*x4^2*z^10 - 5*x1^19*x2^17*x3^12*x4^2*z^10 - 2*x1^18*x2^18*x3^12*x4^2*z^10 - x1^22*x2^13*x3^13*x4^2*z^10 - x1^21*x2^14*x3^13*x4^2*z^10 - x1^20*x2^15*x3^13*x4^2*z^10 - x1^19*x2^16*x3^13*x4^2*z^10 + 5*x1^18*x2^17*x3^13*x4^2*z^10 + 2*x1^19*x2^15*x3^14*x4^2*z^10 - x1^18*x2^16*x3^14*x4^2*z^10 - 2*x1^17*x2^17*x3^14*x4^2*z^10 - x1^18*x2^15*x3^15*x4^2*z^10 + x1^17*x2^16*x3^15*x4^2*z^10 - x1^27*x2^17*x3^3*x4^3*z^10 - x1^25*x2^19*x3^3*x4^3*z^10 - x1^24*x2^20*x3^3*x4^3*z^10 + x1^26*x2^17*x3^4*x4^3*z^10 + x1^24*x2^19*x3^4*x4^3*z^10 - x1^26*x2^16*x3^5*x4^3*z^10 - x1^25*x2^17*x3^5*x4^3*z^10 - x1^28*x2^13*x3^6*x4^3*z^10 - x1^26*x2^15*x3^6*x4^3*z^10 + x1^25*x2^16*x3^6*x4^3*z^10 - 2*x1^24*x2^17*x3^6*x4^3*z^10 + 2*x1^23*x2^18*x3^6*x4^3*z^10 + 3*x1^22*x2^19*x3^6*x4^3*z^10 - x1^21*x2^20*x3^6*x4^3*z^10 + x1^27*x2^13*x3^7*x4^3*z^10 + x1^26*x2^14*x3^7*x4^3*z^10 + x1^25*x2^15*x3^7*x4^3*z^10 - 2*x1^24*x2^16*x3^7*x4^3*z^10 - x1^23*x2^17*x3^7*x4^3*z^10 - x1^22*x2^18*x3^7*x4^3*z^10 + x1^21*x2^19*x3^7*x4^3*z^10 + x1^20*x2^20*x3^7*x4^3*z^10 - 2*x1^27*x2^12*x3^8*x4^3*z^10 - x1^26*x2^13*x3^8*x4^3*z^10 + x1^25*x2^14*x3^8*x4^3*z^10 + 3*x1^24*x2^15*x3^8*x4^3*z^10 - x1^23*x2^16*x3^8*x4^3*z^10 + x1^21*x2^18*x3^8*x4^3*z^10 - x1^20*x2^19*x3^8*x4^3*z^10 + x1^27*x2^11*x3^9*x4^3*z^10 + x1^26*x2^12*x3^9*x4^3*z^10 + 2*x1^25*x2^13*x3^9*x4^3*z^10 + x1^24*x2^14*x3^9*x4^3*z^10 + 2*x1^23*x2^15*x3^9*x4^3*z^10 - 2*x1^21*x2^17*x3^9*x4^3*z^10 + 2*x1^20*x2^18*x3^9*x4^3*z^10 + x1^19*x2^19*x3^9*x4^3*z^10 - x1^26*x2^11*x3^10*x4^3*z^10 - x1^24*x2^13*x3^10*x4^3*z^10 + 2*x1^23*x2^14*x3^10*x4^3*z^10 - x1^22*x2^15*x3^10*x4^3*z^10 + 2*x1^21*x2^16*x3^10*x4^3*z^10 - x1^20*x2^17*x3^10*x4^3*z^10 - 2*x1^19*x2^18*x3^10*x4^3*z^10 + x1^23*x2^13*x3^11*x4^3*z^10 + 2*x1^22*x2^14*x3^11*x4^3*z^10 + 2*x1^21*x2^15*x3^11*x4^3*z^10 - 3*x1^20*x2^16*x3^11*x4^3*z^10 + 2*x1^19*x2^17*x3^11*x4^3*z^10 - x1^23*x2^12*x3^12*x4^3*z^10 - x1^22*x2^13*x3^12*x4^3*z^10 - 2*x1^21*x2^14*x3^12*x4^3*z^10 + x1^20*x2^15*x3^12*x4^3*z^10 + 2*x1^19*x2^16*x3^12*x4^3*z^10 - 2*x1^18*x2^17*x3^12*x4^3*z^10 - x1^21*x2^13*x3^13*x4^3*z^10 - x1^19*x2^15*x3^13*x4^3*z^10 + x1^17*x2^17*x3^13*x4^3*z^10 - x1^19*x2^14*x3^14*x4^3*z^10 - x1^18*x2^15*x3^14*x4^3*z^10 + x1^26*x2^16*x3^4*x4^4*z^10 + x1^24*x2^18*x3^4*x4^4*z^10 + 2*x1^23*x2^19*x3^4*x4^4*z^10 + x1^22*x2^20*x3^4*x4^4*z^10 - 3*x1^25*x2^16*x3^5*x4^4*z^10 - x1^24*x2^17*x3^5*x4^4*z^10 - 3*x1^23*x2^18*x3^5*x4^4*z^10 - x1^22*x2^19*x3^5*x4^4*z^10 - x1^21*x2^20*x3^5*x4^4*z^10 + 2*x1^25*x2^15*x3^6*x4^4*z^10 + 2*x1^24*x2^16*x3^6*x4^4*z^10 + x1^23*x2^17*x3^6*x4^4*z^10 + 3*x1^22*x2^18*x3^6*x4^4*z^10 + x1^21*x2^19*x3^6*x4^4*z^10 + x1^20*x2^20*x3^6*x4^4*z^10 + x1^27*x2^12*x3^7*x4^4*z^10 + x1^25*x2^14*x3^7*x4^4*z^10 - 4*x1^24*x2^15*x3^7*x4^4*z^10 - 4*x1^22*x2^17*x3^7*x4^4*z^10 - 3*x1^21*x2^18*x3^7*x4^4*z^10 - 2*x1^26*x2^12*x3^8*x4^4*z^10 - 2*x1^25*x2^13*x3^8*x4^4*z^10 + 3*x1^23*x2^15*x3^8*x4^4*z^10 + x1^22*x2^16*x3^8*x4^4*z^10 + 4*x1^21*x2^17*x3^8*x4^4*z^10 + 2*x1^20*x2^18*x3^8*x4^4*z^10 + 2*x1^26*x2^11*x3^9*x4^4*z^10 + x1^25*x2^12*x3^9*x4^4*z^10 - 4*x1^23*x2^14*x3^9*x4^4*z^10 - 7*x1^21*x2^16*x3^9*x4^4*z^10 - 3*x1^20*x2^17*x3^9*x4^4*z^10 - x1^25*x2^11*x3^10*x4^4*z^10 - x1^24*x2^12*x3^10*x4^4*z^10 + x1^23*x2^13*x3^10*x4^4*z^10 + x1^22*x2^14*x3^10*x4^4*z^10 + x1^21*x2^15*x3^10*x4^4*z^10 + 6*x1^20*x2^16*x3^10*x4^4*z^10 + x1^23*x2^12*x3^11*x4^4*z^10 - 2*x1^22*x2^13*x3^11*x4^4*z^10 + 3*x1^21*x2^14*x3^11*x4^4*z^10 - 5*x1^20*x2^15*x3^11*x4^4*z^10 - x1^19*x2^16*x3^11*x4^4*z^10 + x1^21*x2^13*x3^12*x4^4*z^10 + x1^20*x2^14*x3^12*x4^4*z^10 + 5*x1^19*x2^15*x3^12*x4^4*z^10 - x1^18*x2^16*x3^12*x4^4*z^10 + 2*x1^20*x2^13*x3^13*x4^4*z^10 - 2*x1^19*x2^14*x3^13*x4^4*z^10 - x1^18*x2^15*x3^13*x4^4*z^10 + x1^17*x2^16*x3^13*x4^4*z^10 + 2*x1^18*x2^14*x3^14*x4^4*z^10 - x1^16*x2^16*x3^14*x4^4*z^10 + 2*x1^23*x2^17*x3^5*x4^5*z^10 + x1^20*x2^20*x3^5*x4^5*z^10 + 2*x1^24*x2^15*x3^6*x4^5*z^10 + x1^22*x2^17*x3^6*x4^5*z^10 - x1^20*x2^19*x3^6*x4^5*z^10 - x1^24*x2^14*x3^7*x4^5*z^10 - x1^23*x2^15*x3^7*x4^5*z^10 - 2*x1^21*x2^17*x3^7*x4^5*z^10 - x1^20*x2^18*x3^7*x4^5*z^10 + x1^25*x2^12*x3^8*x4^5*z^10 + x1^24*x2^13*x3^8*x4^5*z^10 + 4*x1^23*x2^14*x3^8*x4^5*z^10 + 3*x1^21*x2^16*x3^8*x4^5*z^10 + x1^20*x2^17*x3^8*x4^5*z^10 - x1^19*x2^18*x3^8*x4^5*z^10 - 2*x1^24*x2^12*x3^9*x4^5*z^10 + x1^23*x2^13*x3^9*x4^5*z^10 - 2*x1^22*x2^14*x3^9*x4^5*z^10 - 5*x1^20*x2^16*x3^9*x4^5*z^10 + x1^18*x2^18*x3^9*x4^5*z^10 + x1^24*x2^11*x3^10*x4^5*z^10 + x1^23*x2^12*x3^10*x4^5*z^10 + x1^22*x2^13*x3^10*x4^5*z^10 - x1^21*x2^14*x3^10*x4^5*z^10 + 5*x1^20*x2^15*x3^10*x4^5*z^10 + x1^19*x2^16*x3^10*x4^5*z^10 - 2*x1^18*x2^17*x3^10*x4^5*z^10 - x1^22*x2^12*x3^11*x4^5*z^10 - x1^21*x2^13*x3^11*x4^5*z^10 + 2*x1^20*x2^14*x3^11*x4^5*z^10 - 6*x1^19*x2^15*x3^11*x4^5*z^10 - x1^18*x2^16*x3^11*x4^5*z^10 + x1^17*x2^17*x3^11*x4^5*z^10 + x1^21*x2^12*x3^12*x4^5*z^10 - 2*x1^20*x2^13*x3^12*x4^5*z^10 + x1^19*x2^14*x3^12*x4^5*z^10 + 2*x1^18*x2^15*x3^12*x4^5*z^10 - x1^17*x2^16*x3^12*x4^5*z^10 + x1^19*x2^13*x3^13*x4^5*z^10 - x1^18*x2^14*x3^13*x4^5*z^10 + x1^17*x2^15*x3^13*x4^5*z^10 + x1^17*x2^14*x3^14*x4^5*z^10 - x1^16*x2^15*x3^14*x4^5*z^10 + x1^15*x2^15*x3^15*x4^5*z^10 - x1^24*x2^14*x3^6*x4^6*z^10 - 2*x1^22*x2^16*x3^6*x4^6*z^10 - 2*x1^21*x2^17*x3^6*x4^6*z^10 - x1^19*x2^19*x3^6*x4^6*z^10 + 2*x1^22*x2^15*x3^7*x4^6*z^10 + x1^21*x2^16*x3^7*x4^6*z^10 + x1^19*x2^18*x3^7*x4^6*z^10 - x1^23*x2^13*x3^8*x4^6*z^10 - x1^21*x2^15*x3^8*x4^6*z^10 + x1^20*x2^16*x3^8*x4^6*z^10 - x1^19*x2^17*x3^8*x4^6*z^10 - 2*x1^18*x2^18*x3^8*x4^6*z^10 - x1^23*x2^12*x3^9*x4^6*z^10 + x1^21*x2^14*x3^9*x4^6*z^10 + 2*x1^20*x2^15*x3^9*x4^6*z^10 + x1^19*x2^16*x3^9*x4^6*z^10 + 2*x1^18*x2^17*x3^9*x4^6*z^10 + x1^23*x2^11*x3^10*x4^6*z^10 + x1^22*x2^12*x3^10*x4^6*z^10 - x1^21*x2^13*x3^10*x4^6*z^10 - 3*x1^20*x2^14*x3^10*x4^6*z^10 + 3*x1^19*x2^15*x3^10*x4^6*z^10 - 2*x1^18*x2^16*x3^10*x4^6*z^10 - 2*x1^17*x2^17*x3^10*x4^6*z^10 + 2*x1^20*x2^13*x3^11*x4^6*z^10 + 3*x1^18*x2^15*x3^11*x4^6*z^10 + 4*x1^17*x2^16*x3^11*x4^6*z^10 + 2*x1^20*x2^12*x3^12*x4^6*z^10 - 2*x1^19*x2^13*x3^12*x4^6*z^10 + x1^18*x2^14*x3^12*x4^6*z^10 - 3*x1^17*x2^15*x3^12*x4^6*z^10 - x1^16*x2^16*x3^12*x4^6*z^10 + x1^18*x2^13*x3^13*x4^6*z^10 - x1^17*x2^14*x3^13*x4^6*z^10 + 4*x1^16*x2^15*x3^13*x4^6*z^10 - 2*x1^15*x2^15*x3^14*x4^6*z^10 + x1^23*x2^13*x3^7*x4^7*z^10 + 2*x1^20*x2^16*x3^7*x4^7*z^10 - x1^22*x2^13*x3^8*x4^7*z^10 - 2*x1^21*x2^14*x3^8*x4^7*z^10 - 2*x1^20*x2^15*x3^8*x4^7*z^10 - x1^19*x2^16*x3^8*x4^7*z^10 + x1^18*x2^17*x3^8*x4^7*z^10 + x1^21*x2^13*x3^9*x4^7*z^10 + 2*x1^20*x2^14*x3^9*x4^7*z^10 + x1^19*x2^15*x3^9*x4^7*z^10 + x1^18*x2^16*x3^9*x4^7*z^10 + x1^17*x2^17*x3^9*x4^7*z^10 + x1^21*x2^12*x3^10*x4^7*z^10 - x1^20*x2^13*x3^10*x4^7*z^10 - x1^19*x2^14*x3^10*x4^7*z^10 - 2*x1^18*x2^15*x3^10*x4^7*z^10 - 2*x1^17*x2^16*x3^10*x4^7*z^10 - x1^21*x2^11*x3^11*x4^7*z^10 + x1^20*x2^12*x3^11*x4^7*z^10 + 3*x1^19*x2^13*x3^11*x4^7*z^10 - x1^18*x2^14*x3^11*x4^7*z^10 + 2*x1^17*x2^15*x3^11*x4^7*z^10 + x1^16*x2^16*x3^11*x4^7*z^10 - x1^19*x2^12*x3^12*x4^7*z^10 + x1^17*x2^14*x3^12*x4^7*z^10 - 2*x1^16*x2^15*x3^12*x4^7*z^10 + x1^15*x2^15*x3^13*x4^7*z^10 + x1^21*x2^13*x3^8*x4^8*z^10 + 2*x1^20*x2^14*x3^8*x4^8*z^10 + x1^19*x2^15*x3^8*x4^8*z^10 + x1^17*x2^17*x3^8*x4^8*z^10 + x1^20*x2^13*x3^9*x4^8*z^10 - x1^18*x2^15*x3^9*x4^8*z^10 - x1^17*x2^16*x3^9*x4^8*z^10 - x1^19*x2^13*x3^10*x4^8*z^10 + 2*x1^18*x2^14*x3^10*x4^8*z^10 - x1^16*x2^15*x3^11*x4^8*z^10 - x1^18*x2^12*x3^12*x4^8*z^10 + 2*x1^17*x2^13*x3^12*x4^8*z^10 - x1^16*x2^14*x3^12*x4^8*z^10 - x1^16*x2^13*x3^13*x4^8*z^10 - x1^19*x2^13*x3^9*x4^9*z^10 - x1^18*x2^14*x3^9*x4^9*z^10 - x1^16*x2^16*x3^9*x4^9*z^10 + x1^17*x2^14*x3^10*x4^9*z^10 + x1^16*x2^15*x3^10*x4^9*z^10 - 2*x1^17*x2^13*x3^11*x4^9*z^10 - x1^16*x2^14*x3^11*x4^9*z^10 + x1^16*x2^13*x3^12*x4^9*z^10 + x1^17*x2^13*x3^10*x4^10*z^10 + x1^25*x2^18*x3^2*z^9 + x1^24*x2^19*x3^2*z^9 + 2*x1^27*x2^15*x3^3*z^9 - x1^25*x2^17*x3^3*z^9 - x1^24*x2^18*x3^3*z^9 - 2*x1^23*x2^19*x3^3*z^9 - x1^22*x2^20*x3^3*z^9 - x1^27*x2^14*x3^4*z^9 + x1^23*x2^18*x3^4*z^9 + 2*x1^22*x2^19*x3^4*z^9 + x1^21*x2^20*x3^4*z^9 + x1^26*x2^14*x3^5*z^9 + x1^25*x2^15*x3^5*z^9 - 2*x1^23*x2^17*x3^5*z^9 - 2*x1^21*x2^19*x3^5*z^9 - 2*x1^26*x2^13*x3^6*z^9 + 2*x1^20*x2^19*x3^6*z^9 + 2*x1^25*x2^13*x3^7*z^9 + 2*x1^23*x2^15*x3^7*z^9 - x1^22*x2^16*x3^7*z^9 - 2*x1^20*x2^18*x3^7*z^9 - x1^19*x2^19*x3^7*z^9 - 2*x1^25*x2^12*x3^8*z^9 - x1^24*x2^13*x3^8*z^9 - x1^23*x2^14*x3^8*z^9 + x1^21*x2^16*x3^8*z^9 + x1^20*x2^17*x3^8*z^9 + 2*x1^19*x2^18*x3^8*z^9 + 2*x1^24*x2^12*x3^9*z^9 + x1^22*x2^14*x3^9*z^9 - x1^21*x2^15*x3^9*z^9 - 2*x1^19*x2^17*x3^9*z^9 - x1^18*x2^18*x3^9*z^9 + x1^20*x2^15*x3^10*z^9 + x1^19*x2^16*x3^10*z^9 + 2*x1^18*x2^17*x3^10*z^9 - x1^20*x2^14*x3^11*z^9 - 2*x1^18*x2^16*x3^11*z^9 + x1^20*x2^13*x3^12*z^9 + 2*x1^17*x2^16*x3^12*z^9 - x1^19*x2^13*x3^13*z^9 - x1^17*x2^15*x3^13*z^9 - x1^16*x2^16*x3^13*z^9 + x1^16*x2^15*x3^14*z^9 - x1^25*x2^18*x3*x4*z^9 + x1^24*x2^18*x3^2*x4*z^9 + x1^23*x2^19*x3^2*x4*z^9 + x1^22*x2^20*x3^2*x4*z^9 - 2*x1^24*x2^17*x3^3*x4*z^9 - x1^23*x2^18*x3^3*x4*z^9 + x1^22*x2^19*x3^3*x4*z^9 - x1^21*x2^20*x3^3*x4*z^9 - 2*x1^26*x2^14*x3^4*x4*z^9 + x1^24*x2^16*x3^4*x4*z^9 + 3*x1^23*x2^17*x3^4*x4*z^9 + 4*x1^21*x2^19*x3^4*x4*z^9 + x1^20*x2^20*x3^4*x4*z^9 + x1^26*x2^13*x3^5*x4*z^9 + x1^25*x2^14*x3^5*x4*z^9 + x1^24*x2^15*x3^5*x4*z^9 - 2*x1^23*x2^16*x3^5*x4*z^9 - 3*x1^22*x2^17*x3^5*x4*z^9 - x1^21*x2^18*x3^5*x4*z^9 - 5*x1^20*x2^19*x3^5*x4*z^9 - 4*x1^25*x2^13*x3^6*x4*z^9 + 4*x1^22*x2^16*x3^6*x4*z^9 + 6*x1^20*x2^18*x3^6*x4*z^9 + 2*x1^19*x2^19*x3^6*x4*z^9 + 2*x1^25*x2^12*x3^7*x4*z^9 + x1^24*x2^13*x3^7*x4*z^9 + x1^23*x2^14*x3^7*x4*z^9 - x1^22*x2^15*x3^7*x4*z^9 - 2*x1^21*x2^16*x3^7*x4*z^9 - 2*x1^20*x2^17*x3^7*x4*z^9 - 6*x1^19*x2^18*x3^7*x4*z^9 - 5*x1^24*x2^12*x3^8*x4*z^9 - x1^22*x2^14*x3^8*x4*z^9 + 4*x1^21*x2^15*x3^8*x4*z^9 + 6*x1^19*x2^17*x3^8*x4*z^9 + 2*x1^18*x2^18*x3^8*x4*z^9 + 2*x1^24*x2^11*x3^9*x4*z^9 + 3*x1^23*x2^12*x3^9*x4*z^9 + 2*x1^22*x2^13*x3^9*x4*z^9 - x1^21*x2^14*x3^9*x4*z^9 - 2*x1^20*x2^15*x3^9*x4*z^9 - 2*x1^19*x2^16*x3^9*x4*z^9 - 6*x1^18*x2^17*x3^9*x4*z^9 - 2*x1^23*x2^11*x3^10*x4*z^9 - 2*x1^22*x2^12*x3^10*x4*z^9 - 2*x1^21*x2^13*x3^10*x4*z^9 + 4*x1^20*x2^14*x3^10*x4*z^9 + 6*x1^18*x2^16*x3^10*x4*z^9 + 2*x1^17*x2^17*x3^10*x4*z^9 + 2*x1^21*x2^12*x3^11*x4*z^9 - x1^19*x2^14*x3^11*x4*z^9 - 2*x1^18*x2^15*x3^11*x4*z^9 - 6*x1^17*x2^16*x3^11*x4*z^9 - 2*x1^20*x2^12*x3^12*x4*z^9 + x1^19*x2^13*x3^12*x4*z^9 - x1^18*x2^14*x3^12*x4*z^9 + 4*x1^17*x2^15*x3^12*x4*z^9 + 2*x1^16*x2^16*x3^12*x4*z^9 - x1^18*x2^13*x3^13*x4*z^9 - 4*x1^16*x2^15*x3^13*x4*z^9 + 2*x1^15*x2^15*x3^14*x4*z^9 - x1^23*x2^18*x3^2*x4^2*z^9 - x1^22*x2^19*x3^2*x4^2*z^9 - x1^23*x2^17*x3^3*x4^2*z^9 - x1^22*x2^18*x3^3*x4^2*z^9 - x1^21*x2^19*x3^3*x4^2*z^9 + 2*x1^23*x2^16*x3^4*x4^2*z^9 + x1^22*x2^17*x3^4*x4^2*z^9 - x1^21*x2^18*x3^4*x4^2*z^9 + 2*x1^20*x2^19*x3^4*x4^2*z^9 - 4*x1^22*x2^16*x3^5*x4^2*z^9 - 2*x1^20*x2^18*x3^5*x4^2*z^9 - 2*x1^19*x2^19*x3^5*x4^2*z^9 - x1^24*x2^13*x3^6*x4^2*z^9 - x1^23*x2^14*x3^6*x4^2*z^9 + 3*x1^22*x2^15*x3^6*x4^2*z^9 + 2*x1^21*x2^16*x3^6*x4^2*z^9 + 5*x1^19*x2^18*x3^6*x4^2*z^9 + 2*x1^24*x2^12*x3^7*x4^2*z^9 - 3*x1^21*x2^15*x3^7*x4^2*z^9 - x1^20*x2^16*x3^7*x4^2*z^9 - 5*x1^19*x2^17*x3^7*x4^2*z^9 - 2*x1^18*x2^18*x3^7*x4^2*z^9 - x1^23*x2^12*x3^8*x4^2*z^9 - x1^22*x2^13*x3^8*x4^2*z^9 + 2*x1^21*x2^14*x3^8*x4^2*z^9 + 2*x1^20*x2^15*x3^8*x4^2*z^9 + 2*x1^19*x2^16*x3^8*x4^2*z^9 + 6*x1^18*x2^17*x3^8*x4^2*z^9 + x1^23*x2^11*x3^9*x4^2*z^9 + x1^22*x2^12*x3^9*x4^2*z^9 - 3*x1^20*x2^14*x3^9*x4^2*z^9 - 6*x1^18*x2^16*x3^9*x4^2*z^9 - 2*x1^17*x2^17*x3^9*x4^2*z^9 - x1^21*x2^12*x3^10*x4^2*z^9 + x1^20*x2^13*x3^10*x4^2*z^9 + x1^18*x2^15*x3^10*x4^2*z^9 + 6*x1^17*x2^16*x3^10*x4^2*z^9 - x1^19*x2^13*x3^11*x4^2*z^9 + 2*x1^18*x2^14*x3^11*x4^2*z^9 - 3*x1^17*x2^15*x3^11*x4^2*z^9 - 2*x1^16*x2^16*x3^11*x4^2*z^9 - x1^18*x2^13*x3^12*x4^2*z^9 - x1^17*x2^14*x3^12*x4^2*z^9 + 3*x1^16*x2^15*x3^12*x4^2*z^9 + x1^17*x2^13*x3^13*x4^2*z^9 - x1^15*x2^15*x3^13*x4^2*z^9 + 2*x1^24*x2^15*x3^3*x4^3*z^9 + x1^23*x2^16*x3^3*x4^3*z^9 + x1^22*x2^17*x3^3*x4^3*z^9 + x1^21*x2^18*x3^3*x4^3*z^9 - x1^24*x2^14*x3^4*x4^3*z^9 + x1^22*x2^16*x3^4*x4^3*z^9 - x1^21*x2^17*x3^4*x4^3*z^9 - x1^20*x2^18*x3^4*x4^3*z^9 + x1^23*x2^14*x3^5*x4^3*z^9 + 2*x1^21*x2^16*x3^5*x4^3*z^9 + x1^20*x2^17*x3^5*x4^3*z^9 - x1^19*x2^18*x3^5*x4^3*z^9 + x1^26*x2^10*x3^6*x4^3*z^9 + x1^25*x2^11*x3^6*x4^3*z^9 + x1^24*x2^12*x3^6*x4^3*z^9 - x1^23*x2^13*x3^6*x4^3*z^9 - x1^22*x2^14*x3^6*x4^3*z^9 + 2*x1^21*x2^15*x3^6*x4^3*z^9 - x1^20*x2^16*x3^6*x4^3*z^9 - x1^19*x2^17*x3^6*x4^3*z^9 - x1^25*x2^10*x3^7*x4^3*z^9 + x1^24*x2^11*x3^7*x4^3*z^9 - 2*x1^21*x2^14*x3^7*x4^3*z^9 + 3*x1^20*x2^15*x3^7*x4^3*z^9 + 2*x1^19*x2^16*x3^7*x4^3*z^9 - 2*x1^18*x2^17*x3^7*x4^3*z^9 + x1^24*x2^10*x3^8*x4^3*z^9 + 2*x1^22*x2^12*x3^8*x4^3*z^9 - x1^21*x2^13*x3^8*x4^3*z^9 - 2*x1^20*x2^14*x3^8*x4^3*z^9 - x1^19*x2^15*x3^8*x4^3*z^9 + x1^18*x2^16*x3^8*x4^3*z^9 + x1^17*x2^17*x3^8*x4^3*z^9 - x1^23*x2^10*x3^9*x4^3*z^9 - 3*x1^20*x2^13*x3^9*x4^3*z^9 + x1^19*x2^14*x3^9*x4^3*z^9 - 2*x1^17*x2^16*x3^9*x4^3*z^9 + x1^22*x2^10*x3^10*x4^3*z^9 + x1^20*x2^12*x3^10*x4^3*z^9 + x1^19*x2^13*x3^10*x4^3*z^9 - 2*x1^18*x2^14*x3^10*x4^3*z^9 + x1^17*x2^15*x3^10*x4^3*z^9 + x1^16*x2^16*x3^10*x4^3*z^9 + x1^20*x2^11*x3^11*x4^3*z^9 - x1^19*x2^12*x3^11*x4^3*z^9 - x1^16*x2^15*x3^11*x4^3*z^9 + 2*x1^18*x2^12*x3^12*x4^3*z^9 + x1^16*x2^14*x3^12*x4^3*z^9 + x1^16*x2^13*x3^13*x4^3*z^9 - x1^15*x2^14*x3^13*x4^3*z^9 + x1^14*x2^14*x3^14*x4^3*z^9 - 2*x1^23*x2^14*x3^4*x4^4*z^9 - x1^22*x2^15*x3^4*x4^4*z^9 - x1^21*x2^16*x3^4*x4^4*z^9 - x1^19*x2^18*x3^4*x4^4*z^9 + x1^23*x2^13*x3^5*x4^4*z^9 + x1^22*x2^14*x3^5*x4^4*z^9 + x1^21*x2^15*x3^5*x4^4*z^9 + 3*x1^20*x2^16*x3^5*x4^4*z^9 + 2*x1^19*x2^17*x3^5*x4^4*z^9 - 4*x1^22*x2^13*x3^6*x4^4*z^9 - x1^21*x2^14*x3^6*x4^4*z^9 - 3*x1^20*x2^15*x3^6*x4^4*z^9 - x1^19*x2^16*x3^6*x4^4*z^9 - x1^18*x2^17*x3^6*x4^4*z^9 - x1^23*x2^11*x3^7*x4^4*z^9 + x1^22*x2^12*x3^7*x4^4*z^9 + 3*x1^21*x2^13*x3^7*x4^4*z^9 + 5*x1^19*x2^15*x3^7*x4^4*z^9 + 2*x1^18*x2^16*x3^7*x4^4*z^9 + x1^22*x2^11*x3^8*x4^4*z^9 - 3*x1^21*x2^12*x3^8*x4^4*z^9 + x1^20*x2^13*x3^8*x4^4*z^9 - 6*x1^19*x2^14*x3^8*x4^4*z^9 - 2*x1^18*x2^15*x3^8*x4^4*z^9 - x1^22*x2^10*x3^9*x4^4*z^9 + x1^21*x2^11*x3^9*x4^4*z^9 + 2*x1^20*x2^12*x3^9*x4^4*z^9 + 3*x1^19*x2^13*x3^9*x4^4*z^9 + 7*x1^18*x2^14*x3^9*x4^4*z^9 + 2*x1^17*x2^15*x3^9*x4^4*z^9 + x1^21*x2^10*x3^10*x4^4*z^9 - 2*x1^20*x2^11*x3^10*x4^4*z^9 + x1^19*x2^12*x3^10*x4^4*z^9 - 5*x1^18*x2^13*x3^10*x4^4*z^9 - 2*x1^17*x2^14*x3^10*x4^4*z^9 + x1^19*x2^11*x3^11*x4^4*z^9 + 5*x1^17*x2^13*x3^11*x4^4*z^9 - x1^17*x2^12*x3^12*x4^4*z^9 - 3*x1^16*x2^13*x3^12*x4^4*z^9 - x1^20*x2^15*x3^5*x4^5*z^9 - x1^19*x2^16*x3^5*x4^5*z^9 - x1^18*x2^17*x3^5*x4^5*z^9 - x1^21*x2^13*x3^6*x4^5*z^9 - 2*x1^19*x2^15*x3^6*x4^5*z^9 - x1^18*x2^16*x3^6*x4^5*z^9 + x1^17*x2^17*x3^6*x4^5*z^9 + 2*x1^21*x2^12*x3^7*x4^5*z^9 + x1^19*x2^14*x3^7*x4^5*z^9 - x1^22*x2^10*x3^8*x4^5*z^9 - 2*x1^20*x2^12*x3^8*x4^5*z^9 - 5*x1^18*x2^14*x3^8*x4^5*z^9 + x1^17*x2^15*x3^8*x4^5*z^9 + 2*x1^21*x2^10*x3^9*x4^5*z^9 + 2*x1^20*x2^11*x3^9*x4^5*z^9 - x1^19*x2^12*x3^9*x4^5*z^9 + 2*x1^18*x2^13*x3^9*x4^5*z^9 + 3*x1^17*x2^14*x3^9*x4^5*z^9 - x1^16*x2^15*x3^9*x4^5*z^9 - x1^20*x2^10*x3^10*x4^5*z^9 + 2*x1^18*x2^12*x3^10*x4^5*z^9 - 4*x1^17*x2^13*x3^10*x4^5*z^9 + x1^15*x2^15*x3^10*x4^5*z^9 - x1^16*x2^12*x3^12*x4^5*z^9 + x1^15*x2^13*x3^12*x4^5*z^9 - x1^14*x2^13*x3^13*x4^5*z^9 + 2*x1^21*x2^12*x3^6*x4^6*z^9 + x1^19*x2^14*x3^6*x4^6*z^9 + 2*x1^18*x2^15*x3^6*x4^6*z^9 + x1^17*x2^16*x3^6*x4^6*z^9 + x1^20*x2^12*x3^7*x4^6*z^9 - 2*x1^19*x2^13*x3^7*x4^6*z^9 - x1^17*x2^15*x3^7*x4^6*z^9 - x1^16*x2^16*x3^7*x4^6*z^9 + x1^20*x2^11*x3^8*x4^6*z^9 + 3*x1^19*x2^12*x3^8*x4^6*z^9 + 2*x1^18*x2^13*x3^8*x4^6*z^9 - x1^17*x2^14*x3^8*x4^6*z^9 + 2*x1^16*x2^15*x3^8*x4^6*z^9 - x1^20*x2^10*x3^9*x4^6*z^9 - x1^19*x2^11*x3^9*x4^6*z^9 - 3*x1^18*x2^12*x3^9*x4^6*z^9 - 3*x1^16*x2^14*x3^9*x4^6*z^9 - x1^15*x2^15*x3^9*x4^6*z^9 - x1^19*x2^10*x3^10*x4^6*z^9 + x1^18*x2^11*x3^10*x4^6*z^9 + x1^17*x2^12*x3^10*x4^6*z^9 + 3*x1^15*x2^14*x3^10*x4^6*z^9 - 2*x1^17*x2^11*x3^11*x4^6*z^9 + 2*x1^16*x2^12*x3^11*x4^6*z^9 - 3*x1^15*x2^13*x3^11*x4^6*z^9 - 2*x1^14*x2^14*x3^11*x4^6*z^9 - x1^15*x2^12*x3^12*x4^6*z^9 + 3*x1^14*x2^13*x3^12*x4^6*z^9 - x1^13*x2^13*x3^13*x4^6*z^9 - x1^20*x2^11*x3^7*x4^7*z^9 - x1^19*x2^12*x3^7*x4^7*z^9 - x1^17*x2^14*x3^7*x4^7*z^9 + x1^18*x2^12*x3^8*x4^7*z^9 + x1^17*x2^13*x3^8*x4^7*z^9 + 2*x1^16*x2^14*x3^8*x4^7*z^9 - x1^17*x2^12*x3^9*x4^7*z^9 - x1^15*x2^14*x3^9*x4^7*z^9 + x1^17*x2^11*x3^10*x4^7*z^9 - x1^16*x2^12*x3^10*x4^7*z^9 + x1^15*x2^13*x3^10*x4^7*z^9 + x1^14*x2^14*x3^10*x4^7*z^9 - x1^16*x2^11*x3^11*x4^7*z^9 - x1^15*x2^12*x3^11*x4^7*z^9 - x1^14*x2^13*x3^11*x4^7*z^9 - 2*x1^17*x2^12*x3^8*x4^8*z^9 - 2*x1^16*x2^13*x3^8*x4^8*z^9 - x1^15*x2^14*x3^8*x4^8*z^9 - x1^17*x2^11*x3^9*x4^8*z^9 - x1^15*x2^13*x3^9*x4^8*z^9 + x1^14*x2^14*x3^9*x4^8*z^9 - x1^15*x2^12*x3^10*x4^8*z^9 + x1^15*x2^12*x3^9*x4^9*z^9 - x1^13*x2^13*x3^10*x4^9*z^9 + x1^14*x2^11*x3^11*x4^9*z^9 + x1^23*x2^16*x3*z^8 + x1^22*x2^17*x3*z^8 - x1^22*x2^16*x3^2*z^8 - x1^20*x2^18*x3^2*z^8 - x1^24*x2^13*x3^3*z^8 - x1^23*x2^14*x3^3*z^8 + x1^22*x2^15*x3^3*z^8 + x1^21*x2^16*x3^3*z^8 + x1^20*x2^17*x3^3*z^8 + 2*x1^19*x2^18*x3^3*z^8 + 2*x1^24*x2^12*x3^4*z^8 - x1^22*x2^14*x3^4*z^8 - x1^21*x2^15*x3^4*z^8 - 2*x1^19*x2^17*x3^4*z^8 - x1^18*x2^18*x3^4*z^8 - x1^24*x2^11*x3^5*z^8 + x1^20*x2^15*x3^5*z^8 + x1^19*x2^16*x3^5*z^8 + 2*x1^18*x2^17*x3^5*z^8 + x1^24*x2^10*x3^6*z^8 + 2*x1^23*x2^11*x3^6*z^8 + x1^21*x2^13*x3^6*z^8 - 2*x1^20*x2^14*x3^6*z^8 - 2*x1^18*x2^16*x3^6*z^8 - x1^23*x2^10*x3^7*z^8 + 2*x1^17*x2^16*x3^7*z^8 + x1^22*x2^10*x3^8*z^8 + 2*x1^20*x2^12*x3^8*z^8 - x1^19*x2^13*x3^8*z^8 - 2*x1^17*x2^15*x3^8*z^8 - x1^16*x2^16*x3^8*z^8 - x1^21*x2^10*x3^9*z^8 + x1^18*x2^13*x3^9*z^8 + x1^17*x2^14*x3^9*z^8 + 2*x1^16*x2^15*x3^9*z^8 - x1^19*x2^11*x3^10*z^8 - 2*x1^18*x2^12*x3^10*z^8 - 2*x1^16*x2^14*x3^10*z^8 - x1^15*x2^15*x3^10*z^8 + x1^17*x2^12*x3^11*z^8 + 2*x1^15*x2^14*x3^11*z^8 - x1^15*x2^13*x3^12*z^8 + x1^14*x2^13*x3^13*z^8 + x1^22*x2^16*x3*x4*z^8 - 2*x1^22*x2^15*x3^2*x4*z^8 - x1^21*x2^16*x3^2*x4*z^8 - 2*x1^19*x2^18*x3^2*x4*z^8 + 4*x1^21*x2^15*x3^3*x4*z^8 + 2*x1^19*x2^17*x3^3*x4*z^8 + x1^23*x2^12*x3^4*x4*z^8 + x1^22*x2^13*x3^4*x4*z^8 - 3*x1^21*x2^14*x3^4*x4*z^8 - 2*x1^20*x2^15*x3^4*x4*z^8 - 5*x1^18*x2^17*x3^4*x4*z^8 - 3*x1^23*x2^11*x3^5*x4*z^8 + x1^22*x2^12*x3^5*x4*z^8 + 3*x1^20*x2^14*x3^5*x4*z^8 + x1^19*x2^15*x3^5*x4*z^8 + 5*x1^18*x2^16*x3^5*x4*z^8 + 2*x1^17*x2^17*x3^5*x4*z^8 + 2*x1^23*x2^10*x3^6*x4*z^8 + 2*x1^22*x2^11*x3^6*x4*z^8 - x1^21*x2^12*x3^6*x4*z^8 - 2*x1^20*x2^13*x3^6*x4*z^8 - 2*x1^19*x2^14*x3^6*x4*z^8 - 2*x1^18*x2^15*x3^6*x4*z^8 - 6*x1^17*x2^16*x3^6*x4*z^8 - 3*x1^22*x2^10*x3^7*x4*z^8 + 4*x1^19*x2^13*x3^7*x4*z^8 + 6*x1^17*x2^15*x3^7*x4*z^8 + 2*x1^16*x2^16*x3^7*x4*z^8 + 2*x1^21*x2^10*x3^8*x4*z^8 + x1^20*x2^11*x3^8*x4*z^8 - x1^19*x2^12*x3^8*x4*z^8 - 2*x1^18*x2^13*x3^8*x4*z^8 - 2*x1^17*x2^14*x3^8*x4*z^8 - 6*x1^16*x2^15*x3^8*x4*z^8 - 2*x1^20*x2^10*x3^9*x4*z^8 - 2*x1^19*x2^11*x3^9*x4*z^8 + 2*x1^18*x2^12*x3^9*x4*z^8 + 6*x1^16*x2^14*x3^9*x4*z^8 + 2*x1^15*x2^15*x3^9*x4*z^8 + 2*x1^19*x2^10*x3^10*x4*z^8 + x1^18*x2^11*x3^10*x4*z^8 - x1^16*x2^13*x3^10*x4*z^8 - 6*x1^15*x2^14*x3^10*x4*z^8 + x1^17*x2^11*x3^11*x4*z^8 - 2*x1^16*x2^12*x3^11*x4*z^8 + 3*x1^15*x2^13*x3^11*x4*z^8 + 2*x1^14*x2^14*x3^11*x4*z^8 + 2*x1^15*x2^12*x3^12*x4*z^8 - 3*x1^14*x2^13*x3^12*x4*z^8 + x1^13*x2^13*x3^13*x4*z^8 + x1^19*x2^17*x3^2*x4^2*z^8 + x1^21*x2^14*x3^3*x4^2*z^8 + 2*x1^18*x2^17*x3^3*x4^2*z^8 - 3*x1^20*x2^14*x3^4*x4^2*z^8 - x1^19*x2^15*x3^4*x4^2*z^8 - x1^18*x2^16*x3^4*x4^2*z^8 - x1^17*x2^17*x3^4*x4^2*z^8 + 2*x1^20*x2^13*x3^5*x4^2*z^8 + 2*x1^19*x2^14*x3^5*x4^2*z^8 + 4*x1^17*x2^16*x3^5*x4^2*z^8 - 4*x1^19*x2^13*x3^6*x4^2*z^8 - x1^18*x2^14*x3^6*x4^2*z^8 - 3*x1^17*x2^15*x3^6*x4^2*z^8 - 2*x1^16*x2^16*x3^6*x4^2*z^8 - x1^21*x2^10*x3^7*x4^2*z^8 + 3*x1^19*x2^12*x3^7*x4^2*z^8 - x1^18*x2^13*x3^7*x4^2*z^8 + x1^17*x2^14*x3^7*x4^2*z^8 + 6*x1^16*x2^15*x3^7*x4^2*z^8 - 2*x1^18*x2^12*x3^8*x4^2*z^8 + 2*x1^17*x2^13*x3^8*x4^2*z^8 - 6*x1^16*x2^14*x3^8*x4^2*z^8 - 2*x1^15*x2^15*x3^8*x4^2*z^8 + x1^18*x2^11*x3^9*x4^2*z^8 + x1^17*x2^12*x3^9*x4^2*z^8 + 6*x1^15*x2^14*x3^9*x4^2*z^8 - x1^17*x2^11*x3^10*x4^2*z^8 + x1^16*x2^12*x3^10*x4^2*z^8 - 2*x1^15*x2^13*x3^10*x4^2*z^8 - 2*x1^14*x2^14*x3^10*x4^2*z^8 + x1^16*x2^11*x3^11*x4^2*z^8 + 2*x1^14*x2^13*x3^11*x4^2*z^8 - x1^21*x2^13*x3^3*x4^3*z^8 - 2*x1^20*x2^14*x3^3*x4^3*z^8 - 2*x1^19*x2^15*x3^3*x4^3*z^8 - x1^18*x2^16*x3^3*x4^3*z^8 + 2*x1^21*x2^12*x3^4*x4^3*z^8 + x1^20*x2^13*x3^4*x4^3*z^8 - x1^18*x2^15*x3^4*x4^3*z^8 - x1^21*x2^11*x3^5*x4^3*z^8 + x1^19*x2^13*x3^5*x4^3*z^8 - x1^18*x2^14*x3^5*x4^3*z^8 - x1^17*x2^15*x3^5*x4^3*z^8 - x1^23*x2^8*x3^6*x4^3*z^8 - x1^21*x2^10*x3^6*x4^3*z^8 + x1^20*x2^11*x3^6*x4^3*z^8 + x1^19*x2^12*x3^6*x4^3*z^8 + 2*x1^18*x2^13*x3^6*x4^3*z^8 - x1^16*x2^15*x3^6*x4^3*z^8 + x1^22*x2^8*x3^7*x4^3*z^8 - x1^20*x2^10*x3^7*x4^3*z^8 + 3*x1^18*x2^12*x3^7*x4^3*z^8 - 2*x1^17*x2^13*x3^7*x4^3*z^8 + x1^20*x2^9*x3^8*x4^3*z^8 + x1^19*x2^10*x3^8*x4^3*z^8 - 2*x1^18*x2^11*x3^8*x4^3*z^8 + 3*x1^17*x2^12*x3^8*x4^3*z^8 + 3*x1^16*x2^13*x3^8*x4^3*z^8 - 2*x1^15*x2^14*x3^8*x4^3*z^8 - x1^18*x2^10*x3^9*x4^3*z^8 - x1^17*x2^11*x3^9*x4^3*z^8 - 2*x1^16*x2^12*x3^9*x4^3*z^8 + x1^15*x2^13*x3^9*x4^3*z^8 + x1^14*x2^14*x3^9*x4^3*z^8 - x1^17*x2^10*x3^10*x4^3*z^8 - x1^15*x2^12*x3^10*x4^3*z^8 - x1^14*x2^13*x3^10*x4^3*z^8 - x1^15*x2^11*x3^11*x4^3*z^8 + x1^14*x2^12*x3^11*x4^3*z^8 - x1^13*x2^12*x3^12*x4^3*z^8 + x1^20*x2^12*x3^4*x4^4*z^8 + 2*x1^19*x2^13*x3^4*x4^4*z^8 + 2*x1^18*x2^14*x3^4*x4^4*z^8 + x1^17*x2^15*x3^4*x4^4*z^8 - 3*x1^20*x2^11*x3^5*x4^4*z^8 - x1^19*x2^12*x3^5*x4^4*z^8 - x1^18*x2^13*x3^5*x4^4*z^8 - 2*x1^16*x2^15*x3^5*x4^4*z^8 + 2*x1^20*x2^10*x3^6*x4^4*z^8 + 2*x1^19*x2^11*x3^6*x4^4*z^8 + x1^18*x2^12*x3^6*x4^4*z^8 + 5*x1^17*x2^13*x3^6*x4^4*z^8 + x1^16*x2^14*x3^6*x4^4*z^8 - x1^21*x2^8*x3^7*x4^4*z^8 - 2*x1^19*x2^10*x3^7*x4^4*z^8 - x1^18*x2^11*x3^7*x4^4*z^8 - 4*x1^17*x2^12*x3^7*x4^4*z^8 - 2*x1^16*x2^13*x3^7*x4^4*z^8 - x1^15*x2^14*x3^7*x4^4*z^8 + 2*x1^18*x2^10*x3^8*x4^4*z^8 + 8*x1^16*x2^12*x3^8*x4^4*z^8 + 2*x1^15*x2^13*x3^8*x4^4*z^8 - x1^18*x2^9*x3^9*x4^4*z^8 + x1^17*x2^10*x3^9*x4^4*z^8 - 3*x1^16*x2^11*x3^9*x4^4*z^8 - 4*x1^15*x2^12*x3^9*x4^4*z^8 - x1^14*x2^13*x3^9*x4^4*z^8 + 4*x1^15*x2^11*x3^10*x4^4*z^8 + x1^14*x2^12*x3^10*x4^4*z^8 - x1^14*x2^11*x3^11*x4^4*z^8 - x1^13*x2^12*x3^11*x4^4*z^8 + x1^12*x2^12*x3^12*x4^4*z^8 + x1^18*x2^12*x3^5*x4^5*z^8 - x1^17*x2^13*x3^5*x4^5*z^8 + x1^16*x2^14*x3^5*x4^5*z^8 + x1^15*x2^15*x3^5*x4^5*z^8 + x1^15*x2^14*x3^6*x4^5*z^8 + x1^17*x2^11*x3^7*x4^5*z^8 - 3*x1^16*x2^12*x3^7*x4^5*z^8 + x1^14*x2^14*x3^7*x4^5*z^8 - x1^17*x2^10*x3^8*x4^5*z^8 + x1^16*x2^11*x3^8*x4^5*z^8 + x1^15*x2^12*x3^8*x4^5*z^8 - 2*x1^17*x2^9*x3^9*x4^5*z^8 - 3*x1^15*x2^11*x3^9*x4^5*z^8 + x1^14*x2^12*x3^9*x4^5*z^8 - x1^15*x2^10*x3^10*x4^5*z^8 - x1^13*x2^12*x3^10*x4^5*z^8 + x1^12*x2^12*x3^11*x4^5*z^8 - x1^18*x2^10*x3^6*x4^6*z^8 - 2*x1^17*x2^11*x3^6*x4^6*z^8 - x1^15*x2^13*x3^6*x4^6*z^8 - x1^14*x2^14*x3^6*x4^6*z^8 + x1^17*x2^10*x3^7*x4^6*z^8 + x1^14*x2^13*x3^7*x4^6*z^8 - x1^17*x2^9*x3^8*x4^6*z^8 - 4*x1^16*x2^10*x3^8*x4^6*z^8 - x1^15*x2^11*x3^8*x4^6*z^8 - 2*x1^14*x2^12*x3^8*x4^6*z^8 - x1^13*x2^13*x3^8*x4^6*z^8 + x1^16*x2^9*x3^9*x4^6*z^8 + 2*x1^15*x2^10*x3^9*x4^6*z^8 + 3*x1^13*x2^12*x3^9*x4^6*z^8 + x1^14*x2^10*x3^10*x4^6*z^8 - x1^13*x2^11*x3^10*x4^6*z^8 - 2*x1^12*x2^12*x3^10*x4^6*z^8 + x1^12*x2^11*x3^11*x4^6*z^8 + x1^16*x2^10*x3^7*x4^7*z^8 - x1^16*x2^9*x3^8*x4^7*z^8 + x1^14*x2^11*x3^8*x4^7*z^8 - x1^13*x2^12*x3^8*x4^7*z^8 + x1^11*x2^11*x3^11*x4^7*z^8 + x1^13*x2^11*x3^8*x4^8*z^8 + x1^12*x2^12*x3^8*x4^8*z^8 - x1^13*x2^10*x3^9*x4^8*z^8 + x1^12*x2^11*x3^9*x4^8*z^8 + x1^21*x2^14*z^7 - x1^20*x2^14*x3*z^7 - x1^19*x2^15*x3*z^7 - x1^18*x2^16*x3*z^7 + x1^20*x2^13*x3^2*z^7 + x1^19*x2^14*x3^2*z^7 - x1^18*x2^15*x3^2*z^7 + x1^17*x2^16*x3^2*z^7 + x1^22*x2^10*x3^3*z^7 - x1^21*x2^11*x3^3*z^7 + x1^20*x2^12*x3^3*z^7 - x1^19*x2^13*x3^3*z^7 - 2*x1^17*x2^15*x3^3*z^7 - x1^16*x2^16*x3^3*z^7 - x1^21*x2^10*x3^4*z^7 - x1^20*x2^11*x3^4*z^7 + x1^19*x2^12*x3^4*z^7 + x1^18*x2^13*x3^4*z^7 + x1^17*x2^14*x3^4*z^7 + 2*x1^16*x2^15*x3^4*z^7 + 2*x1^21*x2^9*x3^5*z^7 - x1^19*x2^11*x3^5*z^7 - x1^18*x2^12*x3^5*z^7 - 2*x1^16*x2^14*x3^5*z^7 - x1^15*x2^15*x3^5*z^7 - x1^21*x2^8*x3^6*z^7 - x1^20*x2^9*x3^6*z^7 - x1^19*x2^10*x3^6*z^7 + x1^17*x2^12*x3^6*z^7 + x1^16*x2^13*x3^6*z^7 + 2*x1^15*x2^14*x3^6*z^7 + x1^20*x2^8*x3^7*z^7 - 2*x1^17*x2^11*x3^7*z^7 - 2*x1^15*x2^13*x3^7*z^7 + x1^17*x2^10*x3^8*z^7 + 2*x1^14*x2^13*x3^8*z^7 + x1^17*x2^9*x3^9*z^7 - x1^16*x2^10*x3^9*z^7 - 2*x1^14*x2^12*x3^9*z^7 - x1^13*x2^13*x3^9*z^7 + x1^15*x2^10*x3^10*z^7 + x1^14*x2^11*x3^10*z^7 + 2*x1^13*x2^12*x3^10*z^7 - x1^12*x2^12*x3^11*z^7 - x1^20*x2^13*x3*x4*z^7 - x1^17*x2^16*x3*x4*z^7 + 3*x1^19*x2^13*x3^2*x4*z^7 + x1^18*x2^14*x3^2*x4*z^7 + x1^17*x2^15*x3^2*x4*z^7 + x1^16*x2^16*x3^2*x4*z^7 - 2*x1^19*x2^12*x3^3*x4*z^7 - 2*x1^18*x2^13*x3^3*x4*z^7 - 4*x1^16*x2^15*x3^3*x4*z^7 - x1^21*x2^9*x3^4*x4*z^7 + x1^20*x2^10*x3^4*x4*z^7 - x1^19*x2^11*x3^4*x4*z^7 + 4*x1^18*x2^12*x3^4*x4*z^7 + x1^17*x2^13*x3^4*x4*z^7 + 3*x1^16*x2^14*x3^4*x4*z^7 + 2*x1^15*x2^15*x3^4*x4*z^7 + 2*x1^20*x2^9*x3^5*x4*z^7 + x1^19*x2^10*x3^5*x4*z^7 - 3*x1^18*x2^11*x3^5*x4*z^7 - x1^17*x2^12*x3^5*x4*z^7 - x1^16*x2^13*x3^5*x4*z^7 - 6*x1^15*x2^14*x3^5*x4*z^7 - 2*x1^20*x2^8*x3^6*x4*z^7 - 2*x1^19*x2^9*x3^6*x4*z^7 + 4*x1^17*x2^11*x3^6*x4*z^7 + 6*x1^15*x2^13*x3^6*x4*z^7 + 2*x1^14*x2^14*x3^6*x4*z^7 + x1^19*x2^8*x3^7*x4*z^7 + x1^18*x2^9*x3^7*x4*z^7 - 2*x1^17*x2^10*x3^7*x4*z^7 - x1^16*x2^11*x3^7*x4*z^7 - 2*x1^15*x2^12*x3^7*x4*z^7 - 6*x1^14*x2^13*x3^7*x4*z^7 - x1^17*x2^9*x3^8*x4*z^7 + 2*x1^16*x2^10*x3^8*x4*z^7 - x1^15*x2^11*x3^8*x4*z^7 + 5*x1^14*x2^12*x3^8*x4*z^7 + 2*x1^13*x2^13*x3^8*x4*z^7 + x1^15*x2^10*x3^9*x4*z^7 - 5*x1^13*x2^12*x3^9*x4*z^7 - 2*x1^14*x2^10*x3^10*x4*z^7 + x1^13*x2^11*x3^10*x4*z^7 + 2*x1^12*x2^12*x3^10*x4*z^7 - x1^12*x2^11*x3^11*x4*z^7 - 2*x1^18*x2^12*x3^3*x4^2*z^7 - x1^15*x2^15*x3^3*x4^2*z^7 + x1^18*x2^11*x3^4*x4^2*z^7 + x1^17*x2^12*x3^4*x4^2*z^7 + x1^16*x2^13*x3^4*x4^2*z^7 + 3*x1^15*x2^14*x3^4*x4^2*z^7 + x1^20*x2^8*x3^5*x4^2*z^7 - x1^19*x2^9*x3^5*x4^2*z^7 - x1^18*x2^10*x3^5*x4^2*z^7 - 3*x1^17*x2^11*x3^5*x4^2*z^7 - 2*x1^15*x2^13*x3^5*x4^2*z^7 - 2*x1^14*x2^14*x3^5*x4^2*z^7 + 2*x1^17*x2^10*x3^6*x4^2*z^7 + x1^15*x2^12*x3^6*x4^2*z^7 + 5*x1^14*x2^13*x3^6*x4^2*z^7 - 3*x1^16*x2^10*x3^7*x4^2*z^7 + 2*x1^15*x2^11*x3^7*x4^2*z^7 - 2*x1^14*x2^12*x3^7*x4^2*z^7 - 2*x1^13*x2^13*x3^7*x4^2*z^7 + 2*x1^16*x2^9*x3^8*x4^2*z^7 - x1^14*x2^11*x3^8*x4^2*z^7 + 4*x1^13*x2^12*x3^8*x4^2*z^7 - x1^15*x2^9*x3^9*x4^2*z^7 - x1^13*x2^11*x3^9*x4^2*z^7 - 2*x1^12*x2^12*x3^9*x4^2*z^7 + x1^12*x2^11*x3^10*x4^2*z^7 - x1^11*x2^11*x3^11*x4^2*z^7 + x1^19*x2^10*x3^3*x4^3*z^7 + x1^17*x2^12*x3^3*x4^3*z^7 + x1^16*x2^13*x3^3*x4^3*z^7 + 2*x1^15*x2^14*x3^3*x4^3*z^7 - x1^18*x2^10*x3^4*x4^3*z^7 - x1^17*x2^11*x3^4*x4^3*z^7 - 3*x1^16*x2^12*x3^4*x4^3*z^7 + x1^14*x2^14*x3^4*x4^3*z^7 + 2*x1^18*x2^9*x3^5*x4^3*z^7 + x1^17*x2^10*x3^5*x4^3*z^7 - x1^15*x2^12*x3^5*x4^3*z^7 - x1^18*x2^8*x3^6*x4^3*z^7 - 4*x1^15*x2^11*x3^6*x4^3*z^7 - x1^14*x2^12*x3^6*x4^3*z^7 + x1^13*x2^13*x3^6*x4^3*z^7 - x1^18*x2^7*x3^7*x4^3*z^7 + x1^17*x2^8*x3^7*x4^3*z^7 + x1^15*x2^10*x3^7*x4^3*z^7 - x1^14*x2^11*x3^7*x4^3*z^7 - x1^13*x2^12*x3^7*x4^3*z^7 - 2*x1^16*x2^8*x3^8*x4^3*z^7 - 2*x1^14*x2^10*x3^8*x4^3*z^7 - x1^13*x2^11*x3^8*x4^3*z^7 + x1^14*x2^9*x3^9*x4^3*z^7 + 2*x1^13*x2^10*x3^9*x4^3*z^7 + x1^11*x2^11*x3^10*x4^3*z^7 - x1^18*x2^9*x3^4*x4^4*z^7 - x1^16*x2^11*x3^4*x4^4*z^7 - x1^15*x2^12*x3^4*x4^4*z^7 - 2*x1^14*x2^13*x3^4*x4^4*z^7 + 2*x1^17*x2^9*x3^5*x4^4*z^7 + 2*x1^16*x2^10*x3^5*x4^4*z^7 + 4*x1^15*x2^11*x3^5*x4^4*z^7 - 2*x1^17*x2^8*x3^6*x4^4*z^7 - x1^16*x2^9*x3^6*x4^4*z^7 - 3*x1^15*x2^10*x3^6*x4^4*z^7 - 2*x1^14*x2^11*x3^6*x4^4*z^7 - 2*x1^13*x2^12*x3^6*x4^4*z^7 + x1^17*x2^7*x3^7*x4^4*z^7 - x1^15*x2^9*x3^7*x4^4*z^7 + 4*x1^14*x2^10*x3^7*x4^4*z^7 + x1^13*x2^11*x3^7*x4^4*z^7 - x1^14*x2^9*x3^8*x4^4*z^7 - 3*x1^13*x2^10*x3^8*x4^4*z^7 - 3*x1^12*x2^11*x3^8*x4^4*z^7 + x1^13*x2^9*x3^9*x4^4*z^7 + x1^12*x2^10*x3^9*x4^4*z^7 - x1^11*x2^10*x3^10*x4^4*z^7 - x1^16*x2^9*x3^5*x4^5*z^7 - x1^15*x2^10*x3^5*x4^5*z^7 + x1^14*x2^11*x3^5*x4^5*z^7 - x1^13*x2^12*x3^5*x4^5*z^7 + 2*x1^15*x2^9*x3^6*x4^5*z^7 - x1^14*x2^10*x3^6*x4^5*z^7 + x1^13*x2^11*x3^6*x4^5*z^7 + x1^13*x2^10*x3^7*x4^5*z^7 + x1^14*x2^8*x3^8*x4^5*z^7 - x1^13*x2^9*x3^8*x4^5*z^7 + 2*x1^12*x2^10*x3^8*x4^5*z^7 + 2*x1^12*x2^9*x3^9*x4^5*z^7 - x1^11*x2^10*x3^9*x4^5*z^7 + x1^10*x2^10*x3^10*x4^5*z^7 + x1^14*x2^9*x3^6*x4^6*z^7 + x1^12*x2^11*x3^6*x4^6*z^7 - 2*x1^14*x2^8*x3^7*x4^6*z^7 - x1^12*x2^10*x3^7*x4^6*z^7 + x1^13*x2^8*x3^8*x4^6*z^7 + 3*x1^11*x2^10*x3^8*x4^6*z^7 - x1^11*x2^9*x3^9*x4^6*z^7 - x1^10*x2^10*x3^9*x4^6*z^7 + x1^12*x2^8*x3^8*x4^7*z^7 - x1^10*x2^10*x3^8*x4^7*z^7 - 2*x1^18*x2^12*z^6 + x1^18*x2^11*x3*z^6 + 2*x1^15*x2^14*x3*z^6 - x1^17*x2^11*x3^2*z^6 - x1^16*x2^12*x3^2*z^6 - x1^15*x2^13*x3^2*z^6 - x1^20*x2^7*x3^3*z^6 - x1^19*x2^8*x3^3*z^6 + x1^18*x2^9*x3^3*z^6 + x1^17*x2^10*x3^3*z^6 + 2*x1^14*x2^13*x3^3*z^6 + x1^19*x2^7*x3^4*z^6 - x1^18*x2^8*x3^4*z^6 + x1^17*x2^9*x3^4*z^6 - x1^16*x2^10*x3^4*z^6 - 2*x1^14*x2^12*x3^4*z^6 - x1^13*x2^13*x3^4*z^6 - x1^18*x2^7*x3^5*z^6 + x1^16*x2^9*x3^5*z^6 + x1^15*x2^10*x3^5*z^6 + x1^14*x2^11*x3^5*z^6 + 2*x1^13*x2^12*x3^5*z^6 + x1^17*x2^7*x3^6*z^6 - x1^15*x2^9*x3^6*z^6 - 2*x1^13*x2^11*x3^6*z^6 - x1^12*x2^12*x3^6*z^6 - x1^16*x2^7*x3^7*z^6 + x1^14*x2^9*x3^7*z^6 + x1^13*x2^10*x3^7*z^6 + 2*x1^12*x2^11*x3^7*z^6 - x1^14*x2^8*x3^8*z^6 - 2*x1^12*x2^10*x3^8*z^6 - x1^12*x2^9*x3^9*z^6 + 2*x1^11*x2^10*x3^9*z^6 - x1^10*x2^10*x3^10*z^6 + 2*x1^17*x2^11*x3*x4*z^6 + x1^14*x2^14*x3*x4*z^6 - x1^17*x2^10*x3^2*x4*z^6 - x1^16*x2^11*x3^2*x4*z^6 - x1^15*x2^12*x3^2*x4*z^6 - 3*x1^14*x2^13*x3^2*x4*z^6 + 4*x1^16*x2^10*x3^3*x4*z^6 + 2*x1^14*x2^12*x3^3*x4*z^6 + 2*x1^13*x2^13*x3^3*x4*z^6 + x1^18*x2^7*x3^4*x4*z^6 - 2*x1^16*x2^9*x3^4*x4*z^6 - x1^14*x2^11*x3^4*x4*z^6 - 5*x1^13*x2^12*x3^4*x4*z^6 - x1^16*x2^8*x3^5*x4*z^6 + 3*x1^15*x2^9*x3^5*x4*z^6 - x1^14*x2^10*x3^5*x4*z^6 + 4*x1^13*x2^11*x3^5*x4*z^6 + 2*x1^12*x2^12*x3^5*x4*z^6 + x1^16*x2^7*x3^6*x4*z^6 - x1^15*x2^8*x3^6*x4*z^6 - x1^13*x2^10*x3^6*x4*z^6 - 6*x1^12*x2^11*x3^6*x4*z^6 - x1^15*x2^7*x3^7*x4*z^6 + x1^14*x2^8*x3^7*x4*z^6 - 2*x1^13*x2^9*x3^7*x4*z^6 + 4*x1^12*x2^10*x3^7*x4*z^6 + 2*x1^11*x2^11*x3^7*x4*z^6 - x1^13*x2^8*x3^8*x4*z^6 + x1^12*x2^9*x3^8*x4*z^6 - 4*x1^11*x2^10*x3^8*x4*z^6 + x1^10*x2^10*x3^9*x4*z^6 + x1^15*x2^10*x3^3*x4^2*z^6 + x1^14*x2^11*x3^3*x4^2*z^6 + x1^13*x2^12*x3^3*x4^2*z^6 - x1^16*x2^8*x3^4*x4^2*z^6 - 3*x1^15*x2^9*x3^4*x4^2*z^6 + x1^14*x2^10*x3^4*x4^2*z^6 - x1^13*x2^11*x3^4*x4^2*z^6 - 2*x1^12*x2^12*x3^4*x4^2*z^6 - x1^17*x2^6*x3^5*x4^2*z^6 - x1^16*x2^7*x3^5*x4^2*z^6 + x1^15*x2^8*x3^5*x4^2*z^6 + x1^14*x2^9*x3^5*x4^2*z^6 - x1^13*x2^10*x3^5*x4^2*z^6 + 4*x1^12*x2^11*x3^5*x4^2*z^6 - x1^15*x2^7*x3^6*x4^2*z^6 - 2*x1^14*x2^8*x3^6*x4^2*z^6 + x1^13*x2^9*x3^6*x4^2*z^6 - x1^12*x2^10*x3^6*x4^2*z^6 - 2*x1^11*x2^11*x3^6*x4^2*z^6 + x1^13*x2^8*x3^7*x4^2*z^6 + 2*x1^11*x2^10*x3^7*x4^2*z^6 - x1^11*x2^9*x3^8*x4^2*z^6 + x1^10*x2^9*x3^9*x4^2*z^6 - x1^17*x2^7*x3^3*x4^3*z^6 - x1^16*x2^8*x3^3*x4^3*z^6 - x1^15*x2^9*x3^3*x4^3*z^6 - x1^14*x2^10*x3^3*x4^3*z^6 - x1^12*x2^12*x3^3*x4^3*z^6 + x1^16*x2^7*x3^4*x4^3*z^6 - x1^15*x2^8*x3^4*x4^3*z^6 + 2*x1^13*x2^10*x3^4*x4^3*z^6 + x1^12*x2^11*x3^4*x4^3*z^6 - x1^15*x2^7*x3^5*x4^3*z^6 - 3*x1^13*x2^9*x3^5*x4^3*z^6 + x1^11*x2^11*x3^5*x4^3*z^6 + x1^15*x2^6*x3^6*x4^3*z^6 + x1^14*x2^7*x3^6*x4^3*z^6 + x1^13*x2^8*x3^6*x4^3*z^6 + 2*x1^12*x2^9*x3^6*x4^3*z^6 - 3*x1^12*x2^8*x3^7*x4^3*z^6 + x1^10*x2^10*x3^7*x4^3*z^6 + x1^11*x2^8*x3^8*x4^3*z^6 - x1^9*x2^9*x3^9*x4^3*z^6 + x1^14*x2^8*x3^4*x4^4*z^6 + x1^13*x2^9*x3^4*x4^4*z^6 + x1^11*x2^11*x3^4*x4^4*z^6 - x1^13*x2^8*x3^5*x4^4*z^6 - 3*x1^12*x2^9*x3^5*x4^4*z^6 - 2*x1^11*x2^10*x3^5*x4^4*z^6 + 2*x1^12*x2^8*x3^6*x4^4*z^6 + x1^11*x2^9*x3^6*x4^4*z^6 + x1^10*x2^10*x3^6*x4^4*z^6 + x1^9*x2^9*x3^8*x4^4*z^6 + x1^13*x2^7*x3^5*x4^5*z^6 + x1^11*x2^9*x3^5*x4^5*z^6 - x1^12*x2^7*x3^6*x4^5*z^6 - x1^10*x2^9*x3^6*x4^5*z^6 - x1^9*x2^8*x3^8*x4^5*z^6 + x1^9*x2^8*x3^7*x4^6*z^6 - x1^8*x2^8*x3^8*x4^6*z^6 + x1^15*x2^10*z^5 + x1^14*x2^11*z^5 + x1^13*x2^12*z^5 - 2*x1^15*x2^9*x3*z^5 - x1^12*x2^12*x3*z^5 + x1^15*x2^8*x3^2*z^5 + 2*x1^12*x2^11*x3^2*z^5 + x1^17*x2^5*x3^3*z^5 + x1^16*x2^6*x3^3*z^5 - 2*x1^14*x2^8*x3^3*z^5 - 2*x1^12*x2^10*x3^3*z^5 - x1^16*x2^5*x3^4*z^5 + x1^14*x2^7*x3^4*z^5 + 2*x1^11*x2^10*x3^4*z^5 + x1^14*x2^6*x3^5*z^5 - x1^13*x2^7*x3^5*z^5 - 2*x1^11*x2^9*x3^5*z^5 - x1^10*x2^10*x3^5*z^5 + x1^11*x2^8*x3^6*z^5 + 2*x1^10*x2^9*x3^6*z^5 + x1^11*x2^7*x3^7*z^5 - x1^10*x2^8*x3^7*z^5 - x1^9*x2^9*x3^7*z^5 + x1^9*x2^8*x3^8*z^5 - x1^14*x2^9*x3*x4*z^5 - x1^13*x2^10*x3*x4*z^5 - x1^12*x2^11*x3*x4*z^5 + 3*x1^14*x2^8*x3^2*x4*z^5 - x1^13*x2^9*x3^2*x4*z^5 + x1^12*x2^10*x3^2*x4*z^5 + 2*x1^11*x2^11*x3^2*x4*z^5 - x1^14*x2^7*x3^3*x4*z^5 - x1^13*x2^8*x3^3*x4*z^5 - 4*x1^11*x2^10*x3^3*x4*z^5 + 3*x1^13*x2^7*x3^4*x4*z^5 - 3*x1^12*x2^8*x3^4*x4*z^5 + 2*x1^11*x2^9*x3^4*x4*z^5 + 2*x1^10*x2^10*x3^4*x4*z^5 - x1^13*x2^6*x3^5*x4*z^5 - 4*x1^10*x2^9*x3^5*x4*z^5 + x1^12*x2^6*x3^6*x4*z^5 - 2*x1^11*x2^7*x3^6*x4*z^5 + 2*x1^10*x2^8*x3^6*x4*z^5 + 2*x1^9*x2^9*x3^6*x4*z^5 + x1^10*x2^7*x3^7*x4*z^5 - 2*x1^9*x2^8*x3^7*x4*z^5 + x1^8*x2^8*x3^8*x4*z^5 - x1^13*x2^7*x3^3*x4^2*z^5 + x1^12*x2^8*x3^3*x4^2*z^5 - x1^11*x2^9*x3^3*x4^2*z^5 - x1^10*x2^10*x3^3*x4^2*z^5 + 2*x1^12*x2^7*x3^4*x4^2*z^5 + 2*x1^10*x2^9*x3^4*x4^2*z^5 + x1^13*x2^5*x3^5*x4^2*z^5 - x1^12*x2^6*x3^5*x4^2*z^5 + x1^11*x2^7*x3^5*x4^2*z^5 - 2*x1^9*x2^9*x3^5*x4^2*z^5 + x1^11*x2^6*x3^6*x4^2*z^5 + x1^10*x2^7*x3^6*x4^2*z^5 + x1^9*x2^8*x3^6*x4^2*z^5 - x1^8*x2^8*x3^7*x4^2*z^5 + x1^14*x2^5*x3^3*x4^3*z^5 + 2*x1^12*x2^7*x3^3*x4^3*z^5 + x1^11*x2^8*x3^3*x4^3*z^5 + x1^10*x2^9*x3^3*x4^3*z^5 - x1^13*x2^5*x3^4*x4^3*z^5 + x1^10*x2^8*x3^4*x4^3*z^5 - x1^9*x2^9*x3^4*x4^3*z^5 + x1^10*x2^7*x3^5*x4^3*z^5 + x1^9*x2^8*x3^5*x4^3*z^5 - x1^10*x2^6*x3^6*x4^3*z^5 - x1^9*x2^7*x3^6*x4^3*z^5 + x1^8*x2^8*x3^6*x4^3*z^5 + x1^8*x2^7*x3^7*x4^3*z^5 + x1^12*x2^5*x3^4*x4^4*z^5 - x1^10*x2^7*x3^4*x4^4*z^5 - x1^9*x2^8*x3^4*x4^4*z^5 - x1^10*x2^6*x3^5*x4^4*z^5 - x1^9*x2^7*x3^5*x4^4*z^5 + 2*x1^8*x2^8*x3^5*x4^4*z^5 - x1^8*x2^7*x3^6*x4^4*z^5 - x1^7*x2^7*x3^7*x4^4*z^5 + x1^9*x2^6*x3^5*x4^5*z^5 - x1^8*x2^7*x3^5*x4^5*z^5 + x1^8*x2^6*x3^6*x4^5*z^5 - x1^13*x2^7*z^4 + x1^12*x2^8*z^4 - x1^11*x2^9*z^4 - x1^10*x2^10*z^4 + x1^12*x2^7*x3*z^4 + x1^11*x2^8*x3*z^4 + x1^10*x2^9*x3*z^4 - 2*x1^12*x2^6*x3^2*z^4 - x1^9*x2^9*x3^2*z^4 - x1^13*x2^4*x3^3*z^4 + x1^10*x2^7*x3^3*z^4 + 2*x1^9*x2^8*x3^3*z^4 + x1^12*x2^4*x3^4*z^4 + x1^10*x2^6*x3^4*z^4 - 2*x1^9*x2^7*x3^4*z^4 - x1^9*x2^6*x3^5*z^4 + 2*x1^8*x2^7*x3^5*z^4 - x1^7*x2^7*x3^6*z^4 + x1^12*x2^6*x3*x4*z^4 - x1^11*x2^7*x3*x4*z^4 + x1^10*x2^8*x3*x4*z^4 + x1^9*x2^9*x3*x4*z^4 - x1^11*x2^6*x3^2*x4*z^4 - 2*x1^9*x2^8*x3^2*x4*z^4 + x1^12*x2^4*x3^3*x4*z^4 + 3*x1^11*x2^5*x3^3*x4*z^4 + x1^9*x2^7*x3^3*x4*z^4 + 2*x1^8*x2^8*x3^3*x4*z^4 - x1^10*x2^5*x3^4*x4*z^4 + x1^9*x2^6*x3^4*x4*z^4 - 2*x1^8*x2^7*x3^4*x4*z^4 - x1^9*x2^5*x3^5*x4*z^4 + x1^8*x2^6*x3^5*x4*z^4 + x1^7*x2^7*x3^5*x4*z^4 - x1^7*x2^6*x3^6*x4*z^4 - x1^11*x2^5*x3^2*x4^2*z^4 - x1^9*x2^6*x3^3*x4^2*z^4 + x1^8*x2^7*x3^3*x4^2*z^4 - x1^9*x2^5*x3^4*x4^2*z^4 - x1^8*x2^6*x3^4*x4^2*z^4 - x1^7*x2^7*x3^4*x4^2*z^4 - x1^8*x2^5*x3^5*x4^2*z^4 + x1^7*x2^6*x3^5*x4^2*z^4 - x1^6*x2^6*x3^6*x4^2*z^4 - x1^8*x2^6*x3^3*x4^3*z^4 - x1^7*x2^7*x3^3*x4^3*z^4 + x1^9*x2^4*x3^4*x4^3*z^4 + x1^8*x2^5*x3^4*x4^3*z^4 - x1^7*x2^6*x3^4*x4^3*z^4 + x1^7*x2^5*x3^5*x4^3*z^4 - x1^6*x2^6*x3^5*x4^3*z^4 - x1^8*x2^4*x3^4*x4^4*z^4 + x1^6*x2^6*x3^4*x4^4*z^4 + x1^6*x2^5*x3^5*x4^4*z^4 + x1^11*x2^4*z^3 + x1^10*x2^5*z^3 - x1^9*x2^6*z^3 + x1^8*x2^7*z^3 - x1^10*x2^4*x3*z^3 + x1^9*x2^5*x3*z^3 - x1^8*x2^6*x3*z^3 - x1^7*x2^7*x3*z^3 - x1^10*x2^3*x3^2*z^3 + x1^8*x2^5*x3^2*z^3 + x1^7*x2^6*x3^2*z^3 + 2*x1^8*x2^4*x3^3*z^3 - x1^7*x2^5*x3^3*z^3 - x1^6*x2^6*x3^3*z^3 - x1^7*x2^4*x3^4*z^3 + x1^6*x2^5*x3^4*z^3 - x1^9*x2^4*x3*x4*z^3 + x1^8*x2^5*x3*x4*z^3 - x1^7*x2^6*x3*x4*z^3 + x1^9*x2^3*x3^2*x4*z^3 - x1^8*x2^4*x3^2*x4*z^3 + x1^7*x2^5*x3^2*x4*z^3 + x1^6*x2^6*x3^2*x4*z^3 - x1^8*x2^3*x3^3*x4*z^3 - x1^7*x2^4*x3^3*x4*z^3 - 3*x1^6*x2^5*x3^3*x4*z^3 + x1^5*x2^5*x3^4*x4*z^3 + x1^8*x2^3*x3^2*x4^2*z^3 + x1^7*x2^4*x3^2*x4^2*z^3 + x1^5*x2^4*x3^4*x4^2*z^3 - x1^6*x2^3*x3^3*x4^3*z^3 + x1^5*x2^4*x3^3*x4^3*z^3 - x1^8*x2^2*z^2 - x1^7*x2^3*z^2 - x1^6*x2^4*z^2 - x1^6*x2^3*x3*z^2 + x1^5*x2^4*x3*z^2 + x1^6*x2^2*x3^2*z^2 - x1^4*x2^4*x3^2*z^2 + x1^6*x2^2*x3*x4*z^2 + x1^5*x2^3*x3*x4*z^2 + x1^5*x2^2*x3^2*x4*z^2 - x1^4*x2^3*x3^2*x4*z^2 + 2*x1^3*x2^3*x3^3*x4*z^2 - x1^4*x2^2*x3^2*x4^2*z^2 + 2*x1^3*x2^2*z - x1^2*x2*x3*x4*z - 1))

In [32]:
(x1^16*x2^10*x3^6*z^8 + x1^14*x2^9*x3^5*z^7 + 2*x1^13*x2^9*x3^6*z^7 + 2*x1^13*x2^7*x3^4*z^6 + x1^12*x2^8*x3^4*z^6 + x1^12*x2^7*x3^5*z^6 + x1^11*x2^8*x3^5*z^6 + x1^10*x2^8*x3^6*z^6 + x1^11*x2^6*x3^3*z^5 + x1^10*x2^7*x3^3*z^5 - 4*x1^10*x2^6*x3^4*z^5 - 2*x1^9*x2^7*x3^4*z^5 - x1^9*x2^6*x3^5*z^5 - 2*x1^8*x2^7*x3^5*z^5 + x1^10*x2^4*x3^2*z^4 - 2*x1^9*x2^5*x3^2*z^4 - x1^9*x2^4*x3^3*z^4 - 2*x1^8*x2^5*x3^3*z^4 - x1^7*x2^6*x3^3*z^4 - 2*x1^7*x2^5*x3^4*z^4 + x1^6*x2^6*x3^4*z^4 - 2*x1^8*x2^3*x3*z^3 - x1^7*x2^4*x3*z^3 - 2*x1^7*x2^3*x3^2*z^3 - 4*x1^6*x2^4*x3^2*z^3 + x1^6*x2^3*x3^3*z^3 + x1^5*x2^4*x3^3*z^3 + x1^6*x2^2*z^2 + x1^5*x2^2*x3*z^2 + x1^4*x2^3*x3*z^2 + x1^4*x2^2*x3^2*z^2 + 2*x1^3*x2^3*x3^2*z^2 + 2*x1^3*x2*z + x1^2*x2*x3*z + 1) == x1^16*x2^10*x3^6*z^8 + x1^14*x2^9*x3^5*z^7 + 2*x1^13*x2^9*x3^6*z^7 + 2*x1^13*x2^7*x3^4*z^6 + x1^12*x2^8*x3^4*z^6 + x1^12*x2^7*x3^5*z^6 + x1^11*x2^8*x3^5*z^6 + x1^10*x2^8*x3^6*z^6 + x1^11*x2^6*x3^3*z^5 + x1^10*x2^7*x3^3*z^5 - 4*x1^10*x2^6*x3^4*z^5 - 2*x1^9*x2^7*x3^4*z^5 - x1^9*x2^6*x3^5*z^5 - 2*x1^8*x2^7*x3^5*z^5 + x1^10*x2^4*x3^2*z^4 - 2*x1^9*x2^5*x3^2*z^4 - x1^9*x2^4*x3^3*z^4 - 2*x1^8*x2^5*x3^3*z^4 - x1^7*x2^6*x3^3*z^4 - 2*x1^7*x2^5*x3^4*z^4 + x1^6*x2^6*x3^4*z^4 - 2*x1^8*x2^3*x3*z^3 - x1^7*x2^4*x3*z^3 - 2*x1^7*x2^3*x3^2*z^3 - 4*x1^6*x2^4*x3^2*z^3 + x1^6*x2^3*x3^3*z^3 + x1^5*x2^4*x3^3*z^3 + x1^6*x2^2*z^2 + x1^5*x2^2*x3*z^2 + x1^4*x2^3*x3*z^2 + x1^4*x2^2*x3^2*z^2 + 2*x1^3*x2^3*x3^2*z^2 + 2*x1^3*x2*z + x1^2*x2*x3*z + 1


True

P2_2var: correct
P3_3var: correct
P4_3var: correct 
P4_4var: correct 
P5_5var: correct
P11_2var: correct
P21_3var: correct
P22_4var: correct
P31_4var: correct
P32_5var: correct
P41_5var: ~4 days to compute numerator
P111_3var: correct
P211_4var: correct
P1111_4var: correct
P2111_5var: 1 to 3 days to compute numerator
P221_5var: 3 days to 2 weeks to compute numerator
P11111_5var: 1 week to 2 weeks to compute numerator
P311_5var: Unknown since I still need to compute P311_4var

In [46]:
out/den_guess()

(-x1^52*x2^38*x3^20*z^22 + 2*x1^49*x2^36*x3^20*z^21 - x1^48*x2^34*x3^18*z^20 + x1^46*x2^36*x3^18*z^20 + x1^47*x2^34*x3^19*z^20 - x1^46*x2^35*x3^19*z^20 - x1^46*x2^34*x3^20*z^20 - x1^45*x2^35*x3^20*z^20 - x1^44*x2^36*x3^20*z^20 + x1^46*x2^33*x3^16*z^19 - x1^45*x2^34*x3^16*z^19 - x1^46*x2^32*x3^17*z^19 - x1^45*x2^33*x3^17*z^19 + 2*x1^44*x2^34*x3^17*z^19 + x1^45*x2^32*x3^18*z^19 + x1^44*x2^33*x3^18*z^19 - x1^42*x2^35*x3^18*z^19 - x1^45*x2^31*x3^19*z^19 - x1^44*x2^32*x3^19*z^19 + x1^43*x2^33*x3^19*z^19 - x1^42*x2^34*x3^19*z^19 + x1^44*x2^31*x3^20*z^19 - x1^43*x2^32*x3^20*z^19 + x1^42*x2^33*x3^20*z^19 + x1^41*x2^34*x3^20*z^19 - x1^45*x2^31*x3^14*z^18 + 2*x1^44*x2^31*x3^15*z^18 - x1^43*x2^32*x3^15*z^18 - 2*x1^43*x2^31*x3^16*z^18 + x1^42*x2^32*x3^16*z^18 + x1^40*x2^34*x3^16*z^18 + 2*x1^43*x2^30*x3^17*z^18 + x1^42*x2^31*x3^17*z^18 - x1^39*x2^34*x3^17*z^18 - x1^43*x2^29*x3^18*z^18 - 2*x1^40*x2^32*x3^18*z^18 + x1^42*x2^29*x3^19*z^18 + x1^41*x2^30*x3^19*z^18 + x1^40*x2^31*x3^19*z^18 - x1^42*x2^28

In [3]:
factor((1-z*x1**5)*(1 - z**6*x1**15*x2**15)*(1 - z*x1**4*x2)*(1 - z*x1**3*x2**2)*(1 - z*x1**3*x2*x3)*(1 - z**3*x1**9*x2**3*x3**3)*(1 - z**3*x1**6*x2**6*x3**3)*(1 - z**2*x1**4*x2**3*x3**3)*(1 - z**9*x1**15*x2**15*x3**15))

(-1) * (x1^2*x2^2*x3*z - 1) * (x1^3*x2^2*z - 1) * (x1^4*x2*z - 1) * (x1^5*z - 1) * (x1^3*x2*x3*z - 1)^2 * (x1^4*x2^3*x3^3*z^2 - 1) * (x1^4*x2^4*x3^2*z^2 + x1^2*x2^2*x3*z + 1) * (x1^6*x2^2*x3^2*z^2 + x1^3*x2*x3*z + 1) * (x1^5*x2^5*z^2 - 1) * (x1^5*x2^5*x3^5*z^3 - 1) * (x1^10*x2^10*z^4 + x1^5*x2^5*z^2 + 1) * (x1^10*x2^10*x3^10*z^6 + x1^5*x2^5*x3^5*z^3 + 1)

(-1) * (x1^3*x2*x3*z - 1)

(-1) * (x1^3*x2^2*z - 1) * (x1^4*x2*z - 1) * (x1^5*z - 1) * (x1^3*x2*x3*z - 1) * (x1^4*x2^3*x3^3*z^2 - 1) * (x1^4*x2^4*x3^2*z^2 + x1^2*x2^2*x3*z + 1) * (x1^6*x2^2*x3^2*z^2 + x1^3*x2*x3*z + 1) * (x1^5*x2^5*z^2 - 1) * (x1^5*x2^5*x3^5*z^3 - 1) * (x1^10*x2^10*z^4 + x1^5*x2^5*z^2 + 1) * (x1^10*x2^10*x3^10*z^6 + x1^5*x2^5*x3^5*z^3 + 1)

In [ ]:
(1 - x1^3*x2^2*z)*(x1^4*x2*z - 1)*(x1^5*z - 1)*(x1^3*x2*x3*z - 1)
    *(x1^4*x2^3*x3^3*z^2 - 1)
    *(x1^4*x2^4*x3^2*z^2 + x1^2*x2^2*x3*z + 1)
    *(x1^6*x2^2*x3^2*z^2 + x1^3*x2*x3*z + 1)
    *(x1^5*x2^5*z^2 - 1)
    *(x1^5*x2^5*x3^5*z^3 - 1)
    *(x1^10*x2^10*z^4 + x1^5*x2^5*z^2 + 1)
    *(x1^10*x2^10*x3^10*z^6 + x1^5*x2^5*x3^5*z^3 + 1)
    *(1 - z**4*x1**5*x2**5*x3**5*x4**5)
    *(1 - z**2*x1**4*x2**4*x3*x4)
    *(1- z**2*x1**3*x2**3*x3**3*x4)
    *(1 - z*x1**2*x2*x3*x4)
    *(1 - z^3 * x1^4 * x2^4 * x3^4 * x4^3)
    *(1 - z^6 * x1^9 * x2^9 * x3^6 * x4^6)
    *(1 - z*x1^2*x2*x3*x4)
    *(1 - z**3*x1**6*x2**3*x3**3*x4**3)

Execution Loop that tracks ETA

In [4]:
# ==========================================
# CELL 2: Execution Loop with ETA
# ==========================================
import time
import numpy as np

out = 0
target_degree = 45 # Matches the upper bound of your loop
time_history = []
degree_history = []

print("Warming up native denominator expansion... (takes a few seconds)")
get_den_expanded()
print("Expansion complete! Starting degrees...\n")

for d in range(0, target_degree):
    start_time = time.time()
    
    # Calculate the numerator coefficient for the current degree
    CC = calc_num([4,1], d)
    
    elapsed = time.time() - start_time
    time_history.append(elapsed)
    degree_history.append(d)
    
    if d < target_degree and len(time_history) >= 8:
        try:
            # Filter out early degrees to avoid warm-up noise
            mask = np.array(degree_history) > 6
            x_data = np.array(degree_history)[mask]
            y_data = np.array(time_history)[mask]
            
            # Fit Log-Quadratic: log(time) = A*d^2 + B*d + C
            # This perfectly models a multiplier that decays by a constant factor!
            coeffs = np.polyfit(x_data, np.log(y_data), deg=2)
            A, B, C = coeffs[0], coeffs[1], coeffs[2]
            
            # Predict remaining times using the extrapolated parabola
            remaining_degrees = np.arange(d + 1, target_degree)
            predicted_times = np.exp(A * (remaining_degrees**2) + B * remaining_degrees + C)
            
            projected_remaining = np.sum(predicted_times)
            
            # The decay factor (e.g. your ~0.967) is exactly e^(2A)
            decay_factor = np.exp(2 * A)
            
            eta_str = f" | ETA to d={target_degree}: ~{projected_remaining/60:.1f} min (Decay: {decay_factor:.3f})"
        except Exception:
            eta_str = ""
    else:
        eta_str = ""

    if CC:
        CC_list = list(CC)
        if len(CC_list) > 6:
            front = CC_list[:3]
            back = CC_list[-3:]
            print(f"{d:02d} {len(CC_list):4d} FRONT: {front} BACK: {back} | Time: {elapsed:.2f}s{eta_str}")
        else:
            print(f"{d:02d} {len(CC_list):4d} {CC_list} | Time: {elapsed:.2f}s{eta_str}")
    else:
        print(f"{d:02d}    0 [] | Time: {elapsed:.2f}s{eta_str}")
    
    out += z**d * CC

Warming up native denominator expansion... (takes a few seconds)
Expansion complete! Starting degrees...

00    1 [(-1, 1)] | Time: 0.01s
01    5 [(1, x1^4*x2), (1, x1^3*x2^2), (-1, x1^3*x2*x3), (-1, x1^2*x2^2*x3), (1, x1^2*x2*x3*x4)] | Time: 0.01s
02   14 FRONT: [(-1, x1^8*x2^2), (-1, x1^6*x2^4), (-1, x1^5*x2^5)] BACK: [(-1, x1^3*x2^3*x3^3*x4), (-1, x1^4*x2^2*x3^2*x4^2), (1, x1^3*x2^3*x3^2*x4^2)] | Time: 0.02s
03   33 FRONT: [(1, x1^10*x2^5), (1, x1^9*x2^6), (1, x1^8*x2^7)] BACK: [(1, x1^6*x2^3*x3^3*x4^3), (-1, x1^5*x2^4*x3^3*x4^3), (1, x1^4*x2^4*x3^4*x4^3)] | Time: 0.04s
04   62 FRONT: [(-1, x1^13*x2^7), (-1, x1^12*x2^8), (-1, x1^10*x2^10)] BACK: [(2, x1^7*x2^5*x3^4*x4^4), (-1, x1^6*x2^6*x3^4*x4^4), (-2, x1^6*x2^5*x3^5*x4^4)] | Time: 0.09s
05   90 FRONT: [(1, x1^15*x2^10), (1, x1^14*x2^11), (-1, x1^16*x2^8*x3)] BACK: [(2, x1^8*x2^7*x3^5*x4^5), (2, x1^8*x2^6*x3^6*x4^5), (-2, x1^7*x2^7*x3^6*x4^5)] | Time: 0.16s
06  143 FRONT: [(-1, x1^18*x2^12), (1, x1^18*x2^11*x3), (1, x1^17*x2^12*x3)

/tmp/ipykernel_568573/2025407535.py:35: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(x_data, np.log(y_data), deg=Integer(2))


07  213 FRONT: [(-1, x1^21*x2^13*x3), (1, x1^19*x2^15*x3), (1, x1^21*x2^12*x3^2)] BACK: [(-3, x1^11*x2^9*x3^8*x4^7), (2, x1^10*x2^10*x3^8*x4^7), (1, x1^10*x2^9*x3^9*x4^7)] | Time: 0.50s | ETA to d=45: ~0.1 min (Decay: 0.991)


/tmp/ipykernel_568573/2025407535.py:35: RankWarning: Polyfit may be poorly conditioned
  coeffs = np.polyfit(x_data, np.log(y_data), deg=Integer(2))


08  262 FRONT: [(-1, x1^24*x2^14*x3^2), (2, x1^22*x2^16*x3^2), (1, x1^24*x2^13*x3^3)] BACK: [(-3, x1^12*x2^11*x3^9*x4^8), (-1, x1^12*x2^10*x3^10*x4^8), (1, x1^11*x2^11*x3^10*x4^8)] | Time: 0.93s | ETA to d=45: ~72828025717213133022490588610560.0 min (Decay: 1.087)
09  340 FRONT: [(-1, x1^27*x2^15*x3^3), (2, x1^25*x2^17*x3^3), (-1, x1^23*x2^19*x3^3)] BACK: [(-2, x1^13*x2^13*x3^10*x4^9), (1, x1^14*x2^11*x3^11*x4^9), (-1, x1^13*x2^12*x3^11*x4^9)] | Time: 1.51s | ETA to d=45: ~0.3 min (Decay: 0.884)
10  423 FRONT: [(2, x1^28*x2^18*x3^4), (-1, x1^27*x2^19*x3^4), (-1, x1^26*x2^20*x3^4)] BACK: [(3, x1^15*x2^14*x3^11*x4^10), (1, x1^15*x2^13*x3^12*x4^10), (-1, x1^14*x2^14*x3^12*x4^10)] | Time: 2.34s | ETA to d=45: ~0.7 min (Decay: 0.917)
11  475 FRONT: [(1, x1^31*x2^19*x3^5), (-1, x1^29*x2^21*x3^5), (1, x1^29*x2^20*x3^6)] BACK: [(1, x1^16*x2^16*x3^12*x4^11), (-1, x1^17*x2^14*x3^13*x4^11), (1, x1^16*x2^15*x3^13*x4^11)] | Time: 3.69s | ETA to d=45: ~2.8 min (Decay: 0.950)
12  570 FRONT: [(-1, x1^

KeyboardInterrupt: 

In [ ]:
out

In [1]:
factor(out)

NameError: name 'out' is not defined

In [ ]:
[110.46/78.22, 78.22/55.43,55.43/39.04,39.04/26.97]

In [78]:
[1.55868526510481/1.59812796354455, 1.59812796354455/1.63600644771308,1.63600644771308/1.67160660154934,1.72510264489640/1.78203164880041,1.66867769967344/1.72510264489640]

[0.975319436653709,
 0.976846983566917,
 0.978703031082036,
 0.968053876067615,
 0.967291833103445]

In [81]:
total_time = 0
time = 404 
increase = 1.55
for i in range(1,25):
    time = time*increase
    total_time = total_time + time
    increase = increase*0.98
print(total_time/60/60/24)
    

7.49244936610582


In [ ]:
1.4 days. Great.

In [ ]:
22281652504003439123993443243200536813169242413604197979859530875317829057631113616848926274896087107192657470667218147009430597105415618560.0/60/24

In [9]:
((x1^4*x2^4*x3^4*z^3 - 1)*((x1^2*x2*x3*z)^3 - 1)*((x1^3*x2^3*x3^2*z^2)^2 - 1)*(x1^4*z - 1)*(x1^3*x2*z - 1)*((x1^2*x2^2*z)^2 - 1)*((x1^2*x2^2*z)^3 - 1)*((x1^2*x2*x3*z)^2 - 1)*((x1*x2*x3*x4*z)^2 - 1)) == ((x1^4*x2^4*x3^4*z^3 - 1)*(x1^4*x2^4*z^2 + x1^2*x2^2*z + 1)*(x1^4*x2^2*x3^2*z^2 + x1^2*x2*x3*z + 1)*(x1^3*x2^3*x3^2*z^2 + 1)*(x1^3*x2^3*x3^2*z^2 - 1)*(x1^4*z - 1)*(x1^3*x2*z - 1)*(x1^2*x2^2*z + 1)*(x1^2*x2^2*z - 1)^2*(x1^2*x2*x3*z + 1)*(x1^2*x2*x3*z - 1)^2*(x1*x2*x3*x4*z + 1)*(x1*x2*x3*x4*z - 1))

True

In [10]:
factor(((x1^2*z - 1)
*((x1*x2*z)^2 - 1)))

(x1*x2*z - 1) * (x1*x2*z + 1) * (x1^2*z - 1)

In [16]:
factor((((x1^3*x2^3*z^2)^2 - 1)
*(x1^3*z - 1)
*(x1^2*x2*z - 1)
*((x1*x2*x3*z)^2 - 1)))

(x1*x2*x3*z - 1) * (x1*x2*x3*z + 1) * (x1^2*x2*z - 1) * (x1^3*z - 1) * (x1^3*x2^3*z^2 - 1) * (x1^3*x2^3*z^2 + 1)

In [17]:
factor(((x1^4*x2^4*x3^4*z^3 - 1)
*((x1^2*x2*x3*z)^3 - 1)
*((x1^3*x2^3*x3^2*z^2)^2 - 1)
*(x1^4*z - 1)
*(x1^3*x2*z - 1)
*((x1^2*x2^2*z)^2 - 1)
*((x1^2*x2^2*z)^3 - 1)
*((x1^2*x2*x3*z)^2 - 1)
*((x1*x2*x3*x4*z)^2 - 1)))

(x1*x2*x3*x4*z - 1) * (x1*x2*x3*x4*z + 1) * (x1^2*x2*x3*z + 1) * (x1^2*x2^2*z + 1) * (x1^3*x2*z - 1) * (x1^4*z - 1) * (x1^2*x2*x3*z - 1)^2 * (x1^2*x2^2*z - 1)^2 * (x1^3*x2^3*x3^2*z^2 - 1) * (x1^3*x2^3*x3^2*z^2 + 1) * (x1^4*x2^2*x3^2*z^2 + x1^2*x2*x3*z + 1) * (x1^4*x2^4*z^2 + x1^2*x2^2*z + 1) * (x1^4*x2^4*x3^4*z^3 - 1)

In [7]:
# 1. Setup Environment
Sym = SymmetricFunctions(QQ)
Sym.inject_shorthands(verbose=False)
R = PolynomialRing(QQ, 'a, b, x1, x2, x3, x4, x5, z').fraction_field()
R.inject_variables()
x = R.gens()[2:-1]
a = R.gens()[0]
b = R.gens()[1]
z = R.gens()[-1]

# 2. Define Core Functions
def normalize_rational_function(Q):
    S = PolynomialRing(PolynomialRing(QQ, 'a,b'), 'x1,x2,x3,x4,x5,z')
    K = R.fraction_field()
    den = []
    factors = Q.denominator().factor()
    scalar = factors.unit()
    for (factor, exp) in factors:
        c = S(factor).constant_coefficient()
        den.append((K(factor) / c, exp))
        scalar *= c**exp
    den = Factorization(den)
    num = Q.numerator() / scalar
    return (num, den)

def CT(f, g):
    Q = f.subs({z: z / (a * b)}) * g.subs({z: a * b})
    num, den = normalize_rational_function(Q)
    PTa = MacMahonOmega(a, num, den)
    CTa = prod(PTa).subs(b=0)
    return CTa

def P(k):
    return R.one() / R.prod((1 - z * xi**k) for xi in x)

# 3. The Step-by-Step Execution
print("Computing P(1,1)...")
F_11 = CT(P(1), P(1))

print("Computing P(1,1,1)...")
F_111 = CT(P(1), F_11)

print("Computing P(1,1,1,1)...")
F_1111 = CT(P(1), F_111)

print("Computing P(2,1,1,1) [Final Step]...")
F_2111 = CT(P(2), F_1111)

# 4. Extract the Exact Denominator
print("Normalizing final rational function...")
final_num, final_den = normalize_rational_function(F_2111)

print("\n--- EXACT SYMBOLIC DENOMINATOR ---")
print(final_den)

Defining a, b, x1, x2, x3, x4, x5, z
Computing P(1,1)...


KeyboardInterrupt: 